# Viettel AI Race — `predict-v2` Run All (Kaggle)

Pipeline **precision-first**, thay cho pipeline sinh văn bản của lần nộp 01 (14.4255).

```text
GLiNER spans (ngưỡng riêng theo type)
  → Qwen corrector: TRIỆU_CHỨNG → CHẨN_ĐOÁN   (GPU)
  → Qwen consensus additions cho type không có candidate  (GPU, cần 2 teacher)
  → trim generic prefix + loại header
  → exact-alias linking (ICD-10 tiếng Việt TT06 + RxNorm), chỉ emit khi khớp duy nhất
  → assertions rỗng
  → validate → output.zip
```

**Vì sao đổi cách làm.** Scorer của BTC đếm mỗi concept thừa **hai lần** vào mẫu số
của cả ba thành phần. Nên precision đáng giá hơn recall, và candidate thừa còn đắt
hơn nữa: một concept sai mang 3 mã tốn `2×(3+1)=8` đơn vị mẫu số thay vì 2.

**Trước khi Run All:**

1. Settings → Accelerator: **GPU T4 x2** (khuyến nghị) hoặc **P100**.
2. Settings → Internet: **On** — để tải weights và RxNorm.
3. Cần khoảng **12 GB trống** trên `/kaggle/working` cho weights. Nếu đã chạy dở
   lần trước, hãy **Run → Factory reset** trước khi Run All; notebook có cell
   báo cáo và dọn dung lượng ở mục 1.
4. Nếu chưa attach weights: đặt HF token trong Add-ons → Secrets với tên
   `HF_TOKEN`.

**Không cần attach gì cả.** Notebook tự chứa: package `medical_coder`, bảng
ICD-10 tiếng Việt và bản test Vòng 1 đều được nhúng sẵn. Chỉ cần tải lên đúng
một tệp `.ipynb` này.

Attach Dataset input vẫn được và **luôn được ưu tiên** hơn bản nhúng — bắt buộc
làm vậy khi chạy trên private test của BTC.

## 1. Kiểm tra GPU và môi trường

In [ ]:
!nvidia-smi || echo "Không thấy GPU — pipeline vẫn chạy được trên CPU nhưng KHÔNG có bước corrector."

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
IS_KAGGLE = Path("/kaggle").exists()
print("kaggle:", IS_KAGGLE, "| python:", sys.version.split()[0])

try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print("  ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
except ImportError:
    print("torch chưa được cài")

### P100 (`sm_60`)

Wheel torch mặc định của Kaggle có thể không chứa kiến trúc `sm_60`. Cell dưới chỉ
cài lại torch khi phát hiện P100 **và** arch hiện tại thiếu `sm_60`. Sau khi cài
lại phải **Restart Session** rồi Run All lần nữa — không thể tráo binary torch
trong kernel đã import nó.

In [ ]:
NEEDS_RESTART = False
try:
    import torch
    if torch.cuda.is_available():
        caps = {torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())}
        arches = torch.cuda.get_arch_list()
        if (6, 0) in caps and not any("sm_60" in a for a in arches):
            print("P100 nhưng torch thiếu sm_60 — cài lại torch CUDA 12.6")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "torch==2.10.0", "--index-url",
                            "https://download.pytorch.org/whl/cu126"], check=False)
            NEEDS_RESTART = True
        else:
            print("torch arch OK:", [a for a in arches if a.startswith("sm_")])
except Exception as exc:
    print("bỏ qua kiểm tra arch:", exc)

if NEEDS_RESTART:
    print("\n>>> HÃY CHỌN 'Restart Session' RỒI RUN ALL LẠI <<<")

### Dung lượng đĩa

`/kaggle/working` chỉ có khoảng 20 GB và cũng chính là quota output. Weights là
thứ ngốn nhiều nhất, nên cell này báo cáo chỗ trống trước rồi mới dọn cache —
hết đĩa giữa chừng sẽ nổ ra `OSError: [Errno 28]` ở một cell chẳng liên quan gì,
rất khó lần ra nguyên nhân.

In [ ]:
import shutil as _sh

# Đặt True để xoá SẠCH /kaggle/working. Notebook tự tạo lại được mọi thứ nó cần,
# nhưng nếu bạn có tệp riêng ở đó thì sẽ mất.
PURGE_ALL = False

# Tên do các phiên bản notebook trước tạo ra, đều tái tạo được nên xoá an toàn.
REGENERABLE = [
    "models", "hf", "medical_coder_src", "terminology",
    "input_embedded", "output", "output_smoke", "cache",
    "embedded_viettel_ai_race", "viettel_ai_race", "VAIR-NEXTLEVEL",
    "output.zip", "output_v2.zip", "rxnorm.zip",
]

def report_disk():
    for path in ("/kaggle/working", "/tmp", "/"):
        if Path(path).exists():
            usage = _sh.disk_usage(path)
            print(f"  {path:18s} trống {usage.free / 2**30:6.1f} GB "
                  f"/ tổng {usage.total / 2**30:6.1f} GB")

def entry_size(path):
    if path.is_file():
        return path.stat().st_size
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

def show_largest(top=10):
    """Liệt kê thứ đang chiếm chỗ, để cái gì không nằm trong danh sách xoá vẫn nhìn thấy."""
    entries = []
    for path in WORK.iterdir():
        try:
            entries.append((entry_size(path), path))
        except OSError:
            continue
    entries.sort(reverse=True)
    if not entries:
        print("  (trống)")
    for size, path in entries[:top]:
        mark = "  [sẽ xoá]" if path.name in REGENERABLE or PURGE_ALL else ""
        print(f"  {size / 2**30:7.2f} GB  {path.name}{mark}")

print("TRƯỚC khi dọn:")
report_disk()
print("\nĐang chiếm chỗ trong /kaggle/working:")
show_largest()

before = _sh.disk_usage("/kaggle/working").free

# pip cache và wheel đã cài xong thì không còn tác dụng
!rm -rf /root/.cache/pip /tmp/pip-* 2>/dev/null

targets = list(WORK.iterdir()) if PURGE_ALL else [
    WORK / name for name in REGENERABLE if (WORK / name).exists()
]
for target in targets:
    if target.is_dir():
        _sh.rmtree(target, ignore_errors=True)
    else:
        target.unlink(missing_ok=True)

reclaimed = (_sh.disk_usage("/kaggle/working").free - before) / 2**30
print(f"\nSAU khi dọn (giải phóng {reclaimed:.1f} GB):")
report_disk()

FREE_GB = _sh.disk_usage("/kaggle/working").free / 2**30
if FREE_GB < 12:
    print(f"\n>>> CHỈ CÒN {FREE_GB:.1f} GB — teacher chính cần ~9 GB, chưa kể torch.")
    print(">>> Xem danh sách bên trên: thứ nào lớn mà không có nhãn [sẽ xoá] thì")
    print(">>> đặt PURGE_ALL = True rồi chạy lại cell này, hoặc Run → Factory reset.")

## 2. Cài đặt

In [ ]:
%%capture install_log
!python -m pip install -q "gliner>=0.2.13" "transformers>=4.51" accelerate bitsandbytes pydantic

In [ ]:
import importlib
for module in ("gliner", "transformers", "pydantic"):
    try:
        importlib.import_module(module)
        print("ok  ", module)
    except ImportError as exc:
        print("LỖI", module, exc)

import torch
assert torch.cuda.is_available() or True, "không có CUDA"
print("cuda sau khi cài:", torch.cuda.is_available())

## 3. Source code

12 module của package `medical_coder` được ghi thẳng ra đĩa bằng
`%%writefile`, không nén, không mã hoá. Đọc được, sửa được ngay tại chỗ: gặp lỗi
trên Kaggle thì sửa cell rồi Restart & Run All, khỏi phải dựng lại notebook ở máy
rồi tải lên.

Đây là đúng những module mà nhánh predict-v2 cần. `pipeline.py` cùng backend LLM
sinh văn bản của lần nộp 01 **không** có ở đây — chúng không được dùng, và đưa vào
chỉ tổ đặt 43 KB code chết trước mặt người đọc.

> Sửa cell nào thì phải **Restart Session** rồi chạy lại, vì `medical_coder` đã
> được import vào kernel.

In [ ]:
import pathlib
pathlib.Path("/kaggle/working/medical_coder_src/medical_coder").mkdir(parents=True, exist_ok=True)
print("thư mục source:", "/kaggle/working/medical_coder_src/medical_coder")

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/__init__.py
"""Clinical concept extraction for the Viettel AI Race."""

__version__ = "0.1.0"

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/models.py
from __future__ import annotations

from enum import Enum
from typing import Any

from pydantic import BaseModel, ConfigDict, Field


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class EntityType(str, Enum):
    SYMPTOM = "TRIỆU_CHỨNG"
    TEST_NAME = "TÊN_XÉT_NGHIỆM"
    TEST_RESULT = "KẾT_QUẢ_XÉT_NGHIỆM"
    DIAGNOSIS = "CHẨN_ĐOÁN"
    MEDICATION = "THUỐC"


class AssertionType(str, Enum):
    NEGATED = "isNegated"
    FAMILY = "isFamily"
    HISTORICAL = "isHistorical"


class ExtractedMention(StrictModel):
    """Mention returned by the LLM before deterministic offset alignment."""

    text: str = Field(min_length=1)
    type: EntityType
    assertions: list[AssertionType]
    start_hint: int = Field(
        ge=0,
        description="Estimated zero-based start position in the original text.",
    )


class ExtractionResponse(StrictModel):
    entities: list[ExtractedMention]


class CandidateRequest(StrictModel):
    entity_index: int = Field(ge=0)
    type: EntityType
    text: str = Field(min_length=1)
    context: str


class CandidatePrediction(StrictModel):
    entity_index: int = Field(ge=0)
    candidates: list[str]


class NormalizationResponse(StrictModel):
    mappings: list[CandidatePrediction]


class AlignedEntity(StrictModel):
    text: str
    type: EntityType
    assertions: list[AssertionType]
    position: tuple[int, int]
    candidates: list[str] = Field(default_factory=list)

    def to_submission_dict(self) -> dict[str, Any]:
        result: dict[str, Any] = {
            "text": self.text,
            "type": self.type.value,
        }
        if self.type in {EntityType.DIAGNOSIS, EntityType.MEDICATION}:
            result["candidates"] = self.candidates
        result["assertions"] = [item.value for item in self.assertions]
        result["position"] = [self.position[0], self.position[1]]
        return result

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/validation.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable

from .models import AssertionType, EntityType


ICD10_RE = re.compile(r"^[A-Z][0-9]{2}(?:\.[0-9A-Z]{1,4})?$")
RXCUI_RE = re.compile(r"^[0-9]+$")
ALLOWED_TYPES = {item.value for item in EntityType}
ALLOWED_ASSERTIONS = {item.value for item in AssertionType}
CANDIDATE_TYPES = {EntityType.DIAGNOSIS.value, EntityType.MEDICATION.value}
ASSERTION_TYPES = {
    EntityType.SYMPTOM.value,
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}


class SubmissionValidationError(ValueError):
    pass


def sanitize_candidates(
    entity_type: EntityType,
    candidates: Iterable[str],
    max_candidates: int,
    allowlist: set[str] | None = None,
) -> list[str]:
    pattern = ICD10_RE if entity_type == EntityType.DIAGNOSIS else RXCUI_RE
    result: list[str] = []
    for value in candidates:
        candidate = str(value).strip().upper() if entity_type == EntityType.DIAGNOSIS else str(value).strip()
        if not pattern.fullmatch(candidate):
            continue
        if allowlist is not None and candidate not in allowlist:
            continue
        if candidate not in result:
            result.append(candidate)
        if len(result) >= max_candidates:
            break
    return result


def validate_submission_record(raw_text: str, entities: list[dict[str, Any]]) -> None:
    if not isinstance(entities, list):
        raise SubmissionValidationError("Top-level JSON must be a list")

    seen: set[tuple[int, int, str]] = set()
    previous_position: tuple[int, int] | None = None

    for index, entity in enumerate(entities):
        if not isinstance(entity, dict):
            raise SubmissionValidationError(f"Entity {index} must be an object")

        required = {"text", "type", "assertions", "position"}
        missing = required - set(entity)
        if missing:
            raise SubmissionValidationError(f"Entity {index} missing fields: {sorted(missing)}")

        entity_type = entity["type"]
        if entity_type not in ALLOWED_TYPES:
            raise SubmissionValidationError(f"Entity {index} has invalid type: {entity_type!r}")

        has_candidates = "candidates" in entity
        if (entity_type in CANDIDATE_TYPES) != has_candidates:
            raise SubmissionValidationError(
                f"Entity {index}: candidates field does not match type {entity_type}"
            )

        position = entity["position"]
        if (
            not isinstance(position, list)
            or len(position) != 2
            or not all(isinstance(value, int) and not isinstance(value, bool) for value in position)
        ):
            raise SubmissionValidationError(f"Entity {index} has invalid position")
        start, end = position
        if not (0 <= start < end <= len(raw_text)):
            raise SubmissionValidationError(
                f"Entity {index} position [{start}, {end}] is outside text length {len(raw_text)}"
            )
        if raw_text[start:end] != entity["text"]:
            raise SubmissionValidationError(
                f"Entity {index} text does not match raw_text[{start}:{end}]"
            )

        assertions = entity["assertions"]
        if not isinstance(assertions, list) or any(
            item not in ALLOWED_ASSERTIONS for item in assertions
        ):
            raise SubmissionValidationError(f"Entity {index} has invalid assertions")
        if entity_type not in ASSERTION_TYPES and assertions:
            raise SubmissionValidationError(
                f"Entity {index}: assertions are not allowed for {entity_type}"
            )
        if len(assertions) != len(set(assertions)):
            raise SubmissionValidationError(f"Entity {index} has duplicate assertions")

        if has_candidates:
            candidates = entity["candidates"]
            if not isinstance(candidates, list) or any(
                not isinstance(item, str) for item in candidates
            ):
                raise SubmissionValidationError(f"Entity {index} has invalid candidates")
            pattern = ICD10_RE if entity_type == EntityType.DIAGNOSIS.value else RXCUI_RE
            if any(not pattern.fullmatch(item) for item in candidates):
                raise SubmissionValidationError(f"Entity {index} has malformed candidate code")
            if len(candidates) != len(set(candidates)):
                raise SubmissionValidationError(f"Entity {index} has duplicate candidates")

        current = (start, end)
        if previous_position is not None and current < previous_position:
            raise SubmissionValidationError("Entities are not sorted by position")
        previous_position = current

        identity = (start, end, entity_type)
        if identity in seen:
            raise SubmissionValidationError(f"Duplicate entity at index {index}")
        seen.add(identity)


def validate_output_directory(input_dir: Path, output_dir: Path) -> list[str]:
    errors: list[str] = []
    input_files = sorted(input_dir.glob("*.txt"), key=lambda path: int(path.stem))
    expected_names = {f"{path.stem}.json" for path in input_files}
    actual_names = {path.name for path in output_dir.glob("*.json")}

    missing = sorted(expected_names - actual_names)
    extra = sorted(actual_names - expected_names)
    if missing:
        errors.append(f"Missing output files: {', '.join(missing)}")
    if extra:
        errors.append(f"Unexpected output files: {', '.join(extra)}")

    for input_path in input_files:
        output_path = output_dir / f"{input_path.stem}.json"
        if not output_path.exists():
            continue
        try:
            raw_text = input_path.read_text(encoding="utf-8")
            entities = json.loads(output_path.read_text(encoding="utf-8"))
            validate_submission_record(raw_text, entities)
        except (OSError, UnicodeError, json.JSONDecodeError, SubmissionValidationError) as exc:
            errors.append(f"{output_path.name}: {exc}")
    return errors

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/submission.py
"""Packaging and final checks for a submission.

Kept separate from :mod:`medical_coder.pipeline` on purpose: these helpers are
generic, but `pipeline` pulls in the whole generative LLM backend, so importing
them from there would drag ~43 KB of unrelated code into any consumer — notably
the predict-v2 notebook, which uses none of it.
"""
from __future__ import annotations

import zipfile
from pathlib import Path

from .validation import validate_output_directory


def create_submission_zip(output_dir: Path, zip_path: Path) -> None:
    """Write `output/<id>.json` members, then read the archive back to verify."""
    output_files = sorted(
        (path for path in output_dir.glob("*.json") if path.stem.isdigit()),
        key=lambda path: int(path.stem),
    )
    if not output_files:
        raise FileNotFoundError(f"No JSON files found in {output_dir}")
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = zip_path.with_suffix(zip_path.suffix + ".tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in output_files:
            archive.write(path, arcname=f"output/{path.name}")
    temporary.replace(zip_path)

    expected = [f"output/{path.name}" for path in output_files]
    with zipfile.ZipFile(zip_path, "r") as archive:
        if archive.namelist() != expected:
            raise RuntimeError(f"ZIP verification failed: {zip_path}")
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad_member}")


def validate_all(input_dir: Path, output_dir: Path) -> None:
    errors = validate_output_directory(input_dir, output_dir)
    if errors:
        formatted = "\n".join(f"- {error}" for error in errors)
        raise RuntimeError(f"Output validation failed:\n{formatted}")

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/terminology.py
from __future__ import annotations

import csv
import hashlib
import json
import math
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


_NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
_EMBEDDER_CACHE: dict[tuple[str, str], object] = {}


def normalize_term(value: str) -> str:
    """Normalize a term for retrieval only; never use this text for offsets."""

    decomposed = unicodedata.normalize("NFD", value.casefold())
    without_marks = "".join(
        character
        for character in decomposed
        if unicodedata.category(character) != "Mn"
    )
    return _NON_ALNUM_RE.sub(" ", without_marks).strip()


def _character_ngrams(value: str, size: int = 3) -> set[str]:
    compact = value.replace(" ", "_")
    if len(compact) <= size:
        return {compact} if compact else set()
    return {compact[index : index + size] for index in range(len(compact) - size + 1)}


@dataclass(frozen=True)
class TerminologyEntry:
    code: str
    label: str
    aliases: tuple[str, ...]

    @property
    def search_text(self) -> str:
        values = (self.label, *self.aliases)
        return " ; ".join(value for value in values if value)


@dataclass(frozen=True)
class RetrievedCandidate:
    code: str
    label: str
    score: float
    exact: bool


def _iter_jsonl(path: Path) -> Iterable[TerminologyEntry]:
    for line_number, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            code = str(item["code"]).strip()
            label = str(item["label"]).strip()
            raw_aliases = item.get("aliases", [])
            if isinstance(raw_aliases, str):
                raw_aliases = raw_aliases.split("|")
            aliases = tuple(
                str(alias).strip() for alias in raw_aliases if str(alias).strip()
            )
        except (KeyError, TypeError, ValueError, json.JSONDecodeError) as exc:
            raise ValueError(f"{path}:{line_number}: invalid terminology row") from exc
        if code and label:
            yield TerminologyEntry(code=code, label=label, aliases=aliases)


def _iter_delimited(path: Path) -> Iterable[TerminologyEntry]:
    sample = path.read_text(encoding="utf-8")[:8192]
    delimiter = "\t" if path.suffix.lower() == ".tsv" else ","
    try:
        delimiter = csv.Sniffer().sniff(sample, delimiters="\t,").delimiter
    except csv.Error:
        pass

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        fieldnames = {str(name).strip().casefold(): name for name in (reader.fieldnames or [])}
        if "code" not in fieldnames or "label" not in fieldnames:
            raise ValueError(
                f"{path}: expected a header with at least 'code' and 'label' columns"
            )
        for line_number, row in enumerate(reader, start=2):
            code = str(row[fieldnames["code"]] or "").strip()
            label = str(row[fieldnames["label"]] or "").strip()
            aliases_value = (
                str(row.get(fieldnames.get("aliases", ""), "") or "").strip()
                if "aliases" in fieldnames
                else ""
            )
            aliases = tuple(
                alias.strip() for alias in aliases_value.split("|") if alias.strip()
            )
            if not code or not label:
                raise ValueError(f"{path}:{line_number}: code and label are required")
            yield TerminologyEntry(code=code, label=label, aliases=aliases)


def load_terminology(path: Path) -> list[TerminologyEntry]:
    if not path.is_file():
        raise FileNotFoundError(f"Terminology file does not exist: {path}")
    iterator = _iter_jsonl(path) if path.suffix.lower() == ".jsonl" else _iter_delimited(path)
    entries: list[TerminologyEntry] = []
    seen: set[str] = set()
    for entry in iterator:
        if entry.code in seen:
            continue
        seen.add(entry.code)
        entries.append(entry)
    if not entries:
        raise ValueError(f"Terminology file is empty: {path}")
    return entries


class TerminologyIndex:
    """Fast lexical retrieval with an optional local E5 semantic reranker."""

    def __init__(
        self,
        path: Path,
        *,
        embedding_model: str | None = None,
        embedding_device: str = "cpu",
        embedding_cache_dir: Path | None = None,
        embedding_batch_size: int = 128,
    ) -> None:
        self.path = path
        self.entries = load_terminology(path)
        self._normalized_names: list[tuple[str, ...]] = []
        self._tokens: list[set[str]] = []
        self._ngrams: list[set[str]] = []
        self._exact: dict[str, set[int]] = {}
        self._token_postings: dict[str, set[int]] = {}
        self._ngram_postings: dict[str, set[int]] = {}

        for index, entry in enumerate(self.entries):
            names = tuple(
                dict.fromkeys(
                    normalized
                    for normalized in (
                        normalize_term(entry.label),
                        *(normalize_term(alias) for alias in entry.aliases),
                    )
                    if normalized
                )
            )
            token_set = set().union(*(set(name.split()) for name in names))
            ngram_set = set().union(*(_character_ngrams(name) for name in names))
            self._normalized_names.append(names)
            self._tokens.append(token_set)
            self._ngrams.append(ngram_set)
            for name in names:
                self._exact.setdefault(name, set()).add(index)
            for token in token_set:
                if len(token) >= 2:
                    self._token_postings.setdefault(token, set()).add(index)
            for ngram in ngram_set:
                self._ngram_postings.setdefault(ngram, set()).add(index)

        self._embedder = None
        self._embeddings = None
        if embedding_model:
            self._prepare_embeddings(
                embedding_model=embedding_model,
                device=embedding_device,
                cache_dir=embedding_cache_dir,
                batch_size=embedding_batch_size,
            )

    def _prepare_embeddings(
        self,
        *,
        embedding_model: str,
        device: str,
        cache_dir: Path | None,
        batch_size: int,
    ) -> None:
        try:
            import numpy as np
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise RuntimeError(
                "Semantic retrieval requires the local extra: "
                "python -m pip install -e '.[local]'"
            ) from exc

        embedder_key = (embedding_model, device)
        if embedder_key not in _EMBEDDER_CACHE:
            _EMBEDDER_CACHE[embedder_key] = SentenceTransformer(
                embedding_model,
                device=device,
                local_files_only=True,
            )
        self._embedder = _EMBEDDER_CACHE[embedder_key]
        fingerprint_value = (
            f"{self.path.resolve()}:{self.path.stat().st_size}:"
            f"{self.path.stat().st_mtime_ns}:{embedding_model}"
        )
        fingerprint = hashlib.sha256(fingerprint_value.encode("utf-8")).hexdigest()[:20]
        cache_path = None
        if cache_dir is not None:
            cache_dir.mkdir(parents=True, exist_ok=True)
            cache_path = cache_dir / f"{self.path.stem}.{fingerprint}.embeddings.npy"

        if cache_path is not None and cache_path.exists():
            embeddings = np.load(cache_path, mmap_mode="r")
            if embeddings.shape[0] != len(self.entries):
                raise RuntimeError(f"Stale semantic index: {cache_path}")
            self._embeddings = embeddings
            return

        passages = [f"passage: {entry.search_text}" for entry in self.entries]
        embeddings = self._embedder.encode(
            passages,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )
        embeddings = embeddings.astype("float32")
        if cache_path is not None:
            np.save(cache_path, embeddings)
            self._embeddings = np.load(cache_path, mmap_mode="r")
        else:
            self._embeddings = embeddings

    @staticmethod
    def _jaccard(left: set[str], right: set[str]) -> float:
        union = left | right
        return len(left & right) / len(union) if union else 0.0

    def _lexical_score(
        self,
        index: int,
        query: str,
        query_tokens: set[str],
        query_ngrams: set[str],
    ) -> tuple[float, bool]:
        names = self._normalized_names[index]
        exact = query in names
        token_score = self._jaccard(query_tokens, self._tokens[index])
        ngram_score = self._jaccard(query_ngrams, self._ngrams[index])
        contains = any(query in name or name in query for name in names)
        score = (
            (1.0 if exact else 0.0)
            + (0.18 if contains else 0.0)
            + 0.55 * token_score
            + 0.27 * ngram_score
        )
        return min(score, 1.0), exact

    def retrieve(
        self,
        mention: str,
        *,
        context: str = "",
        top_k: int = 20,
        lexical_pool_size: int = 160,
    ) -> list[RetrievedCandidate]:
        query = normalize_term(mention)
        if not query:
            return []
        query_tokens = set(query.split())
        query_ngrams = _character_ngrams(query)

        pool: set[int] = set(self._exact.get(query, set()))
        for token in query_tokens:
            pool.update(self._token_postings.get(token, set()))
        for ngram in query_ngrams:
            pool.update(self._ngram_postings.get(ngram, set()))

        lexical_by_index = {
            index: self._lexical_score(
                index,
                query,
                query_tokens,
                query_ngrams,
            )
            for index in pool
        }

        semantic_scores: dict[int, float] = {}
        if self._embedder is not None and self._embeddings is not None:
            import numpy as np

            semantic_query = normalize_term(f"{mention} {context[:240]}")
            query_embedding = self._embedder.encode(
                [f"query: {semantic_query}"],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )[0]
            # Add a global semantic pool. This is necessary for Vietnamese
            # mentions whose official ICD/RxNorm labels are only in English.
            all_scores = np.asarray(self._embeddings) @ query_embedding
            semantic_pool_size = min(lexical_pool_size, len(self.entries))
            if semantic_pool_size:
                semantic_indices = np.argpartition(
                    all_scores,
                    -semantic_pool_size,
                )[-semantic_pool_size:]
                pool.update(int(index) for index in semantic_indices)
            semantic_scores = {
                int(index): float((all_scores[index] + 1.0) / 2.0)
                for index in pool
            }

        if not pool:
            return []

        lexical = [
            (*lexical_by_index.get(index, (0.0, False)), index)
            for index in pool
        ]
        lexical.sort(key=lambda item: (-item[0], not item[1], self.entries[item[2]].code))
        lexical = lexical[:lexical_pool_size]

        if semantic_scores:
            # Do not discard semantic-only hits just because their lexical
            # score is zero; rerank the union by the final combined score.
            lexical = [
                (*lexical_by_index.get(index, (0.0, False)), index)
                for index in pool
            ]

        ranked: list[RetrievedCandidate] = []
        for lexical_score, exact, index in lexical:
            semantic_score = semantic_scores.get(index)
            combined = (
                lexical_score
                if semantic_score is None
                else 0.52 * lexical_score + 0.48 * semantic_score
            )
            if not math.isfinite(combined):
                continue
            entry = self.entries[index]
            ranked.append(
                RetrievedCandidate(
                    code=entry.code,
                    label=entry.label,
                    score=combined,
                    exact=exact,
                )
            )
        ranked.sort(key=lambda item: (-item.score, not item.exact, item.code))
        return ranked[:top_k]

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/gliner_ner.py
"""Zero-shot span detection with GLiNER, replacing free-form LLM generation.

Why this exists
---------------
Submission 01 asked Qwen3-8B to *write out* every mention as JSON, then aligned
the generated strings back onto the raw text. That design loses on two fronts the
host metric punishes hardest:

* every mention the model paraphrases is dropped by alignment, so recall is
  capped by the model's copying fidelity;
* generated mentions carry no score, so there is no dial between precision and
  recall — and the scorer counts each spurious concept **twice** in every
  denominator.

GLiNER returns character offsets with a calibrated score per span, which gives
both an exact `text[start:end]` guarantee and a per-type threshold to tune.

Long notes are chunked on line boundaries as verbatim slices, so a chunk offset
plus the local offset is always the true global offset.
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass

from .models import EntityType

LOGGER = logging.getLogger(__name__)

# Descriptive English label strings generalise better than the Vietnamese type
# names: GLiNER was trained on English label prompts.
DEFAULT_LABEL_MAP: dict[str, EntityType] = {
    "symptom": EntityType.SYMPTOM,
    "disease or diagnosis": EntityType.DIAGNOSIS,
    "medication or drug": EntityType.MEDICATION,
    "medical test or lab name": EntityType.TEST_NAME,
    "test result or measurement value": EntityType.TEST_RESULT,
}

# Tuned on the reference solution's dev split; the host double-penalty on
# spurious spans makes precision the right side to err on.
DEFAULT_THRESHOLDS: dict[EntityType, float] = {
    EntityType.SYMPTOM: 0.20,
    EntityType.DIAGNOSIS: 0.25,
    EntityType.MEDICATION: 0.30,
    EntityType.TEST_NAME: 0.15,
    EntityType.TEST_RESULT: 0.35,
}

# A trailing token that belongs inside a THUỐC span. The Vòng 1 example shows
# drug spans carrying strength, route and frequency ("amlodipine 10 mg po daily"),
# but GLiNER usually stops at the drug name.
_DOSE_TOKEN = re.compile(
    r"^(\d+([.,\-/]\d+)*%?"
    r"|\d+\s*(mg|mcg|ug|g|ml|iu|meq|mmol)\b"
    r"|mg|mcg|µg|ug|g|ml|iu|x|%"
    r"|po|iv|im|sc|sl|pr|tab"
    r"|(q\d+h|qd|qid|qod|bid|tid|qhs|qam|qpm|prn|daily)(:prn)?"
    r")$",
    re.IGNORECASE,
)
# Vietnamese indication markers that terminate a drug span.
_INDICATION_MARKERS = ("điều trị", "cho", "để", "khi", "nếu")

# A generic lead-in that adds no clinical content ("dấu hiệu điển hình" -> "điển
# hình" is wrong, but "dấu hiệu vàng da kéo dài" -> "vàng da kéo dài" is right).
_GENERIC_PREFIX = re.compile(
    r"^(?:dấu hiệu|biểu hiện|tình trạng|hội chứng)\b[\s:;,.-]*", re.IGNORECASE
)
_WORD_TOKEN = re.compile(r"[0-9A-Za-zÀ-ỹĐđ]+")


def trim_generic_prefix(text: str, span: "ScoredSpan") -> "ScoredSpan":
    """Drop a generic lead-in when a complete concept (>= 2 words) remains.

    This is the one boundary lever the reference solution kept after ablation;
    the others (symptom-edge trimming, drug re-cleaning) measured worse. A
    one-word remainder is left alone because it is unstable.
    """
    match = _GENERIC_PREFIX.match(text[span.start : span.end])
    if match is None:
        return span
    new_start = span.start + match.end()
    if len(_WORD_TOKEN.findall(text[new_start : span.end])) < 2:
        return span
    return ScoredSpan(new_start, span.end, span.type, span.score)


@dataclass(frozen=True)
class ScoredSpan:
    start: int
    end: int
    type: EntityType
    score: float


def iter_chunks(text: str, max_chunk_chars: int) -> list[tuple[int, str]]:
    """Split into verbatim slices on line boundaries, keeping global offsets."""
    chunks: list[tuple[int, str]] = []
    position = 0
    length = len(text)
    while position < length:
        end = min(position + max_chunk_chars, length)
        if end < length:
            newline = text.rfind("\n", position, end)
            if newline > position:
                end = newline + 1
        chunks.append((position, text[position:end]))
        position = end
    return chunks or [(0, "")]


def extend_medication_span(text: str, start: int, end: int) -> int:
    """Absorb trailing dosage/route/frequency tokens into a medication span."""
    line_end = text.find("\n", end)
    if line_end == -1:
        line_end = len(text)
    remainder = text[end:line_end]
    lowered = remainder.lower()
    for marker in _INDICATION_MARKERS:
        position = lowered.find(marker)
        if position != -1:
            remainder = remainder[:position]
            break
    new_end = end
    for match in re.finditer(r"\S+", remainder):
        if _DOSE_TOKEN.match(match.group(0).strip(".,;:")):
            new_end = end + match.end()
        else:
            break
    return new_end


def resolve_overlaps(spans: list[ScoredSpan]) -> list[ScoredSpan]:
    """Keep non-overlapping spans, preferring higher score then tighter bounds."""
    ordered = sorted(spans, key=lambda span: (-span.score, span.start - span.end))
    kept: list[ScoredSpan] = []
    for span in ordered:
        if any(not (span.end <= other.start or span.start >= other.end) for other in kept):
            continue
        kept.append(span)
    kept.sort(key=lambda span: (span.start, span.end))
    return kept


class GlinerSpanExtractor:
    """Per-type-thresholded GLiNER span extraction over raw clinical text."""

    def __init__(
        self,
        model_path: str = "urchade/gliner_multi-v2.1",
        *,
        label_map: dict[str, EntityType] | None = None,
        thresholds: dict[EntityType, float] | None = None,
        raw_floor: float = 0.02,
        max_chunk_chars: int = 800,
        device: str = "cpu",
    ) -> None:
        try:
            from gliner import GLiNER
        except ImportError as error:  # pragma: no cover - dependency guard
            raise RuntimeError(
                "GLiNER backend needs the gliner package: python -m pip install gliner"
            ) from error

        self.label_map = label_map or DEFAULT_LABEL_MAP
        self.labels = list(self.label_map)
        self.thresholds = thresholds or DEFAULT_THRESHOLDS
        self.raw_floor = raw_floor
        self.max_chunk_chars = max_chunk_chars
        LOGGER.info("loading GLiNER %s on %s", model_path, device)
        self.model = GLiNER.from_pretrained(model_path)
        try:
            self.model = self.model.to(device)
        except Exception:  # pragma: no cover - some versions manage device internally
            LOGGER.debug("GLiNER manages its own device placement")
        self.model.eval()
        self.parameters = sum(
            parameter.numel() for parameter in self.model.parameters()
        )

    def scored_spans(self, text: str) -> list[ScoredSpan]:
        """All spans above ``raw_floor``, before per-type gating."""
        spans: list[ScoredSpan] = []
        for chunk_start, chunk in iter_chunks(text, self.max_chunk_chars):
            if not chunk.strip():
                continue
            predictions = self.model.predict_entities(
                chunk, self.labels, threshold=self.raw_floor
            )
            for prediction in predictions:
                entity_type = self.label_map.get(prediction["label"])
                if entity_type is None:
                    continue
                start = chunk_start + prediction["start"]
                end = chunk_start + prediction["end"]
                while start < end and text[start].isspace():
                    start += 1
                while end > start and text[end - 1].isspace():
                    end -= 1
                if entity_type is EntityType.MEDICATION:
                    end = extend_medication_span(text, start, end)
                if start < end:
                    spans.append(
                        ScoredSpan(start, end, entity_type, float(prediction.get("score", 1.0)))
                    )
        return spans

    def gate(self, spans: list[ScoredSpan]) -> list[ScoredSpan]:
        """Apply per-type thresholds, then resolve overlaps."""
        return resolve_overlaps(
            [span for span in spans if span.score >= self.thresholds.get(span.type, 0.5)]
        )

    def extract(self, text: str) -> list[ScoredSpan]:
        return self.gate(self.scored_spans(text))

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/exact_link.py
"""Precision-first candidate linking: one code, only on a unique exact alias.

Rationale from the metric, not from taste. In the host scorer a diagnosis or
medication concept contributes with weight ``w = len(gold_candidates) + 1``:

* when the gold list is **empty**, predicting empty scores Jaccard 1.0 and
  predicting any code scores 0.0;
* when the concept is **spurious** it lands in the denominator as
  ``2 * (len(predicted_candidates) + 1)`` — so emitting three codes on a wrong
  span costs 8 denominator units instead of 2.

Both effects push the same way: emit a code only when it is nearly certain. The
reference solution measured this directly — removing *every* candidate moved
their candidate Jaccard by only 0.0036, while their retrieve-and-rerank variant
(SapBERT + Qwen listwise) scored materially worse than this policy because it
always picked something.

So retrieval breadth and emission breadth are separated: look up widely, emit
only on a unique exact alias match.
"""
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

from .models import EntityType
from .terminology import load_terminology

# Route / frequency / dose noise that is not part of a canonical drug name.
_DRUG_NOISE = re.compile(
    r"\b(po|iv|im|sc|sl|pr|tid|bid|qd|qid|qhs|qam|qpm|prn|q\d+h(:prn)?|daily|twice|once|"
    r"uống|tiêm|truyền|lần|ngày|viên|tab|caps?)\b",
    re.IGNORECASE,
)
_DECIMAL_COMMA = re.compile(r"(\d),(\d)")
_UNIT_SPACING = re.compile(r"\s*(mg|mcg|µg|ug|g|ml|iu|meq|mmol)\b", re.IGNORECASE)

_BARE_ICD_CATEGORY = re.compile(r"^[A-Z]\d\d$")


def normalize_surface(value: str) -> str:
    """Lowercase and collapse whitespace for lookup only; never touches offsets."""
    return " ".join(str(value).split()).strip(" ,;:.").lower()


def strip_diacritics(value: str) -> str:
    decomposed = unicodedata.normalize("NFD", value)
    stripped = "".join(
        character for character in decomposed if unicodedata.category(character) != "Mn"
    )
    return unicodedata.normalize("NFC", stripped).replace("đ", "d")


def clean_mention(mention: str, entity_type: EntityType) -> str:
    text = " ".join(mention.split())
    if entity_type is EntityType.MEDICATION:
        text = _DRUG_NOISE.sub(" ", text)
        text = _DECIMAL_COMMA.sub(r"\1.\2", text)
        text = _UNIT_SPACING.sub(r" \1", text)
        text = " ".join(text.split())
    return text.strip()


class ExactAliasIndex:
    """Alias -> codes map with a diacritic-insensitive fallback."""

    def __init__(self, entries) -> None:
        self.codes: set[str] = set()
        self._exact: dict[str, set[str]] = {}
        self._plain: dict[str, set[str]] = {}
        for entry in entries:
            self.codes.add(entry.code)
            for surface in (entry.label, *entry.aliases):
                key = normalize_surface(surface)
                if not key:
                    continue
                self._exact.setdefault(key, set()).add(entry.code)
                self._plain.setdefault(strip_diacritics(key), set()).add(entry.code)

    @classmethod
    def from_path(cls, path: Path) -> "ExactAliasIndex":
        return cls(load_terminology(path))

    def lookup(self, mention: str) -> set[str]:
        key = normalize_surface(mention)
        if not key:
            return set()
        hit = self._exact.get(key)
        if hit:
            return hit
        return self._plain.get(strip_diacritics(key), set())


class PrecisionFirstLinker:
    """Emit at most one code, and only when exactly one alias matches."""

    def __init__(
        self,
        icd_index: ExactAliasIndex | None = None,
        rxnorm_index: ExactAliasIndex | None = None,
        *,
        max_candidates: int = 1,
        leaf_remap: bool = True,
    ) -> None:
        self.icd_index = icd_index
        self.rxnorm_index = rxnorm_index
        self.max_candidates = max_candidates
        self.leaf_remap = leaf_remap
        self._cache: dict[tuple[str, str], list[str]] = {}

    def _index_for(self, entity_type: EntityType) -> ExactAliasIndex | None:
        if entity_type is EntityType.DIAGNOSIS:
            return self.icd_index
        if entity_type is EntityType.MEDICATION:
            return self.rxnorm_index
        return None

    def _leafify(self, code: str) -> str:
        """Remap a bare 3-character ICD category to its ``.9`` leaf when it exists.

        A bare-category mention carries no complication or severity detail, and
        the target is almost always a leaf, so ``.9`` (unspecified) is the
        conventional resolution. This can only turn a miss into a hit.
        """
        if not self.leaf_remap or self.icd_index is None:
            return code
        if _BARE_ICD_CATEGORY.match(code) and f"{code}.9" in self.icd_index.codes:
            return f"{code}.9"
        return code

    def link(self, mention: str, entity_type: EntityType) -> list[str]:
        index = self._index_for(entity_type)
        if index is None:
            return []
        cleaned = clean_mention(mention, entity_type)
        if not cleaned:
            return []
        key = (entity_type.value, cleaned)
        cached = self._cache.get(key)
        if cached is not None:
            return list(cached)

        codes = index.lookup(cleaned)
        # Precision-first: ambiguity is a reason to stay silent, not to guess.
        if len(codes) != 1:
            result: list[str] = []
        else:
            code = next(iter(codes))
            if entity_type is EntityType.DIAGNOSIS:
                code = self._leafify(code)
            result = [code][: self.max_candidates]
        self._cache[key] = result
        return list(result)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/selector.py
"""Qwen teachers that re-type and extend GLiNER spans using next-token logits.

Two roles, both decided by a *single forward pass* over a fixed label set — never
by generation. That matters: a generated answer can be anything, while a logit
over the digits 0-5 is a closed decision that cannot corrupt the schema.

1. **Corrector** — for every baseline ``TRIỆU_CHỨNG`` span, read five-way logits
   and re-type it to ``CHẨN_ĐOÁN`` when the teacher disagrees. This is GLiNER's
   single largest type error: it reads chronic diseases as symptoms. Our
   CPU-only run produced 529 diagnoses against the reference solution's 668, and
   this step is what closes that gap.

2. **Additions** — for low-scoring raw spans that do not overlap a baseline span,
   read six-way logits (0 = not a concept) in *both* teachers and add the span
   only when they agree on the same type, that type carries no candidates, and
   both clear a margin over ``NONE``.

Additions are restricted to ``TRIỆU_CHỨNG`` / ``TÊN_XÉT_NGHIỆM`` /
``KẾT_QUẢ_XÉT_NGHIỆM`` on purpose. A spurious diagnosis or drug is charged to the
candidate denominator as well (``2 * (n_codes + 1)``), so recall bought in those
two types is paid for twice.

Consensus is required for additions but not for correction: correction moves a
span that already exists, while an addition creates one, and only the second can
manufacture a spurious concept.
"""
from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Iterable, Sequence

from .gliner_ner import ScoredSpan
from .models import EntityType

LOGGER = logging.getLogger(__name__)

# Digit order is fixed: the prompts below number the types 1-5 in this sequence.
ORDERED_TYPES: tuple[EntityType, ...] = (
    EntityType.SYMPTOM,
    EntityType.DIAGNOSIS,
    EntityType.MEDICATION,
    EntityType.TEST_NAME,
    EntityType.TEST_RESULT,
)

# Types safe to add: none of them carry candidates.
DEFAULT_ADDITION_TYPES = frozenset(
    {EntityType.SYMPTOM, EntityType.TEST_NAME, EntityType.TEST_RESULT}
)

FIVE_WAY_SYSTEM = (
    "Bạn là chuyên gia y lâm sàng Việt Nam. Phân loại ý niệm y khoa vào ĐÚNG MỘT loại:\n"
    "1 = TRIỆU_CHỨNG (triệu chứng: đau đầu, sốt, phù, mệt mỏi)\n"
    "2 = CHẨN_ĐOÁN (chẩn đoán bệnh: đái tháo đường, tăng huyết áp, viêm phổi, ung thư)\n"
    "3 = THUỐC (paracetamol, ceftriaxone, amlodipine)\n"
    "4 = TÊN_XÉT_NGHIỆM (công thức máu, MRI, nội soi, X-quang)\n"
    "5 = KẾT_QUẢ_XÉT_NGHIỆM (giá trị số: 120 mg/dL, HbA1c 7.2%, GCS 15)\n"
    "Quy tắc: 'tăng huyết áp/đái tháo đường/xơ gan' = 2; 'đau/phù/sốt/mệt' = 1."
)

SIX_WAY_SYSTEM = (
    "Bạn là chuyên gia y lâm sàng Việt Nam. Phân loại ý niệm y khoa:\n"
    "0 = KHÔNG PHẢI ý niệm y khoa hợp lệ (tiêu đề, từ chung chung, số/liều rời, hành chính)\n"
    "1 = TRIỆU_CHỨNG (triệu chứng: đau đầu, sốt, phù)\n"
    "2 = CHẨN_ĐOÁN (bệnh: đái tháo đường, tăng huyết áp, viêm phổi)\n"
    "3 = THUỐC (paracetamol, ceftriaxone)\n"
    "4 = TÊN_XÉT_NGHIỆM (công thức máu, MRI, nội soi)\n"
    "5 = KẾT_QUẢ_XÉT_NGHIỆM (giá trị số: 120 mg/dL, GCS 15)\n"
    "Quy tắc: 'tăng huyết áp/đái tháo đường'=2; 'đau/phù/sốt'=1; tiêu đề=0."
)


def line_context(text: str, start: int, end: int, line_limit: int = 300) -> str:
    """The mention's line, prefixed by the nearest non-empty line as a section hint."""
    line_start = text.rfind("\n", 0, start) + 1
    line_end = text.find("\n", end)
    if line_end == -1:
        line_end = len(text)
    line = text[line_start:line_end][:line_limit]

    previous = ""
    cursor = line_start - 1
    while cursor > 0:
        candidate_start = text.rfind("\n", 0, cursor) + 1
        candidate_end = text.find("\n", candidate_start)
        if candidate_end == -1:
            candidate_end = len(text)
        candidate = text[candidate_start:candidate_end].strip()
        if candidate:
            previous = candidate[:80]
            break
        cursor = candidate_start - 1
    return f"[mục: {previous}] {line}" if previous else line


def single_token_ids(tokenizer, surfaces: Iterable[str]) -> list[int]:
    """Token ids for surfaces that encode to exactly one token."""
    ids = []
    for surface in surfaces:
        encoded = tokenizer.encode(surface, add_special_tokens=False)
        if len(encoded) == 1:
            ids.append(encoded[0])
    return ids


def digit_token_groups(tokenizer, digits: Iterable[int]) -> list[list[int]]:
    """One id group per digit, covering the bare and space-prefixed forms."""
    return [single_token_ids(tokenizer, (str(d), f" {d}")) for d in digits]


@dataclass
class TeacherConfig:
    model_path: str
    device: str = "cuda"
    quantization: str = "4bit"
    dtype: str = "bfloat16"


class Teacher:
    """A causal LM used only to score a fixed set of next tokens."""

    def __init__(self, config: TeacherConfig) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        torch_dtype = {
            "bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32,
        }.get(config.dtype, torch.bfloat16)

        self.tokenizer = AutoTokenizer.from_pretrained(config.model_path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        if config.quantization == "none":
            self.model = AutoModelForCausalLM.from_pretrained(
                config.model_path, torch_dtype=torch_dtype
            ).to(config.device)
        else:
            from transformers import BitsAndBytesConfig

            if config.quantization == "4bit":
                bnb = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch_dtype,
                    bnb_4bit_use_double_quant=True,
                )
            elif config.quantization == "8bit":
                bnb = BitsAndBytesConfig(load_in_8bit=True)
            else:
                raise ValueError(f"Unknown quantization mode: {config.quantization!r}")
            self.model = AutoModelForCausalLM.from_pretrained(
                config.model_path,
                quantization_config=bnb,
                device_map={"": config.device},
            )
        self.model.eval()
        self.parameters = sum(p.numel() for p in self.model.parameters())
        self.entity_digits = digit_token_groups(self.tokenizer, range(1, 6))
        self.none_digits = single_token_ids(self.tokenizer, ("0", " 0"))
        LOGGER.info(
            "loaded teacher %s (%.3fB params) on %s",
            config.model_path,
            self.parameters / 1e9,
            config.device,
        )

    def chat_prompt(self, system: str, user: str) -> str:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        try:
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:  # older templates have no enable_thinking argument
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

    def group_scores(
        self,
        prompts: Sequence[str],
        groups: Sequence[Sequence[int]],
        batch_size: int,
        max_length: int,
    ) -> list[list[float]]:
        """Log-sum-exp of each token group, from one next-token forward per batch."""
        import torch

        device = next(self.model.parameters()).device
        results: list[list[float]] = []
        for offset in range(0, len(prompts), batch_size):
            batch = list(prompts[offset : offset + batch_size])
            encoded = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length + 8,
            ).to(device)
            with torch.inference_mode():
                try:
                    logits = self.model(**encoded, logits_to_keep=1).logits[:, -1, :]
                except TypeError:  # transformers without logits_to_keep
                    logits = self.model(**encoded).logits[:, -1, :]
            for row in logits:
                results.append(
                    [
                        torch.logsumexp(row[list(ids)], -1).item() if ids else -1e9
                        for ids in groups
                    ]
                )
        return results


class SpanSelector:
    """Type correction, plus consensus additions when a second teacher is present."""

    def __init__(
        self,
        primary: Teacher,
        secondary: Teacher | None = None,
        *,
        batch_size: int = 48,
        max_length: int = 384,
        addition_margin: float = 1.0,
        addition_types: frozenset[EntityType] = DEFAULT_ADDITION_TYPES,
        reject_margin: float | None = None,
    ) -> None:
        self.primary = primary
        self.secondary = secondary
        self.batch_size = batch_size
        self.max_length = max_length
        self.addition_margin = addition_margin
        self.addition_types = addition_types
        # None disables rejection. A positive value is how far NONE must beat the
        # best entity label before a span is dropped, so larger is more cautious.
        self.reject_margin = reject_margin
        # Every margin the rejector has seen, so one run yields the whole
        # threshold curve instead of a single operating point. Picking the next
        # margin from this distribution beats guessing and re-submitting.
        self.margins_seen: list[float] = []

    @property
    def total_parameters(self) -> int:
        total = self.primary.parameters
        if self.secondary is not None:
            total += self.secondary.parameters
        return total

    def _five_way(self, teacher: Teacher, prompts: Sequence[str]) -> list[EntityType]:
        scores = teacher.group_scores(
            prompts, teacher.entity_digits, self.batch_size, self.max_length
        )
        return [ORDERED_TYPES[max(range(5), key=row.__getitem__)] for row in scores]

    def _six_way(
        self, teacher: Teacher, prompts: Sequence[str]
    ) -> list[tuple[EntityType, float]]:
        groups = list(teacher.entity_digits) + [teacher.none_digits]
        scores = teacher.group_scores(prompts, groups, self.batch_size, self.max_length)
        output = []
        for row in scores:
            entity_scores, none_score = row[:5], row[5]
            best = max(range(5), key=entity_scores.__getitem__)
            output.append((ORDERED_TYPES[best], entity_scores[best] - none_score))
        return output

    def correct_types(self, text: str, spans: list[ScoredSpan]) -> list[ScoredSpan]:
        """Re-type TRIỆU_CHỨNG spans the primary teacher reads as CHẨN_ĐOÁN."""
        indices = [i for i, span in enumerate(spans) if span.type is EntityType.SYMPTOM]
        if not indices:
            return list(spans)
        prompts = [
            self.primary.chat_prompt(
                FIVE_WAY_SYSTEM,
                f"Ngữ cảnh: {line_context(text, spans[i].start, spans[i].end)}\n"
                f'Ý niệm: "{text[spans[i].start : spans[i].end]}"\n'
                "Trả lời CHỈ MỘT chữ số 1-5.",
            )
            for i in indices
        ]
        predictions = self._five_way(self.primary, prompts)
        corrected = list(spans)
        for position, prediction in zip(indices, predictions):
            if prediction is EntityType.DIAGNOSIS:
                span = spans[position]
                corrected[position] = ScoredSpan(
                    span.start, span.end, EntityType.DIAGNOSIS, span.score
                )
        return corrected

    def reject_spans(
        self, text: str, spans: list[ScoredSpan], margin: float
    ) -> list[ScoredSpan]:
        """Drop baseline spans the teacher reads as "not a medical concept".

        Fixes an asymmetry that had no justification: adding a span required two
        teachers to agree, while keeping one required nothing beyond GLiNER's
        threshold. The six-way prompt already has a `0 = not a concept` option;
        this simply asks it about spans we were going to emit anyway.

        The arithmetic is favourable. Dropping a spurious span removes 2 units
        from every denominator; dropping a correct one only costs `alpha` from
        the numerator, because the ground-truth concept stays in the denominator
        as a miss either way. Rejection therefore pays off whenever it is right
        more than ``1 / (1 + 2 * text_score / alpha)`` of the time — about 57% at
        our current operating point, against a pipeline precision near 55%.
        """
        if not spans:
            return []
        prompts = [
            self.primary.chat_prompt(
                SIX_WAY_SYSTEM,
                f"Ngữ cảnh: {line_context(text, span.start, span.end)}\n"
                f'Ý niệm: "{text[span.start : span.end]}"\n'
                "Trả lời CHỈ MỘT chữ số 0-5.",
            )
            for span in spans
        ]
        votes = self._six_way(self.primary, prompts)
        if self.secondary is not None:
            # With a second opinion available, require both to call it junk.
            secondary_votes = self._six_way(self.secondary, prompts)
            votes = [
                (t, max(p, s))
                for (t, p), (_, s) in zip(votes, secondary_votes)
            ]
        self.margins_seen.extend(margin_value for _, margin_value in votes)
        kept = [span for span, (_, m) in zip(spans, votes) if m > -margin]
        LOGGER.info("rejector: dropped %d of %d spans", len(spans) - len(kept), len(spans))
        return kept

    CANDIDATE_MARGINS = (-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0)

    def rejection_stats(self) -> dict:
        """Calibration data for the rejector, in a form worth keeping.

        The raw margins matter more than the summary table: with them any future
        threshold can be evaluated offline, without paying for another GPU run.
        """
        ordered = sorted(self.margins_seen)
        total = len(ordered)
        return {
            "margin_used": self.reject_margin,
            "spans_scored": total,
            "curve": [
                {
                    "margin": candidate,
                    "dropped": sum(1 for value in ordered if value <= -candidate),
                }
                for candidate in self.CANDIDATE_MARGINS
            ],
            "margins": [round(value, 4) for value in self.margins_seen],
        }

    def rejection_report(self) -> str:
        """How many spans each candidate margin would drop, over the whole corpus."""
        if not self.margins_seen:
            return "rejector: chưa chạy"
        stats = self.rejection_stats()
        total = stats["spans_scored"]
        lines = [f"rejector: đã chấm {total} span", "  margin  bỏ đi   tỉ lệ"]
        for row in stats["curve"]:
            lines.append(
                f"  {row['margin']:6.1f} {row['dropped']:6d} {row['dropped'] / total:7.1%}"
            )
        return "\n".join(lines)

    def propose_additions(
        self,
        text: str,
        baseline: list[ScoredSpan],
        raw: list[ScoredSpan],
    ) -> list[ScoredSpan]:
        """Add non-overlapping raw spans both teachers agree on."""
        if self.secondary is None:
            return []
        occupied = [(span.start, span.end) for span in baseline]

        def overlaps(start: int, end: int) -> bool:
            return any(not (end <= s or start >= e) for s, e in occupied)

        candidates: list[tuple[int, int]] = []
        seen: set[tuple[int, int]] = set()
        for span in raw:
            key = (span.start, span.end)
            if key in seen or overlaps(*key):
                continue
            if len(text[span.start : span.end].strip()) < 2:
                continue
            seen.add(key)
            candidates.append(key)
        if not candidates:
            return []

        prompts = [
            self.primary.chat_prompt(
                SIX_WAY_SYSTEM,
                f"Ngữ cảnh: {line_context(text, start, end)}\n"
                f'Ý niệm: "{text[start:end]}"\n'
                "Trả lời CHỈ MỘT chữ số 0-5.",
            )
            for start, end in candidates
        ]
        primary_votes = self._six_way(self.primary, prompts)
        secondary_votes = self._six_way(self.secondary, prompts)

        additions: list[ScoredSpan] = []
        for (start, end), (p_type, p_margin), (s_type, s_margin) in zip(
            candidates, primary_votes, secondary_votes
        ):
            if p_type is not s_type or p_type not in self.addition_types:
                continue
            if p_margin < self.addition_margin or s_margin < self.addition_margin:
                continue
            additions.append(ScoredSpan(start, end, p_type, 0.0))
        return additions

    def select(
        self,
        text: str,
        baseline: list[ScoredSpan],
        raw: list[ScoredSpan],
    ) -> list[ScoredSpan]:
        # Reject before correcting: no point asking the corrector to re-type a
        # span that is about to be dropped.
        if self.reject_margin is not None:
            baseline = self.reject_spans(text, baseline, self.reject_margin)
        corrected = self.correct_types(text, baseline)
        corrected.extend(self.propose_additions(text, corrected, raw))
        corrected.sort(key=lambda span: (span.start, span.end))
        return corrected

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/pipeline_v2.py
"""Precision-first pipeline: GLiNER spans + exact-alias linking + empty assertions.

    raw text
    -> GLiNER spans (per-type thresholds, verbatim offsets)
    -> overlap resolution
    -> header/junk rejection
    -> unique-exact-alias linking against the Vietnamese ICD-10 (TT06) and RxNorm
    -> schema validation
    -> output.zip

There is no generative step, so the whole run is CPU-only and finishes in about a
minute for 100 records. That matters beyond speed: every design choice below is
one the host metric rewards directly, and none of them depend on a GPU being
available at submission time.

Assertions are always emitted empty, and that is a measured choice rather than a
gap. A wrong assertion forfeits the whole Jaccard of its concept, while an empty
prediction against an empty gold list scores 1.0; on the reference solution's
split every negation / family / history rule they tried over-fired, and
`isNegated` separated at AUC 0.497 — chance. Submission 01 emitted 266 assertion
labels and scored 20.19 on the component; the reference emitted none and scored
35.27. Restoring assertions is worth revisiting only against labelled validation
data, which is why there is no flag to half-enable it here.
"""
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass
from pathlib import Path

from .exact_link import ExactAliasIndex, PrecisionFirstLinker, strip_diacritics
from .gliner_ner import (
    DEFAULT_THRESHOLDS,
    GlinerSpanExtractor,
    ScoredSpan,
    resolve_overlaps,
    trim_generic_prefix,
)
from .models import AlignedEntity, EntityType
from .validation import validate_submission_record

# The declared budget is on parameter count, not memory: quantization does not
# change what must be reported. GLiNER 0.289B + two 4B teachers = 8.517B.
PARAMETER_BUDGET = 9_000_000_000

LOGGER = logging.getLogger(__name__)

# Bare field labels and section headings. A span is rejected only when it is
# *exactly* one of these: the same words inside a longer mention ("kết quả xét
# nghiệm glucose cao") are part of a real concept and must survive, so this is an
# equality test rather than a substring blacklist.
HEADER_LABELS = frozenset(
    {
        "thuốc", "tên thuốc",
        "triệu chứng", "các triệu chứng",
        "chẩn đoán", "chẩn đoán sơ bộ", "chẩn đoán ra viện", "icd",
        "xét nghiệm", "tên xét nghiệm",
        "kết quả", "kết quả xét nghiệm",
        "khám lâm sàng", "cận lâm sàng",
        "tiền sử", "tiền sử bệnh", "bệnh sử",
        "điều trị", "tình trạng", "diễn biến",
        "lý do", "lý do vào viện", "hỏi bệnh",
        "bệnh nhân",
    }
)
_NUMBERED_HEADING = re.compile(r"^\d+\s*[.)]\s*$")


@dataclass(frozen=True)
class PipelineV2Config:
    input_dir: Path
    output_dir: Path
    model_path: str = "urchade/gliner_multi-v2.1"
    device: str = "cpu"
    icd_kb: Path | None = None
    rxnorm_kb: Path | None = None
    thresholds: dict[EntityType, float] | None = None
    raw_floor: float = 0.02
    max_chunk_chars: int = 800
    max_candidates: int = 1
    selected_ids: frozenset[str] | None = None
    # Optional Qwen teachers. Without them the run is CPU-only; with them the
    # TRIỆU_CHỨNG -> CHẨN_ĐOÁN corrector runs, and additions need both.
    primary_teacher: str | None = None
    secondary_teacher: str | None = None
    teacher_device: str = "cuda"
    teacher_quantization: str = "4bit"
    teacher_batch_size: int = 48
    addition_margin: float = 1.0
    reject_margin: float | None = None


def is_header_span(text: str) -> bool:
    """True when the mention is a bare heading rather than a clinical concept."""
    stripped = " ".join(text.split())
    if len(stripped) < 2:
        return True
    if _NUMBERED_HEADING.match(stripped):
        return True
    normalized = stripped.rstrip(":;.-").strip().lower()
    if normalized in HEADER_LABELS:
        return True
    return strip_diacritics(normalized) in {
        strip_diacritics(label) for label in HEADER_LABELS
    }


def discover_inputs(input_dir: Path, selected_ids: frozenset[str] | None) -> list[Path]:
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")
    files = [path for path in input_dir.glob("*.txt") if path.stem.isdigit()]
    files.sort(key=lambda path: int(path.stem))
    if selected_ids is not None:
        files = [path for path in files if path.stem in selected_ids]
        missing = selected_ids - {path.stem for path in files}
        if missing:
            raise FileNotFoundError(f"Input IDs not found: {sorted(missing, key=int)}")
    if not files:
        raise FileNotFoundError(f"No numeric .txt files found in {input_dir}")
    return files


def build_linker(config: PipelineV2Config) -> PrecisionFirstLinker:
    icd_index = ExactAliasIndex.from_path(config.icd_kb) if config.icd_kb else None
    rxnorm_index = ExactAliasIndex.from_path(config.rxnorm_kb) if config.rxnorm_kb else None
    if icd_index is None:
        LOGGER.warning("No --icd-kb given; CHẨN_ĐOÁN candidates will all be empty")
    if rxnorm_index is None:
        LOGGER.warning("No --rxnorm-kb given; THUỐC candidates will all be empty")
    return PrecisionFirstLinker(
        icd_index=icd_index,
        rxnorm_index=rxnorm_index,
        max_candidates=config.max_candidates,
    )


def build_entities(
    raw_text: str,
    spans: list[ScoredSpan],
    linker: PrecisionFirstLinker,
) -> list[AlignedEntity]:
    entities: list[AlignedEntity] = []
    for span in spans:
        mention = raw_text[span.start : span.end]
        if is_header_span(mention):
            continue
        entities.append(
            AlignedEntity(
                text=mention,
                type=span.type,
                assertions=[],
                position=(span.start, span.end),
                candidates=linker.link(mention, span.type),
            )
        )
    return entities


def _atomic_write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def build_selector(config: PipelineV2Config, gliner_parameters: int):
    """Load the Qwen teachers, refusing to exceed the declared parameter budget."""
    if config.primary_teacher is None:
        return None
    from .selector import SpanSelector, Teacher, TeacherConfig

    def load(path: str):
        return Teacher(
            TeacherConfig(
                model_path=path,
                device=config.teacher_device,
                quantization=config.teacher_quantization,
            )
        )

    primary = load(config.primary_teacher)
    secondary = load(config.secondary_teacher) if config.secondary_teacher else None
    selector = SpanSelector(
        primary,
        secondary,
        batch_size=config.teacher_batch_size,
        addition_margin=config.addition_margin,
        reject_margin=config.reject_margin,
    )
    total = gliner_parameters + selector.total_parameters
    if total > PARAMETER_BUDGET:
        raise RuntimeError(
            f"Declared parameters {total / 1e9:.3f}B exceed the {PARAMETER_BUDGET / 1e9:.0f}B "
            "limit. Drop the secondary teacher or use smaller ones."
        )
    LOGGER.info("declared parameters: %.3fB / %.0fB", total / 1e9, PARAMETER_BUDGET / 1e9)
    if secondary is None:
        LOGGER.info("no secondary teacher: type correction only, no span additions")
    return selector


def run_pipeline_v2(config: PipelineV2Config) -> int:
    inputs = discover_inputs(config.input_dir, config.selected_ids)
    config.output_dir.mkdir(parents=True, exist_ok=True)

    extractor = GlinerSpanExtractor(
        config.model_path,
        thresholds=config.thresholds or DEFAULT_THRESHOLDS,
        raw_floor=config.raw_floor,
        max_chunk_chars=config.max_chunk_chars,
        device=config.device,
    )
    linker = build_linker(config)
    selector = build_selector(config, extractor.parameters)

    total = 0
    for index, input_path in enumerate(inputs, start=1):
        raw_text = input_path.read_text(encoding="utf-8")
        raw_spans = extractor.scored_spans(raw_text)
        spans = extractor.gate(raw_spans)
        if selector is not None:
            spans = selector.select(raw_text, spans, raw_spans)
        spans = resolve_overlaps(
            [trim_generic_prefix(raw_text, span) for span in spans]
        )
        entities = build_entities(raw_text, spans, linker)
        submission = [entity.to_submission_dict() for entity in entities]
        validate_submission_record(raw_text, submission)
        _atomic_write_json(config.output_dir / f"{input_path.stem}.json", submission)
        total += len(submission)
        LOGGER.info(
            "[%d/%d] %s: %d concepts", index, len(inputs), input_path.stem, len(submission)
        )
    if selector is not None and selector.margins_seen:
        LOGGER.info("%s", selector.rejection_report())
        # Ghi ra đĩa, không chỉ log: log của một session Kaggle mất là mất luôn,
        # mà chạy lại tốn hàng chục phút GPU. Đặt CẠNH output_dir chứ không nằm
        # trong nó, để không có đường nào lọt vào ZIP nộp bài.
        stats_path = config.output_dir.parent / "rejection_stats.json"
        _atomic_write_json(stats_path, selector.rejection_stats())
        LOGGER.info("đã lưu hiệu chuẩn bộ loại: %s", stats_path)
    return total

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/icd_vn.py
"""Build a Vietnamese ICD-10 terminology table from the official MoH catalog.

Source: *Phụ lục Bảng danh mục mã ICD-10 tiếng Việt*, issued with Thông tư
06/2026/TT-BYT (TT06), published by Bộ Y tế. The catalog ships in
``data/kb/raw/`` and is the reason a Vietnamese note can be linked at all: the
CDC ICD-10-CM descriptions used by the first submission are English only, so a
mention like ``viêm túi mật`` could never match an alias.

Three Vietnamese surfaces are harvested per row:

* ``TÊN BỆNH`` -> ``MÃ BỆNH``                        (leaf, e.g. A00.0)
* ``TÊN NHÓM BỆNH 3 KÝ TỰ`` -> ``MÃ NHÓM BỆNH 3 KÝ TỰ`` (3-char category, A00)
* ``HƯỚNG DẪN MÃ HÓA BỔ SUNG CỦA WHO 2019`` -> ``MÃ BỆNH`` (WHO synonyms)

The third column is not used by the reference solution. It is a mixed field: some
rows are clean synonyms (``Bệnh tả cổ điển``), others are inclusion notes
(``Bao gồm: ...``) or dagger/asterisk cross-references. :func:`iter_who_synonyms`
keeps only the clean synonym surfaces, which adds ~2.9k aliases and lifts the
unique-exact-match rate on our diagnosis mentions from 9.0% to 10.9%.

Output is the ``code / label / aliases`` TSV that :mod:`medical_coder.terminology`
already reads, so the result is a drop-in for ``--icd-kb``.
"""
from __future__ import annotations

import csv
import re
from pathlib import Path

CATEGORY_CODE_COLUMN = "MÃ NHÓM BỆNH 3 KÝ TỰ"
CATEGORY_NAME_COLUMN = "TÊN NHÓM BỆNH 3 KÝ TỰ"
LEAF_CODE_COLUMN = "MÃ BỆNH"
LEAF_NAME_COLUMN = "TÊN BỆNH"
WHO_SYNONYM_COLUMN = "HƯỚNG DẪN MÃ HÓA BỔ SUNG CỦA WHO 2019"

# COVID-19 codes from QĐ 98, absent from the TT06 annex.
COVID_CODES = (
    ("U07.1", "COVID-19, vi rút được xác định"),
    ("U07.2", "COVID-19, vi rút không được xác định"),
)

# Inclusion/exclusion prose in the WHO guidance column, never a usable synonym.
_GUIDANCE_PROSE = re.compile(
    r"(bao gồm|loại trừ|dùng thêm|sử dụng|xem |mã hóa)", re.IGNORECASE
)


def dotted_code(value: str) -> str:
    """Normalize an ICD-10 code to dotted form (``A001`` -> ``A00.1``)."""
    code = str(value).strip().upper().replace(".", "")
    return f"{code[:3]}.{code[3:]}" if len(code) > 3 else code


def clean_name(value: str) -> str:
    return " ".join(str(value).replace("・", " ").split()).strip(" ,;:")


def iter_who_synonyms(value: str):
    """Yield usable Vietnamese synonyms from the WHO guidance column.

    Rejects the whole cell when it carries cross-reference markup (``†``, ``*``,
    parenthesised codes) or reads as inclusion/exclusion prose, because those
    surfaces are not names a clinician would write in a note.
    """
    raw = str(value).strip()
    if not raw or raw.isdigit():
        return
    if "†" in raw or "*" in raw or "(" in raw or _GUIDANCE_PROSE.search(raw):
        return
    for part in re.split(r"[\n;]+", raw):
        candidate = part.strip().strip("+-–— ")
        if not candidate or candidate.endswith(":") or candidate.isdigit():
            return
        if 4 <= len(candidate) <= 90:
            yield candidate


def build_alias_table(xlsx_path: Path) -> dict[str, dict[str, object]]:
    """Return ``{code: {"label": str, "aliases": list[str]}}`` from the catalog."""
    try:
        import pandas as pd
    except ImportError as error:  # pragma: no cover - dependency guard
        raise RuntimeError(
            "Building the Vietnamese ICD KB needs pandas and openpyxl: "
            "python -m pip install pandas openpyxl"
        ) from error

    sheet = pd.read_excel(xlsx_path, sheet_name=0, header=None, dtype=str).fillna("")

    header_row = None
    for index in range(min(15, len(sheet))):
        if any(str(cell).strip().upper() == LEAF_CODE_COLUMN for cell in sheet.iloc[index]):
            header_row = index
            break
    if header_row is None:
        raise ValueError(f"Could not find a '{LEAF_CODE_COLUMN}' header in {xlsx_path}")

    header = [str(cell).strip() for cell in sheet.iloc[header_row]]
    body = sheet.iloc[header_row + 1 :].reset_index(drop=True)
    body.columns = header

    def column(name: str) -> str | None:
        for candidate in header:
            if candidate.strip().upper() == name:
                return candidate
        return None

    leaf_code = column(LEAF_CODE_COLUMN)
    leaf_name = column(LEAF_NAME_COLUMN)
    if leaf_code is None or leaf_name is None:
        raise ValueError(f"Missing {LEAF_CODE_COLUMN}/{LEAF_NAME_COLUMN} in {xlsx_path}")
    category_code = column(CATEGORY_CODE_COLUMN)
    category_name = column(CATEGORY_NAME_COLUMN)
    synonym_column = column(WHO_SYNONYM_COLUMN)

    table: dict[str, dict[str, object]] = {}

    def add(raw_code: str, raw_name: str) -> None:
        code = dotted_code(raw_code)
        if len(code.replace(".", "")) < 3:
            return
        name = clean_name(raw_name)
        if not name or name.lower() == "nan":
            return
        entry = table.setdefault(code, {"label": name, "aliases": []})
        aliases: list[str] = entry["aliases"]  # type: ignore[assignment]
        if name != entry["label"] and name not in aliases:
            aliases.append(name)

    for _, row in body.iterrows():
        add(row[leaf_code], row[leaf_name])
        if category_code is not None and category_name is not None:
            add(row[category_code], row[category_name])
        if synonym_column is not None:
            for synonym in iter_who_synonyms(row[synonym_column]):
                add(row[leaf_code], synonym)

    for code, name in COVID_CODES:
        add(code, name)
    return table


def write_terminology_tsv(table: dict[str, dict[str, object]], destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["code", "label", "aliases"])
        for code in sorted(table):
            entry = table[code]
            writer.writerow([code, entry["label"], "|".join(entry["aliases"])])  # type: ignore[arg-type]
    return len(table)


def build(xlsx_path: Path, destination: Path) -> int:
    return write_terminology_tsv(build_alias_table(xlsx_path), destination)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/rxnorm_kb.py
"""Build an RxNorm terminology table from RxNorm Current Prescribable Content.

The archive (``RxNorm_full_prescribe_<date>.zip``, public, no UMLS licence) is not
redistributable with the repository, so this module builds the TSV from a local
copy — see KAGGLE.md for the download step.

Only names that a clinician would actually write are kept. The term types below
cover the ingredient / brand / clinical-drug surfaces that appear in the Vòng 1
example (``amlodipine 10 mg po daily`` -> 308135, a clinical drug), while
dropping dose-form and pack noise that never matches a mention.
"""
from __future__ import annotations

import csv
import zipfile
from pathlib import Path

# RXNCONSO.RRF column offsets (pipe-delimited, no header).
RXCUI, SAB, TTY, STR, SUPPRESS = 0, 11, 12, 14, 16

# Concept classes a mention may legitimately resolve to. Every gold RxCUI in the
# Vòng 1 example is an SCD (``amlodipine 10 mg po daily`` -> 308135, "amlodipine
# 10 MG Oral Tablet"), plus ingredients and brand names for bare drug words.
#
# SCDC (ingredient + strength, e.g. 329526 "amlodipine 10 MG") is deliberately
# excluded. A dosed mention cleans to exactly an SCDC surface, so indexing it
# turns "no answer" into a *confidently wrong* answer on precisely the mentions
# most likely to carry a gold code. SCDC maps one-to-many onto SCDs by dose form,
# which is unresolvable from the mention — so under a unique-match policy the
# right output is silence.
CONCEPT_TERM_TYPES = frozenset({"SCD", "SBD", "BN", "IN", "PIN", "MIN"})

# Name rows worth indexing once a concept qualifies above. SY/PSN/TMSY are
# alternate surfaces of the same RxCUI (243670 SY "ASA 81 MG Oral Tablet").
NAME_TERM_TYPES = CONCEPT_TERM_TYPES | {"SY", "PSN", "TMSY"}

# Compact surfaces preferred as the display label.
LABEL_TERM_TYPES = frozenset({"IN", "PIN", "BN"})


def _iter_rows(handle):
    for line in handle:
        fields = line.decode("utf-8", "replace").rstrip("\n").split("|")
        if len(fields) <= SUPPRESS:
            continue
        if fields[SAB] != "RXNORM" or fields[SUPPRESS] not in ("N", ""):
            continue
        name = fields[STR].strip()
        if name:
            yield fields[RXCUI].strip(), fields[TTY], name


def build_alias_table(archive: Path) -> dict[str, dict[str, object]]:
    """Index every name of each RxCUI that qualifies as a linkable concept.

    Qualification is decided per *concept*, not per row: an SCDC RxCUI also
    carries TMSY rows, so filtering by row term type alone would let it back in.
    """
    with zipfile.ZipFile(archive) as bundle:
        member = next(name for name in bundle.namelist() if name.endswith("RXNCONSO.RRF"))

        with bundle.open(member) as handle:
            qualified = {
                rxcui
                for rxcui, term_type, _ in _iter_rows(handle)
                if term_type in CONCEPT_TERM_TYPES
            }

        table: dict[str, dict[str, object]] = {}
        with bundle.open(member) as handle:
            for rxcui, term_type, name in _iter_rows(handle):
                if rxcui not in qualified or term_type not in NAME_TERM_TYPES:
                    continue
                entry = table.setdefault(rxcui, {"label": name, "aliases": []})
                aliases: list[str] = entry["aliases"]  # type: ignore[assignment]
                if term_type in LABEL_TERM_TYPES and len(name) < len(str(entry["label"])):
                    aliases.append(str(entry["label"]))
                    entry["label"] = name
                elif name != entry["label"] and name not in aliases:
                    aliases.append(name)
    return table


def write_terminology_tsv(table: dict[str, dict[str, object]], destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["code", "label", "aliases"])
        for code in sorted(table, key=lambda value: int(value) if value.isdigit() else 0):
            entry = table[code]
            writer.writerow([code, entry["label"], "|".join(entry["aliases"])])  # type: ignore[arg-type]
    return len(table)


def build(archive: Path, destination: Path) -> int:
    return write_terminology_tsv(build_alias_table(archive), destination)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/scoring.py
"""Local reading of the Vòng 1 host metric.

    final_score = 0.3 * text_score + 0.3 * assertions_score + 0.4 * candidates_score

This is *our reading* of the published formula, not the official grader. It is
reconciled against the only two public data points we have:

* our submission 01 (14.4255 from WER 83.5952 / J_assert 20.1874 / J_cand 8.6197)
* the reference solution's 27.8786 (32.1820 / 35.2687 / 19.1084)

Both reproduce exactly under `0.3*text + 0.3*assert + 0.4*cand`, which confirms
the published `WER` field is an *error rate* and `text_score = 1 - WER`.

Choices the organisers left unspecified are all localised here so a single edit
switches convention when the official scorer is released:

* A prediction matches a ground-truth concept iff **same type** and **overlapping
  character span**, matched greedily by largest overlap. A right-text/wrong-type
  prediction therefore cannot match its twin: it is a brand-new concept scoring 0
  everywhere, which is exactly the double penalty described in the rules.
* An unmatched ("spurious") prediction is counted **twice** in every applicable
  denominator. That is the mechanism that makes over-emission so expensive, and
  it is the single most important property to optimise against.
* Aggregation is global over concepts, not a mean of per-record means.

Scoring the ground truth against itself returns 1.0.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Mapping, Sequence

from .models import EntityType

WEIGHTS = {"text": 0.3, "assertions": 0.3, "candidates": 0.4}

ASSERTABLE_TYPES = {
    EntityType.SYMPTOM.value,
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}
CANDIDATE_TYPES = {
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}


def jaccard(prediction: set[str], truth: set[str]) -> float:
    """Jaccard with the host edge cases: both empty -> 1, one empty -> 0."""
    if not prediction and not truth:
        return 1.0
    if not prediction or not truth:
        return 0.0
    return len(prediction & truth) / len(prediction | truth)


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word-level Levenshtein error rate, (S + D + I) / N."""
    ref = reference.split()
    hyp = hypothesis.split()
    if not ref:
        return 0.0 if not hyp else 1.0
    previous = list(range(len(hyp) + 1))
    for i, ref_word in enumerate(ref, start=1):
        current = [i]
        for j, hyp_word in enumerate(hyp, start=1):
            cost = 0 if ref_word == hyp_word else 1
            current.append(
                min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + cost)
            )
        previous = current
    return previous[len(hyp)] / len(ref)


def _span(concept: Mapping) -> tuple[int, int]:
    start, end = concept["position"]
    return int(start), int(end)


def _overlap(left: tuple[int, int], right: tuple[int, int]) -> int:
    return max(0, min(left[1], right[1]) - max(left[0], right[0]))


def match_concepts(
    predictions: Sequence[Mapping],
    truth: Sequence[Mapping],
) -> tuple[dict[int, int], list[int], list[int]]:
    """Greedily pair prediction -> truth on (same type, largest overlap)."""
    pairs = []
    for pi, prediction in enumerate(predictions):
        prediction_span = _span(prediction)
        for ti, gold in enumerate(truth):
            if gold.get("type") != prediction.get("type"):
                continue
            overlap = _overlap(prediction_span, _span(gold))
            if overlap > 0:
                pairs.append((overlap, pi, ti))
    pairs.sort(key=lambda item: (-item[0], item[1], item[2]))

    matched: dict[int, int] = {}
    used_predictions: set[int] = set()
    used_truth: set[int] = set()
    for _, pi, ti in pairs:
        if pi in used_predictions or ti in used_truth:
            continue
        matched[pi] = ti
        used_predictions.add(pi)
        used_truth.add(ti)
    unmatched_predictions = [i for i in range(len(predictions)) if i not in used_predictions]
    unmatched_truth = [i for i in range(len(truth)) if i not in used_truth]
    return matched, unmatched_predictions, unmatched_truth


@dataclass
class RecordScore:
    record_id: str
    text: float
    assertions: float
    candidates: float
    predicted: int
    expected: int


@dataclass
class Score:
    text_score: float
    assertions_score: float
    candidates_score: float
    final_score: float
    per_record: list[RecordScore] = field(default_factory=list)

    def format_report(self) -> str:
        return (
            f"text       {self.text_score * 100:8.4f}\n"
            f"assertions {self.assertions_score * 100:8.4f}\n"
            f"candidates {self.candidates_score * 100:8.4f}\n"
            f"final      {self.final_score * 100:8.4f}"
        )


def score_corpus(
    predictions: Mapping[str, list],
    truth: Mapping[str, list],
    record_ids: Sequence[str] | None = None,
) -> Score:
    """Score predictions against ground truth using the host formula.

    text        = sum_matched (1 - WER) / (n_truth + 2 * n_spurious)
    assertions  = sum_matched J_assert / (n_truth_assertable + 2 * n_spurious_assertable)
    candidates  = sum_matched J_cand * w / (sum_truth w + 2 * sum_spurious w),
                  where w = len(truth candidates) + 1
    """
    if record_ids is None:
        record_ids = sorted(
            truth,
            key=lambda value: (0, int(value)) if value.isdigit() else (1, value),
        )

    text_numerator = text_denominator = 0.0
    assertion_numerator = assertion_denominator = 0.0
    candidate_numerator = candidate_denominator = 0.0
    per_record: list[RecordScore] = []

    for record_id in record_ids:
        gold = truth[record_id]
        predicted = predictions.get(record_id, [])
        matched, spurious, missed = match_concepts(predicted, gold)

        record_text = sum(
            max(0.0, 1.0 - word_error_rate(gold[ti]["text"], predicted[pi]["text"]))
            for pi, ti in matched.items()
        )
        record_text_denominator = len(gold) + 2 * len(spurious)
        text_numerator += record_text
        text_denominator += record_text_denominator

        assertable_gold = [
            i for i in range(len(gold)) if gold[i].get("type") in ASSERTABLE_TYPES
        ]
        assertable_spurious = [
            i for i in spurious if predicted[i].get("type") in ASSERTABLE_TYPES
        ]
        record_assertions = sum(
            jaccard(
                set(predicted[pi].get("assertions") or []),
                set(gold[ti].get("assertions") or []),
            )
            for pi, ti in matched.items()
            if gold[ti].get("type") in ASSERTABLE_TYPES
        )
        record_assertion_denominator = len(assertable_gold) + 2 * len(assertable_spurious)
        assertion_numerator += record_assertions
        assertion_denominator += record_assertion_denominator

        record_candidates = 0.0
        record_candidate_denominator = 0.0
        for pi, ti in matched.items():
            if gold[ti].get("type") not in CANDIDATE_TYPES:
                continue
            weight = len(gold[ti].get("candidates") or []) + 1
            record_candidates += weight * jaccard(
                set(predicted[pi].get("candidates") or []),
                set(gold[ti].get("candidates") or []),
            )
            record_candidate_denominator += weight
        for ti in missed:
            if gold[ti].get("type") in CANDIDATE_TYPES:
                record_candidate_denominator += len(gold[ti].get("candidates") or []) + 1
        for pi in spurious:
            if predicted[pi].get("type") in CANDIDATE_TYPES:
                record_candidate_denominator += 2 * (
                    len(predicted[pi].get("candidates") or []) + 1
                )
        candidate_numerator += record_candidates
        candidate_denominator += record_candidate_denominator

        per_record.append(
            RecordScore(
                record_id=record_id,
                text=record_text / record_text_denominator if record_text_denominator else 1.0,
                assertions=(
                    record_assertions / record_assertion_denominator
                    if record_assertion_denominator
                    else 1.0
                ),
                candidates=(
                    record_candidates / record_candidate_denominator
                    if record_candidate_denominator
                    else 1.0
                ),
                predicted=len(predicted),
                expected=len(gold),
            )
        )

    text = text_numerator / text_denominator if text_denominator else 1.0
    assertions = (
        assertion_numerator / assertion_denominator if assertion_denominator else 1.0
    )
    candidates = (
        candidate_numerator / candidate_denominator if candidate_denominator else 1.0
    )
    final = (
        WEIGHTS["text"] * text
        + WEIGHTS["assertions"] * assertions
        + WEIGHTS["candidates"] * candidates
    )
    return Score(text, assertions, candidates, final, per_record)


def load_records(directory: Path) -> dict[str, list]:
    return {
        path.stem: json.loads(path.read_text(encoding="utf-8"))
        for path in directory.glob("*.json")
        if path.stem.isdigit()
    }

In [ ]:
IMPORT_DIR = Path("/kaggle/working/medical_coder_src")
if str(IMPORT_DIR) not in sys.path:
    sys.path.insert(0, str(IMPORT_DIR))

modules = sorted(p.name for p in (IMPORT_DIR / "medical_coder").glob("*.py"))
print(f"{len(modules)} module ->", IMPORT_DIR)
print(" ", ", ".join(modules))

import medical_coder
from medical_coder import exact_link, gliner_ner, pipeline_v2, selector, submission
print("\nmedical_coder:", medical_coder.__file__)

REPO = None   # không có repo trên đĩa; mọi thứ dựng ra nằm ở /kaggle/working

## 4. Dữ liệu đầu vào

Ưu tiên Dataset đã attach. Nếu không có Dataset nào, notebook dùng bản test Vòng 1
**nhúng sẵn** bên dưới — nhờ vậy chỉ cần tải lên đúng một tệp notebook, không cần
attach gì cả.

> Khi chấm trên private test, Ban Tổ chức sẽ cấp input khác. Lúc đó **phải** attach
> Dataset input mới; cell này sẽ tự ưu tiên nó và in rõ nguồn đang dùng, nhưng nếu
> quên attach thì nó rơi về bản public test nhúng sẵn và điểm sẽ sai. Hãy đọc dòng
> `nguồn input:` mà cell in ra.

In [ ]:
INPUT_FILES = {
    '1.txt': 'THIẾU MEN G6PD là gì? \n\n1. Thiếu men G6PD là bệnh gì?\n\nThiếu men G6PD (Glucose-6-Phosphate Dehydrogenase) là một bệnh di truyền lặn liên kết với nhiễm sắc thể X. Trẻ mắc bệnh là do nhận gen lặn bất thường trên nhiễm sắc thể giới tính từ bố và/hoặc mẹ. Chính vì gen bệnh nằm trên nhiễm sắc thể X nên bé trai có nguy cơ mắc bệnh cao hơn bé gái.\n\nNguyên nhân trực tiếp của bệnh là do đột biến gen G6PD tại vị trí Xq28. Cho đến nay, các nhà khoa học đã xác định được hơn 140 loại đột biến khác nhau tại vị trí này, tất cả đều có thể dẫn đến tình trạng thiếu hụt men G6PD. Các đột biến làm thay đổi cấu trúc bình thường của enzyme, khiến số lượng và hoạt tính của men G6PD trong hồng cầu giảm sút, từ đó gây ra rối loạn quá trình chuyển hóa và bảo vệ hồng cầu.\n\nMen G6PD đóng vai trò quan trọng trong việc bảo vệ hồng cầu khỏi các tác nhân oxy hóa. Khi thiếu men này, hồng cầu trở nên “mong manh” và dễ bị phá hủy, đặc biệt khi cơ thể tiếp xúc với thuốc, thực phẩm hoặc hóa chất có tính oxy hóa cao.\n\n2. Dấu hiệu của trẻ bị thiếu men G6PD\n\nHiện nay, trẻ sơ sinh sau khi chào đời sẽ được sàng lọc sớm các bệnh bẩm sinh, trong đó có xét nghiệm thiếu men G6PD. Các bác sĩ sẽ lấy máu khô ở gót chân trẻ để phân tích. Nếu kết quả nghi ngờ thiếu men G6PD, trẻ sẽ được chỉ định làm thêm các xét nghiệm chuyên sâu nhằm khẳng định chẩn đoán.\n\nTrong trường hợp trẻ không được phát hiện thiếu men G6PD ngay từ giai đoạn sơ sinh, cha mẹ cần đặc biệt lưu ý các dấu hiệu điển hình để sớm đưa trẻ đi khám. Khi trẻ bị thiếu men G6PD và ăn đậu tằm hoặc sử dụng thuốc, thực phẩm chứa chất oxy hóa, bệnh có thể khởi phát đột ngột với các biểu hiện như:\n • Sốt cao\n3.  Đánh giá tại bệnh viện • Tim đập nhanh, khó thở\n • Vàng da, vàng mắt\n\nKhi đi khám, xét nghiệm máu thường cho thấy thiếu máu do tan huyết, hồng cầu bị phá hủy hàng loạt, dẫn đến thiếu máu, vàng da vàng mắt và thậm chí suy thận cấp. Đặc biệt, nếu trẻ sơ sinh bị vàng da nặng sau khoảng 2 tuần tuổi, các tổn thương thần kinh có thể rất nghiêm trọng, gây bại não, chậm phát triển trí tuệ và vận động.\n\nVới những trường hợp biểu hiện nhẹ, trẻ chỉ cần được theo dõi, tránh các yếu tố gây tan huyết và tình trạng thiếu máu thường sẽ ổn định. Tuy nhiên, nếu các triệu chứng rõ ràng và nặng, trẻ cần được nhập viện để điều trị kịp thời.\n\n3. Thiếu men G6PD có nguy hiểm không?\n\nMen G6PD do hồng cầu sản sinh có nhiệm vụ bảo vệ hồng cầu khỏi sự tấn công của các chất oxy hóa. Khi thiếu men này, hồng cầu rất dễ bị phá hủy, dẫn đến thiếu máu tan huyết.\n\nỞ trẻ sơ sinh, vàng da nặng do thiếu men G6PD có thể gây ra nhiều biến chứng nguy hiểm như:\n • Bại não\n • Chậm phát triển trí tuệ\n • Rối loạn vận động\n\nNếu không được điều trị và theo dõi đúng cách, tình trạng thiếu máu tan huyết kéo dài kèm vàng da sơ sinh sẽ trở thành vấn đề nghiêm trọng, ảnh hưởng lâu dài đến sức khỏe và sự phát triển của trẻ. Vì vậy, khi phát hiện trẻ bị thiếu men G6PD thông qua sàng lọc sớm, phụ huynh cần tuân thủ hướng dẫn của bác sĩ, đặc biệt là tránh các tác nhân gây tan huyết như đậu tằm, thuốc và thực phẩm có tính oxy hóa cao.\n\nTuy nhiên, không phải mọi người thiếu men G6PD đều luôn trong tình trạng nguy hiểm. Nếu biết cách phòng tránh và sinh hoạt hợp lý, người bệnh vẫn có thể sống và phát triển hoàn toàn bình thường.\n\n4. Cần làm gì khi con bị thiếu men G6PD?\n\nTheo khuyến cáo của bác sĩ, trẻ thiếu men G6PD cần tránh các yếu tố có thể gây ảnh hưởng xấu đến sức khỏe như:\n • Nhiễm khuẩn, nhiễm virus\n • Thuốc giảm đau, hạ sốt chứa ******* hoặc **********\n • Kháng sinh nhóm ***********, ********\n • Thuốc kháng sốt rét như *******, ***********, **********\n • Vitamin K dùng trong điều trị nhiễm khuẩn tiết niệu\n • Thực phẩm chế biến từ đậu tằm\n • Tiếp xúc với băng phiến, long não\n • Không nên hiến máu\n\nCha mẹ cần đặc biệt ghi nhớ:\n • Không sử dụng thuốc, thực phẩm hay hóa chất có nguy cơ gây tan huyết\n • Luôn thông báo cho nhân viên y tế về tình trạng thiếu men G6PD của trẻ\n • Không dùng long não, băng phiến trong tủ quần áo, chăn màn\n • Cẩn trọng với thuốc nam, thuốc đông y và một số loại đậu\n • Mẹ đang cho con bú không sử dụng các chất chống chỉ định vì có thể truyền qua sữa\n • Không tự ý mua thuốc cho trẻ khi chưa có chỉ định của bác sĩ\n\nPhòng ngừa thiếu men G6PD\n\nĐể hạn chế nguy cơ trẻ mắc thiếu men G6PD, xét nghiệm sàng lọc trước sinh và sau sinh đóng vai trò vô cùng quan trọng. Các gói chăm sóc thai sản hiện nay đã tích hợp xét nghiệm sàng lọc giúp phát hiện sớm bệnh cho trẻ ngay sau sinh, từ đó có kế hoạch theo dõi và chăm sóc phù hợp',
    '2.txt': '1. Bệnh Kawasaki là gì?\n\nBệnh Kawasaki là tình trạng sốt cấp kéo dài, thường đi kèm phát ban toàn thân, với đặc điểm chính là viêm lan tỏa hệ mạch máu nhỏ và vừa, bao gồm động mạch vành – mạch máu quan trọng cung cấp máu cho tim. Bệnh được mô tả lần đầu bởi bác sĩ Tomisaku Kawasaki (Nhật Bản) vào năm 1967.\n • Độ tuổi thường gặp: Chủ yếu ở trẻ dưới 5 tuổi, đặc biệt là nhóm bú mẹ.\n • Giới tính: Trẻ trai mắc bệnh nhiều hơn trẻ gái.\n • Đặc điểm nguy hiểm: Giai đoạn đầu có thể chưa quá nghiêm trọng, nhưng nếu không điều trị kịp thời, bệnh có thể dẫn đến viêm tim, phình giãn động mạch vành, đột tử, nhồi máu cơ tim, hoặc hẹp tắc mạch vành gây suy tim về sau.\n\nTóm lại: Kawasaki là bệnh viêm mạch máu nguy hiểm, cần phát hiện sớm để tránh biến chứng nặng.\n2. Nguyên nhân gây bệnh Kawasaki\nMặc dù được nghiên cứu nhiều, cho đến nay nguyên nhân chính xác vẫn chưa rõ ràng. Nhiều chuyên gia đưa ra các giả thuyết:\nCác yếu tố nghi ngờ liên quan:\n • Nhiễm khuẩn – nhiễm virus hoặc độc tố vi khuẩn.\n • Phản ứng miễn dịch bất thường ở trẻ có cơ địa nhạy cảm.\n • Yếu tố chủng tộc: Trẻ gốc Á, đặc biệt trẻ Nhật Bản và Hàn Quốc, có tỉ lệ mắc cao.\n • Yếu tố môi trường: thay đổi khí hậu, chất độc trong không khí.\nĐiều quan trọng: Bệnh không lây từ trẻ này sang trẻ khác và đến nay chưa có bằng chứng về lây truyền trong cộng đồng.\n3. Triệu chứng bệnh Kawasaki ở trẻ em\nTriệu chứng điển hình\n(Ít nhất 5 trong số các dấu hiệu sau, kèm sốt ≥5 ngày)\n1. Sốt cao kéo dài\n • Sốt 39–40°C trong hơn 5 ngày.\n • Ít đáp ứng với hạ sốt hoặc kháng sinh.\n\n2. Mắt đỏ\n • Viêm kết mạc 2 bên, đỏ nhưng không có ghèn.\n\n3. Thay đổi niêm mạc miệng\n • Môi đỏ, nứt, có thể rỉ máu.\n • Lưỡi đỏ như dâu tây.\n • Họng đỏ.\n4. Tổn thương ở đầu chi\n • Sưng, đỏ mu bàn tay – chân.\n • Đỏ gan bàn tay – chân.\n • Bong da đầu ngón tay, ngón chân vào ngày 7–14 của bệnh.\n\n5. Ban đỏ toàn thân\n • Ban dạng đa hình: dát – sẩn – mảng đỏ.\n\n    - Sẽ không điển hình cho Bệnh đa xơ cứng theo ý kiến của bác sĩ thần kinh\n    - Ảo giác do rượu (suy nghĩ)\n    - Không được coi là thực sự loạn thần • Công thức máu, CRP, máu lắng\n • Men gan, albumin\n • Xét nghiệm nước tiểu\n • Cấy máu, dịch hầu họng\n • Siêu âm tim (quan trọng nhất để đánh giá động mạch vành)\n • ECG điện tâm đồ\n\n6. Điều trị bệnh Kawasaki ở trẻ\nĐiều trị cần được thực hiện tại bệnh viện, đặc biệt là bệnh viện có chuyên khoa tim mạch nhi.\n1. ********************* – Điều trị quan trọng nhất\n • Dùng liều cao truyền tĩnh mạch.\n • Hiệu quả 80% nếu truyền trước ngày thứ 10 của bệnh.\n • Giảm biến chứng động mạch vành rõ rệt.\n2. ******* (ASA)\n • Dùng liều cao giai đoạn cấp tính để giảm viêm.\n • Sau đó giảm liều duy trì để ngừa huyết khối.\n3. Điều trị lặp lại nếu cần\n • Một số trẻ cần truyền **** lần 2 hoặc sử dụng thuốc ức chế miễn dịch khác\n7. Theo dõi trẻ sau điều trị\n\nTrẻ cần được theo dõi 6 tháng – 1 năm:\n • Tái khám định kỳ, siêu âm tim đánh giá động mạch vành.\n • Dùng thuốc đúng chỉ định.\n • Tránh tiêm vắc xin sống ngay sau khi truyền **** (thường phải hoãn 9–11 tháng).\n\n8. Chế độ dinh dưỡng và vận động cho trẻ mắc Kawasaki\nDinh dưỡng\n • Ăn uống lành mạnh, nhiều rau – trái cây.\n • Hạn chế đồ ăn quá nhiều dầu mỡ.\n • Ưu tiên thực phẩm tốt cho tim mạch.\nVận động\n • Trẻ nên tập luyện nhẹ nhàng.\n • Tham gia các hoạt động phù hợp độ tuổi.\n • Tránh hoạt động mạnh nếu có tổn thương vành mạch.\nTránh nguy cơ tim mạch\n • Trẻ lớn nên tránh hút thuốc lá thụ động, môi trường ô nhiễm.\nKết luận\nBệnh Kawasaki là bệnh nguy hiểm nhưng hoàn toàn có thể điều trị hiệu quả nếu được phát hiện sớm. Cha mẹ cần đưa trẻ đến bệnh viện ngay khi trẻ sốt cao kéo dài kèm các dấu hiệu như đỏ mắt, ban đỏ, môi đỏ – nứt, lưỡi đỏ dâu tây hoặc sưng hạch cổ. Điều trị bằng **** trong 10 ngày đầu là yếu tố then chốt giúp giảm biến chứng mạch vành.',
    '3.txt': '1.  Tiền sử bệnh\n nhưng hiện tượng run tay không khỏi hẳn. Em muốn hỏi có phải run tay là phản ứng phụ của thuốc không và mẹ em nên làm gì để cải thiện tình trạng này. Ngoài ra việc sử dụng Nitralmyl lâu dài có gây ảnh hưởng xấu gì không ạ?\n Em xin cảm ơn Bác sĩ.\nCâu trả lời của bác sĩ: \nChào bạn, kể từ tháng 6/2012 thuốcVastarel (trimetazidin) đã được cảnh báo ở Pháp, và sau đó là ở Châu Âu và trên toàn thế giới về việc lưu ý tác dụng bất lợi của thuốc này là lớn hơn lợi ích của thuốc: \nnhư làm nặng hơn hội chứng Parkinson, gây run tay chân, mất thăng bằng khi đi hoặc run rấy toàn thân. Chính vì vậy mà thuốc bị giới hạn chỉ định "điều trị triệu chứng đau thắt ngực ổn địnhkhi bệnh nhân\n không dung nạp với các thuốc khác hoặc các thuốc khác dùng không hiệu quả". Thuốc này chống chỉ định ở người bị chóng mặt, ù tai, rối loạn thị lực, đặc biệt không dùng thuốc ở người bị Parkinson\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện\n    - yếu sức nửa người bên phải\n    - tình trạng tri giác giảm sút\n    - nhìn song thị\n    Thời điểm khởi phát triệu chứng\n    - Thời gian: khoảng 11:00 sáng\n    - Khởi phát cấp tính hay từ từ: Không ghi rõ\n    Diễn biến bệnh\n    - Được con trai phát hiện tại nhà lúc khoảng 11:00 sáng, nằm sải trên sàn.\n    - Than phiền về không thể tự đứng dậy do yếu sức chân phải.\n    - Không nhớ cách mình bị ngã xuống sàn hay lý do tại sao mình bị ngã.\n    - không thể chịu lực ở chân phải, liên tục khuỵu chân khi được giúp đỡ đứng dậy.\n    - Đã được chụp chụp ct sọ não kết quả âm tính.\n    - Đội ngũ y tế tiến hành chọc dò dịch não tủy.\n    - Bị một cơn nhịp tim chậm nặng và hạ huyết áp ngay sau khi study.\n    - Cơn bệnh tự khỏi sau khi dùng intravenous fluids.\n    - Chụp lại chụp ct sọ não, kết quả âm tính.\n    - Chuyển đến bệnh viện để tiếp tục điều trị.\n    - Được gọi là Code tai biến mạch máu não.\n    - Ban đầu tỉnh chậm phản xạ kém.\n    - Dần dần tỉnh táo hơn tỉnh táo và tỉnh khi gọi.\n    - Phủ nhận các bệnh nền lớn tiền sử bệnh hoặc triệu chứng nào khác ngoại trừ nhiễm trùng nhiễm trùng răng miệng gần đây cần dùng kháng sinh.\n    - Phủ nhận tiền sử bị tai biến mạch máu não hoặc co giật.\n    Triệu chứng khi nhập viện\n    - yếu sức nửa người bên phải\n    - tình trạng tri giác giảm sút\n    - nhìn song thị\n    - mệt mỏi\n    Đặc điểm triệu chứng\n    - Vị trí: yếu chân phải\n    - Mức độ nghiêm trọng: Không ghi rõ\n    - Thời gian: Không ghi rõ\n    - Tần suất: Không ghi rõ\n    - Lan tỏa: Không ghi rõ\n    - Các yếu tố làm nặng thêm: Không ghi rõ\n    - Các yếu tố làm giảm bớt: Không ghi rõ\n    - Triệu chứng liên quan: mệt mỏi\n    Các sự kiện trước khi nhập viện\n    - Được con trai phát hiện tại nhà.\n    - Than phiền về yếu sức chân phải.\n    - Đã được chụp chụp ct sọ não kết quả âm tính.\n    - Tiến hành chọc dò dịch não tủy.\n    - Bị nhịp tim chậm nặng và hạ huyết áp, không đặc hiệu.\n    - Cơn bệnh tự khỏi sau khi dùng intravenous fluids.\n    - Chụp lại chụp ct sọ não, kết quả âm tính.\n    - Đã dùng kháng sinh cho nhiễm trùng nhiễm trùng răng miệng gần đây.\n    Tình trạng ngay trước khi nhập viện\n    - Nằm sải trên sàn.\n    - không thể tự đứng dậy do yếu sức chân phải.\n    - không thể chịu lực ở chân phải, liên tục khuỵu chân.\n    - tỉnh chậm, phản xạ kém.\n\n3.  Đánh giá tại bệnh viện\n    Dấu hiệu lâm sàng\n    - yếu nửa người bên phải\n    - suy giảm tri giác\n    - nhìn song thị\n    - nhịp tim chậm\n    - hạ huyết áp, không đặc hiệu\n    - không nhấc chân phải khỏi mặt giường\n    - chân phải liên tục khuỵ xuống\n    - tỉnh chậm, phản xạ kém\n    Kết quả xét nghiệm\n    - Chẩn đoán hình ảnh\n    - chụp ct sọ não: âm tính\n    - Chụp lại chụp ct sọ não: âm tính\n    Các thủ thuật đã thực hiện\n    - chụp ct sọ não\n    - chọc dò dịch não tủy\n    - Truyền dịch tĩnh mạch',
    '4.txt': '1.  Tiền sử bệnh\n    Các tập phát bệnh tương tự trước đây\n    - Tái phát buồn nôn và tiêu chảy cách đây vài năm\n    - Nôn ra máu 2 tuần sau tập viêm dạ dày ruột do virus\n    - Nôn ra máu trong 2 ngày qua\n    Bệnh lý mãn tính: hội chứng ruột kích thích (được chẩn đoán hội chứng ruột kích thích cách đây vài năm)\n    Tiền sử phẫu thuật / thủ thuật\n    - Nội soi (cách đây vài năm, phát hiện loét tá tràng)\n    - Nội soi (cho thấy nhiều loét tá tràng và hồi tràng)\n    - nội soi (Viêm thực quản độ C, loét thực quản dưới 6 mm có điểm sắc tố, nhiều loét nông sạch đáy ở tá tràng và hồi tràng sớm)\n    Thuốc trước khi nhập viện: omeprazole (vừa ngừng để làm test hơi thở h. pylori)\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Nôn ra máu\n    Thời điểm khởi phát triệu chứng\n    - Triệu chứng cách đây vài năm (buồn nôn và tiêu chảy)\n    - Tập viêm dạ dày ruột do virus (trong năm nay)\n    - Tiếp tục khó khăn khi ăn uống qua đường miệng, buồn nôn và tiêu chảy (2 tuần sau tập viêm dạ dày ruột do virus)\n    - Nôn ra máu (2 tuần sau tập viêm dạ dày ruột do virus)\n    - Nôn ra máu (trong 2 ngày qua)\n    - Tình trạng đau bụng và nôn ra máu trở nên tồi tệ hơn (khi ngừng omeprazole)\n    Diễn biến bệnh\n    - Triệu chứng cách đây vài năm: tái phát buồn nôn và tiêu chảy, chẩn đoán hội chứng ruột kích thích, phát hiện loét tá tràng khi nội soi.\n    - Trong năm nay: không thể giữ được bất cứ thứ gì, chẩn đoán viêm dạ dày ruột do virus.\n    - 2 tuần sau điều trị  viêm dạ dày ruột do virus: tiếp tục khó khăn khi ăn uống qua đường miệng, buồn nôn, tiêu chảy, nôn ra máu. nội soi cho thấy nhiều loét tá tràng và hồi tràng. dương tính xét nghiệm phân tìm cryptosporidium.\n    - Gần đây: ngừng sử dụng omeprazole để làm test hơi thở h. pylori.\n    - Khi ngừng omeprazole: phát triển đau bụng và nôn ra máu trở nên tồi tệ hơn.\n    - 2 ngày qua: tiếp tục nôn ra máu.\n    Triệu chứng khi nhập viện\n    - Nôn ra máu\n    - Ăn uống kém.\n    - nausea\n    - diarrhea\n    - abdominal pain\n    Các sự kiện trước khi nhập viện\n    - Chẩn đoán viêm dạ dày ruột do virus (trong năm nay)\n    - Tiếp tục khó khăn khi ăn uống qua đường miệng, buồn nôn và tiêu chảy (2 tuần sau tập viêm dạ dày ruột do virus)\n    - Nhập viện vì nôn ra máu (2 tuần sau tập viêm dạ dày ruột do virus)\n    - nội soi cho thấy nhiều loét tá tràng và hồi tràng (2 tuần sau tập viêm dạ dày ruột do virus)\n    - dương tính xét nghiệm phân tìm cryptosporidium (2 tuần sau tập viêm dạ dày ruột do virus)\n    - Ngừng sử dụng omeprazole để làm test hơi thở h. pylori (gần đây)\n    - Phát triển đau bụng và nôn ra máu trở nên tồi tệ hơn (khi ngừng omeprazole)\n\nChào bạn, bạn mắc ung thư tuyến giáp và chắc đã phẫu thuật, hiện bác sĩ đang kê đơn cho bạn uống levothyroxine với liều 75 microgam/ngày (lưu ý: một viên thuốc Berlthyrox có hàm lượng là 100 microgam chứ không phải 100 miligam như bạn đã nói), là hormone tuyến giáp tổng hợp, nhằm bù lại lượng hormone giáp cơ thể không thể sản sinh ra để duy trì trạng thái sinh lý cân bằng của chức năng tuyến giáp (còn gọi là trạng thái bình giáp). Các nghiên cứu trên phụ nữ cho con bú dùng thuốc levothyroxine cho thấy rằng lượng hormone giáp bài tiết vào sữa mẹ là rất ít (nồng độ khoảng 4 nanogam/mL), thậm chí khi dùng liều cao ************* (200-300 microgam/ngày), và do đó không gây hại cho trẻ bú mẹ. Nói chung, tốt nhất người mẹ cần cho bé bú xong trước khi dùng thuốc để giảm thiểu đến mức thấp nhất những ảnh hưởng với bé. Bạn cũng cần lưu ý khi chia nhỏ viên thuốc để đảm bảo liều lượng chính xác.    - Viêm thực quản độ C ở thực quản dưới\n    - Loét loét thực quản 6 mm có điểm sắc tố\n    - Nhiều loét sạch đáy loét ở tá tràng kéo dài từ hành tá tràng đến hồi tràng sớm',
    '5.txt': '1.  Tiền sử bệnh nội khoa\n    Tiền sử phẫu thuật / thủ thuật: Đã thực hiện phẫu thuật cắt bỏ ống dẫn mật chung, cắt bỏ một phần thùy gan bên trái và nối mật tụy bằng ống dẫn hồi tràng (RNY hepaticojejunostomy) vào ngày [Ngày] vì nghi ngờ ung thư biểu mô tế bào mật không thể cắt bỏ.\n\n2.  Tiền sử bệnh hiện tại\n\nChào bạn\nCó rất nhiều xét nghiệm để đánh giá vô sinh. Bạn xây dựng gia đình 2 năm mà chưa có thai em cần phải đi khám để xác định nguyên nhân. Tối thiểu các xét nghiệm cần có là:\n- Đối với chồng: khám và làm xét nghiệm tinh dịch đồ.\n- Đối với vợ: cần khám và làm tối thiểu các xét nghiệm cận lâm sàng như siêu âm (đánh giá các bất thường về giải phẫu), chụp tử cung vòi trứng (đánh giá hình thái buồng tử cung và độ thông của vòi trứng), xét nghiệm nội tiết (đánh giá chức năng của hệ thống nội tiết, đặc biệt là chức năng buồng trứng).\nKinh nguyệt của bạn thưa, uống thuốc tránh thai kinh nguyệt bình thường bạn cần đi khám để đánh giá bệnh lý cụ thể. Rất có thể bạn bị Hội chứng buồng trứng đa nang (bao gồm rậm lông, béo phì, siêu âm buồng trứng có nhiều nang …) và đây cũng là nguyên nhân gây vô sinh.\nChúc bạn nhiều sức khoẻ!    - Bà ấy phủ nhận bất kỳ buồn nôn, nôn, sốt, ớn lạnh, hoặc dịch rò rỉ quanh ống thông.\n    - Đã thực hiện cholangiogram vào ngày [Ngày] cho thấy tắc nghẽn kéo dài gần chỗ nối mật tụy, ảnh hưởng đến ống dẫn trước và sau bên phải.\n    - Cả hai ống dẫn mật đều được tăng kích cỡ từ 8.5fr lên 10.2 fr vào thời điểm đó.\n    - Đã thực hiện sinh thiết và lấy mẫu bằng bàn chải vào ngày [Ngày] và kết quả lấy mẫu bằng bàn chải cho thấy tế bào bất thường, đáng ngại cho ung thư biểu mô tuyến.\n    - Thảo luận tại cuộc họp [Tên cuộc họp] vào ngày [Ngày] và quyết định rằng bệnh nhân nên được đặt stent thành vách vĩnh viễn.\n    - Bà ấy đã được đặt 3 stent thành vách vĩnh viễn và có 3 ống dẫn mật ngoài-trong tại chỗ.\n    - Bệnh nhân dung nạp thủ thuật tốt.\n    - Sau thủ thuật, bệnh nhân phủ nhận bất kỳ buồn nôn, nôn, đau ngực, hoặc khó thở.\n    - Bà ấy cho biết có một chút đau khi sờ nắn xung quanh vị trí đặt ống dẫn, đặc biệt là khi hít thở sâu.\n    Sự kiện trước khi nhập viện\n    - Được bác sĩ [Tên bác sĩ] khám tại phòng khám vào ngày [Ngày]. Bệnh nhân tình trạng rất tốt.\n    - ăn uống bình thường, cân nặng đã cải thiện, trở lại các hoạt động hàng ngày bình thường.\n    - Có ống dẫn mật xuyên gan tại chỗ.\n    - Phủ nhận buồn nôn, nôn, sốt, ớn lạnh, hoặc dịch rò rỉ quanh ống thông.\n    - Đã thực hiện cholangiogram vào ngày [Ngày] cho thấy tắc nghẽn kéo dài gần chỗ nối mật tụy, ảnh hưởng đến ống dẫn trước và sau bên phải.\n    - Ống dẫn mật được tăng kích cỡ từ 8.5fr lên 10.2 fr vào thời điểm đó.\n    - Đã thực hiện sinh thiết và lấy mẫu bằng bàn chải vào ngày [Ngày] và kết quả lấy mẫu bằng bàn chải cho thấy các tế bào bất thường, đáng lo ngại cho ung thư biểu mô tuyến.\n    - Thảo luận tại cuộc họp [Tên cuộc họp] vào ngày [Ngày] và quyết định rằng bệnh nhân nên được đặt stent thành vách vĩnh viễn.\n\n3.  Đánh giá tại bệnh viện\n    Dấu hiệu lâm sàng: đau khi sờ nắn xung quanh vị trí đặt ống dẫn, đặc biệt là khi hít thở sâu\n    Kết quả phòng thí nghiệm: lấy mẫu bằng bàn chải cho thấy tế bào bất thườngtế bào bất thường, đáng ngại cho ung thư biểu mô tuyến\n    Kết quả chẩn đoán hình ảnh: cholangiogram vào ngày [Ngày] cho thấy tắc nghẽn kéo dài gần chỗ nối mật tụy, ảnh hưởng đến ống dẫn trước và sau bên phải\n    Thủ thuật thực hiện: Đặt 3 stent thành vách vĩnh viễn',
    '6.txt': '1.  Tiền sử bệnh lý\n\nBệnh tim mạch do xơ vữa động mạch.\năng huyết áp.Rối loạn lipid máu.\nĐái tháo đường.2. \nBệnh sử\nLý do vào viện: Cơn rối loạn ý thức thoáng qua, nghi ngờ cơn co giật.\nKhoảng 1 tháng trước nhập viện, bệnh nhân bị ngã và được chẩn đoán xuất huyết dưới nhện. Chụp cắt lớp vi tính sọ não cho hình ảnh  xuất huyết dưới nhện vùng trán phải, bầm dập nhu mô vùng trán phải và trán - thái dương phải, kèm một lớp dịch dưới màng cứng mỏng vùng thùy trán phải, nghĩ nhiều đến nang màng nhện hoặc tụ dịch/tụ máu dưới màng cứng mạn tính. \nCác lần chụp CT theo dõi sau đó ổn định và bệnh nhân được xuất viện, hẹn tái khám chuyên khoa Ngoại thần kinh.\nSau xuất viện, bệnh nhân từng quay lại khoa Cấp cứu vì đau đầu kéo dài. \nChụp kiểm tra ghi nhận tụ máu ngoài màng cứng phải cấp tính trên nền tổn thương mạn tính. \nBệnh nhân tiếp tục được theo dõi, các lần chụp CT kiểm tra không ghi nhận diễn tiến xấu nên được xuất viện về nhà.\nTrong thời gian sau đó, bệnh nhân tương đối ổn định nhưng vẫn còn đau đầu vùng thái dương phải và đỉnh đầu, kèm cảm giác tê bì nửa mặt phải, đặc biệt vùng trán phải và da đầu phải từ sau lần ngã.\nNgày vào viện, buổi sáng bệnh nhân thức dậy tỉnh táo, sinh hoạt bình thường, ăn sáng, đọc báo và sử dụng máy tính. Đến cuối buổi sáng, khi đang đi trong nhà, bệnh nhân nói muốn đi vệ sinh. \nNgười nhà nhận thấy bệnh nhân có biểu hiện bất thường, mất định hướng, đi lại không vững và gần như ngất. \nBệnh nhân lúc mở mắt, lúc nhắm mắt, nhiều lần ngửa đầu ra sau và không đáp ứng được các câu hỏi của người nhà.Sau đó bệnh nhân được dìu nằm nghỉ. \nTrong cơn, bệnh nhân kích thích nhẹ, liên tục cố gắng đứng dậy và tỏ ra không hiểu tình trạng đang xảy ra. \nTình trạng ý thức dao động kéo dài khoảng 20 phút rồi cải thiện dần. \nKhi nhân viên cấp cứu đến nơi, bệnh nhân bắt đầu tỉnh táo trở lại và hồi phục hoàn toàn về trạng thái ban đầu.\nBệnh nhân không ghi nhận run giật tay chân, không cứng đờ, không cắn lưỡi và không tiểu tiện không tự chủ trong cơn.\nTriệu chứng khi nhập viện:\nĐau đầu vùng thái dương phải, đỉnh đầu, đôi khi lan ra sau mắt phải.\nTê bì vùng trán phải, da đầu phải và nửa mặt phải.\nCơn rối loạn ý thức thoáng qua.Mất định hướng.\nMất thăng bằng.Gần ngất.\nTình trạng trước nhập viện: Sau cơn bệnh nhân tỉnh táo hoàn toàn, ý thức trở về mức nền ban đầu.\n3. Đánh giá tại bệnh viện\nKhám lâm sàng:\nCó ghi nhận giai đoạn mất định hướng và đi lại không vững theo lời kể người nhà.\nTrong cơn có biểu hiện kích thích nhẹ, ngửa đầu ra sau và nhắm mắt từng lúc.\nChụp cắt lớp vi tính (CT Scanner): Bụng - Tiểu khung thường quy (máy 64 đến 128 dãy, có tiêm thuốc cản quang) — Không in phim.3. \nThủ thuật - Dịch vụ kỹ thuật\nĐiện tâm đồ: Ghi điện tim cấp cứu tại giường.II. Kết quả xét nghiệm & Cận lâm sàng đã có\nXét nghiệm có kết quả : \nKết quả Cận lâm sàng\n. Xét nghiệm Máu Huyết học & Đông máu:WBC : 14.99 G/L NEUT% : 82.9 % (Tăng) \nHGB (Hemoglobin): 92 g/L PT - INR: 1.05\nCận lâm sàng: CT sọ não trước đó ghi nhận xuất huyết dưới nhện vùng trán phải.Bầm dập nhu mô vùng trán phải và trán - thái dương phải.Lớp dịch dưới màng cứng mỏng vùng thùy trán phải, nghĩ đến nang màng nhện hoặc tụ máu dưới màng cứng mạn tính.Tụ máu ngoài màng cứng phải cấp tính trên nền tổn thương mạn tính.\n',
    '7.txt': 'Hỏi : Kính chào bác sĩ! Em 37 tuổi và hiện đang mang thai lần 2 được 20 tuần \n. Từ năm 24-26 tuổi em có dấu hiệu đau bao tử (thỉnh thoảng đau và sau 1-2 tiếng thì hết do thời gian này em sống xa gia đình nên ăn uống thất thường) nhưng không khám cũng như không uống thuốc gì cả. Năm 27 tuổi em đi nội soi ở BV thì BS nói em có ổ loét trong bao tử và có cho thuốc uống theo lộ trình dài nhưng em chỉ uống hết liều thuốc lần 1 đó mà không tiếp tục điều trị. Thay vào đó em đã chuyển về sống cùng gia đình và ăn uống rất điều độ nên hầu như em không bị đau lại nữa. Sau đó em uống tinh bột nghệ tách tinh dầu  vào mỗi buổi sáng khi ngủ dậy. Em thấy không còn bị đau như trước và cơ thể cũng có cảm giác khỏe lên. Nay em đang có thai lần 2 nên em ngưng không uống bôt nghệ nữa. Tuy thai mới được 20w nhưng em đã tăng 10kg (mặc dù em ăn nhiêu ói bấy nhiêu). Đi khám bác sĩ cũng tư vấn em hạn chế ăn tinh bột. Tối hôm 03/5/2021 em không ăn cơm mà chỉ ăn rau luộc sau đó ăn thêm 2 quả chuối và uống 1 chai nhỏ Yakult. Đến 21h em bắt đầu đau bụng râm ran và cơn đau càng dồn đạp đến 12h đêm. Em đau đến mức chỉ biết ôm bụng mà không thể nhúc nhích được chân tay vì quá đau. Người nhà em cũng rất vất vả mới thay được quần áo cho em để đưa em đi viện. Em có cảm giác hàng ngàn sợi chỉ đang thắt ruột gan em lại. Đến BV BS chích cho em liều giảm đau và cho siêu âm nhưng kết quả không thấy gì cả (chỉ thấy mỗi bên thận có 1 viên sỏi 4mm nhưng không ứ nước). BS nói sỏi thận không phải là lý do làm em đau và nói khả năng em bị viêm bao tử, có nói do em đang có thai nên không thể dùng thuốc đặc trị được mà chỉ cho thuốc tạm thôi. Em muốn hỏi như sau: 1. Danh sách thuốc BS kê có ảnh hưởng đến em bé trong bụng em không: - ****** (trong HDSD có ghi "An toàn của ****** trên phụ nữ có thai chưa được thiết lập") - ******* (*******************, ****************) - ************** 2. Các xét nghiệm/siêu âm/nội soi cần thiết nào cho em mà vẫn an toàn với thai nhi? Em cảm ơn nhiều!\nTrả lời : Chào em,\nThành phần 3 loại thuốc em sử dụng như sau:\nAquima: nhôm hydroxid, magie hydroxid và **********.\nSimenic: Alverin citrate 40mg và Simethicon 100mg\nPimperam: metoclopramide 10mg\nThông tin về độ an toàn khi sử dụng các loại thuốc trên ở phụ nữ mang thai như sau:\nNhôm hydroxid và magie hydroxid có thể sử dụng được cho phụ nữ mang thai để điều trị triệu chứng ợ nóng hoặc trào ngược dạ dày thực quản và cần sử dụng trong mức liều khuyến cáo.\nHiện tại chưa có đủ nghiên cứu về việc sử dụng ********** trên phụ nữ mang thai. Mặc dù chưa khẳng định được thuốc có qua nhau thai hay không nhưng hấp thu của ********** qua ruột bị hạn chế, do đó làm giảm khả năng phơi nhiễm đối với thai nhi.\n    Triệu chứng khi nhập viện\n    - tổn thương vùng âm hộ và mông bên phải\n    - Tổn thương cực kỳ đau đớn\n    - Tình trạng tổn thương dạng bóng nước ngày càng nặng\n    - Lan đến mông bên phải\n    - có dịch giống mủ có màu vàng  chảy ra từ tổn thương\n    Đặc điểm của triệu chứng\nEm nên đến bệnh viện để được bác sĩ chuyên khoa tư vấn kĩ hơn về các xét nghiệm/siêu âm/nội soi cần thiết mà không ảnh hưởng đến thai nhi cũng như được tư vấn về chế độ ăn phù hợp, tránh tình trạng nhịn ăn dẫn đến viêm dạ dày em nhé!\nChúc em luôn vui khỏe',
    '8.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mạn tính\n    - ho Rối loạn cảm xúc (trầm cảm) (tiền sử giai đoạn trầm cảm không đặc hiệu)\n    - hội chứng nghiện rượu\n    Thuốc trước khi nhập viện lần này: seroquel (ngụ ý, vì bệnh nhân cho rằng lú lẫn, chóng mặt, khó nhìn gần có thể liên quan đến thuốc)\n    Các yếu tố nguy cơ liên quan: hội chứng nghiện rượu\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Bày tỏ tự tử với nhân viên khi xuất viện\n    Triệu chứng hiện tại\n    - tự tử\n    - Lú lẫn từng đợt gần đây\n    - cảm giác chóng mặt\n    - khó khăn khi nhìn gần\n    - rối loạn thị giác, nhìn thấy con cái mình ở góc mắt\n    - ảo giác thị giác (ảo giác thị giác)\n    - Ảo thanh (AH) (ảo giác thính giác)\n    Đặc điểm triệu chứng\n    - tự tử: thụ động, diễn ra liên tục, tưởng tượng về cái chết mãn tính kể từ khi ly hôn\n    - Lú lẫn: từng đợt\n    - chóng mặt\n    - Khó khăn về thị lực gần\n    - rối loạn thị giác: nhìn thấy con cái ở góc mắt\n    - ảo giác thị giác, Ảo thanh (AH): có vẻ là ảo giác hơn là ảo ảnh thực sự\n    Các sự kiện trước khi nhập viện\n    - Được đưa đến Cấp cứu sau khi bày tỏ tự tử với nhân viên khi xuất viện\n    - Được yêu cầu ký một mẫu đơn cam kết đảm bảo an toàn trong 24 giờ trước khi bắt đầu chương trình điều trị bán trú, và anh ấy không thể làm được\n    - chụp ct sọ được phát hiện tình cờ tại Cấp cứu; phát hiện bệnh lý chất trắng\n    - chọc dò dịch não tủy là không phát hiện bất thường\n    - đánh giá thần kinh để tìm bệnh lý chất trắng\n    - các băng nhóm oligoclonal đang chờ kết quả\n    - Được bác sĩ khoa tâm thần cho nhập chỉ định viện tâm thần \n    Tình trạng ngay trước khi nhập viện: Phủ nhận tự tử chủ động, ý định, kế hoạch hiện tại khi đã lên tầng, cảm thấy an toàn\n\n3.  Đánh giá tại bệnh viện\n    Kết quả xét nghiệm: chọc dò dịch não tủy là âm tính\n    Kết quả chẩn đoán hình ảnh: chụp ct sọ: phát hiện bệnh lý chất trắng (được phát hiện tình cờ)\n    Các thủ thuật đã thực hiện: chọc dò dịch não tủy (Đột sống thắt lưng)\n    Các phát hiện chẩn đoán khác\n    - đánh giá thần kinh để tìm bệnh lý chất trắng\n    - các băng nhóm oligoclonal đang chờ kết quả\n    - Nguyên nhân chưa rõ (đối với phát hiện bệnh lý chất trắng)\n6. Sưng hạch cổ\n • Thường là 1 hạch >1,5 cm, chắc, không hóa mủ.\n\nLưu ý quan trọng:\nCác triệu chứng của Kawasaki dễ nhầm với sốt siêu vi, sốt phát ban, nhiễm trùng. Nếu trẻ sốt cao 3–4 ngày + đỏ mắt + phát ban + môi đỏ hoặc lưỡi đỏ → cần nghĩ đến Kawasaki và đưa trẻ đi khám sớm.\n4. Biến chứng nguy hiểm của bệnh Kawasaki\nNếu không điều trị kịp thời, bệnh có thể gây:\nBiến chứng tim mạch (nguy hiểm nhất):\n • Phình giãn động mạch vành\n • Hẹp – tắc động mạch vành\n • Thiếu máu cơ tim\n • Nhồi máu cơ tim\n • Suy vành mạn tính\nKhoảng 25–30% trẻ không điều trị đúng cách sẽ bị biến chứng này.\nBiến chứng khác:\n • Viêm cơ tim\n • Tràn dịch màng tim\n • Tăng men gan\n • Rối loạn tiêu hóa\n5. Chẩn đoán bệnh Kawasaki\n\nTiêu chuẩn chẩn đoán\n\nSốt ≥5 ngày + 4/5 tiêu chí sau:\n 1. Viêm kết mạc 2 bên không ghèn\n 2. Môi – miệng thay đổi (nứt, đỏ, lưỡi dâu tây)\n 3. Tổn thương đầu chi (phù, đỏ, bong da)\n 4. Ban đỏ toàn thân\n 5. Hạch cổ to ≥1,5 cm\n\nCác xét nghiệm cần làm\n',
    '9.txt': 'Hỏi : Kính chào bác sĩ! Em 37 tuổi và hiện đang mang thai lần 2 được 20 tuần \n. Từ năm 24-26 tuổi em có dấu hiệu đau bao tử (thỉnh thoảng đau và sau 1-2 tiếng thì hết do thời gian này em sống xa gia đình nên ăn uống thất thường) nhưng không khám cũng như không uống thuốc gì cả. Năm 27 tuổi em đi nội soi ở BV thì BS nói em có ổ loét trong bao tử và có cho thuốc uống theo lộ trình dài nhưng em chỉ uống hết liều thuốc lần 1 đó mà không tiếp tục điều trị. Thay vào đó em đã chuyển về sống cùng gia đình và ăn uống rất điều độ nên hầu như em không bị đau lại nữa. Sau đó em uống tinh bột nghệ tách tinh dầu  vào mỗi buổi sáng khi ngủ dậy. Em thấy không còn bị đau như trước và cơ thể cũng có cảm giác khỏe lên. Nay em đang có thai lần 2 nên em ngưng không uống bôt nghệ nữa. Tuy thai mới được 20w nhưng em đã tăng 10kg (mặc dù em ăn nhiêu ói bấy nhiêu). Đi khám bác sĩ cũng tư vấn em hạn chế ăn tinh bột. Tối hôm 03/5/2021 em không ăn cơm mà chỉ ăn rau luộc sau đó ăn thêm 2 quả chuối và uống 1 chai nhỏ Yakult. Đến 21h em bắt đầu đau bụng râm ran và cơn đau càng dồn đạp đến 12h đêm. Em đau đến mức chỉ biết ôm bụng mà không thể nhúc nhích được chân tay vì quá đau. Người nhà em cũng rất vất vả mới thay được quần áo cho em để đưa em đi viện. Em có cảm giác hàng ngàn sợi chỉ đang thắt ruột gan em lại. Đến BV BS chích cho em liều giảm đau và cho siêu âm nhưng kết quả không thấy gì cả (chỉ thấy mỗi bên thận có 1 viên sỏi 4mm nhưng không ứ nước). BS nói sỏi thận không phải là lý do làm em đau và nói khả năng em bị viêm bao tử, có nói do em đang có thai nên không thể dùng thuốc đặc trị được mà chỉ cho thuốc tạm thôi. Em muốn hỏi như sau: 1. Danh sách thuốc BS kê có ảnh hưởng đến em bé trong bụng em không: - ****** (trong HDSD có ghi "An toàn của ****** trên phụ nữ có thai chưa được thiết lập") - ******* (*******************, ****************) - ************** 2. Các xét nghiệm/siêu âm/nội soi cần thiết nào cho em mà vẫn an toàn với thai nhi? Em cảm ơn nhiều!\nTrả lời : Chào em,\nThành phần 3 loại thuốc em sử dụng như sau:\nAquima: nhôm hydroxid, magie hydroxid và **********.\nSimenic: Alverin citrate 40mg và Simethicon 100mg\nPimperam: metoclopramide 10mg\nThông tin về độ an toàn khi sử dụng các loại thuốc trên ở phụ nữ mang thai như sau:\nNhôm hydroxid và magie hydroxid có thể sử dụng được cho phụ nữ mang thai để điều trị triệu chứng ợ nóng hoặc trào ngược dạ dày thực quản và cần sử dụng trong mức liều khuyến cáo.\nHiện tại chưa có đủ nghiên cứu về việc sử dụng ********** trên phụ nữ mang thai. Mặc dù chưa khẳng định được thuốc có qua nhau thai hay không nhưng hấp thu của ********** qua ruột bị hạn chế, do đó làm giảm khả năng phơi nhiễm đối với thai nhi.\n    Lý do nhập viện: theo dõi đại tràng giãn\n    - Vào đợt kiểm tra sức khoẻ định kỳ sau ngã đang hồi phục phát hiện xét nghiệm máu có tăng bạch cầuNgày nay xuất hiện ý thúc chậm hơn, đã chụp CT ở tuyến trước chưa phát hiện bất thường trên phim chụp,  vào viện\n    Tình trạng vào Khoa Cấp cứu. \n  Bệnh nhân lơ mơ\n3.  Đánh giá tại bệnh viện\n',
    '10.txt': '1.  Tiền sử bệnh\n    Thuốc trước khi nhập viện\n    - metoprolol 25mg po bid\n    - doxycycline cho viêm tuyến mồ hôi\n    - atenolol (uống hôm nay)\n    Các yếu tố nguy cơ liên quan\n    - Theo lời bệnh nhân kể lại bệnh nhân có   Căng thẳng  nhiều trong công việc\n    - Mất việc làm 8 ngày trước\n    -  Một ngày người bệnh có thể uống hàng chục tách cà phê có caffeine\n    -  và 1 tách cà phê không caffeine\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện:  Bệnh nhân vào viện vì xuất hiện triệu chứng đánh trống ngực\n    Thời điểm khởi phát triệu chứng: \n- Cách 10 ngày trước khi vào việc  bệnh nhân xuất hiện cảm giác đánh trống ngực\n    Các triệu chứng hiện tại\n    - đánh trống ngực\n    - Khó thở nhẹ khó thở\n    - cảm giác thắt chặt ngực vùng trước tim (khởi phát lúc 17 giờ)\n    - Tăng đánh trống ngực (khởi phát lúc 17 giờ)\n    - khó thở (khởi phát lúc 17 giờ)\n    - Cảm thấy mệt mỏi nhiều khi gắng sức trong tuần qua\n    - Cảm thấy mệt mỏi nhiều hơn sau khi luyện tập thể dục so với mọi ngày \n    Đặc điểm triệu chứng khi khám tại khoa cấp cứu\n    - đánh trống ngực:  còn cảm giác đánh trống ngực khi nhập viện.\n    - khó thở: Nhẹ, liên quan đến đánh trống ngực, liên quan đến khó thở (khởi phát lúc 17 giờ)\n    - cảm giác thắt chặt ngực: Trung tâm, khởi phát lúc 17 giờ, kéo dài 20 giây, không có khó chịu vùng ngực khi đến tầng\n    - khó thở: Liên quan đến đánh trống ngực, khởi phát lúc 17 giờ, kéo dài 20 giây\n    - giảm dung nạp gắng sức: \n    - Không  buồn nôn, hay nôn, đổ mồ hôi\n    - Không liên quan đến gắng sức hoặc tư thế\n    Các diễn biến  trước khi nhập viện\n    - Được thăm khám bởi bác sĩ phụ trách chính \n    - monitor holter cho thấy Nhịp xoang chiếm ưu thế. Ghi nhận ngoại tâm thu nhĩ và ngoại tâm thu thất xuất hiện thường xuyên.\n    - Bắt đầu dùng metoprolol 25mg po bid, không có cải thiện\n    -  Ở nhà bệnh nhân đã sử dụng atenololtrong ngày\n    - Lên lịch tái khám với bác sĩ tim mạch và được chỉ định siêu âm tim qua thành ngực vào tuần tới\n    - Ngày hôm nay khoảng Lúc 17 giờ,  khi đang mang đồ tạp hóa ra xe, xuất hiện cảm giác thắt chặt ngực vùng trước tim, tăng đánh trống ngực, và khó thở kéo dài 20 giây\n    - Sau đó Đến Khoa Cấp cứu\n    - Được chỉ định điều trị  aspirin 325mg x 1\n    - chụp x-quang ngực không ghi nhận gì bất thường\n    - phân tích nước tiểu không ghi nhận gì bất thường\n    - ecg bình thường\n    Tình trạng ngay trước khi nhập viện: Tiếp tục cảm thấy đánh trống ngực\n\n3.  Đánh giá tại bệnh viện\n    Kết quả khám lâm sàng\n    - VS98.3 12987 56 18 99RA\nViêm gan cấp tính do virus B thể thông thường điển hình mức độ nặng giai đoạn toàn phát\n3/ đơn thuốc: \n    Kết quả xét nghiệm: phân tích nước tiểu không có gì đáng chú ý\n    Kết quả chẩn đoán hình ảnh: chụp x-quang ngực không có gì đáng chú ý\n    Các kết quả chẩn đoán khác\n    - điện tâm đồ là không ghi nhận gì bất thường\n    - monitor holter cho thấy Nhịp xoang chiếm ưu thế. Ghi nhận ngoại tâm thu nhĩ và ngoại tâm thu thất xuất hiện thường xuyên.',
    '11.txt': '1.  Tiền sử bệnh lý\n\nBệnh tim mạch do xơ vữa động mạch.\năng huyết áp.Rối loạn lipid máu.\nĐái tháo đường.2. \nBệnh sử\nLý do vào viện: Cơn rối loạn ý thức thoáng qua, nghi ngờ cơn co giật.\nKhoảng 1 tháng trước nhập viện, bệnh nhân bị ngã và được chẩn đoán xuất huyết dưới nhện. Chụp cắt lớp vi tính sọ não cho hình ảnh  xuất huyết dưới nhện vùng trán phải, bầm dập nhu mô vùng trán phải và trán - thái dương phải, kèm một lớp dịch dưới màng cứng mỏng vùng thùy trán phải, nghĩ nhiều đến nang màng nhện hoặc tụ dịch/tụ máu dưới màng cứng mạn tính. \nCác lần chụp CT theo dõi sau đó ổn định và bệnh nhân được xuất viện, hẹn tái khám chuyên khoa Ngoại thần kinh.\nSau xuất viện, bệnh nhân từng quay lại khoa Cấp cứu vì đau đầu kéo dài. \nChụp kiểm tra ghi nhận tụ máu ngoài màng cứng phải cấp tính trên nền tổn thương mạn tính. \nBệnh nhân tiếp tục được theo dõi, các lần chụp CT kiểm tra không ghi nhận diễn tiến xấu nên được xuất viện về nhà.\nTrong thời gian sau đó, bệnh nhân tương đối ổn định nhưng vẫn còn đau đầu vùng thái dương phải và đỉnh đầu, kèm cảm giác tê bì nửa mặt phải, đặc biệt vùng trán phải và da đầu phải từ sau lần ngã.\nNgày vào viện, buổi sáng bệnh nhân thức dậy tỉnh táo, sinh hoạt bình thường, ăn sáng, đọc báo và sử dụng máy tính. Đến cuối buổi sáng, khi đang đi trong nhà, bệnh nhân nói muốn đi vệ sinh. \nNgười nhà nhận thấy bệnh nhân có biểu hiện bất thường, mất định hướng, đi lại không vững và gần như ngất. \nBệnh nhân lúc mở mắt, lúc nhắm mắt, nhiều lần ngửa đầu ra sau và không đáp ứng được các câu hỏi của người nhà.Sau đó bệnh nhân được dìu nằm nghỉ. \nTrong cơn, bệnh nhân kích thích nhẹ, liên tục cố gắng đứng dậy và tỏ ra không hiểu tình trạng đang xảy ra. \nTình trạng ý thức dao động kéo dài khoảng 20 phút rồi cải thiện dần. \nKhi nhân viên cấp cứu đến nơi, bệnh nhân bắt đầu tỉnh táo trở lại và hồi phục hoàn toàn về trạng thái ban đầu.\nTheo như mô tả của Bạn cháu bé bị bệnh bàn chân bẹt, đây là tật bẩm sinh không ảnh hưởng nhiều đến sức khỏe. Hiện tại cháu chạy nhảy bình thường, không đau, chưa có chỉ định phẫu thuật, Tuy nhiên bạn nên cho cháu đến bệnh viện để kiểm tra đánh giá bất thường về xương, khám lâm sàng để đánh giá các thiếu hụt về phần mềm, trên cơ sở đó chúng tôi sẽ có các hướng dẫn cụ thể về tập vận động, tiên lượng về tiến triển của bệnh một cách cụ thể. \nKhám lâm sàng:\nCó ghi nhận giai đoạn mất định hướng và đi lại không vững theo lời kể người nhà.\nTrong cơn có biểu hiện kích thích nhẹ, ngửa đầu ra sau và nhắm mắt từng lúc.\nKhông ghi nhận co giật, cứng đờ, cắn lưỡi hoặc tiểu tiện không tự chủ.\nCận lâm sàng: CT sọ não trước đó ghi nhận xuất huyết dưới nhện vùng trán phải.Bầm dập nhu mô vùng trán phải và trán - thái dương phải.Lớp dịch dưới màng cứng mỏng vùng thùy trán phải, nghĩ đến nang màng nhện hoặc tụ máu dưới màng cứng mạn tính.Tụ máu ngoài màng cứng phải cấp tính trên nền tổn thương mạn tính.\n',
    '12.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mãn tính\n- Bệnh phổi kẽ do sử dụng corticoid liều cao kéo dài.\n    - Thở oxy tại nhà\n    - Hội chứng kháng enzym tổng hợp protein.\n    - béo phì\n    - Tăng huyết áp nguyên phát.\n    Tiền sử phẫu thuật / thủ thuật: Sinh thiết nội mạc tử cung gần đây\n    Thuốc trước khi nhập viện\n    -Corticoid liều cao kéo dài\n    - Thở oxy tại nhà\n    - Tăng liều bactrim (do bác sĩ chăm sóc chính kê đơn)\n    - doxycycline (do bác sĩ chăm sóc chính kê đơn)\n- Suy giảm miễn dịch do sử dụng corticoid kéo dài\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Tình trạng tổn thương vùng âm hộ và mông bên phải ngày càng nặng\n    Diễn biến bệnh\n- Bệnh nhân nhập viện vì tổn thương vùng âm hộ phải lan rộng sang mông phải, diễn biến tăng dần trong khoảng 5 ngày trước nhập viện.\n- Cách nhập viện 5 ngày, bệnh nhân phát hiện một tổn thương đơn độc vùng âm hộ phải. Sau đó được bác sĩ điều trị khám và tăng liều cotrimoxazol, đồng thời bổ sung doxycyclin do nghi ngờ viêm mô tế bào. Bệnh nhân thấy  ban đỏ quanh tổn thương giảm sau điều trị\n    - Tuy nhiên tổn thương tiếp tục tiến triển, xuất hiện nhiều bọng nước, lan rộng sang mông phải, đau nhiều, một số vị trí rỉ dịch vàng đục giống mủ.\n    - Khám tại phòng khám vào ngày nhập viện\n    - Được chuyển  đến Khoa Cấp cứu để khám và điều trị tiếp\n    Triệu chứng khi nhập viện\n    - tổn thương vùng âm hộ và mông bên phải\n    - Tổn thương cực kỳ đau đớn\n    - Tình trạng tổn thương dạng bóng nước ngày càng nặng\n    - Lan đến mông bên phải\n    - có dịch giống mủ có màu vàng  chảy ra từ tổn thương\n    Đặc điểm của triệu chứng\nTrả lời :\nChào bạn, trước hết xin được chia sẻ về sự lo lắng đôi chút về tình trạng của bé hiện này, về tình trạng bé mắc phải là dị tật thiểu sản vành tai và tịt ống tai ngoài bẩm sinh. Rất may mắn, cấu trúc tai trong đảm nhận chức năng thần kinh thính giác của con thường vẫn phát triển tốt. Việc bé phản xạ được âm thanh là nhờ sự bù trừ tuyệt vời từ chiếc tai trái khỏe mạnh.\n\nBạn hoàn toàn yên tâm vì y học hiện nay có phác đồ điều trị tương đối tốt . Về thẩm mỹ, khi bé từ 6 - 10 tuổi bác sĩ có thể phẫu thuật tạo hình vành tai bằng sụn sườn tự thân hoặc sụn nhân tạo, giúp con có đôi tai hoàn thiện. Về thính lực, dựa trên kết quả chụp cắt lớp đánh giá cấu trúc tai giữa, bé có thể được tạo hình ống tai ngoài hoặc dùng thiết bị trợ thính đường xương, truyền âm thanh xuyên qua hộp sọ thẳng vào tai trong.\n\n    - Được giới thiệu đến Khoa Cấp cứu để đánh giá thêm\n    - Bệnh nhân kiên quyết muốn tiếp tục sử dụng doxycyclinebactrim cho khả năng Viêm mô tế bào\n    - Cảm thấy rằng cô ấy đã có cải thiện do những loại thuốc này và không muốn ngừng chúng cho đến khi được bác sĩ Truyền nhiễm khám\n3.  Đánh giá tại bệnh viện\n    Các phát hiện chẩn đoán khác: Lo ngại về Nhiễm virus Herpes simplex (HSV) hoặc Bệnh thủy đậu/Zona (do Varicella Zoster Virus)',
    '13.txt': 'Câu hỏi từ người dùng:\n\nEm chào bác sỹ\nCho em hỏi em có bị thương nhẹ ở tay nhưng bị chảy máu,sau đó khoảng 10 phút em có chơi với 1 con chó con nhà em (chó chưa tiêm dại) tuổi có có dính nước dãi của chó con vào tay. Hôm qua em vô tình đọc được thông tin là nếu dính nước dãi chó vẫn có nguy cơ lây bệnh dại. Vậy bác sỹ cho em hỏi trong trường hợp của em có bị sao không ạ?\nMong nhận được sự tư vấn của bác sỹ\nCâu trả lời của bác sĩ:\n\nChào bạn,\nĐể giải đáp thắc mắc của bạn, tôi xin chia sẻ một số thông tin như sau:\nBệnh dại chỉ không chỉ lây qua vết cắn của động vật. Đường lây bệnh dại phổ biến nhất là do bị động vật dại cắn. Bệnh dại còn có thể lây truyền từ nước bọt của chó, mèo dại hoặc động vật khác mắc bệnh dại do cào hoặc liếm vào vết thương, những vùng da bị trầy xước của cơ thể.\n1. Bệnh dại có lây không?\nBệnh dại do Lyssavirus, thuộc họ Lyssaviridae gây ra. Sau khi xâm nhập vào cơ thể người và động vật có vú, virus di chuyển theo hệ thần kinh vào tủy sống và não, phá hủy các trung khu thần kinh trong đại não, gây ra trạng thái điên dạiở động vật và người.\nBệnh dại là bệnh đe dọa tính mạng và có thể gây tử vong cho người nếu người bị cắn không rửa vết thương và được điều trị y tế kịp thời sau khi bị cắn. Không có thuốc điều trị khi lên cơn dại. Phòng bệnh bằng tiêm vaccine phòng dại. Bệnh dạithường gia tăng vào mùa hè.\nBệnh dại gây ra bởi virus, và là bệnh lây truyền. Do vậy, việc phòng bệnh dại là vô cùng cần thiết.\n2. Bệnh dại lây truyền qua đường nào?\nVi-rút dại chủ yếu được lây truyền từ nước bọt của các loài động vật bị dại sang người qua vết cắn hoặc qua vết trầy xước trên cơ thể con người.\n- 96% các trường hợp gây bệnh dại ở người tại Đông Nam Á là do chó cắn. Nơi bị chó cắn càng gần thần kinh trung ương thì nạn nhân càng phát bệnh nhanh.\n- Thế giới ghi nhận việc lây bệnh qua không khí có thể xảy ra khi ở trong hang dơi hay tiếp xúc với chất thải của dơi, việc lây bệnh qua không khí này đã được ghi nhận tại 4 báo cáo về ca mắc bệnh dại ở người và liên quan tới công việc thí nghiệm với động vật. Tuy nhiên các ca mắc dạng này chưa ghi nhận tại Việt Nam.\n- Các yếu tố có thể ảnh hưởng đến sự phát triển lây nhiễm bệnh dạibao gồm:\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: đau ngực trái cấp tính và đau sau xương ức lan ra sau lưng\nMức độ nghiêm trọng của vết cắn\nSố lượng vi rút dại xâm nhập vào\nTình trạng miễn dịch của bệnh nhân\nVùng bị cắn - vết thương ở đầu và cổ, cũng như những vết thương ở các khu vực đầu mút thần kinh như ngón tay, thường có thời gian ủ bệnh ngắn hơn do khoảng cách gần hơn cho vi rút xâm nhập vào mô thần kinh.\nTrường hợp của bạn nên đi đến các trung tâm tiêm chủng gần nhất để kiểm tra y tế và chích ngừa bạn nhé!',
    '14.txt': 'Câu hỏi từ người dùng :\n\nDạ em có một người bạn, hiện tại bạn ấy đang bị nổi các dấu mề đay rất to và ngứa khắp người, bạn em bị nổi quanh năm luôn ạ. Những dấu mề đay nổi rất khó chịu và ngứa, được biết bạn em có từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn ấy cũng bị nên em nghỉ chắc là di truyền. Cả hai người đều không phải dị ứng do thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bị nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến và có cách nào để chữa cho bạn không ạ. Em cảm ơn rất nhiều\n    Các bệnh lý mạn tính\n    - Bệnh bạch cầu dòng tủy mãn tính đang dùng gleevec\n    - Tăng huyết áp\n    - tăng lipid máu, không đặc hiệu\n    - Đái tháo đường típ 2\n    - hẹp ống sống\n    - Giả gout\n    - bệnh thận mạn, không đặc hiệu Giai đoạn 4\n    - tăng sản tuyến tiền liệt\n    - Nhiều lần ngã gần đây Ngã\n    - ảo giác\n    Thuốc trước khi nhập viện: gleevec (dừng theo chỉ dẫn sau xuất viện)\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện\n    - Toàn trạng suy kiệt  kể từ khi xuất viện\n    - ảo giác dai dẳng\n    - Lú lẫn ngày càng nặng\n    Thời điểm khởi phát triệu chứng: Cực kỳ yếu kể từ khi xuất viện\n    Diễn biến bệnh\n    - ảo giác được vợ nhận thấy\n    - Lú lẫn ngày càng nặng được vợ nhận thấy\nTổn thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với lympho T và đại thực bào. Các tổn thương khó quan sát trên kính hiển vi quang học thông thường mà cần xem trên kính hiển vi điện tử. Các tế bào mast tăng số lượng vùng hạ bì với các mức độ thoát bọng khác nhau được quan sát thấy. Nhuộm huỳnh quang miễn dịch tổn thương sinh thiết không thấy có hình ảnh của lắng đọng các phức hợp miễn dịch, bổ thể hay sợi fibrin.\nĐối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Mày đay vô căn là một bệnh mạn tính, việc điều trị phải lâu dài và liên tục. Quá trình điều trị chỉ là điều trị triệu chứng.\nĐiều trị phụ thuộc vào mức độ nặng và bệnh lý nền có liên quan. Cần lưu ý cân bằng giữa hiệu quả điều trị kiểm soát triệu chứng và những tác dụng gây độc của liệu pháp điều trị. **************** thể hệ sau không gây buồn ngủ là lựa chọn hàng đầu sau đó đến **************** thể hệ 1, corticoid liều thấp cách ngày hoặc hàng ngày và giảm liều chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không đỡ hoặc có chống chỉ định, xin bạn hãy đến với bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất.\nCảm ơn bạn đã gửi câu hỏi',
    '15.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mãn tính\n- Bệnh phổi kẽ do sử dụng corticoid liều cao kéo dài.\n    - Thở oxy tại nhà\n    - Hội chứng kháng enzym tổng hợp protein.\n    - béo phì\n    - Tăng huyết áp nguyên phát.\n    Tiền sử phẫu thuật / thủ thuật: Sinh thiết nội mạc tử cung gần đây\n    Thuốc trước khi nhập viện\n    -Corticoid liều cao kéo dài\n    - Thở oxy tại nhà\n    - Tăng liều bactrim (do bác sĩ chăm sóc chính kê đơn)\n    - doxycycline (do bác sĩ chăm sóc chính kê đơn)\n- Suy giảm miễn dịch do sử dụng corticoid kéo dài\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Tình trạng tổn thương vùng âm hộ và mông bên phải ngày càng nặng\n    Diễn biến bệnh\n- Bệnh nhân nhập viện vì tổn thương vùng âm hộ phải lan rộng sang mông phải, diễn biến tăng dần trong khoảng 5 ngày trước nhập viện.\n- Cách nhập viện 5 ngày, bệnh nhân phát hiện một tổn thương đơn độc vùng âm hộ phải. Sau đó được bác sĩ điều trị khám và tăng liều cotrimoxazol, đồng thời bổ sung doxycyclin do nghi ngờ viêm mô tế bào. Bệnh nhân thấy  ban đỏ quanh tổn thương giảm sau điều trị\n    - Tuy nhiên tổn thương tiếp tục tiến triển, xuất hiện nhiều bọng nước, lan rộng sang mông phải, đau nhiều, một số vị trí rỉ dịch vàng đục giống mủ.\n    - Khám tại phòng khám vào ngày nhập viện\n    - Được chuyển  đến Khoa Cấp cứu để khám và điều trị tiếp\nTheo thông tin hướng dẫn sử dụng của sản phẩm *******, phụ nữ có thai chỉ được sử dụng khi có chỉ định của bác sĩ.\nTheo thông tin hướng dẫn sử  của sản phẩm Pimperan, nhiều dữ liệu trên đối tượng phụ nữ có thai cho thấy metoclopramide không gây quái thai hoặc gây độc tính cho thai nhi nên có thể dùng trong thai kỳ nếu cần thiết, tuy nhiên nên tránh sử dụng vào cuối thai kỳ.\n    - Vị trí: âm hộ bên phải, mông bên phải\n    - Mức độ nghiêm trọng: cực kỳ đau đớn\n    - Thời gian: Tình trạng ngày càng nặng trong 5 ngày\n    - Các triệu chứng liên quan: ban đỏ, chảy mủ\n    Các sự kiện trước khi nhập viện\n    - Được bác sĩ chăm sóc chính thay thế khám vào ngày, được tăng liều bactrim và doxycycline để điều trị chẩn đoán Viêm mô tế bào\n    - Báo cáo có cải thiện một phần về ban đỏ\n    - dịch tiết có vẻ như mủ từ một số tổn thương vào ngày, nhưng tình trạng này đã tự khỏi\n    - Khám tại phòng khám vào ngày nhập viện\n    - Được giới thiệu đến Khoa Cấp cứu để đánh giá thêm\n    - Bệnh nhân kiên quyết muốn tiếp tục sử dụng doxycyclinebactrim cho khả năng Viêm mô tế bào\n    - Cảm thấy rằng cô ấy đã có cải thiện do những loại thuốc này và không muốn ngừng chúng cho đến khi được bác sĩ Truyền nhiễm khám\n3.  Đánh giá tại bệnh viện\n    Các phát hiện chẩn đoán khác: Lo ngại về Nhiễm virus Herpes simplex (HSV) hoặc Bệnh thủy đậu/Zona (do Varicella Zoster Virus)',
    '16.txt': 'Câu hỏi của người dùng gửi đến hệ thống\nEm chào bác sỹ\nCho em hỏi em có bị thương nhẹ ở tay nhưng bị chảy máu,sau đó khoảng 10 phút em có chơi với 1 con chó con nhà em (chó chưa tiêm dại) tuổi có có dính nước dãi của chó con vào tay. \nHôm qua em vô tình đọc được thông tin là nếu dính nước dãi chó vẫn có nguy cơ lây bệnh dại. Vậy bác sỹ cho em hỏi trong trường hợp của em có bị sao không ạ?\nMong nhận được sự tư vấn của bác sỹ\n\nBác sĩ trả lời \nChào bạn,\nĐể giải đáp thắc mắc của bạn, tôi xin chia sẻ một số thông tin như sau:\nBệnh dại chỉ không chỉ lây qua vết cắn của động vật. Đường lây bệnh dại phổ biến nhất là do bị động vật dại cắn. Bệnh dại còn có thể lây truyền từ nước bọt của chó, mèo dại\n hoặc động vật khác mắc bệnh dại do cào hoặc liếm vào vết thương, những vùng da bị trầy xước của cơ thể.\n1. Bệnh dại có lây không?\nBệnh dại do Lyssavirus, thuộc họ Lyssaviridae gây ra. Sau khi xâm nhập vào cơ thể người và động vật có vú, virus di chuyển theo hệ thần kinh vào tủy sống và não, phá hủy các trung khu thần kinh trong đại não, \ngây ra trạng thái điên dại ở động vật và người.\nBệnh dại là bệnh đe dọa tính mạng và có thể gây tử vong cho người nếu người bị cắn không rửa vết thương và được điều trị y tế kịp thời sau khi bị cắn. Không có thuốc điều trị khi lên cơn dại. \nPhòng bệnh bằng tiêm *****************. Bệnh dại thường gia tăng vào mùa hè.\nBệnh dại gây ra bởi virus, và là bệnh lây truyền. Do vậy, việc phòng bệnh dại là vô cùng cần thiết.\n2. Bệnh dại lây truyền qua đường nào?\nVi-rút dại chủ yếu được lây truyền từ nước bọt của các loài động vật bị dại sang người qua vết cắn hoặc qua vết trầy xước trên cơ thể con người.\n- 96% các trường hợp gây bệnh dại ở người tại Đông Nam Á là do chó cắn. Nơi bị chó cắn càng gần thần kinh trung ương thì nạn nhân càng phát bệnh nhanh.\n- Thế giới ghi nhận việc lây bệnh qua không khí có thể xảy ra khi ở trong hang dơi hay tiếp xúc với chất thải của dơi,\n việc lây bệnh qua không khí này đã được ghi nhận tại 4 báo cáo về ca mắc bệnh dại ở người và liên quan tới công việc thí nghiệm với động vật. Tuy nhiên các ca mắc dạng này chưa ghi nhận tại Việt Nam.\n    Các thuốc đã thực hiện\n    - được cho bumetanide 2mg iv\n    - được cho vancomycin 1 gram\n    - được cho levofloxacin 750mg iv\nSố lượng vi rút dại xâm nhập vào\nTình trạng miễn dịch của bệnh nhân\nVùng bị cắn - vết thương ở đầu và cổ, cũng như những vết thương ở các khu vực đầu mút thần kinh như ngón tay, thường có thời gian ủ bệnh ngắn hơn do khoảng cách gần hơn cho vi rút xâm nhập vào mô thần kinh.\nTrường hợp của bạn nên đi đến các trung tâm tiêm chủng gần nhất để kiểm tra y tế và chích ngừa bạn nhé!',
    '17.txt': 'Câu hỏi từ người dùng:\n\nchào bác sĩ, em 28 tuổi, dạo này em đánh răng hay chảy máu chân răng, miệng thấy hơi thở mùi khó chịu, kèm theo thấy nhiều mảng bám quanh răng nữa, bác sĩ tư vấn giúp em với.\n\nCâu trả lời của bác sĩ:\n\nChào bạn\nVấn đề của bạn nói nguy hiểm hay không nguy hiểm thì còn tùy một số trường hợp, các nguy cơ có thể xảy ra như sau:\nNguyên nhân:\n- Vệ sinh răng miệng kém: Đây được xem là nguyên nhân phổ biến gây ra chảy máu chân răng kèm hôi miệng. Nếu không vệ sinh răng miệng đúng cách sẽ tạo điều kiện cho vi khuẩn xâm nhập, gây tổn thương nướu và làm phát sinh mùi hôi.\n- Viêm quanh răng: Các mô bao quanh cuống răng bị viêm và sưng do nhiễm vi khuẩn, sang chấn răng. \n- Viêm nha chu: Đây là dạng nhiễm trùng lợi nặng, các triệu chứng gồm đau khi nhai, răng lung lay, sưng nướu, dễ chảy máu răng, hôi miệng, có mủ. Viêm nha chu nếu để kéo dài, chân răng có thể bị hư hại dẫn đến tình trạng mất răng hoặc viêm khớp dạng thấp, các bệnh hô hấp, đột quỵ,...\n- Thiếu canxi và vitamin: Thiếu hụt canxi dễ gây loãng xương, sâu răng và có nguy cơ bị viêm nha chu. Hôi miệng và chảy máu chân răng còn có thể do cơ thể thiếu hụt vitamin K.\n- Tiểu đường: Hôi miệng và chảy máu chân răng là dấu hiệu của bệnh tiểu đường. Nguyên nhân là do cơ thể sẽ giảm sản xuất insulin nên người bị tiểu đường suy yếu hệ miễn dịch, dễ gặp phải các vấn đề về răng miệng. Tình trạng khó đông máu khiến chân răng chảy máu kéo dài do nồng độ đường trong máu cao.\n- Tác dụng phụ của thuốc: Sử dụng một số loại thuốc có thể gây hôi miệng kèm chảy máu chân răng. Các loại thuốc gồm: Kháng histamin H1, thuốc chống trầm cảm, thuốc lợi tiểu, thuốc kháng sinh, thuốc chống nôn,..\n-Hút thuốc lá: Thành phần trong thuốc lá gây hỏng men răng, gai lưỡi phát triển quá mức khiến vi khuẩn xâm nhập, gây hôi miệng, chảy máu chân răng và mất răng.\n    - bệnh mạch máu ngoại biên\n    - bệnh phổi tắc nghẽn mạn tính\n    - Ngưng thở khi ngủ do tắc nghẽn đang dùng BiPAP\n    - Ung thư biểu mô tế bào vảy xâm nhập của dương vậtbiệt hóa kém, sau cắt bao quy đầu với bờ diện cắt dương tính.\n    Tiền sử phẫu thuật / thủ thuật\n•\tUống trà gừng và mật ong để loại bỏ vi khuẩn gây hại, khử mùi hôi do viêm, làm dịu niêm mạc, giảm sưng đau\n•\tTrà đinh hương giúp loại bỏ mùi hôi trong khoang miệng, ngăn ngừa chảy máu, giảm tình trạng chảy máu chân răng và hôi miệng.\n•\tThực hiện bào láng gốc răng để ngăn chặn quá trình tích tụ cao răng và hỗ trợ loại bỏ vi khuẩn gây viêm.\n•\tGhép mô mềm ở vòm họng vào vùng nướu bị ảnh hưởng để tái tạo mô và ổn định chân răng.\nĐể đảm bảo sức khỏe răng miệng, bạn nên đi khám răng miệng để biết nguyên nhân chính xác nhé!\n\n',
    '18.txt': 'Cận lâm sàng\n\nĐiện tâm đồ (ECG)\n • ST chênh lên / chênh xuống\n • Sóng T đảo\n • Q bệnh lý\n\nMen tim\n • Troponin I/T ↑ (chẩn đoán nhồi máu)\n • CK-MB ↑\n\nSiêu âm tim\n • Rối loạn vận động vùng\n • Đánh giá chức năng thất trái\n\nNghiệm pháp gắng sức\n • Phát hiện thiếu máu cơ tim khi gắng sức\n\nChụp mạch vành (tiêu chuẩn vàng) \n • Xác định vị trí – mức độ hẹp\n • Quyết định can thiệp\n\nĐiều trị \n\nMục tiêu\n • Giảm đau\n • Ngăn nhồi máu\n • Cải thiện tưới máu tim\n • Giảm tử vong\n\n\nĐiều trị nội khoa\nThuốc nền tảng\n • ******* / *********** \n • ******  (ổn định mảng xơ vữa)\n • ********* \n • ************* \n • ***************** / ***\n\n\nCan thiệp – phẫu thuật\nCan thiệp mạch vành qua da (PCI)\n • Nong bóng\n • Đặt stent\n\nPhẫu thuật bắc cầu mạch vành (CABG)\n • Khi tổn thương nhiều nhánh, nặng\n\n\n6. Dự phòng bệnh mạch vành \n\nDự phòng tiên phát (chưa mắc bệnh)\n\nĂn ít mỡ bão hòa – ít muối\n    - Ban đầu xuất hiện đau bụng trên. sau đó nhập viện để khám và điều trị tại bệnh viện,  trong quá trình nằm viện ngày hôm qua bệnh nhân thấy  đỡ đau hơn. \n    - Bệnh nhân Tự ý bỏ về từ  bệnh viện ngày hôm qua.\n    - Hôm nay, bệnh nhân có sốt nhẹ đến 38.3°C kèm theo đau hạ sườn phải tái phát, ngày càng nặng hơn.\n    - Được gọi lại nhập viện do có kết quả  cấy máu dương tính \n    Triệu chứng hiện tại\n    - sốt\n    - đau bụng\n    - đau hạ sườn phải\n    - sốt nhẹ đến 38.3°C\n    - đau hạ sườn phải tái phát, ngày càng nặng hơn\n    - đau hạ sườn phải liên tục\n    Đặc điểm triệu chứng\n    - Vị trí: Vùng hạ sườn phải \n    - Mức độ nghiêm trọng: ngày càng nặng hơn\n    - Tính chất: liên tục\n    - Triệu chứng liên quan: không thấy  buồn nôn, nôn, ớn lạnh, thay đổi chức năng ruột, đau ngực, khó thở.\n  Diễn biến  trước khi nhập viện\n    - Tự ý rời bệnh viện (AMA) ngày hôm qua sau khi cơn đau đỡ hơn ở  lần nhập viện trước.\n    - Được thực hiện siêu âm vùng gan mật tại phòng cấp cứu cho thấy túi mật căng to với dịch quanh túi mật gợi ý viêm túi mật cấp.\n3.  Đánh giá tại bệnh viện\n    Kết quả laboratory\n    - lipase là tăng lên ở mức 623 (lần nhập viện trước)\n    - tăng men gan nhẹ (lần nhập viện trước)\n    - tbr là cao tới 1.0 sau đó cải thiện (lần nhập viện trước)\n    - các xét nghiệm khác có xu hướng giảm (lần nhập viện trước)\n    - dương tính cấy máu khi tái khám với GPRs\n    Kết quả chẩn đoán hình ảnh\n    - siêu âm vùng gan mật cho thấy sỏi mật nhưng không có viêm túi mật hoặc sỏi ống mật (lần nhập viện trước)\n    - chụp ct bụng chậu là chưa phát hiện bất thường (lần nhập viện trước)\n    - siêu âm vùng gan mật hiện tại cho thấy túi mật căng to với dịch quanh túi mật gợi ý viêm túi mật cấp\n    Các phát hiện chẩn đoán khác: lo ngại viêm túi mật cấp',
    '19.txt': 'Câu hỏi từ người dùng :\n\nDạ em có một người bạn, hiện tại bạn ấy đang bị nổi các dấu mề đay rất to và ngứa khắp người, bạn em bị nổi quanh năm luôn ạ. Những dấu mề đay nổi rất khó chịu và ngứa, được biết bạn em có từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn ấy cũng bị nên em nghỉ chắc là di truyền. Cả hai người đều không phải dị ứng do thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bị nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến và có cách nào để chữa cho bạn không ạ. Em cảm ơn rất nhiều\n\n\n    - gọi cho bác sĩ chăm sóc chính (PCP)\n    - Bác sĩ PCP khuyên dùng tylenol cho đau\n    - bệnh nhân dùng tylenol nhưng phải đến phòng cấp cứu vì đau vẫn tiếp tục\n    Triệu chứng hiện tại\n    - đau bụng\n    - táo bón\n    - buồn nôn\n    Đặc điểm triệu chứng\n    - Vị trí: đau hố chậu\n    - Mức độ nghiêm trọng: cơn đau vẫn tiếp diễn mặc dù đã dùng tylenol\n    - Triệu chứng liên quan: buồn nôn\n    Các sự kiện trước khi nhập viện\n    - đã dùng quả mận khô\n    - đã dùng tylenol\n    - gọi cho bác sĩ PCP\nNhư bạn đã mô tả, trường hợp này bạn  thuộc MÀY ĐAY VÔ CĂN. Chúng ta không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY ĐAY MẠN TÍNH.\nTổn thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với lympho T và đại thực bào. Các tổn thương khó quan sát trên kính hiển vi quang học thông thường mà cần xem trên kính hiển vi điện tử. Các tế bào mast tăng số lượng vùng hạ bì với các mức độ thoát bọng khác nhau được quan sát thấy. Nhuộm huỳnh quang miễn dịch tổn thương sinh thiết không thấy có hình ảnh của lắng đọng các phức hợp miễn dịch, bổ thể hay sợi fibrin.\nĐối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Mày đay vô căn là một bệnh mạn tính, việc điều trị phải lâu dài và liên tục. Quá trình điều trị chỉ là điều trị triệu chứng.\nĐiều trị phụ thuộc vào mức độ nặng và bệnh lý nền có liên quan. Cần lưu ý cân bằng giữa hiệu quả điều trị kiểm soát triệu chứng và những tác dụng gây độc của liệu pháp điều trị. **************** thể hệ sau không gây buồn ngủ là lựa chọn hàng đầu sau đó đến **************** thể hệ 1, corticoid liều thấp cách ngày hoặc hàng ngày và giảm liều chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không đỡ hoặc có chống chỉ định, xin bạn hãy đến với bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất.\nCảm ơn bạn đã gửi câu hỏi',
    '20.txt': 'Câu hỏi của người dùng gửi đến hệ thống\nEm chào bác sỹ\nCho em hỏi em có bị thương nhẹ ở tay nhưng bị chảy máu,sau đó khoảng 10 phút em có chơi với 1 con chó con nhà em (chó chưa tiêm dại) tuổi có có dính nước dãi của chó con vào tay. \nHôm qua em vô tình đọc được thông tin là nếu dính nước dãi chó vẫn có nguy cơ lây bệnh dại. Vậy bác sỹ cho em hỏi trong trường hợp của em có bị sao không ạ?\nMong nhận được sự tư vấn của bác sỹ\n\nBác sĩ trả lời \nChào bạn,\nĐể giải đáp thắc mắc của bạn, tôi xin chia sẻ một số thông tin như sau:\nBệnh dại chỉ không chỉ lây qua vết cắn của động vật. Đường lây bệnh dại phổ biến nhất là do bị động vật dại cắn. Bệnh dại còn có thể lây truyền từ nước bọt của chó, mèo dại\n hoặc động vật khác mắc bệnh dại do cào hoặc liếm vào vết thương, những vùng da bị trầy xước của cơ thể.\n1. Bệnh dại có lây không?\nBệnh dại do Lyssavirus, thuộc họ Lyssaviridae gây ra. Sau khi xâm nhập vào cơ thể người và động vật có vú, virus di chuyển theo hệ thần kinh vào tủy sống và não, phá hủy các trung khu thần kinh trong đại não, \nDa niêm mạc hồngi\n Đau bụng hạ sườn phải\n Buồn nôn, nôn ra thức ăn và dịch dạ dày, không có máu \nKhám thấy\nHuyết áp:130/76 mmHg\nMạch: 93 l/p\nNhiệt độ : 36.3 độ C\nNhịp thở: 14 l/p \nSPO2: 99 %\n   Điều trị tại bệnh viện:\n    - compazine và alevenhưng vẫn còn đau\n    - Uống  morphineoral và  lorazepam đỡ đau  (có thể ngủ\n\n3.  Đánh giá tại bệnh viện\n2. Bệnh dại lây truyền qua đường nào?\nVi-rút dại chủ yếu được lây truyền từ nước bọt của các loài động vật bị dại sang người qua vết cắn hoặc qua vết trầy xước trên cơ thể con người.\n- 96% các trường hợp gây bệnh dại ở người tại Đông Nam Á là do chó cắn. Nơi bị chó cắn càng gần thần kinh trung ương thì nạn nhân càng phát bệnh nhanh.\n- Thế giới ghi nhận việc lây bệnh qua không khí có thể xảy ra khi ở trong hang dơi hay tiếp xúc với chất thải của dơi,\n việc lây bệnh qua không khí này đã được ghi nhận tại 4 báo cáo về ca mắc bệnh dại ở người và liên quan tới công việc thí nghiệm với động vật. Tuy nhiên các ca mắc dạng này chưa ghi nhận tại Việt Nam.\n- Các yếu tố có thể ảnh hưởng đến sự phát triển lây nhiễm bệnh dại bao gồm:\nLoại hình tiếp xúc và loại động vật cắn\nMức độ nghiêm trọng của vết cắn\nSố lượng vi rút dại xâm nhập vào\nTình trạng miễn dịch của bệnh nhân\nVùng bị cắn - vết thương ở đầu và cổ, cũng như những vết thương ở các khu vực đầu mút thần kinh như ngón tay, thường có thời gian ủ bệnh ngắn hơn do khoảng cách gần hơn cho vi rút xâm nhập vào mô thần kinh.\nTrường hợp của bạn nên đi đến các trung tâm tiêm chủng gần nhất để kiểm tra y tế và chích ngừa bạn nhé!',
    '21.txt': 'Câu hỏi từ người dùng:\nEm bị Rối loạn chuyển hóa tinh bột (amyloidosis) đã lâu. Có dùng thuốc tại viện da liễu trung ương nhưng không thấy đỡ. Có cách nào chữa trị tận gốc hoặc giảm thiểu không ạ? Em cảm ơn.\nCâu trả lời của bác sĩ:\nChào bạn! Cảm ơn bạn đã gửi câu hỏi cho chúng tôi.\n Trả lời câu hỏi của bạn:Bệnh thoái hóa tinh bột (hay bệnh amyloidosis) là một bệnh gây lắng đọng protein hiếm gặp và nghiêm trọng. Nguyên nhân là do một loại protein bất thường gọi là amyloid tích tụ trong các mô hoặc cơ quan. Khi lượng protein amyloid lắng đọng tăng lên, chúng sẽ phá vỡ cấu trúc và gây cản trở chức năng sinh lý của mô hoặc cơ quan. Cuối cùng, sự lắng đọng protein amyloid sẽ bắt đầu gây ra các triệu chứng và suy các cơ quan, thậm chí có thể gây tử vong.\nTình trạng lắng đọng của protein amyloid trong bệnh thoái hóa tinh bột có thể khu trú ở bất kỳ cơ quan nào trong cơ thể, chẳng hạn như phổi, da, bàng quang hoặc ruột hay có thể là toàn thân. Đây cũng là dạng phổ biến nhất. Mặc dù bệnh thoái hóa tinh bột không được xếp thành một loại ung thư, bệnh lại có thể liên quan đến một số bệnh ung thư máu như đa u tủy.\nCó nhiều loại amyloidosis khác nhau, bao gồm những loại sau:\n•        Bệnh amyloidosis chuỗi nhẹ\n•        Bệnh amyloidosis tự miễn dịch\n•        Bệnh amyloidosis di truyền hoặc gia đình\nBệnh thoái hóa tinh bột là một bệnh đa hệ thống dẫn đến nhiều biểu hiện lâm sàng khác nhau, Hiện nay chưa có phương pháp điều trị tận gốc amyloidosis.Tuy nhiên, các phương pháp điều trị có thể giúp bạn kiểm soát các triệu chứng bệnh cũng như ngăn chặn tổn thương \n    Các triệu chứng hiện tại\n    - đau bụng vùng hạ sườn phải\n    - chướng bụng \n    - buồn nôn thoáng qua\n    - Nôn mửa\n    - Bệnh nhân có  đau lưng âm ỉ\n    Đặc điểm triệu chứng\n    - Vị trí: đau bụng hạ sườn phải, đau lưng (khu trú vùng cạnh cột sống bên trái đoạn giữa lưng, đôi khi lan ra phía trước bụng.)\n    - Thời gian: Thường xuất hiện sau bữa ăn (đau bụng, chướng bụng)\n    - Các triệu chứng liên quan: chướng bụng,  buồn nôn thoáng qua, nôn, đau lưng kéo dài \n    Các sự kiện trước khi nhập viện\nBệnh nhân được nội soi thực quản - dạ dày - tá tràng ngoại trú, kết quả ghi nhận viêm dạ dày.\nĐã ngừng sử dụng thuốc NSAIDs.\nĐược điều trị bằng omeprazole.\nChụp cộng hưởng từ mật tụy  ghi nhận sỏi đoạn cuối ống mật chủ.\nXét nghiệm chức năng gan cho thấy men gan tăng.\nSau đó bệnh nhân đến khoa Cấp cứu để được đánh giá và điều trị tiếp\n.Đánh giá tại bệnh viện\nXét nghiệm:Xét nghiệm chức năng gan ghi nhận tăng men gan.\n',
    '22.txt': '1. Tiền sử bệnh\nCác tập tương tự trước đây: Từng bị khó chịu tương tự trong quá khứ, luôn thuyên giảm khi dùng thuốc\nThuốc trước khi nhập viện\n- Tự điều trị bằng liều cao acetaminophen 500mg, taking 10 pills at a time khi cơn đau dữ dội, thường 4 viên mỗi giờ\n- làm hỏng dạ dày của cô ấy bằng cách uống 2 chai thuốc an thần trong quá khứ\n\n2. Bệnh sử hiện tại\nLý do nhập viện: đau bụng ngày càng nặng\nThời điểm khởi phát triệu chứng\n- Tiền sử 2 ngày đau bụng ngày càng nặng\n- Tiền sử 2 tháng đau bụng liên tục\nDiễn biến bệnh\n- Tiền sử 2 tháng đau bụng liên tục\n- 2 ngày trước khi nhập viện, đau bụng trở nên tồi tệ hơn\n- Diễn biến trong viện: tăng men gan bắt đầu có xu hướng giảm mặc dù tổng bilirubin bắt đầu tăng lên đạt đỉnh 6.7 vào\nCác triệu chứng hiện tại triệu chứng\n- đau bụng ngày càng nặng\n- đau bụng liên tục\n- mệt mỏi toàn thân\n- yếu sức\n- khó thở khi gắng sức\n- ngứa toàn thân\n- Vài lần tiểu ra máu không đau\nĐặc điểm triệu chứng\n- Vị trí: Đau vùng gan phải (RUQ) và thượng vị lan ra sau lưng\n- Không có buồn nôn\n- Không có nôn\n- Không có thay đổi thói quen đại tiện\n- Không có phân có máu, đen hoặc nhựa đường\nCác sự kiện trước khi nhập viện\n- Tự điều trị bằng liều cao acetaminophen 500mg\n- làm hỏng dạ dày của cô ấy bằng cách uống 2 chai thuốc an thần trong quá khứ\nCâu hỏi từ người dùng : Dạ em có một người, hiện tại bạn ấy đang nổi những nét nhẹ nhàng nhẹ nhàng rất đến và khắp nơi, bạn em nổi quanh năm ạ. Những thuốc giảm đau nổi bật rất khó chịu và tư vấn, được biết bạn đã từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn cũng là người nên Yên tĩnh chắc chắn là dây truyền tải. Cả hai người đều không phải dị ứng thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. Mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bệnh nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến \u200b\u200b\u200b\u200b\u200b\u200b\u200bvà có cách nào để giải quyết cho em không ạ. Em cảm ơn rất nhiều .\n\n3. Đánh giá tại bệnh viện\nKết quả xét nghiệm\n- ast 421\n- alt 336\n- alp 185\n- bilirubin toàn phần 0.9\n- total bili bắt đầu tăng lên đạt đỉnh 6.7\n- bảng xét nghiệm viêm gan virus là âm tính\n- ferritin là bình thường\n- ceruloplasmin vẫn đang chờ kết quả đang chờ kết quả tại thời điểm chuyển viện\nKết quả chẩn đoán hình ảnh\n- siêu âm bụng có doppler âm tính âm tính\n- âm tính chụp hida\n- ercp cho thấy túi mật giãn nở rõ rệt túi mật giãn nhưng không có sỏi hoặc bệnh lý giải phẫu bệnh khác\nThủ thuật thực hiện: ercp\nCác phát hiện chẩn đoán khác\n- tăng men gan\n- tăng bilirubin máu',
    '23.txt': '1.  Tiền sử bệnh\n    Các tập tương tự trước đây\n-        Sử dụng các chất kích thích như cà phê, chè, thuốc lá,..\n-        Sự thay đổi nội tiết tố trong chu kỳ kinh nguyệt và trong thời kỳ mãn kinh có thể đóng một vai trò. Trong thời kỳ mãn kinh, đổ mồ hôi đêm và bốc hỏa thường làm gián đoạn giấc ngủ. Mất ngủ cũng phổ biến với thai kỳ.\nLời khuyên dành cho bạn:\n-        Chủ động giải quyết những sang chấn tâm lý có khả năng gây ra các rối loạn lo âu,trầm cảm, stress.\n-        Thiết lập chế độ làm việc, nghỉ ngơi,  luyện tập hợp lý, khoa học.\n-        Tránh làm việc quá mức và không dùng thuốc, các chất kích thích thần kinh trung ương.\nNếu tình trạng không thuyên giảm, bạn nên  đưa mẹ tới gặp bác sĩ chuyên khoa để được kiểm tra và đưa ra hướng điều trị phù hợp.    Tiền sử phẫu thuật / thủ thuật\n    - Phẫu thuật đặt cảng để điều trị chứng tăng nhãn áp ở thời kỳ sơ sinh\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế khi 3 tháng tuổi\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế lại khi [Số] tuổi\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: đau đầu kéo dài kèm theo mờ mắt\n    Thời điểm khởi phát triệu chứng: nhìn mờ tiến triển trong 2 tuần qua\n    Diễn biến bệnh\n    - Bệnh nhân bị não úng tuỷ từ thời kỳ sơ sinh, được điều trị bằng cách đặt dẫn lưu shunt\n    - Trước đây đã từng nhập viện với chẩn đoán não úng thủy, có biểu hiện tăng đau đầu , triệu chứng cải thiện khi nằm ngửa.\n    - Nhập viện trước đây vào khoa Thần kinh để điều trị chứng đau đầu.\n    - Khám nhãn khoa trong quá trình nhập viện trước đó cho thấy không có phù gai thị.\n    - Đến hôm nay, bệnh nhân bị mờ mắt ở cả hai mắt, bên trái nặng hơn bên phải, tiến triển trong 2 tuần qua.\n    - Đến hôm nay, bệnh nhân bị tăng tần suất đau đầu gián đoạn.\n    Triệu chứng hiện tại\n    - đau đầu kéo dài\n    - nhìn mờ ở cả hai mắt, bên trái nặng hơn bên phải\n    -  cơn  đau đầu nhiều hơn\n    Đặc điểm triệu chứng\n    - nhìn mờ: tiến triển trong 2 tuần qua, bên trái nặng hơn bên phải\n    - đau đầu: nhiều hơn\n    Sự kiện trước khi nhập viện\n    - Được bác sĩ nhãn khoa [Name] khám ngoại trú hôm nay\n    - Được bác sĩ nhãn khoa giới thiệu đến Khoa Cấp cứu vì phù gai thị, bên trái nặng hơn bên phải\n3.  Đánh giá tại bệnh viện\n    Dấu hiệu lâm sàng: Phù gai thị, bên trái nặng hơn bên phải (theo báo cáo của bác sĩ nhãn khoa)\n    Các phát hiện chẩn đoán khác\n    - Tăng nhãn áp so với chụp cắt lớp vi tính (ct) đầu từ [Date] (từ lần nhập viện trước)\n    - Phù gai thị (theo báo cáo của bác sĩ nhãn khoa)\n',
    '24.txt': 'Bệnh nhân nam 40 tuổi, bộ đội vào viện với lý do mệt mỏi, vàng da, vàng mắt ngày thứ 10, hiện tại ngày thứ 17 của bệnh. Bệnh diễn biến với các hội chứng và triệu chứng sau:\n        Hội chứng nhiễm trùng nhiễm độc:\n        Khởi phát từ từ sốt nhẹ (37^∘ 8), sốt nóng liên tục có gai rét, không có cơn rét run, dùng 1 viên paracetamol viên sủi, không thấy đỡ, sau 3 ngày bệnh nhân hết sốt thì xuất hiện vàng da, vàng niêm mạc.\n        Kèm theo sốt bn mệt mỏi nhiều chỉ muốn nằm nghỉ. \n        Hiện tại bn hết sốt, hết mệt mỏi, ăn ngủ tốt \n        BC 5,38 G/l; N 51,4%\n        HBsAg (+), Anti HBe (-)\n        Anti HBc IgG (-); Anti HBc IgM (+)\n        Hội chứng viêm gan vàng da ứ mật:\n        Khởi phát bệnh từ từ, da niêm mạc vàng sau khi hết sốt. Kèm theo có nước tiểu ít hơn so với bình thường, vàng sậm như nước vối. Sau điều trị 7 ngày, \n        hiện tại bn vàng da giảm, nước tiểu trong, số lượng đã nhiều hơn so với lúc vào viện, 1000ml/24h \n        Gan to dưới bờ sườn 3cm, bờ tù, mật độ mềm, bề mặt nhẵn, ấn tức. \n        XN lúc vào viện:\n        Bilirubin  toàn phần: 43 mmol/l; trực tiếp: 27 mmol/l\n        Ure: 5,9 mmol/l; Creatinin: 89 micromol/l\n        Hội chứng hủy hoại tế bào gan\n        Lúc vào viện\n        GOT: 542 U/l; GPT: 628 U/l; GGT: 234 U/l\n        Chỉ số Deritis = GOT/GPT < 1\n        Gần nhất:\n        Tỷ lệ prothrombin: 55%\n        Tiền sử dịch tễ:\n        Tiền sử bản thân: chưa bị vàng da, vàng mắt trước đó. Trước đó 3 tháng có cạo râu chung với người cùng đơn vị, uống rượu ít không thường xuyên \n        Tiền sử gia đình: không ai bị nhiễm virus viêm gan B, C \n        Tiền sử dịch tễ: Trong đơn vị có 3 người mang virus viêm gan B. \n        Hiện tại ngày thứ 7 sau khi vào viện: mệt mỏi hết, vàng da giảm nhẹ, ăn ngủ được, đánh răng không chảy máu, tiểu tiện 1000ml/24h, nước tiểu vàng nhẹ, đại tiện phân thành khuôn. Không xuất hiện thêm triệu chứng gì\n2/ Chẩn đoán:\n    - Khi được chuyển vào khoa điều trị, bệnh nhân không còn cảm giáckhó chịu vùng ngực\n        Glucose 5% x 1000ml truyền tĩnh mạch L giọt/phút, dùng đến khi nước tiểu 1500ml/ngày, nước tiểu trong \n        Philpovin 5g x 2 ống, pha vào **********, truyền tĩnh mạch L giọt/phút, dùng đến khi ALT, AST trở về bình thường \n        Fortex 25mg x 4 viên, sáng 2 viên, chiều 2v sau ăn, dùng đến khi ALT, AST trở về bình thường \n        Vitamin 3B x 4 viên, sáng 2 viên, chiều 2 viên sau ăn cấp tính do virus B thể thông thường điển hình mức độ nặng giai đoạn toàn phát\n',
    '25.txt': 'Hỏi: Bác sĩ ơi ơi, bác sĩ cho em hỏi là nếu mình sd nhiều ********** thì liệu có sao k anh? Tính từ năm ngoái tới năm nay em có sd 6 vien ********** ạ. Có một dạo em uống 4 viên trong 4 tuần, nguyên nhân của tất cả lần đó là tụi em sd bcs invisible và bị sì nước khi tụi em test nên em có sd ạ. Trong 6 vien đó thì có 1 vien là ********** loại 12h ạ. Em thấy sau mỗi lần uống thì em có đau bụng nhưng k chảy máu với trễ kinh ạ, kinh tới rất đều ạ. V thì sau này nó sẽ có những tác dụng phụ gì k ạ?\nTrả lời : \nChào bạn! ************************* được đánh giá là phương pháp tránh thai hữu hiệu và an toàn trong trường hợp cần thiết và khi sử dụng đúng cách. Tuy nhiên khi lạm dụng thuốc sẽ gây ra một số hậu quả nghiêm trọng như:\n-        Gây ra tình trạng tắc ống dẫn trứng, lạm dụng thuốc còn khiến teo niêm mạc tử cung, không rụng trứng dẫn đến vô sinh. Nhiều trường hợp nguy hiểm còn gây ra ung thư cổ tử cung do dùng thuốc quá nhiều.\n-        Sử dụng nhiều ************************* khiến chị em tăng nguy cơ mang thai ngoài tử cung.\n-        Chị em sử dụng thuốc quá liều gây ảnh hưởng đến tim, mạch, gan, thận...\n-        Nồng độ hormone thay đổi nội tiết tố trong cơ thể gây ra các bệnh lý về da có thể xuất hiện nhiều mụn trứng cá, nám, tàn nhang, sạm da…\n-        Ảnh hưởng đến tâm trạng, cảm xúc, hay thấy bồn chồn, lo lắng, bứt rứt trong người, gây giảm ham muốn, lãnh cảm, suy giảm hưng phấn đối với tình dục, ảnh hưởng đến tình cảm gia đình.\n    Lý do nhập viện: xuất hiện nhiều  ban đỏ và đau khi sờ nắn ở vết mổ\n    Triệu chứng Bắt đầu\n    - sau khi xuất viện\n    - ống dẫn lưu JP được rút vào ngày DD MM\n    - bắt đầu có hiện tượng sưng nề đau khi sờ nắn trên vùng da kèm theo ban đỏ\n    Diễn biến bệnh\n    - tăng diện tích đau khi sờ nắn trên vùng da kèm theo ban đỏ\n    - đến khám tại phòng khám\n    - vùng da sưng nề được chọc hút dịch, gửi đi cấy mẫu bệnh phẩm\n    - được cho về vớ đơn thuốci levafloxacin và cephalexin\n    - tái khám với triệu chứng xuất hiện  ban đỏ lan rộng kèm theo chóng mặt\n    - được chuyển đến Khoa Cấp cứu\n    - Tại thời điểm khám bệnh  bệnh nhân có các triệu chứng tương tự\n    Triệu chứng hiện tại\n    -  ban đỏ xuất hiện nhiểu ở vị trí phẫu thuật\n    - đau khi sờ nắn vùng mổ\n    - chóng mặt\n    - không sốt\n    - hiện tại chưa phát hiện các triệu chứng toàn thân khác\n3.  khám tại bệnh viện\n    Dấu hiệu lâm sàng\n    - ban đỏ ở vị trí phẫu thuật\n',
    '26.txt': '\nBỆNH MẠCH VÀNH (Coronary Artery Disease – CAD)\n\n1. Bệnh mạch vành là gì? \n\nBệnh mạch vành là tình trạng hẹp hoặc tắc các động mạch vành – những mạch máu có nhiệm vụ nuôi dưỡng cơ tim. Nguyên nhân chủ yếu là xơ vữa động mạch, làm giảm tưới máu cơ tim, gây thiếu oxy cơ tim, từ đó dẫn đến đau thắt ngực, nhồi máu cơ tim, suy tim hoặc đột tử .\n\n2. Nguyên nhân và cơ chế bệnh sinh \n\nNguyên nhân chính\n\nXơ vữa động mạch vành:\n • LDL-cholesterol tăng\n • Lắng đọng lipid → hình thành mảng xơ vữa\n • Mảng xơ vữa to dần → hẹp lòng mạch\n • Nứt vỡ mảng xơ vữa → huyết khối → tắc mạch cấp \n\nYếu tố nguy cơ\n\nKhông thay đổi được\n • Tuổi cao \n • Nam giới\n • Tiền sử gia đình bệnh tim mạch sớm\n\nCó thể thay đổi\n • Hút thuốc lá \n • Tăng huyết áp\n • Đái tháo đường \n • Rối loạn lipid máu\n • Béo phì, ít vận động\n • Stress kéo dài \n\nTriệu chứng lâm sàng \n\nĐau thắt ngực (triệu chứng điển hình)\n\nVị trí: sau xương ức\nTính chất: đè nặng, bóp nghẹt, thắt chặt\nLan: vai trái – cánh tay trái – cổ – hàm dưới\nThời gian: vài phút\nGiảm khi nghỉ hoặc dùng nitroglycerin\nCác thể lâm sàng\n\nĐau thắt ngực ổn định\n • Xảy ra khi gắng sức\n • Giảm khi nghỉ\n\nHội chứng vành cấp\n • Đau dữ dội, kéo dài\n • Không đỡ khi nghỉ\n • Gồm:\n • Nhồi máu cơ tim ST chênh\n • Nhồi máu không ST chênh\n • Đau thắt ngực không ổn định\n\nTriệu chứng không điển hình (hay gặp ở nữ, người già, ĐTĐ):\n • Khó thở\n • Mệt\n • Buồn nôn\n • Đau thượng vị\n\n\n4. Chẩn đoán \n\nLâm sàng\n • Khai thác đau ngực + yếu tố nguy cơ\n\n1.  Tiền sử bệnh\n    bệnh nhân có tiền sử nhập viện gần đây vìviêm tụy\n    Bệnh lý mãn tính\n    - tiền sử rung nhĩ\n    - phẫu thuật cắt bỏ tuyến tiền liệt (u ác của tuyến tiền liệt)\n    - rối loạn cảm xúc lưỡng cực khác\n    Tiền sử phẫu thuật / thủ thuật: phẫu thuật cắt bỏ tuyến tiền liệt (u ác của tuyến tiền liệt)\n    Thuốc trước khi nhập viện: đang dùng eliquis (cho rung nhĩ)\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: sốt và đau bụng\n    Thời điểm khởi phát triệu chứng\n    - khởi phát đau bụng trên (lần nhập viện trước)\n    - Hôm nay, bệnh nhân có sốt nhẹ đến 38.3°C kèm theo đau hạ sườn phải tái phát, ngày càng nặng hơn.\n    Diễn biến bệnh\nTập thể dục ≥ 150 phút/tuần\nKhông hút thuốc\nGiữ cân nặng hợp lý\nGiảm stress\nKiểm soát HA – đường huyết – lipid\n\nDự phòng thứ phát (đã mắc bệnh)\n\nUống thuốc đều\nTái khám định kỳ\nKiểm soát LDL < 1,8 mmol/L (hoặc <70 mg/dL)\nNhận biết sớm dấu hiệu đau ngực',
    '27.txt': 'Hỏi : Dạ cho em hỏi viêm phổi hoại tử có phải là giai đoạn nặng của viêm phổi và tiến triển tiếp theo là hình thành áp xe phổi. Hay là VP và VPHT là hai bệnh có cơ chế bệnh sinh khác nhau ạ.\nTrả lời : Chào bạn!\nViêm phổi là tình trạng viêm phế quản, phế nang, giai đoạn đầu là xuất tiết dịch, nếu điều trị sớm và kịp thời thì sẽ nhanh chóng hết với những cơ thể có sức đề kháng bình thường.\nViêm phổi tiến triển nặng là giai đoạn sau, có thể do nhiễm vi khuẩn nặng, thuốc kháng sinh không đáp ứng hay cơ thể có sức đề kháng yêu, suy giảm miễn dịch và hoại tử nhu mô phổi.\nViêm phổi hoại tử: là một thể nặng của bệnh lý phổi với sự hình thành của các hang nhỏ, áp-xe nhỏ (<2cm) trong nhu mô phổi, thường không kèm theo tổn thương màng phổi đáng kể.\nÁp xe phổi: là một tình trạng nung mủ, hoại tử chủ mô phổi sau một quá trình viêm cấp. Cơ chế: \nLúc đầu trong nhu mô phổi bị viêm xuất hiện một hay nhiều ổ viêm hóa mủ, nhu mô phổi bị đông đặc, nếu điều trị ở giai đoạn này thì thương tổn có thể phục hội hoàn toàn. Nếu không thì các ổ viêm này sẽ hoại tử lan rộng và kết hợp lại thành một ổ lớn hoại tử và có mủ. Đây là giai đoạn nung mủ cấp và áp xe phổi đã hình thành, có vỏ mỏng bao bọc. Sau đó thương tổn các phế quản lân cận và bệnh nhân sẽ khạc ra mủ, và các tổ chức hoại tử. Sau một thời gian (khoảng 6-8 tuần) thì viêm xơ bắt đầu bao quanh ổ áp xe tạo nên nhiều vách ngăn, hoặc là mủ sẽ lan qua vùng lân cận gây nên các thương tổn mới.\n    - Đến hôm nay, bệnh nhân bị tăng tần suất đau đầu gián đoạn.\n    Triệu chứng hiện tại\n    - đau đầu kéo dài\n    - nhìn mờ ở cả hai mắt, bên trái nặng hơn bên phải\n    -  cơn  đau đầu nhiều hơn\n    Đặc điểm triệu chứng\n    - nhìn mờ: tiến triển trong 2 tuần qua, bên trái nặng hơn bên phải\n    - đau đầu: nhiều hơn\n    Sự kiện trước khi nhập viện\n    - Được bác sĩ nhãn khoa [Name] khám ngoại trú hôm nay\n    - Được bác sĩ nhãn khoa giới thiệu đến Khoa Cấp cứu vì phù gai thị, bên trái nặng hơn bên phải\n3.  Đánh giá tại bệnh viện\n    Dấu hiệu lâm sàng: Phù gai thị, bên trái nặng hơn bên phải (theo báo cáo của bác sĩ nhãn khoa)\n    Các phát hiện chẩn đoán khác\n    - Tăng nhãn áp so với chụp cắt lớp vi tính (ct) đầu từ [Date] (từ lần nhập viện trước)\n- Có nhiều nang chứa khí hay dịch nhỏ (<2cm) (Trong khi áp-xe phổi kích thước ổ hoại tử trên 2cm, vách dày > 2mm)\nChúc bạn học tập tốt!',
    '28.txt': 'Câu hỏi từ người dùng :\n\nDạ em có một người bạn, hiện tại bạn ấy đang bị nổi các dấu mề đay rất to và ngứa khắp người, bạn em bị nổi quanh năm luôn ạ. Những dấu mề đay nổi rất khó chịu và ngứa, được biết bạn em có từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn ấy cũng bị nên em nghỉ chắc là di truyền. Cả hai người đều không phải dị ứng do thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bị nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến và có cách nào để chữa cho bạn không ạ. Em cảm ơn rất nhiều\n\n\nCâu trả lời của bác sĩ: \n\nChào bạn,mình xin trả lời câu hỏi của bạn như sau\nNhư bạn đã mô tả, trường hợp này bạn  thuộc MÀY ĐAY VÔ CĂN. Chúng ta không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY ĐAY MẠN TÍNH.\nTổn thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với lympho T và đại thực bào. Các tổn thương khó quan sát trên kính hiển vi quang học thông thường mà cần xem trên kính hiển vi điện tử. Các tế bào mast tăng số lượng vùng hạ bì với các mức độ thoát bọng khác nhau được quan sát thấy. Nhuộm huỳnh quang miễn dịch tổn thương sinh thiết không thấy có hình ảnh của lắng đọng các phức hợp miễn dịch, bổ thể hay sợi fibrin.\nĐối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Mày đay vô căn là một bệnh mạn tính, việc điều trị phải lâu dài và liên tục. Quá trình điều trị chỉ là điều trị triệu chứng.\nĐiều trị phụ thuộc vào mức độ nặng và bệnh lý nền có liên quan. Cần lưu ý cân bằng giữa hiệu quả điều trị kiểm soát triệu chứng và những tác dụng gây độc của liệu pháp điều trị. **************** thể hệ sau không gây buồn ngủ là lựa chọn hàng đầu sau đó đến **************** thể hệ 1, corticoid liều thấp cách ngày hoặc hàng ngày và giảm liều chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không đỡ hoặc có chống chỉ định, xin bạn hãy đến với bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất.\n3.  Đánh giá tại bệnh viện\n    Kết quả chụp ảnh: chụp cộng hưởng từ (mri) cột sống cổ cho thấy hẹp ống sống C4-5, C5-6, C6-7 và hẹp lỗ liên hợp',
    '29.txt': '1. Tiền sử bệnh nội\nCác bệnh lý mạn tính\n- ho Bệnh bạch cầu dòng tủy mãn tính\n- Tăng huyết áp\n- Đái tháo đường típ 2\n- ho Rung nhĩ kèm đáp ứng thất nhanh\n- bệnh thận mạn, không đặc hiệu\n2. Bệnh sử hiện tại\nLý do vào viện: Khó thở tăng dần, diễn tiến nặng hơn từ sáng ngày nhập viện.\nKhoảng 5 ngày trước nhập viện, bệnh nhân xuất hiện cảm giác nghẹt ngực kèm sốt, nhiệt độ cao nhất khoảng 38,3°C. \nSau đó không còn sốt nhưng tình trạng nghẹt ngực vẫn kéo dài. \nTrong vài ngày gần đây, bệnh nhân xuất hiện khó thở tăng dần, chủ yếu khi gắng sức, kèm phù ngoại vi tăng dần trong vài tuần trở lại đây.\nBệnh nhân có tiền sử nhập viện gần đây vì sốt và đau vai. \nTrong đợt điều trị đó, bệnh nhân diễn tiến suy hô hấp kèm tăng huyết áp, không đáp ứng đáng kể với điều trị lợi tiểu và phải chuyển khoa Hồi sức tích cực để hỗ trợ thở áp lực dương không xâm nhập (BiPAP). \nTình trạng hô hấp cải thiện sau hỗ trợ hô hấp và bệnh nhân ổn định trở lại. \nTrong thời gian nằm viện, bệnh nhân được chẩn đoán nhiễm khuẩn huyết do tụ cầu vàng nhạy cảm methicillin, nghi liên quan đến đường truyền tĩnh mạch trung tâm ngoại vi (PICC). \nSiêu âm mạch máu chi trên không ghi nhận huyết khối, xạ hình thông khí - tưới máu phổi cho thấy xác suất thấp thuyên tắc phổi.\nHỏi: Chào bác sĩ! Cháu có 1 nhóc 3,5 tuổi. Cháu đang nghĩ cháu bị hội chứng bàn chân bẹt bẩm sinh, vì phần gót chân của con cháu khi nhìn ngoại quan nó hơi lệch , lòng bàn chân phẳng. Cháu chạy, nhảy bình thường, chân có hơi vòng kiềng nhưng khó thấy. Cháu rất mong bác sĩ tư vấn cho cháu hướng điều trị để giúp con cháu tránh được những ảnh hưởng về sau. Và mức độ thành công có cao không ạ. Cháu xin cảm ơn bác!\nTrả lời:\nTheo như mô tả của Bạn cháu bé bị bệnh bàn chân bẹt, đây là tật bẩm sinh không ảnh hưởng nhiều đến sức khỏe. Hiện tại cháu chạy nhảy bình thường, không đau, chưa có chỉ định phẫu thuật, Tuy nhiên bạn nên cho cháu đến bệnh viện để kiểm tra đánh giá bất thường về xương, khám lâm sàng để đánh giá các thiếu hụt về phần mềm, trên cơ sở đó chúng tôi sẽ có các hướng dẫn cụ thể về tập vận động, tiên lượng về tiến triển của bệnh một cách cụ thể. \nSiêu âm mạch máu chi trên không ghi nhận huyết khối.\nXạ hình thông khí – tưới máu phổi cho thấy xác suất thấp thuyên tắc phổi.\n',
    '30.txt': '    Các bệnh mãn tính: Không có tiền sử bệnh tim mạch hoặc thận bệnh nào được biết đến\n    Thuốc trước khi nhập viện lần này: đang dùng methadone\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Đánh giá phù, chủ quan sốt và ớn lạnh, và đau tăng dần khi đi lại\n    Các triệu chứng hiện tại\n    - phù hai bên\n    - ban đỏ, RL\n    - chủ quan sốt\n    - ớn lạnh\n    - đau tăng dần khi đi lại\n    Đặc điểm triệu chứng\n    - Vị trí: RL (Chân phải)\n    - Mức độ nghiêm trọng: đau tăng dần\n    - Các triệu chứng liên quan: phù, ban đỏ, chủ quan sốt, ớn lạnh\n    Các sự kiện trước khi nhập viện: được chuyển từ nơi trú ẩn đến Khoa Cấp cứu để đánh giá\n    Tình trạng ngay trước khi nhập viện\n    - lơ mơ khi đến tầng và không thể tỉnh táo đủ lâu để cung cấp tiền sử\n    - được ghi nhận phù hai bên và ban đỏ, RL\nCâu trả lời của bác sĩ: Chào bạn,mình xin trả lời câu hỏi của bạn như sau Như bạn đã mô tả, trường hợp này bạn thuộc MÀY đay VÔ CĂN. Chúng tôi không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY đay MẠN. Tổ thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với bạch huyết T và đại thực bào. Các loại tiền thưởng khó quan sát trên kính hiển thị vi quang học thông thường cần xem trên kính hiển thị vi điện tử. Các tế bào cột sống tăng lượng hạ bì với các mức độ thoát ra khác nhau cũng được quan sát. Quảng cáo quảng cáo thiết bị thương mại miễn phí dịch sinh học không tìm thấy hình ảnh giảm phức tạp, bổ sung hay sợi fibrin. Đối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Bạn đay vô căn hộ là một bệnh mạn tính, việc điều trị phải dài và liên tục. Quá giá trị chỉ là giấy chứng nhận. Điều trị phụ thuộc vào mức độ nghiêm trọng và nền tảng có liên kết. Cần lưu ý cân bằng giữa hiệu quả kiểm soát triệu chứng và những tác hại gây độc của liệu pháp điều trị. *************** có thể hệ sau không gây buồn ngủ là đơn hàng đầu sau đó đến *************** có thể hệ 1, ********* bậc thấp cách ngày hoặc hàng ngày và giảm dần chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không được hỗ trợ hoặc chống chỉ định, xin bạn hãy đến gặp bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất. Cảm ơn bạn đã gửi câu hỏi',
    '31.txt': '1.  Tiền sử bệnh\n    Bệnh nhân có tiền sử dụng thuốc\n    - NSAID để điều trị đau đầu gối (đã ngừng)\n    - omeprazole (bắt đầu dùng)\n\n2.  Tiền sử bệnh hiện tại\nCâu hỏi từ người dùng :\n\nEm trai mình sn 2004, đi khám ở Tuyến Tỉnh bs chẩn đoán giãn thừng tinh 2.5mm 1 bên, đã khám và điều trị thuốc uống 5 tuần, độ giãn không giảm, thỉnh thoảng đau nhẹ, cảm thấy nóng phần tinh hoàn, đã ngừng dùng thuốc, vì tình hình covid không thể vào hcm để vào chuyên khoa khám , nhờ quý bác sĩ tư vấn được không ạ, để lâu em sợ ảnh hưởng tinh trùng ảnh hưởng sinh sản ạ\n\nCâu trả lời của bác sĩ: \n\nChào bạn. Giãn thừng tinh là một trong nhiều nguyên nhân gây vô sinh thứ phát. Với thông tin mà bạn cung cấp thì hãy đến tái khám tại BV tuyến tỉnh nơi bạn sinh sống để được bs chuyên khoa đưa ra chỉ định điều trị phẫu thuật phù hợp nhé. Hầu hết bv tuyến tỉnh đều có thể điều trị phẫu thuật giãn thừng tinh hiệu quả. Chúc bạn nhiều sức khoẻ.    Các triệu chứng hiện tại\n    - đau bụng vùng hạ sườn phải\n    - chướng bụng \n    - buồn nôn thoáng qua\n    - Nôn mửa\n    - Bệnh nhân có  đau lưng âm ỉ\n    Đặc điểm triệu chứng\n    - Vị trí: đau bụng hạ sườn phải, đau lưng (khu trú vùng cạnh cột sống bên trái đoạn giữa lưng, đôi khi lan ra phía trước bụng.)\n    - Thời gian: Thường xuất hiện sau bữa ăn (đau bụng, chướng bụng)\n    - Các triệu chứng liên quan: chướng bụng,  buồn nôn thoáng qua, nôn, đau lưng kéo dài \n    Các sự kiện trước khi nhập viện\nBệnh nhân được nội soi thực quản - dạ dày - tá tràng ngoại trú, kết quả ghi nhận viêm dạ dày.\nĐã ngừng sử dụng thuốc NSAIDs.\nĐược điều trị bằng omeprazole.\nChụp cộng hưởng từ mật tụy  ghi nhận sỏi đoạn cuối ống mật chủ.\nXét nghiệm chức năng gan cho thấy men gan tăng.\nSau đó bệnh nhân đến khoa Cấp cứu để được đánh giá và điều trị tiếp\n.Đánh giá tại bệnh viện\nXét nghiệm:Xét nghiệm chức năng gan ghi nhận tăng men gan.\nChẩn đoán hình ảnh và thăm dò:Nội soi thực quản - dạ dày - tá tràng: viêm dạ dày.\nCộng hưởng từ mật tụy : sỏi đoạn cuối ống mật chủ.\nThủ thuật được thực hiện   :Nội soi mật tụy ngược dòng (ERCP). Trong quá trình ERCP, lấy thành công 02 viên sỏi tại đoạn cuối ống mật chủ (CBD).\n    - chụp cộng hưởng từ mật tụy tụi mật được thực hiện cho thấy sỏi ống dẫn mật chung đoạn cuối\n',
    '32.txt': 'Câu hỏi từ người dùng:\nEm bị Rối loạn chuyển hóa tinh bột (amyloidosis) đã lâu. Có dùng thuốc tại viện da liễu trung ương nhưng không thấy đỡ. Có cách nào chữa trị tận gốc hoặc giảm thiểu không ạ? Em cảm ơn.\nCâu trả lời của bác sĩ:\nChào bạn! Cảm ơn bạn đã gửi câu hỏi cho chúng tôi.\n Trả lời câu hỏi của bạn:Bệnh thoái hóa tinh bột (hay bệnh amyloidosis) là một bệnh gây lắng đọng protein hiếm gặp và nghiêm trọng. Nguyên nhân là do một loại protein bất thường gọi là amyloid tích tụ trong các mô hoặc cơ quan. Khi lượng protein amyloid lắng đọng tăng lên, chúng sẽ phá vỡ cấu trúc và gây cản trở chức năng sinh lý của mô hoặc cơ quan. Cuối cùng, sự lắng đọng protein amyloid sẽ bắt đầu gây ra các triệu chứng và suy các cơ quan, thậm chí có thể gây tử vong.\n    - Bệnh bạch cầu dòng tủy mãn tính đang dùng gleevec\n    - Tăng huyết áp\n    - tăng lipid máu, không đặc hiệu\n    - Đái tháo đường típ 2\n    - hẹp ống sống\n    - Giả gout\n    - bệnh thận mạn, không đặc hiệu Giai đoạn 4\n    - tăng sản tuyến tiền liệt\n    - Nhiều lần ngã gần đây Ngã\n    - ảo giác\n    Thuốc trước khi nhập viện: gleevec (dừng theo chỉ dẫn sau xuất viện)\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện\n    - Toàn trạng suy kiệt  kể từ khi xuất viện\n    - ảo giác dai dẳng\n    - Lú lẫn ngày càng nặng\n    Thời điểm khởi phát triệu chứng: Cực kỳ yếu kể từ khi xuất viện\n    Diễn biến bệnh\n    - ảo giác được vợ nhận thấy\n    - Lú lẫn ngày càng nặng được vợ nhận thấy\n    - yếu cơ dẫn đến ngã hôm nay\n    Triệu chứng hiện tại\n    - toàn trạng suy kiệt\n    - ảo giác\n    - Lú lẫn\n    - Ăn uống kém, ăn vào dễ nôn.\n    - Nôn không ra máu, không ra dịch mật\n•        Bệnh amyloidosis chuỗi nhẹ\n•        Bệnh amyloidosis tự miễn dịch\n•        Bệnh amyloidosis di truyền hoặc gia đình\nBệnh thoái hóa tinh bột là một bệnh đa hệ thống dẫn đến nhiều biểu hiện lâm sàng khác nhau, Hiện nay chưa có phương pháp điều trị tận gốc amyloidosis.Tuy nhiên, các phương pháp điều trị có thể giúp bạn kiểm soát các triệu chứng bệnh cũng như ngăn chặn tổn thương \nthêm. Tùy vào loại amyloidosis mà sẽ có phương pháp điều trị khác nhau như:  Uống thuốc, hóa trị, ghép gan,…\nDo đó bạn nên tới gặp bác sĩ khám lại để có phương pháp điều trị phù hợp.\n',
    '33.txt': '1. Tiền sử bệnh\nCác bệnh lý mạn tính\n- rung nhĩ\n- tăng huyết áp\n- suy tim, không đặc hiệu\n- béo phì\n\nCháu năm nay 18 tuổi da rất nhiều dầu và bị mụn trứng cá từ năm lớp 8. Đến năm lớp 11 cháu đi khám da liễu, thì sau khoảng 3 tháng điều trị da cháu ít dầu hơn và dường như không còn mụn. Nhưng sau 6 tháng da cháu bắt đầu lên mụn trở lại gồm mụn ở trán, má, 2 bên quai hàm và cằm, hầu như là mụn cám và mụn đầu trắng. Cháu không biết có nên đi khám lại không vì lo ngại thuốc sẽ ảnh hưởng đến sức khoẻ sinh sản cộng thêm việc khám sau một thời gian lên lại mụn thì rất tốn kém. Mong các bác sĩ cho cháu lời khuyên về vấn đề chăm sóc da dầu mụn và làm thế nào để da hết mụn không tái phát ạ. Cháu cảm ơn các bác sĩ ạ.\n\n- mệt mỏi và weak\n- Thường xuyên ngủ quên hoặc có thể ngất xỉu\n- giảm khả năng quan hệ tình dục với bạn tình\n- Đau buốt khi đi tiểu\n- chán ăn\n- nôn từng đợt sau khi ăn\nĐặc điểm triệu chứng\n- đau ngực:\n- xảy ra khoảng trước khi nhập viện khi bệnh nhân đang nằm\n- nhói\n- bên trái\n- lan xuống cánh tay trái\n- kèm theo tê bì ở cánh tay trái trên và đau ở cánh tay trái dưới và bàn tay\n- liên tục\n- không nặng hơn/giảm khi làm gì\n- không giảm khi dùng bất kỳ thuốc nào ở phòng cấp cứu\n- shortness of breath:\n- tăng dần trong vài tuần qua\n- yếu:\n- tăng dần trong vài tuần qua\n- đau rát khi đi tiểu:\n- từng đợt, khoảng 2 tuần\n- Ăn uống:\n- kém trong 3 ngày qua\n- Nôn mửa:\n- từng đợt sau khi ăn trong 3 ngày qua\n- cơn ngất xỉu:\n- 2 tháng trước\n- bệnh nhân không nhớ rõ\n- va phải cánh tay phải và bụng phải nhưng không nhớ đập đầu\n- không nhớ bị đánh trống ngực, chóng mặt trước khi bị ngã\nSự kiện trước khi nhập viện\n- Bệnh nhân đã dùng thêm 80mg po lasix ở nhà mà không có tác dụng\n- Bệnh nhân không được chăm sóc y tế thường xuyên trong những tháng gần đây; bác sĩ chăm sóc chính của anh ấy từ trần và anh ấy chưa theo dõi với ai khác\n\n3. Đánh giá tại bệnh viện\nCác thủ thuật đã thực hiện\n- Nhận 2 sl ntg\n- Nhận asa\n- Nhận 80mg lasix iv\n- Nhận 8mg morphine\n- Nhận 1mg dilaudid\n- Được cho po metoprolol\n- Được cho 10mg iv diltiazem\nCác phát hiện chẩn đoán khác\n- Với ntg khó thở cải thiện nhưng đau ngực vẫn còn\n- Với ntg khó thở cải thiện nhưng đau ngực vẫn còn\n- nhịp tim cải thiện thành',
    '34.txt': 'Câu hỏi từ người dùng :\n\nChào bác sĩ\nDạo gần đây em hay bị đau nửa đầu dữ dội, mỗi lần đau có cảm giác ớn lạnh, buồn nôn và hơi sốt? Bạn sĩ tư vấn giúp em là em có khả năng bị bệnh gì và cách đều trị như thế nào ạ?\n\nCâu trả lời của bác sĩ: \n\nChào bạn\nNhững triệu chứng của bạn có vẻ giống Bệnh lý đau nửa đầu hay còn gọi là bệnh đau nửa đầu Migraine, bệnh lý này sẽ khiến người bệnh thấy đau nửa đầu không cố định hoặc đôi lúc đau cả hai bên, kèm theo các triệu chứng mạch đập. Tiền tình trạng đau nửa đầu, người bệnh sẽ có các biểu hiện về thị giác bị nhòe, ruồi bay, buồn nôn, mạch đập ở vùng thái dương. Tùy vào mức độ mà người bệnh có thể có đau vừa hoặc đau dữ dội, cơn đau tăng dần, thậm chí kèm theo các triệu chứng buồn nôn, sợ ánh sáng, sợ tiếng động, căng thẳng.\n-Với tình trạng đau nửa đầu, đa số người bệnh đều bị đau nửa đầu bên trái. Tình trạng đau kéo dài, thường xuyên dễ bị chẩn đoán nguyên nhân đau đầu do viêm xoang và có thể dẫn đến điều trị không đúng bệnh. Người bị cơn đau nửa đầu trái hành hạ để lâu sẽ gây ra các hậu quả nguy hiểm cho sức khỏe như suy giảm trí nhớ, khó tập trung, trầm cảm thậm chí là đột quỵ. Một số trường hợp còn có biến chứng suy thoái võng mạc, mất thị lực và mù vĩnh viễn.\n- Bên cạnh đó, người bệnh cần áp dụng các biện pháp sau giúp giảm đau, giảm tần suất cơn đau nửa đầu xuất hiện:\n    - gleevec (ngừng uống cách nhập viện 5 ngày)\n    - allopurinol (ngừng uống \ncách nhập viện 5 ngày)\n    - Thuốc giảm đau (ngừng)\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: ảo giác\n    Thời điểm khởi phát triệu chứng: Khoảng một tháng trước\n    Diễn biến\n    - ảo giácxuất hiện khoảng một tháng trước.\n    - Đã được điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định.\n    - Thuốc giảm đau đã ngừng.\n    - Không có thay đổi đáng kể về triệu chứng mặc dù đã điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định và ngừng thuốc giảm đau.\n    - Liều gleevec đã ngừng cách nhập viện 5 ngày.\n    - allopurinol đã ngừng cách nhập viện 5 ngày.\n- Quan trọng nhất bạn vẫn nên đi tham khám bác sĩ để chụp các xét nghiệm cần thiết nhé',
    '35.txt': 'Câu hỏi từ người dùng:\n\n2. Tiền sử bệnh bệnh hiện tại\nLý do nhập viện: xét nghiệm gắng sức bất thường\nThời điểm khởi phát triệu chứng: Hôm qua (không rõ ngày cụ thể)\nCác triệu chứng hiện tại\n- khó chịu vùng ngực gián đoạn\n- chóng mặt\n- khó thở (khó thở)\n- đổ mồ hôi\nĐặc điểm triệu chứng\n- Vị trí: Không rõ (ngụ ý vùng ngực)\n- Mức độ nghiêm trọng: Không rõ\n- Thời gian: Không rõ\n- Tần suất: Gián đoạn\n- Chiếu r xạ: Không rõ\n- Các yếu tố làm nặng thêm: Hoạt động gắng sức\n- Các yếu tố làm giảm bớt: Nghỉ ngơi\n- Các triệu chứng liên quan: chóng mặt, khó thở, đổ mồ hôi\nCác sự kiện trước khi nhập viện: xét nghiệm gắng sức bất thường\n\n3. Đánh giá tại bệnh viện\nKết quả thăm khám lâm sàng\n- Không có đau tại phòng cấp cứu\n\nCâu trả lời của bác sĩ:\n\nViêm hang vị sung huyết là tình trạng niêm mạc vùng hang vị dạ dày viêm, các mạch máu vùng viêm giãn nở do ứ máu nhiều. Biểu hiện chủ yếu là đau bụng cồn cào kèm theo ợ hơi, ợ chua, có thể có cảm giác buồn nôn hoặc nôn. Trước kia bệnh dạ dày được coi là bệnh nan y và nguy hiểm nhưng mấy chục năm gần đây nhờ nội soi phát triển nên bệnh đã được điều trị hiệu quả. Để điều trị dứt điểm bệnh viêm sung huyết hang vị dạ dày, trước hết, người bệnh cần được thăm khám chuyên khoa, nội soi dạ dày để tìm nguyên nhân, đánh giá mức độ bệnh. Căn cứ trên kết quả khám, các bác sĩ sẽ đưa ra phác đồ điều trị hiệu quả nhất cho từng bệnh nhân. Điều quan trọng là bệnh nhân cần tuân thủ tuyệt đối sự chỉ dẫn của bác sĩ trong quá trình điều trị. Không được bỏ thuốc giữa chừng cũng như không được tự ý tăng giảm liều lượng thuốc mà chưa có sự đồng ý của bác sĩ. Ngoài ra, người bị viêm sung huyết hang vị dạ dày có thể sử dụng sản phẩm từ thiên nhiên như nghệ, mật ong... để hỗ trợ điều trị. Bên cạnh đó, cần có chế độ ăn uống khoa học, lành mạnh. Nên kiêng ăn các thức ăn có vị chua, cay, nóng và các loại đồ ăn nhiều dầu mỡ. Không uống rượu, bia, nước có ga. Không hút thuốc lá, thuốc lào. Hạn chế uống cà phê... Khi ăn nên ăn chậm, nhai kỹ, ăn uống điều độ đúng giờ, đủ bữa, không ăn quá no hoặc để quá đói. Và cần có chế độ tập luyện phù hợp.',
    '36.txt': 'Bệnh nhân nam 17 tuổi, vào viện với lý do nặng mặt, tức nặng 2 chi dưới, tiểu ít, bệnh khởi phát cách đây 1 tháng, quá trình bệnh diễn biến với các hội chứng và triệu chứng sau: \n        Hội chứng thận hư:\n      Phù: xuất hiện đột ngột, tiến triển nhanh, đầu tiên ở quanh 2 mi mắt, mặt, sau 1 ngày xuất hiện phù 2 chi dưới, phù tăng về sáng, sau ngủ dậy, giảm về chiều, khi ăn mặn phù tăng lên. Hiện tại hết phù mặt, còn phù nhẹ 2 chi dưới.\n       Protein niệu 24h:\n       Protein máu:   Protein: 52 g/l; Albumin: 20 g/l.\n       Triglycerid:6,7 mmol/l\n     Tổn thương cầu thận mạn tính\n     Phù hợp  với các tính chất như trên.   Xét nghiệm nước tiểu protein (–), HC (–)\nTrụ niệu (–): trụ trong, trụ sáp, trụ hình hạt.\n        SA: không có sỏi đài bể thận, niệu quản, theo dõi viêm cầu thận\n        Không có suy thận: Ure: 6,4 mmol/l; Creatinin: 79 micromol/l\n     Không có thiếu máu: HC: 4,49 T/l; HST: 103 g/l\n       Điện giải: Na+: 141 mmol/l; K+: 5 mmol/l; Cl-: 106 mmol/l; Ca++: 2 mmol/l.\n        Tiền sử: Không có viêm họng cấp, viêm nhiễm ngoài da trong 1 tháng trước khi vào viện\n2. Chẩn đoán: Viêm cầu thận mạn - Hội chứng thận hư\n3. Tiên lượng: vừa, bệnh tổn thương mạn tính, tỉ lệ tái phát cao.\n4. Hướng xử trí:\n        Làm thêm các xét nghiệm theo dõi tiến triển bệnh: protein niệu 24h, protein máu, albumin máu, điện giải đồ.\n       Chế độ ăn: cơm nhạt, uống ít nước < 1l/ngày, tăng protein (2 lạng thịt/ngày), không lipid, tăng glucid.\n        Thuốc:\n        Điều trị nguyên nhân: điều trị VCTM theo cơ chế bệnh sinh\n        Điều trị triệu chứng: phù, tăng lipid máu, giảm thải protein.\n5. Đơn thuốc cụ thể cho 1 ngày\n1.        Medrol 16mg x 3 viên, uống 8h sáng sau ăn no\n2.        Omez 20mg x 1 viên, uống 8h sáng.\n3.        Furosemid 40 mg x 1 viên, uống sáng.\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: FNA của một nốt sần tuyến giáp phải có cấu trúc vi nang\n    Triệu chứng hiện tại: đau vùng hạ sườn phải\n    Các sự kiện trước khi nhập viện\n    - siêu âm cho thấy sỏi mật\n5.        Zestril 10mg x 1 viên, uống sáng.',
    '37.txt': 'Hỏi : Bé nhà mình là bé trai, khi bé đc 20 ngày tuổi mình mới phát hiện tai phải bé không có lỗ tai, vẫn có vành tai đầy đủ, mình có đưa bé đi khám ở 1 vài nơi nhưng vì bé còn quá nhỏ nên bs chỉ nhìn bên ngoài và đo âm tai còn lại. Bé nay đã 2 tuổi, nghe nói bình thường, mọi sinh hoạt bình thường, phát triển đều, nhưng mình cảm thấy thật sự lo lắng vì bên tai ko có lỗ kia vành tai không phát triển đều, nó nhỏ hơn tai còn lại. Ngược lại, bên tai trái thì rất to và vành tai đẹp. Mình cũng để ý con xem có phản ứng khi mình gọi không và bé vẫn nghe bìng thường, kể cả khi sắp ngủ mình nói nhỏ bé vẫn nghe và mở mắt nhìn. Mình rất lo lắng về tai của bé, sợ sau này sẽ ảnh hưởng đến sự phát triển của bé và bé bị trêu chọc. Y học hiện nay đã có nghiên cứu gì về trường hợp như thế này chưa nhủ các bạn. Mình có cách gì kiểm tra việc nghe của bé thường xuyên để theo dõi không? Mọi người cho mình xin ý kiến nhé!\n    - Vị trí: âm hộ bên phải, mông bên phải\n    - Mức độ nghiêm trọng: cực kỳ đau đớn\n    - Thời gian: Tình trạng ngày càng nặng trong 5 ngày\n    - Các triệu chứng liên quan: ban đỏ, chảy mủ\n    Các sự kiện trước khi nhập viện\n    - Được bác sĩ chăm sóc chính thay thế khám vào ngày, được tăng liều bactrim và doxycycline để điều trị chẩn đoán Viêm mô tế bào\n    - Báo cáo có cải thiện một phần về ban đỏ\n    - dịch tiết có vẻ như mủ từ một số tổn thương vào ngày, nhưng tình trạng này đã tự khỏi\n    - Khám tại phòng khám vào ngày nhập viện\nDù hiện tại bé nghe khá tốt, nhưng việc chỉ nghe một bên có thể làm con lúng túng khi xác định hướng âm thanh hoặc ở môi trường ồn ào. Để theo dõi tại nhà, bạn hãy thử gọi con từ nhiều góc khuất phía bên phải xem bé có nhanh chóng quay đúng hướng không, và chú ý xem con phát âm có tròn vành rõ chữ không.\n\nBé hai tuổi đã có thể đo thính lực rất chính xác. Bạn nên đưa con đến các bệnh viện lớn có chuyên khoa Tai mũi họng nhi để đánh giá lại toàn diện. Gia đình hãy cứ lạc quan chăm sóc bé như bình thường, lộ trình điều trị phía trước rất rõ ràng và hoàn toàn khả thi.',
    '38.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mạn tính\n    - Bệnh bạch cầu dòng tủy mãn tính đang dùng gleevec\n    - Tăng huyết áp\n    - tăng lipid máu, không đặc hiệu\n    - Đái tháo đường típ 2\n    - hẹp ống sống\n    - Giả gout\n    - bệnh thận mạn, không đặc hiệu Giai đoạn 4\n    - tăng sản tuyến tiền liệt\n    - Nhiều lần ngã gần đây Ngã\n    - ảo giác\n    Thuốc trước khi nhập viện: gleevec (dừng theo chỉ dẫn sau xuất viện)\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện\n    - Toàn trạng suy kiệt  kể từ khi xuất viện\n    - ảo giác dai dẳng\n    - Lú lẫn ngày càng nặng\n    Thời điểm khởi phát triệu chứng: Cực kỳ yếu kể từ khi xuất viện\n    Diễn biến bệnh\n    - ảo giác được vợ nhận thấy\n    - Lú lẫn ngày càng nặng được vợ nhận thấy\n    - yếu cơ dẫn đến ngã hôm nay\n    Triệu chứng hiện tại\n    - toàn trạng suy kiệt\n    - ảo giác\n    - Lú lẫn\n    - Ăn uống kém, ăn vào dễ nôn.\n    - Nôn không ra máu, không ra dịch mật\n    Các sự kiện trước khi nhập viện\n    - Vừa được xuất viện sau  chẩn đoán cho ảo giác bình thường\n    - Được hướng dẫn ngừng gleevec\n    Tình trạng ngay trước khi nhập viện\n    - Toàn trạng suy kiệt\n    - trượt ngã xuống sàn do yếu cơ\n    - Ăn uống kém, ăn vào dễ nôn.\n    - Nôn không ra máu, không ra dịch mật\n\nCâu trả lời của bác sĩ:\n\nChào em,\nPhụ nữ đang cho con bú có thể dùng que cấy tránh thai. Que tránh thai được cấy  trong khoảng 5 ngày đầu tiên của kỳ kinh. Nếu chưa có kinh trở lại sau sanh, em đến khám và khi được xác định không có thai thì sẽ tư vấn việc cấy que và có thể ra về ngay sau đó.\nThân mến    Kết quả xét nghiệm\n    - bạch cầu tăng là 39.2\n    - creatinin là 3.0\n    - troponin là 0.10\n    - inr là 1.4\n    - Tăng men gan. với alt là 176 và ast là 287\n    - tổng phân tích nước tiểu có đái tháo đườngđái tháo đường\n    Kết quả chẩn đoán hình ảnh: chụp x-quang ngực là bình thường\n    Các phát hiện chẩn đoán khác: đánh giá chuyên khoa huyết học - ung bướu: không nghĩ đây là biến đổi cấp tính và có khả năng đại diện cho cml trong bối cảnh ngừng gleevec',
    '39.txt': '1. Tiền sử bệnh\nCác bệnh lý mạn tính\n- Ung thư biểu mô tuyến đại tràng\n- bệnh phổi tắc nghẽn mạn tính, không xác định\n- Đái tháo đường\n- xơ gan do rượu\nTiền sử phẫu thuật / thủ thuật: Dựa trên tiền sử phẫu thuật mới được bổ sung: "Phẫu thuật mở cắt nối trực tràng/đại tràng sigma vì ung thư tuyến đại tràng\nCác yếu tố nguy cơ liên quan: xơ gan do rượu\n\n2. Bệnh sử hiện tại\nLý do nhập viện: tụt huyết áp không rõ nguyên nhân và mệt mỏi\nThời điểm khởi phát triệu chứng: vài ngày mệt mỏi\nDiễn biến bệnh\n- Bệnh nhân nhập viện, quá trình điều trị biến chứng bởi Viêm phổi bệnh viện và rung nhĩ và nhịp nhanh trên thất, sau đó xuất viện về trung tâm phục hồi chức năng.\n- Bệnh nhân tái khám tại phòng khám vào ngày hẹn để cắt bỏ các mũi kẹp còn lại và thay thế vật liệu độn.\n- Hôm nay, bệnh nhân đến khoa Cấp cứu khám vì tình trạng mệt mỏi kéo dài trong vài ngày, kèm theo khó khăn khi ra khỏi giường vào sáng nay.\n- Bệnh nhân đã có đau bụng gián đoạn kể từ khi phẫu thuật.\nTriệu chứng hiện tại\n- hạ huyết áp, không đặc hiệu\n- mệt mỏi\n- mệt mỏi\nTheo như mô tả của Bạn cháu bé bị bệnh bàn chân bẹt, đây là tật bẩm sinh không ảnh hưởng nhiều đến sức khỏe. Hiện tại cháu chạy nhảy bình thường, không đau, chưa có chỉ định phẫu thuật, Tuy nhiên bạn nên cho cháu đến bệnh viện để kiểm tra đánh giá bất thường về xương, khám lâm sàng để đánh giá các thiếu hụt về phần mềm, trên cơ sở đó chúng tôi sẽ có các hướng dẫn cụ thể về tập vận động, tiên lượng về tiến triển của bệnh một cách cụ thể. \n- khó thở khi gắng sức\n- ho mạn tính có đờm vàng loãng\n- ran\n- phù phù\n- dịch thanh dịch lẫn máu từ vết mổ\n- phân nâu dương tính guaiac\nKết quả xét nghiệm: huyết khối 26.3, giảm so với hôm qua, nhưng ổn định ở mức 28 hậu phẫu tuần trước.\nKết quả chẩn đoán hình ảnh: chụp x-quang ngực có xẹp phổi thùy dưới phải do chèn ép kèm tràn dịch màng phổi.\nCác thủ thuật đã thực hiện\n- Truyền dịch tĩnh mạch 750cc\n- Xông khí dung\nCác phát hiện chẩn đoán khác\n- Viêm phổi bệnh viện\n- Rung nhĩ kèm nhịp nhanh trên thất',
    '40.txt': 'Bệnh nhân nữ 75 tuổi  vào viện vì Đi ngoài phân đen, khó thở \nBệnh sử: Cách vào viện 4 ngày \nBN xuất hiện đau bụng quanh rốn, nôn thức ăn lẫn máu kèm đi ngoài phân đen, khó thở liên tục, không rõ sốt => Vào viện tuyến dưới nội soi tiêu hóa, chẩn đoán XHTH quá liều thuốc kháng vitamin K (INR 15), được dùng **********, ********** => Chuyển viện\nTiền sử bệnh: \nBản thân: Van động mạch chủ cơ học điều trị thuốc không đều, thay khớp háng \nKhám lúc vào viện:\nToàn thân: Mạch: 130 lần/phút \nBệnh nhân tỉnh, glasgow 15 điểm \nDa niêm mạc nhợt \nThể trạng nhiễm trùng \nKhông phù, không xuất huyết dưới da \nCác bộ phận: \nTim đều, T1T2 rõ tiếng van cơ học, không tiếng thổi \nRRPN giảm 2 phế trường \nBụng mềm, không chướng \nKhông điểm đau thành bụng \nNhiệt độ: 37°C \nHuyết áp: 130 / 70 mmHg \nNhịp thở : 21 l/p\n\nChẩn đoán : Xuất huyết tiêu hóa theo dõi quá liều kháng vitamin K - Viêm phổi  - Van động mạch chủ cơ \nhọc - Đái tháo đường - TS thay khớp háng\n\nXét nghiệm cần làm : \n1. Xét nghiệm máuKhí máu: Khí máu động mạch (23 thông số).\nHuyết học & Đông máu:\nTổng phân tích tế bào máu ngoại vi (bằng máy đếm laser).\nThời gian Thromboplastin một phần hoạt hóa (aPTT / TCK) bằng máy tự động.Thời gian Prothrombin (PT / TQ / Tỷ lệ Prothrombin) bằng máy tự động.Định lượng Fibrinogen (Yếu tố I) - Phương pháp Clauss trực tiếp bằng máy tự động.Sinh hóa & Men tim:Định lượng Glucose, Urê, Creatinin máu.Điện giải đồ (Na+, K+, cl-).\nMen gan: Đo hoạt độ AST (GOT) và ALT (GPT).\nViêm & Men tim: Định lượng CRP và Troponin T siêu nhạy (hs-Troponin T).\n2. Chẩn đoán hình ảnh\nKhông ghi nhận co giật, cứng đờ, cắn lưỡi hoặc tiểu tiện không tự chủ.\n Sinh hóa & Miễn dịch: CRP: 227.0 mg/L Creatinin : 46 µmol/L Kali +: 3.6 mmol/L\n. Chẩn đoán hình ảnh (Ngày 03/07/2026)\nChụp CT Bụng - Tiểu khung (64–128 dãy, có thuốc cản quang, không in phim):Kết quả: Hình ảnh dày thành một số quai ruột non.2. \nY lệnh Điều trị\nCeftriaxone 1g :Liều dùng: 2 lọ / ngày.Đường dùng: Truyền tĩnh mạch (sáng), tốc độ 30 giọt/phút.\n',
    '41.txt': 'Câu hỏi từ người dùng:\n\nChào bác sĩ\nCháu năm nay 15 tuổi da rất nhiều dầu và bị mụn trứng cá từ năm lớp 7. Đến năm lớp 11 cháu đi khám da liễu, thì sau khoảng 3 tháng điều trị da cháu đã đỡ hẳn. Nhưng sau 4 tháng da cháu bắt đầu lên mụn trở lại gồm mụn ở trán. Mong các bác sĩ cho cháu lời khuyên về vấn đề chăm sóc da dầu mụn và làm thế nào để da hết mụn không tái phát ạ. Cháu cảm ơn các bác sĩ ạ.\n\nCâu trả lời của bác sĩ:\n\n1.Mụn xuất hiện thông qua sự tương tác của 4 yếu tố chính:\nSản xuất quá nhiều chất bã nhờn\nBít tắc nang lông bởi chất bã và tế bào sừng\nVi hệ tại nang lông bởi Propionibacterium acnes (vi khuẩn kỵ khí thông thường)\nGiải phóng các chất trung gian gây viêm\nTrứng cá có thể được phân loại là\n\n- Không viêm: Đặc trưng bởi nhân mụn\n- Viêm: Đặc trưng bởi sẹo, mụn mủ, nốt sần, và nang\n2. Trứng cá bắt nguồn từ bốn yếu tố chính:\n- Tăng sản tuyến bã nhờn\n- Sừng hóa nang lông bất thường\n- Vi khuẩn C. acne\n- Viêm tại chỗ\n3.Điều trị\n- Nhân mụn: Tretinoin tại chỗ\n- Trứng cá viêm nhẹ: ******** tại chỗ đơn độc hoặc phối hợp với kháng sinh tại chỗ, ****************, hoặc cả hai\n    Lý do nhập viện: đau ngực trái cấp tính và đau sau xương ức lan ra sau lưng\n    Thời điểm khởi phát triệu chứng: cấp tính\n- Điều quan trọng là điều trị trứng cá để giảm mức độ bệnh, sẹo và ức chế về tâm lý.\nHầu hết thiếu niên (80%) bị mụn trứng cá, một số trường hợp có thể kéo dài đến tuổi trưởng thành. Bạn điều trị theo chỉ định của bác sỹ chuyên khoa da liễu nhưng không rõ dùng thuốc gì. Nhưng về nguyên tắc khi điều trị thì bệnh sẽ ổn định, ngừng điều trị thì khả năng bệnh sẽ tái phát vì ngoài các yếu tố kể trên, bệnh còn liên quan đến nhiều yếu tố như yếu tố nội tiết, cơ địa, nghề nghiệp, do thói quen chà xát và nặn mụn…\n\nBạn vẫn nên đi khám lại để bác sĩ chuyên khoa tư vấn về thuốc và chăm sóc da mụn. Bạn vẫn có thể điều trị nhiều phương pháp mà không ảnh hưởng đến sức khỏe sinh sản. Chăm sóc da mụn một cách khoa học sẽ làm giảm dầu, giảm nhân mụn và có làn da đẹp.',
    '42.txt': '1.  Tiền sử bệnh lý\n    Các bệnh lý mạn tính\n    - ho đái tháo đường\n    - tăng huyết áp\n    - béo phì\n    - ngưng thở khi ngủ do tắc nghẽn\n    Thuốc trước khi nhập viện\n    - tylenol\n    - mucinex d\n    - thỉnh thoảng tiêu chảy các thuốc của ông ấy\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: ho, thiếu oxy\n    Thời điểm khởi phát triệu chứng: 1 tuần trước khi nhập viện\n    Triệu chứng hiện tại\n\nMột tuần nay em bị đau bụng vùng thượng vị, ợ hơi, ợ chua. Em đã đi nội soi dạ dày thì được biết là hang vị, tiền môn vị, niêm mạc phù nề, sung huyết đỏ. Xin hỏi bác sĩ bệnh có nguy hiểm không? Cần làm gì để điều trị dứt điểm. Em dùng nghệ mật ong có tốt không?\n    - mệt mỏi\n    - sốt lên đến 101\n    - thỉnh thoảng tiêu chảy các thuốc của ông ấy\n    - đi tiểu nhiều lần vào ban đêm, không có đau buốt khi đi tiểu\n    Đặc điểm triệu chứng\n    - khó thở khi gắng sức bao gồm cảm giác khó thở sau khi đi bộ vài khối\n    - ho đặc như bã theo lời bệnh nhân nhưng ông ấy không khạc nhổ gì\n    - sốt lần cuối sốt là vào ngày\n    - tiêu chảy đã tự khỏi vào buổi sáng hôm sau\n    - Phủ nhận đau ngực\n    - Phủ nhận buồn nôn/nôn\n    - Phủ nhận đổ mồ hôi đêm\n    - Phủ nhận khó thở khi nằm\n    - Phủ nhận khó thở khi nằm đột ngột (paroxysmal nocturnal dyspnea)\n    Sự kiện trước khi nhập viện\n    - Vợ có các triệu chứng tương tự 3 tuần trước, được chẩn đoán là giãn phế quản, phản ứng tốt với azithromycin\n    - Tự điều trị bằng tylenol và mucinex d\n    - Đi khám bác sĩ chăm sóc chính (PCP) vào buổi sáng ngày nhập viện\n    - Bác sĩ PCP ghi nhận spo2 là 90-92% spo2 khi thở khí trời, 93% khi không dùng oxy\n    - Bác sĩ PCP đã gửi bệnh nhân  đến phòng cấp cứu\n\n3.  Đánh giá tại bệnh viện\n    Kết quả xét nghiệm\n    - số lượng bạch cầu là 12.5\n    - tổng phân tích nước tiểu đáng chú ý chỉ có vếtvết protein niệu.  \n    - troponin âm tính x1\n    Kết quả chẩn đoán hình ảnh: chụp x-quang ngực được cho là viêm phổi thùy dưới phải (RLL PNA)',
    '43.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mạn tính\n    - Bệnh bạch cầu dòng tủy mãn tính\n    - tăng cholesterol máu đơn thuần\n    - tăng huyết áp\n    Thuốc trước khi nhập viện lần này\n    - gleevec (ngừng uống cách nhập viện 5 ngày)\n    - allopurinol (ngừng uống \ncách nhập viện 5 ngày)\n    - Thuốc giảm đau (ngừng)\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: ảo giác\n    Thời điểm khởi phát triệu chứng: Khoảng một tháng trước\n    Diễn biến\n    - ảo giácxuất hiện khoảng một tháng trước.\n    - Đã được điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định.\n    - Thuốc giảm đau đã ngừng.\n    - Không có thay đổi đáng kể về triệu chứng mặc dù đã điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định và ngừng thuốc giảm đau.\n    - Liều gleevec đã ngừng cách nhập viện 5 ngày.\n    - allopurinol đã ngừng cách nhập viện 5 ngày.\n    - ảo giác xảy ra khoảng mỗi đêm lúc 2 giờ sáng.\n    Triệu chứng hiện tại: ảo giác\n    Đặc điểm triệu chứng\n Bệnh nhân nhìn/nghe thấy có nhiều người (đôi khi tổng cộng khoảng 8 người).\nThấy những người này đang nói chuyện với mình.\nẢo giác không mang tính chất ra lệnh \nBệnh nhân thường hoang tưởng/lo sợ rằng họ sẽ chạm vào da của mình.\n    Diễn biên  trước khi nhập viện\nChào bạn, không biết là tình trạng đi cầu phân sống của bé đã diễn ra lâu chưa? Bé đã uống sữa công thức được bao lâu rồi? Nếu từ khi chuyển sang bú sữa công thức, bé có dấu hiệu đi cầu phân sống, có thể bé không hấp thu được một số protein trong loại sữa này. Vì vậy, bạn có thể gặp bác sĩ dinh dưỡng để đổi sang một loại sữa công thức khác phù hợp hơn cho bé. Về men tiêu hóa hỗ trợ ở độ tuổi của bé có thể dùng men Bacillus clausii 2 tỷ bào tử/5ml/ ống uống 1-2 ống/ ngày (biệt dược là Enterogermina, Biogermin).    - Liều gleevec đã ngừng cách nhập viện 5 ngày\n    - allopurinol đã ngừng cách nhập viện 5 ngày\n    Tình trạng ngay trước khi nhập viện: ảo giác khoảng mỗi đêm lúc 2 giờ sáng\n3.  Đánh giá tại bệnh viện',
    '44.txt': '- Tại bệnh viện OSH, phát hiện tăng men gan với tăng bilirubin máu: ast 421, alt 336, alp 185, bilirubin toàn phần 0.9\n- âm tính siêu âm bụng có doppler\n- âm tính chụp hida\n- Chưa từng xét nghiệm nồng độ acetaminophen\n- Chưa từng dùng nac\n- Chuyển tạm thời để thực hiện ercp\n- ercp cho thấy túi mật giãn nở rõ rệt nhưng không có sỏi hoặc bệnh lý khác\n- Chuyển lại bệnh viện OSH và được cho là đã tống ra một viên sỏi mật\nTình trạng ngay trước khi nhập viện: tăng bilirubin máu và tăng men gan kéo dài\nCâu trả lời của bác sĩ: Chào bạn,mình xin trả lời câu hỏi của bạn như sau Như bạn đã mô tả, trường hợp này bạn thuộc MÀY đay VÔ CĂN. Chúng tôi không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY đay MẠN. Tổ thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với bạch huyết T và đại thực bào. Các loại tiền thưởng khó quan sát trên kính hiển thị vi quang học thông thường cần xem trên kính hiển thị vi điện tử. Các tế bào cột sống tăng lượng hạ bì với các mức độ thoát ra khác nhau cũng được quan sát. Quảng cáo quảng cáo thiết bị thương mại miễn phí dịch sinh học không tìm thấy hình ảnh giảm phức tạp, bổ sung hay sợi fibrin. Đối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Bạn đay vô căn hộ là một bệnh mạn tính, việc điều trị phải dài và liên tục. Quá giá trị chỉ là giấy chứng nhận. Điều trị phụ thuộc vào mức độ nghiêm trọng và nền tảng có liên kết. Cần lưu ý cân bằng giữa hiệu quả kiểm soát triệu chứng và những tác hại gây độc của liệu pháp điều trị. *************** có thể hệ sau không gây buồn ngủ là đơn hàng đầu sau đó đến *************** có thể hệ 1, ********* bậc thấp cách ngày hoặc hàng ngày và giảm dần chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không được hỗ trợ hoặc chống chỉ định, xin bạn hãy đến gặp bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất. Cảm ơn bạn đã gửi câu hỏi',
    '45.txt': '1.  Tiền sử bệnh\n    Các tập tương tự trước đây\n    - Nhập viện khoa Thần kinh từ [Date] với thông não so với chụp cắt lớp vi tính (ct) đầu từ [Date].\n    - Bị tăng đau đầu cải thiện khi nằm ngửa.\n    - Nhập viện khoa Thần kinh vào thời điểm đó để điều trị chứng đau đầu.\n    - Được bác sĩ nhãn khoa khám trong quá trình nhập viện đó và không phát hiện phù gai thị.\n    Bệnh lý mãn tính: não úng thuỷ khác từ thời kỳ sơ sinh\n    Tiền sử phẫu thuật / thủ thuật\n    - Phẫu thuật đặt cảng để điều trị chứng tăng nhãn áp ở thời kỳ sơ sinh\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế khi 3 tháng tuổi\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế lại khi [Số] tuổi\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: đau đầu kéo dài kèm theo mờ mắt\n    Thời điểm khởi phát triệu chứng: nhìn mờ tiến triển trong 2 tuần qua\n    Diễn biến bệnh\n    - Bệnh nhân bị não úng tuỷ từ thời kỳ sơ sinh, được điều trị bằng cách đặt dẫn lưu shunt\n    - Trước đây đã từng nhập viện với chẩn đoán não úng thủy, có biểu hiện tăng đau đầu , triệu chứng cải thiện khi nằm ngửa.\n    - Nhập viện trước đây vào khoa Thần kinh để điều trị chứng đau đầu.\n    - Khám nhãn khoa trong quá trình nhập viện trước đó cho thấy không có phù gai thị.\n    - Đến hôm nay, bệnh nhân bị mờ mắt ở cả hai mắt, bên trái nặng hơn bên phải, tiến triển trong 2 tuần qua.\nXét nghiệm cần làm:\nTổng phân tích tế bào máu ngoại vi (bằng máy \nđếm laser); Thời gian prothrombin (PT: \nProthrombin Time), (Các tên khác: TQ; Tỷ lệ \nProthrombin) bằng máy tự động,; Định lượng \nTroponin Ths ;Định lượng NT - proBNP ( ProBNP); \nĐo hoạt độ AST (GOT); Đo hoạt độ ALT (GPT); \nĐịnh lượng Creatinin (máu); Điện giải đồ (Na, K, \nCl\nChẩn đoán :  Dị tật còn ống động mạch\nĐiều trị : \n***********************\n************************\n*********\n    - Tăng nhãn áp so với chụp cắt lớp vi tính (ct) đầu từ [Date] (từ lần nhập viện trước)\n    - Phù gai thị (theo báo cáo của bác sĩ nhãn khoa)\n',
    '46.txt': 'Bệnh nhân nữ 81 tuổi, vào viện vì \nkhó thở, tiền sử: stent mạch vành - suy tim \n- tăng huyết áp - HoHL  điêu trị ngoại trú \nkhông rõ đơn \nBệnh sử: \nCách vào viện 1 tuần bệnh nhân xuất hiện \nkhó thở tăng lên, NYHA III-IV, mệt mỏi \nnhiều, không sốt, không đau bụng. Bệnh\nnhân đi khám tuyến dưới chẩn đoán phù phổi cấp - suy tim - \nstent mạch vành - tăng huyết áp - HoHL, \nhiện nay bệnh nhân còn khó thở vừa, bệnh \ncải thiện chậm-> vào viện\nKhám: \nTỉnh, Glasgow: 15 điểm \nDa niêm mạc hồng \nKhông phù \nKhông sốt \nTim đều TTT ở mỏm 3/6 \nM: 82 ck/ph; HA: 160/ 80 mmHg \nPhổi RRPN rõ; rale ẩm \nBụng mềm không chướng \n Cận lâm sàng:\nKhí máu: Lactat: 0.8    HCO3: 32.09    \nPO2: 97.0    PCO2: 39    pH: 7.517   \n Ghi điện tim cấp cứu tại giường: nhịp  xoang ts 74, không biến đổi ST-T \nDiễn biến CLS:  Định lượng Lactat (Acid \nLactic): 0.8    HCO3: 32.09    PO2: 97.0    \nPCO2: 39    pH: 7.517   \nSiêu âm doper tim : \n\nPhình thành sau và thành bên thất trái.\nGiảm vận động 2/3 thành bên và 2/3 thành sau dưới thất trái về phía đáy.\nThành thất trái dày, buồng thất trái không giãn, chức năng tâm thu thất trái bảo tồn\n(EF BP: 55%)\nHở hai lá nhiều.\nHở chủ vừa.\nTăng áp lực động mạch phổi nhẹ\n\nChẩn đoán: Phù phổi cấp - Suy tim - Stent mạch vành - Tăng huyết áp\n\nXử trí : \n\n    - Phẫu thuật cắt bỏ u dây thần kinh số VIII \n    - Đặt shunt động tĩnh mạch (AVF) ở tay phải (RUE AVF)\n    - Sinh thiết tuyến tiền liệt\n2.  Bệnh sử hiện tại\n    Lý do nhập viện:  mệt mỏi, mất trí nhớ chi tiết\nBệnh nhân kể cảm thấy khó chịu mệt mỏi nhiều, ăn không ngon miệng, ngứa da toàn thân nhiều, và mất trí nhớ chi tiết, khó thở khi gắng sức. Tuần qua có buồn nôn, và nôn .\nKhám hiện tại thấy\nBệnh nhân tỉnhi\n    - Mệt mỏi nhiều\n    - Ăn không ngon miệng\n    - Ngứa da toàn thân nhiều\n    - mất trí nhớ chi tiết\n    - khó thở khi gắng sức \n    - buồn nôn và nôn \nCận lâm sàng\n ********************* x 1,0 Viên\nNgày uống 1 viên buổi tối ',
    '47.txt': '1. Tiền sử bệnh\nCác bệnh lý mạn tính\n- Ung thư biểu mô tuyến đại tràng\n- bệnh phổi tắc nghẽn mạn tính, không xác định\n- Đái tháo đường\n- xơ gan do rượu\nTiền sử phẫu thuật / thủ thuật: Dựa trên tiền sử phẫu thuật mới được bổ sung: "Phẫu thuật mở cắt nối trực tràng/đại tràng sigma vì ung thư tuyến đại tràng\nCác yếu tố nguy cơ liên quan: xơ gan do rượu\n\n2. Bệnh sử hiện tại\nLý do nhập viện: tụt huyết áp không rõ nguyên nhân và mệt mỏi\nThời điểm khởi phát triệu chứng: vài ngày mệt mỏi\nDiễn biến bệnh\n- Bệnh nhân nhập viện, quá trình điều trị biến chứng bởi Viêm phổi bệnh viện và rung nhĩ và nhịp nhanh trên thất, sau đó xuất viện về trung tâm phục hồi chức năng.\n- Bệnh nhân tái khám tại phòng khám vào ngày hẹn để cắt bỏ các mũi kẹp còn lại và thay thế vật liệu độn.\n\nCâu trả lời của bác sĩ:\n\nChào em\n- đi tiêu không liên quan đến đau bụng, buồn nôn, nôn, tiêu chảy, hoặc táo bón.\n- Bệnh nhân đang có đi tiêu mỗi ngày.\n- Không có sốt, đau ngực, chóng mặt, đánh trống ngực.\n- Không có đau bụng hoặc dịch từ vết mổ.\n- Không có đỏ da, hoặc bục chỉ khâu bụng.\n3. Đánh giá tại bệnh viện\nKết quả khám thực thể\n- Khám thấy ran nổ, phù phù, dịch rỉ huyết thanh từ vết mổ, và phân nâu dương tính guaiac.\n- dấu hiệu sinh tồn là: 98.8 65 9360 20 95 2LNC (tại khoa Cấp cứu)\n- dấu hiệu sinh tồn 98.6 70 10356 17 962LNC (khi đến MICU)\nDấu hiệu lâm sàng\n- hạ huyết áp, không đặc hiệu\n- mệt mỏi\n- mệt mỏi\n- khó thở khi gắng sức\n- ho mạn tính có đờm vàng loãng\n- ran\n- phù phù\n- dịch thanh dịch lẫn máu từ vết mổ\n- phân nâu dương tính guaiac\nKết quả xét nghiệm: huyết khối 26.3, giảm so với hôm qua, nhưng ổn định ở mức 28 hậu phẫu tuần trước.\nKết quả chẩn đoán hình ảnh: chụp x-quang ngực có xẹp phổi thùy dưới phải do chèn ép kèm tràn dịch màng phổi.\nCác thủ thuật đã thực hiện\n- Truyền dịch tĩnh mạch 750cc\n- Xông khí dung\nCác phát hiện chẩn đoán khác\n- Viêm phổi bệnh viện\n- Rung nhĩ kèm nhịp nhanh trên thất',
    '48.txt': 'Hỏi : Bé nhà mình là bé trai, khi bé đc 20 ngày tuổi mình mới phát hiện tai phải bé không có lỗ tai, vẫn có vành tai đầy đủ, mình có đưa bé đi khám ở 1 vài nơi nhưng vì bé còn quá nhỏ nên bs chỉ nhìn bên ngoài và đo âm tai còn lại. Bé nay đã 2 tuổi, nghe nói bình thường, mọi sinh hoạt bình thường, phát triển đều, nhưng mình cảm thấy thật sự lo lắng vì bên tai ko có lỗ kia vành tai không phát triển đều, nó nhỏ hơn tai còn lại. Ngược lại, bên tai trái thì rất to và vành tai đẹp. Mình cũng để ý con xem có phản ứng khi mình gọi không và bé vẫn nghe bìng thường, kể cả khi sắp ngủ mình nói nhỏ bé vẫn nghe và mở mắt nhìn. Mình rất lo lắng về tai của bé, sợ sau này sẽ ảnh hưởng đến sự phát triển của bé và bé bị trêu chọc. Y học hiện nay đã có nghiên cứu gì về trường hợp như thế này chưa nhủ các bạn. Mình có cách gì kiểm tra việc nghe của bé thường xuyên để theo dõi không? Mọi người cho mình xin ý kiến nhé!\nTrả lời :\nChào bạn, trước hết xin được chia sẻ về sự lo lắng đôi chút về tình trạng của bé hiện này, về tình trạng bé mắc phải là dị tật thiểu sản vành tai và tịt ống tai ngoài bẩm sinh. Rất may mắn, cấu trúc tai trong đảm nhận chức năng thần kinh thính giác của con thường vẫn phát triển tốt. Việc bé phản xạ được âm thanh là nhờ sự bù trừ tuyệt vời từ chiếc tai trái khỏe mạnh.\n\n3. Đánh giá tại bệnh việnDù hiện tại bé nghe khá tốt, nhưng việc chỉ nghe một bên có thể làm con lúng túng khi xác định hướng âm thanh hoặc ở môi trường ồn ào. Để theo dõi tại nhà, bạn hãy thử gọi con từ nhiều góc khuất phía bên phải xem bé có nhanh chóng quay đúng hướng không, và chú ý xem con phát âm có tròn vành rõ chữ không.\n\nBé hai tuổi đã có thể đo thính lực rất chính xác. Bạn nên đưa con đến các bệnh viện lớn có chuyên khoa Tai mũi họng nhi để đánh giá lại toàn diện. Gia đình hãy cứ lạc quan chăm sóc bé như bình thường, lộ trình điều trị phía trước rất rõ ràng và hoàn toàn khả thi.',
    '49.txt': 'Câu hỏi từ người dùng:\n\nEm bị bệnh rụng tóc từng mảng. Mong bác cho cách điều trị được không ạ? Em bị 05 năm rùi điều trị nhiều thuốc và bệnh viện rùi mà đến nay vẫn không hết. Cảm ơn Bác sỹ ạ!\n\nUng thư vú di căn, tràn dịch màng phổi trái tái phát\n\n2. Bệnh sử hiện tại\nLý do vào việni: khó thở\n3. Khám tại bệnh viện\nLâm sàng: tràn dịch màng ngoài tim mức độ trung bình\nKết quả chẩn đoán hình ảnh: ung thư di căn theo đường bạch huyết ở hai phổi\nCác thủ thuật đã thực hiện\n- dẫn lưu dịch màng tim\nMình xin tư vấn một số phương pháp điều trị dưới đây\n1.        Liệu pháp **************\nTiêm ************** nội thương tổn được chỉ định trong rụng tóc từng vùng khi diện tích thương tổn dưới 50% diện tích da đầu. Tiêm nhắc lại sau mỗi 4-6 tuần. Corticosteroid bôi tại chỗ thường được chỉ định cho trẻ em. Các thuốc hay dùng là kem *************************** hai lần một ngày hoặc ************************************ (corticoid mức độ mạnh).\nĐối với rụng tóc toàn bộ hoặc rụng tóc toàn thể khó điều trị có thể sử dụng 2,5g ********************e có bao phim che phủ 6 ngày/tuần trong 6 tháng.\nThời gian điều trị ít nhất là ba tháng trước khi tóc mọc lại, duy trì điều trị thường xuyên nếu cần thiết.\n2.        Liệu phát miễn dịch tiếp xúc (contac immunotherapy)\nCác chất gây mẫn cảm như diphenylcyclopropenone (diphencyprone) và dinitrochlorobenzene được quét lên vùng rụng tóc, chúng sẽ gây viêm da tiếp xúc dị ứng ở vùng điều trị, có thể sử dụng 01 lần/tuần trên vùng rụng tóc trong 06 tháng. Các tác dụng phụ bao gồm viêm da tiếp xúc, mày đay, bất thường sắc tố (gồm cả bạch biến).\n3.        PUVA (psoralen kết hợp UVA)\nPUVA toàn thân và tại chỗ đều được sử dụng. Cần 20-40 lần điều trị nhiều bệnh nhân tái phát sau khi ngừng điều trị.\nBạn nên đến gặp các bác sĩ da liễu để đươc tư vấn trực tiếp và lựa chọn phương pháp điều trị phù hợp. \nChúc bạn sức khoẻ\n',
    '50.txt': '1.  Tiền sử bệnh\n    Các tập tương tự trước đây\n    - Nhập viện khoa Thần kinh từ [Date] với thông não so với chụp cắt lớp vi tính (ct) đầu từ [Date].\n    - Bị tăng đau đầu cải thiện khi nằm ngửa.\n    - Nhập viện khoa Thần kinh vào thời điểm đó để điều trị chứng đau đầu.\n    - Được bác sĩ nhãn khoa khám trong quá trình nhập viện đó và không phát hiện phù gai thị.\n    Bệnh lý mãn tính: não úng thuỷ khác từ thời kỳ sơ sinh\n    Tiền sử phẫu thuật / thủ thuật\n    - Phẫu thuật đặt cảng để điều trị chứng tăng nhãn áp ở thời kỳ sơ sinh\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế khi 3 tháng tuổi\n- Hệ thống dẫn lưu được chỉnh sửa/thay thế lại khi [Số] tuổi\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: đau đầu kéo dài kèm theo mờ mắt\n    Thời điểm khởi phát triệu chứng: nhìn mờ tiến triển trong 2 tuần qua\n    Diễn biến bệnh\n    - Bệnh nhân bị não úng tuỷ từ thời kỳ sơ sinh, được điều trị bằng cách đặt dẫn lưu shunt\n    - Trước đây đã từng nhập viện với chẩn đoán não úng thủy, có biểu hiện tăng đau đầu , triệu chứng cải thiện khi nằm ngửa.\n    - Nhập viện trước đây vào khoa Thần kinh để điều trị chứng đau đầu.\n    - Khám nhãn khoa trong quá trình nhập viện trước đó cho thấy không có phù gai thị.\n    - Đến hôm nay, bệnh nhân bị mờ mắt ở cả hai mắt, bên trái nặng hơn bên phải, tiến triển trong 2 tuần qua.\nChẩn đoán phân biệt: \nCT ngực là phương pháp hữu ích, nhạy cảm để giúp chẩn đoán xác định VPHT, đánh giá độ lan rộng của bệnh, và còn giúp phát hiện bất thường bẩm sinh kèm theo hay dị vật đường thở bỏ quên nếu có. Hiện nay, hầu hết các nghiên cứu đều chẩn đoán xác định VPHT bằng CT ngực. Các hình ảnh chính trong VPHT như sau:\n- Có vùng đông đặc phổi nhưng không giảm thể tích.\n- Phá hủy cấu trúc nhu mô phổi bình thường với sự hiện diện của vùng giảm bắt thuốc cản quang. \n    - Phù gai thị (theo báo cáo của bác sĩ nhãn khoa)\n',
    '51.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mãn tính\n    - béo phì\n    - đái tháo đường được kiểm soát bằng chế độ ăn\n    - ngừng thở khi ngủ\n    - tiền sử lâm sàng suy tim, không đặc hiệu\n    Dị ứng: Dị ứng furosemide\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: khó thở\n    Thời điểm khởi phát triệu chứng\n    - tiền sử trước vào viện  ba ngày\n    - bắt đầu nhận thấy khó thở khi xem TV lúc 11:00 sáng ba ngày trước\n    Triệu chứng hiện tại\n    - khó thở\n    - khó nằm vào ban đêm để ngủ vì thở\n    - căng cứng vai trái (ngắt quãng, xảy ra vài lần mỗi ngày, kéo dài 5 phút)\n    - tiền sử ho có đờm trắng\n    - chủ quan sốt\n    - phù chi dưới (ổn định)\n    Đặc điểm triệu chứng\n    - Vị trí: căng cứng vai trái\n    - Thời gian: kéo dài 5 phút (căng cứng vai trái)\n    - Tần suất: xảy ra vài lần mỗi ngày (căng cứng vai trái)\n    - Triệu chứng liên quan: khó nằm vào ban đêm để ngủ vì khó thở, ho có đờm trắng, chủ quan sốt, phù chi dưới\n    Các sự kiện trước khi nhập viện\n    - bỏ lỡ 3 ngày dùng một số thuốc vì quên đổ thuốc vào hộp (tuần qua)\n    - gọi EMS\n    - EMS phát hiện bệnh nhân thiếu oxy với độ bão hòa oxy là 86 khi thở khí trời\n    - EMS cho bệnh nhân thở oxy qua mask không hồi phục (NRB)\n    - EMS đưa bệnh nhân đến khoa Cấp cứu\nTrả lời:\nChào bạn! Nếu các triệu chứng đúng như bạn mô tả, có thể các bác sỹ sẽ xem xét mổ lại (tái tạo lại) dây chằng chéo trước và xử trí tổn thương sụn chêm cho bạn. Chúng tôi đã từng gặp và xử trí nhiều trường hợp như của bạn, tuy nhiên để đề ra phương án điều trị chi tiết chúng tôi cần khám và hội chẩn trên bệnh nhân cụ thể, vì vậy mời bạn đến Bệnh viện Trung ương Quân đội 108 để khám bệnh và được tư vấn cụ thể hơn nhé. Cảm ơn bạn đã gửi câu hỏi.    - được cho aspirin 325mg\n    - được cho albuterolipratropium nebulizer\n    - được cho methylprednisolone 125mg iv\n    - lợi tiểu 600cc',
    '52.txt': 'Câu hỏi từ người dùng :\n\nDạ em có một người bạn, hiện tại bạn ấy đang bị nổi các dấu mề đay rất to và ngứa khắp người, bạn em bị nổi quanh năm luôn ạ. Những dấu mề đay nổi rất khó chịu và ngứa, được biết bạn em có từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn ấy cũng bị nên em nghỉ chắc là di truyền. Cả hai người đều không phải dị ứng do thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bị nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến và có cách nào để chữa cho bạn không ạ. Em cảm ơn rất nhiều\n\n\nCâu trả lời của bác sĩ: \n\nChào bạn,mình xin trả lời câu hỏi của bạn như sau\nNhư bạn đã mô tả, trường hợp này bạn  thuộc MÀY ĐAY VÔ CĂN. Chúng ta không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY ĐAY MẠN TÍNH.\nTổn thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với lympho T và đại thực bào. Các tổn thương khó quan sát trên kính hiển vi quang học thông thường mà cần xem trên kính hiển vi điện tử. Các tế bào mast tăng số lượng vùng hạ bì với các mức độ thoát bọng khác nhau được quan sát thấy. Nhuộm huỳnh quang miễn dịch tổn thương sinh thiết không thấy có hình ảnh của lắng đọng các phức hợp miễn dịch, bổ thể hay sợi fibrin.\nĐối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Mày đay vô căn là một bệnh mạn tính, việc điều trị phải lâu dài và liên tục. Quá trình điều trị chỉ là điều trị triệu chứng.\n1.  Tiền sử bệnh\n    - 2 lần tự tửdo quá liều klonopinclonidine điều trị đặt nội khí quản tại khoa Hồi sức tích cực\n  - Mẹ Đã tử vong trong lần nhập viện gần đây nhất\nCảm ơn bạn đã gửi câu hỏi',
    '53.txt': '1. Bệnh nhân nữ 62 tuổi, vào viện với lý do: mệt mỏi, gầy sút cân., quá trình bệnh biểu hiện với các hội chứng và triệu chứng sau: \n        Ăn nhiều, khát nước, uống nhiều, đi tiểu nhiều, gầy sút cân (3kg/tháng), người mệt mỏi. \n        Lúc vào viện trong tình trạng: \n       Mạch: 89 lần/phút, HA: 180/100 mmHg \n        Xét nghiệm máu: Glucose máu: 13,2 mmol/l \n      XQ: Quai ĐMC vồng cao, chỉ số tim/LN < ½, rốn phổi đậm.\nBệnh nhân đã dùng thuốc điều trị ĐTD typ II (không rõ thuốc gì), các triệu chứng có giảm, nhưng không duy trì thuốc.\nHiện tại: \n\nTorsemide: uống 1 viên/ngày, đôi khi 2 viên/ngày.\nInsulin glargine: theo đơn 100 đơn vị x 2 lần/ngày, hiện đã ngừng sử dụng.\nIsosorbide: đã hết thuốc khoảng 3 tuần trước nhập viện.\nRosuvastatin (Crestor): đã hết thuốc khoảng 3 tuần trước nhập viện.\nCarvedilol: đã hết thuốc khoảng 3 tuần trước nhập viện.\n2. Bệnh sử hiện tại\nLý do nhập viện: xét nghiệm bất thường - tăng kali máu và Creatinine tăng\nCác sự kiện trước khi nhập viện\n- Đi khám bác sĩ chăm sóc chính (PCP) để tái khám hàng năm\n- Báo cáo đang dùng 1 viên Torsemide/ ngày , đôi khi 2 viên\n- Bác sĩ đề nghị ngừng sử dụng torsemide lâu dài\n- Bệnh nhân không kiểm tra đường huyết thường xuyên tại nhà do hết que thử đường máu.\n- glucose là thấp, nên ngừng dùng glargine\n- Báo cáo bị ngã gần đây\n- Dùng xe đẩy hàng làm dụng cụ hỗ trợ đi lại\n       Xét nghiệm lại CTM, SHM, chụp XQ phổi\n       Kiểm soát đường máu, điều chỉnh rối loạn lipid máu, điều chỉnh HA, phòng ngừa biến chứng\n5. Đơn thuốc cho 1 ngày.\n1.        ***************** x 2 viên, uống sáng trước ăn \n2.        ***************** x 2 viên, uống chiều sau ăn \n3.        ************ x 1 viên, uống chiều. \n4.        ********* x 2 viên, uống sáng 1 viên, chiều 1 viên \n5.        ********* x 2 viên, uống sáng 1 viên, chiều 1 viên.',
    '54.txt': 'Hỏi : Xin chào ad.\nEm muốn hỏi 1 vấn đề được không ah?\nEm mang thai được hơn 6 tuần rồi. Nhưng trước đó khoảng 2 tuần e có uống thuốc ************ này. Liệu có bị ảnh hưởng gì đến thai nhi không ạ?E uống hết khoảng 4 ngày và ngày 2 viên?\n\nVà e có nên làm xét nghiệm gì để sàng lọc không ah?\n    - Bệnh bạch cầu dòng tủy mãn tính đang dùng gleevec\n    - Tăng huyết áp\n    - tăng lipid máu, không đặc hiệu\n    - Đái tháo đường típ 2\n    - hẹp ống sống\n    - Giả gout\n    - bệnh thận mạn, không đặc hiệu Giai đoạn 4\n    - tăng sản tuyến tiền liệt\n    - Nhiều lần ngã gần đây Ngã\n    - ảo giác\n    Thuốc trước khi nhập viện: gleevec (dừng theo chỉ dẫn sau xuất viện)\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện\n    - Toàn trạng suy kiệt  kể từ khi xuất viện\n    - ảo giác dai dẳng\n    - Lú lẫn ngày càng nặng\n    Thời điểm khởi phát triệu chứng: Cực kỳ yếu kể từ khi xuất viện\n    Diễn biến bệnh\n    - ảo giác được vợ nhận thấy\n    - Lú lẫn ngày càng nặng được vợ nhận thấy\n    - yếu cơ dẫn đến ngã hôm nay\n    Triệu chứng hiện tại\n    - toàn trạng suy kiệt\n    - ảo giác\n    - Lú lẫn\nNhư bạn cũng biết đấy, bất kỳ một thai kỳ nào ngay cả khi mẹ không sử dụng thuốc gì thì bé vẫn có thể có các nguy cơ về dị tật bẩm sinh bởi tác động từ nhiều yếu tố khác nữa. Không có một xét nghiệm nài có thể sàng lọc được tất cả các bất thường mẹ nha.\nTrước tiên thì khi thai khoảng 9 10 tuần, e có thể làm nipt, hoặc tầm 12 tuần e có thể làm double test. Tuy nhiên, các xét nghiệm này đánh giá nguy cơ bất thường về nhiễm sắc thể, gen. Còn em muốn đánh giá về hình thái bé phải qua siêu âm, theo hẹn của bác sỹ, đặc biệt là các mốc thai 12 tuần, 22 tuần, 32 tuần. Trường hợp của em thì cũng không lo lắng quá na. Em cứ đi khám thai theo hẹn và có vấn đề bất thường gì thì tái khám luôn nha.\nChúc em sức khỏe',
    '55.txt': 'Câu hỏi từ người dùng :\n\nem năm nay 22 tuổi, đợt vừa rồi có đi khám sức khỏe có đo HA là 160/70 mmHg, sau đó đo lại là HA: 170/60 mmHg, đo lại lần nữa là HA:  150/60 mmHg, nhưng lúc đó em cảm thấy hồi hợp, cảm giác rợn người, em có mua máy đo HA điện tử về nhà, tự đo thì HA bình thường, em đo rất nhiều lần, em học y nên kĩ thuật có thể nói là chính xác, nhưng cứ có người đo cho em thì nó HA nhịp tim đều tăng và có cùng một cảm giác rợn người, hồi hộp, cảm nhận được tim đập rất mạnh, em có đi khám ở ĐHYD làm rất nhiều XN nhưng tất cả đều bình thường và không tìm ra nguyên nhân, sau đó bs có hẹn 1 tuần tái khám nhưng do dịch nên không đi được, em muốn hỏi, tình trạng của em có phải là tăng HA thật sự không ạ? Em cảm thấy rất hoang mang, bởi vì khi tự đo, ngồi một mình, thì cảm giác đó không có và huyết áp của em mỗi lần đo đều bình thường. Em cảm ơn ạ\n1.  Tiền sử bệnh\n    Các bệnh lý mạn tính\n    - Suy thận mạn giai đoạn V do đái tháo đường và tăng huyết áp\n    - u ác của tuyến tiền liệt điều trị bằng xạ trị cấy hạt \n    Tiền sử phẫu thuật / thủ thuật\n    - Phẫu thuật cắt bỏ u dây thần kinh số VIII \n    - Đặt shunt động tĩnh mạch (AVF) ở tay phải (RUE AVF)\n    - Sinh thiết tuyến tiền liệt\n2.  Bệnh sử hiện tại\n    Lý do nhập viện:  mệt mỏi, mất trí nhớ chi tiết\nChào bạn! Huyết áp có thể tạm thời thay đổi trong một số hoàn cảnh như:\n\n- Khi ở trong tâm trạng lo âu căng thẳng thì huyết áp tăng lên đáng kể và sẽ trở lại bình thường sau khi thoải mái thư giãn. Vì thế khi đi khám bệnh huyết áp thường hơi cao hơn khi đo ở nhà do đó nên nghỉ vài phút trước khi đo.\n-Nói chuyện khi đo, huyết áp cũng lên cao. Vì thế nên giữ im lặng trong khi đo.\nĐể chắc chắn và yên tâm hơn bạn nên đi kiểm tra lại, nghỉ ngơi vài phút trước khi đo giữ tâm trạng thoải mái để có kết quả chính xác nhất.\n',
    '56.txt': 'Câu hỏi từ người dùng :\n\nCận lâm sàng:\nKết quả chẩn đoán hình ảnh\n- Đo sức bền cơ tim bằng đồng vị phóng xạ kết quả có dị tật cố định vùng trước vách và vách dưới với  giảm chức năng tâm thu thất trái nghiêm trọng kèm theo rối loạn chức năng thất trái.\n- Chụp động mạch vành cho thấy bệnh ba thân động mạch vành nghiêm trọng \nCác thủ thuật đã thực hiện\n- Chụp động mạch vành can thiệp\nCâu trả lời của bác sĩ: \n\nViêm hang vị sung huyết là tình trạng niêm mạc vùng hang vị dạ dày viêm, các mạch máu vùng viêm giãn nở do ứ máu nhiều. Biểu hiện chủ yếu là đau bụng cồn cào kèm theo ợ hơi, ợ chua, có thể có cảm giác buồn nôn hoặc nôn. Trước kia bệnh dạ dày được coi là bệnh nan y và nguy hiểm nhưng mấy chục năm gần đây nhờ nội soi phát triển nên bệnh đã được điều trị hiệu quả. Để điều trị dứt điểm bệnh viêm sung huyết hang vị dạ dày, trước hết, người bệnh cần được thăm khám chuyên khoa, nội soi dạ dày để tìm nguyên nhân, đánh giá mức độ bệnh. Căn cứ trên kết quả khám, các bác sĩ sẽ đưa ra phác đồ điều trị hiệu quả nhất cho từng bệnh nhân. Điều quan trọng là bệnh nhân cần tuân thủ tuyệt đối sự chỉ dẫn của bác sĩ trong quá trình điều trị. Không được bỏ thuốc giữa chừng cũng như không được tự ý tăng giảm liều lượng thuốc mà chưa có sự đồng ý của bác sĩ. Ngoài ra, người bị viêm sung huyết hang vị dạ dày có thể sử dụng sản phẩm từ thiên nhiên như nghệ, mật ong... để hỗ trợ điều trị. Bên cạnh đó, cần có chế độ ăn uống khoa học, lành mạnh. Nên kiêng ăn các thức ăn có vị chua, cay, nóng và các loại đồ ăn nhiều dầu mỡ. Không uống rượu, bia, nước có ga. Không hút thuốc lá, thuốc lào. Hạn chế uống cà phê... Khi ăn nên ăn chậm, nhai kỹ, ăn uống điều độ đúng giờ, đủ bữa, không ăn quá no hoặc để quá đói. Và cần có chế độ tập luyện phù hợp.',
    '57.txt': '1. Tiền sử bệnh\nTiền sử bệnh nội khoa:\nTăng huyết áp.\nĐái tháo đường, có biến chứng bệnh lý thần kinh ngoại biên.\nSuy tim không do thiếu máu cơ tim (theo chẩn đoán trước đây).\nBệnh thận mạn tính.\nThuốc trước khi nhập viện \n\nTorsemide: uống 1 viên/ngày, đôi khi 2 viên/ngày.\nInsulin glargine: theo đơn 100 đơn vị x 2 lần/ngày, hiện đã ngừng sử dụng.\nIsosorbide: đã hết thuốc khoảng 3 tuần trước nhập viện.\nRosuvastatin (Crestor): đã hết thuốc khoảng 3 tuần trước nhập viện.\nCarvedilol: đã hết thuốc khoảng 3 tuần trước nhập viện.\n2. Bệnh sử hiện tại\nLý do nhập viện: xét nghiệm bất thường - tăng kali máu và Creatinine tăng\nCác sự kiện trước khi nhập viện\n- Đi khám bác sĩ chăm sóc chính (PCP) để tái khám hàng năm\n- Báo cáo đang dùng 1 viên Torsemide/ ngày , đôi khi 2 viên\n- Bác sĩ đề nghị ngừng sử dụng torsemide lâu dài\n- Bệnh nhân không kiểm tra đường huyết thường xuyên tại nhà do hết que thử đường máu.\n- glucose là thấp, nên ngừng dùng glargine\n- Báo cáo bị ngã gần đây\n- Dùng xe đẩy hàng làm dụng cụ hỗ trợ đi lại\n- Các loại thuốc (isosorbide, crestor, carvedilol) đã hết khi đi khám PCP\n- Hết isosorbide, crestor, carvedilol khoảng 3 tuần trước\n- Các xét nghiệm xét nghiệm được kiểm tra bởi PCP: kali là 6.3 mẫu không tan máu\n- Được gọi điện bởi bác sĩ nội trú trực và được yêu cầu đến khoa Cấp cứu\n\n3. Đánh giá tại bệnh viện\nKết quả xét nghiệm\n- kali 6.3, mẫu không tan máu\n- kali (k).8\n- ure (bun) 83\n- creatinine 5.7\n- hemoglobin 7.8\n- hba1c 6.5\n- bnp 21,000\n- tổng phân tích nước tiểu bình thường\n đồ uống có cồn, đồ chiên rán,.... Tăng cường sản phẩm Folate, Vitamin C, các carotenoid, các chất chống oxy hóa  Flavonoid. \nCác thủ thuật đã thực hiện\n- Được dùng insulin và dextrose cho tăng kali máu\n- iv lasix 40 mg once',
    '58.txt': '1. Tiền sử bệnh\nTiền sử bệnh nội khoa:\nTăng huyết áp.\nĐái tháo đường, có biến chứng bệnh lý thần kinh ngoại biên.\nSuy tim không do thiếu máu cơ tim (theo chẩn đoán trước đây).\nBệnh thận mạn tính.\nThuốc trước khi nhập viện \n        Ăn 6 bữa 1 ngày (7h, 9h, 11h, 14h, 17h, 20h), người đỡ mệt, đại tiểu tiện bình thường, nước tiểu trong \n        Nhịp tim đều, 85 lần/phút, HA: 130/75mmHg \n        XN máu: \n        Glucose: 5,8 mmol/l \n        Cholesterol: 4,7 mmol/l, Triglycerid: 1,9mmol/l \n        LDL - cholesterol: 2,2 mmol/l; HDL - cholesterol: 2mmol/l \n        HbA1c: 7,5 pP%. \n        Tiền sử bản thân:\n      Uống rượu 30 năm, mỗi ngày khoảng 400ml.\n      Phát hiện tăng HA từ năm 2009, không duy trì thuốc thường xuyên.\n      Tiền sử dị ứng: Không có\n        Tiền sử gia đình: Chưa phát hiện gì bất thường\n2. Chẩn đoán: Đái tháo đường typ II/Tăng HA độ III đáp ứng với thuốc.\n3. Tiên lượng: Bệnh mạn tính, điều trị ổn định từng đợt, cần chú ý biến chứng tại cơ quan đích.\n4. Hướng điều trị:\n- Các loại thuốc (isosorbide, crestor, carvedilol) đã hết khi đi khám PCP\n- Hết isosorbide, crestor, carvedilol khoảng 3 tuần trước\n- Các xét nghiệm xét nghiệm được kiểm tra bởi PCP: kali là 6.3 mẫu không tan máu\n- Được gọi điện bởi bác sĩ nội trú trực và được yêu cầu đến khoa Cấp cứu\n\n3. Đánh giá tại bệnh viện\nKết quả xét nghiệm\n- kali 6.3, mẫu không tan máu\n- kali (k).8\n- ure (bun) 83\n- creatinine 5.7\n- hemoglobin 7.8\n- hba1c 6.5\n- bnp 21,000\n- tổng phân tích nước tiểu bình thường\n- cúm âm tính\nKết quả chẩn đoán hình ảnh: chụp x-quang ngực: Tăng gánh nhẹ tuần hoàn phổi, tràn dịch màng phổi hai bên nhẹ, tim to\nCác thủ thuật đã thực hiện\n- Được dùng insulin và dextrose cho tăng kali máu\n- iv lasix 40 mg once',
    '59.txt': 'Câu hỏi từ người dùng :\n\nCháu năm nay 18 tuổi da rất nhiều dầu và bị mụn trứng cá từ năm lớp 8. Đến năm lớp 11 cháu đi khám da liễu, thì sau khoảng 3 tháng điều trị da cháu ít dầu hơn và dường như không còn mụn. Nhưng sau 6 tháng da cháu bắt đầu lên mụn trở lại gồm mụn ở trán, má, 2 bên quai hàm và cằm, hầu như là mụn cám và mụn đầu trắng. Cháu không biết có nên đi khám lại không vì lo ngại thuốc sẽ ảnh hưởng đến sức khoẻ sinh sản cộng thêm việc khám sau một thời gian lên lại mụn thì rất tốn kém. Mong các bác sĩ cho cháu lời khuyên về vấn đề chăm sóc da dầu mụn và làm thế nào để da hết mụn không tái phát ạ. Cháu cảm ơn các bác sĩ ạ.\n\nCâu trả lời của bác sĩ: \n\nPhổi thông khí đều\nCòn sonde tiểu, nước tiểu có cặn\nCận lâm sàng::\n- lactate 1.1-->0.8\n- Cấy nước tiểu : nhiễm khuẩn đường tiết niệu có các vi khuẩn hỗn hợp\n- cấy máu :âm tính\n-chỉ số marker viêm của anh ấy đã có xu hướng tăng \nchẩn đoán hình ảnh: \n - MRI:  không thấy tình trạng viêm xương tủy nặng hơn\nSừng hóa nang lông bất thường\nVi khuẩn C. acne\nViêm tại chỗ\nHầu hết thiếu niên (80%) bị mụn trứng cá, một số trường hợp có thể kéo dài đến tuổi trưởng thành. Bạn điều trị theo chỉ định của bác sỹ chuyên khoa da liễu nhưng không rõ dùng thuốc gì. Nhưng về nguyên tắc khi điều trị thì bệnh sẽ ổn định, ngừng điều trị thì khả năng bệnh sẽ tái phát vì ngoài các yếu tố kể trên, bệnh còn liên quan đến nhiều yếu tố như yếu tố nội tiết, cơ địa, nghề nghiệp, do thói quen chà xát và nặn mụn…\n\nBạn vẫn nên đi khám lại để bác sĩ chuyên khoa tư vấn về thuốc và chăm sóc da mụn. Bạn vẫn có thể điều trị nhiều phương pháp mà không ảnh hưởng đến sức khỏe sinh sản. Chăm sóc da mụn một cách khoa học sẽ làm giảm dầu, giảm nhân mụn và có làn da đẹp.',
    '60.txt': 'Câu hỏi từ người dùng :\n\nCháu năm nay 18 tuổi da rất nhiều dầu và bị mụn trứng cá từ năm lớp 8. Đến năm lớp 11 cháu đi khám da liễu, thì sau khoảng 3 tháng điều trị da cháu ít dầu hơn và dường như không còn mụn. Nhưng sau 6 tháng da cháu bắt đầu lên mụn trở lại gồm mụn ở trán, má, 2 bên quai hàm và cằm, hầu như là mụn cám và mụn đầu trắng. Cháu không biết có nên đi khám lại không vì lo ngại thuốc sẽ ảnh hưởng đến sức khoẻ sinh sản cộng thêm việc khám sau một thời gian lên lại mụn thì rất tốn kém. Mong các bác sĩ cho cháu lời khuyên về vấn đề chăm sóc da dầu mụn và làm thế nào để da hết mụn không tái phát ạ. Cháu cảm ơn các bác sĩ ạ.\n\nCâu trả lời của bác sĩ: \n\nTrứng cá bắt nguồn từ bốn yếu tố chính:\n\n    - Phẫu thuật thay van hai lá cơ học\n    - ghép thận thất bại\n    Thuốc trước khi nhập viện: coumadin 3.0 mg /ngày\n\n2.  Bệnh sử hiện tại\nLý do vào viện:, INR dưới ngưỡng điều trị\nTheo lời bệnh nhân kể bệnh nhân xuất hiện chảy máu mũi xuất hiện khoảng 01 lần/ tuần. Khi làm xét nghiệm tại khoa chạy thận, phát hiện chỉ số đông máu dưới ngưỡng điều trị ( kết quả là INR 1.7) ,không có biểu hiện bất thường khác, vào viện\n    Khám hiện tại:\nKhông chảy máu mũi\nKhông có đau ngực\nKhông có khó thở\nKhông có đau bụng\n Không có buồn nôn, không nôn\n    Các cơ quan khác chưa phát hiện bất thường\n\n3.  Khám tại bệnh viện\n    Kết quả khám thực thể: dấu hiệu sinh tồn\n Nhiệt độ : 36.5 độ C\nMạch:  88 l/p\nHuyết áp: 120/70 mmHg\nNhịp thở: 20 l/p\nSPO2:  92 %\n\nBạn vẫn nên đi khám lại để bác sĩ chuyên khoa tư vấn về thuốc và chăm sóc da mụn. Bạn vẫn có thể điều trị nhiều phương pháp mà không ảnh hưởng đến sức khỏe sinh sản. Chăm sóc da mụn một cách khoa học sẽ làm giảm dầu, giảm nhân mụn và có làn da đẹp.',
    '61.txt': 'Câu hỏi từ người dùng:\n\nChào Bác sĩ,\nTôi đang tìm hiểu về các biện pháp ngừa thai, cụ thể là ngừa thai bằng phương pháp cấy que, vui lòng cung cấp thông tin chi tiết giúp!\n- Thời gian tốt nhất thực hiện cấy que là khi nào? Tôi mới sinh bé được 10 ngày.\n- Chi phí cho một lần cấy que là bao nhiêu?\n- Với phương pháp cấy Implanon, thời gian ngừa là 3 năm, sau 3 năm tôi phải thực hiện lấy que cũ ra và cấy que mới vào nếu vẫn muốn tiếp tục ngừa thai hay như thế nào?\n- Sau khi cấy que, có những tác dụng phụ nào đáng kể cần lưu ý?\nXin cảm ơn!\n- Hôm nay, bệnh nhân đến khoa Cấp cứu khám vì tình trạng mệt mỏi kéo dài trong vài ngày, kèm theo khó khăn khi ra khỏi giường vào sáng nay.\n- Bệnh nhân đã có đau bụng gián đoạn kể từ khi phẫu thuật.\nTriệu chứng hiện tại\n- hạ huyết áp, không đặc hiệu\n- mệt mỏi\n- mệt mỏi\n- khó khăn khi ra khỏi giường\n- đau bụng gián đoạn\n- khó thở khi gắng sức\n- ho mạn tính có đờm vàng loãng\n- chán ăn trong vài tháng\nĐặc điểm triệu chứng\nĐối với phụ nữ sau sanh 10 ngày như em, thời gian cấy que tránh thai như sau:\n-  Nếu cho con bú:  tốt nhất là 6 tuần sau sanh. Nếu chưa được 6 tuần sau sanh, chỉ cấy que khi không còn biện pháp tránh thai nào khác.\n- Nếu không cho con bú và sau sinh dưới 21 ngày: có thể cấy que bất kỳ lúc nào.\nChi phí một lần cấy que (có tác dụng 3 năm) vào khoảng 2,4 triệu – 2,8 triệu chưa kể chi phí xét nghiệm bổ sung nếu cần thiết và khác nhau tùy theo mỗi phụ nữ.\nSau thời hạn 3 năm, nếu muốn tiếp tục với biện pháp tránh thai bằng que cấy thì sẽ được tháo que cũ và cấy que mới vào ngay trong một lần thủ thuật.\nCũng như mọi biện pháp tránh thai khác, que cấy có vài tác dụng phụ. Thông thường là rỉ máu âm đạo vài tuần sau cấy, kinh nguyệt thưa…',
    '62.txt': 'Để hạn chế sự tiến triển của các hạt tophi thì cần tuân thủ điều trị để hạ lượng acid uric máu, bạn cần đưa ông bạn đến khám chuyên khoa cơ xương khớp để được tư vấn và có lộ trình điều trị cụ thể.2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: ho, thiếu oxy\n    Thời điểm khởi phát triệu chứng: 1 tuần trước khi nhập viện\n    Triệu chứng hiện tại\n    - ho\n    - nghẹt ngực\n    - khó thở nhẹ-vừa khó thở khi gắng sức\n    - mệt mỏi\n    - mệt mỏi\n    - sốt lên đến 101\n    - thỉnh thoảng tiêu chảy các thuốc của ông ấy\n    - đi tiểu nhiều lần vào ban đêm, không có đau buốt khi đi tiểu\n    Đặc điểm triệu chứng\n    - khó thở khi gắng sức bao gồm cảm giác khó thở sau khi đi bộ vài khối\n    - ho đặc như bã theo lời bệnh nhân nhưng ông ấy không khạc nhổ gì\n    - sốt lần cuối sốt là vào ngày\n    - tiêu chảy đã tự khỏi vào buổi sáng hôm sau\n    - Phủ nhận đau ngực\n    - Phủ nhận buồn nôn/nôn\n    - Phủ nhận đổ mồ hôi đêm\n    - Phủ nhận khó thở khi nằm\n    - Phủ nhận khó thở khi nằm đột ngột (paroxysmal nocturnal dyspnea)\n    Sự kiện trước khi nhập viện\n    - Vợ có các triệu chứng tương tự 3 tuần trước, được chẩn đoán là giãn phế quản, phản ứng tốt với azithromycin\n    - Tự điều trị bằng tylenol và mucinex d\n    - Đi khám bác sĩ chăm sóc chính (PCP) vào buổi sáng ngày nhập viện\n    - Bác sĩ PCP ghi nhận spo2 là 90-92% spo2 khi thở khí trời, 93% khi không dùng oxy\n    - Bác sĩ PCP đã gửi bệnh nhân  đến phòng cấp cứu\n\n3.  Đánh giá tại bệnh viện\n    Kết quả xét nghiệm\n    - số lượng bạch cầu là 12.5\n    - tổng phân tích nước tiểu đáng chú ý chỉ có vếtvết protein niệu.  \n    - troponin âm tính x1\n    Kết quả chẩn đoán hình ảnh: chụp x-quang ngực được cho là viêm phổi thùy dưới phải (RLL PNA)',
    '63.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mạn tính\n    - Bệnh bạch cầu dòng tủy mãn tính\n    - tăng cholesterol máu đơn thuần\nE có uống thuốc kháng B-histamin thì triệu chứng giảm nhưng k hết. Và hiện tại vùng mẩn ngứa tập trung chủ yếu tại vùng bẹn khiến sinh hoạt của e rất bất tiện.\nMong muốn được BS tư vấn để cải thiện tình trạng này ạ\ntrả lời : Chào bạn! \n    - gleevec (ngừng uống cách nhập viện 5 ngày)\n    - allopurinol (ngừng uống \ncách nhập viện 5 ngày)\n    - Thuốc giảm đau (ngừng)\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: ảo giác\n    Thời điểm khởi phát triệu chứng: Khoảng một tháng trước\n    Diễn biến\n    - ảo giácxuất hiện khoảng một tháng trước.\n    - Đã được điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định.\n    - Thuốc giảm đau đã ngừng.\n    - Không có thay đổi đáng kể về triệu chứng mặc dù đã điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định và ngừng thuốc giảm đau.\n    - Liều gleevec đã ngừng cách nhập viện 5 ngày.\n    - allopurinol đã ngừng cách nhập viện 5 ngày.\n    - ảo giác xảy ra khoảng mỗi đêm lúc 2 giờ sáng.\n    Triệu chứng hiện tại: ảo giác\n    Đặc điểm triệu chứng\n Bệnh nhân nhìn/nghe thấy có nhiều người (đôi khi tổng cộng khoảng 8 người).\nThấy những người này đang nói chuyện với mình.\nẢo giác không mang tính chất ra lệnh \nBệnh nhân thường hoang tưởng/lo sợ rằng họ sẽ chạm vào da của mình.\n    Diễn biên  trước khi nhập viện\n    - Điều trị nhiễm khuẩn đường tiết niệu, vị trí không xác định\n    - Thuốc giảm đau đã ngừng\n    - Liều gleevec đã ngừng cách nhập viện 5 ngày\n    - allopurinol đã ngừng cách nhập viện 5 ngày\n    Tình trạng ngay trước khi nhập viện: ảo giác khoảng mỗi đêm lúc 2 giờ sáng\n3.  Đánh giá tại bệnh viện',
    '64.txt': 'Câu hỏi của người dùng gửi đến hệ thống\nChào Bác sĩ, mẹ em bị bệnh động mạch vành đã nhiều năm. Đi khám bệnh Bác sĩ kê cho thuốc Vastarel và *********. Tuy nhiên sau nhiều năm sử dụng thuốc thì mẹ em bị run tay và mẹ em đã ngừng sử dụng thuốc Vastarel\n    Các bệnh lý mạn tính\n    - bệnh Graves\n    - tăng lipid máu, không đặc hiệu\n\nhoặc run rẩy tay chân, mất thăng bằng khi đi đứng. Như vậy Mẹ của bạn đã bắt đầu bị run tay, đó có thể do tác dụng phụ của thuốc. Nhưng theo nhiều nghiên cứu thì tác dụng phụ của thuốc trimetazidine sẽ mất đi \nsau khi ngừng thuốc khoảng 4 tháng, bệnh nhân sẽ trở về bình thường. Như vậy, bạn cần theo dõi là Mẹ bạn đã ngừng thuốc được bao lâu rồi? Chờ đến sau 4 tháng nếu không đỡ hiện tượng run tay thì nên \nđi khám bác sĩ nội thần kinh để xác định nguyên nhân. Đối với thuốc Nitramyl 2,5 mg/viên (nitroglycerin) thì tác dụng chủ yếu của thuốc là làm giãn mạch vành, gây tăng tưới máu cho cơ tim, dùng để ngừa cơn đau thắt ngực\n trong bệnh mạch vành. Thuốc được thải ra khỏi cơ thể nhanh và không tích lũy nên không gây hại khi dùng lâu dài, nhưng trái lại thuốc này có nhiều tác dụng phụ rất thường hay gặp (vì tác dụng phụ này gây ra do đặc tính của thuốc) đó là: \nđau đầu, chóng mặt, buồn nôn, đỏ mặt, và tác dụng tụt huyết áp thế đứng. Nếu Mẹ bạn khi dùng thuốc có một trong các tác dụng phụ nói trên thì nên báo với bác sĩ để ngừng dùng thuốc này và thay bằng các thuốc khác. \nHiện nay có nhiều thuốc đã được chứng minh hiệu quả hơn và an toàn hơn, ít tác dụng phụ hơn Nitramyl. Vì vậy bạn có thể đưa Mẹ bạn đến khám 1 bác sĩ chuyên khoa tim mạch để chỉ định thuốc phù hợp. \nCám ơn bạn đã tham gia chương trình. Chúc Mẹ bạn sớm được hồi phục sức khỏe.',
    '65.txt': 'Câu hỏi từ người dùng:\n\nEm bị bệnh rụng tóc từng mảng. Mong bác cho cách điều trị được không ạ? Em bị 05 năm rùi điều trị nhiều thuốc và bệnh viện rùi mà đến nay vẫn không hết. Cảm ơn Bác sỹ ạ!\n\nCâu trả lời của bác sĩ:\n\nChào bạn! Bạn đang có vấn đề về rụng tóc\nMình xin tư vấn một số phương pháp điều trị dưới đây\n1.        Liệu pháp **************\nTiêm ************** nội thương tổn được chỉ định trong rụng tóc từng vùng khi diện tích thương tổn dưới 50% diện tích da đầu. Tiêm nhắc lại sau mỗi 4-6 tuần. Corticosteroid bôi tại chỗ thường được chỉ định cho trẻ em. Các thuốc hay dùng là kem *************************** hai lần một ngày hoặc ************************************ (corticoid mức độ mạnh).\nĐối với rụng tóc toàn bộ hoặc rụng tóc toàn thể khó điều trị có thể sử dụng 2,5g ********************e có bao phim che phủ 6 ngày/tuần trong 6 tháng.\nThời gian điều trị ít nhất là ba tháng trước khi tóc mọc lại, duy trì điều trị thường xuyên nếu cần thiết.\nNghiệm pháp gắng sức dương tính với thiếu máu cơ tim cục bộ, biểu hiện ST chênh xuống ở các chuyển đạo thành dưới và thành bên. \nCác chất gây mẫn cảm như diphenylcyclopropenone (diphencyprone) và dinitrochlorobenzene được quét lên vùng rụng tóc, chúng sẽ gây viêm da tiếp xúc dị ứng ở vùng điều trị, có thể sử dụng 01 lần/tuần trên vùng rụng tóc trong 06 tháng. Các tác dụng phụ bao gồm viêm da tiếp xúc, mày đay, bất thường sắc tố (gồm cả bạch biến).\n3.        PUVA (psoralen kết hợp UVA)\nPUVA toàn thân và tại chỗ đều được sử dụng. Cần 20-40 lần điều trị nhiều bệnh nhân tái phát sau khi ngừng điều trị.\nBạn nên đến gặp các bác sĩ da liễu để đươc tư vấn trực tiếp và lựa chọn phương pháp điều trị phù hợp. \nChúc bạn sức khoẻ\n',
    '66.txt': 'Câu hỏi từ người dùng:\nDạ e chào bác sĩ . Cho e hỏi những người mà có u lành về tuyến vú như u xơ tuyến vú hay u nang tuyến vú chẳng hạn .có thể bổ sung vitamin 3b b1 b6 b12 được ko ạ . E tìm thông tin trên mạng thì có đọc được nói rằng vitamin b12 làm kích thích khối u ung thư phát triển , \ncòn đối với khối u lành tính có ảnh hưởng ko bác . Em cảm ơn ạ\nCâu trả lời của bác sĩ\nChào bạn! \nVitamin B12 là một vitamin rất cần thiết cho cơ thể. Cơ thể người không thể tự tổng hợp được chúng. Nhu cầu hàng ngày khuyến khích là 2mcg đối với nam, nưa trưởng thành; còn đối với phụ nữ có thai thì  và cho con bú thì khoảng 2,2mcg. \nNhững người dễ thiếu vitamin B12 bao gồm những người ăn chay trường diễn, viêm teo niêm mạc dạ dày, phẫu thuật cắt bỏ 1 phần hay toàn bộ dạ dày,.... Khi thiếu B12 sẽ rối loạn sản xuất máu ở tủy xương; các biểu hiện về thần kinh như dị cảm, \ngiảm cảm giác vị thế, khả năng trí óc giảm, hạ huyết áp tư thế đứng,.... Đối với bệnh nhân ung thư cũng được khuyên là không nên sử dụng B12. Đối với bệnh u nang tuyến vú, u xơ tuyến vú thì cũng chưa có nghiên cứu về việc sử dụng B12 có tăng nguy cơ \nung thư hay không. Tuy nhiên nếu u nang tuyến vú, u xơ tuyến vú thì việc bổ sung B12 cũng giúp bệnh nhân củng cố sức khỏe, giảm nguy cơ thiếu máu. Với u xơ tuyến vú, u nang tuyến vú thì những thực phẩm không nên sử dụng là: các sản phẩm chứa nội tiết Estrogen;\n- cúm âm tính\nKết quả chẩn đoán hình ảnh: chụp x-quang ngực: Tăng gánh nhẹ tuần hoàn phổi, tràn dịch màng phổi hai bên nhẹ, tim to\nVới những trường hợp u xơ tuyến vú, u nang tuyến vú nên tự theo dõi tị nhà và  đi khám định kỳ theo hẹn của bác sỹ để theo dõi sự phát triển của khối u.\nChúc bạn sức khỏe!',
    '67.txt': 'Câu hỏi từ người dùng :\n    - suy tim\n    - bệnh mạch máu ngoại biên\n    - bệnh phổi tắc nghẽn mạn tính\n    - Ngưng thở khi ngủ do tắc nghẽn đang dùng BiPAP\n    - Ung thư biểu mô tế bào vảy xâm nhập của dương vậtbiệt hóa kém, sau cắt bao quy đầu với bờ diện cắt dương tính.\n    Tiền sử phẫu thuật / thủ thuật\n\nCâu trả lời của bác sĩ: \n\nViêm hang vị sung huyết là tình trạng niêm mạc vùng hang vị dạ dày viêm, các mạch máu vùng viêm giãn nở do ứ máu nhiều. Biểu hiện chủ yếu là đau bụng cồn cào kèm theo ợ hơi, ợ chua, có thể có cảm giác buồn nôn hoặc nôn. Trước kia bệnh dạ dày được coi là bệnh nan y và nguy hiểm nhưng mấy chục năm gần đây nhờ nội soi phát triển nên bệnh đã được điều trị hiệu quả. Để điều trị dứt điểm bệnh viêm sung huyết hang vị dạ dày, trước hết, người bệnh cần được thăm khám chuyên khoa, nội soi dạ dày để tìm nguyên nhân, đánh giá mức độ bệnh. Căn cứ trên kết quả khám, các bác sĩ sẽ đưa ra phác đồ điều trị hiệu quả nhất cho từng bệnh nhân. Điều quan trọng là bệnh nhân cần tuân thủ tuyệt đối sự chỉ dẫn của bác sĩ trong quá trình điều trị. Không được bỏ thuốc giữa chừng cũng như không được tự ý tăng giảm liều lượng thuốc mà chưa có sự đồng ý của bác sĩ. Ngoài ra, người bị viêm sung huyết hang vị dạ dày có thể sử dụng sản phẩm từ thiên nhiên như nghệ, mật ong... để hỗ trợ điều trị. Bên cạnh đó, cần có chế độ ăn uống khoa học, lành mạnh. Nên kiêng ăn các thức ăn có vị chua, cay, nóng và các loại đồ ăn nhiều dầu mỡ. Không uống rượu, bia, nước có ga. Không hút thuốc lá, thuốc lào. Hạn chế uống cà phê... Khi ăn nên ăn chậm, nhai kỹ, ăn uống điều độ đúng giờ, đủ bữa, không ăn quá no hoặc để quá đói. Và cần có chế độ tập luyện phù hợp.',
    '68.txt': '1. Tiền sử bệnh\nBệnh lý mãn tính: bệnh động mạch vành, bệnh lý thần kinh ngoại biên\n\n2. Bệnh sử hiện tại\n Trong vòng một năm qua bệnh nhân thi thoảng xuất hiện mệt mỏi, nhất là khi vận động gắng sức khó thở tăng.\nLúc vào viện khám\nBệnh nhân tỉnh\n- mệt mỏi\n- Khó thở tăng khi vận động gắng sức\n- Đi lại khó khăn, cần dùng gậy hỗ trợ đi lại\n- Đau hai bàn chân do bệnh lý thần kinh ngoại biên \nCận lâm sàng:\nKết quả chẩn đoán hình ảnh\n- Đo sức bền cơ tim bằng đồng vị phóng xạ kết quả có dị tật cố định vùng trước vách và vách dưới với  giảm chức năng tâm thu thất trái nghiêm trọng kèm theo rối loạn chức năng thất trái.\n- Chụp động mạch vành cho thấy bệnh ba thân động mạch vành nghiêm trọng \nCác thủ thuật đã thực hiện\n- Chụp động mạch vành can thiệp\nChào bạn, bé nhà bạn 29 tháng, nặng 13 kg, liều Augmentin (amoxicillin/acid clavulanic) cho cháu sẽ là: 80 mg/kg/ngày, tương đương 1040 mg/ngày. Như vậy, liều bác sĩ kê 500 mg/lần x 2 lần ngày, dùng trong 7 ngày là đúng với cân nặng của Bé. Uống thuốc cháu có đỡ bệnh, nghĩa là kháng sinh đã có đáp ứng. Tuy nhên khoảng 7-10 ngày sau bệnh tái phát, thì phải hỏi lại bác sĩ về nguyên nhân vì sao bệnh tái lại: do cơ thể Bé, do đặc điểm bệnh, hay do vi khuẩn chưa diệt hết? Cần xem lại các thuốc dùng kèm trong đơn? Nếu được bạn có thể cung cấp cho chúng tôi chi tiết thuốc của Bé để có thể tư vấn kỹ hơn. Dùng Augmentin nhiều lần không ảnh hưởng gì nghiêm trọng, ngoại trừ: thuốc gây tiêu chảy, nên làm giảm hấp thu dinh dưỡng của trẻ (cần bổ sung men tiêu hóa cho Bé), và dùng kéo dài 1 kháng sinh có thể gây phát triển vi khuẩn đề kháng thuốc (vi khuẩn lờn thuốc), sau này sẽ khó để điều trị bệnh nhiễm trùng ở trẻ.',
    '69.txt': '1.  Tiền sử bệnh\n    Nhiều lần có ý định tự tử trước đây do chia tay bạn trai cũ\n    Bệnh lý mãn tính\n    - trầm cảm và  rối loạn lo âu\n2.  Bệnh sử hiện tại\n    Lý do nhập viện:  tổn thương chi dưới do tự tử không thành\n    Theo lời người nhà kể lại bệnh nhân bị sốc khi chia tay bạn trai cũ, bị bạn bè trêu chọc, buồn chán, uống nhiều rượu,hút cần sa. Sau đó lái xe đến một cây cầu, nhảy từ trên cầu xuống  bệnh nhân nhảy cầu tự tử, sau đó tổn thương chi dưới , vào viện\nTình trạng khi đến khoa Cấp cứu:\nBệnh nhân tỉnh, tiếp xúc tốt\nKhông xác nhận đã bị trầm cảm trước đó, và không xác nhận có ý định tự tử\n Khám thấy bị tổn thương tổn thương chi dưới nghiêm trọng.\nChẩn đoán: Phẫu thuật cắt cụt chân trái bên trên gối  và cắt cụt chân phải bên dưới gối / biến chứng thuyên tắc phổi hai bên  - nhiễm trùng chi dưới bên phải do Enterococcus kháng vancomycin\n    Điều trị :\n Đã phẫu thuật cắt cụt chân trái trên gối và chân phải dưới gối.\nDùng kháng sinh tĩnh mạch.\n\nCâu hỏi từ người dùng : Dạ em có một người, hiện tại bạn ấy đang nổi những nét nhẹ nhàng nhẹ nhàng rất đến và khắp nơi, bạn em nổi quanh năm ạ. Những thuốc giảm đau nổi bật rất khó chịu và tư vấn, được biết bạn đã từng đi chữa trị và sử dụng rất nhiều thuốc nhưng không đỡ ạ. Mẹ của bạn cũng là người nên Yên tĩnh chắc chắn là dây truyền tải. Cả hai người đều không phải dị ứng thời tiết hay thức ăn, nó có thể nổi lên mọi lúc. Mọi người cho em hỏi là bệnh này là bệnh gì ạ và để lâu có bệnh nặng hay ảnh hưởng nghiêm trọng gì không ạ. Mọi người cho em xin ý kiến \u200b\u200b\u200b\u200b\u200b\u200b\u200bvà có cách nào để giải quyết cho em không ạ. Em cảm ơn rất nhiều .\n Diễn biến ổn định và được chuyển vào khoa tâm thần nội trú.\n',
    '70.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mãn tính\n    - béo phì\n    - đái tháo đường được kiểm soát bằng chế độ ăn\n    - ngừng thở khi ngủ\n    - tiền sử lâm sàng suy tim, không đặc hiệu\n    Dị ứng: Dị ứng furosemide\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: khó thở\n    Thời điểm khởi phát triệu chứng\n    - tiền sử trước vào viện  ba ngày\n    - bắt đầu nhận thấy khó thở khi xem TV lúc 11:00 sáng ba ngày trước\n    Triệu chứng hiện tại\n    - khó thở\n    - khó nằm vào ban đêm để ngủ vì thở\n    - căng cứng vai trái (ngắt quãng, xảy ra vài lần mỗi ngày, kéo dài 5 phút)\n    - tiền sử ho có đờm trắng\n    - chủ quan sốt\n    - phù chi dưới (ổn định)\n    Đặc điểm triệu chứng\n    - Vị trí: căng cứng vai trái\n    - Thời gian: kéo dài 5 phút (căng cứng vai trái)\n    - Tần suất: xảy ra vài lần mỗi ngày (căng cứng vai trái)\n    - Triệu chứng liên quan: khó nằm vào ban đêm để ngủ vì khó thở, ho có đờm trắng, chủ quan sốt, phù chi dưới\n    Các sự kiện trước khi nhập viện\n    - bỏ lỡ 3 ngày dùng một số thuốc vì quên đổ thuốc vào hộp (tuần qua)\n    - gọi EMS\n    - EMS phát hiện bệnh nhân thiếu oxy với độ bão hòa oxy là 86 khi thở khí trời\n    - EMS cho bệnh nhân thở oxy qua mask không hồi phục (NRB)\n    - EMS đưa bệnh nhân đến khoa Cấp cứu\n    Tình trạng ngay trước khi nhập viện: thiếu oxy với độ bão hòa oxy là 86 khi thở khí trời\n\n3.  Đánh giá tại bệnh viện\n- Các yếu tố có thể ảnh hưởng đến sự phát triển lây nhiễm bệnh dại bao gồm:\nLoại hình tiếp xúc và loại động vật cắn\nMức độ nghiêm trọng của vết cắn\n    - được cho aspirin 325mg\n    - được cho albuterolipratropium nebulizer\n    - được cho methylprednisolone 125mg iv\n    - lợi tiểu 600cc',
    '71.txt': 'Câu hỏi từ người dùng:\nTình trạng khi đến khoa Cấp cứu:\nBệnh nhân tỉnh, tiếp xúc tốt\nKhông xác nhận đã bị trầm cảm trước đó, và không xác nhận có ý định tự tử\n Khám thấy bị tổn thương tổn thương chi dưới nghiêm trọng.\nChẩn đoán: Phẫu thuật cắt cụt chân trái bên trên gối  và cắt cụt chân phải bên dưới gối / biến chứng thuyên tắc phổi hai bên  - nhiễm trùng chi dưới bên phải do Enterococcus kháng vancomycin\n    Điều trị :\n Đã phẫu thuật cắt cụt chân trái trên gối và chân phải dưới gối.\nCâu trả lời của bác sĩ:\n\nChào em, đọc mô tả của em mình có một số câu hỏi sau: Từ khi phát hiện đến nay em có thường xuyên theo dõi tình trạng nổi mẩn ở vùng lưng em hay không? Tiến triển của những vết nổi mẩn đó như thế nào? Nó có lan ra xung quanh, có đỏ hơn trước hay có nổi sần tạo thành lớp sừng không? Nếu lan thì nó có lan xuống vùng hông - mông hay lên vùng cổ vai gáy không? Sau khi phát hiện em điều trị bằng thuốc viêm nang lông có thấy tình trạng bệnh xấu đi không hay là vẫn như cũ? Bình thường ngủ em có hay cởi trần để ngủ hay không? Từ nhỏ đến giờ tình trạng nổi mẩn vùng lưng ấy xuất hiện nhiều lần hay chỉ một lần và thường trong hoàn cảnh nào em phát hiện nó? Vì nổi mẩn đỏ ngứa ở lưng có thể xuất hiện do quá trình ma sát giữa da và quần áo, vệ sinh da kém, thay đổi thời tiết. Ngoài ra tình trạng này còn hình thành do một số bệnh lý, vấn đề liên quan đến sức khỏe. Cụ thể như suy giảm chức năng gan, nổi mề đay, bệnh vảy nến, tác dụng phụ của một số loại thuốc chữa bệnh. Để điều trị, các bác sĩ cần xác định chính xác nguyên nhân gây bệnh. Nếu như em còn điều gì thắc mắc có thể liên hệ lại với mình. Cảm ơn em.',
    '72.txt': 'Điều trị phụ thuộc vào mức độ nặng và bệnh lý nền có liên quan. Cần lưu ý cân bằng giữa hiệu quả điều trị kiểm soát triệu chứng và những tác dụng gây độc của liệu pháp điều trị. **************** thể hệ sau không gây buồn ngủ là lựa chọn hàng đầu sau đó đến **************** thể hệ 1, corticoid liều thấp cách ngày hoặc hàng ngày và giảm liều chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không đỡ hoặc có chống chỉ định, xin bạn hãy đến với bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất.\n    Các bệnh mãn tính\n    - Rối loạn cảm xúc \n    - Rối loạn lưỡng cực\n    - rối loạn lo âu\n    Thuốc đã điều trị trước khi nhập viện lần này\n    - Thuốc kê đơn (tên không được chỉ định)\n    - klonopinclonidine (được kê đơn)\n    - clonidine (mua trên đường phố)\n    - suboxone (bắt đầu 3 tuần trước), không dùng đều 5-6 ngày, dừng thuốc trước nhập viện 01 ngày\n    Các yếu tố nguy cơ liên quan\n    - Lạm dụng chất kích thích, chất gây nghiện opioid\n2.   Bệnh sử hiện tại\n    Lý do nhập viện:i ý định tự tử và ý nghĩ tự bắn vào đầu\n   Theo lời kể bệnh nhân bị trầm cảm ngày càng nặng , vài tháng trước có các giai đoạn giống hưng cảm  biểu hiện bằng tăng hoạt động và giảm nhu cầu ngủ 3-4 ngày, trong vài tuần nay và có ý định tự tử.  Thuốc kê đơn không dùng đều đặn x5-6 ngày, bắt đầu dùng suboxone 3 tuần trước vì rẻ hơn, dừng dùng suboxone ngày hôm qua\n    Triệu chứng khi vào viện\n    - ý nghĩ tự tử, có nghĩ tự bắn vào đầu\n    - lo âu\nhoảng sợ\n    - hoang tưởng như đang chiếm khí oxy của người khác\n Điều trị\n    - Hiện đang được chăm sóc tâm thần bởi bác sĩ\n\n',
    '73.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mạn tính\n    - Viêm loét đại tràng trong nhiều năm qua\n    - bằng chứng Bệnh gan do rượu ở vị trí 35cm\n    Tiền sử phẫu thuật / thủ thuật: đã phẫu thuật tạo hậu môn nhân tạo kiểu túi nối hồi tràng - hậu môn\n    Thuốc trước khi nhập viện: thuốc giảm đau opioid\n\n2.  Tiền sử bệnh hiện tại\n    Thời điểm khởi phát triệu chứng: 48 giờ qua\n    Triệu chứng hiện tại\n    - ăn uống kém do buồn nôn\n    - đau bụng/khó chịu vùng bụng tăng dần, đặc biệt ở vùng hạ vị bên trái (LLQ)\n    - buồn nôn\n    - nôn x 1\n    - chủ quan sốt và run rẩy\n    - mất cảm giác ngon miệng/chán ăn\n    - suy nhược\n    -  Toàn trạng suy kiệt\n    - giảm lượng nước tiểu từ 1800 ml xuống còn 300 ml trong vòng 24 giờ\n    - tăng đáng kể lượng dịch từ ống thông hồi tràng từ 350 ml lên 1200 mltrong cùng khoảng thời gian\n    Đặc điểm triệu chứng\n•        Bệnh amyloidosis di truyền hoặc gia đình\nBệnh thoái hóa tinh bột là một bệnh đa hệ thống dẫn đến nhiều biểu hiện lâm sàng khác nhau, Hiện nay chưa có phương pháp điều trị tận gốc amyloidosis.Tuy nhiên, các phương pháp điều trị có thể giúp bạn kiểm soát các triệu chứng bệnh cũng như ngăn chặn tổn thương \n    - Thời gian: 48 giờ qua\n    - Tần suất: nôn x 1\n    - Triệu chứng liên quan: chủ quan sốt và run rẩy, mất cảm giác ngon miệng/chán ăn, suy nhược, toàn trạng suy kiệt  thiểu niệu, chảy dịch từ lỗ dò tăng đáng kể\n    Sự kiện trước khi nhập viện\n    - ăn uống kém do buồn nôn, được cho là do tác dụng phụ của thuốc giảm đau opioid\n    - ngừng thuốc giảm đau opioid vào hôm qua mà không có thay đổi các triệu chứng\n\n3.  Đánh giá tại bệnh viện',
    '74.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mạn tính\n    - Viêm loét đại tràng trong nhiều năm qua\n    - bằng chứng Bệnh gan do rượu ở vị trí 35cm\n    Tiền sử phẫu thuật / thủ thuật: đã phẫu thuật tạo hậu môn nhân tạo kiểu túi nối hồi tràng - hậu môn\n\nCháu năm nay 18 tuổi. Cháu bị bệnh nấm bẹn mấy tháng nay rồi ạ. Nhưng cháu phòng ngừa nên không bị lây lan, chỉ bị hai bên bẹn nhưng cháu dùng một số loại thuốc rồi mà chỉ được một khoảng thời gian rồi lại bị tái phát lại. Bác sĩ cho cháu hỏi loại thuốc nào có thể chữa khỏi ạ?\n\n2.  Tiền sử bệnh hiện tại\n    Thời điểm khởi phát triệu chứng: 48 giờ qua\n    Triệu chứng hiện tại\n    - ăn uống kém do buồn nôn\n    - đau bụng/khó chịu vùng bụng tăng dần, đặc biệt ở vùng hạ vị bên trái (LLQ)\n    - buồn nôn\n    - nôn x 1\n    - chủ quan sốt và run rẩy\n    - mất cảm giác ngon miệng/chán ăn\n    - suy nhược\n    -  Toàn trạng suy kiệt\n    - giảm lượng nước tiểu từ 1800 ml xuống còn 300 ml trong vòng 24 giờ\n    - tăng đáng kể lượng dịch từ ống thông hồi tràng từ 350 ml lên 1200 mltrong cùng khoảng thời gian\n    Đặc điểm triệu chứng\n    - Vị trí: LLQ\n    - Mức độ nghiêm trọng: đau bụng/khó chịu vùng bụng tăng dần\n    - Thời gian: 48 giờ qua\n    - Tần suất: nôn x 1\n    - Triệu chứng liên quan: chủ quan sốt và run rẩy, mất cảm giác ngon miệng/chán ăn, suy nhược, toàn trạng suy kiệt  thiểu niệu, chảy dịch từ lỗ dò tăng đáng kể\n    Sự kiện trước khi nhập viện\n    - ăn uống kém do buồn nôn, được cho là do tác dụng phụ của thuốc giảm đau opioid\n    - ngừng thuốc giảm đau opioid vào hôm qua mà không có thay đổi các triệu chứng\n\n3.  Đánh giá tại bệnh viện',
    '75.txt': 'Câu hỏi từ người dùng:\n- khối u trực tràng với hình ảnh của một u ác trực tràng đáng kể\n- sinh thiết chỉ cho thấy một u tuyến\n- đến khám để phân giai đoạn phẫu thuật với tem (viết tắt)\nCâu trả lời của bác sĩ:\n\nChào bạn! Nấm bẹn là một dạng nhiễm nấm  thường là do Trichophyton rubrum hoặc là T. mentagrophytes. Các yếu tố nguy cơ chính liên quan đến môi trường ẩm ướt (thời tiết nóng, quần áo ướt và chật, chứng béo phì do có nhiều nếp gấp da). Bệnh do nhiễm nấm da mà thành nên để điều trị nấm bẹn bạn phải được sử dụng thuốc kháng nấm dạng bôi hoặc dạng uống theo chỉ định của bác sĩ da liễu với liệu trình điều trị từ 1-4 tuần. \nNgoài ra để giảm thiểu nguy cơ bị lây nhiễm, hoặc tái phát nấm bẹn bạn nên:\n•        Giữ vùng bẹn khô thoáng: nên lau khô vùng bẹn sau khi tắm hoặc tập thể dục để giữ cho vùng này được khô thoáng. Nếu cơ thể tiết quá nhiều mồ hôi, có thể sử dụng bột để giảm độ ẩm vùng bẹn.\n•        Mặc đồ sạch: bạn nên thay quần lót ít nhất một lần mỗi ngày và có thể thay nhiều hơn nếu cơ thể tiết quá nhiều mồ hôi. Nên giặt đồ thể thao sau khi đã sử dụng.\n•        Lựa chọn đồ lót vừa mặc: không nên chọn đồ lót quá chật, vì ma sát sẽ làm da vùng bẹn tổn thương, từ đó tạo điều kiện thuận lợi cho sự phát triển của vi nấm.\n•        Không dùng chung đồ dùng cá nhân: không nên cho người khác mượn áo quần, khăn tắm và các đồ dùng cá nhân khác của mình, cũng như tránh đi mượn đồ dùng cá nhân của người khác.\nLời khuyên: bạn nên trực tiếp đến khám và gặp bác sĩ da liễu khi có các vấn đề về da để được tư vấn và điều trị triệt để cũng như tránh “tiền mất tật mang”.\n',
    '76.txt': 'Có ý định điều trị và lắp chân giả\nBệnh nhân suy nghĩ tích cực nghĩ cho tương lai ( nói rằng uộc sống không dừng lại vì một chấn thương từ bên ngoài)\nCâu trả lời của bác sĩ: Chào bạn,mình xin trả lời câu hỏi của bạn như sau Như bạn đã mô tả, trường hợp này bạn thuộc MÀY đay VÔ CĂN. Chúng tôi không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY đay MẠN. Tổ thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với bạch huyết T và đại thực bào. Các loại tiền thưởng khó quan sát trên kính hiển thị vi quang học thông thường cần xem trên kính hiển thị vi điện tử. Các tế bào cột sống tăng lượng hạ bì với các mức độ thoát ra khác nhau cũng được quan sát. Quảng cáo quảng cáo thiết bị thương mại miễn phí dịch sinh học không tìm thấy hình ảnh giảm phức tạp, bổ sung hay sợi fibrin. Đối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Bạn đay vô căn hộ là một bệnh mạn tính, việc điều trị phải dài và liên tục. Quá giá trị chỉ là giấy chứng nhận. Điều trị phụ thuộc vào mức độ nghiêm trọng và nền tảng có liên kết. Cần lưu ý cân bằng giữa hiệu quả kiểm soát triệu chứng và những tác hại gây độc của liệu pháp điều trị. *************** có thể hệ sau không gây buồn ngủ là đơn hàng đầu sau đó đến *************** có thể hệ 1, ********* bậc thấp cách ngày hoặc hàng ngày và giảm dần chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không được hỗ trợ hoặc chống chỉ định, xin bạn hãy đến gặp bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất. Cảm ơn bạn đã gửi câu hỏi',
    '77.txt': '1.  Tiền sử bệnh\n    CGhi nhận triệu chứng tương tự trước đây: xuất hiện đau vùng xương bánh chè – đùi phải dữ dội, xảy ra cách đây vài tháng.\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: Tái khám vì đau đầu gối phải\n    Thời điểm khởi phát triệu chứng\n    - Thời gian khởi phát: cách đây vài tháng (đối với lần khám  trước)\n    - Khởi phát cấp tính hay dần dần: Không ghi rõ\n    Diễn biến bệnh\n    - Thứ tự phát triển triệu chứng: Đau bánh chè đùi phải dữ dội cách đây vài tháng. Tiếp tục bị đau đau dữ dội khi thực hiện hầu hết các hoạt động.\n    - Tiến triển bệnh: Không ghi rõ\n    Triệu chứng hiện tại: đau đầu gối phải\nSiêu âm ổ bụng (gan mật, tụy, lách, thận, bàng quang)+ Hình ảnh giãn nh đường mật trong gan hai bên  \nSiêu âm Doppler tim, van tim: -Buồng thất trái giãn, chức năng tâm thu thất trái giảm nhiều (EF BP: 28%). Hở hai lá vừa. Hở chủ nhẹ. Hở ba lá nhiều. Thất phải giãn, chức năng tâm thu thất phải giảm (FAC: 30%). Tăng áp lực động mạch phổi vừa. \n Chụp Xquang ngực thẳng: Hình ảnh bóng tim to\n\nChẩn đoán: suy tim - suy thận mạn giai đoạn 5tăng huyết áp\n\n    - Thời gian: Không ghi rõ\n    - Tần suất: Không ghi rõ\n    - Chiếu xạ: Không ghi rõ\n    - Các yếu tố làm nặng thêm: hầu hết các hoạt động, đặc biệt là leo cầu thang và gập sâu\n    - Các yếu tố làm giảm: Không ghi rõ\n    - Các triệu chứng liên quan: Không ghi rõ\n\n3.  Đánh giá tại bệnh viện\n    Kết quả chẩn đoán hình ảnh\n    - x-quang vào thời điểm đó cho thấy những thay đổi thoái hóa nghiêm trọng của khớp bánh chè đùi phải\n    - chụp cắt lớp vi tính kể từ lần khám cuối cùng cho thấy góc tăng lên',
    '78.txt': 'Câu hỏi từ người dùng :\n\nDạo gần đây, cháu thường bịchảy máu cam, khoảng 1 tháng 1 lần, chảy ở mũi bên phải và rồi người hơi choáng và hết. Vậy bác sĩ cho cháu hỏi thường hay chảy máu cam là dấu hiệu bệnh gì? Phương pháp điều trị thế nào? Cháu nghĩ là do thời tiết cứ khô nóng là cháu chảy và cháu có hay thức đêm muộn đến 2 giờ, 3 giờ sáng nhưng vẫn ngủ đủ. Liệu có nghiêm trọng không thưa bác sĩ? Cháu cảm ơn bác sĩ.\n\nCâu trả lời của bác sĩ: \n\n    - tái khám với triệu chứng xuất hiện  ban đỏ lan rộng kèm theo chóng mặt\n    - được chuyển đến Khoa Cấp cứu\n    - Tại thời điểm khám bệnh  bệnh nhân có các triệu chứng tương tự\n    Triệu chứng hiện tại\n    -  ban đỏ xuất hiện nhiểu ở vị trí phẫu thuật\n    - đau khi sờ nắn vùng mổ\n    - chóng mặt\n    - không sốt\n    - hiện tại chưa phát hiện các triệu chứng toàn thân khác\n3.  khám tại bệnh viện\n    Dấu hiệu lâm sàng\n    - ban đỏ ở vị trí phẫu thuật\n    - đau ấn vùng mổ\n    - không sốt\nVùng mũi luôn có các cấu trúc mạch máu phong phú đặc biệt có các điểm hội tụ mạch máu gọi là các điểm mạch. Khi có thay đổi huyết áp, hắt hơi mạnh, viêm nhiễm hoặc tác động như ngoáy mũi có thể gây tổn thương điểm mạch và gây chảy máu mũi. Đa số là chảy máu ít và tự cầm. Tuy nhiên, nếu chảy máu tái phát nhiều, số lượng chảy nhiều dẫn đến choáng thì bạn phải đến khoa Tai mũi họng để được nội soi mũi họng kiểm tra, cần thiết sẽ đốt điểm mạch ngăn ngừa chảy máu tái phát. Khi chảy máu bạn đừng quá hốt hoảng, cố gắng hít sâu thở đều, cúi đầu nhẹ, hai tay bóp cánh mũi khoảng 5 - 10 phút.',
    '79.txt': 'Câu hỏi từ người dùng:\nEm bị Rối loạn chuyển hóa tinh bột (amyloidosis) đã lâu. Có dùng thuốc tại viện da liễu trung ương nhưng không thấy đỡ. Có cách nào chữa trị tận gốc hoặc giảm thiểu không ạ? Em cảm ơn.\nCâu trả lời của bác sĩ:\nChào bạn! Cảm ơn bạn đã gửi câu hỏi cho chúng tôi.\n Trả lời câu hỏi của bạn:Bệnh thoái hóa tinh bột (hay bệnh amyloidosis) là một bệnh gây lắng đọng protein hiếm gặp và nghiêm trọng. Nguyên nhân là do một loại protein bất thường gọi là amyloid tích tụ trong các mô hoặc cơ quan. Khi lượng protein amyloid lắng đọng tăng lên, chúng sẽ phá vỡ cấu trúc và gây cản trở chức năng sinh lý của mô hoặc cơ quan. Cuối cùng, sự lắng đọng protein amyloid sẽ bắt đầu gây ra các triệu chứng và suy các cơ quan, thậm chí có thể gây tử vong.\nTình trạng lắng đọng của protein amyloid trong bệnh thoái hóa tinh bột có thể khu trú ở bất kỳ cơ quan nào trong cơ thể, chẳng hạn như phổi, da, bàng quang hoặc ruột hay có thể là toàn thân. Đây cũng là dạng phổ biến nhất. Mặc dù bệnh thoái hóa tinh bột không được xếp thành một loại ung thư, bệnh lại có thể liên quan đến một số bệnh ung thư máu như đa u tủy.\nCó nhiều loại amyloidosis khác nhau, bao gồm những loại sau:\n•        Bệnh amyloidosis chuỗi nhẹ\n•        Bệnh amyloidosis tự miễn dịch\n    - Vị trí: LLQ\n    - Mức độ nghiêm trọng: đau bụng/khó chịu vùng bụng tăng dần\nthêm. Tùy vào loại amyloidosis mà sẽ có phương pháp điều trị khác nhau như:  Uống thuốc, hóa trị, ghép gan,…\nDo đó bạn nên tới gặp bác sĩ khám lại để có phương pháp điều trị phù hợp.\n',
    '80.txt': 'Câu hỏi từ người dùng:\n\nChào bác sĩ, ông em năm nay 70 tuổi, bị bệnh gút hai năm nay, cũng điều trị mấy tháng khỏi đau rồi không uống thuốc nữa, mà gần đây xuất hiện hạt nhỏ màu trắng trên da chỗ mấy khớp ngón tay với ngón chân. Cho em hỏi tại sao ông em bị như vậy, có bị nguy hiểm không, nhờ bác sĩ tư vấn giúp ạ?\nCâu trả lời của bác sĩ: \n\nChào bạn!\xa0\n\nVới những mô tả của bạn thì ông bạn đã bị bệnhgout, dạo gần đây không điều trị thường xuyên, và hình thành những hạt nhỏ ở vùng khớp như hình ảnh, đây là hạt tophi trong bệnh gout mạn tính.\nSự hình thành hạt tophi:\xa0Khi mới hình thành hạt tophi có màu trắng, nhỏ, mềm, có thể di động. Về sau, lượng axit urictăng nhanh, tích tụ nhiều sẽ khiến hạt lớn dần lên thành cục u cứng, cố định ở một vị trí.Tại vùng nổi hạt tophi có hiện tượng sưng, tấy đỏ. Tích tụ nhiều sẽ khiến cho những khối u này lớn dần lên, cản trở vận động. Nếu không được xử lý kịp thời, hạt tophi bị vỡ ra có thể gây hoại tử, biến dạng xương khớp, gây nhiễm trùng máu, rất khó lành\n    - Bệnh nhân xơ gan mất bù có tăng áp lực tĩnh mạch cửa, cổ trướng và tràn dịch màng phổi.\n    Thuốc trước khi nhập viện lần này: liệu pháp lợi tiểu\n2.  Tiền sử bệnh hiện tại\nChưa phát hiện bất thường\n3.  Đánh giá tại bệnh viện\n    Các thủ thuật đã thực hiện\n    - chọc dò màng phổi 3L4\n    - chọc dò dịch ổ bụng 7L\n\nĐể hạn chế sự tiến triển của các hạt tophi thì cần tuân thủ điều trị để hạ lượng acid uric máu, bạn cần đưa ông bạn đến khám chuyên khoa cơ xương khớp để được tư vấn và có lộ trình điều trị cụ thể.',
    '81.txt': 'Câu hỏi từ người dùng :\n\nChào bác sĩ em năm nay 27t đã có 1 bé gái. Sau 2 lần sảy thai thì em đi xét nghiệm máu bác sĩ nói là em bị hội chứng tăng đông hay còn gọi là gen đông máu tên y học là thrombophilia .bác sĩ nói nếu em muốn có con thì sau khi que thử 2 vạch báo nay để bác sĩ tiêm thuốc chống đông máu cho em. Em muốn hỏi là bệnh thrombophilia có nghiêm trọng lắm không ạ? Có ảnh hưởng đến sức khỏe và cuộc sống đời thường của người bệnh không ạ ? Nếu có thai tiêm thuốc chống đông máu có ảnh hưởng đến sức khỏe của mẹ và thai nhi nhiều không? Tiêm thuốc đó thì con em sinh ra có bình thường về hành vì và sức khỏe không ạ. Em cảm ơn và mong giải đáp từ bác sĩ ạ.\n\nCâu trả lời của bác sĩ: \n\nChào bạn\n- Thrombophilia là tên khoa học nói về tình trạng máu có xu hướng vón cục nên có người gọi là bệnh tăng đông máu. Nguyên nhân là do đột biến gây ra. Trong giai đoạn mang thai, để tránh tình trạng sẩy thai, sinh non, tiền sản giật....thì bác sỹ thường kê thuốc chống đông cho bạn. Thuốc chống đông này cần chú ý phải dừng ngay khi có dấu hiệu ra máu bất thường hoặc nên dừng trước tuần thai 36 để tránh tình trạng chảy máu khó cầm (việc này bạn nên theo hướng dẫn của bác sỹ sản khoa).\n- Bệnh có ảnh hưởng đến đến con bạn hay không thì phụ thuốc vào gen đột biến của bạn có truyền sang cho con bạn không. Nếu có thì con bạn cũng có thể bị tắc mạch do cục máu đông.\n    Các thủ thuật đã thực hiện: đặt shunt dẫn lưu tĩnh mạch cửa qua da',
    '82.txt': '1.  Tiền sử bệnh\n    Bệnh nhân có tiền sử dụng thuốc\n    - NSAID để điều trị đau đầu gối (đã ngừng)\n    - omeprazole (bắt đầu dùng)\n\n2.  Tiền sử bệnh hiện tại\n2. Tiền sử bệnh hiện tạiBệnh nhân nhập viện vì đau vùng hạ sườn phải kéo dài vài tuần nay.\nKhởi phát bệnh cách thời điểm nhập viện vài tuần với triệu chứng đau vùng hạ sườn phải, kèm chướng bụng và buồn nôn thoáng qua. \nBệnh nhân không ghi nhận nôn ói.Ngoài ra, bệnh nhân có tiền sử đau lưng kéo dài từ lâu, khu trú vùng cạnh cột sống bên trái đoạn giữa lưng, đôi khi lan ra phía trước bụng.\nĐau bụng chủ yếu khu trú vùng hạ sườn phải, xuất hiện nhiều sau ăn, kèm cảm giác chướng bụng. \nCác triệu chứng liên quan gồm buồn nôn thoáng qua, không nôn. Bệnh nhân cũng ghi nhận tình trạng đau lưng kéo dài như trên.Hiện tại, bệnh nhân nhập viện để tiếp tục được thăm khám, chẩn đoán và điều trị.\n    Lý do nhập viện: đau bụng vùng hạ sườn phải\n    Thời điểm khởi phát triệu chứng: Vài tuần\nthêm. Tùy vào loại amyloidosis mà sẽ có phương pháp điều trị khác nhau như:  Uống thuốc, hóa trị, ghép gan,…\nDo đó bạn nên tới gặp bác sĩ khám lại để có phương pháp điều trị phù hợp.\nChẩn đoán hình ảnh và thăm dò:Nội soi thực quản - dạ dày - tá tràng: viêm dạ dày.\nCộng hưởng từ mật tụy : sỏi đoạn cuối ống mật chủ.\nThủ thuật được thực hiện   :Nội soi mật tụy ngược dòng (ERCP). Trong quá trình ERCP, lấy thành công 02 viên sỏi tại đoạn cuối ống mật chủ (CBD).\n    - chụp cộng hưởng từ mật tụy tụi mật được thực hiện cho thấy sỏi ống dẫn mật chung đoạn cuối\n',
    '83.txt': '    - nhiều lần cố gắng giảm cân và tăng cân trở lại\nCâu trả lời của bác sĩ: Chào bạn,mình xin trả lời câu hỏi của bạn như sau Như bạn đã mô tả, trường hợp này bạn thuộc MÀY đay VÔ CĂN. Chúng tôi không tìm được nguyên nhân gây bệnh. Đây là tình trạng hay gặp nhất của MÀY đay MẠN. Tổ thương mô bệnh học chủ yếu là tình trạng xâm nhập của tế bào một nhân kết hợp với bạch huyết T và đại thực bào. Các loại tiền thưởng khó quan sát trên kính hiển thị vi quang học thông thường cần xem trên kính hiển thị vi điện tử. Các tế bào cột sống tăng lượng hạ bì với các mức độ thoát ra khác nhau cũng được quan sát. Quảng cáo quảng cáo thiết bị thương mại miễn phí dịch sinh học không tìm thấy hình ảnh giảm phức tạp, bổ sung hay sợi fibrin. Đối với bệnh lý mày đay vô căn, để lâu thường không có biến chứng gì nặng, chủ yếu bệnh làm ảnh hưởng tới chất lượng cuộc sống. Bạn đay vô căn hộ là một bệnh mạn tính, việc điều trị phải dài và liên tục. Quá giá trị chỉ là giấy chứng nhận. Điều trị phụ thuộc vào mức độ nghiêm trọng và nền tảng có liên kết. Cần lưu ý cân bằng giữa hiệu quả kiểm soát triệu chứng và những tác hại gây độc của liệu pháp điều trị. *************** có thể hệ sau không gây buồn ngủ là đơn hàng đầu sau đó đến *************** có thể hệ 1, ********* bậc thấp cách ngày hoặc hàng ngày và giảm dần chậm 2,5 – 5mg mỗi 2-3 tuần. Nếu không được hỗ trợ hoặc chống chỉ định, xin bạn hãy đến gặp bác sĩ chuyên khoa da liễu để có phương pháp điều trị tốt nhất. Cảm ơn bạn đã gửi câu hỏi',
    '84.txt': 'Câu hỏi từ người dùng :\n    Thuốc trước khi nhập viện: thuốc giảm đau opioid\n\nCâu trả lời của bác sĩ: \n\nChào bạn! Nấm bẹn là một dạng nhiễm nấm  thường là do Trichophyton rubrum hoặc là T. mentagrophytes. Các yếu tố nguy cơ chính liên quan đến môi trường ẩm ướt (thời tiết nóng, quần áo ướt và chật, chứng béo phì do có nhiều nếp gấp da). Bệnh do nhiễm nấm da mà thành nên để điều trị nấm bẹn bạn phải được sử dụng thuốc kháng nấm dạng bôi hoặc dạng uống theo chỉ định của bác sĩ da liễu với liệu trình điều trị từ 1-4 tuần. \nNgoài ra để giảm thiểu nguy cơ bị lây nhiễm, hoặc tái phát nấm bẹn bạn nên:\n•        Giữ vùng bẹn khô thoáng: nên lau khô vùng bẹn sau khi tắm hoặc tập thể dục để giữ cho vùng này được khô thoáng. Nếu cơ thể tiết quá nhiều mồ hôi, có thể sử dụng bột để giảm độ ẩm vùng bẹn.\n•        Mặc đồ sạch: bạn nên thay quần lót ít nhất một lần mỗi ngày và có thể thay nhiều hơn nếu cơ thể tiết quá nhiều mồ hôi. Nên giặt đồ thể thao sau khi đã sử dụng.\n•        Lựa chọn đồ lót vừa mặc: không nên chọn đồ lót quá chật, vì ma sát sẽ làm da vùng bẹn tổn thương, từ đó tạo điều kiện thuận lợi cho sự phát triển của vi nấm.\n•        Không dùng chung đồ dùng cá nhân: không nên cho người khác mượn áo quần, khăn tắm và các đồ dùng cá nhân khác của mình, cũng như tránh đi mượn đồ dùng cá nhân của người khác.\nLời khuyên: bạn nên trực tiếp đến khám và gặp bác sĩ da liễu khi có các vấn đề về da để được tư vấn và điều trị triệt để cũng như tránh “tiền mất tật mang”.\n',
    '85.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh mãn tính\n    - Tiểu đường loại 1 đái tháo đường\n    - tăng huyết áp\n    - tăng lipid máu, không đặc hiệu\n    - béo phì\n    - Nhiễm trùng đường tiết niệu (UTIs) tái phát nhiễm khuẩn đường tiết niệu, vị trí không xác định\n\n2.  Lịch sử bệnh hiện tại\n    Lý do nhập viện: đau bàn chân phải\n    Các triệu chứng hiện tại\n    - đau bàn chân phải\n    - mất thăng bằng khi đi lại\n    Các sự kiện trước khi nhập viện\n    - Gần đây nhập viện vì viêm bể thận và viêm phế quản\n-        Cách ly môi khỏi ánh sáng mặt trời bằng khẩu trang và các phương pháp bảo hộ.\n-        Tránh để bụi bẩn bám vào môi cũng như mọi va chạm, cọ xát mạnh lên khu vực da còn thương tổn này.\n-        Bổ sung nhiều thực phẩm có chưa vitamin C (cam, bưởi,…)\n    - Vẫn cảm thấy không vững khi đứng\n\n3.  Đánh giá tại bệnh viện\n    Kết quả xét nghiệm xét nghiệm\n    - hct (hematocrit) 8.126.3\n    - tiểu cầu (platelets) 478\n    - hco3- (bicarbonate) 20\n    - ag (anion gap) 13\n    - bun/creatinine (ure/creatinine)\n    - glucose (đường huyết) 316\n    - lactate (acid lactat) 1.3\n    - Kết quả ua (urinalysis - tổng phân tích nước tiểu): 12 bạch cầu, không vi khuẩn, 1 hồng cầu, âm tính nitrite\n    Kết quả chẩn đoán hình ảnh chẩn đoán hình ảnh\n    - chụp x-quang ngực cho thấy dòng picc đã đặt, không có quá trình bệnh lý tim phổi cấp tính\n    - chụp x-quang bàn chân phải không phát hiện gãy xương hoặc viêm xương tủy\n    - Bên phải âm tính với huyết khối tĩnh mạch sâu (DVT)',
    '86.txt': 'Câu hỏi từ người dùng:\n    - ho\n    - nghẹt ngực\n    - khó thở nhẹ-vừa khó thở khi gắng sức\n    - mệt mỏi\n\nCâu trả lời của bác sĩ:\n\nViêm hang vị sung huyết là tình trạng niêm mạc vùng hang vị dạ dày viêm, các mạch máu vùng viêm giãn nở do ứ máu nhiều. Biểu hiện chủ yếu là đau bụng cồn cào kèm theo ợ hơi, ợ chua, có thể có cảm giác buồn nôn hoặc nôn. Trước kia bệnh dạ dày được coi là bệnh nan y và nguy hiểm nhưng mấy chục năm gần đây nhờ nội soi phát triển nên bệnh đã được điều trị hiệu quả. Để điều trị dứt điểm bệnh viêm sung huyết hang vị dạ dày, trước hết, người bệnh cần được thăm khám chuyên khoa, nội soi dạ dày để tìm nguyên nhân, đánh giá mức độ bệnh. Căn cứ trên kết quả khám, các bác sĩ sẽ đưa ra phác đồ điều trị hiệu quả nhất cho từng bệnh nhân. Điều quan trọng là bệnh nhân cần tuân thủ tuyệt đối sự chỉ dẫn của bác sĩ trong quá trình điều trị. Không được bỏ thuốc giữa chừng cũng như không được tự ý tăng giảm liều lượng thuốc mà chưa có sự đồng ý của bác sĩ. Ngoài ra, người bị viêm sung huyết hang vị dạ dày có thể sử dụng sản phẩm từ thiên nhiên như nghệ, mật ong... để hỗ trợ điều trị. Bên cạnh đó, cần có chế độ ăn uống khoa học, lành mạnh. Nên kiêng ăn các thức ăn có vị chua, cay, nóng và các loại đồ ăn nhiều dầu mỡ. Không uống rượu, bia, nước có ga. Không hút thuốc lá, thuốc lào. Hạn chế uống cà phê... Khi ăn nên ăn chậm, nhai kỹ, ăn uống điều độ đúng giờ, đủ bữa, không ăn quá no hoặc để quá đói. Và cần có chế độ tập luyện phù hợp.',
    '87.txt': 'BN nam 74 tuổi vào viện vì lí do \nđau ngực \nTS; khỏe mạnh \nBệnh sử; cách vào viện khoảng 2 tháng, bn \nxuất hiện đau ngực T âm ỉ. đau không lan \nđợt này tình trạng đau ngực tăng lên, đau \ntăng khi gắng sức vào viện: \nKhám lúc vào: \nBn tỉnh \nĐau ngực T âm ỉ \nđau không lan \nkhông khó thở\nDa niêm mạc hồng \nKHông phù, không xuất huyết \nTim nhịp đều \nPhổi RRPN thô \nBụng mềm \nTiểu được \nHA: 110/ 70 mmHg               M: 70 l/p \nCận lâm sàng : \n\nXét nghiệm: \nThời gian prothrombin (PT: Prothrombin Time), \n(Các tên khác: TQ; Tỷ lệ Prothrombin) bằng máy \ntự động,; Định lượng Troponin Ths ; Đo hoạt độ AST \n(GOT); Đo hoạt độ ALT (GPT); Định lượng \nCreatinin (máu); Điện giải đồ (Na, K, Cl); Định \nlượng Glucose; Định lượng NT - proBNP ( \nProBNP); Tổng phân tích tế bào máu ngoại vi \n(bằng máy đếm laser); \nSiêu âm: \nSiêu âm Doppler tim, van tim; \nThủ thuật: Ghi điện tim cấp cứu tại giường - \n\nChẩn đoán: Cơn đau thắt ngực không ổn định-bệnh tăng HA vô căn(nguyên phát)\n\nThuốc điều trị : \n\n***************************** x 1,0 Viên\nNgày uống 1 viên buổi sáng sau ăn \n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: đến khám vì sau cân sau phẫu thuật\n    Các sự kiện trước khi nhập viện\n    - nhiều lần cố gắng giảm cân và tăng cân trở lại\n    - nhiều lần thực hiện chế độ ăn kiêng có giám sát với mức giảm cân tối đa 60 pound nhưng sau đó có  tăng cân trở lại\nNgày uống 1 viên buổi sáng sau ăn \n ***********************\n**************\nx 2,0 Viên\nNgày uống 2 viên buổi tối ',
    '88.txt': '1.  Tiền sử bệnh nội khoa\n    Tiền sử phẫu thuật / thủ thuật\n    - cắt bỏ ống dẫn mật chủ\n    - cắt bỏ phân đoạn bên trái gan\n    - Đã phẫu thuật nối ống gan - hỗng tràng vì nghi ngờ ung thư đường mật không thể cắt bỏ\n    - Đã can thiệp chụp đường mật, đặt 3 stent đường mật lưu dài hạn\n\n2.  Tiền sử bệnh hiện tại\n    Thời điểm khởi phát triệu chứng: Đêm trước khi nhập viện\n    Triệu chứng hiện tại: run rẩy kèm sốt kèm theo sốt\n    Các sự kiện trước khi nhập viện\n    - Gần đây nhập viện vì sốt và các ổ dịch trong ổ bụng được cho là áp xe\n    - Điều trị bằng kháng sinh tĩnh mạch trong thời gian nằm viện\n    - Ra viện với đơn Augmentin đường uống trong 10 ngày \n    - Tình trạng ổn định ở nhà cho đến đêm trước khi nhập viện\n3.  Đánh giá tại bệnh viện\n    Kết quả chụp chẩn đoán hình ảnh\n\nBé 14 tháng tuổi có cân nặng trong giới hạn bình thường và không có dấu hiệu gì bất thường khác có thể là 1. Do cơ địa người bé ra mồ hôi nhiều hơn trẻ khác. Thiếu một số vi chất cho cơ thể do mất cân đối trong chế độ ăn so với nhu cầu phát triển của trẻ như calcium, sắt, kẽm,….3. Mắc một bệnh tiềm ẩn chưa được phát hiện. Lời khuyên: gia đình cho cháu dùng nhiều rau quả, tắm nắng, ăn uống nhiều loại thức ăn hơn là ăn một số loại thức ăn bổ dưỡng.Cháu nên đăng ký khám bs nhi khoa để đánh giá và loại trừ một số bệnh tiềm ẩn.    - ba stent mật kim loại đã được đặt\n    - một ổ dịch sau phẫu thuật giảm kích thước so với lần chụp chẩn đoán hình ảnh  trước',
    '89.txt': 'Các bệnh lý mãn tính\n- tiền sử Đái tháo đường (tiền sử Đái tháo đường)\n- Tăng huyết áp (tăng huyết áp vô căn (nguyên phát))\n\n2. Tiền sử bệnh bệnh hiện tại\nLý do nhập viện: xét nghiệm gắng sức bất thường\nThời điểm khởi phát triệu chứng: Hôm qua (không rõ ngày cụ thể)\nCác triệu chứng hiện tại\n- khó chịu vùng ngực gián đoạn\n- chóng mặt\n- khó thở (khó thở)\n- đổ mồ hôi\nĐặc điểm triệu chứng\n- Vị trí: Không rõ (ngụ ý vùng ngực)\n- Mức độ nghiêm trọng: Không rõ\n- Thời gian: Không rõ\n- Tần suất: Gián đoạn\n- Chiếu r xạ: Không rõ\n- Các yếu tố làm nặng thêm: Hoạt động gắng sức\n- Các yếu tố làm giảm bớt: Nghỉ ngơi\n- Các triệu chứng liên quan: chóng mặt, khó thở, đổ mồ hôi\nCác sự kiện trước khi nhập viện: xét nghiệm gắng sức bất thường\n\n3. Đánh giá tại bệnh viện\nKết quả thăm khám lâm sàng\n- Không có đau tại phòng cấp cứu\n- điện tâm đồ bình thường tại phòng cấp cứu\n- Các chỉ số ban đầu tại phòng cấp cứu: Nhiệt độ 36.7°c, Huyết áp 139/68 mmhg, Mạch 67 lần/phút, Nhịp thở 16 lần/phút, SpO2 100% (không thở oxy) (nhiệt độ 36.7°c, huyết áp 139/68 mmhg, nhịp tim 67 lần/phút, nhịp thở 16 lần/phút, độ bão hòa oxy 100% trên khí trời)\nCác thủ thuật đã thực hiện: ống nội khí quản (nghiệm pháp gắng sức trên máy chạy bộ)\nCác phát hiện chẩn đoán khác\n2.        Liệu phát miễn dịch tiếp xúc (contac immunotherapy)\nĐoạn ST chênh xuống kiểu xuống dốc xuất hiện trong giai đoạn hồi phục, còn tồn tại sau 1 phút và kéo dài đến sau 5 phút hồi phục.\n',
    '90.txt': 'Câu trả lời của bác sĩ:\nChào bạn! Cảm ơn bạn đã gửi câu hỏi cho chúng tôi.\n    - rối loạn lo âu\n    - tăng huyết áp (tăng huyết áp)\n    - Táo bón mãn tính \n    - ngưng thở khi ngủ\n\n2. Bệnh sử hiện tại\n    Lý do nhập viện: nhịp thở nhanh và thiếu oxy\n    Theo lời kể , khoảng 2 tuần trước khi nhập viện bệnh nhân xuất hiện mệt mỏi, ho và chảy nước mũi.Nhiều thành viên trong gia đình có các triệu chứng tương tự. Bệnh nhân đã được bác sĩ gia đình khám vì các triệu chứng nhiễm trùng đường hô hấp trên.\n    - Tại phòng khám, bệnh nhân được ghi nhận có độ bão hòa oxy (SPO2) từ 88-92 % khi thở khí trời và nhịp thở nhanh.\n    - Bệnh nhân được chuyển đến phòng cấp cứu để đánh giá và điều trị thêm.\n    Lúc vào phòng cấp cứu trong tình trạng:\n    - mệt mỏi\n    - ho\n    - chảy nước mũi\n    - nhịp thở nhanh\n    - Thiếu oxy\n    - đau\n    Kết quả xét nghiệm\n    - bạch cầu 26.7\n    - bnp 4227\n    - kali 3.2\n    - troponin 0.01\n    - lactate 1.8\n    - tổng phân tích nước tiểu có kết quả dương tính  với 182bạch cầuvài vi khuẩn (vi khuẩn), nitrite\n    Kết quả chẩn đoán hình ảnh\n    - chẩn đoán hình ảnh không cho thấy quá trình bệnh lý tim mạch hoặc hô hấp cấp tính \n    - chụp x-quang ngực phù hợp với viêm phổi kẽ.\nĐã xử trí thuốc và thủ thuật\n    - Khí dung phối hợp Albuterol và Ipratropium 1 lần\n    - Ceftriaxone 1 gram dùng 1 liều\n    - Uống Tylenol 1 gram, 1 liều duy nhất.\n    - cấy máu 2 mẫu và cấy nước tiểu đã được gửi.',
    '91.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mãn tính\n    - hội chứng turner, không đặc hiệu\n    - Tăng huyết áp\n    - Tiền sử phẫu thuật cắt bỏ đại tràng do Ung thư đại tràng\n    - Tiền sử Thuyên tắc phổi đang dùng coumadin (tiền sử Thuyên tắc phổi đang dùng coumadin)\n    Tiền sử phẫu thuật / thủ thuật: Cắt đại tràng nhiều năm trước\n    Thuốc trước khi nhập viện: coumadin\n\n2.  Tiền sử bệnh hiện tại\n    Lý do nhập viện: khó thở\n    Các triệu chứng hiện tại\n    - khó thở \n    - Cảm giác có đờm ở cổ họng\n    - Cần phải ho nó ra\n    - Không sốt\n    - Không ớn lạnh\n    - Không đau ngực\n    - Không  khó thở\n    -Không  thay đổi tính chất cơn đau khi thay đổi tư thế\n    - Không khó thở khi gắng sức\n    - Không đánh trống ngực\n    - Không khó thở tăng lên khi xuất hiện  Cơn nhịp nhanh\n    - Không đau ngực \n    Các diễn biến trước khi nhập viện\n    - Gần đây nhập viện để đánh giá tổng quát về tiêu hóa\n    - đánh giá tổng quát về tiêu hóa vì cảm thấy  khó nuốt\n3.  Đánh giá tại bệnh viện\n    Các thủ thuật đã thực hiện\n    - albuterolipratropium nebs x2\nSỏi niệu quản lớn hơn 7mm thường không điều trị nội khoa được. Nếu điều trị nội khoa 1 tháng không ra được, cần được can thiệp phẫu thuật, nếu để lâu hơn sẽ làm tổn thương chức năng thận nhiều. Bạn nên đưa chồng tới bệnh viện khám và tiến hành phẫu thuật.    - hr cải thiện lên mức 110 sau khi dùng metoprolol\n    - Bệnh nhân cho biết  cảm thấy khỏe khi đến phòng khám\n',
    '92.txt': '1. Tiền sử bệnh\n- viêm tủy xương mãn tính và bàng quang thần kinh gây biến chứng liệt hai chi dưới và loét tì đè giai đoạn IV mãn tính\n- đặt lưu sonde tiểu    gây nhiễm khuẩn đường tiết niệu tái phát\nThuốc đã dùng trước đây\n- Đã sử dụngvancozosyn nhưng hiện tại đang dừng. Sau khi khám chuyên khoa Truyền nhiễm đang dùng bactrim để điều trị nhiễm khuẩn đường tiết niệu\n\n2. Bệnh sử hiện tại\nLý do nhập viện: sốt cao\nCách ngày vào viện 2 tuần bệnh nhân đã vào viện vì hạ huyết áp và được chẩn đoán  nhiễm trùng huyết điều trị bằng kháng sinh  vancozosynbactrim  điều trị  bệnh viêm tuỷ xương.  Sau đó dùng zosyn 8 ngày và dừng tất cả kháng sinh để điều trị viêm tủy xương mãn tính tại khoa Hồi sức tích cực. Ra viện và tái khám không sử dụng thuốc kháng sinh vì khám lâm sàng ổn\n Sáng cùng ngày vào viện bệnh nhân xuất hiện sốt cao 39.7 độ C , vào viện điều trị.\nTình trạng lúc vào:\nBệnh nhân tỉnh\nCảm giác khát nước\nDa khô, nếp véo da mất chậm\nKhông ngực,  không khó thở\nĐau bụng, đau hông từng cơn-\nHạ huyết áp\nMạch nhanh \nTim nhịp nhanh đều\nTrứng cá bắt nguồn từ bốn yếu tố chính:\n\nTăng sản tuyến bã nhờn\n- Xquang ngực thẳng: không thấy thâm nhiễm\nThủ thuật:\n- svo2 là 82\nĐặt ống thông tĩnh mạch trung tâm : đo cvp là 6\nChẩn đoán: nhiễm trùng huyết đường vào tiết niệu- viêm tủy xương mãn tính,\n\nĐiều trị: \n- Truyền dịch : 4000 ml NS 0.9 %\n- Kháng sinh Cefepim và Vancomycin truyền tĩnh mạch\n\n.\n\n \n',
    '93.txt': 'Câu hỏi từ người dùng :\n\nem năm nay 22 tuổi, đợt vừa rồi có đi khám sức khỏe có đo HA là 160/70 mmHg, sau đó đo lại là HA: 170/60 mmHg, đo lại lần nữa là HA:  150/60 mmHg, nhưng lúc đó em cảm thấy hồi hợp, cảm giác rợn người, em có mua máy đo HA điện tử về nhà, tự đo thì HA bình thường, em đo rất nhiều lần, em học y nên kĩ thuật có thể nói là chính xác, nhưng cứ có người đo cho em thì nó HA nhịp tim đều tăng và có cùng một cảm giác rợn người, hồi hộp, cảm nhận được tim đập rất mạnh, em có đi khám ở ĐHYD làm rất nhiều XN nhưng tất cả đều bình thường và không tìm ra nguyên nhân, sau đó bs có hẹn 1 tuần tái khám nhưng do dịch nên không đi được, em muốn hỏi, tình trạng của em có phải là tăng HA thật sự không ạ? Em cảm thấy rất hoang mang, bởi vì khi tự đo, ngồi một mình, thì cảm giác đó không có và huyết áp của em mỗi lần đo đều bình thường. Em cảm ơn ạ\n\nCâu trả lời của bác sĩ: \n\nChào bạn! Huyết áp có thể tạm thời thay đổi trong một số hoàn cảnh như:\n\n- Khi ở trong tâm trạng lo âu căng thẳng thì huyết áp tăng lên đáng kể và sẽ trở lại bình thường sau khi thoải mái thư giãn. Vì thế khi đi khám bệnh huyết áp thường hơi cao hơn khi đo ở nhà do đó nên nghỉ vài phút trước khi đo.\n-Nói chuyện khi đo, huyết áp cũng lên cao. Vì thế nên giữ im lặng trong khi đo.\n    Kết quả chẩn đoán hình ảnh: chụp ctchưa phát hiện bất thường trên phim chụp\n    Các kết quả  khác: Theo dõi đại tràng giãn',
    '94.txt': 'Cận lâm sàng\n   Siêu âm tim: chèn ép tim\n  Viêm hang vị sung huyết là tình trạng niêm mạc vùng hang vị dạ dày viêm, các mạch máu vùng viêm giãn nở do ứ máu nhiều. Biểu hiện chủ yếu là đau bụng cồn cào kèm theo ợ hơi, ợ chua, có thể có cảm giác buồn nôn hoặc nôn. Trước kia bệnh dạ dày được coi là bệnh nan y và nguy hiểm nhưng mấy chục năm gần đây nhờ nội soi phát triển nên bệnh đã được điều trị hiệu quả. Để điều trị dứt điểm bệnh viêm sung huyết hang vị dạ dày, trước hết, người bệnh cần được thăm khám chuyên khoa, nội soi dạ dày để tìm nguyên nhân, đánh giá mức độ bệnh. Căn cứ trên kết quả khám, các bác sĩ sẽ đưa ra phác đồ điều trị hiệu quả nhất cho từng bệnh nhân. Điều quan trọng là bệnh nhân cần tuân thủ tuyệt đối sự chỉ dẫn của bác sĩ trong quá trình điều trị. Không được bỏ thuốc giữa chừng cũng như không được tự ý tăng giảm liều lượng thuốc mà chưa có sự đồng ý của bác sĩ. Ngoài ra, người bị viêm sung huyết hang vị dạ dày có thể sử dụng sản phẩm từ thiên nhiên như nghệ, mật ong... để hỗ trợ điều trị. Bên cạnh đó, cần có chế độ ăn uống khoa học, lành mạnh. Nên kiêng ăn các thức ăn có vị chua, cay, nóng và các loại đồ ăn nhiều dầu mỡ. Không uống rượu, bia, nước có ga. Không hút thuốc lá, thuốc lào. Hạn chế uống cà phê... Khi ăn nên ăn chậm, nhai kỹ, ăn uống điều độ đúng giờ, đủ bữa, không ăn quá no hoặc để quá đói. Và cần có chế độ tập luyện phù hợp.',
    '95.txt': 'Câu hỏi từ người dùng:\n\nChào bác sĩ, ông em năm nay 70 tuổi, bị bệnh gút hai năm nay, cũng điều trị mấy tháng khỏi đau rồi không uống thuốc nữa, mà gần đây xuất hiện hạt nhỏ màu trắng trên da chỗ mấy khớp ngón tay với ngón chân. Cho em hỏi tại sao ông em bị như vậy, có bị nguy hiểm không, nhờ bác sĩ tư vấn giúp ạ?\nCâu trả lời của bác sĩ: \n\nChào bạn!\xa0\n\nVới những mô tả của bạn thì ông bạn đã bị bệnhgout, dạo gần đây không điều trị thường xuyên, và hình thành những hạt nhỏ ở vùng khớp như hình ảnh, đây là hạt tophi trong bệnh gout mạn tính.\nSự hình thành hạt tophi:\xa0Khi mới hình thành hạt tophi có màu trắng, nhỏ, mềm, có thể di động. Về sau, lượng axit urictăng nhanh, tích tụ nhiều sẽ khiến hạt lớn dần lên thành cục u cứng, cố định ở một vị trí.Tại vùng nổi hạt tophi có hiện tượng sưng, tấy đỏ. Tích tụ nhiều sẽ khiến cho những khối u này lớn dần lên, cản trở vận động. Nếu không được xử lý kịp thời, hạt tophi bị vỡ ra có thể gây hoại tử, biến dạng xương khớp, gây nhiễm trùng máu, rất khó lành\n\nCó những trường hợp acid uric tron hạt tophi giải phóng vào máu gây cơn gout cấp, đau các khớp rất nhiều.\n\n1.  Tiền sử bệnh lý\n    Các bệnh lý mạn tính\n    - ho đái tháo đường\n    - tăng huyết áp\n    - béo phì\n    - ngưng thở khi ngủ do tắc nghẽn\n    Thuốc trước khi nhập viện\n    - tylenol\n    - mucinex d\n    - thỉnh thoảng tiêu chảy các thuốc của ông ấy\n\n',
    '96.txt': 'Câu hỏi từ người dùng:\n\n    Sự kiện trước khi nhập viện: chụp cắt lớp vi tính (ct) được thực hiện cho thấy tắc hẹp 80% động mạch thận trái L\n\n3.  Đánh giá tại bệnh viện\n    Kết quả chẩn đoán hình ảnh: chụp cắt lớp vi tính (ct) cho thấy tắc hẹp 80% động mạch thận trái L\n    Thủ thuật thực hiện\nChào bạn!\nKhi phun xăm làn da nhạy cảm ở môi phải chịu một thương tổn không nhỏ từ các thao tác phun xăm. Tạm thời ngay sau thẩm mỹ nó chưa thể hồi phục bình thường được. Do đó, bất kì tác động mạnh nào từ bên ngoài cũng đều có thể khiến đôi môi bị thương tổn và gặp phải những biến chứng nặng nề.\nMôi bong vẩy trắng có thể do sau phun môi, môi khô và nhiều  da chết do đó bạn nên tăng cường dưỡng ẩm cho môi và tuyệt đối không cậy, bóc vảy môi.\nThông thường, ngay sau khi phun xăm môi chưa lên màu ngay mà cần khoảng thời gian nhất định để môi ổn định và sẽ lên màu dần dần (khoảng từ 1-2 tháng). Ngoài ra việc  lên màu môi còn phụ thuốc vào sắc tố da của mỗi người và kỹ thuật của người thao tác phun xăm. Bạn có thể áp dụng một số biện pháp sau để môi lên màu đẹp hơn:\n-        Cách ly môi khỏi ánh sáng mặt trời bằng khẩu trang và các phương pháp bảo hộ.\n-        Tránh để bụi bẩn bám vào môi cũng như mọi va chạm, cọ xát mạnh lên khu vực da còn thương tổn này.\n-        Bổ sung nhiều thực phẩm có chưa vitamin C (cam, bưởi,…)\n',
    '97.txt': '1.  Tiền sử bệnh\n    Các bệnh lý mạn tính\n    - Suy thận mạn giai đoạn V do đái tháo đường và tăng huyết áp\n    - u ác của tuyến tiền liệt điều trị bằng xạ trị cấy hạt \n    Tiền sử phẫu thuật / thủ thuật\n    - Phẫu thuật cắt bỏ u dây thần kinh số VIII \n    - Đặt shunt động tĩnh mạch (AVF) ở tay phải (RUE AVF)\n    - Sinh thiết tuyến tiền liệt\n2.  Bệnh sử hiện tại\n    Lý do nhập viện:  mệt mỏi, mất trí nhớ chi tiết\nBệnh nhân kể cảm thấy khó chịu mệt mỏi nhiều, ăn không ngon miệng, ngứa da toàn thân nhiều, và mất trí nhớ chi tiết, khó thở khi gắng sức. Tuần qua có buồn nôn, và nôn .\nKhám hiện tại thấy\nBệnh nhân tỉnhi\n    - Mệt mỏi nhiều\n    - Ăn không ngon miệng\n    - Ngứa da toàn thân nhiều\n    - mất trí nhớ chi tiết\nNhưng đợt này chỉ nồi to ra thôi ạ\nEm bị mấy năm nay rồi ạ?\n\nCâu trả lời của bác sĩ: \n\nChào bạn!\nĐiều trị bệnh trĩ theo tây y có hai cách nội khoa và ngoại khoa\nNội khoa dùng thuốc ************ uống 1-2 viên x2 sáng chiều\n  Ure tăng từ 69 lên 91 mg/dl ( 24.6 -32.5 mmol/l)trong 2 tháng qua,  photpho 8.4\nĐiều trị: \n    -  Bệnh nhân đã  hoàn thành quá trình đánh giá trước ghép thận nhưng tạm hoãn ghép thận do chẩn đoán u ác của tuyến tiền liệtAnh ấy có một vài người hiến tiềm năng, nhưng anh ấy không muốn nhận thận từ bất kỳ ai trong số họ\n    - Điều trị bắt đầu chạy thận nhân tạo (HD)\n',
    '98.txt': '1.  Tiền sử bệnh\n\nMặt tôi bị tàn nhang, da sạm, tôi nghe nói bôi nghệ mật ong, uống nghệ giúp sang da, bác sĩ cho hỏi lời khuyên trên đúng không, áp dụng sao cho hiệu quả?\n\nCâu trả lời của bác sĩ:\n\n  - Mẹ Đã tử vong trong lần nhập viện gần đây nhất\n    Các bệnh mãn tính\n    - Rối loạn cảm xúc \n    - Rối loạn lưỡng cực\n    - rối loạn lo âu\n    Thuốc đã điều trị trước khi nhập viện lần này\n    - Thuốc kê đơn (tên không được chỉ định)\n    - klonopinclonidine (được kê đơn)\n    - clonidine (mua trên đường phố)\n    - suboxone (bắt đầu 3 tuần trước), không dùng đều 5-6 ngày, dừng thuốc trước nhập viện 01 ngày\n    Các yếu tố nguy cơ liên quan\n    - Lạm dụng chất kích thích, chất gây nghiện opioid\n2.   Bệnh sử hiện tại\n    Lý do nhập viện:i ý định tự tử và ý nghĩ tự bắn vào đầu\n   Theo lời kể bệnh nhân bị trầm cảm ngày càng nặng , vài tháng trước có các giai đoạn giống hưng cảm  biểu hiện bằng tăng hoạt động và giảm nhu cầu ngủ 3-4 ngày, trong vài tuần nay và có ý định tự tử.  Thuốc kê đơn không dùng đều đặn x5-6 ngày, bắt đầu dùng suboxone 3 tuần trước vì rẻ hơn, dừng dùng suboxone ngày hôm qua\n    Triệu chứng khi vào viện\n    - ý nghĩ tự tử, có nghĩ tự bắn vào đầu\n    - lo âu\nhoảng sợ\n    - hoang tưởng như đang chiếm khí oxy của người khác\n Điều trị\n    - Hiện đang được chăm sóc tâm thần bởi bác sĩ\n\n',
    '99.txt': '1.  Tiền sử bệnh nội khoa\n    Các bệnh lý mãn tính: viêm tủy xương và bàng quang thần kinh có biến chứng liệt hai chi dưới\n\n2.  Bệnh sử hiện tại\n    Lý do nhập viện: biến đổi ý thức kèm theo hạ thân nhiệt và hạ huyết áp\n    Thời điểm khởi phát triệu chứng: Được ghi nhận bởi đội cấp cứu trên đường đến Khoa Cấp cứu\nTrước khi nhập viện, bệnh nhân được phát hiện trong tình trạng biến đổi ý thức, hạ thân nhiệt, hạ huyết áp với huyết áp tâm thu là 90\n    Các triệu chứng hiện tại\n    - biến đổi ý thức\n    - hạ thân nhiệt\nKhó thở khi nằm đầu bằng, SpO2 99% thở  khí 2L/ph \nTim nhịp không đều, tần số 105 chu  kì/phút \nT1, T2 rõ, không có tiếng tim bệnh lí\nHuyết áp 110/70  mmHg \nPhổi rì rào phế nang giảm, không có tiếng rales bệnh lí\nBụng mềm, không chướng \nKhông có điểm đau, gan lách không sờ thấy \nKhông phù, tiểu được không rõ số lượng\nChẩn đoán : : Đợt cấp COPD - Tâm phế mạn - Cơn tim nhanh nhĩ - Nhiễm khuẩn đường tiêu hóa - Tăng huyết áp \n    Kết quả chẩn đoán hình ảnh\n    - chẩn đoán hình ảnh: chụp x-quang ngực cho thấy không có hình ảnh tổn thương viêm cấp tính so với chụp x-quang ngực trước đó\n    Các thủ thuật đã thực hiện\n    - Đặt sonde bàng quang\n    - Gửi xét nghiệm máu và cấy nước tiểu\n    Các thủ thuậ khác\n    - điện tâm đồ cho thấy nhịp chậm xoang \n- đường huyết lúc đóiđường huyết thấp)',
    '100.txt': 'Câu hỏi từ người dùng :\n\nChào Bác sĩ, hiện tại em đang mang thai được 22 tuần. Vào tuần thứ 18 em có làm xét nghiệm thì có kết quả nguy cơ tiền sản giật cao, Bác sĩ kê đơn thuốc ************ sử dụng mỗi tối.Từ đó đến giờ, mũi em thường xuyên có cục máu đông và một lần đi tiêu ra máu. Em không biết là do người em nóng hay do tác dụng phụ của thuốc và có cần đi thăm khám ngay không ạ? Mong Bác sĩ giải đáp. Em xin cảm ơn!\n\nCâu trả lời của bác sĩ: \n\nChào bạn, \n\n************ được khuyến cáo đối với những sản phụ có nguy cơ tiền sản giật cao để phòng ngừa tiền sản giật và những biến chứng liên quan đến tiền sản giật. Tuy nhiên, ******* là thuốc có tác dụng chống đông máu do đó tác dụng phụ của thuốc là có thể gây chảy máu. Các triệu chứng của bạn có thể là tác dụng phụ của ******* tuy nhiên bạn cũng đừng nên quá lo lắng vì dựa trên mô tả thì những tác dụng phụ không phải ở mức độ nghiêm trọng.\nBạn vẫn nên đi khám sớm nhất có thể và liệt kê các triệu chứng  này với Bác sĩ chuyên khoa. Bác sĩ sẽ so sánh nguy cơ, lợi ích của việc tiếp tục sử dụng aspirin và cho bạn chỉ định tiếp tục hay ngưng dùng thuốc. \n\n2. Bệnh sử hiện tại\nLý do nhập viện: 3 ngày đại tiện ra máu đỏ tươi gián đoạn\nThời điểm khởi phát triệu chứng: 3 ngày\n',
}

print(f"input nhúng sẵn: {len(INPUT_FILES)} tệp")

In [ ]:
# Đường dẫn Dataset đã biết, thử trước để khỏi quét toàn bộ /kaggle/input.
KNOWN_INPUT_DIRS = [
    Path("/kaggle/input/datasets/thanhhiepvo/viettelairace/input"),
    Path("/kaggle/input/viettelairace/input"),
    Path("/kaggle/input/viettelairace"),
]

def is_input_dir(folder):
    return folder.is_dir() and (folder / "1.txt").exists() and (folder / "100.txt").exists()

def find_attached_input():
    for folder in KNOWN_INPUT_DIRS:
        if is_input_dir(folder):
            return folder
    base = Path("/kaggle/input")
    if base.exists():
        for path in base.rglob("1.txt"):
            if is_input_dir(path.parent):
                return path.parent
    return None

INPUT_DIR = find_attached_input()
INPUT_SOURCE = "Dataset đã attach"

if INPUT_DIR is None:
    INPUT_SOURCE = "BẢN NHÚNG trong notebook (public test Vòng 1)"
    target = WORK / "input_embedded"
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    for name, body in INPUT_FILES.items():
        (target / name).write_text(body, encoding="utf-8")
    INPUT_DIR = target

n = len(list(INPUT_DIR.glob("*.txt")))
print("nguồn input:", INPUT_SOURCE)
print("đường dẫn  :", INPUT_DIR, f"({n} tệp)")
assert n == 100, f"Cần đúng 100 tệp, thấy {n}"
if INPUT_DIR == WORK / "input_embedded":
    print("\n>>> Đang dùng public test nhúng sẵn. Nếu đây là lần chạy cho PRIVATE TEST,")
    print(">>> hãy attach Dataset input của BTC rồi chạy lại cell này. <<<")

## 5. Knowledge base

* **ICD-10 tiếng Việt** — dựng từ Phụ lục TT06/2026/TT-BYT (Bộ Y tế) và **nhúng
  sẵn** trong notebook. Đây là thay đổi quan trọng nhất ở phần gán mã: KB cũ là
  bản tiếng Anh của CDC nên một mention như `viêm túi mật` không thể khớp alias
  nào.
* **RxNorm** — Current Prescribable Content của NLM. Không nhúng được vì là dữ
  liệu của bên thứ ba, nên cell dưới tự tải khi bật Internet. Thiếu nó thì
  candidates `THUỐC` rỗng, pipeline vẫn chạy.

In [ ]:
# /kaggle/input là READ-ONLY, nên mọi thứ dựng ra phải nằm ở /kaggle/working.
# %%writefile KHÔNG tự tạo thư mục cha, nên phải mkdir trước.
TERM_DIR = WORK / "terminology"
TERM_DIR.mkdir(parents=True, exist_ok=True)
ICD_TSV = TERM_DIR / "icd10_vn.tsv"
print("thư mục terminology:", TERM_DIR)

In [ ]:
%%writefile /kaggle/working/terminology/icd10_vn.tsv
code	label	aliases
A00	Bệnh tả	
A00.0	Bệnh tả do vi khuẩn Vibrio cholerae 01, típ sinh học cholerae	Bệnh tả cổ điển
A00.1	Bệnh tả do Vibrio cholerae 01, típ sinh học eltor	Bệnh tả eltor
A00.9	Bệnh tả, không xác định	
A01	Bệnh thương hàn và/hoặc bệnh phó thương hàn	
A01.0	Bệnh thương hàn	Nhiễm trùng do vi khuẩn Salmonella typhi
A01.1	Bệnh phó thương hàn A	
A01.2	Bệnh phó thương hàn B	
A01.3	Bệnh phó thương hàn C	
A01.4	Bệnh phó thương hàn, không xác định	Nhiễm trùng do vi khuẩn Salmonella paratyphi không xác định khác
A02	Nhiễm Salmonella khác	
A02.0	Viêm ruột do Salmonella	Nhiễm Salmonella
A02.1	Nhiễm trùng hệ thống do Salmonella	
A02.2	Nhiễm Salmonella khu trú	
A02.8	Nhiễm trùng Salmonella xác định khác	
A02.9	Nhiễm trùng Salmonella, không xác định	
A03	Bệnh lỵ trực khuẩn	
A03.0	Bệnh lỵ trực khuẩn do Shigella dysenteriae	Bệnh lỵ trực khuẩn nhóm A [lỵ do Shiga-Kruse]
A03.1	Bệnh lỵ trực khuẩn do Shigella flexneri	Bệnh lỵ trực khuẩn nhóm B
A03.2	Bệnh lỵ trực khuẩn do Shigella boydii	Bệnh lỵ trực khuẩn nhóm C
A03.3	Bệnh lỵ trực khuẩn do Shigella sonnei	Bệnh lỵ trực khuẩn nhóm D
A03.8	Bệnh lỵ trực khuẩn do Shigella khác	
A03.9	Bệnh lỵ trực khuẩn, không xác định	Bệnh lỵ trực khuẩn hình que không xác định khác
A04	Nhiễm trùng đường ruột do vi khuẩn khác	
A04.0	Nhiễm khuẩn Escherichia coli gây bệnh đường ruột	
A04.1	Nhiễm khuẩn Escherichia coli gây độc tố ruột (ETEC)	
A04.2	Nhiễm khuẩn Escherichia coli xâm nhập (EIEC)	
A04.3	Nhiễm khuẩn Escherichia coli gây xuất huyết đường ruột (EHEC)	
A04.4	Nhiễm khuẩn Escherichia coli đường ruột khác	Viêm ruột do Escherichia coli không xác định khác
A04.5	Viêm ruột do vi khuẩn Campylobacter	
A04.6	Viêm ruột do vi khuẩn Yersinia enterocolitica	
A04.7	Viêm ruột do vi khuẩn Clostridium difficile	Nhiễm độc thực phẩm do Clostridium difficile|Viêm đại tràng giả mạc
A04.8	Nhiễm trùng đường ruột do vi khuẩn xác định khác	
A04.9	Nhiễm trùng đường ruột do vi khuẩn, không xác định	Viêm ruột do vi khuẩn không xác định khác
A05	Ngộ độc thực phẩm do vi khuẩn khác, không phân loại mục khác	
A05.0	Ngộ độc thực phẩm do độc tố tụ cầu	
A05.1	Ngộ độc botulinum	Nhiễm độc thực phẩm cổ điển do độc tố của Clostridium botulinum
A05.2	Ngộ độc thực phẩm do Clostridium perfringens [Clostridium welchii]	Viêm ruột hoại tử|Pig-bel [viêm ruột non hoại tử]
A05.3	Ngộ độc thực phẩm do Vibrio Parahaemolyticus	
A05.4	Ngộ độc thực phẩm do Bacillus cereus	
A05.8	Ngộ độc thực phẩm do vi khuẩn xác định khác	
A05.9	Ngộ độc thực phẩm do vi khuẩn, không xác định	
A06	Bệnh lỵ a-míp	
A06.0	Bệnh lỵ a-míp cấp tính	Bệnh lỵ a-míp đường ruột không xác định khác
A06.1	Bệnh lỵ a-míp đường ruột mạn tính	
A06.2	Viêm đại tràng do a-míp không gây hội chứng lỵ	
A06.3	U a-míp đường ruột	U a-míp không xác định khác
A06.4†	Áp xe gan do a-míp (K77.0*)	Bệnh a-míp ở gan
A06.5†	Áp xe phổi do a-míp	
A06.6†	Áp xe não do a-míp (G07*)	
A06.7	Nhiễm a-míp ở da	
A06.8	Nhiễm a-míp ở vị trí khác	
A06.9	Bệnh do a-míp, không xác định	
A07	Bệnh đường ruột do đơn bào khác	
A07.0	Bệnh đường ruột do nhiễm balantidium	Bệnh lỵ do Balantidium
A07.1	Bệnh đường ruột do nhiễm giardia [lamblia]	
A07.2	Bệnh đường ruột do nhiễm cryptosporidium	
A07.3	Bệnh đường ruột do nhiễm lsospora	Nhiễm Isospora belli và/hoặc Isospora hominis|Nhiễm coccidia đường ruột|Nhiễm Isospora
A07.8	Bệnh đường ruột do đơn bào xác định khác	Bệnh do nhiễm trichomonas đường ruột|Bệnh do nhiễm Sarcocystosis|Bệnh do nhiễm Sarcosporidiosis
A07.9	Bệnh đường ruột do đơn bào, không xác định	Tiêu chảy do trùng roi
A08	Nhiễm trùng đường ruột do virus và/hoặc tác nhân xác định khác	
A08.0	Viêm ruột do virus rota	
A08.1	Bệnh lý dạ dày - ruột cấp tính do norovirus	Viêm ruột do nhiễm norovirus|Viêm ruột do virus có cấu trúc tròn nhỏ
A08.2	Viêm ruột do virus adeno	
A08.3	Viêm ruột do virus khác	
A08.4	Nhiễm trùng đường ruột do virus, không xác định	
A08.5	Nhiễm trùng đường ruột xác định khác	
A09	Viêm dạ dày - ruột và/hoặc đại tràng do nhiễm trùng và/hoặc không xác định căn nguyên	Viêm dạ dày-ruột và/hoặc đại tràng do nhiễm trùng và/hoặc không xác định căn nguyên
A09.0	Viêm dạ dày - ruột và/hoặc đại tràng do nhiễm trùng khác và/hoặc không xác định	
A09.9	Viêm dạ dày - ruột và/hoặc đại tràng không xác định căn nguyên	
A15	Bệnh lao hô hấp, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A15.0	Bệnh lao phổi, được khẳng định bằng soi đờm có nuôi cấy hoặc không nuôi cấy đờm	
A15.1	Bệnh lao phổi, chỉ được khẳng định bằng nuôi cấy	Bệnh lý được liệt kê trong A15.0, chỉ được khẳng định bằng nuôi cấy đờm
A15.2	Bệnh lao phổi, được khẳng được bằng mô học	Bệnh lý được liệt kê trong A15.0, chỉ được khẳng định bằng mô học
A15.3	Bệnh lao phổi, được khẳng định bằng phương pháp không xác định	
A15.4	Bệnh lao hạch lympho trong khoang ngực, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A15.5	Bệnh lao thanh quản, khí quản và/hoặc phế quản, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A15.6	Bệnh lao màng phổi, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A15.7	Bệnh lao hô hấp sơ nhiễm, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A15.8	Bệnh lao hô hấp khác, được khẳng định bằng vi khuẩn học và/hoặc mô học	Bệnh lao trung thất được khẳng định bằng vi khuẩn học và/hoặc mô học|Bệnh lao vùng hầu mũi được khẳng định bằng vi khuẩn học và/hoặc mô học
A15.9	Bệnh lao hô hấp không xác định, được khẳng định bằng vi khuẩn học và/hoặc mô học	
A16	Bệnh lao hô hấp, không được khẳng định bằng vi khuẩn học hoặc mô học	
A16.0	Bệnh lao phổi, âm tính bằng vi khuẩn học và/hoặc mô học	
A16.1	Bệnh lao phổi, không xét nghiệm vi khuẩn học và/hoặc mô học	Bệnh lý được liệt kê ở A16.0, không thực hiện xét nghiệm vi khuẩn học và/hoặc mô học
A16.2	Bệnh lao phổi, không đề cập đến việc khẳng định bằng vi khuẩn hoặc mô học	
A16.3	Bệnh lao hạch lympho trong khoang ngực, không đề cập đến việc khẳng định bằng vi khuẩn học hoặc mô học	
A16.4	Bệnh lao thanh quản, khí quản và/hoặc phế quản, không đề cập đến việc khẳng định bằng vi khuẩn học hoặc mô học	
A16.5	Bệnh lao màng phổi, không đề cập đến việc khẳng định bằng vi khuẩn học hoặc mô học	
A16.7	Bệnh lao hô hấp sơ nhiễm không đề cập đến việc khẳng định bằng vi khuẩn học hoặc mô học	
A16.8	Bệnh lao hô hấp khác, không đề cập đến việc khẳng định bằng vi khuẩn học hoặc mô học	
A16.9	Bệnh lao hô hấp không xác định, không đề cập đến việc đã được khẳng định bằng vi khuẩn học hoặc mô học	Bệnh lao hô hấp không xác định khác|Bệnh lao không xác định khác
A17	Bệnh lao hệ thần kinh	
A17.0†	Viêm màng não do bệnh lao (G01*)	
A17.1†	U lao màng não (G07*)	U lao màng não
A17.8†	Bệnh lao khác của hệ thần kinh	
A17.9†	Bệnh lao hệ thần kinh, không xác định (G99.8*)	
A17.†	Bệnh lao hệ thần kinh	
A18	Bệnh lao ở cơ quan khác	
A18.0†	Bệnh lao xương và/hoặc khớp	
A18.1†	Bệnh lao ở hệ tiết niệu sinh dục	
A18.2	Bệnh lao hạch bạch huyết ngoại biên	
A18.3	Bệnh lao ở ruột, phúc mạc và/hoặc mạc treo ruột	
A18.4	Bệnh lao ở da và/hoặc mô dưới da	
A18.5	Bệnh lao ở mắt	
A18.6	Bệnh lao ở tai	
A18.7†	Bệnh lao tuyến thượng thận (E35.1*)	Bệnh Addison, do bệnh lao
A18.8	Bệnh lao ở cơ quan xác định khác	
A19	Bệnh lao kê	
A19.0	Bệnh lao kê cấp tính của một vị trí xác định	
A19.1	Bệnh lao kê cấp tính của nhiều vị trí	
A19.2	Bệnh lao kê cấp tính, không xác định	
A19.8	Bệnh lao kê khác	
A19.9	Bệnh lao kê, không xác định	
A20	Bệnh dịch hạch	
A20.0	Bệnh dịch hạch thể hạch	
A20.1	Bệnh dịch hạch thể viêm tế bào da	
A20.2	Bệnh dịch hạch thể phổi	
A20.3	Bệnh dịch hạch thể viêm màng não	
A20.7	Bệnh dịch hạch thể nhiễm khuẩn huyết	
A20.8	Bệnh dịch hạch thể khác	Dịch hạch thể không phát triển|Dịch hạch không triệu chứng|Dịch hạch thể nhẹ
A20.9	Bệnh dịch hạch, không xác định	
A21	Bệnh Tularaemia	
A21.0	Bệnh Tularaemia thể hạch loét	
A21.1	Bệnh Tularaemia thể hạch mắt	Bệnh tularaemia ở mắt
A21.2	Bệnh Tularaemia thể phổi	
A21.3	Bệnh Tularaemia thể dạ dày - ruột	Bệnh tularaemia ở bụng
A21.7	Bệnh Tularaemia toàn thân	
A21.8	Bệnh Tularaemia thể khác	
A21.9	Bệnh do vi khuẩn Tularaemia gây ra, không xác định	
A22	Bệnh than	
A22.0	Bệnh than thể da	
A22.1	Bệnh than thể phổi	Bệnh than do hít|Bệnh than ở người nhặt giẻ rách|Bệnh than ở người phân loại len
A22.2	Bệnh than thể dạ dày - ruột	
A22.7	Bệnh than thể nhiễm trùng hệ thống	
A22.8	Bệnh than thể khác	
A22.9	Bệnh than, không xác định	
A23	Bệnh do Brucella	
A23.0	Bệnh do Brucella melitensis	
A23.1	Bệnh do Brucella abortus	
A23.2	Bệnh do Brucella suis	
A23.3	Bệnh do Brucella canis	
A23.8	Bệnh do Brucella khác	
A23.9	Bệnh do Brucella, không xác định	
A24	Bệnh melioidosis và/hoặc bệnh glanders	
A24.0	Bệnh tỵ thư [glanders]	Nhiễm trùng do Pseudomonas mallei|Nhiễm trùng do Burkholderia mallei|Malleus
A24.1	Bệnh Melioidosis [bệnh Whitmore] cấp tính và/hoặc đột ngột, nặng	
A24.2	Bệnh melioidosis [bệnh Whitmore] mạn tính và/hoặc bán cấp tính	
A24.3	Bệnh melioidosis [bệnh Whitmore] khác	
A24.4	Bệnh melioidosis [bệnh Whitmore], không xác định	
A25	Sốt do chuột cắn	
A25.0	Bệnh do nhiễm spirilla	Bệnh Sodoku
A25.1	Bệnh do nhiễm streptobacilla	Dịch hồng ban viêm khớp|Sốt Haverhill|Sốt do chuột cắn nhiễm streptobacilla
A25.9	Sốt do chuột cắn, không xác định	
A26	Bệnh viêm quầng đỏ do Erysipelothrix rhusiopathiae	
A26.0	Bệnh viêm quầng đỏ ở da do Erysipelothrix rhusiopathiae	Ban đỏ di chuyển
A26.7	Nhiễm trùng hệ thống do Erysipelothrix	
A26.8	Thể khác của viêm quầng đỏ do Erysipelothrix rhusiopathiae	
A26.9	Bệnh viêm quầng đỏ do Erysipelothrix rhusiopathiae, không xác định	
A27	Bệnh nhiễm leptospira	
A27.0	Bệnh vàng da xuất huyết do leptospira	Bệnh do Leptospira interrogans típ huyết thanh gây vàng da xuất huyết
A27.8	Thể khác của nhiễm leptospira	
A27.9	Bệnh nhiễm leptospira, không xác định	
A28	Bệnh nhiễm trùng khác do động vật truyền sang người, không phân loại mục khác	
A28.0	Bệnh tụ huyết trùng do Pasteurella	
A28.1	Bệnh mèo cào	Sốt do mèo cào
A28.2	Nhiễm Yersinia ngoài ruột	
A28.8	Bệnh nhiễm khuẩn do động vật truyền sang người khác, không phân loại mục khác	
A28.9	Bệnh nhiễm khuẩn do động vật truyền sang người, không xác định	
A30	Bệnh phong [bệnh Hansen]	
A30.0	Bệnh phong thể bất định	Bệnh phong thể I
A30.1	Bệnh phong thể củ	Bệnh phong thể củ TT
A30.2	Bệnh phong thể củ trung gian	
A30.3	Bệnh phong thể trung gian	Bệnh phong thể BB
A30.4	Bệnh phong thể u trung gian	Bệnh phong thể BL
A30.5	Bệnh phong thể u	Bệnh phong thể LL
A30.8	Bệnh phong thể khác	
A30.9	Bệnh phong, không xác định	
A31	Nhiễm trùng do mycobacteria khác	
A31.0	Bệnh do nhiễm mycobacteria ở phổi	
A31.1	Bệnh do nhiễm mycobacteria ở da	Loét Buruli
A31.8	Bệnh do nhiễm mycobacteria khác	
A31.9	Bệnh do nhiễm mycobacteria, không xác định	Nhiễm mycobacteria không điển hình không xác định khác|Bệnh lý do mycobacteria không xác định khác
A32	Bệnh do nhiễm listeria	
A32.0	Nhiễm listeria ở da	
A32.1†	Bệnh viêm màng não và/hoặc bệnh viêm não - màng não do listeria	
A32.7	Nhiễm trùng hệ thống do listeria	
A32.8	Thể khác của bệnh nhiễm listeria	
A32.9	Bệnh do nhiễm listeria, không xác định	
A33	Bệnh uốn ván sơ sinh	
A34	Bệnh uốn ván sản khoa	
A35	Bệnh uốn ván khác	
A36	Bệnh bạch hầu	
A36.0	Bệnh bạch hầu họng	Viêm họng giả mạc do bạch hầu|Viêm amydan do bạch hầu
A36.1	Bệnh bạch hầu thể mũi - họng	
A36.2	Bệnh bạch hầu thanh quản	Viêm thanh - khí quản do bạch hầu
A36.3	Bệnh bạch hầu da	
A36.8	Bệnh bạch hầu khác	
A36.9	Bệnh bạch hầu, không xác định	
A37	Bệnh ho gà	
A37.0	Bệnh ho gà do Bordetella pertussis	
A37.1	Bệnh ho gà do Bordetella parapertussis	
A37.8	Bệnh ho gà do Bordetella khác	
A37.9	Bệnh ho gà, không xác định	
A38	Bệnh sốt tinh hồng nhiệt	
A39	Nhiễm trùng do não mô cầu	
A39.0†	Bệnh viêm màng não do não mô cầu (G01*)	
A39.1†	Hội chứng Waterhouse-Friderichsen (E35.1*)	Viêm tuyến thượng thận xuất huyết do não mô cầu|Hội chứng thượng thận do não mô cầu
A39.2	Nhiễm khuẩn huyết não mô cầu cấp tính	
A39.3	Nhiễm khuẩn huyết não mô cầu mạn tính	
A39.4	Nhiễm khuẩn huyết do não mô cầu, không xác định	Vãng khuẩn huyết não mô cầu không xác định khác
A39.5†	Bệnh tim do não mô cầu	
A39.8	Nhiễm khuẩn não mô cầu khác	
A39.9	Nhiễm não mô cầu, không xác định	Bệnh do não mô cầu không xác định khác
A40	Nhiễm trùng hệ thống do liên cầu khuẩn	
A40.0	Nhiễm trùng hệ thống do liên cầu, nhóm A	
A40.1	Nhiễm trùng hệ thống do liên cầu, nhóm B	
A40.2	Nhiễm trùng hệ thống do liên cầu, nhóm D và/hoặc enterococcus	
A40.3	Nhiễm trùng hệ thống do phế cầu khuẩn gây bệnh [Streptococcus pneumoniae]	Nhiễm trùng hệ thống do phế cầu
A40.8	Nhiễm trùng hệ thống do liên cầu khác	
A40.9	Nhiễm trùng hệ thống do liên cầu, không xác định	
A41	Nhiễm trùng hệ thống khác	
A41.0	Nhiễm trùng hệ thống do tụ cầu vàng [Staphylococcus aureus]	
A41.1	Nhiễm trùng hệ thống do tụ cầu khuẩn xác định khác	Nhiễm trùng hệ thống do tụ cầu coagulase âm tính
A41.2	Nhiễm trùng hệ thống do tụ cầu khuẩn không xác định	
A41.3	Nhiễm trùng hệ thống do vi khuẩn Haemophilus cúm [H. influenzae]	
A41.4	Nhiễm trùng hệ thống do vi khuẩn kỵ khí	
A41.5	Nhiễm trùng hệ thống do vi sinh vật Gram âm khác	Nhiễm trùng hệ thống gram âm không xác định khác
A41.8	Nhiễm trùng hệ thống xác định khác	
A41.9	Nhiễm trùng hệ thống, không xác định	Nhiễm trùng huyết
A42	Bệnh do nhiễm actinomyces	
A42.0	Bệnh do nhiễm actinomyces ở phổi	
A42.1	Bệnh do nhiễm actinomyces ở bụng	
A42.2	Bệnh do nhiễm actinomyces ở mặt - cổ	
A42.7	Nhiễm trùng hệ thống do actinomyces	
A42.8	Bệnh do actinomyces thể khác	
A42.9	Bệnh do actinomyces, không xác định	
A43	Bệnh do nhiễm nocardia	
A43.0	Bệnh do nhiễm nocardia ở phổi	
A43.1	Bệnh do nhiễm nocardia ở da	
A43.8	Bệnh do nhiễm nocardia thể khác	
A43.9	Bệnh do nhiễm nocardia, không xác định	
A44	Bệnh do nhiễm bartonella	
A44.0	Bệnh do nhiễm bartonella toàn thân	Sốt Oroya
A44.1	Bệnh do nhiễm bartonella ở da và/hoặc niêm mạc	Mụn cóc Peru
A44.8	Bệnh do nhiễm bartonella thể khác	
A44.9	Bệnh do nhiễm bartonella, không xác định	
A46	Viêm quầng (do liên cầu khuẩn [Streptococcus])	
A48	Bệnh nhiễm khuẩn khác, không phân loại mục khác	
A48.0	Bệnh hoại thư sinh hơi [hoại thư khí]	
A48.1	Bệnh Legionnaire	
A48.2	Bệnh Legionnaire không gây viêm phổi [sốt Pontiac]	
A48.3	Hội chứng sốc nhiễm độc	
A48.4	Sốt ban xuất huyết Brasil	Nhiễm trùng toàn thân do Haemophilus aegyptius
A48.8	Bệnh nhiễm khuẩn xác định khác	
A49	Nhiễm khuẩn ở vị trí không xác định	
A49.0	Nhiễm khuẩn do tụ cầu, vị trí không xác định	
A49.1	Nhiễm khuẩn liên cầu và/hoặc enterococcal, vị trí không xác định	
A49.2	Nhiễm khuẩn Haemophilus cúm [H. influenzae], vị trí không xác định	
A49.3	Nhiễm khuẩn Mycoplasma, vị trí không xác định	
A49.8	Nhiễm khuẩn khác ở vị trí không xác định	
A49.9	Nhiễm khuẩn, không xác định	Vãng khuẩn huyết không xác định khác
A50	Bệnh giang mai bẩm sinh	
A50.0	Bệnh giang mai bẩm sinh sớm, có triệu chứng	
A50.1	Bệnh giang mai bẩm sinh sớm, tiềm ẩn	
A50.2	Bệnh giang mai bẩm sinh giai đoạn sớm, không xác định [có triệu chứng hoặc tiềm ẩn]	Giang mai bẩm sinh không xác định khác, dưới 2 tuổi sau sinh.
A50.3	Bệnh lý nhãn cầu do giang mai bẩm sinh muộn	
A50.4	Bệnh giang mai thần kinh bẩm sinh muộn [giang mai thần kinh ở thiếu niên]	
A50.5	Bệnh giang mai bẩm sinh muộn khác, có triệu chứng	
A50.6	Bệnh giang mai bẩm sinh muộn, tiềm ẩn	
A50.7	Bệnh giang mai bẩm sinh muộn, không xác định	Giang mai bẩm sinh không xác định khác ở người bệnh từ 2 tuổi trở lên.
A50.9	Bệnh giang mai bẩm sinh, không xác định	
A51	Bệnh giang mai sớm	
A51.0	Bệnh giang mai sinh dục sơ nhiễm	Vết loét [săng] [hạ cam] giang mai không xác định khác
A51.1	Bệnh giang mai hậu môn sơ nhiễm	
A51.2	Bệnh giang mai sơ nhiễm ở vị trí khác	
A51.3	Bệnh giang mai thứ phát ở da và/hoặc ở niêm mạc	
A51.4	Bệnh giang mai thứ phát khác	
A51.5	Bệnh giang mai giai đoạn sớm, tiềm ẩn [Thời gian nhiễm khuẩn dưới 2 năm]	
A51.9	Bệnh giang mai giai đoạn sớm, không xác định [Thời gian nhiễm khuẩn dưới 2 năm]	
A52	Bệnh giang mai giai đoạn muộn [Thời gian nhiễm khuẩn từ 2 năm trở lên]	
A52.0†	Bệnh giang mai tim mạch	
A52.1	Bệnh giang mai thần kinh có triệu chứng	
A52.2	Bệnh giang mai thần kinh không triệu chứng	
A52.3	Bệnh giang mai thần kinh, không xác định	
A52.7	Bệnh giang mai giai đoạn muộn khác có triệu chứng [Thời gian nhiễm khuẩn từ 2 năm trở lên]	
A52.8	Bệnh giang mai giai đoạn muộn, tiềm ẩn [Thời gian nhiễm khuẩn từ 2 năm trở lên]	
A52.9	Bệnh giang mai giai đoạn muộn, không xác định [Thời gian nhiễm khuẩn từ 2 năm trở lên]	
A53	Bệnh giang mai khác và/hoặc không xác định	
A53.0	Bệnh giang mai tiềm ẩn, không xác định là giai đoạn sớm hoặc giai đoạn muộn	Giang mai tiềm ẩn không xác định khác|Phản ứng huyết thanh dương tính đối với giang mai
A53.9	Bệnh giang mai, không xác định	
A54	Bệnh lậu [bệnh nhiễm khuẩn lậu cầu]	
A54.0	Bệnh lậu ở đường sinh dục - tiết niệu dưới không có áp xe quanh niệu đạo hoặc tuyến phần phụ	
A54.1	Bệnh lậu ở đường sinh dục - tiết niệu dưới có áp xe quanh niệu đạo và/hoặc tuyến phần phụ	Áp xe tuyến Bartholin do lậu cầu khuẩn
A54.2	Viêm phúc mạc vùng chậu và/hoặc nhiễm trùng sinh dục - tiết niệu khác do lậu cầu khuẩn	
A54.3	Bệnh lậu ở mắt	
A54.4†	Bệnh lậu ở hệ cơ xương khớp	
A54.5	Viêm họng do lậu cầu khuẩn	
A54.6	Bệnh lậu ở hậu môn và/hoặc trực tràng	
A54.8	Bệnh lậu khác	
A54.9	Bệnh lậu, không xác định	
A55	Viêm hạch lympho do Chlamydia (bệnh hột xoài)	Viêm hạch lympho do Chlamydia (Bệnh hột xoài)
A56	Bệnh khác do chlamydia lây truyền qua đường tình dục	
A56.0	Bệnh do chlamydia ở đường sinh dục - tiết niệu dưới	
A56.1	Viêm phúc mạc vùng chậu và/hoặc cơ quan sinh dục - tiết niệu khác do chlamydia	
A56.2	Bệnh do chlamydia ở đường niệu - sinh dục, không xác định	
A56.3	Bệnh do chlamydia ở hậu môn và/hoặc trực tràng	
A56.4	Bệnh do nhiễm chlamydia ở họng	
A56.8	Bệnh do chlamydia lây truyền qua đường tình dục ở vị trí khác	
A57	Bệnh hạ cam mềm	
A58	U hạt ở bẹn	
A59	Bệnh do trichomonas	
A59.0	Bệnh do trichomonas đường sinh dục - tiết niệu	
A59.8	Bệnh do trichomonas ở vị trí khác	
A59.9	Bệnh do nhiễm trichomonas, không xác định	
A60	Bệnh nhiễm virus herpes [herpes simplex] vùng hậu môn sinh dục	
A60.0	Bệnh nhiễm virus herpes [herpes simplex] ở đường sinh dục và/hoặc đường niệu - sinh dục	
A60.1	Bệnh do virus herpes [herpes simplex] ở da quanh hậu môn và/hoặc trực tràng	
A60.9	Nhiễm virus herpes [herpes simplex] ở hậu môn - sinh dục, không xác định	
A63	Bệnh khác lây chủ yếu qua đường tình dục, không phân loại mục khác	
A63.0	Mụn cơm [sùi mào gà, hoa liễu ở hậu môn - sinh dục]	
A63.8	Bệnh lây truyền chủ yếu qua đường tình dục xác định khác	
A64	Bệnh lây truyền qua đường tình dục không xác định	
A65	Bệnh giang mai không lây qua đường tình dục	
A66	Bệnh ghẻ cóc [do Spirochaete Treponema pallidum]	
A66.0	Tổn thương ban đầu của ghẻ cóc	Vết loét [săng] của ghẻ cóc|Vết mẩn đỏ và loét của bệnh ghẻ cóc, giai đoạn đầu hoặc sơ nhiễm|Loét khởi đầu do ghẻ cóc|Ghẻ cóc mẹ [nốt ghẻ mẹ] [ghẻ cái]
A66.1	Đa u nhú mềm và/hoặc ghẻ cóc tổn thương ở lòng bàn tay hoặc lòng bàn chân	Vết mẩn đỏ và loét [giống quả mâm xôi] của bệnh ghẻ cóc|U ghẻ cóc|U nhú ghẻ cóc ở lòng bàn chân hoặc lòng bàn tay
A66.2	Tổn thương da sớm khác của ghẻ cóc	
A66.3	Quá sản sừng của ghẻ cóc	
A66.4	Gôm và/hoặc loét của ghẻ cóc	
A66.5	Bệnh Gangosa [bệnh loét quanh mũi]	Viêm họng - mũi thể phá hủy
A66.6	Tổn thương xương và/hoặc khớp do ghẻ cóc	
A66.7	Biểu hiện khác của bệnh ghẻ cóc	U hạt cạnh khớp do ghẻ cóc|Ghẻ cóc ở niêm mạc
A66.8	Bệnh ghẻ cóc tiềm ẩn	Ghẻ cóc không có biểu hiện lâm sàng, huyết thanh dương tính
A66.9	Bệnh ghẻ cóc [do Spirochaete Treponema pallidum], không xác định	
A67	Bệnh Pinta [do Treponema carateum]	
A67.0	Tổn thương sơ nhiễm của pinta	
A67.1	Tổn thương trung gian của pinta	
A67.2	Tổn thương giai đoạn muộn của pinta	
A67.3	Tổn thương hỗn hợp của pinta	Tổn thương da giảm sắc tố kèm tổn thương da tăng sắc do pinta [carate]
A67.9	Bệnh pinta [do Treponema carateum], không xác định	
A68	Bệnh sốt tái phát	
A68.0	Sốt tái phát do chấy rận	Sốt tái phát do Borrelia recurrentis
A68.1	Sốt tái phát do ve truyền	Sốt tái phát do bất kỳ loài Borrelia nào khác ngoài Borrelia recurrentis
A68.9	Sốt tái phát, không xác định	
A69	Bệnh do xoắn khuẩn khác	
A69.0	Viêm loét hoại tử ở miệng	
A69.1	Viêm họng nhiễm trùng Vincent khác	
A69.2	Bệnh Lyme	Hồng ban di chuyển mạn tính do nhiễm Borrelia burgdorferi
A69.8	Bệnh nhiễm xoắn khuẩn xác định khác	
A69.9	Bệnh nhiễm xoắn khuẩn, không xác định	
A70	Bệnh do nhiễm Chlamydia psittaci	
A71	Bệnh mắt hột	
A71.0	Giai đoạn đầu của bệnh mắt hột	Bệnh mắt hột nghi ngờ
A71.1	Giai đoạn hoạt hóa của bệnh mắt hột	
A71.9	Bệnh mắt hột, không xác định	
A74	Bệnh khác do nhiễm chlamydia	
A74.0†	Viêm kết mạc do nhiễm chlamydia (H13.1*)	Bệnh mắt hột do nhiễm chlamydia ở người lớn [Paratrachoma]
A74.8	Bệnh do nhiễm chlamydia khác	
A74.9	Bệnh do nhiễm chlamydia, không xác định	Nhiễm chlamydia không xác định khác
A75	Bệnh sốt phát ban	
A75.0	Sốt phát ban dịch tễ do chấy rận truyền nhiễm Rickettsia prowazekii	
A75.1	Bệnh sốt phát ban tái phát [bệnh Brill]	Bệnh Brill-Zinsser
A75.2	Bệnh sốt phát ban do nhiễm Rickettsia typhi	
A75.3	Bệnh sốt phát ban do nhiễm Rickettsia tsutsugamushi	
A75.9	Bệnh sốt phát ban, không xác định	
A77	Sốt phát ban đốm [nhiễm Rickttsia do bọ ve truyền]	
A77.0	Bệnh sốt phát ban đốm do nhiễm Rickettsia rikettsii	Sốt màng não miền núi [sốt phát ban đốm vùng núi đá]|Sốt Sao Paulo
A77.1	Sốt phát ban dạng đốm do Rickettsia conoril	Bệnh sốt phát ban do bọ chét ở châu Phi|Sốt Boutonneuse|Bệnh sốt phát ban do bọ ve ở Ấn Độ|Bệnh sốt phát ban do bọ ve ở Kenya|Sốt Marseilles|Sốt ve Địa Trung Hải
A77.2	Sốt phát ban dạng đốm do Rickettsia siberica	Sốt do bọ ve ở Bắc Á|Bệnh sốt phát ban do bọ ve ở Siberia
A77.3	Sốt phát ban dạng đốm do Rickettsia australis	Sốt phát ban do bọ ve ở Queensland
A77.8	Sốt phát ban dạng đốm khác	
A77.9	Sốt phát ban dạng đốm, không xác định	Bệnh sốt phát ban do bọ ve truyền không xác định khác
A78	Sốt Q	
A79	Bệnh khác do nhiễm Rickettsia	
A79.0	Bệnh sốt chiến hào [sốt tái phát khoảng 5 ngày/lần do Bartonella quintana]	Sốt Quintan [sốt tái phát khoảng 5 ngày/lần]|Sốt Wolhynian [sốt tái phát khoảng 5 ngày/lần]
A79.1	Mụn Rickettsia do nhiễm Rickettsia akari	Sốt vườn thực vật hoàng gia Kew|Mụn nước do rickettsia
A79.8	Bệnh sốt do Rickettsia xác định khác	Bệnh sốt Rickettsia do Neorickettsia sennetsu [Ehrlichia sennetsu]
A79.9	Bệnh sốt do Rickettsia khác, không xác định	Bệnh sốt do Rickettsia không xác định khác
A80	Bệnh bại liệt cấp tính	
A80.0	Bệnh bại liệt cấp tính, thể liệt, liên quan đến vắc xin	
A80.1	Bệnh bại liệt cấp tính, thể liệt, do virus hoang dã, ngoại lai	
A80.2	Bệnh bại liệt cấp tính, thể liệt, do virus hoang dã, bản địa	
A80.3	Bệnh bại liệt cấp tính, thể liệt, do nguyên nhân khác và/hoặc không xác định	
A80.4	Bệnh bại liệt cấp tính thể không liệt	
A80.9	Bệnh bại liệt cấp tính, không xác định	
A81	Bệnh do virus không điển hình ở hệ thần kinh trung ương	
A81.0	Bệnh bò điên [Creutzfeidt-Jakob]	Bệnh não xốp bán cấp tính
A81.1	Viêm toàn não xơ hóa bán cấp tính	Viêm não thể vùi Dawson|Bệnh lý não chất trắng xơ cứng Van Bogaert
A81.2	Bệnh lý não chất trắng đa ổ tiến triển	Bệnh lý não chất trắng đa ổ không xác định khác
A81.8	Nhiễm virus không điển hình khác của hệ thần kinh trung ương	Kuru
A81.9	Nhiễm virus không điển hình của hệ thần kinh trung ương, không xác định	Bệnh do prion của hệ thần kinh trung ương không xác định khác
A82	Bệnh dại ở người	
A82.0	Bệnh dại cộng sinh [bệnh dại ở động vật hoang dã]	
A82.1	Bệnh dại đô thị	
A82.9	Bệnh dại, không xác định	
A83	Bệnh viêm não virus do muỗi truyền	
A83.0	Bệnh viêm não Nhật Bản	
A83.1	Bệnh viêm não ngựa miền Tây	
A83.2	Bệnh viêm não ngựa miền Đông	
A83.3	Bệnh viêm não St. Louis	
A83.4	Bệnh viêm não châu Úc	Bệnh do virus Kunjin
A83.5	Bệnh viêm não California	Bệnh viêm não - màng não California|Bệnh viêm não La Crosse
A83.6	Bệnh virus Rocio	
A83.8	Bệnh viêm não virus khác do muỗi truyền	
A83.9	Bệnh viêm não virus do muỗi truyền, không xác định	
A84	Bệnh viêm não virus do bọ ve truyền	
A84.0	Bệnh viêm não Viễn Đông do bọ ve truyền [viêm não xuân hạ Nga]	
A84.1	Bệnh viêm não Trung Âu do bọ ve truyền	
A84.8	Bệnh viêm não virus khác do bọ ve truyền	Bệnh Looping [Bệnh virus cấp tính do bọ ve truyền qua hệ thần kinh trung ương]|Bệnh do virus Powassan
A84.9	Bệnh viêm não virus do bọ ve truyền, không xác định	
A85	Bệnh viêm não khác do virus, không phân loại mục khác	
A85.0†	Bệnh viêm não do virus lây truyền qua đường ruột [enterovirus] (G05.1*)	Bệnh viêm não tủy do virus lây truyền qua đường ruột [enterovirus]
A85.1†	Bệnh viêm não do virus adeno (G05.1*)	Bệnh viêm não - màng não do virus adeno
A85.2	Bệnh viêm não virus do tiết túc truyền [virus Arbo], không xác định	
A85.8	Bệnh viêm não do virus xác định khác	Viêm não ngủ lịm [hôn mê]|Bệnh viêm não mê ngủ [Von Economo-Cruchet]
A86	Bệnh viêm não do virus không xác định	
A87	Bệnh viêm màng não do virus	
A87.0†	Bệnh viêm màng não do virus lây truyền qua đường ruột [enterovirus] (G02.0*)	Bệnh viêm màng não do virus coxsackie|Bệnh viêm màng não do virus echo
A87.1†	Bệnh viêm màng não do virus adeno (G02.0*)	
A87.2	Bệnh viêm màng não đám rối màng mạch tế bào lympho	Viêm não - màng não tế bào lympho
A87.8	Bệnh viêm màng não do virus khác	
A87.9	Bệnh viêm màng não do virus, không xác định	
A88	Nhiễm virus khác của hệ thần kinh trung ương, không phân loại mục khác	
A88.0	Sốt ngoại ban do virus lây truyền qua đường ruột [enterovirus] [ngoại ban Boston]	
A88.1	Chóng mặt dịch tễ [viêm dây thần kinh tiền đình]	
A88.8	Nhiễm virus xác định khác của hệ thần kinh trung ương	
A89	Nhiễm virus không xác định của hệ thần kinh trung ương	
A92	Bệnh sốt virus khác do muỗi truyền	
A92.0	Bệnh do nhiễm virus Chikungunya	
A92.1	Bệnh sốt do virus O’nyong nyong	
A92.2	Bệnh sốt ngựa Venezuela	
A92.3	Bệnh nhiễm virus tây sông Nin	Bệnh sốt tây sông Nin
A92.4	Bệnh sốt thung lũng Rift	
A92.5	Bệnh nhiễm virus Zika	
A92.8	Bệnh sốt virus xác định khác do muỗi truyền	
A92.9	Bệnh sốt virus do muỗi truyền, không xác định	
A93	Bệnh sốt virus khác do tiết túc truyền [virus arbo], không phân loại mục khác	
A93.0	Bệnh do virus Oropouche	Sốt Oropouche
A93.1	Bệnh sốt ruồi cát [sốt Phlebotomus] [sốt 3 ngày]	Sốt Pappataci|Sốt Phlebotomus
A93.2	Bệnh sốt do bọ ve Colorado	
A93.8	Bệnh sốt virus xác định khác do tiết túc truyền [virus arbo]	Bệnh virus Piry|Sốt nặng kèm theo hội chứng giảm tiểu cầu [SFTS]|Bệnh do virus gây viêm miệng có mụn nước [sốt Indiana]
A94	Bệnh sốt virus do tiết túc truyền [virus arbo] không xác định	
A95	Bệnh sốt vàng	
A95.0	Bệnh sốt vàng cộng sinh [bệnh sốt vàng ở động vật hoang dã]	Bệnh sốt vàng ở rừng
A95.1	Bệnh sốt vàng đô thị	
A95.9	Bệnh sốt vàng, không xác định	
A96	Bệnh sốt xuất huyết do virus arena	
A96.0	Bệnh sốt xuất huyết do virus Junin	Bệnh sốt xuất huyết Argentina
A96.1	Bệnh sốt xuất huyết do virus Machupo	Bệnh sốt xuất huyết Bolivia
A96.2	Bệnh sốt xuất huyết do virus Lassa	
A96.8	Bệnh sốt xuất huyết do virus arena khác	
A96.9	Bệnh sốt xuất huyết do virus arena, không xác định	
A97	Bệnh sốt xuất huyết Dengue	
A97.0	Bệnh sốt xuất huyết Dengue không có dấu hiệu cảnh báo	Sốt xuất huyết Dengue độ 1 và/hoặc độ 2|Sốt xuất huyết Dengue không có dấu hiệu cảnh báo
A97.1	Bệnh sốt xuất huyết Dengue có dấu hiệu cảnh báo	
A97.2	Bệnh sốt xuất huyết Dengue nặng	
A97.9	Bệnh sốt xuất huyết Dengue, không xác định	Sốt xuất huyết Dengue [DF] không xác định khác
A98	Bệnh sốt xuất huyết do virus khác, không phân loại mục khác	
A98.0	Bệnh sốt xuất huyết Crimean-Congo	Bệnh sốt xuất huyết Trung Á
A98.1	Bệnh sốt xuất huyết Omsk	
A98.2	Bệnh sốt xuất huyết rừng Kyasanur	
A98.3	Bệnh sốt xuất huyết do virus Marburg	
A98.4	Bệnh sốt xuất huyết do virus Ebola	
A98.5	Bệnh sốt xuất huyết kèm hội chứng thận	
A98.8	Bệnh sốt xuất huyết do virus xác định khác	
A99	Bệnh sốt xuất huyết do virus không xác định	
B00	Bệnh do nhiễm virus herpes [herpes simplex]	
B00.0	Bệnh chàm do nhiễm virus herpes [herpes simplex]	Phát ban dạng thủy đậu Kaposi
B00.1	Bệnh viêm da rộp nước do nhiễm virus herpes [herpes simplex]	
B00.2	Bệnh viêm miệng - lợi [nướu] và/hoặc viêm amydan - hầu do nhiễm virus herpes [herpes simplex]	Viêm họng do virus herpes [herpes simplex]
B00.3†	Bệnh viêm màng não do nhiễm virus herpes [herpes simplex] (G02.0*)	
B00.4†	Bệnh viêm não do nhiễm virus herpes [herpes simplex] (G05.1*)	viêm não - màng não do virus herpes [herpes simples]|Bệnh nhiễm virus Simian B
B00.5	Bệnh mắt do nhiễm virus herpes [herpes simplex]	
B00.7	Bệnh do nhiễm virus herpes [herpes simplex] lan tỏa	Nhiễm trùng hệ thống do virus herpes [herpes simplex]
B00.8	Dạng khác của nhiễm virus herpes [herpes simplex]	
B00.9	Bệnh do nhiễm virus herpes [herpes simplex], không xác định	Bệnh do nhiễm virus herpes [herpes simplex] không xác định khác
B01	Bệnh thủy đậu	
B01.0†	Bệnh viêm màng não do thủy đậu (G02.0*)	
B01.1†	Bệnh viêm não do thủy đậu (G05.1*)	Viêm não sau mắc bệnh thủy đậu|Viêm não - tủy do thủy đậu
B01.2†	Bệnh viêm phổi do thủy đậu (J17.1*)	
B01.8	Bệnh thủy đậu kèm biến chứng khác	
B01.9	Bệnh thủy đậu không biến chứng	Thủy đậu không xác định khác
B02	Bệnh zona [herpes zoster] [bệnh giời leo]	
B02.0†	Viêm não do zona (G05.1*)	Viêm não - màng não do zona
B02.1†	Viêm màng não do zona (G02.0*)	
B02.2†	Tổn thương hệ thần kinh khác do zona	
B02.3	Bệnh mắt do zona	
B02.7	Bệnh zona lan tỏa	
B02.8	Bệnh zona kèm biến chứng khác	
B02.9	Bệnh zona không biến chứng	Bệnh zona không xác định khác
B03	Bệnh đậu mùa	
B04	Bệnh đậu mùa khỉ	
B05	Bệnh sởi	
B05.0†	Bệnh sởi kèm biến chứng viêm não (G05.1*)	Viêm não sau mắc bệnh sởi
B05.1†	Bệnh sởi kèm biến chứng viêm màng não (G02.0*)	Viêm màng não sau mắc bệnh sởi
B05.2†	Bệnh sởi kèm biến chứng viêm phổi (J17.1*)	Viêm phổi sau mắc bệnh sởi
B05.3†	Bệnh sởi kèm biến chứng viêm tai giữa (H67.1*)	Viêm tai giữa sau mắc bệnh sởi
B05.4	Bệnh sởi kèm biến chứng ở ruột	
B05.8	Bệnh sởi kèm biến chứng khác	
B05.9	Bệnh sởi không biến chứng	Bệnh sởi không xác định khác
B06	Bệnh rubella [sởi Đức]	
B06.0†	Bệnh rubella kèm biến chứng thần kinh	
B06.8	Bệnh rubella kèm biến chứng khác	
B06.9	Bệnh rubella không kèm biến chứng	Bệnh rubella không xác định khác
B07	Bệnh mụn cóc do virus	
B08	Nhiễm virus khác có biểu hiện tổn thương tại da và/hoặc niêm mạc, không phân loại mục khác	
B08.0	Nhiễm virus orthopox khác	
B08.1	U mềm lây	
B08.2	Phát ban đột ngột [bệnh thứ sáu] [bệnh ban đào]	
B08.3	Ban đỏ nhiễm khuẩn [bệnh thứ năm]	
B08.4	Viêm họng có mụn nước do virus đường ruột có phát ban	Bệnh tay, chân và miệng
B08.5	Viêm họng có mụn nước do virus đường ruột	Viêm họng do herpes
B08.8	Nhiễm virus xác định khác có biểu hiện tổn thương tại da và/hoặc niêm mạc	Viêm hầu hạch lympho do virus lây truyền qua đường ruột|Bệnh ở chân - và - miệng|Bệnh do nhiễm virus Tanopox|Bệnh do nhiễm virus Yabapox
B09	Nhiễm virus không xác định, có biểu hiện tổn thương tại da và/hoặc niêm mạc	
B15	Bệnh viêm gan A cấp tính	
B15.0	Bệnh viêm gan A có kèm hôn mê gan	
B15.9	Bệnh viêm gan A không kèm hôn mê gan	
B16	Bệnh viêm gan B cấp tính	
B16.0	Bệnh viêm gan B cấp tính có viêm gan D [tác nhân delta] kèm hôn mê gan	
B16.1	Bệnh viêm gan B cấp tính có viêm gan D [tác nhân delta] (đồng nhiễm) không kèm hôn mê gan	
B16.2	Bệnh viêm gan B cấp tính không có viêm gan D [tác nhân delta] có kèm hôn mê gan	
B16.9	Bệnh viêm gan B cấp tính không có viêm gan D [tác nhân delta] và/hoặc không kèm hôn mê gan	
B17	Bệnh viêm gan virus cấp tính khác	
B17.0	Nhiễm virus viêm gan D [tác nhân delta] cấp tính ở người bệnh viêm gan B	
B17.1	Bệnh viêm gan C cấp tính	
B17.2	Bệnh viêm gan E cấp tính	
B17.8	Bệnh viêm gan virus cấp tính xác định khác	
B17.9	Bệnh viêm gan virus cấp tính, không xác định	Viêm gan vius cấp tính không xác định khác|Viêm gan nhiễm khuẩn cấp tính không xác định khác
B18	Bệnh viêm gan virus mạn tính	
B18.0	Bệnh viêm gan virus B mạn tính có viêm gan D [tác nhân delta]	
B18.00	Bệnh viêm gan virus B mạn tính có viêm gan D [tác nhân delta], giai đoạn dung nạp miễn dịch	
B18.09	Bệnh viêm gan virus B mạn tính có viêm gan D [tác nhân delta], giai đoạn khác và/hoặc không xác định	
B18.1	Bệnh viêm gan virus B mạn tính không có viêm gan D [tác nhân delta]	
B18.10	Bệnh viêm gan virus B mạn tính không có viêm gan D [tác nhân delta], giai đoạn dung nạp miễn dịch	
B18.19	Bệnh viêm gan virus B mạn tính không có viêm gan D [tác nhân delta], giai đoạn khác và/hoặc không xác định	
B18.2	Bệnh viêm gan virus C mạn tính	
B18.8	Bệnh viêm gan virus mạn tính khác	
B18.9	Bệnh viêm gan virus mạn tính, không xác định	
B19	Bệnh viêm gan virus không xác định	
B19.0	Bệnh viêm gan virus không xác định kèm hôn mê gan	
B19.9	Bệnh viêm gan virus không xác định không kèm hôn mê gan	Viêm gan do virus không xác định khác
B20	Bệnh do virus gây suy giảm miễn dịch ở người [HIV] gây ra bệnh nhiễm trùng và/hoặc ký sinh trùng	
B20.0	Bệnh do HIV gây ra nhiễm mycobacteria	Bệnh do HIV gây ra nhiễm lao
B20.1	Bệnh do HIV gây ra bệnh nhiễm trùng khác	
B20.2	Bệnh do HIV gây ra bệnh virus đại bào [cytomegalovirus-CMV]	
B20.3	Bệnh do HIV gây ra bệnh nhiễm virus khác	
B20.4	Bệnh do HIV gây ra nhiễm candida	
B20.5	Bệnh do HIV gây ra nhiễm nấm khác	
B20.6	Bệnh do HIV gây ra viêm phổi do Pneumocystis jirovecii	Bệnh do HIV gây ra viêm phổi do Pneumocystis carinii
B20.7	Bệnh do HIV gây ra bội nhiễm	
B20.8	Bệnh do HIV gây ra bệnh nhiễm trùng và/hoặc ký sinh trùng khác	
B20.9	Bệnh HIV gây ra bệnh nhiễm trùng hoặc ký sinh trùng không xác định	Bệnh do HIV gây ra các bệnh truyền nhiễm không xác định khác
B21	Bệnh do virus suy giảm miễn dịch ở người [HIV] gây ra u ác tính	
B21.0	Bệnh do HIV gây ra ung thư Kaposi [Kaposi sarcoma]	
B21.1	Bệnh do HIV gây ra ung thư hạch Burkitt [Burkitt lymphoma]	
B21.2	Bệnh do HIV gây ra loại ung thư hạch không Hodgkin [non-Hodgkin lymphoma] khác	
B21.3	Bệnh do HIV gây ra khối u ác tính khác ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	
B21.7	Bệnh do HIV gây ra đa u ác tính	
B21.8	Bệnh do HIV gây ra u ác tính khác	
B21.9	Bệnh do HIV gây ra u ác tính không xác định	
B22	Bệnh do virus gây suy giảm miễn dịch ở người [HIV] gây ra bệnh xác định khác	
B22.0	Bệnh do HIV gây ra bệnh lý não	Sa sút trí tuệ do HIV
B22.1	Bệnh do HIV gây ra viêm phổi mô kẽ lympho bào	
B22.2	Bệnh do HIV gây ra hội chứng suy mòn	Bệnh HIV dẫn đến suy nhược cơ thể|Bệnh gầy sút [bệnh do suy giảm miễn dịch mắc phải - AIDS]
B22.7	Bệnh do HIV gây ra đa bệnh lý phân loại mục khác	
B23	Bệnh do virus gây suy giảm miễn dịch ở người [HIV] dẫn đến bệnh khác	
B23.0	Hội chứng nhiễm HIV cấp tính	
B23.1	Bệnh HIV gây ra bệnh lý hạch bạch huyết toàn thân (dai dẳng)	
B23.2	Bệnh do HIV gây ra bất thường của huyết học và/hoặc miễn dịch, không phân loại mục khác	
B23.8	Bệnh HIV gây ra bệnh lý xác định khác	
B24	Bệnh do virus gây suy giảm miễn dịch ở người [HIV] không xác định	
B25	Bệnh nhiễm virus đại bào [cytomegalovirus-CMV]	
B25.0†	Bệnh viêm phổi do virus đại bào [cytomegalovirus-CMV] (J17.1*)	
B25.1†	Bệnh viêm gan do virus đại bào [cytomegalovirus-CMV] (K77.0*)	
B25.2†	Bệnh viêm tụy do virus đại bào [cytomegalovirus-CMV] (K87.1*)	
B25.8	Bệnh khác do virus đại bào [cytomegalovirus-CMV]	
B25.9	Bệnh do nhiễm virus đại bào [cytomegalovirus-CMV], không xác định	
B26	Bệnh quai bị	
B26.0†	Bệnh viêm tinh hoàn do bệnh quai bị (N51.1*)	
B26.1†	Bệnh viêm màng não do bệnh quai bị (G02.0*)	
B26.2†	Bệnh viêm não do bệnh quai bị (G05.1*)	
B26.3†	Bệnh viêm tụy do bệnh quai bị (K87.1*)	
B26.8	Bệnh quai bị kèm biến chứng khác	
B26.9	Bệnh quai bị không kèm biến chứng	
B27	Bệnh tăng bạch cầu đơn nhân nhiễm trùng	
B27.0	Bệnh tăng bạch cầu đơn nhân do virus herpes gamma	Bệnh tăng bạch cầu đơn nhân do virus Epstein-Barr
B27.1	Bệnh tăng bạch cầu đơn nhân do virus đại bào [cytomegalovirus-CMV]	
B27.8	Bệnh tăng bạch cầu đơn nhân nhiễm trùng khác	
B27.9	Bệnh tăng bạch cầu đơn nhân nhiễm trùng, không xác định	
B30	Bệnh viêm kết mạc do virus	
B30.0†	Bệnh viêm kết - giác mạc do virus adeno (H19.2*)	Viêm kết - giác mạc do dịch|Bệnh mắt tại nơi đóng tàu
B30.1†	Bệnh viêm kết mạc do virus adeno (H13.1*)	Viêm kết mạc cấp dạng nang do virus adeno|Viêm kết mạc hồ bơi
B30.2	Bệnh viêm họng - kết mạc do virus	
B30.3†	Bệnh viêm kết mạc xuất huyết cấp tính do dịch (virus lây truyền qua đường ruột [enterovirus]) (H13.1*)	
B30.8†	Bệnh viêm kết mạc do virus khác (H13.1*)	Viêm kết mạc do virus Newcastle
B30.9	Bệnh viêm kết mạc do virus, không xác định	
B33	Bệnh nhiễm virus khác, không phân loại mục khác	
B33.0	Bệnh đau cơ do dịch	Bệnh Bornholm
B33.1	Bệnh Ross River	Viêm đa khớp do dịch và/hoặc phát ban|Sốt Ross River
B33.2	Bệnh viêm tim do virus	
B33.3	Nhiễm virus retro, không phân loại mục khác	Nhiễm virus retro không xác định khác
B33.4†	Hội chứng phổi (tim-) do virus Hanta [HPS] [HCPS] (J17.1*)	
B33.8	Bệnh do virus xác định khác	
B34	Nhiễm virus ở vị trí không xác định	
B34.0	Nhiễm virus adeno, vị trí không xác định	
B34.1	Nhiễm virus lây truyền qua đường ruột [enterovirus], vị trí không xác định	Nhiễm virus coxsackie không xác định khác|Nhiễm virus echo không xác định khác
B34.2	Nhiễm virus corona, vị trí không xác định	
B34.3	Nhiễm viurs parvo, vị trí không xác định	
B34.4	Nhiễm virus papova, vị trí không xác định	
B34.8	Nhiễm virus khác ở vị trí không xác định	
B34.9	Nhiễm virus, không xác định	Nhiễm virus huyết không xác định khác
B35	Bệnh da do nấm sợi	
B35.0	Bệnh nấm ở cằm và/hoặc nấm da đầu [chốc đầu]	Bệnh hắc lào ở cằm|Bệnh nấm da đầu Kerion [chốc đầu mưng mủ]|Bệnh hắc lào ở da đầu|Viêm nang râu, do nấm
B35.1	Bệnh nấm móng	Viêm nền móng do nấm sợi|Móng tay nhiễm nấm sợi|Bệnh hắc lào ở móng tay
B35.2	Bệnh nấm da bàn tay	Nhiễm nấm da bàn tay|Bệnh hắc lào ở bàn tay
B35.3	Bệnh nấm da chân	Bệnh nấm bàn chân|Nhiễm nấm da ở chân|Bệnh hắc lào ở chân
B35.4	Bệnh nấm da thân	Bệnh hắc lào trên cơ thể
B35.5	Bệnh vảy rồng [nấm đồng tâm]	Tokelau [Bệnh vảy rồng]
B35.6	Bệnh nấm da đùi [nấm bẹn]	Bệnh ngứa bẹn [Dhobi itch]|Bệnh hắc lào ở bẹn|Bệnh nấm háng [Jock itch]
B35.8	Bệnh nấm da khác	
B35.9	Bệnh da do nấm sợi, không xác định	Bệnh hắc lào không xác định khác
B36	Bệnh nhiễm nấm nông khác ở da	
B36.0	Bệnh lang ben	
B36.1	Bệnh nấm da có thương tổn màu đen [nấm da nigra]	Bệnh nấm đen lòng bàn tay|Bệnh nấm hắc lào mảng xám|Bệnh vẩy phấn đen
B36.2	Bệnh trứng tóc trắng [white piedra]	Bệnh nấm da vảy trắng
B36.3	Bệnh trứng tóc đen [black piedra]	
B36.8	Bệnh nấm nông xác định khác	
B36.9	Bệnh nấm nông, không xác định	
B37	Bệnh do nhiễm nấm candida	Bệnh do nấm candida
B37.0	Bệnh viêm miệng do nhiễm nấm candida	Bệnh do nấm candida ở miệng [bệnh tưa miệng]
B37.1	Bệnh do nhiễm nấm candida ở phổi	
B37.2	Bệnh do nhiễm nấm candida ở da và/hoặc móng	
B37.3†	Bệnh do nhiễm nấm candida ở âm hộ và/hoặc âm đạo (N77.1*)	Viêm âm hộ - âm đạo do nấm candida|Viêm âm hộ - âm đạo do nhiễm monilia|Bệnh do nhiễm nấm candida ở âm đạo
B37.4	Bệnh do nhiễm nấm candida ở vị trí niệu sinh dục khác	
B37.5†	Bệnh viêm màng não do nhiễm nấm candida (G02.1*)	
B37.6†	Bệnh viêm nội tâm mạc do nhiễm nấm candida (I39.8*)	
B37.7	Nhiễm trùng hệ thống do nhiễm nấm candida	
B37.8	Bệnh do nhiễm nấm candida ở vị trí khác	
B37.9	Bệnh do nhiễm nấm candida, không xác định	Bệnh do nhiễm nấm candida không xác định khác
B38	Bệnh nhiễm nấm coccidioides	
B38.0	Bệnh nhiễm nấm coccidioides ở phổi cấp tính	
B38.1	Bệnh nhiễm nấm coccidioides ở phổi mạn tính	
B38.2	Bệnh nhiễm nấm coccidioides ở phổi, không xác định [cấp tính hoặc mạn tính]	
B38.3	Bệnh nhiễm nấm coccidioides ở da	
B38.4†	Bệnh viêm màng não do nhiễm nấm coccidioides (G02.1*)	
B38.7	Bệnh nhiễm nấm coccidioides lan tỏa	Bệnh nhiễm nấm coccidioides toàn thân
B38.8	Bệnh nhiễm nấm coccidioides thể khác	
B38.9	Bệnh nhiễm nấm coccidioides, không xác định	
B39	Bệnh nhiễm nấm histoplasma	
B39.0	Bệnh nhiễm nấm histoplasma capsulatum ở phổi cấp tính	
B39.1	Bệnh nhiễm nấm histoplasma capsulatum ở phổi mạn tính	
B39.2	Bệnh nhiễm nấm histoplasma capsulatum ở phổi, không xác định [cấp tính hoặc mạn tính]	
B39.3	Bệnh nhiễm nấm histoplasma capsulatum lan tỏa	Bệnh nhiễm nấm histoplasma capsulatum toàn thân
B39.4	Bệnh nhiễm nấm histoplasma capsulatum, không xác định [ở phổi hoặc lan tỏa]	Bệnh nhiễm nấm histoplasma châu Mỹ
B39.5	Bệnh nhiễm nấm histoplasma duboisii	Bệnh nhiễm nấm histoplasma châu Phi
B39.9	Bệnh nhiễm nấm histoplasma, không xác định	
B40	Bệnh do nhiễm nấm blastomyces	
B40.0	Bệnh do nhiễm nấm blastomyces ở phổi cấp tính	
B40.1	Bệnh do nhiễm nấm blastomyces ở phổi mạn tính	
B40.2	Bệnh do nhiễm nấm blastomyces ở phổi, không xác định [cấp tính hoặc mạn tính]	
B40.3	Bệnh do nhiễm nấm blastomyces ở da	
B40.7	Bệnh do nhiễm nấm blastomyces lan tỏa	Bệnh do nhiễm nấm blastomyces toàn thân
B40.8	Bệnh do nhiễm nấm blastomyces thể khác	
B40.9	Bệnh do nhiễm nấm blastomyces, không xác định	
B41	Bệnh do nhiễm nấm paracoccidioides	
B41.0	Bệnh do nhiễm nấm paracoccidioides ở phổi	
B41.7	Bệnh do nhiễm nấm paracoccidioides lan tỏa	Bệnh do nhiễm nấm paracoccidioides toàn thân
B41.8	Bệnh do nhiễm nấm paracoccidioides thể khác	
B41.9	Bệnh do nhiễm nấm paracoccidioides, không xác định	
B42	Bệnh do nhiễm nấm sporotrichum	
B42.0†	Bệnh do nhiễm nấm sporotrichum ở phổi (J99.8*)	
B42.1	Bệnh do nhiễm nấm sporotrichum da - bạch huyết	
B42.7	Bệnh do nhiễm nấm sporotrichum lan tỏa	Bệnh do nhiễm nấm sporotrichum toàn thân
B42.8	Bệnh do nhiễm nấm sporotrichum thể khác	
B42.9	Bệnh do nhiễm nấm sporotrichum, không xác định	
B43	Bệnh do nhiễm nấm chromoblastomycosa [nấm màu] và/hoặc áp xe do phaeomyces	
B43.0	Bệnh do nhiễm nấm chromoblastomycosa ở da	Viêm da mụn cóc
B43.1	Áp xe não do phaeomyces	
B43.2	Nang và/hoặc áp xe dưới da do phaeomyces	
B43.8	Dạng khác của nhiễm nấm sâu chromomycois (nấm hạt màu)	
B43.9	Bệnh nhiễm nấm sâu chromomycois (nấm hạt màu), không xác định	
B44	Bệnh nhiễm nấm aspergillus	
B44.0	Bệnh nhiễm nấm aspergillus xâm lấn ở phổi	
B44.1	Bệnh nhiễm nấm aspergillus khác ở phổi	
B44.2	Bệnh nhiễm nấm aspergillus ở amydan	
B44.7	Bệnh nhiễm nấm aspergillus lan tỏa	Bệnh nhiễm nấm aspergillus toàn thân
B44.8	Bệnh nhiễm nấm aspergillus thể khác	
B44.9	Bệnh nhiễm nấm aspergillus, không xác định	
B45	Bệnh nhiễm nấm cryptococcus	
B45.0	Bệnh nhiễm nấm cryptococcus ở phổi	
B45.1	Bệnh nhiễm nấm cryptococcus ở não	
B45.2	Bệnh nhiễm nấm cryptococcus ở da	
B45.3	Bệnh nhiễm nấm cryptococcus ở xương	
B45.7	Bệnh nhiễm nấm cryptococcus lan tỏa	Bệnh nhiễm nấm cryptococcus toàn thân
B45.8	Bệnh nhiễm nấm cryptococcus thể khác	
B45.9	Bệnh nhiễm nấm cryptococcus, không xác định	
B46	Bệnh nhiễm nấm zygomycetes	
B46.0	Bệnh nhiễm nấm mucor ở phổi	
B46.1	Bệnh nhiễm nấm mucor ở mũi - não	
B46.2	Bệnh nhiễm nấm mucor ở đường tiêu hóa	
B46.3	Bệnh nhiễm nấm mucor ở da	Bệnh nhiễm nấm mucor dưới da
B46.4	Bệnh nhiễm nấm mucor lan tỏa	Bệnh nhiễm nấm mucor toàn thân
B46.5	Bệnh nhiễm nấm mucor, không xác định	
B46.8	Bệnh nấm zygomycetes khác	Bệnh nhiễm nấm entomophthoromyces
B46.9	Bệnh nhiễm nấm zygomycetes, không xác định	Bệnh nhiễm nấm phycomycetes không xác định khác
B47	Bệnh u bướu do nhiễm nấm	Bệnh u bướu do nấm
B47.0	Bệnh u nấm do eumycetes	Bệnh u nấm sâu
B47.1	Nhiễm trùng dưới da mạn tính do vi khuẩn actinomyces	
B47.9	Bệnh u bướu do nấm, không xác định	
B48	Bệnh nhiễm nấm khác, không phân loại mục khác	
B48.0	Bệnh nhiễm nấm lobo	Bệnh do nhiễm nấm blastomyces dạng sùi
B48.1	Bệnh nhiễm nấm rhinosporidiosis	
B48.2	Bệnh nhiễm nấm allesscheria	
B48.3	Bệnh nhiễm nấm geotrichum	Bệnh viêm miệng do nhiễm nấm geotrichum
B48.4	Bệnh nhiễm nấm penicillium	
B48.5†	Bệnh nhiễm nấm pneumocystis	
B48.7	Bệnh nhiễm nấm cơ hội	
B48.8	Bệnh nhiễm nấm xác định khác	Bệnh nhiễm nấm adiaspiromyces
B49	Bệnh nhiễm nấm không xác định	
B50	Bệnh sốt rét do plasmodium falciparum	
B50.0	Bệnh sốt rét do plasmodium falciparum kèm biến chứng não	Bệnh sốt rét thể não không xác định khác
B50.8	Bệnh sốt rét do plasmodium falciparum thể nặng và/hoặc biến chứng khác	Bệnh sốt rét do Plasmodium falciparum nặng hoặc có biến chứng không xác định khác
B50.9	Bệnh sốt rét do plasmodium falciparum, không xác định	
B51	Bệnh sốt rét do plasmodium vivax	
B51.0	Bệnh sốt rét do plasmodium vivax kèm vỡ lách	
B51.8	Bệnh sốt rét do plasmodium vivax kèm biến chứng khác	
B51.9	Bệnh sốt rét do plasmodium vivax không kèm biến chứng	Bệnh sốt rét do plasmodium vivax không xác định khác
B52	Bệnh sốt rét do plasmodium malariae	
B52.0	Bệnh sốt rét do plasmodium malariae kèm bệnh lý thận	
B52.8	Bệnh sốt rét do plasmodium malariae kèm biến chứng khác	
B52.9	Bệnh sốt rét do plasmodium malariae không kèm biến chứng	Bệnh sốt rét do plasmodium malariae không xác định khác
B53	Bệnh sốt rét khác được khẳng định về ký sinh trùng học	
B53.0	Bệnh sốt rét do plasmodium ovale	
B53.1	Bệnh sốt rét do plasmodia ở khỉ	
B53.8	Bệnh sốt rét khác được khẳng định bằng ký sinh trùng học, không phân loại mục khác	Bệnh sốt rét được khẳng định bằng ký sinh trùng học không xác định khác
B54	Bệnh sốt rét không xác định	
B55	Bệnh do nhiễm leishmania	
B55.0	Bệnh do nhiễm leishmania nội tạng	Bệnh Kala-azar|Bệnh do nhiễm leishmania da, hậu Kala-azar
B55.1	Bệnh do nhiễm leishmania da	
B55.2	Bệnh do nhiễm leishmania da niêm mạc	
B55.9	Bệnh do nhiễm leishmania, không xác định	
B56	Bệnh do nhiễm trypanosoma Châu Phi [bệnh ngủ Châu Phi]	
B56.0	Bệnh do nhiễm trypanosoma gambiense	Bệnh truyền nhiễm do trypanosoma brucei gambiense|Bệnh ngủ Tây Phi
B56.1	Bệnh do nhiễm trypanosoma rhodesiense	Bệnh ngủ Đông Phi|Bệnh truyền nhiễm do trypanosoma brucei rhodesiense
B56.9	Bệnh do nhiễm trypanosoma Châu Phi [bệnh ngủ Châu Phi], không xác định	Bệnh ngủ không xác định khác
B57	Bệnh Chagas	
B57.0†	Bệnh Chagas cấp tính có tác động đến tim (I41.2*, I98.1*)	
B57.1	Bệnh Chagas cấp tính không tác động đến tim	Bệnh Chagas cấp tính không xác định khác
B57.2	Bệnh Chagas (mạn tính) có tác động đến tim	
B57.3	Bệnh Chagas (mạn tính) có tác động đến hệ tiêu hóa	
B57.4	Bệnh Chagas (mạn tính) có tác động đến hệ thần kinh	
B57.5	Bệnh Chagas (mạn tính) có tác động đến cơ quan khác	
B58	Bệnh do nhiễm toxoplasma	
B58.0†	Bệnh lý mắt do nhiễm toxoplasma	
B58.1†	Bệnh viêm gan do nhiễm toxoplasma (K77.0*)	
B58.2†	Bệnh viêm não - màng não do nhiễm toxoplasma (G05.2*)	
B58.3†	Bệnh do nhiễm toxoplasma ở phổi (J17.3*)	
B58.8	Bệnh do nhiễm toxoplasma có tác động đến cơ quan khác	
B58.9	Bệnh do nhiễm toxoplasma, không xác định	
B60	Bệnh do ký sinh trùng đơn bào khác, không phân loại mục khác	
B60.0	Bệnh do nhiễm babesia	Bệnh do nhiễm piroplasma
B60.1	Bệnh do nhiễm a-míp acanthamoeba	
B60.2	Bệnh do nhiễm naegleria	
B60.8	Bệnh do nhiễm ký sinh trùng đơn bào xác định khác	Bệnh do nhiễm microsporidia
B64	Bệnh do nhiễm ký sinh trùng đơn bào không xác định	
B65	Bệnh nhiễm sán máng [bilharziasis]	
B65.0	Bệnh nhiễm sán máng do Schistosoma haematobium [bệnh sán máng đường tiết niệu]	
B65.1	Bệnh nhiễm sán máng do Schistosoma mansoni [bệnh sán máng đường ruột]	
B65.2	Bệnh nhiễm sán máng do Schistosoma japonicum	Bệnh sán máng châu Á
B65.3	Bệnh viêm da do ấu trùng sán máng	Bệnh ghẻ ngứa ở người bơi lội
B65.8	Bệnh nhiễm sán máng khác	
B65.9	Bệnh nhiễm sán máng, không xác định	
B66	Bệnh nhiễm sán lá khác	
B66.0	Bệnh sán lá nhỏ Opisthorchis	
B66.1	Bệnh sán lá gan nhỏ Clonorchis	Bệnh sán lá gan Trung Quốc|Bệnh sán lá gan nhỏ Clonorchis sinensis|Bệnh sán lá gan phương đông
B66.2	Bệnh nhiễm sán lá Dicrocoelium	Bệnh sán lá Dicrocoelium dendriticum|Bệnh sán lá Lancet
B66.3	Bệnh nhiễm sán lá gan lớn [Fasciola]	
B66.4	Bệnh nhiễm sán lá phổi [Paragonimus]	Bệnh sán lá do Paragonimus|Bệnh sán lá phổi|Nhiễm sán lá ở phổi
B66.5	Bệnh nhiễm sán lá ruột [Fasciolopsis]	Nhiễm sán lá ruột lớn|Nhiễm sán lá ruột
B66.8	Bệnh nhiễm sán lá xác định khác	Bệnh sán lá ruột nhỏ do echinostoma|Bệnh sán lá ruột nhỏ do heterophyes|Bệnh sán lá ruột nhỏ do metagonimus|Bệnh sán lá ruột nhỏ do nanophyetia|Bệnh do watsoniasis
B66.9	Bệnh nhiễm sán lá, không xác định	
B67	Bệnh nang sán [echinococcus]	
B67.0	Bệnh nhiễm nang sán [hydatid] ở gan	
B67.1	Bệnh nhiễm nang sán [hydatid] ở phổi	
B67.2	Bệnh nhiễm nang sán [hydatid] ở xương	
B67.3	Bệnh nhiễm nang sán [hydatid], ở vị trí khác và/hoặc ở nhiều vị trí	
B67.4	Bệnh nhiễm nang sán [hydatid], không xác định [vị trí nhiễm]	
B67.5	Bệnh nhiễm phế nang [thể nang tổ ong] ở gan	
B67.6	Bệnh nhiễm phế nang [thể nang tổ ong], ở vị trí khác và/hoặc ở nhiều vị trí	
B67.7	Bệnh nhiễm phế nang [thể nang tổ ong], không xác định [vị trí nhiễm]	
B67.8	Bệnh nhiễm sán dây nhỏ echinococcus, không xác định loài, ở gan	
B67.9	Bệnh nhiễm sán dây nhỏ echinococcus, khác và/hoặc không xác định	Nhiễm ấu trùng nang sán [echinococcus] không xác định khác
B68	Bệnh sán dây taenia	
B68.0	Bệnh sán dây lợn [taenia solium]	
B68.1	Bệnh sán dây bò [taenia saginata]	
B68.9	Bệnh sán dây taenia, không xác định	
B69	Bệnh nhiễm ấu trùng sán lợn [bệnh gạo lợn]	
B69.0	Bệnh nhiễm ấu trùng sán lợn [bệnh gạo lợn] ở hệ thần kinh trung ương	
B69.1	Bệnh nhiễm ấu trùng sán lợn [bệnh gạo lợn] ở mắt	
B69.8	Bệnh nhiễm ấu trùng sán lợn [bệnh gạo lợn] ở vị trí khác	
B69.9	Bệnh nhiễm ấu trùng sán lợn [bệnh gạo lợn], không xác định	
B70	Bệnh nhiễm sán dây diphyllobothrium và/hoặc bệnh sán nhái sparganum	
B70.0	Bệnh nhiễm sán dây diphyllobothrium	
B70.1	Bệnh nhiễm sán nhái sparganum	
B71	Bệnh nhiễm sán dây khác	
B71.0	Bệnh sán dải lùn [hymenolepis]	
B71.1	Bệnh nhiễm sán dây dipylidium	
B71.8	Bệnh nhiễm sán dây xác định khác	Bệnh sán nhiều đầu
B71.9	Bệnh nhiễm sán dây, không xác định	
B72	Bệnh nhiễm giun dracunculus [giun Guinea]	
B73	Bệnh nhiễm giun chỉ Onchocerca [mù sông]	
B74	Bệnh nhiễm giun chỉ bạch huyết [Filariasis]	
B74.0	Bệnh nhiễm giun chỉ bạch huyết do Wuchereria bancrofti	
B74.1	Bệnh nhiễm giun chỉ bạch huyết do Brugia malayi	
B74.2	Bệnh nhiễm giun chỉ bạch huyết do Brugia timori	
B74.3	Bệnh nhiễm giun chỉ Loa loa	Phù nề Calabar|Bệnh giun chỉ mắt ở châu Phi|Nhiễm Loa loa
B74.4	Bệnh nhiễm giun chỉ Mansonella	
B74.8	Bệnh nhiễm giun chỉ khác	Bệnh giun chỉ Dirofilaria
B74.9	Bệnh nhiễm giun chỉ, không xác định	
B75	Bệnh nhiễm giun xoắn Trichinella	
B76	Bệnh nhiễm giun móc	
B76.0	Bệnh nhiễm giun móc ancylostoma	Nhiễm loài Ancylostoma
B76.1	Bệnh nhiễm giun mỏ necator	Nhiễm Necator americanus
B76.8	Bệnh nhiễm giun móc khác	
B76.9	Bệnh nhiễm giun móc, không xác định	Bệnh ấu trùng di chuyển ở da không xác định khác
B77	Bệnh nhiễm giun đũa	
B77.0	Bệnh nhiễm giun đũa kèm biến chứng đường ruột	
B77.8	Bệnh nhiễm giun đũa kèm biến chứng khác	
B77.9	Bệnh nhiễm giun đũa, không xác định	
B78	Bệnh nhiễm giun lươn strongyloides	
B78.0	Bệnh nhiễm giun lươn strongyloides đường ruột	
B78.1	Bệnh nhiễm giun lươn strongyloides ở da	
B78.7	Bệnh nhiễm giun lươn strongyloides lan tỏa	
B78.9	Bệnh nhiễm giun lươn strongyloides, không xác định	
B79	Bệnh nhiễm giun tóc trichuris	
B80	Bệnh nhiễm giun kim enterobias	
B81	Bệnh nhiễm giun khác ở đường ruột, không phân loại mục khác	
B81.0	Bệnh nhiễm giun anisakis	Nhiễm ấu trùng anisakis
B81.1	Bệnh nhiễm giun capillaria đường ruột	
B81.2	Bệnh nhiễm giun lươn trichostrongylus	
B81.3	Bệnh nhiễm giun angiostrongylus đường ruột	
B81.4	Bệnh nhiễm giun sán ruột phối hợp	Nhiễm giun sán đường ruột có thể xếp vào nhiều bệnh thuộc B65.0-B81.3 và B81.8|Bệnh giun sán phối hợp không xác định khác
B81.8	Bệnh nhiễm giun sán xác định khác ở đường ruột	
B82	Bệnh nhiễm ký sinh trùng không xác định ở đường ruột	
B82.0	Bệnh nhiễm giun sán ở đường ruột, không xác định	
B82.9	Bệnh nhiễm ký sinh trùng đường ruột, không xác định	
B83	Bệnh nhiễm giun sán khác	
B83.0	Ấu trùng di chuyển trong nội tạng	Bệnh giun đũa chó mèo [toxocariasis]
B83.1	Bệnh nhiễm giun đầu gai [gnathostoma]	Sưng phồng lan tỏa
B83.2	Bệnh nhiễm giun angiostrongylus do giun mạch [parastrongylus cantonensis]	
B83.3	Bệnh nhiễm giun tròn syngamia	
B83.4	Bệnh đỉa ký sinh nội tạng	
B83.8	Bệnh nhiễm giun sán xác định khác	Bệnh nhiễm giun acanthocephala|Bệnh nhiễm giun chỉ gongylonema|Bệnh nhiễm giun capillaria ở gan|Bệnh nhiễm giun phổi metastrongylus|Bệnh nhiễm giun mắt thelazia
B83.9	Bệnh nhiễm giun sán, không xác định	
B85	Bệnh nhiễm chấy và/hoặc rận	
B85.0	Bệnh nhiễm chấy do pediculus humanus capitis [nhiễm chấy ở đầu]	Nhiễm chấy ở đầu
B85.1	Bệnh nhiễm chấy do pediculus humanus corporis [nhiễm chấy trên cơ thể]	Nhiễm chấy trên cơ thể
B85.2	Bệnh nhiễm chấy, không xác định	
B85.3	Bệnh nhiễm rận	
B85.4	Bệnh đồng nhiễm chấy và rận	Bệnh ký sinh có thể phân loại vào các mục B85.0-B85.3
B86	Bệnh ghẻ	
B87	Bệnh giòi [dòi]	
B87.0	Bệnh giòi [dòi] ở da	Bệnh giòi [dòi] di chuyển
B87.1	Bệnh giòi [dòi] trên vết thương	Bệnh giòi [dòi] do chấn thương
B87.2	Bệnh giòi [dòi] ở mắt	
B87.3	Bệnh giòi [dòi] ở mũi họng	Bệnh giòi [dòi] thanh quản
B87.4	Bệnh giòi [dòi] ở tai	
B87.8	Bệnh giòi [dòi] ở vị trí khác	Bệnh giòi [dòi] đường tiết niệu - sinh dục|Bệnh giòi [dòi] đường tiêu hóa
B87.9	Bệnh nhiễm trùng giòi [dòi], không xác định	
B88	Bệnh nhiễm ký sinh trùng khác	
B88.0	Bệnh do bọ ve khác	
B88.1	Bệnh da do bọ Tunga penetrans [do nhiễm bọ chét cát]	
B88.2	Bệnh nhiễm ký sinh trùng do tiết túc khác	Bệnh do scarabia
B88.3	Bệnh do đỉa ngoại ký sinh	
B88.8	Bệnh nhiễm ký sinh trùng xác định khác	Bệnh ký sinh trùng do cá Vandellia cirrhoses [Candiru]|Bệnh nhiễm Linguatolo|Bệnh nhiễm Porocephalus
B88.9	Bệnh nhiễm ký sinh trùng, không xác định	
B89	Bệnh do ký sinh trùng không xác định	
B90	Di chứng của bệnh lao	
B90.0	Di chứng của bệnh lao hệ thần kinh trung ương	
B90.1	Di chứng của bệnh lao tiết niệu - sinh dục	
B90.2	Di chứng của bệnh lao xương và/hoặc khớp	
B90.8	Di chứng của bệnh lao cơ quan khác	
B90.9	Di chứng do bệnh lao hô hấp và/hoặc bệnh lao không xác định	Di chứng của bệnh lao không xác định khác
B91	Di chứng của bệnh bại liệt	
B92	Di chứng của bệnh phong	
B94	Di chứng của bệnh nhiễm trùng và/hoặc ký sinh trùng khác và/hoặc không xác định	
B94.0	Di chứng của bệnh mắt hột	
B94.1	Di chứng của bệnh viêm não do virus	
B94.2	Di chứng của bệnh viêm gan do virus	
B94.8	Di chứng của bệnh nhiễm trùng và/hoặc ký sinh trùng xác định khác	
B94.9	Di chứng của bệnh nhiễm trùng hoặc bệnh ký sinh trùng không xác định	
B95	Liên cầu khuẩn và/hoặc tụ cầu khuẩn là nguyên nhân gây bệnh phân loại ở chương khác	
B95.0	Liên cầu khuẩn, nhóm A, là nguyên nhân gây bệnh phân loại ở chương khác	
B95.1	Liên cầu khuẩn, nhóm B, là nguyên nhân gây bệnh phân loại ở chương khác	
B95.2	Liên cầu khuẩn nhóm D và/hoặc enterococcus là nguyên nhân gây bệnh phân loại ở chương khác	
B95.3	Phế cầu khuẩn là nguyên nhân gây bệnh phân loại ở chương khác	
B95.4	Liên cầu khuẩn khác là nguyên nhân gây bệnh phân loại ở chương khác	
B95.5	Liên cầu khuẩn không xác định là nguyên nhân gây bệnh phân loại ở chương khác	
B95.6	Tụ cầu vàng là nguyên nhân gây bệnh phân loại ở chương khác	
B95.7	Tụ cầu khác là nguyên nhân gây bệnh phân loại ở chương khác	
B95.8	Tụ cầu không xác định là nguyên nhân gây bệnh phân loại ở chương khác	
B96	Tác nhân vi khuẩn xác định khác là nguyên nhân gây bệnh phân loại ở chương khác	
B96.0	Mycoplasma pneumoniae [M. pneumoniae] là nguyên nhân gây bệnh phân loại ở chương khác	Sinh vật giống viêm phổi màng phổi [PPLO]
B96.1	Klebsiella pneumoniae [K. pneumoniae] là nguyên nhân gây bệnh phân loại ở chương khác	
B96.2	Escherichia coli [E. coli] là nguyên nhân gây bệnh phân loại ở chương khác	
B96.3	Haemophilus cúm [H. influenzae] là nguyên nhân gây bệnh phân loại ở chương khác	
B96.4	Proteus (mirabilis) (morganii) là nguyên nhân gây bệnh phân loại ở chương khác	
B96.5	Trực khuẩn (mủ xanh) là nguyên nhân gây bệnh phân loại ở chương khác	
B96.6	Bacillus fragilis [B. fragilis] là nguyên nhân gây bệnh phân loại ở chương khác	
B96.7	Clostridium perfringens [C. Perfringens] gây bệnh đã phân loại ở chương khác	
B96.8	Tác nhân vi khuẩn xác định khác là nguyên nhân gây bệnh phân loại ở chương khác	
B97	Tác nhân virus là nguyên nhân gây bệnh phân loại ở chương khác	
B97.0	Virus adeno là nguyên nhân gây bệnh phân loại ở chương khác	
B97.1	Virus lây truyền qua đường ruột [enterovirus] là nguyên nhân gây bệnh phân loại ở chương khác	Virus coxsackie|Virus echo
B97.2	Virus corona là nguyên nhân gây bệnh phân loại ở chương khác	
B97.3	Virus retro là nguyên nhân gây bệnh phân loại ở chương khác	Virus lenti|Virus onco
B97.4	Virus hợp bào hô hấp [RSV] là nguyên nhân gây bệnh phân loại ở chương khác	
B97.5	Reovirus là nguyên nhân gây bệnh phân loại ở chương khác	
B97.6	Virus parvo là nguyên nhân gây bệnh phân loại ở chương khác	
B97.7	Virus papilloma là nguyên nhân gây bệnh phân loại ở chương khác	
B97.8	Tác nhân virus khác là nguyên nhân gây bệnh phân loại ở chương khác	Nhiễm virus gây bệnh đường hô hấp ở người [metapneumovirus]
B98	Tác nhân nhiễm trùng xác định khác là nguyên nhân gây bệnh phân loại ở chương khác	
B98.0	Vi khuẩn Helicobacter pylori [H.pylori] là nguyên nhân gây bệnh phân loại ở chương khác	
B98.1	Vibrio vulnificus là nguyên nhân gây bệnh phân loại ở chương khác	
B99	Bệnh truyền nhiễm khác và/hoặc không xác định	
C00	U ác tính ở môi	
C00.0	U ác tính ở phần ngoài môi trên	
C00.1	U ác tính ở phần ngoài môi dưới	
C00.2	U ác tính ở phần ngoài môi, không xác định môi trên hoặc môi đưới	Đường viền [bờ đỏ son] của môi không xác định khác
C00.3	U ác tính ở môi trên, mặt trong	
C00.4	U ác tính ở môi dưới, mặt trong	
C00.5	U ác tính ở môi, mặt trong, không xác định môi trên hoặc môi dưới	
C00.6	U ác tính ở mép môi	
C00.8	U ác tính có tổn thương chồng lấn ở môi	
C00.9	U ác tính ở môi, không xác định	
C01	U ác tính ở gốc [rễ] lưỡi	
C02	U ác tính ở phần khác và/hoặc không xác định của lưỡi	
C02.0	U ác tính ở mặt lưng của lưỡi	
C02.1	U ác tính ở bờ của lưỡi	Đầu lưỡi
C02.2	U ác tính ở mặt bụng [dưới] của lưỡi	Mặt bụng [dưới] của hai phần ba [2/3] trước của lưỡi|Phanh [thắng] lưỡi
C02.3	U ác tính ở hai phần ba [2/3] trước của lưỡi, phần không xác định	Một phần ba [1/3] giữa của lưỡi không xác định khác|Phần di động của lưỡi không xác định khác
C02.4	U ác tính ở amydan lưỡi	
C02.8	U ác tính có tổn thương chồng lấn ở lưỡi	
C02.9	U ác tính ở lưỡi, không xác định	
C03	U ác tính ở lợi [nướu răng]	
C03.0	U ác tính ở lợi [nướu] hàm trên	
C03.1	U ác tính ở lợi [nướu] hàm dưới	
C03.9	U ác tính ở lợi [nướu], không xác định	
C04	U ác tính ở sàn miệng	
C04.0	U ác tính ở sàn miệng trước	Trước nơi nối răng nanh - răng hàm
C04.1	U ác tính ở sàn miệng bên	
C04.8	U ác tính có tổn thương chồng lấn ở sàn miệng	
C04.9	U ác tính ở sàn miệng, không xác định	
C05	U ác tính ở khẩu cái [vòm miệng]	U ác tính ở khẩu cái
C05.0	U ác tính ở khẩu cái [vòm miệng] cứng	
C05.1	U ác tính ở khẩu cái [vòm miệng] mềm	
C05.2	U ác tính ở lưỡi gà	
C05.8	U ác tính có tổn thương chồng lấn ở khẩu cái [vòm miệng]	
C05.9	U ác tính ở khẩu cái [vòm miệng], không xác định	Vòm miệng
C06	U ác tính ở phần khác và/hoặc không xác định của miệng	
C06.0	U ác tính ở niêm mạc má	Niêm mạc miệng không xác định khác|Bên trong của má
C06.1	U ác tính ở tiền đình của miệng	
C06.2	U ác tính ở vùng hậu hàm	
C06.8	U ác tính có tổn thương chồng lấn của phần khác và/hoặc không xác định ở miệng	
C06.9	U ác tính ở miệng, không xác định	Tuyến nước bọt phụ, vị trí không xác định|Khoang miệng không xác định khác
C07	U ác tính ở tuyến mang tai	
C08	U ác tính ở tuyến nước bọt chính khác và/hoặc không xác định	
C08.0	U ác tính ở tuyến nước bọt dưới hàm	Tuyến nước bọt dưới hàm
C08.1	U ác tính ở tuyến nước bọt dưới lưỡi	
C08.8	U ác tính có tổn thương chồng lấn ở tuyến nước bọt chính	
C08.9	U ác tính ở tuyến nước bọt chính, không xác định	
C09	U ác tính ở amydan	
C09.0	U ác tính ở hố amydan	
C09.1	U ác tính ở trụ amydan (trước) (sau)	
C09.8	U ác tính có tổn thương chồng lấn ở amydan	
C09.9	U ác tính ở amydan, không xác định	
C10	U ác tính ở miệng - hầu	U ác tính ở miệng-hầu
C10.0	U ác tính ở rãnh nhỏ	
C10.1	U ác tính ở mặt trước của nắp thanh môn	
C10.2	U ác tính ở thành bên miệng - hầu	
C10.3	U ác tính ở thành sau miệng - hầu	
C10.4	U ác tính ở khe mang	Nang khe mang [vị trí của u tân sinh]
C10.8	U ác tính với tổn thương chồng lấn ở miệng - hầu	
C10.9	U ác tính ở miệng - hầu, không xác định	
C11	U ác tính ở mũi - hầu	U ác tính ở mũi-hầu
C11.0	U ác tính ở vách trên mũi - hầu	Vòm mũi - hầu
C11.1	U ác tính ở vách sau của mũi - hầu	Amydan họng
C11.2	U ác tính ở vách bên của mũi - hầu	Hố Rosenmüller|Mở vòi nhĩ|Ngách hầu
C11.3	U ác tính ở vách trước của mũi - hầu	
C11.8	U ác tính có tổn thương chồng lấn ở mũi - hầu	
C11.9	U ác tính ở mũi - hầu, không xác định	Vách mũi - hầu không xác định khác
C12	U ác tính ở xoang lê	
C13	U ác tính ở hạ họng	
C13.0	U ác tính ở vùng sau sụn nhẫn	
C13.1	U ác tính ở nếp sụn phễu - nắp thanh quản, phía hạ họng	
C13.2	U ác tính ở thành sau của hạ họng	
C13.8	U ác tính có tổn thương chồng lấn ở hạ họng	
C13.9	U ác tính ở hạ họng, không xác định	Vách hạ họng không xác định khác
C14	U ác tính ở vị trí khác và/hoặc không rõ ràng ở môi, khoang miệng và/hoặc họng	
C14.0	U ác tính ở họng, không xác định	
C14.2	U ác tính ở vòng bạch huyết Waldeyer	
C14.8	U ác tính có tổn thương chồng lấn ở môi, khoang miệng và/hoặc họng	
C15	U ác tính ở thực quản	
C15.0	U ác tính ở thực quản ở phần cổ	
C15.1	U ác tính ở thực quản ở phần ngực	
C15.2	U ác tính ở thực quản ở phần bụng	
C15.3	U ác tính ở một phần ba [1/3] trên thực quản	
C15.4	U ác tính ở một phần ba [1/3] giữa thực quản	
C15.5	U ác tính ở một phần ba [1/3] dưới thực quản	
C15.8	U ác tính với tổn thương chồng lấn ở thực quản	
C15.9	U ác tính ở thực quản, vị trí không xác định	
C16	U ác tính ở dạ dày	
C16.0	U ác tính ở tâm vị	Lỗ tâm vị|Vùng nối tâm vị - thực quản|Vùng nối dạ dày - thực quản|Thực quản và dạ dày
C16.1	U ác tính ở đáy vị	
C16.2	U ác tính ở thân vị	
C16.3	U ác tính ở hang môn vị	Hang vị
C16.4	U ác tính ở môn vị	Tiền môn vị|Ống môn vị
C16.5	U ác tính ở bờ cong nhỏ dạ dày, không xác định	Bờ cong nhỏ dạ dày, không phân loại ở C16.1 - C16.4
C16.6	U ác tính ở bờ cong lớn dạ dày, không xác định	Bờ cong lớn dạ dày, không phân loại ở C16.0- C16.4
C16.8	U ác tính với tổn thương chồng lấn ở dạ dày	
C16.9	U ác tính ở dạ dày, không xác định	Ung thư dạ dày không xác định khác
C17	U ác tính ở ruột non	
C17.0	U ác tính ở tá tràng	
C17.1	U ác tính ở hỗng tràng	
C17.2	U ác tính ở hồi tràng	
C17.3	U ác tính ở túi thừa Meckel	
C17.8	U ác tính với tổn thương chồng lấn ở ruột non	
C17.9	U ác tính ở ruột non, không xác định	
C18	U ác tính ở đại tràng	
C18.0	U ác tính ở manh tràng	Van hồi - manh tràng
C18.1	U ác tính ở ruột thừa	
C18.2	U ác tính ở đại tràng lên [ruột kết lên]	
C18.3	U ác tính ở đại tràng góc gan	
C18.4	U ác tính ở đại tràng ngang [ruột kết ngang]	
C18.5	U ác tính ở đại tràng góc lách	
C18.6	U ác tính ở đại tràng xuống [ruột kết xuống]	
C18.7	U ác tính ở đại tràng sigma	
C18.8	U ác tính với tổn thương chồng lấn ở đại tràng	
C18.9	U ác tính ở đại tràng, không xác định	Ruột già [đại tràng] không xác định khác
C19	U ác tính ở nơi nối trực tràng sigma	
C20	U ác tính ở trực tràng	
C21	U ác tính ở hậu môn và/hoặc ống hậu môn	
C21.0	U ác tính ở hậu môn, không xác định	
C21.1	U ác tính ở ống hậu môn	Cơ vòng [cơ thắt] hậu môn
C21.2	U ác tính ở vùng có nguồn gốc từ ổ nhớp	
C21.8	U ác tính với tổn thương chồng lấn ở đại tràng, hậu môn và/hoặc ống hậu môn	
C22	U ác tính ở gan và/hoặc đường mật trong gan	
C22.0	Ung thư biểu mô tế bào gan	U gan
C22.1	Ung thư biểu mô ống mật trong gan	Ung thư biểu mô đường mật
C22.2	U nguyên bào gan	
C22.3	U ác tính [sarcoma] mạch máu của gan	Sarcoma tế bào Kupffer
C22.4	U ác tính [sarcoma] khác ở gan	
C22.7	Ung thư biểu mô xác định khác ở gan	
C22.9	U ác tính ở gan, không xác định	
C23	U ác tính ở túi mật	
C24	U ác tính ở phần khác và/hoặc phần không xác định của đường mật	
C24.0	U ác tính ở ống mật ngoài gan	Đường mật hoặc ống mật không xác định khác|Ống mật chủ|Ống túi mật|Ống gan
C24.1	U ác tính ở bóng Vater	
C24.8	U ác tính với tổn thương chồng lấn ở đường mật	
C24.9	U ác tính ở đường mật, không xác định	
C25	U ác tính ở tụy	
C25.0	U ác tính ở đầu tụy	
C25.1	U ác tính ở thân tụy	
C25.2	U ác tính ở đuôi tụy	
C25.3	U ác tính ở ống tụy	
C25.4	U ác tính ở tụy nội tiết	Đảo tụy [đảo Langerhans]
C25.7	U ác tính ở phần khác của tụy	Cổ tụy
C25.8	U ác tính với tổn thương chồng lấn ở tụy	
C25.9	U ác tính ở tụỵ, không xác định	
C26	U ác tính ở cơ quan tiêu hóa khác và/hoặc không rõ ràng	
C26.0	U ác tính ở đường ruột, phần không xác định	Đường ruột không xác định khác
C26.1	U ác tính ở lách	
C26.8	U ác tính với tổn thương chồng lấn ở hệ tiêu hóa	
C26.9	U ác tính ở vị trí không rõ ràng của hệ tiêu hóa	Ống hoặc đường tiêu hóa không xác định khác|Ống dạ dày - ruột không xác định khác
C30	U ác tính ở khoang mũi và/hoặc tai giữa	
C30.0	U ác tính ở khoang mũi	
C30.1	U ác tính ở tai giữa	
C31	U ác tính ở xoang phụ	
C31.0	U ác tính ở xoang hàm	
C31.1	U ác tính ở xoang sàng	
C31.2	U ác tính ở xoang trán	
C31.3	U ác tính ở xoang bướm	
C31.8	U ác tính với tổn thương chồng lấn ở xoang phụ	
C31.9	U ác tính ở xoang phụ, không xác định	
C32	U ác tính ở thanh quản	
C32.0	U ác tính ở thanh môn	
C32.1	U ác tính ở vùng trên thanh môn	
C32.2	U ác tính ở vùng dưới thanh môn	
C32.3	U ác tính ở sụn thanh quản	
C32.8	U ác tính với tổn thương chồng lấn ở thanh quản	
C32.9	U ác tính ở thanh quản, không xác định	
C33	U ác tính ở khí quản	
C34	U ác tính ở phế quản và/hoặc phổi	
C34.0	U ác tính ở phế quản chính	Ngã ba khí phế quản [Carina]|Rốn phổi
C34.1	U ác tính ở thùy trên, phế quản hoặc phổi	
C34.2	U ác tính ở thùy giữa, phế quản hoặc phổi	
C34.3	U ác tính ở thùy dưới, phế quản hoặc phổi	
C34.8	U ác tính với tổn thương chồng lấn ở phế quản và/hoặc phổi	
C34.9	U ác tính ở phế quản hoặc phổi, không xác định	
C37	U ác tính ở tuyến ức	
C38	U ác tính ở tim, trung thất và/hoặc màng phổi	
C38.0	U ác tính ở tim	
C38.1	U ác tính ở trung thất trước	
C38.2	U ác tính ở trung thất sau	
C38.3	U ác tính ở trung thất, phần không xác định	
C38.4	U ác tính ở màng phổi	
C38.8	U ác tính với tổn thương chồng lấn ở tim, trung thất và/hoặc màng phổi	
C39	U ác tính ở những vị trí khác và/hoặc không rõ ràng của hệ hô hấp và/hoặc cơ quan trong khoang ngực	
C39.0	U ác tính ở đường hô hấp trên, phần không xác định	
C39.8	U ác tính với tổn thương chồng lấn ở cơ quan hô hấp và/hoặc cơ quan trong khoang ngực	
C39.9	U ác tính ở vị trí không rõ ràng ở hệ hô hấp	Đường hô hấp không xác định khác
C40	U ác tính ở xương và/hoặc sụn khớp của các chi	
C40.0	U ác tính ở xương bả vai và/hoặc xương dài của chi trên	
C40.1	U ác tính ở xương ngắn của chi trên	
C40.2	U ác tính ở xương dài của chi dưới	
C40.3	U ác tính ở xương ngắn của chi dưới	
C40.8	U ác tính với tổn thương chồng lấn ở xương và/hoặc sụn khớp của các chi	
C40.9	U ác tính ở xương và/hoặc sụn khớp của chi, vị trí không xác định	
C41	U ác tính ở vị trí khác và/hoặc không xác định ở xương và/hoặc sụn khớp	
C41.0	U ác tính ở xương sọ và/hoặc xương mặt	
C41.1	U ác tính ở xương hàm dưới	
C41.2	U ác tính ở cột sống	
C41.3	U ác tính ở xương sườn, xương ức và/hoặc xương đòn	
C41.4	U ác tính ở xương chậu, xương cùng và/hoặc xương cụt	
C41.8	U ác tính với tổn thương chồng lấn ở xương và/hoặc sụn khớp	
C41.9	U ác tính ở xương và/hoặc sụn khớp, vị trí không xác định	
C43	U hắc tố ác tính ở da	
C43.0	U hắc tố ác tính ở môi	
C43.1	U hắc tố ác tính ở mi mắt, bao gồm khóe mắt	
C43.2	U hắc tố ác tính ở tai và/hoặc ống tai ngoài	
C43.3	U hắc tố ác tính ở phần khác và/hoặc không xác định ở mặt	
C43.4	U hắc tố ác tính ở đầu và/hoặc cổ	
C43.5	U hắc tố ác tính ở thân	
C43.6	U hắc tố ác tính ở chi trên, bao gồm vai	
C43.7	U hắc tố ác tính ở chi dưới, bao gồm hông	
C43.8	U hắc tố ác tính chồng lấn ở da	
C43.9	U hắc tố ác tính ở da, vị trí không xác định	
C44	U tân sinh ác tính khác ở da	
C44.0	U tân sinh ác tính khác ở da môi	
C44.1	U tân sinh ác tính khác ở da mi mắt, bao gồm khóe mắt	
C44.2	U tân sinh ác tính khác ở da tai và/hoặc ống tai ngoài	
C44.3	U tân sinh ac tính khác ở phần khác và/hoặc không xác định ở da mặt	
C44.4	U tân sinh ác tính khác ở da đầu và/hoặc ở cổ	
C44.5	U tân sinh ác tính khác ở da thân	
C44.6	U tân sinh ác tính khác ở da chi trên, bao gồm vai	
C44.7	U tân sinh ác tính khác ở da chi dưới, bao gồm hông	
C44.8	U tân sinh ác tính khác với tổn thương chồng lấn ở da	
C44.9	U tân sinh ác tính khác ở da, vị trí không xác định	
C45	U trung biểu mô	
C45.0	U trung biểu mô ở màng phổi	
C45.1	U trung biểu mô ở phúc mạc	
C45.2	U trung biểu mô ở màng ngoài tim	
C45.7	U trung biểu mô ở vị trí khác	
C45.9	U trung biểu mô, vị trí không xác định	
C46	Ung thư [sarcoma] Kaposi	
C46.0	Ung thư [sarcoma] Kaposi ở da	
C46.1	Ung thư [sarcoma] Kaposi ở mô mềm	
C46.2	Ung thư [sarcoma] Kaposi ở khẩu cái [vòm miệng]	
C46.3	Ung thư [sarcoma] Kaposi ở hạch lympho	
C46.7	Ung thư [sarcoma] Kaposi ở cơ quan khác	
C46.8	Ung thư [sarcoma] Kaposi ở nhiều cơ quan	
C46.9	Ung thư [sarcoma] Kaposi, không xác định	
C47	U ác tính ở dây thần kinh ngoại biên và/hoặc hệ thần kinh tự động	
C47.0	U ác tính ở dây thần kinh ngoại biên của đầu, mặt và/hoặc cổ	
C47.1	U ác tính ở dây thần kinh ngoại biên của chi trên, bao gồm vai	
C47.2	U ác tính ở dây thần kinh ngoại biên của chi dưới, bao gồm hông	
C47.3	U ác tính ở dây thần kinh ngoại biên của lồng ngực	
C47.4	U ác tính ở dây thần kinh ngoại biên của bụng	
C47.5	U ác tính ở dây thần kinh ngoại biên của vùng chậu	
C47.6	U ác tính ở dây thần kinh ngoại biên của thân, vị trí không xác định	
C47.8	U ác tính với tổn thương chồng lấn ở dây thần kinh ngoại biên và/hoặc hệ thần kinh tự động	
C47.9	U ác tính ở dây thần kinh ngoại biên và/hoặc hệ thần kinh tự động, không xác định	
C48	U ác tính ở vùng sau phúc mạc và/hoặc phúc mạc	
C48.0	U ác tính ở vùng sau phúc mạc	
C48.1	U ác tính ở vùng xác định của phúc mạc	Mạc treo|Mạc treo ruột già [đại tràng]|Mạc nối
C48.2	U ác tính ở phúc mạc, vùng không xác định	
C48.8	U ác tính với tổn thương chồng lấn ở vùng sau phúc mạc và/hoặc phúc mạc	
C49	U ác tính ở mô liên kết và/hoặc mô mềm khác	
C49.0	U ác tính ở mô liên kết và/hoặc mô mềm của đầu, mặt và/hoặc cổ	
C49.1	U ác tính ở mô liên kết và/hoặc mô mềm của chi trên, bao gồm vai	
C49.2	U ác tính ở mô liên kết và/hoặc mô mềm của chi dưới, bao gồm hông	
C49.3	U ác tính ở mô liên kết và/hoặc mô mềm của lồng ngực	
C49.4	U ác tính ở mô liên kết và/hoặc mô mềm của bụng	Thành bụng|Hạ sườn
C49.5	U ác tính ở mô liên kết và/hoặc mô mềm của vùng chậu	Mông|Hội âm
C49.6	U ác tính ở mô liên kết và/hoặc mô mềm của thân, không xác định	Lưng không xác định khác
C49.8	U ác tính với tổn thương chồng lấn ở mô liên kết và/hoặc mô mềm	
C49.9	U ác tính ở mô liên kết và/hoặc mô mềm, không xác định	
C50	U ác tính ở vú	
C50.0	U ác tính ở núm và/hoặc quầng vú	
C50.1	U ác tính ở vùng trung tâm vú	
C50.2	U ác tính ở một phần tư [1/4] trên - trong vú	
C50.3	U ác tính ở một phần tư [1/4] dưới - trong vú	
C50.4	U ác tính ở một phần tư [1/4] trên - ngoài vú	
C50.5	U ác tính ở một phần tư [1/4] dưới - ngoài vú	
C50.6	U ác tính ở đuôi nách của vú	
C50.8	U ác tính với tổn thương chồng lấn ở vú	
C50.9	U ác tính ở vú, không xác định	
C51	U ác tính ở âm hộ	
C51.0	U ác tính ở môi lớn thuộc âm hộ	Tuyến Bartholin [tuyến tiền đình lớn hơn]
C51.1	U ác tính ở môi bé thuộc âm hộ	
C51.2	U ác tính ở âm vật	
C51.8	U ác tính với tổn thương chồng lấn ở âm hộ	
C51.9	U ác tính ở âm hộ, không xác định	Cơ quan sinh dục ngoài của nữ không xác định khác|Âm hộ
C52	U ác tính ở âm đạo	
C53	U ác tính ở cổ tử cung	
C53.0	U ác tính trong cổ tử cung	
C53.1	U ác tính ở cổ tử cung ngoài	
C53.8	U ác tính có tổn thương chồng lấn ở cổ tử cung	
C53.9	U ác tính ở cổ tử cung, không xác định	
C54	U ác tính ở thân tử cung	
C54.0	U ác tính ở eo tử cung	Phần dưới tử cung
C54.1	U ác tính ở nội mạc tử cung	
C54.2	U ác tính ở cơ tử cung	
C54.3	U ác tính ở đáy tử cung	
C54.8	U ác tính có tổn thương chồng lấn ở thân tử cung	
C54.9	U ác tính ở thân tử cung, không xác định	
C55	U ác tính ở tử cung, phần không xác định	
C56	U ác tính ở buồng trứng	
C57	U ác tính ở cơ quan sinh dục khác và/hoặc không xác định ở nữ giới	
C57.0	U ác tính ở vòi trứng [vòi Fallop]	Vòi trứng|Ống dẫn trứng
C57.1	U ác tính ở dây chằng rộng [tử cung]	
C57.2	U ác tính ở dây chằng tròn [tử cung]	
C57.3	U ác tính ở mô cận tử cung	Dây chằng tử cung không xác định khác
C57.4	U ác tính không xác định ở phần phụ tử cung	
C57.7	U ác tính ở cơ quan sinh dục xác định khác ở nữ giới	Thân hoặc ống Wolff
C57.8	U ác tính có tổn thương chồng lấn ở cơ quan sinh dục nữ	
C57.9	U ác tính ở cơ quan sinh dục nữ, không xác định	Ống niệu sinh dục nữ không xác định khác
C58	U ác tính ở rau thai [nhau thai]	
C60	U ác tính ở dương vật	
C60.0	U ác tính ở bao quy đầu	Bao quy đầu
C60.1	U ác tính ở quy đầu dương vật	
C60.2	U ác tính ở thân dương vật	Thể hang
C60.8	U ác tính có tổn thương chồng lấn ở dương vật	
C60.9	U ác tính ở dương vật, không xác định	Da ở dương vật không xác định khác
C61	U ác tính ở tuyến tiền liệt	
C62	U ác tính ở tinh hoàn	
C62.0	U ác tính ở tinh hoàn ẩn [chưa xuống bìu]	Tinh hoàn lạc chỗ [vị trí của u tân sinh]|Tinh hoàn bị giữ lại [vị trí của u tân sinh]
C62.1	U ác tính ở tinh hoàn đã xuống bìu	Tinh hoàn ở trong bìu
C62.9	U ác tính ở tinh hoàn, không xác định	
C63	U ác tính ở cơ quan sinh dục khác và/hoặc không xác định ở nam giới	
C63.0	U ác tính ở mào tinh	
C63.1	U ác tính ở thừng tinh	
C63.2	U ác tính ở bìu	Da bìu
C63.7	U ác tính xác định khác ở cơ quan sinh dục nam	Túi tinh|Tinh mạc
C63.8	U ác tính có tổn thương chồng lấn ở cơ quan sinh dục nam	
C63.9	U ác tính ở cơ quan sinh dục nam, không xác định	Ống niệu sinh dục nam không xác định khác
C64	U ác tính ở thận, ngoại trừ bể thận	
C65	U ác tính ở bể thận	
C66	U ác tính ở niệu quản	
C67	U ác tính ở bàng quang	
C67.0	U ác tính ở tam giác bàng quang	
C67.1	U ác tính ở đáy bàng quang	
C67.2	U ác tính ở thành bên bàng quang	
C67.3	U ác tính ở thành trước bàng quang	
C67.4	U ác tính ở thành sau bàng quang	
C67.5	U ác tính ở cổ bàng quang	Lỗ niệu đạo trong
C67.6	U ác tính ở lỗ niệu quản	
C67.7	U ác tính ở dây treo bàng quang	
C67.8	U ác tính có tổn thương chồng lấn ở bàng quang	
C67.9	U ác tính ở bàng quang, không xác định	
C68	U ác tính ở cơ quan tiết niệu khác và/hoặc không xác định	
C68.0	U ác tính ở niệu đạo	
C68.1	U ác tính ở tuyến cận niệu đạo	
C68.8	U ác tính có tổn thương chồng lấn ở cơ quan tiết niệu	
C68.9	U ác tính ở cơ quan tiết niệu, không xác định	Hệ tiết niệu không xác định khác
C69	U ác tính ở mắt và/hoặc cấu trúc phụ cận của mắt	
C69.0	U ác tính ở kết mạc	
C69.1	U ác tính ở củng mạc	
C69.2	U ác tính ở võng mạc	
C69.3	U ác tính ở màng mạch	
C69.4	U ác tính ở thể mi	
C69.5	U ác tính ở tuyến lệ và/hoặc ống lệ	Túi lệ|Ống mũi lệ
C69.6	U ác tính ở hốc mắt	
C69.8	U ác tính có tổn thương chồng lấn ở mắt và/hoặc cấu trúc phụ cận của mắt	
C69.9	U ác tính ở mắt, không xác định	Nhãn cầu
C70	U ác tính ở màng não tủy	
C70.0	U ác tính ở màng não	
C70.1	U ác tính ở màng tủy	
C70.9	U ác tính ở màng não tủy, không xác định	
C71	U ác tính ở não	
C71.0	U ác tính ở đại não, ngoại trừ thùy não và não thất	Trên lều không xác định khác
C71.1	U ác tính ở thùy trán	
C71.2	U ác tính ở thùy thái dương	
C71.3	U ác tính ở thùy đỉnh	
C71.4	U ác tính ở thùy chẩm	
C71.5	U ác tính ở não thất	
C71.6	U ác tính ở tiểu não	
C71.7	U ác tính ở cuống não	Não thất IV|Dưới lều không xác định khác
C71.8	U ác tính có tổn thương chồng lấn ở não	
C71.9	U ác tính ở não, không xác định	
C72	U ác tính ở tủy sống, dây thần kinh sọ và/hoặc phần khác của hệ thần kinh trung ương	
C72.0	U ác tính ở tủy sống	
C72.1	U ác tính ở chùm đuôi ngựa	
C72.2	U ác tính ở thần kinh khứu giác	Hành khứu giác
C72.3	U ác tính ở thần kinh thị giác	
C72.4	U ác tính ở thần kinh thính giác	
C72.5	U ác tính ở dây thần kinh sọ khác và/hoặc không xác định	Thần kinh sọ không xác định khác
C72.8	U ác tính có tổn thương chồng lấn ở não và/hoặc phần khác của hệ thần kinh trung ương	
C72.9	U ác tính ở hệ thần kinh trung ương, không xác định	Hệ thần kinh không xác định khác
C73	U ác tính ở tuyến giáp	
C74	U ác tính ở tuyến thượng thận	
C74.0	U ác tính ở vỏ tuyến thượng thận	
C74.1	U ác tính ở tủy tuyến thượng thận	
C74.9	U ác tính ở tuyến thượng thận, không xác định	
C75	U ác tính ở tuyến nội tiết khác và/hoặc cấu trúc liên quan	
C75.0	U ác tính ở tuyến cận giáp	
C75.1	U ác tính ở tuyến yên	
C75.2	U ác tính ở ống sọ hầu	
C75.3	U ác tính ở tuyến tùng	
C75.4	U ác tính tiểu thể của động mạch cảnh	
C75.5	U ác tính tiểu thể của động mạch chủ và/hoặc cận hạch thần kinh khác	
C75.8	U ác tính có tác động đến nhiều tuyến, không xác định	
C75.9	U ác tính ở tuyến nội tiết, không xác định	
C76	U ác tính ở vị trí khác và/hoặc không rõ ràng	
C76.0	U ác tính ở vị trí khác và/hoặc không rõ ràng ở đầu, mặt và/hoặc cổ	Má không xác định khác|Mũi không xác định khác
C76.1	U ác tính ở vị trí khác và/hoặc không rõ ràng ở ngực	Nách không xác định khác|Nội lồng ngực không xác định khác|Lồng ngực không xác định khác
C76.2	U ác tính ở vị trí khác và/hoặc vị trí không rõ ràng ở bụng	
C76.3	U ác tính ở vị trí khác và/hoặc không rõ ràng ở vùng chậu	
C76.4	U ác tính ở vị trí khác và/hoặc không rõ ràng ở chi trên	
C76.5	U ác tính ở vị trí khác và/hoặc không rõ ràng ở chi dưới	
C76.7	U ác tính ở vị trí không rõ ràng khác	
C76.8	U ác tính có tổn thương chồng lấn ở vị trí khác và/hoặc không rõ ràng	
C77	U ác tính thứ phát và/hoặc không xác định ở hạch lympho	
C77.0	U ác tính ở hạch vùng đầu, mặt và/hoặc cổ	Hạch trên đòn
C77.1	U ác tính thứ phát và/hoặc không xác định ở hạch trong khoang ngực	
C77.2	U ác tính thứ phát và/hoặc không xác định ở hạch trong ổ bụng	
C77.3	U ác tính thứ phát và/hoặc không xác định ở hạch nách và/hoặc hạch chi trên	Hạch cơ ngực
C77.4	U ác tính thứ phát và/hoặc không xác định ở hạch bẹn và/hoặc hạch chi dưới	
C77.5	U ác tính thứ phát và/hoặc không xác định ở hạch trong vùng chậu	
C77.8	U ác tính thứ phát và/hoặc không xác định ở hạch ở nhiều vùng	
C77.9	U ác tính thứ phát và/hoặc không xác định ở hạch lympho, vị trí không xác định	
C78	U ác tính thứ phát ở cơ quan hô hấp và/hoặc cơ quan tiêu hóa	
C78.0	U ác tính thứ phát ở phổi	
C78.1	U ác tính thứ phát ở trung thất	
C78.2	U ác tính thứ phát ở màng phổi	Tràn dịch màng phổi ác tính không xác định khác
C78.3	U ác tính thứ phát ở cơ quan hô hấp khác và/hoặc không xác định	
C78.4	U ác tính thứ phát ở ruột non	
C78.5	U ác tính thứ phát ở đại tràng và/hoặc trực tràng	
C78.6	U ác tính thứ phát ở vùng sau phúc mạc và/hoặc phúc mạc	Cổ trướng ác tính không xác định khác
C78.7	U ác tính thứ phát ở gan và/hoặc đường mật trong gan	
C78.8	U ác tính thứ phát ở cơ quan tiêu hóa khác và/hoặc không xác định	
C79	U ác tính thứ phát ở vị trí khác và/hoặc không xác định	
C79.0	U ác tính thứ phát ở thận và/hoặc bể thận	
C79.1	U ác tính thứ phát ở bàng quang và/hoặc cơ quan tiết niệu khác và/hoặc không xác định	
C79.2	U ác tính thứ phát ở da	
C79.3	U ác tính thứ phát ở não và/hoặc màng não	
C79.4	U ác tính thứ phát ở phần khác và/hoặc không xác định của hệ thần kinh	
C79.5	U ác tính thứ phát ở xương và/hoặc tủy xương	
C79.6	U ác tính thứ phát ở buồng trứng	
C79.7	U ác tính thứ phát ở tuyến thượng thận	
C79.8	U ác tính thứ phát ở vị trí xác định khác	
C79.9	U ác tính thứ phát, vị trí không xác định	
C80	U ác tính, vị trí không xác định	
C80.0	U ác tính, đã khẳng định, không biết vị trí nguyên phát	Không biết vị trí nguyên phát
C80.9	U ác tính, vị trí nguyên phát không xác định	
C81	U lympho Hodgkin	
C81.0	U lympho Hodgkin dạng nốt ưu thế lympho bào	
C81.1	U lympho Hodgkin (kinh điển) thể xơ nốt	
C81.2	U lympho Hodgkin (kinh điển) thể hỗn hợp tế bào	
C81.3	U lympho Hodgkin (kinh điển) thể giảm lympho bào	
C81.4	U lympho Hodgkin (kinh điển) thể giàu lympho bào	
C81.7	U lympho Hodgkin (kinh điển) thể khác	U lympho Hodgkin kinh điển, loại không xác định
C81.9	U lympho Hodgkin, thể không xác định	
C82	U lympho thể nang	
C82.0	U lympho thể nang độ I	
C82.1	U lympho thể nang độ II	
C82.2	U lympho thể nang độ III, không xác định	
C82.3	U lympho thể nang độ IIIa	
C82.4	U lympho thể nang độ IIIb	
C82.5	U lympho trung tâm nang thể lan tỏa	
C82.6	U lympho trung tâm thể nang da	
C82.7	Loại khác của u lympho thể nang	
C82.9	U lympho thể nang, không xác định	U lympho dạng nốt không xác định khác
C83	U lympho không phải thể nang	
C83.0	U lympho tế bào B nhỏ	
C83.1	U lympho tế bào Mantle	U lympho trung bào|Bệnh polyp thể lympho ác tính
C83.3	U lympho tế bào B lớn lan tỏa	
C83.5	U lympho dòng nguyên bào lympho (lan tỏa)	U lympho tiền tế bào B|U lympho tế bào B nguyên bào lympho|U lympho nguyên bào lympho không xác định khác|U lympho tế bào T nguyên bào lympho|U lympho tiền tế bào T
C83.7	U lympho Burkitt	
C83.8	U lympho không phải thể nang khác	
C83.9	U lympho (lan tỏa) không phải thể nang, không xác định	
C84	U lympho tế bào T/NK trưởng thành	
C84.0	U sùi dạng nấm	
C84.1	Bệnh Sézary	
C84.4	U lympho tế bào T ngoại biên, không phân loại mục khác	U lympho Lennert|U lympho dòng lympho - ái toan
C84.5	U lympho tế bào T/NK trưởng thành khác	
C84.6	U lympho tế bào lớn bất thục sản, ALK dương tính	U lympho tế bào lớn bất thục sản, CD30 - dương tính
C84.7	U lympho tế bào lớn bất thục sản, ALK âm tính	
C84.8	U lympho tế bào T ở da, không xác định	
C84.9	U lympho tế bào T/NK trưởng thành, không xác định	
C85	U lympho không Hodgkin loại khác và/hoặc không xác định	
C85.1	U lympho tế bào B, không xác định	
C85.2	U lympho tế bào B lớn (tuyến ức) trung thất	
C85.7	U lympho không Hodgkin loại xác định khác	
C85.9	U lympho không Hodgkin, loại không xác định	U lympho không xác định khác|U lympho ác tính không xác định khác|U lympho không Hodgkin không xác định khác
C86	Các loại u lympho tế bào T/NK xác định khác	
C86.0	U lympho tế bào NK/T ngoài hạch, thể mũi	
C86.1	U lympho tế bào T gan - lách	
C86.2	U lympho tế bào T thể bệnh lý ruột (EATL)	U lympho tế bào T liên quan bệnh lý ruột
C86.3	U lympho tế bào T thể viêm mô mỡ dưới da	
C86.4	U lympho nguyên bào NK	
C86.5	U lympho tế bào T nguyên bào miễn dịch mạch máu [AILD]	
C86.6	U lympho tế bào T thể da nguyên phát CD30 dương tính	U lympho dòng sẩn|U lympho tế bào lớn bất thục sản nguyên phát tại da|U lympho tế bào T lớn T CD30 - dương tính nguyên phát tại da
C88	Bệnh tăng sinh miễn dịch ác tính	
C88.0	Bệnh macroglobulin huyết Waldenström	
C88.2	Bệnh lý chuỗi nặng gamma	
C88.3	Bệnh tăng sinh miễn dịch ruột non	Bệnh chuỗi alpha nặng|U lympho Địa Trung Hải
C88.4	U lympho tế bào B vùng rìa ngoài hạch của mô lympho liên quan niêm mạc [MALT-lyphoma]	
C88.7	Bệnh lý tăng sinh miễn dịch ác tính khác	
C88.9	Bệnh lý tăng sinh miễn dịch ác tính, không xác định	Bệnh tăng sinh miễn dịch không xác định khác
C90	Đa u tủy xương và/hoặc u tương bào ác tính	
C90.0	Đa u tủy xương	
C90.1	Bệnh bạch cầu dòng plasma [tương bào]	Bệnh bạch cầu dòng tương bào
C90.2	U tương bào ngoài tủy	
C90.3	U tương bào đơn độc	Khối u tương bào ác tính khu trú không xác định khác|U tương bào không xác định khác|U tủy đơn độc
C91	Bệnh bạch cầu dòng lympho	
C91.0	Bệnh bạch cầu cấp dòng lympho [ALL]	
C91.1	Bệnh bạch cầu mạn tính dòng lympho của tế bào B	
C91.3	Bệnh bạch cầu tế bào tiền lympho B	
C91.4	Bệnh bạch cầu tế bào tóc	Bệnh bạch cầu đơn nhân [bệnh lưới nội mô]
C91.5	Bệnh bạch cầu tế bào T trưởng thành	Biến thể cấp tính|Biến thể mạn tính|Biến thể dạng lympho|Biến thể tiềm tàng của tế bào T ở người trưởng thành
C91.6	Bệnh bạch cầu tế bào tiền lympho T	
C91.7	Bệnh bạch cầu dòng lympho khác	
C91.8	Bệnh bạch cầu tế bào B trưởng thành loại Burkitt	
C91.9	Bệnh bạch cầu dòng lympho, không xác định	
C92	Bệnh bạch cầu dòng tủy	
C92.0	Bệnh bạch cầu cấp dòng tủy	
C92.1	Bệnh bạch cầu dòng tủy mạn tính [CML], BCR/ABL - dương tính	
C92.2	Bệnh bạch cầu dòng tủy bán cấp tính không điển hình, BCR/ABL - âm tính	
C92.3	U ác tính [sarcoma] dòng bạch cầu tủy	
C92.4	Bệnh bạch cầu cấp thể tiền tủy bào [PML]	
C92.5	Bệnh bạch cầu cấp dòng tủy đơn nhân	
C92.6	Bệnh bạch cầu cấp dòng tủy có bất thường 11q23	Bệnh bạch cầu cấp dòng tủy với biến thể gen MLL
C92.7	Bệnh bạch cầu dòng tủy khác	
C92.8	Bệnh bạch cầu cấp dòng tủy có loạn sản đa dòng	
C92.9	Bệnh bạch cầu dòng tủy, không xác định	
C93	Bệnh bạch cầu dòng đơn nhân [mono]	
C93.0	Bệnh bạch cầu cấp tính dòng đơn nhân [mono]	AML M5|AML M5a|AML M5b
C93.1	Bệnh bạch cầu mạn tính dòng đơn nhân [mono]	Bệnh bạch cầu dòng đơn nhân mạn tính|CMML-1|CMML-2|CMML tăng bạch cầu ái toan
C93.3	Bệnh bạch cầu dòng tủy đơn nhân [mono] tuổi thiếu niên	
C93.7	Bệnh bạch cầu dòng đơn nhân [mono] khác	
C93.9	Bệnh bạch cầu dòng đơn nhân [mono], không xác định	
C94	Bệnh bạch cầu khác ở dòng tế bào xác định	
C94.0	Bệnh bạch cầu cấp dạng tăng hồng cầu	
C94.2	Bệnh bạch cầu cấp dòng mẫu tiểu cầu	Bệnh bạch cầu cấp dòng tủy M7
C94.3	Bệnh bạch cầu tế bào mast [dưỡng bào]	
C94.4	Tăng sinh dòng tế bào tủy cấp tính kèm xơ tủy	Bệnh xơ hóa tủy xương cấp tính
C94.6	Bệnh loạn sản tủy và/hoặc tăng sản tủy, không phân loại mục khác	
C94.7	Bệnh bạch cầu xác định khác	Bệnh bạch cầu tế bào NK tiến triển nhanh|Bạch cầu cấp dòng ái kiềm
C95	Bệnh bạch cầu ở loại tế bào không xác định	
C95.0	Bệnh bạch cầu cấp không xác định loại tế bào	
C95.1	Bệnh bạch cầu mạn tính không xác định loại tế bào	
C95.7	Bệnh bạch cầu khác không xác định loại tế bào	
C95.9	Bệnh bạch cầu, không xác định	
C96	U ác tính khác và/hoặc không xác định ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	
C96.0	Bệnh mô bào Langerhans (lan tỏa) đa hệ thống và/hoặc đa ổ [bệnh Letterer-Siwe]	Bệnh mô bào X, đa hệ thống
C96.2	U ác tính tế bào mast [dưỡng bào]	
C96.4	U ác tính [sarcoma] tế bào đuôi gai (tế bào phụ)	Ung thư [sarcoma] tế bào đuôi gai liên kết|Ung thư mô liên kết tế bào Langerhans|Ung thư mô liên kết tế bào đuôi gai dạng nang
C96.5	Bệnh mô bào Langerhans đa ổ và/hoặc đơn hệ thống	Bệnh Hand-Schüller-Christian|Bệnh mô bào X, đa ổ
C96.6	Bệnh mô bào Langerhans đơn ổ	U hạt tăng bạch cầu ái toan|Bệnh mô bào X, đơn ổ|Bệnh mô bào X không xác định khác|Bệnh mô bào Langerhans không xác định khác
C96.7	U ác tính xác định khác ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	
C96.8	U ác tính [sarcoma] mô bào	Bệnh mô bào ác tính
C96.9	U ác tính ở mô hê lympho, cơ quan tạo máu và/hoặc mô liên quan, không xác định	
C97	U ác tính ở nhiều vị trí độc lập (nguyên phát)	
D00	Ung thư biểu mô tại chỗ ở khoang miệng, thực quản và/hoặc dạ dày	
D00.0	Ung thư biểu mô tại chỗ ở môi khoang miệng và/hoặc hầu	
D00.1	Ung thư biểu mô tại chỗ ở thực quản	
D00.2	Ung thư biểu mô tại chỗ ở dạ dày	
D01	Ung thư biểu mô tại chỗ ở cơ quan tiêu hóa khác và/hoặc không xác định	
D01.0	Ung thư biểu mô tại chỗ của đại tràng	
D01.1	Ung thư biểu mô tại chỗ nơi nối trực tràng sigma	
D01.2	Ung thư biểu mô tại chỗ của trực tràng	
D01.3	Ung thư biểu mô tại chỗ của hậu môn và/hoặc ống hậu môn	
D01.4	Ung thư biểu mô tại chỗ ở phần khác và/hoặc không xác định của ruột non	
D01.5	Ung thư biểu mô tại chỗ của gan, túi mật và/hoặc ống mật	Bóng Vater
D01.7	Ung thư biểu mô tại chỗ của cơ quan tiêu hóa xác định khác	
D01.9	Ung thư biểu mô tại chỗ của cơ quan tiêu hóa, không xác định	
D02	Ung thư biểu mô tại chỗ ở tai giữa và/hoặc hệ hô hấp	
D02.0	Ung thư biểu mô tại chỗ ở thanh quản	
D02.1	Ung thư biểu mô tại chỗ ở khí quản	
D02.2	Ung thư biểu mô tại chỗ ở phế quản và/hoặc phổi	
D02.3	Ung thư biểu mô tại chỗ ở phần khác ở hệ hô hấp	
D02.4	Ung thư biểu mô tại chỗ ở hệ hô hấp, không xác định	
D03	Ung thư tế bào hắc tố tại chỗ	
D03.0	Ung thư tế bào hắc tố tại chỗ ở môi	
D03.1	Ung thư tế bào hắc tố tại chỗ ở mi mắt, bao gồm góc mắt	
D03.2	Ung thư tế bào hắc tố tại chỗ ở tai và/hoặc ống ngoài tai	
D03.3	Ung thư tế bào hắc tố tại chỗ ở phần khác và/hoặc không xác định ở mặt	
D03.4	Ung thư tế bào hắc tố tại chỗ ở da đầu và/hoặc cổ	
D03.5	Ung thư tế bào hắc tố tại chỗ ở thân	
D03.6	Ung thư tế bào hắc tố tại chỗ ở chi trên, bao gồm vai	
D03.7	Ung thư tế bào hắc tố tại chỗ ở chi dưới, bao gồm hông	
D03.8	Ung thư tế bào hắc tố tại chỗ ở vị trí khác	
D03.9	Ung thư tế bào hắc tố tại chỗ, không xác định	
D04	Ung thư biểu mô tại chỗ ở da	
D04.0	Ung thư biểu mô tại chỗ ở da môi	
D04.1	Ung thư biểu mô tại chỗ ở da mi mắt bao gồm khóe mắt	
D04.2	Ung thư biểu mô tại chỗ ở da tai và/hoặc da ống tai ngoài	
D04.3	Ung thư biểu mô tại chỗ ở da phần khác và/hoặc không xác định của mặt	
D04.4	Ung thư biểu mô tại chỗ ở da đầu và/hoặc da cổ	
D04.5	Ung thư biểu mô tại chỗ ở da thân	
D04.6	Ung thư biểu mô tại chỗ ở da chi trên, bao gồm vai	
D04.7	Ung thư biểu mô tại chỗ ở da chi dưới, bao gồm hông	
D04.8	Ung thư biểu mô tại chỗ ở da vị trí khác	
D04.9	Ung thư biểu mô tại chỗ ở da, không xác định	
D05	Ung thư biểu mô tại chỗ ở vú	
D05.0	Ung thư biểu mô tại chỗ ở tiểu thùy	
D05.1	Ung thư biểu mô tại chỗ ở ống tuyến vú	
D05.7	Ung thư biểu mô khác tại chỗ ở vú	
D05.9	Ung thư biểu mô tại chỗ ở vú, không xác định	
D06	Ung thư biểu mô tại chỗ ở cổ tử cung	
D06.0	Ung thư biểu mô tại chỗ ở nội mạc cổ tử cung	
D06.1	Ung thư biểu mô tại chỗ ở ngoại mạc cổ tử cung	
D06.7	Ung thư biểu mô tại chỗ ở phần khác của cổ tử cung	
D06.9	Ung thư biểu mô tại chỗ ở cổ tử cung, không xác định	
D07	Ung thư biểu mô tại chỗ ở cơ quan sinh dục khác và/hoặc không xác định	Ung thư biểu mô tại chỗ khác và/hoặc không xác định ở cơ quan sinh dục
D07.0	Ung thư biểu mô tại chỗ ở nội mạc tử cung	
D07.1	Ung thư biểu mô tại chỗ ở âm hộ	
D07.2	Ung thư biểu mô tại chỗ ở âm đạo	
D07.3	Ung thư biểu mô tại chỗ ở cơ quan sinh dục nữ khác và/hoặc không xác định	
D07.4	Ung thư biểu mô tại chỗ ở dương vật	Chứng tăng sinh hồng cầu Queyrat không xác định khác
D07.5	Ung thư biểu mô tại chỗ ở tuyến tiền liệt	
D07.6	Ung thư biểu mô tại chỗ ở cơ quan sinh dục nam khác và/hoặc không xác định	
D09	Ung thư biểu mô tại chỗ ở vị trí khác và/hoặc không xác định	
D09.0	Ung thư biểu mô tại chỗ ở bàng quang	
D09.1	Ung thư biểu mô tại chỗ ở cơ quan tiết niệu khác và/hoặc không xác định	
D09.2	Ung thư biểu mô tại chỗ ở mắt	
D09.3	Ung thư biểu mô tại chỗ ở tuyến giáp và/hoặc tuyến nội tiết khác	
D09.7	Ung thư biểu mô tại chỗ ở vị trí xác định khác	
D09.9	Ung thư biểu mô tại chỗ, không xác định	
D10	U lành ở miệng và/hoặc họng	
D10.0	U lành ở môi	
D10.1	U lành ở lưỡi	Amydan lưỡi
D10.2	U lành ở sàn miệng	
D10.3	U lành ở phần khác và/hoặc không xác định của miệng	
D10.4	U lành ở amydan	
D10.5	U lành ở phần khác của hầu - khẩu	
D10.6	U lành ở mũi - hầu	Amydan họng|Bờ sau của vách ngăn mũi và khoang mũi sau
D10.7	U lành ở hạ họng	
D10.9	U lành ở hầu, không xác định	
D11	U lành ở tuyến nước bọt chính	
D11.0	U lành ở tuyến mang tai	
D11.7	U lành ở tuyến nước bọt chính khác	
D11.9	U lành ở tuyến nước bọt chính, không xác định	
D12	U lành ở đại tràng, trực tràng, hậu môn và/hoặc ống hậu môn	
D12.0	U lành ở manh tràng	Van hồi - manh tràng
D12.1	U lành ở ruột thừa	
D12.2	U lành ở đại tràng lên	
D12.3	U lành ở đại tràng ngang [ruột kết ngang]	Góc gan|Góc lách
D12.4	U lành ở đại tràng xuống [ruột kết xuống]	
D12.5	U lành ở đại tràng sigma	
D12.6	U lành ở đại tràng, không xác định	
D12.7	U lành ở nơi nối trực tràng sigma	
D12.8	U lành ở trực tràng	
D12.9	U lành ở hậu môn và/hoặc ống hậu môn	
D13	U lành ở phần khác và/hoặc không rõ ràng ở hệ tiêu hóa	
D13.0	U lành ở thực quản	
D13.1	U lành ở dạ dày	
D13.2	U lành ở tá tràng	
D13.3	U lành ở phần khác và/hoặc không xác định của ruột non	
D13.4	U lành ở gan	Đường mật trong gan
D13.5	U lành ở đường mật ngoài gan	
D13.6	U lành ở tụy	
D13.7	U lành ở tụy nội tiết	U tế bào tiểu đảo|Tiểu đảo Langerhans
D13.9	U lành ở vị trí không rõ ràng trong hệ tiêu hóa	Hệ tiêu hóa không xác định khác|Ruột non không xác định khác|Lách
D14	U lành ở tai giữa và/hoặc hệ hô hấp	
D14.0	U lành ở tai giữa, hốc mũi và/hoặc xoang phụ	
D14.1	U lành ở thanh quản	
D14.2	U lành ở khí quản	
D14.3	U lành ở phế quản và/hoặc phổi	
D14.4	U lành ở hệ hô hấp, không xác định	
D15	U lành ở cơ quan khác và/hoặc không xác định trong khoang ngực	
D15.0	U lành ở tuyến ức	
D15.1	U lành ở tim	
D15.2	U lành ở trung thất	
D15.7	U lành ở cơ quan trong khoang ngực xác định khác	
D15.9	U lành ở cơ quan trong khoang ngực, không xác định	
D16	U lành ở xương và/hoặc sụn khớp	
D16.0	U lành ở xương bả vai và/hoặc xương dài của chi trên	
D16.1	U lành ở xương ngắn của chi trên	
D16.2	U lành ở xương dài của chi dưới	
D16.3	U lành ở xương ngắn của chi dưới	
D16.4	U lành tính ở xương sọ và/hoặc xương mặt	
D16.5	U lành ở xương hàm dưới	
D16.6	U lành ở cột sống	
D16.7	U lành ở xương sườn, xương ức và/hoặc xương đòn	
D16.8	U lành ở xương chậu, xương thiêng và/hoặc xương cụt	
D16.9	U lành ở xương và sụn khớp, không xác định	
D17	U mỡ lành tính	
D17.0	U mỡ lành tính ở da và/hoặc mô dưới da ở đầu, mặt và/hoặc cổ	
D17.1	U mỡ lành tính ở da và/hoặc mô dưới da ở thân	
D17.2	U mỡ lành tính ở da và/hoặc mô dưới da ở các chi	
D17.3	U mỡ lành tính ở da và/hoặc mô dưới da ở vị trí khác và/hoặc không xác định	
D17.4	U mỡ lành tính ở cơ quan trong khoang ngực	
D17.5	U mỡ lành tính ở cơ quan trong ổ bụng	
D17.6	U mỡ lành tính ở thừng tinh	
D17.7	U mỡ lành tính ở vị trí khác	Phúc mạc|Vùng sau phúc mạc
D17.9	U mỡ lành tính, không xác định	U mỡ không xác định khác
D18	U máu và/hoặc u bạch huyết, vị trí bất kỳ	
D18.0	U máu, vị trí bất kỳ	U mạch máu không xác định khác
D18.1	U bạch huyết, vị trí bất kỳ	
D19	U lành trung biểu mô	
D19.0	U lành trung biểu mô ở màng phổi	
D19.1	U lành trung biểu mô ở phúc mạc	
D19.7	U lành trung biểu mô ở vị trí khác	
D19.9	U lành trung biểu mô, không xác định	U lành trung biểu mô không xác định khác
D20	U lành mô mềm ở vùng sau phúc mạc và/hoặc phúc mạc	
D20.0	U lành mô mềm ở vùng sau phúc mạc	
D20.1	U lành mô mềm phúc mạc	
D21	U lành khác ở mô liên kết và/hoặc mô mềm khác	
D21.0	U lành mô liên kết và/hoặc mô mềm khác ở đầu, mặt và/hoặc cổ	
D21.1	U lành mô liên kết và/hoặc mô mềm khác ở chi trên bao gồm vai	
D21.2	U lành mô liên kết và/hoặc mô mềm khác ở chi dưới, bao gồm hông	
D21.3	U lành mô liên kết và/hoặc mô mềm khác ở lồng ngực	
D21.4	U lành mô liên kết và/hoặc mô mềm khác ở bụng	
D21.5	U lành mô liên kết và/hoặc mô mềm khác ở chậu	
D21.6	U lành mô liên kết và/hoặc mô mềm khác ở thân mình, không xác định	Lưng không xác định khác
D21.9	U lành mô liên kết và/hoặc mô mềm khác, không xác định	
D22	Nơ vi [nevi] tế bào melanin [nốt ruồi]	
D22.0	Nơ vi tế bào melanin ở môi	
D22.1	Nơ vi tế bào melanin ở khóe mắt bao gồm mi mắt	
D22.2	Nơ vi tế bào melanin ở tai và/hoặc ống tai ngoài	
D22.3	Nơ vi tế bào melanin ở phần khác và/hoặc phần không xác định của mặt	
D22.4	Nơ vi tế bào melanin ở da đầu và/hoặc cổ	
D22.5	Nơ vi tế bào melanin ở thân	
D22.6	Nơ vi tế bào melanin ở chi trên bao gồm vai	
D22.7	Nơ vi tế bào melanin ở chi dưới bao gồm hông	
D22.9	Nơ vi tế bào melanin, không xác định	
D23	U lành khác ở da	
D23.0	U lành da ở môi	
D23.1	U lành da ở mi mắt kể cả khóe mắt	
D23.2	U lành da ở tai và/hoặc ống tai ngoài	
D23.3	U lành da ở phần khác và/hoặc không xác định của mặt	
D23.4	U lành da ở đầu và/hoặc cổ	
D23.5	U lành da ở thân mình	
D23.6	U lành da ở chi trên, bao gồm vai	
D23.7	U lành da ở chi dưới, bao gồm hông	
D23.9	U lành da, không xác định	
D24	U lành ở vú	
D25	U cơ trơn tử cung	
D25.0	U cơ trơn dưới niêm mạc tử cung	
D25.1	U cơ trơn trong vách tử cung	
D25.2	U cơ trơn dưới thanh mạc tử cung	
D25.9	U cơ trơn tử cung, không xác định	
D26	U lành khác ở tử cung	
D26.0	U lành ở cổ tử cung	
D26.1	U lành ở thân tử cung	
D26.7	U lành ở phần khác của tử cung	
D26.9	U lành ở tử cung, không xác định	
D27	U lành ở buồng trứng	
D28	U lành ở cơ quan sinh dục nữ khác và/hoặc không xác định	
D28.0	U lành ở âm hộ	
D28.1	U lành ở âm đạo	
D28.2	U lành ở vòi và/hoặc dây chằng tử cung	
D28.7	U lành ở cơ quan sinh dục nữ xác định khác	
D28.9	U lành ở cơ quan sinh dục nữ, không xác định	
D29	U lành ở cơ quan sinh dục nam	
D29.0	U lành ở dương vật	
D29.1	U lành ở tuyến tiền liệt	
D29.2	U lành ở tinh hoàn	
D29.3	U lành ở mào tinh hoàn	
D29.4	U lành ở bìu	Da bìu
D29.7	U lành ở cơ quan sinh dục khác ở nam giới	Túi tinh|Thừng tinh|Lớp tinh mạc
D29.9	U lành ở cơ quan sinh dục nam, không xác định	
D30	U lành ở cơ quan tiết niệu	
D30.0	U lành ở thận	
D30.1	U lành ở bể thận	
D30.2	U lành ở niệu quản	
D30.3	U lành ở bàng quang	
D30.4	U lành ở niệu đạo	
D30.7	U lành ở cơ quan tiết niệu khác	Tuyến cận niệu đạo
D30.9	U lành ở cơ quan tiết niệu, không xác định	Hệ tiết niệu không xác định khác
D31	U lành ở mắt và/hoặc cấu trúc phụ cận của mắt	
D31.0	U lành ở kết mạc	
D31.1	U lành ở giác mạc	
D31.2	U lành ở võng mạc	
D31.3	U lành ở màng mạch mắt	
D31.4	U lành ở thể mi	
D31.5	U lành ở tuyến và/hoặc ống lệ	
D31.6	U lành ở hốc mắt, không xác định	
D31.9	U lành ở mắt, không xác định	Nhãn cầu
D32	Khối u lành tính ở màng não	U lành ở màng não
D32.0	U lành ở màng não	
D32.1	U lành ở màng tủy	
D32.9	U lành ở màng não, không xác định	U màng não không xác định khác
D33	U lành ở não và/hoặc phần khác của hệ thần kinh trung ương	
D33.0	U lành ở não trên lều	
D33.1	U lành ở não, lều dưới	Cuống não|Tiểu não|Não thất IV
D33.2	U lành ở não, không xác định	
D33.3	U lành ở thần kinh sọ não	Hành khứu giác
D33.4	U lành ở tủy sống	
D33.7	U lành ở phần xác định khác của hệ thần kinh trung ương	
D33.9	U lành ở hệ thần kinh trung ương, không xác định	
D34	U lành ở tuyến giáp	
D35	U lành ở tuyến nội tiết khác và/hoặc không xác định	
D35.0	U lành ở tuyến thượng thận	
D35.1	U lành ở tuyến cận giáp	
D35.2	U lành ở tuyến yên	
D35.3	U lành ở ống sọ hầu	
D35.4	U lành ở tuyến tùng	
D35.5	U lành ở tiểu thể của động mạch cảnh	
D35.6	U lành ở tiểu thể của động mạch chủ và/hoặc cận hạch thần kinh khác	
D35.7	U lành ở tuyến nội tiết xác định khác	
D35.8	U lành có tác động đến nhiều tuyến nội tiết	
D35.9	U lành ở tuyến nội tiết, không xác định	
D36	U lành ở vị trí khác và/hoặc không xác định	
D36.0	U lành hạch lympho	
D36.1	U lành thần kinh ngoại biên và/hoặc hệ thần kinh tự động	
D36.7	U lành ở vị trí xác định khác	Mũi không xác định khác
D36.9	U lành ở vị trí không xác định	
D37	U tân sinh không tiên lượng được tiến triển và tính chất ở khoang miệng và/hoặc cơ quan tiêu hóa	
D37.0	U tân sinh không tiên lượng được tiến triển và tính chất ở môi, khoang miệng và/hoặc họng	
D37.1	U tân sinh không tiên lượng được tiến triển và tính chất ở dạ dày	
D37.2	U tân sinh không tiên lượng được tiến triển và tính chất ở ruột non	
D37.3	U tân sinh không tiên lượng được tiến triển và tính chất ở ruột thừa	
D37.4	U tân sinh không tiên lượng được tiến triển và tính chất ở đại tràng	
D37.5	U tân sinh không tiên lượng được tiến triển và tính chất ở trực tràng	Nơi nối trực tràng sigma
D37.6	U tân sinh không tiên lượng được tiến triển và tính chất ở gan, túi mật và/hoặc ống mật	Bóng Vater
D37.7	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan tiêu hóa khác	
D37.9	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan tiêu hóa, không xác định	
D38	U tân sinh không tiên lượng được tiến triển và tính chất ở tai giữa, cơ quan hô hấp và/hoặc cơ quan trong khoang ngực	
D38.0	U tân sinh không tiên lượng được tiến triển và tính chất ở thanh quản	
D38.1	U tân sinh không tiên lượng được tiến triển và tính chất ở khí quản, phế quản và/hoặc phổi	
D38.2	U tân sinh không tiên lượng được tiến triển và tính chất ở màng phổi	
D38.3	U tân sinh không tiên lượng được tiến triển và tính chất ở trung thất	
D38.4	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến ức	
D38.5	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan hô hấp khác	
D38.6	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan hô hấp, không xác định	
D39	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nữ	
D39.0	U tân sinh không tiên lượng được tiến triển và tính chất ở tử cung	
D39.1	U tân sinh không tiên lượng được tiến triển và tính chất ở buồng trứng	
D39.2	U tân sinh không tiên lượng được tiến triển và tính chất ở nhau (rau) thai	
D39.7	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nữ khác	Da của cơ quan sinh dục nữ
D39.9	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nữ, không xác định	
D40	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nam	
D40.0	U tân sinh không tiên lượng được tiến triển và tính chất ở tiền liệt tuyến	
D40.1	U tân sinh không tiên lượng được tiến triển và tính chất của tinh hoàn	
D40.7	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nam khác	Da của cơ quan sinh dục nam
D40.9	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan sinh dục nam, không xác định	
D41	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan tiết niệu	
D41.0	U tân sinh không tiên lượng được tiến triển và tính chất ở thận	
D41.1	U tân sinh không tiên lượng được tiến triển và tính chất ở bể thận	
D41.2	U tân sinh không tiên lượng được tiến triển và tính chất ở niệu quản	
D41.3	U tân sinh không tiên lượng được tiến triển và tính chất ở niệu đạo	
D41.4	U tân sinh không tiên lượng được tiến triển và tính chất ở bàng quang	
D41.7	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan tiết niệu khác	
D41.9	U tân sinh không tiên lượng được tiến triển và tính chất ở cơ quan tiết niệu, không xác định	
D42	U tân sinh không tiên lượng được tiến triển và tính chất ở màng não tủy	
D42.0	U tân sinh không tiên lượng được tiến triển và tính chất của màng não	
D42.1	U tân sinh không tiên lượng được tiến triển và tính chất ở màng tủy	
D42.9	U tân sinh không tiên lượng được tiến triển và tính chất của màng não tủy, không xác định	
D43	U tân sinh không tiên lượng được tiến triển và tính chất ở não và/hoặc hệ thần kinh trung ương	
D43.0	U tân sinh không tiên lượng được tiến triển và tính chất ở não, trên lều não	
D43.1	U tân sinh không tiên lượng được tiến triển và tính chất ở não, dưới lều não	Cuống não|Tiểu não|Não thất IV
D43.2	U tân sinh không tiên lượng được tiến triển và tính chất ở não, không xác định	
D43.3	U tân sinh không tiên lượng được tiến triển và tính chất ở thần kinh sọ	
D43.4	U tân sinh không tiên lượng được tiến triển và tính chất ở tủy sống	
D43.7	U tân sinh không tiên lượng được tiến triển và tính chất ở phần khác của hệ thần kinh trung ương	
D43.9	U tân sinh không tiên lượng được tiến triển và tính chất ở hệ thần kinh trung ương, không xác định	
D44	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến nội tiết	
D44.0	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến giáp	
D44.1	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến thượng thận	
D44.2	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến cận giáp	
D44.3	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến yên	
D44.4	U tân sinh không tiên lượng được tiến triển và tính chất ở ống sọ - hầu	
D44.5	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến tùng	
D44.6	U tân sinh tiểu thể động mạch cảnh không tiên lượng được tiến triển và tính chất	
D44.7	U tân sinh tiểu thể động mạch chủ và/hoặc cận hạch thần kinh khác không tiên lượng được tiến triển và tính chất	
D44.8	U tân sinh không tiên lượng được tiến triển và tính chất có tác động đến nhiều tuyến nội tiết	U của nhiều tuyến nội tiết khác
D44.9	U tân sinh không tiên lượng được tiến triển và tính chất ở tuyến nội tiết, không xác định	
D45	Đa hồng cầu	
D46	Hội chứng loạn sản tủy xương	
D46.0	Thiếu máu dai dẳng không có nguyên hồng cầu sắt vòng, đã được khẳng định	
D46.1	Thiếu máu dai dẳng có nguyên hồng cầu sắt vòng	
D46.2	Thiếu máu dai dẳng có tăng tế bào blast [RAEB]	
D46.4	Thiếu máu dai dẳng, không xác định	
D46.5	Thiếu máu dai dẳng có loạn sản đa dòng tế bào [MDS-MLD]	
D46.6	Hội chứng rối loạn sản tủy có bất thường nhiễm sắc thể del(5q) đơn độc	Hội chứng 5 q-minus
D46.7	Hội chứng rối loạn sản tủy khác	
D46.9	Hội chứng rối loạn sản tủy, không xác định	
D47	U tân sinh khác không tiên lượng được tiến triển và tính chất ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	
D47.0	Bệnh mô bào và/hoặc bệnh u tế bào mast [dưỡng bào] không tiên lượng được tiến triển và tính chất	
D47.1	Bệnh bạch cầu dòng trung tính mạn tính	
D47.2	Bệnh lý gamma thể đơn dòng không xác định (MGUS)	
D47.3	Bệnh tăng tiểu cầu (xuất huyết) nguyên phát	Bệnh tăng tiểu cầu xuất huyết vô căn
D47.4	Bệnh xơ hóa tủy xương	
D47.5	Bệnh bạch cầu dòng tế bào ưa acid mạn tính [hội chứng tăng bạch cầu ưa acid]	
D47.7	U tân sinh xác định khác không tiên lượng được tiến triển và tính chất ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	Các u mô bào với biểu hiện không rõ ràng và/hoặc không chắc chắn
D47.9	U tân sinh không tiên lượng được tiến triển và tính chất ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan, không xác định	Bệnh tăng sinh mô lympho, không xác định khác
D48	U tân sinh không tiên lượng được tiến triển và tính chất ở vị trí khác và/hoặc không xác định	
D48.0	U tân sinh không tiên lượng được tiến triển và tính chất ở xương và/hoặc sụn khớp	
D48.1	U tân sinh không tiên lượng được tiến triển và tính chất ở mô liên kết và/hoặc mô mềm khác	
D48.2	U tân sinh không tiên lượng được tiến triển và tính chất ở thần kinh ngoại biên và/hoặc hệ thần kinh tự động	
D48.3	U tân sinh không tiên lượng được tiến triển và tính chất ở vùng sau phúc mạc	
D48.4	U tân sinh không tiên lượng được tiến triển và tính chất ở phúc mạc	
D48.5	U tân sinh không tiên lượng được tiến triển và tính chất ở da	
D48.6	U tân sinh không tiên lượng được tiến triển và tính chất ở vú	
D48.7	U tân sinh không tiên lượng được tiến triển và tính chất ở vị trí xác định khác	
D48.9	U tân sinh không tiên lượng được tiến triển và tính chất , vị trí không xác định	"""Phát triển"" không xác định khác|U tân sinh không xác định khác|U tân sinh mới không xác định khác|U không xác định khác"
D50	Thiếu máu do thiếu sắt	
D50.0	Thiếu máu thiếu sắt thứ phát do mất máu (mạn tính)	
D50.1	Chứng khó nuốt do thiếu sắt	Hội chứng Kelly-Paterson|Hội chứng Plummer-Vinson
D50.8	Thiếu máu thiếu sắt khác	
D50.9	Thiếu máu thiếu sắt, không xác định	
D51	Thiếu máu do thiếu vitamin B12	
D51.0	Thiếu máu thiếu vitamin B12 do thiếu yếu tố nội sinh	
D51.1	Thiếu vitamin B12 do giảm hấp thu chọn lọc vitamin B12 kèm protein niệu	
D51.2	Thiếu Transcobalamin II	
D51.3	Thiếu máu thiếu vitamin B12 trong chế độ ăn khác	Thiếu máu ở người ăn chay trường
D51.8	Thiếu máu do thiếu vitamin B12 khác	
D51.9	Thiếu máu thiếu vitamin B12, không xác định	
D52	Thiếu máu do thiếu folate	
D52.0	Thiếu máu do thiếu folate trong chế độ ăn	Thiếu máu nguyên hồng cầu khổng lồ do dinh dưỡng
D52.1	Thiếu máu thiếu folate do thuốc	
D52.8	Thiếu máu thiếu folate khác	
D52.9	Thiếu máu thiếu folate, không xác định	Thiếu máu thiếu acid folic không xác định khác
D53	Thiếu máu do thiếu dinh dưỡng khác	
D53.0	Thiếu máu do thiếu protein	
D53.1	Thiếu máu nguyên hồng cầu khổng lồ khác, không phân loại mục khác	
D53.2	Thiếu máu thiếu vitamin C	
D53.8	Thiếu máu thiếu dinh dưỡng xác định khác	
D53.9	Thiếu máu do thiếu dinh dưỡng, không xác định	
D55	Thiếu máu do rối loạn men	
D55.0	Thiếu máu do thiếu men glucose-6-phosphate dehydrogenase [G6PD]	Dị ứng đậu tằm [do ăn đậu Fava gây tán huyết ở người thiếu men G6PD]|Thiếu máu do thiếu men G6PD
D55.1	Thiếu máu do rối loạn chuyển hóa glutathione khác	
D55.2	Thiếu máu do rối loạn men phân giải glucose	
D55.3	Thiếu máu do rối loạn chuyển hóa nucleotide	
D55.8	Thiếu máu khác do rối loạn men	
D55.9	Thiếu máu do rối loạn men, không xác định	
D56	Bệnh thalassaemia [tan máu bẩm sinh]	
D56.0	Bệnh thalassaemia [tan máu bẩm sinh] thể alpha	
D56.1	Bệnh thalassaemia [tan máu bẩm sinh] thể beta	Thiếu máu Cooley|Bệnh thalassaemia beta nặng
D56.2	Bệnh thalassaemia [tan máu bẩm sinh] thể delta-beta	
D56.3	Bệnh thalassaemia [tan máu bẩm sinh] thể trung gian	
D56.4	Tồn tại di truyền huyết sắc tố bào thai [HPFH]	
D56.8	Bệnh thalassaemias [tan máu bẩm sinh] thể khác	
D56.9	Bệnh thalassaemia [tan máu bẩm sinh], không xác định	
D57	Rối loạn hồng cầu hình liềm	
D57.0	Thiếu máu hồng cầu hình liềm kèm cơn tắc mạch và tan máu	Thiếu máu hồng cầu hình liềm [HB-SS] có cơn tắc mạch và tan máu
D57.1	Thiếu máu hồng cầu hình liềm không kèm cơn tắc mạch và tan máu	
D57.2	Rối loạn dị hợp tử kép của thiếu máu hồng cầu hình liềm	
D57.3	Hồng cầu hình liềm thể nhẹ	Hb-S thể nhẹ|Huyết sắc tố S dị hợp tử [HbAS]
D57.8	Rối loạn hồng cầu hình liềm khác	
D58	Thiếu máu tan máu di truyền khác	
D58.0	Hồng cầu hình cầu di truyền	
D58.1	Hồng cầu hình bầu dục [oval] di truyền	
D58.2	Bệnh lý huyết sắc tố khác	
D58.8	Thiếu máu tan máu di truyền xác định khác	Bệnh hồng cầu hình răng cưa
D58.9	Thiếu máu tan máu di truyền, không xác định	
D59	Thiếu máu tan máu mắc phải	
D59.0	Thiếu máu tan máu tự miễn do thuốc	
D59.1	Thiếu máu tan máu tự miễn khác	
D59.2	Thiếu máu tan máu do thuốc không phải bệnh tự miễn	
D59.3	Hội chứng tan máu tăng urê máu	
D59.4	Thiếu máu tan máu không phải bệnh tự miễn khác	
D59.5	Đái huyết sắc tố kịch phát ban đêm [Marchifava-Micheli]	
D59.6	Đái huyết sắc tố do tan máu từ những nguyên nhân bên ngoài khác	
D59.8	Thiếu máu tan máu mắc phải khác	
D59.9	Thiếu máu tan máu mắc phải, không xác định	Thiếu máu tan máu vô căn, mạn tính
D60	Suy tủy xương [bất sản hồng cầu] đơn thuần mắc phải [giảm nguyên hồng cầu]	
D60.0	Suy tủy xương [bất sản hồng cầu] đơn thuần mắc phải mạn tính	
D60.1	Suy tủy xương [bất sản hồng cầu] đơn thuần mắc phải thoáng qua	
D60.8	Suy tủy xương [bất sản hồng cầu] đơn thuần mắc phải khác	
D60.9	Suy tủy xương [bất sản hồng cầu] đơn thuần mắc phải, không xác định	
D61	Thể suy tủy xương [thiếu máu bất sản] khác	
D61.0	Suy tủy xương [thiếu máu bất sản] về thể chất	
D61.1	Suy tủy xương [thiếu máu bất sản] do thuốc	
D61.2	Suy tủy xương [thiếu máu bất sản] do tác nhân bên ngoài khác	
D61.3	Suy tủy xương [thiếu máu bất sản] vô căn	
D61.8	Suy tủy xương [thiếu máu bất sản] xác định khác	
D61.9	Suy tủy xương [thiếu máu bất sản], không xác định	Thiếu máu giảm sản không xác định khác|Bệnh giảm sản tủy xương|Bệnh suy tủy xương toàn bộ
D62	Thiếu máu sau chảy máu cấp tính	
D63.*	Thiếu máu do bệnh mạn tính phân loại mục khác	
D63.0*	Thiếu máu do bệnh u tân sinh (C00.- - D48.-†)	
D63.8*	Thiếu máu do bệnh mạn tính khác phân loại mục khác	
D64	Bệnh thiếu máu khác	
D64.0	Thiếu máu nguyên hồng cầu sắt di truyền	Thiếu máu nguyên hồng cầu sắt nhược sắc liên quan giới tính
D64.1	Thiếu máu nguyên hồng cầu sắt thứ phát do bệnh lý	
D64.2	Thiếu máu nguyên hồng cầu sắt thứ phát do thuốc và/hoặc độc chất	
D64.3	Thiếu máu nguyên hồng cầu sắt khác	
D64.4	Thiếu máu loạn sinh hồng cầu bẩm sinh	
D64.8	Thiếu máu xác định khác	Bệnh bạch cầu giả [pseudoleukaemia] ở trẻ nhỏ|Thiếu máu nguyên bạch hồng cầu
D64.9	Thiếu máu, không xác định	
D65	Đông máu rải rác nội mạch [hội chứng tiêu sợi huyết]	
D66	Thiếu hụt yếu tố VIII di truyền	
D67	Thiếu hụt yếu tố IX di truyền	
D68	Rối loạn đông máu khác	
D68.0	Bệnh Von Willebrand	
D68.1	Thiếu hụt yếu tố XI di truyền	Rối loạn đông máu [Haemophilia] C|Bệnh giảm tiền chất huyết khối thromboplastin [yếu tố XI]
D68.2	Thiếu hụt yếu tố đông máu khác do di truyền	
D68.3	Xuất huyết do có kháng đông lưu hành	
D68.4	Thiếu hụt yếu tố đông máu mắc phải	
D68.5	Bệnh tăng đông máu nguyên phát	Kháng protein C hoạt hóa [đột biến yếu tố V Leiden]
D68.6	Bệnh tăng đông máu khác	
D68.8	Rối loạn đông máu xác định khác	
D68.9	Rối loạn đông máu, không xác định	
D69	Ban xuất huyết và/hoặc tình trạng xuất huyết khác	
D69.0	Ban xuất huyết dị ứng	
D69.1	Bất thường chất lượng tiểu cầu	
D69.2	Ban xuất huyết không giảm tiểu cầu khác	
D69.3	Ban xuất huyết giảm tiểu cầu vô căn	Hội chứng Evans
D69.4	Giảm tiểu cầu nguyên phát khác	
D69.5	Giảm tiểu cầu thứ phát	
D69.6	Giảm tiểu cầu, không xác định	
D69.8	Tình trạng xuất huyết xác định khác	
D69.9	Tình trạng xuất huyết, không xác định	
D70	Bệnh mất bạch cầu hạt	
D71	Rối loạn chức năng bạch cầu đa nhân trung tính	
D72	Rối loạn khác của bạch cầu	
D72.0	Bất thường di truyền của bạch cầu	
D72.1	Tăng bạch cầu ái toan	
D72.8	Rối loạn xác định khác của bạch cầu	
D72.9	Rối loạn bạch cầu, không xác định	
D73	Bệnh lý lách	
D73.0	Thiểu năng lách	
D73.1	Chứng cường lách	
D73.2	Lách to xung huyết mạn tính	
D73.3	Áp xe lách	
D73.4	Nang lách	
D73.5	Bệnh nhồi máu lá lách	
D73.8	Bệnh khác ở lách	Xơ hóa lách không xác định khác|Viêm quanh lách|Viêm lách không xác định khác
D73.9	Bệnh lý lách, không xác định	
D74	Bệnh mất sắc tố máu [Methaemoglobinaemia]	
D74.0	Bệnh mất sắc tố máu [Methaemoglobinaemia] bẩm sinh	Bệnh thiếu hụt enzim NADH- methaemoglobin reductase bẩm sinh|Bệnh huyết sắc tố M [Hb-M]|Bệnh mất sắc tố máu [Methaemoglobinaemia], di truyền
D74.8	Bệnh mất sắc tố máu [Methaemoglobinaemia] khác	
D74.9	Bệnh mất sắc tố máu [Methaemoglobinaemia], không xác định	
D75	Bệnh khác của máu và/hoặc cơ quan tạo máu	
D75.0	Bệnh tăng hồng cầu di truyền [có yếu tố gia đình]	
D75.1	Bệnh tăng hồng cầu thứ phát	
D75.8	Bệnh xác định khác của máu và/hoặc cơ quan tạo máu	Chứng tăng bạch cầu ưa bazơ
D75.9	Bệnh của máu và cơ quan tạo máu, không xác định	
D76	Bệnh xác định khác liên quan mô hệ bạch huyết và/hoặc mô bào lưới	
D76.1	Hội chứng thực bào máu	Bệnh tăng tế bào lưới thực bào máu di truyền|Rối loạn tăng sinh tế bào đuôi gai của thực bào đơn nhân
D76.2	Hội chứng thực bào máu, liên quan đến nhiễm trùng	
D76.3	Hội chứng mô bào khác	
D77.*	Rối loạn khác ở máu và/hoặc cơ quan tạo máu do bệnh phân loại mục khác	
D80	Suy giảm miễn dịch chủ yếu do bất thường kháng thể	
D80.0	Giảm gammaglobulin máu di truyền	
D80.1	Suy giảm gammaglobulin máu không di truyền [không có yếu tố gia đình]	Bệnh không có gammaglobulin máu kèm lymphocyte B mang Ig|Bệnh không có gammaglobulin máu biến thiên phổ biến [CVA gamma]|Chứng giảm gammaglobulin máu không xác định khác
D80.2	Suy giảm immunoglobulin A [IgA] chọn lọc	
D80.3	Suy giảm phân nhóm immunoglobulin G [IgG] chọn lọc	
D80.4	Suy giảm immunoglobulin M [IgM] chọn lọc	
D80.5	Suy giảm miễn dịch kèm tăng immunoglobulin M [IgM]	
D80.6	Suy giảm kháng thể với hàm lượng immunoglobulin giảm ít hoặc chứng tăng immuglobulin máu	
D80.7	Suy giảm gammaglobulin máu thoáng qua ở trẻ nhỏ	
D80.8	Suy giảm miễn dịch khác chủ yếu do thiếu kháng thể	Suy giảm chuỗi nhẹ Kappa
D80.9	Suy giảm miễn dịch do thiếu kháng thể là chủ yếu, không xác định	
D81	Suy giảm miễn dịch kết hợp	
D81.0	Suy giảm miễn dịch kết hợp trầm trọng [SCID] với chứng loạn sinh lưới	
D81.1	Suy giảm miễn dịch kết hợp trầm trọng [SCID] với số lượng tế bào T và tế bào B thấp	
D81.2	Suy giảm miễn dịch kết hợp trầm trọng [SCID] với số lượng tế bào B thấp hoặc bình thường	
D81.3	Suy giảm enzim adenosine deaminase [ADA]	
D81.4	Hội chứng Nezelof	
D81.5	Suy giảm emzim purine nucleoside phosphorylase [PNP]	
D81.6	Suy giảm phức hợp kháng nguyên phù hợp tổ chức của người lớp I (MHC I)	Hội chứng lympho trần
D81.7	Suy giảm phức hợp kháng nguyên phù hợp tổ chức của người lớp II (MHC II)	
D81.8	Suy giảm miễn dịch kết hợp khác	Bệnh suy giảm men carboxylase phụ thuộc biotin
D81.9	Suy giảm miễn dịch hỗn hợp, không xác định	Rối loạn suy giảm miễn dịch kết hợp nguy kịch [SCID] không xác định khác
D82	Suy giảm miễn dịch liên quan đến bất thường nghiêm trọng khác	
D82.0	Hội chứng Wiskott-Aldrich	Suy giảm miễn dịch kèm chứng giảm tiểu cầu và/hoặc bệnh chàm
D82.1	Hội chứng Di George	Hội chứng túi hầu họng
D82.2	Suy giảm miễn dịch kèm chứng ngắn chi	
D82.3	Bệnh suy giảm miễn dịch do khiếm khuyết di truyền sau phản ứng với virus Epstein-Barr	Bệnh tăng sinh mô bạch huyết liên kết với nhiễm sắc thể X
D82.4	Hội chứng tăng immunoglobulin E [IgE]	
D82.8	Suy giảm miễn dịch liên quan đến bất thường nghiêm trọng xác định khác	
D82.9	Suy giảm miễn dịch khiếm khuyết chủ yếu, không xác định	
D83	Suy giảm miễn dịch biến thiên phổ biến	
D83.0	Suy giảm miễn dịch biến thiên phổ biến với bất thường chủ yếu về số lượng và chức năng của tế bào B	
D83.1	Suy giảm miễn dịch biến thiên phổ biến với rối loạn tế bào T điều hòa miễn dịch chiếm ưu thế	
D83.2	Suy giảm miễn dịch biến thiên phổ biến với tự kháng thể đối với tế bào T hoặc tế bào B	
D83.8	Suy giảm miễn dịch biến thiên phổ biến khác	
D83.9	Suy giảm miễn dịch biến thiên phổ biến, không xác định	
D84	Suy giảm miễn dịch khác	
D84.0	Bất thường kháng nguyên chức năng 1 của lymphocyte [LFA-1]	
D84.1	Bất thường của hệ thống bổ thể	Thiếu hụt yếu tố ức chế C1 esterase [C1-INH]
D84.8	Suy giảm miễn dịch xác định khác	
D84.9	Suy giảm miễn dịch, không xác định	
D86	Bệnh u hạt	
D86.0	Bệnh u hạt phổi	
D86.1	Bệnh u hạt hạch bạch huyết	
D86.2	Bệnh u hạt phổi và/hoặc bệnh hạch bạch huyết	
D86.3	Bệnh u hạt da	
D86.8	Bệnh u hạt vị trí khác và/hoặc vị trí kết hợp	
D86.9	Bệnh u hạt, không xác định	
D89	Rối loạn khác liên quan cơ chế miễn dịch, không phân loại mục khác	
D89.0	Bệnh tăng gammaglobulin máu đa dòng	Bệnh ban xuất huyết tăng globulin gamma máu lành tính|Bệnh gamma đa dòng không xác định khác
D89.1	Bệnh kháng thể (tăng globulin) ngưng kết lạnh	
D89.2	Bệnh tăng gammaglobulin máu, không xác định	
D89.3	Hội chứng tái tạo miễn dịch	
D89.8	Rối loạn xác định khác liên quan cơ chế miễn dịch, không phân loại mục khác	
D89.9	Rối loạn liên quan cơ chế miễn dịch, không xác định	Bệnh miễn dịch không xác định khác
E00	Hội chứng thiếu iod bẩm sinh	
E00.0	Hội chứng thiếu iod bẩm sinh, thể thần kinh	Bệnh đần lưu hành [đặc hữu], thể thần kinh
E00.1	Hội chứng thiếu iod bẩm sinh, thể phù niêm	
E00.2	Hội chứng thiếu iod bẩm sinh, thể phối hợp	Chứng đần địa phương [đặc hữu], thể phối hợp
E00.9	Hội chứng thiếu iod bẩm sinh, không xác định	Suy giáp bẩm sinh do thiếu iod, không xác định khác|Chậm phát triển tâm thần lưu hành không xác định khác
E01	Rối loạn tuyến giáp liên quan đến thiếu iod và/hoặc bệnh phối hợp	
E01.0	Bướu giáp lan tỏa (đơn thuần) - liên quan đến thiếu iod	
E01.1	Bướu giáp đa nhân (đơn thuần) do thiếu iod	Bướu giáp nhân liên quan đến thiếu iod
E01.2	Bướu giáp (đơn thuần) do thiếu iod, không xác định	Bướu giáp lưu hành không xác định khác
E01.8	Rối loạn tuyến giáp liên quan đến thiếu iod khác và/hoặc những bệnh phối hợp	Suy giáp mắc phải do thiếu iod không xác định khác
E02	Suy tuyến giáp do thiếu iod dưới lâm sàng	
E03	Suy tuyến giáp khác	
E03.0	Suy tuyến giáp bẩm sinh kèm bướu lan tỏa	
E03.1	Suy tuyến giáp bẩm sinh không kèm bướu	
E03.2	Suy tuyến giáp do thuốc điều trị và/hoặc chất ngoại sinh khác	
E03.3	Suy tuyến giáp sau nhiễm trùng	
E03.4	Teo tuyến giáp (mắc phải)	
E03.5	Hôn mê phù niêm	
E03.8	Suy tuyến giáp xác định khác	
E03.9	Suy tuyến giáp, không xác định	Phù niêm không xác định khác
E04	Bướu tuyến giáp không độc khác	
E04.0	Bướu giáp đơn thuần không độc	
E04.1	Bướu giáp đơn nhân không độc	
E04.2	Bướu giáp đa nhân không độc	
E04.8	Bướu giáp không độc xác định khác	
E04.9	Bướu giáp không độc, không xác định	
E05	Nhiễm độc tuyến giáp [cường giáp]	
E05.0	Nhiễm độc tuyến giáp [cường giáp] kèm bướu lan tỏa	Bướu giáp nhiễm độc hoặc lồi mắt không xác định khác|Bệnh Graves [bệnh Basedow hoặc bướu giáp lan tỏa nhiễm độc]|Bướu giáp lan tỏa nhiễm độc
E05.1	Nhiễm độc tuyến giáp [cường giáp] kèm bướu tuyến giáp đơn nhân có nhiễm độc	Nhiễm độc giáp kèm bướu giáp độc đơn nhân
E05.2	Nhiễm độc tuyến giáp [cường giáp] kèm bướu tuyến giáp đa nhân có nhiễm độc	Bướu giáp nhân độc không xác định khác
E05.3	Nhiễm độc tuyến giáp [cường giáp] từ mô giáp lạc chỗ	
E05.4	Nhiễm độc tuyến giáp [cường giáp] giả	
E05.5	Cơn nhiễm độc giáp [cường giáp] cấp tính hoặc cơn bão giáp	
E05.8	Nhiễm độc tuyến giáp [cường giáp] khác	
E05.9	Nhiễm độc tuyến giáp [cường giáp], không xác định	Cường giáp không xác định khác
E06	Viêm tuyến giáp	
E06.0	Viêm tuyến giáp cấp tính	
E06.1	Viêm tuyến giáp bán cấp tính	
E06.2	Viêm tuyến giáp mạn tính kèm nhiễm độc giáp thoáng qua	
E06.3	Viêm tuyến giáp tự miễn	
E06.4	Viêm tuyến giáp do thuốc	
E06.5	Viêm tuyến giáp mạn tính khác	
E06.9	Viêm tuyến giáp, không xác định	
E07	Rối loạn khác của tuyến giáp	
E07.0	Tăng tiết calcitonin	Tăng sản tế bào C của tuyến giáp|Tăng tiết calcitonin tuyến giáp
E07.1	Phình giáp loạn sinh hormon	
E07.8	Rối loạn xác định khác của tuyến giáp	Bất thường về globulin gắn tuyến giáp|Xuất huyết tuyến giáp|Nhồi máu tuyến giáp|Hội chứng bệnh lý khác với chức năng giáp bình thường
E07.9	Rối loạn tuyến giáp, không xác định	
E10	Bệnh đái tháo đường típ 1	
E10.0	Bệnh đái tháo đường típ 1, kèm hôn mê	
E10.1	Bệnh đái tháo đường típ 1, kèm nhiễm toan ceton	
E10.2†	Bệnh đái tháo đường típ 1, kèm biến chứng thận	
E10.3†	Bệnh đái tháo đường típ 1, kèm biến chứng mắt	
E10.4†	Bệnh đái tháo đường típ 1, kèm biến chứng thần kinh	
E10.5	Bệnh đái tháo đường típ 1, kèm biến chứng mạch máu ngoại vi	
E10.6	Bệnh đái tháo đường típ 1, kèm biến chứng xác định khác	
E10.7	Bệnh đái tháo đường típ 1, kèm đa biến chứng	
E10.8	Bệnh đái tháo đường típ 1, kèm biến chứng không xác định	
E10.9	Bệnh đái tháo đường típ 1, không kèm biến chứng	
E11	Bệnh đái tháo đường típ 2	
E11.0	Bệnh đái tháo đường típ 2, kèm hôn mê	
E11.1	Bệnh đái tháo đường típ 2, kèm nhiễm toan ceton	
E11.2†	Bệnh đái tháo đường típ 2, kèm biến chứng thận	
E11.3†	Bệnh đái tháo đường típ 2, kèm biến chứng mắt	
E11.4†	Bệnh đái tháo đường típ 2, kèm biến chứng thần kinh	
E11.5	Bệnh đái tháo đường típ 2, kèm biến chứng mạch máu ngoại vi	
E11.6	Bệnh đái tháo đường típ 2, kèm biến chứng xác định khác	
E11.7	Bệnh đái tháo đường típ 2, kèm đa biến chứng	
E11.8	Bệnh đái tháo đường típ 2, kèm biến chứng không xác định	
E11.9	Bệnh đái tháo đường típ 2, không kèm biến chứng	
E12	Bệnh đái tháo đường liên quan đến suy dinh dưỡng	
E12.0	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm hôn mê	
E12.1	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm nhiễm toan ceton	
E12.2†	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng thận	
E12.3†	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng mắt	
E12.4†	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng thần kinh	
E12.5	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng mạch máu ngoại vi	
E12.6	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng xác định khác	
E12.7	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm đa biến chứng	
E12.8	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, kèm biến chứng không xác định	
E12.9	Bệnh đái tháo đường liên quan đến suy dinh dưỡng, không kèm biến chứng	
E13	Bệnh đái tháo đường xác định khác	
E13.0	Bệnh đái tháo đường xác định khác, kèm hôn mê	
E13.1	Bệnh đái tháo đường xác định khác, kèm nhiễm toan ceton	
E13.2†	Bệnh đái tháo đường xác định khác, kèm biến chứng thận	
E13.3†	Bệnh đái tháo đường xác định khác, kèm biến chứng mắt	
E13.4†	Bệnh đái tháo đường xác định khác, kèm biến chứng thần kinh	
E13.5	Bệnh đái tháo đường xác định khác, kèm biến chứng mạch máu ngoại vi	
E13.6	Bệnh đái tháo đường xác định khác, kèm biến chứng xác định khác	
E13.7	Bệnh đái tháo đường xác định khác, kèm đa biến chứng	
E13.8	Bệnh đái tháo đường xác định khác, kèm biến chứng không xác định	
E13.9	Bệnh đái tháo đường xác định khác, không kèm biến chứng	
E14	Đái tháo đường không xác định	
E14.0	Đái tháo đường không xác định, kèm hôn mê	
E14.1	Đái tháo đường không xác định, kèm nhiễm toan ceton	
E14.2†	Đái tháo đường không xác định, kèm biến chứng thận	
E14.3†	Đái tháo đường không xác định, kèm biến chứng mắt	
E14.4†	Đái tháo đường không xác định, kèm biến chứng thần kinh	
E14.5	Đái tháo đường không xác định, kèm biến chứng mạch máu ngoại vi	
E14.6	Đái tháo đường không xác định, kèm biến chứng xác định khác	
E14.7	Đái tháo đường không xác định, kèm đa biến chứng	
E14.8	Đái tháo đường không xác định, kèm biến chứng không xác định	
E14.9	Đái tháo đường không xác định, không kèm biến chứng	
E15	Hôn mê hạ đường huyết không do đái tháo đường	
E16	Rối loạn khác của tuyến tụy nội tiết	
E16.0	Hạ đường huyết do thuốc, không hôn mê	
E16.1	Hạ đường huyết khác	
E16.2	Hạ đường huyết, không xác định	
E16.3	Tăng tiết glucagon	Tăng sản tế bào tụy nội tiết với tăng tiết glucagon
E16.4	Tiết gastrin bất thường	Tăng gastrin máu|Hội chứng Zollinger-Ellison
E16.8	Rối loạn xác định khác của tuyến tụy nội tiết	
E16.9	Rối loạn tuyến tụy nội tiết, không xác định	Tăng sản tế bào tiểu đảo tụy không xác định khác|Tăng sản tế bào tụy nội tiết không xác định khác
E20	Suy cận giáp	
E20.0	Suy cận giáp vô căn	
E20.1	Giả suy cận giáp	
E20.8	Suy cận giáp khác	
E20.9	Suy cận giáp, không xác định	Cơn co thắt không tự chủ [tetany] do tuyến cận giáp
E21	Cường cận giáp và/hoặc rối loạn khác của tuyến cận giáp	
E21.0	Cường cận giáp nguyên phát	Tăng sản tuyến cận giáp|Viêm xương nang xơ toàn thân [bệnh von Recklinghausen của xương]
E21.1	Cường cận giáp thứ phát, không phân loại mục khác	
E21.2	Cường cận giáp khác [độ 3]	
E21.3	Cường cận giáp, không xác định	
E21.4	Rối loạn xác định khác của tuyến cận giáp	
E21.5	Rối loạn tuyến cận giáp, không xác định	
E22	Tăng cường chức năng tuyến yên	
E22.0	Bệnh to đầu chi và/hoặc bệnh khổng lồ do tuyến yên	
E22.1	Tăng prolactin máu	
E22.2	Hội chứng tiết nội tiết tố [hormon] chống bài niệu không thích hợp	
E22.8	Cường chức năng khác của tuyến yên	Dậy thì sớm trung ương
E22.9	Tăng cường chức năng tuyến yên, không xác định	
E23	Suy chức năng và/hoặc rối loạn khác của tuyến yên	
E23.0	Suy tuyến yên	
E23.1	Suy tuyến yên do thuốc	
E23.2	Đái tháo nhạt	
E23.3	Rối loạn chức năng vùng dưới đồi, không phân loại mục khác	
E23.6	Rối loạn khác của tuyến yên	Áp xe tuyến yên|Loạn dưỡng phì sinh dục
E23.7	Rối loạn tuyến yên, không xác định	
E24	Hội chứng Cushing	
E24.0	Bệnh Cushing phụ thuộc tuyến yên	Tăng sản xuất ACTH tuyến yên|Tăng năng vỏ thượng thận phụ thuộc tuyến yên
E24.1	Hội chứng Nelson	
E24.2	Hội chứng Cushing do thuốc	
E24.3	Hội chứng ACTH lạc chỗ	
E24.4	Hội chứng Cushing giả do rượu	
E24.8	Hội chứng Cushing khác	
E24.9	Hội chứng Cushing, không xác định	
E25	Rối loạn thượng thận - sinh dục	Rối loạn thượng thận-sinh dục
E25.0	Rối loạn thượng thận - sinh dục bẩm sinh liên quan đến thiếu hụt enzym	Tăng sản thượng thận bẩm sinh|Thiếu men 21-Hydroxylase|Tăng sản thượng thận bẩm sinh gây mất muối
E25.8	Rối loạn khác của thượng thận - sinh dục	
E25.9	Rối loạn thượng thận - sinh dục, không xác định	Hội chứng thượng thận - sinh dục không xác định khác
E26	Cường aldosterone	
E26.0	Cường aldosterone nguyên phát	
E26.1	Cường aldosterone thứ phát	
E26.8	Cường aldosterone khác	Hội chứng Bartter
E26.9	Cường aldosterone, không xác định	
E27	Rối loạn khác của tuyến thượng thận	
E27.0	Tăng hoạt vỏ thượng thận khác	
E27.1	Thiểu năng vỏ thượng thận nguyên phát	
E27.2	Cơn Addison	Cơn suy thượng thận cấp|Cơn suy vỏ thượng thận cấp
E27.3	Thiểu năng vỏ thượng thận do thuốc	
E27.4	Thiểu năng vỏ thượng thận khác và/hoặc không xác định	
E27.5	Cường chức năng tủy thượng thận	Tăng sản tủy thượng thận|Tăng tiết catecholamine
E27.8	Rối loạn xác định khác của tuyến thượng thận	Bất thường về globulin gắn cortisol
E27.9	Rối loạn tuyến thượng thận, không xác định	
E28	Rối loạn chức năng buồng trứng	
E28.0	Rối loạn chức năng buồng trứng: Thừa estrogen	
E28.1	Rối loạn chức năng buồng trứng: Thừa androgen	
E28.2	Hội chứng buồng trứng đa nang	Hội chứng buồng trứng xơ nang|Hội chứng Stein-Leventhal
E28.3	Suy buồng trứng nguyên phát	
E28.8	Rối loạn chức năng buồng trứng khác	Cường năng buồng trứng không xác định khác
E28.9	Rối loạn chức năng buồng trứng, không xác định	
E29	Rối loạn chức năng tinh hoàn	
E29.0	Cường chức năng tinh hoàn	Tăng tiết hormon - tinh hoàn
E29.1	Thiểu năng tinh hoàn	
E29.8	Rối loạn chức năng tinh hoàn khác	
E29.9	Rối loạn chức năng tinh hoàn, không xác định	
E30	Rối loạn tuổi dậy thì, không phân loại mục khác	
E30.0	Dậy thì muộn	Dậy thì muộn do thể tạng|Phát triển tình dục muộn
E30.1	Dậy thì sớm	
E30.8	Rối loạn khác của tuổi dậy thì	Vú phát triển sớm
E30.9	Rối loạn tuổi dậy thì, không xác định	
E31	Rối loạn chức năng đa tuyến	
E31.0	Suy đa tuyến tự miễn	Hội chứng tự miễn Schmidt
E31.1	Cường chức năng đa tuyến	
E31.8	Rối loạn chức năng đa tuyến khác	
E31.9	Rối loạn chức năng đa tuyến, không xác định	
E32	Bệnh tuyến ức	
E32.0	Tăng sản tuyến ức dai dẳng	Phì đại tuyến ức
E32.1	Áp xe tuyến ức	
E32.8	Bệnh tuyến ức khác	
E32.9	Bệnh tuyến ức, không xác định	
E34	Rối loạn nội tiết khác	
E34.0	Hội chứng carcinoid	
E34.1	Tăng tiết khác của nội tiết tố [hormon] ruột	
E34.2	Tiết nội tiết tố [hormon] lạc chỗ, không phân loại mục khác	
E34.3	Vóc dáng lùn, không phân loại mục khác	
E34.4	Thể tạng cao	Thể tạng khổng lồ
E34.5	Hội chứng kháng androgen	
E34.8	Rối loạn nội tiết xác định khác	Rối loạn chức năng tuyến tùng|Lão hóa sớm [Progeria]
E34.9	Rối loạn nội tiết, không xác định	
E35.*	Rối loạn tuyến nội tiết do bệnh phân loại mục khác	
E35.0*	Rối loạn tuyến giáp do bệnh phân loại mục khác	
E35.1*	Rối loạn tuyến thượng thận do bệnh phân loại mục khác	
E35.8*	Rối loạn của tuyến nội tiết khác do bệnh phân loại mục khác	
E40	Suy dinh dưỡng thể phù Kwashiorkor	
E41	Suy dinh dưỡng thể teo đét [marasmus]	
E42	Suy dinh dưỡng thể hỗn hợp marasmus-kwashiorkor	Suy dinh dưỡng thể hỗn hợp marasmus - kwashiorkor
E43	Suy dinh dưỡng nặng do thiếu protein - năng lượng, không xác định	
E44	Suy dinh dưỡng vừa và/hoặc nhẹ do thiếu protein - năng lượng	
E44.0	Suy dinh dưỡng vừa do thiếu protein - năng lượng	
E44.1	Suy dinh dưỡng nhẹ do thiếu protein - năng lượng	
E45	Chậm phát triển sau suy dinh dưỡng do thiếu protein - năng lượng	
E46	Suy dinh dưỡng do thiếu protein - năng lượng không xác định	Suy dinh dưỡng do thiếu protein-năng lượng không xác định
E50	Thiếu vitamin A	
E50.0	Khô kết mạc do thiếu vitamin A	
E50.1	Vết Bitot và khô kết mạc do thiếu vitamin A	Vết Bitot ở trẻ nhỏ
E50.2	Khô giác mạc do thiếu vitamin A	
E50.3	Loét và khô giác mạc do thiếu vitamin A	
E50.4	Nhuyễn giác mạc do thiếu vitamin A	
E50.5	Quáng gà do thiếu Vitamin A	
E50.6	Khô giác mạc có sẹo do thiếu vitamin A	
E50.7	Biểu hiện khác ở mắt do thiếu vitamin A	Bệnh khô mắt không xác định khác
E50.8	Biểu hiện khác của thiếu vitamin A	
E50.9	Thiếu vitamin A, không xác định	Thiếu vitamin A không xác định khác
E51	Thiếu thiamine	
E51.1	Bệnh tê phù	
E51.2	Bệnh lý não Wernicke	
E51.8	Biểu hiện khác của thiếu thiamine	
E51.9	Thiếu thiamine, không xác định	
E52	Thiếu niacin [bệnh pellagra]	
E53	Thiếu vitamin nhóm B khác	
E53.0	Thiếu vitamin B2	Thiếu riboflavin
E53.1	Thiếu pyridoxine	
E53.8	Thiếu vitamin nhóm B xác định khác	
E53.9	Thiếu vitamin B, không xác định	
E54	Thiếu acid ascorbic	
E55	Thiếu vitamin D	
E55.0	Bệnh còi xương, tiến triển	
E55.9	Thiếu vitamin D, không xác định	Thiếu vitamin D
E56	Thiếu vitamin khác	
E56.0	Thiếu vitamin E	
E56.1	Thiếu vitamin K	
E56.8	Thiếu vitamin khác	
E56.9	Thiếu vitamin, không xác định	
E58	Thiếu calci do chế độ ăn	
E59	Thiếu selen do chế độ ăn	
E60	Thiếu kẽm do chế độ ăn	
E61	Thiếu yếu tố dinh dưỡng khác	
E61.0	Thiếu đồng	
E61.1	Thiếu sắt	
E61.2	Thiếu magie	
E61.3	Thiếu mangan	
E61.4	Thiếu crôm	
E61.5	Thiếu molypđen	
E61.6	Thiếu vanadin	
E61.7	Thiếu nhiều yếu tố dinh dưỡng	
E61.8	Thiếu yếu tố dinh dưỡng xác định khác	
E61.9	Thiếu yếu tố dinh dưỡng, không xác định	
E63	Thiếu dinh dưỡng khác	
E63.0	Thiếu acid béo thiết yếu [EFA]	
E63.1	Mất cân đối trong thành phần thức ăn	
E63.8	Thiếu dinh dưỡng xác định khác	
E63.9	Thiếu dinh dưỡng, không xác định	
E64	Di chứng của suy dinh dưỡng và/hoặc thiếu hụt dinh dưỡng khác	
E64.0	Di chứng của suy dinh dưỡng protein - năng lượng	
E64.1	Di chứng của bệnh thiếu vitamin A	
E64.2	Di chứng của bệnh thiếu vitamin C	
E64.3	Di chứng của bệnh còi xương	
E64.8	Di chứng của bệnh suy dinh dưỡng khác	
E64.9	Di chứng của bệnh suy dinh dưỡng không xác định	
E65	Bệnh béo phì khu trú	
E66	Bệnh béo phì	
E66.0	Bệnh béo phì do thừa calo	
E66.1	Bệnh béo phì do thuốc	
E66.2	Bệnh béo phì quá mức với giảm thông khí phế nang	
E66.8	Bệnh béo phì khác	Béo phì bệnh lý
E66.9	Bệnh béo phì, không xác định	Béo phì đơn thuần không xác định khác
E67	Tình trạng thừa dinh dưỡng khác	
E67.0	Thừa vitamin A	
E67.1	Tăng carotene máu	
E67.2	Hội chứng Megavitamin-B6	
E67.3	Thừa vitamin D	
E67.8	Thừa dinh dưỡng xác định khác	
E68	Di chứng của thừa dinh dưỡng	
E70	Rối loạn chuyển hóa acid amin thơm	
E70.0	Phenyl-ceton niệu kinh điển	
E70.1	Tăng phenylalanin máu khác	
E70.2	Rối loạn chuyển hóa tyrosine	Alkapton niệu|Tăng tyrosine máu|Bệnh mô xám nâu|Tyrosine máu|Chứng loạn chuyển hóa tyrosin
E70.3	Chứng bạch tạng	
E70.8	Rối loạn chuyển hóa khác của acid amin thơm	
E70.9	Rối loạn chuyển hóa acid amin thơm, không xác định	
E71	Rối loạn chuyển hóa acid amin chuỗi nhánh và/hoặc rối loạn chuyển hóa acid béo	
E71.0	Bệnh siro niệu [Maple-syrup]	
E71.1	Rối loạn khác của chuyển hóa acid amin chuỗi nhánh	Tăng leucine-isoleucin máu|Tăng valine máu|Acid isovaleric máu|Acid methylmalonic máu|Acid propionic máu
E71.2	Rối loạn chuyển hóa acid amin chuỗi nhánh, không xác định	
E71.3	Rối loạn chuyển hóa acid béo	
E72	Rối loạn khác của chuyển hóa acid amin	
E72.0	Rối loạn vận chuyển acid amin	
E72.1	Rối loạn chuyển hóa acid amin chứa sulfur	
E72.2	Rối loạn chuyển hóa chu trình urê	
E72.3	Rối loạn chuyển hóa lysine và hydroxylysine	
E72.4	Rối loạn chuyển hóa ornithine	
E72.5	Rối loạn chuyển hóa glycine	
E72.8	Rối loạn xác định khác của chuyển hóa acid amin	
E72.9	Rối loạn chuyển hóa acid amin, không xác định	
E73	Không dung nạp lactose	
E73.0	Thiếu men lactase bẩm sinh	
E73.1	Thiếu men lactase thứ phát	
E73.8	Không dung nạp lactose khác	
E73.9	Không dung nạp lactose, không xác định	
E74	Rối loạn khác của chuyển hóa carbohydrat	
E74.0	Bệnh tích lũy glycogen	Bệnh tích lũy glycogen ở tim
E74.1	Rối loạn chuyển hóa fructose	Fructose niệu nguyên phát|Thiếu men fructose-1,6-diphosphatase|Bất dung nạp fructose có tính di truyền
E74.2	Rối loạn chuyển hóa galactose	Thiếu men galactokinase|Galactose máu
E74.3	Rối loạn khác của hấp thu carbohydrat ở ruột non	
E74.4	Rối loạn chuyển hóa pyruvat và/hoặc tân tạo glucose	
E74.8	Rối loạn xác định khác của chuyển hóa carbohydrat	Pentose niệu nguyên phát|Tích oxalat|Oxalat niệu|Glucoza niệu do thận
E74.9	Rối loạn chuyển hóa carbohydrat, không xác định	
E75	Rối loạn chuyển hóa sphingolipid và/hoặc rối loạn tích lũy lipid	
E75.0	Bệnh nhiễm gangliosid GM2	
E75.1	Bệnh nhiễm gangliosid khác	
E75.2	Bệnh nhiễm sphingolipid khác	
E75.3	Bệnh nhiễm sphingolipid, không xác định	
E75.4	Bệnh lý tích tụ lipofuscin ở neuron	
E75.5	Rối loạn tích lũy lipid khác	Rối loạn tích lũy cholesterol não [van Bogaert-Scherer- Epstein]|Bệnh Wolman
E75.6	Rối loạn tích lũy lipid, không xác định	
E76	Rối loạn chuyển hóa glycosaminoglycan	
E76.0	Bệnh mucopolysaccharid [MPS], típ I	
E76.1	Bệnh mucopolysaccharid [MPS], típ II	Hội chứng Hunter
E76.2	Bệnh mucopolysaccharid [MPS] khác	
E76.3	Bệnh mucopolysaccharid, không xác định	
E76.8	Rối loạn chuyển hóa glucosaminoglycan khác	
E76.9	Rối loạn chuyển hóa glucosaminoglycan, không xác định	
E77	Rối loạn chuyển hóa glycoprotein	
E77.0	Khiếm khuyết do sự biến đổi sau chuyển mã của men tiêu bào	Bệnh mucolipid II [bệnh tế bào I]|Bệnh mucolipid II [đa loạn dưỡng giả Hurler]
E77.1	Khiếm khuyết do quá trình phân hủy glycoprotein	Aspartylglucosamin niệu|Nhiễm fucosid|Nhiễm mannosid|Nhiễm sialid [bệnh mucolipid I]
E77.8	Rối loạn chuyển hóa glycoprotein khác	
E77.9	Rối loạn chuyển hóa glycoprotein, không xác định	
E78	Rối loạn chuyển hóa lipoprotein và/hoặc tình trạng tăng lipid máu khác	
E78.0	Tăng cholesterol máu đơn thuần	Tăng cholesterol máu di truyền|Tăng lipoprotein Fredrickson, típ IIa|Tăng betalipoprotein máu|Tăng lipid máu, nhóm A|Tăng lipoprotein máu loại tỷ trọng thấp [LDL]
E78.1	Tăng triglycerid máu đơn thuần	Tăng glycerid máu nội sinh|Tăng lipoprotein máu Fredrickson, típ IV|Tăng lipid máu, nhóm B|Tăng tiền beta lipoprotein máu|Tăng lipoprotein máu loại tỷ trọng thấp [VLDL]
E78.2	Tăng lipid máu hỗn hợp	
E78.3	Tăng chylomicron máu	Tăng lipoprotein máu Fredrickson típ I hoặc típ V|Tăng lipid máu, nhóm D|Tăng glycerid máu hỗn hợp
E78.4	Tăng lipid máu khác	Tăng lipid máu phối hợp di truyền
E78.5	Tăng lipid máu, không xác định	
E78.6	Thiếu lipoprotein	
E78.8	Rối loạn chuyển hóa lipoprotein khác	
E78.9	Rối loạn chuyển hóa lipoprotein, không xác định	
E79	Rối loạn chuyển hóa purine và/hoặc pyrimidine	
E79.0	Tăng acid uric máu không có biểu hiện của viêm khớp và/hoặc bệnh tạo sỏi	Tăng acid uric máu không triệu chứng
E79.1	Hội chứng Lesch-Nyhan	
E79.8	Rối loạn khác của chuyển hóa purine và/hoặc pyrimidine	Xanthine niệu di truyền
E79.9	Rối loạn chuyển hóa purine và pyrimidine, không xác định	
E80	Rối loạn chuyển hóa porphyrin và/hoặc bilirubin	
E80.0	Rối loạn chuyển hóa porphyrin sinh hồng cầu di truyền	
E80.1	Rối loạn chuyển hóa porphyrin biểu hiện muộn ở da	
E80.2	Rối loạn chuyển hóa porphyrin máu khác	
E80.3	Khiếm khuyết men catalase và/hoặc peroxidase	Acatalasia [Takahara]
E80.4	Hội chứng Glibert	
E80.5	Hội chứng Crigler-Najjar	
E80.6	Rối loạn khác của chuyển hóa bilirubin	Hội chứng Dubin-Johnson|Hội chứng Rotor
E80.7	Rối loạn chuyển hóa bilirubin, không xác định	
E83	Rối loạn chuyển hóa chất khoáng	
E83.0	Rối loạn chuyển hóa đồng	
E83.1	Rối loạn chuyển hóa sắt	
E83.2	Rối loạn chuyển hóa kẽm	Viêm da đầu chi ruột
E83.3	Rối loạn chuyển hóa phospho và/hoặc phosphatase	
E83.4	Rối loạn chuyển hóa magie	Tăng magie máu|Giảm magie máu
E83.5	Rối loạn chuyển hóa calci	
E83.8	Rối loạn chuyển hóa chất khoáng khác	
E83.9	Rối loạn chuyển hóa chất khoáng, không xác định	
E84	Xơ nang	
E84.0	Xơ nang kèm biểu hiện tại phổi	
E84.1	Xơ nang kèm biểu hiện tại ruột	
E84.8	Xơ nang kèm biểu hiện khác	
E84.9	Xơ nang, không xác định	
E85	Thoái hóa tinh bột	
E85.0	Thoái hóa tinh bột mang tính di truyền gia đình không có bệnh lý thần kinh	Sốt Địa Trung Hải di truyền [có yếu tố gia đình]|Bệnh lý thận tinh bột di truyền
E85.1	Thoái hóa tinh bột mang tính di truyền gia đình, có bệnh lý thần kinh	
E85.2	Thoái hóa tinh bột mang tính di truyền gia đình, không xác định	
E85.3	Thoái hóa dạng bột toàn thân thứ phát	Thoái hóa tinh bột liên quan đến thẩm tách máu [lọc máu]
E85.4	Thoái hóa tinh bột giới hạn ở cơ quan	Thoái hóa tinh bột khu trú
E85.8	Thoái hóa tinh bột khác	
E85.9	Thoái hóa tinh bột, không xác định	
E86	Giảm thể tích	
E87	Rối loạn cân bằng nước, điện giải và/hoặc thăng bằng kiềm toan	
E87.0	Tăng áp suất thẩm thấu và/hoặc tăng natri máu	Thừa natri [Na]|Quá tải natri [Na]
E87.1	Giảm áp suất thẩm thấu và/hoặc giảm natri máu	
E87.2	Nhiễm toan	
E87.3	Nhiễm kiềm	
E87.4	Rối loạn cân bằng kiềm toan phối hợp	
E87.5	Tăng kali máu	Thừa kali [K]|Quá tải kali [K]
E87.6	Hạ kali máu	Thiếu kali [K]
E87.7	Quá tải dịch	
E87.8	Rối loạn khác về cân bằng điện giải và/hoặc nước, không phân loại mục khác	Mất cân bằng điện giải không xác định khác|Tăng clo máu|Hạ clo máu
E88	Rối loạn chuyển hóa khác	
E88.0	Rối loạn chuyển hóa protein huyết tương, không phân loại mục khác	
E88.1	Loạn dưỡng mỡ, không phân loại mục khác	
E88.2	Bệnh u mỡ, không phân loại mục khác	
E88.3	Hội chứng ly giải khối u	
E88.8	Rối loạn chuyển hóa xác định khác	Bệnh u tuyến cơ Launois-Bensaude|Trimethylamin niệu
E88.9	Rối loạn chuyển hóa, không xác định	
E89	Rối loạn nội tiết và/hoặc chuyển hóa sau can thiệp, không phân loại mục khác	
E89.0	Suy giáp sau can thiệp	Suy tuyến giáp thẩm thấu|Suy tuyến giáp sau phẫu thuật
E89.1	Hạ insulin huyết sau can thiệp	Tăng đường huyết sau cắt tụy|Giảm insulin huyết sau phẫu thuật
E89.2	Suy cận giáp sau can thiệp	Cơn co thắt không tự chủ [tetany] do suy giảm chức năng tuyến cận giáp
E89.3	Suy tuyến yên sau can thiệp	Suy tuyến yên sau xạ trị
E89.4	Suy buồng trứng sau can thiệp	
E89.5	Suy tinh hoàn sau can thiệp	
E89.6	Suy vỏ (-tủy) thượng thận sau can thiệp	
E89.8	Rối loạn nội tiết và/hoặc chuyển hóa khác sau can thiệp	
E89.9	Rối loạn nội tiết và/hoặc chuyển hóa sau can thiệp, không xác định	
E90.*	Rối loạn chuyển hóa và/hoặc dinh dưỡng do bệnh phân loại mục khác	
F00.*	Sa sút trí tuệ do bệnh Alzheimer (G30.-†)	
F00.0*	Sa sút trí tuệ do bệnh Alzheimer khởi phát sớm (G30.0†)	Bệnh Alzheimer, típ 2|Sa sút trí tuệ trước tuổi già, loại Alzheimer|Chứng sa sút trí tuệ thoái hóa nguyên phát thuộc loại Alzheimer, khởi phát trước tuổi già
F00.1*	Sa sút trí tuệ do bệnh Alzheimer khởi phát muộn (G30.1†)	Bệnh Alzheimer, típ 1|Chứng sa sút trí tuệ thoái hóa nguyên phát thuộc loại Alzheimer, khởi phát khi về già|Chứng sa sút trí tuệ do tuổi già, loại bệnh Alzheimer
F00.2*	Sa sút trí tuệ do bệnh Alzheimer, thể không điển hình hoặc thể hỗn hợp (G30.8†)	Sa sút trí tuệ không điển hình, thể Alzheimer
F00.9*	Sa sút trí tuệ do bệnh Alzheimer, không xác định (G30.9†)	
F01	Sa sút trí tuệ do bệnh mạch máu	
F01.0	Sa sút trí tuệ do bệnh mạch máu giai đoạn khởi phát cấp tính	
F01.1	Sa sút trí tuệ do nhồi máu não đa ổ	Chủ yếu là chứng sa sút trí tuệ vỏ não
F01.2	Sa sút trí tuệ do bệnh mạch máu dưới vỏ	
F01.3	Sa sút trí tuệ do bệnh mạch máu kết hợp vỏ não và/hoặc dưới vỏ	
F01.8	Sa sút trí tuệ do bệnh mạch máu khác	
F01.9	Sa sút trí tuệ do bệnh mạch máu, không xác định	
F02.*	Sa sút trí tuệ do bệnh khác đã phân loại mục khác	
F02.0*	Sa sút trí tuệ do bệnh Pick (G31.0†)	
F02.1*	Sa sút trí tuệ do bệnh Creutzfeldt-Jakob (A81.0†)	
F02.2*	Sa sút trí tuệ do bệnh Huntington (G10†)	Sa sút trí tuệ do bệnh múa giật Huntington
F02.3*	Sa sút trí tuệ do bệnh Parkinson (G20†)	
F02.4*	Sa sút trí tuệ do bệnh [HIV] nhiễm virus gây suy giảm miễn dịch ở người (B22.0†)	
F02.8*	Sa sút trí tuệ do bệnh lý xác định khác phân loại mục khác	
F03	Sa sút trí tuệ không xác định	
F04	Hội chứng quên thực tổn, không do rượu và/hoặc chất hướng thần khác	
F05	Mê sảng, không do rượu và/hoặc chất hướng thần khác	
F05.0	Mê sảng không xuất hiện đồng thời với sa sút trí tuệ, như đã mô tả	
F05.1	Mê sảng xuất hiện đồng thời với sa sút trí tuệ	
F05.8	Mê sảng khác	Mê sảng căn nguyên hỗn hợp|Mê sảng sau phẫu thuật
F05.9	Mê sảng, không xác định	
F06	Rối loạn tâm thần khác do tổn thương và/hoặc rối loạn chức năng não và/hoặc do bệnh thực thể	
F06.0	Ảo giác thực tổn	
F06.1	Rối loạn căng trương lực thực tổn	
F06.2	Rối loạn hoang tưởng thực tổn [giống tâm thần phân liệt]	
F06.3	Rối loạn khí sắc [cảm xúc] thực tổn	
F06.4	Rối loạn lo âu thực tổn	
F06.5	Rối loạn phân ly thực tổn	
F06.6	Rối loạn cảm xúc không ổn định [suy nhược] thực tổn	
F06.7	Rối loạn nhận thức nhẹ	
F06.8	Rối loạn tâm thần xác định khác do tổn thương và/hoặc rối loạn chức năng não và/hoặc bệnh thực thể	Loạn thần động kinh không xác định khác
F06.9	Rối loạn tâm thần không xác định do tổn thương và/hoặc rối loạn chức năng não và/hoặc bệnh thực thể	
F07	Rối loạn nhân cách và/hoặc hành vi do bệnh lý, tổn thương và/hoặc rối loạn chức năng não	
F07.0	Rối loạn nhân cách thực tổn	
F07.1	Hội chứng sau viêm não	
F07.2	Hội chứng sau chấn động não	
F07.8	Rối loạn nhân cách và/hoặc hành vi thực tổn khác do bệnh lý, tổn thương và/hoặc rối loạn chức năng não	Rối loạn cảm xúc thực tổn bán cầu não phải
F07.9	Rối loạn nhân cách và/hoặc hành vi thực thể không xác định do bệnh lý, tổn thương và/hoặc rối loạn chức năng não	Hội chứng tâm thần thực tổn
F09	Rối loạn tâm thần thực tổn hoặc rối loạn tâm thần có triệu chứng không xác định	
F10	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu	
F10.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, nhiễm độc cấp	
F10.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, sử dụng gây hại	
F10.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, hội chứng nghiện	
F10.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, trạng thái cai	
F10.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, trạng thái cai kèm mê sảng	
F10.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, loạn thần	
F10.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, hội chứng quên [mất trí nhớ]	
F10.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, loạn thần di chứng và/hoặc khởi phát muộn	
F10.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, rối loạn hành vi và/hoặc tâm thần khác	
F10.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng rượu, rối loạn hành vi và/hoặc tâm thần không xác định	
F11	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện	
F11.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, nhiễm độc cấp	
F11.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, sử dụng gây hại	
F11.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, hội chứng nghiện	
F11.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, trạng thái cai	
F11.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, trạng thái cai kèm mê sảng	
F11.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, loạn thần	
F11.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, hội chứng quên [mất trí nhớ]	
F11.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, loạn thần di chứng và/hoặc khởi phát muộn	
F11.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, rối loạn hành vi và/hoặc tâm thần khác	
F11.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất dạng thuốc phiện, rối loạn hành vi và/hoặc tâm thần không xác định	
F12	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa	
F12.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, nhiễm độc cấp	
F12.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, sử dụng gây hại	
F12.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, hội chứng nghiện	
F12.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, trạng thái cai	
F12.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, trạng thái cai kèm mê sảng	
F12.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, loạn thần	
F12.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, hội chứng quên [mất trí nhớ]	
F12.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, loạn thần di chứng và/hoặc khởi phát muộn	
F12.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, rối loạn hành vi và/hoặc tâm thần khác	
F12.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng cần sa, rối loạn hành vi và/hoặc tâm thần không xác định	
F13	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ	
F13.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, nhiễm độc cấp	
F13.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, sử dụng gây hại	
F13.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, hội chứng nghiện	
F13.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, trạng thái cai	
F13.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, trạng thái cai kèm mê sảng	
F13.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, loạn thần	
F13.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, hội chứng quên [mất trí nhớ]	
F13.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, loạn thần di chứng và/hoặc khởi phát muộn	
F13.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, rối loạn hành vi và/hoặc tâm thần khác	
F13.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất an thần hoặc thuốc ngủ, rối loạn hành vi và/hoặc tâm thần không xác định	
F14	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain	
F14.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, nhiễm độc cấp	
F14.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, sử dụng gây hại	
F14.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, hội chứng nghiện	
F14.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, trạng thái cai	
F14.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, trạng thái cai kèm mê sảng	
F14.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, loạn thần	
F14.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, hội chứng quên [mất trí nhớ]	
F14.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, loạn thần di chứng và/hoặc khởi phát muộn	
F14.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, rối loạn hành vi và/hoặc tâm thần khác	
F14.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng cocain, rối loạn hành vi và/hoặc tâm thần không xác định	
F15	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein	
F15.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, nhiễm độc cấp	
F15.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, sử dụng gây hại	
F15.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, hội chứng nghiện	
F15.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, trạng thái cai	
F15.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, trạng thái cai kèm mê sảng	
F15.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, loạn thần	
F15.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, hội chứng quên [mất trí nhớ]	
F15.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, loạn thần di chứng và/hoặc khởi phát muộn	
F15.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, rối loạn hành vi và/hoặc tâm thần khác	
F15.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất kích thích khác, bao gồm cả caffein, rối loạn hành vi và/hoặc tâm thần không xác định	
F16	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác	
F16.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, nhiễm độc cấp	
F16.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, sử dụng gây hại	
F16.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, hội chứng nghiện	
F16.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, trạng thái cai	
F16.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, trạng thái cai kèm mê sảng	
F16.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, loạn thần	
F16.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, hội chứng quên [mất trí nhớ]	
F16.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, loạn thần di chứng và khởi phát muộn	
F16.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, rối loạn hành vi và/hoặc tâm thần khác	
F16.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng chất gây ảo giác, rối loạn hành vi và/hoặc tâm thần không xác định	
F17	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá	
F17.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, nhiễm độc cấp	
F17.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, sử dụng gây hại	
F17.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, hội chứng nghiện	
F17.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, trạng thái cai	
F17.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, trạng thái cai kèm mê sảng	
F17.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, loạn thần	
F17.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, hội chứng quên [mất trí nhớ]	
F17.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, loạn thần di chứng và/hoặc khởi phát muộn	
F17.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, rối loạn hành vi và/hoặc tâm thần khác	
F17.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng thuốc lá, rối loạn hành vi và/hoặc tâm thần không xác định	
F18	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi	
F18.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, nhiễm độc cấp	
F18.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, sử dụng gây hại	
F18.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, hội chứng nghiện	
F18.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, trạng thái cai	
F18.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, trạng thái cai kèm mê sảng	
F18.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, loạn thần	
F18.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, hội chứng quên [mất trí nhớ]	
F18.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, loạn thần di chứng và/hoặc khởi phát muộn	
F18.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, rối loạn hành vi và/hoặc tâm thần khác	
F18.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng dung môi dễ bay hơi, rối loạn hành vi và/hoặc tâm thần không xác định	
F19	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác	
F19.0	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, nhiễm độc cấp	
F19.1	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, sử dụng gây hại	
F19.2	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, hội chứng nghiện	
F19.3	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, trạng thái cai	
F19.4	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, trạng thái cai kèm mê sảng	
F19.5	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, loạn thần	
F19.6	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, hội chứng quên [mất trí nhớ]	
F19.7	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, loạn thần di chứng và/hoặc khởi phát muộn	
F19.8	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, rối loạn hành vi và/hoặc tâm thần khác	
F19.9	Rối loạn tâm thần và/hoặc hành vi do sử dụng nhiều chất ma túy và/hoặc chất hướng thần khác, rối loạn hành vi và/hoặc tâm thần không xác định	
F20	Tâm thần phân liệt	
F20.0	Tâm thần phân liệt thể paranoid	
F20.1	Tâm thần phân liệt thể thanh xuân	Tâm thần phân liệt thể thiếu tổ chức
F20.2	Tâm thần phân liệt thể căng trương lực	Sững sờ căng trương lực
F20.3	Tâm thần phân liệt thể không xác định	
F20.4	Tâm thần phân liệt thể trầm cảm sau phân liệt	
F20.5	Tâm thần phân liệt thể di chứng	
F20.6	Tâm thần phân liệt thể đơn thuần	
F20.8	Tâm thần phân liệt thể khác	
F20.9	Tâm thần phân liệt, không xác định	
F21	Rối loạn dạng phân liệt	
F22	Rối loạn hoang tưởng dai dẳng	
F22.0	Rối loạn hoang tưởng	
F22.8	Rối loạn hoang tưởng dai dẳng khác	
F22.9	Rối loạn hoang tưởng dai dẳng, không xác định	
F23	Loạn thần cấp tính và/hoặc ngắn hạn	
F23.0	Loạn thần cấp tính đa dạng không có triệu chứng của bệnh tâm thần phân liệt	
F23.1	Loạn thần cấp tính đa dạng với triệu chứng của bệnh tâm thần phân liệt	
F23.2	Loạn thần cấp tính giống tâm thần phân liệt	
F23.3	Loạn thần cấp tính khác chủ yếu hoang tưởng	
F23.8	Loạn thần cấp tính và/hoặc ngắn hạn khác	
F23.9	Loạn thần cấp tính và/hoặc ngắn hạn, không xác định	Rối loạn tâm thần phản ứng ngắn gọn không xác định khác|Rối loạn tâm thần phản ứng
F24	Rối loạn hoang tưởng cảm ứng	
F25	Loạn thần dạng rối loạn khí sắc	
F25.0	Loạn thần dạng rối loạn khí sắc, loại hưng cảm	Rối loạn tâm thần phân liệt cảm xúc, loại hưng cảm|Rối loạn tâm thần dạng phân liệt, loại hưng cảm
F25.1	Loạn thần dạng rối loạn khí sắc, loại trầm cảm	Rối loạn tâm thần phân liệt cảm xúc, loại trầm cảm|Rối loạn tâm thần dạng phân liệt, loại trầm cảm
F25.2	Loạn thần dạng rối loạn khí sắc, loại hỗn hợp	Tâm thần phân liệt thể chu kỳ|Tâm thần phân liệt và cảm xúc hỗn hợp
F25.8	Loạn thần dạng rối loạn khí sắc khác	
F25.9	Loạn thần dạng rối loạn khí sắc, không xác định	Loạn thần phân liệt cảm xúc không xác định khác
F28	Loạn thần không thực tổn khác	
F29	Loạn thần không thực tổn không xác định	
F30	Giai đoạn hưng cảm	
F30.0	Hưng cảm nhẹ	
F30.1	Hưng cảm không có triệu chứng loạn thần	
F30.2	Hưng cảm với triệu chứng loạn thần	
F30.8	Giai đoạn hưng cảm khác	
F30.9	Giai đoạn hưng cảm, không xác định	Hưng cảm không xác định khác
F31	Rối loạn cảm xúc lưỡng cực	
F31.0	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn hưng cảm nhẹ	
F31.1	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn hưng cảm không có triệu chứng loạn thần	
F31.2	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn hưng cảm có triệu chứng loạn thần	
F31.3	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn trầm cảm nhẹ hoặc trung bình	
F31.4	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn trầm cảm nặng không có triệu chứng loạn thần	
F31.5	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn trầm cảm nặng với triệu chứng loạn thần	
F31.6	Rối loạn cảm xúc lưỡng cực, hiện tại giai đoạn hỗn hợp	
F31.7	Rối loạn cảm xúc lưỡng cực, hiện tại thuyên giảm	
F31.8	Rối loạn cảm xúc lưỡng cực khác	Rối loạn lưỡng cực II|Các giai đoạn hưng cảm tái diễn không xác định khác
F31.9	Rối loạn cảm xúc lưỡng cực, không xác định	Trầm cảm hưng cảm không xác định khác
F32	Giai đoạn trầm cảm	
F32.0	Giai đoạn trầm cảm nhẹ	
F32.1	Giai đoạn trầm cảm vừa	
F32.2	Giai đoạn trầm cảm nặng không có triệu chứng loạn thần	
F32.3	Giai đoạn trầm cảm nặng kèm triệu chứng loạn thần	
F32.8	Giai đoạn trầm cảm khác	"Trầm cảm không điển hình|Các giai đoạn đơn độc của trầm cảm ""ẩn"" không xác định khác"
F32.9	Giai đoạn trầm cảm, không xác định	Trầm cảm không xác định khác|Rối loạn trầm cảm không xác định khác
F33	Rối loạn trầm cảm tái phát	
F33.0	Rối loạn trầm cảm tái phát, hiện tại giai đoạn nhẹ	
F33.1	Rối loạn trầm cảm tái phát, hiện tại giai đoạn vừa	
F33.2	Rối loạn trầm cảm tái phát, hiện tại giai đoạn nặng không kèm triệu chứng loạn thần	Trầm cảm nội sinh không có các triệu chứng loạn thần|Trầm cảm nặng, tái phát không có triệu chứng loạn thần|Loạn thần hưng trầm cảm, loại trầm cảm không có các triệu chứng loạn thần|Trầm cảm gây tử vong, tái phát không có các triệu chứng loạn thần
F33.3	Rối loạn trầm cảm tái phát, hiện tại giai đoạn nặng kèm triệu chứng loạn thần	Trầm cảm nội sinh kkemf các triệu chứng loạn thần|Loạn thần hưng trầm cảm thể trầm cảm kèm các triệu chứng loạn thần
F33.4	Rối loạn trầm cảm tái phát, hiện đang thuyên giảm	
F33.8	Rối loạn trầm cảm tái phát khác	
F33.9	Rối loạn trầm cảm tái phát, không xác định	Trầm cảm đơn cực không xác định khác
F34	Rối loạn khí sắc [cảm xúc] dai dẳng	
F34.0	Khí sắc chu kỳ	
F34.1	Trầm cảm dai dẳng	
F34.8	Rối loạn khí sắc [cảm xúc] dai dẳng khác	
F34.9	Rối loạn khí sắc [cảm xúc] dai dẳng, không xác định	
F38	Rối loạn khí sắc [cảm xúc] khác	
F38.0	Rối loạn khí sắc [cảm xúc] đơn độc khác	Giai đoạn cảm xúc hỗn hợp
F38.1	Rối loạn khí sắc [cảm xúc] tái phát	Các giai đoạn trầm cảm ngắn tái phát
F38.8	Rối loạn khí sắc [cảm xúc] biệt định khác	
F39	Rối loạn khí sắc [cảm xúc] không xác định	
F40	Rối loạn lo âu ám ảnh sợ	
F40.0	Chứng sợ khoảng trống vắng	
F40.1	Chứng sợ xã hội	Ám ảnh sợ người|Bệnh tâm căn xã hội
F40.2	Ám ảnh sợ (cô lập) xác định cụ thể	
F40.8	Rối loạn lo âu ám ảnh sợ khác	
F40.9	Rối loạn lo âu ám ảnh sợ, không xác định	Chứng ám ảnh không xác định khác|Trạng thái ám ảnh không xác định khác
F41	Rối loạn lo âu khác	
F41.0	Rối loạn hoảng sợ [lo âu kịch phát từng giai đoạn]	
F41.1	Rối loạn lo âu tổng quát	
F41.2	Rối loạn hỗn hợp lo âu và trầm cảm	
F41.3	Rối loạn lo âu hỗn hợp khác	
F41.8	Rối loạn lo âu xác định khác	Hysteria [chứng cuồng loạn] liên quan lo âu
F41.9	Rối loạn lo âu, không xác định	Lo âu không xác định khác
F42	Rối loạn ám ảnh cưỡng chế	
F42.0	Tư duy ám ảnh	
F42.1	Hành vi cưỡng chế chủ yếu [nghi thức ám ảnh]	
F42.2	Ý tưởng và/hoặc hành vi ám ảnh hỗn hợp	
F42.8	Rối loạn ám ảnh cưỡng chế khác	
F42.9	Rối loạn ám ảnh cưỡng chế, không xác định	
F43	Phản ứng với căng thẳng trầm trọng và/hoặc rối loạn điều chỉnh	
F43.0	Phản ứng với căng thẳng cấp tính	
F43.1	Rối loạn căng thẳng sau chấn thương	
F43.2	Rối loạn điều chỉnh [thích ứng]	
F43.8	Phản ứng khác với căng thẳng trầm trọng	
F43.9	Phản ứng với căng thẳng trầm trọng, không xác định	
F44	Rối loạn phân ly [chuyển dạng]	
F44.0	Chứng quên phân ly	
F44.1	Chứng rối loạn phân ly ra đi	
F44.2	Sững sờ phân ly	
F44.3	Rối loạn lên đồng và/hoặc bị nhập	
F44.4	Rối loạn vận động phân ly	
F44.5	Co giật phân ly	ý thức được duy trì hoặc được thay thế bằng trạng thái sững sờ hoặc lên đồng.
F44.6	Tê và/hoặc mất cảm giác phân ly	Điếc do tâm thần
F44.7	Rối loạn phân ly [chuyển dạng] hỗn hợp	Phối hợp những rối loạn đã xác định trong F44.0-F44.6
F44.8	Rối loạn phân ly [chuyển dạng] khác	Hội chứng Ganser|Đa nhân cách
F44.9	Rối loạn phân ly [chuyển dạng], không xác định	
F45	Rối loạn dạng cơ thể	
F45.0	Rối loạn đau cơ thể	
F45.1	Rối loạn dạng cơ thể không phân biệt	Rối loạn tâm thể không phân biệt
F45.2	Rối loạn nghi bệnh	
F45.3	Loạn chức năng thần kinh tự trị dạng cơ thể	
F45.4	Rối loạn đau dạng cơ thể dai dẳng	
F45.8	Rối loạn dạng cơ thể khác	
F45.9	Rối loạn dạng cơ thể, không xác định	Rối loạn tâm thể không xác định khác
F48	Rối loạn tâm căn khác	
F48.0	Suy nhược thần kinh	
F48.1	Hội chứng rối loạn nhân cách giải thể - mất ý thức thực tại	
F48.8	Rối loạn tâm căn xác định khác	
F48.9	Rối loạn tâm căn, không xác định	Rối loạn tâm căn không xác định khác
F50	Rối loạn ăn uống	
F50.0	Chán ăn tâm thần	
F50.1	Chán ăn tâm thần không điển hình	
F50.2	Chứng ăn bừa bãi	
F50.3	Chứng ăn bừa bãi không điển hình	
F50.4	Chứng ăn quá nhiều kết hợp với rối loạn tâm lý khác	
F50.5	Nôn kết hợp với rối loạn tâm lý khác	
F50.8	Rối loạn ăn uống khác	
F50.9	Rối loạn ăn uống, không xác định	
F51	Rối loạn giấc ngủ không thực tổn	
F51.0	Mất ngủ không thực tổn	
F51.1	Ngủ nhiều không thực tổn	
F51.2	Rối loạn lịch trình thức - ngủ không thực tổn	
F51.3	Chứng mộng du [miên hành]	
F51.4	Hoảng sợ trong giấc ngủ [giấc ngủ kinh hoàng]	
F51.5	Ác mộng	
F51.8	Rối loạn giấc ngủ không thực tổn khác	
F51.9	Rối loạn giấc ngủ không thực tổn, không xác định	Rối loạn giấc ngủ cảm xúc không xác định khác
F52	Rối loạn chức năng tình dục, không do rối loạn thực tổn hoặc bệnh lý	
F52.0	Thiếu hoặc mất ham muốn tình dục	Lãnh cảm|Rối loạn giảm ham muốn tình dục
F52.1	Ghét sợ tình dục và/hoặc thiếu thích thú tình dục	
F52.2	Thất bại trong đáp ứng tình dục	
F52.3	Rối loạn cực khoái	
F52.4	Xuất tinh sớm	
F52.5	Chứng đau co thắt âm hộ không do nguyên nhân thực tổn	
F52.6	Đau khi giao hợp không do nguyên nhân thực tổn	
F52.7	Xu hướng tình dục quá độ	Chứng loạn dâm ở nữ giới|Chứng loạn dâm ở nam giới
F52.8	Rối loạn chức năng tình dục khác, không do rối loạn thực tổn hoặc bệnh lý	
F52.9	Rối loạn chức năng tình dục không xác định, không do rối loạn thực tổn hoặc bệnh lý	
F53	Rối loạn tâm thần và/hoặc hành vi liên quan thời kỳ sau đẻ, không phân loại mục khác	
F53.0	Rối loạn tâm thần và/hoặc hành vi thể nhẹ liên quan thời kỳ sau đẻ, không phân loại mục khác	
F53.1	Rối loạn tâm thần và/hoặc hành vi nặng liên quan thời kỳ sau đẻ, không phân loại mục khác	Loạn thần sau thời kỳ sinh đẻ không xác định khác
F53.8	Rối loạn tâm thần và/hoặc hành vi khác liên quan thời kỳ sau đẻ, không phân loại mục khác	
F53.9	Rối loạn tâm thần trong thời kỳ sau đẻ, không xác định	
F54	Nhân tố tâm lý và/hoặc hành vi kết hợp với rối loạn hoặc bệnh phân loại mục khác	
F55	Lạm dụng chất không gây nghiện	
F59	Hội chứng hành vi ứng xử không xác định kết hợp với rối loạn sinh lý và/hoặc yếu tố thực thể	
F60	Rối loạn nhân cách xác định cụ thể	Đây là những rối loạn nghiêm trọng về nhân cách và xu hướng hành vi của cá nhân
F60.0	Rối loạn nhân cách hoang tưởng	
F60.1	Rối loạn nhân cách dạng phân liệt	
F60.2	Rối loạn nhân cách chống đối xã hội	
F60.3	Rối loạn nhân cách cảm xúc không ổn định	
F60.4	Rối loạn dạng nhân cách kịch tính	
F60.5	Rối loạn nhân cách ám ảnh nghi thức	
F60.6	Rối loạn nhân cách lo âu [tránh né]	
F60.7	Rối loạn nhân cách phụ thuộc	
F60.8	Rối loạn nhân cách xác định khác	
F60.9	Rối loạn nhân cách, không xác định	Loạn thần kinh tính cách không xác định khác|Nhân cách bệnh lý không xác định khác
F61	Rối loạn nhân cách khác và/hoặc rối loạn nhân cách hỗn hợp	
F62	Thay đổi nhân cách kéo dài, không thể quy cho một tổn thương hoặc bệnh não	Biến đổi nhân cách kéo dài, không thể quy cho một tổn thương hay bệnh não
F62.0	Thay đổi nhân cách kéo dài sau một sự kiện thảm khốc	
F62.1	Thay đổi nhân cách kéo dài sau bệnh tâm thần	thụ động, giảm hứng thú và giảm sự tham gia vào các hoạt động giải trí|và các vấn đề lâu dài trong hoạt động xã hội và nghề nghiệp.
F62.8	Thay đổi nhân cách kéo dài khác	Hội chứng nhân cách do đau mạn tính
F62.9	Thay đổi nhân cách kéo dài, không xác định	
F63	Rối loạn thói quen và/hoặc xung động	
F63.0	Cá cược bệnh lý	
F63.1	Xung động phóng hỏa [chứng mê đốt phá]	
F63.2	Trộm cắp bệnh lý [chứng ăn cắp vặt]	
F63.3	Chứng giật tóc	
F63.8	Rối loạn thói quen và/hoặc xung động khác	Rối loạn bộc phát từng cơn
F63.9	Rối loạn thói quen và/hoặc xung động, không xác định	
F64	Rối loạn nhận dạng giới tính	
F64.0	Chuyển giới tính	
F64.1	Rối loạn dục tính cải trang, hai vai trò	
F64.2	Rối loạn nhận dạng giới tính ở trẻ nhỏ	
F64.8	Rối loạn nhận dạng giới tính khác	
F64.9	Rối loạn nhận dạng giới tính, không xác định	Rối loạn vai trò giới tính không xác định khác
F65	Rối loạn sở thích tình dục	
F65.0	Rối loạn dục tính với đồ vật	
F65.1	Rối loạn dục tính cải trang với đồ vật	
F65.2	Rối loạn dục tính phô bày [phô dục]	
F65.3	Rối loạn dục tính nhìn trộm [thị dục]	
F65.4	Rối loạn dục tính với trẻ em	
F65.5	Khổ dâm, bạo dâm	Khổ dâm [thống dâm] [rối loạn dục tính thích đau]|Bạo dâm [ác dâm] [rối loạn dục tính gây đau]
F65.6	Đa rối loạn sở thích tình dục	
F65.8	Rối loạn sở thích tình dục khác	
F65.9	Rối loạn ưa chuộng tình dục, không xác định	Lệch lạc tình dục không xác định khác
F66	Rối loạn tâm lý và/hoặc hành vi liên quan với sự phát triển và/hoặc khuynh hướng tình dục	Rối loạn tâm lý và/hoặc hành vi kết hợp với sự phát triển và/hoặc khuynh hướng tình dục
F66.0	Rối loạn về sự trưởng thành tình dục	
F66.1	Khuynh hướng tình dục loạn trương lực bản thân	
F66.2	Rối loạn quan hệ tình dục	
F66.8	Rối loạn phát triển tâm lý tình dục khác	
F66.9	Rối loạn phát triển tâm lý tình dục, không xác định	
F68	Rối loạn khác về nhân cách và/hoặc hành vi ở người trưởng thành	
F68.0	Hình thành triệu chứng thực thể vì lý do tâm lý	Bệnh tâm căn đền bù
F68.1	Cố ý tạo ra hoặc nguỵ tạo triệu chứng hoặc khuyết tật hoặc thực thể hoặc tinh thần [rối loạn giả bệnh]	
F68.8	Rối loạn xác định khác về nhân cách và/hoặc hành vi ở người trưởng thành	Rối loạn tính cách không xác định khác|Rối loạn mối quan hệ không xác định khác
F69	Rối loạn không xác định về nhân cách và/hoặc hành vi ở người trưởng thành	
F70	Chậm phát triển trí tuệ nhẹ	
F70.0	Chậm phát triển trí tuệ nhẹ, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F70.1	Chậm phát triển trí tuệ nhẹ, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F70.8	Chậm phát triển trí tuệ nhẹ, những khiếm khuyết khác về hành vi	
F70.9	Chậm phát triển trí tuệ nhẹ, không đề cập đến suy giảm hành vi	
F71	Chậm phát triển trí tuệ trung bình	
F71.0	Chậm phát triển trí tuệ trung bình, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F71.1	Chậm phát triển trí tuệ trung bình, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F71.8	Chậm phát triển trí tuệ trung bình, những khiếm khuyết khác về hành vi	
F71.9	Chậm phát triển trí tuệ trung bình, không đề cập đến suy giảm hành vi	
F72	Chậm phát triển trí tuệ nặng	
F72.0	Chậm phát triển trí tuệ nặng, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F72.1	Chậm phát triển trí tuệ nặng, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F72.8	Chậm phát triển trí tuệ nặng, những khiếm khuyết khác về hành vi	
F72.9	Chậm phát triển trí tuệ nặng, không đề cập đến suy giảm hành vi	
F73	Chậm phát triển trí tuệ trầm trọng	
F73.0	Chậm phát triển trí tuệ trầm trọng, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F73.1	Chậm phát triển trí tuệ trầm trọng, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F73.8	Chậm phát triển trí tuệ trầm trọng, những khiếm khuyết khác về hành vi	
F73.9	Chậm phát triển trí tuệ trầm trọng, không đề cập đến suy giảm hành vi	
F78	Chậm phát triển trí tuệ khác	
F78.0	Chậm phát triển trí tuệ khác, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F78.1	Chậm phát triển trí tuệ khác, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F78.8	Chậm phát triển trí tuệ khác, những khiếm khuyết khác về hành vi	
F78.9	Chậm phát triển trí tuệ khác, không đề cập đến suy giảm hành vi	
F79	Chậm phát triển trí tuệ không xác định	
F79.0	Chậm phát triển trí tuệ không xác định, ghi nhận là hành vi không suy giảm hoặc chỉ ở mức tối thiểu	
F79.1	Chậm phát triển trí tuệ không xác định, suy giảm đáng kể hành vi cần được chú ý hoặc điều trị	
F79.8	Chậm phát triển trí tuệ không xác định, những khiếm khuyết khác về hành vi	
F79.9	Chậm phát triển trí tuệ không xác định, không đề cập đến suy giảm hành vi	
F80	Rối loạn cụ thể về phát triển lời nói và/hoặc ngôn ngữ	
F80.0	Rối loạn kết âm xác định cụ thể	
F80.1	Rối loạn ngôn ngữ diễn đạt	
F80.2	Rối loạn ngôn ngữ tiếp nhận	
F80.3	Chứng thất ngôn [khó khăn khi nói] mắc phải kèm động kinh [Landau-Kleffner]	
F80.8	Rối loạn phát triển về lời nói và/hoặc ngôn ngữ khác	Nói nhịu
F80.9	Rối loạn phát triển về lời nói và ngôn ngữ, không xác định	Rối loạn ngôn ngữ không xác định khác
F81	Rối loạn cụ thể về phát triển kỹ năng học đường	
F81.0	Rối loạn cụ thể về đọc	
F81.1	Rối loạn cụ thể về đánh vần	
F81.2	Rối loạn cụ thể về kỹ năng tính toán	
F81.3	Rối loạn hỗn hợp kỹ năng học đường	
F81.8	Rối loạn khác về sự phát triển kỹ năng học đường	Rối loạn phát triển khả năng viết diễn đạt
F81.9	Rối loạn phát triển của kỹ năng ở trường, không xác định	Rối loạn khả năng tiếp thu kiến thức không xác định khác
F82	Rối loạn cụ thể của phát triển chức năng vận động	
F83	Rối loạn cụ thể hỗn hợp của sự phát triển	
F84	Rối loạn lan tỏa sự phát triển	
F84.0	Rối loạn phổ tự kỷ ở trẻ em	
F84.1	Rối loạn phổ tự kỷ không điển hình	
F84.2	Hội chứng Rett	
F84.3	Rối loạn phân rã khác ở trẻ nhỏ	
F84.4	Rối loạn tăng hoạt động kết hợp với chậm phát triển trí tuệ và/hoặc động tác định hình	
F84.5	Hội chứng Asperger	Bệnh tâm lý tự kỷ|Rối loạn dạng phân liệt ở tuổi trẻ em
F84.8	Rối loạn lan tỏa khác của sự phát triển	
F84.9	Rối loạn phát triển lan tỏa, không xác định	
F88	Rối loạn khác của phát triển tâm lý	
F89	Rối loạn không xác định của phát triển tâm lý	
F90	Rối loạn tăng động	
F90.0	Rối loạn hoạt động và/hoặc chú ý	
F90.1	Rối loạn hành vi tăng động	Rối loạn tăng vận động liên quan đến rối loạn hành vi
F90.8	Rối loạn tăng động khác	
F90.9	Rối loạn tăng động, không xác định	Phản ứng tăng động của thời thơ ấu hoặc thanh thiếu niên không xác định khác|Hội chứng tăng vận động không xác định khác
F91	Rối loạn ứng xử	
F91.0	Rối loạn ứng xử khu trú trong môi trường gia đình	
F91.1	Rối loạn ứng xử ở những người kém thích ứng xã hội	
F91.2	Rối loạn ứng xử xã hội hóa	
F91.3	Rối loạn chống đối	
F91.8	Rối loạn ứng xử khác	
F91.9	Rối loạn ứng xử, không xác định	
F92	Rối loạn hỗn hợp về ứng xử và/hoặc cảm xúc	
F92.0	Rối loạn ứng xử trầm cảm	
F92.8	Rối loạn hỗn hợp của ứng xử và/hoặc cảm xúc khác	
F92.9	Rối loạn hỗn hợp của ứng xử và cảm xúc, không xác định	
F93	Rối loạn cảm xúc có sự khởi phát đặc trưng ở trẻ em	
F93.0	Rối loạn lo âu chia ly ở trẻ em	
F93.1	Rối loạn ám ảnh sợ ở trẻ em	
F93.2	Rối loạn lo âu xã hội ở trẻ em	Rối loạn tránh né ở trẻ em hoặc thanh thiếu niên
F93.3	Rối loạn ganh đua với anh chị em ruột	Ghen với anh chị em ruột
F93.8	Rối loạn cảm xúc khác ở trẻ em	
F93.9	Rối loạn cảm xúc ở trẻ em, không xác định	
F94	Rối loạn hoạt động xã hội khởi phát đặc trưng ở tuổi trẻ em và/hoặc thanh thiếu niên	
F94.0	Chứng mất nói chọn lọc	
F94.1	Rối loạn phản ứng gắn bó ở trẻ em	
F94.2	Rối loạn giao tiếp buông thả ở trẻ em	
F94.8	Rối loạn khác ở trẻ em về hoạt động xã hội	
F94.9	Rối loạn hoạt động xã hội ở trẻ em, không xác định	
F95	Rối loạn tic [tật máy cơ]	
F95.0	Rối loạn tic ngắn hạn	
F95.1	Rối loạn tic chuyển động hoặc phát âm kiên trì	
F95.2	Rối loạn kết hợp tic phát âm và tic chuyển động nhiều loại [hội chứng Tourette]	
F95.8	Rối loạn tic khác	
F95.9	Rối loạn tic, không xác định	Chứng tic không xác định khác
F98	Rối loạn tác phong và/hoặc cảm xúc khác thường khởi phát ở trẻ em và/hoặc thanh thiếu niên	
F98.0	Rối loạn tiểu tiện không thực tổn	
F98.1	Rối loạn đại tiện không thực tổn	
F98.2	Rối loạn ăn uống ở trẻ sơ sinh và/hoặc trẻ nhỏ	
F98.3	Tật ăn bậy ở trẻ sơ sinh và/hoặc trẻ nhỏ	
F98.4	Rối loạn động tác định hình	
F98.5	Tật nói lắp [nói cà lăm]	
F98.6	Chứng nói nhanh, vội vã, ngắt quãng	
F98.8	Rối loạn hành vi cảm xúc xác định khác, thường khởi phát trong tuổi trẻ em và/hoặc thanh thiếu niên	Rối loạn thiếu sót chú ý không tăng hoạt động|Thủ dâm quá mức|Cắn móng tay|Ngoáy lỗ mũi|Mút ngón tay
F98.9	Rối loạn hành vi và/hoặc cảm xúc không xác định, thường khởi phát trong tuổi trẻ em và/hoặc thanh thiếu niên	
F99	Rối loạn tâm thần, không xác định khác	
G00	Bệnh viêm màng não vi khuẩn, không phân loại mục khác	
G00.0	Bệnh viêm màng não do Haemophilus	Bệnh viêm màng não do Haemophilus cúm [H. influenzae]
G00.1	Bệnh viêm màng não do phế cầu	
G00.2	Bệnh viêm màng não do liên cầu	
G00.3	Bệnh viêm màng não do tụ cầu	
G00.8	Bệnh viêm màng não do vi khuẩn khác	
G00.9	Bệnh viêm màng não vi khuẩn, không xác định	
G01.*	Bệnh viêm màng não do bệnh nhiễm khuẩn phân loại mục khác	
G02.*	Bệnh viêm màng não do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
G02.0*	Bệnh viêm màng não do bệnh nhiễm virus phân loại mục khác	
G02.1*	Bệnh viêm màng não do nhiễm nấm	
G02.8*	Bệnh viêm màng não do bệnh nhiễm trùng và/hoặc ký sinh trùng xác định khác phân loại mục khác	
G03	Bệnh viêm màng não do nguyên nhân khác và/hoặc không xác định	
G03.0	Bệnh viêm màng não không sinh mủ	Bệnh viêm màng não không do vi khuẩn
G03.1	Bệnh viêm màng não mạn tính	
G03.2	Bệnh viêm màng não tái diễn lành tính [Mollaret]	
G03.8	Bệnh viêm màng não do nguyên nhân xác định khác	
G03.9	Bệnh viêm màng não, không xác định	
G04	Bệnh viêm não, viêm tủy và/hoặc viêm não - tủy	Bệnh viêm não, viêm tủy và/hoặc viêm não-tủy
G04.0	Bệnh viêm não rải rác cấp tính	
G04.1	Bệnh lý tủy liên quan virus T-lymphotropic ở người (HTLV)	Liệt cứng nửa người [dưới thắt lưng] vùng nhiệt đới
G04.2	Viêm não - màng não và/hoặc viêm tủy - màng tủy do vi khuẩn, không phân loại mục khác	
G04.8	Viêm não, viêm tủy và/hoặc viêm não - tủy khác	
G04.9	Viêm não, viêm tủy và/hoặc viêm não - tủy, không xác định	Bệnh viêm não thất không xác định khác
G05.*	Viêm não, viêm tủy và/hoặc viêm não - tủy do bệnh phân loại mục khác	Viêm não, viêm tủy và/hoặc viêm não-tủy do bệnh phân loại mục khác
G05.0*	Viêm não, viêm tủy và/hoặc viêm não - tủy do bệnh nhiễm khuẩn phân loại mục khác	
G05.1*	Viêm não, viêm tủy và/hoặc viêm não - tủy do bệnh nhiễm virus phân loại mục khác	
G05.2*	Bệnh viêm não, viêm tủy và/hoặc viêm não - tủy do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác	
G05.8*	Viêm não, viêm tủy và/hoặc viêm não - tủy do bệnh khác phân loại mục khác	
G06	Áp xe và/hoặc u hạt nội sọ và/hoặc nội tủy	
G06.0	Áp xe và/hoặc u hạt nội sọ	
G06.1	Áp xe và/hoặc u hạt nội tủy	
G06.2	Áp xe ngoài màng cứng và/hoặc dưới màng cứng, không xác định	
G07.*	Áp xe và/hoặc u hạt nội sọ và/hoặc nội tủy do bệnh phân loại mục khác	
G08	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối nội sọ và/hoặc nội tủy	
G09	Di chứng của bệnh viêm hệ thần kinh trung ương	
G10	Bệnh Huntington	
G11	Mất điều hòa vận động [thất điều] di truyền	
G11.0	Mất điều hòa vận động [thất điều] bẩm sinh không tiến triển	
G11.1	Mất điều hòa vận động [thất điều] tiểu não khởi phát sớm	
G11.2	Mất điều hòa vận động [thất điều] tiểu não khởi phát muộn	Lưu ý: thường khởi phát sau 20 tuổi
G11.3	Mất điều hòa vận động [thất điều] tiểu não liên quan rối loạn sửa chữa DNA	
G11.4	Liệt cứng nửa người di truyền	
G11.8	Mất điều hòa vận động [thất điều] di truyền khác	
G11.9	Mất điều hòa vận động [thất điều] di truyền, không xác định	
G12	Bệnh teo cơ do tổn thương tủy sống và/hoặc hội chứng liên quan	
G12.0	Bệnh teo cơ do tủy ở trẻ nhỏ, típ I [Werdnig-Hofman]	
G12.1	Bệnh teo cơ do tủy sống di truyền khác	
G12.2	Bệnh thân neuron vận động (MND)	Bệnh tế bào thần kinh vận động di truyền|Bệnh Kennedy
G12.8	Bệnh teo cơ tủy khác và/hoặc hội chứng liên quan	
G12.9	Bệnh teo cơ tủy sống, không xác định	
G13.*	Bệnh teo hệ thống ảnh hưởng chủ yếu tới hệ thần kinh trung ương do bệnh phân loại mục khác	
G13.0*	Bệnh lý thần kinh - cơ và/hoặc thần kinh cận u	
G13.1*	Bệnh teo hệ thống khác ảnh hưởng chủ yếu tới hệ thần kinh trung ương do bệnh u tân sinh	
G13.2*	Bệnh teo hệ thống ảnh hưởng chủ yếu tới hệ thần kinh trung ương do bệnh phù niêm (E00.1†, E03.-†)	
G13.8*	Bệnh teo hệ thống ảnh hưởng chủ yếu tới hệ thần kinh trung ương do bệnh khác phân loại mục khác	
G14	Hội chứng sau bại liệt	
G20	Bệnh Parkinson	
G21	Hội chứng Parkinson thứ phát	Hội chứng parkinson thứ phát
G21.0	Hội chứng an thần kinh ác tính	
G21.1	Hội chứng Parkinson thứ phát do thuốc	
G21.2	Hội chứng Parkinson thứ phát do tác nhân bên ngoài	
G21.3	Hội chứng Parkinson sau viêm não	
G21.4	Hội chứng Parkinson do căn nguyên mạch máu	
G21.8	Hội chứng Parkinson thứ phát khác	
G21.9	Hội chứng Parkinson thứ phát, không xác định	
G22.*	Hội chứng Parkinson do bệnh phân loại mục khác	Hội chứng parkinson do bệnh phân loại mục khác
G23	Bệnh thoái hóa khác của hạch nền não	
G23.0	Bệnh Hallervorden-Spatz	Thoái hóa sắc tố cầu nhợt
G23.1	Liệt vận nhãn trên nhân tiến triển [Steele-Richardson-Olszewski]	Liệt trên nhân tiến triển
G23.2	Bệnh teo đa hệ thống, kiểu Parkinson [MSA-P]	
G23.3	Bệnh teo đa hệ thống, kiểu tiểu não [MSA-C]	
G23.8	Bệnh thoái hóa xác định khác của hạch nền	
G23.9	Bệnh thoái hóa hạch nền, không xác định	
G24	Loạn trương lực cơ	
G24.0	Loạn trương lực cơ do thuốc	
G24.1	Loạn trương lực cơ vô căn có yếu tố gia đình	Loạn trương lực cơ vô căn không xác định khác
G24.2	Loạn trương lực cơ vô căn không có yếu tố gia đình	
G24.3	Vẹo cổ do co thắt	
G24.4	Loạn trương lực cơ mặt - miệng vô căn	Rối loạn vận động mặt - miệng
G24.5	Co thắt cơ vòng mi	
G24.8	Loạn trương lực cơ khác	
G24.9	Loạn trương lực cơ, không xác định	Rối loạn vận động không xác định khác
G25	Hội chứng ngoại tháp và/hoặc rối loạn vận động khác	
G25.0	Run vô căn [nguyên phát]	
G25.1	Run do thuốc	
G25.2	Thể run xác định khác	Run có chủ ý
G25.3	Giật cơ	
G25.4	Múa giật do thuốc	
G25.5	Múa giật khác	
G25.6	Rối loạn tic do thuốc và/hoặc rối loạn tic khác do nguyên nhân thực tổn	
G25.8	Hội chứng ngoại tháp và/hoặc rối loạn vận động	
G25.9	Hội chứng ngoại tháp và/hoặc rối loạn vận động, không xác định	
G26.*	Hội chứng ngoại tháp và/hoặc rối loạn vận động do bệnh phân loại mục khác	
G30	Bệnh Alzheimer	
G30.0	Bệnh Alzheimer khởi phát sớm	Lưu ý: thường khởi phát trước tuổi 65
G30.1	Bệnh Alzheimer khởi phát muộn	Lưu ý: thường khởi phát sau tuổi 65
G30.8	Bệnh Alzheimer khác	
G30.9	Bệnh Alzheimer, không xác định	
G31	Bệnh thoái hóa khác của hệ thần kinh, không phân loại mục khác	
G31.0	Bệnh teo não khu trú	
G31.1	Bệnh thoái hóa não tuổi già, không phân loại mục khác	
G31.2	Bệnh thoái hóa hệ thần kinh do rượu	
G31.8	Bệnh thoái hóa xác định khác của hệ thần kinh	
G31.9	Bệnh thoái hóa hệ thần kinh, không xác định	
G32.*	Rối loạn thoái hóa khác của hệ thần kinh do bệnh phân loại mục khác	
G32.0*	Bệnh thoái hóa phối hợp tủy sống bán cấp tính do bệnh phân loại mục khác	
G32.8*	Bệnh thoái hóa xác định khác của hệ thần kinh do bệnh phân loại mục khác	
G35	Bệnh đa xơ cứng	
G36	Bệnh mất myelin rải rác cấp tính khác	
G36.0	Bệnh viêm tủy thị thần kinh [Devic]	
G36.1	Bệnh viêm não chất trắng chảy máu cấp tính và/hoặc bán cấp tính	
G36.8	Bệnh mất myelin rải rác cấp tính xác định khác	
G36.9	Bệnh mất myelin rải rác cấp tính, không xác định	
G37	Bệnh mất myelin khác của hệ thần kinh trung ương	
G37.0	Bệnh xơ cứng lan tỏa	
G37.1	Bệnh mất myelin trung tâm của thể chai	
G37.2	Bệnh mất myelin trung tâm cầu não	
G37.3	Bệnh viêm tủy ngang cấp tính do bệnh mất myelin của hệ thần kinh trung ương	
G37.4	Bệnh viêm tủy hoại tử bán cấp tính	
G37.5	Bệnh xơ cứng đồng tâm [Baló]	
G37.8	Bệnh mất myelin xác định khác của hệ thần kinh trung ương	
G37.9	Bệnh mất myelin của hệ thần kinh trung ương, không xác định	
G40	Bệnh động kinh	
G40.0	Bệnh động kinh cục bộ vô căn (khu trú) (một phần) và/hoặc hội chứng động kinh với cơn co giật khởi phát khu trú	Động kinh trẻ em lành tính kèm các gai nhọn vùng trung tâm - thái dương trên điện não đồ|Động kinh trẻ em kèm kịch phát vùng chẩm trên điện não đồ
G40.1	Bệnh động kinh cục bộ có triệu chứng (khu trú) (một phần) và/hoặc hội chứng động kinh có cơn co giật cục bộ đơn giản	Cơn co giật không kèm biến đổi ý thức|Cơn co giật cục bộ đơn giản phát triển thành cơn toàn thể thứ phát
G40.2	Bệnh động kinh cục bộ có triệu chứng (khu trú) (một phần) và/hoặc hội chứng động kinh có cơn co giật cục bộ phức tạp	Cơn co giật có kèm biến đổi ý thức, thường có động tác tự động|Cơn co giật cục bộ phức hợp phát triển thành cơn toàn thể thứ phát
G40.3	Hội chứng động kinh và/hoặc động kinh toàn thể vô căn	
G40.4	Hội chứng động kinh và/hoặc động kinh toàn thể khác	
G40.5	Hội chứng động kinh đặc biệt	
G40.6	Cơn co giật toàn thể, không xác định (kèm hoặc không kèm cơn vắng ý thức)	
G40.7	Cơn động kinh nhỏ [cơn vắng ý thức], không xác định, không kèm cơn co giật toàn thể	
G40.8	Bệnh động kinh khác	Bệnh động kinh và/hoặc hội chứng động kinh không xác định được là cục bộ hay toàn thể
G40.9	Bệnh động kinh, không xác định	
G41	Trạng thái động kinh	
G41.0	Trạng thái động kinh cơn co giật toàn thể	
G41.1	Trạng thái động kinh cơn vắng ý thức [động kinh cơn nhỏ]	Trạng thái động kinh cơn vắng ý thức
G41.2	Trạng thái động kinh cục bộ phức tạp	
G41.8	Trạng thái động kinh khác	
G41.9	Trạng thái động kinh, không xác định	
G43	Bệnh đau nửa đầu [migraine]	
G43.0	Bệnh đau nửa đầu [migraine] không có tiền triệu [aura] [migraine thường]	
G43.1	Bệnh đau nửa đầu [Migraine] có tiền triệu [aura] [Migraine cổ điển]	
G43.2	Tình trạng đau nửa đầu [migraine] dai dẳng	
G43.3	Bệnh đau nửa đầu [migraine] biến chứng	
G43.8	Bệnh đau nửa đầu [migraine] khác	Bệnh đau nửa đầu [migraine] liệt mắt|Bệnh đau nửa đầu [migraine] võng mạc
G43.9	Bệnh đau nửa đầu [migraine], không xác định	
G44	Hội chứng đau đầu khác	
G44.0	Hội chứng đau đầu từng cụm [chuỗi] [cluster]	Bệnh đau nửa đầu kịch phát mạn tính
G44.1	Đau đầu căn nguyên mạch, không phân loại mục khác	Đau đầu do rối loạn mạch máu không xác định khác
G44.2	Đau đầu do căng thẳng	Đau đầu do căng thẳng mạn tính|Đau đầu do căng thẳng từng đợt|Đau đầu do căng thẳng không xác định khác
G44.3	Đau đầu mạn tính sau chấn thương	
G44.4	Đau đầu do thuốc, không phân loại mục khác	
G44.8	Hội chứng đau đầu xác định khác	
G45	Cơn thiếu máu não bộ thoáng qua và/hoặc hội chứng liên quan	
G45.0	Hội chứng động mạch đốt sống - nền	
G45.1	Hội chứng động mạch cảnh (bán cầu não)	
G45.2	Hội chứng động mạch não trước rải rác hai bên	
G45.3	Mất thị lực thoáng qua	
G45.4	Chứng quên toàn bộ thoáng qua	
G45.8	Cơn thiếu máu não thoáng qua khác và/hoặc hội chứng liên quan	
G45.9	Cơn thiếu máu não thoáng qua, không xác định	Co thắt mạch máu não|Cơn thiếu máu não thoáng qua không xác định khác
G46.*	Hội chứng mạch máu não do bệnh mạch máu não (I60-I67†)	
G46.0*	Hội chứng động mạch não giữa (I66.0†)	
G46.1*	Hội chứng động mạch não trước (I66.1†)	
G46.2*	Hội chứng động mạch não sau (I66.2†)	
G46.3*	Hội chứng đột quỵ thân não (I60-I67†)	
G46.4*	Hội chứng đột quỵ tiểu não (I60-I67†)	
G46.5*	Hội chứng ổ khuyết vận động đơn thuần (I60-I67†)	
G46.6*	Hội chứng ổ khuyết cảm giác đơn thuần (I60-I67†)	
G46.7*	Hội chứng ổ khuyết khác (I60-I67†)	
G46.8*	Hội chứng mạch máu não khác do bệnh mạch máu não (I60-I67†)	
G47	Rối loạn giấc ngủ	
G47.0	Rối loạn vào giấc và/hoặc rối loạn duy trì giấc ngủ [chứng mất ngủ]	
G47.1	Rối loạn buồn ngủ quá mức [hội chứng ngủ nhiều]	
G47.2	Rối loạn lịch trình thức - ngủ	Hội chứng giai đoạn giấc ngủ bị trì hoãn|Kiểu thức - ngủ thất thường
G47.3	Ngưng thở khi ngủ	
G47.4	Chứng ngủ rũ và/hoặc chứng mất trương lực	
G47.8	Rối loạn giấc ngủ khác	Hội chứng Kleine-Levin
G47.9	Rối loạn giấc ngủ, không xác định	
G50	Rối loạn dây thần kinh sinh ba [tam thoa] [V]	
G50.0	Đau dây thần kinh sinh ba [tam thoa] [V]	Hội chứng đau mặt kịch phát|Đau giật mặt
G50.1	Đau mặt không điển hình	
G50.8	Rối loạn khác của dây thần kinh sinh ba [tam thoa] [V]	
G50.9	Rối loạn dây thần kinh sinh ba [tam thoa] [V], không xác định	
G51	Rối loạn dây thần kinh mặt	
G51.0	Liệt Bell	
G51.1	Viêm hạch gối	
G51.2	Hội chứng Melkersson	Hội chứng Melkersson-Rosenthal
G51.3	Co thắt và/hoặc giật nửa mặt	
G51.4	Chứng co cứng cơ mặt	
G51.8	Rối loạn khác của dây thần kinh mặt	
G51.9	Rối loạn dây thần kinh mặt, không xác định	
G52	Rối loạn dây thần kinh sọ khác	
G52.0	Rối loạn dây thần kinh khứu giác	Bệnh dây thần kinh thứ 1
G52.1	Rối loạn dây thần kinh lưỡi hầu [thiệt hầu]	Rối loạn dây thần kinh thứ 9|Bệnh đau dây thần kinh lưỡi hầu
G52.2	Rối loạn dây thần kinh phế vị	Rối loạn dây phế vị [dây thần kinh thứ 10]
G52.3	Rối loạn dây thần kinh dưới lưỡi [hạ thiệt]	Rối loạn dây thần kinh thứ 12
G52.7	Rối loạn đa dây thần kinh sọ	Bệnh viêm nhiều dây thần kinh sọ
G52.8	Rối loạn dây thần kinh sọ xác định khác	
G52.9	Rối loạn dây thần kinh sọ, không xác định	
G53.*	Rối loạn dây thần kinh sọ do bệnh phân loại mục khác	
G53.0*	Đau dây thần kinh sau zona (B02.2†)	
G53.1*	Liệt nhiều dây thần kinh sọ do bệnh nhiễm trùng và/hoặc nhiễm ký sinh trùng phân loại mục khác (A00-B99†)	
G53.2*	Liệt nhiều dây thần kinh sọ do bệnh u hạt (D86.8†)	
G53.3*	Liệt nhiều dây thần kinh sọ do u tân sinh (C00.- - D48.-†)	
G53.8*	Rối loạn dây thần kinh sọ khác do bệnh khác phân loại mục khác	
G54	Rối loạn rễ thần kinh và/hoặc đám rối thần kinh	
G54.0	Rối loạn đám rối thần kinh cánh tay	Hội chứng cơ bậc thang
G54.1	Rối loạn đám rối thắt lưng - cùng	
G54.2	Rối loạn rễ thần kinh cổ, không phân loại mục khác	
G54.3	Rối loạn rễ thần kinh ngực, không phân loại mục khác	
G54.4	Rối loạn rễ thần kinh thắt lưng - cùng, không phân loại mục khác	
G54.5	Bệnh teo cơ đau thần kinh	Hội chứng Parsonage-Aldren-Turner|Bệnh viêm dây thần kinh đai vai
G54.6	Hội chứng chi ma có kèm đau	
G54.7	Hội chứng chi ma không kèm đau	Hội chứng chi ma không xác định khác
G54.8	Rối loạn rễ và/hoặc đám rối thần kinh khác	
G54.9	Rối loạn rễ và/hoặc đám rối thần kinh, không xác định	
G55.*	Chèn ép rễ thần kinh và/hoặc đám rối do bệnh phân loại mục khác	
G55.0*	Chèn ép rễ thần kinh và/hoặc đám rối do bệnh u tân sinh (C00.- - D48.-†)	
G55.1*	Chèn ép rễ và/hoặc đám rối thần kinh do rối loạn đĩa đệm cột sống (M50-M51†)	
G55.2*	Chèn ép rễ và/hoặc đám rối thần kinh do thoái hóa đốt sống (M47.-†)	
G55.3*	Chèn ép rễ và/hoặc đám rối thần kinh do bệnh lý cột sống khác (M45-M46†, M48.-†, M53-M54†)	
G55.8*	Chèn ép rễ và/hoặc đám rối thần kinh do bệnh khác phân loại mục khác	
G56	Bệnh lý đơn dây thần kinh chi trên	
G56.0	Hội chứng ống cổ tay	
G56.1	Tổn thương khác của dây thần kinh giữa	
G56.2	Tổn thương dây thần kinh trụ	Liệt thần kinh trụ giai đoạn muộn
G56.3	Tổn thương dây thần kinh quay	
G56.8	Bệnh lý đơn dây thần kinh khác của chi trên	U dây thần kinh gian ngón [kẽ ngón] chi trên
G56.9	Bệnh lý đơn dây thần kinh của chi trên, không xác định	
G57	Bệnh lý đơn dây thần kinh của chi dưới	
G57.0	Tổn thương dây thần kinh hông to [dây thần kinh tọa]	
G57.1	Chứng đau đùi dị cảm	Hội chứng dây thần kinh da - đùi ngoài
G57.2	Tổn thương dây thần kinh đùi	
G57.3	Tổn thương dây thần kinh khoeo ngoài	Liệt dây thần kinh mác
G57.4	Tổn thương dây thần kinh khoeo trong	
G57.5	Hội chứng ống cổ chân	
G57.6	Tổn thương dây thần kinh gan bàn chân	Đau đốt bàn ngón chân Morton
G57.8	Bệnh lý đơn dây thần kinh khác của chi dưới	U dây thần kinh gian ngón [kẽ ngón] chi dưới
G57.9	Bệnh lý đơn dây thần kinh của chi dưới, không xác định	
G58	Bệnh lý đơn dây thần kinh khác	
G58.0	Bệnh lý dây thần kinh liên sườn	
G58.7	Bệnh viêm đơn dây thần kinh nhiều ổ	
G58.8	Bệnh lý đơn dây thần kinh xác định khác	
G58.9	Bệnh lý đơn dây thần kinh, không xác định	
G59.*	Bệnh lý đơn dây thần kinh do bệnh phân loại mục khác	
G59.0*	Bệnh lý đơn dây thần kinh do đái tháo đường (E10-E14†) (với ký tự thứ tư chung là .4†)	
G59.8*	Bệnh lý đơn dây thần kinh khác do bệnh phân loại mục khác	
G60	Bệnh lý dây thần kinh di truyền và/hoặc vô căn	
G60.0	Bệnh lý dây thần kinh cảm giác và/hoặc vận động di truyền	
G60.1	Bệnh Refsum	
G60.2	Bệnh lý dây thần kinh kết hợp với mất điều hòa vận động [thất điều] di truyền	
G60.3	Bệnh lý dây thần kinh tiến triển vô căn	
G60.8	Bệnh lý dây thần kinh vô căn và/hoặc di truyền khác	Bệnh Morvan|Hội chứng Nelaton
G60.9	Bệnh lý dây thần kinh di truyền và/hoặc vô căn, không xác định	
G61	Bệnh lý viêm đa dây thần kinh	
G61.0	Hội chứng Guillain-Barré	
G61.1	Bệnh lý dây thần kinh do huyết thanh	
G61.8	Bệnh lý viêm đa dây thần kinh khác	
G61.9	Bệnh lý viêm đa dây thần kinh, không xác định	
G62	Bệnh lý đa dây thần kinh khác	
G62.0	Bệnh lý đa dây thần kinh do thuốc	
G62.1	Bệnh lý đa dây thần kinh do rượu	
G62.2	Bệnh lý đa dây thần kinh do tác nhân gây độc khác	
G62.8	Bệnh lý đa dây thần kinh xác định khác	
G62.9	Bệnh lý đa dây thần kinh, không xác định	Bệnh dây thần kinh không xác định khác
G63.*	Bệnh lý đa dây thần kinh do bệnh phân loại mục khác	
G63.0*	Bệnh lý đa dây thần kinh do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
G63.1*	Bệnh lý đa dây thần kinh do u tân sinh (C00.- - D48.-†)	
G63.2*	Bệnh lý đa dây thần kinh do đái tháo đường (E10-E14 với ký tự thứ tự chung là.4†)	
G63.3*	Bệnh lý đa dây thần kinh do bệnh nội tiết và/hoặc chuyển hóa khác (E00-E07†, E15-E16†, E20-E34†, E70-E89†)	
G63.4*	Bệnh lý đa dây thần kinh do suy dinh dưỡng (E40-E64†)	
G63.5*	Bệnh lý đa dây thần kinh do rối loạn mô liên kết hệ thống (M30-M35†)	
G63.6*	Bệnh lý đa dây thần kinh do rối loạn cơ xương khớp khác (M00-M25†, M40-M96†)	
G63.8*	Bệnh lý đa dây thần kinh do bệnh khác phân loại mục khác	
G64	Rối loạn khác của hệ thần kinh ngoại biên	
G70	Nhược cơ và/hoặc rối loạn thần kinh - cơ khác	Nhược cơ và/hoặc rối loạn thần kinh-cơ khác
G70.0	Nhược cơ [rối loạn thần kinh - cơ tự miễn]	
G70.1	Bệnh thần kinh - cơ do nhiễm độc	
G70.2	Bệnh nhược cơ bẩm sinh và/hoặc trong quá trình phát triển	
G70.8	Bệnh thần kinh - cơ xác định khác	
G70.9	Bệnh thần kinh - cơ, không xác định	
G71	Bệnh cơ nguyên phát	
G71.0	Bệnh loạn dưỡng cơ	
G71.1	Rối loạn trương lực cơ	
G71.2	Bệnh lý cơ bẩm sinh	
G71.3	Bệnh lý cơ do ty lạp thể, không phân loại mục khác	
G71.8	Rối loạn nguyên phát khác của cơ	
G71.9	Rối loạn cơ nguyên phát, không xác định	Bệnh cơ di truyền không xác định khác
G72	Bệnh lý cơ khác	
G72.0	Bệnh lý cơ do thuốc	
G72.1	Bệnh lý cơ do rượu	
G72.2	Bệnh lý cơ do tác nhân gây độc khác	
G72.3	Liệt chu kỳ	
G72.4	Bệnh lý viêm cơ, không phân loại mục khác	
G72.8	Bệnh lý cơ xác định khác	
G72.9	Bệnh lý cơ, không xác định	
G73.*	Bệnh khớp thần kinh - cơ và/hoặc cơ do bệnh phân loại mục khác	Bệnh khớp thần kinh-cơ và/hoặc cơ do bệnh phân loại mục khác
G73.0*	Hội chứng nhược cơ do bệnh nội tiết	
G73.1*	Hội chứng Lambert-Eaton (C00.- - D48.-†)	
G73.2*	Hội chứng nhược cơ khác do bệnh u tân sinh (C00.- - D48.-†)	
G73.3*	Hội chứng nhược cơ do bệnh khác phân loại mục khác	
G73.4*	Bệnh lý cơ do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
G73.5*	Bệnh lý cơ do bệnh nội tiết	
G73.6*	Bệnh lý cơ do bệnh chuyển hóa	
G73.7*	Bệnh lý cơ do bệnh khác, phân loại mục khác	
G80	Bệnh bại não	
G80.0	Bệnh bại não liệt cứng tứ chi	Bệnh não liệt cứng tứ chi
G80.1	Bệnh bại não liệt cứng hai chi dưới hoặc hai chi trên	
G80.2	Bệnh bại não liệt cứng nửa người	
G80.3	Bệnh bại não thể múa vờn	Bệnh bại não loạn trương lực
G80.4	Bệnh bại não thể thất điều	
G80.8	Bệnh bại não thể khác	Hội chứng bại não hỗn hợp
G80.9	Bệnh bại não, không xác định	Bại não không xác định khác
G81	Hội chứng liệt nửa người [liệt một bên người]	
G81.0	Liệt mềm nửa người [liệt một bên người]	
G81.1	Liệt cứng nửa người [liệt một bên người]	
G81.9	Hội chứng liệt nửa người [liệt một bên người], không xác định	
G82	Hội chứng liệt nửa người [dưới thắt lưng] và/hoặc liệt tứ chi	
G82.0	Hội chứng liệt mềm nửa người [dưới thắt lưng]	
G82.1	Hội chứng liệt cứng nửa người [dưới thắt lưng]	
G82.2	Hội chứng liệt nửa người [dưới thắt lưng], không xác định	
G82.3	Liệt mềm tứ chi	
G82.4	Liệt cứng tứ chi	
G82.5	Liệt tứ chi, không xác định	Liệt tứ chi không xác định khác
G83	Hội chứng liệt khác	
G83.0	Liệt hai chi trên	
G83.1	Liệt một chi dưới	
G83.2	Liệt một chi trên	
G83.3	Liệt một chi, không xác định	
G83.4	Hội chứng chùm đuôi ngựa	
G83.5	Hội chứng khóa trong	
G83.6	Liệt dây VII trung ương	
G83.8	Hội chứng liệt xác định khác	
G83.9	Hội chứng liệt, không xác định	
G90	Rối loạn hệ thần kinh tự động	
G90.0	Bệnh lý thần kinh tự động ngoại biên vô căn	Ngất do xoang cảnh
G90.1	Rối loạn thần kinh tự động có yếu tố gia đình [Riley-Day]	
G90.2	Hội chứng Horner	
G90.4	Rối loạn phản xạ tự động	
G90.5	Hội chứng đau phức hợp vùng típ I	Loạn dưỡng giao cảm phản xạ [Sudeck atrophy]|Hội chứng loạn dưỡng giao cảm phản xạ
G90.6	Hội chứng đau phức hợp vùng típ II	Hội chứng Causalgia [đau và nóng rát] [chứng hỏa thống]
G90.7	Hội chứng đau phức hợp vùng, loại khác và/hoặc không xác định	
G90.8	Rối loạn khác của hệ thần kinh tự động	
G90.9	Rối loạn hệ thần kinh tự động, không xác định	
G91	Bệnh não úng thủy	
G91.0	Bệnh não úng thủy thể thông	
G91.1	Bệnh não úng thủy thể tắc nghẽn	
G91.2	Bệnh não úng thủy áp lực bình thường	
G91.3	Bệnh não úng thủy sau chấn thương, không xác định	
G91.8	Bệnh não úng thủy khác	
G91.9	Bệnh não úng thủy, không xác định	
G92	Bệnh lý não nhiễm độc	
G93	Rối loạn khác của não	
G93.0	Bệnh u nang não	
G93.1	Tổn thương não do thiếu oxy, không phân loại mục khác	
G93.2	Tăng áp lực nội sọ lành tính	
G93.3	Hội chứng mệt mỏi sau nhiễm virus	Bệnh viêm não tủy đau cơ
G93.4	Bệnh lý não, không xác định	
G93.5	Chèn ép não	
G93.6	Phù não	
G93.7	Hội chứng Reye	
G93.8	Rối loạn xác định khác của não	
G93.9	Bệnh não, không xác định	
G94.*	Rối loạn khác của não do bệnh phân loại mục khác	
G94.0*	Bệnh não úng thủy do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác (A00-B99†)	
G94.1*	Bệnh não úng thủy do bệnh u tân sinh (C00.- - D48.-†)	
G94.2*	Bệnh não úng thủy do bệnh phân loại mục khác	
G94.3*	Bệnh lý não do bệnh phân loại mục khác	
G94.8*	Rối loạn xác định khác của não do bệnh phân loại mục khác	
G95	Bệnh khác của tủy sống	
G95.0	Bệnh rỗng tủy sống và/hoặc rỗng hành não	
G95.1	Bệnh lý mạch máu tủy	
G95.2	Bệnh chèn ép tủy, không xác định	
G95.8	Bệnh xác định khác của tủy sống	
G95.9	Bệnh của tủy sống, không xác định	Bệnh tủy không xác định khác
G96	Rối loạn khác của hệ thần kinh trung ương	
G96.0	Rò dịch não tủy	
G96.1	Rối loạn màng não, không phân loại mục khác	
G96.8	Rối loạn xác định khác của hệ thần kinh trung ương	
G96.9	Rối loạn hệ thần kinh trung ương, không xác định	
G97	Rối loạn hệ thần kinh sau can thiệp, không phân loại mục khác	
G97.0	Rò dịch não tủy do chọc dò tủy sống	
G97.1	Phản ứng khác đối với chọc dò tủy sống và/hoặc chọc ống sống thắt lưng	
G97.2	Hạ áp lực nội sọ sau dẫn lưu não thất	
G97.8	Rối loạn khác của hệ thần kinh sau can thiệp	
G97.9	Rối loạn hệ thần kinh sau can thiệp, không xác định	
G98	Rối loạn khác của hệ thần kinh, không phân loại mục khác	
G99.*	Rối loạn khác của hệ thần kinh do bệnh phân loại mục khác	
G99.0*	Bệnh lý hệ thần kinh tự động do bệnh nội tiết và/hoặc chuyển hóa	
G99.1*	Rối loạn khác của hệ thần kinh tự động do bệnh phân loại mục khác	
G99.2*	Bệnh lý tủy do bệnh phân loại mục khác	
G99.8*	Rối loạn xác định khác của hệ thần kinh do bệnh phân loại mục khác	
H00	Lẹo và/hoặc chắp	
H00.0	Lẹo và/hoặc viêm sâu khác của mi mắt	Áp xe mí mắt|Nhọt mí mắt|Lẹo mí mắt
H00.1	Chắp	
H01	Viêm khác của mi mắt	
H01.0	Viêm bờ mi	
H01.1	Bệnh da mi mắt không nhiễm trùng	
H01.8	Viêm mi mắt xác định khác	
H01.9	Viêm mi mắt, không xác định	
H02	Rối loạn khác của mi mắt	
H02.0	Quặm mi và/hoặc lông xiêu của mi mắt	
H02.1	Lật mi	
H02.2	Hở mi	
H02.3	Sa da mi	
H02.4	Sụp mi	
H02.5	Rối loạn khác tác động đến chức năng của mi mắt	
H02.6	U vàng mi mắt	
H02.7	Rối loạn thoái hóa khác của mi mắt và/hoặc vùng quanh mắt	Rám da của mi mắt|Rụng lông mi của mi mắt|Bạch biến của mi mắt
H02.8	Rối loạn xác định khác của mi mắt	Bệnh rậm lông mi|Dị vật mi mắt
H02.9	Rối loạn mi mắt, không xác định	
H03.*	Rối loạn mi mắt do bệnh phân loại mục khác	
H03.0*	Nhiễm ký sinh trùng ở mi mắt do bệnh phân loại mục khác	
H03.1*	Tổn thương mi mắt do bệnh nhiễm trùng phân loại mục khác	
H03.8*	Tổn thương mi mắt do bệnh khác phân loại mục khác	
H04	Rối loạn hệ thống lệ	
H04.0	Viêm tuyến lệ	Phì đại tuyến lệ mạn tính
H04.1	Rối loạn khác của tuyến lệ	U nang tuyến lệ [u nang ống dẫn]|Hội chứng khô mắt
H04.2	Chảy nước mắt sống	
H04.3	Viêm lệ đạo cấp tính và/hoặc không xác định	
H04.4	Viêm lệ đạo mạn tính	Viêm túi lệ mạn tính|Viêm lệ quản mạn tính|U nhầy lệ quản mạn tính
H04.5	Hẹp và/hoặc bán tắc lệ đạo	Sỏi lệ đạo|Lật điểm lệ
H04.6	Thay đổi khác trong lệ đạo	Rò lệ đạo
H04.8	Rối loạn khác của hệ thống lệ	
H04.9	Rối loạn hệ thống lệ, không xác định	
H05	Rối loạn hốc mắt	
H05.0	Viêm hốc mắt cấp tính	Áp xe hốc mắt|Viêm mô tế bào hốc mắt|Viêm xương hốc mắt|Viêm màng xương hốc mắt|Viêm bao gân
H05.1	Rối loạn viêm hốc mắt mạn tính	U hạt của hốc mắt
H05.2	Bệnh lý lồi mắt	
H05.3	Biến dạng hốc mắt	Teo hốc mắt|Lồi xương hốc mắt
H05.4	Lõm mắt	
H05.5	Dị vật (cũ) sau chấn thương xuyên hốc mắt	Dị vật hậu nhãn cầu
H05.8	Rối loạn khác của hốc mắt	Nang hốc mắt
H05.9	Rối loạn hốc mắt, không xác định	
H06.*	Rối loạn hệ thống lệ và/hoặc hốc mắt do bệnh phân loại mục khác	
H06.0*	Rối loạn hệ thống lệ do bệnh phân loại mục khác	
H06.1*	Nhiễm ký sinh trùng của hốc mắt do bệnh phân loại mục khác	
H06.2*	Lồi mắt do tuyến giáp (E05.-†)	
H06.3*	Rối loạn khác của hốc mắt do bệnh phân loại mục khác	
H10	Viêm kết mạc	
H10.0	Viêm kết mạc nhầy mủ	
H10.1	Viêm kết mạc dị ứng cấp tính	
H10.2	Viêm kết mạc cấp tính khác	
H10.3	Viêm kết mạc cấp tính, không xác định	
H10.4	Viêm kết mạc mạn tính	
H10.5	Viêm kết mạc mi mắt	
H10.8	Viêm kết mạc khác	
H10.9	Viêm kết mạc, không xác định	
H11	Rối loạn khác của kết mạc	
H11.0	Mộng thịt	
H11.1	Lắng đọng và/hoặc thoái hóa kết mạc	
H11.2	Sẹo kết mạc	Dính mi cầu
H11.3	Xuất huyết kết mạc	Xuất huyết dưới kết mạc
H11.4	Rối loạn mạch máu kết mạch khác và/hoặc nang kết mạc	
H11.8	Rối loạn xác định khác của kết mạc	Mộng thịt giả
H11.9	Rối loạn kết mạc, không xác định	
H13.*	Rối loạn kết mạc do bệnh phân loại mục khác	
H13.0*	Nhiễm giun chỉ ở kết mạc (B74.-†)	
H13.1*	Viêm kết mạc do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
H13.2*	Bệnh viêm kết mạc do bệnh khác phân loại mục khác	
H13.3*	Pemphigoid ở mắt (L12.-†)	
H13.8*	Rối loạn khác của kết mạc do bệnh phân loại mục khác	
H15	Rối loạn của củng mạc	
H15.0	Viêm củng mạc	
H15.1	Viêm thượng củng mạc	
H15.8	Rối loạn khác của củng mạc	
H15.9	Rối loạn củng mạc, không xác định	
H16	Viêm giác mạc	
H16.0	Loét giác mạc	
H16.1	Viêm giác mạc nông khác không có viêm kết mạc	
H16.2	Viêm kết giác mạc	
H16.3	Viêm giác mạc sâu và/hoặc viêm giác mạc kẽ	
H16.4	Tân mạch giác mạc	
H16.8	Viêm giác mạc khác	
H16.9	Viêm giác mạc, không xác định	
H17	Sẹo và/hoặc đục giác mạc	
H17.0	Sẹo dính giác mạc	
H17.1	Đục giác mạc trung tâm khác	
H17.8	Sẹo và/hoặc đục giác mạc khác	
H17.9	Sẹo và/hoặc đục giác mạc, không xác định	
H18	Rối loạn khác của giác mạc	
H18.0	Nhiễm sắc tố và/hoặc lắng đọng ở giác mạc	
H18.1	Bệnh lý giác mạc bọng	
H18.2	Phù giác mạc khác	
H18.3	Thay đổi ở màng giác mạc	Nếp gấp ở màng Descement|Rách màng Descement
H18.4	Thoái hóa giác mạc	
H18.5	Loạn dưỡng giác mạc di truyền	
H18.6	Bệnh giác mạc chóp	
H18.7	Biến dạng giác mạc khác	
H18.8	Rối loạn xác định khác của giác mạc	Mất cảm giác của giác mạc|Giảm cảm giác của giác mạc|Tróc biểu mô tái phát của giác mạc
H18.9	Rối loạn giác mạc, không xác định	
H19.*	Rối loạn củng mạc và/hoặc giác mạc do bệnh phân loại mục khác	
H19.0*	Viêm củng mạc và/hoặc thượng củng mạc do bệnh phân loại mục khác	
H19.1*	Viêm giác mạc và/hoặc kết giác mạc do virus herpes [herpes simplex] (B00.5†)	Bệnh viêm giác mạc hình đĩa và/hoặc dạng đuôi gai
H19.2*	Viêm giác mạc và/hoặc kết giác mạc do bệnh nhiễm trùng và/hoặc nhiễm ký sinh trùng khác phân loại mục khác	
H19.3*	Viêm giác mạc và/hoặc viêm kết giác mạc do bệnh khác phân loại mục khác	
H19.8*	Rối loạn khác của củng mạc và/hoặc giác mạc do bệnh phân loại mục khác	
H20	Bệnh viêm mống mắt thể mi	Viêm mống mắt thể mi
H20.0	Bệnh viêm mống mắt thể mi cấp tính và/hoặc bán cấp tính	Bệnh viêm màng bồ đào trước cấp tính, tái phát hoặc bán cấp tính|Bệnh viêm thể mi cấp tính, tái phát hoặc bán cấp tính|Bệnh viêm mống mắt cấp tính, tái phát hoặc bán cấp tính
H20.1	Bệnh viêm mống mắt thể mi mạn tính	
H20.2	Bệnh viêm mống mắt thể mi do thể thủy tinh	
H20.8	Bệnh viêm mống mắt thể mi khác	
H20.9	Bệnh viêm mống mắt thể mi, không xác định	
H21	Rối loạn khác của mống mắt và/hoặc thể mi	
H21.0	Xuất huyết tiền phòng	
H21.1	Rối loạn mạch máu khác của mống mắt và/hoặc thể mi	Tân mạch mống mắt và/hoặc thể mi|Mống mắt đỏ
H21.2	Thoái hóa mống mắt và/hoặc thể mi	
H21.3	Nang mống mắt, thể mi và/hoặc tiền phòng	
H21.4	Màng đồng từ	Đứt chân mống mắt
H21.5	Dính và/hoặc rách khác của mống mắt và/hoặc thể mi	
H21.8	Rối loạn xác định khác của mống mắt và/hoặc thể mi	
H21.9	Rối loạn của mống mắt và/hoặc thể mi, không xác định	
H22.*	Rối loạn của mống mắt và/hoặc thể mi do bệnh phân loại mục khác	
H22.0*	Viêm mống mắt thể mi do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
H22.1*	Viêm mống mắt thể mi do bệnh khác phân loại mục khác	
H22.8*	Rối loạn khác của mống mắt và/hoặc thể mi do bệnh phân loại mục khác	
H25	Đục thể thủy tinh tuổi già	
H25.0	Đục thể thủy tinh nguyên phát tuổi già	
H25.1	Đục nhân thủy tinh thể tuổi già	Đục thủy tinh thể có màu nâu|đục thể thủy tinh nhân trung tâm xơ cứng
H25.2	Đục thể thủy tinh, loại hình Morgagni	Đục thủy tinh thể do tuổi già
H25.8	Đục thể thủy tinh tuổi già khác	Dạng phối hợp của đục thủy tinh thể tuổi già
H25.9	Đục thể thủy tinh tuổi già, không xác định	
H26	Đục thể thủy tinh khác	
H26.0	Đục thể thủy tinh ở trẻ nhỏ, người trẻ và/hoặc trước tuổi già	
H26.1	Đục thể thủy tinh do chấn thương	
H26.2	Đục thể thủy tinh biến chứng	
H26.3	Đục thể thủy tinh do thuốc	
H26.4	Đục bao sau thể thủy tinh	Đục thủy tinh thể thứ phát|Vòng Soemmerring
H26.8	Đục thể thủy tinh xác định khác	
H26.9	Đục thể thủy tinh, không xác định	
H27	Rối loạn khác của thể thủy tinh	
H27.0	Không có thể thủy tinh	
H27.1	Lệch thể thủy tinh	
H27.8	Rối loạn thể thủy tinh xác định khác	
H27.9	Rối loạn thể thủy tinh, không xác định	
H28.*	Đục thể thủy tinh và/hoặc rối loạn khác của thể thủy tinh do bệnh phân loại mục khác	
H28.0*	Đục thể thủy tinh do đái tháo đường (E10-E14 với ký tự thứ tư chung là .3†)	
H28.1*	Đục thể thủy tinh do bệnh nội tiết, dinh dưỡng và/hoặc chuyển hóa khác	
H28.2*	Đục thể thủy tinh do bệnh khác phân loại mục khác	
H28.8*	Rối loạn khác của thể thủy tinh do bệnh phân loại mục khác	
H30	Viêm hắc võng mạc	
H30.0	Viêm hắc võng mạc khu trú	
H30.1	Viêm hắc [màng mạch] võng mạc lan tỏa	
H30.2	Viêm thể mi sau	Viêm vùng pars plana
H30.8	Viêm hắc võng mạc khác	Bệnh Harada
H30.9	Viêm hắc võng mạc, không xác định	Viêm hắc võng mạc không xác định khác|Viêm hắc mạc không xác định khác|Viêm võng mạc không xác định khác|Viêm võng hắc mạc không xác định khác
H31	Rối loạn khác của hắc mạc [màng mạch]	
H31.0	Sẹo hắc võng mạc	
H31.1	Thoái hóa hắc mạc	
H31.2	Loạn dưỡng hắc mạc di truyền	
H31.3	Xuất huyết và/hoặc rách hắc mạc	
H31.4	Bong hắc mạc	
H31.8	Rối loạn xác định khác của hắc mạc [màng mạch]	Tân mạch hắc mạc
H31.9	Rối loạn hắc mạc [màng mạch], không xác định	
H32.*	Rối loạn hắc [màng mạch] võng mạc do bệnh phân loại mục khác	
H32.0*	Viêm hắc [màng mạch] - võng mạc do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
H32.8*	Rối loạn khác của hắc [màng mạch] võng mạc do bệnh phân loại mục khác	
H33	Bong và/hoặc rách võng mạc	
H33.0	Bong võng mạc có vết rách	
H33.1	Tách lớp võng mạc và/hoặc nang võng mạc	
H33.2	Bong võng mạc thanh dịch	
H33.3	Rách võng mạc không bong	
H33.4	Bong võng mạc co kéo	Bệnh dịch kính võng mạc tăng sinh kèm bong võng mạc
H33.5	Bong võng mạc khác	
H34	Tắc mạch máu võng mạc	
H34.0	Tắc động mạch võng mạc thoáng qua	
H34.1	Tắc động mạch trung tâm võng mạc	
H34.2	Tắc động mạch võng mạc khác	Mảng bám Hollenhost
H34.8	Tắc mạch máu võng mạc khác	
H34.9	Tắc mạch máu võng mạc, không xác định	
H35	Rối loạn võng mạc khác	
H35.0	Bệnh lý võng mạc nền và/hoặc thay đổi mạch máu võng mạc	Thay đổi hình dạng mạch máu võng mạc
H35.1	Bệnh lý võng mạc ở trẻ sinh non	Xơ hóa võng mạc
H35.2	Bệnh lý võng mạc tăng sinh khác	
H35.3	Thoái hóa hoàng điểm và/hoặc cực sau	
H35.4	Thoái hóa võng mạc ngoại biên	
H35.5	Loạn dưỡng võng mạc di truyền	
H35.6	Xuất huyết võng mạc	
H35.7	Bong các lớp võng mạc	Bệnh hắc võng mạc trung tâm thanh dịch|Bong biểu mô sắc tố võng mạc
H35.8	Rối loạn võng mạc xác định khác	
H35.9	Rối loạn võng mạc, không xác định	
H36.*	Rối loạn võng mạc do bệnh phân loại mục khác	
H36.0*	Bệnh lý võng mạc đái tháo đường (E10-E14 với ký tự thứ tư chung là .3†)	
H36.8*	Rối loạn võng mạc khác do bệnh phân loại mục khác	
H40	Glôcôm	
H40.0	Nghi ngờ Glôcôm	Tăng nhãn áp
H40.1	Glôcôm góc mở nguyên phát	
H40.2	Glôcôm góc đóng nguyên phát	
H40.3	Glôcôm thứ phát do chấn thương mắt	
H40.4	Glôcôm thứ phát do viêm mắt	
H40.5	Glôcôm thứ phát do rối loạn mắt khác	
H40.6	Glôcôm thứ phát do thuốc	
H40.8	Glôcôm khác	
H40.9	Glôcôm, không xác định	
H42.*	Glôcôm do bệnh phân loại mục khác	
H42.0*	Glôcôm do bệnh nội tiết, dinh dưỡng và/hoặc chuyển hóa	
H42.8*	Glôcôm do bệnh khác phân loại mục khác	
H43	Rối loạn dịch kính	
H43.0	Phòi dịch kính	
H43.1	Xuất huyết dịch kính	
H43.2	Lắng đọng tinh thể trong dịch kính	
H43.3	Vẩn đục dịch kính khác	Màng và dải sợi dịch kính
H43.8	Rối loạn khác của dịch kính	
H43.9	Rối loạn dịch kính, không xác định	
H44	Rối loạn nhãn cầu	
H44.0	Viêm mủ nội nhãn	Viêm toàn nhãn|Áp xe dịch kính
H44.1	Viêm nội nhãn khác	Viêm nội nhãn do ký sinh trùng không xác định khác|Nhãn viêm giao cảm
H44.2	Cận thị thoái hóa	
H44.3	Rối loạn thoái hóa khác của nhãn cầu	Nhiễm đồng|Nhiễm sắt
H44.4	Hạ nhãn áp	
H44.5	Bệnh thoái hóa của nhãn cầu	Glôcôm tuyệt đối|Teo nhãn cầu|Tụt nhãn cầu
H44.6	Dị vật nội nhãn, có từ tính	
H44.7	Dị vật nội nhãn, không từ tính	
H44.8	Rối loạn khác của nhãn cầu	Xuất huyết nhãn cầu|Lệch nhãn cầu
H44.9	Rối loạn nhãn cầu, không xác định	
H45.*	Rối loạn dịch kính và/hoặc nhãn cầu do bệnh phân loại mục khác	
H45.0*	Xuất huyết dịch kính do bệnh phân loại mục khác	
H45.1*	Viêm nội nhãn do bệnh phân loại mục khác	
H45.8*	Rối loạn khác của dịch kính và/hoặc nhãn cầu do bệnh phân loại mục khác	
H46	Viêm thần kinh thị giác	
H47	Rối loạn khác của thần kinh thị giác [II] và/hoặc đường thị giác	
H47.0	Rối loạn thần kinh thị giác, không phân loại mục khác	
H47.1	Phù gai thị, không xác định	
H47.2	Teo thị thần kinh	Bạc màu đĩa thị vùng thái dương
H47.3	Rối loạn khác của đĩa thị	Drusen đĩa thị|Giả phù gai
H47.4	Rối loạn giao thoa thị giác	
H47.5	Rối loạn đường thị giác khác	Rối loạn dải thị giác, nhân thể gối ngoài và/hoặc tia thị giác
H47.6	Rối loạn vỏ não thị giác	
H47.7	Rối loạn đường thị giác, không xác định	
H48.*	Rối loạn thần kinh thị giác [II] và/hoặc đường thị giác do bệnh phân loại mục khác	
H48.0*	Teo thần kinh thị giác do bệnh phân loại mục khác	
H48.1*	Viêm thần kinh hậu nhãn cầu do bệnh phân loại mục khác	
H48.8*	Rối loạn khác của thần kinh thị giác và/hoặc đường dẫn truyền thị giác do bệnh phân loại mục khác	
H49	Lác liệt	
H49.0	Liệt dây thần kinh III [vận nhãn]	
H49.1	Liệt dây thần kinh IV [ròng rọc]	
H49.2	Liệt dây thần kinh VI [vận nhãn ngoài]	
H49.3	Liệt vận nhãn (ngoài) toàn bộ	
H49.4	Liệt vận nhãn ngoài tiến triển	
H49.8	Lác liệt khác	Liệt vận nhãn ngoài không xác định khác|Hội chứng Kearns-Sayre
H49.9	Lác liệt, không xác định	
H50	Lác khác	
H50.0	Lác trong [Lác hội tụ đồng hành]	
H50.1	Lác ngoài [Lác phân kỳ đồng hành]	
H50.2	Lác dọc	Lác hướng lên trên|Lác hướng xuống dưới
H50.3	Lác luân hồi [khi lác khi không]	
H50.4	Lác khác và/hoặc không xác định	Lác phối hợp không xác định khác|Lác vòng|Vi lác|Hội chứng định thị một mắt [góc lác nhỏ]
H50.5	Lác ẩn	Lác luân phiên|Lác trong ẩn|Lác ngoài ẩn
H50.6	Lác cơ học	Hội chứng Brown bao cơ chéo trên|Lác do dính|Hạn chế hoạt động cơ mắt do chấn thương
H50.8	Lác xác định khác	Hội chứng Duane
H50.9	Lác, không xác định	
H51	Rối loạn khác của vận nhãn hai mắt	
H51.0	Liệt động tác liên hợp hai mắt [liệt đồng trục thị giác]	
H51.1	Thiểu năng và/hoặc gia tăng quy tụ	
H51.2	Liệt vận nhãn liên nhân	
H51.8	Rối loạn vận nhãn hai mắt xác định khác	
H51.9	Rối loạn vận nhãn hai mắt, không xác định	
H52	Rối loạn khúc xạ và/hoặc điều tiết	
H52.0	Viễn thị	
H52.1	Cận thị	
H52.2	Loạn thị	
H52.3	Lệch khúc xạ hai mắt và/hoặc lệch hình ảnh võng mạc hai mắt	
H52.4	Lão thị	
H52.5	Rối loạn điều tiết	
H52.6	Tật khúc xạ khác	
H52.7	Tật khúc xạ, không xác định	
H53	Rối loạn thị giác	
H53.0	Nhược thị do mắt không nhìn	
H53.1	Rối loạn thị giác chủ quan	
H53.2	Song thị	Song thị [nhìn đôi]
H53.3	Rối loạn khác của thị giác hai mắt	Tương ứng võng mạc bất thường|Hợp thị không có phù thị|Đồng thị không có hợp thị|Ức chế thị giác hai mắt
H53.4	Tổn hại thị trường	
H53.5	Rối loạn sắc giác	
H53.6	Quáng gà	
H53.8	Rối loạn thị giác khác	
H53.9	Rối loạn thị giác, không xác định	
H54	Thị lực giảm và/hoặc khiếm thị (hai mắt hoặc một mắt)	
H54.0	Mù, hai mắt	Các loại giảm thị lực độ 3, 4, 5 cả hai mắt
H54.1	Thị lực giảm mức độ nặng, hai mắt	Giảm thị lực độ 2
H54.2	Thị lực giảm mức độ vừa, hai mắt	Giảm thị lực độ 1
H54.3	Thị lực giảm mức độ nhẹ hoặc không giảm, hai mắt	Giảm thị lực độ 0
H54.4	Mù, một mắt	Phân loại giảm thị lực độ 3, 4, 5 một mắt và phân loại độ 0, 1, 2 hoặc 9 ở mắt bên kia.
H54.5	Thị lực giảm mức độ nặng, một mắt	Giảm thị lực độ 2 ở một mắt và độ 0, 1 hoặc 9 ở mắt bên kia.
H54.6	Thị lực giảm mức độ vừa, một mắt	Giảm thị lực độ 1 ở một mắt và độ 0 hoặc 9 ở mắt bên kia.
H54.9	Thị lực giảm mức độ không xác định (hai mắt)	
H55	Rung giật nhãn cầu và/hoặc rối loạn vận nhãn khác	
H57	Rối loạn khác của mắt và/hoặc cấu trúc phụ cận của mắt	
H57.0	Bất thường chức năng đồng tử	
H57.1	Đau nhức mắt	
H57.8	Rối loạn xác định khác của mắt và/hoặc cấu trúc phụ cận của mắt	
H57.9	Rối loạn mắt và/hoặc cấu trúc phụ cận của mắt, không xác định	
H58.*	Rối loạn khác của mắt và/hoặc cấu trúc phụ cận của mắt do bệnh phân loại mục khác	
H58.0*	Bất thường chức năng đồng tử do bệnh phân loại mục khác	
H58.1*	Rối loạn thị giác do bệnh phân loại mục khác	
H58.8*	Rối loạn xác định khác của mắt và/hoặc cấu trúc phụ cận của mắt do bệnh phân loại mục khác	
H59	Rối loạn của mắt và/hoặc cấu trúc phụ cận của mắt sau can thiệp không phân loại mục khác	
H59.0	Bệnh lý giác mạc (bọng không có thể thủy tinh) sau phẫu thuật đục thể thủy tinh	
H59.8	Rối loạn khác của mắt và/hoặc cấu trúc phụ cận của mắt sau can thiệp	
H59.9	Rối loạn mắt và/hoặc cấu trúc phụ cận của mắt sau can thiệp, không xác định	
H60	Viêm tai ngoài	
H60.0	Áp xe tai ngoài	Nhọt vành tai hoặc ống tai ngoài|Nhọt cụm vành tai hoặc ống tai ngoài|Nhọt bọc vành tai hoặc ống tai ngoài
H60.1	Viêm mô tế bào tai ngoài	
H60.2	Viêm tai ngoài ác tính	
H60.3	Viêm tai ngoài khác do nhiễm trùng	
H60.4	Viêm tai ngoài có Cholesteatoma	
H60.5	Viêm tai ngoài cấp tính, không do nhiễm trùng	
H60.8	Viêm tai ngoài khác	Viêm tai ngoài mạn tính không xác định khác
H60.9	Viêm tai ngoài, không xác định	
H61	Rối loạn khác của tai ngoài	
H61.0	Viêm màng sụn tai ngoài	Viêm da sụn dạng nốt mạn của vành tai|Viêm màng bao sụn của: viêm màng sụn vành tai của|loa tai|vành tai
H61.1	Rối loạn không do nhiễm trùng của vành tai	
H61.2	Nút ráy tai	Ráy tai
H61.3	Hẹp ống tai ngoài mắc phải	Xẹp ống tai ngoài
H61.8	Rối loạn xác định khác của tai ngoài	Chồi xương ống tai ngoài
H61.9	Rối loạn tai ngoài, không xác định	
H62.*	Rối loạn tai ngoài ở bệnh phân loại mục khác	
H62.0*	Viêm tai ngoài do nhiễm khuẩn phân loại mục khác	
H62.1*	Viêm tai ngoài do virus phân loại mục khác	
H62.2*	Viêm tai ngoài do nhiễm nấm	
H62.3*	Viêm tai ngoài do bệnh nhiễm trùng và/hoặc do nhiễm ký sinh trùng khác phân loại mục khác	
H62.4*	Viêm tai ngoài do bệnh khác phân loại mục khác	
H62.8*	Rối loạn khác của tai ngoài ở bệnh phân loại mục khác	
H65	Viêm tai giữa không có mủ	
H65.0	Viêm tai giữa tiết dịch cấp tính	Viêm tai giữa tiết dịch cấp tính và/hoặc bán cấp tính
H65.1	Viêm tai giữa cấp tính không có mủ khác	
H65.2	Viêm tai giữa tiết dịch mạn tính	Xuất tiết vòi nhĩ mạn tính
H65.3	Viêm tai giữa tiết dịch nhày mạn tính	
H65.4	Viêm tai giữa không có mủ mạn tính khác	
H65.9	Viêm tai giữa không có mủ, không xác định	
H66	Viêm tai giữa có mủ và/hoặc viêm tai giữa không xác định	
H66.0	Viêm tai giữa có mủ cấp tính	
H66.1	Viêm tai giữa có mủ ở vòi hòm nhĩ mạn tính	Viêm tai giữa có mủ mạn tính lành tính|Bệnh lý vòi - hòm nhĩ mạn tính
H66.2	Viêm tai giữa có mủ ở thượng nhĩ mạn tính	Bệnh lý thượng nhĩ sào bào mạn tính
H66.3	Viêm tai giữa có mủ mạn tính khác	Viêm tai giữa có mủ mạn tính không xác định khác
H66.4	Viêm tai giữa có mủ, không xác định	Viêm tai giữa có mủ không xác định khác
H66.9	Viêm tai giữa, không xác định	
H67.*	Viêm tai giữa do bệnh phân loại mục khác	
H67.0*	Viêm tai giữa do nhiễm vi khuẩn phân loại mục khác	
H67.1*	Viêm tai giữa do virus phân loại mục khác	
H67.8*	Viêm tai giữa do bệnh khác phân loại mục khác	
H68	Viêm và/hoặc tắc vòi tai [vòi Eustache, vòi nhĩ]	
H68.0	Viêm vòi tai [vòi Eustache, vòi nhĩ]	
H68.1	Tắc vòi tai [vòi Eustache, vòi nhĩ]	Chèn ép vòi tai [vòi Eustache, vòi nhĩ]|Chít hẹp vòi tai [vòi Eustache, vòi nhĩ]
H69	Rối loạn khác của vòi tai [vòi Eustache, vòi nhĩ]	
H69.0	Giãn rộng vòi tai [vòi Eustache, vòi nhĩ]	
H69.8	Rối loạn xác định khác của vòi tai [vòi Eustache, vòi nhĩ]	
H69.9	Rối loạn vòi tai [vòi Eustache, vòi nhĩ], không xác định	
H70	Viêm xương chũm và/hoặc bệnh liên quan	
H70.0	Viêm xương chũm cấp tính	Áp xe xương chũm|Phù nề xương chũm
H70.1	Viêm xương chũm mạn tính	
H70.2	Viêm xương đá	
H70.8	Viêm xương chũm khác và/hoặc bệnh liên quan	
H70.9	Viêm xương chũm, không xác định	
H71	Viêm tai giữa có Cholesteatoma	
H72	Thủng màng nhĩ	
H72.0	Thủng màng nhĩ trung tâm	
H72.1	Thủng màng nhĩ ở vị trí thượng nhĩ	Thủng màng chùng
H72.2	Thủng màng nhĩ ở vùng rìa khác	
H72.8	Thủng màng nhĩ khác	
H72.9	Thủng màng nhĩ, không xác định	
H73	Rối loạn khác của màng nhĩ	
H73.0	Viêm màng nhĩ cấp tính	
H73.1	Viêm màng nhĩ mạn tính	
H73.8	Rối loạn xác định khác của màng nhĩ	
H73.9	Rối loạn màng nhĩ, không xác định	
H74	Rối loạn khác của tai giữa và/hoặc xương chũm	
H74.0	Xơ hóa màng nhĩ và niêm mạc tai giữa	
H74.1	Viêm tai giữa dính	
H74.2	Gián đoạn và/hoặc trật khớp chuỗi xương con	
H74.3	Bất thường mắc phải khác của chuỗi xương con	Cứng khớp chuỗi xương con|Mất một phần trong chuỗi xương con
H74.4	Polyp ở tai giữa	
H74.8	Rối loạn xác định khác của tai giữa và/hoặc xương chũm	
H74.9	Rối loạn tai giữa và/hoặc xương chũm, không xác định	
H75.*	Rối loạn tai giữa và/hoặc xương chũm do bệnh phân loại mục khác	
H75.0*	Viêm xương chũm do nhiễm trùng và/hoặc do nhiễm ký sinh trùng phân loại mục khác	
H75.8*	Những rối loạn xác định khác của tai giữa và/hoặc xương chũm do bệnh đã phân loại mục khác	
H80	Xốp xơ tai	
H80.0	Xốp xơ tai ở cửa sổ bầu dục, không gây bít tắc	
H80.1	Xốp xơ tai ở cửa sổ bầu dục, có gây bít tắc	
H80.2	Xốp xơ ốc tai	
H80.8	Xốp xơ tai khác	
H80.9	Xốp xơ tai, không xác định	
H81	Rối loạn chức năng tiền đình	
H81.0	Bệnh Ménière	Phù mê nhĩ|Hội chứng Ménière hoặc chóng mặt
H81.1	Chóng mặt kịch phát lành tính	
H81.2	Viêm dây thần kinh tiền đình	
H81.3	Chóng mặt do rối loạn tiền đình ngoại biên khác	Hội chứng Lermoyer
H81.4	Chóng mặt do rối loạn tiền đình trung ương	Rung giật nhãn cầu vị trí trung tâm
H81.8	Rối loạn chức năng tiền đình khác	
H81.9	Rối loạn chức năng tiền đình, không xác định	Hội chứng chóng mặt không xác định khác
H82.*	Hội chứng chóng mặt do bệnh được phân loại mục khác	
H83	Bệnh khác của tai trong	
H83.0	Viêm mê nhĩ	
H83.1	Rò mê nhĩ	
H83.2	Rối loạn chức năng mê nhĩ	Mê nhĩ quá mẫn cảm|Suy giảm chức năng mê nhĩ|Mất chức năng mê nhĩ
H83.3	Ảnh hưởng của tiếng ồn lên tai trong	Chấn thương do âm thanh|Giảm thính lực do tiếng ồn
H83.8	Bệnh xác định khác của tai trong	
H83.9	Bệnh của tai trong, không xác định	
H90	Giảm thính lực dẫn truyền và/hoặc giảm thính lực thần kinh giác quan	
H90.0	Giảm thính lực dẫn truyền, hai tai	
H90.1	Giảm thính lực dẫn truyền một bên tai, không hạn chế sức nghe ở tai còn lại	
H90.2	Giảm thính lực dẫn truyền, không xác định	Điếc dẫn truyền không xác định khác
H90.3	Giảm thính lực thần kinh giác quan, hai tai	
H90.4	Giảm thính lực thần kinh giác quan một bên tai, không hạn chế sức nghe ở tai còn lại	
H90.5	Giảm thính lực thần kinh giác quan, không xác định	Điếc bẩm sinh không xác định khác
H90.6	Giảm thính lực hỗn hợp dẫn truyền và/hoặc giảm thính lực thần kinh giác quan, hai tai	
H90.7	Giảm thính lực hỗn hợp dẫn truyền và/hoặc giảm thính lực thần kinh giác quan một tai, không hạn chế sức nghe ở tai còn lại	
H90.8	Giảm thính lực thần kinh giác quan và/hoặc dẫn truyền hỗn hợp, không xác định	
H91	Giảm thính lực khác	
H91.0	Giảm thính lực độc tai [chất hóa học gây ảnh hưởng đến thính lực]	
H91.1	Giảm thính lực ở người cao tuổi [lão thính]	Giảm thính lực liên quan đến tuổi tác
H91.2	Giảm thính lực đột ngột vô căn	Giảm thính lực đột ngột không xác định khác
H91.3	Câm điếc, không phân loại mục khác	
H91.8	Giảm thính lực xác định khác	
H91.9	Giảm thính lực, không xác định	
H92	Đau tai và/hoặc tràn dịch tai	
H92.0	Đau tai	
H92.1	Chảy dịch tai	
H92.2	Chảy máu tai	
H93	Rối loạn khác của tai, không phân loại mục khác	
H93.0	Thoái hóa và/hoặc rối loạn mạch máu tai	
H93.1	Ù tai	
H93.2	Nhận thức thính giác bất thường khác	
H93.3	Rối loạn thần kinh thính giác	Rối loạn dây thần kinh sọ số 8
H93.8	Rối loạn xác định khác của tai	
H93.9	Rối loạn tai, không xác định	
H94.*	Rối loạn khác của tai do bệnh phân loại mục khác	
H94.0*	Viêm dây thần kinh thính giác do nhiễm trùng và/hoặc nhiễm ký sinh trùng phân loại mục khác	
H94.8*	Rối loạn xác định khác của tai do bệnh phân loại mục khác	
H95	Rối loạn tai và/hoặc xương chũm sau can thiệp, không phân loại mục khác	
H95.0	Cholesteatoma tái phát sau phẫu thuật khoét hang chũm	
H95.1	Rối loạn khác sau phẫu thuật khoét chũm	Viêm mạn tính sau phẫu thuật khoét chũm|Lên mô hạt sau phẫu thuật khoét chũm|Nang nhầy sau phẫu thuật khoét chũm
H95.8	Rối loạn khác của tai và/hoặc xương chũm sau can thiệp	
H95.9	Rối loạn của tai và/hoặc xương chũm sau can thiệp, không xác định	
I00	Bệnh sốt thấp không đề cập tác động đến tim	
I01	Bệnh sốt thấp có tác động đến tim	
I01.0	Viêm màng ngoài tim cấp tính do bệnh thấp	
I01.1	Viêm nội tâm mạc cấp tính do bệnh thấp	
I01.2	Viêm cơ tim cấp tính do bệnh thấp	Các bệnh lý trong mục I00 kèm viêm cơ tim
I01.8	Bệnh tim cấp tính khác do bệnh thấp	Bất kỳ bệnh lý nào trong I00 có kèm một hoặc nhiều tổn thương ở tim|Viêm tim toàn bộ do bệnh thấp
I01.9	Bệnh tim cấp tính do bệnh thấp, không xác định	Bất kỳ bệnh lý nào trong I00 kèm theo tổn thương tim, kiểu không xác định.
I02	Múa giật do bệnh thấp	
I02.0	Múa giật do bệnh thấp có tác động đến tim	Múa giật không xác định khác kèm tổn thương tim|Múa giật do bệnh thấp có ảnh hưởng đến tim với bất kỳ kiểu phân loại nào trong I01.
I02.9	Múa giật do thấp tim không có tác động đến tim	Múa giật do bệnh thấp không xác định khác
I05	Bệnh van hai lá do bệnh thấp	
I05.0	Hẹp van hai lá	
I05.1	Hở van hai lá do bệnh thấp	
I05.2	Hẹp hở van hai lá	Hẹp van hai lá kèm theo hở van hai lá
I05.8	Bệnh khác của van hai lá	Suy van hai lá
I05.9	Bệnh van hai lá, không xác định	
I06	Bệnh van động mạch chủ do bệnh thấp	
I06.0	Hẹp van động mạch chủ do bệnh thấp	
I06.1	Hở van động mạch chủ do bệnh thấp	
I06.2	Hẹp kèm hở van động mạch chủ do bệnh thấp	Hẹp động mạch chủ do bệnh thấp kèm thiểu năng hoặc hở
I06.8	Bệnh van động mạch chủ khác do bệnh thấp	
I06.9	Bệnh van động mạch chủ do bệnh thấp, không xác định	
I07	Bệnh van ba lá do bệnh thấp	
I07.0	Hẹp van ba lá	
I07.1	Hở van ba lá	
I07.2	Hẹp hở van ba lá	
I07.8	Bệnh khác của van ba lá	
I07.9	Bệnh lý van ba lá khác, không xác định	Rối loạn van ba lá không xác định khác
I08	Bệnh lý của nhiều van tim	
I08.0	Rối loạn cả van hai lá và van động mạch chủ	Tổn thương cả van hai lá và van động mạch chủ do thấp hoặc không rõ nguyên nhân.
I08.1	Rối loạn cả van hai lá và van ba lá	
I08.2	Rối loạn cả van động mạch chủ và van ba lá	
I08.3	Rối loạn kết hợp van hai lá, van động mạch chủ và van ba lá	
I08.8	Bệnh lý của nhiều van tim khác	
I08.9	Bệnh lý của nhiều van tim khác, không xác định	
I09	Bệnh thấp tim khác	
I09.0	Viêm cơ tim do bệnh thấp	
I09.1	Bệnh nội tâm mạc do bệnh thấp, không xác định van	
I09.2	Viêm màng ngoài tim mạn tính do bệnh thấp	
I09.8	Bệnh thấp tim xác định khác	Bệnh van động mạch phổi do bệnh thấp
I09.9	Bệnh thấp tim, không xác định	
I10	Bệnh tăng huyết áp vô căn (nguyên phát)	
I11	Bệnh tim do tăng huyết áp	
I11.0	Bệnh tim do tăng huyết áp kèm suy tim (xung huyết)	Suy tim do tăng huyết áp
I11.9	Bệnh tim do tăng huyết áp không kèm suy tim (xung huyết)	Bệnh tim do tăng huyết áp không xác định khác
I12	Bệnh thận do tăng huyết áp	
I12.0	Bệnh thận do tăng huyết áp có kèm suy thận	Suy thận do tăng huyết áp
I12.9	Bệnh thận do tăng huyết áp không kèm suy thận	Bệnh thận do tăng huyết áp không xác định khác
I13	Bệnh tim và bệnh thận do tăng huyết áp	
I13.0	Bệnh tim và bệnh thận do tăng huyết áp kèm suy tim (xung huyết)	
I13.1	Bệnh tim và bệnh thận do tăng huyết áp kèm có suy thận	
I13.2	Bệnh tim và bệnh thận do tăng huyết áp kèm cả suy tim (xung huyết) và suy thận	
I13.9	Bệnh tim và bệnh thận do tăng huyết áp, không xác định	
I15	Tăng huyết áp thứ phát	
I15.0	Tăng huyết áp do nguyên nhân mạch thận	
I15.1	Tăng huyết áp thứ phát do rối loạn thận khác	
I15.2	Tăng huyết áp thứ phát do rối loạn nội tiết	
I15.8	Tăng huyết áp thứ phát khác	
I15.9	Tăng huyết áp thứ phát, không xác định	
I20	Cơn đau thắt ngực	
I20.0	Cơn đau thắt ngực không ổn định	
I20.1	Cơn đau thắt ngực có bằng chứng co thắt động mạch vành	
I20.8	Cơn đau thắt ngực thể khác	Cơn đau thắt ngực khi gắng sức|Hội chứng dòng chảy chậm của động mạch vành|Cơn đau thắt ngực ổn định|Chứng đau thắt ngực
I20.9	Cơn đau thắt ngực, không xác định	
I21	Nhồi máu cơ tim cấp tính	
I21.0	Nhồi máu cơ tim xuyên thành cấp tính của thành trước	
I21.1	Nhồi máu cơ tim xuyên thành cấp tính của thành dưới	
I21.2	Nhồi máu xuyên thành cấp tính ở vị trí khác	
I21.3	Nhồi máu cơ tim xuyên thành cấp tính ở vị trí không xác định	Nhồi máu cơ tim xuyên thành không xác định khác
I21.4	Nhồi máu cơ tim dưới nội tâm mạc cấp tính	Nhồi máu cơ tim không có ST chênh|Nhồi máu cơ tim không xuyên thành không xác định khác
I21.9	Nhồi máu cơ tim cấp tính, không xác định	
I22	Nhồi máu cơ tim tái phát	
I22.0	Nhồi máu cơ tim tái phát của thành trước	
I22.1	Nhồi máu cơ tim tái phát của thành dưới	
I22.8	Nhồi máu cơ tim tái phát ở vị trí khác	
I22.9	Nhồi máu cơ tim tái phát ở vị trí không xác định	
I23	Biến chứng hiện tại xác định sau nhồi máu cơ tim cấp tính	
I23.0	Biến chứng tràn máu màng ngoài tim sau nhồi máu cơ tim cấp tính	
I23.1	Biến chứng thủng vách liên nhĩ sau nhồi máu cơ tim cấp tính	
I23.2	Biến chứng thủng vách liên thất sau nhồi máu cơ tim cấp tính	
I23.3	Biến chứng vỡ thành tim không có tràn máu màng ngoài tim sau nhồi máu cơ tim cấp tính	
I23.4	Biến chứng đứt dây chằng van hai lá sau nhồi máu cơ tim cấp tính	
I23.5	Biến chứng đứt cơ nhú sau nhồi máu cơ tim cấp tính	
I23.6	Huyết khối trong buồng tim tâm nhĩ, tiểu nhĩ và/hoặc tâm thất là biến chứng hiện tại sau nhồi máu cơ tim cấp tính	
I23.8	Biến chứng hiện tại khác xảy ra sau nhồi máu cơ tim cấp tính	
I24	Bệnh tim thiếu máu cục bộ cấp tính khác	
I24.0	Huyết khối mạch vành không gây nhồi máu cơ tim	
I24.1	Hội chứng Dressler	Hội chứng sau nhồi máu cơ tim
I24.8	Thể khác của bệnh tim thiếu máu cục bộ cấp tính	
I24.9	Bệnh tim do thiếu máu cục bộ cấp tính, không xác định	
I25	Bệnh tim thiếu máu cục bộ mạn tính	
I25.0	Bệnh tim mạch do xơ vữa động mạch vành, như đã mô tả	
I25.1	Bệnh tim mạch do xơ vữa động mạch	
I25.2	Nhồi máu cơ tim cũ	Nhồi máu cơ tim đã được chữa
I25.3	Phình thành tim	
I25.4	Phình và/hoặc tách động mạch vành	
I25.5	Bệnh lý cơ tim do thiếu máu cục bộ	
I25.6	Thiếu máu cơ tim thầm lặng	
I25.8	Thể khác của bệnh tim thiếu máu cục bộ mạn tính	
I25.9	Bệnh tim thiếu máu cục bộ mạn tính, không xác định	
I26	Thuyên tắc mạch phổi	
I26.0	Thuyên tắc mạch phổi có tâm phế cấp tính	Tâm phế cấp không xác định khác
I26.9	Thuyên tắc mạch phổi không có tâm phế cấp tính	Thuyên tắc phổi không xác định khác
I27	Bệnh tim phổi khác	
I27.0	Tăng huyết áp động mạch phổi nguyên phát	
I27.1	Bệnh tim do gù vẹo cột sống	
I27.2	Tăng huyết áp động mạch phổi thứ phát	
I27.8	Bệnh tim phổi xác định khác	
I27.9	Bệnh tim phổi, không xác định	
I28	Bệnh mạch máu phổi khác	
I28.0	Rò động - tĩnh mạch phổi	
I28.1	Phình động mạch phổi	
I28.8	Bệnh mạch máu phổi xác định khác	Vỡ mạch máu phổi|Hẹp mạch máu phổi|Co hẹp mạch máu phổi
I28.9	Bệnh mạch máu phổi, không xác định	
I30	Viêm màng ngoài tim cấp tính	
I30.0	Viêm màng ngoài tim cấp tính vô căn, không xác định	
I30.1	Viêm màng ngoài tim do nhiễm trùng	
I30.8	Thể khác của viêm màng ngoài tim cấp	
I30.9	Viêm màng ngoài tim cấp tính, không xác định	
I31	Bệnh màng ngoài tim khác	
I31.0	Viêm màng ngoài tim dính mạn tính	
I31.1	Viêm màng ngoài tim co thắt mạn tính	Viêm màng ngoài tim dính bên trong|Vôi hóa màng ngoài tim
I31.2	Tràn máu ngoại tâm mạc, không phân loại mục khác	
I31.3	Tràn dịch màng ngoài tim (không do viêm)	Tràn dịch dưỡng chấp màng ngoài tim
I31.8	Bệnh màng ngoài tim xác định khác	Xơ vữa thượng tâm mạc|Dính khu trú màng ngoài tim
I31.9	Bệnh ngoại tâm mạc, không xác định	
I32.*	Viêm màng ngoài tim do bệnh phân loại mục khác	
I32.0*	Viêm màng ngoài tim do bệnh nhiễm khuẩn phân loại mục khác	
I32.1*	Viêm màng ngoài tim do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác	
I32.8*	Viêm màng ngoài tim do bệnh khác phân loại mục khác	
I33	Viêm nội tâm mạc cấp và/hoặc bán cấp	
I33.0	Viêm nội tâm mạc do nhiễm trùng cấp tính và/hoặc bán cấp tính	
I33.9	Viêm nội tâm mạc cấp, không xác định	Viêm nội tâm mạc cấp tính hoặc bán cấp tính|Viêm cơ tim - nội tâm mạc cấp tính hoặc bán cấp tính|Viêm nội - màng ngoài tim cấp tính hoặc bán cấp tính
I34	Rối loạn van hai lá không do bệnh thấp	
I34.0	Hở (van) hai lá	
I34.1	Sa (van) hai lá	
I34.2	Hẹp (van) hai lá không do bệnh thấp	
I34.8	Rối loạn van hai lá khác không do bệnh thấp	
I34.9	Rối loạn van hai lá không do bệnh thấp, không xác định	
I35	Rối loạn van động mạch chủ không do bệnh thấp	
I35.0	Hẹp (van) động mạch chủ	
I35.1	Hở (van) động mạch chủ	
I35.2	Hẹp kèm hở (van) động mạch chủ	
I35.8	Rối loạn van động mạch chủ khác	
I35.9	Bệnh van động mạch chủ, không xác định	
I36	Rối loạn van ba lá không do bệnh thấp	
I36.0	Hẹp (van) ba lá không do bệnh thấp	
I36.1	Hở (van) ba lá không do bệnh thấp	
I36.2	Hẹp kèm hở van ba lá không do bệnh thấp	
I36.8	Rối loạn van ba lá khác không do bệnh thấp	
I36.9	Rối loạn van ba lá khác không do bệnh thấp, không xác định	
I37	Rối loạn van động mạch phổi	
I37.0	Hẹp van động mạch phổi	
I37.1	Hở van động mạch phổi	
I37.2	Hẹp kèm hở van động mạch phổi	
I37.8	Rối loạn van động mạch phổi khác	
I37.9	Rối loạn van động mạch phổi, không xác định	
I38	Viêm nội tâm mạc, không xác định van	
I39.*	Viêm nội tâm mạc và/hoặc bệnh van tim do bệnh phân loại mục khác	
I39.0*	Rối loạn van hai lá do bệnh phân loại mục khác	
I39.1*	Rối loạn van động mạch chủ do bệnh phân loại mục khác	
I39.2*	Rối loạn van ba lá do bệnh phân loại mục khác	
I39.3*	Rối loạn van động mạch phổi do bệnh phân loại mục khác	
I39.4*	Rối loạn của nhiều van do bệnh phân loại mục khác	
I39.8*	Bệnh viêm nội tâm mạc, không xác định van, do bệnh phân loại mục khác	
I40	Bệnh viêm cơ tim cấp	
I40.0	Bệnh viêm cơ tim do nhiễm trùng	
I40.1	Bệnh viêm cơ tim đơn thuần	
I40.8	Bệnh viêm cơ tim cấp khác	
I40.9	Bệnh viêm cơ tim cấp, không xác định	
I41.*	Bệnh viêm cơ tim do bệnh phân loại mục khác	
I41.0*	Bệnh viêm cơ tim do bệnh nhiễm trùng phân loại mục khác	
I41.1*	Bệnh viêm cơ tim do bệnh do virus phân loại mục khác	
I41.2*	Bệnh viêm cơ tim do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác	
I41.8*	Bệnh viêm cơ tim do bệnh khác phân loại mục khác	
I42	Bệnh lý cơ tim	
I42.0	Bệnh lý cơ tim giãn	Bệnh cơ tim xung huyết
I42.1	Bệnh lý cơ tim phì đại có tắc nghẽn	Hẹp dưới động mạch chủ phì đại
I42.2	Bệnh lý cơ tim phì đại khác	Bệnh cơ tim phì đại không gây tắc nghẽn
I42.3	Bệnh cơ - nội tâm mạc (tăng bạch cầu ái toan)	
I42.4	Xơ chun nội tâm mạc	Bệnh cơ tim bẩm sinh
I42.5	Bệnh lý cơ tim hạn chế	Bệnh cơ tim hạn chế không xác định khác
I42.6	Bệnh lý cơ tim do rượu	
I42.7	Bệnh lý cơ tim do thuốc và tác nhân bên ngoài khác	
I42.8	Bệnh lý cơ tim khác	
I42.9	Bệnh lý cơ tim, không xác định	
I43.*	Bệnh lý cơ tim do bệnh phân loại mục khác	
I43.0*	Bệnh lý cơ tim do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
I43.1*	Bệnh lý cơ tim do bệnh chuyển hóa	
I43.2*	Bệnh lý cơ tim do bệnh về dinh dưỡng	
I43.8*	Bệnh lý cơ tim do bệnh khác phân loại mục khác	
I44	Blốc nhĩ thất và/hoặc nhánh trái	
I44.0	Blốc nhĩ thất cấp I	
I44.1	Blốc nhĩ thất cấp II	Blốc nhĩ thất, típ I và II|Blốc nhĩ thất kiểu Möbitz|Blốc cấp II, típ I và II|Blốc nhĩ thất kiểu chu kỳ Wenckebach
I44.2	Blốc nhĩ thất, hoàn toàn	Blốc tim hoàn toàn không xác định khác|Blốc nhĩ thất cấp III
I44.3	Blốc nhĩ thất khác và/hoặc không xác định	Blốc nhĩ thất không xác định khác
I44.4	Blốc nhánh trước trái	
I44.5	Blốc nhánh sau trái	
I44.6	Blốc nhánh khác và/hoặc không xác định	Blốc phân nhánh trái không xác định khác
I44.7	Blốc nhánh trái, không xác định	
I45	Rối loạn dẫn truyền khác	
I45.0	Blốc nhánh phải	
I45.1	Blốc nhánh phải khác và/hoặc không xác định	Blốc nhánh phải không xác định khác
I45.2	Blốc hai nhánh	
I45.3	Blốc ba nhánh	
I45.4	Blốc trong thất không xác định cụ thể	Blốc nhánh không xác định khác
I45.5	Blốc tim xác định khác	
I45.6	Hội chứng kích thích sớm (hội chứng tiền kích thích)	Kích thích nhĩ - thất bất thường
I45.8	Rối loạn dẫn truyền xác định khác	
I45.9	Rối loạn dẫn truyền, không xác định	Blốc tim [chẹn tim] không xác định khác|Hội chứng Stokes-Adams
I46	Ngưng tim	
I46.0	Ngưng tim được hồi sức thành công	
I46.1	Đột tử do tim, như đã mô tả	
I46.9	Ngưng tim, không xác định	
I47	Nhịp nhanh kịch phát	
I47.0	Rối loạn nhịp thất do cơ chế vòng vào lại	
I47.1	Nhịp nhanh trên thất	
I47.2	Nhịp nhanh thất	
I47.9	Nhịp nhanh kịch phát, không xác định	
I48	Rung nhĩ và/hoặc cuồng nhĩ	
I48.0	Rung nhĩ kịch phát	
I48.1	Rung nhĩ dai dẳng	
I48.2	Rung nhĩ mạn tính	
I48.3	Cuồng nhĩ điển hình	Cuồng nhĩ típ I
I48.4	Cuồng nhĩ không điển hình	Cuồng nhĩ típ II
I48.9	Rung nhĩ và/hoặc cuồng nhĩ, không xác định	
I49	Loạn nhịp tim khác	
I49.0	Rung thất và/hoặc cuồng thất	
I49.1	Ngoại tâm thu nhĩ	
I49.2	Khử cực sớm vùng bộ nối	
I49.3	Ngoại tâm thu thất	
I49.4	Khử cực sớm khác và/hoặc không xác định	Nhịp lạc chỗ|Ngoại tâm thu|Loạn nhịp ngoại tâm thu
I49.5	Hội chứng suy nút xoang	Hội chứng nhịp nhanh - nhịp chậm
I49.8	Loạn nhịp tim xác định khác	Hội chứng Brugada|Hội chứng QT kéo dài
I49.9	Rối loạn nhịp tim, không xác định	
I50	Suy tim	
I50.0	Suy tim xung huyết	
I50.1	Suy thất trái	Hen tim|Suy tim trái|Phù phổi kèm theo bệnh tim không xác định khác hoặc suy tim|Phù tại phổi kèm theo bệnh tim không xác định khác hoặc suy tim
I50.9	Suy tim, không xác định	Suy tim hoặc suy cơ tim không xác định khác
I51	Biến chứng tim và/hoặc bệnh tim mô tả không rõ ràng	
I51.0	Thông vách ngăn tim, mắc phải	
I51.1	Đứt dây chằng van tim, không phân loại mục khác	
I51.2	Đứt cơ nhú, không phân loại mục khác	
I51.3	Huyết khối trong tim, không phân loại mục khác	
I51.4	Viêm cơ tim, không xác định	
I51.5	Thoái hóa cơ tim	
I51.6	Bệnh tim mạch, không xác định	
I51.7	Tim to	
I51.8	Bệnh tim không rõ ràng khác	
I51.9	Bệnh tim, không xác định	
I52.*	Rối loạn tim khác do bệnh phân loại mục khác	
I52.0*	Rối loạn tim khác do bệnh nhiễm trùng phân loại mục khác	
I52.1*	Rối loạn tim khác do bệnh nhiễm trùng và ký sinh trùng khác phân loại mục khác	
I52.8*	Rối loạn tim khác do bệnh khác phân loại mục khác	
I60	Xuất huyết dưới nhện	
I60.0	Xuất huyết dưới nhện từ hành cảnh và/hoặc chỗ chia nhánh động mạch cảnh	
I60.1	Xuất huyết dưới nhện từ động mạch não giữa	
I60.2	Xuất huyết dưới nhện từ động mạch thông trước	
I60.3	Xuất huyết dưới nhện từ động mạch thông sau	
I60.4	Xuất huyết dưới nhện từ động mạch nền	
I60.5	Xuất huyết dưới nhện từ động mạch đốt sống	
I60.6	Xuất huyết dưới nhện từ động mạch nội sọ khác	Tổn thương nhiều động mạch nội sọ
I60.7	Xuất huyết dưới nhện từ động mạch nội sọ, không xác định	
I60.8	Xuất huyết dưới nhện khác	Xuất huyết màng não|Vỡ dị dạng động tĩnh mạch não
I60.9	Xuất huyết dưới nhện, không xác định	
I61	Xuất huyết não	
I61.0	Xuất huyết não tại bán cầu, vùng dưới vỏ	Xuất huyết sâu trong não
I61.1	Xuất huyết não tại bán cầu, vùng vỏ	Xuất huyết thùy não|Xuất huyết nông trong não
I61.2	Xuất huyết não tại bán cầu, không xác định	
I61.3	Xuất huyết thân não	
I61.4	Xuất huyết tiểu não	
I61.5	Xuất huyết não, tràn máu não thất	
I61.6	Xuất huyết não đa ổ	
I61.8	Xuất huyết não khác	
I61.9	Xuất huyết não, không xác định	
I62	Xuất huyết não không do chấn thương khác	
I62.0	Xuất huyết dưới màng cứng không do chấn thương	
I62.1	Xuất huyết ngoài màng cứng không do chấn thương	
I62.9	Xuất huyết não không do chấn thương, không xác định	
I63	Nhồi máu não	
I63.0	Nhồi máu não do huyết khối động mạch trước não	
I63.1	Nhồi máu não do thuyên tắc động mạch trước não	
I63.2	Nhồi máu não không xác định do tắc hoặc hẹp ở động mạch trước não	
I63.3	Nhồi máu não do huyết khối động mạch não	
I63.4	Nhồi máu não do thuyên tắc động mạch não	
I63.5	Nhồi máu não không xác định do tắc hoặc hẹp ở động mạch não	
I63.6	Nhồi máu não do huyết khối tĩnh mạch não, không sinh mủ	
I63.8	Nhồi máu não khác	
I63.9	Nhồi máu não, không xác định	
I64	Đột quỵ, không xác định do xuất huyết hoặc nhồi máu	
I65	Tắc và/hoặc hẹp động mạch trước não, không dẫn đến nhồi máu não	
I65.0	Tắc và/hoặc hẹp động mạch đốt sống	
I65.1	Tắc và/hoặc hẹp động mạch nền	
I65.2	Tắc và/hoặc hẹp động mạch cảnh	
I65.3	Tắc và/hoặc hẹp nhiều động mạch và/hoặc động mạch trước não hai bên	
I65.8	Tắc và/hoặc hẹp của động mạch trước não khác	
I65.9	Tắc và/hoặc hẹp của động mạch trước não không xác định	Động mạch trước não không xác định khác
I66	Tắc và/hoặc hẹp động mạch não, không dẫn đến nhồi máu não	
I66.0	Tắc và/hoặc hẹp động mạch não giữa	
I66.1	Tắc và/hoặc hẹp động mạch não trước	
I66.2	Tắc và/hoặc hẹp động mạch não sau	
I66.3	Tắc và/hoặc hẹp động mạch tiểu não	
I66.4	Tắc và/hoặc hẹp nhiều động mạch não hai bên	
I66.8	Tắc và/hoặc hẹp động mạch não khác	Tắc và hẹp động mạch xuyên
I66.9	Tắc và/hoặc hẹp động mạch não, không xác định	
I67	Bệnh mạch máu não khác	
I67.0	Tách thành động mạch não, không vỡ	
I67.1	Phình động mạch não, không vỡ	
I67.2	Xơ vữa động mạch não	Mảng xơ vữa của động mạch não
I67.3	Bệnh lý chất trắng não tiến triển do căn nguyên mạch máu	
I67.4	Bệnh lý não do tăng huyết áp	
I67.5	Bệnh Moyamoya	
I67.6	Huyết khối không sinh mủ của hệ tĩnh mạch nội sọ	
I67.7	Viêm động mạch não, không phân loại mục khác	
I67.8	Bệnh mạch máu não xác định khác	
I67.9	Bệnh mạch máu não, không xác định	
I68.*	Rối loạn mạch máu não do bệnh phân loại mục khác	
I68.0*	Bệnh lý mạch máu não do thoái hóa tinh bột (E85.-†)	
I68.1*	Viêm động mạch não do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
I68.2*	Viêm động mạch não do bệnh khác phân loại mục khác	
I68.8*	Rối loạn mạch máu não khác do bệnh phân loại mục khác	
I69	Di chứng bệnh mạch máu não	
I69.0	Di chứng xuất huyết dưới nhện	
I69.1	Di chứng xuất huyết não	
I69.2	Di chứng xuất huyết nội sọ không do chấn thương khác	
I69.3	Di chứng nhồi máu não	
I69.4	Di chứng đột quỵ, không xác định do xuất huyết hoặc nhồi máu	
I69.8	Di chứng bệnh mạch máu não khác và/hoặc không xác định	
I70	Xơ vữa [xơ cứng] động mạch	
I70.0	Xơ vữa [xơ cứng] động mạch chủ	
I70.00	Xơ vữa [xơ cứng] động mạch chủ, không kèm hoại thư	
I70.01	Xơ vữa [xơ cứng] động mạch chủ, kèm hoại thư	
I70.1	Xơ vữa [xơ cứng] động mạch thận	
I70.10	Xơ vữa [xơ cứng] động mạch thận, không kèm hoại thư	
I70.11	Xơ vữa [xơ cứng] động mạch thận, kèm hoại thư	
I70.2	Xơ vữa [xơ cứng] động mạch ngoại vi [tứ chi]	
I70.20	Xơ vữa [xơ cứng] động mạch ngoại vi [tứ chi], không kèm hoại thư	
I70.21	Xơ vữa [xơ cứng] động mạch ngoại vi [tứ chi], kèm hoại thư	
I70.8	Xơ vữa [xơ cứng] động mạch khác	
I70.80	Xơ vữa [xơ cứng] động mạch khác, không kèm hoại thư	
I70.81	Xơ vữa [xơ cứng] động mạch khác, kèm hoại thư	
I70.9	Xơ vữa động mạch toàn thể và/hoặc xơ vữa động mạch không xác định	
I70.90	Xơ vữa động mạch toàn thể và/hoặc xơ vữa động mạch không xác định, không kèm hoại thư	
I70.91	Xơ vữa động mạch toàn thể và/hoặc xơ vữa động mạch không xác định, kèm hoại thư	
I71	Phình và/hoặc tách thành động mạch chủ	Phình và tách thành động mạch chủ
I71.0	Tách thành động mạch chủ [bất kỳ đoạn nào]	
I71.1	Phình vỡ động mạch chủ ngực	
I71.2	Phình động mạch chủ ngực, không vỡ	
I71.3	Phình vỡ động mạch chủ bụng	
I71.4	Phình động mạch chủ bụng, không vỡ	
I71.5	Phình vỡ động mạch chủ ngực - bụng	
I71.6	Phình động mạch chủ ngực - bụng, không vỡ	
I71.8	Phình vỡ động mạch chủ, vị trí không xác định	Vỡ động mạch chủ không xác định khác
I71.9	Phình động mạch chủ vị trí không xác định, không vỡ	Phình động mạch chủ|Giãn động mạch chủ|Hoại tử màng trong động mạch chủ
I72	Phình và/hoặc tách động mạch khác	
I72.0	Phình và/hoặc tách động mạch cảnh	
I72.1	Phình và/hoặc tách động mạch chi trên	
I72.2	Phình và/hoặc tách động mạch thận	
I72.3	Phình và/hoặc tách động mạch chậu	
I72.4	Phình và/hoặc tách động mạch chi dưới	
I72.5	Phình và/hoặc tách động mạch nền (thân)	
I72.6	Phình và/hoặc tách động mạch đốt sống	
I72.8	Phình và/hoặc tách động mạch xác định khác	
I72.9	Phình và/hoặc tách động mạch, vị trí không xác định	
I73	Bệnh mạch máu ngoại vi khác	
I73.0	Hội chứng Raynaud	
I73.1	Viêm tắc mạch huyết khối [Buerger]	
I73.8	Bệnh mạch máu ngoại vi xác định khác	Chứng xanh tím đầu chi
I73.9	Bệnh mạch máu ngoại vi, không xác định	Đau cách hồi|Cơ thắt động mạch
I74	Thuyên tắc và/hoặc huyết khối động mạch	
I74.0	Thuyên tắc và/hoặc huyết khối động mạch chủ bụng	Hội chứng chạc ba động mạch chủ|Hội chứng Leriche
I74.1	Thuyên tắc và/hoặc huyết khối, đoạn động mạch chủ khác và/hoặc đoạn không xác định của động mạch chủ	
I74.2	Thuyên tắc và/hoặc huyết khối động mạch chi trên	
I74.3	Thuyên tắc và/hoặc huyết khối động mạch chi dưới	
I74.4	Thuyên tắc và/hoặc huyết khối động mạch chi, không xác định	Thuyên tắc động mạch ngoại vi
I74.5	Thuyên tắc và/hoặc huyết khối động mạch chậu	
I74.8	Thuyên tắc và/hoặc huyết khối động mạch khác	
I74.9	Thuyên tắc và/hoặc huyết khối động mạch, không xác định	
I77	Bệnh khác của hệ động mạch và/hoặc tiểu động mạch	
I77.0	Rò động tĩnh mạch, mắc phải	
I77.1	Co hẹp động mạch	
I77.2	Vỡ động mạch	
I77.3	Loạn sản sợi cơ của động mạch	
I77.4	Hội chứng chèn ép động mạch tạng	
I77.5	Hoại tử động mạch	
I77.6	Viêm động mạch, không xác định	
I77.8	Rối loạn xác định khác của động mạch và/hoặc tiểu động mạch	Trợt động mạch|Loét động mạch
I77.9	Rối loạn của động mạch và/hoặc tiểu động mạch, không xác định	
I78	Bệnh về mao mạch	
I78.0	Giãn mao mạch xuất huyết di truyền	Bệnh Rendu-Osles-Weber
I78.1	Nơ vi do căn nguyên mạch máu, không tân sinh	
I78.8	Bệnh khác của mao mạch	
I78.9	Bệnh khác của mao mạch, không xác định	
I79.*	Rối loạn động mạch, tiểu động mạch và/hoặc mao mạch do bệnh phân loại mục khác	
I79.0*	Phình động mạch chủ do bệnh phân loại mục khác	
I79.1*	Viêm động mạch chủ do bệnh phân loại mục khác	
I79.2*	Bệnh lý mạch máu ngoại vi do bệnh phân loại mục khác	
I79.8*	Rối loạn khác của hệ động mạch, tiểu động mạch và/hoặc mao mạch do bệnh phân loại mục khác	
I80	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối	
I80.0	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối ở tĩnh mạch nông chi dưới	
I80.1	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyêt khối ở tĩnh mạch đùi	
I80.2	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối của tĩnh mạch sâu khác ở chi dưới	Huyết khối tĩnh mạch sâu không xác định khác
I80.3	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối ở chi dưới, không xác định	Thuyên tắc hoặc huyết khối ở chi dưới không xác định khác
I80.8	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối ở vị trí khác	
I80.9	Viêm tĩnh mạch và/hoặc viêm [tắc] tĩnh mạch huyết khối ở vị trí không xác định	
I81	Huyết khối tĩnh mạch cửa	
I82	Thuyên tắc và/hoặc huyết khối tĩnh mạch khác	
I82.0	Hội chứng Budd-Chiari	
I82.1	Viêm tĩnh mạch huyết khối di chuyển	
I82.2	Thuyên tắc và/hoặc huyết khối tĩnh mạch chủ	
I82.3	Thuyên tắc và/hoặc huyết khối tĩnh mạch thận	
I82.8	Thuyên tắc và/hoặc huyết khối tĩnh mạch xác định khác	
I82.9	Thuyên tắc và/hoặc huyết khối, không xác định tĩnh mạch	
I83	Giãn tĩnh mạch chi dưới	
I83.0	Giãn tĩnh mạch chi dưới kèm loét	
I83.1	Giãn tĩnh mạch chi dưới kèm viêm	Bất kỳ tình trạng não trong I83.9 có viêm hay xác định như viêm|Viêm da ứ nước không xác định khác
I83.2	Giãn tĩnh mạch chi dưới có cả loét và viêm	Bất kỳ tình trạng nào trong I83.9 cả loét và viêm
I83.9	Giãn tĩnh mạch chi dưới không loét hoặc viêm	Giãn tĩnh mạch của chi dưới [bất kỳ đoạn nào] hay vị trí không xác định|Tĩnh mạch giãn của chi dưới [bất kỳ đoạn nào] hay vị trí không xác định|Giãn mạch của chi dưới [bất kỳ đoạn nào] hay vị trí không xác định
I85	Giãn tĩnh mạch thực quản	
I85.0	Giãn tĩnh mạch thực quản kèm chảy máu	
I85.9	Giãn tĩnh mạch thực quản không kèm chảy máu	Giãn tĩnh mạch thực quản không xác định khác
I86	Giãn tĩnh mạch vị trí khác	
I86.0	Giãn tĩnh mạch dưới lưỡi	
I86.1	Giãn tĩnh mạch bìu	Giãn tĩnh mạch thừng tinh
I86.2	Giãn tĩnh mạch chậu	
I86.3	Giãn tĩnh mạch âm hộ	
I86.4	Giãn tĩnh mạch dạ dày	
I86.8	Giãn tĩnh mạch ở vị trí xác định khác	Loét biến dạng vách ngăn mũi
I87	Rối loạn khác của tĩnh mạch	
I87.0	Hội chứng sau huyết khối	Hội chứng sau viêm tĩnh mạch
I87.1	Ép tĩnh mạch	
I87.2	Suy tĩnh mạch (mạn tính) (ngoại vi)	
I87.8	Rối loạn xác định khác của tĩnh mạch	
I87.9	Rối loạn tĩnh mạch, không xác định	
I88	Viêm hạch bạch huyết không xác định cụ thể	Viêm hạch bạch huyết nguyên phát [vô căn]
I88.0	Viêm hạch bạch huyết mạc treo không xác định cụ thể	
I88.1	Viêm hạch bạch huyết mạn tính, ngoại trừ mạc treo	Viêm hạch mạn tính, bất kỳ hạch bạch huyết nào, ngoại trừ mạc treo|Viêm hạch bạch huyết mạn tính, bất kỳ hạch bạch huyết nào, ngoại trừ mạc treo
I88.8	Viêm hạch bạch huyết khác không xác định cụ thể	
I88.9	Viêm hạch bạch huyết không xác định cụ thể, không xác định loại	Viêm hạch bạch huyết không xác định khác
I89	Rối loạn mạch bạch huyết và/hoặc hạch bạch huyết khác không do nhiễm trùng	
I89.0	Phù bạch huyết, không phân loại mục khác	Giãn mạch bạch huyết
I89.1	Viêm mạch bạch huyết	
I89.8	Rối loạn mạch bạch huyết và/hoặc hạch bạch huyết xác định khác không do nhiễm trùng	
I89.9	Rối loạn mạch bạch huyết và/hoặc hạch bạch huyết không do nhiễm trùng, không xác định	Bệnh mạch bạch huyết không xác định khác
I95	Huyết áp thấp (hạ huyết áp)	
I95.0	Hạ huyết áp vô căn	
I95.1	Hạ huyết áp tư thế đứng	
I95.2	Hạ huyết áp do thuốc	
I95.8	Hạ huyết áp khác	Hạ huyết áp mạn tính
I95.9	Hạ huyết áp, không xác định	
I97	Rối loạn hệ tuần hoàn sau can thiệp, không phân loại mục khác	
I97.0	Hội chứng sau phẫu thuật tim	
I97.1	Rối loạn chức năng khác sau phẫu thuật tim	Suy tim sau phẫu thuật tim hoặc do có mặt thiết bị thay thế tim
I97.2	Hội chứng phù hạch bạch huyết sau cắt bỏ tuyến vú	Bệnh phù chân voi do cắt bỏ tuyến vú|Tắc mạch bạch huyết do cắt bỏ tuyến vú
I97.8	Rối loạn hệ tuần hoàn khác sau can thiệp, không phân loại mục khác	
I97.9	Rối loạn hệ tuần hoàn sau can thiệp, không xác định	
I98.*	Rối loạn khác của hệ tuần hoàn do bệnh phân loại mục khác	
I98.0*	Giang mai tim mạch	
I98.1*	Rối loạn tim mạch do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác	
I98.2*	Giãn tĩnh mạch thực quản không chảy máu do bệnh phân loại mục khác	
I98.3*	Giãn tĩnh mạch thực quản chảy máu do bệnh phân loại mục khác	
I98.8*	Rối loạn hệ tuần hoàn xác định khác do bệnh phân loại mục khác	
I99	Rối loạn hệ tuần hoàn khác và/hoặc không xác định	
J00	Viêm mũi họng cấp tính [cảm thường]	
J01	Viêm xoang cấp tính	
J01.0	Viêm xoang hàm cấp tính	
J01.1	Viêm xoang trán cấp tính	
J01.2	Viêm xoang sàng cấp tính	
J01.3	Viêm xoang bướm cấp tính	
J01.4	Viêm toàn bộ xoang cấp tính	
J01.8	Viêm xoang cấp tính khác	Viêm xoang cấp ảnh hưởng nhiều hơn một xoang nhưng không viêm toàn bộ các xoang
J01.9	Viêm xoang cấp tính, không xác định	
J02	Viêm họng cấp tính	
J02.0	Viêm họng do liên cầu khuẩn [Streptococcus]	
J02.8	Viêm họng cấp tính do vi sinh vật xác định khác	
J02.9	Viêm họng cấp tính, không xác định	
J03	Viêm amydan cấp tính	
J03.0	Viêm amydan do liên cầu khuẩn [Streptococcus]	
J03.8	Viêm amydan cấp tính do vi sinh vật xác định khác	
J03.9	Viêm amydan cấp tính, không xác định	
J04	Viêm thanh quản và/hoặc khí quản cấp tính	
J04.0	Viêm thanh quản cấp tính	
J04.1	Viêm khí quản cấp tính	
J04.2	Viêm thanh khí quản cấp tính	
J05	Viêm thanh quản tắc nghẽn [tắc nghẽn thanh quản] và/hoặc nắp thanh môn cấp tính	
J05.0	Viêm thanh quản tắc nghẽn cấp tính [yết hầu]	Viêm thanh quản tắc nghẽn không xác định khác
J05.1	Viêm nắp thanh môn cấp tính	Viêm nắp thanh môn không xác định khác
J06	Nhiễm trùng đường hô hấp trên cấp tính ở nhiều vị trí và/hoặc vị trí không xác định	
J06.0	Viêm họng - thanh quản cấp tính	
J06.8	Nhiễm trùng đường hô hấp trên cấp tính khác ở nhiều vị trí	
J06.9	Nhiễm trùng đường hô hấp trên cấp tính, không xác định	
J09	Cúm do virus cúm đã xác định truyền từ động vật hoặc đại dịch	
J10	Cảm cúm do virus cúm mùa đã xác định	
J10.0	Cảm cúm kèm viêm phổi, đã xác định được virus cúm mùa	
J10.1	Cúm với biểu hiện hô hấp khác, đã xác định được virus cúm mùa	Do virus cúm mùa đã xác định|nhiễm trùng đường hô hấp trên cấp tính do virus cúm mùa đã xác định|viêm thanh quản do virus cúm mùa đã xác định|viêm họng do virus cúm mùa đã xác định|tràn dịch màng phổi do virus cúm mùa đã xác định
J10.8	Cảm cúm với biểu hiện khác, đã xác định được virus cúm mùa	
J11	Cúm, không xác định được virus	
J11.0	Cúm kèm viêm phổi, không xác định được virus	
J11.1	Cúm có biểu hiện hô hấp khác, không xác định được virus	Cúm không xác định khác|Do cúm: không xác định hoặc không định danh virus cụ thể|nhiễm trùng hô hấp trên cấp không xác định hoặc không định danh virus cụ thể|viêm thanh quản không xác định hoặc không định danh virus cụ thể|viêm họng không xác định hoặc không định danh virus cụ thể|tràn dịch màng phổi không xác định hoặc không định danh virus cụ thể
J11.8	Cúm có biểu hiện khác, không xác định được virus	
J12	Viêm phổi do virus, không phân loại mục khác	
J12.0	Viêm phổi do virus adeno	
J12.1	Viêm phổi do virus hợp bào hô hấp [RSV]	
J12.2	Viêm phổi do virus parainfluenza	
J12.3	Viêm phổi do virus gây bệnh đường hô hấp ở người [metapneumovirus]	
J12.8	Viêm phổi do virus khác	
J12.9	Viêm phổi do virus, không xác định	
J13	Viêm phổi do vi khuẩn phế cầu khuẩn [Streptococcus pneumoniae]	
J14	Viêm phổi do vi khuẩn Haemophilus cúm [H. influenzae]	
J15	Viêm phổi do vi khuẩn, không phân loại mục khác	
J15.0	Viêm phổi do Klebsiella pneumoniae	
J15.1	Viêm phổi do pseudomonas	
J15.2	Viêm phổi do tụ cầu khuẩn [staphylococcus]	
J15.3	Viêm phổi do liên cầu khuẩn [streptoccoccus], nhóm B	
J15.4	Viêm phổi do liên cầu khuẩn [streptoccoccus] khác	
J15.5	Viêm phổi do Escherichia coli	
J15.6	Viêm phổi do vi khuẩn gram âm (-) khác	
J15.7	Viêm phổi do Mycoplasma pneumoniae	
J15.8	Viêm phổi do vi khuẩn khác	
J15.9	Viêm phổi do vi khuẩn, không xác định	
J16	Viêm phổi do vi sinh vật truyền nhiễm khác, không phân loại mục khác	
J16.0	Viêm phổi do nhiễm chlamydia	
J16.8	Viêm phổi do nhiễm vi sinh vật xác định khác	
J17.*	Viêm phổi ở bệnh phân loại mục khác	
J17.0*	Viêm phổi ở bệnh do vi khuẩn phân loại mục khác	
J17.1*	Viêm phổi ở bệnh do virus phân loại mục khác	
J17.2*	Viêm phổi do nhiễm nấm	
J17.3*	Viêm phổi ở bệnh ký sinh trùng	
J17.8*	Viêm phổi ở bệnh khác phân loại mục khác	
J18	Viêm phổi, không xác định vi sinh vật	
J18.0	Viêm phế quản phổi, không xác định vi sinh vật	
J18.1	Viêm phổi thùy, không xác định vi sinh vật	
J18.2	Viêm phổi do nằm lâu ngày ở một tư thế, không xác định vi sinh vật	
J18.8	Viêm phổi khác, vi sinh vật không xác định	
J18.9	Viêm phổi, không xác định	
J20	Viêm phế quản cấp tính	Viêm phế quản cấp
J20.0	Viêm phế quản cấp tính do Mycoplasma pneumoniae	
J20.1	Viêm phế quản cấp tính do Haemophilus cúm [H. influenzae]	
J20.2	Viêm phế quản cấp tính do liên cầu khuẩn [streptococcus]	
J20.3	Viêm phế quản cấp tính do virus coxsackie	
J20.4	Viêm phế quản cấp tính do virus parainfluenza	
J20.5	Viêm phế quản cấp tính do virus hợp bào hô hấp	
J20.6	Viêm phế quản cấp tính do rhinovirus	
J20.7	Viêm phế quản cấp tính do echovirus	
J20.8	Viêm phế quản cấp tính do vi sinh vật khác đã được xác định	
J20.9	Viêm phế quản cấp tính, không xác định	
J21	Viêm tiểu phế quản cấp tính	
J21.0	Viêm tiểu phế quản cấp tính do virus hợp bào hô hấp	
J21.1	Viêm tiểu phế quản cấp tính do virus gây bệnh đường hô hấp ở người [metapneumovirus]	
J21.8	Viêm tiểu phế quản cấp do vi sinh vật xác định khác	
J21.9	Viêm tiểu phế quản cấp tính, không xác định	
J22	Nhiễm trùng hô hấp dưới cấp tính không xác định	
J30	Viêm mũi vận mạch và/hoặc viêm mũi dị ứng	
J30.0	Viêm mũi vận mạch	
J30.1	Viêm mũi dị ứng do phấn hoa	Dị ứng phấn hoa không xác định khác|Sốt do dị ứng phấn hoa [viêm mũi dị ứng]|Bệnh dị ứng phấn hoa
J30.2	Viêm mũi dị ứng theo mùa khác	
J30.3	Viêm mũi dị ứng khác	Viêm mũi dị ứng quanh năm
J30.4	Viêm mũi dị ứng, không xác định	
J31	Viêm mũi, viêm mũi họng và/hoặc viêm họng mạn tính	
J31.0	Viêm mũi mạn tính	
J31.1	Viêm mũi họng mạn tính	
J31.2	Viêm họng mạn tính	
J32	Viêm xoang mạn tính	
J32.0	Viêm xoang hàm mạn tính	
J32.1	Viêm xoang trán mạn tính	Viêm xoang trán không xác định khác
J32.2	Viêm xoang sàng mạn tính	Viêm xoang sàng không xác định khác
J32.3	Viêm xoang bướm mạn tính	Viêm xoang bướm mạn tính không xác định khác
J32.4	Viêm đa xoang mạn tính	Viêm toàn bộ xoang không xác định khác
J32.8	Viêm xoang mạn tính khác	
J32.9	Viêm xoang mạn tính, không xác định	
J33	Polyp mũi	
J33.0	Polyp khoang mũi	
J33.1	Thoái hóa xoang dạng polyp	Hội chứng Woakes hoặc viêm xoang sàng
J33.8	Polyp khác của xoang	
J33.9	Polyp mũi, không xác định	
J34	Rối loạn khác của mũi và/hoặc xoang	
J34.0	Áp xe, nhọt và/hoặc nhọt tiền đình mũi	
J34.1	U nang và/hoặc nang nhầy của mũi và/hoặc xoang mũi	
J34.2	Vẹo vách ngăn mũi	
J34.3	Phì đại [quá phát] cuốn mũi	
J34.8	Rối loạn xác định khác của mũi và/hoặc xoang	Thủng vách mũi không xác định khác|Sỏi ở mũi
J35	Bệnh mạn tính của amydan và khối mô lympho ở vòm họng [VA]	
J35.0	Viêm amydan mạn tính	
J35.1	Phì đại amydan	
J35.2	Phì đại khối mô lympho ở vòm họng [VA]	VA lớn
J35.3	Phì đại amydan kèm phì đại khối mô lympho ở vòm họng [VA]	
J35.8	Bệnh mạn tính khác của amydan và/hoặc khối mô lympho ở vòm họng [VA]	
J35.9	Bệnh mạn tính của amydan và/hoặc khối mô lympho ở vòm họng [VA], không xác định	
J36	Áp xe quanh amydan	
J37	Viêm thanh quản và/hoặc viêm thanh khí quản mạn tính	
J37.0	Viêm thanh quản mạn tính	
J37.1	Viêm thanh khí quản mạn tính	
J38	Bệnh của dây thanh âm và/hoặc thanh quản, không phân loại mục khác	
J38.0	Liệt dây thanh âm và/hoặc thanh quản	Liệt thanh quản|Liệt thanh môn
J38.1	Polyp của dây thanh âm và/hoặc thanh quản	
J38.2	Hạt xơ dây thanh	
J38.3	Bệnh lý khác của dây thanh âm	
J38.4	Phù thanh quản	
J38.5	Co thắt thanh quản	
J38.6	Hẹp thanh quản	
J38.7	Bệnh khác của thanh quản	Áp xe thanh quản|Viêm mô tế bào thanh quản|Bệnh thanh quản không xác định khác|Hoại tử thanh quản|Dày thanh quản|Viêm màng sụn thanh quản|Loét thanh quản
J39	Bệnh khác của đường hô hấp trên	
J39.0	Áp xe sau họng và/hoặc áp xe cận họng	
J39.1	Áp xe khác của họng	Viêm mô tế bào ở họng|Áp xe mũi họng
J39.2	Bệnh khác của họng	
J39.3	Phản ứng quá mẫn đường hô hấp trên, vị trí không xác định	
J39.8	Bệnh xác định khác của đường hô hấp trên	
J39.9	Bệnh ở đường hô hấp trên, không xác định	
J40	Viêm phế quản, không xác định được là cấp tính hoặc mạn tính	
J41	Viêm phế quản mạn tính đơn thuần và/hoặc nhầy mủ	
J41.0	Viêm phế quản mạn tính đơn thuần	
J41.1	Viêm phế quản mạn tính nhầy mủ	
J41.8	Viêm phế quản mạn tính hỗn hợp (đơn thuần và nhầy mủ)	
J42	Viêm phế quản mạn tính không xác định	
J43	Khí phế thũng [khí thũng phổi]	
J43.0	Hội chứng MacLeod	
J43.1	Khí phế thũng [khí thũng phổi] toàn tiểu thùy	Khí phế thũng [khí thũng phổi] toàn bộ tiểu thùy
J43.2	Khí phế thũng [khí thũng phổi] trung tâm tiểu thùy	
J43.8	Khí phế thũng [khí thũng phổi] khác	
J43.9	Khí phế thũng [khí thũng phổi], không xác định	
J44	Bệnh phổi tắc nghẽn mạn tính khác	
J44.0	Bệnh phổi tắc nghẽn mạn tính kèm nhiễm trùng đường hô hấp dưới cấp tính	
J44.1	Bệnh phổi tắc nghẽn mạn tính đợt cấp, không xác định	
J44.8	Bệnh phổi tắc nghẽn mạn tính xác định khác	
J44.9	Bệnh phổi tắc nghẽn mạn tính, không xác định	
J45	Hen phế quản [hen suyễn] [hen]	
J45.0	Hen phế quản [hen suyễn] [hen] chủ yếu do dị ứng	
J45.1	Hen phế quản [hen suyễn] [hen] không dị ứng	Hen phế quản đặc ứng|Hen phế quản nội sinh không dị ứng
J45.8	Hen phế quản [hen suyễn] [hen] hỗn hợp	Phối hợp các bệnh lý liệt kê ở J45.0 và J54.1
J45.9	Hen phế quản [hen suyễn] [hen], không xác định	Viêm phế quản dạng hen không xác định khác|Hen phế quản khởi phát muộn
J46	Cơn hen phế quản ác tính	
J47	Giãn phế quản	
J60	Bệnh bụi phổi than nghề nghiệp	
J61	Bệnh bụi phổi amiăng và/hoặc sợi khoáng khác	
J62	Bệnh bụi phổi silic	
J62.0	Bệnh bụi phổi talc	
J62.8	Bệnh bụi phổi silic khác	Bệnh bụi phổi silic không xác định khác
J63	Bệnh bụi phổi do bụi vô cơ khác	
J63.0	Nhiễm bụi nhôm (ở phổi)	
J63.1	Bệnh xơ phổi do quặng bô-xít	
J63.2	Bệnh bụi phổi do beryllium	
J63.3	Bệnh xơ phổi do than chì	
J63.4	Bệnh bụi phổi sắt	
J63.5	Bệnh bụi phổi thiếc	
J63.8	Bệnh phổi nghề nghiệp do bụi phổi vô cơ xác định khác	
J64	Bệnh bụi phổi không xác định	
J65	Bệnh bụi phổi liên quan đến bệnh lao	
J66	Bệnh đường thở do bụi hữu cơ cụ thể	Bệnh đường thở do bụi hữu cơ cụ thể khác
J66.0	Bệnh bụi phổi bông	Bệnh đường thở do bụi bông
J66.1	Bệnh phổi Flax-Dresser [bệnh phổi tắc nghẽn mạn tính do hít phải hạt lanh chưa qua chế biến]	
J66.2	Bệnh bụi phổi do hít phải bụi trong quá trình chế biến cây gai dầu	
J66.8	Bệnh đường thở do bụi hữu cơ cụ thể khác	
J67	Viêm phổi quá mẫn do bụi hữu cơ	
J67.0	Bệnh phổi của người nông dân	Bệnh phổi của người thu hoạch ngũ cốc|Bệnh phổi của người cắt cỏ|Bệnh dị ứng do cỏ khô bị mốc
J67.1	Bệnh phổi do bã mía	
J67.2	Bệnh phổi ở người nuôi chim	
J67.3	Bệnh phổi do xơ cây bần	Bệnh bụi phổi quá mẫn do tiếp xúc với bụi bần mốc
J67.4	Bệnh phổi của công nhân tiếp xúc với bào tử nấm từ mạch nha bị mốc	Viêm phế nang do nhiễm vi nấm Aspergillus clavatus
J67.5	Bệnh bụi phổi của công nhân trồng nấm	
J67.6	Bệnh bụi phổi của công nhân bóc vỏ cây thích [cây phong]	Viêm phế nang do Cryptostroma corticale|Nhiễm Cryptostroma
J67.7	Bệnh phổi do máy làm ẩm và/hoặc máy điều hoà không khí	
J67.8	Viêm phổi quá mẫn do bụi hữu cơ khác	Bệnh phổi của công nhân tiếp xúc với pho mát bị mốc|Bệnh phổi của công nhân rang cà phê|Bệnh phổi của công nhân tiếp xúc với bột cá dành để nuôi động vật|Bệnh phổi của người tiếp xúc với lông động vật|Bệnh phổi do tiếp xúc với mạt cưa gỗ đỏ bị mốc
J67.9	Viêm phổi quá mẫn do bụi hữu cơ không xác định	
J68	Bệnh hô hấp do hít hóa chất, khí, khói và/hoặc hơi	
J68.0	Viêm phế quản và/hoặc viêm phổi do hóa chất, chất khí, khói và/hoặc hơi	
J68.1	Phù phổi do hóa chất, khí, khói và/hoặc hơi	
J68.2	Viêm đường hô hấp trên do hóa chất, khí, khói và/hoặc chất bay hơi, không phân loại mục khác	
J68.3	Bệnh hô hấp cấp tính và/hoặc bán cấp tính khác do hóa chất, khí, khói và/hoặc hơi	Hội chứng rối loạn chức năng phản ứng của đường dẫn khí
J68.4	Bệnh hô hấp mạn tính do hóa chất, khí, khói và/hoặc hơi	
J68.8	Bệnh hô hấp khác do hóa chất, khí, khói và/hoặc hơi	
J68.9	Bệnh hô hấp không xác định do hóa chất, khí, khói và/hoặc hơi	
J69	Viêm phổi do chất rắn và/hoặc chất lỏng	
J69.0	Viêm phổi hít phải thức ăn và/hoặc chất nôn	
J69.1	Viêm phổi hít phải dầu và/hoặc hương liệu	Viêm phổi hít phải chất béo
J69.8	Viêm phổi hít phải chất rắn và/hoặc chất lỏng khác	Viêm phổi hít phải máu
J70	Bệnh hô hấp do tác nhân bên ngoài khác	
J70.0	Biểu hiện cấp tính ở phổi do xạ trị	Viêm phổi do xạ trị
J70.1	Biểu hiện mạn tính và biểu hiện khác ở phổi do xạ trị	Xơ phổi do xạ trị
J70.2	Rối loạn phổi mô kẽ cấp tính do thuốc	
J70.3	Rối loạn phổi mô kẽ mạn tính do thuốc	
J70.4	Rối loạn phổi mô kẽ do thuốc, không xác định	
J70.8	Bệnh lý hô hấp do tác nhân bên ngoài xác định khác	
J70.9	Bệnh lý hô hấp do tác nhân bên ngoài không xác định	
J80	Hội chứng suy hô hấp ở người lớn	
J81	Phù phổi	
J82	Tăng bạch cầu ái toan ở phổi, không phân loại mục khác	
J84	Bệnh phổi mô kẽ khác	
J84.0	Bệnh lý phế nang và/hoặc thành phế nang	Tích tụ protein ở phế nang|Sạn [sỏi nhỏ] ở phế nang phổi
J84.1	Bệnh phổi mô kẽ khác kèm xơ phổi	
J84.8	Bệnh phổi mô kẽ xác định khác	
J84.9	Bệnh phổi mô kẽ, không xác định	Bệnh phổi mô kẽ không xác định khác
J85	Áp xe phổi và/hoặc trung thất	
J85.0	Hoại thư và/hoặc hoại tử ở phổi	
J85.1	Áp xe phổi kèm viêm phổi	
J85.2	Áp xe phổi không kèm viêm phổi	Áp xe phổi không xác định khác
J85.3	Áp xe trung thất	
J86	Viêm mủ lồng ngực	
J86.0	Viêm mủ lồng ngực có lỗ rò	
J86.9	Viêm mủ lồng ngực không có lỗ rò	
J90	Tràn dịch màng phổi, không phân loại mục khác	
J91.*	Tràn dịch màng phổi do bệnh phân loại mục khác	
J92	Mảng xơ cứng màng phổi	
J92.0	Mảng xơ cứng màng phổi có amiăng	
J92.9	Mảng xơ cứng màng phổi không có amiăng	Mảng xơ cứng màng phổi không xác định khác
J93	Tràn khí màng phổi	
J93.0	Tràn khí màng phổi áp lực tự phát	
J93.1	Tràn khí màng phổi tự phát khác	
J93.8	Tràn khí màng phổi khác	
J93.9	Tràn khí màng phổi, không xác định	
J94	Bệnh màng phổi khác	
J94.0	Tràn dịch dưỡng chấp	
J94.1	Xơ hóa màng phổi	
J94.2	Tràn máu màng phổi	Tràn máu và khí màng phổi
J94.8	Bệnh màng phổi xác định khác	Tràn dịch ngực [tích dịch màng phổi]
J94.9	Bệnh màng phổi, không xác định	
J95	Bệnh hô hấp sau can thiệp, không phân loại mục khác	
J95.0	Biến chứng lỗ mở khí quản	Chảy máu tại chỗ mở khí quản|Tắc đường thở sau mở khí quản|Nhiễm trùng hệ thống chỗ mở khí quản|Rò khí quản - thực quản sau mở khí quản
J95.1	Suy phổi cấp tính sau phẫu thuật lồng ngực	
J95.2	Suy phổi cấp tính sau phẫu thuật không phải ở lồng ngực	
J95.3	Suy phổi mạn tính sau phẫu thuật	
J95.4	Hội chứng Mendelson	
J95.5	Hẹp dưới thanh môn sau can thiệp	
J95.8	Rối loạn hô hấp khác sau can thiệp	
J95.9	Rối loạn hô hấp sau can thiệp, không xác định	
J96	Suy hô hấp [tiến triển], không phân loại mục khác	Suy hô hấp, không phân loại mục khác
J96.0	Suy hô hấp [tiến triển] cấp tính	
J96.00	Suy hô hấp [tiến triển] cấp tính, típ I [giảm oxy máu]	
J96.01	Suy hô hấp [tiến triển] cấp tính, típ II [tăng CO2 máu]	
J96.09	Suy hô hấp [tiến triển] cấp tính, típ không xác định	
J96.1	Suy hô hấp [tiến triển] mạn tính	
J96.10	Suy hô hấp [tiến triển] mạn tính, típ I [giảm oxy máu]	
J96.11	Suy hô hấp [tiến triển] mạn tính, típ II [tăng CO2 máu]	
J96.19	Suy hô hấp [tiến triển] mạn tính, típ không xác định	
J96.9	Suy hô hấp [tiến triển], tính không xác định	
J96.90	Suy hô hấp [tiến triển], tính không xác định, típ I [giảm oxy máu]	
J96.91	Suy hô hấp [tiến triển], tính không xác định, típ II [tăng CO2 máu]	
J96.99	Suy hô hấp [tiến triển], tính không xác định, típ không xác định	
J98	Rối loạn hô hấp khác	
J98.0	Bệnh phế quản, không phân loại mục khác	Sỏi phế quản|Vôi hóa phế quản|Hẹp phế quản|Loét phế quản
J98.1	Xẹp phổi	
J98.2	Giãn phế nang mô kẽ	
J98.3	Giãn phế nang còn bù	
J98.4	Rối loạn khác của phổi	
J98.5	Bệnh của trung thất, không phân loại mục khác	
J98.6	Rối loạn cơ hoành	
J98.7	Nhiễm trùng hô hấp, không phân loại mục khác	
J98.8	Rối loạn hô hấp xác định khác	
J98.9	Rối loạn hô hấp, không xác định	
J99.*	Rối loạn hô hấp do bệnh phân loại mục khác	
J99.0*	Bệnh phổi dạng thấp (M05.1†)	
J99.1*	Rối loạn hô hấp do bệnh mô liên kết lan tỏa khác	
J99.8*	Rối loạn hô hấp do bệnh khác phân loại mục khác	
K00	Rối loạn phát triển răng và/hoặc mọc răng	
K00.0	Tật không răng bẩm sinh	Chứng thiếu ít răng [thiếu từ 1 đến 5 răng]|Chứng thiếu nhiều răng [thiếu từ 6 răng trở lên]
K00.1	Răng thừa	Răng cối xa|Răng cối thứ tư|Răng kẽ giữa|Răng cối thừa|Răng bù trừ [răng dư]
K00.2	Bất thường kích thước và/hoặc hình dạng răng	
K00.3	Răng đốm màu	
K00.4	Rối loạn tạo răng	
K00.5	Rối loạn di truyền cấu trúc răng, không phân loại mục khác	Bệnh tạo men răng bất toàn|Bệnh tạo ngà răng bất toàn|Bệnh tạo răng bất toàn|Loạn sinh ngà|Răng vỏ sò
K00.6	Rối loạn mọc răng	Răng mọc sớm|Răng lúc sinh|Răng sơ sinh
K00.7	Hội chứng mọc răng	
K00.8	Rối loạn khác về phát triển răng	Biến đổi màu trong quá trình tạo răng|Nhuộm màu răng do yếu tố nội sinh không xác định khác
K00.9	Rối loạn phát triển răng, không xác định	Rối loạn tạo răng không xác định khác
K01	Răng mọc kẹt và/hoặc răng ngầm	
K01.0	Răng ngầm	Răng ngầm là răng không mọc được, không có cản trở của răng khác.
K01.1	Răng mọc kẹt	Một răng mọc kẹt là răng không mọc được do cản trở của răng khác.
K02	Sâu răng	
K02.0	Sâu men răng	Tổn thương đốm trắng [sâu mới chớm]
K02.1	Sâu ngà răng	
K02.2	Sâu xi măng răng	
K02.3	Sâu răng ngưng tiến triển	
K02.4	Hủy khoáng mô cứng nhiều răng	Răng đen ở trẻ nhỏ|Hủy răng đen
K02.5	Sâu răng có hở tủy	
K02.8	Sâu răng khác	
K02.9	Sâu răng, không xác định	
K03	Bệnh mô cứng khác của răng	
K03.0	Mòn răng quá mức	
K03.1	Mòn răng cơ học	
K03.2	Mòn răng hóa học	
K03.3	Tiêu răng bệnh lý	
K03.4	Quá sản xi măng răng	Quá sản chất tạo răng
K03.5	Dính khớp răng	
K03.6	Cao răng	
K03.7	Đổi màu mô cứng sau mọc răng	
K03.8	Bệnh mô cứng xác định khác của răng	
K03.9	Bệnh mô cứng của răng, không xác định	
K04	Bệnh tủy và/hoặc mô quanh chân răng	
K04.0	Viêm tủy răng	
K04.1	Hoại tử tủy răng	Hoại thư tủy
K04.2	Thoái hóa tủy răng	Răng nhỏ
K04.3	Hình thành mô cứng bất thường trong tủy	Men răng thứ phát hoặc không đều
K04.4	Viêm quanh chóp răng cấp tính có nguồn gốc từ tủy răng	Viêm nha chu chóp răng cấp tính không xác định khác
K04.5	Viêm quanh chóp răng mạn tính	U hạt chóp răng và quanh chóp răng [răng cận chóp]|Viêm nha chu chóp răng không xác định khác
K04.6	Áp xe quanh chóp răng có lỗ rò	Áp xe răng kèm xoang|Áp xe ổ răng kèm xoang
K04.7	Áp xe quanh chóp răng không có lỗ rò	Áp xe răng không xác định khác|Áp xe ổ răng không xác định khác|Áp xe răng cận chóp không xác định khác
K04.8	Nang chân răng	
K04.9	Bệnh khác và/hoặc không xác định của tủy và/hoặc mô quanh chóp răng	
K05	Viêm lợi [nướu] và/hoặc bệnh viêm nha chu	
K05.0	Viêm lợi [nướu] cấp tính	
K05.1	Viêm lợi [nướu] mạn tính	
K05.2	Viêm quanh răng [nha chu] cấp tính	
K05.3	Viêm quanh răng [nha chu] mạn tính	Viêm quanh thân răng mạn tính
K05.4	Thoái hóa quanh răng [nha chu]	Thoái hóa nha chu ở thanh thiếu niên
K05.5	Bệnh quanh răng [nha chu] khác	
K05.6	Bệnh quanh răng [nha chu], không xác định	
K06	Rối loạn khác của lợi [nướu] và/hoặc sống hàm đã mất răng	
K06.0	Tụt lợi [nướu]	
K06.1	Phì đại lợi [nướu]	Bệnh u xơ lợi [nướu]
K06.2	Tổn thương lợi [nướu] răng và/hoặc sống hàm đã mất răng liên quan chấn thương	
K06.8	Bệnh lý xác định khác của lợi [nướu] và/hoặc sống hàm đã mất răng	U lợi [nướu] dạng xơ|Sống hàm di động|U nướu [lợi] tế bào khổng lồ|U hạt tế bào khổng lồ ngoại biên|U hạt sinh mủ ở nướu [lợi]
K06.9	Rối loạn ở lợi [nướu] và/hoặc sống hàm đã mất răng, không xác định	
K07	Bất thường hàm mặt [bao gồm khớp cắn lệch]	
K07.0	Bất thường lớn về kích thước xương hàm	
K07.1	Bất thường tương quan sọ mặt	
K07.2	Bất thường tương quan cung răng	
K07.3	Bất thường vị trí răng mặt	
K07.4	Khớp cắn lệch, không xác định	
K07.5	Bất thường chức năng răng mặt	
K07.6	Loạn năng khớp thái dương hàm	
K07.8	Bất thường răng mặt khác	
K07.9	Bất thường răng mặt, không xác định	
K08	Rối loạn khác của răng và/hoặc cấu trúc nâng đỡ	
K08.0	Bong tróc răng do nguyên nhân hệ thống	
K08.1	Mất răng do tai nạn, do nhổ răng hoặc do bệnh nha chu khu trú	
K08.2	Teo sống hàm đã mất răng	
K08.3	Chân răng còn sót	
K08.8	Rối loạn xác định khác của răng và/hoặc cấu trúc nâng đỡ	
K08.9	Rối loạn của răng và/hoặc cấu trúc nâng đỡ, không xác định	
K09	Nang vùng miệng, không phân loại mục khác	
K09.0	Nang nguồn gốc răng	
K09.1	Nang vùng miệng không có nguồn gốc răng	
K09.2	Nang khác của xương hàm	
K09.8	Nang khác ở vùng miệng, không phân loại mục khác	Nang dạng bì của miệng|Nang biểu bì của miệng|Nang biểu mô Lympho của miệng|Bệnh hạt kê bẩm sinh
K09.9	Nang của vùng miệng, không xác định	
K10	Bệnh khác của xương hàm	
K10.0	Rối loạn phát triển của xương hàm	Nang xương hàm tiềm ẩn|Nang Stafne
K10.1	U hạt tế bào khổng lồ, trung tâm	
K10.2	Tình trạng viêm của xương hàm	
K10.3	Viêm ổ răng của xương hàm	Viêm xương ổ răng|Khô ổ răng
K10.8	Bệnh xác định khác của xương hàm	Loạn sản xương hàm|Lồi xương hàm|Loạn sản xơ xương hàm
K10.9	Bệnh của xương hàm, không xác định	
K11	Bệnh tuyến nước bọt	
K11.0	Teo tuyến nước bọt	
K11.1	Phì đại tuyến nước bọt	
K11.2	Viêm tuyến nước bọt	
K11.3	Áp xe tuyến nước bọt	
K11.4	Lỗ rò tuyến nước bọt	
K11.5	Sỏi tuyến nước bọt	Bệnh sỏi của tuyến nước bọt hoặc ống dẫn nước bọt|Sỏi của tuyến tuyến nước bọt hoặc ống dẫn nước bọt
K11.6	Nang nhầy của tuyến nước bọt	
K11.7	Rối loạn tiết nước bọt	
K11.8	Bệnh khác của tuyến nước bọt	
K11.9	Bệnh của tuyến nước bọt, không xác định	Viêm tuyến nước bọt không xác định khác
K12	Viêm miệng và/hoặc tổn thương liên quan	
K12.0	Loét miệng tái phát	
K12.1	Trạng thái khác của viêm miệng	
K12.2	Viêm mô tế bào và/hoặc áp xe của miệng	
K12.3	Viêm niêm mạc miệng (loét)	
K13	Bệnh khác của môi và/hoặc niêm mạc miệng	
K13.0	Bệnh của môi	
K13.1	Tật cắn môi và/hoặc tật cắn má	
K13.2	Bạch sản [mảng trắng] và/hoặc thay đổi của niêm mạc miệng, bao gồm cả lưỡi	
K13.3	Bạch sản dạng tóc ở miệng	
K13.4	U hạt và/hoặc tổn thương dạng u hạt của niêm mạc miệng	U hạt nhiễm bạch cầu ái toan của niêm mạc miệng|U hạt sinh mủ của niêm mạc miệng|U dạng mụn cơm của niêm mạc miệng
K13.5	Xơ hóa dưới niêm mạc miệng	Xơ hóa dưới niêm mạc lưỡi
K13.6	Tăng sản do kích thích của niêm mạc miệng	
K13.7	Tổn thương khác và/hoặc không xác định của niêm mạc miệng	Bệnh viêm niêm mạc miệng khu trú
K14	Bệnh của lưỡi	
K14.0	Viêm lưỡi	
K14.1	Lưỡi bản đồ	Viêm lưỡi di chuyển lành tính|Viêm lưỡi loang
K14.2	Viêm lưỡi giữa hình trám	
K14.3	Phì đại gai lưỡi	Lưỡi lông đen|Lưỡi bựa|Phì đại gai lưỡi hình lá|Hội chứng lưỡi lông đen
K14.4	Teo gai lưỡi	Viêm teo lưỡi
K14.5	Lưỡi nứt	
K14.6	Đau lưỡi	Cảm giác bỏng lưỡi|Lưỡi đau
K14.8	Bệnh khác của lưỡi	
K14.9	Bệnh của lưỡi, không xác định	Bệnh lý lưỡi không xác định khác
K20	Viêm thực quản	
K21	Bệnh trào ngược dạ dày - thực quản	Bệnh trào ngược dạ dày-thực quản
K21.0	Bệnh trào ngược dạ dày - thực quản kèm viêm thực quản	Viêm thực quản do trào ngược
K21.9	Bệnh trào ngược dạ dày - thực quản không kèm viêm thực quản	Trào ngược thực quản không xác định khác
K22	Bệnh khác của thực quản	
K22.0	Co thắt tâm vị	
K22.1	Loét thực quản	
K22.2	Tắc nghẽn thực quản	
K22.3	Thủng thực quản	
K22.4	Rối loạn vận động thực quản	
K22.5	Túi thừa thực quản, mắc phải	
K22.6	Hội chứng rách - chảy máu dạ dày - thực quản	Hội chứng Mallory Weiss
K22.7	Thực quản Barrett	
K22.8	Bệnh xác định khác của thực quản	Xuất huyết thực quản không xác định khác
K22.9	Bệnh của thực quản, không xác định	
K23.*	Rối loạn thực quản do bệnh phân loại mục khác	
K23.0*	Viêm thực quản do lao (A18.8†)	
K23.1*	Phình đại thực quản do bệnh Chagas (B57.3†)	
K23.8*	Rối loạn thực quản do bệnh khác phân loại mục khác	
K25	Loét dạ dày	
K25.0	Loét dạ dày, cấp tính kèm xuất huyết	
K25.1	Loét dạ dày, cấp tính kèm thủng	
K25.2	Loét dạ dày, cấp tính kèm cả xuất huyết và thủng	
K25.3	Loét dạ dày, cấp tính không xuất huyết hoặc thủng	
K25.4	Loét dạ dày, mạn tính hoặc không xác định kèm xuất huyết	
K25.5	Loét dạ dày, mạn tính hoặc không xác định kèm thủng	
K25.6	Loét dạ dày, mạn tính hoặc không xác định kèm cả xuất huyết và thủng	
K25.7	Loét dạ dày, mạn tính không xuất huyết hoặc thủng	
K25.9	Loét dạ dày, không xác định mạn tính hoặc cấp tính, không xuất huyết hoặc thủng	
K26	Loét tá tràng	
K26.0	Loét tá tràng, cấp tính kèm xuất huyết	
K26.1	Loét tá tràng, cấp tính kèm thủng	
K26.2	Loét tá tràng, cấp tính kèm cả xuất huyết và thủng	
K26.3	Loét tá tràng, cấp tính không xuất huyết hoặc thủng	
K26.4	Loét tá tràng, mạn tính hoặc không xác định kèm xuất huyết	
K26.5	Loét tá tràng, mạn tính hoặc không xác định kèm thủng	
K26.6	Loét tá tràng, mạn tính hoặc không xác định kèm cả xuất huyết và thủng	
K26.7	Loét tá tràng, mạn tính không xuất huyết hoặc thủng	
K26.9	Loét tá tràng, không xác định mạn tính hoặc cấp tính, không xuất huyết hoặc thủng	
K27	Loét dạ dày - tá tràng, vị trí không xác định	Loét dạ dày-tá tràng, vị trí không xác định
K27.0	Loét dạ dày - tá tràng, vị trí không xác định, cấp tính kèm xuất huyết	
K27.1	Loét dạ dày - tá tràng, vị trí không xác định, cấp tính kèm thủng	
K27.2	Loét dạ dày - tá tràng, vị trí không xác định, cấp tính kèm cả xuất huyết và thủng	
K27.3	Loét dạ dày - tá tràng, vị trí không xác định, cấp tính không xuất huyết hoặc thủng	
K27.4	Loét dạ dày - tá tràng, vị trí không xác định, mạn tính hoặc không xác định kèm xuất huyết	
K27.5	Loét dạ dày - tá tràng, vị trí không xác định, mạn tính hoặc không xác định kèm thủng	
K27.6	Loét dạ dày - tá tràng, vị trí không xác định, mạn tính hoặc không xác định kèm cả xuất huyết và thủng	
K27.7	Loét dạ dày - tá tràng, vị trí không xác định, mạn tính không xuất huyết hoặc thủng	
K27.9	Loét dạ dày - tá tràng, không xác định mạn tính hoặc cấp tính, không xuất huyết hoặc thủng	
K28	Loét dạ dày - hỗng tràng	Loét dạ dày-hỗng tràng
K28.0	Loét dạ dày - hỗng tràng, cấp tính kèm xuất huyết	
K28.1	Loét dạ dày - hỗng tràng, cấp tính kèm thủng	
K28.2	Loét dạ dày - hỗng tràng, cấp tính kèm cả xuất huyết và thủng	
K28.3	Loét dạ dày - hỗng tràng, cấp tính không xuất huyết hoặc thủng	
K28.4	Loét dạ dày - hỗng tràng, mạn tính hoặc không xác định kèm xuất huyết	
K28.5	Loét dạ dày - hỗng tràng, mạn tính hoặc không xác định kèm thủng	
K28.6	Loét dạ dày - hỗng tràng, mạn tính hoặc không xác định kèm cả xuất huyết và thủng	
K28.7	Loét dạ dày - hỗng tràng, mạn tính không xuất huyết hoặc thủng	
K28.9	Loét dạ dày - hỗng tràng, không xác định mạn tính hoặc cấp tính, không xuất huyết hoặc thủng	
K29	Viêm dạ dày và/hoặc tá tràng	
K29.0	Viêm dạ dày xuất huyết cấp tính	
K29.1	Viêm dạ dày cấp tính khác	
K29.2	Viêm dạ dày do rượu	
K29.3	Viêm nông niêm mạc dạ dày mạn tính	
K29.4	Viêm teo niêm mạc dạ dày mạn tính	Teo dạ dày
K29.5	Viêm dạ dày mạn tính, không xác định	
K29.6	Viêm dạ dày khác	Viêm dạ dày thể phì đại khổng lồ|Viêm dạ dày dạng hạt|Bệnh Menetrier
K29.7	Viêm dạ dày, không xác định	
K29.8	Viêm tá tràng	
K29.9	Viêm dạ dày tá tràng, không xác định	
K30	Rối loạn tiêu hóa chức năng [không do loét]	
K31	Bệnh khác của dạ dày và/hoặc tá tràng	
K31.0	Giãn dạ dày cấp tính	Căng dạ dày cấp tính
K31.1	Hẹp môn vị do phì đại ở người lớn	
K31.2	Hẹp và/hoặc co hẹp dạ dày dạng đồng hồ cát	
K31.3	Co thắt môn vị, không phân loại mục khác	
K31.4	Túi thừa dạ dày	
K31.5	Tắc tá tràng	
K31.6	Rò dạ dày và/hoặc tá tràng	Rò dạ dày - ruột|Rò dạ dày - hỗng tràng - đại tràng
K31.7	Polyp dạ dày và/hoặc tá tràng	
K31.8	Bệnh xác định khác của dạ dày và/hoặc tá tràng	Giảm tiết dịch vị|Sa dạ dày|Co thắt dạ dày dạng đồng hồ cát
K31.9	Bệnh không xác định của dạ dày và/hoặc tá tràng	
K35	Viêm ruột thừa cấp tính	
K35.2	Viêm ruột thừa cấp tính kèm viêm phúc mạc toàn bộ	
K35.3	Viêm ruột thừa cấp tính kèm viêm phúc mạc khu trú	
K35.8	Viêm ruột thừa cấp tính, khác và/hoặc không xác định	Viêm ruột thừa cấp tính không đề cập đến viêm phúc mạc khu trú hoặc viêm phúc mạc toàn bộ
K36	Viêm ruột thừa khác	
K37	Viêm ruột thừa không xác định	
K38	Bệnh khác của ruột thừa	
K38.0	Tăng sản ruột thừa	
K38.1	Kết sỏi ở ruột thừa	Sỏi phân của ruột thừa|Phân cứng như sỏi trong ruột thừa
K38.2	Túi thừa của ruột thừa	
K38.3	Rò ruột thừa	
K38.8	Bệnh xác định khác của ruột thừa	Chứng lồng ruột của ruột thừa
K38.9	Bệnh không xác định của ruột thừa	
K40	Thoát vị bẹn	
K40.0	Thoát vị bẹn hai bên, kèm tắc nghẽn, không hoại thư	
K40.1	Thoát vị bẹn hai bên, kèm hoại thư	
K40.2	Thoát vị bẹn hai bên, không tắc nghẽn hoặc hoại thư	
K40.3	Thoát vị bẹn một bên hoặc không xác định, kèm tắc nghẽn, không hoại thư	
K40.4	Thoát vị bẹn một bên hoặc không xác định, kèm hoại thư	Thoát vị bẹn không xác định khác kèm hoại thư
K40.9	Thoát vị bẹn một bên hoặc không xác định, không tắc nghẽn hoặc hoại thư	
K41	Thoát vị đùi	
K41.0	Thoát vị đùi hai bên, kèm tắc nghẽn, không hoại thư	
K41.1	Thoát vị đùi hai bên, kèm hoại thư	
K41.2	Thoát vị đùi hai bên, không tắc nghẽn hoặc hoại thư	Thoát vị đùi hai bên không xác định khác
K41.3	Thoát vị đùi một bên hoặc không xác định, kèm tắc nghẽn, không hoại thư	
K41.4	Thoát vị đùi một bên hoặc không xác định, kèm hoại thư	
K41.9	Thoát vị đùi một bên hoặc không xác định, không tắc nghẽn hoặc hoại thư	
K42	Thoát vị rốn	
K42.0	Thoát vị rốn kèm tắc nghẽn, không hoại thư	
K42.1	Thoát vị rốn kèm hoại thư	Thoát vị rốn hoại thư
K42.9	Thoát vị rốn không tắc nghẽn hoặc hoại thư	Thoát vị rốn không xác định khác
K43	Thoát vị thành bụng	
K43.0	Thoát vị qua vết mổ kèm tắc nghẽn, không hoại thư	
K43.1	Thoát vị qua vết mổ kèm hoại thư	Thoát vị qua vết mổ có hoại thư
K43.2	Thoát vị qua vết mổ không tắc nghẽn hoặc hoại thư	Thoát vị qua vết mổ không xác định khác
K43.3	Thoát vị quanh lỗ mở thông ruột, đường tiết niệu ra da kèm tắc nghẽn, không hoại thư	
K43.4	Thoát vị quanh lỗ mở thông ruột, đường tiết niệu ra da kèm hoại thư	
K43.5	Thoát vị quanh lỗ mở thông ruột, đường tiết niệu ra da không tắc nghẽn hoặc hoại thư	Thoát vị quanh lỗ mở thông ruột, đường tiết niệu ra da không xác định khác
K43.6	Thoát vị thành bụng khác và/hoặc không xác định kèm tắc nghẽn, không hoại thư	
K43.7	Thoát vị thành bụng khác và/hoặc không xác định kèm hoại thư	Bất kỳ tình trạng nào được liệt kê trong mã K43.6 được xác định là có hoại thư
K43.9	Thoát vị thành bụng khác và/hoặc không xác định không tắc nghẽn hoặc hoại thư	Thoát vị thành bụng không xác định khác
K44	Thoát vị cơ hoành	
K44.0	Thoát vị cơ hoành kèm tắc nghẽn, không hoại thư	
K44.1	Thoát vị cơ hoành kèm hoại thư	Thoát vị hoành có hoại thư
K44.9	Thoát vị cơ hoành, không tắc nghẽn hoặc hoại thư	Thoát vị hoành không xác định khác
K45	Thoát vị bụng khác	
K45.0	Thoát vị bụng xác định khác có tắc nghẽn, không hoại thư	
K45.1	Thoát vị bụng xác định khác, có hoại thư	Bất kỳ tình trạng nào liệt kê ở K45.- được xác định có hoại thư
K45.8	Thoát vị bụng xác định khác, không tắc nghẽn hoặc hoại thư	
K46	Thoát vị bụng không xác định	
K46.0	Thoát vị bụng không xác định có tắc nghẽn, không hoại thư	Bất kỳ tình trạng nào được liệt kê ở K46.|gây tắc không hoại thư|giữ chặt không hoại thư|không đẩy lên được không hoại thư|nghẹt không hoại thư
K46.1	Thoát vị bụng không xác định, có hoại thư	Bất kỳ tình trạng nào liệt kê ở K46.- được xác định có hoại tử
K46.9	Thoát vị bụng không xác định, không tắc nghẽn hoặc hoại thư	Thoát vị bụng không xác định khác
K50	Bệnh Crohn [viêm ruột từng vùng]	
K50.0	Bệnh Crohn của ruột non	
K50.1	Bệnh Crohn của đại tràng	
K50.8	Bệnh Crohn khác	Bệnh Crohn của cả ruột non và đại tràng
K50.9	Bệnh Crohn, không xác định	Viêm ruột khu vực không xác định khác
K51	Viêm loét đại tràng	
K51.0	Viêm loét toàn ruột (mạn tính)	Viêm hồi tràng trào ngược
K51.2	Viêm loét trực tràng (mạn tính)	
K51.3	Viêm loét đại tràng sigma (mạn tính)	
K51.4	Nhiều polyp viêm	
K51.5	Viêm loét đại tràng trái	Viêm nửa khung đại tràng trái
K51.8	Viêm loét khác của đại tràng	
K51.9	Viêm loét không xác định của đại tràng	
K52	Viêm dạ dày - ruột và/hoặc viêm đại tràng khác không do nhiễm trùng	Viêm dạ dày-ruột và/hoặc viêm đại tràng khác không do nhiễm trùng
K52.0	Viêm dạ dày - ruột và/hoặc đại tràng do xạ trị	
K52.1	Viêm dạ dày - ruột và/hoặc đại tràng do nhiễm độc	
K52.2	Viêm dạ dày - ruột và/hoặc đại tràng do dị ứng và/hoặc do chế độ ăn	Viêm dạ dày - ruột và đại tràng do tăng nhạy cảm với thực phẩm
K52.3	Viêm đại tràng không xác định	
K52.8	Viêm dạ dày - ruột và/hoặc đại tràng xác định khác không do nhiễm trùng	
K52.9	Viêm dạ dày - ruột và/hoặc viêm đại tràng không do nhiễm trùng, không xác định	
K55	Rối loạn của mạch máu ruột	
K55.0	Rối loạn mạch máu ruột cấp tính	
K55.1	Rối loạn mạch máu ruột mạn tính	
K55.2	Loạn sản mạch máu của đại tràng	Chứng loạn sản mạch máu của ruột không xác định khác
K55.3	Loạn sản mạch máu của ruột non	
K55.8	Rối loạn khác của mạch máu ruột	
K55.9	Rối loạn của mạch máu ruột, không xác định	
K56	Liệt ruột và/hoặc tắc nghẽn ruột không có thoát vị	
K56.0	Liệt ruột	
K56.1	Lồng ruột	
K56.2	Xoắn ruột	Nghẹt đại tràng hoặc ruột|Xoắn đại tràng hoặc ruột|Vặn đại tràng hoặc ruột
K56.3	Tắc ruột do sỏi	
K56.4	Nghẹt ruột khác	
K56.5	Dính ruột [thành dải] có tắc nghẽn	Dính phúc mạc [thành dải] có tắc ruột
K56.6	Tắc nghẽn ruột khác và/hoặc không xác định	Hẹp đường ruột|Tắc ruột không xác định khác|Tắc đại tràng hay ruột|Hẹp đại tràng hay ruột|Chít hẹp đại tràng hay ruột
K56.7	Tắc ruột, không xác định	
K57	Bệnh túi thừa của ruột	
K57.0	Bệnh túi thừa của ruột non, có thủng và/hoặc áp xe	
K57.1	Bệnh túi thừa của ruột non, không thủng hoặc áp xe	
K57.2	Bệnh túi thừa của đại tràng có thủng và/hoặc áp xe	
K57.3	Bệnh túi thừa của đại tràng không thủng hoặc áp xe	
K57.4	Bệnh túi thừa của cả ruột non và đại tràng có thủng và/hoặc áp xe	Bệnh túi thừa của ruột non và/hoặc đại tràng có viêm phúc mạc
K57.5	Bệnh túi thừa của cả ruột non và đại tràng không thủng hoặc áp xe	Bệnh túi thừa của cả ruột non và đại tràng không xác định khác
K57.8	Bệnh túi thừa của ruột, phần không xác định, kèm thủng và áp xe	Bệnh túi thừa của ruột không xác định khác có viêm phúc mạc
K57.9	Bệnh túi thừa của ruột, phần không xác định, không thủng hoặc áp xe	Bệnh túi thừa của ruột không xác định khác
K58	Hội chứng ruột kích thích	
K58.1	Hội chứng ruột kích thích thể tiêu chảy (IBS-D)	
K58.2	Hội chứng ruột kích thích thể táo bón (IBS-C)	
K58.3	Hội chứng ruột kích thích hỗn hợp (IBS-M)	
K58.8	Hội chứng khác và/hoặc không xác định của ruột kích thích	Hội chứng ruột kích thích không xác định khác
K59	Rối loạn chức năng khác của ruột	
K59.0	Táo bón	
K59.1	Tiêu chảy chức năng	
K59.2	Rối loạn ruột do nguyên nhân thần kinh, không phân loại mục khác	
K59.3	Phình đại tràng, không phân loại mục khác	
K59.4	Co thắt hậu môn	Đau trực tràng thoáng qua
K59.8	Rối loạn chức năng xác định khác của ruột	Mất trương lực đại tràng
K59.9	Rối loạn chức năng của ruột, không xác định	
K60	Nứt kẽ và/hoặc rò vùng hậu môn và/hoặc trực tràng	
K60.0	Nứt kẽ hậu môn cấp tính	
K60.1	Nứt kẽ hậu môn mạn tính	
K60.2	Nứt ống hậu môn, không xác định	
K60.3	Rò hậu môn	
K60.4	Rò trực tràng	
K60.5	Rò hậu môn trực tràng	
K61	Áp xe vùng hậu môn và/hoặc trực tràng	
K61.0	Áp xe hậu môn	
K61.1	Áp xe trực tràng	
K61.2	Áp xe hậu môn trực tràng	
K61.3	Áp xe ụ ngồi - trực tràng	Áp xe hố ụ ngồi - trực tràng
K61.4	Áp xe trong cơ thắt hậu môn	
K62	Bệnh khác của hậu môn và/hoặc trực tràng	
K62.0	Polyp hậu môn	
K62.1	Polyp trực tràng	
K62.2	Sa hậu môn	Sa ống hậu môn
K62.3	Sa trực tràng qua hậu môn [toàn bộ]	Sa niêm mạc trực tràng
K62.4	Hẹp trực tràng và/hoặc ống hậu môn	
K62.5	Xuất huyết hậu môn và/hoặc trực tràng	
K62.6	Loét hậu môn và/hoặc trực tràng	
K62.7	Viêm trực tràng do xạ trị	
K62.8	Bệnh xác định khác của hậu môn và/hoặc trực tràng	Viêm trực tràng không xác định khác
K62.9	Bệnh của hậu môn và/hoặc trực tràng, không xác định	
K63	Bệnh khác của ruột	
K63.0	Áp xe ruột	
K63.1	Thủng ruột (không do chấn thương)	
K63.2	Rò ruột	
K63.3	Loét ruột	
K63.4	Sa ruột	
K63.5	Polyp đại tràng	
K63.8	Bệnh xác định khác của ruột	
K63.9	Bệnh của ruột, không xác định	
K64	Bệnh trĩ và/hoặc huyết khối tĩnh mạch quanh hậu môn	
K64.0	Bệnh trĩ độ I	
K64.1	Bệnh trĩ độ II	
K64.2	Bệnh trĩ độ III	
K64.3	Bệnh trĩ độ IV	
K64.4	Vạt da thừa còn sót lại của bệnh trĩ	Da thừa ở hậu môn
K64.5	Huyết khối tĩnh mạch quanh hậu môn	Tụ máu quanh hậu môn
K64.8	Bệnh trĩ xác định khác	
K64.9	Bệnh trĩ, không xác định	
K65	Viêm phúc mạc	
K65.0	Viêm phúc mạc cấp tính	
K65.8	Viêm phúc mạc khác	Viêm phúc mạc tăng sinh mạn tính
K65.9	Viêm phúc mạc, không xác định	
K66	Rối loạn khác của phúc mạc	
K66.0	Dính phúc mạc	
K66.1	Tràn máu phúc mạc	
K66.2	Xơ hóa vùng sau phúc mạc	Bệnh Ormond
K66.8	Rối loạn xác định khác của phúc mạc	
K66.9	Rối loạn của phúc mạc, không xác định	
K67.*	Rối loạn của phúc mạc do bệnh nhiễm trùng phân loại mục khác	
K67.0*	Viêm phúc mạc do nhiễm chlamydia (A74.8†)	
K67.1*	Viêm phúc mạc do bệnh lậu cầu khuẩn (A54.8†)	
K67.2*	Viêm phúc mạc do bệnh giang mai (A52.7†)	
K67.3*	Viêm phúc mạc do bệnh lao (A18.3†)	
K67.8*	Rối loạn khác của phúc mạc do bệnh nhiễm trùng phân loại mục khác	
K70	Bệnh gan do rượu	
K70.0	Gan nhiễm mỡ do rượu	
K70.1	Viêm gan do rượu	
K70.2	Bệnh xơ gan và/hoặc xơ hóa gan do rượu	
K70.3	Xơ gan do rượu	Bệnh xơ gan do rượu không xác định khác
K70.4	Suy gan do rượu	
K70.9	Bệnh gan do rượu, không xác định	
K71	Bệnh gan nhiễm độc	
K71.0	Bệnh gan nhiễm độc kèm ứ mật	"Ứ mật với tổn thương tế bào gan|Ứ mật ""đơn thuần"""
K71.1	Bệnh gan nhiễm độc kèm hoại tử gan	
K71.2	Bệnh gan nhiễm độc kèm viêm gan cấp tính	
K71.3	Bệnh gan nhiễm độc kèm viêm gan mạn tính dai dẳng	
K71.4	Bệnh gan nhiễm độc kèm viêm tiểu thùy gan mạn tính	
K71.5	Bệnh gan nhiễm độc kèm viêm gan mạn tính hoạt động	Bệnh gan nhiễm độc kèm theo viêm gan dạng lupoid
K71.6	Bệnh gan nhiễm độc kèm viêm gan, không phân loại mục khác	
K71.7	Bệnh gan nhiễm độc kèm xơ hóa gan và/hoặc xơ gan	
K71.8	Bệnh gan nhiễm độc kèm rối loạn khác của gan	
K71.9	Bệnh gan nhiễm độc, không xác định	
K72	Suy gan, không phân loại mục khác	
K72.0	Suy gan cấp tính và/hoặc bán cấp tính	Viêm gan cấp tính không do virus, không xác định khác|Suy gan khởi phát muộn
K72.1	Suy gan mạn tính	
K72.9	Suy gan, không xác định	
K73	Viêm gan mạn tính, không phân loại mục khác	
K73.0	Viêm gan mạn tính dai dẳng, không phân loại mục khác	
K73.1	Viêm tiểu thùy gan mạn, không phân loại mục khác	
K73.2	Viêm gan mạn tính hoạt động, không phân loại mục khác	
K73.8	Viêm gan mạn tính khác, không phân loại mục khác	
K73.9	Viêm gan mạn tính, không xác định	
K74	Xơ hóa gan và/hoặc xơ gan	
K74.0	Xơ hóa gan	
K74.1	Xơ cứng gan	
K74.2	Gan xơ hóa với gan xơ cứng	
K74.3	Xơ gan mật nguyên phát	Viêm đường mật phá hủy không có mủ mạn tính
K74.4	Xơ gan mật thứ phát	
K74.5	Xơ gan mật, không xác định	
K74.6	Xơ gan khác và/hoặc không xác định	
K75	Bệnh viêm gan khác	
K75.0	Áp xe gan	
K75.1	Viêm tĩnh mạch cửa	
K75.2	Viêm gan phản ứng, không xác định cụ thể	
K75.3	Viêm gan dạng u hạt, không phân loại mục khác	
K75.4	Viêm gan tự miễn	Viêm gan lupus không phân loại mục khác
K75.8	Bệnh viêm gan xác định khác	Viêm gan nhiễm mỡ không do rượu [NASH]
K75.9	Bệnh viêm gan, không xác định	Viêm gan không xác định khác
K76	Bệnh gan khác	
K76.0	Gan (biến đổi của gan) nhiễm mỡ, không phân loại mục khác	
K76.1	Xung huyết thụ động mạn tính ở gan	
K76.2	Hoại tử xuất huyết vùng trung tâm của gan	
K76.3	Nhồi máu gan	
K76.4	Rối loạn mạch máu trong gan	Bệnh u mạch máu gan
K76.5	Bệnh tắc tĩnh mạch trên gan	
K76.6	Tăng áp lực tĩnh mạch cửa	
K76.7	Hội chứng gan - thận	
K76.8	Bệnh xác định khác của gan	Shunt mạch máu trong gan mắc phải|Tăng sản nốt khu trú của gan|Bệnh sa gan|Nang gan đơn giản
K76.9	Bệnh của gan, không xác định	
K77.*	Rối loạn gan do bệnh phân loại mục khác	
K77.0*	Rối loạn gan do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
K77.8*	Rối loạn gan do bệnh khác phân loại mục khác	
K80	Bệnh sỏi mật	
K80.0	Sỏi túi mật kèm viêm túi mật cấp tính	Bất kỳ tình trạng nào liệt kê ở K80.2 có viêm túi mật cấp
K80.1	Sỏi túi mật kèm viêm túi mật khác	
K80.2	Sỏi túi mật không kèm viêm túi mật	
K80.3	Sỏi ống mật kèm viêm đường dẫn mật	Bất kỳ tình trạng nào liệt kê ở K80.5 có viêm đường dẫn mật
K80.4	Sỏi đường mật kèm viêm túi mật	
K80.5	Sỏi mật không viêm đường dẫn mật hay viêm túi mật	
K80.8	Sỏi mật khác	
K81	Viêm túi mật	
K81.0	Viêm túi mật cấp tính	
K81.1	Viêm túi mật mạn tính	
K81.8	Viêm túi mật thể khác	
K81.9	Viêm túi mật, không xác định	
K82	Bệnh khác của túi mật	
K82.0	Tắc nghẽn túi mật	
K82.1	Tràn dịch túi mật	U nhầy túi mật
K82.2	Thủng túi mật	Vỡ ống túi mật hoặc túi mật
K82.3	Rò túi mật	Rò túi mật đại tràng|Rò túi mật tá tràng
K82.4	Tích tụ cholesterol ở túi mật	Túi mật hình quả dâu tây
K82.8	Bệnh xác định khác của túi mật	Dính túi mật hay ống mật|Teo túi mật hay ống mật|U nang của túi mật hay ống mật|Rối loạn vận động của túi mật hay ống mật|Phì đại túi mật hay ống mật|Mất chức năng của túi mật hay ống mật|Loét túi mật hay ống mật
K82.9	Bệnh của túi mật, không xác định	
K83	Bệnh khác của đường mật	
K83.0	Viêm đường mật	
K83.1	Tắc nghẽn ống mật	
K83.2	Thủng đường mật	Vỡ đường mật
K83.3	Rò đường mật	Rò ống mật chủ - tá tràng
K83.4	Co thắt cơ vòng Oddi	
K83.5	U nang đường mật	
K83.8	Bệnh xác định khác của đường mật	Dính đường mật|Teo đường mật|Phì đại đường mật|Loét đường mật
K83.9	Bệnh đường mật, không xác định	
K85	Viêm tụy cấp tính	
K85.0	Viêm tụy cấp tính vô căn	
K85.1	Viêm tụy cấp tính do mật	Viêm tụy do sỏi túi mật
K85.2	Viêm tụy cấp tính do rượu	
K85.3	Viêm tụy cấp tính do thuốc	
K85.8	Viêm tụy cấp tính khác	
K85.9	Viêm tụy cấp tính, không xác định	
K86	Bệnh khác của tụy	
K86.0	Viêm tụy mạn tính do rượu	
K86.1	Viêm tụy mạn tính thể khác	
K86.2	U nang của tụy	
K86.3	U nang giả của tụy	
K86.8	Bệnh xác định khác của tụy	Teo tụy|Sỏi tụy|Xơ tụy|Xơ hóa tụy
K86.9	Bệnh của tụy, không xác định	
K87.*	Rối loạn của túi mật, đường mật và/hoặc tụy do bệnh phân loại mục khác	
K87.0*	Rối loạn của túi mật và/hoặc đường mật do bệnh phân loại mục khác	
K87.1*	Rối loạn của tụy do bệnh phân loại mục khác	
K90	Suy giảm hấp thu ở ruột	
K90.0	Bệnh Coeliac	Bệnh lý đường ruột nhạy cảm gluten|Tiêu chảy phân mỡ không rõ nguyên nhân|Tiêu chảy phân mỡ và suy giảm hấp thu dinh dưỡng [sprue], không gặp ở vùng nhiệt đới
K90.1	Bệnh tiêu chảy nhiệt đới	Tiêu chảy phân mỡ và suy giảm hấp thu dinh dưỡng [sprue] không xác định khác|Phân nhiễm mỡ nhiệt đới
K90.2	Hội chứng quai ruột, không phân loại mục khác	
K90.3	Tiêu chảy phân mỡ [suy giảm hấp thu chất béo] do tụy	
K90.4	Suy giảm hấp thu do không dung nạp, không phân loại mục khác	
K90.8	Suy giảm hấp thu khác ở ruột	
K90.9	Suy giảm hấp thu ở ruột, không xác định	
K91	Rối loạn của hệ tiêu hóa sau can thiệp, không phân loại mục khác	
K91.0	Nôn sau phẫu thuật đường tiêu hóa	
K91.1	Hội chứng sau phẫu thuật dạ dày	
K91.2	Suy giảm hấp thu sau phẫu thuật, không phân loại mục khác	
K91.3	Tắc nghẽn ruột sau mổ	
K91.4	Suy chức năng sau mở thông đại tràng và/hoặc mở thông ruột non ra da	
K91.5	Hội chứng sau cắt túi mật	
K91.8	Rối loạn khác của hệ tiêu hóa sau can thiệp, không phân loại mục khác	
K91.9	Rối loạn của hệ tiêu hóa sau can thiệp, không xác định	
K92	Bệnh khác của hệ tiêu hóa	
K92.0	Nôn ra máu [thổ huyết]	
K92.1	Đi ngoài phân đen	
K92.2	Xuất huyết tiêu hóa [phân đen], không xác định	
K92.8	Bệnh xác định khác của hệ tiêu hóa	
K92.9	Bệnh của hệ tiêu hóa, không xác định	
K93.*	Rối loạn của cơ quan tiêu hóa khác do bệnh phân loại mục khác	
K93.0*	Rối loạn ở ruột, phúc mạc và/hoặc tuyến mạc treo do bệnh lao (A18.3†)	
K93.1*	Phình đại tràng do bệnh Chagas (B57.3†)	
K93.8*	Rối loạn cơ quan tiêu hóa xác định khác do bệnh phân loại mục khác	
L00	Hội chứng bong vảy da do tụ cầu khuẩn	
L01	Bệnh chốc	
L01.0	Bệnh chốc [bất kỳ vi sinh vật nào] [bất kỳ vị trí nào]	Chốc Bockhart
L01.1	Chốc hóa của bệnh da khác	
L02	Áp xe da, nhọt và/hoặc cụm nhọt	
L02.0	Áp xe da, nhọt và/hoặc cụm nhọt ở mặt	
L02.1	Áp xe da, nhọt và/hoặc cụm nhọt ở cổ	
L02.2	Áp xe da, nhọt và/hoặc cụm nhọt ở thân	
L02.3	Áp xe da, nhọt và/hoặc cụm nhọt ở mông	
L02.4	Áp xe da, nhọt và/hoặc cụm nhọt ở chi	Nách|Háng
L02.8	Áp xe da, nhọt và/hoặc cụm nhọt ở vị trí khác	Đầu [bất cứ phần nào, ngoại trừ mặt]|Da đầu
L02.9	Áp xe da, nhọt và/hoặc cụm nhọt, không xác định	Bệnh nhọt không xác định khác
L03	Viêm mô bào	
L03.0	Viêm mô bào ở ngón tay và/hoặc ngón chân	Nhiễm trùng móng|Viêm nền móng|Chín mé|Viêm quanh móng
L03.1	Viêm mô bào ở phần khác của chi	Nách|Háng
L03.2	Viêm mô bào ở mặt	
L03.3	Viêm mô bào ở thân	
L03.8	Viêm mô bào ở vị trí khác	Đầu [bất kỳ phần nào, ngoại trừ mặt]|Da đầu
L03.9	Viêm mô bào, không xác định	
L04	Viêm hạch bạch huyết cấp tính	
L04.0	Viêm hạch bạch huyết cấp tính ở mặt, đầu và/hoặc cổ	
L04.1	Viêm hạch bạch huyết cấp tính ở thân	
L04.2	Viêm hạch bạch huyết cấp tính ở chi trên	Nách
L04.3	Viêm hạch bạch huyết cấp tính ở chi dưới	Háng
L04.8	Viêm hạch bạch huyết cấp tính ở vị trí khác	
L04.9	Viêm hạch bạch huyết cấp tính, không xác định	
L05	U nang lông	
L05.0	U nang lông có áp xe	
L05.9	U nang lông không áp xe	U nang lông không xác định khác
L08	Nhiễm trùng khu trú khác của da và/hoặc mô dưới da	
L08.0	Viêm da mủ	
L08.1	Bệnh viêm đỏ nếp kẽ do vi khuẩn [Erythrasma]	
L08.8	Nhiễm trùng khu trú xác định khác ở da và/hoặc mô dưới da	
L08.9	Nhiễm trùng khu trú ở da và/hoặc mô dưới da, không xác định	
L10	Bệnh pemphigus	
L10.0	Bệnh pemphigus thông thường	
L10.1	Bệnh pemphigus sùi	
L10.2	Bệnh pemphigus vảy lá	
L10.3	Bệnh pemphigus Brazil	
L10.4	Bệnh pemphigus đỏ da	Hội chứng Senear-Usher
L10.5	Bệnh pemphigus do thuốc	
L10.8	Bệnh pemphigus khác	
L10.9	Bệnh pemphigus, không xác định	
L11	Rối loạn da ly gai khác	
L11.0	Dày sừng nang lông mắc phải	
L11.1	Bệnh viêm da ly gai thoáng qua [Grover]	
L11.8	Rối loạn ly gai xác định khác	
L11.9	Rối loạn ly gai, không xác định	
L12	Bệnh bọng nước dạng pemphigus	
L12.0	Bệnh pemphigoid bọng nước	
L12.1	Bệnh pemphigoid có sẹo	Bệnh pemphigoid màng nhầy lành tính
L12.2	Bệnh da bọng nước mạn tính ở trẻ em	Viêm da dạng herpes ở tuổi thanh thiếu niên
L12.3	Bệnh ly thượng bì bọng nước mắc phải	
L12.8	Bệnh pemphigoid khác	
L12.9	Bệnh pemphigoid, không xác định	
L13	Rối loạn da bọng nước khác	
L13.0	Viêm da dạng herpes	Bệnh Duhring
L13.1	Viêm da mụn mủ dưới lớp sừng	Bệnh Sneddon-Wilkinson
L13.8	Bệnh lý da bọng nước xác định khác	
L13.9	Bệnh lý da bọng nước, không xác định	
L14.*	Bệnh lý da bọng nước do bệnh phân loại mục khác	
L20	Viêm da cơ địa	
L20.0	Sẩn ngứa Besnier	
L20.8	Viêm da cơ địa khác	
L20.9	Viêm da cơ địa, không xác định	
L21	Viêm da dầu	
L21.0	Viêm da dầu ở đầu	Tróc thành mảng lớn trên đầu
L21.1	Viêm da dầu ở trẻ nhỏ	
L21.8	Viêm da dầu khác	
L21.9	Viêm da dầu, không xác định	
L22	Viêm da vùng tã (bỉm) [hăm tã]	
L23	Viêm da tiếp xúc dị ứng	
L23.0	Viêm da tiếp xúc dị ứng do kim loại	Crom|Niken
L23.1	Viêm da tiếp xúc dị ứng do keo dính	
L23.2	Viêm da tiếp xúc dị ứng do mỹ phẩm	
L23.3	Viêm da tiếp xúc dị ứng do thuốc bôi	
L23.4	Viêm da tiếp xúc dị ứng do thuốc nhuộm	
L23.5	Viêm da tiếp xúc dị ứng do hóa chất khác	Xi măng|Thuốc trừ sâu|Nhựa dẻo|Cao su
L23.6	Viêm da tiếp xúc dị ứng do thực phẩm	
L23.7	Viêm da tiếp xúc dị ứng do thực vật, ngoại trừ thực phẩm	
L23.8	Viêm da tiếp xúc dị ứng do tác nhân khác	
L23.9	Viêm da tiếp xúc dị ứng, nguyên nhân không xác định	Chàm tiếp xúc dị ứng không xác định khác
L24	Viêm da tiếp xúc kích ứng	
L24.0	Viêm da tiếp xúc kích ứng do chất tẩy rửa	
L24.1	Viêm da tiếp xúc kích ứng do dầu và/hoặc mỡ bôi trơn	
L24.2	Viêm da tiếp xúc kích ứng do dung môi	
L24.3	Viêm da tiếp xúc kích ứng do mỹ phẩm	
L24.4	Viêm da tiếp xúc kích ứng do thuốc bôi	
L24.5	Viêm da tiếp xúc kích ứng do hóa chất khác	Xi măng|Thuốc trừ sâu
L24.6	Viêm da tiếp xúc kích ứng do thực phẩm	
L24.7	Viêm da tiếp xúc kích ứng do thực vật, ngoại trừ thực phẩm	
L24.8	Viêm da tiếp xúc kích ứng do tác nhân khác	Thuốc nhuộm
L24.9	Viêm da tiếp xúc kích ứng, nguyên nhân không xác định	Chàm tiếp xúc kích ứng không xác định khác
L25	Viêm da tiếp xúc không xác định	
L25.0	Viêm da tiếp xúc không xác định, do mỹ phẩm	
L25.1	Viêm da tiếp xúc không xác định, do thuốc tiếp xúc với da	
L25.2	Viêm da tiếp xúc không xác định do thuốc nhuộm	
L25.3	Viêm da tiếp xúc không xác định do hóa chất khác	Xi măng|Thuốc trừ sâu
L25.4	Viêm da tiếp xúc không xác định do thực phẩm	
L25.5	Viêm da tiếp xúc không xác định, do thực vật, trừ thực phẩm	
L25.8	Viêm da tiếp xúc không xác định, do tác nhân khác	
L25.9	Viêm da tiếp xúc, không xác định	
L26	Đỏ da toàn thân [viêm da tróc vảy]	
L27	Viêm da do chất được đưa vào trong cơ thể	
L27.0	Phát ban toàn thân do dược chất và/hoặc thuốc điều trị	
L27.1	Phát ban khu trú do dược chất và/hoặc thuốc điều trị	
L27.2	Viêm da do thức ăn	
L27.8	Viêm da do chất khác đưa vào trong cơ thể	
L27.9	Viêm da do chất không xác định đưa vào trong cơ thể	
L28	Lichen đơn dạng mạn tính và/hoặc sẩn ngứa	
L28.0	Lichen đơn dạng mạn tính	Viêm da thần kinh khu trú|Lichen không xác định khác
L28.1	Sẩn cục	
L28.2	Sẩn ngứa khác	
L29	Ngứa	
L29.0	Ngứa hậu môn	
L29.1	Ngứa bìu	
L29.2	Ngứa âm hộ	
L29.3	Ngứa hậu môn - sinh dục, không xác định	
L29.8	Ngứa khác	
L29.9	Ngứa, không xác định	Ngứa không xác định khác
L30	Viêm da khác	
L30.0	Viêm da dạng đồng tiền	
L30.1	Tổ đỉa [chàm tổ đỉa]	
L30.2	Viêm da tự mẫn cảm	Do Candida [ban đỏ do nấm men]|Do nấm sợi|Do chàm
L30.3	Viêm da nhiễm trùng	Viêm da dạng chàm do nhiễm trùng
L30.4	Viêm đỏ nếp kẽ	
L30.5	Vảy phấn trắng [chàm khô]	
L30.8	Viêm da xác định khác	
L30.9	Viêm da, không xác định	Chàm không xác định khác
L40	Bệnh vảy nến	
L40.0	Bệnh vảy nến thể thông thường	Vảy nến thể đồng tiền|Vảy nến thể mảng
L40.1	Bệnh vảy nến thể mủ toàn thân	Bệnh chốc dạng herpes|Bệnh Von Zumbusch
L40.2	Viêm da mụn mủ đầu chi liên tục	
L40.3	Bệnh vảy nến thể mủ khu trú ở lòng bàn tay hoặc lòng bàn chân	
L40.4	Bệnh vảy nến thể giọt	
L40.5†	Bệnh vảy nến thể khớp (M07.0-M07.3*, M09.0*)	
L40.8	Bệnh vảy nến khác	Bệnh vảy nến thể đảo ngược
L40.9	Bệnh vảy nến, không xác định	
L41	Á vảy nến	
L41.0	Bệnh vảy phấn dạng lichen và/hoặc bệnh đậu mùa cấp tính	Bệnh Mucha-Habermann
L41.1	Vảy phấn dạng lichen mạn tính	
L41.3	Á vảy nến thể mảng nhỏ	
L41.4	Á vảy nến thể mảng lớn	
L41.5	Á vảy nến dạng lưới	
L41.8	Á vảy nến khác	
L41.9	Á vảy nến, không xác định	
L42	Bệnh vảy phấn hồng	
L43	Lichen phẳng	
L43.0	Lichen phẳng phì đại	
L43.1	Lichen phẳng bọng nước	
L43.2	Phản ứng thuốc dạng lichen	
L43.3	Lichen phẳng bán cấp tính (hoạt tính)	Lichen phẳng nhiệt đới
L43.8	Lichen phẳng khác	
L43.9	Lichen phẳng, không xác định	
L44	Rối loạn sẩn có vảy khác	
L44.0	Bệnh vảy phấn đỏ nang lông	
L44.1	Liken phẳng thể chấm	
L44.2	Lichen thể vạch	
L44.3	Lichen dạng vằn	
L44.4	Viêm da đầu chi dạng sẩn ở trẻ nhỏ [Glannotti-Crosti]	
L44.8	Rối loạn sẩn tróc vảy xác định khác	
L44.9	Rối loạn sẩn tróc vảy da, không xác định	
L45.*	Rối loạn của sẩn tróc vảy da do bệnh phân loại mục khác	
L50	Bệnh mày đay	
L50.0	Bệnh mày đay dị ứng	
L50.1	Bệnh mày đay vô căn	
L50.2	Bệnh mày đay do lạnh và/hoặc nóng	
L50.3	Bệnh da vẽ nổi	
L50.4	Bệnh mày đay do rung động	
L50.5	Bệnh mày đay do Cholin	
L50.6	Bệnh mày đay tiếp xúc	
L50.8	Bệnh mày đay khác	
L50.9	Bệnh mày đay, không xác định	
L51	Hồng ban đa dạng	
L51.0	Hồng ban đa dạng không có bọng nước	
L51.1	Hồng ban đa dạng có bọng nước	Hội chứng Stevens-Johnson
L51.2	Hoại tử thượng bì nhiễm độc [hội chứng Lyell]	
L51.8	Hồng ban đa dạng khác	
L51.9	Hồng ban đa dạng, không xác định	
L52	Hồng ban nút	
L53	Trạng thái hồng ban khác	
L53.0	Hồng ban do nhiễm độc	
L53.1	Hồng ban vòng ly tâm	
L53.2	Hồng ban vòng	
L53.3	Hồng ban mạn tính khác	
L53.8	Tình trạng hồng ban xác định khác	
L53.9	Tình trạng hồng ban, không xác định	Hồng ban không xác định khác|Đỏ da toàn thân không xác định khác
L54.*	Hồng ban do bệnh phân loại mục khác	
L54.0*	Hồng ban vòng do bệnh sốt thấp cấp tính (I00†)	
L54.8*	Hồng ban do bệnh khác phân loại mục khác	
L55	Bỏng [cháy] nắng	
L55.0	Bỏng [cháy] nắng độ một	
L55.1	Bỏng [cháy] nắng độ hai	
L55.2	Bỏng [cháy] nắng độ ba	
L55.8	Bỏng [cháy] nắng khác	
L55.9	Bỏng [cháy] nắng, không xác định	
L56	Thay đổi cấp tính khác của da do tia cực tím [tia tử ngoại]	
L56.0	Phản ứng với thuốc gây ngộ độc ánh sáng	
L56.1	Phản ứng dị ứng ánh sáng do thuốc	
L56.2	Viêm da tiếp xúc ánh sáng [viêm da berloque]	
L56.3	Bệnh mày đay do ánh nắng	
L56.4	Phát ban đa dạng do ánh sáng	
L56.8	Thay đổi cấp tính xác định khác của da do tia cực tím [tia tử ngoại]	
L56.9	Thay đổi cấp tính không xác định của da do tia cực tím [tia tử ngoại]	
L57	Thay đổi của da do phơi nhiễm mạn tính với phóng xạ [bức xạ] không ion hóa	
L57.0	Dày sừng quang hóa	
L57.1	Viêm da ánh sáng dạng mạng lưới	
L57.2	Bệnh dày da gáy [vân hình thoi]	
L57.3	Bệnh da loang lổ vùng cổ	
L57.4	Bệnh nhão da ở người già	Thoái hóa mô đàn hồi ở người già
L57.5	U hạt do ánh sáng	
L57.8	Thay đổi khác ở da do phơi nhiễm mạn tính với phóng xạ [bức xạ] không ion hóa	Da của nhà nông|Da của thủy thủ|Viêm da do ánh nắng
L57.9	Thay đổi khác ở da do phơi nhiễm mạn tính với phóng xạ [bức xạ] không ion hóa, không xác định	
L58	Viêm da do phóng xạ	
L58.0	Viêm da cấp tính do phóng xạ	
L58.1	Viêm da do phóng xạ mạn tính	
L58.9	Viêm da do phóng xạ, không xác định	
L59	Rối loạn khác ở da và/hoặc mô dưới da liên quan đến phóng xạ [bức xạ]	
L59.0	Hồng ban nhiệt [viêm da do nhiệt]	
L59.8	Rối loạn xác định khác ở da và/hoặc mô dưới da liên quan đến phóng xạ [bức xạ]	
L59.9	Rối loạn kở da và/hoặc mô dưới da liên quan đến phóng xạ [bức xạ], không xác định	
L60	Rối loạn của móng	
L60.0	Móng chọc thịt	
L60.1	Bong móng	
L60.2	Móng dày và cong	
L60.3	Loạn dưỡng móng	
L60.4	Đường lõm Beau ở móng	
L60.5	Hội chứng vàng móng	
L60.8	Rối loạn khác của móng	
L60.9	Rối loạn móng khác, không xác định	
L62.*	Rối loạn móng do bệnh phân loại mục khác	
L62.0*	Móng tay dùi trống do tăng sinh màng xương (M89.4†)	
L62.8*	Rối loạn của móng do bệnh khác phân loại mục khác	
L63	Rụng tóc thể mảng	
L63.0	Rụng tóc toàn phần (ở đầu)	
L63.1	Rụng tóc toàn bộ	
L63.2	Rụng tóc thể rắn bò	
L63.8	Rụng tóc thể mảng khác	
L63.9	Rụng tóc thể mảng, không xác định	
L64	Rụng tóc do nội tiết tố nam	
L64.0	Rụng tóc do thuốc nội tiết tố nam	
L64.8	Rụng tóc khác do nội tiết tố nam	
L64.9	Rụng tóc do nội tiết tố nam, không xác định	
L65	Rụng tóc không sẹo khác	
L65.0	Rụng tóc ở giai đoạn tóc ngừng phát triển [telogen efluvium]	
L65.1	Rụng tóc ở giai đoạn tóc đang phát triển [anagen efluvium]	
L65.2	Rụng tóc do lắng đọng chất nhầy	
L65.8	Rụng tóc không sẹo xác định khác	
L65.9	Rụng tóc không sẹo, không xác định	Rụng tóc không xác định khác
L66	Rụng tóc có sẹo	
L66.0	Chứng rụng tóc teo da [pelade]	
L66.1	Bệnh Lichen phẳng nang lông	Lichen phẳng vùng tóc nang lông
L66.2	Viêm nang lông gây rụng tóc	
L66.3	Viêm nang lông da đầu	
L66.4	Viêm nang lông sẹo đỏ dạng lưới	
L66.8	Rụng tóc có sẹo khác	
L66.9	Rụng tóc có sẹo, không xác định	
L67	Bất thường về màu và/hoặc sợi tóc	
L67.0	Bệnh tóc gãy giòn có hạt	
L67.1	Thay đổi màu tóc	
L67.8	Bất thường khác về màu tóc và/hoặc sợi tóc	Chứng tóc giòn
L67.9	Bất thường về màu tóc và/hoặc sợi tóc, không xác định	
L68	Chứng rậm lông tóc	
L68.0	Chứng rậm lông ở phụ nữ	
L68.1	Chứng rậm lông tơ mắc phải	
L68.2	Chứng rậm lông khu trú	
L68.3	Đa nang tóc (có từ 5 sợi lông/tóc mọc trên cùng một lỗ nang)	
L68.8	Chứng rậm lông tóc khác	
L68.9	Chứng rậm lông tóc, không xác định	
L70	Trứng cá	
L70.0	Trứng cá thể thông thường	
L70.1	Trứng cá bọc	
L70.2	Trứng cá dạng thủy đậu	Trứng cá kê hoại tử
L70.3	Trứng cá nhiệt đới	
L70.4	Trứng cá ở trẻ nhỏ	
L70.5	Hội chứng ám ảnh mụn [Acné excoriée]	Trứng cá ở thiếu nữ
L70.8	Trứng cá khác	
L70.9	Trứng cá, không xác định	
L71	Trứng cá đỏ	
L71.0	Viêm da quanh miệng	
L71.1	Bệnh mũi sư tử	
L71.8	Trứng cá đỏ khác	
L71.9	Trứng cá đỏ, không xác định	
L72	Kén nang lông của da và/hoặc mô dưới da	
L72.0	Kén/nang thượng bì	
L72.1	Kén ở nang lông/tuyến bã	U nang bì|U nang bã
L72.2	Đa u nang tuyến bã	
L72.8	Kén/nang khác ở da và/hoặc mô dưới da	
L72.9	Kén nang lông ở da và/hoặc mô dưới da, không xác định	
L73	Bệnh lý nang lông khác	
L73.0	Trứng cá sẹo lồi	
L73.1	Viêm nang râu	
L73.2	Viêm tuyến mồ hôi mủ [nhọt ổ gà]	
L73.8	Bệnh lý nang lông xác định khác	Viêm nang lông ở cằm
L73.9	Bệnh lý nang lông, không xác định	
L74	Rối loạn của tuyến mồ hôi toàn hủy [Eccrine]	
L74.0	Rôm sảy đỏ	
L74.1	Rôm sảy dạng mụn nước	
L74.2	Rôm sảy sâu	Rôm sảy nhiệt đới
L74.3	Rôm sảy, không xác định	
L74.4	Giảm tiết mồ hôi	
L74.8	Rối loạn khác của tuyến mồ hôi toàn hủy [Eccrine]	
L74.9	Rối loạn tuyến mồ hôi toàn hủy [Eccrine], không xác định	Bệnh tuyến mồ hôi không xác định khác
L75	Rối loạn tuyến mồ hôi đầu hủy [Apocrine]	
L75.0	Chứng mồ hôi nặng mùi	
L75.1	Chứng mồ hôi màu	
L75.2	Rôm tuyến mồ hôi đầu hủy [Apocrine]	Bệnh Fox-Fordyce
L75.8	Rối loạn tuyến mồ hôi đầu hủy [Apocrine] khác	
L75.9	Rối loạn tuyến mồ hôi đầu hủy [Apocrine], không xác định	
L80	Bệnh bạch biến	
L81	Rối loạn sắc tố khác	
L81.0	Tăng sắc tố sau viêm	
L81.1	Rám má	
L81.2	Tàn nhang	
L81.3	Dát cà phê sữa	
L81.4	Bệnh tăng sắc tố do melanin khác	Đồi mồi
L81.5	Bệnh da mất sắc tố, không phân loại mục khác	
L81.6	Rối loạn khác do giảm sự hình thành sắc tố melanin	
L81.7	Bệnh da xuất huyết nhiễm sắc tố	U máu dạng gai
L81.8	Rối loạn sắc tố xác định khác	Nhiễm sắc tố do sắt|Nhiễm sắc tố do xăm
L81.9	Rối loạn sắc tố, không xác định	
L82	Bệnh dày sừng tiết bã	
L83	Bệnh gai đen	
L84	Mắt cá và/hoặc vết chai chân	
L85	Dày thượng bì khác	
L85.0	Bệnh da vảy cá mắc phải	
L85.1	Bệnh dày sừng mắc phải [da dày] ở lòng bàn tay và/hoặc bàn chân	
L85.2	Chứng dày sừng đốm (lòng bàn tay và bàn chân)	
L85.3	Da khô và đóng vảy	Viêm da do khô da
L85.8	Dày thượng bì xác định khác	Sừng da
L85.9	Dày thượng bì, không xác định	
L86.*	Dày sừng do bệnh phân loại mục khác	
L87	Rối loạn của bệnh bài tiết qua thượng bì	
L87.0	Dày sừng nang lông và quanh nang lông [Bệnh Kyrle]	Dày sừng nang lông đục lỗ
L87.1	Bệnh tạo keo đục lỗ phản ứng [đào thải collagen biến đổi qua thượng bì]	
L87.2	Bệnh sợi chun đục lỗ ngoằn ngoèo [đào thải sợi chun qua nhú bì]	
L87.8	Rối loạn khác của bệnh bài tiết qua thượng bì	
L87.9	Bệnh loại bỏ dị vật qua thượng bì, không xác định	
L88	Viêm da mủ hoại thư	
L89	Loét do tì đè và/hoặc loét vùng đè ép	
L89.0	Loét do tì đè và/hoặc loét vùng đè ép giai đoạn I	
L89.1	Loét do tì đè, độ II	
L89.2	Loét do tì đè, độ III	
L89.3	Loét do tì đè, độ IV	
L89.9	Loét do tì đè và/hoặc loét vùng đè ép, không xác định	loét do tì đè không đề cập đến giai đoạn
L90	Rối loạn teo da	
L90.0	Lichen xơ teo	
L90.1	Chứng teo da Schweninger-Buzzi	
L90.2	Chứng teo da Jadassohn-Pellizzari	
L90.3	Chứng teo da Pasini và Pierini	
L90.4	Viêm da đầu chi teo mạn tính	
L90.5	Tình trạng sẹo và/hoặc xơ hóa của da	
L90.6	Rạn da teo	
L90.8	Rối loạn teo da khác	
L90.9	Rối loạn teo da, không xác định	
L91	Rối loạn phì đại của da	
L91.0	Sẹo phì đại [sẹo lồi]	
L91.8	Rối loạn phì đại khác của da	
L91.9	Bệnh phì đại của da, không xác định	
L92	U hạt của da và/hoặc mô dưới da	
L92.0	U hạt vòng	U hạt vòng loét
L92.1	Hoại tử mỡ, không phân loại mục khác	
L92.2	U hạt ở mặt [u hạt nhiễm bạch ái toan của da]	
L92.3	U hạt ở da và/hoặc mô dưới da do dị vật	
L92.8	Bệnh u hạt ở da và/hoặc mô dưới da khác	
L92.9	Bệnh u hạt ở da và/hoặc mô dưới da, không xác định	
L93	Bệnh lupus ban đỏ	
L93.0	Bệnh lupus ban đỏ dạng đĩa	Bệnh lupus ban đỏ không xác định khác
L93.1	Bệnh lupus ban đỏ bán cấp	
L93.2	Bệnh lupus ban đỏ khu trú khác	
L94	Rối loạn mô liên kết khu trú khác	
L94.0	Bệnh xơ cứng bì khu trú [dạng mảng]	Bệnh xơ cứng bì thể mảng
L94.1	Bệnh xơ cứng bì dạng dải	Tổn thương hình dao chém
L94.2	Bệnh lắng đọng calci ở da	
L94.3	Xơ hóa da ở đầu chi [cứng ngón]	
L94.4	Sẩn Gottron	
L94.5	Chứng da đốm teo giãn mạch	
L94.6	Bệnh Ainhum	
L94.8	Rối loạn mô liên kết khu trú xác định khác	
L94.9	Bệnh mô liên kết khu trú, không xác định	
L95	Viêm mao mạch ở da, không phân loại mục khác	
L95.0	Viêm mạch dạng mạng lưới	
L95.1	Hồng ban rắn	
L95.8	Viêm mạch khác giới hạn ở da	
L95.9	Viêm mạch giới hạn ở da, không xác định	
L97	Loét chi dưới, không phân loại mục khác	
L98	Rối loạn khác của da và/hoặc mô dưới da, không phân loại mục khác	
L98.0	U hạt sinh mủ	
L98.1	Viêm da tự tạo	Trợt da do tâm thần [chứng giật da do tâm thần]
L98.2	Bệnh da tăng bạch cầu trung tính có sốt [Hội chứng Sweet]	
L98.3	Viêm mô bào tăng bạch cầu ái toan [Hội chứng Wells]	
L98.4	Loét da mạn tính, không phân loại mục khác	
L98.5	Bệnh thoái hóa nhày ở da	
L98.6	Rối loạn có thâm nhiễm khác ở da và/hoặc mô dưới da	
L98.7	Tình trạng thừa da và/hoặc thừa mô dưới da	
L98.8	Rối loạn xác định khác ở da và/hoặc mô dưới da	
L98.9	Rối loạn ở da và/hoặc mô dưới da, không xác định	
L99.*	Rối loạn khác của da và/hoặc mô dưới da do bệnh phân loại mục khác	
L99.0*	Thoái hóa dạng bột ở da (E85.-†)	Lichen thoái hóa tinh bột|Thoái hóa tinh bột thể vàng da
L99.8*	Rối loạn xác định khác của da và/hoặc mô dưới da do bệnh phân loại mục khác	
M00	Viêm khớp nhiễm khuẩn sinh mủ	
M00.0	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn	
M00.00	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, nhiều vị trí	
M00.01	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, vùng vai	
M00.02	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, cánh tay trên	
M00.03	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, cẳng tay	
M00.04	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, bàn tay	
M00.05	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, vùng chậu và/hoặc đùi	
M00.06	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, cẳng chân	
M00.07	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, cổ chân và/hoặc bàn chân	
M00.08	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, vị trí khác	
M00.09	Viêm khớp và/hoặc viêm đa khớp do tụ cầu khuẩn, vị trí không xác định	
M00.1	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn	
M00.10	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, nhiều vị trí	
M00.11	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, vùng vai	
M00.12	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, cánh tay trên	
M00.13	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, cẳng tay	
M00.14	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, bàn tay	
M00.15	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, vùng chậu và/hoặc đùi	
M00.16	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, cẳng chân	
M00.17	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, cổ chân và/hoặc bàn chân	
M00.18	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, vị trí khác	
M00.19	Viêm khớp và/hoặc viêm đa khớp do phế cầu khuẩn, vị trí không xác định	
M00.2	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn	
M00.20	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, nhiều vị trí	
M00.21	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, vùng vai	
M00.22	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, cánh tay trên	
M00.23	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, cẳng tay	
M00.24	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, bàn tay	
M00.25	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, vùng chậu và/hoặc đùi	
M00.26	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, cẳng chân	
M00.27	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, cổ chân và/hoặc bàn chân	
M00.28	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, vị trí khác	
M00.29	Viêm khớp và/hoặc viêm đa khớp khác do liên cầu khuẩn, vị trí không xác định	
M00.8	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác	
M00.80	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, nhiều vị trí	
M00.81	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, vùng vai	
M00.82	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, cánh tay trên	
M00.83	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, cẳng tay	
M00.84	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, bàn tay	
M00.85	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, vùng chậu và/hoặc đùi	
M00.86	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, cẳng chân	
M00.87	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, cổ chân và/hoặc bàn chân	
M00.88	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, vị trí khác	
M00.89	Viêm khớp và/hoặc viêm đa khớp do tác nhân vi khuẩn xác định khác, vị trí không xác định	
M00.9	Viêm khớp nhiễm khuẩn sinh mủ, không xác định	Viêm khớp nhiễm khuẩn không xác định khác
M00.90	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, nhiều vị trí	
M00.91	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, vùng vai	
M00.92	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, cánh tay trên	
M00.93	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, cẳng tay	
M00.94	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, bàn tay	
M00.95	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, vùng chậu và/hoặc đùi	
M00.96	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, cẳng chân	
M00.97	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, cổ chân và/hoặc bàn chân	
M00.98	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, vị trí khác	
M00.99	Viêm khớp nhiễm khuẩn sinh mủ, không xác định, vị trí không xác định	
M01.*	Viêm khớp nhiễm trùng trực tiếp do bệnh nhiễm khuẩn và/hoặc ký sinh trùng phân loại mục khác	
M01.0*	Viêm khớp do nhiễm não mô cầu (A39.8†)	
M01.00*	Viêm khớp do nhiễm não mô cầu (A39.8†), nhiều vị trí	
M01.01*	Viêm khớp do nhiễm não mô cầu (A39.8†), vùng vai	
M01.02*	Viêm khớp do nhiễm não mô cầu (A39.8†), cánh tay trên	
M01.03*	Viêm khớp do nhiễm não mô cầu (A39.8†), cẳng tay	
M01.04*	Viêm khớp do nhiễm não mô cầu (A39.8†), bàn tay	
M01.05*	Viêm khớp do nhiễm não mô cầu (A39.8†), vùng chậu và/hoặc đùi	
M01.06*	Viêm khớp do nhiễm não mô cầu (A39.8†), cẳng chân	
M01.07*	Viêm khớp do nhiễm não mô cầu (A39.8†), cổ chân và/hoặc bàn chân	
M01.08*	Viêm khớp do nhiễm não mô cầu (A39.8†), vị trí khác	
M01.09*	Viêm khớp do nhiễm não mô cầu (A39.8†), vị trí không xác định	
M01.1*	Viêm khớp do lao (A18.0†)	
M01.10*	Viêm khớp do lao (A18.0†), nhiều vị trí	
M01.11*	Viêm khớp do lao (A18.0†), vùng vai	
M01.12*	Viêm khớp do lao (A18.0†), cánh tay trên	
M01.13*	Viêm khớp do lao (A18.0†), cẳng tay	
M01.14*	Viêm khớp do lao (A18.0†), bàn tay	
M01.15*	Viêm khớp do lao (A18.0†), vùng chậu và/hoặc đùi	
M01.16*	Viêm khớp do lao (A18.0†), cẳng chân	
M01.17*	Viêm khớp do lao (A18.0†), cổ chân và/hoặc bàn chân	
M01.18*	Viêm khớp do lao (A18.0†), vị trí khác	
M01.19*	Viêm khớp do lao (A18.0†), vị trí không xác định	
M01.2*	Viêm khớp do bệnh Lyme (A69.2†)	
M01.20*	Viêm khớp do bệnh Lyme (A69.2†), nhiều vị trí	
M01.21*	Viêm khớp do bệnh Lyme (A69.2†), vùng vai	
M01.22*	Viêm khớp do bệnh Lyme (A69.2†), cánh tay trên	
M01.23*	Viêm khớp do bệnh Lyme (A69.2†), cẳng tay	
M01.24*	Viêm khớp do bệnh Lyme (A69.2†), bàn tay	
M01.25*	Viêm khớp do bệnh Lyme (A69.2†), vùng chậu và/hoặc đùi	
M01.26*	Viêm khớp do bệnh Lyme (A69.2†), cẳng chân	
M01.27*	Viêm khớp do bệnh Lyme (A69.2†), cổ chân và/hoặc bàn chân	
M01.28*	Viêm khớp do bệnh Lyme (A69.2†), vị trí khác	
M01.29*	Viêm khớp do bệnh Lyme (A69.2†), vị trí không xác định	
M01.3*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác	
M01.30*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, nhiều vị trí	
M01.31*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, vùng vai	
M01.32*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, cánh tay trên	
M01.33*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, cẳng tay	
M01.34*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, bàn tay	
M01.35*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, vùng chậu và/hoặc đùi	
M01.36*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, cẳng chân	
M01.37*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, cổ chân và/hoặc bàn chân	
M01.38*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, vị trí khác	
M01.39*	Viêm khớp do bệnh nhiễm khuẩn khác phân loại mục khác, vị trí không xác định	
M01.4*	Viêm khớp do bệnh rubella (B06.8†)	
M01.40*	Viêm khớp do bệnh rubella (B06.8†), nhiều vị trí	
M01.41*	Viêm khớp do bệnh rubella (B06.8†), vùng vai	
M01.42*	Viêm khớp do bệnh rubella (B06.8†), cánh tay trên	
M01.43*	Viêm khớp do bệnh rubella (B06.8†), cẳng tay	
M01.44*	Viêm khớp do bệnh rubella (B06.8†), bàn tay	
M01.45*	Viêm khớp do bệnh rubella (B06.8†), vùng chậu và/hoặc đùi	
M01.46*	Viêm khớp do bệnh rubella (B06.8†), cẳng chân	
M01.47*	Viêm khớp do bệnh rubella (B06.8†), cổ chân và/hoặc bàn chân	
M01.48*	Viêm khớp do bệnh rubella (B06.8†), vị trí khác	
M01.49*	Viêm khớp do bệnh rubella (B06.8†), vị trí không xác định	
M01.5*	Viêm khớp do bệnh nhiễm virus phân loại mục khác	
M01.50*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, nhiều vị trí	
M01.51*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, vùng vai	
M01.52*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, cánh tay trên	
M01.53*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, cẳng tay	
M01.54*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, bàn tay	
M01.55*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, vùng chậu và/hoặc đùi	
M01.56*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, cẳng chân	
M01.57*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, cổ chân và/hoặc bàn chân	
M01.58*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, vị trí khác	
M01.59*	Viêm khớp do bệnh nhiễm virus phân loại mục khác, vị trí không xác định	
M01.6*	Viêm khớp do nấm (B35-B49†)	
M01.60*	Viêm khớp do nấm (B35-B49†), nhiều vị trí	
M01.61*	Viêm khớp do nấm (B35-B49†), vùng vai	
M01.62*	Viêm khớp do nấm (B35-B49†), cánh tay trên	
M01.63*	Viêm khớp do nấm (B35-B49†), cẳng tay	
M01.64*	Viêm khớp do nấm (B35-B49†), bàn tay	
M01.65*	Viêm khớp do nấm (B35-B49†), vùng chậu và/hoặc đùi	
M01.66*	Viêm khớp do nấm (B35-B49†), cẳng chân	
M01.67*	Viêm khớp do nấm (B35-B49†), cổ chân và/hoặc bàn chân	
M01.68*	Viêm khớp do nấm (B35-B49†), vị trí khác	
M01.69*	Viêm khớp do nấm (B35-B49†), vị trí không xác định	
M01.8*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác	
M01.80*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, nhiều vị trí	
M01.81*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, vùng vai	
M01.82*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, cánh tay trên	
M01.83*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, cẳng tay	
M01.84*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, bàn tay	
M01.85*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, vùng chậu và/hoặc đùi	
M01.86*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, cẳng chân	
M01.87*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, cổ chân và/hoặc bàn chân	
M01.88*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, vị trí khác	
M01.89*	Viêm khớp do bệnh nhiễm trùng và/hoặc ký sinh trùng khác phân loại mục khác, vị trí không xác định	
M02	Bệnh lý khớp phản ứng	
M02.0	Bệnh lý khớp sau phẫu thuật nối tắt ruột	
M02.00	Bệnh lý khớp sau phẫu thuật nối tắt ruột, nhiều vị trí	
M02.01	Bệnh lý khớp sau phẫu thuật nối tắt ruột, vùng vai	
M02.02	Bệnh lý khớp sau phẫu thuật nối tắt ruột, cánh tay trên	
M02.03	Bệnh lý khớp sau phẫu thuật nối tắt ruột, cẳng tay	
M02.04	Bệnh lý khớp sau phẫu thuật nối tắt ruột, bàn tay	
M02.05	Bệnh lý khớp sau phẫu thuật nối tắt ruột, vùng chậu và/hoặc đùi	
M02.06	Bệnh lý khớp sau phẫu thuật nối tắt ruột, cẳng chân	
M02.07	Bệnh lý khớp sau phẫu thuật nối tắt ruột, cổ chân và/hoặc bàn chân	
M02.08	Bệnh lý khớp sau phẫu thuật nối tắt ruột, vị trí khác	
M02.09	Bệnh lý khớp sau phẫu thuật nối tắt ruột, vị trí không xác định	
M02.1	Bệnh lý khớp sau bệnh lỵ	
M02.10	Bệnh lý khớp sau bệnh lỵ, nhiều vị trí	
M02.11	Bệnh lý khớp sau bệnh lỵ, vùng vai	
M02.12	Bệnh lý khớp sau bệnh lỵ, cánh tay trên	
M02.13	Bệnh lý khớp sau bệnh lỵ, cẳng tay	
M02.14	Bệnh lý khớp sau bệnh lỵ, bàn tay	
M02.15	Bệnh lý khớp sau bệnh lỵ, vùng chậu và/hoặc đùi	
M02.16	Bệnh lý khớp sau bệnh lỵ, cẳng chân	
M02.17	Bệnh lý khớp sau bệnh lỵ, cổ chân và/hoặc bàn chân	
M02.18	Bệnh lý khớp sau bệnh lỵ, vị trí khác	
M02.19	Bệnh lý khớp sau bệnh lỵ, vị trí không xác định	
M02.2	Bệnh lý khớp sau tiêm vắc xin	
M02.20	Bệnh lý khớp sau tiêm vắc xin, nhiều vị trí	
M02.21	Bệnh lý khớp sau tiêm vắc xin, vùng vai	
M02.22	Bệnh lý khớp sau tiêm vắc xin, cánh tay trên	
M02.23	Bệnh lý khớp sau tiêm vắc xin, cẳng tay	
M02.24	Bệnh lý khớp sau tiêm vắc xin, bàn tay	
M02.25	Bệnh lý khớp sau tiêm vắc xin, vùng chậu và/hoặc đùi	
M02.26	Bệnh lý khớp sau tiêm vắc xin, cẳng chân	
M02.27	Bệnh lý khớp sau tiêm vắc xin, cổ chân và/hoặc bàn chân	
M02.28	Bệnh lý khớp sau tiêm vắc xin, vị trí khác	
M02.29	Bệnh lý khớp sau tiêm vắc xin, vị trí không xác định	
M02.3	Bệnh Reiter [viêm khớp phản ứng]	
M02.30	Bệnh Reiter [viêm khớp phản ứng], nhiều vị trí	
M02.31	Bệnh Reiter [viêm khớp phản ứng], vùng vai	
M02.32	Bệnh Reiter [viêm khớp phản ứng], cánh tay trên	
M02.33	Bệnh Reiter [viêm khớp phản ứng], cẳng tay	
M02.34	Bệnh Reiter [viêm khớp phản ứng], bàn tay	
M02.35	Bệnh Reiter [viêm khớp phản ứng], vùng chậu và/hoặc đùi	
M02.36	Bệnh Reiter [viêm khớp phản ứng], cẳng chân	
M02.37	Bệnh Reiter [viêm khớp phản ứng], cổ chân và/hoặc bàn chân	
M02.38	Bệnh Reiter [viêm khớp phản ứng], vị trí khác	
M02.39	Bệnh Reiter [viêm khớp phản ứng], vị trí không xác định	
M02.8	Bệnh lý khớp phản ứng khác	
M02.80	Bệnh lý khớp phản ứng khác, nhiều vị trí	
M02.81	Bệnh lý khớp phản ứng khác, vùng vai	
M02.82	Bệnh lý khớp phản ứng khác, cánh tay trên	
M02.83	Bệnh lý khớp phản ứng khác, cẳng tay	
M02.84	Bệnh lý khớp phản ứng khác, bàn tay	
M02.85	Bệnh lý khớp phản ứng khác, vùng chậu và/hoặc đùi	
M02.86	Bệnh lý khớp phản ứng khác, cẳng chân	
M02.87	Bệnh lý khớp phản ứng khác, cổ chân và/hoặc bàn chân	
M02.88	Bệnh lý khớp phản ứng khác, vị trí khác	
M02.89	Bệnh lý khớp phản ứng khác, vị trí không xác định	
M02.9	Bệnh lý khớp phản ứng, không xác định	
M02.90	Bệnh lý khớp phản ứng, không xác định, nhiều vị trí	
M02.91	Bệnh lý khớp phản ứng, không xác định, vùng vai	
M02.92	Bệnh lý khớp phản ứng, không xác định, cánh tay trên	
M02.93	Bệnh lý khớp phản ứng, không xác định, cẳng tay	
M02.94	Bệnh lý khớp phản ứng, không xác định, bàn tay	
M02.95	Bệnh lý khớp phản ứng, không xác định, vùng chậu và/hoặc đùi	
M02.96	Bệnh lý khớp phản ứng, không xác định, cẳng chân	
M02.97	Bệnh lý khớp phản ứng, không xác định, cổ chân và/hoặc bàn chân	
M02.98	Bệnh lý khớp phản ứng, không xác định, vị trí khác	
M02.99	Bệnh lý khớp phản ứng, không xác định, vị trí không xác định	
M03.*	Bệnh lý khớp sau nhiễm trùng và/hoặc bệnh lý khớp phản ứng phân loại mục khác	
M03.0*	Viêm khớp sau nhiễm não mô cầu (A39.8†)	
M03.00*	Viêm khớp sau nhiễm não mô cầu (A39.8†), nhiều vị trí	
M03.01*	Viêm khớp sau nhiễm não mô cầu (A39.8†), vùng vai	
M03.02*	Viêm khớp sau nhiễm não mô cầu (A39.8†), cánh tay trên	
M03.03*	Viêm khớp sau nhiễm não mô cầu (A39.8†), cẳng tay	
M03.04*	Viêm khớp sau nhiễm não mô cầu (A39.8†), bàn tay	
M03.05*	Viêm khớp sau nhiễm não mô cầu (A39.8†), vùng chậu và/hoặc đùi	
M03.06*	Viêm khớp sau nhiễm não mô cầu (A39.8†), cẳng chân	
M03.07*	Viêm khớp sau nhiễm não mô cầu (A39.8†), cổ chân và/hoặc bàn chân	
M03.08*	Viêm khớp sau nhiễm não mô cầu (A39.8†), vị trí khác	
M03.09*	Viêm khớp sau nhiễm não mô cầu (A39.8†), vị trí không xác định	
M03.1*	Bệnh lý khớp sau nhiễm giang mai	
M03.10*	Bệnh lý khớp sau nhiễm giang mai, nhiều vị trí	
M03.11*	Bệnh lý khớp sau nhiễm giang mai, vùng vai	
M03.12*	Bệnh lý khớp sau nhiễm giang mai, cánh tay trên	
M03.13*	Bệnh lý khớp sau nhiễm giang mai, cẳng tay	
M03.14*	Bệnh lý khớp sau nhiễm giang mai, bàn tay	
M03.15*	Bệnh lý khớp sau nhiễm giang mai, vùng chậu và/hoặc đùi	
M03.16*	Bệnh lý khớp sau nhiễm giang mai, cẳng chân	
M03.17*	Bệnh lý khớp sau nhiễm giang mai, cổ chân và/hoặc bàn chân	
M03.18*	Bệnh lý khớp sau nhiễm giang mai, vị trí khác	
M03.19*	Bệnh lý khớp sau nhiễm giang mai, vị trí không xác định	
M03.2*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác	
M03.20*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, nhiều vị trí	
M03.21*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, vùng vai	
M03.22*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, cánh tay trên	
M03.23*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, cẳng tay	
M03.24*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, bàn tay	
M03.25*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, vùng chậu và/hoặc đùi	
M03.26*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, cẳng chân	
M03.27*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, cổ chân và/hoặc bàn chân	
M03.28*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, vị trí khác	
M03.29*	Bệnh lý khớp sau nhiễm trùng khác do bệnh phân loại mục khác, vị trí không xác định	
M03.6*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác	
M03.60*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, nhiều vị trí	
M03.61*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, vùng vai	
M03.62*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, cánh tay trên	
M03.63*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, cẳng tay	
M03.64*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, bàn tay	
M03.65*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, vùng chậu và/hoặc đùi	
M03.66*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, cẳng chân	
M03.67*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, cổ chân và/hoặc bàn chân	
M03.68*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, vị trí khác	
M03.69*	Bệnh lý khớp phản ứng do bệnh khác phân loại mục khác, vị trí không xác định	
M05	Viêm khớp dạng thấp huyết thanh dương tính	
M05.0	Hội chứng Felty	Viêm khớp dạng thấp có biểu hiện phì đại lách và/hoặc giảm bạch cầu
M05.00	Hội chứng Felty, nhiều vị trí	
M05.01	Hội chứng Felty, vùng vai	
M05.02	Hội chứng Felty, cánh tay trên	
M05.03	Hội chứng Felty, cẳng tay	
M05.04	Hội chứng Felty, bàn tay	
M05.05	Hội chứng Felty, vùng chậu và/hoặc đùi	
M05.06	Hội chứng Felty, cẳng chân	
M05.07	Hội chứng Felty, cổ chân và/hoặc bàn chân	
M05.08	Hội chứng Felty, vị trí khác	
M05.09	Hội chứng Felty, vị trí không xác định	
M05.10†	Bệnh phổi dạng thấp (J99.0*), nhiều vị trí	
M05.11†	Bệnh phổi dạng thấp (J99.0*), vùng vai	
M05.12†	Bệnh phổi dạng thấp (J99.0*), cánh tay trên	
M05.13†	Bệnh phổi dạng thấp (J99.0*), cẳng tay	
M05.14†	Bệnh phổi dạng thấp (J99.0*), bàn tay	
M05.15†	Bệnh phổi dạng thấp (J99.0*), vùng chậu và/hoặc đùi	
M05.16†	Bệnh phổi dạng thấp (J99.0*), cẳng chân	
M05.17†	Bệnh phổi dạng thấp (J99.0*), cổ chân và/hoặc bàn chân	
M05.18†	Bệnh phổi dạng thấp (J99.0*), vị trí khác	
M05.19†	Bệnh phổi dạng thấp (J99.0*), vị trí không xác định	
M05.1†	Bệnh phổi dạng thấp (J99.0*)	
M05.2	Viêm mạch máu dạng thấp	
M05.20	Viêm mạch máu dạng thấp, nhiều vị trí	
M05.21	Viêm mạch máu dạng thấp, vùng vai	
M05.22	Viêm mạch máu dạng thấp, cánh tay trên	
M05.23	Viêm mạch máu dạng thấp, cẳng tay	
M05.24	Viêm mạch máu dạng thấp, bàn tay	
M05.25	Viêm mạch máu dạng thấp, vùng chậu và/hoặc đùi	
M05.26	Viêm mạch máu dạng thấp, cẳng chân	
M05.27	Viêm mạch máu dạng thấp, cổ chân và/hoặc bàn chân	
M05.28	Viêm mạch máu dạng thấp, vị trí khác	
M05.29	Viêm mạch máu dạng thấp, vị trí không xác định	
M05.30†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, nhiều vị trí	
M05.31†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, vùng vai	
M05.32†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, cánh tay trên	
M05.33†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, cẳng tay	
M05.34†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, bàn tay	
M05.35†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, vùng chậu và/hoặc đùi	
M05.36†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, cẳng chân	
M05.37†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, cổ chân và/hoặc bàn chân	
M05.38†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, vị trí khác	
M05.39†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác, vị trí không xác định	
M05.3†	Viêm khớp dạng thấp có tác động đến tạng và/hoặc hệ thống khác	
M05.8	Viêm khớp dạng thấp huyết thanh dương tính khác	
M05.80	Viêm khớp dạng thấp huyết thanh dương tính khác, nhiều vị trí	
M05.81	Viêm khớp dạng thấp huyết thanh dương tính khác, vùng vai	
M05.82	Viêm khớp dạng thấp huyết thanh dương tính khác, cánh tay trên	
M05.83	Viêm khớp dạng thấp huyết thanh dương tính khác, cẳng tay	
M05.84	Viêm khớp dạng thấp huyết thanh dương tính khác, bàn tay	
M05.85	Viêm khớp dạng thấp huyết thanh dương tính khác, vùng chậu và/hoặc đùi	
M05.86	Viêm khớp dạng thấp huyết thanh dương tính khác, cẳng chân	
M05.87	Viêm khớp dạng thấp huyết thanh dương tính khác, cổ chân và/hoặc bàn chân	
M05.88	Viêm khớp dạng thấp huyết thanh dương tính khác, vị trí khác	
M05.89	Viêm khớp dạng thấp huyết thanh dương tính khác, vị trí không xác định	
M05.9	Viêm khớp dạng thấp huyết thanh dương tính, không xác định	
M05.90	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, nhiều vị trí	
M05.91	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, vùng vai	
M05.92	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, cánh tay trên	
M05.93	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, cẳng tay	
M05.94	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, bàn tay	
M05.95	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, vùng chậu và/hoặc đùi	
M05.96	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, cẳng chân	
M05.97	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, cổ chân và/hoặc bàn chân	
M05.98	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, vị trí khác	
M05.99	Viêm khớp dạng thấp huyết thanh dương tính, không xác định, vị trí không xác định	
M06	Viêm khớp dạng thấp khác	
M06.0	Viêm khớp dạng thấp huyết thanh âm tính	
M06.00	Viêm khớp dạng thấp huyết thanh âm tính, nhiều vị trí	
M06.01	Viêm khớp dạng thấp huyết thanh âm tính, vùng vai	
M06.02	Viêm khớp dạng thấp huyết thanh âm tính, cánh tay trên	
M06.03	Viêm khớp dạng thấp huyết thanh âm tính, cẳng tay	
M06.04	Viêm khớp dạng thấp huyết thanh âm tính, bàn tay	
M06.05	Viêm khớp dạng thấp huyết thanh âm tính, vùng chậu và/hoặc đùi	
M06.06	Viêm khớp dạng thấp huyết thanh âm tính, cẳng chân	
M06.07	Viêm khớp dạng thấp huyết thanh âm tính, cổ chân và/hoặc bàn chân	
M06.08	Viêm khớp dạng thấp huyết thanh âm tính, vị trí khác	
M06.09	Viêm khớp dạng thấp huyết thanh âm tính, vị trí không xác định	
M06.1	Bệnh Still khởi phát ở người lớn	
M06.10	Bệnh Still khởi phát ở người lớn, nhiều vị trí	
M06.11	Bệnh Still khởi phát ở người lớn, vùng vai	
M06.12	Bệnh Still khởi phát ở người lớn, cánh tay trên	
M06.13	Bệnh Still khởi phát ở người lớn, cẳng tay	
M06.14	Bệnh Still khởi phát ở người lớn, bàn tay	
M06.15	Bệnh Still khởi phát ở người lớn, vùng chậu và/hoặc đùi	
M06.16	Bệnh Still khởi phát ở người lớn, cẳng chân	
M06.17	Bệnh Still khởi phát ở người lớn, cổ chân và/hoặc bàn chân	
M06.18	Bệnh Still khởi phát ở người lớn, vị trí khác	
M06.19	Bệnh Still khởi phát ở người lớn, vị trí không xác định	
M06.2	Viêm bao hoạt dịch dạng thấp	
M06.20	Viêm bao hoạt dịch dạng thấp, nhiều vị trí	
M06.21	Viêm bao hoạt dịch dạng thấp, vùng vai	
M06.22	Viêm bao hoạt dịch dạng thấp, cánh tay trên	
M06.23	Viêm bao hoạt dịch dạng thấp, cẳng tay	
M06.24	Viêm bao hoạt dịch dạng thấp, bàn tay	
M06.25	Viêm bao hoạt dịch dạng thấp, vùng chậu và/hoặc đùi	
M06.26	Viêm bao hoạt dịch dạng thấp, cẳng chân	
M06.27	Viêm bao hoạt dịch dạng thấp, cổ chân và/hoặc bàn chân	
M06.28	Viêm bao hoạt dịch dạng thấp, vị trí khác	
M06.29	Viêm bao hoạt dịch dạng thấp, vị trí không xác định	
M06.3	Nốt dạng thấp	
M06.30	Nốt dạng thấp, nhiều vị trí	
M06.31	Nốt dạng thấp, vùng vai	
M06.32	Nốt dạng thấp, cánh tay trên	
M06.33	Nốt dạng thấp, cẳng tay	
M06.34	Nốt dạng thấp, bàn tay	
M06.35	Nốt dạng thấp, vùng chậu và/hoặc đùi	
M06.36	Nốt dạng thấp, cẳng chân	
M06.37	Nốt dạng thấp, cổ chân và/hoặc bàn chân	
M06.38	Nốt dạng thấp, vị trí khác	
M06.39	Nốt dạng thấp, vị trí không xác định	
M06.4	Bệnh lý viêm đa khớp	
M06.40	Bệnh lý viêm đa khớp, nhiều vị trí	
M06.41	Bệnh lý viêm đa khớp, vùng vai	
M06.42	Bệnh lý viêm đa khớp, cánh tay trên	
M06.43	Bệnh lý viêm đa khớp, cẳng tay	
M06.44	Bệnh lý viêm đa khớp, bàn tay	
M06.45	Bệnh lý viêm đa khớp, vùng chậu và/hoặc đùi	
M06.46	Bệnh lý viêm đa khớp, cẳng chân	
M06.47	Bệnh lý viêm đa khớp, cổ chân và/hoặc bàn chân	
M06.48	Bệnh lý viêm đa khớp, vị trí khác	
M06.49	Bệnh lý viêm đa khớp, vị trí không xác định	
M06.8	Viêm khớp dạng thấp xác định khác	
M06.80	Viêm khớp dạng thấp xác định khác, nhiều vị trí	
M06.81	Viêm khớp dạng thấp xác định khác, vùng vai	
M06.82	Viêm khớp dạng thấp xác định khác, cánh tay trên	
M06.83	Viêm khớp dạng thấp xác định khác, cẳng tay	
M06.84	Viêm khớp dạng thấp xác định khác, bàn tay	
M06.85	Viêm khớp dạng thấp xác định khác, vùng chậu và/hoặc đùi	
M06.86	Viêm khớp dạng thấp xác định khác, cẳng chân	
M06.87	Viêm khớp dạng thấp xác định khác, cổ chân và/hoặc bàn chân	
M06.88	Viêm khớp dạng thấp xác định khác, vị trí khác	
M06.89	Viêm khớp dạng thấp xác định khác, vị trí không xác định	
M06.9	Viêm khớp dạng thấp, không xác định	
M06.90	Viêm khớp dạng thấp, không xác định, nhiều vị trí	
M06.91	Viêm khớp dạng thấp, không xác định, vùng vai	
M06.92	Viêm khớp dạng thấp, không xác định, cánh tay trên	
M06.93	Viêm khớp dạng thấp, không xác định, cẳng tay	
M06.94	Viêm khớp dạng thấp, không xác định, bàn tay	
M06.95	Viêm khớp dạng thấp, không xác định, vùng chậu và/hoặc đùi	
M06.96	Viêm khớp dạng thấp, không xác định, cẳng chân	
M06.97	Viêm khớp dạng thấp, không xác định, cổ chân và/hoặc bàn chân	
M06.98	Viêm khớp dạng thấp, không xác định, vị trí khác	
M06.99	Viêm khớp dạng thấp, không xác định, vị trí không xác định	
M07.*	Bệnh lý khớp do vảy nến và/hoặc do bệnh lý viêm ruột	
M07.0*	Bệnh lý khớp do vảy nến có tổn thương khớp ngón xa (L40.5†)	
M07.00*	Bệnh lý khớp do vảy nến có tổn thương khớp ngón xa (L40.5†), nhiều vị trí	
M07.04*	Bệnh lý khớp do vảy nến có tổn thương khớp ngón xa (L40.5†), bàn tay	
M07.07*	Bệnh lý khớp do vảy nến có tổn thương khớp ngón xa (L40.5†), cổ chân và/hoặc bàn chân	
M07.09*	Bệnh lý khớp do vảy nến có tổn thương khớp ngón xa (L40.5†), vị trí không xác định	
M07.1*	Viêm khớp thể nặng (L40.5†)	
M07.10*	Viêm khớp thể nặng (L40.5†), nhiều vị trí	
M07.11*	Viêm khớp thể nặng (L40.5†), vùng vai	
M07.12*	Viêm khớp thể nặng (L40.5†), cánh tay trên	
M07.13*	Viêm khớp thể nặng (L40.5†), cẳng tay	
M07.14*	Viêm khớp thể nặng (L40.5†), bàn tay	
M07.15*	Viêm khớp thể nặng (L40.5†), vùng chậu và/hoặc đùi	
M07.16*	Viêm khớp thể nặng (L40.5†), cẳng chân	
M07.17*	Viêm khớp thể nặng (L40.5†), cổ chân và/hoặc bàn chân	
M07.18*	Viêm khớp thể nặng (L40.5†), vị trí khác	
M07.19*	Viêm khớp thể nặng (L40.5†), vị trí không xác định	
M07.2*	Viêm cột sống do vảy nến (L40.5†)	
M07.3*	Bệnh lý khớp khác do vảy nến (L40.5†)	
M07.30*	Bệnh lý khớp khác do vảy nến (L40.5†), nhiều vị trí	
M07.31*	Bệnh lý khớp khác do vảy nến (L40.5†), vùng vai	
M07.32*	Bệnh lý khớp khác do vảy nến (L40.5†), cánh tay trên	
M07.33*	Bệnh lý khớp khác do vảy nến (L40.5†), cẳng tay	
M07.34*	Bệnh lý khớp khác do vảy nến (L40.5†), bàn tay	
M07.35*	Bệnh lý khớp khác do vảy nến (L40.5†), vùng chậu và/hoặc đùi	
M07.36*	Bệnh lý khớp khác do vảy nến (L40.5†), cẳng chân	
M07.37*	Bệnh lý khớp khác do vảy nến (L40.5†), cổ chân và/hoặc bàn chân	
M07.38*	Bệnh lý khớp khác do vảy nến (L40.5†), vị trí khác	
M07.39*	Bệnh lý khớp khác do vảy nến (L40.5†), vị trí không xác định	
M07.4*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†)	
M07.40*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), nhiều vị trí	
M07.41*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), vùng vai	
M07.42*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), cánh tay trên	
M07.43*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), cẳng tay	
M07.44*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), bàn tay	
M07.45*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), vùng chậu và/hoặc đùi	
M07.46*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), cẳng chân	
M07.47*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), cổ chân và/hoặc bàn chân	
M07.48*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), vị trí khác	
M07.49*	Bệnh lý khớp do bệnh Crohn [Viêm ruột từng vùng] (K50.-†), vị trí không xác định	
M07.5*	Bệnh lý khớp do viêm loét đại tràng (K51.-†)	
M07.50*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), nhiều vị trí	
M07.51*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), vùng vai	
M07.52*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), cánh tay trên	
M07.53*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), cẳng tay	
M07.54*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), bàn tay	
M07.55*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), vùng chậu và/hoặc đùi	
M07.56*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), cẳng chân	
M07.57*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), cổ chân và/hoặc bàn chân	
M07.58*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), vị trí khác	
M07.59*	Bệnh lý khớp do viêm loét đại tràng (K51.-†), vị trí không xác định	
M07.6*	Bệnh lý khớp do bệnh lý viêm ruột khác	
M07.60*	Bệnh lý khớp do bệnh lý viêm ruột khác, nhiều vị trí	
M07.61*	Bệnh lý khớp do bệnh lý viêm ruột khác, vùng vai	
M07.62*	Bệnh lý khớp do bệnh lý viêm ruột khác, cánh tay trên	
M07.63*	Bệnh lý khớp do bệnh lý viêm ruột khác, cẳng tay	
M07.64*	Bệnh lý khớp do bệnh lý viêm ruột khác, bàn tay	
M07.65*	Bệnh lý khớp do bệnh lý viêm ruột khác, vùng chậu và/hoặc đùi	
M07.66*	Bệnh lý khớp do bệnh lý viêm ruột khác, cẳng chân	
M07.67*	Bệnh lý khớp do bệnh lý viêm ruột khác, cổ chân và/hoặc bàn chân	
M07.68*	Bệnh lý khớp do bệnh lý viêm ruột khác, vị trí khác	
M07.69*	Bệnh lý khớp do bệnh lý viêm ruột khác, vị trí không xác định	
M08	Viêm khớp thiếu niên	
M08.0	Viêm khớp dạng thấp ở thiếu niên	Viêm khớp dạng thấp ở thiếu niên có hoặc không có yếu tố dạng thấp
M08.00	Viêm khớp dạng thấp ở thiếu niên, nhiều vị trí	
M08.01	Viêm khớp dạng thấp ở thiếu niên, vùng vai	
M08.02	Viêm khớp dạng thấp ở thiếu niên, cánh tay trên	
M08.03	Viêm khớp dạng thấp ở thiếu niên, cẳng tay	
M08.04	Viêm khớp dạng thấp ở thiếu niên, bàn tay	
M08.05	Viêm khớp dạng thấp ở thiếu niên, vùng chậu và/hoặc đùi	
M08.06	Viêm khớp dạng thấp ở thiếu niên, cẳng chân	
M08.07	Viêm khớp dạng thấp ở thiếu niên, cổ chân và/hoặc bàn chân	
M08.08	Viêm khớp dạng thấp ở thiếu niên, vị trí khác	
M08.09	Viêm khớp dạng thấp ở thiếu niên, vị trí không xác định	
M08.1	Viêm cột sống dính khớp ở thiếu niên	
M08.10	Viêm cột sống dính khớp ở thiếu niên, nhiều vị trí	
M08.11	Viêm cột sống dính khớp ở thiếu niên, vùng vai	
M08.12	Viêm cột sống dính khớp ở thiếu niên, cánh tay trên	
M08.13	Viêm cột sống dính khớp ở thiếu niên, cẳng tay	
M08.14	Viêm cột sống dính khớp ở thiếu niên, bàn tay	
M08.15	Viêm cột sống dính khớp ở thiếu niên, vùng chậu và/hoặc đùi	
M08.16	Viêm cột sống dính khớp ở thiếu niên, cẳng chân	
M08.17	Viêm cột sống dính khớp ở thiếu niên, cổ chân và/hoặc bàn chân	
M08.18	Viêm cột sống dính khớp ở thiếu niên, vị trí khác	
M08.19	Viêm cột sống dính khớp ở thiếu niên, vị trí không xác định	
M08.2	Viêm khớp thiếu niên khởi phát hệ thống	
M08.20	Viêm khớp thiếu niên khởi phát hệ thống, nhiều vị trí	
M08.21	Viêm khớp thiếu niên khởi phát hệ thống, vùng vai	
M08.22	Viêm khớp thiếu niên khởi phát hệ thống, cánh tay trên	
M08.23	Viêm khớp thiếu niên khởi phát hệ thống, cẳng tay	
M08.24	Viêm khớp thiếu niên khởi phát hệ thống, bàn tay	
M08.25	Viêm khớp thiếu niên khởi phát hệ thống, vùng chậu và/hoặc đùi	
M08.26	Viêm khớp thiếu niên khởi phát hệ thống, cẳng chân	
M08.27	Viêm khớp thiếu niên khởi phát hệ thống, cổ chân và/hoặc bàn chân	
M08.28	Viêm khớp thiếu niên khởi phát hệ thống, vị trí khác	
M08.29	Viêm khớp thiếu niên khởi phát hệ thống, vị trí không xác định	
M08.3	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên	Viêm khớp mạn tính ở thiếu niên
M08.30	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, nhiều vị trí	
M08.31	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, vùng vai	
M08.32	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, cánh tay trên	
M08.33	Viêm khớp huyết thanh âm tính ở thiếu niên, cẳng tay	
M08.34	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, bàn tay	
M08.35	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, vùng chậu và/hoặc đùi	
M08.36	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, cẳng chân	
M08.37	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, cổ chân và/hoặc bàn chân	
M08.38	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, vị trí khác	
M08.39	Viêm đa khớp (huyết thanh âm tính) ở thiếu niên, vị trí không xác định	
M08.4	Viêm khớp thiếu niên loại viêm ít khớp	
M08.40	Viêm khớp thiếu niên loại viêm ít khớp, nhiều vị trí	
M08.41	Viêm khớp thiếu niên loại viêm ít khớp, vùng vai	
M08.42	Viêm khớp thiếu niên loại viêm ít khớp, cánh tay trên	
M08.43	Viêm khớp thiếu niên loại viêm ít khớp, cẳng tay	
M08.44	Viêm khớp thiếu niên loại viêm ít khớp, bàn tay	
M08.45	Viêm khớp thiếu niên loại viêm ít khớp, vùng chậu và/hoặc đùi	
M08.46	Viêm khớp thiếu niên loại viêm ít khớp, cẳng chân	
M08.47	Viêm khớp thiếu niên loại viêm ít khớp, cổ chân và/hoặc bàn chân	
M08.48	Viêm khớp thiếu niên loại viêm ít khớp, vị trí khác	
M08.49	Viêm khớp thiếu niên loại viêm ít khớp, vị trí không xác định	
M08.8	Viêm khớp thiếu niên khác	
M08.80	Viêm khớp thiếu niên khác, nhiều vị trí	
M08.81	Viêm khớp thiếu niên khác, vùng vai	
M08.82	Viêm khớp thiếu niên khác, cánh tay trên	
M08.83	Viêm khớp thiếu niên khác, cẳng tay	
M08.84	Viêm khớp thiếu niên khác, bàn tay	
M08.85	Viêm khớp thiếu niên khác, vùng chậu và/hoặc đùi	
M08.86	Viêm khớp thiếu niên khác, cẳng chân	
M08.87	Viêm khớp thiếu niên khác, cổ chân và/hoặc bàn chân	
M08.88	Viêm khớp thiếu niên khác, vị trí khác	
M08.89	Viêm khớp thiếu niên khác, vị trí không xác định	
M08.9	Viêm khớp thiếu niên, không xác định	
M08.90	Viêm khớp thiếu niên, không xác định, nhiều vị trí	
M08.91	Viêm khớp thiếu niên, không xác định, vùng vai	
M08.92	Viêm khớp thiếu niên, không xác định, cánh tay trên	
M08.93	Viêm khớp thiếu niên, không xác định, cẳng tay	
M08.94	Viêm khớp thiếu niên, không xác định, bàn tay	
M08.95	Viêm khớp thiếu niên, không xác định, vùng chậu và/hoặc đùi	
M08.96	Viêm khớp thiếu niên, không xác định, cẳng chân	
M08.97	Viêm khớp thiếu niên, không xác định, cổ chân và/hoặc bàn chân	
M08.98	Viêm khớp thiếu niên, không xác định, vị trí khác	
M08.99	Viêm khớp thiếu niên, không xác định, vị trí không xác định	
M09.*	Viêm khớp thiếu niên do bệnh phân loại mục khác	
M09.0*	Viêm khớp thiếu niên do vảy nến (L40.5†)	
M09.00*	Viêm khớp thiếu niên do vảy nến (L40.5†), nhiều vị trí	
M09.01*	Viêm khớp thiếu niên do vảy nến (L40.5†), vùng vai	
M09.02*	Viêm khớp thiếu niên do vảy nến (L40.5†), cánh tay trên	
M09.03*	Viêm khớp thiếu niên do vảy nến (L40.5†), cẳng tay	
M09.04*	Viêm khớp thiếu niên do vảy nến (L40.5†), bàn tay	
M09.05*	Viêm khớp thiếu niên do vảy nến (L40.5†), vùng chậu và/hoặc đùi	
M09.06*	Viêm khớp thiếu niên do vảy nến (L40.5†), cẳng chân	
M09.07*	Viêm khớp thiếu niên do vảy nến (L40.5†), cổ chân và/hoặc bàn chân	
M09.08*	Viêm khớp thiếu niên do vảy nến (L40.5†), vị trí khác	
M09.09*	Viêm khớp thiếu niên do vảy nến (L40.5†), vị trí không xác định	
M09.1*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†)	
M09.10*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), nhiều vị trí	
M09.11*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), vùng vai	
M09.12*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), cánh tay trên	
M09.13*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), cẳng tay	
M09.14*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), bàn tay	
M09.15*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), vùng chậu và/hoặc đùi	
M09.16*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), cẳng chân	
M09.17*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), cổ chân và/hoặc bàn chân	
M09.18*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), vị trí khác	
M09.19*	Viêm khớp thiếu niên do bệnh Crohn [viêm ruột từng vùng] (K50.-†), vị trí không xác định	
M09.2*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†)	
M09.20*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), nhiều vị trí	
M09.21*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), vùng vai	
M09.22*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), cánh tay trên	
M09.23*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), cẳng tay	
M09.24*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), bàn tay	
M09.25*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), vùng chậu và/hoặc đùi	
M09.26*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), cẳng chân	
M09.27*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), cổ chân và/hoặc bàn chân	
M09.28*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), vị trí khác	
M09.29*	Viêm khớp thiếu niên do bệnh viêm loét đại tràng (K51.-†), vị trí không xác định	
M09.8*	Viêm khớp thiếu niên do bệnh khác phân loại mục khác	
M09.80*	Viêm khớp thiếu niên do bệnh phân loại mục khác, nhiều vị trí	
M09.81*	Viêm khớp thiếu niên do bệnh phân loại mục khác, vùng vai	
M09.82*	Viêm khớp thiếu niên do bệnh phân loại mục khác, cánh tay trên	
M09.83*	Viêm khớp thiếu niên do bệnh phân loại mục khác, cẳng tay	
M09.84*	Viêm khớp thiếu niên do bệnh phân loại mục khác, bàn tay	
M09.85*	Viêm khớp thiếu niên do bệnh phân loại mục khác, vùng chậu và/hoặc đùi	
M09.86*	Viêm khớp thiếu niên do bệnh phân loại mục khác, cẳng chân	
M09.87*	Viêm khớp thiếu niên do bệnh phân loại mục khác, cổ chân và/hoặc bàn chân	
M09.88*	Viêm khớp thiếu niên do bệnh phân loại mục khác, vị trí khác	
M09.89*	Viêm khớp thiếu niên do bệnh phân loại mục khác, vị trí không xác định	
M10	Gút [thống phong]	
M10.0	Bệnh gút vô căn	
M10.00	Bệnh gút vô căn, nhiều vị trí	
M10.01	Bệnh gút vô căn, vùng vai	
M10.02	Bệnh gút vô căn, cánh tay trên	
M10.03	Bệnh gút vô căn, cẳng tay	
M10.04	Bệnh gút vô căn, bàn tay	
M10.05	Bệnh gút vô căn, vùng chậu và/hoặc đùi	
M10.06	Bệnh gút vô căn, cẳng chân	
M10.07	Bệnh gút vô căn, cổ chân và/hoặc bàn chân	
M10.08	Bệnh gút vô căn, vị trí khác	
M10.09	Bệnh gút vô căn, vị trí không xác định	
M10.1	Bệnh gút do nhiễm độc chì	
M10.10	Bệnh gút do nhiễm độc chì, nhiều vị trí	
M10.11	Bệnh gút do nhiễm độc chì, vùng vai	
M10.12	Bệnh gút do nhiễm độc chì, cánh tay trên	
M10.13	Bệnh gút do nhiễm độc chì, cẳng tay	
M10.14	Bệnh gút do nhiễm độc chì, bàn tay	
M10.15	Bệnh gút do nhiễm độc chì, vùng chậu và/hoặc đùi	
M10.16	Bệnh gút do nhiễm độc chì, cẳng chân	
M10.17	Bệnh gút do nhiễm độc chì, cổ chân và/hoặc bàn chân	
M10.18	Bệnh gút do nhiễm độc chì, vị trí khác	
M10.19	Bệnh gút do nhiễm độc chì, vị trí không xác định	
M10.2	Bệnh gút do thuốc	
M10.20	Bệnh gút do thuốc, nhiều vị trí	
M10.21	Bệnh gút do thuốc, vùng vai	
M10.22	Bệnh gút do thuốc, cánh tay trên	
M10.23	Bệnh gút do thuốc, cẳng tay	
M10.24	Bệnh gút do thuốc, bàn tay	
M10.25	Bệnh gút do thuốc, vùng chậu và/hoặc đùi	
M10.26	Bệnh gút do thuốc, cẳng chân	
M10.27	Bệnh gút do thuốc, cổ chân và/hoặc bàn chân	
M10.28	Bệnh gút do thuốc, vị trí khác	
M10.29	Bệnh gút do thuốc, vị trí không xác định	
M10.3	Bệnh gút do suy giảm chức năng thận	
M10.30	Bệnh gút do suy giảm chức năng thận, nhiều vị trí	
M10.31	Bệnh gút do suy giảm chức năng thận, vùng vai	
M10.32	Bệnh gút do suy giảm chức năng thận, cánh tay trên	
M10.33	Bệnh gút do suy giảm chức năng thận, cẳng tay	
M10.34	Bệnh gút do suy giảm chức năng thận, bàn tay	
M10.35	Bệnh gút do suy giảm chức năng thận, vùng chậu và/hoặc đùi	
M10.36	Bệnh gút do suy giảm chức năng thận, cẳng chân	
M10.37	Bệnh gút do suy giảm chức năng thận, cổ chân và/hoặc bàn chân	
M10.38	Bệnh gút do suy giảm chức năng thận, vị trí khác	
M10.39	Bệnh gút do suy giảm chức năng thận, vị trí không xác định	
M10.4	Bệnh gút thứ phát khác	
M10.40	Bệnh gút thứ phát khác, nhiều vị trí	
M10.41	Bệnh gút thứ phát khác, vùng vai	
M10.42	Bệnh gút thứ phát khác, cánh tay trên	
M10.43	Bệnh gút thứ phát khác, cẳng tay	
M10.44	Bệnh gút thứ phát khác, bàn tay	
M10.45	Bệnh gút thứ phát khác, vùng chậu và/hoặc đùi	
M10.46	Bệnh gút thứ phát khác, cẳng chân	
M10.47	Bệnh gút thứ phát khác, cổ chân và/hoặc bàn chân	
M10.48	Bệnh gút thứ phát khác, vị trí khác	
M10.49	Bệnh gút thứ phát khác, vị trí không xác định	
M10.9	Bệnh gút, không xác định	
M10.90	Bệnh gút, không xác định, nhiều vị trí	
M10.91	Bệnh gút, không xác định, vùng vai	
M10.92	Bệnh gút, không xác định, cánh tay trên	
M10.93	Bệnh gút, không xác định, cẳng tay	
M10.94	Bệnh gút, không xác định, bàn tay	
M10.95	Bệnh gút, không xác định, vùng chậu và/hoặc đùi	
M10.96	Bệnh gút, không xác định, cẳng chân	
M10.97	Bệnh gút, không xác định, cổ chân và/hoặc bàn chân	
M10.98	Bệnh gút, không xác định, vị trí khác	
M10.99	Bệnh gút, không xác định, vị trí không xác định	
M11	Bệnh lý khớp do vi tinh thể khác	
M11.0	Bệnh lắng đọng tinh thể Hydroxyapatite	
M11.00	Bệnh lắng đọng tinh thể Hydroxyapatite, nhiều vị trí	
M11.01	Bệnh lắng đọng tinh thể Hydroxyapatite, vùng vai	
M11.02	Bệnh lắng đọng tinh thể Hydroxyapatite, cánh tay trên	
M11.03	Bệnh lắng đọng tinh thể Hydroxyapatite, cẳng tay	
M11.04	Bệnh lắng đọng tinh thể Hydroxyapatite, bàn tay	
M11.05	Bệnh lắng đọng tinh thể Hydroxyapatite, vùng chậu và/hoặc đùi	
M11.06	Bệnh lắng đọng tinh thể Hydroxyapatite, cẳng chân	
M11.07	Bệnh lắng đọng tinh thể Hydroxyapatite, cổ chân và/hoặc bàn chân	
M11.08	Bệnh lắng đọng tinh thể Hydroxyapatite, vị trí khác	
M11.09	Bệnh lắng đọng tinh thể Hydroxyapatite, vị trí không xác định	
M11.1	Bệnh vôi hóa sụn khớp có yếu tố gia đình	
M11.10	Bệnh vôi hóa sụn khớp có yếu tố gia đình, nhiều vị trí	
M11.11	Bệnh vôi hóa sụn khớp có yếu tố gia đình, vùng vai	
M11.12	Bệnh vôi hóa sụn khớp có yếu tố gia đình, cánh tay trên	
M11.13	Bệnh vôi hóa sụn khớp có yếu tố gia đình, cẳng tay	
M11.14	Bệnh vôi hóa sụn khớp có yếu tố gia đình, bàn tay	
M11.15	Bệnh vôi hóa sụn khớp có yếu tố gia đình, vùng chậu và/hoặc đùi	
M11.16	Bệnh vôi hóa sụn khớp có yếu tố gia đình, cẳng chân	
M11.17	Bệnh vôi hóa sụn khớp có yếu tố gia đình, cổ chân và/hoặc bàn chân	
M11.18	Bệnh vôi hóa sụn khớp có yếu tố gia đình, vị trí khác	
M11.19	Bệnh vôi hóa sụn khớp có yếu tố gia đình, vị trí không xác định	
M11.2	Bệnh vôi hóa sụn khớp khác	Bệnh vôi hóa sụn khớp không xác định khác
M11.20	Bệnh vôi hóa sụn khớp khác, nhiều vị trí	
M11.21	Bệnh vôi hóa sụn khớp khác, vùng vai	
M11.22	Bệnh vôi hóa sụn khớp khác, cánh tay trên	
M11.23	Bệnh vôi hóa sụn khớp khác, cẳng tay	
M11.24	Bệnh vôi hóa sụn khớp khác, bàn tay	
M11.25	Bệnh vôi hóa sụn khớp khác, vùng chậu và/hoặc đùi	
M11.26	Bệnh vôi hóa sụn khớp khác, cẳng chân	
M11.27	Bệnh vôi hóa sụn khớp khác, cổ chân và/hoặc bàn chân	
M11.28	Bệnh vôi hóa sụn khớp khác, vị trí khác	
M11.29	Bệnh vôi hóa sụn khớp khác, vị trí không xác định	
M11.8	Bệnh lý khớp do vi tinh thể xác định khác	
M11.80	Bệnh lý khớp do vi tinh thể xác định khác, nhiều vị trí	
M11.81	Bệnh lý khớp do vi tinh thể xác định khác, vùng vai	
M11.82	Bệnh lý khớp do vi tinh thể xác định khác, cánh tay trên	
M11.83	Bệnh lý khớp do vi tinh thể xác định khác, cẳng tay	
M11.84	Bệnh lý khớp do vi tinh thể xác định khác, bàn tay	
M11.85	Bệnh lý khớp do vi tinh thể xác định khác, vùng chậu và/hoặc đùi	
M11.86	Bệnh lý khớp do vi tinh thể xác định khác, cẳng chân	
M11.87	Bệnh lý khớp do vi tinh thể xác định khác, cổ chân và/hoặc bàn chân	
M11.88	Bệnh lý khớp do vi tinh thể xác định khác, vị trí khác	
M11.89	Bệnh lý khớp do vi tinh thể xác định khác, vị trí không xác định	
M11.9	Bệnh lý khớp do vi tinh thể, không xác định	
M11.90	Bệnh lý khớp do vi tinh thể, không xác định, nhiều vị trí	
M11.91	Bệnh lý khớp do vi tinh thể, không xác định, vùng vai	
M11.92	Bệnh lý khớp do vi tinh thể, không xác định, cánh tay trên	
M11.93	Bệnh lý khớp do vi tinh thể, không xác định, cẳng tay	
M11.94	Bệnh lý khớp do vi tinh thể, không xác định, bàn tay	
M11.95	Bệnh lý khớp do vi tinh thể, không xác định, vùng chậu và/hoặc đùi	
M11.96	Bệnh lý khớp do vi tinh thể, không xác định, cẳng chân	
M11.97	Bệnh lý khớp do vi tinh thể, không xác định, cổ chân và/hoặc bàn chân	
M11.98	Bệnh lý khớp do vi tinh thể, không xác định, vị trí khác	
M11.99	Bệnh lý khớp do vi tinh thể, không xác định, vị trí không xác định	
M12	Bệnh lý khớp xác định khác	
M12.0	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud]	
M12.00	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], nhiều vị trí	
M12.01	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], vùng vai	
M12.02	Bệnh lý khớp mạn tính sau bẹnh thấp [Jaccoud], cánh tay trên	
M12.03	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], cẳng tay	
M12.04	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], bàn tay	
M12.05	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], vùng chậu và/hoặc đùi	
M12.06	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], cẳng chân	
M12.07	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], cổ chân và/hoặc bàn chân	
M12.08	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], vị trí khác	
M12.09	Bệnh lý khớp mạn tính sau bệnh thấp [Jaccoud], vị trí không xác định	
M12.1	Bệnh Kaschin-Beck	
M12.10	Bệnh Kaschin-Beck, nhiều vị trí	
M12.11	Bệnh Kaschin-Beck, vùng vai	
M12.12	Bệnh Kaschin-Beck, cánh tay trên	
M12.13	Bệnh Kaschin-Beck, cẳng tay	
M12.14	Bệnh Kaschin-Beck, bàn tay	
M12.15	Bệnh Kaschin-Beck, vùng chậu và/hoặc đùi	
M12.16	Bệnh Kaschin-Beck, cẳng chân	
M12.17	Bệnh Kaschin-Beck, cổ chân và/hoặc bàn chân	
M12.18	Bệnh Kaschin-Beck, vị trí khác	
M12.19	Bệnh Kaschin-Beck, vị trí không xác định	
M12.2	Viêm màng hoạt dịch thể lông nốt (sắc tố)	
M12.20	Viêm màng hoạt dịch thể lông nốt (sắc tố), nhiều vị trí	
M12.21	Viêm màng hoạt dịch thể lông nốt (sắc tố), vùng vai	
M12.22	Viêm màng hoạt dịch thể lông nốt (sắc tố), cánh tay trên	
M12.23	Viêm màng hoạt dịch thể lông nốt (sắc tố), cẳng tay	
M12.24	Viêm màng hoạt dịch thể lông nốt (sắc tố), bàn tay	
M12.25	Viêm màng hoạt dịch thể lông nốt (sắc tố), vùng chậu và/hoặc đùi	
M12.26	Viêm màng hoạt dịch thể lông nốt (sắc tố), cẳng chân	
M12.27	Viêm màng hoạt dịch thể lông nốt (sắc tố), cổ chân và/hoặc bàn chân	
M12.28	Viêm màng hoạt dịch thể lông nốt (sắc tố), vị trí khác	
M12.29	Viêm màng hoạt dịch thể lông nốt (sắc tố), vị trí không xác định	
M12.3	Bệnh thấp khớp tái phát (PR)	
M12.30	Bệnh thấp khớp tái phát (PR), nhiều vị trí	
M12.31	Bệnh thấp khớp tái phát (PR), vùng vai	
M12.32	Bệnh thấp khớp tái phát (PR), cánh tay trên	
M12.33	Bệnh thấp khớp tái phát (PR), cẳng tay	
M12.34	Bệnh thấp khớp tái phát (PR), bàn tay	
M12.35	Bệnh thấp khớp tái phát (PR), vùng chậu và/hoặc đùi	
M12.36	Bệnh thấp khớp tái phát (PR), cẳng chân	
M12.37	Bệnh thấp khớp tái phát (PR), cổ chân và/hoặc bàn chân	
M12.38	Bệnh thấp khớp tái phát (PR), vị trí khác	
M12.39	Bệnh thấp khớp tái phát (PR), vị trí không xác định	
M12.4	Tràn dịch khớp gián đoạn	
M12.40	Tràn dịch khớp gián đoạn, nhiều vị trí	
M12.41	Tràn dịch khớp gián đoạn, vùng vai	
M12.42	Tràn dịch khớp gián đoạn, cánh tay trên	
M12.43	Tràn dịch khớp gián đoạn, cẳng tay	
M12.44	Tràn dịch khớp gián đoạn, bàn tay	
M12.45	Tràn dịch khớp gián đoạn, vùng chậu và/hoặc đùi	
M12.46	Tràn dịch khớp gián đoạn, cẳng chân	
M12.47	Tràn dịch khớp gián đoạn, cổ chân và/hoặc bàn chân	
M12.48	Tràn dịch khớp gián đoạn, vị trí khác	
M12.49	Tràn dịch khớp gián đoạn, vị trí không xác định	
M12.5	Bệnh lý khớp do chấn thương	
M12.50	Bệnh lý khớp do chấn thương, nhiều vị trí	
M12.51	Bệnh lý khớp do chấn thương, vùng vai	
M12.52	Bệnh lý khớp do chấn thương, cánh tay trên	
M12.53	Bệnh lý khớp do chấn thương, cẳng tay	
M12.54	Bệnh lý khớp do chấn thương, bàn tay	
M12.55	Bệnh lý khớp do chấn thương, vùng chậu và/hoặc đùi	
M12.56	Bệnh lý khớp do chấn thương, cẳng chân	
M12.57	Bệnh lý khớp do chấn thương, cổ chân và/hoặc bàn chân	
M12.58	Bệnh lý khớp do chấn thương, vị trí khác	
M12.59	Bệnh lý khớp do chấn thương, vị trí không xác định	
M12.8	Bệnh lý khớp xác định khác, không phân loại mục khác	Bệnh khớp thoáng qua
M12.80	Bệnh lý khớp xác định khác, không phân loại mục khác, nhiều vị trí	
M12.81	Bệnh lý khớp xác định khác, không phân loại mục khác, vùng vai	
M12.82	Bệnh lý khớp xác định khác, không phân loại mục khác, cánh tay trên	
M12.83	Bệnh lý khớp xác định khác, không phân loại mục khác, cẳng tay	
M12.84	Bệnh lý khớp xác định khác, không phân loại mục khác, bàn tay	
M12.85	Bệnh lý khớp xác định khác, không phân loại mục khác, vùng chậu và/hoặc đùi	
M12.86	Bệnh lý khớp xác định khác, không phân loại mục khác, cẳng chân	
M12.87	Bệnh lý khớp xác định khác, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M12.88	Bệnh lý khớp xác định khác, không phân loại mục khác, vị trí khác	
M12.89	Bệnh lý khớp xác định khác, không phân loại mục khác, vị trí không xác định	
M13	Viêm khớp khác	
M13.0	Viêm đa khớp, không xác định	
M13.00	Viêm đa khớp, không xác định, nhiều vị trí	
M13.01	Viêm đa khớp, không xác định, vùng vai	
M13.02	Viêm đa khớp, không xác định, cánh tay trên	
M13.03	Viêm đa khớp, không xác định, cẳng tay	
M13.04	Viêm đa khớp, không xác định, bàn tay	
M13.05	Viêm đa khớp, không xác định, vùng chậu và/hoặc đùi	
M13.06	Viêm đa khớp, không xác định, cẳng chân	
M13.07	Viêm đa khớp, không xác định, cổ chân và/hoặc bàn chân	
M13.08	Viêm đa khớp, không xác định, vị trí khác	
M13.09	Viêm đa khớp, không xác định, vị trí không xác định	
M13.1	Viêm đơn khớp, không phân loại mục khác	
M13.10	Viêm đơn khớp, không phân loại mục khác, nhiều vị trí	
M13.11	Viêm đơn khớp, không phân loại mục khác, vùng vai	
M13.12	Viêm đơn khớp, không phân loại mục khác, cánh tay trên	
M13.13	Viêm đơn khớp, không phân loại mục khác, cẳng tay	
M13.14	Viêm đơn khớp, không phân loại mục khác, bàn tay	
M13.15	Viêm đơn khớp, không phân loại mục khác, vùng chậu và/hoặc đùi	
M13.16	Viêm đơn khớp, không phân loại mục khác, cẳng chân	
M13.17	Viêm đơn khớp, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M13.18	Viêm đơn khớp, không phân loại mục khác, vị trí khác	
M13.19	Viêm đơn khớp, không phân loại mục khác, vị trí không xác định	
M13.8	Viêm khớp xác định khác	Viêm khớp dị ứng
M13.80	Viêm khớp xác định khác, nhiều vị trí	
M13.81	Viêm khớp xác định khác, vùng vai	
M13.82	Viêm khớp xác định khác, cánh tay trên	
M13.83	Viêm khớp xác định khác, cẳng tay	
M13.84	Viêm khớp xác định khác, bàn tay	
M13.85	Viêm khớp xác định khác, vùng chậu và/hoặc đùi	
M13.86	Viêm khớp xác định khác, cẳng chân	
M13.87	Viêm khớp xác định khác, cổ chân và/hoặc bàn chân	
M13.88	Viêm khớp xác định khác, vị trí khác	
M13.89	Viêm khớp xác định khác, vị trí không xác định	
M13.9	Viêm khớp, không xác định	
M13.90	Viêm khớp, không xác định, nhiều vị trí	
M13.91	Viêm khớp, không xác định, vùng vai	
M13.92	Viêm khớp, không xác định, cánh tay trên	
M13.93	Viêm khớp, không xác định, cẳng tay	
M13.94	Viêm khớp, không xác định, bàn tay	
M13.95	Viêm khớp, không xác định, vùng chậu và/hoặc đùi	
M13.96	Viêm khớp, không xác định, cẳng chân	
M13.97	Viêm khớp, không xác định, cổ chân và/hoặc bàn chân	
M13.98	Viêm khớp, không xác định, vị trí khác	
M13.99	Viêm khớp, không xác định, vị trí không xác định	
M14.*	Bệnh lý khớp do bệnh phân loại mục khác	
M14.0*	Bệnh lý khớp gút do thiếu enzym và/hoặc rối loạn di truyền khác	
M14.1*	Bệnh lý khớp dạng vi tinh thể do rối loạn chuyển hóa khác	
M14.2*	Bệnh lý khớp do đái tháo đường (E10-E14 có chung ký tự thứ tư .6†)	
M14.3*	Viêm khớp do viêm tế bào mỡ dưới da (E78.8†)	
M14.4*	Bệnh lý khớp do thoái hóa tinh bột (E85.-†)	
M14.5*	Bệnh lý khớp do bệnh rối loạn nội tiết, dinh dưỡng và/hoặc chuyển hóa khác	
M14.6*	Bệnh lý khớp do bệnh thần kinh	
M14.8*	Bệnh lý khớp do bệnh xác định khác phân loại mục khác	
M15	Thoái hóa đa khớp	
M15.0	Thoái hóa khớp nguyên phát toàn phần	
M15.1	Nốt Heberden (kèm bệnh lý khớp)	
M15.2	Nốt Bouchard (kèm bệnh lý khớp)	
M15.3	Thoái hóa đa khớp thứ phát	Thoái hóa đa khớp sau chấn thương
M15.4	Thoái hóa khớp bào mòn	
M15.8	Thoái hóa đa khớp khác	
M15.9	Thoái hóa đa khớp, không xác định	Thoái hóa khớp toàn thể
M16	Thoái hóa khớp háng [thoái hóa xương hông]	
M16.0	Thoái hóa khớp háng nguyên phát, hai bên	
M16.1	Thoái hóa khớp háng nguyên phát khác	
M16.2	Thoái hóa khớp háng do loạn sản, hai bên	
M16.3	Thoái hóa khớp háng do loạn sản khác	
M16.4	Thoái hóa khớp háng sau chấn thương, hai bên	
M16.5	Thoái hóa khớp háng sau chấn thương khác	
M16.6	Thoái hóa khớp háng thứ phát khác, hai bên	
M16.7	Thoái hóa khớp háng thứ phát khác	
M16.9	Thoái hóa khớp háng, không xác định	
M17	Thoái hóa khớp gối	
M17.0	Thoái hóa khớp gối nguyên phát, hai bên	
M17.1	Thoái hóa khớp gối nguyên phát khác	
M17.2	Thoái hóa khớp gối sau chấn thương, hai bên	
M17.3	Thoái hóa khớp gối sau chấn thương khác	
M17.4	Thoái hóa khớp gối thứ phát khác, hai bên	
M17.5	Thoái hóa khớp gối thứ phát khác	
M17.9	Thoái hóa khớp gối, không xác định	
M18	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái	Thoái hóa khớp cổ tay-khớp gốc ngón tay cái
M18.0	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái nguyên phát, hai bên	
M18.1	Thoái hóa nguyên phát khác của khớp cổ tay - khớp gốc ngón tay cái	
M18.2	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái sau chấn thương, hai bên	
M18.3	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái sau chấn thương khác	
M18.4	Thoái hóa khớp cổ tay - khớp gốc ngón tay thứ phát khác, hai bên	
M18.5	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái thứ phát khác	
M18.9	Thoái hóa khớp cổ tay - khớp gốc ngón tay cái, không xác định	
M19	Thoái hóa khớp khác	
M19.0	Thoái hóa khớp nguyên phát ở khớp khác	Thoái hóa khớp nguyên phát không xác định khác
M19.00	Thoái hóa khớp nguyên phát ở khớp khác, nhiều vị trí	
M19.01	Thoái hóa khớp nguyên phát ở khớp khác, vùng vai	
M19.02	Thoái hóa khớp nguyên phát ở khớp khác, cánh tay trên	
M19.03	Thoái hóa khớp nguyên phát ở khớp khác, cẳng tay	
M19.04	Thoái hóa khớp nguyên phát ở khớp khác, bàn tay	
M19.05	Thoái hóa khớp nguyên phát ở khớp khác, vùng chậu và/hoặc đùi	
M19.06	Thoái hóa khớp nguyên phát ở khớp khác, cẳng chân	
M19.07	Thoái hóa khớp nguyên phát ở khớp khác, cổ chân và/hoặc bàn chân	
M19.08	Thoái hóa khớp nguyên phát ở khớp khác, vị trí khác	
M19.09	Thoái hóa khớp nguyên phát ở khớp khác, vị trí không xác định	
M19.1	Thoái hóa khớp sau chấn thương ở khớp khác	Thoái hóa khớp sau chấn thương không xác định khác
M19.10	Thoái hóa khớp sau chấn thương ở khớp khác, nhiều vị trí	
M19.11	Thoái hóa khớp sau chấn thương ở khớp khác, vùng vai	
M19.12	Thoái hóa khớp sau chấn thương ở khớp khác, cánh tay trên	
M19.13	Thoái hóa khớp sau chấn thương ở khớp khác, cẳng tay	
M19.14	Thoái hóa khớp sau chấn thương ở khớp khác, bàn tay	
M19.15	Thoái hóa khớp sau chấn thương ở khớp khác, vùng chậu và/hoặc đùi	
M19.16	Thoái hóa khớp sau chấn thương ở khớp khác, cẳng chân	
M19.17	Thoái hóa khớp sau chấn thương ở khớp khác, cổ chân và/hoặc bàn chân	
M19.18	Thoái hóa khớp sau chấn thương ở khớp khác, vị trí khác	
M19.19	Thoái hóa khớp sau chấn thương ở khớp khác, vị trí không xác định	
M19.2	Thoái hóa khớp thứ phát khác	Thoái hóa khớp thứ phát không xác định khác
M19.20	Thoái hóa khớp thứ phát khác, nhiều vị trí	
M19.21	Thoái hóa khớp thứ phát khác, vùng vai	
M19.22	Thoái hóa khớp thứ phát khác, cánh tay trên	
M19.23	Thoái hóa khớp thứ phát khác, cẳng tay	
M19.24	Thoái hóa khớp thứ phát khác, bàn tay	
M19.25	Thoái hóa khớp thứ phát khác, vùng chậu và/hoặc đùi	
M19.26	Thoái hóa khớp thứ phát khác, cẳng chân	
M19.27	Thoái hóa khớp thứ phát khác, cổ chân và/hoặc bàn chân	
M19.28	Thoái hóa khớp thứ phát khác, vị trí khác	
M19.29	Thoái hóa khớp thứ phát khác, vị trí không xác định	
M19.8	Thoái hóa khớp xác định khác	
M19.80	Thoái hóa khớp xác định khác, nhiều vị trí	
M19.81	Thoái hóa khớp xác định khác, vùng vai	
M19.82	Thoái hóa khớp xác định khác, cánh tay trên	
M19.83	Thoái hóa khớp xác định khác, cẳng tay	
M19.84	Thoái hóa khớp xác định khác, bàn tay	
M19.85	Thoái hóa khớp xác định khác, vùng chậu và/hoặc đùi	
M19.86	Thoái hóa khớp xác định khác, cẳng chân	
M19.87	Thoái hóa khớp xác định khác, cổ chân và/hoặc bàn chân	
M19.88	Thoái hóa khớp xác định khác, vị trí khác	
M19.89	Thoái hóa khớp xác định khác, vị trí không xác định	
M19.9	Thoái hóa khớp, không xác định	
M19.90	Thoái hóa khớp, không xác định, nhiều vị trí	
M19.91	Thoái hóa khớp, không xác định, vùng vai	
M19.92	Thoái hóa khớp, không xác định, cánh tay trên	
M19.93	Thoái hóa khớp, không xác định, cẳng tay	
M19.94	Thoái hóa khớp, không xác định, bàn tay	
M19.95	Thoái hóa khớp, không xác định, vùng chậu và/hoặc đùi	
M19.96	Thoái hóa khớp, không xác định, cẳng chân	
M19.97	Thoái hóa khớp, không xác định, cổ chân và/hoặc bàn chân	
M19.98	Thoái hóa khớp, không xác định, vị trí khác	
M19.99	Thoái hóa khớp, không xác định, vị trí không xác định	
M20	Biến dạng mắc phải của ngón tay và/hoặc ngón chân	
M20.0	Biến dạng ngón tay	
M20.1	Biến dạng vẹo ngoài của ngón chân cái (mắc phải) [Hallux valgus]	Biến dạng khớp ngón chân cái
M20.2	Biến dạng viêm cứng khớp ngón chân cái [Hallux rigidus]	
M20.3	Biến dạng khác của ngón chân cái (mắc phải)	Ngón chân cái vẹo vào trong
M20.4	Biến dạng ngón chân hình búa (mắc phải)	
M20.5	Biến dạng khác của ngón chân (mắc phải)	
M20.6	Biến dạng mắc phải của ngón chân, không xác định	
M21	Biến dạng mắc phải khác của các chi	
M21.0	Biến dạng vẹo ngoài, không phân loại mục khác	
M21.00	Biến dạng vẹo ngoài, không phân loại mục khác, nhiều vị trí	
M21.01	Biến dạng vẹo ngoài, không phân loại mục khác, vùng vai	
M21.02	Biến dạng vẹo ngoài, không phân loại mục khác, cánh tay trên	
M21.03	Biến dạng vẹo ngoài, không phân loại mục khác, cẳng tay	
M21.04	Biến dạng vẹo ngoài, không phân loại mục khác, bàn tay	
M21.05	Biến dạng vẹo ngoài, không phân loại mục khác, vùng chậu và/hoặc đùi	
M21.06	Biến dạng vẹo ngoài, không phân loại mục khác, cẳng chân	
M21.07	Biến dạng vẹo ngoài, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M21.08	Biến dạng vẹo ngoài, không phân loại mục khác, vị trí khác	
M21.09	Biến dạng vẹo ngoài, không phân loại mục khác, vị trí không xác định	
M21.1	Biến dạng vẹo trong, không phân loại mục khác	
M21.10	Biến dạng vẹo trong, không phân loại mục khác, nhiều vị trí	
M21.11	Biến dạng vẹo trong, không phân loại mục khác, vùng vai	
M21.12	Biến dạng vẹo trong, không phân loại mục khác, cánh tay trên	
M21.13	Biến dạng vẹo trong, không phân loại mục khác, cẳng tay	
M21.14	Biến dạng vẹo trong, không phân loại mục khác, bàn tay	
M21.15	Biến dạng vẹo trong, không phân loại mục khác, vùng chậu và/hoặc đùi	
M21.16	Biến dạng vẹo trong, không phân loại mục khác, cẳng chân	
M21.17	Biến dạng vẹo trong, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M21.18	Biến dạng vẹo trong, không phân loại mục khác, vị trí khác	
M21.19	Biến dạng vẹo trong, không phân loại mục khác, vị trí không xác định	
M21.2	Biến dạng gấp	
M21.20	Biến dạng gấp, nhiều vị trí	
M21.21	Biến dạng gấp, vùng vai	
M21.22	Biến dạng gấp, cánh tay trên	
M21.23	Biến dạng gấp, cẳng tay	
M21.24	Biến dạng gấp, bàn tay	
M21.25	Biến dạng gấp, vùng chậu và/hoặc đùi	
M21.26	Biến dạng gấp, cẳng chân	
M21.27	Biến dạng gấp, cổ chân và/hoặc bàn chân	
M21.28	Biến dạng gấp, vị trí khác	
M21.29	Biến dạng gấp, vị trí không xác định	
M21.3	Biến dạng cổ tay hoặc bàn chân rủ (mắc phải)	
M21.33	Biến dạng cổ tay hoặc bàn chân rủ (mắc phải), cẳng tay	
M21.34	Biến dạng cổ tay hoặc bàn chân rủ (mắc phải), bàn tay	
M21.37	Biến dạng cổ tay hoặc bàn chân rủ (mắc phải), cổ chân và/hoặc bàn chân	
M21.39	Biến dạng cổ tay hoặc bàn chân rủ (mắc phải), vị trí không xác định	
M21.4	Biến dạng bàn chân bẹt (mắc phải)	
M21.5	Biến dạng bàn tay quặp, bàn tay vẹo, bàn chân quặp, bàn chân khoèo (mắc phải)	
M21.50	Biến dạng bàn tay quặp, bàn tay vẹo, bàn chân quặp, bàn chân khoèo (mắc phải), nhiều vị trí	
M21.54	Biến dạng bàn tay quặp, bàn tay vẹo, bàn chân quặp, bàn chân khoèo (mắc phải), bàn tay	
M21.57	Biến dạng bàn tay quặp, bàn tay vẹo, bàn chân quặp, bàn chân khoèo (mắc phải), cổ chân và/hoặc bàn chân	
M21.59	Biến dạng bàn tay quặp, bàn tay vẹo, bàn chân quặp, bàn chân khoèo (mắc phải), vị trí không xác định	
M21.6	Biến dạng mắc phải khác của cổ chân và/hoặc bàn chân	
M21.60	Biến dạng mắc phải khác của cổ chân và/hoặc bàn chân, nhiều vị trí	
M21.67	Biến dạng mắc phải khác của cổ chân và/hoặc bàn chân, cổ chân và/hoặc bàn chân	
M21.69	Biến dạng mắc phải khác của cổ chân và/hoặc bàn chân, vị trí không xác định	
M21.7	Mất cân đối chi (mắc phải)	
M21.70	Mất cân đối chi (mắc phải), nhiều vị trí	
M21.71	Mất cân đối chi (mắc phải), vùng vai	
M21.72	Mất cân đối chi (mắc phải), cánh tay trên	
M21.73	Mất cân đối chi (mắc phải), cẳng tay	
M21.74	Mất cân đối chi (mắc phải), bàn tay	
M21.75	Mất cân đối chi (mắc phải), vùng chậu và/hoặc đùi	
M21.76	Mất cân đối chi (mắc phải), cẳng chân	
M21.77	Mất cân đối chi (mắc phải), cổ chân và/hoặc bàn chân	
M21.78	Mất cân đối chi (mắc phải), vị trí khác	
M21.79	Mất cân đối chi (mắc phải), vị trí không xác định	
M21.8	Biến dạng mắc phải xác định khác của chi	
M21.80	Biến dạng mắc phải xác định khác của chi, nhiều vị trí	
M21.81	Biến dạng mắc phải xác định khác của chi, vùng vai	
M21.82	Biến dạng mắc phải xác định khác của chi, cánh tay trên	
M21.83	Biến dạng mắc phải xác định khác của chi, cẳng tay	
M21.84	Biến dạng mắc phải xác định khác của chi, bàn tay	
M21.85	Biến dạng mắc phải xác định khác của chi, vùng chậu và/hoặc đùi	
M21.86	Biến dạng mắc phải xác định khác của chi, cẳng chân	
M21.87	Biến dạng mắc phải xác định khác của chi, cổ chân và/hoặc bàn chân	
M21.88	Biến dạng mắc phải xác định khác của chi, vị trí khác	
M21.89	Biến dạng mắc phải xác định khác của chi, vị trí không xác định	
M21.9	Biến dạng mắc phải của chi, không xác định	
M21.90	Biến dạng mắc phải của chi, không xác định, nhiều vị trí	
M21.91	Biến dạng mắc phải của chi, không xác định, vùng vai	
M21.92	Biến dạng mắc phải của chi, không xác định, cánh tay trên	
M21.93	Biến dạng mắc phải của chi, không xác định, cẳng tay	
M21.94	Biến dạng mắc phải của chi, không xác định, bàn tay	
M21.95	Biến dạng mắc phải của chi, không xác định, vùng chậu và/hoặc đùi	
M21.96	Biến dạng mắc phải của chi, không xác định, cẳng chân	
M21.97	Biến dạng mắc phải của chi, không xác định, cổ chân và/hoặc bàn chân	
M21.98	Biến dạng mắc phải của chi, không xác định, vị trí khác	
M21.99	Biến dạng mắc phải của chi, không xác định, vị trí không xác định	
M22	Rối loạn của xương bánh chè	
M22.0	Trật khớp bánh chè tái phát	
M22.1	Bán trật khớp bánh chè tái phát	
M22.2	Rối loạn của khớp gối (xương bánh chè - xương đùi)	
M22.3	Trật khác của khớp bánh chè	
M22.4	Bệnh nhuyễn sụn khớp bánh chè	
M22.8	Rối loạn khác của khớp bánh chè	
M22.9	Rối loạn của khớp bánh chè, không xác định	
M23	Tổn thương bên trong khớp gối	
M23.0	Nang sụn chêm	
M23.00	Nang sụn chêm, nhiều vị trí	
M23.01	Nang sụn chêm, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.02	Nang sụn chêm, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.03	Nang sụn chêm, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.04	Nang sụn chêm, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.05	Nang sụn chêm, sừng sau của sụn chêm ngoài	
M23.06	Nang sụn chêm, sụn chêm ngoài khác và/hoặc không xác định	
M23.07	Nang sụn chêm, dây chằng bao khớp	
M23.09	Nang sụn chêm, không xác định dây chằng hoặc sụn chêm	
M23.1	Sụn chêm dạng đĩa (bẩm sinh)	
M23.10	Sụn chêm dạng đĩa (bẩm sinh), nhiều vị trí	
M23.11	Sụn chêm dạng đĩa (bẩm sinh), dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.12	Sụn chêm dạng đĩa (bẩm sinh), dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.13	Sụn chêm dạng đĩa (bẩm sinh), dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.14	Sụn chêm dạng đĩa (bẩm sinh), dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.15	Sụn chêm dạng đĩa (bẩm sinh), sừng sau của sụn chêm ngoài	
M23.16	Sụn chêm dạng đĩa (bẩm sinh), sụn chêm ngoài khác và/hoặc không xác định	
M23.17	Sụn chêm dạng đĩa (bẩm sinh), dây chằng bao khớp	
M23.19	Sụn chêm dạng đĩa (bẩm sinh), không xác định dây chằng hoặc sụn chêm	
M23.2	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ	Vết rách cũ kiểu quai xô
M23.20	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, nhiều vị trí	
M23.21	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.22	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.23	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.24	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.25	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, sừng sau của sụn chêm ngoài	
M23.26	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, sụn chêm ngoài khác và/hoặc không xác định	
M23.27	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, dây chằng bao khớp	
M23.29	Sụn chêm biến dạng do tổn thương rách hay chấn thương cũ, không xác định dây chằng hoặc sụn chêm	
M23.3	Biến dạng khác của sụn chêm	Thoái hóa sụn chêm|Bong sụn chêm|Kẹt sụn chêm
M23.30	Biến dạng khác của sụn chêm, nhiều vị trí	
M23.31	Biến dạng khác của sụn chêm, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.32	Biến dạng khác của sụn chêm, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.33	Biến dạng khác của sụn chêm, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.34	Biến dạng khác của sụn chêm, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.35	Biến dạng khác của sụn chêm, sừng sau của sụn chêm ngoài	
M23.36	Biến dạng khác của sụn chêm, sụn chêm ngoài khác và/hoặc không xác định	
M23.37	Biến dạng khác của sụn chêm, dây chằng bao khớp	
M23.39	Biến dạng khác của sụn chêm, không xác định dây chằng hoặc sụn chêm	
M23.4	Dị vật khớp gối	
M23.40	Dị vật khớp gối, nhiều vị trí	
M23.41	Dị vật khớp gối, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.42	Dị vật khớp gối, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.43	Dị vật khớp gối, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.44	Dị vật khớp gối, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.45	Dị vật khớp gối, sừng sau của sụn chêm ngoài	
M23.46	Dị vật khớp gối, sụn chêm ngoài khác và/hoặc không xác định	
M23.47	Dị vật khớp gối, dây chằng bao khớp	
M23.49	Dị vật khớp gối, không xác định dây chằng hoặc sụn chêm	
M23.5	Đầu gối mất vững mạn tính	
M23.50	Đầu gối mất vững mạn tính, nhiều vị trí	
M23.51	Đầu gối mất vững mạn tính, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.52	Đầu gối mất vững mạn tính, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.53	Đầu gối mất vững mạn tính, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.54	Đầu gối mất vững mạn tính, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.55	Đầu gối mất vững mạn tính, sừng sau của sụn chêm ngoài	
M23.56	Đầu gối mất vững mạn tính, sụn chêm ngoài khác và/hoặc không xác định	
M23.57	Đầu gối mất vững mạn tính, dây chằng bao khớp	
M23.59	Đầu gối mất vững mạn tính, không xác định dây chằng hoặc sụn chêm	
M23.6	Đứt dây chằng khớp gối tự phát khác	
M23.60	Đứt dây chằng khớp gối tự phát khác, nhiều vị trí	
M23.61	Đứt dây chằng khớp gối tự phát khác, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.62	Đứt dây chằng khớp gối tự phát khác, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.63	Đứt dây chằng khớp gối tự phát khác, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.64	Đứt dây chằng khớp gối tự phát khác, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.65	Đứt dây chằng khớp gối tự phát khác, sừng sau của sụn chêm ngoài	
M23.66	Đứt dây chằng khớp gối tự phát khác, sụn chêm ngoài khác và/hoặc không xác định	
M23.67	Đứt dây chằng khớp gối tự phát khác, dây chằng bao khớp	
M23.69	Đứt dây chằng khớp gối tự phát khác, không xác định dây chằng hoặc sụn chêm	
M23.8	Tổn thương khác bên trong khớp gối	Dây chằng khớp gối lỏng lẻo|Khớp gối lục khục
M23.80	Tổn thương khác bên trong khớp gối, nhiều vị trí	
M23.81	Tổn thương khác bên trong khớp gối, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.82	Tổn thương khác bên trong khớp gối, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.83	Tổn thương khác bên trong khớp gối, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.84	Tổn thương khác bên trong khớp gối, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.85	Tổn thương khác bên trong khớp gối, sừng sau của sụn chêm ngoài	
M23.86	Tổn thương khác bên trong khớp gối, sụn chêm ngoài khác và/hoặc không xác định	
M23.87	Tổn thương khác bên trong khớp gối, dây chằng bao khớp	
M23.89	Tổn thương khác bên trong khớp gối, không xác định dây chằng hoặc sụn chêm	
M23.9	Tổn thương không xác định bên trong khớp gối	
M23.90	Tổn thương không xác định bên trong khớp gối, nhiều vị trí	
M23.91	Tổn thương không xác định bên trong khớp gối, dây chằng chéo trước hoặc sừng trước của sụn chêm trong	
M23.92	Tổn thương không xác định bên trong khớp gối, dây chằng chéo sau hoặc sừng sau của sụn chêm trong	
M23.93	Tổn thương không xác định bên trong khớp gối, dây chằng bên trong hoặc sụn chêm trong khác và/hoặc không xác định	
M23.94	Tổn thương không xác định bên trong khớp gối, dây chằng bên ngoài hoặc sừng trước của sụn chêm ngoài	
M23.95	Tổn thương không xác định bên trong khớp gối, sừng sau của sụn chêm ngoài	
M23.96	Tổn thương không xác định bên trong khớp gối, sụn chêm ngoài khác và/hoặc không xác định	
M23.97	Tổn thương không xác định bên trong khớp gối, dây chằng bao khớp	
M23.99	Tổn thương không xác định bên trong khớp gối, không xác định dây chằng hoặc sụn chêm	
M24	Tổn thương xác định khác ở khớp	
M24.0	Dị vật nội khớp	
M24.00	Dị vật nội khớp, nhiều vị trí	
M24.01	Dị vật nội khớp, vùng vai	
M24.02	Dị vật nội khớp, cánh tay trên	
M24.03	Dị vật nội khớp, cẳng tay	
M24.04	Dị vật nội khớp, bàn tay	
M24.05	Dị vật nội khớp, vùng chậu và/hoặc đùi	
M24.06	Dị vật nội khớp, cẳng chân	
M24.07	Dị vật nội khớp, cổ chân và/hoặc bàn chân	
M24.08	Dị vật nội khớp, vị trí khác	
M24.09	Dị vật nội khớp, vị trí không xác định	
M24.1	Rối loạn sụn khớp khác	
M24.10	Rối loạn sụn khớp khác, nhiều vị trí	
M24.11	Rối loạn sụn khớp khác, vùng vai	
M24.12	Rối loạn sụn khớp khác, cánh tay trên	
M24.13	Rối loạn sụn khớp khác, cẳng tay	
M24.14	Rối loạn sụn khớp khác, bàn tay	
M24.15	Rối loạn sụn khớp khác, vùng chậu và/hoặc đùi	
M24.16	Rối loạn sụn khớp khác, cẳng chân	
M24.17	Rối loạn sụn khớp khác, cổ chân và/hoặc bàn chân	
M24.18	Rối loạn sụn khớp khác, vị trí khác	
M24.19	Rối loạn sụn khớp khác, vị trí không xác định	
M24.2	Bệnh lý dây chằng	
M24.20	Bệnh lý dây chằng, nhiều vị trí	
M24.21	Bệnh lý dây chằng, vùng vai	
M24.22	Bệnh lý dây chằng, cánh tay trên	
M24.23	Bệnh lý dây chằng, cẳng tay	
M24.24	Bệnh lý dây chằng, bàn tay	
M24.25	Bệnh lý dây chằng, vùng chậu và/hoặc đùi	
M24.26	Bệnh lý dây chằng, cẳng chân	
M24.27	Bệnh lý dây chằng, cổ chân và/hoặc bàn chân	
M24.28	Bệnh lý dây chằng, vị trí khác	
M24.29	Bệnh lý dây chằng, vị trí không xác định	
M24.3	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác	
M24.30	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, nhiều vị trí	
M24.31	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, vùng vai	
M24.32	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, cánh tay trên	
M24.33	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, cẳng tay	
M24.34	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, bàn tay	
M24.35	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, vùng chậu và/hoặc đùi	
M24.36	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, cẳng chân	
M24.37	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M24.38	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, vị trí khác	
M24.39	Trật và/hoặc bán trật bệnh lý của khớp, không phân loại mục khác, vị trí không xác định	
M24.4	Trật và/hoặc bán trật khớp tái phát	
M24.40	Trật và/hoặc bán trật khớp tái phát, nhiều vị trí	
M24.41	Trật và/hoặc bán trật khớp tái phát, vùng vai	
M24.42	Trật và/hoặc bán trật khớp tái phát, cánh tay trên	
M24.43	Trật và/hoặc bán trật khớp tái phát, cẳng tay	
M24.44	Trật và/hoặc bán trật khớp tái phát, bàn tay	
M24.45	Trật và/hoặc bán trật khớp tái phát, vùng chậu và/hoặc đùi	
M24.46	Trật và/hoặc bán trật khớp tái phát, cẳng chân	
M24.47	Trật và/hoặc bán trật khớp tái phát, cổ chân và/hoặc bàn chân	
M24.48	Trật và/hoặc bán trật khớp tái phát, vị trí khác	
M24.49	Trật và/hoặc bán trật khớp tái phát, vị trí không xác định	
M24.5	Co cứng khớp	
M24.50	Co cứng khớp, nhiều vị trí	
M24.51	Co cứng khớp, vùng vai	
M24.52	Co cứng khớp, cánh tay trên	
M24.53	Co cứng khớp, cẳng tay	
M24.54	Co cứng khớp, bàn tay	
M24.55	Co cứng khớp, vùng chậu và/hoặc đùi	
M24.56	Co cứng khớp, cẳng chân	
M24.57	Co cứng khớp, cổ chân và/hoặc bàn chân	
M24.58	Co cứng khớp, vị trí khác	
M24.59	Co cứng khớp, vị trí không xác định	
M24.6	Dính khớp	
M24.60	Dính khớp, nhiều vị trí	
M24.61	Dính khớp, vùng vai	
M24.62	Dính khớp, cánh tay trên	
M24.63	Dính khớp, cẳng tay	
M24.64	Dính khớp, bàn tay	
M24.65	Dính khớp, vùng chậu và/hoặc đùi	
M24.66	Dính khớp, cẳng chân	
M24.67	Dính khớp, cổ chân và/hoặc bàn chân	
M24.68	Dính khớp, vị trí khác	
M24.69	Dính khớp, vị trí không xác định	
M24.7	Lồi cầu ngoài ổ cối	
M24.75	Lồi cầu ngoài ổ cối, vùng chậu và/hoặc đùi	
M24.79	Lồi cầu ngoài ổ cối, vị trí không xác định	
M24.8	Tổn thương xác định khác của khớp, không phân loại mục khác	
M24.80	Tổn thương xác định khác của khớp, không phân loại mục khác, nhiều vị trí	
M24.81	Tổn thương xác định khác của khớp, không phân loại mục khác, vùng vai	
M24.82	Tổn thương xác định khác của khớp, không phân loại mục khác, cánh tay trên	
M24.83	Tổn thương xác định khác của khớp, không phân loại mục khác, cẳng tay	
M24.84	Tổn thương xác định khác của khớp, không phân loại mục khác, bàn tay	
M24.85	Tổn thương xác định khác của khớp, không phân loại mục khác, vùng chậu và/hoặc đùi	
M24.86	Tổn thương xác định khác của khớp, không phân loại mục khác, cẳng chân	
M24.87	Tổn thương xác định khác của khớp, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M24.88	Tổn thương xác định khác của khớp, không phân loại mục khác, vị trí khác	
M24.89	Tổn thương xác định khác của khớp, không phân loại mục khác, vị trí không xác định	
M24.9	Tổn thương khớp, không xác định	
M24.90	Tổn thương khớp, không xác định, nhiều vị trí	
M24.91	Tổn thương khớp, không xác định, vùng vai	
M24.92	Tổn thương khớp, không xác định, cánh tay trên	
M24.93	Tổn thương khớp, không xác định, cẳng tay	
M24.94	Tổn thương khớp, không xác định, bàn tay	
M24.95	Tổn thương khớp, không xác định, vùng chậu và/hoặc đùi	
M24.96	Tổn thương khớp, không xác định, cẳng chân	
M24.97	Tổn thương khớp, không xác định, cổ chân và/hoặc bàn chân	
M24.98	Tổn thương khớp, không xác định, vị trí khác	
M24.99	Tổn thương khớp, không xác định, vị trí không xác định	
M25	Rối loạn khớp khác, không phân loại mục khác	
M25.0	Tràn máu khớp	
M25.00	Tràn máu khớp, nhiều vị trí	
M25.01	Tràn máu khớp, vùng vai	
M25.02	Tràn máu khớp, cánh tay trên	
M25.03	Tràn máu khớp, cẳng tay	
M25.04	Tràn máu khớp, bàn tay	
M25.05	Tràn máu khớp, vùng chậu và/hoặc đùi	
M25.06	Tràn máu khớp, cẳng chân	
M25.07	Tràn máu khớp, cổ chân và/hoặc bàn chân	
M25.08	Tràn máu khớp, vị trí khác	
M25.09	Tràn máu khớp, vị trí không xác định	
M25.1	Rò khớp	
M25.10	Rò khớp, nhiều vị trí	
M25.11	Rò khớp, vùng vai	
M25.12	Rò khớp, cánh tay trên	
M25.13	Rò khớp, cẳng tay	
M25.14	Rò khớp, bàn tay	
M25.15	Rò khớp, vùng chậu và/hoặc đùi	
M25.16	Rò khớp, cẳng chân	
M25.17	Rò khớp, cổ chân và/hoặc bàn chân	
M25.18	Rò khớp, vị trí khác	
M25.19	Rò khớp, vị trí không xác định	
M25.2	Khớp lỏng lẻo	
M25.20	Khớp lỏng lẻo, nhiều vị trí	
M25.21	Khớp lỏng lẻo, vùng vai	
M25.22	Khớp lỏng lẻo, cánh tay trên	
M25.23	Khớp lỏng lẻo, cẳng tay	
M25.24	Khớp lỏng lẻo, bàn tay	
M25.25	Khớp lỏng lẻo, vùng chậu và/hoặc đùi	
M25.26	Khớp lỏng lẻo, cẳng chân	
M25.27	Khớp lỏng lẻo, cổ chân và/hoặc bàn chân	
M25.28	Khớp lỏng lẻo, vị trí khác	
M25.29	Khớp lỏng lẻo, vị trí không xác định	
M25.3	Sự mất vững khác của khớp	
M25.30	Sự mất vững khác của khớp, nhiều vị trí	
M25.31	Sự mất vững khác của khớp, vùng vai	
M25.32	Sự mất vững khác của khớp, cánh tay trên	
M25.33	Sự mất vững khác của khớp, cẳng tay	
M25.34	Sự mất vững khác của khớp, bàn tay	
M25.35	Sự mất vững khác của khớp, vùng chậu và/hoặc đùi	
M25.36	Sự mất vững khác của khớp, cẳng chân	
M25.37	Sự mất vững khác của khớp, cổ chân và/hoặc bàn chân	
M25.38	Sự mất vững khác của khớp, vị trí khác	
M25.39	Sự mất vững khác của khớp, vị trí không xác định	
M25.4	Tràn dịch khớp	
M25.40	Tràn dịch khớp, nhiều vị trí	
M25.41	Tràn dịch khớp, vùng vai	
M25.42	Tràn dịch khớp, cánh tay trên	
M25.43	Tràn dịch khớp, cẳng tay	
M25.44	Tràn dịch khớp, bàn tay	
M25.45	Tràn dịch khớp, vùng chậu và/hoặc đùi	
M25.46	Tràn dịch khớp, cẳng chân	
M25.47	Tràn dịch khớp, cổ chân và/hoặc bàn chân	
M25.48	Tràn dịch khớp, vị trí khác	
M25.49	Tràn dịch khớp, vị trí không xác định	
M25.5	Đau khớp	
M25.50	Đau khớp, nhiều vị trí	
M25.51	Đau khớp, vùng vai	
M25.52	Đau khớp, cánh tay trên	
M25.53	Đau khớp, cẳng tay	
M25.54	Đau khớp, bàn tay	
M25.55	Đau khớp, vùng chậu và/hoặc đùi	
M25.56	Đau khớp, cẳng chân	
M25.57	Đau khớp, cổ chân và/hoặc bàn chân	
M25.58	Đau khớp, vị trí khác	
M25.59	Đau khớp, vị trí không xác định	
M25.6	Cứng khớp, không phân loại mục khác	
M25.60	Cứng khớp, không phân loại mục khác, nhiều vị trí	
M25.61	Cứng khớp, không phân loại mục khác, vùng vai	
M25.62	Cứng khớp, không phân loại mục khác, cánh tay trên	
M25.63	Cứng khớp, không phân loại mục khác, cẳng tay	
M25.64	Cứng khớp, không phân loại mục khác, bàn tay	
M25.65	Cứng khớp, không phân loại mục khác, vùng chậu và/hoặc đùi	
M25.66	Cứng khớp, không phân loại mục khác, cẳng chân	
M25.67	Cứng khớp, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M25.68	Cứng khớp, không phân loại mục khác, vị trí khác	
M25.69	Cứng khớp, không phân loại mục khác, vị trí không xác định	
M25.7	Gai xương	
M25.70	Gai xương, nhiều vị trí	
M25.71	Gai xương, vùng vai	
M25.72	Gai xương, cánh tay trên	
M25.73	Gai xương, cẳng tay	
M25.74	Gai xương, bàn tay	
M25.75	Gai xương, vùng chậu và/hoặc đùi	
M25.76	Gai xương, cẳng chân	
M25.77	Gai xương, cổ chân và/hoặc bàn chân	
M25.78	Gai xương, vị trí khác	
M25.79	Gai xương, vị trí không xác định	
M25.8	Rối loạn khớp xác định khác	
M25.80	Rối loạn khớp xác định khác, nhiều vị trí	
M25.81	Rối loạn khớp xác định khác, vùng vai	
M25.82	Rối loạn khớp xác định khác, cánh tay trên	
M25.83	Rối loạn khớp xác định khác, cẳng tay	
M25.84	Rối loạn khớp xác định khác, bàn tay	
M25.85	Rối loạn khớp xác định khác, vùng chậu và/hoặc đùi	
M25.86	Rối loạn khớp xác định khác, cẳng chân	
M25.87	Rối loạn khớp xác định khác, cổ chân và/hoặc bàn chân	
M25.88	Rối loạn khớp xác định khác, vị trí khác	
M25.89	Rối loạn khớp xác định khác, vị trí không xác định	
M25.9	Rối loạn khớp, không xác định	Bệnh khớp không xác định khác
M25.90	Rối loạn khớp, không xác định, nhiều vị trí	
M25.91	Rối loạn khớp, không xác định, vùng vai	
M25.92	Rối loạn khớp, không xác định, cánh tay trên	
M25.93	Rối loạn khớp, không xác định, cẳng tay	
M25.94	Rối loạn khớp, không xác định, bàn tay	
M25.95	Rối loạn khớp, không xác định, vùng chậu và/hoặc đùi	
M25.96	Rối loạn khớp, không xác định, cẳng chân	
M25.97	Rối loạn khớp, không xác định, cổ chân và/hoặc bàn chân	
M25.98	Rối loạn khớp, không xác định, vị trí khác	
M25.99	Rối loạn khớp, không xác định, vị trí không xác định	
M30	Viêm nốt quanh động mạch và/hoặc bệnh liên quan	
M30.0	Viêm nốt quanh động mạch	
M30.1	Viêm đa động mạch có tác động đến phổi [Churg-Strauss]	Bệnh viêm mạch u hạt dị ứng
M30.2	Viêm đa động mạch ở thiếu niên	
M30.3	Hội chứng hạch bạch huyết niêm mạc [Kawasaki]	
M30.8	Bệnh lý khác liên quan đến viêm nốt quanh động mạch	Hội chứng viêm đa mạch chồng lấp
M31	Bệnh lý mạch máu hoại tử khác	Bệnh mạch máu hoại tử khác
M31.0	Viêm mạch máu quá mẫn	Hội chứng Goodpasture
M31.1	Bệnh lý huyết khối vi mạch máu	Ban xuất huyết giảm tiểu cầu huyết khối
M31.3	Bệnh u hạt Wegener	
M31.4	Hội chứng quai động mạch chủ [Takayasu]	
M31.5	Viêm động mạch tế bào khổng lồ kèm đau đa cơ dạng thấp	
M31.6	Bệnh viêm động mạch tế bào khổng lồ khác	
M31.7	Viêm đa vi mạch máu	
M31.8	Bệnh lý mạch máu hoại tử xác định khác	Viêm mạch máu kèm giảm bổ thể
M31.9	Bệnh lý viêm mạch máu hoại tử, không xác định	
M32	Bệnh lupus ban đỏ hệ thống	
M32.0	Bệnh lupus ban đỏ hệ thống do dùng thuốc	
M32.1†	Bệnh lupus ban đỏ hệ thống có tác động đến tạng hoặc hệ thống	
M32.8	Dạng khác của bệnh lupus ban đỏ hệ thống	
M32.9	Bệnh lupus ban đỏ hệ thống, không xác định	
M33	Bệnh viêm đa cơ và da	
M33.0	Bệnh viêm cơ và da ở thiếu niên	
M33.1	Bệnh viêm cơ và da khác	
M33.2	Bệnh viêm đa cơ	
M33.9	Bệnh viêm đa cơ và da, không xác định	
M34	Bệnh xơ cứng bì toàn thể	
M34.0	Bệnh xơ cứng bì toàn thể tiến triển	
M34.1	Hội chứng CR(E)ST	
M34.2	Bệnh xơ cứng bì do dùng thuốc và/hoặc hóa chất	
M34.8	Những dạng khác của bệnh xơ cứng bì toàn thể	
M34.9	Bệnh xơ cứng bì toàn thể, không xác định	
M35	Tổn thương hệ thống khác của mô liên kết	
M35.0	Hội chứng khô [Hội chứng Sjögren]	
M35.1	Hội chứng trùng lắp khác	
M35.2	Bệnh Behçet	
M35.3	Bệnh đau đa cơ do thấp	
M35.4	Viêm cân mạc lan tỏa (tăng bạch cầu ái toan)	
M35.5	Xơ cứng đa ổ	
M35.6	Viêm tế bào mỡ dưới da tái phát [Weber-Christian]	
M35.7	Hội chứng người dẻo [hypermobility]	
M35.8	Tổn thương hệ thống xác định khác của mô liên kết	
M35.9	Tổn thương hệ thống của mô liên kết, không xác định	
M36.*	Tổn thương hệ thống của mô liên kết do bệnh phân loại mục khác	
M36.0*	Viêm (đa) cơ và da do bệnh u tân sinh (C00.- - D48.-†)	
M36.1*	Bệnh lý khớp do bệnh u tân sinh (C00.- - D48.-†)	
M36.2*	Bệnh lý khớp do rối loạn đông máu [haemophilia] (D66-D68†)	
M36.3*	Bệnh lý khớp do rối loạn máu khác (D50-D76†)	
M36.4*	Bệnh lý khớp do phản ứng quá mẫn phân loại mục khác	
M36.8*	Tổn thương hệ thống của mô liên kết do bệnh khác phân loại mục khác	
M40	Gù và/hoặc ưỡn cột sống [lưng]	
M40.0	Gù cột sống [lưng] do tư thế	
M40.00	Gù cột sống [lưng] do tư thế, nhiều vị trí của cột sống	
M40.01	Gù cột sống [lưng] do tư thế, vùng trục - đội - chẩm	
M40.02	Gù cột sống [lưng] do tư thế, vùng cổ	
M40.03	Gù cột sống [lưng] do tư thế, vùng cổ - ngực	
M40.04	Gù cột sống [lưng] do tư thế, vùng (lồng) ngực	
M40.05	Gù cột sống [lưng] do tư thế, vùng ngực - thắt lưng	
M40.06	Gù cột sống [lưng] do tư thế, vùng thắt lưng	
M40.07	Gù cột sống [lưng] do tư thế, vùng thắt lưng - cùng	
M40.08	Gù cột sống [lưng] do tư thế, vùng cùng và/hoặc cùng - cụt	
M40.09	Gù cột sống [lưng] do tư thế, vị trí không xác định	
M40.1	Gù cột sống [lưng] thứ phát khác	
M40.10	Gù cột sống [lưng] thứ phát khác, nhiều vị trí của cột sống	
M40.11	Gù cột sống [lưng] thứ phát khác, vùng trục - đội - chẩm	
M40.12	Gù cột sống [lưng] thứ phát khác, vùng cổ	
M40.13	Gù cột sống [lưng] thứ phát khác, vùng cổ - ngực	
M40.14	Gù cột sống [lưng] thứ phát khác, vùng (lồng) ngực	
M40.15	Gù cột sống [lưng] thứ phát khác, vùng ngực - thắt lưng	
M40.16	Gù cột sống [lưng] thứ phát khác, vùng thắt lưng	
M40.17	Gù cột sống [lưng] thứ phát khác, vùng thắt lưng - cùng	
M40.18	Gù cột sống [lưng] thứ phát khác, vùng cùng và/hoặc cùng - cụt	
M40.19	Gù cột sống [lưng] thứ phát khác, vị trí không xác định	
M40.2	Gù cột sống [lưng] khác và/hoặc không xác định	
M40.20	Gù cột sống [lưng] khác và/hoặc không xác định, nhiều vị trí của cột sống	
M40.21	Gù cột sống [lưng] khác và/hoặc không xác định, vùng trục - đội - chẩm	
M40.22	Gù cột sống [lưng] khác và/hoặc không xác định, vùng cổ	
M40.23	Gù cột sống [lưng] khác và/hoặc không xác định, vùng cổ - ngực	
M40.24	Gù cột sống [lưng] khác và/hoặc không xác định, vùng (lồng) ngực	
M40.25	Gù cột sống [lưng] khác và/hoặc không xác định, vùng ngực - thắt lưng	
M40.26	Gù cột sống [lưng] khác và/hoặc không xác định, vùng thắt lưng	
M40.27	Gù cột sống [lưng] khác và/hoặc không xác định, vùng thắt lưng - cùng	
M40.28	Gù cột sống [lưng] khác và/hoặc không xác định, vùng cùng và/hoặc cùng - cụt	
M40.29	Gù cột sống [lưng] khác và/hoặc không xác định, vị trí không xác định	
M40.3	Hội chứng lưng phẳng	
M40.30	Hội chứng lưng phẳng, nhiều vị trí của cột sống	
M40.31	Hội chứng lưng phẳng, vùng trục - đội - chẩm	
M40.32	Hội chứng lưng phẳng, vùng cổ	
M40.33	Hội chứng lưng phẳng, vùng cổ - ngực	
M40.34	Hội chứng lưng phẳng, vùng (lồng) ngực	
M40.35	Hội chứng lưng phẳng, vùng ngực - thắt lưng	
M40.36	Hội chứng lưng phẳng, vùng thắt lưng	
M40.37	Hội chứng lưng phẳng, vùng thắt lưng - cùng	
M40.38	Hội chứng lưng phẳng, vùng cùng và/hoặc cùng - cụt	
M40.39	Hội chứng lưng phẳng, vị trí không xác định	
M40.4	Ưỡn cột sống [võng lưng] khác	
M40.40	Ưỡn cột sống [võng lưng] khác, nhiều vị trí của cột sống	
M40.41	Ưỡn cột sống [võng lưng] khác, vùng trục - đội - chẩm	
M40.42	Ưỡn cột sống [võng lưng] khác, vùng cổ	
M40.43	Ưỡn cột sống [võng lưng] khác, vùng cổ - ngực	
M40.44	Ưỡn cột sống [võng lưng] khác, vùng (lồng) ngực	
M40.45	Ưỡn cột sống [võng lưng] khác, vùng ngực - thắt lưng	
M40.46	Ưỡn cột sống [võng lưng] khác, vùng thắt lưng	
M40.47	Ưỡn cột sống [võng lưng] khác, vùng thắt lưng - cùng	
M40.48	Ưỡn cột sống [võng lưng] khác, vùng cùng và/hoặc cùng - cụt	
M40.49	Ưỡn cột sống [võng lưng] khác, vị trí không xác định	
M40.5	Ưỡn cột sống [võng lưng], không xác định	
M40.50	Ưỡn cột sống [võng lưng], không xác định, nhiều vị trí của cột sống	
M40.51	Ưỡn cột sống [võng lưng], không xác định, vùng trục - đội - chẩm	
M40.52	Ưỡn cột sống [võng lưng], không xác định, vùng cổ	
M40.53	Ưỡn cột sống [võng lưng], không xác định, vùng cổ - ngực	
M40.54	Ưỡn cột sống [võng lưng], không xác định, vùng (lồng) ngực	
M40.55	Ưỡn cột sống [võng lưng], không xác định, vùng ngực - thắt lưng	
M40.56	Ưỡn cột sống [võng lưng], không xác định, vùng thắt lưng	
M40.57	Ưỡn cột sống [võng lưng], không xác định, vùng thắt lưng - cùng	
M40.58	Ưỡn cột sống [võng lưng], không xác định, vùng cùng và/hoặc cùng - cụt	
M40.59	Ưỡn cột sống [võng lưng], không xác định, vị trí không xác định	
M41	Vẹo cột sống	
M41.0	Vẹo cột sống vô căn ở trẻ nhỏ	
M41.00	Vẹo cột sống vô căn ở trẻ nhỏ, nhiều vị trí của cột sống	
M41.01	Vẹo cột sống vô căn ở trẻ nhỏ, vùng trục - đội - chẩm	
M41.02	Vẹo cột sống vô căn ở trẻ nhỏ, vùng cổ	
M41.03	Vẹo cột sống vô căn ở trẻ nhỏ, vùng cổ - ngực	
M41.04	Vẹo cột sống vô căn ở trẻ nhỏ, vùng (lồng) ngực	
M41.05	Vẹo cột sống vô căn ở trẻ nhỏ, vùng ngực - thắt lưng	
M41.06	Vẹo cột sống vô căn ở trẻ nhỏ, vùng thắt lưng	
M41.07	Vẹo cột sống vô căn ở trẻ nhỏ, vùng thắt lưng - cùng	
M41.08	Vẹo cột sống vô căn ở trẻ nhỏ, vùng cùng và/hoặc cùng - cụt	
M41.09	Vẹo cột sống vô căn ở trẻ nhỏ, vị trí không xác định	
M41.1	Vẹo cột sống vô căn thiếu niên	Vẹo cột sống ở thiếu niên
M41.10	Vẹo cột sống vô căn thiếu niên, nhiều vị trí của cột sống	
M41.11	Vẹo cột sống vô căn thiếu niên, vùng trục - đội - chẩm	
M41.12	Vẹo cột sống vô căn thiếu niên, vùng cổ	
M41.13	Vẹo cột sống vô căn thiếu niên, vùng cổ - ngực	
M41.14	Vẹo cột sống vô căn thiếu niên, vùng (lồng) ngực	
M41.15	Vẹo cột sống vô căn thiếu niên, vùng ngực - thắt lưng	
M41.16	Vẹo cột sống vô căn thiếu niên, vùng thắt lưng	
M41.17	Vẹo cột sống vô căn thiếu niên, vùng thắt lưng - cùng	
M41.18	Vẹo cột sống vô căn thiếu niên, vùng cùng và/hoặc cùng - cụt	
M41.19	Vẹo cột sống vô căn thiếu niên, vị trí không xác định	
M41.2	Vẹo cột sống vô căn khác	
M41.20	Vẹo cột sống vô căn khác, nhiều vị trí của cột sống	
M41.21	Vẹo cột sống vô căn khác, vùng trục - đội - chẩm	
M41.22	Vẹo cột sống vô căn khác, vùng cổ	
M41.23	Vẹo cột sống vô căn khác, vùng cổ - ngực	
M41.24	Vẹo cột sống vô căn khác, vùng (lồng) ngực	
M41.25	Vẹo cột sống vô căn khác, vùng ngực - thắt lưng	
M41.26	Vẹo cột sống vô căn khác, vùng thắt lưng	
M41.27	Vẹo cột sống vô căn khác, vùng thắt lưng - cùng	
M41.28	Vẹo cột sống vô căn khác, vùng cùng và/hoặc cùng - cụt	
M41.29	Vẹo cột sống vô căn khác, vị trí không xác định	
M41.3	Vẹo cột sống do bất thường vùng ngực	
M41.30	Vẹo cột sống do bất thường vùng ngực, nhiều vị trí của cột sống	
M41.34	Vẹo cột sống do bất thường vùng ngực, vùng (lồng) ngực	
M41.35	Vẹo cột sống do bất thường vùng ngực, vùng ngực - thắt lưng	
M41.39	Vẹo cột sống do bất thường vùng ngực, vị trí không xác định	
M41.4	Vẹo cột sống do nguyên nhân thần kinh - cơ	
M41.40	Vẹo cột sống do nguyên nhân thần kinh - cơ, nhiều vị trí của cột sống	
M41.41	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng trục - đội - chẩm	
M41.42	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng cổ	
M41.43	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng cổ - ngực	
M41.44	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng (lồng) ngực	
M41.45	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng ngực - thắt lưng	
M41.46	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng thắt lưng	
M41.47	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng thắt lưng - cùng	
M41.48	Vẹo cột sống do nguyên nhân thần kinh - cơ, vùng cùng và/hoặc cùng - cụt	
M41.49	Vẹo cột sống do nguyên nhân thần kinh - cơ, vị trí không xác định	
M41.5	Vẹo cột sống thứ phát khác	
M41.50	Vẹo cột sống thứ phát khác, nhiều vị trí của cột sống	
M41.51	Vẹo cột sống thứ phát khác, vùng trục - đội - chẩm	
M41.52	Vẹo cột sống thứ phát khác, vùng cổ	
M41.53	Vẹo cột sống thứ phát khác, vùng cổ - ngực	
M41.54	Vẹo cột sống thứ phát khác, vùng (lồng) ngực	
M41.55	Vẹo cột sống thứ phát khác, vùng ngực - thắt lưng	
M41.56	Vẹo cột sống thứ phát khác, vùng thắt lưng	
M41.57	Vẹo cột sống thứ phát khác, vùng thắt lưng - cùng	
M41.58	Vẹo cột sống thứ phát khác, vùng cùng và/hoặc cùng - cụt	
M41.59	Vẹo cột sống thứ phát khác, vị trí không xác định	
M41.8	Dạng khác của vẹo cột sống	
M41.80	Dạng khác của vẹo cột sống, nhiều vị trí của cột sống	
M41.81	Dạng khác của vẹo cột sống, vùng trục - đội - chẩm	
M41.82	Dạng khác của vẹo cột sống, vùng cổ	
M41.83	Dạng khác của vẹo cột sống, vùng cổ - ngực	
M41.84	Dạng khác của vẹo cột sống, vùng (lồng) ngực	
M41.85	Dạng khác của vẹo cột sống, vùng ngực - thắt lưng	
M41.86	Dạng khác của vẹo cột sống, vùng thắt lưng	
M41.87	Dạng khác của vẹo cột sống, vùng thắt lưng - cùng	
M41.88	Dạng khác của vẹo cột sống, vùng cùng và/hoặc cùng - cụt	
M41.89	Dạng khác của vẹo cột sống, vị trí không xác định	
M41.9	Vẹo cột sống, không xác định	
M41.90	Vẹo cột sống, không xác định, nhiều vị trí của cột sống	
M41.91	Vẹo cột sống, không xác định, vùng trục - đội - chẩm	
M41.92	Vẹo cột sống, không xác định, vùng cổ	
M41.93	Vẹo cột sống, không xác định, vùng cổ - ngực	
M41.94	Vẹo cột sống, không xác định, vùng (lồng) ngực	
M41.95	Vẹo cột sống, không xác định, vùng ngực - thắt lưng	
M41.96	Vẹo cột sống, không xác định, vùng thắt lưng	
M41.97	Vẹo cột sống, không xác định, vùng thắt lưng - cùng	
M41.98	Vẹo cột sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M41.99	Vẹo cột sống, không xác định, vị trí không xác định	
M42	Bệnh thoái hóa xương sụn cột sống	
M42.0	Thoái hóa xương sụn cột sống thiếu niên	
M42.00	Thoái hóa xương sụn cột sống thiếu niên, nhiều vị trí của cột sống	
M42.01	Thoái hóa xương sụn cột sống thiếu niên, vùng trục - đội - chẩm	
M42.02	Thoái hóa xương sụn cột sống thiếu niên, vùng cổ	
M42.03	Thoái hóa xương sụn cột sống thiếu niên, vùng cổ - ngực	
M42.04	Thoái hóa xương sụn cột sống thiếu niên, vùng (lồng) ngực	
M42.05	Thoái hóa xương sụn cột sống thiếu niên, vùng ngực - thắt lưng	
M42.06	Thoái hóa xương sụn cột sống thiếu niên, vùng thắt lưng	
M42.07	Thoái hóa xương sụn cột sống thiếu niên, vùng thắt lưng - cùng	
M42.08	Thoái hóa xương sụn cột sống thiếu niên, vùng cùng và/hoặc cùng - cụt	
M42.09	Thoái hóa xương sụn cột sống thiếu niên, vị trí không xác định	
M42.1	Thoái hóa xương sụn cột sống ở người lớn	
M42.10	Thoái hóa xương sụn cột sống ở người lớn, nhiều vị trí của cột sống	
M42.11	Thoái hóa xương sụn cột sống ở người lớn, vùng trục - đội - chẩm	
M42.12	Thoái hóa xương sụn cột sống ở người lớn, vùng cổ	
M42.13	Thoái hóa xương sụn cột sống ở người lớn, vùng cổ - ngực	
M42.14	Thoái hóa xương sụn cột sống ở người lớn, vùng (lồng) ngực	
M42.15	Thoái hóa xương sụn cột sống ở người lớn, vùng ngực - thắt lưng	
M42.16	Thoái hóa xương sụn cột sống ở người lớn, vùng thắt lưng	
M42.17	Thoái hóa xương sụn cột sống ở người lớn, vùng thắt lưng - cùng	
M42.18	Thoái hóa xương sụn cột sống ở người lớn, vùng cùng và/hoặc cùng - cụt	
M42.19	Thoái hóa xương sụn cột sống ở người lớn, vị trí không xác định	
M42.9	Thoái hóa xương sụn cột sống, không xác định	
M42.90	Thoái hóa xương sụn cột sống, không xác định, nhiều vị trí của cột sống	
M42.91	Thoái hóa xương sụn cột sống, không xác định, vùng trục - đội - chẩm	
M42.92	Thoái hóa xương sụn cột sống, không xác định, vùng cổ	
M42.93	Thoái hóa xương sụn cột sống, không xác định, vùng cổ - ngực	
M42.94	Thoái hóa xương sụn cột sống, không xác định, vùng (lồng) ngực	
M42.95	Thoái hóa xương sụn cột sống, không xác định, vùng ngực - thắt lưng	
M42.96	Thoái hóa xương sụn cột sống, không xác định, vùng thắt lưng	
M42.97	Thoái hóa xương sụn cột sống, không xác định, vùng thắt lưng - cùng	
M42.98	Thoái hóa xương sụn cột sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M42.99	Thoái hóa xương sụn cột sống, không xác định, vị trí không xác định	
M43	Bệnh lý biến dạng khác của cột sống	
M43.0	Bệnh lý gãy eo đốt sống	
M43.00	Bệnh lý gãy eo đốt sống, nhiều vị trí của cột sống	
M43.01	Bệnh lý gãy eo đốt sống, vùng trục - đội - chẩm	
M43.02	Bệnh lý gãy eo đốt sống, vùng cổ	
M43.03	Bệnh lý gãy eo đốt sống, vùng cổ - ngực	
M43.04	Bệnh lý gãy eo đốt sống, vùng (lồng) ngực	
M43.05	Bệnh lý gãy eo đốt sống, vùng ngực - thắt lưng	
M43.06	Bệnh lý gãy eo đốt sống, vùng thắt lưng	
M43.07	Bệnh lý gãy eo đốt sống, vùng thắt lưng - cùng	
M43.08	Bệnh lý gãy eo đốt sống, vùng cùng và/hoặc cùng - cụt	
M43.09	Bệnh lý gãy eo đốt sống, vị trí không xác định	
M43.1	Bệnh lý trượt đốt sống do gãy eo	
M43.10	Bệnh lý trượt đốt sống do gãy eo, nhiều vị trí của cột sống	
M43.11	Bệnh lý trượt đốt sống do gãy eo, vùng trục - đội - chẩm	
M43.12	Bệnh lý trượt đốt sống do gãy eo, vùng cổ	
M43.13	Bệnh lý trượt đốt sống do gãy eo, vùng cổ - ngực	
M43.14	Bệnh lý trượt đốt sống do gãy eo, vùng (lồng) ngực	
M43.15	Bệnh lý trượt đốt sống do gãy eo, vùng ngực - thắt lưng	
M43.16	Bệnh lý trượt đốt sống do gãy eo, vùng thắt lưng	
M43.17	Bệnh lý trượt đốt sống do gãy eo, vùng thắt lưng - cùng	
M43.18	Bệnh lý trượt đốt sống do gãy eo, vùng cùng và/hoặc cùng - cụt	
M43.19	Bệnh lý trượt đốt sống do gãy eo, vị trí không xác định	
M43.2	Dính cột sống khác	
M43.20	Dính cột sống khác, nhiều vị trí của cột sống	
M43.21	Dính cột sống khác, vùng trục - đội - chẩm	
M43.22	Dính cột sống khác, vùng cổ	
M43.23	Dính cột sống khác, vùng cổ - ngực	
M43.24	Dính cột sống khác, vùng (lồng) ngực	
M43.25	Dính cột sống khác, vùng ngực - thắt lưng	
M43.26	Dính cột sống khác, vùng thắt lưng	
M43.27	Dính cột sống khác, vùng thắt lưng - cùng	
M43.28	Dính cột sống khác, vùng cùng và/hoặc cùng - cụt	
M43.29	Dính cột sống khác, vị trí không xác định	
M43.3	Trượt khớp trục - đội tái phát có bệnh lý tủy sống	
M43.4	Trượt khớp trục - đội tái phát khác	
M43.5	Trượt đốt sống tái phát khác	
M43.50	Trượt đốt sống tái phát khác, nhiều vị trí của cột sống	
M43.51	Trượt đốt sống tái phát khác, vùng trục - đội - chẩm	
M43.52	Trượt đốt sống tái phát khác, vùng cổ	
M43.53	Trượt đốt sống tái phát khác, vùng cổ - ngực	
M43.54	Trượt đốt sống tái phát khác, vùng (lồng) ngực	
M43.55	Trượt đốt sống tái phát khác, vùng ngực - thắt lưng	
M43.56	Trượt đốt sống tái phát khác, vùng thắt lưng	
M43.57	Trượt đốt sống tái phát khác, vùng thắt lưng - cùng	
M43.58	Trượt đốt sống tái phát khác, vùng cùng và/hoặc cùng - cụt	
M43.59	Trượt đốt sống tái phát khác, vị trí không xác định	
M43.6	Vẹo cổ	
M43.60	Vẹo cổ, nhiều vị trí của cột sống	
M43.61	Vẹo cổ, vùng trục - đội - chẩm	
M43.62	Vẹo cổ, vùng cổ	
M43.63	Vẹo cổ, vùng cổ - ngực	
M43.69	Vẹo cổ, vị trí không xác định	
M43.8	Bệnh lý biến dạng cột sống (mắc phải) xác định khác	
M43.80	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, nhiều vị trí của cột sống	
M43.81	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng trục - đội - chẩm	
M43.82	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng cổ	
M43.83	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng cổ - ngực	
M43.84	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng (lồng) ngực	
M43.85	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng ngực - thắt lưng	
M43.86	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng thắt lưng	
M43.87	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng thắt lưng - cùng	
M43.88	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vùng cùng và/hoặc cùng - cụt	
M43.89	Bệnh lý biến dạng cột sống (mắc phải) xác định khác, vị trí không xác định	
M43.9	Bệnh lý biến dạng cột sống (mắc phải), không xác định	Cột sống cong không xác định khác
M43.90	Bệnh lý biến dạng cột sống (mắc phải), không xác định, nhiều vị trí của cột sống	
M43.91	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng trục - đội - chẩm	
M43.92	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng cổ	
M43.93	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng cổ - ngực	
M43.94	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng (lồng) ngực	
M43.95	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng ngực - thắt lưng	
M43.96	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng thắt lưng	
M43.97	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng thắt lưng - cùng	
M43.98	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vùng cùng và/hoặc cùng - cụt	
M43.99	Bệnh lý biến dạng cột sống (mắc phải), không xác định, vị trí không xác định	
M45	Viêm đốt sống dính khớp	
M45.0	Viêm đốt sống dính khớp, nhiều vị trí của cột sống	
M45.1	Viêm đốt sống dính khớp, vùng trục - đội - chẩm	
M45.2	Viêm đốt sống dính khớp, vùng cổ	
M45.3	Viêm đốt sống dính khớp, vùng cổ - ngực	
M45.4	Viêm đốt sống dính khớp, vùng (lồng) ngực	
M45.5	Viêm đốt sống dính khớp, vùng ngực - thắt lưng	
M45.6	Viêm đốt sống dính khớp, vùng thắt lưng	
M45.7	Viêm đốt sống dính khớp, vùng thắt lưng - cùng	
M45.8	Viêm đốt sống dính khớp, vùng cùng và/hoặc cùng - cụt	
M45.9	Viêm đốt sống dính khớp, vị trí không xác định	
M46	Bệnh lý viêm đốt sống khác	
M46.0	Bệnh lý điểm bám gân - dây chằng cột sống	Bệnh lý dây chằng hoặc cơ bám của cột sống
M46.00	Bệnh lý điểm bám gân - dây chằng cột sống, nhiều vị trí của cột sống	
M46.01	Bệnh lý điểm bám gân - dây chằng cột sống, vùng trục - đội - chẩm	
M46.02	Bệnh lý điểm bám gân - dây chằng cột sống, vùng cổ	
M46.03	Bệnh lý điểm bám gân - dây chằng cột sống, vùng cổ - ngực	
M46.04	Bệnh lý điểm bám gân - dây chằng cột sống, vùng (lồng) ngực	
M46.05	Bệnh lý điểm bám gân - dây chằng cột sống, vùng ngực - thắt lưng	
M46.06	Bệnh lý điểm bám gân - dây chằng cột sống, vùng thắt lưng	
M46.07	Bệnh lý điểm bám gân - dây chằng cột sống, vùng thắt lưng - cùng	
M46.08	Bệnh lý điểm bám gân - dây chằng cột sống, vùng cùng và/hoặc cùng - cụt	
M46.09	Bệnh lý điểm bám gân - dây chằng cột sống, vị trí không xác định	
M46.1	Viêm khớp xương cùng - xương hông, không phân loại mục khác	
M46.2	Viêm xương tủy đốt sống	
M46.20	Viêm xương tủy đốt sống, nhiều vị trí của cột sống	
M46.21	Viêm xương tủy đốt sống, vùng trục - đội - chẩm	
M46.22	Viêm xương tủy đốt sống, vùng cổ	
M46.23	Viêm xương tủy đốt sống, vùng cổ - ngực	
M46.24	Viêm xương tủy đốt sống, vùng (lồng) ngực	
M46.25	Viêm xương tủy đốt sống, vùng ngực - thắt lưng	
M46.26	Viêm xương tủy đốt sống, vùng thắt lưng	
M46.27	Viêm xương tủy đốt sống, vùng thắt lưng - cùng	
M46.28	Viêm xương tủy đốt sống, vùng cùng và/hoặc cùng - cụt	
M46.29	Viêm xương tủy đốt sống, vị trí không xác định	
M46.3	Nhiễm trùng đĩa đệm cột sống (sinh mủ)	
M46.30	Nhiễm trùng đĩa đệm cột sống (sinh mủ), nhiều vị trí của cột sống	
M46.31	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng trục - đội - chẩm	
M46.32	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng cổ	
M46.33	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng cổ - ngực	
M46.34	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng (lồng) ngực	
M46.35	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng ngực - thắt lưng	
M46.36	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng thắt lưng	
M46.37	nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng thắt lưng - cùng	
M46.38	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vùng cùng và/hoặc cùng - cụt	
M46.39	Nhiễm trùng đĩa đệm cột sống (sinh mủ), vị trí không xác định	
M46.4	Viêm đĩa đệm, không xác định	
M46.40	Viêm đĩa đệm, không xác định, nhiều vị trí của cột sống	
M46.41	Viêm đĩa đệm, không xác định, vùng trục - đội - chẩm	
M46.42	Viêm đĩa đệm, không xác định, vùng cổ	
M46.43	Viêm đĩa đệm, không xác định, vùng cổ - ngực	
M46.44	Viêm đĩa đệm, không xác định, vùng (lồng) ngực	
M46.45	Viêm đĩa đệm, không xác định, vùng ngực - thắt lưng	
M46.46	Viêm đĩa đệm, không xác định, vùng thắt lưng	
M46.47	Viêm đĩa đệm, không xác định, vùng thắt lưng - cùng	
M46.48	Viêm đĩa đệm, không xác định, vùng cùng và/hoặc cùng - cụt	
M46.49	Viêm đĩa đệm, không xác định, vị trí không xác định	
M46.5	Bệnh lý đốt sống khác do nhiễm trùng	
M46.50	Bệnh lý đốt sống khác do nhiễm trùng, nhiều vị trí của cột sống	
M46.51	Bệnh lý đốt sống khác do nhiễm trùng, vùng trục - đội - chẩm	
M46.52	Bệnh lý đốt sống khác do nhiễm trùng, vùng cổ	
M46.53	Bệnh lý đốt sống khác do nhiễm trùng, vùng cổ - ngực	
M46.54	Bệnh lý đốt sống khác do nhiễm trùng, vùng (lồng) ngực	
M46.55	Bệnh lý đốt sống khác do nhiễm trùng, vùng ngực - thắt lưng	
M46.56	Bệnh lý đốt sống khác do nhiễm trùng, vùng thắt lưng	
M46.57	Bệnh lý đốt sống khác do nhiễm trùng, vùng thắt lưng - cùng	
M46.58	Bệnh lý đốt sống khác do nhiễm trùng, vùng cùng và/hoặc cùng - cụt	
M46.59	Bệnh lý đốt sống khác do nhiễm trùng, vị trí không xác định	
M46.8	Bệnh lý viêm đốt sống xác định khác	
M46.80	Bệnh lý viêm đốt sống xác định khác, nhiều vị trí của cột sống	
M46.81	Bệnh lý viêm đốt sống xác định khác, vùng trục - đội - chẩm	
M46.82	Bệnh lý viêm đốt sống xác định khác, vùng cổ	
M46.83	Bệnh lý viêm đốt sống xác định khác, vùng cổ - ngực	
M46.84	Bệnh lý viêm đốt sống xác định khác, vùng (lồng) ngực	
M46.85	Bệnh lý viêm đốt sống xác định khác, vùng ngực - thắt lưng	
M46.86	Bệnh lý viêm đốt sống xác định khác, vùng thắt lưng	
M46.87	Bệnh lý viêm đốt sống xác định khác, vùng thắt lưng - cùng	
M46.88	Bệnh lý viêm đốt sống xác định khác, vùng cùng và/hoặc cùng - cụt	
M46.89	Bệnh lý viêm đốt sống xác định khác, vị trí không xác định	
M46.9	Bệnh lý viêm đốt sống, không xác định	
M46.90	Bệnh lý viêm đốt sống, không xác định, nhiều vị trí của cột sống	
M46.91	Bệnh lý viêm đốt sống, không xác định, vùng trục - đội - chẩm	
M46.92	Bệnh lý viêm đốt sống, không xác định, vùng cổ	
M46.93	Bệnh lý viêm đốt sống, không xác định, vùng cổ - ngực	
M46.94	Bệnh lý viêm đốt sống, không xác định, vùng (lồng) ngực	
M46.95	Bệnh lý viêm đốt sống, không xác định, vùng ngực - thắt lưng	
M46.96	Bệnh lý viêm đốt sống, không xác định, vùng thắt lưng	
M46.97	Bệnh lý viêm đốt sống, không xác định, vùng thắt lưng - cùng	
M46.98	Bệnh lý viêm đốt sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M46.99	Bệnh lý viêm đốt sống, không xác định, vị trí không xác định	
M47	Thoái hóa đốt sống	
M47.00†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), nhiều vị trí của cột sống	
M47.01†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng trục - đội - chẩm	
M47.02†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng cổ	
M47.03†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng cổ - ngực	
M47.04†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng (lồng) ngực	
M47.05†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng ngực - thắt lưng	
M47.06†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng thắt lưng	
M47.07†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng thắt lưng - cùng	
M47.08†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vùng cùng và/hoặc cùng - cụt	
M47.09†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*), vị trí không xác định	
M47.0†	Hội chứng chèn ép động mạch đốt sống và/hoặc động mạch cột sống trước (G99.2*)	
M47.1	Thoái hóa đốt sống khác kèm bệnh lý tủy sống	
M47.10	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, nhiều vị trí của cột sống	
M47.11	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng trục - đội - chẩm	
M47.12	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng cổ	
M47.13	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng cổ - ngực	
M47.14	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng (lồng) ngực	
M47.15	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng ngực - thắt lưng	
M47.16	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng thắt lưng	
M47.17	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng thắt lưng - cùng	
M47.18	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vùng cùng và/hoặc cùng - cụt	
M47.19	Thoái hóa đốt sống khác kèm bệnh lý tủy sống, vị trí không xác định	
M47.2	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh	
M47.20	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, nhiều vị trí của cột sống	
M47.21	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng trục - đội - chẩm	
M47.22	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng cổ	
M47.23	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng cổ - ngực	
M47.24	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng (lồng) ngực	
M47.25	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng ngực - thắt lưng	
M47.26	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng thắt lưng	
M47.27	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng thắt lưng - cùng	
M47.28	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vùng cùng và/hoặc cùng - cụt	
M47.29	Thoái hóa đốt sống khác kèm bệnh lý rễ thần kinh, vị trí không xác định	
M47.8	Thoái hóa đốt sống khác	Thoái hóa cột sống cổ không có bệnh lý tủy hoặc bệnh lý rễ|Thoái hóa cột sống lưng không có bệnh lý tủy hoặc bệnh lý rễ|Thoái hóa thắt lưng cùng không có bệnh lý tủy hoặc bệnh lý rễ
M47.80	Thoái hóa đốt sống khác, nhiều vị trí của cột sống	
M47.81	Thoái hóa đốt sống khác, vùng trục - đội - chẩm	
M47.82	Thoái hóa đốt sống khác, vùng cổ	
M47.83	Thoái hóa đốt sống khác, vùng cổ - ngực	
M47.84	Thoái hóa đốt sống khác, vùng (lồng) ngực	
M47.85	Thoái hóa đốt sống khác, vùng ngực - thắt lưng	
M47.86	Thoái hóa đốt sống khác, vùng thắt lưng	
M47.87	Thoái hóa đốt sống khác, vùng thắt lưng - cùng	
M47.88	Thoái hóa đốt sống khác, vùng cùng và/hoặc cùng - cụt	
M47.89	Thoái hóa đốt sống khác, vị trí không xác định	
M47.9	Thoái hóa đốt sống, không xác định	
M47.90	Thoái hóa đốt sống, không xác định, nhiều vị trí của cột sống	
M47.91	Thoái hóa đốt sống, không xác định, vùng trục - đội - chẩm	
M47.92	Thoái hóa đốt sống, không xác định, vùng cổ	
M47.93	Thoái hóa đốt sống, không xác định, vùng cổ - ngực	
M47.94	Thoái hóa đốt sống, không xác định, vùng (lồng) ngực	
M47.95	Thoái hóa đốt sống, không xác định, vùng ngực - thắt lưng	
M47.96	Thoái hóa đốt sống, không xác định, vùng thắt lưng	
M47.97	Thoái hóa đốt sống, không xác định, vùng thắt lưng - cùng	
M47.98	Thoái hóa đốt sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M47.99	Thoái hóa đốt sống, không xác định, vị trí không xác định	
M48	Bệnh lý đốt sống khác	Bệnh đốt sống khác
M48.0	Hẹp ống sống	Hẹp phần đuôi [ống sống]
M48.00	Hẹp ống sống, nhiều vị trí của cột sống	
M48.01	Hẹp ống sống, vùng trục - đội - chẩm	
M48.02	Hẹp ống sống, vùng cổ	
M48.03	Hẹp ống sống, vùng cổ - ngực	
M48.04	Hẹp ống sống, vùng (lồng) ngực	
M48.05	Hẹp ống sống, vùng ngực - thắt lưng	
M48.06	Hẹp ống sống, vùng thắt lưng	
M48.07	Hẹp ống sống, vùng thắt lưng - cùng	
M48.08	Hẹp ống sống, vùng cùng và/hoặc cùng - cụt	
M48.09	Hẹp ống sống, vị trí không xác định	
M48.1	Dính khớp do tăng tạo xương [Forestier]	Hội chứng tăng tạo xương lan tỏa nguyên phát [DISH]
M48.10	Dính khớp do tăng tạo xương [Forestier], nhiều vị trí của cột sống	
M48.11	Dính khớp do tăng tạo xương [Forestier], vùng trục - đội - chẩm	
M48.12	Dính khớp do tăng tạo xương [Forestier], vùng cổ	
M48.13	Dính khớp do tăng tạo xương [Forestier], vùng cổ - ngực	
M48.14	Dính khớp do tăng tạo xương [Forestier], vùng (lồng) ngực	
M48.15	Dính khớp do tăng tạo xương [Forestier], vùng ngực - thắt lưng	
M48.16	Dính khớp do tăng tạo xương [Forestier], vùng thắt lưng	
M48.17	Dính khớp do tăng tạo xương [Forestier], vùng thắt lưng - cùng	
M48.18	Dính khớp do tăng tạo xương [Forestier], vùng cùng và/hoặc cùng - cụt	
M48.19	Dính khớp do tăng tạo xương [Forestier], vị trí không xác định	
M48.2	Bệnh quá phát gai sau	
M48.20	Bệnh quá phát gai sau, nhiều vị trí của cột sống	
M48.21	Bệnh quá phát gai sau, vùng trục - đội - chẩm	
M48.22	Bệnh quá phát gai sau, vùng cổ	
M48.23	Bệnh quá phát gai sau, vùng cổ - ngực	
M48.24	Bệnh quá phát gai sau, vùng (lồng) ngực	
M48.25	Bệnh quá phát gai sau, vùng ngực - thắt lưng	
M48.26	Bệnh quá phát gai sau, vùng thắt lưng	
M48.27	Bệnh quá phát gai sau, vùng thắt lưng - cùng	
M48.28	Bệnh quá phát gai sau, vùng cùng và/hoặc cùng - cụt	
M48.29	Bệnh quá phát gai sau, vị trí không xác định	
M48.3	Bệnh lý đốt sống do chấn thương	
M48.30	Bệnh lý đốt sống do chấn thương, nhiều vị trí của cột sống	
M48.31	Bệnh lý đốt sống do chấn thương, vùng trục - đội - chẩm	
M48.32	Bệnh lý đốt sống do chấn thương, vùng cổ	
M48.33	Bệnh lý đốt sống do chấn thương, vùng cổ - ngực	
M48.34	Bệnh lý đốt sống do chấn thương, vùng (lồng) ngực	
M48.35	Bệnh lý đốt sống do chấn thương, vùng ngực - thắt lưng	
M48.36	Bệnh lý đốt sống do chấn thương, vùng thắt lưng	
M48.37	Bệnh lý đốt sống do chấn thương, vùng thắt lưng - cùng	
M48.38	Bệnh lý đốt sống do chấn thương, vùng cùng và/hoặc cùng - cụt	
M48.39	Bệnh lý đốt sống do chấn thương, vị trí không xác định	
M48.4	Gãy đốt sống do mỏi	Gãy đốt sống do gắng sức
M48.40	Gãy đốt sống do mỏi, nhiều vị trí của cột sống	
M48.41	Gãy đốt sống do mỏi, vùng trục - đội - chẩm	
M48.42	Gãy đốt sống do mỏi, vùng cổ	
M48.43	Gãy đốt sống do mỏi, vùng cổ - ngực	
M48.44	Gãy đốt sống do mỏi, vùng (lồng) ngực	
M48.45	Gãy đốt sống do mỏi, vùng ngực - thắt lưng	
M48.46	Gãy đốt sống do mỏi, vùng thắt lưng	
M48.47	Gãy đốt sống do mỏi, vùng thắt lưng - cùng	
M48.48	Gãy đốt sống do mỏi, vùng cùng và/hoặc cùng - cụt	
M48.49	Gãy đốt sống do mỏi, vị trí không xác định	
M48.5	Xẹp đốt sống, không phân loại mục khác	
M48.50	Xẹp đốt sống, không phân loại mục khác, nhiều vị trí của cột sống	
M48.51	Xẹp đốt sống, không phân loại mục khác, vùng trục - đội - chẩm	
M48.52	Xẹp đốt sống, không phân loại mục khác, vùng cổ	
M48.53	Xẹp đốt sống, không phân loại mục khác, vùng cổ - ngực	
M48.54	Xẹp đốt sống, không phân loại mục khác, vùng (lồng) ngực	
M48.55	Xẹp đốt sống, không phân loại mục khác, vùng ngực - thắt lưng	
M48.56	Xẹp đốt sống, không phân loại mục khác, vùng thắt lưng	
M48.57	Xẹp đốt sống, không phân loại mục khác, vùng thắt lưng - cùng	
M48.58	Xẹp đốt sống, không phân loại mục khác, vùng cùng và/hoặc cùng - cụt	
M48.59	Xẹp đốt sống, không phân loại mục khác, vị trí không xác định	
M48.8	Bệnh lý đốt sống xác định khác	Cốt hóa dây chằng dọc sau
M48.80	Bệnh lý đốt sống xác định khác, nhiều vị trí của cột sống	
M48.81	Bệnh lý đốt sống xác định khác, vùng trục - đội - chẩm	
M48.82	Bệnh lý đốt sống xác định khác, vùng cổ	
M48.83	Bệnh lý đốt sống xác định khác, vùng cổ - ngực	
M48.84	Bệnh lý đốt sống xác định khác, vùng (lồng) ngực	
M48.85	Bệnh lý đốt sống xác định khác, vùng ngực - thắt lưng	
M48.86	Bệnh lý đốt sống xác định khác, vùng thắt lưng	
M48.87	Bệnh lý đốt sống xác định khác, vùng thắt lưng - cùng	
M48.88	Bệnh lý đốt sống xác định khác, vùng cùng và/hoặc cùng - cụt	
M48.89	Bệnh lý đốt sống xác định khác, vị trí không xác định	
M48.9	Bệnh lý đốt sống, không xác định	
M48.90	Bệnh lý đốt sống, không xác định, nhiều vị trí của cột sống	
M48.91	Bệnh lý đốt sống, không xác định, vùng trục - đội - chẩm	
M48.92	Bệnh lý đốt sống, không xác định, vùng cổ	
M48.93	Bệnh lý đốt sống, không xác định, vùng cổ - ngực	
M48.94	Bệnh lý đốt sống, không xác định, vùng (lồng) ngực	
M48.95	Bệnh lý đốt sống, không xác định, vùng ngực - thắt lưng	
M48.96	Bệnh lý đốt sống, không xác định, vùng thắt lưng	
M48.97	Bệnh lý đốt sống, không xác định, vùng thắt lưng - cùng	
M48.98	Bệnh lý đốt sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M48.99	Bệnh lý đốt sống, không xác định, vị trí không xác định	
M49.*	Bệnh lý đốt sống do bệnh phân loại mục khác	
M49.0*	Lao cột sống (A18.0†)	Cong cột sống Pott [cong cột sống do lao] [lao cột sống Pott]
M49.00*	Lao cột sống (A18.0†), nhiều vị trí của cột sống	
M49.01*	Lao cột sống (A18.0†), vùng trục - đội - chẩm	
M49.02*	Lao cột sống (A18.0†), vùng cổ	
M49.03*	Lao cột sống (A18.0†), vùng cổ - ngực	
M49.04*	Lao cột sống (A18.0†), vùng (lồng) ngực	
M49.05*	Lao cột sống (A18.0†), vùng ngực - thắt lưng	
M49.06*	Lao cột sống (A18.0†), vùng thắt lưng	
M49.07*	Lao cột sống (A18.0†), vùng thắt lưng - cùng	
M49.08*	Lao cột sống (A18.0†), vùng cùng và/hoặc cùng - cụt	
M49.09*	Lao cột sống (A18.0†), vị trí không xác định	
M49.1*	Viêm đốt sống do Brucella (A23.-†)	
M49.10*	Viêm đốt sống do Brucella (A23.-†), nhiều vị trí của cột sống	
M49.11*	Viêm đốt sống do Brucella (A23.-†), vùng trục - đội - chẩm	
M49.12*	Viêm đốt sống do Brucella (A23.-†), vùng cổ	
M49.13*	Viêm đốt sống do Brucella (A23.-†), vùng cổ - ngực	
M49.14*	Viêm đốt sống do Brucella (A23.-†), vùng (lồng) ngực	
M49.15*	Viêm đốt sống do Brucella (A23.-†), vùng ngực - thắt lưng	
M49.16*	Viêm đốt sống do Brucella (A23.-†), vùng thắt lưng	
M49.17*	Viêm đốt sống do Brucella (A23.-†), vùng thắt lưng - cùng	
M49.18*	Viêm đốt sống do Brucella (A23.-†), vùng cùng và/hoặc cùng - cụt	
M49.19*	Viêm đốt sống do Brucella (A23.-†), vị trí không xác định	
M49.2*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†)	
M49.20*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), nhiều vị trí của cột sống	
M49.21*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng trục - đội - chẩm	
M49.22*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng cổ	
M49.23*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng cổ - ngực	
M49.24*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng (lồng) ngực	
M49.25*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng ngực - thắt lưng	
M49.26*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng thắt lưng	
M49.27*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng thắt lưng - cùng	
M49.28*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vùng cùng và/hoặc cùng - cụt	
M49.29*	Viêm đốt sống do vi khuẩn đường ruột (A01-A04†), vị trí không xác định	
M49.3*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
M49.30*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, nhiều vị trí của cột sống	
M49.31*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng trục - đội - chẩm	
M49.32*	Bệnh lý đốt sống do các bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng cổ	
M49.33*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng cổ - ngực	
M49.34*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng (lồng) ngực	
M49.35*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng ngực - thắt lưng	
M49.36*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng thắt lưng	
M49.37*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng thắt lưng - cùng	
M49.38*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vùng cùng và/hoặc cùng - cụt	
M49.39*	Bệnh lý đốt sống do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác, vị trí không xác định	
M49.4*	Bệnh lý đốt sống thần kinh	
M49.40*	Bệnh lý đốt sống thần kinh, nhiều vị trí của cột sống	
M49.41*	Bệnh lý đốt sống thần kinh, vùng trục - đội - chẩm	
M49.42*	Bệnh lý đốt sống thần kinh, vùng cổ	
M49.43*	Bệnh lý đốt sống thần kinh, vùng cổ - ngực	
M49.44*	Bệnh lý đốt sống thần kinh, vùng (lồng) ngực	
M49.45*	Bệnh lý đốt sống thần kinh, vùng ngực - thắt lưng	
M49.46*	Bệnh lý đốt sống thần kinh, vùng thắt lưng	
M49.47*	Bệnh lý đốt sống thần kinh, vùng thắt lưng - cùng	
M49.48*	Bệnh lý đốt sống thần kinh, vùng cùng và/hoặc cùng - cụt	
M49.49*	Bệnh lý đốt sống thần kinh, vị trí không xác định	
M49.5*	Xẹp đốt sống do bệnh phân loại mục khác	
M49.50*	Xẹp đốt sống do bệnh phân loại mục khác, nhiều vị trí của cột sống	
M49.51*	Xẹp đốt sống do bệnh phân loại mục khác, vùng trục - đội - chẩm	
M49.52*	Xẹp đốt sống do bệnh phân loại mục khác, vùng cổ	
M49.53*	Xẹp đốt sống do bệnh phân loại mục khác, vùng cổ - ngực	
M49.54*	Xẹp đốt sống do bệnh phân loại mục khác, vùng (lồng) ngực	
M49.55*	Xẹp đốt sống do bệnh phân loại mục khác, vùng ngực - thắt lưng	
M49.56*	Xẹp đốt sống do bệnh phân loại mục khác, vùng thắt lưng	
M49.57*	Xẹp đốt sống do bệnh phân loại mục khác, vùng thắt lưng - cùng	
M49.58*	Xẹp đốt sống do bệnh phân loại mục khác, vùng cùng và/hoặc cùng - cụt	
M49.59*	Xẹp đốt sống do bệnh phân loại mục khác, vị trí không xác định	
M49.8*	Bệnh lý đốt sống do bệnh khác phân loại mục khác	
M49.80*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, nhiều vị trí của cột sống	
M49.81*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng trục - đội - chẩm	
M49.82*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng cổ	
M49.83*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng cổ - ngực	
M49.84*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng (lồng) ngực	
M49.85*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng ngực - thắt lưng	
M49.86*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng thắt lưng	
M49.87*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng thắt lưng - cùng	
M49.88*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vùng cùng và/hoặc cùng - cụt	
M49.89*	Bệnh lý đốt sống do bệnh khác phân loại mục khác, vị trí không xác định	
M50	Rối loạn đĩa đệm đốt sống cổ	
M50.0†	Rối loạn đĩa đệm đốt sống cổ có bệnh lý tủy sống (G99.2*)	
M50.1	Rối loạn đĩa đệm cột sống cổ có bệnh lý rễ thần kinh	
M50.2	Thoát vị đĩa đệm cột sống cổ khác	
M50.3	Thoái hóa đĩa đệm đốt sống cổ khác	
M50.8	Rối loạn đĩa đệm đốt sống cổ khác	
M50.9	Rối loạn đĩa đệm đốt sống cổ, không xác định	
M51	Rối loạn đĩa đệm cột sống khác	
M51.0†	Rối loạn đĩa đệm cột sống thắt lưng và/hoặc đĩa đệm đốt sống khác kèm bệnh lý tủy sống (G99.2*)	
M51.1†	Rối loạn đĩa đệm cột sống thắt lưng và/hoặc đĩa đệm đốt sống khác kèm bệnh lý rễ thần kinh (G55.1*)	
M51.2	Thoát vị đĩa đệm cột sống xác định khác	Đau lưng do thoát vị đĩa đệm
M51.3	Thoái hóa đĩa đệm cột sống xác định khác	
M51.4	Thoát vị đĩa đệm nội xốp [nốt Schmorl]	
M51.8	Rối loạn xác định khác của đĩa đệm cột sống khác	
M51.9	Rối loạn đĩa đệm cột sống, không xác định	
M53	Bệnh lý cột sống khác, không phân loại mục khác	
M53.0	Hội chứng đầu - cổ	Hội chứng giao cảm cổ sau
M53.00	Hội chứng đầu - cổ, nhiều vị trí của cột sống	
M53.01	Hội chứng đầu - cổ, vùng trục - đội - chẩm	
M53.02	Hội chứng đầu - cổ, vùng cổ	
M53.03	Hội chứng đầu - cổ, vùng cổ - ngực	
M53.09	Hội chứng đầu - cổ, vị trí không xác định	
M53.1	Hội chứng cổ vai cánh tay	
M53.10	Hội chứng cổ vai cánh tay, nhiều vị trí của cột sống	
M53.11	Hội chứng cổ vai cánh tay, vùng trục - đội - chẩm	
M53.12	Hội chứng cổ vai cánh tay, vùng cổ	
M53.13	Hội chứng cổ vai cánh tay, vùng cổ - ngực	
M53.19	Hội chứng cổ vai cánh tay, vị trí không xác định	
M53.2	Cột sống mất vững	
M53.20	Cột sống mất vững, nhiều vị trí của cột sống	
M53.21	Cột sống mất vững, vùng trục - đội - chẩm	
M53.22	Cột sống mất vững, vùng cổ	
M53.23	Cột sống mất vững, vùng cổ - ngực	
M53.24	Cột sống mất vững, vùng (lồng) ngực	
M53.25	Cột sống mất vững, vùng ngực - thắt lưng	
M53.26	Cột sống mất vững, vùng thắt lưng	
M53.27	Cột sống mất vững, vùng thắt lưng - cùng	
M53.28	Cột sống mất vững, vùng cùng và/hoặc cùng - cụt	
M53.29	Cột sống mất vững, vị trí không xác định	
M53.3	Rối loạn cùng cụt, không phân loại mục khác	Đau xương cụt
M53.8	Bệnh lý cột sống xác định khác	
M53.80	Bệnh lý cột sống xác định khác, nhiều vị trí của cột sống	
M53.81	Bệnh lý cột sống xác định khác, vùng trục - đội - chẩm	
M53.82	Bệnh lý cột sống xác định khác, vùng cổ	
M53.83	Bệnh lý cột sống xác định khác, vùng cổ - ngực	
M53.84	Bệnh lý cột sống xác định khác, vùng (lồng) ngực	
M53.85	Bệnh lý cột sống xác định khác, vùng ngực - thắt lưng	
M53.86	Bệnh lý cột sống xác định khác, vùng thắt lưng	
M53.87	Bệnh lý cột sống xác định khác, vùng thắt lưng - cùng	
M53.88	Bệnh lý cột sống xác định khác, vùng cùng và/hoặc cùng - cụt	
M53.89	Bệnh lý cột sống xác định khác, vị trí không xác định	
M53.9	Bệnh lý cột sống, không xác định	
M53.90	Bệnh lý cột sống, không xác định, nhiều vị trí của cột sống	
M53.91	Bệnh lý cột sống, không xác định, vùng trục - đội - chẩm	
M53.92	Bệnh lý cột sống, không xác định, vùng cổ	
M53.93	Bệnh lý cột sống, không xác định, vùng cổ - ngực	
M53.94	Bệnh lý cột sống, không xác định, vùng (lồng) ngực	
M53.95	Bệnh lý cột sống, không xác định, vùng ngực - thắt lưng	
M53.96	Bệnh lý cột sống, không xác định, vùng thắt lưng	
M53.97	Bệnh lý cột sống, không xác định, vùng thắt lưng - cùng	
M53.98	Bệnh lý cột sống, không xác định, vùng cùng và/hoặc cùng - cụt	
M53.99	Bệnh lý cột sống, không xác định, vị trí không xác định	
M54	Đau lưng	
M54.0	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng	
M54.00	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, nhiều vị trí của cột sống	
M54.01	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng trục - đội - chẩm	
M54.02	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng cổ	
M54.03	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng cổ - ngực	
M54.04	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng (lồng) ngực	
M54.05	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng ngực - thắt lưng	
M54.06	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng thắt lưng	
M54.07	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng thắt lưng - cùng	
M54.08	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vùng cùng và/hoặc cùng - cụt	
M54.09	Viêm mô mỡ dưới da tại vùng cổ và/hoặc lưng, vị trí không xác định	
M54.1	Bệnh lý rễ thần kinh	
M54.10	Bệnh lý rễ thần kinh, nhiều vị trí của cột sống	
M54.11	Bệnh lý rễ thần kinh, vùng trục - đội - chẩm	
M54.12	Bệnh lý rễ thần kinh, vùng cổ	
M54.13	Bệnh lý rễ thần kinh, vùng cổ - ngực	
M54.14	Bệnh lý rễ thần kinh, vùng (lồng) ngực	
M54.15	Bệnh lý rễ thần kinh, vùng ngực - thắt lưng	
M54.16	Bệnh lý rễ thần kinh, vùng thắt lưng	
M54.17	Bệnh lý rễ thần kinh, vùng thắt lưng - cùng	
M54.18	Bệnh lý rễ thần kinh, vùng cùng và/hoặc cùng - cụt	
M54.19	Bệnh lý rễ thần kinh, vị trí không xác định	
M54.2	Đau vai gáy	
M54.20	Đau vai gáy, nhiều vị trí của cột sống	
M54.21	Đau vai gáy, vùng trục - đội - chẩm	
M54.22	Đau vai gáy, vùng cổ	
M54.23	Đau vai gáy, vùng cổ - ngực	
M54.29	Đau vai gáy, vị trí không xác định	
M54.3	Đau dây thần kinh hông to [dây thần kinh tọa]	
M54.30	Đau dây thần kinh hông to [dây thần kinh tọa], nhiều vị trí của cột sống	
M54.35	Đau dây thần kinh hông to [dây thần kinh tọa], vùng ngực - thắt lưng	
M54.36	Đau dây thần kinh hông to [dây thần kinh tọa], vùng thắt lưng	
M54.37	Đau dây thần kinh hông to [dây thần kinh tọa], vùng thắt lưng - cùng	
M54.38	Đau dây thần kinh hông to [dây thần kinh tọa], vùng cùng và/hoặc cùng - cụt	
M54.39	Đau dây thần kinh hông to [dây thần kinh tọa], vị trí không xác định	
M54.4	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa]	
M54.40	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], nhiều vị trí của cột sống	
M54.45	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], vùng ngực - thắt lưng	
M54.46	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], vùng thắt lưng	
M54.47	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], vùng thắt lưng - cùng	
M54.48	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], vùng cùng và/hoặc cùng - cụt	
M54.49	Đau thắt lưng cấp tính kèm đau dây thần kinh hông to [dây thần kinh tọa], vị trí không xác định	
M54.5	Đau thắt lưng	
M54.50	Đau thắt lưng, nhiều vị trí của cột sống	
M54.55	Đau thắt lưng, vùng ngực - thắt lưng	
M54.56	Đau thắt lưng, vùng thắt lưng	
M54.57	Đau thắt lưng, vùng thắt lưng - cùng	
M54.58	Đau thắt lưng, vùng cùng và/hoặc cùng - cụt	
M54.59	Đau thắt lưng, vị trí không xác định	
M54.6	Đau cột sống lồng ngực	
M54.60	Đau cột sống lồng ngực, nhiều vị trí của cột sống	
M54.63	Đau cột sống lồng ngực, vùng cổ - ngực	
M54.64	Đau cột sống lồng ngực, vùng (lồng) ngực	
M54.65	Đau cột sống lồng ngực, vùng ngực - thắt lưng	
M54.69	Đau cột sống lồng ngực, vị trí không xác định	
M54.8	Đau lưng khác	
M54.80	Đau lưng khác, nhiều vị trí của cột sống	
M54.81	Đau lưng khác, vùng trục - đội - chẩm	
M54.82	Đau lưng khác, vùng cổ	
M54.83	Đau lưng khác, vùng cổ - ngực	
M54.84	Đau lưng khác, vùng (lồng) ngực	
M54.85	Đau lưng khác, vùng ngực - thắt lưng	
M54.86	Đau lưng khác, vùng thắt lưng	
M54.87	Đau lưng khác, vùng thắt lưng - cùng	
M54.88	Đau lưng khác, vùng cùng và/hoặc cùng - cụt	
M54.89	Đau lưng khác, vị trí không xác định	
M54.9	Đau lưng, không xác định	Đau lưng không xác định khác
M54.90	Đau lưng, không xác định, nhiều vị trí của cột sống	
M54.91	Đau lưng, không xác định, vùng trục - đội - chẩm	
M54.92	Đau lưng, không xác định, vùng cổ	
M54.93	Đau lưng, không xác định, vùng cổ - ngực	
M54.94	Đau lưng, không xác định, vùng (lồng) ngực	
M54.95	Đau lưng, không xác định, vùng ngực - thắt lưng	
M54.96	Đau lưng, không xác định, vùng thắt lưng	
M54.97	Đau lưng, không xác định, vùng thắt lưng - cùng	
M54.98	Đau lưng, không xác định, vùng cùng và/hoặc cùng - cụt	
M54.99	Đau lưng, không xác định, vị trí không xác định	
M60	Viêm cơ	
M60.0	Viêm cơ do nhiễm trùng	
M60.00	Viêm cơ do nhiễm trùng, nhiều vị trí	
M60.01	Viêm cơ do nhiễm trùng, vùng vai	
M60.02	Viêm cơ do nhiễm trùng, cánh tay trên	
M60.03	Viêm cơ do nhiễm trùng, cẳng tay	
M60.04	Viêm cơ do nhiễm trùng, bàn tay	
M60.05	Viêm cơ do nhiễm trùng, vùng chậu và/hoặc đùi	
M60.06	Viêm cơ do nhiễm trùng, cẳng chân	
M60.07	Viêm cơ do nhiễm trùng, cổ chân và/hoặc bàn chân	
M60.08	Viêm cơ do nhiễm trùng, vị trí khác	
M60.09	Viêm cơ do nhiễm trùng, vị trí không xác định	
M60.1	Viêm tổ chức kẽ của cơ	
M60.10	Viêm tổ chức kẽ của cơ, nhiều vị trí	
M60.11	Viêm tổ chức kẽ của cơ, vùng vai	
M60.12	Viêm tổ chức kẽ của cơ, cánh tay trên	
M60.13	Viêm tổ chức kẽ của cơ, cẳng tay	
M60.14	Viêm tổ chức kẽ của cơ, bàn tay	
M60.15	Viêm tổ chức kẽ của cơ, vùng chậu và/hoặc đùi	
M60.16	Viêm tổ chức kẽ của cơ, cẳng chân	
M60.17	Viêm tổ chức kẽ của cơ, cổ chân và/hoặc bàn chân	
M60.18	Viêm tổ chức kẽ của cơ, vị trí khác	
M60.19	Viêm tổ chức kẽ của cơ, vị trí không xác định	
M60.2	U hạt mô mềm do dị vật, không phân loại mục khác	
M60.20	U hạt mô mềm do dị vật, không phân loại mục khác, nhiều vị trí	
M60.21	U hạt mô mềm do dị vật, không phân loại mục khác, vùng vai	
M60.22	U hạt mô mềm do dị vật, không phân loại mục khác, cánh tay trên	
M60.23	U hạt mô mềm do dị vật, không phân loại mục khác, cẳng tay	
M60.24	U hạt mô mềm do dị vật, không phân loại mục khác, bàn tay	
M60.25	U hạt mô mềm do dị vật, không phân loại mục khác, vùng chậu và/hoặc đùi	
M60.26	U hạt mô mềm do dị vật, không phân loại mục khác, cẳng chân	
M60.27	U hạt mô mềm do dị vật, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M60.28	U hạt mô mềm do dị vật, không phân loại mục khác, vị trí khác	
M60.29	U hạt mô mềm do dị vật, không phân loại mục khác, vị trí không xác định	
M60.8	Viêm cơ khác	
M60.80	Viêm cơ khác, nhiều vị trí	
M60.81	Viêm cơ khác, vùng vai	
M60.82	Viêm cơ khác, cánh tay trên	
M60.83	Viêm cơ khác, cẳng tay	
M60.84	Viêm cơ khác, bàn tay	
M60.85	Viêm cơ khác, vùng chậu và/hoặc đùi	
M60.86	Viêm cơ khác, cẳng chân	
M60.87	Viêm cơ khác, cổ chân và/hoặc bàn chân	
M60.88	Viêm cơ khác, vị trí khác	
M60.89	Viêm cơ khác, vị trí không xác định	
M60.9	Viêm cơ, không xác định	
M60.90	Viêm cơ, không xác định, nhiều vị trí	
M60.91	Viêm cơ, không xác định, vùng vai	
M60.92	Viêm cơ, không xác định, cánh tay trên	
M60.93	Viêm cơ, không xác định, cẳng tay	
M60.94	Viêm cơ, không xác định, bàn tay	
M60.95	Viêm cơ, không xác định, vùng chậu và/hoặc đùi	
M60.96	Viêm cơ, không xác định, cẳng chân	
M60.97	Viêm cơ, không xác định, cổ chân và/hoặc bàn chân	
M60.98	Viêm cơ, không xác định, vị trí khác	
M60.99	Viêm cơ, không xác định, vị trí không xác định	
M61	Vôi hóa và/hoặc cốt hóa cơ	
M61.0	Viêm cơ cốt hóa do chấn thương	
M61.00	Viêm cơ cốt hóa do chấn thương, nhiều vị trí	
M61.01	Viêm cơ cốt hóa do chấn thương, vùng vai	
M61.02	Viêm cơ cốt hóa do chấn thương, cánh tay trên	
M61.03	Viêm cơ cốt hóa do chấn thương, cẳng tay	
M61.04	Viêm cơ cốt hóa do chấn thương, bàn tay	
M61.05	Viêm cơ cốt hóa do chấn thương, vùng chậu và/hoặc đùi	
M61.06	Viêm cơ cốt hóa do chấn thương, cẳng chân	
M61.07	Viêm cơ cốt hóa do chấn thương, cổ chân và/hoặc bàn chân	
M61.08	Viêm cơ cốt hóa do chấn thương, vị trí khác	
M61.09	Viêm cơ cốt hóa do chấn thương, vị trí không xác định	
M61.1	Viêm cơ cốt hóa tiến triển	
M61.10	Viêm cơ cốt hóa tiến triển, nhiều vị trí	
M61.11	Viêm cơ cốt hóa tiến triển, vùng vai	
M61.12	Viêm cơ cốt hóa tiến triển, cánh tay trên	
M61.13	Viêm cơ cốt hóa tiến triển, cẳng tay	
M61.14	Viêm cơ cốt hóa tiến triển, bàn tay	
M61.15	Viêm cơ cốt hóa tiến triển, vùng chậu và/hoặc đùi	
M61.16	Viêm cơ cốt hóa tiến triển, cẳng chân	
M61.17	Viêm cơ cốt hóa tiến triển, cổ chân và/hoặc bàn chân	
M61.18	Viêm cơ cốt hóa tiến triển, vị trí khác	
M61.19	Viêm cơ cốt hóa tiến triển, vị trí không xác định	
M61.2	Vôi hóa và/hoặc cốt hóa cơ do liệt	Viêm cơ cốt hóa liên quan đến liệt tứ chi hoặc liệt nửa người [dưới thắt lưng]
M61.20	Vôi hóa và/hoặc cốt hóa cơ do liệt, nhiều vị trí	
M61.21	Vôi hóa và/hoặc cốt hóa cơ do liệt, vùng vai	
M61.22	Vôi hóa và/hoặc cốt hóa cơ do liệt, cánh tay trên	
M61.23	Vôi hóa và/hoặc cốt hóa cơ do liệt, cẳng tay	
M61.24	Vôi hóa và/hoặc cốt hóa cơ do liệt, bàn tay	
M61.25	Vôi hóa và/hoặc cốt hóa cơ do liệt, vùng chậu và/hoặc đùi	
M61.26	Vôi hóa và/hoặc cốt hóa cơ do liệt, cẳng chân	
M61.27	Vôi hóa và/hoặc cốt hóa cơ do liệt, cổ chân và/hoặc bàn chân	
M61.28	Vôi hóa và/hoặc cốt hóa cơ do liệt, vị trí khác	
M61.29	Vôi hóa và/hoặc cốt hóa cơ do liệt, vị trí không xác định	
M61.3	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng	Viêm cơ cốt hóa liên quan đến bỏng
M61.30	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, nhiều vị trí	
M61.31	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, vùng vai	
M61.32	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, cánh tay trên	
M61.33	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, cẳng tay	
M61.34	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, bàn tay	
M61.35	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, vùng chậu và/hoặc đùi	
M61.36	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, cẳng chân	
M61.37	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, cổ chân và/hoặc bàn chân	
M61.38	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, vị trí khác	
M61.39	Vôi hóa và/hoặc cốt hóa cơ liên quan đến bỏng, vị trí không xác định	
M61.4	Vôi hóa khác của cơ	
M61.40	Vôi hóa khác của cơ, nhiều vị trí	
M61.41	Vôi hóa khác của cơ, vùng vai	
M61.42	Vôi hóa khác của cơ, cánh tay trên	
M61.43	Vôi hóa khác của cơ, cẳng tay	
M61.44	Vôi hóa khác của cơ, bàn tay	
M61.45	Vôi hóa khác của cơ, vùng chậu và/hoặc đùi	
M61.46	Vôi hóa khác của cơ, cẳng chân	
M61.47	Vôi hóa khác của cơ, cổ chân và/hoặc bàn chân	
M61.48	Vôi hóa khác của cơ, vị trí khác	
M61.49	Vôi hóa khác của cơ, vị trí không xác định	
M61.5	Cốt hóa cơ khác	
M61.50	Cốt hóa cơ khác, nhiều vị trí	
M61.51	Cốt hóa cơ khác, vùng vai	
M61.52	Cốt hóa cơ khác, cánh tay trên	
M61.53	Cốt hóa cơ khác, cẳng tay	
M61.54	Cốt hóa khác của cơ, bàn tay	
M61.55	Cốt hóa khác của cơ, vùng chậu và/hoặc đùi	
M61.56	Cốt hóa khác của cơ, cẳng chân	
M61.57	Cốt hóa khác của cơ, cổ chân và/hoặc bàn chân	
M61.58	Cốt hóa khác của cơ, vị trí khác	
M61.59	Cốt hóa khác của cơ, vị trí không xác định	
M61.9	Vôi hóa và/hoặc cốt hóa cơ, không xác định	
M61.90	Vôi hóa và/hoặc cốt hóa cơ, không xác định, nhiều vị trí	
M61.91	Vôi hóa và/hoặc cốt hóa cơ, không xác định, vùng vai	
M61.92	Vôi hóa và/hoặc cốt hóa cơ, không xác định, cánh tay trên	
M61.93	Vôi hóa và/hoặc cốt hóa cơ, không xác định, cẳng tay	
M61.94	Vôi hóa và/hoặc cốt hóa cơ, không xác định, bàn tay	
M61.95	Vôi hóa và/hoặc cốt hóa cơ, không xác định, vùng chậu và/hoặc đùi	
M61.96	Vôi hóa và/hoặc cốt hóa cơ, không xác định, cẳng chân	
M61.97	Vôi hóa và/hoặc cốt hóa cơ, không xác định, cổ chân và/hoặc bàn chân	
M61.98	Vôi hóa và/hoặc cốt hóa cơ, không xác định, vị trí khác	
M61.99	Vôi hóa và/hoặc cốt hóa cơ, không xác định, vị trí không xác định	
M62	Rối loạn khác của cơ	
M62.0	Tách cơ bụng	
M62.00	Tách cơ bụng, nhiều vị trí	
M62.01	Tách cơ bụng, vùng vai	
M62.02	Tách cơ bụng, cánh tay trên	
M62.03	Tách cơ bụng, cẳng tay	
M62.04	Tách cơ bụng, bàn tay	
M62.05	Tách cơ bụng, vùng chậu và/hoặc đùi	
M62.06	Tách cơ bụng, cẳng chân	
M62.07	Tách cơ bụng, cổ chân và/hoặc bàn chân	
M62.08	Tách cơ bụng, vị trí khác	
M62.09	Tách cơ bụng, vị trí không xác định	
M62.1	Rách khác của cơ (không do chấn thương)	
M62.10	Rách khác của cơ (không do chấn thương), nhiều vị trí	
M62.11	Rách khác của cơ (không do chấn thương), vùng vai	
M62.12	Rách khác của cơ (không do chấn thương), cánh tay trên	
M62.13	Rách khác của cơ (không do chấn thương), cẳng tay	
M62.14	Rách khác của cơ (không do chấn thương), bàn tay	
M62.15	Rách khác của cơ (không do chấn thương), vùng chậu và/hoặc đùi	
M62.16	Rách khác của cơ (không do chấn thương), cẳng chân	
M62.17	Rách khác của cơ (không do chấn thương), cổ chân và/hoặc bàn chân	
M62.18	Rách khác của cơ (không do chấn thương), vị trí khác	
M62.19	Rách khác của cơ (không do chấn thương), vị trí không xác định	
M62.2	Nhồi máu cơ do thiếu máu cục bộ	
M62.20	Nhồi máu cơ do thiếu máu cục bộ, nhiều vị trí	
M62.21	Nhồi máu cơ do thiếu máu cục bộ, vùng vai	
M62.22	Nhồi máu cơ do thiếu máu cục bộ, cánh tay trên	
M62.23	Nhồi máu cơ do thiếu máu cục bộ, cẳng tay	
M62.24	Nhồi máu cơ do thiếu máu cục bộ, bàn tay	
M62.25	Nhồi máu cơ do thiếu máu cục bộ, vùng chậu và/hoặc đùi	
M62.26	Nhồi máu cơ do thiếu máu cục bộ, cẳng chân	
M62.27	Nhồi máu cơ do thiếu máu cục bộ, cổ chân và/hoặc bàn chân	
M62.28	Nhồi máu cơ do thiếu máu cục bộ, vị trí khác	
M62.29	Nhồi máu cơ do thiếu máu cục bộ, vị trí không xác định	
M62.3	Hội chứng bất động (liệt nửa người)	
M62.30	Hội chứng bất động (liệt nửa người), nhiều vị trí	
M62.31	Hội chứng bất động (liệt nửa người), vùng vai	
M62.32	Hội chứng bất động (liệt nửa người), cánh tay trên	
M62.33	Hội chứng bất động (liệt nửa người), cẳng tay	
M62.34	Hội chứng bất động (liệt nửa người), bàn tay	
M62.35	Hội chứng bất động (liệt nửa người), vùng chậu và/hoặc đùi	
M62.36	Hội chứng bất động (liệt nửa người), cẳng chân	
M62.37	Hội chứng bất động (liệt nửa người), cổ chân và/hoặc bàn chân	
M62.38	Hội chứng bất động (liệt nửa người), vị trí khác	
M62.39	Hội chứng bất động (liệt nửa người), vị trí không xác định	
M62.4	Co cứng cơ	
M62.40	Co cứng cơ, nhiều vị trí	
M62.41	Co cứng cơ, vùng vai	
M62.42	Co cứng cơ, cánh tay trên	
M62.43	Co cứng cơ, cẳng tay	
M62.44	Co cứng cơ, bàn tay	
M62.45	Co cứng cơ, vùng chậu và/hoặc đùi	
M62.46	Co cứng cơ, cẳng chân	
M62.47	Co cứng cơ, cổ chân và/hoặc bàn chân	
M62.48	Co cứng cơ, vị trí khác	
M62.49	Co cứng cơ, vị trí không xác định	
M62.5	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác	Teo do không vận động, không phân loại mục khác|Sarcopenia [mất cơ và chức năng hoạt động của cơ]
M62.50	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, nhiều vị trí	
M62.51	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, vùng vai	
M62.52	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, cánh tay trên	
M62.53	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, cẳng tay	
M62.54	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, bàn tay	
M62.55	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, vùng chậu và/hoặc đùi	
M62.56	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, cẳng chân	
M62.57	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M62.58	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, vị trí khác	
M62.59	Teo cơ và/hoặc suy mòn cơ, không phân loại mục khác, vị trí không xác định	
M62.6	Căng cơ	
M62.60	Căng cơ, nhiều vị trí	
M62.61	Căng cơ, vùng vai	
M62.62	Căng cơ, cánh tay trên	
M62.63	Căng cơ, cẳng tay	
M62.64	Căng cơ, bàn tay	
M62.65	Căng cơ, vùng chậu và/hoặc đùi	
M62.66	Căng cơ, cẳng chân	
M62.67	Căng cơ, cổ chân và/hoặc bàn chân	
M62.68	Căng cơ, vị trí khác	
M62.69	Căng cơ, vị trí không xác định	
M62.8	Rối loạn xác định khác của cơ	
M62.80	Rối loạn xác định khác của cơ, nhiều vị trí	
M62.81	Rối loạn xác định khác của cơ, vùng vai	
M62.82	Rối loạn xác định khác của cơ, cánh tay trên	
M62.83	Rối loạn xác định khác của cơ, cẳng tay	
M62.84	Rối loạn xác định khác của cơ, bàn tay	
M62.85	Rối loạn xác định khác của cơ, vùng chậu và/hoặc đùi	
M62.86	Rối loạn xác định khác của cơ, cẳng chân	
M62.87	Rối loạn xác định khác của cơ, cổ chân và/hoặc bàn chân	
M62.88	Rối loạn xác định khác của cơ, vị trí khác	
M62.89	Rối loạn xác định khác của cơ, vị trí không xác định	
M62.9	Rối loạn cơ, không xác định	
M62.90	Rối loạn cơ, không xác định, nhiều vị trí	
M62.91	Rối loạn cơ, không xác định, vùng vai	
M62.92	Rối loạn cơ, không xác định, cánh tay trên	
M62.93	Rối loạn cơ, không xác định, cẳng tay	
M62.94	Rối loạn cơ, không xác định, bàn tay	
M62.95	Rối loạn cơ, không xác định, vùng chậu và/hoặc đùi	
M62.96	Rối loạn cơ, không xác định, cẳng chân	
M62.97	Rối loạn cơ, không xác định, cổ chân và/hoặc bàn chân	
M62.98	Rối loạn cơ, không xác định, vị trí khác	
M62.99	Rối loạn cơ, không xác định, vị trí không xác định	
M63.*	Rối loạn cơ do bệnh phân loại mục khác	
M63.0*	Viêm cơ do bệnh nhiễm khuẩn phân loại mục khác	
M63.1*	Viêm cơ do nhiễm ký sinh trùng và/hoặc động vật đơn bào phân loại mục khác	
M63.2*	Viêm cơ do bệnh nhiễm trùng khác phân loại mục khác	
M63.3*	Viêm cơ do bệnh u hạt (D86.8†)	
M63.8*	Rối loạn cơ khác do bệnh phân loại mục khác	
M65	Viêm màng hoạt dịch và/hoặc viêm bao gân	
M65.0	Áp xe bao gân	
M65.00	Áp xe bao gân, nhiều vị trí	
M65.01	Áp xe bao gân, vùng vai	
M65.02	Áp xe bao gân, cánh tay trên	
M65.03	Áp xe bao gân, cẳng tay	
M65.04	Áp xe bao gân, bàn tay	
M65.05	Áp xe bao gân, vùng chậu và/hoặc đùi	
M65.06	Áp xe bao gân, cẳng chân	
M65.07	Áp xe bao gân, cổ chân và/hoặc bàn chân	
M65.08	Áp xe bao gân, vị trí khác	
M65.09	Áp xe bao gân, vị trí không xác định	
M65.1	Viêm màng hoạt dịch hoặc viêm bao gân khác do nhiễm trùng	
M65.10	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, nhiều vị trí	
M65.11	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, vùng vai	
M65.12	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, cánh tay trên	
M65.13	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, cẳng tay	
M65.14	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, bàn tay	
M65.15	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, vùng chậu và/hoặc đùi	
M65.16	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, cẳng chân	
M65.17	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, cổ chân và/hoặc bàn chân	
M65.18	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, vị trí khác	
M65.19	Viêm màng hoạt dịch hoặc viêm bao gân nhiễm trùng khác, vị trí không xác định	
M65.2	Viêm gân vôi hóa	
M65.20	Viêm gân vôi hóa, nhiều vị trí	
M65.22	Viêm gân vôi hóa, cánh tay trên	
M65.23	Viêm gân vôi hóa, cẳng tay	
M65.24	Viêm gân vôi hóa, bàn tay	
M65.25	Viêm gân vôi hóa, vùng chậu và/hoặc đùi	
M65.26	Viêm gân vôi hóa, cẳng chân	
M65.27	Viêm gân vôi hóa, cổ chân và/hoặc bàn chân	
M65.28	Viêm gân vôi hóa, vị trí khác	
M65.29	Viêm gân vôi hóa, vị trí không xác định	
M65.3	Ngón tay lò xo [cò súng]	Bệnh viêm gân gấp [xuất hiện hạt xơ]
M65.4	Viêm bao gân mỏm trâm quay cổ tay [de Quervain]	
M65.8	Viêm màng hoạt dịch và/hoặc viêm bao gân khác	Khớp háng dễ bị kích thích
M65.80	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, nhiều vị trí	
M65.81	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, vùng vai	
M65.82	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, cánh tay trên	
M65.83	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, cẳng tay	
M65.84	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, bàn tay	
M65.85	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, vùng chậu và/hoặc đùi	
M65.86	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, cẳng chân	
M65.87	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, cổ chân và/hoặc bàn chân	
M65.88	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, vị trí khác	
M65.89	Viêm màng hoạt dịch và/hoặc viêm bao gân khác, vị trí không xác định	
M65.9	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định	
M65.90	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, nhiều vị trí	
M65.91	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, vùng vai	
M65.92	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, cánh tay trên	
M65.93	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, cẳng tay	
M65.94	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, bàn tay	
M65.95	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, vùng chậu và/hoặc đùi	
M65.96	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, cẳng chân	
M65.97	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, cổ chân và/hoặc bàn chân	
M65.98	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, vị trí khác	
M65.99	Viêm màng hoạt dịch và/hoặc viêm bao gân, không xác định, vị trí không xác định	
M66	Rách (đứt) tự phát của màng hoạt dịch và/hoặc gân	
M66.0	Vỡ u nang hoạt dịch khoeo chân	
M66.1	Rách màng hoạt dịch	
M66.10	Rách màng hoạt dịch, nhiều vị trí	
M66.11	Rách màng hoạt dịch, vùng vai	
M66.12	Rách màng hoạt dịch, cánh tay trên	
M66.13	Rách màng hoạt dịch, cẳng tay	
M66.14	Rách màng hoạt dịch, bàn tay	
M66.15	Rách màng hoạt dịch, vùng chậu và/hoặc đùi	
M66.16	Rách màng hoạt dịch, cẳng chân	
M66.17	Rách màng hoạt dịch, cổ chân và/hoặc bàn chân	
M66.18	Rách màng hoạt dịch, vị trí khác	
M66.19	Rách màng hoạt dịch, vị trí không xác định	
M66.2	Rách gân duỗi tự phát	
M66.20	Rách gân duỗi tự phát, nhiều vị trí	
M66.21	Rách gân duỗi tự phát, vùng vai	
M66.22	Rách gân duỗi tự phát, cánh tay trên	
M66.23	Rách gân duỗi tự phát, cẳng tay	
M66.24	Rách gân duỗi tự phát, bàn tay	
M66.25	Rách gân duỗi tự phát, vùng chậu và/hoặc đùi	
M66.26	Rách gân duỗi tự phát, cẳng chân	
M66.27	Rách gân duỗi tự phát, cổ chân và/hoặc bàn chân	
M66.28	Rách gân duỗi tự phát, vị trí khác	
M66.29	Rách gân duỗi tự phát, vị trí không xác định	
M66.3	Rách gân cơ gấp tự phát	
M66.30	Rách gân cơ gấp tự phát, nhiều vị trí	
M66.31	Rách gân cơ gấp tự phát, vùng vai	
M66.32	Rách gân cơ gấp tự phát, cánh tay trên	
M66.33	Rách gân cơ gấp tự phát, cẳng tay	
M66.34	Rách gân cơ gấp tự phát, bàn tay	
M66.35	Rách gân cơ gấp tự phát, vùng chậu và/hoặc đùi	
M66.36	Rách gân cơ gấp tự phát, cẳng chân	
M66.37	Rách gân cơ gấp tự phát, cổ chân và/hoặc bàn chân	
M66.38	Rách gân cơ gấp tự phát, vị trí khác	
M66.39	Rách gân cơ gấp tự phát, vị trí không xác định	
M66.4	Rách tự phát gân khác	
M66.40	Rách tự phát gân khác, nhiều vị trí	
M66.41	Rách tự phát gân khác, vùng vai	
M66.42	Rách tự phát gân khác, cánh tay trên	
M66.43	Rách tự phát gân khác, cẳng tay	
M66.44	Rách tự phát gân khác, bàn tay	
M66.45	Rách tự phát gân khác, vùng chậu và/hoặc đùi	
M66.46	Rách tự phát gân khác, cẳng chân	
M66.47	Rách tự phát gân khác, cổ chân và/hoặc bàn chân	
M66.48	Rách tự phát gân khác, vị trí khác	
M66.49	Rách tự phát gân khác, vị trí không xác định	
M66.5	Rách tự phát gân không xác định	Rách tại điểm nối cơ - gân, không do tổn thương
M66.50	Rách tự phát gân không xác định, nhiều vị trí	
M66.51	Rách tự phát gân không xác định, vùng vai	
M66.52	Rách tự phát gân không xác định, cánh tay trên	
M66.53	Rách tự phát gân không xác định, cẳng tay	
M66.54	Rách tự phát gân không xác định, bàn tay	
M66.55	Rách tự phát gân không xác định, vùng chậu và/hoặc đùi	
M66.56	Rách tự phát gân không xác định, cẳng chân	
M66.57	Rách tự phát gân không xác định, cổ chân và/hoặc bàn chân	
M66.58	Rách tự phát gân không xác định, vị trí khác	
M66.59	Rách tự phát gân không xác định, vị trí không xác định	
M67	Rối loạn khác của màng hoạt dịch và/hoặc gân	
M67.0	Gân gót chân (Achille) ngắn (mắc phải)	
M67.1	Chứng co rút gân (bao gân) khác	
M67.2	Phì đại màng hoạt dịch, không phân loại mục khác	
M67.3	Viêm màng hoạt dịch thoáng qua	
M67.4	Nang hạch	
M67.8	Rối loạn của màng hoạt dịch và/hoặc gân xác định khác	
M67.9	Rối loạn màng hoạt dịch và/hoặc gân, không xác định	
M68.*	Rối loạn màng hoạt dịch và/hoặc gân do bệnh phân loại mục khác	
M68.0*	Viêm màng hoạt dịch và/hoặc viêm bao gân do bệnh nhiễm khuẩn được xếp loại ở mục khác	
M68.8*	Rối loạn màng hoạt dịch và/hoặc gân khác do bệnh phân loại mục khác	
M70	Rối loạn mô mềm liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép	
M70.0	Viêm màng hoạt dịch khô mạn tính ở bàn tay và/hoặc cổ tay	
M70.03	Viêm màng hoạt dịch khô mạn tính ở bàn tay và/hoặc cổ tay, cẳng tay	
M70.04	Viêm màng hoạt dịch khô mạn tính ở bàn tay và/hoặc cổ tay, bàn tay	
M70.09	Viêm màng hoạt dịch khô mạn tính ở bàn tay và/hoặc cổ tay, vị trí không xác đinh	
M70.1	Viêm bao hoạt dịch bàn tay	
M70.13	Viêm bao hoạt dịch bàn tay, cẳng tay	
M70.14	Viêm bao hoạt dịch bàn tay, bàn tay	
M70.19	Viêm bao hoạt dịch bàn tay, vị trí không xác định	
M70.2	Viêm bao hoạt dịch mỏm khuỷu	
M70.3	Viêm bao hoạt dịch khác ở khuỷu tay	
M70.4	Viêm bao hoạt dịch trước xương bánh chè	
M70.5	Viêm bao hoạt dịch khác ở khớp gối	
M70.6	Viêm bao hoạt dịch mấu chuyển	Viêm bao gân quanh mấu chuyển
M70.7	Viêm bao hoạt dịch khác ở khớp háng	Viêm bao hoạt dịch xương ụ ngồi
M70.8	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép	
M70.80	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, nhiều vị trí	
M70.81	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vùng vai	
M70.82	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cánh tay trên	
M70.83	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cẳng tay	
M70.84	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, bàn tay	
M70.85	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vùng chậu và/hoặc đùi	
M70.86	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cẳng chân	
M70.87	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cổ chân và/hoặc bàn chân	
M70.88	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vị trí khác	
M70.89	Rối loạn mô mềm khác liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vị trí không xác định	
M70.9	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép	
M70.90	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, nhiều vị trí	
M70.91	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vùng vai	
M70.92	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cánh tay trên	
M70.93	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cẳng tay	
M70.94	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, bàn tay	
M70.95	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vùng chậu và/hoặc đùi	
M70.96	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cẳng chân	
M70.97	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, cổ chân và/hoặc bàn chân	
M70.98	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vị trí khác	
M70.99	Rối loạn mô mềm không xác định liên quan đến vận động, vận động quá mức và/hoặc bị chèn ép, vị trí không xác định	
M71	Bệnh lý bao hoạt dịch khác	
M71.0	Áp xe bao hoạt dịch	
M71.00	Áp xe bao hoạt dịch, nhiều vị trí	
M71.01	Áp xe bao hoạt dịch, vùng vai	
M71.02	Áp xe bao hoạt dịch, cánh tay trên	
M71.03	Áp xe bao hoạt dịch, cẳng tay	
M71.04	Áp xe bao hoạt dịch, bàn tay	
M71.05	Áp xe bao hoạt dịch, vùng chậu và/hoặc đùi	
M71.06	Áp xe bao hoạt dịch, cẳng chân	
M71.07	Áp xe bao hoạt dịch, cổ chân và/hoặc bàn chân	
M71.08	Áp xe bao hoạt dịch, vị trí khác	
M71.09	Áp xe bao hoạt dịch, vị trí không xác định	
M71.1	Viêm bao hoạt dịch khác do nhiễm trùng	
M71.10	Viêm bao hoạt dịch khác do nhiễm trùng, nhiều vị trí	
M71.11	Viêm bao hoạt dịch khác do nhiễm trùng, vùng vai	
M71.12	Viêm bao hoạt dịch khác do nhiễm trùng, cánh tay trên	
M71.13	Viêm bao hoạt dịch khác do nhiễm trùng, cẳng tay	
M71.14	Viêm bao hoạt dịch khác do nhiễm trùng, bàn tay	
M71.15	Viêm bao hoạt dịch khác do nhiễm trùng, vùng chậu và/hoặc đùi	
M71.16	Viêm bao hoạt dịch khác do nhiễm trùng, cẳng chân	
M71.17	Viêm bao hoạt dịch khác do nhiễm trùng, cổ chân và/hoặc bàn chân	
M71.18	Viêm bao hoạt dịch khác do nhiễm trùng, vị trí khác	
M71.19	Viêm bao hoạt dịch khác do nhiễm trùng, vị trí không xác định	
M71.2	U nang màng hoạt dịch vùng khoeo [Baker]	
M71.3	U nang bao hoạt dịch khác	
M71.30	U nang bao hoạt dịch khác, nhiều vị trí	
M71.31	U nang bao hoạt dịch khác, vùng vai	
M71.32	U nang bao hoạt dịch khác, cánh tay trên	
M71.33	U nang bao hoạt dịch khác, cẳng tay	
M71.34	U nang bao hoạt dịch khác, bàn tay	
M71.35	U nang bao hoạt dịch khác, vùng chậu và/hoặc đùi	
M71.36	U nang bao hoạt dịch khác, cẳng chân	
M71.37	U nang bao hoạt dịch khác, cổ chân và/hoặc bàn chân	
M71.38	U nang bao hoạt dịch khác, vị trí khác	
M71.39	U nang bao hoạt dịch khác, vị trí không xác định	
M71.4	Lắng đọng calci bao hoạt dịch	
M71.40	Lắng đọng calci bao hoạt dịch, nhiều vị trí	
M71.42	Lắng đọng calci bao hoạt dịch, cánh tay trên	
M71.43	Lắng đọng calci bao hoạt dịch, cẳng tay	
M71.44	Lắng đọng calci bao hoạt dịch, bàn tay	
M71.45	Lắng đọng calci bao hoạt dịch, vùng chậu và/hoặc đùi	
M71.46	Lắng đọng calci bao hoạt dịch, cẳng chân	
M71.47	Lắng đọng calci bao hoạt dịch, cổ chân và/hoặc bàn chân	
M71.48	Lắng đọng calci bao hoạt dịch, vị trí khác	
M71.49	Lắng đọng calci bao hoạt dịch, vị trí không xác định	
M71.5	Viêm bao hoạt dịch khác, không phân loại mục khác	
M71.50	Viêm bao hoạt dịch khác, không phân loại mục khác, nhiều vị trí	
M71.52	Viêm bao hoạt dịch khác, không phân loại mục khác, cánh tay trên	
M71.53	Viêm bao hoạt dịch khác, không phân loại mục khác, cẳng tay	
M71.54	Viêm bao hoạt dịch khác, không phân loại mục khác, bàn tay	
M71.55	Viêm bao hoạt dịch khác, không phân loại mục khác, vùng chậu và/hoặc đùi	
M71.57	Viêm bao hoạt dịch khác, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M71.58	Viêm bao hoạt dịch khác, không phân loại mục khác, vị trí khác	
M71.59	Viêm bao hoạt dịch khác, không phân loại mục khác, vị trí không xác định	
M71.8	Bệnh lý bao hoạt dịch xác định khác	
M71.80	Bệnh lý bao hoạt dịch xác định khác, nhiều vị trí	
M71.81	Bệnh lý bao hoạt dịch xác định khác, vùng vai	
M71.82	Bệnh lý bao hoạt dịch xác định khác, cánh tay trên	
M71.83	Bệnh lý bao hoạt dịch xác định khác, cẳng tay	
M71.84	Bệnh lý bao hoạt dịch xác định khác, bàn tay	
M71.85	Bệnh lý bao hoạt dịch xác định khác, vùng chậu và/hoặc đùi	
M71.86	Bệnh lý bao hoạt dịch xác định khác, cẳng chân	
M71.87	Bệnh lý bao hoạt dịch xác định khác, cổ chân và/hoặc bàn chân	
M71.88	Bệnh lý bao hoạt dịch xác định khác, vị trí khác	
M71.89	Bệnh lý bao hoạt dịch xác định khác, vị trí không xác định	
M71.9	Bệnh lý bao hoạt dịch, không xác định	Viêm bao hoạt dịch không xác định khác
M71.90	Bệnh lý bao hoạt dịch, không xác định, nhiều vị trí	
M71.91	Bệnh lý bao hoạt dịch, không xác định, vùng vai	
M71.92	Bệnh lý bao hoạt dịch, không xác định, cánh tay trên	
M71.93	Bệnh lý bao hoạt dịch, không xác định, cẳng tay	
M71.94	Bệnh lý bao hoạt dịch, không xác định, bàn tay	
M71.95	Bệnh lý bao hoạt dịch, không xác định, vùng chậu và/hoặc đùi	
M71.96	Bệnh lý bao hoạt dịch, không xác định, cẳng chân	
M71.97	Bệnh lý bao hoạt dịch, không xác định, cổ chân và/hoặc bàn chân	
M71.98	Bệnh lý bao hoạt dịch, không xác định, vị trí khác	
M71.99	Bệnh lý bao hoạt dịch, không xác định, vị trí không xác định	
M72	Rối loạn nguyên bào sợi	
M72.0	U xơ màng cơ bàn tay [Dupuytren]	
M72.1	U xơ phía sau khớp ngón tay và/hoặc ngón chân [nốt Garrod]	
M72.14	U xơ phía sau khớp [nốt Garrod], ngón tay	
M72.17	U xơ phía sau khớp liên đốt [nốt Garrod], ngón chân	
M72.19	U xơ phía sau khớp liên đốt [nốt Garrod], vị trí không xác định	
M72.2	U xơ màng cơ bàn chân	Viêm can gan bàn chân
M72.4	Bệnh u xơ giả sarcoma	U xơ màng cơ
M72.40	Bệnh u xơ giả sarcoma, nhiều vị trí	
M72.41	Bệnh u xơ giả sarcoma, vùng vai	
M72.42	Bệnh u xơ giả sarcoma, cánh tay trên	
M72.43	Bệnh u xơ giả sarcoma, cẳng tay	
M72.44	Bệnh u xơ giả sarcoma, bàn tay	
M72.45	Bệnh u xơ giả sarcoma, vùng chậu và/hoặc đùi	
M72.46	Bệnh u xơ giả sarcoma, cẳng chân	
M72.47	Bệnh u xơ giả sarcoma, cổ chân và/hoặc bàn chân	
M72.48	Bệnh u xơ giả sarcoma, vị trí khác	
M72.49	Bệnh u xơ giả sarcoma, vị trí không xác định	
M72.6	Viêm hoại tử cân mạc	
M72.60	Viêm hoại tử cân mạc, nhiều vị trí	
M72.61	Viêm hoại tử cân mạc, vùng vai	
M72.62	Viêm hoại tử cân mạc, cánh tay trên	
M72.63	Viêm hoại tử cân mạc, cẳng tay	
M72.64	Viêm hoại tử cân mạc, bàn tay	
M72.65	Viêm hoại tử cân mạc, vùng chậu và/hoặc đùi	
M72.66	Viêm hoại tử cân mạc, cẳng chân	
M72.67	Viêm hoại tử cân mạc, cổ chân và/hoặc bàn chân	
M72.68	Viêm hoại tử cân mạc, vị trí khác	
M72.69	Viêm hoại tử cân mạc, vị trí không xác định	
M72.8	Rối loạn nguyên bào sợi khác	
M72.80	Rối loạn nguyên bào sợi khác, nhiều vị trí	
M72.81	Rối loạn nguyên bào sợi khác, vùng vai	
M72.82	Rối loạn nguyên bào sợi khác, cánh tay trên	
M72.83	Rối loạn nguyên bào sợi khác, cẳng tay	
M72.84	Rối loạn nguyên bào sợi khác, bàn tay	
M72.85	Rối loạn nguyên bào sợi khác, vùng chậu và/hoặc đùi	
M72.86	Rối loạn nguyên bào sợi khác, cẳng chân	
M72.87	Rối loạn nguyên bào sợi khác, cổ chân và/hoặc bàn chân	
M72.88	Rối loạn nguyên bào sợi khác, vị trí khác	
M72.89	Rối loạn nguyên bào sợi khác, vị trí không xác định	
M72.9	Rối loạn nguyên bào sợi, không xác định	Viêm cân mạc không xác định khác|U xơ không xác định khác
M72.90	Rối loạn nguyên bào sợi, không xác định, nhiều vị trí	
M72.91	Rối loạn nguyên bào sợi, không xác định, vùng vai	
M72.92	Rối loạn nguyên bào sợi, không xác định, cánh tay trên	
M72.93	Rối loạn nguyên bào sợi, không xác định, cẳng tay	
M72.94	Rối loạn nguyên bào sợi, không xác định, bàn tay	
M72.95	Rối loạn nguyên bào sợi, không xác định, vùng chậu và/hoặc đùi	
M72.96	Rối loạn nguyên bào sợi, không xác định, cẳng chân	
M72.97	Rối loạn nguyên bào sợi, không xác định, cổ chân và/hoặc bàn chân	
M72.98	Rối loạn nguyên bào sợi, không xác định, vị trí khác	
M72.99	Rối loạn nguyên bào sợi, không xác định, vị trí không xác định	
M73.*	Rối loạn mô mềm do bệnh phân loại mục khác	Rối loạn mô mềm trong bệnh phân loại mục khác
M73.0*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†)	
M73.00*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), nhiều vị trí	
M73.01*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), vùng vai	
M73.02*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), cánh tay trên	
M73.03*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), cẳng tay	
M73.04*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), bàn tay	
M73.05*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), vùng chậu và/hoặc đùi	
M73.06*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), cẳng chân	
M73.07*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), cổ chân và/hoặc bàn chân	
M73.08*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), vị trí khác	
M73.09*	Viêm bao hoạt dịch do lậu cầu khuẩn (A54.4†), vị trí không xác định	
M73.1*	Viêm bao hoạt dịch do giang mai (A52.7†)	
M73.10*	Viêm bao hoạt dịch do giang mai (A52.7†), nhiều vị trí	
M73.11*	Viêm bao hoạt dịch do giang mai (A52.7†), vùng vai	
M73.12*	Viêm bao hoạt dịch do giang mai (A52.7†), cánh tay trên	
M73.13*	Viêm bao hoạt dịch do giang mai (A52.7†), cẳng tay	
M73.14*	Viêm bao hoạt dịch do giang mai (A52.7†), bàn tay	
M73.15*	Viêm bao hoạt dịch do giang mai (A52.7†), vùng chậu và/hoặc đùi	
M73.16*	Viêm bao hoạt dịch do giang mai (A52.7†), cẳng chân	
M73.17*	Viêm bao hoạt dịch do giang mai (A52.7†), cổ chân và/hoặc bàn chân	
M73.18*	Viêm bao hoạt dịch do giang mai (A52.7†), vị trí khác	
M73.19*	Viêm bao hoạt dịch do giang mai (A52.7†), vị trí không xác định	
M73.8*	Rối loạn mô mềm khác do bệnh phân loại mục khác	
M73.80*	Rối loạn mô mềm khác do bệnh phân loại mục khác, nhiều vị trí	
M73.81*	Rối loạn mô mềm khác do bệnh phân loại mục khác, vùng vai	
M73.82*	Rối loạn mô mềm khác do bệnh phân loại mục khác, cánh tay trên	
M73.83*	Rối loạn mô mềm khác do bệnh phân loại mục khác, cẳng tay	
M73.84*	Rối loạn mô mềm khác do bệnh phân loại mục khác, bàn tay	
M73.85*	Rối loạn mô mềm khác do bệnh phân loại mục khác, vùng chậu và/hoặc đùi	
M73.86*	Rối loạn mô mềm khác do bệnh phân loại mục khác, cẳng chân	
M73.87*	Rối loạn mô mềm khác do bệnh phân loại mục khác, cổ chân và/hoặc bàn chân	
M73.88*	Rối loạn mô mềm khác do bệnh phân loại mục khác, vị trí khác	
M73.89*	Rối loạn mô mềm khác do bệnh phân loại mục khác, vị trí không xác định	
M75	Tổn thương vai	
M75.0	Viêm cứng khớp vai	Đông cứng khớp vai [viêm dính khớp vai]|Viêm quanh khớp vai
M75.1	Hội chứng chóp xoay	
M75.2	Viêm gân cơ nhị đầu	
M75.3	Viêm gân vôi hóa ở vai	Viêm bao hoạt dịch vôi hóa ở vai
M75.4	Hội chứng chèn ép vùng vai	
M75.5	Viêm túi thanh mạc ở vai	
M75.6	Rách sụn viền do thoái hóa khớp vai	
M75.8	Tổn thương khác ở vai	
M75.9	Tổn thương vai, không xác định	
M76	Bệnh lý điểm bám gân của chân, ngoại trừ bàn chân	
M76.0	Viêm gân vùng mông	
M76.1	Viêm gân cơ thắt lưng	
M76.2	Gai xương mào chậu	
M76.3	Hội chứng dải chậu chày	
M76.35	Hội chứng dải chậu chày, vùng chậu và/hoặc đùi	
M76.36	Hội chứng dải chậu chày, cẳng chân	
M76.39	Hội chứng dải chậu chày, vị trí không xác định	
M76.4	Viêm túi thanh mạc bên của xương chày [Pellegrini-Stieda]	
M76.5	Viêm gân bánh chè	
M76.6	Viêm gân Achille	Viêm bao hoạt dịch gót chân
M76.7	Viêm gân cơ mác	
M76.8	Bệnh lý điểm bám gân - dây chằng khác của chân, ngoại trừ bàn chân	Hội chứng xương chày trước|Viêm bao gân xương chày sau
M76.80	Bệnh lý điểm bám gân - dây chằng khác của chân, ngoại trừ bàn chân, nhiều vị trí	
M76.85	Bệnh lý điểm bám gân - dây chằng khác của chân, ngoại trừ bàn chân, vùng chậu và/hoặc đùi	
M76.86	Bệnh lý điểm bám gân - dây chằng khác của chân, ngoại trừ bàn chân, cẳng chân	
M76.89	Bệnh lý điểm bám gân - dây chằng khác của chân, ngoại trừ bàn chân, vị trí không xác định	
M76.9	Bệnh lý điểm bám gân - dây chằng của chân, không xác định	
M76.90	Bệnh lý điểm bám gân - dây chằng của chân, không xác định, nhiều vị trí	
M76.95	Bệnh lý điểm bám gân - dây chằng của chân, không xác định, vùng chậu và/hoặc đùi	
M76.96	Bệnh lý điểm bám gân - dây chằng của chân, không xác định, cẳng chân	
M76.97	Bệnh lý điểm bám gân - dây chằng của chân, không xác định, cổ chân và/hoặc bàn chân	
M76.99	Bệnh lý điểm bám gân - dây chằng của chân, không xác định, vị trí không xác định	
M77	Bệnh lý điểm bám gân - dây chằng khác	Bệnh lý điểm bám gân-dây chằng khác
M77.0	Viêm điểm bám gân lồi cầu trong xương cánh tay	
M77.1	Viêm điểm bám gân lồi cầu ngoài xương cánh tay	Hội chứng viêm lồi cầu ngoài xương cánh tay
M77.2	Viêm quanh khớp cổ tay	
M77.23	Viêm quanh khớp cổ tay, cẳng tay	
M77.24	Viêm quanh khớp cổ tay, bàn tay	
M77.29	Viêm quanh khớp cổ tay, vị trí không xác định	
M77.3	Gai xương gót chân	
M77.4	Đau đốt bàn ngón bàn chân	
M77.5	Bệnh lý điểm bám gân - dây chằng khác của bàn chân	
M77.8	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác	
M77.80	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, nhiều vị trí	
M77.81	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, vùng vai	
M77.82	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, cánh tay trên	
M77.83	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, cẳng tay	
M77.84	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, bàn tay	
M77.85	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, vùng chậu và/hoặc đùi	
M77.86	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, cẳng chân	
M77.87	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M77.88	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, vị trí khác	
M77.89	Bệnh lý điểm bám gân - dây chằng khác, không phân loại mục khác, vị trí không xác định	
M77.9	Bệnh lý điểm bám gân - dây chằng, không xác định	Gai xương không xác định khác|Viêm bao khớp không xác định khác|Viêm quanh khớp không xác định khác|Viêm gân không xác định khác
M77.90	Bệnh lý điểm bám gân - dây chằng, không xác định, nhiều vị trí	
M77.91	Bệnh lý điểm bám gân - dây chằng, không xác định, vùng vai	
M77.92	Bệnh lý điểm bám gân - dây chằng, không xác định, cánh tay trên	
M77.93	Bệnh lý điểm bám gân - dây chằng, không xác định, cẳng tay	
M77.94	Bệnh lý điểm bám gân - dây chằng, không xác định, bàn tay	
M77.95	Bệnh lý điểm bám gân - dây chằng, không xác định, vùng chậu và/hoặc đùi	
M77.96	Bệnh lý điểm bám gân - dây chằng, không xác định, cẳng chân	
M77.97	Bệnh lý điểm bám gân - dây chằng, không xác định, cổ chân và/hoặc bàn chân	
M77.98	Bệnh lý điểm bám gân - dây chằng, không xác định, vị trí khác	
M77.99	Bệnh lý điểm bám gân - dây chằng, không xác định, vị trí không xác định	
M79	Rối loạn khác của mô mềm, không phân loại mục khác	
M79.0	Bệnh thấp, không xác định	
M79.00	Bệnh thấp, không xác định, nhiều vị trí	
M79.01	Bệnh thấp, không xác định, vùng vai	
M79.02	Bệnh thấp, không xác định, cánh tay trên	
M79.03	Bệnh thấp, không xác định, cẳng tay	
M79.04	Bệnh thấp, không xác định, bàn tay	
M79.05	Bệnh thấp, không xác định, vùng chậu và/hoặc đùi	
M79.06	Bệnh thấp, không xác định, cẳng chân	
M79.07	Bệnh thấp, không xác định, cổ chân và/hoặc bàn chân	
M79.08	Bệnh thấp, không xác định, vị trí khác	
M79.09	Bệnh thấp, không xác định, vị trí không xác định	
M79.1	Đau cơ	
M79.10	Đau cơ, nhiều vị trí	
M79.11	Đau cơ, vùng vai	
M79.12	Đau cơ, cánh tay trên	
M79.13	Đau cơ, cẳng tay	
M79.14	Đau cơ, bàn tay	
M79.15	Đau cơ, vùng chậu và/hoặc đùi	
M79.16	Đau cơ, cẳng chân	
M79.17	Đau cơ, cổ chân và/hoặc bàn chân	
M79.18	Đau cơ, vị trí khác	
M79.19	Đau cơ, vị trí không xác định	
M79.2	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định	
M79.20	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, nhiều vị trí	
M79.21	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, vùng vai	
M79.22	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, cánh tay trên	
M79.23	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, cẳng tay	
M79.24	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, bàn tay	
M79.25	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, vùng chậu và/hoặc đùi	
M79.26	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, cẳng chân	
M79.27	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, cổ chân và/hoặc bàn chân	
M79.28	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, vị trí khác	
M79.29	Đau dây thần kinh và/hoặc viêm dây thần kinh, không xác định, vị trí không xác định	
M79.3	Viêm tế bào mỡ dưới da, không xác định	
M79.30	Viêm tế bào mỡ dưới da, không xác định, nhiều vị trí	
M79.31	Viêm tế bào mỡ dưới da, không xác định, vùng vai	
M79.32	Viêm tế bào mỡ dưới da, không xác định, cánh tay trên	
M79.33	Viêm tế bào mỡ dưới da, không xác định, cẳng tay	
M79.34	Viêm tế bào mỡ dưới da, không xác định, bàn tay	
M79.35	Viêm tế bào mỡ dưới da, không xác định, vùng chậu và/hoặc đùi	
M79.36	Viêm tế bào mỡ dưới da, không xác định, cẳng chân	
M79.37	Viêm tế bào mỡ dưới da, không xác định, cổ chân và/hoặc bàn chân	
M79.38	Viêm tế bào mỡ dưới da, không xác định, vị trí khác	
M79.39	Viêm tế bào mỡ dưới da, không xác định, vị trí không xác định	
M79.4	Phì đại khối mỡ (dưới xương bánh chè)	
M79.5	Dị vật còn lại trong mô mềm	
M79.50	Dị vật còn lại trong mô mềm, nhiều vị trí	
M79.51	Dị vật còn lại trong mô mềm, vùng vai	
M79.52	Dị vật còn lại trong mô mềm, cánh tay trên	
M79.53	Dị vật còn lại trong mô mềm, cẳng tay	
M79.54	Dị vật còn lại trong mô mềm, bàn tay	
M79.55	Dị vật còn lại trong mô mềm, vùng chậu và/hoặc đùi	
M79.56	Dị vật còn lại trong mô mềm, cẳng chân	
M79.57	Dị vật còn lại trong mô mềm, cổ chân và/hoặc bàn chân	
M79.58	Dị vật còn lại trong mô mềm, vị trí khác	
M79.59	Dị vật còn lại trong mô mềm, vị trí không xác định	
M79.6	Đau ở chi	
M79.60	Đau ở chi, nhiều vị trí	
M79.61	Đau ở chi, vùng vai	
M79.62	Đau ở chi, cánh tay trên	
M79.63	Đau ở chi, cẳng tay	
M79.64	Đau ở chi, bàn tay	
M79.65	Đau ở chi, vùng chậu và/hoặc đùi	
M79.66	Đau ở chi, cẳng chân	
M79.67	Đau ở chi, cổ chân và/hoặc bàn chân	
M79.68	Đau ở chi, vị trí khác	
M79.69	Đau ở chi, vị trí không xác định	
M79.7	Đau xơ cơ	Viêm xơ cơ|Viêm mô xơ|Viêm mô quanh cơ
M79.8	Rối loạn mô mềm xác định khác	
M79.80	Rối loạn mô mềm xác định khác, nhiều vị trí	
M79.81	Rối loạn mô mềm xác định khác, vùng vai	
M79.82	Rối loạn mô mềm xác định khác, cánh tay trên	
M79.83	Rối loạn mô mềm xác định khác, cẳng tay	
M79.84	Rối loạn mô mềm xác định khác, bàn tay	
M79.85	Rối loạn mô mềm xác định khác, vùng chậu và/hoặc đùi	
M79.86	Rối loạn mô mềm xác định khác, cẳng chân	
M79.87	Rối loạn mô mềm xác định khác, cổ chân và/hoặc bàn chân	
M79.88	Rối loạn mô mềm xác định khác, vị trí khác	
M79.89	Rối loạn mô mềm xác định khác, vị trí không xác định	
M79.9	Rối loạn mô mềm, không xác định	
M79.90	Rối loạn mô mềm, không xác định, nhiều vị trí	
M79.91	Rối loạn mô mềm, không xác định, vùng vai	
M79.92	Rối loạn mô mềm, không xác định, cánh tay trên	
M79.93	Rối loạn mô mềm, không xác định, cẳng tay	
M79.94	Rối loạn mô mềm, không xác định, bàn tay	
M79.95	Rối loạn mô mềm, không xác định, vùng chậu và/hoặc đùi	
M79.96	Rối loạn mô mềm, không xác định, cẳng chân	
M79.97	Rối loạn mô mềm, không xác định, cổ chân và/hoặc bàn chân	
M79.98	Rối loạn mô mềm, không xác định, vị trí khác	
M79.99	Rối loạn mô mềm, không xác định, vị trí không xác định	
M80	Loãng xương kèm gãy xương bệnh lý	
M80.0	Loãng xương sau mãn kinh kèm gãy xương bệnh lý	
M80.00	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, nhiều vị trí	
M80.01	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, vùng vai	
M80.02	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, cánh tay trên	
M80.03	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, cẳng tay	
M80.04	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, bàn tay	
M80.05	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.06	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, cẳng chân	
M80.07	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.08	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, vị trí khác	
M80.09	Loãng xương sau mãn kinh kèm gãy xương bệnh lý, vị trí không xác định	
M80.1	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý	
M80.10	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, nhiều vị trí	
M80.11	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, vùng vai	
M80.12	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, cánh tay trên	
M80.13	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, cẳng tay	
M80.14	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, bàn tay	
M80.15	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.16	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, cẳng chân	
M80.17	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.18	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, vị trí khác	
M80.19	Loãng xương sau cắt bỏ buồng trứng kèm gãy xương bệnh lý, vị trí không xác định	
M80.2	Loãng xương do không vận động kèm gãy xương bệnh lý	
M80.20	Loãng xương do không vận động kèm gãy xương bệnh lý, nhiều vị trí	
M80.21	Loãng xương do không vận động kèm gãy xương bệnh lý, vùng vai	
M80.22	Loãng xương do không vận động kèm gãy xương bệnh lý, cánh tay trên	
M80.23	Loãng xương do không vận động kèm gãy xương bệnh lý, cẳng tay	
M80.24	Loãng xương do không vận động kèm gãy xương bệnh lý, bàn tay	
M80.25	Loãng xương do không vận động kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.26	Loãng xương do không vận động kèm gãy xương bệnh lý, cẳng chân	
M80.27	Loãng xương do không vận động kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.28	Loãng xương do không vận động kèm gãy xương bệnh lý, vị trí khác	
M80.29	Loãng xương do không vận động kèm gãy xương bệnh lý, vị trí không xác định	
M80.3	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý	
M80.30	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, nhiều vị trí	
M80.31	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, vùng vai	
M80.32	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, cánh tay trên	
M80.33	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, cẳng tay	
M80.34	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, bàn tay	
M80.35	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.36	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, cẳng chân	
M80.37	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.38	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, vị trí khác	
M80.39	Loãng xương do suy giảm hấp thu sau phẫu thuật kèm gãy xương bệnh lý, vị trí không xác định	
M80.4	Loãng xương do dùng thuốc kèm gãy xương bệnh lý	
M80.40	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, nhiều vị trí	
M80.41	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, vùng vai	
M80.42	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, cánh tay trên	
M80.43	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, cẳng tay	
M80.44	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, bàn tay	
M80.45	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.46	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, cẳng chân	
M80.47	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.48	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, vị trí khác	
M80.49	Loãng xương do dùng thuốc kèm gãy xương bệnh lý, vị trí không xác định	
M80.5	Loãng xương vô căn kèm gãy xương bệnh lý	
M80.50	Loãng xương vô căn kèm gãy xương bệnh lý, nhiều vị trí	
M80.51	Loãng xương vô căn kèm gãy xương bệnh lý, vùng vai	
M80.52	Loãng xương vô căn kèm gãy xương bệnh lý, cánh tay trên	
M80.53	Loãng xương vô căn kèm gãy xương bệnh lý, cẳng tay	
M80.54	Loãng xương vô căn kèm gãy xương bệnh lý, bàn tay	
M80.55	Loãng xương vô căn kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.56	Loãng xương vô căn kèm gãy xương bệnh lý, cẳng chân	
M80.57	Loãng xương vô căn kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.58	Loãng xương vô căn kèm gãy xương bệnh lý, vị trí khác	
M80.59	Loãng xương vô căn kèm gãy xương bệnh lý, vị trí không xác định	
M80.8	Loãng xương khác kèm gãy xương bệnh lý	
M80.80	Loãng xương khác kèm gãy xương bệnh lý, nhiều vị trí	
M80.81	Loãng xương khác kèm gãy xương bệnh lý, vùng vai	
M80.82	Loãng xương khác kèm gãy xương bệnh lý, cánh tay trên	
M80.83	Loãng xương khác kèm gãy xương bệnh lý, cẳng tay	
M80.84	Loãng xương khác kèm gãy xương bệnh lý, bàn tay	
M80.85	Loãng xương khác kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.86	Loãng xương khác kèm gãy xương bệnh lý, cẳng chân	
M80.87	Loãng xương khác kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.88	Loãng xương khác kèm gãy xương bệnh lý, vị trí khác	
M80.89	Loãng xương khác kèm gãy xương bệnh lý, vị trí không xác định	
M80.9	Loãng xương không xác định kèm gãy xương bệnh lý	
M80.90	Loãng xương không xác định kèm gãy xương bệnh lý, nhiều vị trí	
M80.91	Loãng xương không xác định kèm gãy xương bệnh lý, vùng vai	
M80.92	Loãng xương không xác định kèm gãy xương bệnh lý, cánh tay trên	
M80.93	Loãng xương không xác định kèm gãy xương bệnh lý, cẳng tay	
M80.94	Loãng xương không xác định kèm gãy xương bệnh lý, bàn tay	
M80.95	Loãng xương không xác định kèm gãy xương bệnh lý, vùng chậu và/hoặc đùi	
M80.96	Loãng xương không xác định kèm gãy xương bệnh lý, cẳng chân	
M80.97	Loãng xương không xác định kèm gãy xương bệnh lý, cổ chân và/hoặc bàn chân	
M80.98	Loãng xương không xác định kèm gãy xương bệnh lý, vị trí khác	
M80.99	Loãng xương không xác định kèm gãy xương bệnh lý, vị trí không xác định	
M81	Loãng xương không kèm gãy xương bệnh lý	
M81.0	Loãng xương sau mãn kinh	
M81.00	Loãng xương sau mãn kinh, nhiều vị trí	
M81.01	Loãng xương sau mãn kinh, vùng vai	
M81.02	Loãng xương sau mãn kinh, cánh tay trên	
M81.03	Loãng xương sau mãn kinh, cẳng tay	
M81.04	Loãng xương sau mãn kinh, bàn tay	
M81.05	Loãng xương sau mãn kinh, vùng chậu và/hoặc đùi	
M81.06	Loãng xương sau mãn kinh, cẳng chân	
M81.07	Loãng xương sau mãn kinh, cổ chân và/hoặc bàn chân	
M81.08	Loãng xương sau mãn kinh, vị trí khác	
M81.09	Loãng xương sau mãn kinh, vị trí không xác định	
M81.1	Loãng xương sau cắt bỏ buồng trứng	
M81.10	Loãng xương sau cắt bỏ buồng trứng, nhiều vị trí	
M81.11	Loãng xương sau cắt bỏ buồng trứng, vùng vai	
M81.12	Loãng xương sau cắt bỏ buồng trứng, cánh tay trên	
M81.13	Loãng xương sau cắt bỏ buồng trứng, cẳng tay	
M81.14	Loãng xương sau cắt bỏ buồng trứng, bàn tay	
M81.15	Loãng xương sau cắt bỏ buồng trứng, vùng chậu và/hoặc đùi	
M81.16	Loãng xương sau cắt bỏ buồng trứng, cẳng chân	
M81.17	Loãng xương sau cắt bỏ buồng trứng, cổ chân và/hoặc bàn chân	
M81.18	Loãng xương sau cắt bỏ buồng trứng, vị trí khác	
M81.19	Loãng xương sau cắt bỏ buồng trứng, vị trí không xác định	
M81.2	Loãng xương do không vận động	
M81.20	Loãng xương do không vận động, nhiều vị trí	
M81.21	Loãng xương do không vận động, vùng vai	
M81.22	Loãng xương do không vận động, cánh tay trên	
M81.23	Loãng xương do không vận động, cẳng tay	
M81.24	Loãng xương do không vận động, bàn tay	
M81.25	Loãng xương do không vận động, vùng chậu và/hoặc đùi	
M81.26	Loãng xương do không vận động, cẳng chân	
M81.27	Loãng xương do không vận động, cổ chân và/hoặc bàn chân	
M81.28	Loãng xương do không vận động, vị trí khác	
M81.29	Loãng xương do không vận động, vị trí không xác định	
M81.3	Loãng xương do suy giảm hấp thu sau phẫu thuật	
M81.30	Loãng xương do suy giảm hấp thu sau phẫu thuật, nhiều vị trí	
M81.31	Loãng xương do suy giảm hấp thu sau phẫu thuật, vùng vai	
M81.32	Loãng xương do suy giảm hấp thu sau phẫu thuật, cánh tay trên	
M81.33	Loãng xương do suy giảm hấp thu sau phẫu thuật, cẳng tay	
M81.34	Loãng xương do suy giảm hấp thu sau phẫu thuật, bàn tay	
M81.35	Loãng xương do suy giảm hấp thu sau phẫu thuật, vùng chậu và/hoặc đùi	
M81.36	Loãng xương do suy giảm hấp thu sau phẫu thuật, cẳng chân	
M81.37	Loãng xương do suy giảm hấp thu sau phẫu thuật, cổ chân và/hoặc bàn chân	
M81.38	Loãng xương do suy giảm hấp thu sau phẫu thuật, vị trí khác	
M81.39	Loãng xương do suy giảm hấp thu sau phẫu thuật, vị trí không xác định	
M81.4	Loãng xương do dùng thuốc	
M81.40	Loãng xương do dùng thuốc, nhiều vị trí	
M81.41	Loãng xương do dùng thuốc, vùng vai	
M81.42	Loãng xương do dùng thuốc, cánh tay trên	
M81.43	Loãng xương do dùng thuốc, cẳng tay	
M81.44	Loãng xương do dùng thuốc, bàn tay	
M81.45	Loãng xương do dùng thuốc, vùng chậu và/hoặc đùi	
M81.46	Loãng xương do dùng thuốc, cẳng chân	
M81.47	Loãng xương do dùng thuốc, cổ chân và/hoặc bàn chân	
M81.48	Loãng xương do dùng thuốc, vị trí khác	
M81.49	Loãng xương do dùng thuốc, vị trí không xác định	
M81.5	Loãng xương vô căn	
M81.50	Loãng xương vô căn, nhiều vị trí	
M81.51	Loãng xương vô căn, vùng vai	
M81.52	Loãng xương vô căn, cánh tay trên	
M81.53	Loãng xương vô căn, cẳng tay	
M81.54	Loãng xương vô căn, bàn tay	
M81.55	Loãng xương vô căn, vùng chậu và/hoặc đùi	
M81.56	Loãng xương vô căn, cẳng chân	
M81.57	Loãng xương vô căn, cổ chân và/hoặc bàn chân	
M81.58	Loãng xương vô căn, vị trí khác	
M81.59	Loãng xương vô căn, vị trí không xác định	
M81.6	Loãng xương khu trú [Lequesne]	
M81.60	Loãng xương khu trú [Lequesne], nhiều vị trí	
M81.61	Loãng xương khu trú [Lequesne], vùng vai	
M81.62	Loãng xương khu trú [Lequesne], cánh tay trên	
M81.63	Loãng xương khu trú [Lequesne], cẳng tay	
M81.64	Loãng xương khu trú [Lequesne], bàn tay	
M81.65	Loãng xương khu trú [Lequesne], vùng chậu và/hoặc đùi	
M81.66	Loãng xương khu trú [Lequesne], cẳng chân	
M81.67	Loãng xương khu trú [Lequesne], cổ chân và/hoặc bàn chân	
M81.68	Loãng xương khu trú [Lequesne], vị trí khác	
M81.69	Loãng xương khu trú [Lequesne], vị trí không xác định	
M81.8	Bệnh loãng xương khác	Loãng xương ở tuổi già
M81.80	Bệnh loãng xương khác, nhiều vị trí	
M81.81	Bệnh loãng xương khác, vùng vai	
M81.82	Bệnh loãng xương khác, cánh tay trên	
M81.83	Bệnh loãng xương khác, cẳng tay	
M81.84	Bệnh loãng xương khác, bàn tay	
M81.85	Bệnh loãng xương khác, vùng chậu và/hoặc đùi	
M81.86	Bệnh loãng xương khác, cẳng chân	
M81.87	Bệnh loãng xương khác, cổ chân và/hoặc bàn chân	
M81.88	Bệnh loãng xương khác, vị trí khác	
M81.89	Bệnh loãng xương khác, vị trí không xác định	
M81.9	Loãng xương, không xác định	
M81.90	Loãng xương, không xác định, nhiều vị trí	
M81.91	Loãng xương, không xác định, vùng vai	
M81.92	Loãng xương, không xác định, cánh tay trên	
M81.93	Loãng xương, không xác định, cẳng tay	
M81.94	Loãng xương, không xác định, bàn tay	
M81.95	Loãng xương, không xác định, vùng chậu và/hoặc đùi	
M81.96	Loãng xương, không xác định, cẳng chân	
M81.97	Loãng xương, không xác định, cổ chân và/hoặc bàn chân	
M81.98	Loãng xương, không xác định, vị trí khác	
M81.99	Loãng xương, không xác định, vị trí không xác định	
M82.*	Loãng xương do bệnh phân loại mục khác	
M82.0*	Loãng xương do bệnh đa u tủy xương (C90.0†)	
M82.00*	Loãng xương do bệnh đa u tủy xương (C90.0†), nhiều vị trí	
M82.01*	Loãng xương do bệnh đa u tủy xương (C90.0†), vùng vai	
M82.02*	Loãng xương do bệnh đa u tủy xương (C90.0†), cánh tay trên	
M82.03*	Loãng xương do bệnh đa u tủy xương (C90.0†), cẳng tay	
M82.04*	Loãng xương do bệnh đa u tủy xương (C90.0†), bàn tay	
M82.05*	Loãng xương do bệnh đa u tủy xương (C90.0†), vùng chậu và/hoặc đùi	
M82.06*	Loãng xương do bệnh đa u tủy xương (C90.0†), cẳng chân	
M82.07*	Loãng xương do bệnh đa u tủy xương (C90.0†), cổ chân và/hoặc bàn chân	
M82.08*	Loãng xương do bệnh đa u tủy xương (C90.0†), vị trí khác	
M82.09*	Loãng xương do bệnh đa u tủy xương (C90.0†), vị trí không xác định	
M82.1*	Loãng xương do rối loạn nội tiết (E00-E34†)	
M82.10*	Loãng xương do rối loạn nội tiết (E00-E34†), nhiều vị trí	
M82.11*	Loãng xương do rối loạn nội tiết (E00-E34†), vùng vai	
M82.12*	Loãng xương do rối loạn nội tiết (E00-E34†), cánh tay trên	
M82.13*	Loãng xương do rối loạn nội tiết (E00-E34†), cẳng tay	
M82.14*	Loãng xương do rối loạn nội tiết (E00-E34†), bàn tay	
M82.15*	Loãng xương do rối loạn nội tiết (E00-E34†), vùng chậu và/hoặc đùi	
M82.16*	Loãng xương do rối loạn nội tiết (E00-E34†), cẳng chân	
M82.17*	Loãng xương do rối loạn nội tiết (E00-E34†), cổ chân và/hoặc bàn chân	
M82.18*	Loãng xương do rối loạn nội tiết (E00-E34†), vị trí khác	
M82.19*	Loãng xương do rối loạn nội tiết (E00-E34†), vị trí không xác định	
M82.8*	Loãng xương do bệnh khác phân loại mục khác	
M82.80*	Loãng xương do bệnh khác phân loại mục khác, nhiều vị trí	
M82.81*	Loãng xương do bệnh khác phân loại mục khác, vùng vai	
M82.82*	Loãng xương do bệnh khác phân loại mục khác, cánh tay trên	
M82.83*	Loãng xương do bệnh khác phân loại mục khác, cẳng tay	
M82.84*	Loãng xương do bệnh khác phân loại mục khác, bàn tay	
M82.85*	Loãng xương do bệnh khác phân loại mục khác, vùng chậu và/hoặc đùi	
M82.86*	Loãng xương do bệnh khác phân loại mục khác, cẳng chân	
M82.87*	Loãng xương do bệnh khác phân loại mục khác, cổ chân và/hoặc bàn chân	
M82.88*	Loãng xương do bệnh khác phân loại mục khác, vị trí khác	
M82.89*	Loãng xương do bệnh khác phân loại mục khác, vị trí không xác định	
M83	Bệnh nhuyễn xương ở người lớn	
M83.0	Bệnh nhuyễn xương thời kỳ sau đẻ	
M83.00	Bệnh nhuyễn xương thời kỳ sau đẻ, nhiều vị trí	
M83.01	Bệnh nhuyễn xương thời kỳ sau đẻ, vùng vai	
M83.02	Bệnh nhuyễn xương thời kỳ sau đẻ, cánh tay trên	
M83.03	Bệnh nhuyễn xương thời kỳ sau đẻ, cẳng tay	
M83.04	Bệnh nhuyễn xương thời kỳ sau đẻ, bàn tay	
M83.05	Bệnh nhuyễn xương thời kỳ sau đẻ, vùng chậu và/hoặc đùi	
M83.06	Bệnh nhuyễn xương thời kỳ sau đẻ, cẳng chân	
M83.07	Bệnh nhuyễn xương thời kỳ sau đẻ, cổ chân và/hoặc bàn chân	
M83.08	Bệnh nhuyễn xương thời kỳ sau đẻ, vị trí khác	
M83.09	Bệnh nhuyễn xương thời kỳ sau đẻ, vị trí không xác định	
M83.1	Bệnh nhuyễn xương ở người già	
M83.10	Bệnh nhuyễn xương ở người già, nhiều vị trí	
M83.11	Bệnh nhuyễn xương ở người già, vùng vai	
M83.12	Bệnh nhuyễn xương ở người già, cánh tay trên	
M83.13	Bệnh nhuyễn xương ở người già, cẳng tay	
M83.14	Bệnh nhuyễn xương ở người già, bàn tay	
M83.15	Bệnh nhuyễn xương ở người già, vùng chậu và/hoặc đùi	
M83.16	Bệnh nhuyễn xương ở người già, cẳng chân	
M83.17	Bệnh nhuyễn xương ở người già, cổ chân và/hoặc bàn chân	
M83.18	Bệnh nhuyễn xương ở người già, vị trí khác	
M83.19	Bệnh nhuyễn xương ở người già, vị trí không xác định	
M83.2	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu	Bệnh nhuyễn xương do suy giảm hấp thu sau can thiệp ở người lớn
M83.20	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, nhiều vị trí	
M83.21	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, vùng vai	
M83.22	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, cánh tay trên	
M83.23	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, cẳng tay	
M83.24	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, bàn tay	
M83.25	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, vùng chậu và/hoặc đùi	
M83.26	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, cẳng chân	
M83.27	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, cổ chân và/hoặc bàn chân	
M83.28	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, vị trí khác	
M83.29	Bệnh nhuyễn xương ở người lớn do suy giảm hấp thu, vị trí không xác định	
M83.3	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng	
M83.30	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, nhiều vị trí	
M83.31	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, vùng vai	
M83.32	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, cánh tay trên	
M83.33	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, cẳng tay	
M83.34	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, bàn tay	
M83.35	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, vùng chậu và/hoặc đùi	
M83.36	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, cẳng chân	
M83.37	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, cổ chân và/hoặc bàn chân	
M83.38	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, vị trí khác	
M83.39	Bệnh nhuyễn xương ở người lớn do suy dinh dưỡng, vị trí không xác định	
M83.4	Bệnh xương nhiễm nhôm	
M83.40	Bệnh xương nhiễm nhôm, nhiều vị trí	
M83.41	Bệnh xương nhiễm nhôm, vùng vai	
M83.42	Bệnh xương nhiễm nhôm, cánh tay trên	
M83.43	Bệnh xương nhiễm nhôm, cẳng tay	
M83.44	Bệnh xương nhiễm nhôm, bàn tay	
M83.45	Bệnh xương nhiễm nhôm, vùng chậu và/hoặc đùi	
M83.46	Bệnh xương nhiễm nhôm, cẳng chân	
M83.47	Bệnh xương nhiễm nhôm, cổ chân và/hoặc bàn chân	
M83.48	Bệnh xương nhiễm nhôm, vị trí khác	
M83.49	Bệnh xương nhiễm nhôm, vị trí không xác định	
M83.5	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc	
M83.50	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, nhiều vị trí	
M83.51	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, vùng vai	
M83.52	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, cánh tay trên	
M83.53	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, cẳng tay	
M83.54	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, bàn tay	
M83.55	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, vùng chậu và/hoặc đùi	
M83.56	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, cẳng chân	
M83.57	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, cổ chân và/hoặc bàn chân	
M83.58	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, vị trí khác	
M83.59	Bệnh nhuyễn xương khác ở người lớn do dùng thuốc, vị trí không xác định	
M83.8	Bệnh nhuyễn xương khác ở người lớn	
M83.80	Bệnh nhuyễn xương khác ở người lớn, nhiều vị trí	
M83.81	Bệnh nhuyễn xương khác ở người lớn, vùng vai	
M83.82	Bệnh nhuyễn xương khác ở người lớn, cánh tay trên	
M83.83	Bệnh nhuyễn xương khác ở người lớn, cẳng tay	
M83.84	Bệnh nhuyễn xương khác ở người lớn, bàn tay	
M83.85	Bệnh nhuyễn xương khác ở người lớn, vùng chậu và/hoặc đùi	
M83.86	Bệnh nhuyễn xương khác ở người lớn, cẳng chân	
M83.87	Bệnh nhuyễn xương khác ở người lớn, cổ chân và/hoặc bàn chân	
M83.88	Bệnh nhuyễn xương khác ở người lớn, vị trí khác	
M83.89	Bệnh nhuyễn xương khác ở người lớn, vị trí không xác định	
M83.9	Bệnh nhuyễn xương ở người lớn, không xác định	
M83.90	Bệnh nhuyễn xương ở người lớn, không xác định, nhiều vị trí	
M83.91	Bệnh nhuyễn xương ở người lớn, không xác định, vùng vai	
M83.92	Bệnh nhuyễn xương ở người lớn, không xác định, cánh tay trên	
M83.93	Bệnh nhuyễn xương ở người lớn, không xác định, cẳng tay	
M83.94	Bệnh nhuyễn xương ở người lớn, không xác định, bàn tay	
M83.95	Bệnh nhuyễn xương ở người lớn, không xác định, vùng chậu và/hoặc đùi	
M83.96	Bệnh nhuyễn xương ở người lớn, không xác định, cẳng chân	
M83.97	Bệnh nhuyễn xương ở người lớn, không xác định, cổ chân và/hoặc bàn chân	
M83.98	Bệnh nhuyễn xương ở người lớn, không xác định, vị trí khác	
M83.99	Bệnh nhuyễn xương ở người lớn, không xác định, vị trí không xác định	
M84	Rối loạn về tính liên tục của xương	
M84.0	Gãy xương di lệch	
M84.00	Gãy xương di lệch, nhiều vị trí	
M84.01	Gãy xương di lệch, vùng vai	
M84.02	Gãy xương di lệch, cánh tay trên	
M84.03	Gãy xương di lệch, cẳng tay	
M84.04	Gãy xương di lệch, bàn tay	
M84.05	Gãy xương di lệch, vùng chậu và/hoặc đùi	
M84.06	Gãy xương di lệch, cẳng chân	
M84.07	Gãy xương di lệch, cổ chân và/hoặc bàn chân	
M84.08	Gãy xương di lệch, vị trí khác	
M84.09	Gãy xương di lệch, vị trí không xác định	
M84.1	Gãy xương không liền [khớp giả]	
M84.10	Gãy xương không liền [khớp giả], nhiều vị trí	
M84.11	Gãy xương không liền [khớp giả], vùng vai	
M84.12	Gãy xương không liền [khớp giả], cánh tay trên	
M84.13	Gãy xương không liền [khớp giả], cẳng tay	
M84.14	Gãy xương không liền [khớp giả], bàn tay	
M84.15	Gãy xương không liền [khớp giả], vùng chậu và/hoặc đùi	
M84.16	Gãy xương không liền [khớp giả], cẳng chân	
M84.17	Gãy xương không liền [khớp giả], cổ chân và/hoặc bàn chân	
M84.18	Gãy xương không liền [khớp giả], vị trí khác	
M84.19	Gãy xương không liền [khớp giả], vị trí không xác định	
M84.2	Gãy xương chậm liền	
M84.20	Gãy xương chậm liền, nhiều vị trí	
M84.21	Gãy xương chậm liền, vùng vai	
M84.22	Gãy xương chậm liền, cánh tay trên	
M84.23	Gãy xương chậm liền, cẳng tay	
M84.24	Gãy xương chậm liền, bàn tay	
M84.25	Gãy xương chậm liền, vùng chậu và/hoặc đùi	
M84.26	Gãy xương chậm liền, cẳng chân	
M84.27	Gãy xương chậm liền, cổ chân và/hoặc bàn chân	
M84.28	Gãy xương chậm liền, vị trí khác	
M84.29	Gãy xương chậm liền, vị trí không xác định	
M84.3	Gãy xương do gắng sức, không phân loại mục khác	
M84.30	Gãy xương do gắng sức, không phân loại mục khác, nhiều vị trí	
M84.31	Gãy xương do gắng sức, không phân loại mục khác, vùng vai	
M84.32	Gãy xương do gắng sức, không phân loại mục khác, cánh tay trên	
M84.33	Gãy xương do gắng sức, không phân loại mục khác, cẳng tay	
M84.34	Gãy xương do gắng sức, không phân loại mục khác, bàn tay	
M84.35	Gãy xương do gắng sức, không phân loại mục khác, vùng chậu và/hoặc đùi	
M84.36	Gãy xương do gắng sức, không phân loại mục khác, cẳng chân	
M84.37	Gãy xương do gắng sức, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M84.38	Gãy xương do gắng sức, không phân loại mục khác, vị trí khác	
M84.39	Gãy xương do gắng sức, không phân loại mục khác, vị trí không xác định	
M84.4	Gãy xương bệnh lý, không phân loại mục khác	
M84.40	Gãy xương bệnh lý, không phân loại mục khác, nhiều vị trí	
M84.41	Gãy xương bệnh lý, không phân loại mục khác, vùng vai	
M84.42	Gãy xương bệnh lý, không phân loại mục khác, cánh tay trên	
M84.43	Gãy xương bệnh lý, không phân loại mục khác, cẳng tay	
M84.44	Gãy xương bệnh lý, không phân loại mục khác, bàn tay	
M84.45	Gãy xương bệnh lý, không phân loại mục khác, vùng chậu và/hoặc đùi	
M84.46	Gãy xương bệnh lý, không phân loại mục khác, cẳng chân	
M84.47	Gãy xương bệnh lý, không phân loại mục khác, cổ chân và/hoặc bàn chân	
M84.48	Gãy xương bệnh lý, không phân loại mục khác, vị trí khác	
M84.49	Gãy xương bệnh lý, không phân loại mục khác, vị trí không xác định	
M84.8	Rối loạn khác về tính liên tục của xương	
M84.80	Rối loạn khác về tính liên tục của xương, nhiều vị trí	
M84.81	Rối loạn khác về tính liên tục của xương, vùng vai	
M84.82	Rối loạn khác về tính liên tục của xương, cánh tay trên	
M84.83	Rối loạn khác về tính liên tục của xương, cẳng tay	
M84.84	Rối loạn khác về tính liên tục của xương, bàn tay	
M84.85	Rối loạn khác về tính liên tục của xương, vùng chậu và/hoặc đùi	
M84.86	Rối loạn khác về tính liên tục của xương, cẳng chân	
M84.87	Rối loạn khác về tính liên tục của xương, cổ chân và/hoặc bàn chân	
M84.88	Rối loạn khác về tính liên tục của xương, vị trí khác	
M84.89	Rối loạn khác về tính liên tục của xương, vị trí không xác định	
M84.9	Rối loạn về tính liên tục của xương, không xác định	
M84.90	Rối loạn về tính liên tục của xương, không xác định, nhiều vị trí	
M84.91	Rối loạn về tính liên tục của xương, không xác định, vùng vai	
M84.92	Rối loạn về tính liên tục của xương, không xác định, cánh tay trên	
M84.93	Rối loạn về tính liên tục của xương, không xác định, cẳng tay	
M84.94	Rối loạn về tính liên tục của xương, không xác định, bàn tay	
M84.95	Rối loạn về tính liên tục của xương, không xác định, vùng chậu và/hoặc đùi	
M84.96	Rối loạn về tính liên tục của xương, không xác định, cẳng chân	
M84.97	Rối loạn về tính liên tục của xương, không xác định, cổ chân và/hoặc bàn chân	
M84.98	Rối loạn về tính liên tục của xương, không xác định, vị trí khác	
M84.99	Rối loạn về tính liên tục của xương, không xác định, vị trí không xác định	
M85	Rối loạn khác về mật độ và/hoặc cấu trúc xương	
M85.0	Loạn sản xơ xương (một xương)	
M85.00	Loạn sản xơ xương (một xương), nhiều vị trí	
M85.01	Loạn sản xơ xương (một xương), vùng vai	
M85.02	Loạn sản xơ xương (một xương), cánh tay trên	
M85.03	Loạn sản xơ xương (một xương), cẳng tay	
M85.04	Loạn sản xơ xương (một xương), bàn tay	
M85.05	Loạn sản xơ xương (một xương), vùng chậu và/hoặc đùi	
M85.06	Loạn sản xơ xương (một xương), cẳng chân	
M85.07	Loạn sản xơ xương (một xương), cổ chân và/hoặc bàn chân	
M85.08	Loạn sản xơ xương (một xương), vị trí khác	
M85.09	Loạn sản xơ xương (một xương), vị trí không xác định	
M85.1	Xương nhiễm độc fluor	
M85.10	Xương nhiễm độc fluor, nhiều vị trí	
M85.11	Xương nhiễm độc fluor, vùng vai	
M85.12	Xương nhiễm độc fluor, cánh tay trên	
M85.13	Xương nhiễm độc fluor, cẳng tay	
M85.14	Xương nhiễm độc fluor, bàn tay	
M85.15	Xương nhiễm độc fluor, vùng chậu và/hoặc đùi	
M85.16	Xương nhiễm độc fluor, cẳng chân	
M85.17	Xương nhiễm độc fluor, cổ chân và/hoặc bàn chân	
M85.18	Xương nhiễm độc fluor, vị trí khác	
M85.19	Xương nhiễm độc fluor, vị trí không xác định	
M85.2	Chứng dày xương sọ	
M85.3	Bệnh đặc xương	
M85.30	Bệnh đặc xương, nhiều vị trí	
M85.31	Bệnh đặc xương, vùng vai	
M85.32	Bệnh đặc xương, cánh tay trên	
M85.33	Bệnh đặc xương, cẳng tay	
M85.34	Bệnh đặc xương, bàn tay	
M85.35	Bệnh đặc xương, vùng chậu và/hoặc đùi	
M85.36	Bệnh đặc xương, cẳng chân	
M85.37	Bệnh đặc xương, cổ chân và/hoặc bàn chân	
M85.38	Bệnh đặc xương, vị trí khác	
M85.39	Bệnh đặc xương, vị trí không xác định	
M85.4	Nang xương đơn độc	
M85.40	Nang xương đơn độc, nhiều vị trí	
M85.41	Nang xương đơn độc, vùng vai	
M85.42	Nang xương đơn độc, cánh tay trên	
M85.43	Nang xương đơn độc, cẳng tay	
M85.44	Nang xương đơn độc, bàn tay	
M85.45	Nang xương đơn độc, vùng chậu và/hoặc đùi	
M85.46	Nang xương đơn độc, cẳng chân	
M85.47	Nang xương đơn độc, cổ chân và/hoặc bàn chân	
M85.48	Nang xương đơn độc, vị trí khác	
M85.49	Nang xương đơn độc, vị trí không xác định	
M85.5	Nang xương phình mạch	
M85.50	Nang xương phình mạch, nhiều vị trí	
M85.51	Nang xương phình mạch, vùng vai	
M85.52	Nang xương phình mạch, cánh tay trên	
M85.53	Nang xương phình mạch, cẳng tay	
M85.54	Nang xương phình mạch, bàn tay	
M85.55	Nang xương phình mạch, vùng chậu và/hoặc đùi	
M85.56	Nang xương phình mạch, cẳng chân	
M85.57	Nang xương phình mạch, cổ chân và/hoặc bàn chân	
M85.58	Nang xương phình mạch, vị trí khác	
M85.59	Nang xương phình mạch, vị trí không xác định	
M85.6	Nang xương khác	
M85.60	Nang xương khác, nhiều vị trí	
M85.61	Nang xương khác, vùng vai	
M85.62	Nang xương khác, cánh tay trên	
M85.63	Nang xương khác, cẳng tay	
M85.64	Nang xương khác, bàn tay	
M85.65	Nang xương khác, vùng chậu và/hoặc đùi	
M85.66	Nang xương khác, cẳng chân	
M85.67	Nang xương khác, cổ chân và/hoặc bàn chân	
M85.68	Nang xương khác, vị trí khác	
M85.69	Nang xương khác, vị trí không xác định	
M85.8	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương	
M85.80	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, nhiều vị trí	
M85.81	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, vùng vai	
M85.82	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, cánh tay trên	
M85.83	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, cẳng tay	
M85.84	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, bàn tay	
M85.85	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, vùng chậu và/hoặc đùi	
M85.86	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, cẳng chân	
M85.87	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, cổ chân và/hoặc bàn chân	
M85.88	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, vị trí khác	
M85.89	Rối loạn xác định khác về mật độ và/hoặc cấu trúc xương, vị trí không xác định	
M85.9	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương	
M85.90	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, nhiều vị trí	
M85.91	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, vùng vai	
M85.92	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, cánh tay trên	
M85.93	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, cẳng tay	
M85.94	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, bàn tay	
M85.95	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, vùng chậu và/hoặc đùi	
M85.96	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, cẳng chân	
M85.97	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, cổ chân và/hoặc bàn chân	
M85.98	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, vị trí khác	
M85.99	Rối loạn không xác định về mật độ và/hoặc cấu trúc xương, vị trí không xác định	
M86	Viêm xương tủy	
M86.0	Viêm xương tủy lây qua đường máu cấp tính	
M86.00	Viêm xương tủy lây qua đường máu cấp tính, nhiều vị trí	
M86.01	Viêm xương tủy lây qua đường máu cấp tính, vùng vai	
M86.02	Viêm xương tủy lây qua đường máu cấp tính, cánh tay trên	
M86.03	Viêm xương tủy lây qua đường máu cấp tính, cẳng tay	
M86.04	Viêm xương tủy lây qua đường máu cấp tính, bàn tay	
M86.05	Viêm xương tủy lây qua đường máu cấp tính, vùng chậu và/hoặc đùi	
M86.06	Viêm xương tủy lây qua đường máu cấp tính, cẳng chân	
M86.07	Viêm xương tủy lây qua đường máu cấp tính, cổ chân và/hoặc bàn chân	
M86.08	Viêm xương tủy lây qua đường máu cấp tính, vị trí khác	
M86.09	Viêm xương tủy lây qua đường máu cấp tính, vị trí không xác định	
M86.1	Viêm xương tủy cấp tính khác	
M86.10	Viêm xương tủy cấp tính khác, nhiều vị trí	
M86.11	Viêm xương tủy cấp tính khác, vùng vai	
M86.12	Viêm xương tủy cấp tính khác, cánh tay trên	
M86.13	Viêm xương tủy cấp tính khác, cẳng tay	
M86.14	Viêm xương tủy cấp tính khác, bàn tay	
M86.15	Viêm xương tủy cấp tính khác, vùng chậu và/hoặc đùi	
M86.16	Viêm xương tủy cấp tính khác, cẳng chân	
M86.17	Viêm xương tủy cấp tính khác, cổ chân và/hoặc bàn chân	
M86.18	Viêm xương tủy cấp tính khác, vị trí khác	
M86.19	Viêm xương tủy cấp tính khác, vị trí không xác định	
M86.2	Viêm xương tủy bán cấp tính	
M86.20	Viêm xương tủy bán cấp tính, nhiều vị trí	
M86.21	Viêm xương tủy bán cấp tính, vùng vai	
M86.22	Viêm xương tủy bán cấp tính, cánh tay trên	
M86.23	Viêm xương tủy bán cấp tính, cẳng tay	
M86.24	Viêm xương tủy bán cấp tính, bàn tay	
M86.25	Viêm xương tủy bán cấp tính, vùng chậu và/hoặc đùi	
M86.26	Viêm xương tủy bán cấp tính, cẳng chân	
M86.27	Viêm xương tủy bán cấp tính, cổ chân và/hoặc bàn chân	
M86.28	Viêm xương tủy bán cấp tính, vị trí khác	
M86.29	Viêm xương tủy bán cấp tính, vị trí không xác định	
M86.3	Viêm xương tủy mạn tính đa ổ	
M86.30	Viêm xương tủy mạn tính đa ổ, nhiều vị trí	
M86.31	Viêm xương tủy mạn tính đa ổ, vùng vai	
M86.32	Viêm xương tủy mạn tính đa ổ, cánh tay trên	
M86.33	Viêm xương tủy mạn tính đa ổ, cẳng tay	
M86.34	Viêm xương tủy mạn tính đa ổ, bàn tay	
M86.35	Viêm xương tủy mạn tính đa ổ, vùng chậu và/hoặc đùi	
M86.36	Viêm xương tủy mạn tính đa ổ, cẳng chân	
M86.37	Viêm xương tủy mạn tính đa ổ, cổ chân và/hoặc bàn chân	
M86.38	Viêm xương tủy mạn tính đa ổ, vị trí khác	
M86.39	Viêm xương tủy mạn tính đa ổ, vị trí không xác định	
M86.4	Viêm xương tủy mạn tính có đường rò	
M86.40	Viêm xương tủy mạn tính có đường rò, nhiều vị trí	
M86.41	Viêm xương tủy mạn tính có đường rò, vùng vai	
M86.42	Viêm xương tủy mạn tính có đường rò, cánh tay trên	
M86.43	Viêm xương tủy mạn tính có đường rò, cẳng tay	
M86.44	Viêm xương tủy mạn tính có đường rò, bàn tay	
M86.45	Viêm xương tủy mạn tính có đường rò, vùng chậu và/hoặc đùi	
M86.46	Viêm xương tủy mạn tính có đường rò, cẳng chân	
M86.47	Viêm xương tủy mạn tính có đường rò, cổ chân và/hoặc bàn chân	
M86.48	Viêm xương tủy mạn tính có đường rò, vị trí khác	
M86.49	Viêm xương tủy mạn tính có đường rò, vị trí không xác định	
M86.5	Viêm xương tủy lây qua đường máu mạn tính khác	
M86.50	Viêm xương tủy lây qua đường máu mạn tính khác, nhiều vị trí	
M86.51	Viêm xương tủy lây qua đường máu mạn tính khác, vùng vai	
M86.52	Viêm xương tủy lây qua đường máu mạn tính khác, cánh tay trên	
M86.53	Viêm xương tủy lây qua đường máu mạn tính khác, cẳng tay	
M86.54	Viêm xương tủy lây qua đường máu mạn tính khác, bàn tay	
M86.55	Viêm xương tủy lây qua đường máu mạn tính khác, vùng chậu và/hoặc đùi	
M86.56	Viêm xương tủy lây qua đường máu mạn tính khác, cẳng chân	
M86.57	Viêm xương tủy lây qua đường máu mạn tính khác, cổ chân và/hoặc bàn chân	
M86.58	Viêm xương tủy lây qua đường máu mạn tính khác, vị trí khác	
M86.59	Viêm xương tủy lây qua đường máu mạn tính khác, vị trí không xác định	
M86.6	Viêm xương tủy mạn tính khác	
M86.60	Viêm xương tủy mạn tính khác, nhiều vị trí	
M86.61	Viêm xương tủy mạn tính khác, vùng vai	
M86.62	Viêm xương tủy mạn tính khác, cánh tay trên	
M86.63	Viêm xương tủy mạn tính khác, cẳng tay	
M86.64	Viêm xương tủy mạn tính khác, bàn tay	
M86.65	Viêm xương tủy mạn tính khác, vùng chậu và/hoặc đùi	
M86.66	Viêm xương tủy mạn tính khác, cẳng chân	
M86.67	Viêm xương tủy mạn tính khác, cổ chân và/hoặc bàn chân	
M86.68	Viêm xương tủy mạn tính khác, vị trí khác	
M86.69	Viêm xương tủy mạn tính khác, vị trí không xác định	
M86.8	Viêm xương tủy khác	Áp xe Brodie
M86.80	Viêm xương tủy khác, nhiều vị trí	
M86.81	Viêm xương tủy khác, vùng vai	
M86.82	Viêm xương tủy khác, cánh tay trên	
M86.83	Viêm xương tủy khác, cẳng tay	
M86.84	Viêm xương tủy khác, bàn tay	
M86.85	Viêm xương tủy khác, vùng chậu và/hoặc đùi	
M86.86	Viêm xương tủy khác, cẳng chân	
M86.87	Viêm xương tủy khác, cổ chân và/hoặc bàn chân	
M86.88	Viêm xương tủy khác, vị trí khác	
M86.89	Viêm xương tủy khác, vị trí không xác định	
M86.9	Viêm xương tủy, không xác định	Nhiễm trùng xương không xác định khác|Viêm màng xương không xác định khác
M86.90	Viêm xương tủy, không xác định, nhiều vị trí	
M86.91	Viêm xương tủy, không xác định, vùng vai	
M86.92	Viêm xương tủy, không xác định, cánh tay trên	
M86.93	Viêm xương tủy, không xác định, cẳng tay	
M86.94	Viêm xương tủy, không xác định, bàn tay	
M86.95	Viêm xương tủy, không xác định, vùng chậu và/hoặc đùi	
M86.96	Viêm xương tủy, không xác định, cẳng chân	
M86.97	Viêm xương tủy, không xác định, cổ chân và/hoặc bàn chân	
M86.98	Viêm xương tủy, không xác định, vị trí khác	
M86.99	Viêm xương tủy, không xác định, vị trí không xác định	
M87	Hoại tử xương	
M87.0	Hoại tử vô khuẩn xương vô căn	
M87.00	Hoại tử vô khuẩn xương vô căn, nhiều vị trí	
M87.01	Hoại tử vô khuẩn xương vô căn, vùng vai	
M87.02	Hoại tử vô khuẩn xương vô căn, cánh tay trên	
M87.03	Hoại tử vô khuẩn xương vô căn, cẳng tay	
M87.04	Hoại tử vô khuẩn xương vô căn, bàn tay	
M87.05	Hoại tử vô khuẩn xương vô căn, vùng chậu và/hoặc đùi	
M87.06	Hoại tử vô khuẩn xương vô căn, cẳng chân	
M87.07	Hoại tử vô khuẩn xương vô căn, cổ chân và/hoặc bàn chân	
M87.08	Hoại tử vô khuẩn xương vô căn, vị trí khác	
M87.09	Hoại tử vô khuẩn xương vô căn, vị trí không xác định	
M87.1	Hoại tử xương do thuốc	
M87.10	Hoại tử xương do thuốc, nhiều vị trí	
M87.11	Hoại tử xương do thuốc, vùng vai	
M87.12	Hoại tử xương do thuốc, cánh tay trên	
M87.13	Hoại tử xương do thuốc, cẳng tay	
M87.14	Hoại tử xương do thuốc, bàn tay	
M87.15	Hoại tử xương do thuốc, vùng chậu và/hoặc đùi	
M87.16	Hoại tử xương do thuốc, cẳng chân	
M87.17	Hoại tử xương do thuốc, cổ chân và/hoặc bàn chân	
M87.18	Hoại tử xương do thuốc, vị trí khác	
M87.19	Hoại tử xương do thuốc, vị trí không xác định	
M87.2	Hoại tử xương do chấn thương từ trước	
M87.20	Hoại tử xương do chấn thương từ trước, nhiều vị trí	
M87.21	Hoại tử xương do chấn thương từ trước, vùng vai	
M87.22	Hoại tử xương do chấn thương từ trước, cánh tay trên	
M87.23	Hoại tử xương do chấn thương từ trước, cẳng tay	
M87.24	Hoại tử xương do chấn thương từ trước, bàn tay	
M87.25	Hoại tử xương do chấn thương từ trước, vùng chậu và/hoặc đùi	
M87.26	Hoại tử xương do chấn thương từ trước, cẳng chân	
M87.27	Hoại tử xương do chấn thương từ trước, cổ chân và/hoặc bàn chân	
M87.28	Hoại tử xương do chấn thương từ trước, vị trí khác	
M87.29	Hoại tử xương do chấn thương từ trước, vị trí không xác định	
M87.3	Hoại tử xương thứ phát khác	
M87.30	Hoại tử xương thứ phát khác, nhiều vị trí	
M87.31	Hoại tử xương thứ phát khác, vùng vai	
M87.32	Hoại tử xương thứ phát khác, cánh tay trên	
M87.33	Hoại tử xương thứ phát khác, cẳng tay	
M87.34	Hoại tử xương thứ phát khác, bàn tay	
M87.35	Hoại tử xương thứ phát khác, vùng chậu và/hoặc đùi	
M87.36	Hoại tử xương thứ phát khác, cẳng chân	
M87.37	Hoại tử xương thứ phát khác, cổ chân và/hoặc bàn chân	
M87.38	Hoại tử xương thứ phát khác, vị trí khác	
M87.39	Hoại tử xương thứ phát khác, vị trí không xác định	
M87.8	Hoại tử xương khác	
M87.80	Hoại tử xương khác, nhiều vị trí	
M87.81	Hoại tử xương khác, vùng vai	
M87.82	Hoại tử xương khác, cánh tay trên	
M87.83	Hoại tử xương khác, cẳng tay	
M87.84	Hoại tử xương khác, bàn tay	
M87.85	Hoại tử xương khác, vùng chậu và/hoặc đùi	
M87.86	Hoại tử xương khác, cẳng chân	
M87.87	Hoại tử xương khác, cổ chân và/hoặc bàn chân	
M87.88	Hoại tử xương khác, vị trí khác	
M87.89	Hoại tử xương khác, vị trí không xác định	
M87.9	Hoại tử xương, không xác định	
M87.90	Hoại tử xương, không xác định, nhiều vị trí	
M87.91	Hoại tử xương, không xác định, vùng vai	
M87.92	Hoại tử xương, không xác định, cánh tay trên	
M87.93	Hoại tử xương, không xác định, cẳng tay	
M87.94	Hoại tử xương, không xác định, bàn tay	
M87.95	Hoại tử xương, không xác định, vùng chậu và/hoặc đùi	
M87.96	Hoại tử xương, không xác định, cẳng chân	
M87.97	Hoại tử xương, không xác định, cổ chân và/hoặc bàn chân	
M87.98	Hoại tử xương, không xác định, vị trí khác	
M87.99	Hoại tử xương, không xác định, vị trí không xác định	
M88	Bệnh Paget xương [viêm xương biến dạng]	
M88.0	Bệnh Paget xương sọ	
M88.8	Bệnh Paget xương khác (ngoài xương sọ)	
M88.80	Bệnh Paget xương khác (ngoài xương sọ), nhiều vị trí	
M88.81	Bệnh Paget xương khác (ngoài xương sọ), vùng vai	
M88.82	Bệnh Paget xương khác (ngoài xương sọ), cánh tay trên	
M88.83	Bệnh Paget xương khác (ngoài xương sọ), cẳng tay	
M88.84	Bệnh Paget xương khác (ngoài xương sọ), bàn tay	
M88.85	Bệnh Paget xương khác (ngoài xương sọ), vùng chậu và/hoặc đùi	
M88.86	Bệnh Paget xương khác (ngoài xương sọ), cẳng chân	
M88.87	Bệnh Paget xương khác (ngoài xương sọ), cổ chân và/hoặc bàn chân	
M88.88	Bệnh Paget xương khác (ngoài xương sọ), vị trí khác	
M88.89	Bệnh Paget xương khác (ngoài xương sọ), vị trí không xác định	
M88.9	Bệnh Paget xương, không xác định	
M88.90	Bệnh Paget xương, không xác định, nhiều vị trí	
M88.91	Bệnh Paget xương, không xác định, vùng vai	
M88.92	Bệnh Paget xương, không xác định, cánh tay trên	
M88.93	Bệnh Paget xương, không xác định, cẳng tay	
M88.94	Bệnh Paget xương, không xác định, bàn tay	
M88.95	Bệnh Paget xương, không xác định, vùng chậu và/hoặc đùi	
M88.96	Bệnh Paget xương, không xác định, cẳng chân	
M88.97	Bệnh Paget xương, không xác định, cổ chân và/hoặc bàn chân	
M88.98	Bệnh Paget xương, không xác định, vị trí khác	
M88.99	Bệnh Paget xương, không xác định, vị trí không xác định	
M89	Rối loạn khác của xương	
M89.0	Hội chứng loạn dưỡng đau thần kinh	
M89.00	Hội chứng loạn dưỡng đau thần kinh, nhiều vị trí	
M89.01	Hội chứng loạn dưỡng đau thần kinh, vùng vai	
M89.02	Hội chứng loạn dưỡng đau thần kinh, cánh tay trên	
M89.03	Hội chứng loạn dưỡng đau thần kinh, cẳng tay	
M89.04	Hội chứng loạn dưỡng đau thần kinh, bàn tay	
M89.05	Hội chứng loạn dưỡng đau thần kinh, vùng chậu và/hoặc đùi	
M89.06	Hội chứng loạn dưỡng đau thần kinh, cẳng chân	
M89.07	Hội chứng loạn dưỡng đau thần kinh, cổ chân và/hoặc bàn chân	
M89.08	Hội chứng loạn dưỡng đau thần kinh, vị trí khác	
M89.09	Hội chứng loạn dưỡng đau thần kinh, vị trí không xác định	
M89.1	Ngừng phát triển đầu xương	
M89.10	Ngừng phát triển đầu xương, nhiều vị trí	
M89.11	Ngừng phát triển đầu xương, vùng vai	
M89.12	Ngừng phát triển đầu xương, cánh tay trên	
M89.13	Ngừng phát triển đầu xương, cẳng tay	
M89.14	Ngừng phát triển đầu xương, bàn tay	
M89.15	Ngừng phát triển đầu xương, vùng chậu và/hoặc đùi	
M89.16	Ngừng phát triển đầu xương, cẳng chân	
M89.17	Ngừng phát triển đầu xương, cổ chân và/hoặc bàn chân	
M89.18	Ngừng phát triển đầu xương, vị trí khác	
M89.19	Ngừng phát triển đầu xương, vị trí không xác định	
M89.2	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương	
M89.20	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, nhiều vị trí	
M89.21	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, vùng vai	
M89.22	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, cánh tay trên	
M89.23	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, cẳng tay	
M89.24	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, bàn tay	
M89.25	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, vùng chậu và/hoặc đùi	
M89.26	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, cẳng chân	
M89.27	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, cổ chân và/hoặc bàn chân	
M89.28	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, vị trí khác	
M89.29	Rối loạn khác về sự phát triển và/hoặc tăng trưởng của xương, vị trí không xác định	
M89.3	Phì đại xương	
M89.30	Phì đại xương, nhiều vị trí	
M89.31	Phì đại xương, vùng vai	
M89.32	Phì đại xương, cánh tay trên	
M89.33	Phì đại xương, cẳng tay	
M89.34	Phì đại xương, bàn tay	
M89.35	Phì đại xương, vùng chậu và/hoặc đùi	
M89.36	Phì đại xương, cẳng chân	
M89.37	Phì đại xương, cổ chân và/hoặc bàn chân	
M89.38	Phì đại xương, vị trí khác	
M89.39	Phì đại xương, vị trí không xác định	
M89.4	Bệnh lý phì đại xương khớp khác	Bệnh Maire - Bamberger|Chứng dày da viêm màng xương
M89.40	Bệnh lý phì đại xương khớp khác, nhiều vị trí	
M89.41	Bệnh lý phì đại xương khớp khác, vùng vai	
M89.42	Bệnh lý phì đại xương khớp khác, cánh tay trên	
M89.43	Bệnh lý phì đại xương khớp khác, cẳng tay	
M89.44	Bệnh lý phì đại xương khớp khác, bàn tay	
M89.45	Bệnh lý phì đại xương khớp khác, vùng chậu và/hoặc đùi	
M89.46	Bệnh lý phì đại xương khớp khác, cẳng chân	
M89.47	Bệnh lý phì đại xương khớp khác, cổ chân và/hoặc bàn chân	
M89.48	Bệnh lý phì đại xương khớp khác, vị trí khác	
M89.49	Bệnh lý phì đại xương khớp khác, vị trí không xác định	
M89.5	Bệnh tiêu xương	
M89.50	Bệnh tiêu xương, nhiều vị trí	
M89.51	Bệnh tiêu xương, vùng vai	
M89.52	Bệnh tiêu xương, cánh tay trên	
M89.53	Bệnh tiêu xương, cẳng tay	
M89.54	Bệnh tiêu xương, bàn tay	
M89.55	Bệnh tiêu xương, vùng chậu và/hoặc đùi	
M89.56	Bệnh tiêu xương, cẳng chân	
M89.57	Bệnh tiêu xương, cổ chân và/hoặc bàn chân	
M89.58	Bệnh tiêu xương, vị trí khác	
M89.59	Bệnh tiêu xương, vị trí không xác định	
M89.6	Bệnh lý xương sau bệnh bại liệt	
M89.60	Bệnh lý xương sau bệnh bại liệt, nhiều vị trí	
M89.61	Bệnh lý xương sau bệnh bại liệt, vùng vai	
M89.62	Bệnh lý xương sau bệnh bại liệt, cánh tay trên	
M89.63	Bệnh lý xương sau bệnh bại liệt, cẳng tay	
M89.64	Bệnh lý xương sau bệnh bại liệt, bàn tay	
M89.65	Bệnh lý xương sau bệnh bại liệt, vùng chậu và/hoặc đùi	
M89.66	Bệnh lý xương sau bệnh bại liệt, cẳng chân	
M89.67	Bệnh lý xương sau bệnh bại liệt, cổ chân và/hoặc bàn chân	
M89.68	Bệnh lý xương sau bệnh bại liệt, vị trí khác	
M89.69	Bệnh lý xương sau bệnh bại liệt, vị trí không xác định	
M89.8	Rối loạn xác định khác của xương	Bệnh phì đại màng xương ở trẻ nhỏ|Cốt hóa dưới màng xương sau chấn thương
M89.80	Rối loạn xác định khác của xương, nhiều vị trí	
M89.81	Rối loạn xác định khác của xương, vùng vai	
M89.82	Rối loạn xác định khác của xương, cánh tay trên	
M89.83	Rối loạn xác định khác của xương, cẳng tay	
M89.84	Rối loạn xác định khác của xương, bàn tay	
M89.85	Rối loạn xác định khác của xương, vùng chậu và/hoặc đùi	
M89.86	Rối loạn xác định khác của xương, cẳng chân	
M89.87	Rối loạn xác định khác của xương, cổ chân và/hoặc bàn chân	
M89.88	Rối loạn xác định khác của xương, vị trí khác	
M89.89	Rối loạn xác định khác của xương, vị trí không xác định	
M89.9	Rối loạn xương, không xác định	
M89.90	Rối loạn xương, không xác định, nhiều vị trí	
M89.91	Rối loạn xương, không xác định, vùng vai	
M89.92	Rối loạn xương, không xác định, cánh tay trên	
M89.93	Rối loạn xương, không xác định, cẳng tay	
M89.94	Rối loạn xương, không xác định, bàn tay	
M89.95	Rối loạn xương, không xác định, vùng chậu và/hoặc đùi	
M89.96	Rối loạn xương, không xác định, cẳng chân	
M89.97	Rối loạn xương, không xác định, cổ chân và/hoặc bàn chân	
M89.98	Rối loạn xương, không xác định, vị trí khác	
M89.99	Rối loạn xương, không xác định, vị trí không xác định	
M90.*	Bệnh lý xương do bệnh phân loại mục khác	
M90.0*	Lao xương (A18.0†)	
M90.00*	Lao xương (A18.0†), nhiều vị trí	
M90.01*	Lao xương (A18.0†), vùng vai	
M90.02*	Lao xương (A18.0†), cánh tay trên	
M90.03*	Lao xương (A18.0†), cẳng tay	
M90.04*	Lao xương (A18.0†), bàn tay	
M90.05*	Lao xương (A18.0†), vùng chậu và/hoặc đùi	
M90.06*	Lao xương (A18.0†), cẳng chân	
M90.07*	Lao xương (A18.0†), cổ chân và/hoặc bàn chân	
M90.08*	Lao xương (A18.0†), vị trí khác	
M90.09*	Lao xương (A18.0†), vị trí không xác định	
M90.1*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác	
M90.10*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, nhiều vị trí	
M90.11*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, vùng vai	
M90.12*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, cánh tay trên	
M90.13*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, cẳng tay	
M90.14*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, bàn tay	
M90.15*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, vùng chậu và/hoặc đùi	
M90.16*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, cẳng chân	
M90.17*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, cổ chân và/hoặc bàn chân	
M90.18*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, vị trí khác	
M90.19*	Viêm màng xương do bệnh nhiễm trùng phân loại mục khác, vị trí không xác định	
M90.2*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác	
M90.20*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, nhiều vị trí	
M90.21*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, vùng vai	
M90.22*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, cánh tay trên	
M90.23*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, cẳng tay	
M90.24*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, bàn tay	
M90.25*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, vùng chậu và/hoặc đùi	
M90.26*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, cẳng chân	
M90.27*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, cổ chân và/hoặc bàn chân	
M90.28*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, vị trí khác	
M90.29*	Bệnh lý xương do bệnh nhiễm trùng phân loại mục khác, vị trí không xác định	
M90.3*	Hoại tử xương do bệnh giảm áp (T70.3†)	
M90.30*	Hoại tử xương do bệnh giảm áp (T70.3†), nhiều vị trí	
M90.31*	Hoại tử xương do bệnh giảm áp (T70.3†), vùng vai	
M90.32*	Hoại tử xương do bệnh giảm áp (T70.3†), cánh tay trên	
M90.33*	Hoại tử xương do bệnh giảm áp (T70.3†), cẳng tay	
M90.34*	Hoại tử xương do bệnh giảm áp (T70.3†), bàn tay	
M90.35*	Hoại tử xương do bệnh giảm áp (T70.3†), vùng chậu và/hoặc đùi	
M90.36*	Hoại tử xương do bệnh giảm áp (T70.3†), cẳng chân	
M90.37*	Hoại tử xương do bệnh giảm áp (T70.3†), cổ chân và/hoặc bàn chân	
M90.38*	Hoại tử xương do bệnh giảm áp (T70.3†), vị trí khác	
M90.39*	Hoại tử xương do bệnh giảm áp (T70.3†), vị trí không xác định	
M90.4*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†)	
M90.40*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), nhiều vị trí	
M90.41*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), vùng vai	
M90.42*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), cánh tay trên	
M90.43*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), cẳng tay	
M90.44*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), bàn tay	
M90.45*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), vùng chậu và/hoặc đùi	
M90.46*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), cẳng chân	
M90.47*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), cổ chân và/hoặc bàn chân	
M90.48*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), vị trí khác	
M90.49*	Hoại tử xương do bệnh lý huyết sắc tố (D50-D64†), vị trí không xác định	
M90.5*	Hoại tử xương do bệnh phân loại mục khác	
M90.50*	Hoại tử xương do bệnh phân loại mục khác, nhiều vị trí	
M90.51*	Hoại tử xương do bệnh phân loại mục khác, vùng vai	
M90.52*	Hoại tử xương do bệnh phân loại mục khác, cánh tay trên	
M90.53*	Hoại tử xương do bệnh phân loại mục khác, cẳng tay	
M90.54*	Hoại tử xương do bệnh phân loại mục khác, bàn tay	
M90.55*	Hoại tử xương do bệnh phân loại mục khác, vùng chậu và/hoặc đùi	
M90.56*	Hoại tử xương do bệnh phân loại mục khác, cẳng chân	
M90.57*	Hoại tử xương do bệnh phân loại mục khác, cổ chân và/hoặc bàn chân	
M90.58*	Hoại tử xương do bệnh phân loại mục khác, vị trí khác	
M90.59*	Hoại tử xương do bệnh phân loại mục khác, vị trí không xác định	
M90.6*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†)	
M90.60*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), nhiều vị trí	
M90.61*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), vùng vai	
M90.62*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), cánh tay trên	
M90.63*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), cẳng tay	
M90.64*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), bàn tay	
M90.65*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), vùng chậu và/hoặc đùi	
M90.66*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), cẳng chân	
M90.67*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), cổ chân và/hoặc bàn chân	
M90.68*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), vị trí khác	
M90.69*	Viêm xương biến dạng do bệnh u tân sinh (C00.- - D48.-†), vị trí không xác định	
M90.7*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†)	
M90.70*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), nhiều vị trí	
M90.71*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), vùng vai	
M90.72*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), cánh tay trên	
M90.73*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), cẳng tay	
M90.74*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), bàn tay	
M90.75*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), vùng chậu và/hoặc đùi	
M90.76*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), cẳng chân	
M90.77*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), cổ chân và/hoặc bàn chân	
M90.78*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), vị trí khác	
M90.79*	Gãy xương do bệnh u tân sinh (C00.- - D48.-†), vị trí không xác định	
M90.8*	Bệnh lý xương do bệnh khác phân loại mục khác	
M90.80*	Bệnh lý xương do bệnh phân loại mục khác, nhiều vị trí	
M90.81*	Bệnh lý xương do bệnh phân loại mục khác, vùng vai	
M90.82*	Bệnh lý xương do bệnh phân loại mục khác, cánh tay trên	
M90.83*	Bệnh lý xương do bệnh phân loại mục khác, cẳng tay	
M90.84*	Bệnh lý xương do bệnh phân loại mục khác, bàn tay	
M90.85*	Bệnh lý xương do bệnh phân loại mục khác, vùng chậu và/hoặc đùi	
M90.86*	Bệnh lý xương do bệnh phân loại mục khác, cẳng chân	
M90.87*	Bệnh lý xương do bệnh phân loại mục khác, cổ chân và/hoặc bàn chân	
M90.88*	Bệnh lý xương do bệnh phân loại mục khác, vị trí khác	
M90.89*	Bệnh lý xương do bệnh phân loại mục khác, vị trí không xác định	
M91	Thoái hóa xương sụn khớp háng và/hoặc khung chậu ở thiếu niên	
M91.0	Thoái hóa xương sụn khung chậu ở thiếu niên	
M91.1	Thoái hóa xương sụn chỏm xương đùi ở thiếu niên [Legg-Calvé-Pethès]	
M91.2	Xẹp chỏm xương đùi	Biến dạng khớp háng do bệnh thoái hóa xương sụn ở tuổi thiếu niên trước đó
M91.3	Bệnh hoại tử vô khuẩn chỏm xương đùi ở trẻ em [Legg-Calvé-Perthes]	
M91.8	Thoái hóa xương sụn khác khớp háng và/hoặc khung chậu ở tuổi thiếu niên	Thoái hóa xương sụn thiếu niên sau nắn xương do trật khớp háng bẩm sinh
M91.9	Thoái hóa xương sụn khớp háng và/hoặc khung chậu ở tuổi thiếu niên, không xác định	
M92	Thoái hóa xương sụn khác ở tuổi thiếu niên	
M92.0	Thoái hóa xương sụn cánh tay ở tuổi thiếu niên	
M92.1	Thoái hóa xương sụn của xương trụ và/hoặc xương quay ở tuổi thiếu niên	
M92.2	Thoái hóa xương sụn bàn tay ở tuổi thiếu niên	
M92.3	Thoái hóa xương sụn chi trên khác ở thiếu niên	
M92.4	Thoái hóa xương sụn xương bánh chè ở tuổi thiếu niên	
M92.5	Thoái hóa xương sụn xương chày và/hoặc xương mác ở tuổi thiếu niên	
M92.6	Thoái hóa xương sụn xương cổ chân ở tuổi thiếu niên	
M92.7	Thoái hóa xương sụn xương bàn chân ở thiếu niên	
M92.8	Thoái hóa xương sụn xác định khác ở tuổi thiếu niên	Viêm mỏm xương gót
M92.9	Thoái hóa xương sụn ở tuổi thiếu niên, vị trí không xác định	Viêm mỏm xương xác định ở tuổi thiếu niên, ở vị trí không xác định|Viêm đầu xương xác định ở tuổi thiếu niên, ở vị trí không xác định|Viêm xương sụn xác định ở tuổi thiếu niên, ở vị trí không xác định|Thoái hóa xương sụn xác định ở tuổi thiếu niên, ở vị trí không xác định
M93	Bệnh lý xương sụn khác	
M93.0	Trượt đầu trên xương đùi (không do chấn thương)	
M93.1	Bệnh Kienböck ở người lớn	Thoái hóa xương sụn xương bán nguyệt cổ tay ở người lớn
M93.2	Viêm xương sụn bóc tách	
M93.8	Bệnh lý xương sụn xác định khác	
M93.9	Bệnh lý xương sụn, không xác định	Viêm mỏm xương xác định ở tuổi thiếu niên, ở vị trí không xác định|Viêm đầu xương xác định ở tuổi thiếu niên, ở vị trí không xác định|Viêm xương sụn xác định ở tuổi thiếu niên, ở vị trí không xác định|Thoái hóa xương sụn xác định ở tuổi thiếu niên, ở vị trí không xác định
M94	Rối loạn khác của sụn	
M94.0	Hội chứng khớp sụn sườn [Tietze]	Viêm khớp sụn sườn
M94.1	Viêm đa sụn tái phát	
M94.10	Viêm đa sụn tái phát, nhiều vị trí	
M94.11	Viêm đa sụn tái phát, vùng vai	
M94.12	Viêm đa sụn tái phát, cánh tay trên	
M94.13	Viêm đa sụn tái phát, cẳng tay	
M94.14	Viêm đa sụn tái phát, bàn tay	
M94.15	Viêm đa sụn tái phát, vùng chậu và/hoặc đùi	
M94.16	Viêm đa sụn tái phát, cẳng chân	
M94.17	Viêm đa sụn tái phát, cổ chân và/hoặc bàn chân	
M94.18	Viêm đa sụn tái phát, vị trí khác	
M94.19	Viêm đa sụn tái phát, vị trí không xác định	
M94.2	Bệnh nhuyễn sụn	
M94.20	Bệnh nhuyễn sụn, nhiều vị trí	
M94.21	Bệnh nhuyễn sụn, vùng vai	
M94.22	Bệnh nhuyễn sụn, cánh tay trên	
M94.23	Bệnh nhuyễn sụn, cẳng tay	
M94.24	Bệnh nhuyễn sụn, bàn tay	
M94.25	Bệnh nhuyễn sụn, vùng chậu và/hoặc đùi	
M94.26	Bệnh nhuyễn sụn, cẳng chân	
M94.27	Bệnh nhuyễn sụn, cổ chân và/hoặc bàn chân	
M94.28	Bệnh nhuyễn sụn, vị trí khác	
M94.29	Bệnh nhuyễn sụn, vị trí không xác định	
M94.3	Bệnh tiêu sụn	
M94.30	Bệnh tiêu sụn, nhiều vị trí	
M94.31	Bệnh tiêu sụn, vùng vai	
M94.32	Bệnh tiêu sụn, cánh tay trên	
M94.33	Bệnh tiêu sụn, cẳng tay	
M94.34	Bệnh tiêu sụn, bàn tay	
M94.35	Bệnh tiêu sụn, vùng chậu và/hoặc đùi	
M94.36	Bệnh tiêu sụn, cẳng chân	
M94.37	Bệnh tiêu sụn, cổ chân và/hoặc bàn chân	
M94.38	Bệnh tiêu sụn, vị trí khác	
M94.39	Bệnh tiêu sụn, vị trí không xác định	
M94.8	Rối loạn xác định khác của sụn	
M94.80	Rối loạn xác định khác của sụn, nhiều vị trí	
M94.81	Rối loạn xác định khác của sụn, vùng vai	
M94.82	Rối loạn xác định khác của sụn, cánh tay trên	
M94.83	Rối loạn xác định khác của sụn, cẳng tay	
M94.84	Rối loạn xác định khác của sụn, bàn tay	
M94.85	Rối loạn xác định khác của sụn, vùng chậu và/hoặc đùi	
M94.86	Rối loạn xác định khác của sụn, cẳng chân	
M94.87	Rối loạn xác định khác của sụn, cổ chân và/hoặc bàn chân	
M94.88	Rối loạn xác định khác của sụn, vị trí khác	
M94.89	Rối loạn xác định khác của sụn, vị trí không xác định	
M94.9	Rối loạn của sụn, không xác định	
M94.90	Rối loạn của sụn, không xác định, nhiều vị trí	
M94.91	Rối loạn của sụn, không xác định, vùng vai	
M94.92	Rối loạn của sụn, không xác định, cánh tay trên	
M94.93	Rối loạn của sụn, không xác định, cẳng tay	
M94.94	Rối loạn của sụn, không xác định, bàn tay	
M94.95	Rối loạn của sụn, không xác định, vùng chậu và/hoặc đùi	
M94.96	Rối loạn của sụn, không xác định, cẳng chân	
M94.97	Rối loạn của sụn, không xác định, cổ chân và/hoặc bàn chân	
M94.98	Rối loạn của sụn, không xác định, vị trí khác	
M94.99	Rối loạn của sụn, không xác định, vị trí không xác định	
M95	Biến dạng mắc phải khác của hệ cơ xương khớp và/hoặc mô liên kết	
M95.0	Biến dạng mắc phải của mũi	
M95.1	Biến dạng tai súp lơ	
M95.2	Biến dạng mắc phải khác của đầu	
M95.3	Biến dạng mắc phải của cổ	
M95.4	Biến dạng mắc phải của lồng ngực và/hoặc sườn	
M95.5	Biến dạng mắc phải của khung chậu	
M95.8	Biến dạng mắc phải xác định khác của hệ cơ xương khớp	
M95.9	Biến dạng mắc phải của hệ cơ xương khớp, không xác định	
M96	Rối loạn của hệ cơ xương khớp sau can thiệp, không phân loại mục khác	
M96.0	Khớp giả sau thủ thuật làm cứng khớp hay cố định khớp	
M96.1	Hội chứng sau phẫu thuật cắt cung sau đốt sống, không phân loại mục khác	
M96.2	Gù sau xạ trị	
M96.3	Gù sau phẫu thuật cắt cung sau đốt sống	
M96.4	Ưỡn cột sống sau phẫu thuật	
M96.5	Vẹo cột sống sau xạ trị	
M96.6	Gãy xương sau cấy thiết bị chỉnh hình, khớp giả và/hoặc nẹp vít cố định bên trong	
M96.8	Rối loạn cơ xương khớp khác sau can thiệp	Sự không ổn định thứ phát của khớp sau khi tháo khớp giả
M96.9	Rối loạn cơ xương khớp sau can thiệp, không xác định	
M99	Tổn thương sinh - cơ học, không phân loại mục khác	Tổn thương sinh-cơ học, không phân loại mục khác
M99.0	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể	
M99.00	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng đầu	
M99.01	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng cổ	
M99.02	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng ngực	
M99.03	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng thắt lưng	
M99.04	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng xương cùng	
M99.05	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, vùng chậu	
M99.06	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, chi dưới	
M99.07	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, chi trên	
M99.08	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, lồng ngực	
M99.09	Rối loạn chức năng bộ phận và/hoặc rối loạn chức năng cơ thể, bụng và/hoặc vùng khác	
M99.1	Hội chứng trượt đốt sống	
M99.10	Hội chứng trượt đốt sống, vùng đầu	
M99.11	Hội chứng trượt đốt sống, vùng cổ	
M99.12	Hội chứng trượt đốt sống, vùng ngực	
M99.13	Hội chứng trượt đốt sống, vùng thắt lưng	
M99.14	Hội chứng trượt đốt sống, vùng xương cùng	
M99.2	Hẹp ống tủy sống do trượt đốt sống	
M99.20	Hẹp ống tủy sống do trượt đốt sống, vùng đầu	
M99.21	Hẹp ống tủy sống do trượt đốt sống, vùng cổ	
M99.22	Hẹp ống tủy sống do trượt đốt sống, vùng ngực	
M99.23	Hẹp ống tủy sống do trượt đốt sống, vùng thắt lưng	
M99.24	Hẹp ống tủy sống do trượt đốt sống, vùng xương cùng	
M99.3	Hẹp ống tủy sống do rối loạn xương	
M99.30	Hẹp ống tủy sống do rối loạn xương, vùng đầu	
M99.31	Hẹp ống tủy sống do rối loạn xương, vùng cổ	
M99.32	Hẹp ống tủy sống do rối loạn xương, vùng ngực	
M99.33	Hẹp ống tủy sống do rối loạn xương, vùng thắt lưng	
M99.34	Hẹp ống tủy sống do rối loạn xương, vùng xương cùng	
M99.4	Hẹp ống tủy sống do rối loạn mô liên kết	
M99.40	Hẹp ống tủy sống do rối loạn mô liên kết, vùng đầu	
M99.41	Hẹp ống tủy sống do rối loạn mô liên kết, vùng cổ	
M99.42	Hẹp ống tủy sống do rối loạn mô liên kết, vùng ngực	
M99.43	Hẹp ống tủy sống do rối loạn mô liên kết, vùng thắt lưng	
M99.44	Hẹp ống tủy sống do rối loạn mô liên kết, vùng xương cùng	
M99.5	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống	
M99.50	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống, vùng đầu	
M99.51	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống, vùng cổ	
M99.52	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống, vùng ngực	
M99.53	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống, vùng thắt lưng	
M99.54	Hẹp ống tủy sống do rối loạn đĩa đệm cột sống, vùng xương cùng	
M99.6	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống	
M99.60	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống, vùng đầu	
M99.61	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống, vùng cổ	
M99.62	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống, vùng ngực	
M99.63	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống, vùng thắt lưng	
M99.64	Hẹp lỗ gian đốt sống do cốt hóa và/hoặc trượt đốt sống, vùng xương cùng	
M99.7	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm	
M99.70	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm, vùng đầu	
M99.71	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm, vùng cổ	
M99.72	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm, vùng ngực	
M99.73	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm, vùng thắt lưng	
M99.74	Hẹp lỗ gian đốt sống do rối loạn mô liên kết và/hoặc đĩa đệm, vùng xương cùng	
M99.8	Tổn thương sinh - cơ học khác	
M99.80	Tổn thương sinh - cơ học khác, vùng đầu	
M99.81	Tổn thương sinh - cơ học khác, vùng cổ	
M99.82	Tổn thương sinh - cơ học khác, vùng ngực	
M99.83	Tổn thương sinh - cơ học khác, vùng thắt lưng	
M99.84	Tổn thương sinh - cơ học khác, vùng xương cùng	
M99.85	Tổn thương sinh - cơ học khác, vùng chậu	
M99.86	Tổn thương sinh - cơ học khác, chi dưới	
M99.87	Tổn thương sinh - cơ học khác, chi trên	
M99.88	Tổn thương sinh - cơ học khác, lồng ngực	
M99.89	Tổn thương sinh - cơ học khác, bụng và/hoặc vùng khác	
M99.9	Tổn thương sinh - cơ học, không xác định	
M99.90	Tổn thương sinh - cơ học, không xác định, vùng đầu	
M99.91	Tổn thương sinh - cơ học, không xác định, vùng cổ	
M99.92	Tổn thương sinh - cơ học, không xác định, vùng ngực	
M99.93	Tổn thương sinh - cơ học, không xác định, vùng thắt lưng	
M99.94	Tổn thương sinh - cơ học, không xác định, vùng xương cùng	
M99.95	Tổn thương sinh - cơ học, không xác định, vùng chậu	
M99.96	Tổn thương sinh - cơ học, không xác định, chi dưới	
M99.97	Tổn thương sinh - cơ học, không xác định, chi trên	
M99.98	Tổn thương sinh - cơ học, không xác định, lồng ngực	
M99.99	Tổn thương sinh - cơ học, không xác định, bụng và/hoặc vùng khác	
N00	Hội chứng viêm thận cấp tính	
N00.0	Hội chứng viêm thận cấp tính, bất thường nhỏ ở cầu thận	
N00.1	Hội chứng viêm thận cấp tính, tổn thương cầu thận ổ - cục bộ	
N00.2	Hội chứng viêm thận cấp tính, viêm thận cầu thận màng lan tỏa	
N00.3	Hội chứng viêm thận cấp tính, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N00.4	Hội chứng viêm thận cấp tính, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N00.5	Hội chứng viêm thận cấp tính, viêm thận cầu thận màng tăng sinh lan tỏa	
N00.6	Hội chứng viêm thận cấp tính, bệnh lắng đọng đặc	
N00.7	Hội chứng viêm thận cấp tính, viêm thận cầu thận hình liềm lan tỏa	
N00.8	Hội chứng viêm thận cấp tính, kết quả mô bệnh học khác	
N00.9	Hội chứng viêm thận cấp tính, kết quả mô bệnh học không xác định	
N01	Hội chứng viêm thận tiến triển nhanh	
N01.0	Hội chứng viêm thận tiến triển nhanh, bất thường nhỏ ở cầu thận	
N01.1	Hội chứng viêm thận tiến triển nhanh, tổn thương cầu thận ổ - cục bộ	
N01.2	Hội chứng viêm thận tiến triển nhanh, viêm thận cầu thận màng lan tỏa	
N01.3	Hội chứng viêm thận tiến triển nhanh, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N01.4	Hội chứng viêm thận tiến triển nhanh, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N01.5	Hội chứng viêm thận tiến triển nhanh, viêm thận cầu thận màng tăng sinh lan tỏa	
N01.6	Hội chứng viêm thận tiến triển nhanh, bệnh lắng đọng đặc	
N01.7	Hội chứng viêm thận tiến triển nhanh, viêm thận cầu thận hình liềm lan tỏa	
N01.8	Hội chứng viêm thận tiến triển nhanh, kết quả mô bệnh học khác	
N01.9	Hội chứng viêm thận tiến triển nhanh, kết quả mô bệnh học không xác định	
N02	Tiểu máu dai dẳng và/hoặc tái phát	
N02.0	Tiểu máu dai dẳng và/hoặc tái phát, bất thường nhỏ ở cầu thận	
N02.1	Tiểu máu dai dẳng và/hoặc tái phát, tổn thương cầu thận ổ - cục bộ	
N02.2	Tiểu máu dai dẳng và/hoặc tái phát, viêm thận cầu thận màng lan tỏa	
N02.3	Tiểu máu dai dẳng và/hoặc tái phát, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N02.4	Tiểu máu dai dẳng và/hoặc tái phát, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N02.5	Tiểu máu dai dẳng và/hoặc tái phát, viêm thận cầu thận màng tăng sinh lan tỏa	
N02.6	Tiểu máu dai dẳng và/hoặc tái phát, bệnh lắng đọng đặc	
N02.7	Tiểu máu dai dẳng và/hoặc tái phát, viêm thận cầu thận hình liềm lan tỏa	
N02.8	Tiểu máu dai dẳng và/hoặc tái phát, kết quả mô bệnh học khác	
N02.9	Tiểu máu dai dẳng và/hoặc tái phát, kết quả mô bệnh học không xác định	
N03	Hội chứng viêm thận mạn tính	
N03.0	Hội chứng viêm thận mạn tính, bất thường nhỏ ở cầu thận	
N03.1	Hội chứng viêm thận mạn tính, tổn thương cầu thận ổ - cục bộ	
N03.2	Hội chứng viêm thận mạn tính, viêm thận cầu thận màng lan tỏa	
N03.3	Hội chứng viêm thận mạn tính, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N03.4	Hội chứng viêm thận mạn tính, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N03.5	Hội chứng viêm thận mạn tính, viêm thận cầu thận màng tăng sinh lan tỏa	
N03.6	Hội chứng viêm thận mạn tính, bệnh lắng đọng đặc	
N03.7	Hội chứng viêm thận mạn tính, viêm thận cầu thận hình liềm lan tỏa	
N03.8	Hội chứng viêm thận mạn tính, kết quả mô bệnh học khác	
N03.9	Hội chứng viêm thận mạn tính, kết quả mô bệnh học không xác định	
N04	Hội chứng thận hư	
N04.0	Hội chứng thận hư, bất thường nhỏ ở cầu thận	
N04.1	Hội chứng thận hư, tổn thương cầu thận ổ - cục bộ	
N04.2	Hội chứng thận hư, viêm thận cầu thận màng lan tỏa	
N04.3	Hội chứng thận hư, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N04.4	Hội chứng thận hư, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N04.5	Hội chứng thận hư, viêm thận cầu thận màng tăng sinh lan tỏa	
N04.6	Hội chứng thận hư, bệnh lắng đọng đặc	
N04.7	Hội chứng thận hư, viêm thận cầu thận hình liềm lan tỏa	
N04.8	Hội chứng thận hư, kết quả mô bệnh học khác	
N04.9	Hội chứng thận hư, kết quả mô bệnh học không xác định	
N05	Hội chứng viêm thận không xác định	
N05.0	Hội chứng viêm thận không xác định, bất thường nhỏ ở cầu thận	
N05.1	Hội chứng viêm thận không xác định, tổn thương cầu thận ổ - cục bộ	
N05.2	Hội chứng viêm thận không xác định, viêm thận cầu thận màng lan tỏa	
N05.3	Hội chứng viêm thận không xác định, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N05.4	Hội chứng viêm thận không xác định, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N05.5	Hội chứng viêm thận không xác định, viêm thận cầu thận màng tăng sinh lan tỏa	
N05.6	Hội chứng viêm thận không xác định, bệnh lắng đọng đặc	
N05.7	Hội chứng viêm thận không xác định, viêm thận cầu thận hình liềm lan tỏa	
N05.8	Hội chứng viêm thận không xác định, kết quả mô bệnh học khác	
N05.9	Hội chứng không xác định của viêm thận, kết quả mô bệnh học không xác định	
N06	Protein niệu đơn độc với tổn thương hình thái xác định	
N06.0	Protein niệu đơn độc với tổn thương hình thái xác định, bất thường nhỏ ở cầu thận	
N06.1	Protein niệu đơn độc với tổn thương hình thái xác định, tổn thương cầu thận ổ - cục bộ	
N06.2	Protein niệu đơn độc với tổn thương hình thái xác định, viêm thận cầu thận màng lan tỏa	
N06.3	Protein niệu đơn độc với tổn thương hình thái xác định, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N06.4	Protein niệu đơn độc với tổn thương hình thái xác định, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N06.5	Protein niệu đơn độc với tổn thương hình thái xác định, viêm thận cầu thận màng tăng sinh lan tỏa	
N06.6	Protein niệu đơn độc với tổn thương hình thái xác định, bệnh lắng đọng đặc	
N06.7	Protein niệu đơn độc với tổn thương hình thái xác định, viêm thận cầu thận hình liềm lan tỏa	
N06.8	Protein niệu đơn độc với tổn thương hình thái xác định, kết quả mô bệnh học khác	
N06.9	Protein niệu đơn độc với tổn thương hình thái xác định, kết quả mô bệnh học không xác định	
N07	Bệnh lý thận di truyền, không phân loại mục khác	
N07.0	Bệnh lý thận di truyền, không phân loại mục khác, bất thường nhỏ ở cầu thận	
N07.1	Bệnh lý thận di truyền, không phân loại mục khác, tổn thương cầu thận ổ - cục bộ	
N07.2	Bệnh lý thận di truyền, không phân loại mục khác, viêm thận cầu thận màng lan tỏa	
N07.3	Bệnh lý thận di truyền, không phân loại mục khác, viêm thận cầu thận tăng sinh gian mạch lan tỏa	
N07.4	Bệnh lý thận di truyền, không phân loại mục khác, viêm thận cầu thận tăng sinh nội mạch lan tỏa	
N07.5	Bệnh lý thận di truyền, không phân loại mục khác, viêm thận cầu thận màng tăng sinh lan tỏa	
N07.6	Bệnh lý thận di truyền, không phân loại mục khác, bệnh lắng đọng đặc	
N07.7	Bệnh lý thận di truyền, không phân loại mục khác, viêm thận cầu thận hình liềm lan tỏa	
N07.8	Bệnh lý thận di truyền khác, không phân loại mục khác, kết quả mô bệnh học khác	
N07.9	Bệnh lý thận di truyền không xác định, không phân loại mục khác, kết quả mô bệnh học không xác định	
N08.*	Rối loạn cầu thận do bệnh phân loại mục khác	
N08.0*	Rối loạn cầu thận do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
N08.1*	Rối loạn cầu thận do bệnh tân sinh	
N08.2*	Biến đổi cầu thận do bệnh về máu và/hoặc rối loạn liên quan cơ chế miễn dịch	
N08.3*	Rối loạn cầu thận do đái tháo đường (nhóm E10-E14 với ký tự thứ tư chung là .2†)	
N08.4*	Rối loạn cầu thận do bệnh nội tiết, dinh dưỡng và/hoặc chuyển hóa khác	
N08.5*	Rối loạn cầu thận do rối loạn mô liên kết hệ thống	
N08.8*	Rối loạn cầu thận do bệnh khác phân loại mục khác	
N10	Viêm ống thận mô kẽ cấp tính	
N11	Viêm ống thận mô kẽ mạn tính	
N11.0	Viêm thận bể thận mạn tính liên quan đến trào ngược không tắc nghẽn	
N11.1	Viêm thận bể thận mạn tính do tắc nghẽn	
N11.8	Viêm ống thận mô kẽ mạn tính khác	Viêm thận bể thận mạn tính không tắc nghẽn không xác định khác
N11.9	Viêm mô kẽ ống thận mạn tính, không xác định	
N12	Viêm ống thận mô kẽ, không xác định cấp tính hay mạn tính	
N13	Bệnh lý tiết niệu do tắc nghẽn và/hoặc trào ngược	
N13.0	Thận ứ nước kèm hẹp khúc nối bể thận niệu quản	
N13.1	Thận ứ nước kèm co hẹp khúc nối bể thận niệu quản, không phân loại mục khác	
N13.2	Thận ứ nước kèm tắc nghẽn thận và/hoặc niệu quản	
N13.3	Thận ứ nước khác và/hoặc không xác định	
N13.4	Niệu quản ứ nước	
N13.5	Niệu quản gấp khúc và/hoặc co hẹp không gây ứ nước thận	
N13.6	Thận ứ mủ	
N13.7	Bệnh lý tiết niệu liên quan đến trào ngược bàng quang niệu quản	
N13.8	Bệnh lý tiết niệu do tắc nghẽn và/hoặc trào ngược khác	
N13.9	Bệnh lý tiết niệu do tắc nghẽn và/hoặc trào ngược, không xác định	Tắc nghẽn đường tiết niệu không xác định khác
N14	Bệnh lý ống thận và/hoặc ống thận mô kẽ do thuốc và/hoặc kim loại nặng	
N14.0	Bệnh lý thận do thuốc giảm đau	
N14.1	Bệnh lý thận do dược chất, thuốc điều trị và/hoặc sinh phẩm khác	
N14.2	Bệnh lý thận do dược chất, thuốc điều trị và/hoặc sinh phẩm không xác định	
N14.3	Bệnh lý thận do kim loại nặng	
N14.4	Bệnh lý thận nhiễm độc, không phân loại mục khác	
N15	Bệnh ống thận mô kẽ khác	
N15.0	Bệnh lý thận vùng Balkan	Bệnh thận lưu hành vùng Balkan
N15.1	Áp xe thận và/hoặc quanh thận	
N15.8	Bệnh ống thận mô kẽ xác định khác	
N15.9	Bệnh ống thận mô kẽ, không xác định	
N16.*	Rối loạn ống thận mô kẽ do bệnh phân loại mục khác	
N16.0*	Rối loạn ống thận mô kẽ do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
N16.1*	Rối loạn ống thận mô kẽ do bệnh tân sinh	
N16.2*	Rối loạn ống thận mô kẽ do bệnh máu và/hoặc rối loạn liên quan đến cơ chế miễn dịch	
N16.3*	Rối loạn ống thận mô kẽ do bệnh chuyển hóa	
N16.4*	Rối loạn ống thận mô kẽ do rối loạn mô liên kết hệ thống	
N16.5*	Rối loạn ống thận mô kẽ do thải ghép (T86.-†)	
N16.8*	Rối loạn ống thận mô kẽ do bệnh khác phân loại mục khác	
N17	Suy thận cấp tính	
N17.0	Suy thận cấp tính kèm hoại tử ống thận	
N17.1	Suy thận cấp tính kèm hoại tử cấp tính vỏ thận	
N17.2	Suy thận cấp tính kèm hoại tử tủy thận	
N17.8	Suy thận cấp tính khác	
N17.9	Suy thận cấp tính, không xác định	
N18	Bệnh thận mạn tính	
N18.1	Bệnh thận mạn tính, giai đoạn 1	
N18.2	Bệnh thận mạn tính, giai đoạn 2	
N18.3	Bệnh thận mạn tính, giai đoạn 3	
N18.4	Bệnh thận mạn tính, giai đoạn 4	
N18.5	Bệnh thận mạn tính, giai đoạn 5	
N18.9	Bệnh thận mạn tính, không xác định	Suy thận mạn tính|Urê máu cao mạn tính không xác định khác|Viêm cầu thận xơ cứng lan tỏa không xác định khác
N19	Suy thận không xác định	
N20	Sỏi thận và/hoặc niệu quản	
N20.0	Sỏi thận	Sỏi thận không xác định|Sỏi thận hoặc sạn thận|Sỏi san hô|Sỏi trong thận
N20.1	Sỏi niệu quản	
N20.2	Sỏi thận kèm sỏi niệu quản	
N20.9	Sỏi tiết niệu, không xác định	
N21	Sỏi đường tiết niệu dưới	
N21.0	Sỏi bàng quang	
N21.1	Sỏi niệu đạo	
N21.8	Sỏi đường tiết niệu dưới khác	
N21.9	Sỏi đường tiết niệu dưới, không xác định	
N22.*	Sỏi đường tiết niệu do bệnh phân loại mục khác	
N22.0*	Sỏi tiết niệu do bệnh sán máng [bilharziasis] (B65.0†)	
N22.8*	Sỏi đường tiết niệu do bệnh khác phân loại mục khác	
N23	Cơn đau quặn thận không xác định	
N25	Rối loạn do suy giảm chức năng ống thận	
N25.0	Loạn dưỡng xương do thận	Loạn dưỡng xương tăng azote máu|Rối loạn ống thận gây giảm phosphate
N25.1	Đái tháo nhạt do thận	
N25.8	Rối loạn khác do suy giảm chức năng ống thận	Hội chứng Lightwood-Albright|Nhiễm toan ống thận không xác định khác|Cường cận giáp thứ phát do thận
N25.9	Rối loạn do suy giảm chức năng ống thận, không xác định	
N26	Thận nhỏ không xác định	
N27	Thận teo nhỏ không rõ nguyên nhân	
N27.0	Thận teo nhỏ, một bên	
N27.1	Thận teo nhỏ, hai bên	
N27.9	Thận teo nhỏ, không xác định	
N28	Rối loạn khác của thận và/hoặc niệu quản, không phân loại mục khác	
N28.0	Thiếu máu cực bộ và/hoặc nhồi máu thận	
N28.1	U nang thận	
N28.8	Rối loạn xác định khác của thận và/hoặc niệu quản	Thận phì đại|Niệu quản phình to|Sa thận|Viêm bể thận bàng quang|Viêm bể thận niệu quản bàng quang|Viêm niệu quản bàng quang|Thoát vị niệu quản
N28.9	Rối loạn của thận và/hoặc niệu quản, không xác định	
N29.*	Rối loạn khác của thận và/hoặc niệu quản do bệnh phân loại mục khác	
N29.0*	Bệnh thận do bệnh giang mai giai đoạn muộn (A52.7†)	
N29.1*	Rối loạn khác của thận và/hoặc niệu quản do các bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
N29.8*	Rối loạn khác của thận và/hoặc niệu quản do bệnh khác phân loại mục khác	
N30	Viêm bàng quang	
N30.0	Viêm bàng quang cấp tính	
N30.1	Viêm bàng quang kẽ (mạn tính)	
N30.2	Viêm bàng quang mạn tính khác	
N30.3	Viêm tam giác bàng quang	Viêm tam giác bàng quang niệu đạo
N30.4	Viêm bàng quang do xạ trị	
N30.8	Viêm khác của bàng quang	Áp xe bàng quang
N30.9	Viêm bàng quang, không xác định	
N31	Rối loạn chức năng thần kinh - cơ bàng quang, không phân loại mục khác	Rối loạn chức năng thần kinh-cơ bàng quang, không phân loại mục khác
N31.0	Bàng quang thần kinh do tổn thương ở não và/hoặc tủy sống, không phân loại mục khác	
N31.1	Bàng quang thần kinh phản xạ do tổn thương phần trên của tủy sống, không phân loại mục khác	
N31.2	Bàng quang thần kinh mất trương lực, không phân loại mục khác	
N31.8	Rối loạn chức năng khác của thần kinh - cơ bàng quang	
N31.9	Rối loạn chức năng của thần kinh - cơ bàng quang, không xác định	Rối loạn chức năng bàng quang thần kinh không xác định khác
N32	Rối loạn khác của bàng quang	
N32.0	Chít hẹp cổ bàng quang	
N32.1	Rò bàng quang - ruột	Rò bàng quang trực tràng
N32.2	Rò bàng quang, không phân loại mục khác	
N32.3	Túi thừa bàng quang	
N32.4	Vỡ bàng quang, không do chấn thương	
N32.8	Rối loạn xác định khác của bàng quang	
N32.9	Rối loạn bàng quang, không xác định	
N33.*	Rối loạn bàng quang do bệnh phân loại mục khác	
N33.0*	Viêm bàng quang do bệnh lao (A18.1†)	
N33.8*	Rối loạn của bàng quang do bệnh phân loại mục khác	
N34	Viêm niệu đạo và/hoặc hội chứng niệu đạo	
N34.0	Áp xe niệu đạo	
N34.1	Viêm niệu đạo không xác định cụ thể	
N34.2	Viêm niệu đạo khác	
N34.3	Hội chứng niệu đạo, không xác định	
N35	Co hẹp niệu đạo	
N35.0	Co hẹp niệu đạo sau chấn thương	
N35.1	Co hẹp niệu đạo sau nhiễm trùng, không phân loại mục khác	
N35.8	Co hẹp niệu đạo khác	
N35.9	Co hẹp niệu đạo, không xác định	Lỗ sáo châm kim không xác định khác
N36	Rối loạn khác của niệu đạo	
N36.0	Rò niệu đạo	
N36.1	Túi thừa niệu đạo	
N36.2	Núm niệu đạo	
N36.3	Sa niêm mạc niệu đạo	
N36.8	Rối loạn xác định khác của niệu đạo	
N36.9	Rối loạn niệu đạo, không xác định	
N37.*	Rối loạn niệu đạo do bệnh phân loại mục khác	
N37.0*	Viêm niệu đạo do bệnh phân loại mục khác	
N37.8*	Rối loạn niệu đạo khác do bệnh phân loại mục khác	
N39	Rối loạn khác của hệ tiết niệu	
N39.0	Nhiễm trùng đường tiết niệu, vị trí không xác định	
N39.1	Protein niệu [tiểu đạm] dai dẳng, không xác định	
N39.2	Protein niệu [tiểu đạm] tư thế đứng, không xác định	
N39.3	Tiểu tiện không tự chủ do căng thẳng	
N39.4	Tiểu tiện không tự chủ xác định khác	
N39.8	Rối loạn xác định khác của hệ tiết niệu	
N39.9	Rối loạn hệ tiết niệu, không xác định	
N40	Phì đại [tăng sản] tuyến tiền liệt	
N41	Bệnh viêm tuyến tiền liệt	
N41.0	Viêm tuyến tiền liệt cấp tính	
N41.1	Viêm tuyến tiền liệt mạn tính	
N41.2	Áp xe tuyến tiền liệt	
N41.3	Viêm tuyến tiền liệt - bàng quang	
N41.8	Bệnh viêm khác của tuyến tiền liệt	
N41.9	Bệnh viêm tuyến tiền liệt, không xác định	Viêm tuyến tiền liệt không xác định khác
N42	Rối loạn khác của tuyến tiền liệt	
N42.0	Sỏi tuyến tiền liệt	
N42.1	Xung huyết và/hoặc xuất huyết tuyến tiền liệt	
N42.2	Teo tuyến tiền liệt	
N42.3	Loạn sản tuyến tiền liệt	
N42.8	Rối loạn xác định khác của tuyến tiền liệt	
N42.9	Rối loạn tuyến tiền liệt, không xác định	
N43	Tràn dịch màng tinh hoàn và/hoặc nang mào tinh hoàn	
N43.0	Tràn dịch màng tinh hoàn nang hóa	
N43.1	Tràn dịch màng tinh hoàn nhiễm trùng	
N43.2	Tràn dịch màng tinh hoàn khác	
N43.3	Tràn dịch màng tinh hoàn, không xác định	
N43.4	Nang mào tinh hoàn	
N44	Xoắn tinh hoàn	
N45	Viêm tinh hoàn và/hoặc viêm mào tinh hoàn	
N45.0	Viêm tinh hoàn, viêm mào tinh hoàn và/hoặc viêm tinh hoàn - mào tinh hoàn kèm áp xe	Áp xe mào tinh hoàn hoặc tinh hoàn
N45.9	Viêm tinh hoàn, mào tinh hoàn và/hoặc viêm tinh hoàn - mào tinh hoàn không kèm áp xe	Viêm mào tinh hoàn không xác định khác|Viêm tinh hoàn không xác định khác
N46	Vô sinh ở nam giới	
N47	Bao quy đầu dài, hẹp bao quy đầu và/hoặc thắt nghẹt bao quy đầu	
N48	Rối loạn khác của dương vật	
N48.0	Bạch sản dương vật	
N48.1	Viêm bao quy đầu	
N48.2	Rối loạn viêm khác của dương vật	
N48.3	Chứng cương đau dương vật	Cương đau
N48.4	Bất lực do nguyên nhân thực tổn	
N48.5	Loét dương vật	
N48.6	Xơ cứng dương vật	Bệnh dương vật cong [Peyronie]
N48.8	Rối loạn xác định khác của dương vật	Teo thể hang và dương vật|Phì đại thể hang và dương vật|Nghẽn mạch thể hang và dương vật
N48.9	Rối loạn dương vật, không xác định	
N49	Rối loạn viêm của cơ quan sinh dục nam, không phân loại mục khác	
N49.0	Rối loạn viêm của túi tinh	Viêm túi tinh không xác định khác
N49.1	Rối loạn viêm của thừng tinh, màng tinh và/hoặc ống dẫn tinh	Viêm ống dẫn tinh
N49.2	Rối loạn viêm của bìu	
N49.8	Rối loạn viêm của các cơ quan xác định khác của cơ quan sinh dục nam	Viêm nhiều vị trí của cơ quan sinh dục nam
N49.9	Rối loạn viêm không xác định của cơ quan sinh dục nam	Áp xe ở cơ quan không xác định của sinh dục nam|Mụn ở cơ quan không xác định của sinh dục nam|Nhọt ở cơ quan không xác định của sinh dục nam|Viêm mô tế bào ở cơ quan không xác định của sinh dục nam
N50	Rối loạn khác của cơ quan sinh dục nam	
N50.0	Teo tinh hoàn	
N50.1	Rối loạn mạch máu của cơ quan sinh dục nam	Tràn máu màng tinh hoàn không xác định khác ở cơ quan sinh dục nam|Chảy máu ở cơ quan sinh dục nam|Tắc mạch ở cơ quan sinh dục nam
N50.8	Rối loạn xác định khác của cơ quan sinh dục nam	
N50.9	Rối loạn cơ quan sinh dục nam, không xác định	
N51.*	Rối loạn cơ quan sinh dục nam do bệnh phân loại mục khác	
N51.0*	Rối loạn tuyến tiền liệt do bệnh phân loại mục khác	
N51.1*	Rối loạn tinh hoàn và/hoặc mào tinh hoàn do bệnh phân loại mục khác	
N51.2*	Viêm quy đầu do bệnh phân loại mục khác	
N51.8*	Rối loạn khác của cơ quan sinh dục nam do bệnh phân loại mục khác	
N60	Loạn sản vú lành tính	
N60.0	U nang đơn độc của vú	U nang vú
N60.1	U nang tuyến vú lan tỏa	
N60.2	U xơ tuyến vú	
N60.3	Xơ teo tuyến vú	U nang vú kèm tăng sinh biểu mô
N60.4	Giãn ống tuyến vú	
N60.8	Loạn sản lành tính khác của tuyến vú	
N60.9	Loạn sản lành tính của tuyến vú, không xác định	
N61	Rối loạn viêm của vú	
N62	Phì đại vú	
N63	Khối u không xác định ở vú	
N64	Rối loạn khác ở vú	
N64.0	Nứt và/hoặc rò ở núm vú	
N64.1	Hoại tử mỡ của vú	
N64.2	Teo vú	
N64.3	Tiết sữa không liên quan đến sinh đẻ	
N64.4	Đau vú	
N64.5	Dấu hiệu và/hoặc triệu chứng khác ở vú	Xơ cứng ở vú|Núm vú tiết dịch|Núm vú co rút
N64.8	Rối loạn xác định khác của vú	
N64.9	Rối loạn của vú, không xác định	
N70	Viêm vòi trứng và/hoặc viêm buồng trứng	
N70.0	Viêm vòi trứng và/hoặc viêm buồng trứng cấp tính	
N70.1	Viêm vòi trứng và/hoặc viêm buồng trứng mạn tính	Tràn dịch trong vòi trứng
N70.9	Viêm vòi trứng và/hoặc viêm buồng trứng, không xác định	
N71	Bệnh viêm tử cung, trừ cổ tử cung	
N71.0	Bệnh viêm tử cung cấp tính	
N71.1	Bệnh viêm tử cung mạn tính	
N71.9	Bệnh viêm tử cung, không xác định	
N72	Bệnh viêm cổ tử cung	
N73	Bệnh viêm khác của tiểu khung ở nữ giới	
N73.0	Viêm mô cận tử cung và/hoặc viêm mô tế bào tiểu khung	
N73.1	Viêm mô cận tử cung và viêm mô tế bào tiểu khung mạn tính	Bất cứ tình trạng nào ở N73.0 xác định là mạn tính
N73.2	Viêm mô cận tử cung và/hoặc viêm mô tế bào tiểu khung không xác định	Bất cứ tình trạng nào ở N73.0 xác định là cấp tính hoặc mạn tính
N73.3	Viêm phúc mạc tiểu khung cấp tính ở nữ giới	
N73.4	Viêm phúc mạctiểu khung mạn tính ở nữ giới	
N73.5	Viêm phúc mạc của tiểu khung ở nữ giới, không xác định	
N73.6	Dính phúc mạc tiểu khung ở nữ giới	
N73.8	Bệnh viêm tiểu khung ở nữ giới xác định khác	
N73.9	Bệnh viêm tiểu khung ở nữ giới, không xác định	Viêm hoặc nhiễm khuẩn vùng chậu ở nữ giới không xác định khác
N74.*	Rối loạn viêm tiểu khung ở nữ giới do bệnh phân loại mục khác	
N74.0*	Lao cổ tử cung (A18.1†)	
N74.1*	Bệnh viêm tiểu khung do bệnh lao ở nữ giới (A18.1†)	Lao nội mạc tử cung
N74.2*	Bệnh viêm tiểu khung do bệnh giang mai ở nữ giới (A51.4†, A52.7†)	
N74.3*	Bệnh viêm tiểu khung do bệnh lậu cầu khuẩn ở nữ giới (A54.2†)	
N74.4*	Bệnh viêm tiểu khung do bệnh nhiễm chlamydia ở nữ giới (A56.1†)	
N74.8*	Rối loạn viêm tiểu khung ở nữ giới do bệnh khác phân loại mục khác	
N75	Bệnh của tuyến Bartholin	
N75.0	Nang tuyến Bartholin	
N75.1	Áp xe tuyến Bartholin	
N75.8	Bệnh khác của tuyến Bartholin	Viêm Bartholin
N75.9	Bệnhcủa tuyến Bartholin, không xác định	
N76	Viêm khác của âm đạo và/hoặc âm hộ	
N76.0	Viêm âm đạo cấp tính	Viêm âm đạo không xác định khác
N76.1	Viêm âm đạo bán cấp và/hoặc mạn tính	
N76.2	Viêm âm hộ cấp tính	Viêm âm hộ không xác định khác
N76.3	Viêm âm hộ bán cấp và/hoặc mạn tính	
N76.4	Áp xe âm hộ	Đinh nhọt ở âm hộ
N76.5	Loét âm đạo	
N76.6	Loét âm hộ	
N76.8	Viêm âm đạo và/hoặc âm hộ xác định khác	
N77.*	Viêm và/hoặc loét âm hộ âm đạo do bệnh phân loại mục khác	
N77.0*	Loét âm hộ do bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
N77.1*	Viêm âm đạo, viêm âm hộ và/hoặc viêm âm hộ âm đạo do các bệnh nhiễm trùng và/hoặc ký sinh trùng phân loại mục khác	
N77.8*	Viêm và/hoặc loét âm hộ âm đạo do các bệnh khác phân loại mục khác	
N80	Bệnh lạc nội mạc tử cung	
N80.0	Lạc nội mạc tử cung	Bệnh lạc cơ tuyến
N80.1	Lạc nội mạc tử cung ở buồng trứng	
N80.2	Lạc nội mạc tử cung ở vòi trứng	
N80.3	Lạc nội mạc tử cung ở phúc mạc chậu	
N80.4	Lạc nội mạc tử cung ở vách trực tràng - âm đạo và/hoặc âm đạo	
N80.5	Lạc nội mạc tử cung ở ruột	
N80.6	Lạc nội mạc tử cung ở sẹo da	
N80.8	Lạc nội mạc tử cung khác	Lạc nội mạc tử cung của lồng ngực
N80.9	Lạc nội mạc tử cung, không xác định	
N81	Sa sinh dục nữ	
N81.0	Sa niệu đạo vào thành trước của âm đạo ở nữ giới	
N81.1	Sa bàng quang	
N81.2	Sa tử cung âm đạo không hoàn toàn	Sa cổ tử cung không xác định khác
N81.3	Sa tử cung âm đạo hoàn toàn	
N81.4	Sa tử cung âm đạo, không xác định	Sa tử cung không xác định khác
N81.5	Sa ruột âm đạo	
N81.6	Sa trực tràng âm đạo hình túi	
N81.8	Sa sinh dục khác ở nữ giới	Tầng sinh môn yếu|Sẹo rách cũ của cơ đáy chậu
N81.9	Sa sinh dục ở nữ giới, không xác định	
N82	Rò đường sinh dục ở nữ giới	
N82.0	Rò bàng quang âm đạo	
N82.1	Rò khác của đường tiết niệu sinh dục ở nữ giới	
N82.2	Rò âm đạo vào ruột non	
N82.3	Rò âm đạo vào đại tràng	Rò trực tràng - âm đạo
N82.4	Rò khác của đường sinh dục - ruột ở nữ giới	Rò ruột - tử cung
N82.5	Rò đường sinh dục ra da ở nữ giới	
N82.8	Rò khác của đường sinh dục ở nữ giới	
N82.9	Rò đường sinh dục ở nữ giới, không xác định	
N83	Rối loạn không do viêm của buồng trứng, vòi trứng và/hoặc dây chằng rộng	
N83.0	U nang nang trứng	
N83.1	U nang hoàng thể	U nang hoàng thể chảy máu
N83.2	U nang buồng trứng khác và/hoặc không xác định	
N83.3	Teo buồng trứng và/hoặc teo vòi trứng mắc phải	
N83.4	Thoát vị và/hoặc sa buồng trứng và/hoặc vòi trứng	
N83.5	Xoắn buồng trứng, cuống trứng và/hoặc vòi trứng	
N83.6	Vòi trứng ứ máu	
N83.7	Khối máu tụ trong dây chằng rộng	
N83.8	Rối loạn khác không do viêm của buồng trứng, vòi trứng và/hoặc dây chằng rộng	Hội chứng rách dây chằng rộng [Alien-Masters]
N83.9	Rối loạn không do viêm của buồng trứng, vòi trứng và/hoặc dây chằng rộng, không xác định	
N84	Polyp đường sinh dục ở nữ giới	
N84.0	Polyp thân tử cung	
N84.1	Polyp cổ tử cung	Polyp nhầy cổ tử cung
N84.2	Polyp âm đạo	
N84.3	Polyp âm hộ	Polyp môi âm hộ
N84.8	Polyp các phần khác của đường sinh dục ở nữ giới	
N84.9	Polyp đường sinh dục ở nữ giới, không xác định	
N85	Rối loạn khác không do viêm của tử cung, trừ cổ tử cung	
N85.0	Tăng sản tuyến nội mạc	
N85.1	Tăng sản u tuyến nội mạc tử cung	
N85.2	Phì đại tử cung	
N85.3	Bán co tử cung	
N85.4	Vị trí bất thường của tử cung	
N85.5	Lộn tử cung	
N85.6	Dính buồng tử cung	
N85.7	Ứ máu tử cung	
N85.8	Rối loạn xác định khác không do viêm của tử cung	Teo tử cung, mắc phải|Xơ hóa tử cung không xác định khác
N85.9	Rối loạn không do viêm của tử cung, không xác định	Rối loạn tử cung không xác định khác
N86	Trợt và/hoặc lộn cổ tử cung	
N87	Loạn sản cổ tử cung	
N87.0	Loạn sản cổ tử cung mức độ nhẹ	
N87.1	Loạn sản cổ tử cung mức độ trung bình	Ung thư nội biểu mô cổ tử cung [CIN] độ II
N87.2	Loạn sản cổ tử cung mức độ nặng, không phân loại mục khác	
N87.9	Loạn sản cổ tử cung, không xác định	
N88	Rối loạn khác không do viêm của cổ tử cung	
N88.0	Bạch sản cổ tử cung	
N88.1	Vết rách cũ của cổ tử cung	
N88.2	Co hẹp và/hoặc hẹp cổ tử cung	
N88.3	Cổ tử cung yếu [không đủ khả năng]	
N88.4	Giãn dài cổ tử cung do phì đại	
N88.8	Rối loạn xác định khác không do viêm của cổ tử cung	
N88.9	Rối loạn không do viêm của cổ tử cung, không xác định	
N89	Rối loạn khác không do viêm của âm đạo	
N89.0	Loạn sản âm đạo nhẹ	Ung thư nội biểu mô âm đạo [VAIN], độ I
N89.1	Loạn sản âm đạo mức độ trung bình	Ung thư nội biểu mô âm đạo [VAIN], độ II
N89.2	Loạn sản âm đạo mức độ nặng, không phân loại mục khác	
N89.3	Loạn sản âm đạo, không xác định	
N89.4	Bạch sản âm đạo	
N89.5	Co hẹp và/hoặc teo âm đạo	
N89.6	Vòng màng trinh hẹp	
N89.7	Ứ máu âm đạo	Ứ máu âm đạo kèm theo ứ máu tử cung hay ứ máu vòi trứng
N89.8	Rối loạn xác định khác không do viêm của âm đạo	
N89.9	Rối loạn không do viêm của âm đạo, không xác định	
N90	Rối loạn khác không do viêm của âm hộ và/hoặc tầng sinh môn	
N90.0	Loạn sản âm hộ mức độ nhẹ	Ung thư nội biểu mô âm hộ [VIN], độ I
N90.1	Loạn sản âm hộ mức độ trung bình	Ung thư nội biểu mô âm hộ [VIN], độ II
N90.2	Loạn sản âm hộ mức độ nặng, không phân loại mục khác	
N90.3	Loạn sản âm hộ, không xác định	
N90.4	Bạch sản âm hộ	Loạn dưỡng âm hộ|Teo xơ âm hộ
N90.5	Teo âm hộ	Hẹp âm hộ
N90.6	Phì đại âm hộ	Phì đại môi âm hộ
N90.7	U nang âm hộ	
N90.8	Rối loạn xác định khác không do viêm của âm hộ và/hoặc tầng sinh môn	Dính âm hộ|Phì đại âm vật
N90.9	Rối loạn không do viêm của âm hộ và/hoặc tầng sinh môn, không xác định	
N91	Vô kinh, thiểu kinh và/hoặc hiếm kinh	
N91.0	Vô kinh nguyên phát	Không thấy kinh ở tuổi dậy thì.
N91.1	Vô kinh thứ phát	Không thấy kinh ở phụ nữ trước đó đã có kinh nguyệt.
N91.2	Vô kinh, không xác định	Vô kinh không xác định khác
N91.3	Thiểu kinh [kinh thưa] nguyên phát	Kinh nguyệt ít hoặc hiếm ngay từ đầu.
N91.4	Thiểu kinh [kinh thưa] thứ phát	Kinh nguyệt ít và hiếm ở phụ nữ trước đó có chu kỳ kinh bình thường.
N91.5	Thiểu kinh [kinh thưa], không xác định	Chứng ít kinh nguyệt không xác định khác
N92	Kinh nguyệt quá nhiều, thường xuyên và/hoặc thất thường	
N92.0	Kinh nguyệt quá nhiều và/hoặc thường xuyên với chu kỳ đều	Rong kinh không xác định khác|Chứng đa kinh không xác định khác|Rối loạn kinh nguyệt thể kinh mau
N92.1	Kinh nguyệt quá nhiều và/hoặc thường xuyên với chu kỳ thất thường	Chảy máu giữa chu kỳ kinh nguyệt bất thường|Khoảng thời gian không đều, rút ngắn giữa các chu kỳ kinh nguyệt|Đa kinh kéo dài|Rong huyết
N92.2	Rong kinh tuổi dậy thì	Chảy máu nhiều lúc bắt đầu thấy kinh|Chảy máu tuổi dậy thì
N92.3	Chảy máu lúc rụng trứng	Chảy máu đều giữa các chu kỳ kinh nguyệt
N92.4	Chảy máu nặng thời kỳ tiền mãn kinh	
N92.5	Kinh nguyệt thất thường xác định khác	
N92.6	Kinh nguyệt thất thường, không xác định	
N93	Chảy máu bất thường khác của tử cung và/hoặc âm đạo	
N93.0	Chảy máu sau tiếp xúc và/hoặc giao hợp	
N93.8	Chảy máu bất thường xác định khác của tử cung và/hoặc âm đạo	
N93.9	Chảy máu bất thường của tử cung và/hoặc âm đạo, không xác định	
N94	Đau và/hoặc các tình trạng khác liên quan đến cơ quan sinh dục nữ và/hoặc chu kỳ kinh nguyệt	
N94.0	Hội chứng Mittelschmerz [đau bụng dưới, do rụng trứng]	
N94.1	Đau khi giao hợp	
N94.2	Chứng đau co thắt âm đạo	
N94.3	Hội chứng căng thẳng trước kỳ kinh	
N94.4	Đau bụng kinh nguyên phát	
N94.5	Đau bụng kinh thứ phát	
N94.6	Đau bụng kinh, không xác định	
N94.8	Các tình trạng xác định khác liên quan đến cơ quan sinh dục nữ và/hoặc chu kỳ kinh nguyệt	
N94.9	Tình trạng không xác định liên quan đến cơ quan sinh dục nữ và/hoặc chu kỳ kinh nguyệt	
N95	Rối loạn mãn kinh và/hoặc rối loạn tiền mãn kinh	
N95.0	Chảy máu sau mãn kinh	
N95.1	Trạng thái mãn kinh và/hoặc suy buồng trứng	
N95.2	Viêm teo âm đạo sau mãn kinh	
N95.3	Tình trạng liên quan đến mãn kinh nhân tạo	Hội chứng sau mãn kinh nhân tạo
N95.8	Rối loạn xác định khác của thời kỳ mãn kinh và/hoặc tiền mãn kinh	
N95.9	Rối loạn của thời kỳ mãn kinh và/hoặc tiền mãn kinh, không xác định	
N96	Chứng sảy thai liên tục [tái phát]	
N97	Vô sinh ở nữ giới	
N97.0	Vô sinh ở nữ giới liên quan đến không rụng trứng	
N97.1	Vô sinh ở nữ giới do nguyên nhân vòi trứng	Liên quan đến dị dạng bẩm sinh vòi trứng
N97.2	Vô sinh ở nữ giới do nguyên nhân tử cung	Liên quan đến dị dạng bẩm sinh tử cung|Trứng không làm tổ
N97.3	Vô sinh ở nữ giới do nguyên nhân cổ tử cung	
N97.4	Vô sinh ở nữ giới liên quan đến các yếu tố nam	
N97.8	Vô sinh ở nữ giới do nguyên nhân khác	
N97.9	Vô sinh ở nữ giới, không xác định	
N98	Biến chứng liên quan đến thụ tinh nhân tạo	
N98.0	Nhiễm trùng liên quan đến thụ tinh nhân tạo	
N98.1	Quá kích buồng trứng	
N98.2	Biến chứng liên quan đưa trứng đã thụ tinh [hợp tử] vào tử cung sau khi thụ tinh trong ống nghiệm	
N98.3	Biến chứng liên quan đưa phôi vào tử cung trong quá trình chuyển phôi	
N98.8	Biến chứng khác liên quan đến thụ tinh nhân tạo	
N98.9	Biến chứng liên quan đến thụ thai nhân tạo, không xác định	
N99	Rối loạn sau can thiệp của hệ sinh dục tiết niệu, không phân loại mục khác	
N99.0	Suy thận sau can thiệp	
N99.1	Co hẹp niệu đạo sau can thiệp	Hẹp niệu đạo sau thông niệu đạo
N99.2	Dính âm đạo sau phẫu thuật	
N99.3	Sa vòm âm đạo sau cắt bỏ tử cung	
N99.4	Dính phúc mạc chậu sau can thiệp	
N99.5	Hoạt động kém của các lỗ mở thông ngoài da của đường tiết niệu	
N99.8	Rối loạn khác của hệ sinh dục tiết niệu sau can thiệp	Hội chứng buồng trứng còn lại
N99.9	Rối loạn của hệ sinh dục tiết niệu sau can thiệp, không xác định	
O00	Thai ngoài tử cung	
O00.0	Thai trong ổ bụng	
O00.1	Thai ở vòi tử cung	Thai ở vòi trứng|Vỡ vòi tử cung do thai|Thai sảy qua loa vòi tử cung
O00.2	Thai ở buồng trứng	
O00.8	Thai ngoài tử cung khác	
O00.9	Thai ngoài tử cung, không xác định	
O01	Thai trứng dạng nang	
O01.0	Thai trứng cổ điển	Thai trứng hoàn toàn
O01.1	Thai trứng không hoàn toàn và/hoặc thai trứng bán phần	
O01.9	Thai trứng, không xác định	Bệnh lý nguyên bào nuôi không xác định khác|Thai trứng không xác định khác
O02	Bất thường khác của mô được tạo ra do thụ thai	
O02.0	Trứng trống và/hoặc rau thai thoái hóa nước	
O02.1	Sảy thai sót trong tử cung	
O02.8	Bất thường xác định khác của mô được tao ra do thụ thai	
O02.9	Bất thường của mô được tạo ra do thụ thai, không xác định	
O03	Sảy thai tự nhiên	
O03.0	Sảy thai tự nhiên, không hoàn toàn, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O03.1	Sảy thai tự nhiên, không hoàn toàn, kèm biến chứng rong huyết hoặc băng huyết	
O03.2	Sảy thai tự nhiên, không hoàn toàn, kèm biến chứng thuyên tắc mạch	
O03.3	Sảy thai tự nhiên, không hoàn toàn, kèm biến chứng khác và/hoặc không xác định	
O03.4	Sảy thai tự nhiên, không hoàn toàn, không kèm biến chứng	
O03.5	Sảy thai tự nhiên, hoàn toàn hoặc không xác định, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O03.6	Sảy thai tự nhiên, hoàn toàn hoặc không xác định, kèm biến chứng rong huyết hoặc băng huyết	
O03.7	Sảy thai tự nhiên, hoàn toàn hoặc không xác định, kèm biến chứng thuyên tắc mạch	
O03.8	Sảy thai tự nhiên, hoàn toàn hoặc không xác định, kèm biến chứng khác và/hoặc không xác định	
O03.9	Sảy thai tự nhiên, hoàn toàn hoặc không xác định, không kèm biến chứng	
O04	Phá thai bằng thuốc	
O04.0	Phá thai bằng thuốc, không hoàn toàn, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O04.1	Phá thai bằng thuốc, không hoàn toàn, kèm biến chứng rong huyết hoặc băng huyết	
O04.2	Phá thai bằng thuốc, không hoàn toàn, kèm biến chứng thuyên tắc mạch	
O04.3	Phá thai bằng thuốc, không hoàn toàn, kèm biến chứng khác và/hoặc không xác định	
O04.4	Phá thai bằng thuốc, không hoàn toàn, không kèm biến chứng	
O04.5	Phá thai bằng thuốc, hoàn toàn hoặc không xác định, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O04.6	Phá thai bằng thuốc, hoàn toàn hoặc không xác định, kèm biến chứng rong huyết hoặc băng huyết	
O04.7	Phá thai bằng thuốc, hoàn toàn hoặc không xác định, kèm biến chứng thuyên tắc mạch	
O04.8	Phá thai bằng thuốc, hoàn toàn hoặc không xác định, kèm biến chứng khác và/hoặc không xác định	
O04.9	Phá thai bằng thuốc, hoàn toàn hoặc không xác định, không kèm biến chứng	
O05	Phá thai khác	
O05.0	Phá thai khác, không hoàn toàn, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O05.1	Phá thai khác, không hoàn toàn, kèm biến chứng rong huyết hoặc băng huyết	
O05.2	Phá thai khác, không hoàn toàn, kèm biến chứng thuyên tắc mạch	
O05.3	Phá thai khác, không hoàn toàn, kèm biến chứng khác và/hoặc không xác định	
O05.4	Phá thai khác, không hoàn toàn, không kèm biến chứng	
O05.5	Phá thai khác, hoàn toàn hoặc không xác định, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O05.6	Phá thai khác, hoàn toàn hoặc không xác định, kèm biến chứng rong huyết hoặc băng huyết	
O05.7	Phá thai khác, hoàn toàn hoặc không xác định, kèm biến chứng thuyên tắc mạch	
O05.8	Phá thai khác, hoàn toàn hoặc không xác định, kèm biến chứng khác và/hoặc không xác định	
O05.9	Phá thai khác, hoàn toàn hoặc không xác định, không kèm biến chứng	
O06	Sảy/phá thai không xác định	
O06.0	Sảy/phá thai không xác định, không hoàn toàn, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O06.1	Sảy/phá thai không xác định, không hoàn toàn, kèm biến chứng rong huyết hoặc băng huyết	
O06.2	Sảy/phá thai không xác định, không hoàn toàn, kèm biến chứng thuyên tắc mạch	
O06.3	Sảy/phá thai không xác định, không hoàn toàn, kèm biến chứng khác và/hoặc không xác định	
O06.4	Sảy/phá thai không xác định, không hoàn toàn, không kèm biến chứng	
O06.5	Sảy/phá thai không xác định, hoàn toàn hoặc không xác định, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	
O06.6	Sảy/phá thai không xác định, hoàn toàn hoặc không xác định, kèm biến chứng rong huyết hoặc băng huyết	
O06.7	Sảy/phá thai không xác định, hoàn toàn hoặc không xác định, kèm biến chứng thuyên tắc mạch	
O06.8	Sảy/phá thai không xác định, hoàn toàn hoặc không xác định, kèm biến chứng khác và/hoặc không xác định	
O06.9	Sảy/phá thai không xác định, hoàn toàn hoặc không xác định, không kèm biến chứng	
O07	Phá thai thất bại	
O07.0	Phá thai bằng thuốc thất bại, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	Kèm biến chứng phân loại ở O08.0
O07.1	Phá thai bằng thuốc thất bại, kèm biến chứng rong huyết hoặc băng huyết	Kèm biến chứng phân loại ở O08.1
O07.2	Phá thai bằng thuốc thất bại, kèm biến chứng thuyên tắc mạch	Kèm biến chứng phân loại ở O08.2
O07.3	Phá thai bằng thuốc thất bại, gây biến chứng khác và/hoặc không xác định	Kèm biến chứng phân loại ở O08.3-O08.9
O07.4	Phá thai bằng thuốc thất bại, không kèm biến chứng	Gây sảy thai bằng thuốc thất bại không xác định khác
O07.5	Phá thai thất bại khác và/hoặc không xác định, kèm biến chứng nhiễm trùng đường sinh dục và/hoặc tiểu khung	Kèm biến chứng phân loại ở O08.0
O07.6	Phá thai thất bại khác và/hoặc không xác định, kèm biến chứng rong huyết hoặc băng huyết	Kèm biến chứng phân loại ở O08.1
O07.7	Phá thai thất bại khác và/hoặc không xác định, kèm biến chứng thuyên tắc mạch	Kèm biến chứng phân loại ở O08.2
O07.8	Phá thai thất bại khác và/hoặc không xác định, kèm biến chứng khác và/hoặc không xác định	Kèm biến chứng phân loại ở O08.3-O08.9
O07.9	Phá thai thất bại khác và/hoặc không xác định, không kèm biến chứng	Gây sảy thai thất bại không xác định khác
O08	Biến chứng sau sảy thai, thai ngoài tử cung và/hoặc thai trứng	
O08.0	Nhiễm trùng đường sinh dục và/hoặc tiểu khung sau sảy thai, thai ngoài tử cung và/hoặc thai trứng	
O08.1	Rong huyết hoặc băng huyết sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	Giảm fibrinogen huyết sau bệnh lý phân loại ở O00.- - O07.|Hội chứng tiêu sợi huyết sau bệnh lý phân loại ở O00.- - O07.|Đông máu nội mạch sau bệnh lý phân loại ở O00.- - O07.
O08.2	Thuyên tắc mạch sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	
O08.3	Sốc sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	
O08.4	Suy thận sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	
O08.5	Rối loạn chuyển hóa sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	Mất cân bằng điện giải sau bệnh lý phân loại ở O00.- - O07.
O08.6	Tổn thương các tạng và tổ chức ở tiểu khung sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	
O08.7	Biến chứng tĩnh mạch khác sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	
O08.8	Biến chứng khác sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	Ngừng tim sau bệnh lý phân loại ở O00.- - O07.|Nhiễm trùng đường tiết niệu sau bệnh lý phân loại ở O00.- - O07.
O08.9	Biến chứng sau sảy thai và/hoặc thai ngoài tử cung và/hoặc thai trứng	Biến chứng không xác định sau bệnh lý phân loại ở O00.- - O07.
O10	Tăng huyết áp mắc từ trước gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.0	Tăng huyết áp vô căn mắc từ trước gây biến chứng thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.1	Bệnh tim do tăng huyết áp mắc từ trước gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.2	Bệnh thận do tăng huyết áp mắc từ trước gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.3	Bệnh tim và thận do tăng huyết áp mắc từ trước gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.4	Tăng huyết áp thứ phát mắc từ trước gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O10.9	Tăng huyết áp mắc từ trước không xác định, gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O11	Tiền sản giật kèm tăng huyết áp mạn tính	
O12	Phù và/ hoặc protein niệu trong thai kỳ [do thai nghén gây ra] không kèm tăng huyết áp	
O12.0	Phù trong thai kỳ	
O12.1	Protein niệu trong thai kỳ	
O12.2	Phù kèm protein niệu trong thai kỳ	
O13	Tăng huyết áp thai kỳ [do mang thai gây ra]	
O14	Tiền sản giật	
O14.0	Tiền sản giật thể nhẹ đến thể trung bình	
O14.1	Tiền sản giật thể nặng	
O14.2	Hội chứng HELLP	Sự kết hợp của tan máu, tăng men gan và số lượng tiểu cầu thấp
O14.9	Tiền sản giật, không xác định	
O15	Sản giật	
O15.0	Sản giật do thai kỳ	
O15.1	Sản giật trong khi chuyển dạ	
O15.2	Sản giật trong thời kỳ sau đẻ	
O15.9	Sản giật, không xác định thời kỳ xảy ra	Sản giật không xác định khác
O16	Tăng huyết áp thai sản không xác định	
O20	Xuất huyết đầu thai kỳ	
O20.0	Dọa sảy thai	Băng huyết được xác định do dọa sảy thai
O20.8	Xuất huyết khác đầu thai kỳ	
O20.9	Xuất huyết đầu thai kỳ, không xác định	
O21	Nôn quá mức trong thai kỳ	
O21.0	Nôn nghén thể trung bình	
O21.1	Nôn nghén kèm rối loạn chuyển hóa	
O21.2	Nôn muộn trong thai kỳ	Nôn nhiều bắt đầu sau 22 tuần của thai kỳ
O21.8	Nôn khác gây biến chứng cho thai kỳ	Nôn do các bệnh phân loại mục khác, gây biến chứng cho thai kỳ
O21.9	Nôn trong thai kỳ, không xác định	
O22	Biến chứng tĩnh mạch và/hoặc trĩ trong thai kỳ	
O22.0	Giãn tĩnh mạch chi dưới trong thai kỳ	Giãn tĩnh mạch không xác định khác do thai kỳ
O22.1	Giãn tĩnh mạch sinh dục trong thai kỳ	Giãn tĩnh mạch tầng sinh môn do thai kỳ|Giãn tĩnh mạch âm đạo do thai kỳ|Giãn tĩnh mạch âm hộ do thai kỳ
O22.2	Viêm [tắc] tĩnh mạch huyết khối nông trong thai kỳ	Viêm [tắc] tĩnh mạch huyết khối ở chân trong thai kỳ
O22.3	Huyết khối tĩnh mạch sâu trong thai kỳ	Huyết khối tĩnh mạch sâu, trước khi sinh
O22.4	Trĩ trong thai kỳ	
O22.5	Huyết khối tĩnh mạch não trong thai kỳ	Huyết khối xoang tĩnh mạch não do thai kỳ
O22.8	Biến chứng tĩnh mạch khác trong thai kỳ	
O22.9	Biến chứng tĩnh mạch trong thai kỳ, không xác định	
O23	Nhiễm trùng đường tiết niệu - sinh dục trong thai kỳ	Nhiễm trùng đường tiết niệu-sinh dục trong thai kỳ
O23.0	Nhiễm trùng thận trong thai kỳ	
O23.1	Nhiễm trùng bàng quang trong thai kỳ	
O23.2	Nhiễm trùng niệu đạo trong thai kỳ	
O23.3	Nhiễm trùng phần khác của đường tiết niệu trong thai kỳ	
O23.4	Nhiễm trùng không xác định của đường tiết niệu trong thai kỳ	
O23.5	Nhiễm trùng đường sinh dục trong thai kỳ	
O23.9	Nhiễm trùng đường tiết niệu sinh dục khác và/hoặc không xác định trong thai kỳ	Nhiễm trùng đường tiết niệu sinh dục do thai kỳ không xác định khác
O24	Đái tháo đường trong thai kỳ	
O24.0	Đái tháo đường trong thai kỳ: Đái tháo đường típ 1 mắc từ trước khi mang thai	
O24.1	Đái tháo đường trong thai kỳ: Đái tháo đường típ 2 mắc từ trước khi mang thai	
O24.2	Đái tháo đường trong thai kỳ: Đái tháo đường có liên quan đến thiếu dinh dưỡng mắc từ trước khi mang thai	
O24.3	Đái tháo đường trong thai kỳ: Đái tháo đường mắc từ trước khi mang thai, không xác định	
O24.4	Đái tháo đường phát sinh trong thai kỳ	Đái tháo đường thai kỳ không xác định khác
O24.9	Đái tháo đường trong thai kỳ, không xác định	
O25	Thiếu dinh dưỡng trong thai kỳ	
O26	Chăm sóc thai sản đối với bệnh lý khác chủ yếu liên quan đến thai kỳ	
O26.0	Tăng cân quá mức trong thai kỳ	
O26.1	Tăng cân ít trong thai kỳ	
O26.2	Chăm sóc thai kỳ người sảy thai liên tục [tái phát]	
O26.3	Tồn lưu dụng cụ tránh thai trong khi mang thai	
O26.4	Bệnh herpes thai kỳ	
O26.5	Hội chứng hạ huyết áp thai sản	Hội chứng hạ huyết áp ở tư thế nằm ngửa
O26.6	Rối loạn gan trong thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O26.7	Giãn khớp mu trong thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O26.8	Bệnh lý xác định khác liên quan đến thai kỳ	Kiệt sức và mệt mỏi liên quan đến thai kỳ|Viêm thần kinh ngoại biên liên quan đến thai kỳ|Bệnh thận liên quan đến thai kỳ
O26.9	Bệnh lý liên quan đến thai kỳ, không xác định	
O28	Phát hiện bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.0	Phát hiện huyết học bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.1	Phát hiện hóa sinh bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.2	Phát hiện tế bào học bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.3	Phát hiện siêu âm bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.4	Phát hiện X-quang bất thường ở thai phụ khi khám sàng lọc trước sinh	
O28.5	Phát hiện nhiễm sắc thể và/hoặc di truyền bất thương ở thai phụ khi khám sàng lọc trước sinh	
O28.8	Phát hiện bất thường khác ở thai phụ khi khám khám sàng lọc trước sinh	
O28.9	Phát hiện bất thường ở thai phụ khi khám sàng lọc trước sinh, không xác định	
O29	Biến chứng do gây mê trong thai kỳ	
O29.0	Biến chứng phổi do gây mê trong thai kỳ	Viêm phổi hít do gây mê trong thai kỳ|Viêm phổi hít phải hóa chất do gây mê trong thai kỳ|Hít phải dịch dạ dày hay các chất tiết không xác định khác do gây mê trong thai kỳ|Hội chứng Mendelson do gây mê trong thai kỳ|Xẹp phổi do gây mê trong thai kỳ
O29.1	Biến chứng tim do gây mê trong thai kỳ	
O29.2	Biến chứng hệ thống thần kinh trung ương do gây mê trong thai kỳ	Thiếu oxy não do gây mê trong thai kỳ
O29.3	Phản ứng ngộ độc do gây tê tại chỗ trong thai kỳ	
O29.4	Đau đầu do gây tê tủy sống và/hoặc ngoài màng cứng trong thai kỳ	
O29.5	Biến chứng khác của gây tê tủy sống và ngoài màng cứng trong thai kỳ	
O29.6	Không đặt được hoặc khó đặt nội khí quản trong thai kỳ	
O29.8	Biến chứng khác do gây mê trong thai kỳ	
O29.9	Biến chứng không xác định do gây mê trong thai kỳ	
O30	Đa thai	
O30.0	Song thai	
O30.1	Thai ba	
O30.2	Thai tư	
O30.8	Đa thai khác	
O30.9	Đa thai, không xác định	Đa thai không xác định khác
O31	Biến chứng xác định của đa thai	
O31.0	Thai giấy	Thai bị chèn ép
O31.1	Thai kỳ tiếp tục tiến triển sau khi sảy một hay nhiều thai	
O31.2	Thai kỳ tiếp tục tiến triển sau khi một hay nhiều thai chết trong tử cung	
O31.8	Biến chứng xác định khác do đa thai	
O32	Chăm sóc thai sản vì biết hoặc nghi ngờ ngôi thai bất thường	
O32.0	Chăm sóc thai sản vì ngôi thai không ổn định	
O32.1	Chăm sóc thai sản vì thai ngôi mông	
O32.2	Chăm sóc thai sản vì thai ngôi ngang và/hoặc ngôi chếch	
O32.3	Chăm sóc thai sản vì ngôi mặt, ngôi trán và/hoặc ngôi thóp trước	
O32.4	Chăm sóc thai sản vì thai đủ tháng ngôi đầu cao	Đầu thai không vào được eo trên
O32.5	Chăm sóc thai sản vì đa thai, trong đó một hay nhiều thai có ngôi không thuận [bất thường]	
O32.6	Chăm sóc thai sản vì ngôi thai phức tạp	
O32.8	Chăm sóc thai sản vì ngôi thai bất thường khác	
O32.9	Chăm sóc thai sản vì ngôi thai bất thường, không xác định	
O33	Chăm sóc thai sản vì biết hay nghi ngờ có bất tương xứng đầu chậu	
O33.0	Chăm sóc thai sản vì bất tương xứng do biến dạng khung xương chậu	Biến dạng khung chậu gây ra bất tương xứng không xác định khác
O33.1	Chăm sóc thai sản vì bất tương xứng do khung chậu hẹp toàn bộ	Khung chậu hẹp gây ra bất tương xứng không xác định khác
O33.2	Chăm sóc thai sản vì bất tương xứng do hẹp eo trên	Hẹp eo trên khung chậu gây ra bất tương xứng
O33.3	Chăm sóc thai sản vì bất tương xứng do hẹp eo dưới	Hẹp eo giữa gây bất tương xứng|Hẹp eo dưới gây bất tương xứng
O33.4	Chăm sóc thai sản vì bất tương xứng có nguồn gốc phối hợp cả mẹ và thai	
O33.5	Chăm sóc thai sản vì bất tương xứng do thai to bất thường	Bất tương xứng nguồn gốc do thai với hình dạng thai bình thường|Bất tương xứng do thai không xác định khác
O33.6	Chăm sóc thai sản vì bất tương xứng do thai có não úng thủy	
O33.7	Chăm sóc thai sản vì bất tương xứng do dị dạng khác của thai	Cặp song sinh dính liền gây ra sự không cân xứng
O33.8	Chăm sóc thai sản vì bất tương xứng do nguyên nhân khác	
O33.9	Chăm sóc thai sản vì bất tương xứng, không xác định	Bất tương xứng đầu thai và khung chậu không xác định khác|Bất tương xứng thai khung chậu không xác định khác
O34	Chăm sóc thai sản vì có hoặc nghi ngờ bất thường ở cơ quan tiểu khung	
O34.0	Chăm sóc thai sản vì tử cung dị dạng bẩm sinh	
O34.1	Chăm sóc thai sản vì có khối u ở thân tử cung	
O34.2	Chăm sóc thai sản vì tử cung có sẹo mổ trước đó	
O34.3	Chăm sóc thai sản vì hở eo tử cung	
O34.4	Chăm sóc thai sản vì có bất thường khác ở cổ tử cung	
O34.5	Chăm sóc thai sản vì có bất thường khác ở tử cung có thai	
O34.6	Chăm sóc thai sản vì có bất thường ở âm đạo	
O34.7	Chăm sóc thai sản vì có bất thường ở âm hộ và/hoặc tầng sinh môn	
O34.8	Chăm sóc thai sản vì có bất thường khác ở tạng trong tiểu khung	
O34.9	Chăm sóc thai sản vì có bất thường ở tạng trong tiểu khung, không xác định	
O35	Chăm sóc thai sản vì biết hoặc nghi ngờ có bất thường và/hoặc tổn thương ở thai nhi	
O35.0	Chăm sóc thai sản vì (nghi ngờ) có dị tật hệ thống thần kinh trung ương ở thai nhi	
O35.1	Chăm sóc thai sản vì (nghi ngờ) có bất thường nhiễm sắc thể ở thai nhi	
O35.2	Chăm sóc thai sản vì (nghi ngờ) có bệnh di truyền ở thai nhi	
O35.3	Chăm sóc thai sản vì (nghi ngờ) thai nhi tổn thương do bệnh virus ở người mẹ	
O35.4	Chăm sóc thai sản vì (nghi ngờ) thai nhi tổn thương do rượu	
O35.5	Chăm sóc thai sản vì (nghi ngờ) thai nhi tổn thương do ma túy	
O35.6	Chăm sóc thai sản vì (nghi ngờ) thai nhi tổn thương do nguồn phóng xạ [bức xạ] [tia xạ]	
O35.7	Chăm sóc thai sản vì (nghi ngờ) thai nhi tổn thương do can thiệp khác	
O35.8	Chăm sóc thai sản vì (nghi ngờ) có bất thường và/hoặc tổn thương khác ở thai nhi	
O35.9	Chăm sóc thai sản vì (nghi ngờ) có bất thường và/hoặc tổn thương ở thai nhi, không xác định	
O36	Chăm sóc thai sản vì biết hoặc nghi ngờ có vấn đề khác ở thai nhi	
O36.0	Chăm sóc thai sản vì miễn dịch đồng loại Rh	
O36.1	Chăm sóc thai sản vì miễn dịch đồng loại khác	
O36.2	Chăm sóc thai sản vì phù thai	
O36.3	Chăm sóc thai sản vì dấu hiệu thai thiếu oxy	
O36.4	Chăm sóc thai sản vì thai chết trong tử cung [chết lưu từ khi hoàn thành tuần thứ 20]	
O36.5	Chăm sóc thai sản vì thai kém phát triển	
O36.6	Chăm sóc thai sản vì thai phát triển quá mức	Chăm sóc thai sản vì biết hay nghi ngờ thai to so với tuổi thai
O36.7	Chăm sóc thai sản đối với thai lạc chỗ còn sống [phát triển] trong ổ bụng	
O36.8	Chăm sóc thai sản vì vấn đề xác định khác ở thai nhi	
O36.9	Chăm sóc thai sản vì vấn đề ở thai nhi, không xác định	
O40	Đa ối	
O41	Rối loạn khác của nước ối và/hoặc màng ối	
O41.0	Thiểu ối	Chứng ít nước ối không đề cập đến vỡ ối
O41.1	Nhiễm trùng ối và/hoặc màng ối	Viêm màng ối|Viêm màng đệm|Viêm màng|Viêm rau thai
O41.8	Rối loạn xác định khác của nước ối và/hoặc màng ối	
O41.9	Rối loạn của màng ối và/hoặc nước ối, không xác định	
O42	Vỡ ối sớm	
O42.0	Vỡ ối sớm, xuất hiện chuyển dạ trong vòng 24 giờ	
O42.1	Vỡ ối sớm, xuất hiện chuyển dạ sau 24 giờ	
O42.2	Vỡ ối sớm, có điều trị để trì hoãn chuyển dạ	
O42.9	Vỡ ối sớm, không xác định	
O43	Rối loạn của rau thai	
O43.0	Hội chứng truyền máu qua rau thai	
O43.1	Dị dạng rau thai	Bánh rau bất thường không xác định khác|Bánh rau có rãnh vây quanh
O43.2	Rau cài răng lược	
O43.8	Rối loạn khác của rau thai	
O43.9	Rối loạn của rau thai, không xác định	
O44	Rau tiền đạo	
O44.0	Rau tiền đạo không có xuất huyết	Rau bám thấp không có xuất huyết
O44.1	Rau tiền đạo có xuất huyết	
O45	Rau bong non	
O45.0	Rau bong non do rối loạn đông máu	
O45.8	Rau bong non khác	
O45.9	Rau bong non, không xác định	Rau bong non không xác định khác
O46	Xuất huyết trước đẻ, không phân loại mục khác	
O46.0	Xuất huyết trước đẻ do rối loạn đông máu	
O46.8	Xuất huyết trước đẻ do yếu tố khác	
O46.9	Xuất huyết trước đẻ, không xác định	
O47	Chuyển dạ giả	
O47.0	Chuyển dạ giả trước kết thúc tuần thứ 37 của thai kỳ	
O47.1	Chuyển dạ giả trong hay sau tuần thứ 37 của thai kỳ	
O47.9	Chuyển dạ giả, không xác định	
O48	Thai kỳ quá ngày sinh	
O60	Chuyển dạ và đẻ sớm	
O60.0	Chuyển dạ sớm không đẻ	
O60.1	Chuyển dạ sớm tự nhiên và đẻ non	Chuyển dạ sớm và đẻ không xác định khác|Chuyển dạ sớm tự nhiên và đẻ sớm bằng kỹ thuật mổ lấy thai
O60.2	Chuyển dạ sớm tự nhiên và đẻ đúng kỳ	Chuyển dạ sớm tự nhiên và đẻ đúng kỳ bằng kỹ thuật mổ lấy thai
O60.3	Đẻ sớm không có chuyển dạ tự nhiên	
O61	Can thiệp gây chuyển dạ nhân tạo thất bại	
O61.0	Can thiệp gây chuyển dạ bằng thuốc thất bại	
O61.1	Can thiệp gây chuyển dạ bằng dụng cụ thất bại	
O61.8	Can thiệp gây chuyển dạ thất bại khác	
O61.9	Can thiệp gây chuyển dạ thất bại, không xác định	
O62	Bất thường về động lực chuyển dạ	
O62.0	Cơn co tử cung yếu nguyên phát	Thất bại mở cổ tử cung|Rối loạn tử cung giảm trương lực cơ nguyên phát|Cơn co tử cung yếu từ pha tiềm tàng [giai đoạn đầu] của chuyển dạ
O62.1	Cơn co tử cung yếu thứ phát	Chuyển dạ ngừng ở pha tích cực|Rối loạn tử cung giảm trương lực cơ thứ phát
O62.2	Cơn co tử cung thưa yếu khác	
O62.3	Chuyển dạ nhanh	
O62.4	Cơn co tử cung tăng trương lực cơ, không đồng bộ và kéo dài	
O62.8	Bất thường khác của động lực chuyển dạ	
O62.9	Bất thường không xác định của động lực chuyển dạ	
O63	Chuyển dạ kéo dài	
O63.0	Chuyển dạ giai đoạn đầu kéo dài	
O63.1	Chuyển dạ giai đoạn thứ hai kéo dài	
O63.2	Đẻ chậm thai thứ 2, thứ 3 trong chuyển dạ đẻ đa thai	
O63.9	Chuyển dạ kéo dài, không xác định	Chuyển dạ kéo dài không xác định khác
O64	Chuyển dạ đình trệ do ngôi và/hoặc do thế của thai bất thường	
O64.0	Chuyển dạ đình trệ do đầu thai quay không hoàn toàn	Ngừng ở tư thế ngang
O64.1	Chuyển dạ đình trệ do ngôi mông	
O64.2	Chuyển dạ đình trệ do ngôi mặt	Chuyển dạ đình trệ do ngôi cằm
O64.3	Chuyển dạ đình trệ do ngôi trán	
O64.4	Chuyển dạ đình trệ do ngôi vai	
O64.5	Chuyển dạ đình trệ do ngôi thai phức tạp	
O64.8	Chuyển dạ đình trệ do ngôi và/hoặc do thế bất thường khác	
O64.9	Chuyển dạ đình trệ do ngôi và thế bất thường, không xác định	
O65	Chuyển dạ đình trệ do bất thường ở khung chậu sản phụ	
O65.0	Chuyển dạ đình trệ do biến dạng khung chậu sản phụ	
O65.1	Chuyển dạ đình trệ do khung chậu hẹp hoàn toàn	
O65.2	Chuyển dạ đình trệ do hẹp khung chậu eo trên	
O65.3	Chuyển dạ đình trệ do hẹp khung chậu eo dưới và/hoặc hẹp eo giữa	
O65.4	Chuyển dạ đình trệ do bất tương xứng thai - khung chậu, không xác định	
O65.5	Chuyển dạ đình trệ do bất thường tạng trong tiểu khung sản phụ	Chuyển dạ đình trệ do các bệnh lý phân loại ở O34.
O65.8	Chuyển dạ đình trệ do bất thường khác của khung chậu sản phụ	
O65.9	Chuyển dạ đình trệ do bất thường khung chậu sản phụ, không xác định	
O66	Chuyển dạ đình trệ khác	
O66.0	Chuyển dạ đình trệ do đẻ khó do kẹt vai	Ngôi vai găm chặt
O66.1	Chuyển dạ đình trệ do thai sinh đôi ngôi mông - ngôi đầu cản trở nhau	
O66.2	Chuyển dạ đình trệ do thai to bất thường	
O66.3	Chuyển dạ đình trệ do bất thường khác của thai nhi	
O66.4	Chuyển dạ đẻ thường thất bại, không xác định	Chuyển dạ đẻ thường thất bại phải mổ lấy thai
O66.5	Đẻ forcep hoặc giác hút thất bại, không xác định	Đẻ forcep thất bại phải mổ lấy thai hoặc đẻ giác hút thất bại phải đẻ forcep
O66.8	Chuyển dạ đình trệ xác định khác	
O66.9	Chuyển dạ đình trệ, không xác định	
O67	Chuyển dạ và đẻ kèm biến chứng xuất huyết trong đẻ không phân loại mục khác	
O67.0	Xuất huyết trong đẻ do rối loạn đông máu	
O67.8	Xuất huyết trong đẻ do yếu tố khác	Băng huyết trong đẻ
O67.9	Xuất huyết trong đẻ, không xác định	
O68	Chuyển dạ và đẻ kèm biến chứng suy thai	
O68.0	Chuyển dạ và đẻ kèm biến chứng nhịp tim thai bất thường	
O68.1	Chuyển dạ và đẻ kèm biến chứng nước ối lẫn phân su	
O68.2	Chuyển dạ và đẻ kèm biến chứng nhịp tim thai bất thường và nước ối lẫn phân su	
O68.3	Chuyển dạ và đẻ kèm biến chứng dấu hiệu sinh hóa của suy thai	
O68.8	Chuyển dạ và đẻ kèm biến chứng dấu hiệu khác của suy thai	
O68.9	Chuyển dạ và đẻ kèm biến chứng suy thai, không xác định	
O69	Chuyển dạ và đẻ kèm biến chứng dây rốn	
O69.0	Chuyển dạ và đẻ kèm biến chứng sa dây rốn	
O69.1	Chuyển dạ và đẻ kèm biến chứng dây rốn quấn quanh cổ, có chèn ép	
O69.2	Chuyển dạ và đẻ kèm biến chứng vướng mắc dây rốn, có chèn ép	Chèn ép dây rốn không xác định khác|Vướng mắc dây rốn trong sinh đôi một ối|Thắt nút dây rốn
O69.3	Chuyển dạ và đẻ kèm biến chứng do dây rốn ngắn	
O69.4	Chuyển dạ và đẻ kèm biến chứng do mạch máu dây rốn của thai nhi tiền đạo	Xuất huyết từ mạch máu dây rốn của thai nhi tiền đạo
O69.5	Chuyển dạ và đẻ kèm biến chứng tổn thương mạch dây rốn	
O69.8	Chuyển dạ và đẻ kèm biến chứng khác của dây rốn	Dây rốn quấn quanh cổ không có chèn ép
O69.9	Chuyển dạ và đẻ kèm biến chứng khác của dây rốn, không xác định	
O70	Rách tầng sinh môn trong đẻ	
O70.0	Rách tầng sinh môn độ I trong đẻ	
O70.1	Rách tầng sinh môn độ II trong đẻ	
O70.2	Rách tầng sinh môn độ III trong đẻ	
O70.3	Rách tầng sinh môn độ IV trong đẻ	
O70.9	Rách tầng sinh môn trong đẻ, không xác định mức độ	
O71	Chấn thương sản khoa khác	
O71.0	Vỡ tử cung trước chuyển dạ	
O71.1	Vỡ tử cung trong chuyển dạ	Vỡ tử cung chưa xác định rõ là xảy ra trước chuyển dạ
O71.2	Lộn tử cung sau đẻ	
O71.3	Rách cổ tử cung sản khoa	Đứt rời cổ tử cung
O71.4	Rách âm đạo đoạn cao do sản khoa	
O71.5	Thương tổn tạng tiểu khung khác do sản khoa	
O71.6	Tổn thương khớp và/hoặc dây chằng vùng chậu do sản khoa	Rạn sụn trong khớp mu do sản khoa|Tổn thương xương cụt do sản khoa|Giãn khớp mu do chấn thương sản khoa
O71.7	Khối máu tụ tiểu khung sản khoa	
O71.8	Chấn thương sản khoa xác định khác	
O71.9	Chấn thương sản khoa, không xác định	
O72	Băng huyết sau đẻ	
O72.0	Băng huyết giai đoạn 3 (chuyển dạ)	
O72.1	Băng huyết ngay sau đẻ	
O72.2	Băng huyết muộn và/hoặc thứ phát sau đẻ	Xuất huyết liên quan đến sót một phần bánh rau hay màng rau|Sót phần thai hay phần phụ của thai không xác định khác sau đẻ
O72.3	Rối loạn đông máu sau đẻ	
O73	Sót rau thai và/hoặc màng rau thai không có băng huyết	
O73.0	Sót rau thai không có băng huyết	
O73.1	Sót phần rau thai và/hoặc màng rau thai, không có băng huyết	Sót phần thai hay phần phụ của thai sau đẻ, không có xuất huyết
O74	Biến chứng do gây mê trong chuyển dạ và đẻ	
O74.0	Viêm phổi hít do gây mê trong chuyển dạ và đẻ	Viêm phổi hít hóa chất do gây mê trong chuyển dạ và đẻ|Hội chứng Mendelson do gây mê trong chuyển dạ và đẻ|Xẹp phổi do tràn khí màng phổi
O74.1	Biến chứng khác ở phổi do gây mê trong chuyển dạ và đẻ	Xẹp phổi do gây mê trong chuyển dạ và đẻ
O74.2	Biến chứng tim do gây mê trong chuyển dạ và đẻ	
O74.3	Biến chứng hệ thống thần kinh trung ương do gây mê trong chuyển dạ và đẻ	Thiếu máu não do gây mê trong chuyển dạ và đẻ
O74.4	Phản ứng độc thuốc gây tê tại chỗ trong chuyển dạ và đẻ	
O74.5	Đau đầu do gây tê tủy sống và/hoặc ngoài màng cứng trong chuyển dạ và đẻ	
O74.6	Biến chứng khác của gây tê tủy sống và/hoặc gây tê ngoài màng cứng trong chuyển dạ và đẻ	
O74.7	Không đặt được hay khó đặt ống nội khí quản trong chuyển dạ và đẻ	
O74.8	Biến chứng khác do gây mê trong chuyển dạ và đẻ	
O74.9	Biến chứng do gây mê trong chuyển dạ và đẻ, không xác định	
O75	Biến chứng khác của chuyển dạ và đẻ, không phân loại mục khác	
O75.0	Tình trạng nguy cấp của sản phụ trong chuyển dạ và đẻ	
O75.1	Sốc trong hay sau chuyển dạ và đẻ	Sốc sản khoa
O75.2	Sốt trong khi chuyển dạ, không phân loại mục khác	
O75.3	Nhiễm trùng khác trong khi chuyển dạ	Nhiễm trùng hệ thống trong khi chuyển dạ
O75.4	Biến chứng khác của can thiệp sản khoa	
O75.5	Cuộc đẻ đình trệ sau khi bấm ối	
O75.6	Cuộc đẻ đình trệ sau khi vỡ ối tự nhiên hoặc vỡ ối không xác định	
O75.7	Đẻ đường âm đạo sau mổ lấy thai cũ	
O75.8	Biến chứng xác định khác của chuyển dạ và đẻ	
O75.9	Biến chứng của chuyển dạ và đẻ, không xác định	
O80	Đẻ thường một thai	
O80.0	Đẻ thường một thai ngôi đầu	
O80.1	Đẻ thường một thai ngôi mông	
O80.8	Đẻ thường một thai khác	
O80.9	Đẻ thường một thai, không xác định	Đẻ thường tự nhiên không xác định khác
O81	Đẻ một thai bằng forcep hoặc giác hút	
O81.0	Đẻ một thai đặt forcep thấp	
O81.1	Đẻ một thai đặt forcep trung bình	
O81.2	Đẻ một thai đặt forcep trung bình có quay	
O81.3	Đẻ một thai đặt forcep ở vị trí khác và/hoặc không xác định	
O81.4	Đẻ một thai đặt giác hút	Đặt giác hút
O81.5	Đẻ một thai đặt phối hợp cả forcep và giác hút	Đặt forcep và giác hút
O82	Mổ lấy thai cho một thai	
O82.0	Mổ lấy thai chủ động	Mổ lấy thai lại không xác định khác
O82.1	Mổ lấy thai cấp cứu	
O82.2	Mổ lấy thai và cắt bỏ tử cung	
O82.8	Mổ lấy thai khác cho một thai	
O82.9	Mổ lấy thai, không xác định	
O83	Đẻ một thai có can thiệp hỗ trợ khác	
O83.0	Kéo thai trong ngôi mông	
O83.1	Hỗ trợ đẻ khác trong ngôi mông	Đẻ ngôi mông không xác định khác
O83.2	Đẻ có hỗ trợ khác bằng tay	Xoay thai kèm kéo thai
O83.3	Đẻ thai lạc chỗ còn sống [phát triển] trong ổ bụng	
O83.4	Can thiệp hủy thai để hỗ trợ đẻ thai chết lưu qua âm đạo	Thủ thuật rạch phá xương đòn để hỗ trợ đẻ thai chết lưu|Thủ thuật mở nắp sọ để hỗ trợ đẻ thai chết lưu|Thủ thuật cắt thai để hỗ trợ đẻ thai chết lưu
O83.8	Can thiệp xác định khác hỗ trợ đẻ một thai	
O83.9	Can thiệp hỗ trợ đẻ một thai, không xác định	Thủ thuật trong đẻ không xác định khác
O84	Đẻ đa thai	
O84.0	Đẻ đa thai, tất cả đẻ tự nhiên	
O84.1	Đẻ đa thai, tất cả đẻ bằng forcep hoặc giác hút	
O84.2	Đẻ đa thai, tất cả đẻ bằng mổ lấy thai	
O84.8	Đẻ đa thai bằng phương pháp khác	Đẻ nhiều con bằng kết hợp phương pháp
O84.9	Đẻ đa thai, không xác định phương pháp	
O85	Nhiễm trùng hệ thống trong thời kỳ sau đẻ	
O86	Nhiễm trùng khác trong thời kỳ sau đẻ	
O86.0	Nhiễm trùng vết thương do phẫu thuật sản khoa	
O86.1	Nhiễm trùng đường sinh dục khác sau đẻ	Viêm cổ tử cung sau đẻ|Viêm âm đạo sau đẻ
O86.2	Nhiễm trùng đường tiết niệu sau đẻ	Bệnh lý phân loại ở N10-N12, N15.-, N30.-, N34.-, N39.0 sau đẻ
O86.3	Nhiễm trùng khác của đường tiết niệu sinh dục trong thời kỳ sau đẻ	Nhiễm trùng đường tiết niệu sinh dục trong thời kỳ sau đẻ không xác định khác
O86.4	Sốt không rõ nguyên nhân sau đẻ	
O86.8	Nhiễm trùng xác định khác trong thời kỳ sau đẻ	
O87	Biến chứng tĩnh mạch và/hoặc trĩ trong thời kỳ sau đẻ	
O87.0	Viêm [tắc] tĩnh mạch huyết khối nông trong thời kỳ sau đẻ	
O87.1	Huyết khối tĩnh mạch sâu trong thời kỳ sau đẻ	Huyết khối tĩnh mạch sâu sau đẻ|Viêm tĩnh mạch huyết khối tiểu khung sau đẻ
O87.2	Trĩ trong thời kỳ sau đẻ	
O87.3	Huyết khối tĩnh mạch não trong thời kỳ sau đẻ	Huyết khối xoang tĩnh mạch não trong thời kỳ sau đẻ
O87.8	Biến chứng tĩnh mạch khác trong thời kỳ sau đẻ	Giãn tĩnh mạch cơ quan sinh dục trong thời kỳ sau đẻ
O87.9	Biến chứng tĩnh mạch trong thời kỳ sau đẻ, không xác định	
O88	Thuyên tắc mạch sản khoa	
O88.0	Thuyên tắc [tắc mạch máu] khí sản khoa	
O88.1	Thuyên tắc [tắc mạch máu] ối	Hội chứng phản vệ khi mang thai
O88.2	Thuyên tắc mạch sản khoa do cục máu đông	
O88.3	Thuyên tắc mạch do mủ huyết và/hoặc nhiễm trùng sản khoa	
O88.8	Thuyên tắc mạch sản khoa khác	Thuyên tắc [tắc mạch máu] mỡ sản khoa
O89	Biến chứng do gây mê trong thời kỳ sau đẻ	
O89.0	Biến chứng phổi do gây mê trong thời kỳ sau đẻ	Viêm phổi hít do gây mê trong thời kỳ sau đẻ|Viêm phổi hít phải hóa chất gây mê trong thời kỳ sau đẻ|Hội chứng Mendelson do gây mê trong thời kỳ sau đẻ|Xẹp phổi do gây mê trong thời kỳ sau đẻ
O89.1	Biến chứng tim do gây mê trong thời kỳ sau đẻ	
O89.2	Biến chứng hệ thống thần kinh trung ương do gây mê trong thời kỳ sau đẻ	Thiếu oxy não do gây mê trong thời kỳ sau đẻ
O89.3	Phản ứng độc do gây tê tại chỗ trong thời kỳ sau đẻ	
O89.4	Đau đầu do gây tê tủy sống và/hoặc ngoài màng cứng trong thời kỳ sau đẻ	
O89.5	Biến chứng khác của gây tê tủy sống và/hoặc ngoài màng cứng trong thời kỳ sau đẻ	
O89.6	Đặt nội khí quản khó hoặc thất bại trong thời kỳ sau đẻ	
O89.8	Biến chứng khác do gây mê trong thời kỳ sau đẻ	
O89.9	Biến chứng do gây mê trong thời kỳ sau đẻ, không xác định	
O90	Biến chứng trong thời kỳ sau đẻ, không phân loại mục khác	
O90.0	Toác vết mổ lấy thai	
O90.1	Toác vết khâu tầng sinh môn	
O90.2	Khối máu tụ ở vết khâu sản khoa	
O90.3	Bệnh lý cơ tim trong thời kỳ sau đẻ	Bệnh lý phân loại ở I42.
O90.4	Suy thận cấp tính sau đẻ	Hội chứng gan-thận sau chuyển dạ và đẻ
O90.5	Viêm tuyến giáp sau đẻ	
O90.8	Biến chứng khác trong thời kỳ sau đẻ, không phân loại mục khác	Polyp rau thai
O90.9	Biến chứng trong thời kỳ sau đẻ, không xác định	
O91	Nhiễm trùng vú liên quan đến sinh đẻ	
O91.0	Nhiễm trùng núm vú liên quan đến sinh đẻ	
O91.1	Áp xe vú liên quan đến sinh đẻ	Áp xe tuyến vú trong thai kỳ hoặc sau đẻ|Viêm vú có mủ trong thai kỳ hoặc sau đẻ|Áp xe dưới quầng vú trong thai kỳ hoặc sau đẻ
O91.2	Viêm vú không thành mủ liên quan đến sinh đẻ	Viêm hạch ở vú trong thai kỳ hoặc sau đẻ
O92	Rối loạn khác của vú và/hoặc tiết sữa liên quan đến sinh đẻ	
O92.0	Tụt núm vú liên quan đến sinh đẻ	
O92.1	Nứt đầu vú liên quan đến sinh đẻ	Nứt đầu vú trong thai kỳ hoặc sau đẻ
O92.2	Rối loạn khác và/hoặc không xác định của vú liên quan đến sinh đẻ	
O92.3	Không có sữa	Không có sữa nguyên phát
O92.4	Thiếu sữa, ít sữa	
O92.5	Cắt sữa	
O92.6	Tiết sữa	
O92.7	Rối loạn tiết sữa khác và/hoặc không xác định	Nang sữa sau đẻ
O94	Di chứng của biến chứng thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O95	Tử vong sản khoa không rõ nguyên nhân	
O96	Tử vong vì bất kỳ nguyên nhân sản khoa nào xảy ra trên 42 ngày và dưới 1 năm sau đẻ	
O96.0	Tử vong do nguyên nhân sản khoa trực tiếp xảy ra trên 42 ngày nhưng dưới một năm sau đẻ	
O96.1	Tử vong do nguyên nhân sản khoa gián tiếp xảy ra trên 42 ngày nhưng dưới một năm sau đẻ	
O96.9	Tử vong do nguyên nhân sản khoa không xác định xảy ra hơn 42 ngày nhưng dưới một năm sau đẻ	
O97	Tử vong vì di chứng có nguyên nhân sản khoa	
O97.0	Tử vong vì di chứng có nguyên nhân sản khoa trực tiếp	
O97.1	Tử vong vì di chứng có nguyên nhân sản khoa gián tiếp	
O97.9	Tử vong vì di chứng có nguyên nhân sản khoa, không xác định	
O98	Bệnh nhiễm trùng và/hoặc ký sinh trùng ở người mẹ phân loại mục khác có gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O98.0	Bệnh lao gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở A15.- - A19.
O98.1	Giang mai gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở A50.- - A53.
O98.2	Bệnh lậu gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở A54.
O98.3	Nhiễm trùng khác ở người mẹ chủ yếu lây truyền qua đường tình dục gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở A55-A64
O98.4	Viêm gan do virus gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở B15.- - B19.
O98.5	Bệnh do virus khác gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở A80.- -B09, B25.- - B34.
O98.6	Bệnh do đơn bào gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở B50.- - B64
O98.7	Bệnh do virus gây suy giảm miễn dịch ở người [HIV] gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O98.8	Bệnh nhiễm trùng khác ở người mẹ gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O98.9	Bệnh nhiễm trùng và/hoặc ký sinh trùng không xác định ở người mẹ có gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99	Bệnh thai sản khác phân loại mục khác có gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.0	Thiếu máu gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở D50.- - D64.
O99.1	Bệnh khác của máu, cơ quan tạo máu và/hoặc rối loại cơ chế miễn dịch gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.2	Bệnh nội tiết, dinh dưỡng và/hoặc chuyển hóa gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.3	Rối loạn tâm thần và/hoặc bệnh lý hệ thần kinh gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.4	Bệnh của hệ tuần hoàn gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.5	Bệnh của hệ hô hấp gây biến chứng thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	Bệnh lý phân loại ở J00-J99.
O99.6	Bệnh của hệ tiêu hóa gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.7	Bệnh của da và/hoặc mô dưới da gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
O99.8	Bệnh và bệnh lý xác định khác gây biến chứng cho thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
P00	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý của mẹ có thể không liên quan đến lần mang thai hiện nay	
P00.0	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do rối loạn tăng huyết áp của mẹ	
P00.1	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh thận và/hoặc bệnh đường tiết niệu của mẹ	
P00.2	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh nhiễm trùng và/hoặc ký sinh trùng của mẹ	
P00.3	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh khác về tuần hoàn và/hoặc hô hấp của mẹ	
P00.4	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do rối loạn dinh dưỡng của mẹ	Thai nhi hoặc trẻ sơ sinh bị ảnh hưởng do các rối loạn của mẹ có thể phân loại ở E40-E64.|Suy dinh dưỡng của mẹ không xác định khác.
P00.5	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do người mẹ bị chấn thương	
P00.6	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do can thiệp ở người mẹ	
P00.7	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do can thiệp khác ở người mẹ, không phân loại mục khác	
P00.8	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý khác của mẹ	
P00.9	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý không xác định của mẹ	
P01	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng ở thai phụ trong thai kỳ	
P01.0	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do hở eo cổ tử cung	
P01.1	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do vỡ ối sớm	
P01.2	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do thiểu ối	
P01.3	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đa ối	Đa ối
P01.4	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do thai lạc chỗ	Có thai ngoài tử cung
P01.5	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đa thai	Mang thai ba|Mang thai đôi
P01.6	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do người mẹ tử vong	
P01.7	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do ngôi bất thường trước khi chuyển dạ	Ngôi mông trước chuyển dạ|Ngoại xoay thai trước chuyển dạ|Ngôi mặt trước chuyển dạ|Ngôi ngang trước chuyển dạ|Ngôi thai không ổn định trước chuyển dạ
P01.8	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng khác ở người mẹ trong thai kỳ	Sảy thai tự nhiên
P01.9	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng ở người mẹ trong thai kỳ, không xác định	
P02	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng của rau thai, dây rốn và/hoặc màng thai	
P02.0	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do rau tiền đạo	
P02.1	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng bong rau và/hoặc xuất huyết khác	Rau thai bong non|Xuất huyết do tai nạn|Xuất huyết sau đẻ|Tổn thương rau thai do chọc dò ối, mổ lấy thai hoặc bấm ối khởi phạt chuyển dạ|Mất máu ở mẹ|Tách rau thai sớm
P02.2	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bất thường về hình thái và/hoặc chức năng của rau thai khác và/hoặc không xác định	
P02.3	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do hội chứng truyền máu trong rau thai	
P02.4	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do sa dây rốn	
P02.5	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do chèn ép khác của dây rốn	
P02.6	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý khác và/hoặc không xác định của dây rốn	
P02.7	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do viêm màng ối	Viêm ối|Viêm màng ối|Viêm bánh rau
P02.8	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bất thường khác của màng thai	
P02.9	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do bất thường của màng thai, không xác định	
P03	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng của chuyển dạ và đẻ	
P03.0	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đẻ ngôi mông và/hoặc kéo thai ngôi mông	
P03.1	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do ngôi, thế bất thường, bất tương xứng trong chuyển dạ và đẻ	Hẹp khung chậu|Thai nhi hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý phân loại ở O64.- - O66.|Kiểu thế chẩm sau không xoay|Ngôi ngang
P03.2	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đẻ bằng forcep	
P03.3	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đẻ bằng giác hút	
P03.4	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mổ lấy thai	
P03.5	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do đẻ nhanh	Giai đoạn 2 nhanh
P03.6	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do cơn co tử cung bất thường	Thai nhi hoặc trẻ sơ sinh bị ảnh hưởng do bệnh lý phân loại ở O62.- trừ O62.3|Chuyển dạ khi tử cung tăng trương lực cơ [cơn co mau mạnh]|Tử cung giảm trương lực cơ
P03.8	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng xác định khác của chuyển dạ và đẻ	
P03.9	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do biến chứng của chuyển dạ và đẻ, không xác định	
P04	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do chất độc hại truyền qua rau thai hoặc qua sữa mẹ	
P04.0	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do thuốc gây mê và/hoặc gây tê cho mẹ trong thai kỳ, chuyển dạ và đẻ	
P04.1	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ dùng thuốc khác	
P04.2	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ sử dụng thuốc lá	
P04.3	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ dùng rượu	
P04.4	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ sử dụng chất gây nghiện	
P04.5	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ sử dụng chất hóa học dinh dưỡng	
P04.6	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do mẹ phơi nhiễm với hóa chất trong môi trường	
P04.8	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do chất độc hại khác truyền từ mẹ	
P04.9	Thai nhi và/hoặc trẻ sơ sinh bị ảnh hưởng do chất độc hại truyền từ mẹ, không xác định	
P05	Thai nhi chậm phát triển và/hoặc thai suy dinh dưỡng	
P05.0	Nhẹ cân so với tuổi thai	Thường dành cho nhẹ cân nhưng chiều dài ở trên bách phân vị thứ 10 so với tuổi thai|Thai nhẹ hơn tuổi
P05.1	Nhỏ so với tuổi thai	Thường dành cho trẻ có cân nặng và chiều dài dưới bách phân vị thứ 10 so với tuổi thai.|Nhỏ và nhẹ so với tuổi thai
P05.2	Thai suy dinh dưỡng mà không đề cập đến nhỏ hoặc nhẹ cân so với tuổi thai	
P05.9	Thai chậm phát triển, không xác định	Thai nhi chậm phát triển không xác định khác
P07	Rối loạn liên quan đến sinh non và/hoặc nhẹ cân lúc sinh, không phân loại mục khác	
P07.0	Trẻ có cân nặng cực thấp khi sinh	Cân nặng khi đẻ từ 999g trở xuống.
P07.1	Trẻ sơ sinh nhẹ cân khác	Cân nặng khi đẻ từ 1000g đến 2499g.
P07.2	Trẻ cực non	
P07.3	Trẻ sinh non khác	
P08	Rối loạn liên quan đến thai già tháng và/hoặc trẻ nặng cân khi sinh	
P08.0	Trẻ nặng cân	
P08.1	Trẻ nặng cân so với tuổi thai khác	
P08.2	Trẻ già tháng nhưng không nặng so với tuổi thai	
P10	Rách và/hoặc xuất huyết nội sọ do chấn thương khi sinh	
P10.0	Xuất huyết dưới màng cứng do chấn thương khi sinh	
P10.1	Xuất huyết não do chấn thương khi sinh	
P10.2	Xuất huyết trong não thất do chấn thương khi sinh	
P10.3	Xuất huyết dưới màng nhện do chấn thương khi sinh	
P10.4	Rách lều não do chấn thương khi sinh	
P10.8	Rách và/hoặc xuất huyết nội sọ khác do chấn thương khi sinh	
P10.9	Rách và/hoặc xuất huyết nội sọ do chấn thương khi sinh, không xác định	
P11	Chấn thương khi sinh khác ở hệ thần kinh trung ương	
P11.0	Phù não do chấn thương khi sinh	
P11.1	Tổn thương não khác được xác định do chấn thương khi sinh	
P11.2	Tổn thương não không xác định, do chấn thương khi sinh	
P11.3	Tổn thương thần kinh mặt do chấn thương khi sinh	Liệt mặt do chấn thương khi sinh
P11.4	Tổn thương dây thần kinh sọ khác do chấn thương khi sinh	
P11.5	Tổn thương cột sống và/hoặc tủy sống do chấn thương khi sinh	Gãy cột sống do chấn thương khi sinh
P11.9	Tổn thương hệ thần kinh trung ương do chấn thương khi sinh, không xác định	
P12	Tổn thương da đầu khi sinh	
P12.0	Khối máu tụ dưới cốt mạc do chấn thương khi sinh	
P12.1	Bong da đầu do chấn thương khi sinh	
P12.2	Chảy máu dưới cân ngoài sọ do chấn thương khi sinh	Tụ máu dưới cân Galea do chấn thương khi sinh
P12.3	Bầm tím da đầu do chấn thương khi sinh	
P12.4	Tổn thương da đầu trẻ sơ sinh do máy monitor	
P12.8	Tổn thương da đầu khi sinh khác	
P12.9	Tổn thương da đầu khi sinh, không xác định	
P13	Chấn thương hệ xương khi sinh	
P13.0	Vỡ xương sọ do chấn thương khi sinh	
P13.1	Chấn thương khi sinh khác lên xương sọ	
P13.2	Chấn thương xương đùi khi sinh	
P13.3	Chấn thương xương dài khác khi sinh	
P13.4	Gãy xương đòn do chấn thương khi sinh	
P13.8	Chấn thương xương khác khi sinh	
P13.9	Chấn thương hệ xương khi sinh, không xác định	
P14	Chấn thương hệ thần kinh ngoại biên khi sinh	
P14.0	Liệt Erb do chấn thương khi sinh	
P14.1	Liệt Klumpke do chấn thương khi sinh	
P14.2	Liệt dây thần kinh hoành do chấn thương khi sinh	
P14.3	Chấn thương khác của đám rối cánh tay khi sinh	
P14.8	Chấn thương phần khác của hệ thần kinh ngoại biên khi sinh	
P14.9	Chấn thương hệ thần kinh ngoại biên khi sinh, không xác định	
P15	Chấn thương khác khi sinh	
P15.0	Chấn thương gan khi sinh	Vỡ gan do chấn thương khi sinh
P15.1	Chấn thương lách khi sinh	Vỡ lách do chấn thương khi sinh
P15.2	Chấn thương cơ ức - đòn - chũm khi sinh	
P15.3	Chấn thương mắt khi sinh	Xuất huyết dưới kết mạc do chấn thương khi sinh|Glôcôm do chấn thương khi sinh
P15.4	Chấn thương mặt khi sinh	Xung huyết mặt do chấn thương khi sinh
P15.5	Chấn thương bộ phận sinh dục ngoài khi sinh	
P15.6	Hoại tử tổ chức mỡ dưới da do chấn thương khi sinh	
P15.8	Chấn thương xác định khác khi sinh	
P15.9	Chấn thương khi sinh, không xác định	
P20	Hạ oxy huyết trong tử cung	
P20.0	Hạ oxy máu của thai nhi trong tử cung từ trước khi bắt đầu chuyển dạ	
P20.1	Hạ oxy máu của thai nhi trong tử cung trong chuyển dạ và đẻ	
P20.9	Hạ oxy máu của thai nhi trong tử cung, không xác định	
P21	Ngạt trong khi sinh	
P21.0	Ngạt nặng trong khi sinh	Chỉ số Apgar phút thứ nhất: từ 0 đến 3|Ngạt trắng
P21.1	Ngạt nhẹ và/hoặc ngạt trung bình trong khi sinh	Ngạt thở với chỉ số Apgar phút thứ nhất: 4-7|Ngạt tím
P21.9	Ngạt trong khi sinh, không xác định	Thiếu oxy không xác định khác|Ngạt không xác định khác|Giảm oxy không xác định khác
P22	Suy hô hấp ở trẻ sơ sinh	
P22.0	Hội chứng suy hô hấp ở trẻ sơ sinh	Bệnh màng trong [Hyaline]
P22.1	Nhịp thở nhanh thoáng qua ở trẻ sơ sinh [chậm tiêu dịch phổi sau sinh]	
P22.8	Suy hô hấp khác ở trẻ sơ sinh	
P22.9	Suy hô hấp ở trẻ sơ sinh, không xác định	
P23	Viêm phổi bẩm sinh	
P23.0	Viêm phổi bẩm sinh do tác nhân virus	
P23.1	Viêm phổi bẩm sinh do nhiễm chlamydia	
P23.2	Viêm phổi bẩm sinh do nhiễm tụ cầu khuẩn	
P23.3	Viêm phổi bẩm sinh do nhiễm liên cầu khuẩn, nhóm B	
P23.4	Viêm phổi bẩm sinh do nhiễm E Coli	
P23.5	Viêm phổi bẩm sinh do nhiễm Pseudomonas	
P23.6	Viêm phổi bẩm sinh do tác nhân vi khuẩn khác	
P23.8	Viêm phổi bẩm sinh do vi sinh vật khác	
P23.9	Viêm phổi bẩm sinh, không xác định	
P24	Hội chứng hít phải ở trẻ sơ sinh	
P24.0	Trẻ sơ sinh hít phải phân su	
P24.1	Trẻ sơ sinh hít phải nước ối và/hoặc dịch nhầy	Hít nước ối
P24.2	Trẻ sơ sinh hít phải máu	
P24.3	Trẻ sơ sinh hít phải sữa và/hoặc thức ăn nôn trớ	
P24.8	Hội chứng hít phải khác ở trẻ sơ sinh	
P24.9	Hội chứng hít phải ở sơ sinh, không xác định	Viêm phổi hít [sặc] ở trẻ sơ sinh không xác định khác
P25	Tràn khí tổ chức kẽ phổi và/hoặc bệnh lý liên quan xuất phát trong thời kỳ chu sinh	
P25.0	Tràn khí tổ chức kẽ phổi xuất phát trong thời kỳ chu sinh	
P25.1	Tràn khí màng phổi trong thời kỳ chu sinh	
P25.2	Tràn khí trung thất trong thời kỳ chu sinh	
P25.3	Tràn khí màng tim trong thời kỳ chu sinh	
P25.8	Bệnh lý khác liên quan đến tràn khí tổ chức kẽ phổi trong thời kỳ chu sinh	
P26	Xuất huyết phổi trong thời kỳ chu sinh	
P26.0	Xuất huyết khí phế quản chu sinh	
P26.1	Xuất huyết phổi nặng trong thời kỳ chu sinh	
P26.8	Xuất huyết phổi khác trong thời kỳ chu sinh	
P26.9	Xuất huyết phổi trong thời kỳ chu sinh, không xác định	
P27	Bệnh hô hấp mạn tính xuất phát trong thời kỳ chu sinh	
P27.0	Hội chứng Wilson-Mikity	Rối loạn phổi non trong thai đủ tháng gây tràn khí sau sinh
P27.1	Loạn sản phế quản phổi xuất phát trong thời kỳ chu sinh	
P27.8	Bệnh phổi mạn tính khác xuất phát trong thời kỳ chu sinh	Xơ hóa phổi bẩm sinh|Bệnh phổi do thở máy ở trẻ sơ sinh
P27.9	Bệnh hô hấp mạn tính không xác định xuất phát trong thời kỳ chu sinh	
P28	Bệnh lý hô hấp khác xuất phát từ thời kỳ chu sinh	
P28.0	Xẹp phổi nguyên phát ở trẻ sơ sinh	
P28.1	Xẹp phổi khác và/hoặc không xác định ở trẻ sơ sinh	
P28.2	Cơn tím tái ở trẻ sơ sinh	
P28.3	Ngừng thở khi ngủ nguyên phát ở trẻ sơ sinh	
P28.4	Ngừng thở khác ở trẻ sơ sinh	
P28.5	Suy hô hấp [tiến triển] ở trẻ sơ sinh	
P28.8	Bệnh lý hô hấp xác định khác ở trẻ sơ sinh	
P28.9	Bệnh lý hô hấp ở trẻ sơ sinh, không xác định	
P29	Rối loạn tim mạch xuất phát trong thời kỳ chu sinh	
P29.0	Suy tim sơ sinh	
P29.1	Loạn nhịp tim sơ sinh	
P29.2	Tăng huyết áp sơ sinh	
P29.3	Tuần hoàn thai nhi dai dẳng	
P29.4	Thiếu máu cơ tim thoáng qua ở trẻ sơ sinh	
P29.8	Rối loạn tim mạch khác xuất phát trong thời kỳ chu sinh	
P29.9	Rối loạn tim mạch xuất phát trong thời kỳ chu sinh, không xác định	
P35	Bệnh nhiễm virus bẩm sinh	
P35.0	Hội chứng rubella bẩm sinh	Viêm phổi bẩm sinh do rubella
P35.1	Nhiễm virus đại bào [cytomegalovirus-CMV] bẩm sinh	
P35.2	Nhiễm virus herpes [herpes simplex] bẩm sinh	
P35.3	Viêm gan virus bẩm sinh	
P35.4	Bệnh do nhiễm virus Zika bẩm sinh	Tật đầu [não] nhỏ bẩm sinh do bệnh virus Zika
P35.8	Bệnh nhiễm virus bẩm sinh khác	Bệnh thủy đậu [chickenpox] bẩm sinh
P35.9	Bệnh virus bẩm sinh, không xác định	
P36	Nhiễm khuẩn hệ thống ở trẻ sơ sinh	
P36.0	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm liên cầu khuẩn, nhóm B	
P36.1	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm liên cầu khuẩn khác và/hoặc không xác định	
P36.2	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm tụ cầu vàng	
P36.3	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm tụ cầu khuẩn khác và/hoặc không xác định	
P36.4	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm E. Coli	
P36.5	Nhiễm khuẩn hệ thống ở trẻ sơ sinh do nhiễm vi khuẩn kỵ khí	
P36.8	Nhiễm khuẩn hệ thống khác ở trẻ sơ sinh	
P36.9	Nhiễm khuẩn hệ thống ở trẻ sơ sinh, không xác định	
P37	Bệnh nhiễm trùng và/hoặc ký sinh trùng bẩm sinh khác	
P37.0	Bệnh lao bẩm sinh	
P37.1	Bệnh do nhiễm toxoplasma bẩm sinh	Bệnh não úng thủy do nhiễm toxoplasma bẩm sinh
P37.2	Bệnh do nhiễm listeria (lan tỏa) sơ sinh	
P37.3	Bệnh sốt rét do nhiễm falciparum bẩm sinh	
P37.4	Bệnh sốt rét bẩm sinh khác	
P37.5	Bệnh do nhiễm nấm candida sơ sinh	
P37.8	Bệnh nhiễm trùng và/hoặc ký sinh trùng bẩm sinh xác định khác	
P37.9	Bệnh nhiễm trùng và/hoặc ký sinh trùng bẩm sinh, không xác định	
P38	Viêm rốn ở trẻ sơ sinh có hoặc không có chảy máu mức độ nhẹ	
P39	Nhiễm trùng khác đặc trưng của thời kỳ chu sinh	
P39.0	Viêm vú nhiễm khuẩn sơ sinh	
P39.1	Viêm kết mạc và/hoặc túi lệ sơ sinh	
P39.2	Nhiễm trùng thai trong buồng ối, không phân loại mục khác	
P39.3	Nhiễm trùng đường tiết niệu sơ sinh	
P39.4	Nhiễm trùng da sơ sinh	
P39.8	Nhiễm trùng xác định khác đặc trưng của thời kỳ chu sinh	
P39.9	Nhiễm trùng đặc trưng của thời kỳ chu sinh, không xác định	
P50	Mất máu thai nhi	
P50.0	Mất máu thai nhi do mạch máu dây rốn tiền đạo	
P50.1	Mất máu thai nhi do đứt dây rốn	
P50.2	Mất máu thai nhi từ bánh rau	
P50.3	Truyền máu song thai	
P50.4	Xuất huyết vào tuần hoàn người mẹ	
P50.5	Mất máu thai nhi do cắt vào dây rốn chung song thai	
P50.8	Mất máu thai nhi khác	
P50.9	Mất máu thai nhi, không xác định	Xuất huyết bào thai không xác định khác
P51	Xuất huyết rốn ở trẻ sơ sinh	
P51.0	Xuất huyết rốn mức độ nặng ở trẻ sơ sinh	
P51.8	Xuất huyết rốn khác ở trẻ sơ sinh	Tuột nút buộc rốn không xác định khác
P51.9	Xuất huyết rốn ở trẻ sơ sinh, không xác định	
P52	Xuất huyết nội sọ không do chấn thương ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.0	Xuất huyết trong não thất (không do chấn thương), độ 1, ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.1	Xuất huyết trong não thất (không do chấn thương), độ 2, ở thai nhi và/hoặc ở trẻ sơ sinh	Xuất huyết dưới màng não thất lan vào trong não thất
P52.2	Xuất huyết trong não thất (không do chấn thương), độ 3, và/hoặc độ 4 ở thai nhi và/hoặc ở trẻ sơ sinh	Xuất huyết dưới màng não thất lan vào trong não thất và vào trong não
P52.3	Xuất huyết trong não thất (không do chấn thương) không xác định ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.4	Xuất huyết trong não (không do chấn thương) ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.5	Xuất huyết dưới màng nhện (không do chấn thương) ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.6	Xuất huyết tiểu não (không do chấn thương) và/hoặc hố sau ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.8	Xuất huyết nội sọ khác (không do chấn thương) ở thai nhi và/hoặc ở trẻ sơ sinh	
P52.9	Xuất huyết nội sọ (không do chấn thương) ở thai nhi và/hoặc ở trẻ sơ sinh, không xác định	
P53	Bệnh xuất huyết của thai nhi và/hoặc trẻ sơ sinh	
P54	Xuất huyết khác ở trẻ sơ sinh	
P54.0	Nôn ra máu ở trẻ sơ sinh	
P54.1	Xuất huyết tiêu hóa [phân máu] ở trẻ sơ sinh	
P54.2	Xuất huyết trực tràng ở trẻ sơ sinh	
P54.3	Xuất huyết dạ dày - ruột khác ở trẻ sơ sinh	
P54.4	Xuất huyết thượng thận ở trẻ sơ sinh	
P54.5	Xuất huyết dưới da ở trẻ sơ sinh	
P54.6	Xuất huyết âm đạo ở trẻ sơ sinh	Kinh nguyệt giả
P54.8	Xuất huyết sơ sinh xác định khác	
P54.9	Xuất huyết ở trẻ sơ sinh, không xác định	
P55	Bệnh tan máu ở thai nhi và/hoặc ở trẻ sơ sinh	
P55.0	Tan máu do bất đồng nhóm máu Rh ở thai nhi và/hoặc ở trẻ sơ sinh	
P55.1	Tan máu do bất đồng nhóm máu ABO ở thai nhi và/hoặc ở trẻ sơ sinh	
P55.8	Bệnh tan máu khác ở thai nhi và/hoặc ở trẻ sơ sinh	
P55.9	Bệnh tan máu của thai nhi và/hoặc trẻ sơ sinh, không xác định	
P56	Phù thai do bệnh tan máu	
P56.0	Phù thai do miễn dịch đồng loại	
P56.9	Phù thai do bệnh lý tan máu khác và/hoặc không xác định	
P57	Vàng da nhân xám	
P57.0	Vàng da nhân xám do bất đồng miễn dịch	
P57.8	Vàng da nhân xám xác định khác	
P57.9	Vàng da nhân xám, không xác định	
P58	Vàng da sơ sinh do tan máu quá mức	
P58.0	Vàng da sơ sinh do bầm tím	
P58.1	Vàng da sơ sinh do xuất huyết	
P58.2	Vàng da sơ sinh do nhiễm trùng	
P58.3	Vàng da sơ sinh do đa hồng cầu	
P58.4	Vàng da sơ sinh do thuốc hoặc độc tố truyền từ mẹ hoặc dùng cho trẻ sơ sinh	
P58.5	Vàng da sơ sinh do nuốt phải máu mẹ	
P58.8	Vàng da sơ sinh do tan máu quá mức xác định khác	
P58.9	Vàng da sơ sinh do tan máu quá mức, không xác định	
P59	Vàng da sơ sinh do các nguyên nhân khác và/hoặc không xác định	
P59.0	Vàng da sơ sinh có liên quan đến đẻ non	Tăng bilirubin huyết ở trẻ sinh non|Vàng da do liên hợp bilirubin chậm liên quan đến sinh non
P59.1	Hội chứng mật đặc	
P59.2	Vàng da sơ sinh vì các tổn thương tế bào gan khác và/hoặc không xác định	
P59.3	Vàng da sơ sinh do ức chế sữa mẹ	
P59.8	Vàng da sơ sinh vì nguyên nhân xác định khác	
P59.9	Vàng da sơ sinh, không xác định	
P60	Bệnh đông máu rải rác nội mạch của thai nhi và/hoặc trẻ sơ sinh	
P61	Rối loạn huyết học chu sinh khác	
P61.0	Giảm tiểu cầu thoáng qua ở trẻ sơ sinh	
P61.1	Bệnh đa hồng cầu ở trẻ sơ sinh	
P61.2	Thiếu máu ở trẻ non tháng	
P61.3	Thiếu máu bẩm sinh do mất máu thai nhi	
P61.4	Thiếu máu bẩm sinh khác, không phân loại mục khác	Thiếu máu bẩm sinh không xác định khác
P61.5	Bệnh giảm bạch cầu trung tính thoáng qua ở trẻ sơ sinh	
P61.6	Rối loạn đông máu khác thoáng qua ở trẻ sơ sinh	
P61.8	Rối loạn huyết học chu sinh xác định khác	
P61.9	Rối loạn huyết học chu sinh, không xác định	
P70	Rối loạn chuyển hóa carbon hydrat tạm thời đặc trưng ở thai nhi và/hoặc trẻ sơ sinh	
P70.0	Hội chứng ở trẻ sơ sinh của mẹ mắc đái tháo đường thai kỳ	
P70.1	Hội chứng ở trẻ sơ sinh của mẹ mắc đái tháo đường	
P70.2	Đái tháo đường ở trẻ sơ sinh	
P70.3	Hạ đường huyết ở trẻ sơ sinh do trị liệu	
P70.4	Hạ đường huyết khác ở trẻ sơ sinh	Hạ đường huyết thoáng qua ở trẻ sơ sinh
P70.8	Rối loạn tạm thời khác của chuyển hóa cacbon hydrat đặc trưng ở thai nhi và/hoặc trẻ sơ sinh	
P70.9	Rối loạn tạm thời của chuyển hóa cacbon hydrat đặc trưng ở thai nhi và/hoặc trẻ sơ sinh, không xác định	
P71	Rối loạn chuyển hóa calci và/hoặc magie tạm thời ở trẻ sơ sinh	
P71.0	Hạ calci huyết khi dùng sữa bò ở trẻ sơ sinh	
P71.1	Hạ calci huyết khác ở trẻ sơ sinh	
P71.2	Hạ magie máu ở trẻ sơ sinh	
P71.3	Cơn co thắt không tự chủ [tetany] ở trẻ sơ sinh không do thiếu calci và/hoặc magie	Cơn co thắt không tự chủ [tetany] ở trẻ sơ sinh không xác định khác
P71.4	Suy tuyến cận giáp tạm thời ở trẻ sơ sinh	
P71.8	Rối loạn tạm thời khác của chuyển hóa calci và/hoặc magie ở trẻ sơ sinh	
P71.9	Rối loạn tạm thời của chuyển hóa calci và/hoặc magie ở trẻ sơ sinh, không xác định	
P72	Rối loạn nội tiết tạm thời khác ở trẻ sơ sinh	
P72.0	Bướu giáp ở trẻ sơ sinh, không phân loại mục khác	Bướu giáp bẩm sinh tạm thời với chức năng bình thường
P72.1	Cường giáp tạm thời ở trẻ sơ sinh	Nhiễm độc tuyến giáp ở trẻ sơ sinh
P72.2	Rối loạn chức năng tuyến giáp tạm thời ở trẻ sơ sinh, không phân loại mục khác	Suy giáp tạm thời ở trẻ sơ sinh
P72.8	Rối loạn nội tiết tạm thời xác định khác ở trẻ sơ sinh	
P72.9	Rối loạn nội tiết tạm thời ở trẻ sơ sinh, không xác định	
P74	Rối loạn điện giải và/hoặc chuyển hóa tạm thời khác ở trẻ sơ sinh	
P74.0	Toan chuyển hóa muộn ở trẻ sơ sinh	
P74.1	Mất nước ở trẻ sơ sinh	
P74.2	Rối loạn cân bằng natri ở trẻ sơ sinh	
P74.3	Rối loạn cân bằng kali ở trẻ sơ sinh	
P74.4	Rối loạn điện giải tạm thời khác ở trẻ sơ sinh	
P74.5	Tăng tyrosin máu tạm thời ở trẻ sơ sinh	
P74.8	Rối loạn chuyển hóa tạm thời khác ở trẻ sơ sinh	
P74.9	Rối loạn chuyển hóa tạm thời ở trẻ sơ sinh, không xác định	
P75.*	Tắc ruột phân su do bệnh xơ nang (E84.1†)	
P76	Tắc nghẽn ruột non khác ở trẻ sơ sinh	
P76.0	Hội chứng nút phân su	Tắc ruột phân su trong các trường hợp không có xơ nang.
P76.1	Tắc nghẽn ruột tạm thời ở trẻ sơ sinh	
P76.2	Tắc nghẽn ruột non do sữa đông	
P76.8	Tắc nghẽn ruột non xác định khác ở trẻ sơ sinh	
P76.9	Tắc nghẽn ruột non ở trẻ sơ sinh, không xác định	
P77	Viêm ruột hoại tử ở thai nhi và/hoặc trẻ sơ sinh	
P78	Rối loạn khác của hệ tiêu hóa ở thời kỳ chu sinh	
P78.0	Thủng ruột non chu sinh	Viêm phúc mạc phân su
P78.1	Viêm phúc mạc khác ở trẻ sơ sinh	Viêm phúc mạc sơ sinh không xác định khác
P78.2	Nôn ra máu và/hoặc xuất huyết tiêu hóa [phân máu] do nuốt phải máu mẹ ở trẻ sơ sinh	
P78.3	Tiêu chảy ở trẻ sơ sinh không do nhiễm trùng	
P78.8	Rối loạn xác định khác của hệ tiêu hóa ở thời kỳ chu sinh	
P78.9	Rối loạn của hệ tiêu hóa ở thời kỳ chu sinh, không xác định	
P80	Hạ thân nhiệt ở trẻ sơ sinh	
P80.0	Hội chứng tổn thương do lạnh	
P80.8	Hạ thân nhiệt khác ở trẻ sơ sinh	Hạ thân nhiệt nhẹ ở trẻ sơ sinh
P80.9	Hạ thân nhiệt ở trẻ sơ sinh, không xác định	
P81	Rối loạn điều nhiệt khác ở trẻ sơ sinh	
P81.0	Tăng thân nhiệt sơ sinh do môi trường	
P81.8	Rối loạn điều nhiệt xác định khác ở trẻ sơ sinh	
P81.9	Rối loạn điều nhiệt ở trẻ sơ sinh, không xác định	Sốt ở trẻ sơ sinh không xác định khác
P83	Bệnh lý khác của da đặc trưng của thai nhi và/hoặc trẻ sơ sinh	
P83.0	Chứng phù cứng bì ở trẻ sơ sinh	
P83.1	Ban đỏ nhiễm độc ở trẻ sơ sinh	
P83.2	Phù thai không do bệnh tan máu	Phù thai không xác định khác
P83.3	Phù da khác và/hoặc không xác định ở thai nhi và/hoặc trẻ sơ sinh	
P83.4	Cương tức vú ở trẻ sơ sinh	Viêm vú không nhiễm khuẩn ở trẻ sơ sinh
P83.5	Tràn dịch màng tinh hoàn bẩm sinh	
P83.6	Polyp rốn ở trẻ sơ sinh	
P83.8	Bệnh lý xác định khác của hệ vỏ bọc [da] đặc trưng ở thai nhi và/hoặc trẻ sơ sinh	Hội chứng da màu đồng [sơ sinh]|Xơ cứng bì ở trẻ sơ sinh|Mày đay ở trẻ sơ sinh
P83.9	Bệnh lý hệ vỏ bọc [da] đặc trưng ở thai nhi và/hoặc trẻ sơ sinh, không xác định	
P90	Cơn co giật ở trẻ sơ sinh	
P91	Rối loạn khác của tình trạng não trẻ sơ sinh	
P91.0	Bệnh thiếu máu não ở trẻ sơ sinh	
P91.1	Nang quanh não thất mắc phải ở trẻ sơ sinh	
P91.2	Bệnh nhuyễn hóa [nhũn] chất trắng não ở trẻ sơ sinh	
P91.3	Não dễ kích thích ở trẻ sơ sinh	
P91.4	Não suy yếu [ngạt] ở trẻ sơ sinh	
P91.5	Hôn mê ở trẻ sơ sinh	
P91.6	Bệnh lý não do thiếu oxy thiếu máu cục bộ ở trẻ sơ sinh	
P91.7	Não úng thủy mắc phải ở trẻ sơ sinh	Não úng thủy sau xuất huyết ở trẻ sơ sinh
P91.8	Rối loạn xác định khác của tình trạng não trẻ sơ sinh	
P91.9	Rối loạn của tình trạng não trẻ sơ sinh, không xác định	
P92	Rối loạn ăn uống ở trẻ sơ sinh	
P92.0	Nôn ở trẻ sơ sinh	
P92.1	Trào ngược và/hoặc nhai lại ở trẻ sơ sinh	
P92.2	Ăn chậm ở trẻ sơ sinh	
P92.3	Ăn kém ở trẻ sơ sinh	
P92.4	Ăn quá mức ở trẻ sơ sinh	
P92.5	Rối loạn trong việc bú mẹ ở trẻ sơ sinh	
P92.8	Rối loạn ăn uống khác ở trẻ sơ sinh	
P92.9	Rối loạn ăn uông ở trẻ sơ sinh, không xác định	
P93	Phản ứng và/hoặc ngộ độc thuốc dùng cho thai nhi và/hoặc trẻ sơ sinh	
P94	Rối loạn trương lực cơ ở trẻ sơ sinh	
P94.0	Nhược cơ thoáng qua ở trẻ sơ sinh	
P94.1	Tăng trương lực cơ bẩm sinh	
P94.2	Giảm trương lực cơ bẩm sinh	Hội chứng giảm trương lực cơ sơ sinh không cụ thể
P94.8	Rối loạn trương lực cơ khác ở trẻ sơ sinh	
P94.9	Rối loạn trương lực cơ ở trẻ sơ sinh, không xác định	
P95	Thai chết không rõ nguyên nhân	
P96	Bệnh lý khác xuất phát trong thời kỳ chu sinh	
P96.0	Suy thận bẩm sinh	Tăng urê huyết ở trẻ sơ sinh
P96.1	Triệu chứng cai nghiện ở trẻ sơ sinh do mẹ sử dụng chất gây nghiện	
P96.2	Triệu chứng cai thuốc sử dụng điều trị trẻ sơ sinh	
P96.3	Giãn đường khớp sọ ở trẻ sơ sinh	Nhuyễn sọ ở trẻ sơ sinh
P96.4	Chấm dứt thai kỳ (thất bai), ảnh hưởng đến thai nhi và/hoặc trẻ sơ sinh	
P96.5	Biến chứng của can thiệp trong tử cung, không phân loại mục khác	
P96.8	Bệnh lý xác định khác xuất phát trong thời kỳ chu sinh	
P96.9	Bệnh lý xuất phát trong thời kỳ chu sinh, không xác định	Suy nhược bẩm sinh không xác định khác
Q00	Dị tật khuyết não và dị tật tương tự	
Q00.0	Dị tật khuyết não	
Q00.1	Dị tật tật nứt sọ - cột sống	
Q00.2	Dị tật thoát vị não chẩm	
Q01	Dị tật thoát vị não	
Q01.0	Dị tật thoát vị não thùy trán	
Q01.1	Dị tật thoát vị não qua vùng mũi trán	
Q01.2	Dị tật thoát vị não vùng chẩm	
Q01.8	Dị tật thoát vị não ở những vị trí khác	
Q01.9	Dị tật thoát vị não, không xác định	
Q02	Dị tật đầu nhỏ	
Q03	Bệnh não úng thủy bẩm sinh	
Q03.0	Dị tật kênh Sylvius	
Q03.1	Dị tật teo lỗ Magendie và/hoặc Luschka	Hội chứng Dandy-Walker
Q03.8	Bệnh não úng thủy bẩm sinh khác	
Q03.9	Bệnh não úng thủy bẩm sinh, không xác định	
Q04	Dị tật bẩm sinh khác của não	
Q04.0	Dị tật bẩm sinh thể chai	Bất sản thể chai
Q04.1	Dị tật không khứu não	
Q04.2	Dị tật toàn bộ não trước	
Q04.3	Dị tật khuyết thiếu khác của não	
Q04.4	Loạn sản vách ngăn thần kinh thị giác	
Q04.5	Dị tật não to	
Q04.6	Nang não bẩm sinh	
Q04.8	Dị tật bẩm sinh xác định khác của não	Hồi não to
Q04.9	Dị tật bẩm sinh não, không xác định	
Q05	Dị tật bẩm sinh nứt đốt sống [cột sống chẻ đôi] [gai đôi cột sống]	
Q05.0	Dị tật nứt đốt sống cổ kết hợp bệnh não úng thủy	
Q05.1	Dị tật nứt đốt sống ngực kết hợp bệnh não úng thủy	
Q05.2	Dị tật nứt đốt sống thắt lưng kết hợp não úng thủy	Nứt đốt sống thắt lưng - cùng cụt kèm não úng thủy
Q05.3	Dị tật nứt đốt sống cùng cụt kết hợp não úng thủy	
Q05.4	Dị tật nứt đốt sống không xác định kết hợp não úng thủy	
Q05.5	Dị tật nứt đốt sống cổ không có não úng thủy	
Q05.6	Dị tật nứt đốt sống ngực không có não úng thủy	
Q05.7	Dị tật nứt đốt sống thắt lưng không có não úng thủy	Nứt đốt sống thắt lưng cùng không xác định khác
Q05.8	Dị tật nứt đốt cùng cụt không có não úng thủy	
Q05.9	Dị tật nứt đốt sống, không xác định	
Q06	Dị tật bẩm sinh khác của tủy sống	
Q06.0	Dị tật thiếu tủy sống	
Q06.1	Thiểu sản và/hoặc loạn sản tủy sống	Tủy sống phát triển bất toàn|Sự phát triển không hoàn toàn tủy sống|Loạn sản tủy sống
Q06.2	Dị tật bẩm sinh tủy sống chẻ đôi	
Q06.3	Dị tật bẩm sinh khác ở vùng đuôi ngựa	
Q06.4	Dị tật ứ nước ống nội tủy	Ứ nước ống sống
Q06.8	Dị tật bẩm sinh xác định khác của tủy sống	
Q06.9	Dị tật bẩm sinh của tủy sống, không xác định	
Q07	Dị tật bẩm sinh khác của hệ thần kinh	
Q07.0	Hội chứng Arnold Chiari	
Q07.8	Dị tật bẩm sinh của hệ thần kinh xác định khác	Bất sản dây thần kinh|Đám rối thần kinh cánh tay lạc chỗ|Hội chứng đồng động mi - hàm|Hội chứng Marcus Gunn
Q07.9	Dị tật bẩm sinh hệ thần kinh, không xác định	
Q10	Dị tật bẩm sinh của mi mắt, hệ lệ và/hoặc hốc mắt	
Q10.0	Dị tật sụp mí bẩm sinh	
Q10.1	Dị tật lộn mí bẩm sinh	
Q10.2	Dị tật quặm bẩm sinh	
Q10.3	Dị tật mí mắt bẩm sinh khác	Dị tật không có mí mắt
Q10.4	Dị tật thiếu hoặc bất sản hệ lệ	Dị tật không có điểm lệ
Q10.5	Dị tật hẹp hoặc co hẹp bẩm sinh ống dẫn lệ	
Q10.6	Dị tật bẩm sinh khác của hệ lệ	Dị tật bẩm sinh của hệ lệ không xác định khác
Q10.7	Dị tật bẩm sinh hốc mắt	
Q11	Dị tật không có nhãn cầu, mắt bé, mắt to	
Q11.0	Dị tật nhãn cầu dạng túi	
Q11.1	Dị tật không có mắt khác	Bất sản nhãn cầu|Thiểu sản nhãn cầu
Q11.2	Dị tật mắt bé	
Q11.3	Dị tật mắt to	
Q12	Dị tật bẩm sinh của thủy tinh thể	
Q12.0	Đục thủy tinh thể bẩm sinh	
Q12.1	Dị tật lệch thủy tinh thể bẩm sinh	
Q12.2	Dị tật Coloboma thủy tinh thể [dị tật ở dây treo thể thủy tinh]	
Q12.3	Thiếu thủy tinh thể bẩm sinh	
Q12.4	Dị tật thủy tinh thể hình cầu	
Q12.8	Dị tật bẩm sinh khác của thủy tinh thể	
Q12.9	Dị tật thủy tinh thể bẩm sinh, không xác định	
Q13	Dị tật bẩm sinh bán phần trước của mắt	
Q13.0	Dị tật Coloboma mống mắt [hội chứng mắt mèo]	Dị tật thiếu hụt ở mắt [Coloboma] không xác định khác
Q13.1	Dị tật thiếu mống mắt	Dị tật không có mống mắt
Q13.2	Dị tật bẩm sinh khác của mống mắt	Tật đồng tử không đều bẩm sinh|Teo đồng tử|Dị tật bẩm sinh ở mống mắt không xác định khác|Đồng tử lạc chỗ
Q13.3	Đục giác mạc bẩm sinh	
Q13.4	Dị tật bẩm sinh khác của giác mạc	Dị tật bẩm sinh của giác mạc không xác định khác|Giác mạc bé
Q13.5	Dị tật củng mạc xanh	
Q13.8	Dị tật bẩm sinh khác ở phần trước của mắt	Hội chứng Axenfeld-Rieger|Dị thường Rieger
Q13.9	Dị tật bẩm sinh ở phần trước của mắt, không xác định	
Q14	Dị tật bẩm sinh phần sau của mắt	
Q14.0	Dị tật bẩm sinh thủy tinh dịch	Đục thủy tinh dịch bẩm sinh
Q14.1	Dị tật bẩm sinh võng mạc	Giãn mạch võng mạc bẩm sinh
Q14.2	Dị tật bẩm sinh đĩa thị	Dị tật thiếu hụt [Coloboma] ở đĩa thị
Q14.3	Dị tật bẩm sinh màng mạch	
Q14.8	Dị tật bẩm sinh khác ở phần sau của mắt	Dị tật thiếu hụt [Coloboma] đáy mắt
Q14.9	Dị tật bẩm sinh ở phần sau của mắt, không xác định	
Q15	Dị tật bẩm sinh khác của mắt	
Q15.0	Glôcôm [tăng nhãn áp] bẩm sinh	Dị tật dạng mắt trâu [tăng nhãn áp]|Glôcôm ở trẻ sơ sinh|Phù nề nhãn cầu [thứ cấp do bệnh khác]|Lồi nhãn cầu bẩm sinh có tăng nhãn áp [Glôcôm]|Giác mạc to có tăng nhãn áp|Mắt to do Glôcôm bẩm sinh
Q15.8	Dị tật bẩm sinh xác định khác của mắt	
Q15.9	Dị tật bẩm sinh mắt, không xác định	
Q16	Dị tật bẩm sinh ở tai gây suy giảm thính lực	
Q16.0	Dị tật thiếu tai ngoài [vành tai] bẩm sinh	
Q16.1	Dị tật thiếu, teo và/hoặc hẹp ống tai ngoài bẩm sinh	Dị tật teo hoặc hẹp lỗ tai phần xương
Q16.2	Dị tật thiếu vòi nhĩ [eustache]	
Q16.3	Dị tật bẩm sinh các xương con của tai	Dị tật dính các xương con của tai
Q16.4	Dị tật bẩm sinh khác của tai giữa	Dị tật bẩm sinh không xác định khác của tai giữa
Q16.5	Dị tật bẩm sinh tai trong	
Q16.9	Dị tật bẩm sinh ở tai gây khiếm thính, không xác định	Thiếu tai bẩm sinh không xác định khác
Q17	Dị tật bẩm sinh khác ở tai	
Q17.0	Dị tật thừa vành tai [dị tật nhiều tai]	Dị tật thừa gờ bình tai|Dị tật thừa tai|Thịt thừa ngoài tai
Q17.1	Dị tật vành tai to	
Q17.2	Dị tật tai bé [Thiểu sản vành tai]	
Q17.3	Dị đạng vành tai [tai ngoài]	Dị hình tai nhọn
Q17.4	Dị tật vành tai ở vị trí bất thường	
Q17.5	Dị dạng vành tai nhô [vểnh, to]	Tật tai hình vợt [voi]
Q17.8	Dị tật bẩm sinh xác định khác của tai	Dị tật bẩm sinh không có dái tai
Q17.9	Dị tật bẩm sinh tai, không xác định	Dị tật bẩm sinh của tai không xác định khác
Q18	Dị tật bẩm sinh khác của mặt và/hoặc cổ	
Q18.0	Dị tật xoang, đường rò và/hoặc nang khe mang	Dấu tích khe mang
Q18.1	Di tật rò và/hoặc nang luân nhĩ	
Q18.2	Dị tật khe mang khác	Dị tật khe mang không xác định khác|Dị tật vành tai ở vị trí cổ|Não tai
Q18.3	Dị tật màng da cổ	Mảng da thừa hai bên cổ [dị tật mang cánh bướm]
Q18.4	Dị tật miệng rộng [dị tật khe hở ngang mặt]	
Q18.5	Dị tật miệng nhỏ	
Q18.6	Dị tật môi to	Phì đại môi bẩm sinh
Q18.7	Dị tật môi nhỏ	
Q18.8	Dị tật bẩm sinh xác định khác ở mặt và/hoặc cổ	
Q18.9	Dị tật bẩm sinh ở mặt và/hoặc cổ, không xác định	Dị tật bẩm sinh không xác định khác ở mặt và cổ
Q20	Dị tật bẩm sinh của buồng tim và bộ phận nối	
Q20.0	Dị tật thân chung động mạch	Tồn tại thân động mạch chung
Q20.1	Dị tật thất phải hai đường ra	Hội chứng Taussig-Bing
Q20.2	Dị tật thất trái hai đường ra	
Q20.3	Dị tật bộ phận nối tâm thất - động mạch không phù hợp	
Q20.4	Dị tật thất hai đường vào	Thất chung|Tim ba buồng có hai tâm nhĩ|Một buồng thất
Q20.5	Dị tật bộ phận nối nhĩ - thất không phù hợp	Chuyển gốc động mạch tự sửa chữa|Chuyển vị trí sang trái|Đảo thất
Q20.6	Dị tật đồng dạng của tiểu nhĩ	Đồng dạng tiểu nhĩ kèm không có lách hoặc đa lách
Q20.8	Dị tật bẩm sinh khác của buồng tim và bộ phận nối	
Q20.9	Dị tật bẩm sinh của buồng tim và bộ phận nối, không xác định	
Q21	Dị tật bẩm sinh của vách tim	
Q21.0	Dị tật thông liên thất	
Q21.1	Dị tật thông liên nhĩ	
Q21.2	Dị tật thông vách nhĩ thất	
Q21.3	Tứ chứng Fallot	
Q21.4	Dị tật thông vách động mạch chủ - phổi	Thông vách động mạch chủ|Cửa sổ động mạch chủ - phổi
Q21.8	Dị tật bẩm sinh khác của vách tim	
Q21.9	Dị tật bẩm sinh của vách tim, không xác định	
Q22	Dị tật bẩm sinh của van động mạch phổi và/hoặc van ba lá	
Q22.0	Dị tật teo van động mạch phổi	
Q22.1	Dị tật hẹp van động mạch phổi bẩm sinh	
Q22.2	Dị tật hở van động mạch phổi bẩm sinh	Trào ngược máu từ van động mạch phổi về lại buồng thất phải bẩm sinh
Q22.3	Dị tật bẩm sinh khác của van động mạch phổi	Dị tật bẩm sinh của van động mạch phổi không xác định khác
Q22.4	Dị tật hẹp van ba lá bẩm sinh	Teo van ba lá
Q22.5	Bất thường [dị tật] Ebstein	
Q22.6	Hội chứng tim phải thiểu sản	
Q22.8	Dị tật bẩm sinh khác của van ba lá	
Q22.9	Dị tật bẩm sinh van ba lá, không xác định	
Q23	Dị tật bẩm sinh của van động mạch chủ và/hoặc van hai lá	
Q23.0	Dị tật hẹp van động mạch chủ bẩm sinh	
Q23.1	Dị tật hở van động mạch chủ bẩm sinh	Van động mạch chủ có hai lá van|Hở động mạch chủ bẩm sinh
Q23.2	Dị tật hẹp van hai lá bẩm sinh	Teo van hai lá bẩm sinh
Q23.3	Dị tật hở van hai lá bẩm sinh	
Q23.4	Hội chứng tim trái thiểu sản	
Q23.8	Dị tật bẩm sinh khác của van động mạch chủ và/hoặc van hai lá	
Q23.9	Dị tật bẩm sinh của van động mạch chủ và/hoặc van hai lá, không xác định	
Q24	Dị tật bẩm sinh khác của tim	
Q24.0	Dị tật tim lệch phải	
Q24.1	Dị tật tim nằm bên trái (trong đảo ngược phủ tạng)	
Q24.2	Dị tật tim ba buồng nhĩ	
Q24.3	Dị tật hẹp phễu động mạch phổi	
Q24.4	Dị tật hẹp dưới van động mạch chủ bẩm sinh	
Q24.5	Dị tật của mạch vành	
Q24.6	Block nhĩ thất [block tim] [rối loạn dẫn truyền] bẩm sinh	
Q24.8	Dị tật tim bẩm sinh xác định khác	
Q24.9	Dị tật tim bẩm sinh, không xác định	
Q25	Dị tật bẩm sinh của động mạch lớn	
Q25.0	Dị tật còn ống động mạch	Còn ống Botalli|Tồn tại ống động mạch
Q25.1	Dị tật hẹp eo động mạch chủ	
Q25.2	Dị tật teo van động mạch chủ	
Q25.3	Dị tật hẹp van động mạch chủ	
Q25.4	Dị tật bẩm sinh khác của động mạch chủ	
Q25.5	Dị tật teo động mạch phổi	
Q25.6	Dị tật hẹp động mạch phổi	Hẹp trên van động mạch phổi
Q25.7	Dị tật bẩm sinh khác của động mạch phổi	Động mạch phổi bất thường|Dị tật không có động mạch phổi|Phình động mạch phổi bẩm sinh|Bất thường động mạch phổi|Thiểu sản động mạch phổi|Phình động - tĩnh mạch phổi
Q25.8	Dị tật bẩm sinh khác của động mạch lớn	
Q25.9	Dị tật bẩm sinh của động mạch lớn, không xác định	
Q26	Dị tật bẩm sinh của tĩnh mạch lớn	
Q26.0	Dị tật hẹp tĩnh mạch chủ bẩm sinh	
Q26.1	Tồn tại [còn] tĩnh mạch chủ trên bên trái	
Q26.2	Di tật hồi lưu tĩnh mạch phổi bất thường hoàn toàn	
Q26.3	Dị tật hồi lưu tĩnh mạch phổi bất thường một phần	
Q26.4	Dị tật hồi lưu tĩnh mạch phổi bất thường, không xác định	
Q26.5	Dị tật hồi lưu tĩnh mạch phổi bất thường	
Q26.6	Dị tật rò động mạch gan - tĩnh mạch cửa	
Q26.8	Dị tật bẩm sinh khác của tĩnh mạch lớn	
Q26.9	Dị tật bẩm sinh của tĩnh mạch lớn, không xác định	
Q27	Dị tật bẩm sinh khác của hệ thống mạch máu ngoại vi	
Q27.0	Dị tật thiếu và/hoặc thiểu sản động mạch dây rốn bẩm sinh	Dây rốn một động mạch
Q27.1	Dị tật hẹp động mạch thận bẩm sinh	
Q27.2	Dị tật bẩm sinh khác của động mạch thận	Dị tật bẩm sinh động mạch thận không xác định khác|Nhiều động mạch thận
Q27.3	Dị tật động - tĩnh mạch ngoại vi	
Q27.4	Giãn tĩnh mạch bẩm sinh	
Q27.8	Dị tật bẩm sinh xác định khác của hệ thống mạch ngoại vi	
Q27.9	Dị tật bẩm sinh của hệ thống mạch ngoại vi, không xác định	Bất thường của động mạch hoặc tĩnh mạch không xác định khác
Q28	Dị tật bẩm sinh khác của hệ thống tuần hoàn	
Q28.0	Dị dạng thông động - tĩnh mạch của mạch máu trước não	
Q28.1	Dị tật khác của mạch máu trước não	
Q28.2	Dị dạng thông động - tĩnh mạch của mạch máu não	
Q28.3	Dị tật khác của mạch máu não	
Q28.8	Dị tật bẩm sinh xác định khác của hệ tuần hoàn	Phình mạch bẩm sinh, vị trí xác định không phân loại mục khác
Q28.9	Dị tật bẩm sinh hệ tuần hoàn, không xác định	
Q30	Dị tật bẩm sinh ở mũi	
Q30.0	Dị tật hẹp [teo] lỗ mũi sau bẩm sinh	
Q30.1	Dị tật bất sản và/hoặc khuyết thiếu trong sự phát triển của mũi	Dị tật thiếu mũi bẩm sinh
Q30.2	Mũi có rãnh, có lõm hoặc nứt kẽ bẩm sinh	
Q30.3	Dị tật thủng vách mũi bẩm sinh	
Q30.8	Dị tật bẩm sinh khác ở mũi	Dị tật thừa mũi|Bất thường bẩm sinh của thành xoang mũi
Q30.9	Dị tật bẩm sinh ở mũi, không xác định	
Q31	Dị tật bẩm sinh của thanh quản	
Q31.0	Màng thanh quản	
Q31.1	Dị tật hẹp vùng hạ thanh môn bẩm sinh	
Q31.2	Giảm sản thanh quản	
Q31.3	Thoát vị thanh quản	
Q31.5	Dị tật nhuyễn cơ thanh quản bẩm sinh	
Q31.8	Dị tật bẩm sinh khác của thanh quản	Dị tật thiếu sụn nhân, nắp thanh quản, buồng thanh âm, thanh quản, sụn giáp|Bất sản sụn nhân, nắp thanh quản, buồng thanh âm, thanh quản, sụn giáp|Dị tật teo sụn nhân, nắp thanh quản, buồng thanh âm, thanh quản, sụn giáp|Dị tật nứt kẽ sụn giáp|Dị tật hẹp bẩm sinh của thanh quản không phân Loại mục khác|Dị tật nứt kẽ nắp thanh quản|Dị tật nứt mặt sau sụn nhẫn
Q31.9	Dị tật thanh quản bẩm sinh, không xác định	
Q32	Dị tật bẩm sinh ở khí quản và/hoặc phế quản	
Q32.0	Dị tật nhuyễn khí quản bẩm sinh	
Q32.1	Dị tật bẩm sinh khác của khí quản	Dị tật sụn khí quản|Dị tật teo khí quản
Q32.2	Dị tật nhuyễn phế quản bẩm sinh	
Q32.3	Dị tật hẹp phế quản bẩm sinh	
Q32.4	Dị tật bẩm sinh khác của phế quản	Dị tật thiếu phế quản|Dị tật bất sản phế quản|Dị tật teo phế quản|Dị tật bẩm sinh không xác định khác của phế quản|Dị tật túi thừa ở phế quản
Q33	Dị tật bẩm sinh của phổi	
Q33.0	Dị tật nang (kén) phổi bẩm sinh	
Q33.1	Dị tật phổi thừa thùy	
Q33.2	Dị tật phổi biệt lập	
Q33.3	Dị tật bất sản phổi	
Q33.4	Dị tật giãn phế quản bẩm sinh	
Q33.5	Dị tật mô lạc chỗ trong phổi	
Q33.6	Giảm sản và/hoặc loạn sản phổi	
Q33.8	Dị tật bẩm sinh khác của phổi	
Q33.9	Dị tật bẩm sinh ở phổi, không xác định	
Q34	Dị tật bẩm sinh khác của hệ hô hấp	
Q34.0	Bất thường ở màng phổi	
Q34.1	Nang (kén) trung thất bẩm sinh	
Q34.8	Dị tật bẩm sinh xác định khác của hệ hô hấp	Dị tật teo khoang mũi - hầu
Q34.9	Dị tật bẩm sinh hệ hô hấp, không xác định	
Q35	Khe hở vòm miệng	
Q35.1	Khe hở vòm miệng cứng	
Q35.3	Khe hở vòm miệng mềm	
Q35.5	Khe hở vòm miệng cứng và mềm	
Q35.7	Khe hở lưỡi gà	
Q35.9	Khe hở vòm miệng, một bên, không xác định	
Q36	Khe hở môi	
Q36.0	Khe hở môi, hai bên	
Q36.1	Khe hở giữa môi	
Q36.9	Khe hở môi, một bên	Khe hở môi không xác định khác
Q37	Khe hở vòm miệng kết hợp khe hở môi	
Q37.0	Khe hở vòm miệng cứng kết hợp khe hở môi hai bên	
Q37.1	Khe hở vòm miệng cứng kết hợp khe hở môi một bên	Khe hở vòm miệng cứng và môi không xác định khác
Q37.2	Khe hở vòm miệng mềm kết hợp khe hở môi hai bên	
Q37.3	Khe hở vòm miệng mềm kết hợp khe hở môi một bên	Khe hở vòm miệng mềm và môi không xác định khác
Q37.4	Khe hở vòm miệng cứng và mềm, kết hợp khe hở môi hai bên	
Q37.5	Khe hở vòm miệng cứng và mềm, kết hợp khe hở môi một bên	Khe hở vòm miệng mềm và môi không xác định khác
Q37.8	Khe hở vòm miệng không xác định kết hợp khe hở môi hai bên	
Q37.9	Khe hở vòm miệng không xác định kết hợp khe hở môi một bên không xác định	Khe hở vòm miệng và khe hở môi không xác định khác
Q38	Dị tật bẩm sinh khác của lưỡi, miệng và/hoặc họng	
Q38.0	Dị tật bẩm sinh của môi, không phân loại ở mục khác	
Q38.1	Dị tật dính [thắng; phanh] lưỡi	Dị tật cứng lưỡi [ngắn hãm lưỡi]
Q38.2	Dị tật lưỡi to	
Q38.3	Dị tật bẩm sinh khác của lưỡi	Dị tật thiếu lưỡi|Dị tật lưỡi chẻ đôi
Q38.4	Dị tật bẩm sinh của tuyến và/hoặc ống dẫn nước bọt	
Q38.5	Các dị tật bẩm sinh của vòm miệng, không phân loại mục khác	
Q38.6	Dị tật bẩm sinh khác của miệng	Dị tật bẩm sinh của miệng không xác định khác
Q38.7	Dị tật túi họng	
Q38.8	Dị tật bẩm sinh khác của họng	Dị tật bẩm sinh của họng không xác định khác
Q39	Dị tật bẩm sinh của thực quản	
Q39.0	Dị tật teo thực quản không có đường rò	Teo thực quản không xác định khác
Q39.1	Dị tật teo thực quản có đường rò thực quản - khí quản	Teo thực quản có đường rò thực quản - phế quản
Q39.2	Đường rò thực quản - khí quản bẩm sinh, không có teo thực quản	Đường rò thực quản - khí quản bẩm sinh không xác định khác
Q39.3	Dị tật bẩm sinh hẹp và/hoặc co hẹp thực quản	
Q39.4	Dị tật màng thực quản bẩm sinh	
Q39.5	Dị tật giãn thực quản bẩm sinh	
Q39.6	Dị tật bẩm sinh túi thừa thực quản	Dị tật bẩm sinh túi thực quản
Q39.8	Dị tật bẩm sinh khác của thực quản	Dị tật thiếu thực quản|Dị tật chuyển vị thực quản bẩm sinh|Dị tật thực quản đôi
Q39.9	Dị tật bẩm sinh thực quản, không xác định	
Q40	Dị tật bẩm sinh khác của đường tiêu hóa trên	
Q40.0	Dị tật hẹp môn vị phì đại bẩm sinh	
Q40.1	Dị tật bẩm sinh thoát vị cơ hoành qua khe thực quản	
Q40.2	Dị tật bẩm sinh xác định khác của dạ dày	
Q40.3	Dị tật bẩm sinh của dạ dày, không xác định	
Q40.8	Dị tật bẩm sinh xác định khác của đường tiêu hóa trên	
Q40.9	Dị tật bẩm sinh đường tiêu hóa trên, không xác định	
Q41	Dị tật thiếu, teo và/hoặc hẹp ruột non bẩm sinh	
Q41.0	Dị tật thiếu, teo và/hoặc hẹp tá tràng bẩm sinh	
Q41.1	Dị tật thiếu, teo và/hoặc hẹp hỗng tràng bẩm sinh	Hội chứng vỏ táo|Hỗng tràng không thủng
Q41.2	Dị tật thiếu, teo và/hoặc hẹp hồi tràng bẩm sinh	
Q41.8	Dị tật thiếu, teo và/hoặc hẹp phần xác định khác của ruột non bẩm sinh	
Q41.9	Dị tật thiếu, teo và/hoặc hẹp ruột non bẩm sinh, phần không xác định	Dị tật không có, teo và/hoặc hẹp ruột bẩm sinh không xác định khác
Q42	Dị tật thiếu, teo và/hoặc hẹp đại tràng bẩm sinh	
Q42.0	Dị tật thiếu, teo và/hoặc hẹp trực tràng bẩm sinh có đường rò	
Q42.1	Dị tật thiếu, teo và/hoặc hẹp trực tràng bẩm sinh không có đường rò	Trực tràng không thủng
Q42.2	Dị tật thiếu, teo và/hoặc hẹp hậu môn bẩm sinh có đường rò	
Q42.3	Dị tật thiếu, teo và/hoặc hẹp hậu môn bẩm sinh không có đường rò	Hậu môn không thủng
Q42.8	Dị tật thiếu, teo và/hoặc hẹp phần khác của đại tràng bẩm sinh	
Q42.9	Dị tật thiếu, teo và/hoặc hẹp đại tràng bẩm sinh, phần không xác định	
Q43	Dị tật bẩm sinh khác của ruột	
Q43.0	Dị tật bẩm sinh túi thừa Meckel	
Q43.1	Bệnh Hirschsprung	
Q43.2	Rối loạn chức năng bẩm sinh khác của đại tràng	Giãn đại tràng bẩm sinh
Q43.3	Dị tật cố định ruột bẩm sinh	
Q43.4	Dị tật ruột đôi	
Q43.5	Dị tật hậu môn lạc chỗ	
Q43.6	Đường rò bẩm sinh của hậu môn và/hoặc trực tràng	
Q43.7	Dị tật còn ổ nhớp	Ổ nhớp không xác định khác
Q43.8	Dị tật bẩm sinh xác định khác của ruột	
Q43.9	Dị tật bẩm sinh ruột, không xác định	
Q44	Dị tật bẩm sinh của túi mật, đường [ống] mật và/hoặc gan	
Q44.0	Bất sản, thiểu sản và/hoặc giảm sản túi mật	Dị tật thiếu túi mật bẩm sinh
Q44.1	Dị tật bẩm sinh khác của túi mật	Dị tật bẩm sinh của túi mật không xác định khác|Dị tật túi mật trong gan
Q44.2	Dị tật teo đường mật	
Q44.3	Hẹp và co hẹp bẩm sinh khác của đường mật	
Q44.4	Dị tật nang ống mật chủ	
Q44.5	Dị tật bẩm sinh khác của đường mật	Dị tật thừa ống gan|Dị tật bẩm sinh đường mật không xác định khác
Q44.6	Bệnh nang gan	Bệnh gan đa nang xơ hóa
Q44.7	Dị tật bẩm sinh khác của gan	Dị tật thừa gan|Hội chứng Alagille
Q45	Dị tật bẩm sinh khác của hệ tiêu hóa	
Q45.0	Bất sản, thiểu sản và/hoặc giảm sản tụy	Dị tật thiếu tụy bẩm sinh
Q45.1	Dị tật tụy hình vòng	
Q45.2	Dị tật nang tụy bẩm sinh	
Q45.3	Dị tật bẩm sinh khác của tụy và/hoặc ống tụy	
Q45.8	Dị tật bẩm sinh xác định khác của hệ tiêu hóa	
Q45.9	Dị tật bẩm sinh của hệ tiêu hóa, không xác định	
Q50	Dị tật bẩm sinh của buồng trứng, vòi trứng và/hoặc dây chằng rộng	
Q50.0	Dị tật thiếu buồng trứng bẩm sinh	
Q50.1	Nang buồng trứng bẩm sinh	
Q50.2	Xoắn bẩm sinh của buồng trứng	
Q50.3	Dị tật bẩm sinh khác của buồng trứng	Dị tật thừa buồng trứng|Dị tật bẩm sinh của buồng trứng không xác định khác|Dị tật buồng trứng sọc [loạn sản buồng trứng]
Q50.4	Nang nguồn gốc bào thai của vòi trứng	U nang tua viền [diềm]
Q50.5	Nang nguồn gốc bào thai của dây chằng rộng	
Q50.6	Dị tật bẩm sinh khác của vòi trứng và/hoặc dây chằng rộng	Dị tật không có vòi trứng hoặc dây chằng rộng|Dị tật thừa vòi trứng hoặc dây chằng rộng|Dị tật teo vòi trứng hoặc dây chằng rộng|Dị tật bẩm sinh vòi trứng và dây chằng rộng không xác định khác
Q51	Dị tật bẩm sinh của tử cung và/hoặc cổ tử cung	
Q51.0	Bất sản và/hoặc thiểu sản tử cung	Dị tật thiếu tử cung bẩm sinh
Q51.1	Tử cung đôi với cổ tử cung và âm đạo đôi	
Q51.2	Các loại tử cung đôi khác	Tử cung đôi không xác định khác
Q51.3	Dị tật tử cung hai sừng	
Q51.4	Dị tật tử cung một sừng [đơn]	
Q51.5	Bất sản và/hoặc thiểu sản cổ tử cung	Dị tật thiếu cổ tử cung bẩm sinh
Q51.6	Nang nguồn gốc bào thai của cổ tử cung	
Q51.7	Dị tật rò bẩm sinh giữa tử cung với ống tiêu hóa và/hoặc đường tiết niệu	
Q51.8	Dị tật bẩm sinh khác của tử cung và/hoặc cổ tử cung	Giảm sản của tử cung và cổ tử cung
Q51.9	Dị tật bẩm sinh của tử cung và/hoặc cổ tử cung, không xác định	
Q52	Dị tật bẩm sinh khác của cơ quan sinh dục nữ	
Q52.0	Dị tật thiếu âm đạo bẩm sinh	
Q52.1	Dị tật am đạo đôi	
Q52.2	Rò trực tràng âm đạo bẩm sinh	
Q52.3	Màng trinh không thủng	
Q52.4	Dị tật bẩm sinh khác của âm đạo	Dị tật bẩm sinh của âm đạo không xác định khác
Q52.5	Dị tật dính môi lớn của âm hộ	
Q52.6	Dị tật bẩm sinh của âm vật	
Q52.7	Dị tật bẩm sinh khác của âm hộ	
Q52.8	Dị tật bẩm sinh xác định khác của cơ quan sinh dục nữ	
Q52.9	Dị tật bẩm sinh của cơ quan sinh dục nữ, không xác định	
Q53	Dị tật tinh hoàn ẩn [lạc chỗ]	
Q53.0	Dị tật tinh hoàn ẩn	Dị tật tinh hoàn lạc chỗ một hoặc hai bên
Q53.1	Tinh hoàn chưa xuống bìu một bên	
Q53.2	Tinh hoàn chưa xuống bìu hai bên	
Q53.9	Tinh hoàn không xuống bìu, không xác định	Tinh hoàn ẩn không xác định khác
Q54	Dị tật bẩm sinh lỗ tiểu lệch thấp	
Q54.0	Dị tật bẩm sinh lỗ tiểu lệch thấp, thể quy đầu	
Q54.1	Dị tật bẩm sinh lỗ tiểu thấp, thể thân dương vật	
Q54.2	Dị tật bẩm sinh lỗ tiểu thấp, thể dương vật - bìu	
Q54.3	Dị tật bẩm sinh lỗ tiểu thấp, tầng sinh môn	
Q54.4	Dị tật bẩm sinh cong dương vật	
Q54.8	Dị tật bẩm sinh lỗ tiểu lệch thấp khác	
Q54.9	Dị tật bẩm sinh lỗ tiểu lệch thấp, không xác định	
Q55	Dị tật bẩm sinh khác của cơ quan sinh dục nam	
Q55.0	Dị tật thiếu và/hoặc thiểu sản tinh hoàn	Dị tật một tinh hoàn
Q55.1	Giảm sản tinh hoàn và bìu	Dính hai tinh hoàn
Q55.2	Dị tật bẩm sinh khác của tinh hoàn và/hoặc bìu	Dị tật bẩm sinh của tinh hoàn hoặc bìu không xác định khác|Hội chứng đa tinh hoàn|Tinh hoàn lò xo|Tinh hoàn di chuyển
Q55.3	Dị tật teo ống dẫn tinh	
Q55.4	Dị tật bẩm sinh khác của ống dẫn tinh, mào tinh, túi tinh và tuyến tiền liệt	
Q55.5	Dị tật thiếu và/hoặc thiểu sản dương vật bẩm sinh	
Q55.6	Dị tật bẩm sinh khác của dương vật	
Q55.8	Dị tật bẩm sinh xác định khác của cơ quan sinh dục nam	
Q55.9	Dị tật bẩm sinh của cơ quan sinh dục nam, không xác định	Bẩm sinh: không xác định khác cơ quan sinh dục nam|bất thường không xác định khác của cơ quan sinh dục nam|dị dạng không xác định khác của cơ quan sinh dục nam
Q56	Giới tính chưa định hình chính xác và/hoặc hội chứng giả lưỡng giới	
Q56.0	Hội chứng lưỡng giới, không phân loại mục khác	Tuyến sinh dục lưỡng giới
Q56.1	Hội chứng lưỡng giới giả nam, không phân loại mục khác	Hiện tượng lưỡng giới giả ở nam giới không xác định khác
Q56.2	Hội chứng lưỡng giới giả nữ, không phân loại mục khác	Hiện tượng lưỡng giới giả ở nữ giới không xác định khác
Q56.3	Hội chứng lưỡng giới giả, không xác định	
Q56.4	Giới tính chưa định hình chính xác, không xác định	Lưỡng giới [mơ hồ giới tính]
Q60	Dị tật bất sản và/hoặc khuyết thiếu khác của thận	
Q60.0	Bất sản một bên thận bẩm sinh	
Q60.1	Bất sản cả hai bên thận bẩm sinh	
Q60.2	Bất sản thận bẩm sinh, không xác định	
Q60.3	Dị tật bẩm sinh giảm sản thận, một bên	
Q60.4	Dị tật bẩm sinh giảm sản thận, hai bên	
Q60.5	Dị tật bẩm sinh giảm sản thận, không xác định	
Q60.6	Hội chứng Potter	
Q61	Bệnh nang thận	
Q61.0	Nang thận (đơn) bẩm sinh	
Q61.1	Bệnh thận đa nang, di truyền lặn trên nhiễm sắc thể thường	Bệnh thận đa nang, thể bệnh ở trẻ nhỏ
Q61.2	Bệnh thận đa nang, di truyền trội trên nhiễm sắc thể thường	Bệnh thận đa nang, thể bệnh ở người lớn
Q61.3	Bệnh thận đa nang bẩm sinh, không xác định	
Q61.4	Loạn sản thận	
Q61.5	Bệnh nang tủy thận di truyền	Bệnh xốp tủy thận không xác định khác
Q61.8	Bệnh nang thận khác	
Q61.9	Bệnh nang thận, không xác định	Hội chứng Meckel-Gruber
Q62	Dị tật tắc nghẽn bẩm sinh của bể thận và dị tật bẩm sinh của niệu quản	
Q62.0	Bệnh lý thận ứ nước bẩm sinh	
Q62.1	Dị tật teo và hẹp niệu quản	
Q62.2	Phình to niệu quản bẩm sinh	Dị tật giãn niệu quản bẩm sinh
Q62.3	Dị tật tắc nghẽn khác của bể thận và/hoặc niệu quản	Dị tật túi sa [nang, thoát vị, tràn dịch] niệu quản bẩm sinh
Q62.4	Bất sản niệu quản	Dị tật thiếu niệu quản
Q62.5	Dị tật niệu quản nhân đôi	Dị tật thừa niệu quản|Dị tật niệu quản đôi
Q62.6	Niệu quản lạc chỗ	Dị lệch niệu quản hay lỗ niệu quản|Chuyển vị của niệu quản hay lỗ niệu quản|Niệu quản hay lỗ niệu quản lạc chỗ
Q62.7	Trào ngược bàng quang - niệu quản - thận bẩm sinh	
Q62.8	Dị tật bẩm sinh khác của niệu quản	Bất thường niệu quản không xác định khác
Q63	Dị tật bẩm sinh khác của thận	
Q63.0	Dị tật thừa thận	
Q63.1	Dị tật thận dính, thận phân thùy, thận móng ngựa	
Q63.2	Dị tât thận lạc chỗ	Dị tật thận lạc chỗ bẩm sinh|Dị tật thận quay bất thường
Q63.3	Thận tăng sản và/hoặc khổng lồ	
Q63.8	Dị tật bẩm sinh xác định khác của thận	Sỏi thận bẩm sinh
Q63.9	Dị tật bẩm sinh của thận, không xác định	
Q64	Dị tật bẩm sinh khác của hệ tiết niệu	
Q64.0	Dị tật bẩm sinh lỗ tiểu lệch cao	
Q64.1	Dị tật lộn bàng quang ra ngoài cơ thể	Dị tật lạc vị bàng quang [lạc chỗ]|Bàng quang lồi ra ngoài bụng
Q64.2	Dị tật van niệu đạo sau bẩm sinh	
Q64.3	Dị tật teo và hẹp niệu đạo và/hoặc cổ bàng quang khác	
Q64.4	Dị tật ống niệu rốn	Dị tật nang ống niệu rốn|Dị tật còn ống niệu rốn|Dị tật sa ống niệu rốn
Q64.5	Dị tật thiếu bàng quang và/hoặc niệu đạo bẩm sinh	
Q64.6	Di tật bẩm sinh túi thừa bàng quang	
Q64.7	Dị tật bẩm sinh khác của bàng quang và/hoặc niệu đạo	
Q64.8	Dị tật bẩm sinh xác định khác của hệ tiết niệu	
Q64.9	Dị tật bẩm sinh của hệ tiết niệu, không xác định	
Q65	Dị dạng bẩm sinh của khớp háng	
Q65.0	Dị tật trật khớp háng bẩm sinh, một bên	
Q65.1	Dị tật trật khớp háng bẩm sinh, hai bên	
Q65.2	Dị tật trật khớp háng bẩm sinh, không xác định	
Q65.3	Dị tật bán trật khớp háng bẩm sinh, một bên	
Q65.4	Dị tật bán trật khớp háng bẩm sinh, hai bên	
Q65.5	Dị tật bán trật khớp háng bẩm sinh, không xác định	
Q65.6	Dị tật khớp háng không ổn định	Khớp háng dễ bị trật|Khớp háng dễ bị bán trật
Q65.8	Dị dạng bẩm sinh khác của khớp háng	Cổ xương đùi xoay trước|Loạn sản ổ cối bẩm sinh
Q65.9	Dị dạng bẩm sinh của khớp háng, không xác định	
Q66	Dị dạng bẩm sinh của bàn chân	
Q66.0	Dị dạng bàn chân khoèo [chân vẹo]	
Q66.1	Dị dạng bàn chân gót vẹo vào trong	
Q66.2	Dị dạng xương đốt bàn chân vẹo vào trong	
Q66.3	Dị dạng bẩm sinh vẹo vào trong khác của bàn chân	Dị dạng ngón chân cái vẹo vào trong, bẩm sinh
Q66.4	Dị dạng bàn chân gót vẹo ra ngoài	
Q66.5	Dị dạng bàn chân bẹt bẩm sinh	
Q66.6	Dị dạng bẩm sinh vẹo ra ngoài khác của bàn chân	Dị tật xương đốt bàn chân vẹo ra ngoài
Q66.7	Dị dạng bẩm sinh bàn chân lõm [bàn chân quặp, có vòm cao]	
Q66.8	Dị dạng bẩm sinh khác của bàn chân	Bàn chân khoèo không xác định khác|Ngón chân quặp, bẩm sinh
Q66.9	Dị dạng bẩm sinh của bàn chân, không xác định	
Q67	Dị dạng cơ xương bẩm sinh của đầu, mặt, cột sống và/hoặc ngực	
Q67.0	Dị dạng mất cân đối mặt	
Q67.1	Dị dạng mặt do chèn ép	
Q67.2	Dị tật đầu dài [dị tật đầu hình thuyền]	
Q67.3	Dị tật đầu méo [Dị tật sọ nghiêng] [dị tật đầu dẹt]	
Q67.4	Dị dạng bẩm sinh khác của sọ, mặt và/hoặc xương hàm	
Q67.5	Dị dạng cột sống bẩm sinh	
Q67.6	Dị tật ngực lõm	Ngực hình phễu bẩm sinh
Q67.7	Dị tật ngực lồi	Ngực hình chim bồ câu [ngực gà] bẩm sinh
Q67.8	Dị dạng bẩm sinh khác của ngực	Dị dạng bẩm sinh của thành ngực không xác định khác
Q68	Dị dạng cơ xương bẩm sinh khác	
Q68.0	Dị dạng bẩm sinh của cơ ức đòn chũm	
Q68.1	Dị dạng bẩm sinh của bàn tay	
Q68.2	Dị dạng bẩm sinh của đầu gối	
Q68.3	Dị dạng cong xương đùi bẩm sinh	
Q68.4	Dị dạng xương chày và xương mác cong bẩm sinh	
Q68.5	Dị dạng xương dài của chân cong bẩm sinh, không xác định	
Q68.8	Dị dạng cơ xương khớp bẩm sinh xác định khác	
Q69	Dị tật thừa ngón	
Q69.0	Dị tật thừa ngón tay	
Q69.1	Dị tật thừa ngón tay cái	
Q69.2	Dị tật thừa ngón chân	Dị tật thừa ngón chân cái
Q69.9	Dị tật thừa ngón, không xác định	Dị tật thừa ngón không xác định khác
Q70	Dị tật dính ngón	
Q70.0	Dị tật ngón tay dính nhau	Dị tật dính ngón tay kèm dính xương
Q70.1	Dị tật ngón tay dính mô mềm có màng như chân vịt	Dị tật dính ngón tay không dính xương
Q70.2	Dị tật ngón chân dính nhau	Dị tật dính ngón chân kèm dính xương
Q70.3	Dị tật ngón chân dính mô mềm có màng như chân vịt	Dị tật dính ngón chân không dính xương
Q70.4	Dị tật dính nhiều ngón	
Q70.9	Dị tật dính ngón, không xác định	Dị tật dính đốt ngón không xác định khác
Q71	Dị tật bẩm sinh thiếu hụt chi trên	Khuyết tật thiếu hụt của chi trên
Q71.0	Dị tật thiếu toàn bộ chi trên bẩm sinh	
Q71.1	Dị tật thiếu cánh - cẳng tay bẩm sinh có bàn tay	
Q71.2	Dị tật thiếu cả bàn tay và cẳng tay bẩm sinh	
Q71.3	Dị tật thiếu bàn tay và/hoặc (các) ngón tay bẩm sinh	
Q71.4	Dị tật thiếu hụt xương quay theo chiều dọc [dài]	
Q71.5	Dị tật thiếu hụt xương trụ theo chiều dọc	
Q71.6	Dị tật bàn tay hình càng tôm hùm [dị tật bàn tay chẻ]	
Q71.8	Dị tật bẩm sinh thiếu hụt chi trên khác	Tật ngắn chi trên bẩm sinh
Q71.9	Dị tật bẩm sinh thiếu hụt chi trên, không xác định	
Q72	Dị tật bẩm sinh thiếu hụt chi dưới	Khuyết tật thiếu hụt của chi dưới
Q72.0	Dị tật thiếu toàn phần (các) chi dưới bẩm sinh	
Q72.1	Dị tật thiếu đùi và/hoặc cẳng chân có bàn chân bẩm sinh	
Q72.2	Di tật thiếu cả cẳng chân và bàn chân bẩm sinh	
Q72.3	Dị tật thiếu bàn chân và/hoặc ngón chân bẩm sinh	
Q72.4	Dị tật bẩm sinh thiếu hụt xương đùi theo chiều dọc [dài]	Dị tật bẩm sinh thiếu hụt đầu trên và chiều dài xương đùi
Q72.5	Dị tật thiếu hụt xương chày theo chiều dọc [dài]	
Q72.6	Dị tật thiếu hụt xương mác theo chiều dọc [dài]	
Q72.7	Dị tật bàn chân chẻ [dị tật bàn chân hình càng tôm hùm]	
Q72.8	Dị tật thiếu hụt chi dưới khác	Dị tật ngắn chân bẩm sinh
Q72.9	Dị tật thiếu hụt chi dưới, không xác định	
Q73	Dị tật thiếu hụt của chi không xác định	
Q73.0	Dị tật bẩm sinh thiếu chi không xác định	Dị tật khuyết chi không xác định khác
Q73.1	Dị tật tay chân hải cẩu [phocomelia], không xác định chi	Dị tật tay chân hải cẩu [phocomelia] không xác định khác
Q73.8	Dị tật thiếu hụt chi khác của chi không xác định	Biến dạng thu nhỏ theo chiều dài của chi không xác định|Tật giảm sinh chi không xác định khác|Tật thiếu nửa ngoài không xác định khác|Khuyết tật thiếu hụt không xác định khác
Q74	Dị tật bẩm sinh khác của chi	
Q74.0	Dị tật bẩm sinh khác của chi trên, kể cả vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan]	Dị tật thừa xương cổ tay|Dị tật loạn phát xương đòn sọ|Khớp giả của xương đòn bẩm sinh|Tật to ngón|Biến dạng Madelung|Tật dính xương quay trụ|Biến dạng Sprengel|Ngón cái ba đốt
Q74.1	Dị tật bẩm sinh của gối	
Q74.2	Dị tật bẩm sinh khác của chi dưới, kể cả đai chậu [khung chậu, xương cùng và xương cụt]	
Q74.3	Co cứng đa khớp bẩm sinh	
Q74.8	Dị tật bẩm sinh xác định khác của chi	
Q74.9	Dị tật bẩm sinh không xác định của (các) chi	Bất thường bẩm sinh của chi không xác định khác
Q75	Dị tật bẩm sinh khác của xương sọ và xương mặt	
Q75.0	Dị tật liền sớm khớp sọ [dị tật hẹp sọ]	Dị tật đầu hình cao|Kết hợp không hoàn chỉnh của sọ|Dị tật đầu hình tháp|Dị tật đầu hình tam giác
Q75.1	Loạn sản xương sọ mặt	Bệnh Crouzon
Q75.2	Dị tật quá cách xa nhau (thường ở mắt)	
Q75.3	Dị tật đầu to	
Q75.4	Loạn sản xương hàm mặt	
Q75.5	Loạn sản xương mắt - hàm	
Q75.8	Dị tật bẩm sinh xác định khác của xương sọ và/hoặc mặt	Dị tật thiếu xương sọ bẩm sinh|Dị dạng bẩm sinh của trán|Dị tật đáy chẩm dịch trên
Q75.9	Dị tật bẩm sinh của xương sọ và/hoặc xương mặt, không xác định	
Q76	Dị tật bẩm sinh của xương sống và/hoặc xương lồng ngực	
Q76.0	Dị tật cột sống chẻ đôi thể kín; dị tật nứt đốt sống ẩn	
Q76.1	Hội chứng Klippel-Feil	Hội chứng tổng hợp đốt sống cổ [hội chứng dính đốt sống cổ]
Q76.2	Bệnh lý trượt đốt sống do gãy eo bẩm sinh	
Q76.3	Vẹo cột sống bẩm sinh do dị tật xương bẩm sinh	Dị tật kết dính một bên đốt sống hoặc phân chia đốt sống không hoàn chỉnh có vẹo cột sống
Q76.4	Dị tật bẩm sinh khác của cột sống, không liên quan đến vẹo cột sống	
Q76.5	Dị tật xương sườn cổ	Dị tật xương sườn thừa [phụ] tại vùng cổ
Q76.6	Dị tật bẩm sinh khác ở xương sườn	
Q76.7	Dị tật bẩm sinh của xương ức	Dị tật không có xương ức bẩm sinh|Xương ức chẻ đôi
Q76.8	Dị tật bẩm sinh khác của xương ngực	
Q76.9	Dị tật bẩm sinh của xương ngực, không xác định	
Q77	Loạn sản xương sụn có khuyết tật phát triển xương dài và/hoặc cột sống	
Q77.0	Loạn sản sụn xương	Thiểu sản sụn
Q77.1	Loạn sản hệ xương gây tầm vóc thấp [hội chứng người lùn]	
Q77.2	Hội chứng xương sườn ngắn	Loạn sản ngực gây ngạt [hội chứng Jeune]
Q77.3	Loạn sản sụn từng đám nhỏ	
Q77.4	Loạn sản sụn achondroplasia [hội chứng ngắn chi]	Dị tật thiểu sản sụn|Xơ cứng xương bẩm sinh
Q77.5	Loạn sản loạn dưỡng [Chứng loạn sản Diastrophic]	
Q77.6	Loạn sản sụn ngoại bì	Hội chứng Ellis-van Creveld
Q77.7	Loạn sản đầu đốt xương cột sống	
Q77.8	Loạn sản xương sụn khác với khuyết tật phát triển xương dài và/hoặc cột sống	
Q77.9	Loạn sản xương sụn với khuyết tật phát triển xương dài và/hoặc cột sống, không xác định	
Q78	Loạn sản xương sụn khác	
Q78.0	Chứng tạo xương bất toàn [bệnh xương thủy tinh]	Dị tật giòn xương|Bệnh tạo xương bất toàn [bệnh xương dễ gãy]
Q78.1	Loạn sản sợi - đa xương	
Q78.2	Dị tật xương hóa đá	Hội chứng Albers-Schönberg
Q78.3	Loạn sản thân xương tiến triển	Hội chứng Camurati-Engelmann
Q78.4	U sụn xương bẩm sinh	Hội chứng Maffucci|Bệnh Ollier
Q78.5	Loạn sản hành xương	Hội chứng Pyle
Q78.6	Dị tật đa chồi xương bẩm sinh	Loạn sản cục bộ thân xương
Q78.8	Loạn sản xương sụn xác định khác	Bệnh xương đặc đốm
Q78.9	Loạn sản xương sụn, không xác định	Loạn dưỡng sụn không xác định khác|Loạn dưỡng xương không xác định khác
Q79	Dị tật bẩm sinh của hệ cơ xương, không phân loại mục khác	
Q79.0	Thoát vị hoành bẩm sinh	
Q79.1	Dị tật bẩm sinh khác của cơ hoành	Dị tật thiếu cơ hoành|Dị tật bẩm sinh của cơ hoành không xác định khác|Dị tật tạng trong ổ bụng đi lên lồng ngực do lồi cơ hoành
Q79.2	Dị tật thoát vị trong dây rốn	
Q79.3	Dị tật sổ tạng bẩm sinh [qua lỗ bên cạnh rốn]	
Q79.4	Hội chứng bụng quả mận [bao gồm sự thiếu hụt cơ bụng, dị tật đường tiểu, và tinh hoàn ẩn nằm trong ổ bụng trong]	
Q79.5	Dị tật bẩm sinh khác của thành bụng	
Q79.6	Hội chứng Ehlers-Danlos	
Q79.8	Dị tật bẩm sinh khác của hệ cơ xương khớp	
Q79.9	Dị tật bẩm sinh hệ cơ xương, không xác định	
Q80	Bệnh da vảy cá bẩm sinh	
Q80.0	Bệnh da vảy cá thể thông thường	
Q80.1	Bệnh da vẩy cá liên quan đến nhiễm sắc thể X	
Q80.2	Bệnh da vảy cá bẩm sinh không có bóng nước	Bệnh da dạng keo ở trẻ nhỏ [hội chứng trẻ sơ sinh bọc màng toàn thân]
Q80.3	Chứng đỏ da bóng nước dưới dạng vảy cá	
Q80.4	Bệnh da vảy cá sơ sinh dạng harlequin	
Q80.8	Bệnh vảy cá bẩm sinh khác	
Q80.9	Bệnh vảy cá bẩm sinh, không xác định	
Q81	Bệnh ly thượng bì bọng nước bẩm sinh	
Q81.0	Bệnh ly thượng bì bọng nước bẩm sinh thể đơn giản	
Q81.1	Bệnh ly thượng bì bọng nước bẩm sinh thể nguy kịch	Hội chứng Herlitz
Q81.2	Bệnh ly thượng bì bọng nước bẩm sinh thể loạn dưỡng	
Q81.8	Bệnh ly thượng bì bọng nước bẩm sinh khác	
Q81.9	Bệnh ly thượng bì bọng nước bẩm sinh, không xác định	
Q82	Dị tật bẩm sinh khác của da	
Q82.0	Phù bạch huyết di truyền	
Q82.1	Bệnh khô da sắc tố	
Q82.2	Bệnh dưỡng bào [bệnh tế bào mast]	
Q82.3	Bệnh sắc tố dầm dề [sắc tố incontinentia]	
Q82.4	Loạn sản ngoại bì (làm giảm tiết mồ hôi)	
Q82.5	Nơ vi bẩm sinh không tân sinh	
Q82.8	Dị tật bẩm sinh xác định khác của da	
Q82.9	Dị tật bẩm sinh của da, không xác định	
Q83	Dị tật bẩm sinh của vú	
Q83.0	Dị tật thiếu vú và/hoặc núm vú bẩm sinh	
Q83.1	Dị dạng vú phụ	Dị dạng thừa vú
Q83.2	Dị tật thiếu núm vú	
Q83.3	Dị dạng núm vú phụ	Dị dạng thừa núm vú
Q83.8	Dị tật bẩm sinh khác của vú	Dị tật giảm sản vú
Q83.9	Dị tật bẩm sinh của vú, không xác định	
Q84	Dị tật bẩm sinh khác của hệ vỏ bọc [da]	
Q84.0	Rụng lông tóc bẩm sinh	Dị tật không có tóc bẩm sinh
Q84.1	Rối loạn bẩm sinh hình thái của tóc không phân loại mục khác	
Q84.2	Dị tật bẩm sinh khác của lông tóc	
Q84.3	Dị tật không móng	
Q84.4	Móng đốm trắng bẩm sinh	
Q84.5	Móng to và/hoặc phì đại	Dày móng bẩm sinh|Dày móng
Q84.6	Dị tật bẩm sinh khác của móng	
Q84.8	Dị tật bẩm sinh xác định khác của hệ vỏ bọc [da]	Bất sản da bẩm sinh
Q84.9	Dị tật bẩm sinh khác của hệ vỏ bọc [da], không xác định	
Q85	Hội chứng u thần kinh - da ngoại bì, không phân loại mục khác	
Q85.0	U xơ thần kinh (không ác tính)	Bệnh Von Recklinghausen
Q85.1	Bệnh xơ cứng củ	Bệnh Bourneville|Xơ não củ
Q85.8	Hội chứng u thần kinh da ngoại bì khác, không phân loại mục khác	
Q85.9	Hội chứng u thần kinh da ngoại bì, không xác định	Bệnh u mô thừa [u hamartoma] không xác định khác
Q86	Hội chứng dị tật bẩm sinh do nguyên nhân bên ngoài đã biết, không phân loại mục khác	
Q86.0	Hội chứng cồn [rượu] bào thai (dị dạng)	
Q86.1	Hội chứng hydantoin bào thai	Hội chứng Meadow
Q86.2	Dị dạng do warfarin	
Q86.8	Dị tật bẩm sinh khác do nguyên nhân bên ngoài đã biết	
Q87	Hội chứng dị tật bẩm sinh khác tác động nhiều hệ thống	
Q87.0	Hội chứng dị tật bẩm sinh tác đọng chủ yếu vào diện mạo [hình dạng] của mặt	Đầu hình tháp dính nhiều ngón tay [Apert]|Đầu hình tháp ngón tay [Apert]|Hội chứng ẩn mắt|Dị tật một hốc mắt
Q87.1	Hội chứng dị tật bẩm sinh liên quan chủ yếu đến tầm vóc thấp	
Q87.2	Hội chứng dị tật bẩm sinh liên quan chủ yếu đến các chi	
Q87.3	Hội chứng dị tật bẩm sinh có phát triển sớm quá mức	
Q87.4	Hội chứng Marfan	
Q87.5	Hội chứng dị tật bẩm sinh khác đi kèm thay đổi xương khác	
Q87.8	Hội chứng dị tật bẩm sinh xác định khác, không phân loại mục khác	
Q89	Dị tật bẩm sinh khác, không phân loại mục khác	
Q89.0	Dị tật bẩm sinh của lách	
Q89.1	Dị tật bẩm sinh của tuyến thượng thận	
Q89.2	Dị tật bẩm sinh của tuyến nội tiết khác	
Q89.3	Đảo ngược phủ tạng	
Q89.4	Sinh đôi dính nhau	Sinh đôi dính liền đầu|Sinh con hai đầu|Sinh đôi dính liền|Sinh đôi dính liền mông|Sinh đôi dính liền ngực
Q89.7	Da dị tật bẩm sinh, không phân loại mục khác	
Q89.8	Dị tật bẩm sinh xác định khác	
Q89.9	Dị tật bẩm sinh, không xác định	
Q90	Hội chứng Down	
Q90.0	Thể tam nhiễm sắc thể, không phân ly trong giảm phân	
Q90.1	Thể tam nhiễm sắc thể 21, thể khảm (không phân ly trong nguyên phân)	
Q90.2	Thể tam nhiễm sắc thể 21, chuyển đoạn	
Q90.9	Hội chứng Down, không xác định	Thể tam nhiễm sắc thể 21, không xác định khác
Q91	Hội chứng Edwards và/hoặc hội chứng Patau	
Q91.0	Thể tam nhiễm sắc thể 18, không phân ly trong giảm phân	
Q91.1	Thể tam nhiễm sắc thể 18, thể khảm (không phân ly trong nguyên phân)	
Q91.2	Thể tam nhiễm sắc thể 18, chuyển đoạn	
Q91.3	Hội chứng Edwards, không xác định	
Q91.4	Thể tam nhiễm sắc thể 13, không phân ly trong giảm phân	
Q91.5	Thể tam nhiễm sắc thể 13, thể khảm (không phân ly trong nguyên phân)	
Q91.6	Thể tam nhiễm sắc thể 13, chuyển đoạn	
Q91.7	Hội chứng Patau, không xác định	
Q92	Thể tam nhiễm sắc thể thường hoàn toàn và/hoặc một phần khác, không phân loại mục khác	
Q92.0	Thể tam nhiễm sắc thể hoàn toàn, không phân ly trong giảm phân	
Q92.1	Thể tam nhiễm sắc thể hoàn toàn, thể khảm (không phân ly trong nguyên phân)	
Q92.2	Thể tam nhiễm sắc thể một phần, lớn	Nhân đôi [sao chép] một phần hay toàn thể nhiễm sắc thể
Q92.3	Thể tam nhiễm sắc thể một phần, nhỏ	Nhân đôi [so chép] một phần nhiễm sắc thể.
Q92.4	Sự nhân đôi [sao chép] nhiễm sắc tử chỉ thấy ở cuối kỳ đầu [đầu kỳ giữa]	
Q92.5	Sự nhân đôi [sao chép] nhiễm sắc tử kèm đột biến cấu trúc phức tạp khác	
Q92.6	Nhiễm sắc thể có dấu ấn phụ [siêu số, thừa]	
Q92.7	Thể tam bội và/hoặc thể đa bội	
Q92.8	Thể tam nhiễm sắc thể thường hoàn toàn và/hoặc một phần xác định khác	
Q92.9	Thể tam nhiễm sắc thể thường hoàn toàn và/hoặc một phần, không xác định	
Q93	Thể đơn nhiễm sắc thể thường và/hoặc mất đoạn nhiễm sắc thể thường, không phân loại mục khác	
Q93.0	Thể đơn nhiễm sắc thể hoàn toàn không phân ly trong giảm phân	
Q93.1	Thể đơn nhiễm sắc thể hoàn toàn, thể khảm (không phân ly trong nguyên phân)	
Q93.2	Nhiễm sắc thể hình vòng và/hoặc hai tâm động	
Q93.3	Mất đoạn cánh ngắn nhiễm sắc thể số 4	Hội chứng Wolff-Hirschorn
Q93.4	Mất đoạn cánh ngắn nhiễm sắc thể số 5	Hội chứng mèo kêu
Q93.5	Mất đoạn khác của nhiễm sắc thể	Hội chứng Angelman
Q93.6	Mất đoạn nhiễm sắc thể chỉ thấy ở cuối kỳ đầu [đầu kỳ giữa]	
Q93.7	Mất đoạn nhiễm sắc thể kèm đột biến cấu trúc phức tạp khác	
Q93.8	Mất đoạn của nhiễm sắc thể thường khác	
Q93.9	Mất đoạn nhiễm sắc thể thường, không xác định	
Q95	Đột biến cấu trúc nhiễm sắc thể không gây mất cân bằng gen và/hoặc cá thể bình thường có dấu ấn đột biến cấu trúc, không phân loại ở mục khác	
Q95.0	Chuyển đoạn và lặp đoạn không gây mất cân bằng gen ở cá thể bình thường	
Q95.1	Đảo đoạn nhiễm sắc thể ở cá thể bình thường	
Q95.2	Đột biến cấu trúc nhiễm sắc thể thường không gây mất cân bằng gen ở cá thể bất thường	
Q95.3	Đột biến cấu trúc nhiễm sắc thể giới tính/thường không gây mất cân bằng gen ở cá thể bất thường	
Q95.4	Cá thể bình thường có mang bất thường vùng dị nhiễm sắc	
Q95.5	Cá thể có vị trí dễ gãy ở nhiễm sắc thể thường	
Q95.8	Đột biến cấu trúc nhiễm sắc thể không gây mất cân bằng gen và/hoặc dấu ấn bất thường cấu trúc khác	
Q95.9	Đột biến cấu trúc nhiễm sắc thể không gây mất cân bằng gen và/hoặc dấn ấn bất thường cấu trúc, không xác định	
Q96	Hội chứng Turner	
Q96.0	Công thức nhiễm sắc thể 45,X	
Q96.1	Công thức nhiễm sắc thể 46, X iso (Xq)	
Q96.2	Công thức nhiễm sắc thể 46,X với nhiễm sắc thể giới tính bất thường, trừ iso (Xq)	
Q96.3	Thể khảm, 45,X/46,XX hay XY	
Q96.4	Thể khảm, 45,X /dòng tế bào khác có nhiễm sắc thể giới tính bất thường	
Q96.8	Thể khác của hội chứng Turner	
Q96.9	Hội chứng Turner, không xác định	
Q97	Bất thường nhiễm sắc thể giới tính khác, kiểu hình nữ, không phân loại mục khác	
Q97.0	Công thức nhiễm sắc thể 47,XXX	
Q97.1	Nữ có hơn 3 nhiễm sắc thể X	
Q97.2	Thể khảm, dòng có số lượng nhiễm sắc thể X khác nhau	
Q97.3	Nữ có công thức nhiễm sắc thể 46,XY	
Q97.8	Bất thường nhiễm sắc thể giới tính xác định khác, kiểu hình nữ	
Q97.9	Bất thường nhiễm sắc thể giới tính, kiểu hình nữ, không xác định	
Q98	Bất thường nhiễm sắc thể giới tính khác, kiểu hình nam không phân loại mục khác	
Q98.0	Hội chứng Klinefelter có công thức nhiễm sắc thể 47,XXY	
Q98.1	Hội chứng Klinefelter, nam giới có hơn 2 nhiễm sắc thể X	
Q98.2	Hội chứng Klinefelter, nam giới có công thức nhiễm sắc thể 46,XX	
Q98.3	Nam giới khác có công thức nhiễm sắc thể 46,XX	
Q98.4	Hội chứng Klinefelter, không xác định	
Q98.5	Công thức nhiễm sắc thể 47,XYY	
Q98.6	Nam có cấu trúc nhiễm sắc thể giới tính bất thường	
Q98.7	Nam có thể khảm nhiễm sắc thể giới tính	
Q98.8	Bất thường nhiễm sắc thể giới tính xác định khác, kiểu hình nam	
Q98.9	Bất thường nhiễm sắc thể giới tính, kiểu hình nam, không xác định	
Q99	Bất thường nhiễm sắc thể khác không phân loại mục khác	
Q99.0	Hợp thể khảm 46, XX/46,XY	Hợp thể khảm 46, XX/46,XY lưỡng giới thật
Q99.1	Lưỡng tính thật 46,XX	46,XX với tuyến sinh dục sọc|46,XY với tuyến sinh dục sọc|Loạn sản tuyến sinh dục đơn thuần [bất sản hoặc thiểu sản buồng trứng]
Q99.2	Nhiễm sắc thể X dễ gãy [Fragile X]	Hội chứng nhiễm sắc thể X dễ gãy
Q99.8	Bất thường nhiễm sắc thể xác định khác	
Q99.9	Bất thường nhiễm sắc thể, không xác định	
R00	Nhịp tim bất thường	
R00.0	Nhịp tim nhanh, không xác định	Tim đập nhanh
R00.1	Nhịp tim chậm, không xác định	
R00.2	Đánh trống ngực	Nhận biết nhịp tim
R00.3	Ngừng tim với hoạt động điện vô mạch, không phân loại mục khác	
R00.8	Nhịp tim bất thường khác và/hoặc không xác định	
R01	Tiếng thổi của tim và/hoặc các âm thanh khác của tim	
R01.0	Tiếng thổi lành tính và/hoặc không có hại của tim	Tiếng thổi chức năng của tim
R01.1	Tiếng thổi của tim, không xác định	Tiếng thổi mạch máu không xác định khác|Tiếng thổi tâm thu không xác định khác
R01.2	Âm thanh khác của tim	Diện đục tim, tăng hay giảm|Tiếng cọ sát vùng thượng vị
R02	Hoại thư, không phân loại mục khác	
R03	Chỉ số huyết áp bất thường, không có chẩn đoán	
R03.0	Chỉ số huyết áp tăng, không chẩn đoán tăng huyết áp	
R03.1	Chỉ số huyết áp thấp không xác định số cụ thể	
R04	Chảy máu đường hô hấp	
R04.0	Chảy máu cam	Chảy máu từ mũi|Chảy máu mũi
R04.1	Chảy máu họng	
R04.2	Ho ra máu	Đờm nhuốm máu|Ho có chảy máu
R04.8	Xuất huyết từ các vị trí khác của đường hô hấp	
R04.9	Xuất huyết đường hô hấp, không xác định	
R05	Ho	
R06	Nhịp thở bất thường	
R06.0	Khó thở	
R06.1	Thở rít	
R06.2	Thở khò khè	
R06.3	Thở có tính chu kỳ	Nhịp thở Cheyne-Stokes
R06.4	Tăng thông khí	
R06.5	Thở bằng miệng	
R06.6	Nấc cụt	
R06.7	Hắt hơi	
R06.8	Nhịp thở bất thường khác và/hoặc không xác định	
R07	Đau họng và/hoặc đau ngực	
R07.0	Đau họng	
R07.1	Đau ngực khi thở	Đau khi thở
R07.2	Đau vùng trước tim	
R07.3	Đau ngực khác	Đau thành ngực trước không xác định khác
R07.4	Đau ngực, không xác định	
R09	Triệu chứng và/hoặc dấu hiệu khác liên quan đến hệ tuần hoàn và/hoặc hệ hô hấp	
R09.0	Ngạt thở	
R09.1	Viêm màng phổi	
R09.2	Ngừng hô hấp	Suy tim - hô hấp
R09.3	Đờm bất thường	
R09.8	Triệu chứng và/hoặc dấu hiệu xác định khác liên quan tới hệ tuần hoàn và/hoặc hô hấp	
R10	Đau vùng bụng và/hoặc chậu	
R10.0	Đau bụng cấp tính	
R10.1	Đau khu trú bụng trên	
R10.2	Đau vùng chậu và/hoặc tầng sinh môn	
R10.3	Đau khu trú tại vùng khác của bụng dưới	
R10.4	Đau bụng khác và/hoặc không xác định	Bụng mềm ấn đau không xác định khác
R11	Buồn nôn và/hoặc nôn	
R12	Ợ nóng [đau rát ngực]	
R13	Khó nuốt	
R14	Đầy hơi và/hoặc tình trạng liên quan	
R15	Đại tiện không tự chủ	
R16	Gan to và/hoặc lá lách to, không phân loại mục khác	
R16.0	Gan to, không phân loại mục khác	Gan to không xác định khác
R16.1	Lách to, không phân loại mục khác	Lách to không xác định khác
R16.2	Gan to kèm lách to, không phân loại mục khác	Gan lách to không xác định khác
R17	Tăng bilirubin máu, có hoặc không vàng da, không phân loại mục khác	
R17.0	Tăng bilirubin máu, có đề cập vàng da, không phân loại mục khác	Vàng da không xác định khác
R17.9	Tăng bilirubin máu, không đề cập vàng da, không phân loại mục khác	Tăng bilirubin máu không xác định khác
R18	Chứng cổ trướng	
R19	Triệu chứng và/hoặc dấu hiệu khác liên quan tới hệ tiêu hóa và/hoặc bụng	
R19.0	Khối và/hoặc mảng sưng phồng ở vùng bụng và/hoặc vùng chậu	
R19.1	Tiếng ruột bất thường	Không có âm thanh của ruột|Âm thanh của ruột quá mức
R19.2	Nhu động ruột nổi [có thể nhìn thấy được]	Tăng nhu động ruột
R19.3	Cứng bụng	
R19.4	Thay đổi thói quen đại tiện	
R19.5	Bất thường khác của phân	
R19.6	Chứng hôi miệng	
R19.8	Triệu chứng và/hoặc dấu hiệu xác định khác liên quan tới hệ tiêu hóa và/hoặc bụng	
R20	Rối loạn cảm giác da	
R20.0	Mất cảm giác da	
R20.1	Giảm cảm giác da	
R20.2	Dị cảm da	
R20.3	Tăng cảm giác	
R20.8	Rối loạn cảm giác da khác và/ hoặc không xác định	
R21	Ban da và/hoặc phát ban không xác định cụ thể	
R22	Sưng cục bộ, khối và/hoặc u ở da và/hoặc mô dưới da	
R22.0	Sưng khu trú, khối và/hoặc u ở đầu	
R22.1	Sưng khu trú, khối và/hoặc u ở cổ	
R22.2	Sưng khu trú, khối và/hoặc u ở thân mình	
R22.3	Sưng khu trú, khối và/hoặc u ở chi trên	
R22.4	Sưng khu trú, khối và/hoặc u ở chi dưới	
R22.7	Sưng khu trú, khối và/hoặc u ở nhiều vị trí	
R22.9	Sưng khu trú, khối và/hoặc u, không xác định	
R23	Thay đổi khác của da	
R23.0	Chứng xanh tím da	
R23.1	Da xanh xao, tái nhợt	Da ẩm và lạnh
R23.2	Chứng đỏ bừng mặt	
R23.3	Bầm máu tự phát	
R23.4	Thay đổi kết cấu da	
R23.8	Thay đổi khác và/hoặc không xác định của da	
R25	Vận động không tự chủ bất thường	
R25.0	Cử động đầu bất thường	
R25.1	Run, không xác định	
R25.2	Chuột rút và/hoặc co thắt	
R25.3	Co cứng cơ cục bộ	Giật cơ không xác định khác
R25.8	Vận động không tự chủ bất thường khác và/hoặc không xác định	
R26	Bất thường về dáng đi và/hoặc vận động	
R26.0	Dáng đi mất điều hòa vận động	Dáng đi lảo đảo [loạng choạng]
R26.1	Dáng đi liệt	Dáng đi co cứng [co giật]
R26.2	Đi bộ khó khăn, không phân loại mục khác	
R26.3	Bất động	Liệt giường|Ngồi liệt [không thể di chuyển ra khỏi ghế ngồi]
R26.8	Bất thường dáng đi và/hoặc vận động khác và/hoặc không xác định	Không đứng vững không xác định khác
R27	Thiếu phối hợp vận động khác	
R27.0	Mất điều hòa vận động [thất điều], không xác định	
R27.8	Thiếu phối hợp vận động khác và/hoặc không xác định	
R29	Triệu chứng và/hoặc dấu hiệu khác liên quan tới hệ thần kinh và/hoặc cơ xương khớp	
R29.0	Cơn co thắt không tự chủ [tetany]	
R29.1	Kích thích màng não [biểu hiện bằng gáy cứng, đau đầu]	
R29.2	Phản xạ bất thường	
R29.3	Tư thế bất thường	
R29.4	Khớp háng kêu lách cách	
R29.6	Dễ ngã, không phân loại mục khác	
R29.8	Triệu chứng và/hoặc dấu hiệu khác và/hoặc không xác định liên quan đến hệ thần kinh và/hoặc hệ cơ xương khớp	
R30	Đau liên quan với tiểu tiện	
R30.0	Khó tiểu tiện	
R30.1	Buồn tiểu sau khi tiểu tiện do cơ bàng quang quá kích thích	
R30.9	Tiểu tiện gây đau, không xác định	Tiểu tiện đau không xác định khác
R31	Tiểu máu không xác định	
R32	Tiểu tiện không tự chủ không xác định	
R33	Bí tiểu	
R34	Vô niệu và/hoặc thiểu niệu	
R35	Đa niệu	
R36	Dịch tiết từ niệu đạo	
R39	Triệu chứng và/hoặc dấu hiệu khác liên quan tới hệ tiết niệu	
R39.0	Thoát mạch nước tiểu [Viêm tấy do nước tiểu]	
R39.1	Khó tiểu tiện khác	Chứng ngại tiểu tiện|Dòng nước tiểu yếu|Dòng nước tiểu phân tách
R39.2	Tăng urê huyết ngoài thận	Tăng urê huyết trước thận [trước tuyến thượng thận]
R39.8	Triệu chứng và dấu hiệu khác và/hoặc không xác định liên quan tới hệ tiết niệu	
R40	Buồn ngủ, ngẩn ngơ và/hoặc hôn mê	
R40.0	Buồn ngủ	Ngủ lơ mơ [ngủ gà]
R40.1	Trạng thái ngẩn ngơ	
R40.2	Hôn mê, không xác định	Bất tỉnh không xác định khác
R41	Triệu chứng và/hoặc dấu hiệu khác liên quan đến chức năng nhận thức và/hoặc tri giác	
R41.0	Mất ý thức, không xác định	
R41.1	Chứng quên thuận chiều [không có khả năng hình thành ký ức mới sau khi biến cố gây bệnh xảy ra]	
R41.2	Chứng quên ngược chiều [không nhớ được các ký ức cũ trước khi biến cố gây bệnh xảy ra]	
R41.3	Chứng quên khác	
R41.8	Triệu chứng và dấu hiệu khác và/hoặc không xác định liên quan đến chức năng nhận thức và/hoặc nhận biết	
R42	Hoa mắt và/hoặc chóng mặt	
R43	Rối loạn khứu giác và/hoặc vị giác	
R43.0	Mất khứu giác	
R43.1	Rối loạn khứu giác	
R43.2	Rối loạn vị giác	
R43.8	Rối loạn khứu giác và/ hoặc vị giác khác và/hoặc không xác định	Rối loạn hỗn hợp khứu giác và vị giác
R44	Triệu chứng và/hoặc dấu hiệu khác về cảm giác và/hoặc tri giác	
R44.0	Ảo thanh	
R44.1	Ảo thị giác	
R44.2	Ảo giác khác	
R44.3	Ảo giác, không xác định	
R44.8	Triệu chứng và dấu hiệu khác và/hoặc không xác định liên quan đến cảm giác và/hoặc tri giác	
R45	Triệu chứng và/hoặc dấu hiệu về trạng thái cảm xúc	
R45.0	Lo lắng [bồn chồn]	Căng thẳng thần kinh
R45.1	Tình trạng bứt rứt [không yên] và/hoặc kích động	
R45.2	Buồn phiền [không hài lòng]	Lo lắng không xác định khác
R45.3	Làm mất tinh thần và/hoặc thờ ơ	
R45.4	Cáu gắt và/hoặc tức giận	
R45.5	Thái độ thù địch [phản đối] [chống đối]	
R45.6	Bạo lực thể xác	
R45.7	Trạng thái sốc tâm lý và/hoặc căng thẳng, không xác định	
R45.8	Triệu chứng và/hoặc dấu hiệu khác liên quan đến trạng thái cảm xúc	
R46	Triệu chứng và/hoặc dấu hiệu liên quan đến diện mạo bên ngoài và/hoặc hành vi	Triệu chứng và/hoặc dấu hiệu khác liên quan đến diện mạo bên ngoài và/hoặc hành vi
R46.0	Vệ sinh cá nhân rất kém	
R46.1	Diện mạo cá nhân kỳ lạ	
R46.2	Hành vi kỳ lạ và/hoặc khó hiểu	
R46.3	Tăng động	
R46.4	Phản ứng kém và/hoặc chậm chạp	
R46.5	Tính đa nghi và/hoặc lảng tránh rõ rệt	
R46.6	Lo lắng và/hoặc bận tâm thái quá vì các sự kiện căng thẳng	
R46.7	Chứng nói dài và/hoặc quá rườm rà gây khó hiểu lý do tiếp cận dịch vụ y tế	
R46.8	Triệu chứng và/hoặc dấu hiệu khác liên quan đến diện mạo bên ngoài và/hoặc hành vi	
R47	Rối loạn ngôn ngữ, không phân loại mục khác	
R47.0	Loạn ngôn [giọng nói bất thường] và/hoặc chứng thất ngôn [khó khăn khi nói]	
R47.1	Rối loạn vận ngôn và/hoặc mất vận ngôn	
R47.8	Rối loạn ngôn ngữ khác và/hoặc không xác định	
R48	Chứng khó đọc và/hoặc các rối loạn chức năng biểu đạt khác, không phân loại mục khác	
R48.0	Chứng khó đọc và/hoặc mất khả năng đọc	
R48.1	Mất nhận thức [vong tri]	
R48.2	Chứng mất phối hợp động tác	
R48.8	Rối loạn chức năng biểu đạt khác và/hoặc không xác định	Mất khả năng tính toán|Mất khả năng viết
R49	Rối loạn giọng nói	
R49.0	Rối loạn phát âm	Khàn giọng
R49.1	Chứng mất tiếng	Mất giọng
R49.2	Giọng mũi nhiều và/hoặc giọng mũi ít	
R49.8	Rối loạn giọng nói khác và/hoặc không xác định	Thay đổi giọng nói không xác định khác
R50	Sốt không rõ nguyên nhân và/hoặc sốt khác	
R50.2	Sốt do dùng thuốc	
R50.8	Sốt xác định khác	Sốt kèm rùng mình|Sốt kèm rét run|Sốt dai dẳng
R50.9	Sốt, không xác định	
R51	Đau đầu	
R52	Đau, không phân loại mục khác	
R52.0	Đau cấp tính	
R52.1	Đau mạn tính khó chữa hoặc quản lý	
R52.2	Đau mạn tính khác	
R52.9	Đau, không xác định	Đau toàn thân [đau ở ba bộ phận cơ thể trở lên] không xác định khác
R53	Khó chịu và/hoặc mệt mỏi	
R54	Suy yếu do tuổi già	
R55	Ngất và/hoặc ngã quỵ	
R56	Co giật, không phân loại mục khác	
R56.0	Co giật do sốt	
R56.8	Co giật khác và/hoặc không xác định	
R57	Sốc, không phân loại mục khác	
R57.0	Sốc tim	
R57.1	Sốc giảm thể tích	
R57.2	Sốc nhiễm trùng	
R57.8	Sốc khác	Sốc do nội độc tố
R57.9	Sốc, không xác định	Suy tuần hoàn ngoại vi không xác định khác
R58	Xuất huyết, không phân loại mục khác	
R59	Chứng phì đại hạch bạch huyết	
R59.0	Chứng phì đại hạch bạch huyết khu trú	
R59.1	Chứng phì đại hạch bạch huyết toàn thân	
R59.9	Chứng phì đại hạch bạch huyết, không xác định	
R60	Phù, không phân loại mục khác	
R60.0	Phù khu trú	
R60.1	Phù toàn thân	
R60.9	Phù, không xác định	Ứ dịch không xác định khác
R61	Tăng tiết mồ hôi	
R61.0	Tăng tiết mồ hôi khu trú	
R61.1	Tăng tiết mồ hôi toàn thân	
R61.9	Tăng tiết mồ hôi, không xác định	Tiết mồ hôi quá mức|Tiết mồ hôi ban đêm
R62	Phát triển sinh lý không bình thường như kỳ vọng	
R62.0	Chậm đạt các mốc phát triển	Chậm đạt được giai đoạn phát triển sinh lý như kỳ vọng
R62.8	Phát triển sinh lý không bình thường như kỳ vọng khác	
R62.9	Phát triển sinh lý không bình thường như kỳ vọng, không xác định	
R63	Triệu chứng và/hoặc dấu hiệu liên quan đến ăn uống thức ăn và/hoặc đồ uống	
R63.0	Chán ăn [lười ăn]	
R63.1	Chứng khát nước nhiều	Khát nước quá mức
R63.2	Đói quá mức	Ăn quá mức|Thừa dinh dưỡng không xác định khác
R63.3	Khó khăn và/hoặc quản lý kém về cho ăn	
R63.4	Giảm cân bất thường	
R63.5	Tăng cân bất thường	
R63.6	Ăn uống không đủ thực phẩm và/hoặc nước uống	
R63.8	Triệu chứng và/hoặc dấu hiệu khác liên quan đến ăn uống thức ăn và/hoặc đồ uống	
R64	Suy mòn	
R65	Hội chứng đáp ứng viêm hệ thống [SIRS]	
R65.0	Hội chứng đáp ứng viêm hệ thống do nhiễm trùng không kèm suy tạng	
R65.1	Hội chứng đáp ứng viêm hệ thống do nhiễm trùng có kèm suy tạng	Nhiễm trùng hệ thống nặng
R65.2	Hội chứng đáp ứng viêm hệ thống không do nhiễm trùng không kèm suy tạng	
R65.3	Hội chứng đáp ứng viêm hệ thống không do nhiễm trùng có kèm suy tạng	
R65.9	Hội chứng đáp ứng viêm hệ thống, không xác định	
R68	Triệu chứng và/hoặc dấu hiệu toàn thể khác	
R68.0	Hạ thân nhiệt, không liên quan đến nhiệt độ môi trường thấp	
R68.1	Triệu chứng không cụ thể đặc trưng của trẻ sơ sinh	
R68.2	Chứng khô miệng, không xác định	
R68.3	Ngón tay dùi trống	
R68.8	Triệu chứng và/hoặc dấu hiệu toàn thể xác định khác	
R69	Nguyên nhân mắc bệnh không xác định và/hoặc không rõ	
R70	Tăng tốc độ lắng hồng cầu và/hoặc độ nhớt huyết tương bất thường	
R70.0	Tăng tốc độ lắng hồng cầu	
R70.1	Độ nhớt huyết tương bất thường	
R71	Bất thường về hồng cầu	
R72	Bất thường bạch cầu, không phân loại mục khác	
R73	Tăng nồng độ đường huyết	
R73.0	Xét nghiệm dung nạp đường bất thường	
R73.9	Tăng đường huyết, không xác định	
R74	Nồng độ enzym huyết thanh bất thường	
R74.0	Tăng nồng độ enzyme tranzaminase và/hoặc nồng độ acid lactic dehydrogenase (LDH)	
R74.8	Nồng độ enzym huyết thanh khác bất thường	
R74.9	Nồng độ enzym huyết thanh bất thường, không xác định	
R75	Bằng chứng cận lâm sàng của virus gây suy giảm miễn dịch ở người [HIV]	
R76	Phát hiện miễn dịch khác bất thường trong huyết thanh	
R76.0	Tăng độ chuẩn kháng thể	
R76.1	Phản ứng xét nghiệm tuberculin bất thường	Kết quả xét nghiệm Mantoux bất thường
R76.2	Xét nghiệm huyết thanh dương tính giả với giang mai	Phản ứng Wasserman dương tính giả
R76.8	Phát hiện miễn dịch bất thường xác định khác trong huyết thanh	Tăng nồng độ immunoglobulin không xác định khác
R76.9	Phát hiện miễn dịch bất thường trong huyết thanh, không xác định	
R77	Bất thường khác của protein huyết tương	
R77.0	Bất thường của albumin	
R77.1	Bất thường của globulin	Chứng tăng globulin máu không xác định khác
R77.2	Bất thường của alphafetoprotein	
R77.8	Bất thường xác định khác của protein huyết tương	
R77.9	Bất thường của protein huyết tương, không xác định	
R78	Phát hiện ma túy và/hoặc chất khác, không thường có trong máu	
R78.0	Phát hiện có cồn trong máu	
R78.1	Phát hiện có thuốc có nguồn gốc thuốc phiện trong máu	
R78.2	Phát hiện cocain trong máu	
R78.3	Phát hiện chất gây ảo giác trong máu	
R78.4	Phát hiện thuốc có khả năng gây nghiện khác trong máu	
R78.5	Phát hiện thuốc hướng thần trong máu	
R78.6	Phát hiện tác nhân steroid trong máu	
R78.7	Phát hiện nồng độ kim loại nặng bất thường trong máu	
R78.8	Phát hiện chất xác định khác, không thường có trong máu	Phát hiện nồng độ lithium bất thường trong máu
R78.9	Phát hiện chất không xác định, không thường có trong máu	
R79	Phát hiện sinh hóa máu bất thường khác	
R79.0	Bất thường nồng độ khoáng chất trong máu	
R79.8	Phát hiện sinh hóa máu bất thường xác định khác	Nồng độ khí máu bất thường
R79.9	Phát hiện sinh hóa máu bất thường, không xác định	
R80	Protein niệu đơn độc	
R81	Glucoza niệu	
R82	Phát hiện bất thường khác trong nước tiểu	
R82.0	Dưỡng chấp niệu	
R82.1	Myoglobin niệu	
R82.2	Đái sắc tố mật	
R82.3	Đái huyết sắc tố [Haemoglobin niệu]	
R82.4	Aceton niệu	Xeton niệu
R82.5	Tăng nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm trong nước tiểu	
R82.6	Nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	Nồng độ kim loại nặng bất thường trong nước tiểu
R82.7	Phát hiện xét nghiệm vi sinh bất thường trong nước tiểu	Các phát hiện về nuôi cấy dương tính
R82.8	Phát hiện xét nghiệm tế bào và/hoặc mô học bất thường trong nước tiểu	
R82.9	Phát hiện bất thường khác và/hoặc không xác định trong nước tiểu	Các tế bào niệu và trụ niệu|Tinh thể niệu|Melanin niệu
R83	Phát hiện bất thường trong dịch não tủy	
R83.0	Phát hiện bất thường trong dịch não tủy, nồng độ enzym bất thường	
R83.1	Phát hiện bất thường trong dịch não tủy, nồng độ nội tiết tố [hormon] bất thường	
R83.2	Phát hiện bất thường trong dịch não tủy, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R83.3	Phát hiện bất thường trong dịch não tủy, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R83.4	Phát hiện bất thường trong dịch não tủy, phát hiện miễn dịch bất thường	
R83.5	Phát hiện bất thường trong dịch não tủy, phát hiện vi sinh bất thường	
R83.6	Phát hiện bất thường trong dịch não tủy, phát hiện tế bào bất thường	
R83.7	Phát hiện bất thường trong dịch não tủy, phát hiện mô học bất thường	
R83.8	Phát hiện bất thường trong dịch não tủy, phát hiện bất thường khác	
R83.9	Phát hiện bất thường trong dịch não tủy, phát hiện bất thường không xác định	
R84	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực	
R84.0	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, nồng độ enzym bất thường	
R84.1	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, nồng độ nội tiết tố [hormon] bất thường	
R84.2	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R84.3	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R84.4	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện miễn dịch bất thường	
R84.5	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện vi sinh bất thường	
R84.6	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện tế bào bất thường	
R84.7	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện mô học bất thường	
R84.8	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện bất thường khác	
R84.9	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan hô hấp và/hoặc lồng ngực, phát hiện bất thường không xác định	
R85	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng	
R85.0	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, nồng độ enzym bất thường	
R85.1	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, nồng độ nội tiết tố [hormon] bất thường	
R85.2	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R85.3	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R85.4	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện miễn dịch bất thường	
R85.5	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện vi sinh bất thường	
R85.6	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện tế bào bất thường	
R85.7	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện mô học bất thường	
R85.8	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện bất thường khác	
R85.9	Phát hiện bất thường trong mẫu bệnh phẩm từ cơ quan tiêu hóa và/hoặc ổ bụng, phát hiện bất thường không xác định	
R86	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam	
R86.0	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, nồng độ enzym bất thường	
R86.1	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, nồng độ nội tiết tố [hormon] bất thường	
R86.2	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R86.3	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R86.4	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện miễn dịch bất thường	
R86.5	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện vi sinh bất thường	
R86.6	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện tế bào bất thường	
R86.7	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện mô học bất thường	
R86.8	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện bất thường khác	
R86.9	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nam, phát hiện bất thường không xác định	
R87	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ	
R87.0	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, nồng độ enzym bất thường	
R87.1	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, nồng độ nội tiết tố [hormon] bất thường	
R87.2	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R87.3	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R87.4	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện miễn dịch bất thường	
R87.5	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện vi sinh bất thường	
R87.6	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện tế bào bất thường	
R87.7	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện mô học bất thường	
R87.8	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện bất thường khác	
R87.9	Phát hiện bất thường trong mẫu bệnh phẩm của cơ quan sinh dục nữ, phát hiện bất thường không xác định	
R89	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác	
R89.0	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, nồng độ enzym bất thường	
R89.1	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, nồng độ nội tiết tố [hormon] bất thường	
R89.2	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, nồng độ dược chất, thuốc điều trị và/hoặc sinh phẩm bất thường	
R89.3	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, nồng độ bất thường của chất có nguồn gốc chủ yếu không phải là thuốc	
R89.4	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện miễn dịch bất thường	
R89.5	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện vi sinh bất thường	
R89.6	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện tế bào bất thường	
R89.7	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện mô học bất thường	
R89.8	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện bất thường khác	
R89.9	Phát hiện bất thường trong mẫu bệnh phẩm của các cơ quan, hệ thống và/hoặc mô khác, phát hiện bất thường không xác định	
R90	Phát hiện bất thường của chẩn đoán hình ảnh ở hệ thần kinh trung ương	
R90.0	Tổn thương choán chỗ nội sọ	
R90.8	Phát hiện bất thường khác của chẩn đoán hình ảnh ở hệ thần kinh trung ương	Siêu âm não bất thường|Bệnh thoái hóa chất trắng không xác định khác
R91	Phát hiện bất thường của chẩn đoán hình ảnh ở phổi	
R92	Phát hiện bất thường của chẩn đoán hình ảnh ở ngực	
R93	Phát hiện bất thường của chẩn đoán hình ảnh ở cơ quan khác	
R93.0	Phát hiện bất thường của chẩn đoán hình ảnh ở xương sọ và/hoặc đầu, không phân loại mục khác	
R93.1	Phát hiện bất thường của chẩn đoán hình ảnh ở tim và/hoặc mạch vành	
R93.2	Phát hiện bất thường của chẩn đoán hình ảnh ở gan và/hoặc ống mật	Không nhận dạng được túi mật
R93.3	Phát hiện bất thường của chẩn đoán hình ảnh phần đường tiêu hóa khác	
R93.4	Phát hiện bất thường của chẩn đoán hình ảnh ở cơ quan tiết niệu	
R93.5	Phát hiện bất thường của chẩn đoán hình ảnh ở vùng bụng khác, bao gồm vùng sau phúc mạc	
R93.6	Phát hiện bất thường của chẩn đoán hình ảnh ở chi	
R93.7	Phát hiện bất thường của chẩn đoán hình ảnh ở phần khác của hệ cơ xương khớp	
R93.8	Phát hiện bất thường của chẩn đoán hình ảnh ở cấu trúc cơ thể xác định khác	Phát hiện hình ảnh bất thường ở da và/hoặc mô dưới da|Di lệch cấu trúc trung thất sang một bên
R94	Kết quả bất thường của thăm dò chức năng	
R94.0	Kết quả bất thường của thăm dò chức năng hệ thần kinh trung ương	Điện não đồ bất thường [EEG]
R94.1	Kết quả bất thường của thăm dò chức năng hệ thần kinh ngoại biên và/hoặc giác quan đặc biệt [đặc biệt bao gồm thị, thính, khứu, vị giác]	
R94.2	Kết quả bất thường của thăm dò chức năng phổi	
R94.3	Kết quả bất thường của thăm dò chức năng tuần hoàn	
R94.4	Kết quả bất thường của thăm dò chức năng thận	Xét nghiệm chức năng thận bất thường
R94.5	Kết quả bất thường của thăm dò chức năng gan	
R94.6	Kết quả bất thường của thăm dò chức năng tuyến giáp	
R94.7	Kết quả bất thường của thăm dò chức năng nội tiết khác	
R94.8	Kết quả bất thường của thăm dò chức năng cơ quan và/hoặc hệ thống khác	
R95	Hội chứng đột tử ở trẻ sơ sinh	
R95.0	Hội chứng đột tử ở trẻ sơ sinh, có đề cập đến khám nghiệm tử thi	
R95.9	Hội chứng đột tử ở trẻ sơ sinh, không đề cập đến khám nghiệm tử thi	Hội chứng đột tử ở trẻ sơ sinh, không xác định
R96	Đột tử khác, không rõ nguyên nhân	
R96.0	Đột tử	
R96.1	Tử vong dưới 24 giờ sau khi khởi phát các triệu chứng, không giải thích được bằng cách khác	Tử vong được xác định không do bạo lực hay đột tử không phát hiện được nguyên nhân|Tử vong không có dấu hiệu bệnh
R98	Tử vong một mình [không ai biết]	
R99	Nguyên nhân tử vong không rõ ràng và/hoặc không xác định khác	
S00	Tổn thương nông ở đầu	
S00.0	Tổn thương nông của da đầu	
S00.1	Đụng giập mi mắt và/hoặc vùng quanh mắt	
S00.2	Tổn thương nông khác ở mi mắt và/hoặc vùng quanh mắt	
S00.3	Tổn thương nông ở mũi	
S00.4	Tổn thương nông ở tai	
S00.5	Tổn thương nông ở môi và/hoặc khoang miệng	
S00.7	Đa tổn thương nông ở đầu	
S00.8	Tổn thương nông ở phần khác của đầu	
S00.9	Tổn thương nông ở đầu, phần không xác định	
S01	Vết thương hở ở đầu	
S01.0	Vết thương hở ở da đầu	
S01.1	Vết thương hở ở mi mắt và/hoặc vùng quanh mắt	Vết thương hở ở mi mắt và/hoặc quanh mắt có hoặc không liên quan đến tuyến lệ
S01.2	Vết thương hở ở mũi	
S01.3	Vết thương hở ở tai	
S01.4	Vết thương hở ở má và/hoặc vùng thái dương - xương hàm dưới	
S01.5	Vết thương hở ở môi và/hoặc khoang miệng	
S01.7	Đa vết thương hở ở đầu	
S01.8	Vết thương hở ở phần khác của đầu	
S01.9	Vết thương hở ở đầu, phần không xác định	
S02	Vỡ xương sọ và/hoặc gãy xương mặt	
S02.0	Vỡ xương vòm sọ	Xương trán|Xương đỉnh
S02.00	Vỡ xương vòm sọ, vỡ kín	
S02.01	Vỡ xương vòm sọ, vỡ hở	
S02.1	Vỡ xương nền sọ	
S02.10	Vỡ xương nền sọ, vỡ kín	
S02.11	Vỡ xương nền sọ, vỡ hở	
S02.2	Gãy xương chính mũi	
S02.20	Gãy xương chính mũi, gãy kín	
S02.21	Gãy xương chính mũi, gãy hở	
S02.3	Gãy xương sàn hốc mắt	
S02.30	Gãy xương sàn hốc mắt, gãy kín	
S02.31	Gãy xương sàn hốc mắt, gãy hở	
S02.4	Gãy xương gò má và/hoặc xương hàm trên	
S02.40	Gãy xương gò má và/hoặc xương hàm trên, gãy kín	
S02.41	Gãy xương gò má và/hoặc xương hàm trên, gãy hở	
S02.5	Gãy răng	Răng vỡ
S02.50	Gãy răng, gãy kín	
S02.51	Gãy răng, gãy hở	
S02.6	Gãy xương hàm dưới	
S02.60	Gãy xương hàm dưới, gãy kín	
S02.61	Gãy xương hàm dưới, gãy hở	
S02.7	Gãy [vỡ] nhiều xương và/hoặc nhiều vị trí ở xương sọ và/hoặc xương mặt	
S02.70	Gãy [vỡ] nhiều xương và/hoặc nhiều vị trí ở xương sọ và/hoặc xương mặt, vỡ kín	
S02.71	Gãy [vỡ] nhiều xương và/hoặc nhiều vị trí ở xương sọ và/hoặc xương mặt, vỡ hở	
S02.8	Gãy xương khác của sọ và/hoặc mặt	
S02.80	Gãy xương khác của sọ và/hoặc mặt, gãy kín	
S02.81	Gãy xương khác của sọ và/hoặc mặt, gãy hở	
S02.9	Vỡ xương sọ và/hoặc gãy xương mặt, phần không xác định	
S02.90	Vỡ xương sọ và/hoặc gãy xương mặt, phần không xác định, vỡ kín	
S02.91	Vỡ xương sọ và/hoặc gãy xương mặt, phần không xác định, vỡ hở	
S03	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng của đầu	
S03.0	Trật khớp hàm	
S03.1	Lệch sụn vách mũi	
S03.2	Trật khớp răng	
S03.3	Di lệch của phần không xác định ở đầu	
S03.4	Giãn dây chằng [bong gân] và/hoặc căng cơ của hàm	
S03.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng của phần đầu khác và/hoặc không xác định	
S04	Tổn thương dây thần kinh sọ	
S04.0	Tổn thương dây thần kinh thị giác và/hoặc đường dẫn truyền thị giác	Giao thoa thị giác|Dây thần kinh sọ số II|Vỏ não thị giác
S04.1	Tổn thương dây thần kinh vận nhãn	Dây thần kinh sọ số III
S04.2	Tổn thương dây thần kinh ròng rọc	Dây thần kinh sọ số IV
S04.3	Tổn thương dây thần kinh sinh ba	Dây thần kinh sọ số V
S04.4	Tổn thương dây thần kinh vận nhãn ngoài	Dây thần kinh sọ số VI
S04.5	Tổn thương dây thần kinh mặt	Dây thần kinh sọ số VII
S04.6	Tổn thương dây thần kinh thính giác	Dây thần kinh tiền đình - ốc tai|Dây thần kinh sọ số VIII
S04.7	Tổn thương dây thần kinh phụ	Dây thần kinh sọ số XI
S04.8	Tổn thương dây thần kinh sọ khác	Dây thần kinh thiệt hầu [số IX]|Dây thần kinh hạ thiệt [số XII]|Dây thần kinh khứu giác [số I]|Dây thần kinh phế vị [số X]
S04.9	Tổn thương dây thần kinh sọ không xác định	
S05	Tổn thương mắt và/hoặc hốc mắt	
S05.0	Tổn thương kết mạc và/hoặc xước giác mạc không đề cập việc có dị vật	
S05.1	Đụng giập nhãn cầu và/hoặc mô hốc mắt	
S05.2	Rách xé nhãn cầu và/hoặc thủng kèm sa hoặc mất mô nội nhãn	
S05.3	Rách xé nhãn cầu không kèm sa hoặc mất mô nội nhãn	Rách xé mắt không xác định khác
S05.4	Vết thương xuyên thấu hốc mắt có hay không có dị vật	
S05.5	Vết thương xuyên thấu nhãn cầu có dị vật	
S05.6	Vết thương xuyên thấu nhãn cầu không có dị vật	Xuyên thấu nhãn cầu không xác định khác
S05.7	Giật đứt nhãn cầu	Khoét bỏ [bóc] nhãn cầu do chấn thương
S05.8	Tổn thương khác ở mắt và/hoặc hốc mắt	Tổn thương lệ đạo
S05.9	Tổn thương ở mắt và/hoặc hốc mắt, không xác định	Tổn thương mắt không xác định khác
S06	Tổn thương nội sọ	
S06.0	Chấn động não	
S06.00	Chấn động não, không có vết thương nội sọ hở	
S06.01	Chấn động não, có vết thương nội sọ hở	
S06.1	Phù não do chấn thương	
S06.10	Phù não do chấn thương, không có vết thương nội sọ hở	
S06.11	Phù não do chấn thương, có vết thương nội sọ hở	
S06.2	Tổn thương não lan tỏa	
S06.20	Tổn thương não lan tỏa, không có vết thương nội sọ hở	
S06.21	Tổn thương não lan tỏa, có vết thương nội sọ hở	
S06.3	Tổn thương não khu trú	
S06.30	Tổn thương não khu trú, không có vết thương nội sọ hở	
S06.31	Tổn thương não khu trú, có vết thương nội sọ hở	
S06.4	Xuất huyết ngoài màng cứng	
S06.40	Xuất huyết ngoài màng cứng, không có vết thương nội sọ hở	
S06.41	Xuất huyết ngoài màng cứng, có vết thương nội sọ hở	
S06.5	Xuất huyết dưới màng cứng do chấn thương	
S06.50	Xuất huyết dưới màng cứng do chấn thương, không có vết thương nội sọ hở	
S06.51	Xuất huyết dưới màng cứng do chấn thương, có vết thương nội sọ hở	
S06.6	Xuất huyết dưới màng nhện do chấn thương	
S06.60	Xuất huyết dưới màng nhện do chấn thương, không có vết thương nội sọ hở	
S06.61	Xuất huyết dưới màng nhện do chấn thương, có vết thương nội sọ hở	
S06.7	Tổn thương nội sọ có hôn mê kéo dài	
S06.70	Tổn thương nội sọ có hôn mê kéo dài, không có vết thương nội sọ hở	
S06.71	Tổn thương nội sọ có hôn mê kéo dài, có vết thương nội sọ hở	
S06.8	Tổn thương nội sọ khác	
S06.80	Tổn thương nội sọ khác, không có vết thương nội sọ hở	
S06.81	Tổn thương nội sọ khác, có vết thương nội sọ hở	
S06.9	Tổn thương nội sọ, không xác định	
S06.90	Tổn thương nội sọ, không xác định, không có vết thương nội sọ hở	
S06.91	Tổn thương nội sọ, không xác định, có vết thương nội sọ hở	
S07	Tổn thương dập nát ở đầu	
S07.0	Tổn thương dập nát ở mặt	
S07.1	Tổn thương dập nát ở hộp sọ	
S07.8	Tổn thương dập nát ở phần khác của đầu	
S07.9	Tổn thương dập nát của đầu, phần không xác định	
S08	Đứt rời một phần của đầu do chấn thương	
S08.0	Nhổ giật mảng da đầu	
S08.1	Đứt rời tai do chấn thương	
S08.8	Đứt rời phần khác của đầu do chấn thương	
S08.9	Đứt rời phần không xác định của đầu do chấn thương	
S09	Tổn thương khác và/hoặc không xác định ở đầu	
S09.0	Tổn thương mạch máu ở đầu, không phân loại mục khác	
S09.1	Tổn thương cơ và/hoặc gân ở đầu	
S09.2	Rách màng nhĩ do chấn thương	
S09.7	Đa tổn thương ở đầu	Tổn thương có thể phân loại ở nhiều mục trong S00.- - S09.2
S09.8	Tổn thương xác định khác ở đầu	
S09.9	Tổn thương không xác định ở đầu	
S10	Tổn thương nông ở cổ	
S10.0	Đụng giập họng	Thực quản|Thanh quản|Họng|Phế quản
S10.1	Tổn thương nông khác và/hoặc không xác định ở họng	
S10.7	Đa tổn thương nông ở cổ	
S10.8	Tổn thương nông ở phần khác của cổ	
S10.9	Tổn thương nông ở cổ, phần không xác định	
S11	Vết thương hở ở cổ	
S11.0	Vết thương hở tác động đến thanh quản và/hoặc khí quản	
S11.1	Vết thương hở liên quan đến tuyến giáp	
S11.2	Vết thương hở bao gồm hầu và/hoặc thực quản phần cổ	
S11.7	Đa vết thương hở ở cổ	
S11.8	Vết thương hở ở phần khác của cổ	
S11.9	Vết thương hở ở cổ, phần không xác định	
S12	Gãy cổ	
S12.0	Gãy đốt sống cổ thứ nhất	Đốt sống đội
S12.00	Gãy đốt sống cổ thứ nhất, gãy kín	
S12.01	Gãy đốt sống cổ thứ nhất, gãy hở	
S12.1	Gãy đốt sống cổ thứ hai	Đốt sống trục
S12.10	Gãy đốt sống cổ thứ hai, gãy kín	
S12.11	Gãy đốt sống cổ thứ hai, gãy hở	
S12.2	Gãy đốt sống cổ xác định khác	
S12.20	Gãy đốt sống cổ xác định khác, gãy kín	
S12.21	Gãy đốt sống cổ xác định khác, gãy hở	
S12.7	Gãy cột sống cổ, nhiều đốt xương và/hoặc nhiều vị trí	
S12.70	Gãy cột sống cổ, nhiều đốt xương và/hoặc nhiều vị trí, gãy kín	
S12.71	Gãy cột sống cổ, nhiều đốt xương và/hoặc nhiều vị trí, gãy hở	
S12.8	Gãy phần khác của cổ	Xương móng|Thanh quản|Sụn giáp|Khí quản
S12.80	Gãy phần khác của cổ, gãy kín	
S12.81	Gãy phần khác của cổ, gãy hở	
S12.9	Gãy cổ, phần không xác định	
S12.90	Gãy cổ, phần không xác định, gãy kín	
S12.91	Gãy cổ, phần không xác định, gãy hở	
S13	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng vùng cổ	
S13.0	Vỡ đĩa đệm cột sống cổ do chấn thương	
S13.1	Trật khớp đốt sống cổ	Cột sống cổ không xác định khác
S13.2	Trật khớp phần khác và/hoặc không xác định của cổ	
S13.3	Trật khớp phức tạp [nhiều vị trí] của cổ	
S13.4	Giãn dây chằng [bong gân] và/hoặc căng cơ ở cột sống cổ	
S13.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở vùng giáp	
S13.6	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng của phần cổ khác và/hoặc không xác định	
S14	Tổn thương dây thần kinh và/hoặc tủy sống, vùng cổ	
S14.0	Chấn động và/hoặc phù tủy sống cổ	
S14.1	Tổn thương khác và/hoặc không xác định của tủy sống cổ	Tổn thương tủy sống cổ không xác định khác
S14.2	Tổn thương rễ thần kinh của cột sống cổ	
S14.3	Tổn thương đám rối thần kinh cánh tay	
S14.4	Tổn thương đám rối thần kinh ngoại biên ở cổ	
S14.5	Tổn thương dây thần kinh giao cảm cổ	
S14.6	Tổn thương dây thần kinh khác và/hoặc không xác định của cổ	
S15	Tổn thương mạch máu, vùng cổ	
S15.0	Tổn thương động mạch cảnh	
S15.1	Tổn thương động mạch đốt sống	
S15.2	Tổn thương tĩnh mạch cảnh ngoài	
S15.3	Tổn thương tĩnh mạch cảnh trong	
S15.7	Tổn thương nhiều mạch máu, vùng cổ	
S15.8	Tổn thương mạch máu khác, vùng cổ	
S15.9	Tổn thương mạch máu không xác định, vùng cổ	
S16	Tổn thương cơ và/hoặc gân, vùng cổ	
S17	Tổn thương dập nát ở cổ	
S17.0	Tổn thương dập nát ở thanh quản và/hoặc phế quản	
S17.8	Tổn thương dập nát ở phần khác của cổ	
S17.9	Tổn thương dập nát ở cổ, phần không xác định	
S18	Đứt rời vùng cổ do chấn thương	
S19	Tổn thương khác và/hoặc không xác định ở cổ	
S19.7	Đa tổn thương ở cổ	Tổn thương có thể phân loại ở nhiều mục trong S10.- - S18
S19.8	Tổn thương xác định khác ở cổ	
S19.9	Tổn thương không xác định ở cổ	
S20	Tổn thương nông ở ngực	
S20.0	Đụng giập vú	
S20.1	Tổn thương nông khác và/hoặc không xác định ở vú	
S20.2	Đụng giập ngực	
S20.3	Tổn thương nông khác ở thành trước ngực	
S20.4	Tổn thương nông khác ở thành sau ngực	
S20.7	Đa tổn thương nông ở ngực	
S20.8	Tổn thương nông ở phần khác và/hoặc không xác định của ngực	Thành ngực không xác định khác
S21	Vết thương hở ở ngực	
S21.0	Vết thương hở ở vú	
S21.1	Vết thương hở ở thành trước ngực	
S21.2	Vết thương hở ở thành sau ngực	
S21.7	Đa vết thương hở ở thành ngực	
S21.8	Vết thương hở ở phần khác của ngực	
S21.9	Vết thương hở ở ngực, phần không xác định	Thành ngực không xác định khác
S22	Gãy xương sườn, xương ức và/hoặc cột sống ngực	
S22.0	Gãy đốt sống ngực	Gãy cột sống ngực không xác định khác
S22.00	Gãy đốt sống ngực, gãy kín	
S22.01	Gãy đốt sống ngực, gãy hở	
S22.1	Gãy cột sống ngực, nhiều đốt xương và/hoặc nhiều vị trí	
S22.10	Gãy cột sống ngực, nhiều đốt xương và/hoặc nhiều vị trí, gãy kín	
S22.11	Gãy cột sống ngực, nhiều đốt xương và/hoặc nhiều vị trí, gãy hở	
S22.2	Gãy xương ức	
S22.20	Gãy xương ức, gãy kín	
S22.21	Gãy xương ức, gãy hở	
S22.3	Gãy xương sườn	
S22.30	Gãy xương sườn, gãy kín	
S22.31	Gãy xương sườn, gãy hở	
S22.4	Gãy xương sườn, nhiều xương và/hoặc nhiều vị trí	
S22.40	Gãy xương sườn, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S22.41	Gãy xương sườn, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S22.5	Mảng sườn di động	
S22.50	Mảng sườn di động, gãy kín	
S22.51	Mảng sườn di động, gãy hở	
S22.8	Gãy phần khác của xương ngực	
S22.80	Gãy phần khác của xương ngực, gãy kín	
S22.81	Gãy phần khác của xương ngực, gãy hở	
S22.9	Gãy xương ngực, phần không xác định	
S22.90	Gãy xương ngực, phần không xác định, gãy kín	
S22.91	Gãy xương ngực, phần không xác định, gãy hở	
S23	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng ngực	
S23.0	Vỡ đĩa đệm cột sống ngực do chấn thương	
S23.1	Trật khớp đốt sống ngực	Cột sống ngực không xác định khác
S23.2	Trật khớp ở phần khác và/hoặc không xác định của ngực	
S23.3	Giãn dây chằng [bong gân] và/hoặc căng cơ ở cột sống ngực	
S23.4	Giãn dây chằng [bong gân] và/hoặc căng cơ của xương sườn và/hoặc xương ức	
S23.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở phần khác và/hoặc không xác định của ngực	
S24	Tổn thương dây thần kinh và/hoặc tủy sống, vùng ngực	
S24.0	Chấn động và/hoặc phù tủy sống ngực	
S24.1	Tổn thương khác và/hoặc không xác định của tủy sống ngực	
S24.2	Tổn thương rễ thần kinh của cột sống ngực	
S24.3	Tổn thương dây thần kinh ngoại biên của ngực	
S24.4	Tổn thương dây thần kinh giao cảm ở ngực	Đám rối thần kinh tim|Đám rối thần kinh thực quản|Đám rối thần kinh phổi|Hạch hình sao|Hạch giao cảm ngực
S24.5	Tổn thương dây thần kinh khác của ngực	
S24.6	Tổn thương dây thần kinh không xác định của ngực	
S25	Tổn thương mạch máu ở ngực	Tổn thương mạch máu của ngực
S25.0	Tổn thương động mạch chủ ngực	Động mạch chủ không xác định khác
S25.1	Tổn thương động mạch vô danh hoặc động mạch dưới đòn	
S25.2	Tổn thương tĩnh mạch chủ trên	Tĩnh mạch chủ không xác định khác
S25.3	Tổn thương tĩnh mạch vô danh hoặc tĩnh mạch dưới đòn	
S25.4	Tổn thương mạch máu phổi	
S25.5	Tổn thương mạch máu liên sườn	
S25.7	Tổn thương nhiều mạch máu của ngực	
S25.8	Tổn thương mạch máu khác của ngực	Tĩnh mạch đơn|Động mạch hoặc tĩnh mạch vú
S25.9	Tổn thương mạch máu không xác định của vùng ngực	
S26	Tổn thương tim	
S26.0	Tổn thương tim có tràn máu màng tim	
S26.00	Tổn thương tim có tràn máu màng tim, không có vết thương thấu khoang ngực	
S26.01	Tổn thương tim có tràn máu màng tim, có vết thương thấu khoang ngực	
S26.8	Tổn thương khác của tim	
S26.80	Tổn thương khác của tim, không có vết thương thấu khoang ngực	
S26.81	Tổn thương khác của tim, có vết thương thấu khoang ngực	
S26.9	Tổn thương tim, không xác định	
S26.90	Tổn thương tim, không xác định, không có vết thương thấu khoang ngực	
S26.91	Tổn thương tim, không xác định, có vết thương thấu khoang ngực	
S27	Tổn thương nội tạng khác và/hoặc không xác định trong khoang ngực	
S27.0	Tràn khí màng phổi do chấn thương	
S27.00	Tràn khí màng phổi do chấn thương, không có vết thương thấu khoang ngực	
S27.01	Tràn khí màng phổi do chấn thương, có vết thương thấu khoang ngực	
S27.1	Tràn máu màng phổi do chấn thương	
S27.10	Tràn máu màng phổi do chấn thương, không có vết thương thấu khoang ngực	
S27.11	Tràn máu màng phổi do chấn thương, có vết thương thấu khoang ngực	
S27.2	Tràn máu và khí trong màng phổi do chấn thương	
S27.20	Tràn máu và khí trong màng phổi do chấn thương, không có vết thương thấu khoang ngực	
S27.21	Tràn máu và khí trong màng phổi do chấn thương, có vết thương thấu khoang ngực	
S27.3	Tổn thương khác của phổi	
S27.30	Tổn thương khác của phổi, không có vết thương thấu khoang ngực	
S27.31	Tổn thương khác của phổi, có vết thương thấu khoang ngực	
S27.4	Tổn thương phế quản	
S27.40	Tổn thương phế quản, không có vết thương thấu khoang ngực	
S27.41	Tổn thương phế quản, có vết thương thấu khoang ngực	
S27.5	Tổn thương khí quản ngực	
S27.50	Tổn thương khí quản ngực, không có vết thương thấu khoang ngực	
S27.51	Tổn thương khí quản ngực, có vết thương thấu khoang ngực	
S27.6	Tổn thương màng phổi	
S27.60	Tổn thương màng phổi, không có vết thương thấu khoang ngực	
S27.61	Tổn thương màng phổi, có vết thương thấu khoang ngực	
S27.7	Đa tổn thương của nội tạng trong khoang ngực	
S27.70	Đa tổn thương của nội tạng trong khoang ngực, không có vết thương thấu khoang ngực	
S27.71	Đa tổn thương của nội tạng trong khoang ngực, có vết thương thấu khoang ngực	
S27.8	Tổn thương ở tạng xác định khác trong khoang ngực	
S27.80	Tổn thương ở nội tạng xác định khác trong khoang ngực, không có vết thương thấu khoang ngực	
S27.81	Tổn thương ở nội tạng xác định khác trong khoang ngực, có vết thương thấu khoang ngực	
S27.9	Tổn thương nội tạng không xác định trong khoang ngực	
S27.90	Tổn thương nội tạng không xác định trong khoang ngực, không có vết thương thấu khoang ngực	
S27.91	Tổn thương nội tạng không xác định trong khoang ngực, có vết thương thấu khoang ngực	
S28	Tổn thương dập nát và/hoặc đứt rời phần của ngực do chấn thương	
S28.0	Dập nát ngực	
S28.1	Đứt rời một phần của ngực do chấn thương	
S29	Tổn thương khác và/hoặc không xác định ở ngực	
S29.0	Tổn thương cơ và/hoặc tổn thương gân, vùng ngực	
S29.7	Đa tổn thương ở ngực	Tổn thương có thể phân loại ở nhiều mục trong S20.- - S29.0
S29.8	Tổn thương xác định khác ở ngực	
S29.9	Tổn thương khoang ngực không xác định	
S30	Tổn thương nông ở bụng, thắt lưng và/hoặc vùng chậu	
S30.0	Đụng giập thắt lưng và/hoặc vùng chậu	Mông
S30.1	Đụng giập thành bụng	Hông|Háng
S30.2	Đụng giập cơ quan sinh dục ngoài	
S30.7	Đa tổn thương nông ở bụng, thắt lưng và/hoặc vùng chậu	
S30.8	Tổn thương nông khác ở bụng, thắt lưng và/hoặc vùng chậu	
S30.9	Tổn thương nông ở bụng, thắt lưng và/hoặc vùng chậu, phần không xác định	
S31	Vết thương hở ở bụng, thắt lưng và/hoặc vùng chậu	
S31.0	Vết thương hở ở thắt lưng và/hoặc vùng chậu	Mông
S31.1	Vết thương hở ở thành bụng	Sườn|Háng
S31.2	Vết thương hở ở dương vật	
S31.3	Vết thương hở ở bìu và/hoặc tinh hoàn	
S31.4	Vết thương hở ở âm đạo và/hoặc âm hộ	
S31.5	Vết thương hở ở cơ quan sinh dục ngoài khác và/hoặc không xác định	
S31.7	Đa vết thương hở ở bụng, thắt lưng và/hoặc vùng chậu	
S31.8	Vết thương hở ở vị trí khác và/hoặc không xác định ở bụng	
S32	Gãy cột sống thắt lưng và/hoặc khung chậu	
S32.0	Gãy đốt sống thắt lưng	Gãy cột sống thắt lưng
S32.00	Gãy đốt sống thắt lưng, gãy kín	
S32.01	Gãy đốt sống thắt lưng, gãy hở	
S32.1	Gãy xương cùng	
S32.10	Gãy xương cùng, gãy kín	
S32.11	Gãy xương cùng, gãy hở	
S32.2	Gãy xương cụt	
S32.20	Gãy xương cụt, gãy kín	
S32.21	Gãy xương cụt, gãy hở	
S32.3	Gãy xương chậu	
S32.30	Gãy xương chậu, gãy kín	
S32.31	Gãy xương chậu, gãy hở	
S32.4	Gãy ổ cối	
S32.40	Gãy ổ cối, gãy kín	
S32.41	Gãy ổ cối, gãy hở	
S32.5	Gãy xương mu	
S32.50	Gãy xương mu, gãy kín	
S32.51	Gãy xương mu, gãy hở	
S32.7	Gãy xương cột sống thắt lưng và/hoặc khung chậu, nhiều xương và/hoặc nhiều vị trí	
S32.70	Gãy xương cột sống thắt lưng và/hoặc khung chậu, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S32.71	Gãy xương cột sống thắt lưng và/hoặc khung chậu, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S32.8	Gãy phần khác và/hoặc không xác định của xương cột sống thắt lưng và/hoặc khung chậu	
S32.80	Gãy phần khác và/hoặc không xác định của xương cột sống thắt lưng và/hoặc khung chậu, gãy kín	
S32.81	Gãy phần khác và/hoặc không xác định của xương cột sống thắt lưng và/hoặc khung chậu, gãy hở	
S33	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng cột sống thắt lưng và/hoặc vùng chậu	
S33.0	Vỡ đĩa đệm cột sống thắt lưng do chấn thương	
S33.1	Trật đốt sống thắt lưng	Trật khớp cột sống thắt lưng không xác định khác
S33.2	Trật khớp cùng chậu và/hoặc khớp cùng cụt	
S33.3	Trật khớp phần khác và/hoặc không xác định của cột sống thắt lưng và/hoặc khung chậu	
S33.4	Vỡ khớp mu do chấn thương	
S33.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở cột sống thắt lưng	
S33.6	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp cùng chậu	
S33.7	Giãn dây chằng [bong gân] và/hoặc căng cơ ở phần khác và/hoặc không xác định của cột sống thắt lưng và/hoặc khung chậu	
S34	Tổn thương dây thần kinh và/hoặc tủy sống thắt lưng, vùng bụng, thắt lưng và/hoặc chậu	
S34.0	Chấn động và/hoặc phù của tủy sống thắt lưng	
S34.1	Tổn thương khác của tủy sống thắt lưng	
S34.2	Vết thương rễ thần kinh của cột sống thắt lưng và/hoặc cột sống cùng	
S34.3	Tổn thương dây thần kinh chùm đuôi ngựa	
S34.4	Tổn thương đám rối thần kinh thắt lưng - cùng	
S34.5	Tổn thương dây thần kinh giao cảm vùng thắt lưng, cùng và/hoặc chậu	
S34.6	Tổn thương dây thần kinh ngoại biên của bụng, thắt lưng và/hoặc vùng chậu	
S34.8	Tổn thương dây thần kinh khác và/hoặc không xác định, vùng bụng, thắt lưng và/hoặc chậu	
S35	Tổn thương mạch máu, vùng bụng, thắt lưng và/hoặc chậu	
S35.0	Tổn thương động mạch chủ bụng	
S35.1	Tổn thương tĩnh mạch chủ dưới	
S35.2	Tổn thương động mạch bụng hoặc động mạch mạc treo tràng	
S35.3	Tổn thương tĩnh mạch cửa hoặc tĩnh mạch lách	
S35.4	Tổn thương mạch máu thận	Động mạch hay tĩnh mạch thận
S35.5	Tổn thương mạch máu chậu	Động mạch hay tĩnh mạch hạ vị|Động mạch hay tĩnh mạch chậu|Động mạch hay tĩnh mạch tử cung
S35.7	Đa tổn thương mạch máu, vùng bụng, thắt lưng và/hoặc chậu	
S35.8	Tổn thương mạch máu khác, vùng bụng, thắt lưng và/hoặc chậu	Động mạch hoặc tĩnh mạch buồng trứng
S35.9	Tổn thương mạch máu không xác định, vùng bụng, thắt lưng và/hoặc chậu	
S36	Tổn thương tạng trong ổ bụng	
S36.0	Tổn thương lách	
S36.00	Tổn thương lách, không có vết thương thấu ổ bụng	
S36.01	Tổn thương lách, có vết thương thấu ổ bụng	
S36.1	Tổn thương gan hoặc túi mật	Ống mật
S36.10	Tổn thương gan hoặc túi mật, không có vết thương thấu ổ bụng	
S36.11	Tổn thương gan hoặc túi mật, có vết thương thấu ổ bụng	
S36.2	Tổn thương tụy	
S36.20	Tổn thương tụy, không có vết thương thấu ổ bụng	
S36.21	Tổn thương tụy, có vết thương thấu ổ bụng	
S36.3	Tổn thương dạ dày	
S36.30	Tổn thương dạ dày, không có vết thương thấu ổ bụng	
S36.31	Tổn thương dạ dày, có vết thương thấu ổ bụng	
S36.4	Tổn thương ruột non	
S36.40	Tổn thương ruột non, không có vết thương thấu ổ bụng	
S36.41	Tổn thương ruột non, có vết thương thấu ổ bụng	
S36.5	Tổn thương đại tràng	
S36.50	Tổn thương đại tràng, không có vết thương thấu ổ bụng	
S36.51	Tổn thương đại tràng, có vết thương thấu ổ bụng	
S36.6	Tổn thương trực tràng	
S36.60	Tổn thương trực tràng, không có vết thương thấu ổ bụng	
S36.61	Tổn thương trực tràng, có vết thương thấu ổ bụng	
S36.7	Tổn thương đa tạng trong ổ bụng	
S36.70	Tổn thương đa tạng trong ổ bụng, không có vết thương thấu ổ bụng	
S36.71	Tổn thương đa tạng trong ổ bụng, có vết thương thấu ổ bụng	
S36.8	Tổn thương tạng khác trong ổ bụng	Phúc mạc|Vùng sau phúc mạc
S36.80	Tổn thương tạng khác trong ổ bụng, không có vết thương thấu ổ bụng	
S36.81	Tổn thương tạng khác trong ổ bụng, có vết thương thấu ổ bụng	
S36.9	Tổn thương tạng trong ổ bụng không xác định	
S36.90	Tổn thương tạng trong ổ bụng không xác định, không có vết thương thấu ổ bụng	
S36.91	Tổn thương tạng trong ổ bụng không xác định, có vết thương thấu ổ bụng	
S37	Tổn thương tạng của hệ tiết niệu và/hoặc vùng chậu	
S37.0	Tổn thương thận	
S37.00	Tổn thương thận, không có vết thương thấu ổ bụng	
S37.01	Tổn thương thận, có vết thương thấu ổ bụng	
S37.1	Tổn thương niệu quản	
S37.10	Tổn thương niệu quản, không có vết thương thấu ổ bụng	
S37.11	Tổn thương niệu quản, có vết thương thấu ổ bụng	
S37.2	Tổn thương bàng quang	
S37.20	Tổn thương bàng quang, không có vết thương thấu ổ bụng	
S37.21	Tổn thương bàng quang, có vết thương thấu ổ bụng	
S37.3	Tổn thương niệu đạo	
S37.30	Tổn thương niệu đạo, không có vết thương thấu ổ bụng	
S37.31	Tổn thương niệu đạo, có vết thương thấu ổ bụng	
S37.4	Tổn thương buồng trứng	
S37.40	Tổn thương buồng trứng, không có vết thương thấu ổ bụng	
S37.41	Tổn thương buồng trứng, có vết thương thấu ổ bụng	
S37.5	Tổn thương vòi trứng	
S37.50	Tổn thương vòi trứng, không có vết thương thấu ổ bụng	
S37.51	Tổn thương vòi trứng, có vết thương thấu ổ bụng	
S37.6	Tổn thương tử cung	
S37.60	Tổn thương tử cung, không có vết thương thấu ổ bụng	
S37.61	Tổn thương tử cung, có vết thương thấu ổ bụng	
S37.7	Tổn thương đa tạng vùng chậu	
S37.70	Tổn thương đa tạng vùng chậu, không có vết thương thấu ổ bụng	
S37.71	Tổn thương đa tạng vùng chậu, có vết thương thấu ổ bụng	
S37.8	Tổn thương tạng khác ở vùng chậu	
S37.80	Tổn thương tạng khác ở vùng chậu, không có vết thương thấu ổ bụng	
S37.81	Tổn thương tạng khác ở vùng chậu, có vết thương thấu ổ bụng	
S37.9	Tổn thương cơ quan không xác định ở vùng chậu	
S37.90	Tổn thương cơ quan không xác định ở vùng chậu, không có vết thương thấu ổ bụng	
S37.91	Tổn thương cơ quan không xác định ở vùng chậu, có vết thương thấu ổ bụng	
S38	Tổn thương dập nát và/hoặc đứt rời do chấn thương ở một phần của bụng, thắt lưng và/hoặc vùng chậu	
S38.0	Tổn thương dập nát cơ quan sinh dục ngoài	
S38.1	Tổn thương dập nát ở phần khác và/hoặc không xác định của bụng, lưng dưới và/hoặc vùng chậu	
S38.2	Đứt rời cơ quan sinh dục ngoài do chấn thương	
S38.3	Đứt rời phần khác và/hoặc không xác định của bụng, thắt lưng và/hoặc vùng chậu do chấn thương	
S39	Tổn thương khác và/hoặc không xác định ở vùng bụng, thắt lưng và/hoặc chậu	
S39.0	Tổn thương cơ và/hoặc tổn thương gân ở vùng bụng, thắt lưng và/hoặc vùng chậu	
S39.6	Tổn thương tạng trong ổ bụng kèm tổn thương tạng vùng chậu	
S39.7	Đa tổn thương ở vùng bụng, thắt lưng và/hoặc vùng chậu	
S39.8	Tổn thương xác định khác của vùng bụng, thắt lưng và/hoặc vùng chậu	
S39.9	Tổn thương không xác định ở vùng bụng, thắt lưng và/hoặc vùng chậu	
S40	Tổn thương nông ở vai và/hoặc cánh tay trên	
S40.0	Đụng giập vai và/hoặc cánh tay trên	
S40.7	Đa tổn thương nông của vai và/hoặc cánh tay trên	
S40.8	Tổn thương nông khác của vai và/hoặc cánh tay trên	
S40.9	Tổn thương nông của vai và/hoặc cánh tay trên, không xác định	
S41	Vết thương hở ở vai và/hoặc cánh tay trên	
S41.0	Vết thương hở ở vai	
S41.1	Vết thương hở ở cánh tay trên	
S41.7	Đa vết thương hở ở vai và/hoặc cánh tay trên	
S41.8	Vết thương hở ở phần khác và/hoặc không xác định của vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan]	
S42	Gãy xương vùng vai [xương đòn, vai] và/hoặc xương cánh tay trên	
S42.0	Gãy xương đòn	
S42.00	Gãy xương đòn, gãy kín	
S42.01	Gãy xương đòn, gãy hở	
S42.1	Gãy xương vai	
S42.10	Gãy xương vai, gãy kín	
S42.11	Gãy xương vai, gãy hở	
S42.2	Gãy phần trên xương cánh tay trên	Cổ giải phẫu xương cánh tay trên|Củ/mấu động lớn|Đầu gần|Cổ tiếp [cổ phẫu thuật xương cánh tay]|Đầu xương trên
S42.20	Gãy phần trên xương cánh tay trên, gãy kín	
S42.21	Gãy phần trên xương cánh tay trên, gãy hở	
S42.3	Gãy thân xương cánh tay trên	Xương cánh tay không xác định khác|Xương cánh tay trên không xác định khác
S42.30	Gãy thân xương cánh tay, gãy kín	
S42.31	Gãy thân xương cánh tay, gãy hở	
S42.4	Gãy phần dưới xương cánh tay	
S42.40	Gãy phần dưới xương cánh tay, gãy kín	
S42.41	Gãy phần dưới xương cánh tay, gãy hở	
S42.7	Gãy xương đòn, xương vai và/ hoặc xương cánh tay trên, nhiều xương và/hoặc nhiều vị trí	
S42.70	Gãy xương đòn, xương vai và/ hoặc xương cánh tay trên, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S42.71	Gãy xương đòn, xương vai và/ hoặc xương cánh tay trên, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S42.8	Gãy phần khác của vùng vai và/hoặc xương cánh tay trên	
S42.80	Gãy phần khác của vùng vai và/hoặc xương cánh tay trên, gãy kín	
S42.81	Gãy phần khác của vùng vai và/hoặc xương cánh tay trên, gãy hở	
S42.9	Gãy xương của vòng ngực [xương đòn, vai, cánh tay trên], phần không xác định	Gãy xương vai không xác định khác
S42.90	Gãy xương của vòng ngực [xương đòn, vai, cánh tay trên], phần không xác định, gãy kín	
S42.91	Gãy xương của vòng ngực [xương đòn, vai, cánh tay trên], phần không xác định, gãy hở	
S43	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng của vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan]	
S43.0	Trật khớp vai	Khớp ổ chảo cánh tay
S43.1	Trật khớp cùng - đòn	
S43.2	Trật khớp ức - đòn	
S43.3	Trật khớp phần khác và/hoặc không xác định của vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan]	Trật khớp vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan] không xác định khác
S43.4	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp vai	
S43.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp mỏm cùng vai - đòn	Dây chằng quạ - đòn
S43.6	Giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp ức đòn	
S43.7	Giãn dây chằng [bong gân] và/hoặc căng cơ ở phần khác và/hoặc không xác định của vòng ngực [xương đòn, vai, cánh tay trên và khớp liên quan]	
S44	Tổn thương dây thần kinh, vùng vai và cánh tay trên	
S44.0	Tổn thương dây thần kinh trụ, vùng vai và cánh tay trên	
S44.1	Tổn thương dây thần kinh giữa, vùng cánh tay trên	
S44.2	Tổn thương dây thân kinh quay, vùng cánh tay trên	
S44.3	Tổn thương dây thần kinh nách	
S44.4	Tổn thương dây thần kinh cơ - da	
S44.5	Tổn thương dây thần kinh cảm giác - da, vùng vai và cánh tay trên	
S44.7	Tổn thương nhiều dây thần kinh, vùng vai và cánh tay trên	
S44.8	Tổn thương dây thần kinh khác, vùng vai và cánh tay trên	
S44.9	Tổn thương dây thần kinh không xác định, vùng vai và cánh tay trên	
S45	Tổn thương mạch máu, vùng vai và cánh tay trên	
S45.0	Tổn thương động mạch nách	
S45.1	Tổn thương động mạch cánh tay	
S45.2	Tổn thương tĩnh mạch nách hoặc cánh tay	
S45.3	Tổn thương nông tĩnh mạch máu, vùng vai và cánh tay trên	
S45.7	Tổn thương nhiều mạch máu, vùng vai và cánh tay trên	
S45.8	Tổn thương mạch máu khác, vùng vai và cánh tay trên	
S45.9	Tổn thương mạch máu không xác định, vùng vai và cánh tay trên	
S46	Tổn thương cơ và/hoặc gân, vùng vai và cánh tay trên	
S46.0	Tổn thương cơ và/hoặc gân của chỏm xoay ở vai	
S46.1	Tổn thương cơ và/hoặc gân đầu dài cơ nhị đầu cánh tay	
S46.2	Tổn thương cơ và/hoặc gân ở phần khác của cơ nhị đầu cánh tay	
S46.3	Tổn thương cơ và/hoặc gân cơ tam đầu	
S46.7	Tổn thương nhiều cơ và/hoặc gân, vùng vai và cánh tay trên	
S46.8	Tổn thương cơ và/hoặc gân khác, vùng vai và cánh tay trên	
S46.9	Tổn thương cơ và/hoặc gân không xác định, vùng vai và cánh tay trên	
S47	Tổn thương dập nát vai và/hoặc cánh tay trên	
S48	Đứt rời vai và/hoặc cánh tay trên do chấn thương	
S48.0	Đứt rời khớp vai do chấn thương	
S48.1	Đứt rời cánh tay trên vùng giữa vai và khuỷu tay do chấn thương	
S48.9	Đứt rời vai và/hoặc cánh tay trên do chấn thương, tầm không xác định	
S49	Tổn thương khác và/hoặc không xác định ở vai và/hoặc cánh tay trên	
S49.7	Đa tổn thương ở vai và/hoặc cánh tay trên	Tổn thương có thể phân loại ở nhiều mục trong S40.- - S48.
S49.8	Tổn thương xác định khác ở vai và/hoặc cánh tay trên	
S49.9	Tổn thương không xác định ở vai và/hoặc cánh tay trên	
S50	Tổn thương nông ở cẳng tay	
S50.0	Đụng giập khuỷu tay	
S50.1	Đụng giập phần khác và/hoặc không xác định của cẳng tay	
S50.7	Đa tổn thương nông khác ở cẳng tay	
S50.8	Tổn thương nông khác ở cẳng tay	
S50.9	Tổn thương nông ở cẳng tay, không xác định	Tổn thương nông ở cẳng tay không xác định khác
S51	Vết thương hở ở cẳng tay	
S51.0	Vết thương hở ở khuỷu tay	
S51.7	Nhiều vết thương hở ở cẳng tay	
S51.8	Vết thương hở ở phần khác của cẳng tay	
S51.9	Vết thương hở ở cẳng tay, phần không xác định	
S52	Gãy xương cẳng tay	
S52.0	Gãy đầu trên xương trụ	Xương mỏm quạ|Khuỷu tay không xác định khác|Gãy xương - trật khớp Monteggia|Mỏm khuỷu|Gốc gần
S52.00	Gãy đầu trên xương trụ, gãy kín	
S52.01	Gãy đầu trên xương trụ, gãy hở	
S52.1	Gãy đầu trên xương quay	Gốc gần
S52.10	Gãy đầu trên xương quay, gãy kín	
S52.11	Gãy đầu trên xương quay, gãy hở	
S52.2	Gãy thân xương trụ	
S52.20	Gãy thân xương trụ, gãy kín	
S52.21	Gãy thân xương trụ, gãy hở	
S52.3	Gãy thân xương quay	
S52.30	Gãy thân xương quay, gãy kín	
S52.31	Gãy thân xương quay, gãy hở	
S52.4	Gãy thân cả xương trụ và xương quay	
S52.40	Gãy thân cả xương trụ và xương quay, gãy kín	
S52.41	Gãy thân cả xương trụ và xương quay, gãy hở	
S52.5	Gãy đầu dưới xương quay	Gãy xương Colles|Gãy xương Smith
S52.50	Gãy đầu dưới xương quay, gãy kín	
S52.51	Gãy đầu dưới xương quay, gãy hở	
S52.6	Gãy đầu dưới cả xương trụ và xương quay	
S52.60	Gãy đầu dưới cả xương trụ và xương quay, gãy kín	
S52.61	Gãy đầu dưới cả xương trụ và xương quay, gãy hở	
S52.7	Gãy xương cẳng tay, nhiều xương và/hoặc nhiều vị trí	
S52.70	Gãy xương cẳng tay, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S52.71	Gãy xương cẳng tay, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S52.8	Gãy phần khác của xương cẳng tay	Đầu dưới của xương trụ|Đầu xương trụ
S52.80	Gãy phần khác của xương cẳng tay, gãy kín	
S52.81	Gãy phần khác của xương cẳng tay, gãy hở	
S52.9	Gãy xương cẳng tay, phần không xác định	
S52.90	Gãy xương cẳng tay, phần không xác định, gãy kín	
S52.91	Gãy xương cẳng tay, phần không xác định, gãy hở	
S53	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng khuỷu tay	
S53.0	Trật khớp đầu xương quay	
S53.1	Trật khớp khuỷu tay, không xác định	
S53.2	Chấn thương đứt dây chằng bên xương quay	
S53.3	Chấn thương đứt dây chằng bên xương trụ	
S53.4	Giãn dây chằng [bong gân] và/hoặc căng cơ khuỷu tay	
S54	Tổn thương dây thần kinh, tầm cẳng tay	
S54.0	Tổn thương dây thần kinh trụ, tầm cẳng tay	Dây thần kinh xương trụ không xác định khác
S54.1	Tổn thương dây thần kinh giữa, tầm cẳng tay	Dây thần kinh giữa không xác định khác
S54.2	Tổn thương dây thần kinh quay, tầm cẳng tay	Dây thần kinh quay không xác định khác
S54.3	Tổn thương dây thần kinh cảm giác - da, tầm cẳng tay	
S54.7	Tổn thương đa dây thần kinh, tầm cẳng tay	
S54.8	Tổn thương dây thần kinh khác, tầm cẳng tay	
S54.9	Tổn thương dây thần kinh không xác định, tầm cẳng tay	
S55	Tổn thương mạch máu, tầm cẳng tay	
S55.0	Tổn thương động mạch trụ, tầm cẳng tay	
S55.1	Tổn thương động mạch quay, tầm cẳng tay	
S55.2	Tổn thương tĩnh mạch, tầm cẳng tay	
S55.7	Tổn thương đa mạch máu, tầm cẳng tay	
S55.8	Tổn thương mạch máu khác, tầm cẳng tay	
S55.9	Tổn thương mạch máu không xác định, tầm cẳng tay	
S56	Tổn thương cơ và/hoặc gân, tầm cẳng tay	
S56.0	Tổn thương cơ và/hoặc gân gấp ngón tay cái, tầm cẳng tay	
S56.1	Tổn thương cơ và/hoặc gân gấp dài ngón tay khác, tầm cẳng tay	
S56.2	Tổn thương cơ và/hoặc gân gấp khác, tầm cẳng tay	
S56.3	Tổn thương cơ duỗi hoặc cơ giạng và/hoặc gân duỗi hoặc gân giạng ngón tay cái, tầm cẳng tay	
S56.4	Tổn thương cơ và/hoặc gân duỗi của ngón tay khác, tầm cẳng tay	
S56.5	Tổn thương cơ và/hoặc gân duỗi khác, tầm cẳng tay	
S56.7	Tổn thương đa cơ và/hoặc đa gân, tầm cẳng tay	
S56.8	Tổn thương cơ và/hoặc gân khác và/hoặc không xác định, tầm cẳng tay	
S57	Tổn thương dập nát ở cẳng tay	
S57.0	Tổn thương dập nát ở khuỷu tay	
S57.8	Tổn thương dập nát ở phần khác của cẳng tay	
S57.9	Tổn thương dập nát ở cẳng tay, phần không xác định	
S58	Đứt rời cẳng tay do chấn thương	
S58.0	Đứt rời cằng tay tầm khuỷu tay do chấn thương	
S58.1	Đứt rời cẳng tay tầm giữa khuỷu và cổ tay do chấn thương	
S58.9	Đứt rời cẳng tay do chấn thương, tầm chi không xác định	
S59	Tổn thương khác và/hoặc không xác định của cẳng tay	
S59.7	Đa tổn thương cẳng tay	Tổn thương có thể phân loại ở nhiều mục trong S50.- - S58.
S59.8	Tổn thương xác định khác ở cẳng tay	
S59.9	Tổn thương không xác định ở cẳng tay	
S60	Tổn thương nông ở cổ tay và/hoặc bàn tay	
S60.0	Đụng giập (các) ngón tay không có tổn thương móng	
S60.1	Đụng giập (các) ngón tay kèm tổn thương móng	
S60.2	Đụng giập phần khác của cổ tay và/hoặc bàn tay	
S60.7	Đa tổn thương nông ở cổ tay và/hoặc bàn tay	
S60.8	Tổn thương nông khác ở cổ tay và/hoặc bàn tay	
S60.9	Tổn thương nông ở cổ tay và/hoặc bàn tay, không xác định	
S61	Vết thương hở ở cổ tay và/hoặc bàn tay	
S61.0	Vết thương hở của (các) ngón tay không có tổn thương móng	
S61.1	Vết thương hở của (các) ngón tay kèm tổn thương móng	
S61.7	Đa vết thương hở ở cổ tay và/hoặc bàn tay	
S61.8	Vết thương hở ở phần khác của cổ tay và/hoặc bàn tay	
S61.9	Vết thương hở ở cổ tay và/hoặc bàn tay, phần không xác định	
S62	Gãy xương, tầm cổ tay và bàn tay	
S62.0	Gãy xương thuyền bàn tay	
S62.00	Gãy xương thuyền bàn tay, gãy kín	
S62.01	Gãy xương thuyền bàn tay, gãy hở	
S62.1	Gãy xương khớp cổ tay khác	Xương cả [xương to]|Xương móc [hình móc]|Xương bán nguyệt [hình bán nguyệt]|Xương đậu|Xương thang [nhiều góc lớn]|Xương thê [nhiều góc nhỏ]|Xương tháp [xương chêm của khớp xương cổ tay]
S62.10	Gãy xương khớp cổ tay khác, gãy kín	
S62.11	Gãy xương khớp cổ tay khác, gãy hở	
S62.2	Gãy xương đốt bàn tay của ngón 1 [ngón tay cái]	Gãy xương Bennett [gãy nền xương bàn tay]
S62.20	Gãy xương đốt bàn tay của ngón 1 [ngón tay cái], gãy kín	
S62.21	Gãy xương đốt bàn tay của ngón 1 [ngón tay cái], gãy hở	
S62.3	Gãy xương đốt khác của bàn tay	
S62.30	Gãy xương đốt khác của bàn tay, gãy kín	
S62.31	Gãy xương đốt khác của bàn tay, gãy hở	
S62.4	Gãy xương đốt bàn tay, nhiều xương và/hoặc nhiều vị trí	
S62.40	Gãy xương đốt bàn tay, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S62.41	Gãy xương đốt bàn tay, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S62.5	Gãy xương đốt ngón tay cái	
S62.50	Gãy xương đốt ngón tay cái, gãy kín	
S62.51	Gãy xương đốt ngón tay cái, gãy hở	
S62.6	Gãy xương đốt ngón tay khác	
S62.60	Gãy xương đốt ngón tay khác, gãy kín	
S62.61	Gãy xương đốt ngón tay khác, gãy hở	
S62.7	Gãy xương đốt ngón tay, nhiều xương và/hoặc nhiều vị trí	
S62.70	Gãy xương đốt ngón tay, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S62.71	Gãy xương đốt ngón tay, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S62.8	Gãy xương phần khác và/hoặc không xác định của cổ tay và/hoặc bàn tay	
S62.80	Gãy xương phần khác và/hoặc không xác định của cổ tay và/hoặc bàn tay, gãy kín	
S62.81	Gãy xương phần khác và/hoặc không xác định của cổ tay và/hoặc bàn tay, gãy hở	
S63	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ khớp và/hoặc dây chằng, tầm cổ tay và bàn tay	
S63.0	Trật khớp cổ tay	
S63.1	Trật khớp ngón tay	
S63.2	Trật khớp ngón tay, nhiều ngón và/hoặc nhiều vị trí	
S63.3	Chấn thương đứt dây chằng cổ tay và/hoặc xương cổ tay	
S63.4	Chấn thương đứt dây chằng ngón tay tại xương đốt bàn tay - ngón tay và/hoặc khớp gian đốt ngón	Ở bên|Cung gan tay|gan bàn tay phẳng
S63.5	Giãn dây chằng [bong gân] và/hoặc căng cơ cổ tay	
S63.6	Giãn dây chằng [bong gân] và/hoặc căng cơ (các) ngón tay	
S63.7	Giãn dây chằng [bong gân] và/hoặc căng cơ phần bàn tây khác và/hoặc không xác định	
S64	Tổn thương dây thần kinh, tầm cổ tay và bàn tay	
S64.0	Tổn thương dây thần kinh trụ, tầm cổ tay và bàn tay	
S64.1	Tổn thương dây thần kinh giữa, tầm cổ tay và bàn tay	
S64.2	Tổn thương dây thần kinh quay, tầm cổ tay và bàn tay	
S64.3	Tổn thương dây thần kinh ngón tay của ngón tay cái	
S64.4	Tổn thương dây thần kinh ngón tay của ngón tay khác	
S64.7	Tổn thương đa dây thần kinh, tầm cổ tay và bàn tay	
S64.8	Tổn thương dây thần kinh khác, tầm cổ tay và bàn tay	
S64.9	Tổn thương dây thần kinh không xác định, tầm cổ tay và bàn tay	
S65	Tổn thương mạch máu, tầm cổ tay và bàn tay	
S65.0	Tổn thương động mạch trụ, tầm cổ tay và bàn tay	
S65.1	Tổn thương động mạch quay, tầm cổ tay và bàn tay	
S65.2	Tổn thương của cung gan tay nông	
S65.3	Tổn thương của cung gan tay sâu	
S65.4	Tổn thương mạch máu ngón tay cái	
S65.5	Tổn thương mạch máu ngón tay khác	
S65.7	Tổn thương đa mạch máu, tầm cổ tay và bàn tay	
S65.8	Tổn thương mạch máu khác, tầm cổ tay và bàn tay	
S65.9	Tổn thương mạch máu không xác định, tầm cổ tay và bàn tay	
S66	Tổn thương cơ bắp và/hoặc gân, tầm cổ tay và bàn tay	
S66.0	Tổn thương cơ và/hoặc gân gấp dài ngón tay cái, tầm cổ tay và bàn tay	
S66.1	Tổn thương cơ và/hoặc gân gấp ngón tay khác, tầm cổ tay và bàn tay	
S66.2	Tổn thương cơ và/hoặc gân duỗi ngón tay cái, tầm cổ tay và bàn tay	
S66.3	Tổn thương cơ và/hoặc gân duỗi ngón tay khác, tầm cổ tay và bàn tay	
S66.4	Tổn thương cơ và/hoặc gân nội tại của ngón tay cái, tầm cổ tay và bàn tay	
S66.5	Tổn thương cơ và/hoặc gân nội tại của ngón tay khác, tầm cổ tay và bàn tay	
S66.6	Tổn thương đa cơ và/hoặc đa gân gấp, tầm cổ tay và bàn tay	
S66.7	Tổn thương đa cơ và/hoặc đa gân duỗi, tầm cổ tay và bàn tay	
S66.8	Tổn thương cơ và/hoặc gân khác, tầm cổ tay và bàn tay	
S66.9	Tổn thương cơ và/hoặc gân không xác định, tầm cổ tay và bàn tay	
S67	Tổn thương dập nát cổ tay và/hoặc bàn tay	
S67.0	Tổn thương dập nát ngón tay cái và/hoặc các ngón tay khác	
S67.8	Tổn thương dập nát phần khác và/hoặc không xác định của cổ tay và/hoặc bàn tay	
S68	Đứt rời cổ tay và/hoặc bàn tay do chấn thương	
S68.0	Đứt rời ngón tay cái (toàn phần) (một phần) do chấn thương	
S68.1	Đứt rời ngón tay khác (toàn phần) (một phần) do chấn thương	
S68.2	Đứt rời hai hoặc nhiều ngón tay (toàn phần) (một phần) không kèm phần tay khác do chấn thương	
S68.3	Đứt rời kết hợp một phần ngón tay với phần khác của cổ tay và/hoặc bàn tay do chấn thương	
S68.4	Đứt rời bàn tay tầm cổ tay do chấn thương	
S68.8	Đứt rời phần khác của cổ tay và/hoặc bàn tay do chấn thương	
S68.9	Đứt rời cổ tay và bàn tay do chấn thương, tầm chi không xác định	
S69	Tổn thương khác và/hoặc không xác định ở cổ tay và/hoặc bàn tay	
S69.7	Đa tổn thương cổ tay và/hoặc bàn tay	Tổn thương có thể phân loại ở nhiều mục trong S60.- - S68.
S69.8	Tổn thương xác định khác của cổ tay và/hoặc bàn tay	
S69.9	Tổn thương không xác định của cổ tay và/hoặc bàn tay	
S70	Tổn thương nông ở hông và/hoặc đùi	
S70.0	Đụng giập tại hông	
S70.1	Đụng giập tại đùi	
S70.7	Đa tổn thương nông tại hông và/hoặc đùi	
S70.8	Tổn thương nông khác tại hông và/hoặc đùi	
S70.9	Tổn thương nông tại hông và/hoặc đùi, không xác định	
S71	Vết thương hở ở hông và/hoặc đùi	
S71.0	Vết thương hở tại hông	
S71.1	Vết thương hở tại đùi	
S71.7	Đa vết thương hở tại hông và/hoặc đùi	
S71.8	Vết thương hở phần khác và/hoặc không xác định của đai chậu [khung chậu, xương cùng và xương cụt]	
S72	Gãy xương đùi	
S72.0	Gãy cổ xương đùi	Gãy xương háng không xác định khác
S72.00	Gãy cổ xương đùi, gãy kín	
S72.01	Gãy cổ xương đùi, gãy hở	
S72.1	Gãy mấu chuyển	Gãy liên mấu chuyển
S72.10	Gãy mấu chuyển, gãy kín	
S72.11	Gãy mấu chuyển, gãy hở	
S72.2	Gãy mấu chuyển phụ	
S72.20	Gãy mấu chuyển phụ, gãy kín	
S72.21	Gãy mấu chuyển phụ, gãy hở	
S72.3	Gãy thân xương đùi	
S72.30	Gãy thân xương đùi, gãy kín	
S72.31	Gãy thân xương đùi, gãy hở	
S72.4	Gãy xương đầu dưới xương đùi	
S72.40	Gãy xương đầu dưới xương đùi, gãy kín	
S72.41	Gãy xương đầu dưới xương đùi, gãy hở	
S72.7	Gãy xương đùi, nhiều xương và/hoặc nhiều vị trí	
S72.70	Gãy xương đùi, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S72.71	Gãy xương đùi, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S72.8	Gãy phần khác của xương đùi	
S72.80	Gãy phần khác của xương đùi, gãy kín	
S72.81	Gãy phần khác của xương đùi, gãy hở	
S72.9	Gãy xương đùi, phần không xác định	
S72.90	Gãy xương đùi, phần không xác định, gãy kín	
S72.91	Gãy xương đùi, phần không xác định, gãy hở	
S73	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ khớp háng	
S73.0	Trật khớp háng	
S73.1	Giãn dây chằng [bong gân] và/hoặc căng cơ khớp háng	
S74	Tổn thương dây thần kinh, tầm hông và đùi	
S74.0	Tổn thương dây thần kinh tọa, tầm hông và đùi	
S74.1	Tổn thương dây thần kinh đùi, tầm hông và đùi	
S74.2	Tổn thương dây thần kinh cảm giác - da, tầm hông và đùi	
S74.7	Tổn thương đa dây thần kinh, tầm hông và đùi	
S74.8	Tổn thương dây thần kinh khác, tầm hông và đùi	
S74.9	Tổn thương dây thần kinh không xác định, tầm hông và đùi	
S75	Tổn thương mạch máu, tầm hông và đùi	
S75.0	Tổn thương động mạch đùi	
S75.1	Tổn thương tĩnh mạch đùi, tầm hông và đùi	
S75.2	Tổn thương tĩnh mạch hiển lớn, tầm hông và đùi	
S75.7	Tổn thương đa mạch máu, tầm hông và đùi	
S75.8	Tổn thương mạch máu khác, tầm hông và đùi	
S75.9	Tổn thương mạch máu không xác định, tầm hông và đùi	
S76	Tổn thương cơ và/hoặc gân, tầm hông và đùi	
S76.0	Tổn thương cơ và/hoặc gân ở hông	
S76.1	Tổn thương cơ tứ đầu và/hoặc gân	
S76.2	Tổn thương cơ khép và/hoặc gân đùi	
S76.3	Tổn thương cơ và/hoặc gân của nhóm cơ sau, tầm đùi	
S76.4	Tổn thương cơ và/hoặc gân khác và/hoặc không xác định, tầm đùi	
S76.7	Tổn thương đa cơ và/hoặc gân, tầm hông và đùi	
S77	Tổn thương dập nát ở hông và/hoặc đùi	
S77.0	Tổn thương dập nát ở hông	
S77.1	Tổn thương dập nát ở đùi	
S77.2	Tổn thương dập nát ở hông và đùi	
S78	Đứt rời hông và/hoặc đùi do chấn thương	
S78.0	Đứt rời chi dưới tầm khớp háng do chấn thương	
S78.1	Đứt rời chi dưới tầm giữa hông và gối do chấn thương	
S78.9	Đứt rời hông và/hoặc đùi do chấn thương, tầm chi không xác định	
S79	Tổn thương khác và/hoặc không xác định ở hông và/hoặc đùi	
S79.7	Đa tổn thương ở hông và/hoặc đùi	Tổn thương có thể phân loại ở nhiều mục trong S70.- - S78.
S79.8	Tổn thương xác định khác ở hông và/hoặc đùi	
S79.9	Tổn thương không xác định ở hông và/hoặc đùi	
S80	Tổn thương nông tại cẳng chân	Tổn thương nông ở cẳng chân
S80.0	Đụng giập đầu gối	
S80.1	Đụng giập phần khác và/hoặc không xác định của cẳng chân	
S80.7	Đa tổn thương nông ở cẳng chân	
S80.8	Tổn thương nông khác ở cẳng chân	
S80.9	Tổn thương nông ở cẳng chân, không xác định	
S81	Vết thương hở ở cẳng chân	
S81.0	Vết thương hở ở đầu gối	
S81.7	Đa vết thương hở ở cẳng chân	
S81.8	Vết thương hở ở phần khác của chi dưới	
S81.9	Vết thương hở ở cẳng chân, phần không xác định	
S82	Gãy xương cẳng chân, bao gồm cổ chân	
S82.0	Gãy xương bánh chè	Xương bánh chè [đầu gối]
S82.00	Gãy xương bánh chè, gãy kín	
S82.01	Gãy xương bánh chè, gãy hở	
S82.1	Gãy đầu trên xương chày	
S82.10	Gãy đầu trên xương chày, gãy kín	
S82.11	Gãy đầu trên xương chày, gãy hở	
S82.2	Gãy thân xương chày	Có hoặc không đề cập đến gãy xương mác
S82.20	Gãy thân xương chày, gãy kín	
S82.21	Gãy thân xương chày, gãy hở	
S82.3	Gãy đầu dưới xương chày	
S82.30	Gãy đầu dưới xương chày, gãy kín	
S82.31	Gãy đầu dưới xương chày, gãy hở	
S82.4	Gãy xương mác đơn thuần	
S82.40	Gãy xương mác đơn thuần, gãy kín	
S82.41	Gãy xương mác đơn thuần, gãy hở	
S82.5	Gãy mắt cá trong	
S82.50	Gãy mắt cá trong, gãy kín	
S82.51	Gãy mắt cá trong, gãy hở	
S82.6	Gãy mắt cá ngoài	
S82.60	Gãy mắt cá ngoài, gãy kín	
S82.61	Gãy mắt cá ngoài, gãy hở	
S82.7	Gãy xương ở cẳng chân, nhiều xương và/hoặc nhiều vị trí	
S82.70	Gãy xương ở cẳng chân, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S82.71	Gãy xương ở cẳng chân, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S82.8	Gãy xương ở phần khác của cẳng chân	
S82.80	Gãy xương ở phần khác của cẳng chân, gãy kín	
S82.81	Gãy xương ở phần khác của cẳng chân, gãy hở	
S82.9	Gãy xương cẳng chân, phần không xác định	
S82.90	Gãy xương cẳng chân, phần không xác định, gãy kín	
S82.91	Gãy xương cẳng chân, phần không xác định, gãy hở	
S83	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng đầu gối	
S83.0	Trật xương bánh chè	
S83.1	Trật khớp gối	
S83.2	Rách sụn chêm, vết rách hiện tại	
S83.3	Rách sụn khớp gối, vết rách hiện tại	
S83.4	Giãn dây chằng [bong gân] và/hoặc căng cơ ở dây chằng bên (xương mác) (xương chày) của khớp gối	
S83.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ở dây chằng chéo (phía trước) (phía sau) khớp gối	
S83.6	Giãn dây chằng [bong gân] và/hoặc căng cơ ở phần khớp gối khác và/hoặc không xác định	
S83.7	Tổn thương đa cấu trúc khớp gối	
S84	Tổn thương dây thần kinh, tầm cẳng chân	
S84.0	Tổn thương dây thần kinh chằng, tầm cẳng chân	
S84.1	Tổn thương dây thần kinh mác, tầm cẳng chân	
S84.2	Tổn thương dây thần kinh cảm giác - da, tầm cẳng chân	
S84.7	Tổn thương đa dây thần kinh, tầm cẳng chân	
S84.8	Tổn thương dây thần kinh khác tầm cẳng chân	
S84.9	Tổn thương dây thần kinh không xác định, tầm cẳng chân	
S85	Tổn thương mạch máu, tầm cẳng chân	
S85.0	Tổn thương động mạch khoeo	
S85.1	Tổn thương động mạch chày (phía trước) (phía sau)	
S85.2	Tổn thương động mạch mác	
S85.3	Tổn thương tĩnh mạch hiển lớn, tầm cẳng chân	Tĩnh mạch hiển lớn không xác định khác
S85.4	Tổn thương tĩnh mạch hiển nhỏ, tầm cẳng chân	
S85.5	Tổn thương tĩnh mạch khoeo	
S85.7	Tổn thương đa mạch máu, tầm cẳng chân	
S85.8	Tổn thương mạch máu khác, tầm cẳng chân	
S85.9	Tổn thương mạch máu không xác định, tầm cẳng chân	
S86	Tổn thương cơ và/hoặc gân, tầm cẳng chân	
S86.0	Tổn thương gân Achilles	
S86.1	Tổn thương cơ khác và/hoặc gân nhóm cơ sau khác, tầm cẳng chân	
S86.2	Tổn thương cơ và/hoặc gân nhóm cơ trước, tầm cẳng chân	
S86.3	Tổn thương cơ và/hoặc gân nhóm cơ mác, tầm cẳng chân	
S86.7	Tổn thương đa cơ và/hoặc đa gân, tầm cẳng chân	
S86.8	Tổn thương cơ và/hoặc gân khác, tầm cẳng chân	
S86.9	Tổn thương cơ và/hoặc gân không xác định, tầm cẳng chân	
S87	Tổn thương dập nát cẳng chân	
S87.0	Tổn thương dập nát đầu gối	
S87.8	Tổn thương dập nát phần khác và/hoặc không xác định của cẳng chân	
S88	Đứt rời cẳng chân do chấn thương	
S88.0	Đứt rời cẳng chân tầm đầu gối do chấn thương	
S88.1	Đứt rời cẳng chân tầm giữa khớp gối và cổ chân do chấn thương	
S88.9	Đứt rời cẳng chân do chấn thương, tầm chi không xác định	
S89	Tổn thương khác và/hoặc không xác định ở cẳng chân	
S89.7	Đa tổn thương ở cẳng chân	Tổn thương có thể phân loại ở nhiều mục trong S80.- - S88.
S89.8	Tổn thương xác định khác ở cẳng chân	
S89.9	Tổn thương không xác định ở cẳng chân	
S90	Tổn thương nông ở cổ chân và/hoặc bàn chân	
S90.0	Đụng giập cổ chân	
S90.1	Đụng giập ngón chân không có tổn thương móng	Đụng giập ngón chân không xác định khác
S90.2	Đụng giập ngón chân có tổn thương móng	
S90.3	Đụng giập phần khác và/hoặc không xác định của bàn chân	
S90.7	Đa tổn thương nông ở cổ chân và/hoặc bàn chân	
S90.8	Tổn thương nông khác ở cổ chân và/hoặc bàn chân	
S90.9	Tổn thương nông ở cổ chân và/hoặc bàn chân, không xác định	
S91	Vết thương hở ở cổ chân và/hoặc bàn chân	
S91.0	Vết thương hở ở cổ chân	
S91.1	Vết thương hở ở ngón chân không có tổn thương móng	Vết thương hở của ngón chân không xác định khác
S91.2	Vết thương hở ở ngón chân có tổn thương móng	
S91.3	Vết thương hở ở phần khác của bàn chân	Vết thương hở ở bàn chân không xác định khác
S91.7	Đa vết thương hở ở cẳng chân và/hoặc bàn chân	
S92	Gãy xương bàn chân, trừ cổ chân	
S92.0	Gãy xương gót	Xương gót chân|Xương gót
S92.00	Gãy xương gót, gãy kín	
S92.01	Gãy xương gót, gãy hở	
S92.1	Gãy xương sên	Xương sên
S92.10	Gãy xương sên, gãy kín	
S92.11	Gãy xương sên, gãy hở	
S92.2	Gãy xương cổ chân khác	
S92.20	Gãy xương cổ chân khác, gãy kín	
S92.21	Gãy xương cổ chân khác, gãy hở	
S92.3	Gãy xương đốt bàn chân	
S92.30	Gãy xương đốt bàn chân, gãy kín	
S92.31	Gãy xương đốt bàn chân, gãy hở	
S92.4	Gãy xương ngón chân cái	
S92.40	Gãy xương ngón chân cái, gãy kín	
S92.41	Gãy xương ngón chân cái, gãy hở	
S92.5	Gãy xương ngón chân khác	
S92.50	Gãy xương ngón chân khác, gãy kín	
S92.51	Gãy xương ngón chân khác, gãy hở	
S92.7	Gãy xương bàn chân, nhiều xương và/hoặc nhiều vị trí	
S92.70	Gãy xương bàn chân, nhiều xương và/hoặc nhiều vị trí, gãy kín	
S92.71	Gãy xương bàn chân, nhiều xương và/hoặc nhiều vị trí, gãy hở	
S92.9	Gãy xương bàn chân, không xác định	
S92.90	Gãy xương bàn chân, không xác định, gãy kín	
S92.91	Gãy xương bàn chân, không xác định, gãy hở	
S93	Trật khớp, giãn dây chằng [bong gân], căng cơ khớp và/hoặc dây chằng tầm cẳng chân và bàn chân	
S93.0	Trật khớp cổ chân	Xương sên|Xương mác, đầu dưới|Xương mắt cá|Xương chày, đầu dưới
S93.1	Trật khớp ngón chân	
S93.2	Đứt dây chằng, tầm cổ chân và bàn chân	
S93.3	Trật khớp ở phần khác và/hoặc không xác định của bàn chân	
S93.4	Giãn dây chằng [bong gân] và/hoặc căng cơ cổ chân	
S93.5	Giãn dây chằng [bong gân] và/hoặc căng cơ ngón chân	
S93.6	Giãn dây chằng [bong gân] và/hoặc căng cơ ở phần khác và/hoặc không xác định của bàn chân	
S94	Tổn thương dây thần kinh, tầm cổ chân và bàn chân	
S94.0	Tổn thương dây thần kinh gan bàn chân ngoài	
S94.1	Tổn thương dây thần kinh gan bàn chân trong	
S94.2	Tổn thương dây thần kinh mác sâu, tầm cổ chân và bàn chân	Nhánh bên, tận cùng của dây thần kinh sâu xương mác
S94.3	Tổn thương dây thần kinh cảm giác - da, tầm cổ chân và bàn chân	
S94.7	Tổn thương đa dây thần kinh, tầm cổ chân và bàn chân	
S94.8	Tổn thương dây thần kinh khác, tầm cổ chân và bàn chân	
S94.9	Tổn thương dây thần kinh không xác định, tầm cổ chân và bàn chân	
S95	Tổn thương mạch máu tầm cổ chân và bàn chân	
S95.0	Tổn thương động mạch mu bàn chân	
S95.1	Tổn thương động mạch gan bàn chân	
S95.2	Tổn thương tĩnh mạch mu bàn chân	
S95.7	Tổn thương đa mạch máu, tầm cổ chân và bàn chân	
S95.8	Tổn thương mạch máu khác, tầm cổ chân và bàn chân	
S95.9	Tổn thương mạch máu không xác định, tầm cổ chân và bàn chân	
S96	Tổn thương cơ và/hoặc gân, tầm cổ chân và bàn chân	
S96.0	Tổn thương cơ và/hoặc gân gấp dài của ngón, tầm cổ chân và bàn chân	
S96.1	Tổn thương cơ và/hoặc gân duỗi dài ngón chân, tầm cổ chân và bàn chân	
S96.2	Tổn thương cơ và/hoặc gân nội tại, tầm cổ chân và bàn chân	
S96.7	Tổn thương đa cơ và/hoặc gân, tầm cổ chân và bàn chân	
S96.8	Tổn thương cơ và/hoặc gân khác, tầm cổ chân và bàn chân	
S96.9	Tổn thương cơ và/hoặc gân không xác định, tầm cổ chân và bàn chân	
S97	Tổn thương dập nát cổ chân và/hoặc bàn chân	
S97.0	Tổn thương dập nát cổ chân	
S97.1	Tổn thương dập nát ngón chân	
S97.8	Tổn thương dập nát phần khác của cổ chân và/hoặc bàn chân	Tổn thương dập nát bàn chân không xác định khác
S98	Đứt rời cổ chân và/hoặc bàn chân do chấn thương	
S98.0	Đứt rời bàn chân tầm cổ chân do chấn thương	
S98.1	Đứt rời một ngón chân do chấn thương	
S98.2	Đứt rời hai hoặc nhiều ngón chân do chấn thương	
S98.3	Đứt rời phần khác của bàn chân do chấn thương	Đứt rời ngón kết hợp phần khác của bàn chân do chấn thương
S98.4	Đứt rời bàn chân do chấn thương, tầm không xác định	
S99	Tổn thương khác và/hoặc không xác định ở cổ chân và/hoặc bàn chân	
S99.7	Đa tổn thương ở cổ chân và/hoặc bàn chân	Tổn thương có thể phân loại ở nhiều mục trong S90.- - S98.
S99.8	Tổn thương xác định khác ở cổ chân và/hoặc bàn chân	
S99.9	Tổn thương không xác định ở cổ chân và/hoặc bàn chân	
T00	Tổn thương nông tác động đến nhiều vùng cơ thể	
T00.0	Tổn thương nông tác động đến vùng đầu kết hợp vùng cổ	
T00.1	Tổn thương nông tác động đến vùng ngực kết hợp vùng bụng, thắt lưng và/hoặc vùng chậu	
T00.2	Tổn thương nông tác động đến nhiều vùng của (các) chi trên	
T00.3	Tổn thương nông tác động đến nhiều vùng của (các) chi dưới	
T00.6	Tổn thương nông tác động đến nhiều vùng của (các) chi trên và (các) chi dưới	
T00.8	Tổn thương nông tác động kết hợp khác ở nhiều vùng cơ thể	
T00.9	Đa tổn thương nông, không xác định	
T01	Vết thương hở tác động đến nhiều vùng cơ thể	
T01.0	Vết thương hở tác động đến vùng đầu kết hợp vùng cổ	
T01.1	Vết thương hở tác động đến vùng ngực kết hợp vùng bụng, thắt lưng và/hoặc chậu	
T01.2	Vết thương hở ở tác động đến nhiều vùng của (các) chi trên	
T01.3	Vết thương hở tác động đến nhiều vùng của (các) chi dưới	
T01.6	Vết thương hở tác động đến nhiều vùng của (các) chi trên và (các) chi dưới	
T01.8	Vết thương hở tác động kết hợp khác ở nhiều vùng cơ thể	
T01.9	Đa vết thương hở, không xác định	
T02	Gãy xương tác động đến nhiều vùng cơ thể	
T02.0	Gãy xương tác động đến vùng đầu kết hợp vùng cổ	
T02.00	Gãy xương tác động đến vùng đầu kết hợp vùng cổ, gãy kín	
T02.01	Gãy xương tác động đến vùng đầu kết hợp vùng cổ, gãy hở	
T02.1	Gãy xương tác động đến vùng ngực, thắt lưng và/hoặc vùng chậu	
T02.10	Gãy xương tác động đến vùng ngực, thắt lưng và/hoặc chậu, gãy kín	
T02.11	Gãy xương tác động đến vùng ngực, thắt lưng và/hoặc chậu, gãy hở	
T02.2	Gãy xương tác động đến nhiều vùng của một chi trên	
T02.20	Gãy xương tác động đến nhiều vùng của một chi trên, gãy kín	
T02.21	Gãy xương tác động đến nhiều vùng của một chi trên, gãy hở	
T02.3	Gãy xương tác động đến nhiều vùng của một chi dưới	
T02.30	Gãy xương tác động đến nhiều vùng của một chi dưới, gãy kín	
T02.31	Gãy xương tác động đến nhiều vùng của một chi dưới, gãy hở	
T02.4	Gãy xương tác động đến nhiều vùng của cả hai chi trên	
T02.40	Gãy xương tác động đến nhiều vùng của cả hai chi trên, gãy kín	
T02.41	Gãy xương tác động đến nhiều vùng của cả hai chi trên, gãy hở	
T02.5	Gãy xương tác động đến nhiều vùng của cả hai chi dưới	
T02.50	Gãy xương tác động đến nhiều vùng của cả hai chi dưới, gãy kín	
T02.51	Gãy xương tác động đến nhiều vùng của cả hai chi dưới, gãy hở	
T02.6	Gãy xương tác động đến nhiều vùng của (các) chi trên và (các) chi dưới	
T02.60	Gãy xương tác động đến nhiều vùng của (các) chi trên và (các) chi dưới, gãy kín	
T02.61	Gãy xương tác động đến nhiều vùng của (các) chi trên và (các) chi dưới, gãy hở	
T02.7	Gãy xương tác động đến vùng ngực và vùng thắt lưng và/hoặc chậu và (các) chi	
T02.70	Gãy xương tác động đến vùng ngực và vùng thắt lưng và/hoặc chậu và (các) chi, gãy kín	
T02.71	Gãy xương tác động đến vùng ngực và vùng thắt lưng và/hoặc chậu và (các) chi, gãy hở	
T02.8	Gãy xương tác động kết hợp khác ở nhiều vùng cơ thể	
T02.80	Gãy xương tác động kết hợp khác ở nhiều vùng cơ thể, gãy kín	
T02.81	Gãy xương tác động kết hợp khác ở nhiều vùng cơ thể, gãy hở	
T02.9	Gãy xương, nhiều xương và/hoặc nhiều vị trí, không xác định	
T02.90	Gãy xương, nhiều xương và/hoặc nhiều vị trí, không xác định, gãy kín	
T02.91	Gãy xương, nhiều xương và/hoặc nhiều vị trí, không xác định, gãy hở	
T03	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở nhiều vùng cơ thể	
T03.0	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở vùng đầu kết hợp vùng cổ	
T03.1	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở vùng ngực kết hợp thắt lưng và/hoặc vùng chậu	
T03.2	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ tác động đến nhiều vùng của (các) chi trên	
T03.3	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở nhiều vùng của (các) chi dưới	
T03.4	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ tác động đến nhiều vùng của (các) chi trên và (các) chi dưới	
T03.8	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ tác động kết hợp khác ở nhiều vùng cơ thể	Đa trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ, không xác định
T03.9	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở nhiều vị trí, không xác định	
T04	Tổn thương dập nát tác động đến nhiều vùng cơ thể	
T04.0	Tổn thương dập nát tác động đến vùng đầu kết hợp vùng cổ	
T04.1	Tổn thương dập nát tác động đến vùng ngực kết hợp vùng bụng, thắt lưng và/hoặc chậu	
T04.2	Tổn thương dập nát tác động đến nhiều vùng của (các) chi trên	
T04.3	Tổn thương dập nát tác động đến nhiều vùng của (các) chi dưới	
T04.4	Tổn thương dập nát tác động đến nhiều vùng của (các) chi trên và (các) chi dưới	
T04.7	Tổn thương dập nát vùng ngực kết hợp vùng bụng, thắt lưng và/hoặc chậu và (các) chi	
T04.8	Tổn thương dập nát tác động kết hợp khác ở nhiều vùng cơ thể	
T04.9	Tổn thương dập nát nhiều vị trí, không xác định	
T05	Đứt rời do chấn thương tác động đến nhiều vùng cơ thể	
T05.0	Đứt rời cả hai bàn tay do chấn thương	
T05.1	Đứt rời một bàn tay và cánh tay khác [bất kỳ tầm chi nào, trừ bàn tay] do chấn thương	
T05.2	Đứt rời cả hai cánh tay [bất kỳ tầm chi nào] do chấn thương	
T05.3	Đứt rời cả hai bàn chân do chấn thương	
T05.4	Đứt rời một bàn chân và chân khác [bất kỳ tầm chi nào trừ bàn chân] do chấn thương	
T05.5	Đứt rời cả hai chân [bất kỳ tầm chi nào] do chấn thương	
T05.6	Đứt rời chi trên và chi dưới, bất kỳ sự kết hợp nào [bất kỳ tầm chi nào] do chấn thương	
T05.8	Đứt rời do chấn thương tác động kết hợp khác ở nhiều vùng cơ thể	
T05.9	Đứt rời do chấn thương ở nhiều phần cơ thể, không xác định	
T06	Tổn thương khác tác động đến nhiều vùng cơ thể, không phân loại mục khác	
T06.0	Tổn thương não và/hoặc dây thần kinh sọ kết hợp tổn thương của dây thần kinh và/hoặc tủy sống vùng cổ	Tổn thương có thể phân loại vào S04.- và S06.- kèm tổn thương có thể phân loại vào S14.
T06.1	Tổn thương tủy sống kết hợp dây thần kinh tác động đến nhiều vùng khác của cơ thể	
T06.2	Tổn thương dây thần kinh tác động đến nhiều vùng khác của cơ thể	
T06.3	Tổn thương mạch máu tác động đến nhiều vùng khác của cơ thể	
T06.4	Tổn thương cơ và/hoặc gân tác động đến nhiều vùng khác của cơ thể	
T06.5	Tổn thương tạng trong khoang ngực kết hợp tạng trong ổ bụng và/hoặc vùng chậu	
T06.8	Tổn thương xác định khác tác động đến nhiều vùng khác của cơ thể	
T07	Đa tổn thương không xác định	
T08	Gãy cột sống, đoạn không xác định	
T08.0	Gãy cột sống, đoạn không xác định, gãy kín	
T08.1	Gãy cột sống, đoạn không xác định, gãy hở	
T09	Tổn thương khác ở cột sống và/hoặc thân, vùng không xác định	
T09.0	Tổn thương nông tại thân, vùng không xác định	
T09.1	Vết thương hở tại thân, vùng không xác định	
T09.2	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng không xác định của thân	
T09.3	Tổn thương tủy sống, đoạn không xác định	
T09.4	Tổn thương không xác định của dây thần kinh, rễ dây thần kinh cột sống và/hoặc đám rối của thân	
T09.5	Tổn thương không xác định của cơ và/hoặc gân của thân	
T09.6	Đứt rời thân do chấn thương, vùng không xác định	
T09.8	Tổn thương xác định khác tại thân, vùng không xác định	
T09.9	Tổn thương không xác định của thân, vùng không xác định	
T10	Gãy xương chi trên, tầm chi không xác định	
T10.0	Gãy xương chi trên, tầm chi không xác định, gãy kín	
T10.1	Gãy xương chi trên, tầm chi không xác định, gãy hở	
T11	Tổn thương khác ở chi trên, tầm không xác định	
T11.0	Tổn thương nông của chi trên, tầm chi không xác định	
T11.1	Vết thương hở của chi trên, tầm chi không xác định	
T11.2	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ khớp không xác định và/hoặc dây chằng chi trên, tầm chi không xác định	
T11.3	Tổn thương dây thần kinh không xác định tại chi trên, tầm chi không xác định	
T11.4	Tổn thương mạch máu không xác định tại chi trên, tầm chi không xác định	
T11.5	Tổn thương cơ không xác định và/hoặc gân tại chi trên, tầm chi không xác định	
T11.6	Đứt rời chi trên do chấn thương, tầm chi không xác định	Chấn thương gây đứt rời tay không xác định khác
T11.8	Tổn thương xác định khác của chi trên, tầm chi không xác định	
T11.9	Tổn thương không xác định của chi trên, tầm chi không xác định	Chấn thương cánh tay không xác định khác
T12	Gãy xương chi dưới, tầm chi không xác định	
T12.0	Gãy xương chi dưới, tầm chi không xác định, gãy kín	
T12.1	Gãy xương chi dưới, tầm chi không xác định, gãy hở	
T13	Tổn thương khác ở chi dưới, tầm không xác định	
T13.0	Tổn thương nông của chi dưới, tầm chi không xác định	
T13.1	Vết thương hở của chi dưới, tầm chi không xác định	
T13.2	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở khớp và/hoặc dây chằng chi dưới, tầm chi không xác định	
T13.3	Tổn thương dây thần kinh không xác định tại chi dưới, tầm chi không xác định	
T13.4	Tổn thương mạch máu không xác định tại chi dưới, tầm chi không xác định	
T13.5	Tổn thương cơ và/hoặc gân không xác định tại chi dưới, tầm chi không xác định	
T13.6	Đứt rời chi dưới do chấn thương, tầm chi không xác định	Chấn thương gây đứt rời chân không xác định khác
T13.8	Tổn thương xác định khác tại chi dưới, tầm chi không xác định	
T13.9	Tổn thương không xác định tại chi dưới, tầm chi không xác định	Tổn thương chân không xác định khác
T14	Tổn thương ở vùng cơ thể không xác định	
T14.0	Tổn thương nông ở vùng cơ thể không xác định	
T14.1	Vết thương hở ở vùng cơ thể không xác định	
T14.2	Gãy xương ở vùng cơ thể không xác định	
T14.20	Gãy xương ở vùng cơ thể không xác định, gãy kín	
T14.21	Gãy xương ở vùng cơ thể không xác định, gãy hở	
T14.3	Trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ ở vùng cơ thể không xác định	
T14.4	Tổn thương dây thần kinh ở vùng cơ thể không xác định	
T14.5	Tổn thương mạch máu ở vùng cơ thể không xác định	
T14.6	Tổn thương cơ và/hoặc gân ở vùng cơ thể không xác định	
T14.7	Tổn thương dập nát và/hoặc đứt rời do chấn thương ở vùng cơ thể không xác định	
T14.8	Tổn thương khác ở vùng cơ thể không xác định	
T14.9	Tổn thương, không xác định	
T15	Dị vật ở mắt ngoài	
T15.0	Dị vật trong giác mạc	
T15.1	Dị vật trong túi kết mạc	
T15.8	Dị vật ở các vị trí khác và/hoặc nhiều vị trí của mắt ngoài	Dị vật trong điểm lệ
T15.9	Dị vật ở mắt ngoài, phần không xác định	
T16	Dị vật ở tai	
T17	Dị vật đường hô hấp	
T17.0	Dị vật trong xoang mũi	
T17.1	Dị vật trong lỗ mũi	Mũi không xác định khác
T17.2	Dị vật trong họng	Mũi - hầu|Họng không xác định khác
T17.3	Dị vật trong thanh quản	
T17.4	Dị vật trong khí quản	
T17.5	Dị vật trong phế quản	
T17.8	Dị vật ở vị trí khác và/hoặc ở nhiều vị trí của đường hô hấp	Tiểu phế quản|Phổi
T17.9	Dị vật đường hô hấp, phần không xác định	
T18	Dị vật đường tiêu hóa	
T18.0	Dị vật trong miệng	
T18.1	Dị vật trong thực quản	
T18.2	Dị vật trong dạ dày	
T18.3	Dị vật trong ruột non	
T18.4	Dị vật trong đại tràng	
T18.5	Dị vật trong hậu môn và/hoặc trực tràng	
T18.8	Dị vật ở vị trí khác và/hoặc ở nhiều vị trí của đường tiêu hóa	
T18.9	Dị vật trong đường tiêu hóa, phần không xác định	Hệ tiêu hóa không xác định khác|Nuốt dị vật không xác định khác
T19	Dị vật đường sinh dục - tiết niệu	
T19.0	Dị vật trong niệu đạo	
T19.1	Dị vật trong bàng quang	
T19.2	Dị vật trong âm hộ và/hoặc âm đạo	
T19.3	Dị vật trong tử cung [bất kỳ vị trí nào]	
T19.8	Dị vật ở các vị trí khác và/hoặc ở nhiều vị trí của đường sinh dục - tiết niệu	
T19.9	Dị vật đường sinh dục - tiết niệu, phần không xác định	
T20	Bỏng và/hoặc ăn mòn vùng đầu và/hoặc cổ	Bỏng và/hoặc ăn mòn ở đầu và/hoặc cổ
T20.0	Bỏng vùng đầu và/hoặc cổ, không xác định mức độ	
T20.1	Bỏng độ I (biểu bì) vùng đầu và/hoặc cổ	
T20.2	Bỏng độ II (trung bì) vùng đầu và/hoặc cổ	
T20.3	Bỏng độ III (hết lớp da hoặc sâu hơn) vùng đầu và/hoặc cổ	
T20.4	Ăn mòn độ chưa xác định tvùng đầu và/hoặc cổ	
T20.5	Ăn mòn độ I (biểu bì) vùng đầu và/hoặc cổ	
T20.6	Ăn mòn độ II (trung bì) vùng đầu và/hoặc cổ	
T20.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn) vùng đầu và/hoặc cổ	
T21	Bỏng và/hoặc ăn mòn tại thân	
T21.0	Bỏng tại thân, không xác định mức độ	
T21.1	Bỏng độ I (biểu bì) tại thân	
T21.2	Bỏng độ II (trung bì) tại thân	
T21.3	Bỏng độ III (hết lớp da hoặc sâu hơn) tại thân	
T21.4	Ăn mòn tại thân, không xác định mức độ	
T21.5	Ăn mòn độ I (biểu bì) tại thân	
T21.6	Ăn mòn độ II (trung bì) tại thân	
T21.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn) tại thân	
T22	Bỏng và/hoặc ăn mòn tại vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.0	Bỏng ở vai và/hoặc chi trên, trừ cổ tay và bàn tay, độ bỏng không xác định	
T22.1	Bỏng độ I (biểu bì) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.2	Bỏng độ II (trung bì) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.3	Bỏng độ III (hết lớp da hoặc sâu hơn) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.4	Ăn mòn ở vai và/hoặc chi trên không xác định mức độ, trừ cổ tay và bàn tay	
T22.5	Ăn mòn độ I (biểu bì) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.6	Ăn mòn độ II (trung bì) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T22.7	Ăn mòn độ III (hết hớp da hoặc sâu hơn) vùng vai và/hoặc chi trên, trừ cổ tay và bàn tay	
T23	Bỏng và/hoặc ăn mòn tại cổ tay và/hoặc bàn tay	
T23.0	Bỏng ở cổ tay và bàn tay, không xác định mức độ	
T23.1	Bỏng độ I (biểu bì) cổ tay và/hoặc bàn tay	
T23.2	Bỏng độ II (trung bì) cổ tay và/hoặc bàn tay	
T23.3	Bỏng độ III (hết lớp da hoặc sâu hơn) cổ tay và/hoặc bàn tay	
T23.4	Ăn mòn của cổ tay và/hoặc bàn tay, không xác định mức độ	
T23.5	Ăn mòn độ I (biểu bì) cổ tay và/hoặc bàn tay	
T23.6	Ăn mòn độ II (trung bì) cổ tay và/hoặc bàn tay	
T23.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn) cổ tay và/hoặc bàn tay	
T24	Bỏng và/hoặc ăn mòn tại háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	Bỏng và/hoặc ăn mòn tại hông và/hoặc chi dưới, trừ cổ chân và bàn chân
T24.0	Bỏng tại hông và/hoặc chi dưới, trừ cổ chân và bàn chân, độ bỏng không xác định	
T24.1	Bỏng độ I (biểu bì) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T24.2	Bỏng độ II (trung bì) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T24.3	Bỏng độ III (hết lớp da hoặc sâu hơn) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T24.4	Ăn mòn vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân, không xác định mức độ	
T24.5	Ăn mòn độ I (biểu bì) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T24.6	Ăn mòn độ II (trung bì) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T24.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn) vùng háng và/hoặc chi dưới, trừ cổ chân và/hoặc bàn chân	
T25	Bỏng và/hoặc ăn mòn tại cổ chân và/hoặc bàn chân	
T25.0	Bỏng tại cổ chân và/hoặc bàn chân, không xác định mức độ	
T25.1	Bỏng độ I (biểu bì) tại cổ chân và/hoặc bàn chân	
T25.2	Bỏng độ II (trung bì) tại cổ chân và/hoặc bàn chân	
T25.3	Bỏng độ III (hết lớp da hoặc sâu hơn) tại cổ chân và/hoặc bàn chân	
T25.4	Ăn mòn tại cổ chân và/hoặc bàn chân, không xác định mức độ	
T25.5	Ăn mòn độ I (biểu bì) tại cổ chân và/hoặc bàn chân	
T25.6	Ăn mòn độ II (trung bì) tại cổ chân và/hoặc bàn chân	
T25.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn) tại cổ chân và/hoặc bàn chân	
T26	Bỏng và/hoặc ăn mòn giới hạn tại mắt và/hoặc phần phụ của mắt	Bỏng và/hoặc ăn mòn giới hạn tại mắt và/hoặc cấu trúc phụ cận của mắt
T26.0	Bỏng tại mi mắt và/hoặc vùng quanh nhãn cầu	
T26.1	Bỏng tại giác mạc và/hoặc túi kết mạc	
T26.2	Bỏng với hậu quả gây vỡ và/hoặc phá hủy nhãn cầu	
T26.3	Bỏng tại phần khác của mắt và/hoặc phần phụ của mắt	
T26.4	Bỏng tại mắt và/hoặc cấu trúc phụ của mắt, phần chưa xác định	
T26.5	Ăn mòn tại mi mắt và/hoặc vùng quanh nhãn cầu	
T26.6	Ăn mòn tại giác mạc và/hoặc túi kết mạc	
T26.7	Ăn mòn với hậu quả gây vỡ và/hoặc phá hủy nhãn cầu	
T26.8	Ăn mòn tại phần khác của mắt và/hoặc cấu trúc phụ của mắt	
T26.9	Ăn mòn của mắt và/hoặc cấu trúc phụ của mắt, không xác định vùng	
T27	Bỏng và/hoặc ăn mòn đường hô hấp	
T27.0	Bỏng thanh quản và/hoặc khí quản	
T27.1	Bỏng tác động thanh quản và/hoặc khí quản với phổi	
T27.2	Bỏng phần khác của đường hô hấp	Khoang ngực
T27.3	Bỏng đường hô hấp, không xác định vùng	
T27.4	Ăn mòn thanh quản và/hoặc khí quản	
T27.5	Ăn mòn tác động thanh quản và/hoặc khí quản với phổi	
T27.6	Ăn mòn ở các vị trí khác của đường hô hấp	
T27.7	Ăn mòn đường hô hấp, không xác định vị trí	
T28	Bỏng và/hoặc ăn mòn nội tạng khác	
T28.0	Bỏng tại miệng và/hoặc hầu họng	
T28.1	Bỏng tại thực quản	
T28.2	Bỏng tại các vị trí khác của đường tiêu hóa	
T28.3	Bỏng tại phần trong của đường tiết niệu sinh dục	
T28.4	Bỏng nơi khác và/hoặc các nội tạng không xác định	
T28.5	Ăn mòn tại miệng và/hoặc thanh quản	
T28.6	Ăn mòn thực quản	
T28.7	Ăn mòn tại các vị trí khác của đường tiêu hóa	
T28.8	Ăn mòn các cơ quan bên trong hệ sinh dục - tiết niệu	
T28.9	Ăn mòn các cơ quan nội tạng khác và/hoặc cơ quan nội tạng không xác định	
T29	Bỏng và/hoặc ăn mòn nhiều vùng cơ thể	
T29.0	Bỏng nhiều vùng, độ bỏng không xác định	Bỏng nhiều vùng không xác định khác
T29.1	Bỏng nhiều vùng, không có vết bỏng nào nặng hơn độ I (biểu bì)	
T29.2	Bỏng nhiều vùng, không có vết bỏng nào nặng hơn độ II (trung bì)	
T29.3	Bỏng nhiều vùng, ít nhất có một vết bỏng độ III (hết lớp da hoặc sâu hơn)	
T29.4	Ăn mòn nhiều vùng, độ ăn mòn không xác định	Ăn mòn nhiều vùng không xác định khác
T29.5	Ăn mòn nhiều vùng, không có vùng nào bị ăn mòn nặng hơn độ I (biểu bì)	
T29.6	Ăn mòn nhiều vùng, không có vùng nào bị ăn mòn nặng hơn độ II (trung bì)	
T29.7	Ăn mòn nhiều vùng, ít nhất có một vùng bị ăn mòn độ III (hết lớp da hoặc sâu hơn)	
T30	Bỏng và/hoặc ăn mòn, không xác định vùng cơ thể	Bỏng và/hoặc ăn mòn, vùng cơ thể không xác định
T30.0	Bỏng không xác định vùng cơ thể và mức độ	Bỏng không xác định khác
T30.1	Bỏng độ I (biểu bì), không xác định vùng cơ thể	Bỏng độ một không xác định khác
T30.2	Bỏng độ II (trung bì), không xác định vùng cơ thể	Bỏng độ hai không xác định khác
T30.3	Bỏng độ III (hết lớp da hoặc sâu hơn), không xác định vùng cơ thể	Bỏng độ ba không xác định khác
T30.4	Ăn mòn không xác định vùng cơ thể và mức độ	Ăn mòn không xác định khác
T30.5	Ăn mòn độ I (biểu bì), không xác định vùng cơ thể	Ăn mòn độ một không xác định khác
T30.6	Ăn mòn độ II (trung bì), không xác định vùng cơ thể	Ăn mòn độ hai không xác định khác
T30.7	Ăn mòn độ III (hết lớp da hoặc sâu hơn), không xác định vùng cơ thể	Ăn mòn độ ba không xác định khác
T31	Phân loại bỏng theo diện tích bề mặt cơ thể bị tổn thương	
T31.0	Bỏng ít hơn 10% diện tích cơ thể	
T31.1	Bỏng từ 10 đến 19% diện tích cơ thể	
T31.2	Bỏng từ 20 đến 29% diện tích cơ thể	
T31.3	Bỏng từ 30 đến 39% diện tích cơ thể	
T31.4	Bỏng từ 40 đến 49% diện tích cơ thể	
T31.5	Bỏng từ 50 đến 59% diện tích cơ thể	
T31.6	Bỏng từ 60 đến 69% diện tích cơ thể	
T31.7	Bỏng từ 70 đến 79% diện tích cơ thể	
T31.8	Bỏng từ 80 đến 89% diện tích cơ thể	
T31.9	Bỏng lớn hơn 90% diện tích cơ thể	
T32	Phân loại ăn mòn theo diện tích bề mặt cơ thể bị tổn thương	
T32.0	Ăn mòn dưới 10% diện tích cơ thể	
T32.1	Ăn mòn từ 10 đến 19% diện tích cơ thể	
T32.2	Ăn mòn từ 20 đến 29% diện tích cơ thể	
T32.3	Ăn mòn từ 30 đến 39% diện tích cơ thể	
T32.4	Ăn mòn từ 40 đến 49% diện tích cơ thể	
T32.5	Ăn mòn từ 50 đến 59% diện tích cơ thể	
T32.6	Ăn mòn từ 60-69% diện tích cơ thể	
T32.7	Ăn mòn từ 70 đến 79% diện tích cơ thể	
T32.8	Ăn mòn từ 80 đến 89% diện tích cơ thể	
T32.9	Ăn mòn lớn hơn 90% diện tích cơ thể	
T33	Bỏng lạnh nông	
T33.0	Bỏng lạnh nông ở đầu	
T33.1	Bỏng lạnh nông ở cổ	
T33.2	Bỏng lạnh nông ở ngực	
T33.3	Bỏng lạnh nông ở thành bụng, thắt lưng và/hoặc vùng chậu	
T33.4	Bỏng lạnh nông ở cánh tay	
T33.5	Bỏng lạnh nông ở cổ tay và/hoặc bàn tay	
T33.6	Bỏng lạnh nông ở hông và/hoặc đùi	
T33.7	Bỏng lạnh nông ở đầu gối và/hoặc cẳng chân	
T33.8	Bỏng lạnh nông ở cổ chân và/hoặc bàn chân	
T33.9	Bỏng lạnh nông ở vị trí khác và/hoặc không xác định	
T34	Bỏng lạnh có hoại tử mô	
T34.0	Bỏng lạnh có hoại tử mô ở đầu	
T34.1	Bỏng lạnh có hoại tử mô ở cổ	
T34.2	Bỏng lạnh có hoại tử mô ở ngực	
T34.3	Bỏng lạnh có hoại tử mô ở thành bụng, thắt lưng và/hoặc vùng chậu	
T34.4	Bỏng lạnh có hoại tử mô ở cánh tay	
T34.5	Bỏng lạnh có hoại tử mô ở cổ tay và/hoặc bàn tay	
T34.6	Bỏng lạnh có hoại tử mô ở hông và/hoặc đùi	
T34.7	Bỏng lạnh có hoại tử mô ở đầu gối và/hoặc cẳng chân	
T34.8	Bỏng lạnh có hoại tử mô ở cổ chân và/hoặc bàn chân	
T34.9	Bỏng lạnh có hoại tử mô ở vị trí khác và/hoặc không xác định	
T35	Bỏng lạnh tác động đến nhiều vùng cơ thể và/hoặc bỏng lạnh không xác định	
T35.0	Bỏng lạnh tác động đến nhiều vùng cơ thể	Đa vết bỏng lạnh nông không xác định khác
T35.1	Bỏng lạnh có hoại tử mô tác động đến nhiều vùng cơ thể	Đa vết bỏng lạnh có hoại tử mô không xác định khác
T35.2	Bỏng lạnh không xác định, ở đầu và/hoặc cổ	
T35.3	Bỏng lạnh không xác định, ở vùng ngực, bụng, thắt lưng và/hoặc chậu	Bỏng lạnh ở thân không xác định khác
T35.4	Bỏng lạnh không xác định, ở chi trên	
T35.5	Bỏng lạnh không xác định, ở chi dưới	
T35.6	Bỏng lạnh không xác định, tác động đến nhiều vùng cơ thể	Đa vết bỏng lạnh không xác định khác
T35.7	Bỏng lạnh không xác định, ở vị trí không xác định	Bỏng lạnh không xác định khác
T36	Ngộ độc kháng sinh toàn thân	
T36.0	Ngộ độc kháng sinh penicillin	
T36.1	Ngộ độc kháng sinh Cefalosporin và/hoặc kháng sinh beta-lactam khác	
T36.2	Ngộ độc kháng sinh nhóm chloramphenicol	
T36.3	Ngộ độc kháng sinh macrolid	
T36.4	Ngộ độc kháng sinh Tetracyclin	
T36.5	Ngộ độc kháng sinh aminoglycosid	Streptomycin
T36.6	Ngộ độc kháng sinh Rifamycin	
T36.7	Ngộ độc thuốc kháng nấm, sử dụng toàn thân	
T36.8	Ngộ độc kháng sinh toàn thân loại khác	
T36.9	Ngộ độc kháng sinh toàn thân, không xác định	
T37	Ngộ độc do thuốc chống nhiễm trùng và/hoặc thuốc chống ký sinh trùng toàn thân khác	
T37.0	Ngộ độc kháng sinh sulfonamid	
T37.1	Ngộ độc thuốc kháng mycobacteria	
T37.2	Ngộ độc thuốc chống sốt rét và/hoặc thuốc tác động lên động vật nguyên sinh trong máu khác	
T37.3	Ngộ độc thuốc chống động vật nguyên sinh khác	
T37.4	Ngộ độc thuốc tẩy giun sán	
T37.5	Ngộ độc thuốc kháng virus	
T37.8	Ngộ độc thuốc chống nhiễm trùng và/hoặc thuốc chống ký sinh trùng toàn thân xác định khác	
T37.9	Ngộ độc thuốc chống nhiễm trùng và/hoặc thuốc chống ký sinh trùng toàn thân, không xác định	
T38	Ngộ độc do nội tiết tố [hormon] và/hoặc chất tổng hợp thay thế và/hoặc đối kháng của chúng, không phân loại mục khác	
T38.0	Ngộ độc glucocorticoid và/hoặc thuốc tổng hợp tương tự	
T38.1	Ngộ độc nội tiết tố [hormon] tuyến giáp và/hoặc chất thay thế	
T38.2	Ngộ độc thuốc chống tuyến giáp	
T38.3	Ngộ độc insulin và/hoặc thuốc hạ đường huyết dạng uống [chống đái tháo đường]	
T38.4	Ngộ độc thuốc tránh thai đường uống	Chế phẩm một và/hoặc nhiều thành phần
T38.5	Ngộ độc estrogen và/hoặc progestogen khác	Hỗn hợp và/hoặc thay thế
T38.6	Ngộ độc thuốc kháng gonadotropin, kháng estrogen, kháng androgen, không phân loại mục khác	Tamoxifen
T38.7	Ngộ độc androgen và/hoặc sản phẩm đồng hóa tương tự	
T38.8	Ngộ độc nội tiết tố [hormon] khác và/hoặc không xác định và/hoặc chất tổng hợp thay thế của chúng	Hormon [adenohypophyseal] thùy trước tuyến yên
T38.9	Ngộ độc kháng nội tiết tố [hormon] khác và/hoặc không xác định	
T39	Ngộ độc do thuốc giảm đau, hạ nhiệt và/hoặc chống viêm khớp dạng thấp không có chất dạng thuốc phiện	
T39.0	Ngộ độc salicylat	
T39.1	Ngộ độc chất dẫn xuất 4-aminophenol	
T39.2	Ngộ độc chất dẫn xuất pyrazolone	
T39.3	Ngộ độc thuốc chống viêm không steroid [NSAID] khác	
T39.4	Ngộ độc thuốc chống viêm khớp dạng thấp, không phân loại mục khác	
T39.8	Ngộ độc thuốc giảm đau và/hoặc hạ sốt nonopioid khác, không phân loại mục khác	
T39.9	Ngộ độc thuốc giảm đau, hạ sốt, chống viêm khớp dạng thấp không có chất dạng thuốc phiện, không xác định	
T40	Ngộ độc chất ma túy và/hoặc chất gây ảo giác	
T40.0	Ngộ độc thuốc phiện [opium]	
T40.1	Ngộ độc heroin	
T40.2	Ngộ độc thuốc có nguồn gốc thuốc phiện khác	Codeine|Morphine
T40.3	Ngộ độc methadone	
T40.4	Ngộ độc chất ma túy tổng hợp khác	Pethidine
T40.5	Ngộ độc cocaine	
T40.6	Ngộ độc chất ma túy khác và/hoặc không xác định	
T40.7	Ngộ độc cần sa (dẫn xuất)	
T40.8	Ngộ độc lysergid [LSD]	
T40.9	Ngộ độc chất gây ảo giác [gây hoang tưởng] khác và/hoặc không xác định	Mescaline|Psilocin|Psilocybine
T41	Ngộ độc chất gây tê/gây tê và/hoặc khí trị liệu	
T41.0	Ngộ độc chất gây mê đường thở	
T41.1	Ngộ độc chất gây mê đường tĩnh mạch	Thiobarbiturat
T41.2	Ngộ độc chất gây mê toàn thân khác và/hoặc không xác định	
T41.3	Ngộ độc thuốc gây tê tại chỗ	
T41.4	Ngộ độc chất gây mê/gây tê, không xác định	
T41.5	Ngộ độc khí trị liệu	CO2 [Carbon dioxide] [cacbon dioxyt]|Oxy [Oxygen]
T42	Ngộ độc thuốc chống động kinh, thuốc an thần - gây ngủ và/hoặc thuốc chống hội chứng parkison	
T42.0	Ngộ độc dẫn xuất của hydantoin	
T42.1	Ngộ độc iminostiben	Carbamazepin
T42.2	Ngộ độc succinimid và/hoặc oxazoildinedion	
T42.3	Ngộ độc barbiturat	
T42.4	Ngộ độc benzodiazepin	
T42.5	Ngộ độc hỗn hợp thuốc chống động kinh, không phân loại mục khác	
T42.6	Ngộ độc thuốc chống động kinh và/hoặc thuốc an thần - gây ngủ khác	
T42.7	Ngộ độc thuốc chống động kinh, an thần gây ngủ, không xác định	
T42.8	Ngộ độc thuốc chống hội chứng parkinson và/hoặc thuốc ức chế trương lực cơ trung tâm khác	Amantadin
T43	Ngộ độc thuốc hướng thần, không phân loại mục khác	
T43.0	Ngộ độc thuốc chống trầm cảm ba vòng và/hoặc bốn vòng	
T43.1	Ngộ độc thuốc chống trầm cảm nhóm ức chế men IMAO	
T43.2	Ngộ độc thuốc chống trầm cảm khác và/hoặc không xác định	
T43.3	Ngộ độc thuốc chống loạn thân và/hoặc thuốc an thần dạng phenothiazin	
T43.4	Ngộ độc thuốc an thần butyrophenon và/hoặc thioxanthen	
T43.5	Ngộ độc thuốc chống loạn thần và/hoặc an thần khác và/hoặc không xác định	
T43.6	Ngộ độc thuốc kích thích tâm thần có thể bị lạm dụng	
T43.8	Ngộ độc thuốc hướng thần khác, không phân loại mục khác	
T43.9	Ngộ độc thuốc hướng thần, không xác định	
T44	Ngộ độc do thuốc tác động chủ yếu đến hệ thần kinh tự động	
T44.0	Ngộ độc tác nhân ức chế [kháng] enzyme cholinesterase	
T44.1	Ngộ độc chất tác dụng giống như kích thích phó giao cảm [Thuốc kích thích hệ cholinergic]	
T44.2	Ngộ độc thuốc ngăn chạn hạch [chẹn], chưa phân loại mục khác	
T44.3	Ngộ độc chất ức chế phó giao cảm (kháng cholinergic và/hoặc kháng hệ muscarinic) và chống co thắt khác, không phân loại mục khác	Papaverine
T44.4	Ngộ độc chủ yếu chủ vận alpha [chất chủ vận thụ thể tuyến thượng thận alpha], không phân loại mục khác	Metaraminol
T44.5	Ngộ độc chất chủ yếu chủ vận beta [chất chủ vận thụ thể nội tiết tố tuyến thượng thận Beta], không phân loại mục khác	
T44.6	Ngộ độc chất đối kháng thụ thể tuyến thượng thận alpha [thuốc chẹn alpha], không phân loại mục khác	
T44.7	Ngộ độc chất đối kháng thụ thể tuyến thượng thận beta [thuốc chẹn beta], không phân loại mục khác	
T44.8	Ngộ độc tác nhân tác động trung tâm và/hoặc tác nhân đối kháng thụ thể tuyến thượng thận, không phân loại mục khác	
T44.9	Ngộ độc thuốc tác động chủ yếu đến hệ thần kinh tự động [hệ thần kinh thực vật] khác và/hoặc không xác định	Thuốc kích thích cả thụ thể tuyến thượng thận alpha và beta
T45	Ngộ độc tác nhân chủ yếu tác động toàn thân và/hoặc huyết học, không phân loại mục khác	
T45.0	Ngộ độc thuốc chống nôn và/hoặc thuốc chống dị ứng	
T45.1	Ngộ độc thuốc điều trị ung thư và/hoặc thuốc ức chế miễn dịch	
T45.2	Ngộ độc vitamin, không phân loại mục khác	
T45.3	Ngộ độc enzym, không phân loại mục khác	
T45.4	Ngộ độc sắt và/hoặc hợp chất của sắt	
T45.5	Ngộ độc thuốc chống đông máu	
T45.6	Ngộ độc thuốc tác động phân hủy fibrin	
T45.7	Ngộ độc thuốc đối kháng chống đông máu, vitamin K, và/hoặc chất làm đông máu [cầm máu] khác	
T45.8	Ngộ độc tác nhân chủ yếu tác động toàn thân và/hoặc huyết học khác	
T45.9	Ngộ độc tác nhân chủ yếu tác động toàn thân và/hoặc huyết học, không xác định	
T46	Ngộ độc tác nhân chủ yếu tác động đến hệ tim mạch	
T46.0	Ngộ độc glycoside kích thích tim và/hoặc thuốc có tác dụng tương tự	
T46.1	Ngộ độc thuốc chẹn kênh canxi	
T46.2	Ngộ độc thuốc chống loạn nhịp khác, không phân loại mục khác	
T46.3	Ngộ độc thuốc giãn động mạch vành, không phân loại mục khác	
T46.4	Ngộ độc thuốc ức chế men chuyển angiotensin	
T46.5	Ngộ độc thuốc hạ huyết áp khác, không phân loại mục khác	
T46.6	Ngộ độc thuốc hạ lipid máu và/hoặc chống xơ cứng động mạch	
T46.7	Ngộ độc thuốc giãn động mạch ngoại vi	
T46.8	Ngộ độc thuốc chống giãn tĩnh mạch, kể cả tác nhân gây xơ	
T46.9	Ngộ độc tác nhân tác động chủ yếu đến hệ thống tim mạch khác và/hoặc không xác định	
T47	Ngộ độc do tác nhân tác động chủ yếu trên hệ thống tiêu hóa	
T47.0	Ngộ độc chất đối kháng thụ thể Histamin H2	
T47.1	Ngộ độc chất chống acid và chống tiết dịch vị khác	
T47.2	Ngộ độc chất kích thích nhuận tràng	
T47.3	Ngộ độc chất nhuận tràng thẩm thấu và muối	
T47.4	Ngộ độc chất nhuận tràng khác	Thuốc điều trị giảm trương lực ruột
T47.5	Ngộ độc chất lợi tiêu hóa	
T47.6	Ngộ độc thuốc chống tiêu chảy	
T47.7	Ngộ độc chất gây nôn	
T47.8	Ngộ độc tác nhân khác tác động chủ yếu trên hệ thống tiêu hóa	
T47.9	Ngộ độc tác nhân tác động chủ yếu trên hệ tiêu hóa, không xác định	
T48	Ngộ độc do tác nhân tác động chủ yếu trên cơ trơn và/hoặc cơ xương và/hoặc hệ hô hấp	
T48.0	Ngộ độc thuốc trợ đẻ [tác động gây hoặc tăng cường co thắt tử cung]	
T48.1	Ngộ độc tác nhân giãn cơ xương (thuốc ức chế thần kinh - cơ)	
T48.2	Ngộ độc tác nhân chủ yếu tác động trên cơ khác và/hoặc không xác định	
T48.3	Ngộ độc chất chống ho	
T48.4	Ngộ độc thuốc long đờm	
T48.5	Ngộ độc chất chống cảm lạnh	
T48.6	Ngộ độc chất trị hen phế quản, không phân loại mục khác	
T48.7	Ngộ độc tác nhân tác động chủ yếu trên hệ hô hấp khác và/hoặc không xác định	
T49	Ngộ độc tác nhân dùng tại chỗ tác động chủ yếu trên da và/hoặc niêm mạc và/hoặc do thuốc mắt, tai - mũi - họng và/hoặc nha khoa	Ngộ độc tác nhân dùng tại chỗ tác động chủ yếu trên da và/hoặc niêm mạc và/hoặc do thuốc mắt, tai-mũi - họng và/hoặc nha khoa
T49.0	Ngộ độc thuốc kháng nấm, chống nhiễm trùng, kháng viêm tại chỗ không phân loại mục khác	
T49.1	Ngộ độc chất chống ngứa	
T49.2	Ngộ độc chất làm săn da [làm se khít] tại chỗ và/hoặc thuốc sát [kh] trùng tại chỗ	
T49.3	Ngộ độc chất làm mềm da, làm dịu da và/hoặc bảo vệ da	
T49.4	Ngộ độc thuốc làm tróc lớp sừng, tạo hình lớp sừng và/hoặc thuốc và/hoặc các chế phẩm điều trị tóc khác	
T49.5	Ngộ độc thuốc và/hoặc chế phẩm điều trị mắt	Thuốc chống nhiễm trùng mắt
T49.6	Ngộ độc thuốc và/hoặc chế phẩm điều trị tai, mũi, họng	Thuốc chống nhiễm trùng ở tai, mũi họng
T49.7	Ngộ độc thuốc nha khoa dùng tại chỗ	
T49.8	Ngộ độc tác nhân dùng tại chỗ khác	Thuốc diệt tinh trùng
T49.9	Ngộ độc tác nhân dùng tại chỗ, không xác định	
T50	Ngộ độc thuốc lợi tiểu và/hoặc dược chất, thuốc điều trị, sinh phẩm khác và/hoặc không xác định	
T50.0	Ngộ độc thuốc Mineralocorticoid và/hoặc chất đối kháng của chúng	
T50.1	Ngộ độc thuốc lợi tiểu quai tiểu quản thận	
T50.2	Ngộ độc thuốc ức chế anhydrase carbonic, benzothladiazid và lợi tiểu khác	Acetazolamid
T50.3	Ngộ độc tác nhân cân bằng điện giải, nhiệt lượng và/hoặc nước	Ngộ độc thuốc bù nước và điện giải [ORS]
T50.4	Ngộ độc thuốc tác động chuyển hóa acid uric	
T50.5	Ngộ độc ức chế sự thèm ăn [thuốc giảm ngon miệng]	
T50.6	Ngộ độc tác nhân giải độc và/hoặc đối khoáng [giải độc] kim loại nặng, không phân loại mục khác	Ngộ độc thuốc cai nghiện rượu
T50.7	Ngộ độc thuốc phục hồi sức khỏe [sức lực] và/hoặc chất đối kháng thụ thể opioid	
T50.8	Ngộ độc tác nhân dùng để chẩn đoán	
T50.9	Ngộ độc dược chất, thuốc điều trị và/hoặc sinh phẩm khác và/hoặc không xác định	Tác nhân toan hóa|Tác nhân kiềm hóa|Globulin miễn dịch|Chất miễn dịch|Thuốc tăng cường đốt mỡ|Hormon cận giáp trạng và dẫn xuất
T51	Tác động độc hại của cồn	
T51.0	Tác động độc hại của ethanol	
T51.1	Tác động độc hại của methanol	Cồn methyl
T51.2	Tác động độc hại của 2-Propanol	Cồn isopropanol
T51.3	Tác động độc hại của dầu fusel [loại rượu bậc cao]	
T51.8	Tác động độc hại của cồn khác	
T51.9	Tác động độc hại của cồn, không xác định	
T52	Tác động độc hại của dung môi hữu cơ	
T52.0	Tác động độc hại của sản phẩm dầu hỏa	Xăng dầu [dầu hỏa]|Dầu lửa, dầu hỏa [dầu paraffin]|Sáp paraffin
T52.1	Tác động độc hại của benzene	
T52.2	Tác động độc hại của chất đồng đẳng của benzen	Toluen [methylbenzene]|Xylen [dimethylbenzene]
T52.3	Tác động độc hại của glycol	
T52.4	Tác động độc hại của xeton	
T52.8	Tác động độc hại của dung môi hữu cơ khác	
T52.9	Tác động độc hại của dung môi hữu cơ, không xác định	
T53	Tác động độc hại của dẫn xuất halogen của hydrocarbon béo và/hoặc hydrocarbon thơm	
T53.0	Tác động độc hại của carbon tetrachlorid	Tetrachloromethan
T53.1	Tác động độc hại của chloroform	Trichloromethane
T53.2	Tác động độc hại của trichloroethylene	Trichloroethene
T53.3	Tác động độc hại của tetrachloroethylen	Perchloroethylene|Tetrachloroethene
T53.4	Tác động độc hại của dichloromethan	Methylene chloride
T53.5	Tác động độc hại của chlorofluorocarbon [CFC]	
T53.6	Tác động độc hại của dẫn xuất halogen của hydrocarbon béo	
T53.7	Tác động độc hại của dẫn xuất halogen của hydrocarbon thơm	
T53.9	Tác động độc hại của dẫn xuất halogen của hydrocarbon béo và/hoặc hydrocarbon thơm, không xác định	
T54	Tác động độc hại của chất ăn mòn	
T54.0	Tác động độc hại của phenol và/hoặc các chất đồng đẳng của phenol	
T54.1	Tác động độc hại của hợp chất hữu cơ ăn mòn khác	
T54.2	Tác động độc hại của chất acid ăn mòn và/hoặc chất giống acid	
T54.3	Tác động độc hại của chất kiềm ăn mòn và/hoặc các chất giống kiềm	Kali hydroxid|Natri hydroxid
T54.9	Tác động độc hại của chất ăn mòn, không xác định	
T55	Tác động độc hại của xà phòng và/hoặc chất tẩy rửa	
T56	Tác động độc hại của kim loại	
T56.0	Tác động độc hại của chì và/hoặc các hợp chất của nó	
T56.1	Tác động độc hại của thủy ngân và/hoặc các hợp chất của nó	
T56.2	Tác động độc hại của crôm và/hoặc các hợp chất của nó	
T56.3	Tác động độc hại của cadmi và/hoặc các hợp chất của nó	
T56.4	Tác động độc hại của đồng và/hoặc các hợp chất của nó	
T56.5	Tác động độc hại của kẽm và/hoặc các hợp chất của nó	
T56.6	Tác động độc hại của thiếc và/hoặc các hợp chất của nó	
T56.7	Tác động độc hại của beryllium và/hoặc các hợp chất của nó	
T56.8	Tác động độc hại của kim loại khác	Thallium
T56.9	Tác động độc hại của kim loại, không xác định	
T57	Tác động độc hại của chất vô cơ khác	
T57.0	Tác động độc hại của arsen và/hoặc các hợp chất của nó	
T57.1	Tác động độc hại của phospho và/hoặc các hợp chất của nó	
T57.2	Tác động độc hại của mangan và/hoặc các hợp chất của nó	
T57.3	Tác động độc hại của khí hydro xyanua [HCN]	
T57.8	Tác động độc hại của chất vô cơ xác định khác	
T57.9	Tác động độc hại của chất vô cơ, không xác định	
T58	Tác động độc hại của carbon monoxide	
T59	Tác động độc hại của khí, khói và/hoặc hơi khác	
T59.0	Nhiễm độc nitrogen oxid [NO hoặc NO2]	
T59.1	Nhiễm độc sulfur dioxid	
T59.2	Nhiễm độc formaldehyd	
T59.3	Nhiễm độc khí lacrimogenic	Hơi cay
T59.4	Nhiễm độc khí chlorin	
T59.5	Nhiễm độc khí fluorin và/hoặc hydrogen fluorid	
T59.6	Nhiễm độc hydrogen sulfid	
T59.7	Nhiễm độc carbon dioxid [CO2]	
T59.8	Nhiễm độc khí, khói và/hoặc hơi xác định khác	
T59.9	Nhiễm độc khí, khói và/hoặc hơi, không xác định	
T60	Tác động độc hại của thuốc trừ sâu	
T60.0	Tác động độc hại của thuốc trừ sâu organophosphate và/hoặc carbamate	
T60.1	Tác động độc hại của thuốc trừ sâu có chứa halogen	
T60.2	Tác động độc hại của thuốc trừ sâu khác và/hoặc không xác đinh	
T60.3	Tác động độc hại của thuốc diệt cỏ và/hoặc thuốc diệt nấm	
T60.4	Tác động độc hại của thuốc diệt loài gặm nhấm	
T60.8	Tác động độc hại của thuốc trừ sâu khác	
T60.9	Tác động độc hại của thuốc trừ sâu, không xác định	
T61	Tác động độc hại do ăn hải sản có chất độc	
T61.0	Tác động độc hại của cá Ciguarera	
T61.1	Tác động độc hại của cá Scombroid	Hội chứng giống histamin
T61.2	Tác động độc hại của cá và/hoặc động vật có vỏ khác	
T61.8	Tác động độc hại của hải sản khác	
T61.9	Tác động độc hại của hải sản không xác định	
T62	Tác động độc hại do ăn phải thực phẩm có chất độc	
T62.0	Tác động độc hại do ăn nấm	
T62.1	Tác động độc hại do ăn quả mọng [berry]	
T62.2	Tác động độc hại do ăn (bộ phận của) (các) thực vật khác	
T62.8	Tác động độc hại do ăn thực phẩm có chất độc xác định khác	
T62.9	Tác động độc hại do ăn thực phẩm có chất độc, không xác định	
T63	Tác động độc hại do tiếp xúc với động vật có nọc độc	
T63.0	Tác động độc hại của nọc độc rắn	Nọc độc rắn biển
T63.1	Tác động độc hại của nọc độc loài bò sát khác	Nọc độc thằn lằn
T63.2	Tác động độc hại của nọc độc bọ cạp	
T63.3	Tác động độc hại của nọc độc nhện	
T63.4	Tác động độc hại của nọc độc tiết túc [virus arbo] khác	Vết cắn hoặc vết đốt của côn trùng, có nọc độc
T63.5	Tác động độc hại do tiếp xúc với cá	
T63.6	Tác động độc hại do tiếp xúc với động vật biển khác	
T63.8	Tác động độc hại do tiếp xúc với các động vật có nọc độc khác	Nọc độc của động vật lưỡng cư
T63.9	Tác động độc hại do tiếp xúc với động vật có nọc độc không xác định	
T64	Tác động độc hại của aflatoxin và/hoặc độc tố nấm khác gây ô nhiễm thực phẩm	
T65	Tác động độc hại của chất khác và/hoặc không xác định	
T65.0	Tác động độc hại của xyanua	
T65.1	Tác động độc hại của strychnin và/hoặc muối của strychnin	
T65.2	Tác động độc hại của thuốc lá và/hoặc nicotin	
T65.3	Tác động độc hại của dẫn xuất của nitrogen và/hoặc acid amin của benzen và/hoặc chất đồng đẳng	Anilin [benzenamine]|Nitrobenzen|Trinitrotoluen
T65.4	Tác động độc hại của carbon disulfid	
T65.5	Tác động độc hại của nitroglycerin và/hoặc acid nitric và/hoặc ester khác	1,2,3-Propanetriol trinitrate
T65.6	Tác động độc hại của sơn và/hoặc thuốc nhuộm, không phân loại mục khác	
T65.8	Tác động độc hại của chất xác định khác	
T65.9	Tác động độc hại của chất không xác định	Nhiễm độc không xác định khác
T66	Tác động không xác định của phóng xạ [bức xạ]	
T67	Tác động của nhiệt và/hoặc ánh sáng	
T67.0	Sốc nhiệt và/hoặc say nắng	
T67.1	Ngất do nhiệt	Ngã quỵ do nhiệt
T67.2	Chuột rút do nhiệt	
T67.3	Kiệt sức do nhiệt, thể giảm tiết mồ hôi	
T67.4	Kiệt sức do nhiệt vì mất muối	
T67.5	Kiệt sức do nhiệt, không xác định	Mệt mỏi vì nóng không xác định khác
T67.6	Mệt mỏi do nhiệt, thoáng qua	
T67.7	Phù do nhiệt	
T67.8	Tác động khác của nhiệt và/hoặc ánh sáng	
T67.9	Tác động của nhiệt và/hoặc ánh sáng, không xác định	
T68	Hạ thân nhiệt	
T69	Tác động khác của hạ nhiệt độ	
T69.0	Ngâm nước bàn tay và/hoặc bàn chân	Chứng bợt da chân [do ngâm nước lâu]
T69.1	Bệnh cước	
T69.8	Tác động xác định khác của hạ nhiệt độ	
T69.9	Tác động của hạ nhiệt độ, không xác định	
T70	Tác động của áp suất không khí và/hoặc áp suất nước	
T70.0	Tổn thương tai do chấn thương khí áp	Viêm tai giữa do áp suất khí|Ảnh hưởng đến tai do biến đổi áp suất khí quyển hoặc áp suất nước xung quanh
T70.1	Tổn thương xoang do chấn thương khí áp	Viêm xoang do áp suất khí|Ảnh hưởng đến tai do biến đổi áp suất không khí xung quanh
T70.2	Tác động khác và/hoặc không xác định của độ cao	
T70.3	Bệnh giảm áp [bệnh thợ lặn]	Bệnh khí nén|Liệt nhẹ và liệt của thợ lặn
T70.4	Tác động của chất lỏng áp suất cao	
T70.8	Tác động khác của áp suất không khí và/hoặc áp suất nước	Hội chứng chấn thương do vụ nổ
T70.9	Tác động áp lực khí và/hoặc áp lực nước, không xác định	
T71	Ngạt thở do chấn thương	Ngạt thở
T73	Tác động của thiếu hụt khác	
T73.0	Tác động của đói	Thiếu thực phẩm
T73.1	Tác động của khát	Thiếu nước
T73.2	Kiệt sức do tiếp xúc với điều kiện thời tiết ngoài trời khắc nghiệt hoặc kéo dài	
T73.3	Kiệt sức do cố gắng quá sức	Gắng sức quá mức
T73.8	Tác động khác của thiếu hụt	
T73.9	Tác động của thiếu hút, không xác định	
T74	Hội chứng ngược đãi	
T74.0	Thờ ơ hoặc bỏ rơi	
T74.1	Bạo hành thể xác	
T74.2	Lạm dụng tình dục	
T74.3	Thao túng tâm lý	
T74.8	Hội chứng ngược đãi khác	Thể hỗn hợp
T74.9	Hội chứng ngược đãi, không xác định	
T75	Tác động của nguyên nhân bên ngoài khác	
T75.0	Tác động của sét/chớp	Choáng do sét/chớp|Bị sét đánh không xác định khác
T75.1	Ngạt nước và/hoặc bị chìm không tử vong	Nhấn chìm|Chuột rút ở người bơi
T75.2	Tác động của rung động	Hội chứng búa khí nén|Hội chứng co thắt mạch do chấn thương|Chóng mặt do sóng hạ âm
T75.3	Chứng say tàu xe [say do chuyển động]	Say tàu bay|Say sóng|Say xe
T75.4	Tác động của dòng điện	Điện giật|Sốc do dòng điện
T75.8	Tác động xác định khác của nguyên nhân bên ngoài khác	
T76	Tác động không xác định của nguyên nhân bên ngoài	
T78	Tác dụng bất lợi, không phân loại mục khác	
T78.0	Sốc phản vệ do phản ứng dị ứng đối với thực phẩm	
T78.1	Phản ứng có hại khác với thực phẩm, không phân loại mục khác	
T78.2	Sốc phản vệ, không xác định	
T78.3	Phù mạch	
T78.4	Dị ứng, không xác định	
T78.8	Tác dụng bất lợi khác, không phân loại mục khác	
T78.9	Tác dụng bất lợi, không xác định	
T79	Một số biến chứng ban đầu của chấn thương, không phân loại mục khác	
T79.0	Thuyên tắc khí (do chấn thương)	
T79.1	Thuyên tắc mỡ (do chấn thương)	
T79.2	Xuất huyết thứ phát và/hoặc tái phát do chấn thương	
T79.3	Nhiễm trùng vết thương sau chấn thương, không phân loại mục khác	
T79.4	Sốc do chấn thương	
T79.5	Vô niệu do chấn thương	Hội chứng do dập nát|Suy thận sau chấn thương dập nát
T79.6	Thiếu máu cơ do chấn thương	
T79.7	Tràn khí dưới da do chấn thương	
T79.8	Biến chứng sớm khác của chấn thương	
T79.9	Biến chứng sớm không xác định của chấn thương	
T80	Biến chứng sau tiêm truyền, truyền máu và/hoặc tiêm thuốc điều trị	
T80.0	Thuyên tắc khí sau tiêm truyền, truyền máu và/hoặc tiêm thuốc	
T80.1	Biến chứng mạch máu sau tiêm truyền, truyền máu và/hoặc tiêm thuốc	
T80.2	Nhiễm trùng sau tiêm truyền, truyền máu và/hoặc tiêm thuốc điều trị	
T80.3	Phản ứng ABO không tương thích	Truyền máu không tương thích|Phản ứng với sự tương kỵ nhóm máu khi tiêm truyền hoặc truyền máu
T80.4	Phản ứng Rh không tương thích	Phản ứng do yếu tố Rh trong tiêm truyền hoặc truyền máu
T80.5	Sốc phản vệ do huyết thanh	
T80.6	Phản ứng huyết thanh khác	
T80.8	Biến chứng khác sau tiêm truyền, truyền máu và/hoặc tiêm thuốc điều trị	
T80.9	Biến chứng không xác định sau tiêm truyền, truyền máu và/hoặc tiêm điều trị	Phản ứng truyền máu không xác định khác
T81	Biến chứng can thiệp, không phân loại mục khác	
T81.0	Biến chứng chảy máu và/hoặc tụ máu do can thiệp, không phân loại mục khác	
T81.1	Sốc kéo dài hoặc hậu quả do can thiệp, không phân loại mục khác	
T81.2	Vô tình thủng và/hoặc rách trong khi can thiệp, không phân loại mục khác	
T81.3	Toác vết mổ, không phân loại mục khác	
T81.4	Nhiễm trùng sau can thiệp, không phân loại mục khác	
T81.5	Dị vật vô tình bị để lại trong khoang cơ thể hoặc vết mổ sau can thiệp	
T81.6	Phản ứng cấp tính với dị vật vô tình bị để lại trong khi thực hiện can thiệp	
T81.7	Biến chứng mạch máu sau can thiệp, không phân loại mục khác	
T81.8	Biến chứng khác do can thiệp, không phân loại mục khác	
T81.9	Biến chứng không xác định của can thiệp	
T82	Biến chứng của thiết bị nhân tạo tim và/hoặc mạch máu, cấy và/hoặc ghép	
T82.0	Biến chứng cơ học của van tim nhân tạo	
T82.1	Biến chứng cơ học của thiết bị điện tử tại tim	
T82.2	Biến chứng cơ học của bắc cầu động mạch vành và/hoặc ghép van	Bệnh lý liệt kê tại T82.0 do mảnh ghép bắc cầu động mạch vành và/hoặc ghép van
T82.3	Biến chứng cơ học của ghép mạch máu khác	
T82.4	Biến chứng cơ học của catheter thẩm tách mạch máu	
T82.5	Biến chứng cơ học của thiết bị/dụng cụ và/hoặc vật tư cấy ghép khác ở tim và/hoặc mạch máu	
T82.6	Nhiễm trùng và/hoặc phản ứng viêm do van tim nhân tạo	
T82.7	Nhiễm trùng và/hoặc phản ứng viêm do các thiết bị/dụng cụ ở tim và/hoặc mạch máu khác, cấy và/hoặc ghép	
T82.8	Biến chứng xác định khác của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu	Thuyên tắc mạch do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu|Xơ hóa do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu mạch|Chảy máu do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu|Đau do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu|Hẹp do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu|Huyết khối do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/howajc mạch máu
T82.9	Biến chứng không xác định của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở tim và/hoặc mạch máu	
T83	Biến chứng của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép ở hệ sinh dục - tiết niệu	
T83.0	Biến chứng cơ học của ống thông tiểu (đặt ống thông tại chỗ)	
T83.1	Biến chứng cơ học của các thiết bị tiết niệu và/hoặc các cấy khác	
T83.2	Biến chứng cơ học của ghép cơ quan tiết niệu	Bệnh lý liệt kê tại T82.0 do ghép cơ quan tiết niệu
T83.3	Biến chứng cơ học của dụng cụ tránh thai trong tử cung	Bệnh lý liệt kê tại T82.0 do dụng cụ tử cung tránh thai
T83.4	Biến chứng cơ học của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép khác trong đường sinh dục	
T83.5	Nhiễm trùng và/hoặc phản ứng viêm do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép hệ tiết niệu	
T83.6	Nhiễm trùng và/hoặc phản ứng viêm do thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép đường sinh dục	
T83.8	Biến chứng khác của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép hệ sinh dục - tiết niệu	
T83.9	Biến chứng không xác định của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép hệ sinh dục - tiết niệu	
T84	Biến chứng do thiết bị/dụng cụ, cấy và/hoặc ghép chỉnh hình bên trong	
T84.0	Biến chứng cơ học của khớp nhân tạo bên trong	Bệnh lý liệt kê tại T82.0 do khớp nhân tạo
T84.1	Biến chứng cơ học của thiết bị/dụng cụ cố định bên trong xương của chi	Bệnh lý liệt kê tại T82.0 do thiết bị/dụng cụ cố định bên trong xương của chi
T84.2	Biến chứng cơ học của thiết bị/dụng cụ cố định bên trong của xương khác	Bệnh lý liệt kê tại T82.0 do thiết bị/dụng cụ cố định bên trong của các xương khác
T84.3	Biến chứng cơ học của thiết bị/dụng cụ và/hoặc mô xương, cấy và/hoặc ghép	
T84.4	Biến chứng cơ học của thiết bị/dụng cụ chỉnh hình khác, cấy và/hoặc ghép	Bệnh lý liệt kê tại tại T82.0 do ghép cơ và/hoặc gân
T84.5	Nhiễm trùng và/hoặc phản ứng viêm do khớp nhân tạo bên trong	
T84.6	Nhiễm trùng và/hoặc phản ứng viêm do thiết bị/dụng cụ cố định bên trong [bất kỳ vị trí nào]	
T84.7	Nhiễm trùng và/hoặc phản ứng viêm do thiết bị/dụng cụ nhân tạo chỉnh hình, cấy và/hoặc ghép bên trong	
T84.8	Biến chứng khác của thiết bị nhân tạo chỉnh hình, cấy và/hoặc ghép bên trong	
T84.9	Biến chứng không xác định của thiết bị/dụng cụ nhân tạo chỉnh hình, cấy và/hoặc ghép bên trong	
T85	Biến chứng của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép bên trong khác	
T85.0	Biến chứng cơ học của dẫn lưu (thông) não thất nội sọ	
T85.1	Biến chứng cơ học của máy kích thích điện tử được cấy ghép ở hệ thần kinh	
T85.2	Biến chứng cơ học của thủy tinh thể nhân tạo nội nhãn	Bệnh lý liệt kê tại T82.0 do thủy tinh thể nội nhãn
T85.3	Biến chứng cơ học của thiết bị/dụng cụ mắt nhân tạo khác, cấy và/hoặc ghép	
T85.4	Biến chứng cơ học của ngực giả và/hoặc túi độn ngực cấy ghép	Bệnh lý liệt kê tại T82.0 do vú nhân tạo và/hoặc cấy
T85.5	Biến chứng cơ học của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép dạ dày - ruột	
T85.6	Biến chứng cơ học của thiết bị/dụng cụ nhân tạo xác định khác, cấy và/hoặc ghép bên trong	
T85.7	Nhiễm trùng và/hoặc phản ứng viêm do thiết bị/dụng cụ nhân tạo khác, cấy và/hoặc ghép bên trong	
T85.8	Biến chứng khác của thiết bị/dụng cụ nhân tạo, cấy và/hoặc ghép bên trong, không phân loại mục khác	
T85.9	Biến chứng không xác định của thiết bị/dụng cụ nhân tạo cấy và/hoặc ghép bên trong	Biến chứng của thiết bị nhân tạo, cấy và/hoặc ghép bên trong không xác định khác
T86	Thất bại và/hoặc thải ghép tạng và/hoặc mô	
T86.0	Thải ghép tủy xương	Phản ứng hoặc bệnh mảnh ghép chống lại ký chủ
T86.1	Thất bại và/hoặc thải ghép thận	
T86.2	Thất bại và/hoặc thải ghép tim	
T86.3	Thất bại và/hoặc thải ghép tim - phổi	
T86.4	Thất bại và/hoặc thải ghép gan	
T86.8	Thất bại và/hoặc thải khác của ghép tạng và/hoặc mô	
T86.9	Thất bại và/hoặc thải ghép trong cấy ghép nội tạng và/hoặc mô	
T87	Biến chứng đặc trưng của phẫu thuật nối lại và/hoặc cắt cụt	
T87.0	Biến chứng của (một phần của) chi trên được nối lại	
T87.1	Biến chứng của (một phần của) chi dưới được nối lại	
T87.2	Biến chứng của phần cơ thể được nối lại khác	
T87.3	U thần kinh của mỏm cụt	
T87.4	Nhiễm trùng mỏm cụt	
T87.5	Hoại tử mỏm cụt	
T87.6	Biến chứng khác và/hoặc không xác định của mỏm cụt	
T88	Biến chứng khác của chăm sóc ngoại khoa và/hoặc nội khoa không phân loại mục khác	
T88.0	Nhiễm trùng sau tiêm chủng	Nhiễm trùng hệ thống sau tiêm chủng
T88.1	Biến chứng khác sau tiêm chủng, không phân loại mục khác	
T88.2	Sốc do gây mê	
T88.3	Tăng thân nhiệt ác tính do gây mê	
T88.4	Thất bại hoặc khó đặt nội khí quản	
T88.5	Biến chứng khác của gây mê	Hạ thân nhiệt sau khi gây mê
T88.6	Sốc phản vệ do tác dụng bất lợi của dược chất hoặc thuốc điều trị thích hợp sử dụng đúng quy cách	
T88.7	Tác dụng bất lợi không xác định của dược chất hoặc thuốc điều trị	
T88.8	Biến chứng xác định khác của chăm sóc ngoại khoa và/hoặc nội khoa, không phân loại mục khác	
T88.9	Biến chứng ngoại khoa và/hoặc nội khoa, không xác định	
T90	Di chứng tổn thương ở đầu	
T90.0	Di chứng tổn thương nông ở đầu	Di chứng tổn thương phân loại vào S00.
T90.1	Di chứng vết thương hở ở đầu	Di chứng tổn thương phân loại vào S01.
T90.2	Di chứng vỡ xương sọ và/hoặc gãy xương mặt	Di chứng tổn thương phân loại vào S02.
T90.3	Di chứng tổn thương dây thần kinh sọ não	Di chứng tổn thương phân loại vào S04.
T90.4	Di chứng tổn thương ở mắt và/hoặc hốc mắt	Di chứng tổn thương phân loại vào S05.
T90.5	Di chứng tổn thương nội sọ	Di chứng tổn thương phân loại vào S06.
T90.8	Di chứng tổn thương xác định khác ở đầu	Di chứng tổn thương phân loại vào S03.-, S07.- - S08.- và S09.0-S09.8
T90.9	Di chứng tổn thương không xác định ở đầu	Di chứng tổn thương phân loại vào S09.9
T91	Di chứng tổn thương cổ và/hoặc thân	
T91.0	Di chứng tổn thương nông và/hoặc vết thương hở tại cổ và/hoặc thân	
T91.1	Di chứng gãy xương cột sống	Di chứng tổn thương phân loại vào S12.-, S22.0-S22.1, S32.0, S32.7 và T08
T91.2	Di chứng gãy xương ngực và/hoặc vùng chậu khác	Di chứng tổn thương phân loại vào S22.2-S22.9, S32.1-S32.5 và S32.8
T91.3	Di chứng tổn thương tủy sống	Di chứng tổn thương phân loại vào S14.0-S14.1, S24.0-S24.1, S34.0-S34.1 và T09.3
T91.4	Di chứng tổn thương nội tạng trong khoang ngực	Di chứng tổn thương phân loại vào S26.- - S27.
T91.5	Di chứng tổn thương nội tạng trong bụng và/hoặc vùng chậu	Di chứng tổn thương phân loại vào S36.- - S37.
T91.8	Di chứng tổn thương xác định khác tại cổ và thân	
T91.9	Di chứng tổn thương không xác định ở cổ và/hoặc thân	Di chứng tổn thương phân loại vào S19.9, S29.9, S39.9 và T09.9
T92	Di chứng tổn thương chi trên	
T92.0	Di chứng vết thương hở chi trên	Di chứng tổn thương phân loại vào S41.-, S51.-, S61.- và T11.1
T92.1	Di chứng gãy xương cánh tay	Di chứng tổn thương phân loại vào S42.-, S52.- và T10
T92.2	Di chứng gãy xương tầm cổ tay và/hoặc bàn tay	Di chứng tổn thương phân loại vào S62.
T92.3	Di chứng trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ chi trên	Di chứng tổn thương phân loại vào S43.-, S53.-, S63.- và T11.2
T92.4	Di chứng tổn thương dây thần kinh chi trên	Di chứng tổn thương phân loại vào S44.-, S54.-, S64.- và T11.3
T92.5	Di chứng tổn thương cơ và gân chi trên	Di chứng tổn thương phân loại vào S46.-, S56.-, S66.- và T11.5
T92.6	Di chứng tổn thương dập nát và/hoặc đứt rời chi trên do chấn thương	Di chứng tổn thương phân loại vào S47.- - S48.-, S57.- - S58.-, S67.- - S68.- và T11.6
T92.8	Di chứng tổn thương xác định khác của chi trên	
T92.9	Di chứng tổn thương không xác định của chi trên	Di chứng tổn thương phân loại vào S49.9, S59.9, S69.9 và T11.9
T93	Di chứng tổn thương chi dưới	
T93.0	Di chứng vết thương hở chi dưới	Di chứng tổn thương phân loại vào S71.-, S81.-, S91.- và T13.1
T93.1	Di chứng gãy xương đùi	Di chứng tổn thương phân loại vào S72.
T93.2	Di chứng gãy xương khác của chi dưới	Di chứng tổn thương phân loại từ S82.-, S92.- và T12
T93.3	Di chứng trật khớp, giãn dây chằng [bong gân] và/hoặc căng cơ chi dưới	Di chứng tổn thương phân loại vào S73.-, S83.-, S93.- và T13.2
T93.4	Di chứng tổn thương dây thần kinh chi dưới	Di chứng tổn thương phân loại vào S74.-, S84.-, S94.- và T13.3
T93.5	Di chứng tổn thương cơ và/hoặc gân chi dưới	Di chứng tổn thương phân loại vào S76.-, S86.-, S96.- và T13.5
T93.6	Di chứng tổn thương dập nát và/hoặc đứt rời chi dưới do chấn thương	Di chứng tổn thương phân loại vào S77.- - S78.-, S87.- - S88.-, S97.- - S98.- và T13.6
T93.8	Di chứng tổn thương xác định khác của chi dưới	
T93.9	Di chứng tổn thương không xác định của chi dưới	Di chứng tổn thương phân loại vào S79.9, S89.9, S99.9 và T13.9
T94	Di chứng tổn thương tác động đến nhiều vùng cơ thể và/hoặc vùng không xác định	
T94.0	Di chứng tổn thương tác động đến nhiều vùng cơ thể	Di chứng tổn thương phân loại vào T00.- - T07
T94.1	Di chứng tổn thương, không xác định vùng cơ thể	Di chứng tổn thương phân loại vào T14.
T95	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh	
T95.0	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh ở đầu và/hoặc cổ	Di chứng tổn thương phân loại vào T20.-, T33.0-T33.1, T34.0-T34.1 và T35.2
T95.1	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh ở thân	Di chứng tổn thương phân loại vào T21.-, T33.2-T33.3, T34.2-T34.3 và T35.3
T95.2	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh ở chi trên	Di chứng tổn thương phân loại vào T22.- - T23.-, T33.4-T33.5, T34.4-T34.5 và T35.4
T95.3	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh ở chi dưới	Di chứng tổn thương phân loại vào T24.- - T25.-, T33.6-T33.8, T34.6-T34.8 và T35.5
T95.4	Di chứng bỏng, ăn mòn phân loại chỉ theo phạm vi bề mặt cơ thể tổn thương	Di chứng tổn thương phân loại vào T31.- -T32.
T95.8	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh xác định khác	Di chứng tổn thương phân loại vào T26.- - T29.-, T35.0-T35.1 và T35.6
T95.9	Di chứng bỏng, ăn mòn và/hoặc bỏng lạnh không xác định	Di chứng tổn thương phân loại vào T30.-, T33.9, T34.9 và T35.7
T96	Di chứng ngộ độc dược chất, thuốc điều trị và/hoặc sinh phẩm	
T97	Di chứng của tác động độc hại do chất chủ yếu có nguồn gốc không phải là thuốc	
T98	Di chứng do tác động từ nguyên nhân bên ngoài khác và/hoặc không xác định	
T98.0	Di chứng do tác động của dị vật xâm nhập qua lỗ tự nhiên	Di chứng tác động phân loại vào T15.- - T19.
T98.1	Di chứng do tác động từ nguyên nhân bên ngoài khác và/hoặc không xác định	Di chứng tác động phân loại vào T66- T78.
T98.2	Di chứng của một số biến chứng sớm của chấn thương	Di chứng biến chứng phân loại vào T79.
T98.3	Di chứng của biến chứng ngoại khoa và/hoặc nội khoa, không phân loại mục khác	Di chứng biến chứng phân loại vào T80.- - T88.
U04	Hội chứng suy hô hấp cấp tính nặng [SARS]	
U04.9	Hội chứng suy hô hấp cấp tính nặng [SARS], không xác định	
U07	Sử dụng mã U07 trong tình huống khẩn cấp	
U07.0	Rối loạn liên quan sử dụng thuốc lá điện tử	
U07.1	COVID-19, virus được xác định	COVID-19, vi rút được xác định
U07.2	COVID-19, virus chưa được xác định	COVID-19, vi rút không được xác định
U08	Tiền sử cá nhân mắc COVID-19	
U08.9	Tiền sử cá nhân mắc COVID-19, không xác định	
U09	Tình trạng bệnh lý hậu COVID-19	
U09.9	Tình trạng bệnh lý hậu COVID-19, không xác định	
U10	Hội chứng viêm đa cơ quan liên quan đến COVID-19	
U10.9	Hội chứng viêm đa cơ quan liên quan đến COVID-19, không xác định	
U11	Nhu cầu tiêm phòng COVID-19	
U11.9	Nhu cầu tiêm phòng COVID-19, không xác định	
U12	Vắc xin COVID-19 gây tác dụng bất lợi trong điều trị	
U12.9	Vắc xin COVID-19 gây tác dụng bất lợi trong điều trị, không xác định	
U13	Sử dụng mã U13 trong tình huống khẩn cấp	
U13./9	Sử dụng mã U13.9 trong tình huống khẩn cấp	
U14	Sử dụng mã U14 trong tình huống khẩn cấp	
U14.9	Sử dụng mã U14.9 trong tình huống khẩn cấp	
U15	Sử dụng mã U15 trong tình huống khẩn cấp	
U15.9	Sử dụng mã U15.9 trong tình huống khẩn cấp	
U16	Sử dụng mã U16 trong tình huống khẩn cấp	
U16.9	Sử dụng mã U16.9 trong tình huống khẩn cấp	
U17	Sử dụng mã U17 trong tình huống khẩn cấp	
U17.9	Sử dụng mã U17.9 trong tình huống khẩn cấp	
U18	Sử dụng mã U18 trong tình huống khẩn cấp	
U18.9	Sử dụng mã U18.9 trong tình huống khẩn cấp	
U19	Sử dụng mã U19 trong tình huống khẩn cấp	
U19.9	Sử dụng mã U19.9 trong tình huống khẩn cấp	
U20	Sử dụng mã U20 trong tình huống khẩn cấp	
U20.9	Sử dụng mã U20.9 trong tình huống khẩn cấp	
U21	Sử dụng mã U21 trong tình huống khẩn cấp	
U21.9	Sử dụng mã U21.9 trong tình huống khẩn cấp	
U22	Sử dụng mã U22 trong tình huống khẩn cấp	
U22.9	Sử dụng mã U22.9 trong tình huống khẩn cấp	
U23	Sử dụng mã U23 trong tình huống khẩn cấp	
U23.9	Sử dụng mã U23.9 trong tình huống khẩn cấp	
U24	Sử dụng mã U24 trong tình huống khẩn cấp	
U24.9	Sử dụng mã U24.9 trong tình huống khẩn cấp	
U25	Sử dụng mã U25 trong tình huống khẩn cấp	
U25.9	Sử dụng mã U25.9 trong tình huống khẩn cấp	
U26	Sử dụng mã U26 trong tình huống khẩn cấp	
U26.9	Sử dụng mã U26.9 trong tình huống khẩn cấp	
U27	Sử dụng mã U27 trong tình huống khẩn cấp	
U27.9	Sử dụng mã U27.9 trong tình huống khẩn cấp	
U28	Sử dụng mã U28 trong tình huống khẩn cấp	
U28.9	Sử dụng mã U28.9 trong tình huống khẩn cấp	
U29	Sử dụng mã U29 trong tình huống khẩn cấp	
U29.9	Sử dụng mã U29.9 trong tình huống khẩn cấp	
U30	Sử dụng mã U30 trong tình huống khẩn cấp	
U30.9	Sử dụng mã U30.9 trong tình huống khẩn cấp	
U31	Sử dụng mã U31 trong tình huống khẩn cấp	
U31.9	Sử dụng mã U31.9 trong tình huống khẩn cấp	
U32	Sử dụng mã U32 trong tình huống khẩn cấp	
U32.9	Sử dụng mã U32.9 trong tình huống khẩn cấp	
U33	Sử dụng mã U33 trong tình huống khẩn cấp	
U33.9	Sử dụng mã U33.9 trong tình huống khẩn cấp	
U34	Sử dụng mã U34 trong tình huống khẩn cấp	
U34.9	Sử dụng mã U34.9 trong tình huống khẩn cấp	
U35	Sử dụng mã U35 trong tình huống khẩn cấp	
U35.9	Sử dụng mã U35.9 trong tình huống khẩn cấp	
U36	Sử dụng mã U36 trong tình huống khẩn cấp	
U36.9	Sử dụng mã U36.9 trong tình huống khẩn cấp	
U37	Sử dụng mã U37 trong tình huống khẩn cấp	
U37.9	Sử dụng mã U37.9 trong tình huống khẩn cấp	
U38	Sử dụng mã U38 trong tình huống khẩn cấp	
U38.9	Sử dụng mã U38.9 trong tình huống khẩn cấp	
U39	Sử dụng mã U39 trong tình huống khẩn cấp	
U39.9	Sử dụng mã U39.9 trong tình huống khẩn cấp	
U40	Sử dụng mã U40 trong tình huống khẩn cấp	
U40.9	Sử dụng mã U40.9 trong tình huống khẩn cấp	
U41	Sử dụng mã U41 trong tình huống khẩn cấp	
U41.9	Sử dụng mã U41.9 trong tình huống khẩn cấp	
U42	Sử dụng mã U42 trong tình huống khẩn cấp	
U42.9	Sử dụng mã U42.9 trong tình huống khẩn cấp	
U43	Sử dụng mã U43 trong tình huống khẩn cấp	
U43.9	Sử dụng mã U43.9 trong tình huống khẩn cấp	
U44	Sử dụng mã U44 trong tình huống khẩn cấp	
U44.9	Sử dụng mã U44.9 trong tình huống khẩn cấp	
U45	Sử dụng mã U45 trong tình huống khẩn cấp	
U45.9	Sử dụng mã U45.9 trong tình huống khẩn cấp	
U46	Sử dụng mã U46 trong tình huống khẩn cấp	
U46.9	Sử dụng mã U46.9 trong tình huống khẩn cấp	
U47	Sử dụng mã U47 trong tình huống khẩn cấp	
U47.9	Sử dụng mã U47.9 trong tình huống khẩn cấp	
U48	Sử dụng mã U48 trong tình huống khẩn cấp	
U48.9	Sử dụng mã U48.9 trong tình huống khẩn cấp	
U49	Sử dụng mã U49 trong tình huống khẩn cấp	Sử dụng mã U49 trong trường hợp khẩn cấp
U49.9	Sử dụng mã U49.9 trong tình huống khẩn cấp	
U82	Kháng kháng sinh họ betalactam	
U82.0	Kháng penicillin	
U82.1	Kháng methicillin	
U82.2	Kháng betalactamase phổ mở rộng (ESBL)	
U82.8	Kháng kháng sinh khác thuộc họ betalactam	
U82.9	Kháng kháng sinh họ betalactam, không xác định	
U83	Kháng kháng sinh khác	
U83.0	Kháng vancomycin	
U83.1	Kháng kháng sinh khác thuộc họ vancomycin khác	
U83.2	Kháng nhóm quinolon	
U83.7	Kháng đa kháng sinh	
U83.8	Kháng một loại kháng sinh xác định khác	
U83.9	Kháng kháng sinh không xác định	Kháng kháng sinh không xác định khác
U84	Kháng thuốc kháng vi sinh vật khác	
U84.0	Kháng thuốc diệt ký sinh trùng	Kháng ký ninh và/hoặc các hợp chất liên quan.
U84.1	Kháng thuốc chống nấm	
U84.2	Kháng thuốc diệt virus	
U84.3	Kháng thuốc chống lao	
U84.7	Đa kháng thuốc kháng vi sinh vật	
U84.8	Kháng thuốc kháng vi sinh vật xác định khác	
U84.9	Kháng thuốc kháng vi sinh vật không xác định	Kháng thuốc không xác định khác
U85	Kháng thuốc chống ung thư	
V01	Người đi bộ bị thương do va chạm với xe đạp	
V01.0	Người đi bộ bị thương do va chạm với xe đạp, tai nạn không do giao thông	
V01.1	Người đi bộ bị thương do va chạm với xe đạp, tai nạn giao thông	
V01.9	Người đi bộ bị thương do va chạm với xe đạp, không xác định tai nạn giao thông hay không do giao thông	
V02	Người đi bộ bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V02.0	Người đi bộ bị thương do va chạm với xe máy 2 hoặc 3 bánh, tai nạn không do giao thông	
V02.1	Người đi bộ bị thương do va chạm với xe máy 2 hoặc 3 bánh, tai nạn giao thông	
V02.9	Người đi bộ bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định tai nạn giao thông hay không do giao thông	
V03	Người đi bộ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V03.0	Người đi bộ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, tai nạn không do giao thông	
V03.1	Người đi bộ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, tai nạn giao thông	
V03.9	Người đi bộ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định tai nạn giao thông hay không do giao thông	
V04	Người đi bộ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V04.0	Người đi bộ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, tai nạn không do giao thông	
V04.1	Người đi bộ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, tai nạn giao thông	
V04.9	Người đi bộ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định tai nạn giao thông hay không do giao thông	
V05	Người đi bộ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V05.0	Người đi bộ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, tai nạn không do giao thông	
V05.1	Người đi bộ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, tai nạn giao thông	
V05.9	Người đi bộ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định tai nạn giao thông hay không do giao thông	
V06	Người đi bộ bị thương do va chạm với xe không có động cơ khác	
V06.0	Người đi bộ bị thương do va chạm với xe không có động cơ khác, tai nạn không do giao thông	
V06.1	Người đi bộ bị thương do va chạm với xe không có động cơ khác, tai nạn giao thông	
V06.9	Người đi bộ bị thương do va chạm với xe không có động cơ khác, không xác định tai nạn giao thông hay không do giao thông	
V09	Người đi bộ bị thương trong tai nạn giao thông khác và/hoặc không xác định	
V09.0	Người đi bộ bị thương trong tai nạn không do giao thông nhưng có liên quan đến xe cơ giới khác và/hoặc không xác định	
V09.1	Người đi bộ bị thương trong tai nạn không do giao thông không xác định	
V09.2	Người đi bộ bị thương trong tai nạn giao thông có liên quan đến xe cơ giới khác và/hoặc không xác định	
V09.3	Người đi bộ bị thương trong tai nạn giao thông đường bộ không xác định	
V09.9	Người đi bộ bị thương trong tai nạn giao thông không xác định	
V10	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật	
V10.0	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V10.1	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V10.2	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V10.3	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V10.4	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V10.5	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V10.9	Người đi xe đạp bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V11	Người đi xe đạp bị thương do va chạm với xe đạp khác	
V11.0	Người đi xe đạp bị thương do va chạm với xe đạp khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V11.1	Người đi xe đạp bị thương do va chạm với xe đạp khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V11.2	Người đi xe đạp bị thương do va chạm với xe đạp khác, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V11.3	Người đi xe đạp bị thương do va chạm với xe đạp khác, người bị thương khi lên xe hoặc xuống xe	
V11.4	Người đi xe đạp bị thương do va chạm với xe đạp khác, người điều khiển xe bị thương trong tai nạn giao thông	
V11.5	Người đi xe đạp bị thương do va chạm với xe đạp khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V11.9	Người đi xe đạp bị thương do va chạm với xe đạp khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V12	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V12.0	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V12.1	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V12.2	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V12.3	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V12.4	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V12.5	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V12.9	Người đi xe đạp bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V13	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V13.0	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V13.1	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V13.2	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V13.3	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V13.4	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V13.5	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V13.9	Người đi xe đạp bị thương khi va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V14	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt	
V14.0	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V14.1	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V14.2	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V14.3	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V14.4	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V14.5	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V14.9	Người đi xe đạp bị thương khi va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V15	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt	
V15.0	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V15.1	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V15.2	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V15.3	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V15.4	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V15.5	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V15.9	Người đi xe đạp bị thương khi va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V16	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác	
V16.0	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V16.1	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V16.2	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V16.3	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V16.4	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V16.5	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V16.9	Người đi xe đạp bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V17	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật	
V17.0	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V17.1	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V17.2	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V17.3	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V17.4	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V17.5	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V17.9	Người đi xe đạp bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V18	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm	
V18.0	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V18.1	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V18.2	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V18.3	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V18.4	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V18.5	Người đi xe đạp bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V18.9	Người đi xe đạp bị thương trong tai nạn giao thông không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V19	Người đi xe đạp bị thương do những tai nạn giao thông vận tải khác và/hoặc không xác định	
V19.0	Người điều khiển xe đạp bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V19.1	Người ngồi trên xe bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V19.2	Người đi xe đạp (không xác định là điều khiển hoặc ngồi trên xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Xe đạp va chạm không xác định khác, không do giao thông
V19.3	Người đi xe đạp [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn không do giao thông	Tai nạn xe đạp không xác định khác, không do giao thông|Người đi xe đạp bị thương trong tai nạn không do giao thông không xác định khác
V19.4	Người điều khiển xe đạp bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V19.5	Người ngồi trên xe đạp bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V19.6	Người đi xe đạp vai trò không xác định bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V19.8	Người đi xe đạp [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào một bộ phận của xe đạp
V19.9	Người đi xe đạp [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn xe đạp không xác định khác
V20	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc va chạm với động vật
V20.0	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V20.1	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V20.2	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V20.3	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V20.4	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V20.5	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V20.9	Người đi xe máy 2 bánh bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V21	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp	
V21.0	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V21.1	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V21.2	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V21.3	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V21.4	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V21.5	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V21.9	Người đi xe máy 2 bánh bị thương khi va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V22	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V22.0	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V22.1	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V22.2	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V22.3	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V22.4	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V22.5	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V22.9	Người đi xe máy 2 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V23	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V23.0	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V23.1	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V23.2	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V23.3	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V23.4	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V23.5	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V23.9	Người đi xe máy 2 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V24	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V24.0	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V24.1	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V24.2	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V24.3	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V24.4	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V24.5	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V24.9	Người đi xe máy 2 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V25	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V25.0	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V25.1	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V25.2	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V25.3	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V25.4	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V25.5	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V25.9	Người đi xe máy 2 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người đi xe máy bị thương trong tai nạn giao thông	
V26	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác	
V26.0	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V26.1	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V26.2	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V26.3	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V26.4	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V26.5	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V26.9	Người đi xe máy 2 bánh bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V27	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật	
V27.0	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V27.1	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V27.2	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V27.3	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V27.4	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V27.5	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V27.9	Người đi xe máy 2 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V28	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm	
V28.0	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V28.1	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V28.2	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn không do giao thông	
V28.3	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V28.4	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V28.5	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V28.9	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V29	Người đi xe máy 2 bánh bị thương trong tai nạn giao thông khác và/hoặc không xác định	
V29.0	Người điều khiển xe máy 2 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V29.1	Người ngồi trên xe máy 2 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V29.2	Người đi xe máy 2 bánh (không xác định là điều khiển hoặc ngồi trên xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm xe máy không xác định khác, không do giao thông
V29.3	Người đi xe máy [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn không xác định không do giao thông	Tai nạn xe máy không xác định khác, không do giao thông|Người lái xe máy bị thương do tai nạn không do giao thông không xác định khác
V29.4	Người điều khiển xe máy 2 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V29.5	Người ngồi trên xe bị thương khi va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V29.6	Người đi xe máy 2 bánh không xác định bị thương khi va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	Va chạm xe máy không xác định khác, do giao thông
V29.8	Người đi xe máy [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào một bộ phận của xe máy
V29.9	Người đi xe máy [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn xe máy không xác định khác
V30	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật	
V30.0	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V30.1	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V30.2	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V30.3	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V30.4	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V30.5	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V30.6	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V30.7	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V30.9	Người đi xe máy 3 bánh bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V31	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp	
V31.0	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V31.1	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V31.2	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V31.3	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V31.4	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V31.5	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V31.6	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V31.7	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn giao thông	
V31.9	Người đi xe máy 3 bánh bị thương do va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V32	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 bánh hoặc xe máy 3 bánh
V32.0	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V32.1	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V32.2	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V32.3	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V32.4	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V32.5	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V32.6	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V32.7	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn giao thông	
V32.9	Người đi xe máy 3 bánh bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V33	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V33.0	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V33.1	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V33.2	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V33.3	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V33.4	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V33.5	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V33.6	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V33.7	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn giao thông	
V33.9	Người đi xe máy 3 bánh bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V34	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V34.0	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V34.1	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V34.2	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V34.3	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V34.4	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V34.5	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V34.6	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V34.7	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn giao thông	
V34.9	Người đi xe máy 3 bánh bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V35	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V35.0	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V35.1	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V35.2	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V35.3	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V35.4	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V35.5	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V35.6	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V35.7	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn giao thông	
V35.9	Người đi xe máy 3 bánh bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V36	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác	
V36.0	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V36.1	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V36.2	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V36.3	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V36.4	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người bị thương khi lên xe hoặc xuống xe	
V36.5	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người điều khiển xe bị thương trong tai nạn giao thông	
V36.6	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V36.7	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, người bên ngoài xe bị thương trong tai nạn giao thông	
V36.9	Người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V37	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật	
V37.0	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V37.1	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V37.2	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V37.3	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V37.4	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V37.5	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V37.6	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V37.7	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V37.9	Người đi xe máy 3 bánh bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V38	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm	
V38.0	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V38.1	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V38.2	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V38.3	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được người đi xe máy ba bánh bị thương trong tai nạn không do giao thông	
V38.4	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V38.5	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V38.6	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V38.7	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn giao thông	
V38.9	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V39	Người đi xe máy 3 bánh bị thương trong tai nạn giao thông vận tải khác và/hoặc không xác định	
V39.0	Người điều khiển xe máy 3 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V39.1	Người ngồi trên xe máy 3 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V39.2	Người đi xe máy 3 bánh (không xác định là điều khiển, ngồi trên, bám ngoài xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm với xe máy 3 bánh không xác định khác, không do giao thông
V39.3	Người đi xe máy 3 bánh [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn không do giao thông	Tai nạn không xác định khác của xe máy 3 bánh, không do giao thông|Người trên xe máy 3 bánh bị thương trong tai nạn không do giao thông, không xác định khác
V39.4	Người điều khiển xe máy 3 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V39.5	Người ngồi trên xe máy 3 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V39.6	Không xác định được người đi xe máy 3 bánh bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V39.8	Người đi xe máy 3 bánh [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào cửa hoặc bộ phận khác của xe máy 3 bánh
V39.9	Người đi xe máy 3 bánh [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn không xác định khác của xe máy 3 bánh
V40	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật	
V40.0	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V40.1	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V40.2	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V40.3	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V40.4	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V40.5	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V40.6	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V40.7	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V40.9	Người đi xe ô tô bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V41	Người đi xe ô tô bị thương do va chạm với xe đạp	
V41.0	Người đi xe ô tô bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V41.1	Người đi xe ô tô bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V41.2	Người đi xe ô tô bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V41.3	Người đi xe ô tô bị thương do va chạm với xe đạp, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V41.4	Người đi xe ô tô bị thương do va chạm với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V41.5	Người đi xe ô tô bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V41.6	Người đi xe ô tô bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V41.7	Người đi xe ô tô bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn giao thông	
V41.9	Người đi xe ô tô bị thương do va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V42	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V42.0	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V42.1	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V42.2	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V42.3	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V42.4	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V42.5	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V42.6	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V42.7	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn giao thông	
V42.9	Người đi xe ô tô bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V43	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ	
V43.0	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V43.1	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V43.2	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V43.3	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V43.4	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V43.5	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V43.6	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V43.7	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn giao thông	
V43.9	Người đi xe ô tô bị thương do va chạm với xe ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V44	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V44.0	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V44.1	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V44.2	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V44.3	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V44.4	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V44.5	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V44.6	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V44.7	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn giao thông	
V44.9	Người đi xe ô tô bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V45	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V45.0	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V45.1	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V45.2	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V45.3	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V45.4	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V45.5	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V45.6	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V45.7	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn giao thông	
V45.9	Người đi xe ô tô bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V46	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác	
V46.0	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V46.1	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V46.2	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V46.3	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V46.4	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V46.5	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V46.6	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V46.7	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn giao thông	
V46.9	Người đi xe ô tô bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V47	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật	
V47.0	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V47.1	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V47.2	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V47.3	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V47.4	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V47.5	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V47.6	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V47.7	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V47.9	Người đi xe ô tô bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V48	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm	
V48.0	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V48.1	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V48.2	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V48.3	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được người đi xe ô tô bị thương trong tai nạn không do giao thông	
V48.4	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V48.5	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V48.6	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V48.7	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn giao thông	
V48.9	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V49	Người đi xe ô tô bị thương trong tai nạn giao thông vận tải khác và/hoặc không xác định	
V49.0	Người điều khiển xe ô tô bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V49.1	Người ngồi trên xe ô tô bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V49.2	Người đi xe ô tô (không xác định là điều khiển, ngồi trên, bám ngoài xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm ô tô không xác định khác, không do giao thông
V49.3	Người đi xe [người điều khiển hoặc người ngồi trên xe] ô tô bị thương trong tai nạn không do giao thông	Va chạm ô tô không xác định khác, không do giao thông
V49.4	Người điều khiển xe ô tô bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V49.5	Người ngồi trên xe ô tô bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V49.6	Không xác định được người đi xe ô tô bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V49.8	Người đi xe ô tô [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào cửa hoặc bộ phận khác của xe ô tô
V49.9	Người đi xe ô tô [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn ô tô không xác định khác
V50	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật	
V50.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V50.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V50.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V50.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V50.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V50.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V50.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V50.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V50.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V51	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp	
V51.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V51.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V51.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V51.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V51.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V51.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V51.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V51.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn giao thông	
V51.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V52	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V52.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V52.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V52.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V52.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V52.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V52.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V52.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V52.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn giao thông	
V52.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V53	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V53.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V53.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V53.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V53.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V53.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V53.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V53.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V53.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn giao thông	
V53.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V54	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V54.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V54.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V54.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V54.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V54.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V54.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V54.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V54.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn giao thông	
V54.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V55	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V55.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V55.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V55.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V55.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V55.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V55.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V55.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V55.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn giao thông	
V55.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V56	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác	
V56.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V56.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V56.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V56.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V56.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V56.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V56.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V56.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn giao thông	
V56.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V57	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật	
V57.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V57.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V57.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V57.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V57.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V57.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V57.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V57.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V57.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V58	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm	
V58.0	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V58.1	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V58.2	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V58.3	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn không do giao thông	
V58.4	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V58.5	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V58.6	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V58.7	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn giao thông	
V58.9	Người đi xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V59	Người đi xe bán tải hoặc xe tải nhỏ bị thương tai nạn giao thông vận tải khác và/hoặc không xác định	Người đi xe bán tải hoặc xe tải van bị thương tai nạn giao thông vận tải khác và/hoặc không xác định
V59.0	Người điều khiển xe bán tải hoặc xe tải van bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V59.1	Người ngồi trên xe bán tải hoặc xe tải van bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V59.2	Người đi xe bán tải (không xác định là điều khiển, ngồi trên, bám ngoài xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm không xác định khác liên quan đến xe bán tải hoặc xe tải, không do giao thông
V59.3	Người đi xe bán tải hoặc xe tải nhỏ [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn không xác định và không do giao thông	Tai nạn không xác định khác liên quan đến xe bán tải hoặc xe tải, không do giao thông
V59.4	Người điều khiển xe bán tải hoặc xe tải van bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V59.5	Người ngồi trên xe bán tải hoặc xe tải van bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V59.6	Không xác định được người đi xe bán tải hoặc xe tải nhỏ bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V59.8	Người đi [người điều khiển hoặc người ngồi trên] xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông xác định khác	Bị mắc vào cửa hoặc bộ phận khác của xe bán tải hoặc xe tải
V59.9	Người đi [người điều khiển hoặc người ngồi trên] xe bán tải hoặc xe tải nhỏ bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn không xác định khác liên quan đến xe bán tải hoặc xe tải
V60	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật	
V60.0	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V60.1	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V60.2	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V60.3	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V60.4	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V60.5	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V60.6	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V60.7	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V60.9	Người đi xe tải hạng nặng bị thương do va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V61	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp	
V61.0	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V61.1	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V61.2	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V61.3	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V61.4	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V61.5	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V61.6	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V61.7	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, người bên ngoài xe bị thương trong tai nạn giao thông	
V61.9	Người đi xe tải hạng nặng bị thương do va chạm với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V62	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V62.0	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V62.1	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V62.2	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V62.3	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V62.4	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V62.5	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V62.6	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V62.7	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn giao thông	
V62.9	Người đi xe tải hạng nặng bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V63	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V63.0	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V63.1	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V63.2	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V63.3	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V63.4	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V63.5	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V63.6	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V63.7	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn giao thông	
V63.9	Người đi xe tải hạng nặng bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V64	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V64.0	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V64.1	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V64.2	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V64.3	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V64.4	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V64.5	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V64.6	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V64.7	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn giao thông	
V64.9	Người đi xe tải hạng nặng bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V65	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V65.0	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V65.1	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V65.2	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V65.3	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V65.4	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V65.5	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V65.6	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V65.7	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn giao thông	
V65.9	Người đi xe tải hạng nặng bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V66	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác	
V66.0	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V66.1	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V66.2	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V66.3	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V66.4	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V66.5	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V66.6	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V66.7	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn giao thông	
V66.9	Người đi xe tải hạng nặng bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V67	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật	
V67.0	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V67.1	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V67.2	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V67.3	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V67.4	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V67.5	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V67.6	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V67.7	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V67.9	Người đi xe tải hạng nặng bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V68	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm	
V68.0	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V68.1	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V68.2	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V68.3	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được người đi xe tải hạng nặng bị thương trong tai nạn không do giao thông	
V68.4	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V68.5	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V68.6	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V68.7	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn giao thông	
V68.9	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V69	Người đi xe tải hạng nặng bị thương trong tai nạn giao thông vận tải khác và/hoặc không xác định	
V69.0	Người điều khiển xe tải hạng nặng bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V69.1	Người ngồi trên xe tải hạng nặng bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V69.2	Người đi xe tải hạng nặng (không xác định là điều khiển, ngồi trên, bám ngoài) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm không xác định khác liên quan đến xe tải nặng, không do giao thông
V69.3	Người đi xe tải hạng nặng [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn không do giao thông	Tai nạn không xác định khác liên quan đến xe tải nặng, không do giao thông|Người ngồi trên xe tải nặng bị thương do tai nạn không do giao thông không xác định khác
V69.4	Người điều khiển xe tải hạng nặng bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V69.5	Người ngồi trên xe tải hạng nặng bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V69.6	Không xác định được người đi xe tải hạng nặng bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V69.8	Người đi xe tải hạng nặng [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào cửa hoặc bộ phận khác của xe tải nặng
V69.9	Người đi xe tải hạng nặng [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn không xác định khác liên quan đến xe tải nặng
V70	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật	
V70.0	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V70.1	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V70.2	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V70.3	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V70.4	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người bị thương khi lên xe hoặc xuống xe	
V70.5	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người điều khiển xe bị thương trong tai nạn giao thông	
V70.6	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V70.7	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V70.9	Người đi xe buýt bị thương khi va chạm với người đi bộ hoặc động vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V71	Người đi xe buýt bị thương do va chạm với với xe đạp	
V71.0	Người đi xe buýt bị thương do va chạm với với xe đạp, người điều khiển xe bị thương trong tai nạn không do giao thông	
V71.1	Người đi xe buýt bị thương do va chạm với với xe đạp, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V71.2	Người đi xe buýt bị thương do va chạm với với xe đạp, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V71.3	Người đi xe buýt bị thương do va chạm với với xe đạp, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V71.4	Người đi xe buýt bị thương do va chạm với với xe đạp, người bị thương khi lên xe hoặc xuống xe	
V71.5	Người đi xe buýt bị thương do va chạm với với xe đạp, người điều khiển xe bị thương trong tai nạn giao thông	
V71.6	Người đi xe buýt bị thương do va chạm với với xe đạp, người ngồi trên xe bị thương trong tai nạn giao thông	
V71.7	Người đi xe buýt bị thương do va chạm với với xe đạp, người bên ngoài xe bị thương trong tai nạn giao thông	
V71.9	Người đi xe buýt bị thương do va chạm với với xe đạp, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V72	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V72.0	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn không do giao thông	
V72.1	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V72.2	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V72.3	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V72.4	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bị thương khi lên xe hoặc xuống xe	
V72.5	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người điều khiển xe bị thương trong tai nạn giao thông	
V72.6	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người ngồi trên xe bị thương trong tai nạn giao thông	
V72.7	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, người bên ngoài xe bị thương trong tai nạn giao thông	
V72.9	Người đi xe buýt bị thương do va chạm với xe máy 2 hoặc 3 bánh, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V73	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ	
V73.0	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn không do giao thông	
V73.1	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V73.2	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V73.3	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V73.4	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bị thương khi lên xe hoặc xuống xe	
V73.5	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người điều khiển xe bị thương trong tai nạn giao thông	
V73.6	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người ngồi trên xe bị thương trong tai nạn giao thông	
V73.7	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, người bên ngoài xe bị thương trong tai nạn giao thông	
V73.9	Người đi xe buýt bị thương do va chạm với ô tô, xe bán tải hoặc xe tải nhỏ, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V74	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt	
V74.0	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V74.1	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V74.2	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V74.3	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V74.4	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bị thương khi lên xe hoặc xuống xe	
V74.5	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người điều khiển xe bị thương trong tai nạn giao thông	
V74.6	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người ngồi trên xe bị thương trong tai nạn giao thông	
V74.7	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, người bên ngoài xe bị thương trong tai nạn giao thông	
V74.9	Người đi xe buýt bị thương do va chạm với xe tải hạng nặng hoặc xe buýt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V75	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V75.0	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn không do giao thông	
V75.1	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V75.2	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V75.3	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V75.4	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bị thương khi lên xe hoặc xuống xe	
V75.5	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người điều khiển xe bị thương trong tai nạn giao thông	
V75.6	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người ngồi trên xe bị thương trong tai nạn giao thông	
V75.7	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, người bên ngoài xe bị thương trong tai nạn giao thông	
V75.9	Người đi xe buýt bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V76	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác	
V76.0	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn không do giao thông	
V76.1	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V76.2	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V76.3	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V76.4	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người bị thương khi lên xe hoặc xuống xe	
V76.5	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người điều khiển xe bị thương trong tai nạn giao thông	
V76.6	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người ngồi trên xe bị thương trong tai nạn giao thông	
V76.7	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, người bên ngoài xe bị thương trong tai nạn giao thông	
V76.9	Người đi xe buýt bị thương do va chạm với xe không có động cơ khác, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V77	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật	
V77.0	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn không do giao thông	
V77.1	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V77.2	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V77.3	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V77.4	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người bị thương khi lên xe hoặc xuống xe	
V77.5	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người điều khiển xe bị thương trong tai nạn giao thông	
V77.6	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người ngồi trên xe bị thương trong tai nạn giao thông	
V77.7	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, người bên ngoài xe bị thương trong tai nạn giao thông	
V77.9	Người đi xe buýt bị thương do va chạm với vật cố định hoặc tĩnh vật, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V78	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm	
V78.0	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn không do giao thông	
V78.1	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn không do giao thông	
V78.2	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn không do giao thông	
V78.3	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được người đi xe buýt bị thương trong tai nạn không do giao thông	
V78.4	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người bị thương khi lên xe hoặc xuống xe	
V78.5	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người điều khiển xe bị thương trong tai nạn giao thông	
V78.6	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người ngồi trên xe bị thương trong tai nạn giao thông	
V78.7	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, người bên ngoài xe bị thương trong tai nạn giao thông	
V78.9	Người đi xe buýt bị thương trong tai nạn giao thông vận tải không có va chạm, không xác định được vai trò của người bị thương trong tai nạn giao thông	
V79	Người đi xe buýt bị thương trong tai nạn giao thông vận tải khác và/hoặc không xác định	
V79.0	Người điều khiển xe buýt bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V79.1	Người ngồi trên xe buýt bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	
V79.2	Người đi xe buýt (không xác định là điều khiển, ngồi trên, bám ngoài xe) bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn không do giao thông	Va chạm xe buýt không xác định khác, không do giao thông
V79.3	Người đi xe buýt [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn không do giao thông	Tai nạn xe buýt không xác định khác, không do giao thông|Người ngồi trên xe buýt bị thương trong tai nạn không do giao thông không xác định khác
V79.4	Người điều khiển xe buýt bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V79.5	Người ngồi trên xe buýt bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V79.6	Không xác định được người đi xe buýt bị thương do va chạm với xe cơ giới khác và/hoặc không xác định trong tai nạn giao thông	
V79.8	Người đi xe buýt [người điều khiển hoặc người ngồi trên xe] bị thương trong tai nạn giao thông xác định khác	Bị mắc vào cửa hoặc một bộ phận của xe buýt
V79.9	Người đi xe buýt [người điều khiển xe hoặc người ngồi trên xe] bị thương trong tai nạn giao thông đường bộ không xác định	Tai nạn xe buýt không xác định khác
V80	Người cưỡi động vật hoặc người trên xe động vật kéo bị thương trong tai nạn giao thông	
V80.0	Người điều khiển hoặc người ngồi trên bị thương do ngã từ hoặc bị ném từ động vật hoặc xe động vật kéo trong tai nạn không có va chạm	
V80.1	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương do va chạm với người đi bộ hoặc động vật	
V80.2	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương do va chạm với xe đạp	
V80.3	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương do va chạm với xe máy 2 hoặc 3 bánh	
V80.4	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương do va chạm với ô tô, xe bán tải, xe tải nhỏ, xe tải hạng nặng hoặc xe buýt	
V80.5	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo trên bị thương do va chạm với xe cơ giới xác định khác	
V80.6	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo trên bị thương do va chạm với tàu hỏa hoặc phương tiện đường sắt	
V80.7	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương do va chạm với xe không có động cơ khác	
V80.8	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương trong va chạm với vật cố định hoặc tĩnh vật	
V80.9	Người điều khiển động vật hoặc người ngồi trên xe do động vật kéo bị thương trong tai nạn giao thông vận tải khác và/hoặc không xác định	Tai nạn xe động vật kéo không xác định khác|Tai nạn cưỡi động vật không xác định khác
V81	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương trong tai nạn giao thông vận tải	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương trong tai nạn giao thông
V81.0	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do va chạm với xe cơ giới trong tai nạn không do giao thông	
V81.1	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do va chạm với xe cơ giới trong tai nạn giao thông	
V81.2	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do va chạm với hoặc va quệt với đầu máy toa xe	
V81.3	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do va chạm với vật thể khác	Va chạm đường sắt không xác định khác
V81.4	Người bị thương khi lên hoặc xuống tàu hỏa hoặc phương tiện đường sắt	
V81.5	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do ngã trên tàu hỏa hoặc phương tiện đường sắt	
V81.6	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do ngã từ tàu hỏa hoặc phương tiện đường sắt	
V81.7	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương do trật bánh mà không có va chạm trước	
V81.8	Người đi tàu hỏa hoặc phương tiện đường sắt bị thương trong tai nạn đường sắt xác định khác	
V81.9	Người có mặt trên tàu hỏa hoặc phương tiện đường sắt bị thương trong tai nạn đường sắt không xác định	Tai nạn đường sắt không xác định khác
V82	Người đi xe điện bị thương trong tai nạn giao thông vận tải	Người đi xe điện bị thương trong tai nạn giao thông
V82.0	Người đi xe điện bị thương do va chạm với xe cơ giới trong tai nạn không do giao thông	
V82.1	Người đi xe điện bị thương do va chạm với xe cơ giới trong tai nạn giao thông	
V82.2	Người đi xe điện bị thương do va chạm hoặc va quệt với toa xe	
V82.3	Người đi xe điện bị thương do va chạm với vật thể khác	
V82.4	Người đi xe điện bị thương khi lên hoặc xuống xe	
V82.5	Người đi xe điện bị thương do ngã trên xe điện	
V82.6	Người đi xe điện bị thương do ngã từ xe điện	
V82.7	Người đi xe điện bị thương do trật bánh mà không xảy ra va chạm trước	
V82.8	Người đi xe điện bị thương trong tai nạn giao thông vận tải xác định khác	Va chạm với tàu hỏa hoặc xe không có động cơ khác
V82.9	Người đi xe điện bị thương trong tai nạn giao thông không xác định	Tai nạn xe điện không xác định khác
V83	Người đi xe chuyên dùng chủ yếu sử dụng trong các cơ sở công nghiệp bị thương trong tai nạn giao thông	
V83.0	Người điều khiển xe chuyên dùng trong công nghiệp bị thương trong tai nạn giao thông	
V83.1	Người ngồi trên xe chuyên dùng cho công nghiệp bị thương trong tai nạn giao thông	
V83.2	Người ở bên ngoài xe chuyên dùng cho công nghiệp bị thương trong tai nạn giao thông	
V83.3	Không xác định được người đi xe chuyên dùng cho công nghiệp bị thương trong tai nạn giao thông	
V83.4	Người bị thương khi lên xuống từ xe chuyên dùng cho công nghiệp	
V83.5	Người điều khiển xe chuyên dùng trong công nghiệp bị thương trong tai nạn không do giao thông	
V83.6	Người ngồi trên xe chuyên dùng cho công nghiệp bị thương trong tai nạn không do giao thông	
V83.7	Người ở ngoài xe chuyên dùng cho công nghiệp bị thương trong tai nạn không do giao thông	
V83.9	Không xác định được người ngồi trên xe chuyên dụng cho công nghiệp bị thương trong tai nạn không do giao thông	Tai nạn xe chuyên dùng trong công nghiệp không xác định khác
V84	Người đi xe chuyên dùng sử dụng chủ yếu trong nông nghiệp bị thương trong tai nạn giao thông vận tải	Người đi xe chuyên dùng sử dụng chủ yếu trong nông nghiệp bị thương trong tai nạn giao thông
V84.0	Người điều khiển xe chuyên dùng trong nông nghiệp bị thương trong tai nạn giao thông	
V84.1	Người ngồi trên xe chuyên dùng trong nông nghiệp bị thương trong tai nạn giao thông	
V84.2	Người ở bên ngoài xe chuyên dùng trong nông nghiệp bị thương trong tai nạn giao thông	
V84.3	Không xác định được người đi xe chuyên dùng trong nông nghiệp bị thương trong tai nạn giao thông	
V84.4	Người bị thương khi lên xuống xe chuyên dùng trong nông nghiệp	
V84.5	Người điều khiển xe chuyên dùng trong nông nghiệp bị thương trong tai nạn không do giao thông	
V84.6	Người ngồi trên xe chuyên dùng trong nông nghiệp bị thương trong tai nạn không do giao thông	
V84.7	Người ở bên ngoài xe chuyên dùng trong nông nghiệp bị thương trong tai nạn không do giao thông	
V84.9	Không xác định được người đi xe chuyên dùng trong nông nghiệp bị thương trong tai nạn không do giao thông	Tai nạn xe chuyên dùng trong nông nghiệp không xác định khác
V85	Người đi xe chuyên dùng trong xây dựng bị thương trong tai nạn giao thông	
V85.0	Người điều khiển xe chuyên dùng trong xây dựng bị thương trong tai nạn giao thông	
V85.1	Người ngồi trên xe chuyên dùng trong xây dựng bị thương trong tai nạn giao thông	
V85.2	Người ở bên ngoài xe chuyên dùng trong xây dựng bị thương trong tai nạn giao thông	
V85.3	Không xác định được người trên xe chuyên dùng trong xây dựng bị thương trong tai nạn giao thông	
V85.4	Người bị thương khi lên hoặc xuống xe chuyên dùng trong xây dựng	
V85.5	Người điều khiển xe chuyên dùng trong xây dựng bị thương trong tai nạn không do giao thông	
V85.6	Người ngồi trên xe chuyên dùng trong xây dựng bị thương trong tai nạn không do giao thông	
V85.7	Người ở bên ngoài xe chuyên dùng trong xây dựng bị thương trong tai nạn không do giao thông	
V85.9	Không xác định được người đi xe chuyên dụng trong xây dựng bị thương trong tai nạn không do giao thông	Tai nạn xe chuyên dùng trong xây dựng không xác định khác
V86	Người đi xe chuyên dùng mọi địa hình hoặc xe máy khác được thiết kế chủ yếu cho sử dụng ngoài quốc lộ, bị thương trong tai nạn giao thông	Người đi xe chuyên dùng mọi địa hình hoặc xe mô tô khác được thiết kế chủ yếu cho sử dụng ngoài quốc lộ, bị thương trong tai nạn giao thông
V86.0	Người điều khiển xe địa hình hoặc xe việt dã khác bị thương trong tai nạn giao thông	
V86.1	Người ngồi trên xe địa hình hoặc xe việt dã bị thương trong tai nạn giao thông	
V86.2	Người ở bên ngoài xe địa hình hoặc xe việt dã bị thương trong tai nạn giao thông	
V86.3	Không xác định người đi xe địa hình hoặc xe việt dã bị thương trong tai nạn giao thông	
V86.4	Người bị thương trong khi lên hoặc xuống xe địa hình hoặc xe việt dã	
V86.5	Người điều khiển xe địa hình hoặc xe việt dã bị thương trong tai nạn không do giao thông	
V86.6	Người ngồi trên xe địa hình hoặc xe việt dã bị thương trong tai nạn không do giao thông	
V86.7	Người ở bên ngoài xe địa hình hoặc xe việt dã bị thương trong tai nạn không do giao thông	
V86.9	Không xác định người đi xe địa hình hoặc xe việt dã bị thương tích trong tai nạn nhưng không tham gia giao thông, không xác định	Tai nạn xe cơ giới mọi địa hình không xác định khác|Tai nạn xe cơ giới ngoài quốc lộ không xác định khác
V87	Tai nạn giao thông xác định nhưng không rõ phương tiện vận tải của nạn nhân	
V87.0	Người bị thương do va chạm giữa ô tô và xe máy 2 hoặc 3 bánh (do tham gia giao thông)	
V87.1	Người bị thương do va chạm giữa xe cơ giới khác và xe máy 2 hoặc 3 bánh (do tham gia giao thông)	
V87.2	Người bị thương do va chạm giữa ô tô và xe bán tải hoặc xe tải nhỏ (do tham gia giao thông)	
V87.3	Người bị thương do va chạm giữa ô tô và xe buýt (do tham gia giao thông)	
V87.4	Người bị thương do va chạm giữa ô tô và xe tải hạng nặng (do tham gia giao thông)	
V87.5	Người bị thương trong va chạm giữa xe tải nặng và xe buýt (do tham gia giao thông)	
V87.6	Người bị thương do va chạm giữa tàu hỏa và/hoặc phương tiện đường sắt và ô tô (do tham gia giao thông)	
V87.7	Người bị thương do va chạm các xe cơ giới chuyên dùng khác (do tham gia giao thông)	
V87.8	Người bị thương trong tai nạn giao thông vận tải không có va chạm xác định khác liên quan đến xe cơ giới (do tham gia giao thông)	
V87.9	Người bị thương trong tai nạn giao thông vận tải xác định khác (có va chạm) (không va chạm) liên quan đến xe không có động cơ (do tham gia giao thông)	
V88	Tai nạn xác định không do giao thông nhưng không rõ phương tiện vận tải của nạn nhân	Tai nạn xác định không liên quan đến giao thông nhưng không rõ phương tiện vận tải của nạn nhân
V88.0	Người bị thương do va chạm giữa ô tô và xe máy 2 hoặc 3 bánh, không do giao thông	
V88.1	Người bị thương do va chạm giữa xe cơ giới khác và xe máy 2 hoặc 3 bánh, không do giao thông	
V88.2	Người bị thương do va chạm giữa ô tô và xe bán tải hoặc xe tải nhỏ, không do giao thông	
V88.3	Người bị thương do va chạm giữa ô tô và xe buýt, không do giao thông	
V88.4	Người bị thương do va chạm giữa ô tô và xe tải hạng nặng, không do giao thông	
V88.5	Người bị thương trong va chạm giữa xe tải nặng và buýt, không do giao thông	
V88.6	Người bị thương trong va chạm giữa tầu hỏa hoặc phương tiện đi trên ray với ô tô, không do giao thông	
V88.7	Người bị thương trong va chạm giữa các xe có động cơ biết rõ đặc điểm khác, không do giao thông	
V88.8	Người bị thương trong tai nạn giao thông biết rõ đặc điểm khác, không có va chạm, liên quan đến xe động cơ, không do giao thông	
V88.9	Người bị thương tích trong tai nạn giao thông cụ thể khác rõ đặc điểm (có va chạm) (không có va chạm) liên quan đến xe không động cơ, không tham gia giao thông	
V89	Tai nạn xe cơ giới hoặc xe không có động cơ, không xác định loại phương tiện	
V89.0	Người bị thương trong tai nạn xe cơ giới không xác định, không do giao thông	Tai nạn xe cơ giới không xác định khác, không do giao thông
V89.1	Người bị thương trong tai nạn xe không có động cơ không xác định, không do giao thông	
V89.2	Người bị thương trong tai nạn xe cơ giới không xác định, do tham gia giao thông	Tai nạn xe có động cơ [MVA] không xác định khác|Tai nạn giao thông trên quốc lộ [RTA] không xác định khác
V89.3	Người bị thương trong tai nạn xe không có động cơ không xác định, do tham gia giao thông	Tai nạn giao thông xe không có động cơ không xác định khác
V89.9	Người bị thương trong tai nạn phương tiện giao thông không xác định	Va chạm không xác định khác
V90	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu	Tai nạn tàu thủy gây đuối nước và/hoặc chìm tàu
V90.0	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, tàu chở hàng	
V90.1	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, tàu chở khách	
V90.2	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, tàu đánh cá	
V90.3	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy khác có thủy lực	
V90.4	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, thuyền buồm	
V90.5	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, ca nô hoặc thuyền kayak	
V90.6	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, thuyền thủ công bơm hơi (không có thủy lực)	
V90.7	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, ván trượt nước	
V90.8	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy khác không có thủy lực	
V90.9	Tai nạn phương tiện đường thủy gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy không xác định	
V91	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác	
V91.0	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, tàu chở hàng	
V91.1	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, tàu chở khách	
V91.2	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, tàu đánh cá	
V91.3	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, phương tiện đường thủy khác có thủy lực	
V91.4	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, thuyền buồm	
V91.5	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, ca nô hoặc thuyền kayak	
V91.6	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, thuyền thủ công bơm hơi (không có thủy lực)	
V91.7	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, ván trượt nước	
V91.8	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, phương tiện đường thủy khác không có thủy lực	
V91.9	Tai nạn phương tiện giao thông đường thủy gây chấn thương khác, phương tiện đường thủy không xác định	
V92	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra	
V92.0	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, tàu chở hàng	
V92.1	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, tàu chở khách	
V92.2	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, tàu đánh cá	
V92.3	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, phương tiện đường thủy khác có thủy lực	
V92.4	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, thuyền buồm	
V92.5	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, ca nô hoặc thuyền kayak	
V92.6	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, thuyền thủ công bơm hơi (không có thủy lực)	
V92.7	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, ván trượt nước	
V92.8	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, phương tiện đường thủy khác không có thủy lực	
V92.9	Đuối nước và/hoặc ngạt nước liên quan tới giao thông đường thủy không có tai nạn do phương tiện giao thông đường thủy gây ra, phương tiện đường thủy không xác định	
V93	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu	
V93.0	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, tàu chở hàng	
V93.1	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, tàu chở khách	
V93.2	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, tàu đánh cá	
V93.3	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy khác có thủy lực	
V93.4	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, thuyền buồm	
V93.5	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, ca nô hoặc thuyền kayak	
V93.6	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, thuyền thủ công bơm hơi (không có thủy lực)	
V93.7	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, ván trượt nước	
V93.8	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy khác không có thủy lực	
V93.9	Tai nạn trên phương tiện thủy không có tai nạn của phương tịên, không gây đuối nước và/hoặc chìm tàu, phương tiện đường thủy không xác định	
V94	Tai nạn giao thông đường thủy khác và/hoặc không xác định	
V94.0	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, tàu chở hàng	
V94.1	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, tàu chở khách	
V94.2	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, tàu đánh cá	
V94.3	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, phương tiện đường thủy khác có thủy lực	
V94.4	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, thuyền buồm	
V94.5	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, ca nô hoặc thuyền kayak	
V94.6	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, thuyền thủ công bơm hơi (không có thủy lực)	
V94.7	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, ván trượt nước	
V94.8	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, phương tiện đường thủy khác không có thủy lực	
V94.9	Các tai nạn giao thông đường thủy khác và/hoặc không xác định, phương tiện đường thủy không xác định	
V95	Tai nạn do phương tiện bay có động cơ gây tổn thương cho người sử dụng phương tiện	
V95.0	Tai nạn máy bay trực thăng gây tổn thương cho người đi máy bay	
V95.1	Tai nạn tàu lượn nhẹ, siêu nhẹ hoặc tàu lượn có động cơ gây thương tích cho người tham gia	
V95.2	Tai nạn của máy bay cánh bằng khác của tư nhân, gây thương tích cho người đi máy bay	
V95.3	Tai nạn của máy bay cánh bằng thương mại, gây thương tích cho người đi máy bay	
V95.4	Tai nạn tàu vũ trụ gây thương tích cho người trên tàu	
V95.8	Tai nạn máy bay khác gây thương tích cho người trên máy bay	
V95.9	Người đi máy bay bị thương trong tai nạn máy bay không xác định	Tai nạn máy bay không xác định khác|Tai nạn vận tải hàng không không xác định khác
V96	Tai nạn của phương tiện bay không động cơ làm bị thương người trên tàu	
V96.0	Tai nạn khinh khí cầu tổn thương người trên khinh khí cầu	
V96.1	Tai nạn tàu lượn làm người ngồi trên tàu bị thương	
V96.2	Tai nạn tàu lượn (không động cơ) làm người ngồi trên tàu bị thương	
V96.8	Các tai nạn phương tiện bay không động cơ khác, làm người ngồi trên tàu bị thương	Diều chở người
V96.9	Người tham gia sử dụng tàu bay không người lái hoặc các phương tiện bay siêu nhẹ bị thương tích không xác định	Tai nạn phương tiện bay không động cơ không xác định khác
V97	Các tai nạn do phương tịên bay xác định khác	
V97.0	Người ở trên phương tiện bay bị thương trong tai nạn vận tải hàng không xác định khác	
V97.1	Người bị thương khi lên xuống máy bay	
V97.2	Người nhảy dù bị thương trong tai nạn vận tải hàng không	
V97.3	Người trên mặt đất bị thương trong tai nạn vận tải hàng không	Bị va đập do vật rơi từ máy bay|Bị thương do cánh quạt quay|Bị hút, cuốn vào máy bay
V97.8	Các tai nạn vận tải hàng không khác, không phân loại mục khác	
V98	Các tai nạn giao thông xác định khác	
V99	Tai nạn giao thông không xác định	
W00	Ngã trên cùng một mặt bằng liên quan băng và/hoặc tuyết	
W01	Ngã trên cùng mặt bằng do trượt, vấp và/hoặc lộn nhào	
W02	Ngã liên quan trượt băng, ski, trượt patin hoặc ván trượt	
W03	Ngã khác trên cùng mặt bằng do va chạm hoặc bị người khác xô đẩy	
W04	Ngã khi đang được người khác bế hoặc cõng	
W05	Ngã liên quan đến xe lăn	
W06	Ngã liên quan đến giường	
W07	Ngã liên quan đến ghế	
W08	Ngã liên quan đến đồ nội thất khác	
W09	Ngã liên quan đến thiết bị sân chơi	
W10	Ngã trên và ngã từ bậc thang và/hoặc bậc thềm	
W11	Ngã trên và/hoặc ngã từ thang	
W12	Ngã trên và/hoặc ngã từ giàn giáo	
W13	Ngã từ, ngã ra ngoài hoặc ngã xuyên qua tòa nhà hoặc công trình xây dựng	
W14	Ngã từ cây	
W15	Rơi từ vách đã	
W16	Lặn hoặc nhảy xuống nước gây thương tích không phải là chết đuối hoặc chìm trong nước	
W17	Ngã khác từ cấp độ này sang cấp độ khác	
W18	Ngã khác trên cùng mặt bằng	
W19	Ngã không xác định	
W20	Bị tác động của (các) vật ném, tung hoặc đang rơi	
W21	Va phải hoặc bị đập bởi dụng cụ thể thao	
W22	Va phải hoặc bị đập bởi các vật thể khác	
W23	Bị bắt, kẹp, ép hoặc bị chèn ép trong hoặc giữa các vật	
W24	Tiếp xúc với (các) thiết bị nâng và/hoặc truyền tải, không phân loại mục khác	
W25	Tiếp xúc với kính sắc nhọn	
W26	Tiếp xúc với (các) vật thể sắc nhọn khác	
W26.0	Tiếp xúc với dao, kiếm hoặc dao găm	
W26.8	Tiếp xúc với (các) vật thể sắc nhọn khác, không phân loại mục khác	Cạnh giấy cứng|Nắp lon thiếc
W26.9	Tiếp xúc với (các) vật thể sắc nhọn không xác định	
W27	Tiếp xúc với dụng cụ cầm tay không có động cơ	
W28	Tiếp xúc với máy cắt cỏ có động cơ	
W29	Tiếp xúc với các dụng cụ cầm tay và máy móc gia dụng có động cơ khác	
W30	Va chạm với máy móc nông nghiệp	
W31	Va chạm với máy móc khác và/hoặc máy móc không xác định	
W32	Xả súng ngắn	
W33	Xả súng trường, súng bắn đạn ghém và/hoặc súng lớn hơn	
W34	Xả súng khác và/hoặc không xác định	
W35	Nổ và/hoặc vỡ lò hơi	
W36	Nổ và/hoặc vỡ bình ga	
W37	Nổ và/hoặc vỡ lốp, đường ống hoặc ống có áp suất	
W38	Nổ và vỡ các thiết bị điều áp xác định khác	
W39	Nổ pháo hoa	
W40	Nổ các vật liệu khác	
W41	Tiếp xúc với tia áp lực cao	
W42	Phơi nhiễm tiếng ồn	
W43	Phơi nhiễm với chấn động	
W44	Dị vật xâm nhập vào hoặc qua mắt hoặc lỗ tự nhiên	
W45	Dị vật xâm nhập qua da	
W46	Tiếp xúc với kim tiêm dưới da	
W49	Bị ảnh hưởng của lực cơ học bất động khác và/hoặc không xác định	
W50	Bị người khác đập đánh, đá, vặn, cắn, cào	
W51	Bị người khác tấn công chống lại hoặc đâm vào	
W52	Bị đám đông hoặc đám người dẫm lên, ép hoặc xô đẩy	
W53	Bị chuột cắn	
W54	Bị chó cắn hoặc tấn công	
W55	Bị cắn hoặc tấn công do loài động vật có vú khác	
W56	Tiếp xúc với động vật biển	
W57	Bị côn trùng không có nọc và/hoặc các loài tiết túc [virus arbo] không có nọc độc cắn hoặc đốt	
W58	Bị cá sấu châu Phi hoặc cá sấu Mỹ cắn hoặc tấn công	
W59	Bị loài bò sát khác cắn hoặc kẹp	
W60	Tiếp xúc với cây gai, gai, lá nhọn	
W64	Tiếp xúc với các lực cơ học động khác và/hoặc không xác định	
W65	Đuối nước và/hoặc ngập nước khi ở trong bồn tắm	
W66	Đuối nước và/hoặc ngập nước sau khi ngã vào bồn tắm	
W67	Đuối nước và/hoặc ngập nước khi ở bể bơi	
W68	Đuối nước và/hoặc ngập nước sau khi ngã vào bể bơi	
W69	Đuối nước và/hoặc ngập nước khi ở trong nước tự nhiên	
W70	Đuối nước và/hoặc ngập nước sau khi ngã xuống nước tự nhiên	
W73	Đuối nước và/hoặc ngập nước xác định khác	
W74	Đuối nước và/hoặc ngập nước không xác định	Đuối nước và/hoặc ngập nước xác định khác
W75	Vô tình bị ngạt thở và/hoặc bị siết cổ trên giường	
W76	Vô tình bị treo cổ và/hoặc siết cổ khác	
W77	Đe dọa hô hấp do sụt đất, sạt lở đất và/hoặc các chất khác	
W78	Hít phải các chất trong dạ dày	
W79	Hít và/hoặc nuốt thức ăn gây tắc nghẽn đường hô hấp	
W80	Hít và/hoặc nuốt phải các vật thể khác gây tắc nghẽn đường hô hấp	
W81	Bị giam giữ hoặc bị mắc kẹt trong môi trường thiếu oxy	
W83	Các mối đe dọa xác định khác đối với hô hấp	
W84	Mối đe dọa không xác định đối với hô hấp	
W85	Tiếp xúc với đường dây tải điện	
W86	Tiếp xúc với dòng điện xác định khác	
W87	Tiếp xúc với dòng điện không xác định	
W88	Phơi nhiễm với nguồn phóng xạ [bức xạ] ion hóa	
W89	Mắt phơi nhiễm với ánh sáng nhân tạo và/hoặc tia cực tím	
W90	Phơi nhiễm với bức xạ không ion hóa khác	
W91	Phơi nhiễm với nguồn phóng xạ [bức xạ] không xác định	
W92	Tiếp xúc với nhiệt độ quá cao có nguồn gốc nhân tạo	
W93	Tiếp xúc quá mức với nguồn nhiệt lạnh nhân tạo	
W94	Tiếp xúc với áp suất không khí cao và/hoặc thấp và/hoặc thay đổi áp suất không khí	
W99	Tiếp xúc với các yếu tố môi trường khác và/hoặc không xác định do con người tạo ra	
X00	Tiếp xúc với đám cháy không kiểm soát được ở tòa nhà hoặc công trình xây dựng	
X01	Tiếp xúc với đám cháy không kiểm soát được, không phải ở tòa nhà hoặc công trình xây dựng	
X02	Tiếp xúc với đám cháy kiểm soát được của tòa nhà hoặc công trình xây dựng	
X03	Tiếp xúc với đám cháy có kiểm soát, không phải ở tòa nhà hoặc công trình xây dựng	
X04	Tiếp xúc với vật liệu dễ cháy có tính bắt lửa cao	
X05	Tiếp xúc với quần áo hoặc trang phục dễ nóng chảy	
X06	Phơi nhiễm với quần áo hoặc trang phục khác dễ bắt lửa	
X08	Phơi nhiễm với khói, cháy và/hoặc lửa xác định khác	
X09	Phơi nhiễm với khói, cháy và/hoặc lửa không xác định	
X10	Tiếp xúc với đồ uống, thức ăn, mỡ và/hoặc dầu ăn nóng	
X11	Tiếp xúc với vòi nước nóng	
X12	Tiếp xúc với các chất lỏng nóng khác	
X13	Tiếp xúc với hơi nước và/hoặc hơi nóng	
X14	Tiếp xúc với không khí nóng và/hoặc hơi nóng	
X15	Tiếp xúc với các thiết bị gia dụng nóng	
X16	Tiếp xúc với thiết bị sưởi, bộ tản nhiệt và/hoặc đường ống nóng	
X17	Tiếp xúc với động cơ, máy móc và/hoặc công cụ nóng	
X18	Tiếp xúc với các kim loại nóng khác	
X19	Tiếp xúc với các chất nóng và nhiệt khác và/hoặc không xác định	
X20	Tiếp xúc với rắn độc và/hoặc thằn lằn độc	
X21	Tiếp xúc với nhện độc	
X22	Tiếp xúc với bọ cạp	
X23	Tiếp xúc với ong bắp cày, tò vò và/hoặc ong vò vẽ	
X24	Tiếp xúc với các loại rết và/hoặc các loại động vật nhiều chân (nhiệt đới)	
X25	Tiếp xúc với các động vật thân đốt có độc tính khác	
X26	Tiếp xúc với các thực vật và/hoặc động vật biển có độc tính khác	
X27	Tiếp xúc với động vật có nọc độc xác định khác	
X28	Tiếp xúc với các loài thực vật độc xác định khác	
X29	Tiếp xúc với động vật hoặc thực vật độc không xác định	
X30	Phơi nhiễm với thời tiết cực nóng	
X31	Phơi nhiễm với thời tiết cực lạnh	
X32	Phơi nhiễm với ánh nắng	
X33	Nạn nhân của sét	
X34	Nạn nhân của động đất	
X34.0	Nạn nhân của cơn đại hồng thủy do động đất gây ra	Bị mắc kẹt hoặc bị thương do sập tòa nhà hoặc cấu trúc khác do động đất
X34.1	Nạn nhân của sóng thần	
X34.8	Nạn nhân của các ảnh hưởng xác định khác của động đất	
X34.9	Nạn nhân của ảnh hưởng không xác định của động đất	
X35	Nạn nhân của vụ phun trào núi lửa	
X36	Nạn nhân của tuyết lở, đất trượt, hoặc các chuyển động khác của đất	
X37	Nạn nhân của cơn bão đại hồng thủy	
X38	Nạn nhân của lũ lụt	
X39	Phơi nhiễm với các lực lượng thiên nhiên khác và/hoặc không xác định	
X40	Ngộ độc do vô tình và/hoặc phơi nhiễm với thuốc giảm đau, hạ sốt và/hoặc chống viêm khớp dạng thấp không có chất dạng thuốc phiện	
X41	Ngộ độc do vô tình và/hoặc phơi nhiễm với thuốc chống động kinh, thuốc an thần - thuốc ngủ, thuốc chống hội chứng parkinson và/hoặc thuốc hướng thần, không phân loại mục khác	
X42	Vô tình ngộ độc do và tiếp xúc với ma túy và thuốc an thần [chất gây ảo giác], không phân loại mục khác	
X43	Ngộ độc do vô tình và/hoặc phơi nhiễm với các thuốc khác tác động lên hệ thống thần kinh tự động	
X44	Ngộ độc và/hoặc phơi nhiễm vô tình với dược chất, thuốc điều trị và/hoặc sinh phẩm khác và/hoặc không xác định	
X45	Ngộ độc do vô tình và/hoặc phơi nhiễm với rượu	
X46	Ngộ độc do vô tình và/hoặc phơi nhiễm với dung môi hữu cơ và/hoặc hydrocacbon đã halogen hóa và/hoặc hơi của chúng	
X47	Ngộ độc do vô tình và/hoặc phơi nhiễm với khí và/hoặc hơi khác	
X47.0	Ngộ độc do vô tình và/hoặc phơi nhiễm với khí carbon monoxide từ khí thải động cơ đốt trong	
X47.1	Ngộ độc do vô tình và/hoặc phơi nhiễm với khí carbon monoxide từ khí đốt	
X47.2	Ngộ độc do vô tình và/hoặc phơi nhiễm với khí carbon monoxide từ các nhiên liệu trong nước [nội địa] khác	
X47.3	Ngộ độc do vô tình và/hoặc phơi nhiễm với carbon monoxide từ các nguồn khác	
X47.4	Ngộ độc do vô tình và/hoặc phơi nhiễm với carbon monoxide từ các nguồn không xác định	
X47.8	Ngộ độc do vô tình và/hoặc phơi nhiễm với các khí và/hoặc hơi xác định khác	
X47.9	Ngộ độc do vô tình và/hoặc phơi nhiễm với khí và hơi không xác định	
X48	Ngộ độc do vô tình và/hoặc phơi nhiễm với thuốc trừ sâu	
X49	Ngộ độc do vô tình và/hoặc phơi nhiễm với chất hóa học và/hoặc chất có hại khác và/hoặc không xác định	
X50	Vận động quá sức và/hoặc gắng sức hoặc lặp đi lặp lại	
X51	Di chuyển và/hoặc vận động	
X52	Ở lâu trong môi trường không trọng lượng	
X53	Thiếu thức ăn	
X54	Thiếu nước	
X57	Tình trạng thiếu thốn không xác định	
X58	Phơi nhiễm với các yếu tố xác định khác	
X59	Phơi nhiễm với các yếu tố không xác định	
X59.0	Phơi nhiễm với yếu tố không xác định gây gãy xương	
X59.9	Phơi nhiễm với yếu tố không xác định gây chấn thương khác và/hoặc không xác định	Tai nạn không xác định khác|Phơi nhiễm không xác định khác
X60	Cố ý tự đầu độc bằng và/hoặc cố tình phơi nhiễm với thuốc giảm đau, hạ sốt và/hoặc chống viêm khớp dạng thấp không có chất dạng thuốc phiện	
X61	Cố tình tự đầu độc bằng và/hoặc cố tình phơi nhiễm với thuốc chống động kinh, thuốc an thần - gây ngủ, thuốc chống hội chứng parkinson và/hoặc thuốc tâm thần, không phân loại mục khác	
X62	Cố ý tự đầu độc và/hoặc cố tình phơi nhiễm với chất ma túy và/hoặc thuốc an thần [chất gây ảo giác], không phân loại mục khác	
X63	Cố tình tự đầu độc bằng và/hoặc cố tình phơi nhiễm với các thuốc khác tác dụng lên hệ thống thần kinh tự động	
X64	Cố tình tự đầu độc bằng và/hoặc phơi nhiễm với dược chất, thuốc điều trị, sinh phẩm khác và/hoặc không xác định	
X65	Cố tình tự đầu độc bằng và/hoặc cố tình phơi nhiễm với rượu	
X66	Cố tình tự đầu độc bằng và/hoặc cố tình phơi nhiễm với dung môi hữu cơ, hydrocacbon halogen hóa và/hoặc hơi của chúng	
X67	Cố tình tự đầu độc bằng và/hoặc cố tình phơi nhiễm với các khí và/hoặc hơi khác	
X67.0	Cố ý tự đầu độc bằng cách tiếp xúc với khí carbon monoxide từ khí thải động cơ đốt	
X67.1	Cố ý tự đầu độc bằng cách tiếp xúc với khí carbon monoxide từ khí đốt	
X67.2	Cố ý tự đầu độc bằng cách tiếp xúc với khí carbon monoxide từ các nhiên liệu khác trong gia đình	
X67.3	Cố ý tự đầu độc bằng cách tiếp xúc với carbon monoxide từ các nguồn khác	
X67.4	Cố ý tự đầu độc bằng cách tiếp xúc với carbon monoxide từ các nguồn không xác định	
X67.8	Cố ý tự đầu độc bằng cách tiếp xúc với khí và/hoặc hơi xác định khác	
X67.9	Cố ý tự đầu độc bằng cách tiếp xúc với khí và/hoặc hơi không xác định	
X68	Cố ý tự đầu độc bằng và/hoặc cố tình phơi nhiễm với thuốc trừ sâu	
X69	Cố ý tự đầu độc bằng và/hoặc cố tình phơi nhiễm với hóa chất và hoặc chất độc hại khác và/hoặc không xác định	
X70	Cố tình tự làm hại bản thân bằng cách treo cổ, bóp cổ và/hoặc tự làm ngạt thở	
X71	Cố tình tự hại bằng nhảy xuống nước và/hoặc trầm mình dưới nước	
X72	Cố tình tự làm hại bản thân bằng xả súng ngắn	
X73	Cố tình tự làm hại bản thân bằng xả súng trường, xúng bắn đạn ghém và/hoặc súng lớn hơn	
X74	Cố tình tự làm hại bản thân bằng xả súng khác và/hoặc không xác định	
X75	Cố tình tự làm hại bản thân bằng vật liệu nổ	
X76	Cố tình tự làm hại bản thân bằng khói, cháy và/hoặc lửa	
X77	Cố tình tự làm hại bản thân bằng hơi nước, hơi nóng và/hoặc vật nóng	
X78	Cố tình tự làm hại bản thân bằng vật sắc nhọn	
X79	Cố tình tự làm hại bản thân bằng vật cùn	
X80	Cố tình ý tự làm hại bản thân bằng cách nhảy từ trên cao xuống	
X81	Cố tình tự làm hại bản thân bằng cách nhảy hoặc nằm trước khi di chuyển vật thể	
X82	Cố tình tự làm hại bản thân bằng cách đâm vào xe cơ giới	
X83	Cố tình tự làm hại bản thân bằng các phương tiện xác định khác	
X84	Cố tình tự hại bằng các phương tiện không xác định	
X85	Tấn công bằng dược chất, thuốc điều trị và/hoặc sinh phẩm	
X86	Tấn công bằng hóa chất ăn mòn	
X87	Tấn công bằng thuốc trừ sâu	
X88	Tấn công bằng carbon monoxide và/hoặc các loại khí và/hoặc hơi khác	
X88.0	Tấn công bằng carbon monoxide từ khí thải động cơ đốt trong	
X88.1	Tấn công bằng carbon monoxide từ khí đốt tiện ích	
X88.2	Tấn công bằng carbon monoxide từ các nhiên liệu khác trong nhà	
X88.3	Tấn công bằng carbon monoxide từ các nguồn khác	
X88.4	Tấn công bằng carbon monoxide từ các nguồn không xác định	
X88.8	Tấn công bởi các loại khí và/hoặc hơi xác định khác	
X88.9	Tấn công bằng khí và/hoặc hơi không xác định	
X89	Tấn công bằng hóa chất và/hoặc chất độc hại xác định khác	
X90	Tấn công bằng hóa chất và/hoặc chất độc hại không xác định	
X91	Tấn công bằng cách treo cổ, bóp cổ và/hoặc làm ngạt thở	
X92	Tấn công bằng cách dìm nước và/hoặc làm ngập nước	
X93	Tấn công bằng xả súng ngắn	
X94	Tấn công bằng xả súng trường, súng bắn đạn ghém và/hoặc súng lớn hơn	
X95	Tấn công bằng cách xả súng khác và/hoặc không xác định	
X96	Tấn công bằng vật liệu nổ	
X97	Tấn công bằng khói, hỏa hoạn và/hoặc bằng lửa	
X98	Tấn công bằng hơi nước, hơi nóng và/hoặc các vật thể nóng	
X99	Tấn công bằng vật sắc nhọn	
Y00	Tấn công bằng vật thể không sắc nhọn	
Y01	Tấn công bằng cách đẩy từ nơi cao	
Y02	Tấn công bằng cách đẩy hoặc đặt nạn nhân trước vật đang chuyển động	
Y03	Tấn công bằng đâm xe cơ giới	
Y04	Tấn công bằng vũ lực	
Y05	Tấn công tình dục bằng vũ lực	
Y06	Thờ ơ và/hoặc bỏ rơi	
Y06.0	Bị vợ/chồng hoặc đối tác bỏ rơi	
Y06.1	Bị cha mẹ bỏ rơi	
Y06.2	Bị bạn hoặc người quen bỏ rơi	
Y06.8	Bị bỏ rơi bởi người xác định khác	
Y06.9	Bị bỏ rơi bởi người không xác định	
Y07	Ngược đãi khác	
Y07.0	Bị vợ/chồng hoặc đối tác ngược đãi	
Y07.1	Bị cha mẹ ngược đãi	
Y07.2	Bị bạn hoặc người quen ngược đãi	
Y07.3	Bị công chức có thẩm quyền ngược đãi	
Y07.8	Bị ngược đãi bởi người xác định khác	
Y07.9	Bị ngược đãi bởi người không xác định	
Y08	Tấn công bằng các phương tiện xác định khác	
Y09	Tấn công bằng phương tiện khác không xác định	
Y10	Ngộ độc và/hoặc phơi nhiễm với thuốc giảm đau, hạ sốt, chống viêm khớp dạng thấp không có chất dạng thuốc phiện, không rõ mục đích	
Y11	Ngộ độc và/hoặc phơi nhiễm với thuốc chống động kinh, thuốc an thần - gây ngủ, thuốc chống hội chứng parkinson và hướng thần, không phân loại mục khác và/hoặc không rõ mục đích	
Y12	Ngộ độc và/hoặc phơi nhiễm với thuốc mê và/hoặc thuốc làm hưng phấn tinh thần (sinh hoang tưởng), không phân loại mục khác và/hoặc không rõ mục đích	
Y13	Ngộ độc và/hoặc phơi nhiễm với các thuốc khác tác dụng trên hệ thần kinh tự động, không rõ mục đích	
Y14	Ngộ độc và/hoặc phơi nhiễm với dược chất, thuốc điều trị, sinh phẩm khác và/hoặc không xác định, không rõ ý định sử dụng	
Y15	Ngộ độc và/hoặc phơi nhiễm với rượu, không rõ mục đích	
Y16	Ngộ độc và/hoặc phơi nhiễm với chất dung môi hữu cơ và/hoặc hologenat carbon và/hoặc các chất hơi của chúng, không rõ mục đích	
Y17	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide và/hoặc các khí và/hoặc hơi khác, không rõ mục đích	
Y17.0	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide từ khí thải của động cơ đốt, không rõ mục đích	
Y17.1	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide từ khí đốt tiện ích, không rõ mục đích	
Y17.2	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide từ các nhiên liệu khác trong gia đình, không rõ mục đích	
Y17.3	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide từ các nguồn khác, không rõ mục đích	
Y17.4	Ngộ độc và/hoặc phơi nhiễm với carbon monoxide từ các nguồn không xác định, không rõ mục đích	
Y17.8	Ngộ độc và/hoặc phơi nhiễm với các khí và/hoặc hơi xác định khác, không rõ mục đích	
Y17.9	Ngộ độc và/hoặc phơi nhiễm với khí và/hoặc hơi không xác định, không rõ mục đích	
Y18	Ngộ độc và/hoặc phơi nhiễm với thuốc trừ sâu, không rõ mục đích	
Y19	Ngộ độc và/hoặc phơi nhiễm với hóa chất, chất độc hại khác và/hoặc không xác định, không rõ mục đích	
Y20	Treo cổ, bóp cổ và/hoặc nghẹt thở, không rõ mục đích	
Y21	Ngã xuống nước và/hoặc ngập nước, không rõ mục đích	
Y22	Xả súng ngắn, không rõ mục đích	
Y23	Xả súng trường, súng bắn đạn ghém và/hoặc súng lớn hơn, không rõ mục đích	
Y24	Xả súng khác và/hoặc không xác định, không rõ mục đích	
Y25	Tiếp xúc với vật liệu nổ, không rõ mục đích	
Y26	Phơi nhiễm khói, cháy và/hoặc lửa, không rõ mục đích	
Y27	Phơi nhiễm hơi nước, hơi nước nóng và/hoặc vật nóng, không rõ mục đích	
Y28	Tiếp xúc với vật sắc nhọn, không rõ mục đích	
Y29	Tiếp xúc với vật thể cùn, không rõ mục đích	
Y30	Ngã, nhảy hoặc bị đẩy từ một chỗ cao, không rõ mục đích	
Y31	Ngã, nằm hoặc chạy trước hoặc chạy vào vật đang chuyển động, không rõ mục đích	
Y32	Bị xe máy đâm, không rõ mục đích	Bị xe mô tô đâm, không rõ mục đích
Y33	Các sự cố xác định khác, không rõ mục đích	
Y34	Sự cố không xác định, không rõ mục đích	
Y35	Can thiệp pháp lý	
Y35.0	Can thiệp pháp lý liên quan đến xả súng	
Y35.1	Can thiệp pháp lý liên quan đến chất nổ	
Y35.2	Can thiệp pháp lý liên quan đến khí	Ngạt khí gas do sự can thiệp pháp lý|Bị thương bởi hơi cay do sự can thiệp pháp lý|Ngộ độc khí gas do can thiệp pháp lý
Y35.3	Can thiệp pháp lý liên quan đến các vật không sắc nhọn	
Y35.4	Can thiệp pháp lý liên quan đến vật sắc nhọn	Bị cắt khi can thiệp pháp lý|Bị thương do lưỡi lê trong quá trình can thiệp pháp lý|Bị đâm trong quá trình can thiệp pháp lý
Y35.5	Hành hình theo luật	
Y35.6	Can thiệp pháp lý liên quan đến các phương tiện xác định khác	Cư xử thô bạo
Y35.7	Can thiệp pháp lý, các phương tiện không xác định	
Y36	Các hoạt động trong chiến tranh	
Y36.0	Hoạt động chiến tranh liên quan đến vụ nổ vũ khí trên biển	Bom phá tàu ngầm|Mìn thủy|Mìn không xác định khác, trên biển hoặc ở bến cảng|Đạn pháo trên biển|Ngư lôi|Vụ nổ dưới nước
Y36.1	Hoạt động chiến tranh liên quan đến phá hủy phương tiện bay	
Y36.2	Hoạt động chiến tranh liên quan đến các vụ nổ và/hoặc mảnh vỡ khác	
Y36.3	Hành động chiến tranh iên quan đến cháy, hỏa hoạn và/hoặc các chất nóng	Bom xăng
Y36.4	Hoạt động chiến tranh liên quan đến xả súng và hình thức khác của chiến tranh quy ước	
Y36.5	Hoạt động chiến tranh liên quan đến vũ khí hạt nhân	Hiệu ứng vụ nổ|Phơi nhiễm với bức xạ ion hóa từ vũ khí hạt nhân|Hiệu ứng quả cầu lửa|Nhiệt|Các tác động trực tiếp và phụ khác của vũ khí hạt nhân
Y36.6	Hoạt động chiến tranh liên quan đến vũ khí sinh học	
Y36.7	Hoạt động chiến tranh liên quan đến vũ khí hóa học và/hoặc vũ khí chiến tranh phi quy ước khác	Khí, khói và hóa chất|Tia laze
Y36.8	Các hoạt động chiến tranh xảy ra sau khi chấm dứt chiến sự	
Y36.9	Các hoạt động chiến tranh, không xác định	
Y40	Tác động bất lợi của kháng sinh toàn thân sử dụng cho mục đích điều trị	Kháng sinh toàn thân
Y40.0	Tác động bất lợi của penicillin	
Y40.1	Tác động bất lợi của cefalosporin và/hoặc kháng sinh beta-lactam khác	
Y40.2	Tác động bất lợi của nhóm chloramphenicol	
Y40.3	Tác động bất lợi của macrolid	
Y40.4	Tác động bất lợi của tetracyclin	
Y40.5	Tác động bất lợi của aminoglycosid	Streptomycin
Y40.6	Tác động bất lợi của rifamycin	
Y40.7	Tác động bất lợi của thuốc kháng sinh chống nấm, sử dụng toàn thân	
Y40.8	Tác động bất lợi của kháng sinh toàn thân khác	
Y40.9	Tác động bất lợi của kháng sinh toàn thân, không xác định	
Y41	Tác động bất lợi của thuốc chống nhiễm trùng và/hoặc chống ký sinh trùng toàn thân khác	Thuốc chống nhiễm trùng và/hoặc chống ký sinh trùng toàn thân khác
Y41.0	Tác động bất lợi của sulfonamid	
Y41.1	Tác động bất lợi của thuốc chống mycobacteria	
Y41.2	Tác động bất lợi của thuốc chống sốt rét và/hoặc thuốc tác động lên động vật nguyên sinh khác trong máu	
Y41.3	Tác động bất lợi của thuốc chống động vật nguyên sinh khác	
Y41.4	Tác động bất lợi của thuốc tẩy giun sán	
Y41.5	Tác động bất lợi của thuốc kháng virus	
Y41.8	Tác động bất lợi của thuốc chống nhiễm trùng toàn thân và/hoặc chống ký sinh trùng xác định khác	
Y41.9	Tác động bất lợi của thuốc chống nhiễm khuẩn và/hoặc ký sinh trùng toàn thân, không xác định	
Y42	Tác động bất lợi của nội tiệt tố [hormon] và/hoặc chất thay thế tổng hợp và/hoặc chất đối kháng, không phân loại mục khác	Nội tiệt tố [hormon] và/hoặc chất thay thế tổng hợp và/hoặc chất đối kháng, không phân loại mục khác
Y42.0	Tác động bất lợi của glucocorticoid và/hoặc các chất tương tự tổng hợp	
Y42.1	Tác động bất lợi của nội tiệt tố [hormon] tuyến giáp và/hoặc chất thay thế	
Y42.2	Tác động bất lợi của thuốc kháng giáp	
Y42.3	Tác động bất lợi của insulin và/hoặc thuốc uống hạ đường huyết [chống đái tháo đường]	
Y42.4	Tác động bất lợi của thuốc uống tránh thai	
Y42.5	Tác động bất lợi của estrogen và/hoặc progestogen khác	Hỗn hợp và các chất thay thế
Y42.6	Tác động bất lợi của kháng gonadotrophin, kháng estrogen, kháng androgen, không phân loại mục khác	Tamoxifen
Y42.7	Tác động bất lợi của androgen và/hoặc các đồng loại đồng hóa	
Y42.8	Tác động bất lợi của nội tiệt tố [hormon] khác và/hoặc không xác định và/hoặc chất thay thế của chúng	Nội tiết tố [adenohypophyseal] thùy trước tuyến yên
Y42.9	Tác động bất lợi của thuốc kháng nội tiệt tố [hormon] khác và/hoặc không xác định	
Y43	Tác động bất lợi của tác nhân tác dụng toàn thân chủ yếu	Tác nhân tác dụng toàn thân chủ yếu
Y43.0	Tác động bất lợi của thuốc chống dị ứng và/hoặc thuốc chống nôn	
Y43.1	Tác động bất lợi của chất chống chuyển hóa chống ung thư	Cytarabine
Y43.2	Tác động bất lợi của sản phẩm tự nhiên chống ung thư	
Y43.3	Tác động bất lợi của thuốc chống ung thư khác	
Y43.4	Tác động bất lợi của thuốc ức chế miễn dịch	
Y43.5	Tác động bất lợi của chất toan hóa và/hoặc chất kiềm hóa	
Y43.6	Tác động bất lợi của men, không phân loại mục khác	
Y43.8	Tác động bất lợi của chất tác dụng toàn thân chủ yếu khác, không phân loại mục khác	Chất đối kháng kim loại nặng
Y43.9	Tác động bất lợi của chất tác dụng toàn thân chủ yếu, không xác định	
Y44	Tác động bất lợi của chất tác dụng chủ yếu lên các thành phần của máu	Chất tác dụng chủ yếu lên các thành phần của máu
Y44.0	Tác động bất lợi của chế phẩm sắt và/hoặc chế phẩm chống thiếu máu giảm huyết sắc tố khác	
Y44.1	Tác động bất lợi của vitamin B12, axit folic và/hoặc chế phẩm chống thiếu máu nguyên bào khổng lồ khác	
Y44.2	Tác động bất lợi của thuốc chống đông máu	
Y44.3	Tác động bất lợi của thuốc đối kháng chống đông máu, vitamin K và/hoặc chất đông máu khác	
Y44.4	Tác động bất lợi của thuốc chống huyết khối [thuốc ức chế kết tập tiểu cầu]	
Y44.5	Tác động bất lợi của thuốc làm tan huyết khối	
Y44.6	Tác động bất lợi của máu tự nhiên và/hoặc chế phẩm máu	
Y44.7	Tác động bất lợi của chất thay thế huyết tương	
Y44.9	Tác động bất lợi của chất khác và/hoặc không xác định ảnh hưởng đến các thành phần của máu	
Y45	Tác động bất lợi của thuốc giảm đau, thuốc hạ sốt và/hoặc thuốc chống viêm	Thuốc giảm đau, thuốc hạ sốt và/hoặc thuốc chống viêm
Y45.0	Tác động bất lợi của thuốc phiện và/hoặc thuốc giảm đau liên quan	
Y45.1	Tác động bất lợi của salicylat	
Y45.2	Tác động bất lợi của các dẫn xuất của axit propionic	Các dẫn xuất của axit propanoic
Y45.3	Tác động bất lợi của thuốc chống viêm không steroid khác [NSAID]	
Y45.4	Tác động bất lợi của thuốc chống viêm khớp dạng thấp	
Y45.5	Tác động bất lợi của các dẫn xuất 4-aminophenol	
Y45.8	Tác động bất lợi của thuốc giảm đau và/hoặc hạ sốt khác	
Y45.9	Tác động bất lợi của thuốc giảm đau, hạ sốt và/hoặc chống viêm, không xác định	
Y46	Tác động bất lợi của thuốc chống động kinh và/hoặc parkinson	Thuốc chống động kinh và/hoặc parkinson
Y46.0	Tác động bất lợi của succinimid	
Y46.1	Tác động bất lợi của oxazolidinedion	
Y46.2	Tác động bất lợi của các dẫn xuất hydantoin	
Y46.3	Tác động bất lợi của deoxybarbiturat	
Y46.4	Tác động bất lợi của iminostilben	Carbamazepine
Y46.5	Tác động bất lợi của acid valproic	
Y46.6	Tác động bất lợi của thuốc chống động kinh khác và/hoặc không xác định	
Y46.7	Tác động bất lợi của thuốc chống parkinson	Thuốc chống virus và thuốc chống giun sán [Amantadine]
Y46.8	Tác động bất lợi của thuốc trị co thắt	
Y47	Tác động bất lợi của thuốc an thần, thuốc ngủ và/hoặc thuốc chống lo lắng	Thuốc an thần, thuốc ngủ và/hoặc thuốc chống lo lắng
Y47.0	Tác động bất lợi của barbiturat, không phân loại mục khác	
Y47.1	Tác động bất lợi benzodiazepin	
Y47.2	Tác động bất lợi của các dẫn xuất cloral	
Y47.3	Tác động bất lợi của paraldehyd	
Y47.4	Tác động bất lợi của hợp chất brom	
Y47.5	Tác động bất lợi của thuốc an thần và/hoặc thuốc ngủ hỗn hợp, không phân loại mục khác	
Y47.8	Tác động bất lợi của thuốc an thần, thuốc ngủ và/hoặc thuốc chống lo âu khác	Methaqualone
Y47.9	Tác động bất lợi của thuốc an thần, thuốc ngủ và/hoặc thuốc chống lo âu, không xác định	
Y48	Tác động bất lợi của thuốc mê và/hoặc khí điều trị	Thuốc mê và/hoặc khí điều trị
Y48.0	Tác động bất lợi của thuốc mê dạng hít	
Y48.1	Tác động bất lợi của thuốc gây mê đường tĩnh mạch	Thiobarbiturates
Y48.2	Tác động bất lợi của thuốc gây mê toàn thân khác và/hoặc không xác định	
Y48.3	Tác động bất lợi của thuốc gây tê cục bộ	
Y48.4	Tác động bất lợi của thuốc gây mê, không xác định	
Y48.5	Tác động bất lợi của khí điều trị	
Y49	Tác động bất lợi của thuốc hướng thần, không phân loại mục khác	Thuốc hướng thần, không phân loại mục khác
Y49.0	Tác động bất lợi của thuốc chống trầm cảm ba vòng và/hoặc bốn vòng	
Y49.1	Tác động bất lợi của thuốc chống trầm cảm ức chế men monoamin-oxidase	
Y49.2	Tác động bất lợi của thuốc chống trầm cảm khác và/hoặc không xác định	
Y49.3	Tác động bất lợi của thuốc chống loạn thần và/hoặc an thần dạng phenothiazine	
Y49.4	Tác động bất lợi của thuốc an thần loại butyrophenon và/hoặc thioxanthen	
Y49.5	Tác động bất lợi của thuốc chống loạn thần và/hoặc thuốc an thần khác	
Y49.6	Tác động bất lợi của thuốc an thần [chất gây ảo giác]	
Y49.7	Tác động bất lợi của chất kích thích tâm thần có khả năng lạm dụng	
Y49.8	Tác động bất lợi của thuốc hướng thần khác, không phân loại mục khác	
Y49.9	Tác động bất lợi của thuốc hướng tâm thần, không xác định	
Y50	Tác động bất lợi của thuốc kích thích hệ thần kinh trung ương, không phân loại mục khác	Thuốc kích thích hệ thần kinh trung ương, không phân loại mục khác
Y50.0	Tác động bất lợi của thuốc tăng cường sức khỏe	
Y50.1	Tác động bất lợi của thuốc đối kháng thụ thể opioid	
Y50.2	Tác động bất lợi của methylxanthines, không phân loại mục khác	
Y50.8	Tác động bất lợi của chất kích thích hệ thần kinh trung ương khác	
Y50.9	Tác động bất lợi của chất kích thích hệ thần kinh trung ương, không xác định	
Y51	Tác động bất lợi của thuốc tác dụng chủ yếu trên hệ thần kinh tự động	Thuốc tác dụng chủ yếu trên hệ thần kinh tự động
Y51.0	Tác động bất lợi của chất kháng cholinesterase	
Y51.1	Tác động bất lợi của các thuốc phó giao cảm khác [cholinergics]	
Y51.2	Tác động bất lợi của thuốc ức chế dẫn truyền qua hạch, không phân loại mục khác	
Y51.3	Tác động bất lợi của thuốc giải ký sinh trùng khác [thuốc kháng cholinergic và antimuscarinics] và thuốc giảm co thắt, không phân loại mục khác	Papaverin
Y51.4	Tác động bất lợi của thuốc chủ vận ưu thế tác dụng trên thụ cảm thể alpha adrenergic, không phân loại mục khác	Metaraminol
Y51.5	Tác động bất lợi của chất ưu tiên trên cảm thụ beta-adrenergic, không phân loại mục khác	
Y51.6	Tác động bất lợi của chất đối kháng cảm thụ alpha-adrenalin, không phân loại mục khác	
Y51.7	Tác động bất lợi của thuốc kháng thụ cảm thể beta-adrenergic, không phân loại mục khác	
Y51.8	Tác động bất lợi của chất tác dụng trung ương và/hoặc tác nhân ngăn chặn tế bào thần kinh adrenergic, không phân loại mục khác	
Y51.9	Tác động bất lợi của thuốc khác và/hoặc không xác định chủ yếu tác động trên hệ thần kinh tự chủ	Thuốc kích thích cả thụ thể alpha và beta-adrenoreceptors
Y52	Tác động bất lợi của thuốc tác dụng chủ yếu lên hệ tim mạch	Các thuốc tác dụng chủ yếu lên hệ tim mạch
Y52.0	Tác động bất lợi của glycoside kích thích tim và/hoặc các loại thuốc có tác dụng tương tự	
Y52.1	Tác động bất lợi của thuốc chẹn kênh calci	
Y52.2	Tác động bất lợi của thuốc chống loạn nhịp khác, không phân loại mục khác	
Y52.3	Tác động bất lợi của thuốc giãn mạch vành, không phân loại mục khác	
Y52.4	Tác động bất lợi của thuốc ức chế men chuyển angiotensin	
Y52.5	Tác động bất lợi của thuốc hạ huyết áp khác, không phân loại mục khác	
Y52.6	Tác động bất lợi của thuốc chống tăng mỡ máu và/hoặc thuốc chống xơ cứng động mạch	
Y52.7	Tác động bất lợi của thuốc giãn mạch ngoại vi	
Y52.8	Tác động bất lợi của thuốc chống giãn tĩnh mạch, bao gồm chất gây xơ	
Y52.9	Tác động bất lợi của thuốc khác và/hoặc không xác định chủ yếu gây ảnh hưởng đến hệ tim mạch	
Y53	Tác động bất lợi của thuốc tác dụng chủ yếu đến hệ tiêu hóa	Thuốc tác dụng chủ yếu đến hệ tiêu hóa
Y53.0	Tác động bất lợi của chất đối kháng thụ thể histamin H2	
Y53.1	Tác động bất lợi của thuốc kháng axit và/hoặc thuốc chống tiết dịch vị khác	
Y53.2	Tác động bất lợi của thuốc kích thích nhuận tràng	
Y53.3	Tác động bất lợi của nước muối và/hoặc thuốc nhuận tràng thẩm thấu	
Y53.4	Tác động bất lợi của thuốc nhuận tràng khác	Thuốc giảm trương lực ruột
Y53.5	Tác động bất lợi của thuốc kích thích tiêu hoá	
Y53.6	Tác động bất lợi của thuốc chống tiêu chảy	
Y53.7	Tác động bất lợi của thuốc gây nôn	
Y53.8	Tác động bất lợi của thuốc tác dụng chủ yếu lên hệ tiêu hoá	
Y53.9	Tác động bất lợi của thuốc tác dụng chủ yếu lên hệ tiêu hoá, không xác định	
Y54	Tác động bất lợi của thuốc chủ yếu ảnh hưởng tới cân bằng nước và/hoặc chuyển hóa acid và hoặc khoáng chất	Các thuốc chủ yếu ảnh hưởng tới cân bằng nước và/hoặc chuyển hoá acid và hoặc khoáng chất
Y54.0	Tác động bất lợi của corticoid chuyển hóa muối nước	
Y54.1	Tác động bất lợi của thuốc đối kháng corticoid khoáng [thuốc kháng aldosteron]	
Y54.2	Tác động bất lợi của thuốc ức chế cacbonic-anhydrase	Acetazolamide
Y54.3	Tác động bất lợi của các dẫn xuất benzothiadiazin	
Y54.4	Tác động bất lợi của thuốc lợi tiểu quai	
Y54.5	Tác động bất lợi của thuốc lợi tiểu khác	
Y54.6	Tác động bất lợi của thuốc cân bằng nước và/hoặc nhiệt, điện giải	
Y54.7	Tác động bất lợi của tác nhân ảnh hưởng đến quá trình vôi hóa	Hormone tuyến cận giáp và các dẫn xuất|Nhóm vitamin D
Y54.8	Tác động bất lợi của tác nhân ảnh hưởng đến chuyển hóa acid uric	
Y54.9	Tác động bất lợi của muối khoáng, không phân loại mục khác	
Y55	Tác động bất lợi của tác nhân chủ yếu tác động lên cơ trơn, cơ xương và/hoặc hệ hô hấp	Tác nhân chủ yếu tác động lên cơ trơn, cơ xương và/hoặc hệ hô hấp
Y55.0	Tác động bất lợi của thuốc tăng co bóp tử cung	
Y55.1	Tác động bất lợi của thuốc giãn cơ xương [thuốc phong bế thần kinh - cơ] [thuốc chẹn thần kinh - cơ]	
Y55.2	Tác động bất lợi của thuốc khác và/hoặc không xác định hoạt động chủ yếu trên các cơ	
Y55.3	Tác động bất lợi của thuốc chống ho	
Y55.4	Tác động bất lợi của thuốc long đờm	
Y55.5	Tác động bất lợi của thuốc chống cảm cúm thông thường	
Y55.6	Tác động bất lợi của thuốc chống hen phế quản, không phân loại mục khác	
Y55.7	Tác động bất lợi của tác nhân khác và/hoặc không xác định chủ yếu tác động lên hệ hô hấp	
Y56	Tác động bất lợi của tác nhân dùng tại chỗ chủ yếu ảnh hưởng đến da, màng nhầy và/hoặc các loại thuốc nhãn khoa, tai mũi họng và/hoặc nha khoa	Tác nhân dùng tại chỗ chủ yếu ảnh hưởng đến da, màng nhầy và/hoặc các loại thuốc nhãn khoa, tai mũi họng và/hoặc nha khoa
Y56.0	Tác động bất lợi của thuốc chống nấm, chống nhiễm trùng và/hoặc chống viêm tại chỗ, không phân loại mục khác	
Y56.1	Tác động bất lợi của thuốc trị ngứa	
Y56.2	Tác động bất lợi của chất làm se tại chỗ và/hoặc chất tẩy tại chỗ	
Y56.3	Tác động bất lợi của chất làm mềm, chất khử mùi và/hoặc chất bảo vệ	
Y56.4	Tác động bất lợi của thuốc và/hoặc chế phẩm phân giải keratin, tạo keratin và chữa tóc khác	
Y56.5	Tác động bất lợi của thuốc và/hoặc chế phẩm nhãn khoa	
Y56.6	Tác động bất lợi của thuốc và/hoặc chế phẩm tai mũi họng	
Y56.7	Tác động bất lợi của thuốc răng, dùng tại chỗ	
Y56.8	Tác động bất lợi của tác nhân dùng tại chỗ khác	Thuốc diệt tinh trùng
Y56.9	Tác động bất lợi của tác nhân dùng tại chỗ, không xác định	
Y57	Tác động bất lợi của dược chất và/hoặc thuốc điều trị khác và/hoặc không xác định	Dược chất và/hoặc thuốc điều trị khác và/hoặc không xác định
Y57.0	Tác động bất lợi của thuốc làm giảm cảm giác thèm ăn [biếng ăn]	
Y57.1	Tác động bất lợi của thuốc tăng cường đốt mỡ	
Y57.2	Tác động bất lợi của thuốc giải độc và/hoặc chất chelat hóa, không phân loại mục khác	
Y57.3	Tác động bất lợi của thuốc giải rượu	
Y57.4	Tác động bất lợi của tá dược	
Y57.5	Tác động bất lợi của thuốc cản quang	
Y57.6	Tác động bất lợi của chất khác sử dụng trong chẩn đoán	
Y57.7	Tác động bất lợi của vitamin, không phân loại mục khác	
Y57.8	Tác động bất lợi của dược chất và/hoặc thuốc điều trị khác	
Y57.9	Tác động bất lợi của dược chất hoặc thuốc điều trị, không xác định	
Y58	Tác động bất lợi của vắc xin chống vi khuẩn	Vắc xin chống vi khuẩn
Y58.0	Tác động bất lợi của vắc xin BCG	
Y58.1	Tác động bất lợi của vắc xin thương hàn và/hoặc vắc xin phó thương hàn	
Y58.2	Tác động bất lợi của vắc xin tả	
Y58.3	Tác động bất lợi của vắc xin dịch hạch	
Y58.4	Tác động bất lợi của vắc xin uốn ván	
Y58.5	Tác động bất lợi của vắc xin bạch hầu	
Y58.6	Tác động bất lợi của vắc xin ho gà, bao gồm vắc xin kết hợp có thành phần ho gà	
Y58.8	Tác động bất lợi của vắc xin kết hợp chống vi khuẩn, trừ kết hợp có thành phần ho gà	
Y58.9	Tác động bất lợi của vắc xin chống vi khuẩn khác và/hoặc không xác định	
Y59	Tác động bất lợi của vắc xin và sinh phẩm khác và/hoặc không xác định	Vắc xin và sinh phẩm khác và/hoặc không xác định
Y59.0	Tác động bất lợi của vắc xin chống virus	
Y59.1	Tác động bất lợi của vắc xin Rickettsia	
Y59.2	Tác động bất lợi của vắc xin chống đơn bào	
Y59.3	Tác động bất lợi của globulin miễn dịch	
Y59.8	Tác động bất lợi của vắc xin và/hoặc sinh phẩm xác định khác	
Y59.9	Tác động bất lợi của vắc xin hoặc sinh phẩm, không xác định	
Y60	Vết cắt, đâm thủng, thủng hoặc xuất huyết ngoài ý muốn trong chăm sóc ngoại khoa và/hoặc nội khoa	
Y60.0	Sự cố y khoa trong khi phẫu thuật	
Y60.1	Sự cố y khoa trong quá trình tiêm truyền hoặc truyền máu	
Y60.2	Sự cố y khoa trong quá trình chạy thận nhân tạo hoặc truyền dịch khác	
Y60.3	Sự cố y khoa trong quá trình tiêm hoặc tiêm chủng	
Y60.4	Sự cố y khoa trong quá trình khám nội soi	
Y60.5	Sự cố y khoa trong quá trình thông tim	
Y60.6	Sự cố y khoa trong quá trình hút, chọc dò và/hoặc đặt ống thông khác	
Y60.7	Sự cố y khoa trong quá trình thụt tháo	
Y60.8	Sự cố y khoa trong chăm sóc ngoại khoa và/hoặc nội khoa	
Y60.9	Sự cố y khoa trong chăm sóc ngoại khoa và/hoặc nội khoa không xác định	
Y61	Dị vật vô tình để lại trong cơ thể trong chăm sóc ngoại khoa và/hoặc nội khoa	
Y61.0	Dị vật vô tình để lại trong cơ thể trong quá trình phẫu thuật	
Y61.1	Dị vật vô tình để lại trong cơ thể trong quá trình truyền dịch hoặc truyền máu	
Y61.2	Dị vật vô tình để lại trong cơ thể trong quá trình chạy thận nhân tạo hoặc truyền dịch khác	
Y61.3	Dị vật vô tình để lại trong cơ thể trong quá trình tiêm thuốc hoặc tiêm chủng	
Y61.4	Dị vật vô tình để lại trong cơ thể trong quá trình khám nội soi	
Y61.5	Dị vật vô tình để lại trong cơ thể trong quá trình thông tim	
Y61.6	Dị vật vô tình để lại trong cơ thể trong quá trình hút, chọc dò và/hoặc đặt ống thông khác	
Y61.7	Dị vật vô tình để lại trong khi rút catheter hoặc băng dán cố định catheter	
Y61.8	Dị vật vô tình để lại trong chăm sóc ngoại khoa và/hoặc y khoa khác	
Y61.9	Dị vật vô tình để lại trong chăm sóc ngoại khoa và/hoặc nội khoa không xác định	
Y62	Không đảm bảo vô khuẩn trong chăm sóc ngoại khoa và/hoặc nội khoa	
Y62.0	Không đảm bảo vô khuẩn trong khi phẫu thuật	
Y62.1	Không đảm bảo vô khuẩn trong quá trình truyền dịch hoặc truyền máu	
Y62.2	Không đảm bảo vô khuẩn trong quá trình chạy thận nhân tạo hoặc truyền dịch khác	
Y62.3	Không đảm bảo vô khuẩn trong quá trình tiêm hoặc tiêm chủng	
Y62.4	Không đảm bảo vô khuẩn trong khi khám nội soi	
Y62.5	Không đảm bảo vô khuẩn trong quá trình thông tim	
Y62.6	Không đảm bảo vô khuẩn trong quá trình hút, chọc dò và/hoặc đặt ống thông khác	
Y62.8	Không đảm bảo vô khuẩn trong chăm sóc ngoại khoa và/hoặc nội khoa khác	
Y62.9	Không đảm bảo vô khuẩn trong chăm sóc ngoại khoa và/hoặc nội khoa không xác định	
Y63	Sai sót về liều lượng thuốc trong chăm sóc ngoại khoa và/hoặc nội khoa	
Y63.0	Truyền quá nhiều máu hoặc dịch truyền trong quá trình truyền dịch hoặc truyền máu	
Y63.1	Pha loãng dịch truyền không chính xác sử dụng trong quá trình truyền dịch	
Y63.2	Sử dụng quá liều lượng bức xạ trong quá trình điều trị	
Y63.3	Người bệnh vô tình phơi nhiễm với nguồn phóng xạ [bức xạ] trong quá trình chăm sóc y tế	
Y63.4	Sai sót về liều lượng trong điều trị sốc điện hoặc sốc insulin	
Y63.5	Nhiệt độ không phù hợp trong điều trị áp tại chỗ hoặc đắp quanh người	
Y63.6	Không sử dụng dược chất, thuốc điều trị và/hoặc sinh phẩm cần thiết	
Y63.8	Sai sót về liều lượng thuốc trong chăm sóc ngoại khoa và/hoặc nội khoa khác	
Y63.9	Sai sót về liều lượng thuốc của chăm sóc ngoại khoa và/hoặc nội khoa không xác định	
Y64	Chế phẩm y tế, sinh phẩm bị ô nhiễm	
Y64.0	Truyền các chế phẩm y tế, hoặc sinh phẩm bị ô nhiễm	
Y64.1	Tiêm hoặc gây miễn dịch bằng chế phẩm y tế hoặc sinh phẩm bị ô nhiễm	
Y64.8	Chất liệu y tế hoặc sinh phẩm bị ô nhiễm được đưa vào cơ thể bằng phương tiện khác	
Y64.9	Chất liệu y tế hoặc sinh phẩm bị ô nhiễm được đưa vào cơ thể bằng phương tiện không xác định	Đưa vào cơ thể chất liệu y tế hoặc sinh phẩm bị ô nhiễm không xác định khác
Y65	Sai sót khác của chăm sóc ngoại khoa và/hoặc nội khoa	
Y65.0	Truyền sai nhóm máu	
Y65.1	Truyền sai dịch	
Y65.2	Khâu hoặc buộc chỉ không đúng trong quá trình phẫu thuật	
Y65.3	Đặt ống nội khí quản sai vị trí khi thực hiện kỹ thuật gây mê	
Y65.4	Thất bại trong đưa vào hoặc lấy ra ống hoặc dụng cụ khác	
Y65.5	Thực hiện phẫu thuật không phù hợp	
Y65.8	Sai sót xác định khác của chăm sóc ngoại khoa và/hoặc nội khoa	
Y66	Không thực hiện can thiệp ngoại khoa và/hoặc nội khoa	
Y69	Sai sót không xác định của chăm sóc ngoại khoa và/hoặc nội khoa	
Y70	Thiết bị gây mê có liên quan đến biến cố bất lợi	Dụng cụ gây mê kết hợp với tai biến
Y70.0	Thiết bị gây mê có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y70.1	Thiết bị gây mê có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y70.2	Thiết bị gây mê có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y70.3	Thiết bị gây mê có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y70.8	Thiết bị gây mê có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y71	Thiết bị tim mạch có liên quan đến biến cố bất lợi	
Y71.0	Thiết bị tim mạch có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y71.1	Thiết bị tim mạch có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y71.2	Thiết bị tim mạch có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y71.3	Thiết bị tim mạch có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y71.8	Thiết bị tim mạch có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y72	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi	
Y72.0	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y72.1	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y72.2	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y72.3	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y72.8	Thiết bị tai mũi họng có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y73	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi	
Y73.0	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y73.1	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y73.2	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y73.3	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y73.8	Thiết bị tiêu hóa và/hoặc tiết niệu có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y74	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi	
Y74.0	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y74.1	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y74.2	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y74.3	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y74.8	Bệnh viện đa khoa và/hoặc các thiết bị sử dụng cá nhân có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y75	Thiết bị thần kinh có liên quan đến biến cố bất lợi	
Y75.0	Thiết bị thần kinh có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y75.1	Thiết bị thần kinh có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y75.2	Thiết bị thần kinh có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y75.3	Thiết bị thần kinh có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y75.8	Thiết bị thần kinh có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y76	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi	
Y76.0	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y76.1	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y76.2	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y76.3	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y76.8	Thiết bị sản phụ khoa có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y77	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi	
Y77.0	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y77.1	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y77.2	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y77.3	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y77.8	Thiết bị nhãn khoa có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y78	Thiết bị phóng xạ có liên quan đến biến cố bất lợi	
Y78.0	Thiết bị phóng xạ có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y78.1	Thiết bị phóng xạ có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y78.2	Thiết bị phóng xạ có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y78.3	Thiết bị phóng xạ có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y78.8	Thiết bị phóng xạ có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y79	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi	
Y79.0	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y79.1	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y79.2	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y79.3	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y79.8	Thiết bị chỉnh hình có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y80	Thiết bị y tế liên quan đến sự cố y khoa	
Y80.0	Thiết bị y tế liên quan đến sự cố y khoa, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y80.1	Thiết bị y tế liên quan đến sự cố y khoa, thiết bị trị liệu (không phẫu thuật) và/hoặc phục hồi chức năng	
Y80.2	Thiết bị y tế liên quan đến sự cố y khoa, thiết bị, vật tư và/hoặc phụ kiện thay thế và/hoặc cấy ghép khác	
Y80.3	Thiết bị y tế liên quan đến sự cố y khoa, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ phẫu thuật)	
Y80.8	Thiết bị y tế liên quan đến sự cố y khoa, thiết bị khác, không phân loại mục khác	
Y81	Thiết bị ngoại khoa và/hoặc tạo hình liên quan đến sự cố y khoa	
Y81.0	Thiết bị phẫu thuật nói chung và/hoặc tạo hình có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y81.1	Thiết bị phẫu thuật nói chung và/hoặc tạo hình có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y81.2	Thiết bị phẫu thuật nói chung và/hoặc tạo hình có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y81.3	Thiết bị phẫu thuật nói chung và/hoặc tạo hình có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y81.8	Thiết bị phẫu thuật nói chung và/hoặc tạo hình có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y82	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi	
Y82.0	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi, thiết bị theo dõi và/hoặc chẩn đoán hình ảnh	
Y82.1	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi, thiết bị trị liệu (không phẫu thuật) và/hoặc thiết bị phục hồi chức năng	
Y82.2	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi, thiết bị nhân tạo và/hoặc các thiết bị cấy ghép, vật liệu và/hoặc phụ kiện khác	
Y82.3	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi, dụng cụ, vật liệu và/hoặc thiết bị phẫu thuật (bao gồm cả chỉ khâu)	
Y82.8	Thiết bị khác và/hoặc không xác định có liên quan đến biến cố bất lợi, các thiết bị khác, không phân loại mục khác	
Y83	Can thiệp ngoại khoa là nguyên nhân của phản ứng bất thường của người bệnh hoặc biến chứng sau này, không đề cập đến sai sót tại thời điểm thực hiện	
Y83.0	Tai biến do phẫu thuật ghép toàn bộ tạng	
Y83.1	Tai biến do phẫu thuật cấy thiết bị nhân tạo bên trong cơ thể	
Y83.2	Tai biến do phẫu thuật với khâu nối, bắc cầu hoặc ghép	
Y83.3	Tai biến do phẫu thuật làm hậu môn nhân tạo	
Y83.4	Tai biến do phẫu thuật tái tạo khác	
Y83.5	Tai biến do phẫu thuật cắt cụt (các) chi	
Y83.6	Tai biến do phẫu thuật cắt bỏ tạng khác (một phần) (toàn bộ)	
Y83.8	Tai biến do can thiệp ngoại khoa khác	
Y83.9	Tai biến do can thiệp ngoại khoa, không xác định	
Y84	Can thiệp khác là nguyên nhân của phản ứng bất thường ở người bệnh hay biến chứng sau này, không đề cập đến sai sót tại thời điểm thực hiện	
Y84.0	Tai biến do thông tim	
Y84.1	Tai biến do chạy thận nhân tạo	
Y84.2	Tai biến do kỹ thuật chẩn đoán hình ảnh và/hoặc xạ trị	
Y84.3	Tai biến do liệu pháp sốc	
Y84.4	Tai biến do hút dịch	
Y84.5	Tai biến do đặt ống thông dạ dày hoặc tá tràng	
Y84.6	Tai biến do thông tiểu	
Y84.7	Tai biến do lấy mẫu máu	
Y84.8	Tai biến do can thiệp khác	
Y84.9	Tai biến do can thiệp, không xác định	
Y85	Di chứng của tai nạn giao thông vận tải	
Y85.0	Di chứng của tai nạn xe cơ giới	
Y85.9	Di chứng của các tại nạn giao thông khác và/hoặc không xác định	
Y86	Di chứng của các tai nạn khác	
Y87	Di chứng của hành vi cố ý tự làm hại bản thân, tấn công và/hoặc các sự kiện không xác định được chủ đích	
Y87.0	Di chứng của hành vi cố ý tự làm hại bản thân	
Y87.1	Di chứng của tấn công	
Y87.2	Di chứng của các sự kiện không xác định được chủ đích	
Y88	Di chứng của chăm sóc ngoại khoa và/hoặc nội khoa là nguyên nhân bên ngoài	
Y88.0	Di chứng của tác dụng bất lợi do sử dụng dược chất, thuốc điều trị và/hoặc sinh phẩm để điều trị bệnh	
Y88.1	Di chứng trong rủi ro đối với người bệnh trong can thiệp ngoại và/hoặc nội khoa	
Y88.2	Di chứng của các biến cố bất lợi liên quan đến thiết bị y tế sử dụng trong chẩn đoán và/hoặc điều trị	
Y88.3	Di chứng của can thiệp ngoại và/hoặc nội khoa là nguyên nhân gây phản ứng bất thường ở người bệnh, hoặc biến chứng về sau, không đề cập đến sai sót chuyên môn	
Y89	Di chứng của các nguyên nhân bên ngoài khác	
Y89.0	Di chứng của can thiệp pháp luật	
Y89.1	Di chứng của hoạt động chiến tranh	
Y89.9	Di chứng của nguyên nhân bên ngoài không xác định	
Y90	Bằng chứng về tác động do rượu xác định bởi nồng độ cồn trong máu	
Y90.0	Nồng độ cồn trong máu dưới 20 mg/100 ml	
Y90.1	Nồng độ cồn trong máu ở 20-39mg/100 ml	
Y90.2	Nồng độ cồn trong máu ở 40-59mg/100 ml	
Y90.3	Nồng độ cồn trong máu ở 60-79mg/100 ml	
Y90.4	Nồng độ cồn trong máu ở 80-99mg/100 ml	
Y90.5	Mức cồn trong máu ở 100-119mg/100 ml	
Y90.6	Nồng độ cồn trong máu ở 120-119mg/100 ml	
Y90.7	Nồng độ cồn trong máu ở 220-239mg/100 ml	
Y90.8	Nồng độ cồn trong máu ở 240mg/ml hay hơn	
Y90.9	Có cồn trong máu, không xác định nồng độ	
Y91	Bằng chứng về tác động do rượu xác định bằng mức độ nhiễm độc	
Y91.0	Nhiễm độc rượu mức độ nhẹ	
Y91.1	Nhiễm độc rượu mức độ vừa	
Y91.2	Nhiễm độc rượu nghiêm trọng	
Y91.3	Nhiễm độc rượu rất nghiêm trọng	
Y91.9	Tác động do rượu, không xác định khác	Bị nghi ngờ liên quan đến rượu không xác định khác
Y95	Nhiễm trùng mắc phải tại bệnh viện	
Y96	Bệnh mắc phải liên quan đến công việc	
Y97	Bệnh mắc phải liên quan đến ô nhiễm môi trường	
Y98	Bệnh mắc phải liên quan đến lối sống	
Z00	Khám lâm sàng và/hoặc cận lâm sàng cho những người không có than phiền về sức khỏe và/hoặc chẩn đoán được ghi nhận	
Z00.0	Khám sức khỏe tổng quát	
Z00.1	Khám sức khỏe thường quy cho trẻ em	
Z00.2	Khám thời kỳ phát triển nhanh của trẻ em	
Z00.3	Khám tình trạng phát triển ở trẻ vị thành niên	Tình trạng phát triển tuổi dậy thì
Z00.4	Khám tâm thần tổng quát, không phân loại mục khác	
Z00.5	Khám cho người muốn hiến tạng và/hoặc mô	
Z00.6	Khám để so sánh bình thường và/hoặc đối chứng trong chương trình nghiên cứu lâm sàng	
Z00.8	Khám lâm sàng khác	Khám sức khỏe trong điều tra dân số
Z01	Khám lâm sàng và/hoặc cận lâm sàng chuyên khoa cho những người không có than phiền về sức khỏe hoặc chẩn đoán được ghi nhận	
Z01.0	Khám mắt và/hoặc kiểm tra thị lực	
Z01.1	Khám tai và/hoặc kiểm tra thính lực	
Z01.2	Khám răng	
Z01.3	Đo huyết áp	
Z01.4	Khám phụ khoa (tổng quát) (thường quy)	
Z01.5	Chẩn đoán về da và/hoặc các test về độ nhạy cảm	Test dị ứng
Z01.6	Chẩn đoán hình ảnh, không phân loại mục khác	
Z01.7	Xét nghiệm cận lâm sàng	
Z01.8	Khám chuyên khoa xác định khác	
Z01.9	Khám chuyên khoa, không xác định	
Z02	Khám vì lý do hành chính	
Z02.0	Khám để nhập trường học	
Z02.1	Khám trước khi nhận công tác	
Z02.2	Khám để nhập vào nơi cư trú tập trung	
Z02.3	Khám tuyển quân cho các lực lượng vũ trang	
Z02.4	Khám để thi lấy giấy phép lái xe	
Z02.5	Khám để tham gia thể thao	
Z02.6	Khám vì mục đích bảo hiểm	
Z02.7	Cấp giấy chứng nhận y tế	
Z02.8	Khám khác vì lý do hành chính	
Z02.9	Khám vì mục đích hành chính, không xác định	
Z03	Theo dõi và đánh giá y tế đối với các bệnh và/hoặc bệnh lý nghi ngờ, bệnh được loại trừ	
Z03.0	Theo dõi khi nghi ngờ mắc bệnh lao	
Z03.1	Theo dõi khi nghi ngờ u ác tính	
Z03.2	Theo dõi khi nghi ngờ các rối loạn tâm thần và/hoặc hành vi	
Z03.3	Theo dõi khi nghi ngờ có rối loạn hệ thần kinh	
Z03.4	Theo dõi khi nghi ngờ có nhồi máu cơ tim	
Z03.5	Theo dõi khi nghi ngờ có các bệnh tim mạch khác	
Z03.6	Theo dõi khi nghi ngờ tác dụng độc của chất được nuốt vào	
Z03.8	Theo dõi khi nghi ngờ mắc bệnh và/hoặc bệnh lý khác	
Z03.9	Theo dõi bệnh hoặc bệnh lý nghi ngờ, không xác định	
Z04	Khám và/hoặc theo dõi vì những lý do khác	
Z04.0	Xét nghiệm phát hiện rượu và/hoặc ma túy trong máu	
Z04.1	Khám và/hoặc theo dõi sau tai nạn giao thông	
Z04.2	Khám và/hoặc theo dõi sau tai nạn lao động	
Z04.3	Khám và/hoặc theo dõi sau tai nạn khác	
Z04.4	Khám và/hoặc theo dõi sau cáo buộc hiếp dâm và/hoặc dụ dỗ	Khám nạn nhân hoặc bị cáo sau cáo buộc hiếp dâm hoặc dụ dỗ
Z04.5	Khám và/hoặc theo dõi sau vụ gây thương tích khác	Khám nạn nhân hoặc thủ phạm sau vụ gây thương tích khác
Z04.6	Khám tâm thần tổng quát, theo yêu cầu của cơ quan có thẩm quyền	
Z04.8	Khám và/hoặc theo dõi vì lý do xác định khác	Theo yêu cầu cấp bằng chứng chuyên môn
Z04.9	Khám và/hoặc quan sát vì lý do không xác định	Quan sát không xác định khác
Z08	Tái khám sau điều trị u ác tính	
Z08.0	Tái khám sau phẫu thuật u ác tính	
Z08.1	Tái khám sau xạ trị liệu u ác tính	
Z08.2	Tái khám sau hóa trị liệu u ác tính	
Z08.7	Tái khám sau điều trị kết hợp bệnh u ác tính	
Z08.8	Tái khám sau điều trị u ác tính khác	
Z08.9	Tái khám sau điều trị u ác tính không xác định	
Z09	Tái khám sau điều trị bệnh lý ngoài u ác tính	
Z09.0	Tái khám sau phẫu thuật bệnh lý khác	
Z09.1	Tái khám sau xạ trị liệu bệnh lý khác	
Z09.2	Tái khám sau hóa trị liệu bệnh lý khác	
Z09.3	Tái khám sau trị liệu tâm lý	
Z09.4	Tái khám sau điều trị gãy xương	
Z09.7	Tái khám sau điều trị kết hợp đối với bệnh lý khác	
Z09.8	Tái khám sau điều trị khác đối với bệnh lý khác	
Z09.9	Tái khám sau điều trị không xác định đối với bệnh lý khác	
Z10	Khám sức khỏe tổng quát định kỳ cho nhóm đối tượng xác định	
Z10.0	Khám sức khỏe nghề nghiệp	
Z10.1	Kiểm tra sức khỏe tổng quát thường quy cho người cư trú tâp trung	
Z10.2	Kiểm tra sức khỏe tổng quát thường quy của lực lượng vũ trang	
Z10.3	Kiểm tra sức khỏe tổng quát thường quy cho đội thể thao	
Z10.8	Kiểm tra sức khỏe tổng quát thường quy cho nhóm đối tượng xác định khác	Học sinh|Sinh viên
Z11	Khám sàng lọc chuyên khoa bệnh nhiễm trùng và/hoặc ký sinh trùng	
Z11.0	Khám sàng lọc chuyên khoa các bệnh nhiễm trùng đường ruột	
Z11.1	Khám sàng lọc chuyên khoa bệnh lao hô hấp	
Z11.2	Khám sàng lọc chuyên khoa các bệnh nhiễm khuẩn khác	
Z11.3	Khám sàng lọc chuyên khoa về các bệnh nhiễm trùng lây truyền chủ yếu qua đường tình dục	
Z11.4	Khám sàng lọc chuyên khoa bệnh nhiễm virus suy giảm miễn dịch ở người [HIV]	
Z11.5	Khám sàng lọc chuyên khoa bệnh nhiễm virus khác	
Z11.6	Khám sàng lọc chuyên khoa bệnh do động vật đơn bào và/hoặc giun sán khác	
Z11.8	Khám sàng lọc chuyên khoa bệnh nhiễm trùng và/hoặc ký sinh trùng khác	Bệnh do nhiễm Chlamydia|Bệnh còi xương|Bệnh do nhiễm xoắn khuẩn|Bệnh do nhiễm nấm
Z11.9	Khám sàng lọc chuyên khoa bệnh nhiễm trùng và/hoặc ký sinh trùng, không xác định	
Z12	Khám sàng lọc chuyên khoa u tân sinh	
Z12.0	Khám sàng lọc chuyên khoa u tân sinh dạ dày	
Z12.1	Khám sàng lọc chuyên khoa u tân sinh đường ruột	
Z12.2	Khám sàng lọc chuyên khoa u tân sinh cơ quan hô hấp	
Z12.3	Khám sàng lọc chuyên khoa u tân sinh vú	
Z12.4	Khám sàng lọc chuyên khoa u tân sinh cổ tử cung	
Z12.5	Khám sàng lọc chuyên khoa u tân sinh tiền liệt tuyến	
Z12.6	Khám sàng lọc chuyên khoa u tân sinh bàng quang	
Z12.8	Khám sàng lọc chuyên khoa u tân sinh ở vị trí khác	
Z12.9	Khám sàng lọc chuyên khoa u tân sinh, không xác định	
Z13	Khám sàng lọc chuyên khoa bệnh và/hoặc rối loạn khác	
Z13.0	Khám sàng lọc chuyên khoa bệnh về máu và/hoặc cơ quan tạo máu và/hoặc một số rối loạn liên quan đến cơ chế miễn dịch	
Z13.1	Khám sàng lọc chuyên khoa bệnh đái tháo đường	
Z13.2	Khám sàng lọc chuyên khoa rối loạn dinh dưỡng	
Z13.3	Khám sàng lọc chuyên khoa rối loạn tâm thần và/hoặc hành vi	Nghiện rượu|Trầm cảm|Chậm phát triển trí tuệ
Z13.4	Khám sàng lọc chuyên khoa một số rối loạn phát triển của trẻ em	
Z13.5	Khám sàng lọc chuyên khoa rối loạn thị giác và/hoặc thính giác	
Z13.6	Khám sàng lọc chuyên khoa rối loạn tim mạch	
Z13.7	Khám sàng lọc chuyên khoa dị tật, biến dạng và/hoặc bất thường nhiễm sắc thể bẩm sinh	
Z13.8	Khám sàng lọc chuyên khoa bệnh và/hoặc rối loạn xác định khác	
Z13.9	Khám sàng lọc chuyên khoa, không xác định	
Z20	Tiếp xúc và/hoặc phơi nhiễm với bệnh truyền nhiễm	
Z20.0	Tiếp xúc và/hoặc phơi nhiễm với bệnh nhiễm trùng đường ruột	
Z20.1	Tiếp xúc và/hoặc phơi nhiễm với bệnh lao	
Z20.2	Tiếp xúc và/hoặc phơi nhiễm với bệnh nhiễm trùng lây truyền chủ yếu qua đường tình dục	
Z20.3	Tiếp xúc và/hoặc phơi nhiễm với virus gây bệnh dại	
Z20.4	Tiếp xúc và/hoặc phơi nhiễm với virus rubella	
Z20.5	Tiếp xúc và/hoặc phơi nhiễm với virus viêm gan	
Z20.6	Tiếp xúc và/hoặc phơi nhiễm với virus gây suy giảm miễn dịch ở người (HIV)	
Z20.7	Tiếp xúc và/hoặc phơi nhiễm với chấy rận, giun đũa và/hoặc ký sinh trùng khác	
Z20.8	Tiếp xúc và/hoặc phơi nhiễm với bệnh truyền nhiễm khác	
Z20.9	Tiếp xúc và/hoặc phơi nhiễm với bệnh truyền nhiễm không xác định	
Z21	Tình trạng nhiễm virus suy giảm miễn dịch ở người [HIV] không có triệu chứng	
Z22	Người mang mầm bệnh nhiễm trùng	
Z22.0	Người mang mầm bệnh thương hàn	
Z22.1	Người mang mầm bệnh nhiễm trùng đường ruột khác	
Z22.2	Người mang mầm bệnh bạch hầu	
Z22.3	Người mang mầm bệnh nhiễm khuẩn xác định khác	
Z22.4	Người mang mầm bệnh nhiễm trùng lây truyền chủ yếu qua đường tình dục	
Z22.6	Người mang mầm virus T-lymphotropic típ 1 [HTLV.1 ] gây bệnh ở người	
Z22.7	Người nhiễm lao tiềm ẩn	
Z22.8	Người mang mầm bệnh nhiễm trùng khác	
Z22.9	Người mang mầm bệnh nhiễm trùng, không xác định	
Z23	Nhu cầu tiêm chủng phòng các bệnh nhiễm khuẩn đơn lẻ	
Z23.0	Nhu cầu tiêm chủng phòng bệnh tả đơn lẻ	
Z23.1	Nhu cầu tiêm chủng phòng bệnh thương hàn - phó thương hàn đơn lẻ [TAB]	
Z23.2	Nhu cầu tiêm chủng phòng bệnh lao [BCG]	
Z23.3	Nhu cầu tiêm chủng phòng bệnh dịch hạch	
Z23.4	Nhu cầu tiêm chủng phòng bệnh tularemia	
Z23.5	Nhu cầu tiêm chủng phòng bệnh uốn ván đơn lẻ	
Z23.6	Nhu cầu tiêm chủng phòng bệnh bạch hầu đơn lẻ	
Z23.7	Nhu cầu tiêm chủng phòng bệnh ho gà đơn lẻ	
Z23.8	Nhu cầu tiêm chủng phòng các bệnh nhiễm khuẩn đơn lẻ khác	
Z24	Nhu cầu tiêm chủng phòng một số bệnh do nhiễm virus đơn lẻ	
Z24.0	Nhu cầu tiêm chủng phòng bệnh bại liệt	
Z24.1	Nhu cầu tiêm chủng phòng viêm não virus do tiết túc truyền [virus arbo]	
Z24.2	Nhu cầu tiêm chủng phòng bệnh dại	
Z24.3	Nhu cầu tiêm chủng phòng bệnh sốt vàng da	
Z24.4	Nhu cầu tiêm chủng phòng bệnh sởi đơn lẻ	
Z24.5	Nhu cầu tiêm chủng phòng bệnh rubella đơn lẻ	
Z24.6	Nhu cầu tiêm chủng phòng bệnh viêm gan do virus	
Z25	Nhu cầu tiêm chủng phòng bệnh do nhiễm virus đơn lẻ khác	
Z25.0	Nhu cầu tiêm chủng phòng bệnh quai bị đơn lẻ	
Z25.1	Nhu cầu tiêm chủng phòng bệnh cúm	
Z25.8	Nhu cầu tiêm chủng phòng các bệnh virus đơn lẻ xác định khác	
Z26	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng đơn lẻ khác	
Z26.0	Nhu cầu tiêm chủng phòng bệnh do nhiễm leishmania	
Z26.8	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng đơn lẻ xác định khác	
Z26.9	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng, không xác định	Cần tiêm chủng không xác định khác
Z27	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng kết hợp	
Z27.0	Nhu cầu tiêm chủng phòng tả kèm thương hàn - phó thương hàn [tả + TAB]	
Z27.1	Nhu cầu tiêm chủng phòng bạch hầu - uốn ván - ho gà phối hợp [DTP]	
Z27.2	Nhu cầu tiêm chủng phòng bạch hầu - uốn ván - ho gà kèm thương hàn - phó thương hàn [DPT + TAB]	
Z27.3	Nhu cầu tiêm chủng phòng bạch hầu - uốn ván - ho gà kèm bại liệt [DPT+ bại liệt]	
Z27.4	Nhu cầu tiêm chủng phòng bệnh sởi - quai bị - rubella [MMR]	
Z27.8	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng kết hợp khác	
Z27.9	Nhu cầu tiêm chủng phòng các bệnh nhiễm trùng kết hợp không xác định	
Z28	Không thực hiện được tiêm chủng	
Z28.0	Không thực hiện được tiêm chủng do chống chỉ định	
Z28.1	Không thực hiện được tiêm chủng vì người bệnh chưa quyết định do tín ngưỡng hay áp lực của nhóm	
Z28.2	Không thực hiện được tiêm chủng vì quyết định của người bệnh do lý do khác và/hoặc không xác định	
Z28.8	Không thực hiện được tiêm chủng vì lý do khác	
Z28.9	Không thực hiện được tiêm chủng vì lý do không xác định	
Z29	Nhu cầu thực hiện biện pháp dự phòng khác	
Z29.0	Cách ly	
Z29.1	Miễn dịch liệu pháp dự phòng	Điều trị bằng globulin miễn dịch
Z29.2	Hóa trị liệu dự phòng khác	Hóa trị liệu dự phòng|Liệu pháp kháng sinh dự phòng
Z29.8	Can thiệp dự phòng xác định khác	
Z29.9	Can thiệp dự phòng, không xác định	
Z30	Quản lý biện pháp tránh thai	
Z30.0	Tư vấn chung và/hoặc tư vấn về các biện pháp tránh thai	Tư vấn kế hoạch hóa gia đình không xác định khác|Chỉ dẫn ban đầu về tránh thai
Z30.1	Đặt dụng cụ tránh thai (trong tử cung)	
Z30.2	Triệt sản	Nhập viện để thắt vòi trứng hoặc thắt ống dẫn tinh
Z30.3	Hút điều hòa kinh nguyệt	Đình chỉ phôi thai|Điều hòa kinh nguyệt
Z30.4	Theo dõi sử dụng thuốc tránh thai	Kê lại đơn viên uống tránh thai hoặc thuốc tránh thai khác|Khám thường quy để duy trì biện pháp tránh thai
Z30.5	Theo dõi dụng cụ tránh thai (trong tử cung)	
Z30.8	Quản lý biện pháp tránh thai khác	Đếm tinh trùng sau phẫu thuật cắt bỏ ống dẫn tinh
Z30.9	Quản lý biện pháp tránh thai, không xác định	
Z31	Quản lý hỗ trợ sinh sản	
Z31.0	Tái tạo vòi trứng hay ống dẫn tinh sau khi triệt sản trước đó	
Z31.1	Thụ tinh nhân tạo	
Z31.2	Thụ tinh trong ống nghiệm	Nhập viện để thụ tinh hoặc cấy ghép buồng trứng
Z31.3	Phương pháp hỗ trợ sinh sản khác	
Z31.4	Thăm dò và/hoặc xét nghiệm khả năng sinh sản	
Z31.5	Tư vấn di truyền học	
Z31.6	Tư vấn và khuyến bảo liên quan hỗ trợ sinh sản	
Z31.8	Quản lý hỗ trợ sinh sản khác	
Z31.9	Quản lý hỗ trợ sinh sản, không xác định	
Z32	Thăm khám và/hoặc xét nghiệm xác định có thai	
Z32.0	Có thai, chưa được khẳng định	
Z32.1	Có thai đã được khẳng định	
Z33	Trạng thái mang thai, mã phụ trợ	
Z34	Theo dõi thai kỳ bình thường	
Z34.0	Theo dõi thai kỳ bình thường con đầu lòng	
Z34.8	Theo dõi thai kỳ bình thường khác	
Z34.9	Theo dõi thai kỳ bình thường, không xác định	
Z35	Theo dõi thai phụ có thai kỳ nguy cơ cao	
Z35.0	Theo dõi thai phụ có tiền sử vô sinh	
Z35.1	Theo dõi thai phụ có tiền sử sảy thai	
Z35.2	Theo dõi thai phụ có tiền sử sinh sản hoặc sản khoa khó khăn	
Z35.3	Theo dõi thai phụ có tiền sử chăm sóc trước sinh không đầy đủ	
Z35.4	Theo dõi thai phụ đã đẻ nhiều lần	
Z35.5	Theo dõi thai phụ nhiều tuổi mang thai lần đầu	
Z35.6	Theo dõi thai phụ quá trẻ tuổi mang thai lần đầu	
Z35.7	Theo dõi thai phụ có nguy cơ cao do vấn đề xã hội	
Z35.8	Theo dõi thai phụ có nguy cơ cao khác	
Z35.9	Theo dõi thai phụ có nguy cơ cao, không xác định	
Z36	Khám sàng lọc trước sinh	
Z36.0	Khám sàng lọc trước sinh về bất thường nhiễm sắc thể	
Z36.1	Khám sàng lọc trước sinh về mức alphafetoprotein tăng	
Z36.2	Khám sàng lọc trước sinh khác dựa vào chọc rò màng ối qua bụng	
Z36.3	Khám sàng lọc dị tật trước sinh qua siêu âm hay phương pháp khám thực thể khác	
Z36.4	Khám sàng lọc trước sinh về thai chậm lớn bằng siêu âm hoặc các phương pháp khám thực thể khác	
Z36.5	Khám sàng lọc trước sinh về miễn dịch đồng loại	
Z36.8	Khám sàng lọc trước sinh khác	Khám sàng lọc bệnh lý hemoglobin
Z36.9	Khám sàng lọc trước sinh, không xác định	
Z37	Kết quả sinh đẻ	
Z37.0	Đẻ một thai, trẻ sống	
Z37.1	Đẻ một thai, chết lưu	
Z37.2	Đẻ song thai, cả hai trẻ sống	
Z37.3	Đẻ song thai, một trẻ sinh ra sống, một trẻ chết lưu	
Z37.4	Đẻ song thai, cả hai trẻ chết lưu	
Z37.5	Đẻ đa thai, tất cả đều sống	
Z37.6	Đẻ đa thai, trong đó một số trẻ sinh ra sống	
Z37.7	Đẻ đa thai, tất cả đều chết lưu	
Z37.9	Kết quả sinh đẻ, không xác định	Sinh đa thai không xác định khác|Sinh một thai không xác định khác
Z38	Trẻ sinh ra sống theo nơi sinh	
Z38.0	Trẻ sinh ra sống, một con, tại bệnh viện	
Z38.1	Trẻ sinh ra sống, một con, bên ngoài bệnh viện	
Z38.2	Trẻ sinh ra sống, một con, không xác định nơi sinh	Trẻ sinh ra sống không xác định khác
Z38.3	Trẻ sinh ra sống, sinh đôi, tại bệnh viện	
Z38.4	Trẻ sinh ra sống, sinh đôi, bên ngoài bệnh viện	
Z38.5	Trẻ sinh ra sống, sinh đôi, không xác định nơi sinh	
Z38.6	Trẻ sinh ra sống, sinh nhiều con, tại bệnh viện	
Z38.7	Trẻ sinh ra sống, sinh nhiều con, bên ngoài bệnh viện	
Z38.8	Trẻ sinh ra sống, sinh nhiều con, không xác định nơi sinh	
Z39	Chăm sóc và/hoặc thăm khám sau sinh	
Z39.0	Chăm sóc và/hoặc thăm khám ngay sau khi sinh	
Z39.1	Chăm sóc và/hoặc thăm khám người mẹ cho con bú	
Z39.2	Tái khám định kỳ sau sinh	
Z40	Phẫu thuật dự phòng	
Z40.0	Phẫu thuật dự phòng nguy cơ liên quan đến khối u ác tính	Nhập viện để cắt bỏ tạng dự phòng
Z40.8	Phẫu thuật dự phòng khác	
Z40.9	Phẫu thuật dự phòng, không xác định	
Z41	Can thiệp vì mục đích khác ngoài phục hồi tình trạng sức khỏe	
Z41.0	Cấy tóc	
Z41.1	Phẫu thuật tạo hình khác do diện mạo bên ngoài không như mong muốn	
Z41.2	Cắt bao quy đầu theo nghi tục và/hoặc thường quy	
Z41.3	Bấm lỗ tai	
Z41.8	Can thiệp khác vì mục đích khác ngoài phục hồi tình trạng sức khỏe	
Z41.9	Can thiệp vì mục đích khác ngoài phục hồi tình trạng sức khỏe, không xác định	
Z42	Chăm sóc y tế tiếp có can thiệp tạo hình	
Z42.0	Chăm sóc y tế tiếp có can thiệp tạo hình ở đầu và/hoặc cổ	
Z42.1	Chăm sóc y tế tiếp có can thiệp tạo hình ở ngực	
Z42.2	Chăm sóc y tế tiếp có can thiệp tạo hình ở phần khác của thân	
Z42.3	Chăm sóc y tế tiếp có can thiệp tạo hình ở chi trên	
Z42.4	Chăm sóc y tế tiếp có can thiệp tạo hình ở chi dưới	
Z42.8	Chăm sóc y tế tiếp có can thiệp tạo hình ở phần khác của người bệnh	
Z42.9	Chăm sóc y tế tiếp có can thiệp tạo hình, không xác định	
Z43	Chăm sóc lỗ mở nhân tạo	
Z43.0	Chăm sóc lỗ mở khí quản	
Z43.1	Chăm sóc lỗ mở thông dạ dày	
Z43.2	Chăm sóc lỗ mở thông hồi tràng	
Z43.3	Chăm sóc lỗ mở thông đại tràng	
Z43.4	Chăm sóc lỗ mở nhân tạo khác của đường tiêu hóa	
Z43.5	Chăm sóc lỗ mở thông bàng quang	
Z43.6	Chăm sóc các lỗ mở nhân tạo khác của đường tiết niệu	Lỗ mở thông thận|Lỗ mở niệu đạo|Lỗ mở niệu quản
Z43.7	Chăm sóc âm đạo nhân tạo	
Z43.8	Chăm sóc lỗ mở nhân tạo khác	
Z43.9	Chăm sóc lỗ mở thông nhân tạo không xác định	
Z44	Lắp và/hoặc điều chỉnh bộ phận giả bên ngoài	
Z44.0	Lắp và/hoặc điều chỉnh tay giả (toàn bộ) (một phần)	
Z44.1	Lắp và/hoặc điều chỉnh chân giả (toàn bộ) (một phần)	
Z44.2	Lắp và/hoặc điều chỉnh mắt giả	
Z44.3	Lắp và/hoặc điều chỉnh ngực giả bên ngoài	
Z44.8	Lắp và/hoặc điều chỉnh bộ phận giả bên ngoài khác	
Z44.9	Lắp và/hoặc điều chỉnh bộ phận giả bên ngoài, không xác định	
Z45	Điều chỉnh thiết bị cấy ghép	
Z45.0	Điều chỉnh và quản lý hoạt động thiết bị trợ tim	Kiểm tra và thử thiết bị tim
Z45.1	Điều chỉnh vả quản lý hoạt động bơm truyền dịch điện	
Z45.2	Điều chỉnh và quản lý hoạt động thiết bị truy cập mạch máu	
Z45.3	Điều chỉnh và quản lý hoạt động thiết bị thính giác được cấy ghép	Thiết bị dẫn truyền xương|Thiết bị ốc tai
Z45.8	Điều chỉnh và quản lý hoạt động thiết bị cấy ghép khác	
Z45.9	Điều chỉnh và quản lý hoạt động thiết bị cấy ghép không xác định	
Z46	Điều chỉnh thiết bị khác	
Z46.0	Điều chỉnh kính thuốc và/hoặc kính áp tròng	
Z46.1	Điều chỉnh thiết bị trợ thính	
Z46.2	Điều chỉnh thiết bị khác liên quan tới hệ thần kinh và/hoặc giác quan đặc biệt [thị lực, thính lực, vị giác, khứu giác]	
Z46.3	Điều chỉnh dụng cụ phục hình răng miệng	
Z46.4	Điều chỉnh thiết bị chỉnh nha	
Z46.5	Điều chỉnh ống mở thông hỗng tràng và/hoặc thiết bị khác ở ruột non	
Z46.6	Điều chỉnh thiết bị tiết niệu	
Z46.7	Điều chỉnh thiết bị chỉnh hình	
Z46.8	Điều chỉnh thiết bị xác định khác	Xe lăn
Z46.9	Điều chỉnh thiết bị không xác định	
Z47	Tái khám sau chỉnh hình khác	
Z47.0	Tái khám liên quan tháo bỏ nẹp xương gãy và/hoặc thiết bị cố định bên trong khác	
Z47.8	Tái khám sau chỉnh hình xác định khác	
Z47.9	Tái khám sau chỉnh hình, không xác định	
Z48	Chăm sóc sau phẫu thuật khác	
Z48.0	Chăm sóc liên quan băng và/hoặc vết khâu phẫu thuật	Thay băng|Cắt chỉ
Z48.8	Chăm sóc sau phẫu thuật xác định khác	
Z48.9	Chăm sóc sau phẫu thuật, không xác định	
Z49	Chăm sóc liên quan đến lọc máu nhân tạo	
Z49.0	Chăm sóc chuẩn bị cho lọc máu nhân tạo	
Z49.1	Lọc máu ngoài cơ thể	
Z49.2	Lọc máu khác	Thẩm phân phúc mạc [lọc máu qua màng bụng]
Z50	Chăm sóc có sử dụng kỹ thuật phục hồi chức năng	
Z50.0	Phục hồi chức năng tim	
Z50.1	Vật lý trị liệu khác	Bài tập trị liệu và phục hồi sức khỏe
Z50.2	Cai rượu	
Z50.3	Cai nghiện ma túy	
Z50.4	Liệu pháp tâm lý, không phân loại mục khác	
Z50.5	Liệu pháp ngôn ngữ	
Z50.6	Huấn luyện thị giác	
Z50.7	Hoạt động trị liệu và phục hồi chức năng lao động, không phân loại mục khác	
Z50.8	Chăm sóc liên quan đến sử dụng kỹ thuật phục hồi chức năng khác	Cai nghiện thuốc lá|Đào tạo về các hoạt động sinh hoạt hàng ngày [ADL] không phân loại mục khác
Z50.9	Chăm sóc liên quan đến sử dụng kỹ thuật phục hồi chức năng, không xác định	Phục hồi chức năng không xác định khác
Z51	Chăm sóc y tế khác	
Z51.0	Đợt xạ trị liệu	
Z51.1	Đợt hóa trị liệu điều trị u tân sinh	
Z51.2	Hóa trị liệu khác	
Z51.3	Truyền máu (không ghi nhận chẩn đoán)	
Z51.4	Chăm sóc chuẩn bị cho điều trị tiếp theo, không phân loại mục khác	
Z51.5	Chăm sóc giảm nhẹ	
Z51.6	Giải mẫn cảm với các dị nguyên	
Z51.8	Chăm sóc y tế xác định khác	
Z51.9	Chăm sóc y tế, không xác định	
Z52	Hiến tạng và/hoặc mô	
Z52.0	Hiến máu	Các thành phần máu như bạch cầu lympho, tiểu cầu hoặc tế bào gốc
Z52.1	Hiến da	
Z52.2	Hiến xương	
Z52.3	Hiến tủy xương	
Z52.4	Hiến thận	
Z52.5	Hiến giác mạc	
Z52.6	Hiến gan	
Z52.7	Hiến tim	
Z52.8	Hiến tạng và/hoặc hiến mô khác	
Z52.9	Hiến tạng hoặc hiến mô không xác định	Hiến tạng không xác định khác
Z53	Người bệnh đến cơ sở y tế để thực hiện can thiệp xác định cụ thể, không thực hiện được	
Z53.0	Không thực hiện can thiệp vì chống chỉ định	
Z53.1	Không thực hiện can thiệp vì quyết định của người bệnh liên quan tín ngưỡng và/hoặc áp lực nhóm	
Z53.2	Không thực hiện can thiệp vì quyết định của người bệnh do lý do khác và/hoặc không xác định	
Z53.8	Không thực hiện can thiệp vì những lý do khác	
Z53.9	Không thực hiện can thiệp, vì lý do không xác định	
Z54	Dưỡng sức	
Z54.0	Dưỡng sức sau phẫu thuật	
Z54.1	Dưỡng sức sau xạ trị liệu	
Z54.2	Dưỡng sức sau hóa trị liệu	
Z54.3	Dưỡng sức sau liệu pháp tâm lý	
Z54.4	Dưỡng sức sau điều trị gãy xương	
Z54.7	Dưỡng sức sau điều trị kết hợp	Dưỡng sức sau bất kỳ điều trị kết hợp nào được phân loại vào Z54.0-Z54.4
Z54.8	Dưỡng sức sau điều trị khác	
Z54.9	Dưỡng sức sau điều trị không xác định	
Z55	Những vấn đề liên quan đến giáo dục và/hoặc kỹ năng đọc viết	
Z55.0	Mù chữ và/hoặc kỹ năng đọc viết mức thấp	
Z55.1	Không có trường học và/hoặc không thể đến trường	
Z55.2	Thi trượt	
Z55.3	Không đạt kết quả học tập ở trường	
Z55.4	Bất hoà với giáo viên và/hoặc bạn cùng lớp	
Z55.8	Vấn đề khác liên quan đến giáo dục và/hoặc kỹ năng đọc viết	Giảng dạy không đầy đủ
Z55.9	Vấn đề liên quan đến giáo dục và/hoặc đọc viết, không xác định	
Z56	Những vấn đề liên quan đến việc làm và/hoặc thất nghiệp	
Z56.0	Thất nghiệp, không xác định	
Z56.1	Thay đổi công việc	
Z56.2	Sợ mất việc	
Z56.3	Lịch làm việc căng thẳng	
Z56.4	Bất hoà với chủ và/hoặc đồng nghiệp	
Z56.5	Công việc không phù hợp	Điều kiện làm việc khó khăn
Z56.6	Căng thẳng thể chất và/hoặc tinh thần liên quan đến công việc	
Z56.7	Vấn đề liên quan việc làm khác và/ hoặc không xác định	
Z57	Phơi nhiễm nghề nghiệp với các yếu tố nguy cơ	
Z57.0	Phơi nhiễm nghề nghiệp với tiếng ồn	
Z57.1	Phơi nhiễm nghề nghiệp với nguồn phóng xạ [bức xạ] [tia xạ]	
Z57.2	Phơi nhiễm nghề nghiệp với bụi	
Z57.3	Phơi nhiễm nghề nghiệp với ô nhiễm không khí	
Z57.4	Phơi nhiễm nghề nghiệp với tác nhân gây độc trong nông nghiệp	Chất rắn, chất lỏng, khí hoặc hơi
Z57.5	Phơi nhiễm nghề nghiệp với tác nhân gây độc trong ngành khác	Chất rắn, chất lỏng, khí hoặc hơi
Z57.6	Phơi nhiễm nghề nghiệp với nhiệt độ cao	
Z57.7	Phơi nhiễm nghề nghiệp với độ rung	
Z57.8	Phơi nhiễm nghề nghiệp với yếu tố nguy cơ khác	
Z57.9	Phơi nhiễm nghề nghiệp với yếu tố nguy cơ không xác định	
Z58	Vấn đề liên quan đến môi trường sống	
Z58.0	Phơi nhiễm với tiếng ồn	
Z58.1	Phơi nhiễm ô nhiễm không khí	
Z58.2	Phơi nhiễm với nước ô nhiễm	
Z58.3	Phơi nhiễm với đất ô nhiễm	
Z58.4	Phơi nhiễm với nguồn phóng xạ [bức xạ] [tia phóng xạ]	
Z58.5	Phơi nhiễm với loại ô nhiễm khác	
Z58.6	Cung cấp nước uống không đầy đủ	
Z58.7	Phơi nhiễm với khói thuốc lá	
Z58.8	Vấn đề khác liên quan đến môi trường sống	
Z58.9	Vấn đề liên quan đến môi trường sống, không xác định	
Z59	Những vấn đề liên quan đến nhà ở và/hoặc hoàn cảnh kinh tế	
Z59.0	Tình trạng vô gia cư	
Z59.1	Nơi cư trú thiếu tiện nghi	
Z59.2	Bất hoà với hàng xóm, người thuê nhà và/hoặc chủ nhà	
Z59.3	Những vấn đề liên quan đến sinh sống tại nơi cư trú tập trung	
Z59.4	Thiếu thực phẩm	
Z59.5	Quá nghèo	
Z59.6	Thu nhập thấp	
Z59.7	Trợ cấp xã hội và/hoặc bảo hiểm xã hội không đảm bảo	
Z59.8	Những vấn đề khác liên quan đến nhà ở và/hoặc hoàn cảnh kinh tế	Phát mại tài sản thế chấp vay|Nhà ở cô lập, xa xôi hẻo lánh|Vấn đề với chủ nợ
Z59.9	Vấn đề liên quan đến nhà ở và/hoặc hoàn cảnh kinh tế, không xác định	
Z60	Vấn đề liên quan đến môi trường xã hội	
Z60.0	Vấn đề thích nghi chuyển tiếp vòng đời	Thích nghi với việc về hưu|Hội chứng tổ ấm trống trải
Z60.1	Hoàn cảnh nuôi dạy con cái không điển hình	
Z60.2	Sống đơn độc	
Z60.3	Khó khăn hoặc thiếu khả năng thích nghi môi trường hoặc văn hóa mới	Di cư|Chuyển từ một nhóm xã hội sang nhóm khác
Z60.4	Đối tượng bị xã hội loại trừ và/hoặc bác bỏ	
Z60.5	Đối tượng đích cảm thấy bị phân biệt đối xử bất lợi và/hoặc ngược đãi	
Z60.8	Vấn đề khác liên quan đến môi trường xã hội	
Z60.9	Vấn đề liên quan đến môi trường xã hội, không xác định	
Z61	Vấn đề liên quan đến sự kiện tiêu cực trong cuộc sống thời thơ ấu	
Z61.0	Mất mối quan hệ thân yêu trong thời thơ ấu	
Z61.1	Tách khỏi hộ gia đình trong thời thơ ấu	
Z61.2	Mô hình của mối quan hệ gia đình biến đổi trong thời kỳ thơ ấu	
Z61.3	Những sự kiện làm trẻ mất đi lòng tự trọng trong thời thơ ấu	biết hoặc phát hiện ra một sự việc xấu hoặc đáng hổ thẹn của cá nhân hoặc của gia đình|và những sự việc đáng bẽ mặt khác.
Z61.4	Vấn đề liên quan đến cáo buộc lạm dụng tình dục trẻ em của người trong nhóm hỗ trợ chính	
Z61.5	Vấn đề liên quan đến cáo buộc lạm dụng tình dục trẻ em của một người không thuộc nhóm hỗ trợ chính	
Z61.6	Vấn đề liên quan đến cáo buộc lạm dụng thể xác trẻ em	
Z61.7	Trải nghiệm đáng sợ của cá nhân trong thời thơ ấu	
Z61.8	Sự kiện tiêu cực khác trong cuộc sống thời thơ ấu	
Z61.9	Sự kiện tiêu cực trong cuộc sống thời thơ ấu, không xác định	
Z62	Vấn đề khác liên quan đến nuôi dạy trẻ	
Z62.0	Theo dõi và/hoặc kiểm soát không đầy đủ của cha mẹ	Thiếu hiểu biết của cha mẹ về những gì trẻ đang làm hoặc trẻ đang ở đâu|thiếu kiểm soát|thiếu quan tâm hoặc thiếu nỗ lực can thiệp khi trẻ gặp các tình huống rủi ro.
Z62.1	Cha mẹ bảo vệ quá mức	Mô hình giáo dục dẫn đến nhi tính hóa trẻ và ngăn cản tính tự lập của trẻ.
Z62.2	Trẻ được nuôi dạy trong cơ sở nuôi dưỡng tập trung	
Z62.3	Thái độ thù địch với trẻ và đổ lỗi cho trẻ	
Z62.4	Thái độ thờ ơ cảm xúc với trẻ	
Z62.5	Vấn đề khác liên quan đến thờ ơ trong nuôi dạy trẻ	Thiếu trải nghiệm học tập và vui chơi
Z62.6	Áp đặt không phù hợp của cha mẹ và/hoặc những phẩm chất bất thường khác trong quá trình nuôi dạy trẻ	
Z62.8	Vấn đề xác định khác liên quan tới nuôi dạy trẻ	
Z62.9	Vấn đề liên quan tới nuôi dạy trẻ, không xác định	
Z63	Vấn đề khác liên quan đến nhóm hỗ trợ chính, bao gồm cả hoàn cảnh gia đình	
Z63.0	Vấn đề trong mối quan hệ vợ chồng hoặc bạn đời	
Z63.1	Vấn đề trong mối quan hệ với cha mẹ và/hoặc thông gia	
Z63.2	Sự hỗ trợ không đầy đủ của gia đình	
Z63.3	Thiếu vắng thành viên trong gia đình	
Z63.4	Sự biến mất hay qua đời của thành viên gia đình	Người thân trong gia đình được cho rằng là đã qua đời
Z63.5	Sự tan vỡ của gia đình do ly thân và/hoặc ly hôn	Sự mất cảm tính|ly thân
Z63.6	Người thân sống lệ thuộc có nhu cầu chăm sóc tại nhà	
Z63.7	Sự kiện căng thẳng khác trong cuộc sống ảnh hưởng đến gia đình và/hoặc hộ gia đình	
Z63.8	Vấn đề xác định khác liên quan đến nhóm hỗ trợ chính	Gia đình bất hòa không xác định khác|Mức độ biểu lộ cảm xúc cao trong gia đình|Giao tiếp trong gia đình không đầy đủ hoặc không chuẩn mực
Z63.9	Vấn đề liên quan đến nhóm hỗ trợ chính, không xác định	
Z64	Vấn đề liên quan đến một số hoàn cảnh tâm lý xã hội nhất định	
Z64.0	Vấn đề liên quan đến mang thai không mong muốn	
Z64.1	Vấn đề liên quan đến sinh nhiều con	
Z64.2	Tìm kiếm và/hoặc chấp nhận can thiệp thực thể, dinh dưỡng và hóa học được biết là nguy hiểm và có hại	
Z64.3	Tìm kiếm và/hoặc chấp nhận các can thiệp hành vi và tâm lý được biết là nguy hiểm và có hại	
Z64.4	Bất hoà với người tư vấn	
Z65	Vấn đề liên quan đến hoàn cảnh tâm lý xã hội khác	
Z65.0	Bị kết án trong tố tụng dân sự và/hoặc hình sự, không bị phạt tù	
Z65.1	Tống giam và/hoặc giam cầm khác	
Z65.2	Vấn đề liên quan ra tù	
Z65.3	Vấn đề liên quan đến hoàn cảnh pháp luật khác	Bắt giữ|Khởi kiện quyền nuôi con và nghĩa vụ cấp dưỡng nuôi con|Kiện tụng|Truy tố
Z65.4	Nạn nhân của tội ác và/hoặc khủng bố	Nạn nhân của tra tấn
Z65.5	Phơi nhiễm rủi ro do thảm họa, chiến tranh và/hoặc các hành động thù địch khác	
Z65.8	Vấn đề xác định khác liên quan đến hoàn cảnh tâm lý xã hội	
Z65.9	Vấn đề liên quan đến hoàn cảnh tâm lý xã hội không xác định	
Z70	Tư vấn liên quan đến thái độ, hành vi và/hoặc khuynh hướng tình dục	
Z70.0	Tư vấn liên quan đến thái độ tình dục	
Z70.1	Tư vấn về thái độ và/hoặc khuynh hướng tình dục của người bệnh	
Z70.2	Tư vấn về thái độ và/hoặc khuynh hướng tình dục của người thứ ba	
Z70.3	Tư vấn liên quan đến sự lo lắng hỗn hợp về hành vi, thái độ và/hoặc khuynh hướng tình dục	
Z70.8	Tư vấn khác về tình dục	Giáo dục tình dục
Z70.9	Tư vấn liên quan đến tình dục, không xác định	
Z71	Những người đến cơ sở y tế để được tư vấn y tế và/hoặc tư vấn khác, không phân loại mục khác	
Z71.0	Người thay mặt cho người khác đến tư vấn	
Z71.1	Người than phiền lo sợ những bệnh không được chẩn đoán	
Z71.2	Tư vấn cá nhân để giải thích kết quả cận lâm sàng	
Z71.3	Tư vấn và/hoặc theo dõi chế độ ăn uống	
Z71.4	Tư vấn chống lạm dụng rượu và/hoặc theo dõi lạm dụng rượu	
Z71.5	Tư vấn chống lạm dụng ma túy và/hoặc theo dõi lạm dụng ma túy	
Z71.6	Tư vấn chống lạm dụng thuốc lá	
Z71.7	Tư vấn về virus suy giảm miễn dịch ở người [HIV]	
Z71.8	Tư vấn xác định khác	Tư vấn về huyết thống
Z71.9	Tư vấn, không xác định	Tư vấn y tế không xác định khác
Z72	Vấn đề liên quan đến lối sống	
Z72.0	Sử dụng thuốc lá	
Z72.1	Sử dụng rượu	
Z72.2	Sử dụng ma túy	
Z72.3	Không luyện tập thể dục	
Z72.4	Chế độ ăn uống và/hoặc thói quen ăn uống không phù hợp	
Z72.5	Hành vi tình dục nguy cơ cao	
Z72.6	Đánh bạc và/hoặc cá cược	
Z72.8	Vấn đề khác liên quan đến lối sống	Hành vi tự hủy hoại bản thân
Z72.9	Vấn đề liên quan đến lối sống, không xác định	
Z73	Vấn đề liên quan đến khó khăn trong quản lý cuộc sống	
Z73.0	Kiệt sức	Tình trạng kiệt sức nghiêm trọng
Z73.1	Rối loạn nhấn mạnh nét cá tính quá mức [đến mức bệnh lý]	
Z73.2	Thiếu thư giãn và/hoặc giải trí	
Z73.3	Căng thẳng, không phân loại mục khác	
Z73.4	Thiếu kỹ năng xã hội, không phân loại mục khác	
Z73.5	Xung đột vai trò xã hội, không phân loại mục khác	
Z73.6	Hạn chế các hoạt động do khuyết tật	
Z73.8	Vấn đề khác liên quan đến khó khăn trong việc quản lý cuộc sống	
Z73.9	Vấn đề liên quan đến khó khăn trong quản lý cuộc sống, không xác định	
Z74	Vấn đề liên quan tình trạng phụ thuộc vào người chăm sóc	
Z74.0	Nhu cầu trợ giúp do giảm khả năng vận động	
Z74.1	Nhu cầu trợ giúp chăm sóc cá nhân	
Z74.2	Nhu cầu trợ giúp chăm sóc tại nhà và trong hộ không có ai có thể chăm sóc	
Z74.3	Nhu cầu theo dõi liên tục	
Z74.8	Vấn đề khác liên quan đến tình trạng phụ thuộc vào người chăm sóc	
Z74.9	Vấn đề liên quan đến tình trạng phụ thuộc vào người chăm sóc, không xác định	
Z75	Vấn đề liên quan đến cơ sở y tế và/hoặc dịch vụ chăm sóc sức khỏe khác	
Z75.0	Dịch vụ y tế không có sẵn tại nhà	
Z75.1	Người đang chờ được chuyển đến [nhập viện tại] cơ sở khác phù hợp	
Z75.2	Giai đoạn chờ đợi khác để khám cận lâm sàng tra và/hoặc điều trị	
Z75.3	Không có sẵn và/hoặc không tiếp cận được cơ sở y tế	
Z75.4	Không có sẵn và/hoặc không tiếp cận được với tổ chức trợ giúp khác	
Z75.5	Chăm sóc hỗ trợ trong kỳ nghỉ (của người thân chăm sóc người bệnh)	Chăm sóc thay thế trong thời gian người chăm sóc chính nghỉ dưỡng
Z75.8	Vấn đề khác liên quan đến cơ sở y tế và/hoặc dịch vụ chăm sóc sức khỏe khác	
Z75.9	Vấn đề liên quan đến cơ sở y tế và/hoặc dịch vụ chăm sóc sức khỏe khác, không xác định	
Z76	Những người tiếp cận dịch vụ y tế trong hoàn cảnh khác	
Z76.0	Chỉ định nhắc lại y lệnh	
Z76.1	Theo dõi sức khỏe và/hoặc chăm sóc sức khỏe trẻ bị bỏ rơi	
Z76.2	Theo dõi sức khỏe và/hoặc chăm sóc sức khỏe trẻ sơ sinh và/hoặc trẻ em khỏe mạnh khác	
Z76.3	Người khỏe mạnh đi cùng với người bệnh	
Z76.4	Người bệnh ở lại cơ sở y tế dù chưa được nhập viện nội trú	
Z76.5	Người giả ốm [cố ý giả vờ]	
Z76.8	Những người tiếp cận cơ sở y tế trong hoàn cảnh xác định khác	
Z76.9	Những người tiếp cận cơ sở y tế trong hoàn cảnh không xác định	
Z80	Tiền sử gia đình có u ác tính	
Z80.0	Tiền sử gia đình có u ác tính ở cơ quan tiêu hóa	Bệnh lý phân loại tại C15.- - C26.
Z80.1	Tiền sử gia đình có u ác tính ở khí quản, phế quản và/hoặc phổi	Bệnh lý phân loại tại C33-C34.
Z80.2	Tiền sử gia đình có u ác tính ở cơ quan hô hấp và/hoặc trong khoang ngực khác	Bệnh lý phân loại tại C30.- - C32.-, C37-C39.
Z80.3	Tiền sử gia đình có u ác tính ở vú	Bệnh lý phân loại tại C50.
Z80.4	Tiền sử gia đình có u ác tính ở cơ quan sinh dục	Bệnh lý phân loại tại C51.- - C63.
Z80.5	Tiền sử gia đình có u ác tính ở đường tiết niệu	Bệnh lý phân loại tại C64-C68.
Z80.6	Tiền sử gia đình có bệnh bạch cầu	Bệnh lý phân loại tại C91.- - C95.
Z80.7	Tiền sử gia đình có u ác tính ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	Bệnh lý phân loại tại C81.- - C90.-, C96.
Z80.8	Tiền sử gia đình có u ác tính ở tạng và/hoặc hệ thống khác	Bệnh lý phân loại tại C00.- - C14.-, C40.- - C49.-, C69.- - C79.-, C97
Z80.9	Tiền sử gia đình có u ác tính, không xác định	Bệnh lý phân loại tại C80.
Z81	Tiền sử gia đình có rối loạn tâm thần và/hoặc hành vi	
Z81.0	Tiền sử gia đình có chậm phát triển trí tuệ	Bệnh lý phân loại tại F70.- - F79.
Z81.1	Tiền sử gia đình có lạm dụng rượu	Bệnh lý phân loại tại F10.
Z81.2	Tiền sử gia đình có lạm dụng thuốc lá	Bệnh lý phân loại tại F17.
Z81.3	Tiền sử gia đình có lạm dụng chất hướng thần khác	Bệnh lý phân loại tại F11.- -F16.-, F18.- - F19.
Z81.4	Tiền sử gia đình có lạm dụng chất khác	Bệnh lý phân loại tại F55
Z81.8	Tiền sử gia đình có rối loạn tâm thần và/hoặc hành vi khác	Bệnh lý phân loại ở những mục khác trong F00.- - F99
Z82	Tiền sử gia đình có một số khuyết tật và bệnh mạn tính dẫn đến tàn tật	
Z82.0	Tiền sử gia đình có động kinh và các bệnh khác của hệ thần kinh	Bệnh lý phân loại tại G00.- - G99.
Z82.1	Tiền sử gia đình có mù loà và/hoặc mất thị giác	Bệnh lý phân loại tại H54.
Z82.2	Tiền sử gia đình có điếc và/hoặc mất thính giác	Bệnh lý phân loại tại H90.- - H91.
Z82.3	Tiền sử gia đình có tai biến mạch máu não	Bệnh lý phân loại tại I60.- - I64
Z82.4	Tiền sử gia đình có bệnh tim thiếu máu cực bộ và/hoặc các bệnh khác của hệ tuần hoàn	Bệnh lý phân loại tại I00-I52.-, I65.- - I99
Z82.5	Tiền sử gia đình có hen phế quản và/hoặc bệnh đường hô hấp dưới mạn tính khác	Bệnh lý phân loại tại J40-J47
Z82.6	Tiền sử gia đình có viêm khớp và các bệnh khác của hệ cơ xương và/hoặc mô liên kết	Bệnh lý phân loại tại M00.- - M99.
Z82.7	Tiền sử gia đình có dị tật hoặc dị dạng bẩm sinh và/hoặc bất thường nhiễm sắc thể	Bệnh lý phân loại tại Q00.- - Q99.
Z82.8	Tiền sử gia đình có khuyết tật và/hoặc bệnh mạn tính khác dẫn đến tàn tật, không phân loại mục khác	
Z83	Tiền sử gia đình có rối loạn xác định cụ thể khác	
Z83.0	Tiền sử gia đình có bệnh suy giảm miễn dịch virus người [HIV]	Bệnh lý phân loại tại B20.- - B24, O98.7
Z83.1	Tiền sử gia đình có bệnh truyền nhiễm và/hoặc ký sinh trùng khác	Bệnh lý phân loại tại A00.- - B19.-, B25.- - B94.-, B99
Z83.2	Tiền sử gia đình có bệnh về máu và/hoặc cơ quan tạo máu và/hoặc một số rối loạn về cơ chế miễn dịch	Bệnh lý phân loại tại D50.- - D89.
Z83.3	Tiền sử gia đình có bệnh đái tháo đường	Bệnh lý phân loại tại E10.- - E14.- , O24
Z83.4	Tiền sử gia đình có bệnh về nội tiết, dinh dưỡng và/hoặc chuyển hóa khác	Bệnh lý phân loại tại E00.- - E07.-, E15-E90
Z83.5	Tiền sử gia đình có rối loạn thị giác và/hoặc thính giác	
Z83.6	Tiền sử gia đình có bệnh của hệ hô hấp	
Z83.7	Tiền sử gia đình có bệnh của hệ tiêu hóa	Bệnh lý phân loại tại K00.- - K93.
Z84	Tiền sử gia đình có bệnh lý khác	
Z84.0	Tiền sử gia đình có bệnh về da và/hoặc mô dưới da	Bệnh lý phân loại tại L00-L99.
Z84.1	Tiền sử gia đình có rối loạn về thận và/hoặc niệu quản	Bệnh lý phân loại tại N00.- - N29.
Z84.2	Tiền sử gia đình có bệnh của hệ sinh dục tiết niệu	Bệnh lý phân loại tại N30.- - N99.
Z84.3	Tiền sử gia đình có bệnh về huyết thống	
Z84.8	Tiền sử gia đình có bệnh lý xác định khác	
Z85	Tiền sử cá nhân có u ác tính	
Z85.0	Tiền sử cá nhân có u ác tính của cơ quan tiêu hóa	Bệnh lý phân loại tại C15.- - C26.
Z85.1	Tiền sử cá nhân có u ác tính ở khí quản, phế quản và/hoặc phổi	Bệnh lý phân loại tại C33-C34.
Z85.2	Tiền sử cá nhân có u ác tính ở cơ quan hô hấp và/hoặc cơ quan khác trong khoang ngực	Bệnh phân loại tại C30.- - C32.-, C37-C39.
Z85.3	Tiền sử cá nhân có u ác tính ở vú	Bệnh lý phân loại tại C50.
Z85.4	Tiền sử cá nhân có u ác tính ở cơ quan sinh dục	Bệnh lý phân loại tại C51.- - C63.
Z85.5	Tiền sử cá nhân có u ác tính của đường tiết niệu	Bệnh lý phân loại tại C64-C68.
Z85.6	Tiền sử cá nhân có bệnh bạch cầu	Bệnh lý phân loại tại C91.- - C95.
Z85.7	Tiền sử cá nhân có u ác tính ở mô hệ lympho, cơ quan tạo máu và/hoặc mô liên quan	Bệnh lý phân loại tại C81.- - C90.-, C96.
Z85.8	Tiền sử cá nhân có u ác tính ở cơ quan và/hoặc hệ thống khác	Bệnh lý phân loại tại C00.- - C14.-, C40.- - C49.-, C69.- - C79.-, C97
Z85.9	Tiền sử cá nhân có u ác tính, không xác định	Bệnh lý phân loại tại C80.
Z86	Tiền sử cá nhân có một số bệnh khác	
Z86.0	Tiền sử cá nhân có u tân sinh khác	
Z86.1	Tiền sử cá nhân có bệnh nhiễm trùng và/hoặc ký sinh trùng	
Z86.2	Tiền sử cá nhân có bệnh về máu và/hoặc cơ quan tạo máu và/hoặc một số rối loạn về cơ chế miễn dịch	Bệnh lý phân loại tại D50.- - D89.
Z86.3	Tiền sử cá nhân có bệnh về nội tiết, dinh dưỡng và/hoặc chuyển hóa	Bệnh lý phân loại tại E00.- - E90
Z86.4	Tiền sử cá nhân có lạm dụng chất hướng thần	
Z86.5	Tiền sử cá nhân có rối loạn tâm thần và/hoặc hành vi khác	Bệnh lý phân loại tại F00.- - F09.-, F20.- - F99
Z86.6	Tiền sử cá nhân có bệnh của hệ thần kinh và/hoặc cơ quan cảm nhận	Bệnh lý phân loại tại G00.- - G99.-, H00.- - H95.
Z86.7	Tiền sử cá nhân có bệnh của hệ tuần hoàn	
Z87	Tiền sử cá nhân có bệnh và/hoặc bệnh lý khác	
Z87.0	Tiền sử cá nhân có bệnh của hệ hô hấp	Bệnh lý phân loại tại J00 - J99.
Z87.1	Tiền sử cá nhân có các bệnh của hệ tiêu hóa	Bệnh lý phân loại tại K00.- - K93.
Z87.2	Tiền sử cá nhân có bệnh về da và/hoặc mô dưới da	Bệnh lý phân loại tại L00-L99.
Z87.3	Tiền sử cá nhân có bệnh của hệ cơ xương khớp và/hoặc mô liên kết	Bệnh lý phân loại tại M00.- - M99.
Z87.4	Tiền sử cá nhân có bệnh của hệ sinh dục tiết niệu	Bệnh lý phân loại tại N00.- - N99.
Z87.5	Tiền sử cá nhân có các biến chứng của thai kỳ, sinh đẻ và/hoặc thời kỳ sau đẻ	
Z87.6	Tiền sử cá nhân có một số bệnh phát sinh trong thời kỳ chu sinh	Bệnh lý phân loại tại P00.- - P96.
Z87.7	Tiền sử cá nhân có dị tật hoặc dị dạng bẩm sinh và/hoặc bất thường nhiễm sắc thể	Bệnh lý phân loại tại Q00.- - Q99.
Z87.8	Tiền sử cá nhân có bệnh lý xác định khác	
Z88	Tiền sử cá nhân có dị ứng với dược chất, thuốc điều trị và/hoặc sinh phẩm	
Z88.0	Tiền sử cá nhân dị ứng với penicillin	
Z88.1	Tiền sử cá nhân dị ứng với tác nhân kháng sinh khác	
Z88.2	Tiền sử cá nhân dị ứng với kháng sinh họ sulfonamides	
Z88.3	Tiền sử cá nhân dị ứng với các tác nhân chống nhiễm trùng khác	
Z88.4	Tiền sử cá nhân dị ứng với tác nhân gây mê	
Z88.5	Tiền sử cá nhân dị ứng với tác nhân ma túy	
Z88.6	Tiền sử cá nhân dị ứng với các tác nhân giảm đau	
Z88.7	Tiền sử cá nhân dị ứng với huyết thanh và/hoặc vắc xin	
Z88.8	Tiền sử cá nhân dị ứng với dược chất, thuốc điều trị và/hoặc sinh phẩm khác	
Z88.9	Tiền sử cá nhân dị ứng với dược chất, thuốc điều trị và/hoặc sinh phẩm không xác định	
Z89	Mất chi mắc phải	
Z89.0	Mất nhiều ngón tay mắc phải [bao gồm ngón cái], một bên	
Z89.1	Mất bàn tay và/hoặc cổ tay mắc phải	
Z89.2	Mất chi trên từ trên cổ tay mắc phải	Cánh tay không xác định khác
Z89.3	Mất cả hai chi trên mắc phải [bất kỳ tầm nào]	Mất ngón tay mắc phải, hai bên
Z89.4	Mất bàn chân và/hoặc mắt cá chân mắc phải	Ngón chân
Z89.5	Mất cẳng chân từ khớp gối hoặc dưới mắc phải	
Z89.6	Mất cẳng chân từ trên khớp gối mắc phải	Chân không xác định khác
Z89.7	Mất cả hai chi dưới mắc phải [bất kỳ tầm nào, trừ riêng các ngón chân]	
Z89.8	Mất cả chi trên và/hoặc chi dưới mắc phải [bất kỳ tầm nào]	
Z89.9	Mất chi mắc phải, không xác định	
Z90	Mất tạng [cơ quan] mắc phải, không phân loại mục khác	
Z90.0	Mất một phần của đầu và/hoặc cổ mắc phải	
Z90.1	Mất (một hoặc hai bên) vú mắc phải	
Z90.2	Mất [một phần] phổi mắc phải	
Z90.3	Mất một phần dạ dày mắc phải	
Z90.4	Mất các phần khác của đường tiêu hóa mắc phải	
Z90.5	Mất thận mắc phải	
Z90.6	Mất phần khác của đường tiết niệu mắc phải	
Z90.7	Mất (nhiều) cơ quan sinh dục mắc phải	
Z90.8	Mất cơ quan khác mắc phải	
Z91	Tiền sử cá nhân có yếu tố nguy cơ, không phân loại mục khác	
Z91.0	Tiền sử cá nhân có dị ứng, ngoài dị ứng thuốc hoặc sinh phẩm	
Z91.1	Tiền sử cá nhân không tuân thủ chỉ định và/hoặc chế độ điều trị	
Z91.2	Tiền sử cá nhân vệ sinh cá nhân kém	
Z91.3	Tiền sử cá nhân về lịch trình ngủ - thức không lành mạnh	
Z91.4	Tiền sử cá nhân có sang chấn tâm lý, không phân loại mục khác	
Z91.5	Tiền sử cá nhân tự làm hại bản thân	Tự sát không thành|Tự đầu độc|Cố gắng tự tử
Z91.6	Tiền sử cá nhân có chấn thương thực thể khác	
Z91.7	Tiền sử cá nhân đã bị cắt âm vật	Cắt âm vật cho nữ|Cắt bộ phận sinh dục nữ|FGM các loại 1-4
Z91.8	Tiền sử cá nhân có yếu tố nguy cơ khác, không phân loại mục khác	Lạm dụng không xác định khác|Ngược đãi không xác định khác
Z92	Tiền sử cá nhân liên quan can thiệp y tế	
Z92.0	Tiền sử cá nhân về can thiệp tránh thai	
Z92.1	Tiền sử cá nhân (đang) sử dụng dài ngày chất chống đông máu	
Z92.2	Tiền sử cá nhân sử dụng dài ngày (hoặc hiện đang dùng) thuốc điều trị	Aspirin
Z92.3	Tiền sử cá nhân có điều trị bằng xạ trị	
Z92.4	Tiền sử cá nhân có phẫu thuật lớn, không phân loại mục khác	
Z92.5	Tiền sử cá nhân có can thiệp phục hồi chức năng	
Z92.6	Tiền sử cá nhân có hóa trị liệu bệnh u tân sinh	
Z92.8	Tiền sử cá nhân có can tiệp y tế khác	
Z92.9	Tiền sử cá nhân có can thiệp y tế, không xác định	
Z93	Tình trạng lỗ mở nhân tạo	
Z93.0	Tình trạng lỗ mở khí quản	
Z93.1	Tình trạng lỗ mở thông dạ dày	
Z93.2	Tình trạng lỗ mở thông hồi tràng [hậu môn nhân tạo]	
Z93.3	Tình trạng lỗ mở thông đại tràng [hậu môn nhân tạo]	
Z93.4	Tình trạng lỗ mở nhân tạo khác của dạ dày - ruột	
Z93.5	Tình trạng có ống dẫn lưu bàng quang trên xương mu	
Z93.6	Tình trạng có ống dẫn lưu đường tiết niệu qua da khác	Ống dẫn lưu bể thận|Ống dẫn lưu niệu quản|Ống dẫn lưu niệu đạo
Z93.8	Tình trạng lỗ mở nhân tạo khác	
Z93.9	Tình trạng lỗ mở nhân tạo, không xác định	
Z94	Tình trạng tạng và/hoặc mô được cấy ghép	
Z94.0	Tình trạng ghép thận	
Z94.1	Tình trạng ghép tim	
Z94.2	Tình trạng ghép phổi	
Z94.3	Tình trạng ghép cả tim và phổi	
Z94.4	Tình trạng ghép gan	
Z94.5	Tình trạng ghép da	Tình trạng ghép da tự thân
Z94.6	Tình trạng ghép xương	
Z94.7	Tình trạng ghép giác mạc	
Z94.8	Tình trạng tạng và/hoặc mô khác được cấy ghép	Tủy xương|Ruột|Tuyến tụy|Tế bào gốc
Z94.9	Tình trạng tạng và/hoặc mô được cấy ghép, không xác định	
Z95	Sự có mặt thiết bị/dụng cụ cấy ghép tim và/hoặc mạch máu	
Z95.0	Sự có mặt của thiết bị điện tử ở tim	
Z95.1	Sự có mặt của ghép bắc cầu động mạch vành	
Z95.2	Sự có mặt của van tim nhân tạo cơ học	
Z95.3	Sự có mặt của van tim nhân tạo sinh học	
Z95.4	Sự có mặt của van tim thay thế khác	
Z95.5	Sự có mặt của thiết bị/dụng cụ được cấy ghép trong can thiệp mạch vành	Sự có mặt của stent động mạch vành|Tình trạng sau nong mạch vành không xác định khác
Z95.8	Sự có mặt của thiết bị/dụng cụ tim và/hoặc mạch máu khác được cấy ghép	
Z95.9	Sự có mặt của thiết bị/dụng cụ tim và/hoặc mạch máu được cấy ghép, không xác định	
Z96	Sự có mặt của thiết bị/dụng cụ cấy ghép chức năng khác	
Z96.0	Sự có mặt thiết bị/dụng cụ cấy ghép tiết niệu sinh dục	
Z96.1	Sự có mặt của thấu kính nội nhãn	Thủy tinh thể nhân tạo
Z96.2	Sự có mặt của thiết bị/dụng cụ cấy ghép tai và/hoặc thính giác	Thiết bị thính giác dẫn truyền qua xương|Thiết bị cấy ốc tai|Khung đỡ vòi nhĩ|Ống thông khí màng nhĩ|Xương bàn đạp thay thế
Z96.3	Sự có mặt của thanh quản nhân tạo	
Z96.4	Sự có mặt của thiết bị/dụng cụ cấy ghép nội tiết	Bơm insulin
Z96.5	Sự có mặt của chân [trụ] răng và/hoặc thiết bị/dụng cụ tạo hình hàm được cấy ghép	
Z96.6	Sự có mặt của thiết bị/dụng cụ chỉnh hình khớp	
Z96.7	Sự có mặt của mô xương và/hoặc gân được ghép	Bản xương sọ [mảnh ghép sọ nhân tạo]
Z96.8	Sự có mặt của thiết bị/dụng cụ cấy ghép chức năng xác định khác	
Z96.9	Sự có mặt của thiết bị/dụng cụ cấy ghép chức năng, không xác định	
Z97	Sự có mặt thiết bị/dụng cụ cấy ghép khác	
Z97.0	Sự có mặt mắt giả	
Z97.1	Sự có mặt của chi giả (toàn bộ) (một phần)	
Z97.2	Sự có mặt của thiết bị/dụng cụ phục hình răng (toàn bộ) (một phần)	
Z97.3	Sự có mặt của kính thuốc và/hoặc kính áp tròng	
Z97.4	Sự có mặt của máy trợ thính bên ngoài	
Z97.5	Sự có mặt dụng cụ tránh thai (trong tử cung)	
Z97.8	Sự có mặt thiết bị/dụng cụ xác định khác	
Z98	Tình trạng khác sau can thiệp	
Z98.0	Tình trạng nối tắt ruột và/hoặc nối ruột	
Z98.1	Tình trạng cố định khớp	
Z98.2	Sự có mặt của thiết bị/dụng cụ dẫn lưu dịch não tủy	Ống dẫn lưu dịch não tủy
Z98.8	Tình trạng sau can thiệp xác định khác	
Z99	Tình trạng phụ thuộc vào thiết bị và/hoặc máy móc hỗ trợ, không phân loại mục khác	
Z99.0	Tình trạng phụ thuộc vào máy hút dịch	
Z99.1	Tình trạng phụ thuộc vào máy thở	Phụ thuộc thiết bị thông khí nhân tạo
Z99.2	Tình trạng phụ thuộc phụ thuộc vào lọc máu nhân tạo [thận nhân tạo]	
Z99.3	Tình trạng phụ thuộc vào xe lăn	
Z99.4	Tình trạng phụ thuộc vào tim nhân tạo	
Z99.8	Tình trạng phụ thuộc vào máy móc và/hoặc thiết bị hỗ trợ khác	
Z99.9	Tình trạng phụ thuộc vào thiết bị và/hoặc máy móc hỗ trợ không xác định	

In [ ]:
rows = ICD_TSV.read_text(encoding="utf-8").splitlines()
print(f"ICD KB: {len(rows) - 1:,} mã -> {ICD_TSV} ({ICD_TSV.stat().st_size:,} bytes)")
print("ví dụ:", rows[1])

In [ ]:
from medical_coder.rxnorm_kb import build as build_rxnorm

RX_TSV = TERM_DIR / "rxnorm.tsv"
RX_URL = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_07062026.zip"

def find_rxnorm_archive():
    for base in (Path("/kaggle/input"), WORK):
        if base.exists():
            for path in base.rglob("RxNorm_full_prescribe_*.zip"):
                return path
    return None

if RX_TSV.exists():
    print("dùng RxNorm TSV có sẵn:", RX_TSV)
else:
    archive = find_rxnorm_archive()
    if archive is None:
        archive = WORK / "rxnorm.zip"
        print("tải RxNorm …")
        rc = subprocess.run(["curl", "-sSL", "--max-time", "600", "-o", str(archive), RX_URL],
                            check=False).returncode
        if rc != 0 or not archive.exists() or archive.stat().st_size < 10_000_000:
            print("CẢNH BÁO: tải RxNorm thất bại — candidates THUỐC sẽ rỗng")
            archive = None
    if archive is not None:
        print("dựng RxNorm KB từ:", archive)
        print("số RxCUI:", build_rxnorm(archive, RX_TSV))
    else:
        RX_TSV = None

## 6. Weights

| Model | Vai trò | Tham số | Đĩa (bf16) |
|---|---|---:|---:|
| `urchade/gliner_multi-v2.1` | NER | 0.289B | ~1.2 GB |
| `Qwen/Qwen3-4B-Instruct-2507` | corrector | 4.022B | ~8.0 GB |
| `Qwen/Qwen3.5-4B` | teacher phụ (additions) | 4.206B | ~8.4 GB |

Tham số thì cả ba cộng lại là 8.517B, vẫn dưới 9B. Nhưng **đĩa mới là ràng buộc
thật**: cả ba là ~17.6 GB, trong khi `/kaggle/working` chỉ có ~20 GB và còn phải
chứa torch, output và cache. Đó chính là nguyên nhân `Errno 28`.

Nên **mặc định chỉ tải teacher chính** (~9.2 GB tổng cộng): corrector chạy, chỉ
bỏ bước additions. Đây cũng là đánh đổi hợp lý — corrector sửa type cho span đã
có, còn additions chỉ thêm span cho các type không mang candidate.

Muốn bật additions thì attach cả hai Qwen dưới dạng **Kaggle Dataset**: đọc từ
`/kaggle/input` là read-only, không tính vào quota `/kaggle/working`. Khi đó đặt
`SECONDARY = "Qwen/Qwen3.5-4B"` và cell dưới sẽ tự tìm thấy.

In [ ]:
GLINER_MODEL = "urchade/gliner_multi-v2.1"
PRIMARY   = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY = None    # đặt tên repo để bật additions — CHỈ nên làm khi đã attach Dataset

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for name in ("HF_TOKEN", "HF_KEY", "HUGGINGFACE_TOKEN", "HUGGINGFACE_KEY"):
        try:
            TOKEN = UserSecretsClient().get_secret(name)
            if TOKEN:
                print("dùng secret:", name)
                break
        except Exception:
            continue
except ImportError:
    pass

# Tải vào cache rồi dùng thẳng đường dẫn cache trả về. Dùng local_dir= sẽ giữ
# thêm một bản trong cache nữa, tức gấp đôi đĩa cho cùng một model.
HF_CACHE = WORK / "hf"
HF_CACHE.mkdir(parents=True, exist_ok=True)

def find_attached_model(repo_id):
    leaf = repo_id.split("/")[-1]
    base = Path("/kaggle/input")
    if base.exists():
        for path in base.rglob(leaf):
            if path.is_dir() and (path / "config.json").exists():
                return str(path)
    return None

def resolve_model(repo_id, need_gb):
    attached = find_attached_model(repo_id)
    if attached:
        print(f"  {repo_id}: dùng Dataset đã attach (không tốn quota)")
        return attached
    free_gb = _sh.disk_usage("/kaggle/working").free / 2**30
    if free_gb < need_gb + 2:
        raise RuntimeError(
            f"{repo_id} cần ~{need_gb} GB nhưng chỉ còn {free_gb:.1f} GB trống. "
            "Factory reset session, hoặc attach model này dưới dạng Dataset."
        )
    from huggingface_hub import snapshot_download
    print(f"  {repo_id}: tải về (~{need_gb} GB, còn trống {free_gb:.1f} GB) …")
    return snapshot_download(
        repo_id=repo_id, cache_dir=str(HF_CACHE), token=TOKEN,
        ignore_patterns=["*.pth", "*.onnx", "*.msgpack", "*.h5", "*.gguf"],
    )

GLINER_PATH = resolve_model(GLINER_MODEL, 1.2)
PRIMARY_PATH = resolve_model(PRIMARY, 8.0)

SECONDARY_PATH = None
if SECONDARY:
    try:
        SECONDARY_PATH = resolve_model(SECONDARY, 8.4)
    except Exception as exc:
        print(f"  bỏ qua teacher phụ: {exc}")

print()
print("gliner   :", GLINER_PATH)
print("primary  :", PRIMARY_PATH)
print("secondary:", SECONDARY_PATH or "(không có — chỉ chạy corrector, bỏ additions)")
print()
report_disk()

In [ ]:
# Sau bước provision, khoá offline: inference hoàn toàn self-host, không gọi API.
TOKEN = None
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("đã khoá chế độ offline")

## 7. Cấu hình

In [ ]:
import torch
from medical_coder.gliner_ner import DEFAULT_THRESHOLDS
from medical_coder.models import EntityType
from medical_coder.pipeline_v2 import PipelineV2Config, run_pipeline_v2

HAS_CUDA = torch.cuda.is_available()
NGPU = torch.cuda.device_count() if HAS_CUDA else 0

OUTPUT_DIR = WORK / "output"
ZIP_PATH   = WORK / "output.zip"

# Ngưỡng theo từng type. GLiNER có phân bố score khác nhau theo label nên một
# ngưỡng chung là sai; các giá trị này lấy từ lời giải tham chiếu 27.8786.
# Giữ ngưỡng gốc. Sàn 0.30 đã được thử và LỖ (27.5217 -> 26.8959): dải 0.15-0.30
# chỉ chứa ~57% rác, dưới vạch hoà vốn 60.8%. Đo đó cho thấy điểm tin cậy của
# GLiNER gần như không phân biệt được đúng/sai — precision tổng thể cũng chỉ
# 50-64% — nên vặn ngưỡng là ngõ cụt theo cả hai hướng.
THRESHOLDS = dict(DEFAULT_THRESHOLDS)
for k, v in THRESHOLDS.items():
    print(f"  {k.value:22s} {v}")

CONFIG = dict(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_path=GLINER_PATH,
    device="cuda" if HAS_CUDA else "cpu",
    icd_kb=ICD_TSV,
    rxnorm_kb=RX_TSV,
    thresholds=THRESHOLDS,
    max_candidates=1,          # >1 làm phình mẫu số candidate
    primary_teacher=PRIMARY_PATH if HAS_CUDA else None,
    secondary_teacher=SECONDARY_PATH if HAS_CUDA else None,
    teacher_device="cuda:0" if HAS_CUDA else "cpu",
    teacher_quantization="4bit",
    teacher_batch_size=48 if NGPU else 8,
    # Bộ loại span: hỏi teacher xem span sắp emit có thực sự là khái niệm y khoa
    # không; đặt None để tắt. GIỮ NGUYÊN 1.0 — đã đo được +0.38 điểm ở giá trị
    # này (bỏ 131 span, precision 65-69%, hoà vốn 60.8%). Lượt này chỉ đổi ngưỡng
    # GLiNER, nên đừng đổi thêm margin: hai thay đổi cùng lúc thì không quy được
    # nguyên nhân. Bảng hiệu chuẩn in cuối lượt chạy để chọn margin cho lượt sau.
    reject_margin=1.0 if HAS_CUDA else None,
)
print("\nGPU:", NGPU, "| corrector:", bool(CONFIG["primary_teacher"]),
      "| additions:", bool(CONFIG["secondary_teacher"]))

## 8. Smoke test (2 bản ghi)

Chạy thử trước khi chạy đủ 100 để bắt lỗi cấu hình sớm. Bước này **không** tạo ZIP.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

SMOKE_DIR = WORK / "output_smoke"
smoke = run_pipeline_v2(PipelineV2Config(**{**CONFIG, "output_dir": SMOKE_DIR,
                                            "selected_ids": frozenset({"1", "2"})}))
print("\nsmoke concepts:", smoke)

for stem in ("1", "2"):
    data = json.loads((SMOKE_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    raw = (INPUT_DIR / f"{stem}.txt").read_text(encoding="utf-8")
    assert all(raw[c["position"][0]:c["position"][1]] == c["text"] for c in data), "offset sai"
    print(f"\n--- {stem}.json ({len(data)} concept) ---")
    for c in data[:6]:
        print(f"  {c['position']} {c['type']:20s} {c['text'][:44]!r} {c.get('candidates', '')}")
print("\noffset khớp nguyên văn trên cả hai bản ghi")

## 9. Chạy đủ 100 bản ghi

Cuối lượt chạy, log in ra bảng hiệu chuẩn bộ loại:

```text
rejector: đã chấm 2497 span
  margin  bỏ đi   tỉ lệ
    -1.0     ...
     0.0     ...
     1.0     ...
```

Bảng này cho biết mỗi ngưỡng sẽ bỏ bao nhiêu span trên toàn corpus, nhờ đó chọn
`reject_margin` cho lần nộp sau bằng dữ liệu thay vì đoán — một lượt chạy GPU cho
cả đường cong thay vì một điểm.

Nó **cũng được ghi ra** `/kaggle/working/rejection_stats.json`, kèm margin thô của
từng span. Tải tệp đó về ở mục 11: log của một session mất là mất luôn, mà chạy
lại tốn hàng chục phút GPU. Tệp này nằm ngoài `output/` nên không lọt vào ZIP.

In [ ]:
import time

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

start = time.time()
total = run_pipeline_v2(PipelineV2Config(**CONFIG))
print(f"\n{total} concept trong {time.time() - start:.0f}s")

## 10. Kiểm tra và đóng gói

In [ ]:
from medical_coder.submission import create_submission_zip, validate_all

validate_all(INPUT_DIR, OUTPUT_DIR)
print("validator: PASS")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
create_submission_zip(OUTPUT_DIR, ZIP_PATH)

import zipfile, hashlib
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
    assert names == [f"output/{i}.json" for i in range(1, 101)], "cấu trúc ZIP sai"
    assert archive.testzip() is None, "ZIP hỏng"
print(f"ZIP OK: {len(names)} tệp, {ZIP_PATH.stat().st_size:,} bytes")
print("sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())

In [ ]:
from collections import Counter

records = {p.stem: json.loads(p.read_text(encoding="utf-8"))
           for p in OUTPUT_DIR.glob("*.json") if p.stem.isdigit()}
concepts = [c for v in records.values() for c in v]
types = Counter(c["type"] for c in concepts)
with_codes = [c for c in concepts if c.get("candidates")]

print(f"tổng concept        : {len(concepts)}")
print(f"trung bình / bản ghi: {len(concepts) / len(records):.2f}")
print(f"bản ghi rỗng        : {[k for k, v in records.items() if not v]}")
for k in ("TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"):
    print(f"  {k:22s} {types.get(k, 0)}")
print(f"concept có candidate: {len(with_codes)}")
print(f"tổng mã xuất ra     : {sum(len(c['candidates']) for c in with_codes)}")
print(f"nhãn assertion      : {sum(len(c['assertions']) for c in concepts)} (chủ ý để rỗng)")

## 11. Tải kết quả

`/kaggle/working/output.zip` — nộp trực tiếp tệp này.

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(ZIP_PATH)))

# Hiệu chuẩn bộ loại span. Không nằm trong ZIP nộp bài; tải riêng về để chọn
# reject_margin cho lượt sau mà không phải chạy lại GPU. Chứa cả margin thô của
# từng span, nên mọi ngưỡng đều đánh giá lại được offline.
STATS_PATH = WORK / "rejection_stats.json"
if STATS_PATH.exists():
    print(f"hiệu chuẩn bộ loại: {STATS_PATH} ({STATS_PATH.stat().st_size:,} bytes)")
    display(FileLink(str(STATS_PATH)))
else:
    print("không có rejection_stats.json — bộ loại không chạy (thiếu GPU?)")

## 12. Ghi chú

**Đã cố ý bỏ:**

* **Assertions để rỗng.** Lời giải tham chiếu đo được `isNegated` tách biệt ở AUC
  0.497 (ngang ngẫu nhiên); mọi rule đều emit thừa. Một assertion sai mất trọn
  Jaccard của concept đó, trong khi dự đoán rỗng đúng với ground truth rỗng được
  1.0. Chỉ nên bật lại khi đã có dữ liệu gán nhãn.
* **Candidate tối đa 1 và chỉ khi alias khớp duy nhất.** Bỏ toàn bộ candidate chỉ
  làm candidate Jaccard của họ giảm 0.0036 — thành phần 40% này gần như hoàn toàn
  do chất lượng khớp concept quyết định, không phải do tra đúng mã.
* **Additions chỉ cho type không có candidate.** Thêm nhầm một CHẨN_ĐOÁN/THUỐC
  còn bị tính vào mẫu số candidate.

**Chưa kiểm chứng:** chưa có ground truth nên chưa đo được điểm cục bộ. Sau khi
gán nhãn 15–20 bản ghi, dùng:

```bash
medical-coder score --output-dir output --truth-dir data/labelled --per-record
```

Scorer tự chấm ground truth bằng 1.0 và tái lập đúng cả hai mốc điểm đã công bố
(14.4255 và 27.8786).